# Adaptive Signals · Hat Yai

Run the first two code cells on a CPU runtime. Select **Start / reset run** in the second cell's output to start SUMO. The controls, road network, simulated vehicles and signal states are displayed inside Colab.

The bundled OpenStreetMap extract covers 1,054 directed road edges and seven signal-controller groups in Hat Yai. Traffic demand is simulated. The lane, turn and signal configurations are for simulation and require field validation before operational use.

Map data © OpenStreetMap contributors, ODbL 1.0.


In [ ]:
# Cell 1 — bundled real-road model and source, pinned dependencies, validation.
import os,sys,subprocess,shutil,time,json,urllib.request,zipfile
from pathlib import Path
if "LIVE_PROCESS" in globals() and LIVE_PROCESS.poll() is None:
    LIVE_PROCESS.terminate();LIVE_PROCESS.wait(timeout=10)
BASE=Path(os.environ.get("ADAPTIVE_SIGNALS_HOME","/content" if Path("/content").exists() else str(Path.cwd())))
PROJECT=BASE/"AdaptiveSignals_Stage3_RealRoad"
PROJECT.mkdir(parents=True,exist_ok=True)
PAYLOAD={'experiment.py': '"""Reproducible SUMO pilot: synthetic left-hand grid, C++ control kernels.\n\nNo RL training, real CCTV, calibrated Thai demand, or deployment claims.\nRun `python experiment.py --help` for smoke, pilot and extended campaigns.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport bisect\nimport csv\nimport ctypes as ct\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport random\nimport statistics\nimport subprocess\nimport sys\nimport time\nimport xml.etree.ElementTree as ET\nfrom collections import defaultdict, deque\n\nimport numpy as np\nimport sumolib\nimport libsumo as sim\n\nROOT = Path(__file__).resolve().parent\nCONTROLLERS = [\'fixed\', \'queue_actuated\', \'max_pressure\', \'receiving_aware\', \'coordinated\']\nMODE = {\'queue_actuated\':0,\'max_pressure\':1,\'receiving_aware\':2,\'coordinated\':3,\n        \'calibrated_pressure\':1,\'message_only\':3,\'moving_only\':3,\'predictive\':3}\nSCENARIOS = [\'balanced\', \'directional_switch\', \'surge\']\nSETTINGS = {\n    \'version\': \'stage3-v1-realroad\', \'grid\': 3, \'spacing_m\': 160, \'boundary_m\': 220,\n    \'phase_layout\': \'incoming\',\n    \'step_s\': 1, \'minimum_green_s\': 10, \'maximum_green_s\': 50,\n    \'yellow_s\': 3, \'all_red_s\': 2, \'decision_interval_s\': 5, \'maximum_red_target_s\':180,\n    \'hysteresis_vehicles\': 2.0, \'vehicle_length_m\': 5, \'min_gap_m\': 2.5,\n    \'observation_sample_s\': 5, \'observation_delay_s\': 5,\n    \'dropout_probability\': 0.10, \'outage_fraction\': [0.40, 0.55],\n    \'coordination_weight\': 0.30, \'moving_weight\': 0.20,\n    \'fixed_candidates_s\': [20, 30, 45], \'development_seed\': 101,\n    \'evaluation_seeds\': [4777, 5777, 6777],\n}\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    temp.write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\n\')\n    temp.replace(path)\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef binary(name):\n    import sumo\n    candidate = Path(sumo.__file__).parent / \'bin\' / name\n    return str(candidate) if candidate.exists() else sumolib.checkBinary(name)\n\n\nclass Kernel:\n    def __init__(self, config=None):\n        self.config=config or {\'horizon_s\':20,\'receiving_power\':1,\'message_weight\':0.3,\'moving_weight\':0.2}\n        source = ROOT / \'src/controllers.cpp\'\n        library = ROOT / \'build/controllers.so\'\n        library.parent.mkdir(exist_ok=True)\n        if not library.exists() or library.stat().st_mtime < source.stat().st_mtime:\n            subprocess.run([\'g++\', \'-std=c++17\', \'-O3\', \'-Wall\', \'-Wextra\',\n                            \'-Werror\', \'-shared\', \'-fPIC\', str(source), \'-o\', str(library)], check=True)\n        self.lib = ct.CDLL(str(library))\n        ptr = ct.POINTER(ct.c_double)\n        self.lib.phase_score.argtypes = [ct.c_int, ct.c_int] + [ptr]*5\n        self.lib.phase_score.restype = ct.c_double\n        self.lib.should_switch.argtypes = [ct.c_double]*6\n        self.lib.should_switch.restype = ct.c_int\n        self.lib.discharge_score.argtypes=[ct.c_int]+[ptr]*5+[ct.c_double]*5\n        self.lib.discharge_score.restype=ct.c_double\n\n    def score(self, mode, rows):\n        if not rows:\n            return 0.0\n        arrays = [np.ascontiguousarray(a, dtype=np.float64) for a in zip(*rows)]\n        return self.lib.phase_score(mode, len(rows), *[\n            a.ctypes.data_as(ct.POINTER(ct.c_double)) for a in arrays])\n\n    def switch(self, current, next_, elapsed):\n        return bool(self.lib.should_switch(current, next_, elapsed,\n            SETTINGS[\'minimum_green_s\'], SETTINGS[\'maximum_green_s\'],\n            SETTINGS[\'hysteresis_vehicles\']))\n\n    def predictive(self,rows,lost_time):\n        if not rows:return 0.0\n        arrays=[np.ascontiguousarray(a,dtype=np.float64) for a in zip(*rows)]\n        c=self.config\n        return self.lib.discharge_score(len(rows),*[a.ctypes.data_as(ct.POINTER(ct.c_double)) for a in arrays],\n            c[\'horizon_s\'],lost_time,c[\'receiving_power\'],c[\'message_weight\'],c[\'moving_weight\'])\n\n\ndef build_network(folder):\n    folder.mkdir(parents=True, exist_ok=True)\n    netfile = folder / \'grid.net.xml\'\n    if not netfile.exists():\n        command = [binary(\'netgenerate\'), \'--grid\', \'--grid.number\', str(SETTINGS[\'grid\']),\n            \'--grid.length\', str(SETTINGS[\'spacing_m\']), \'--grid.attach-length\', str(SETTINGS[\'boundary_m\']),\n            \'--default-junction-type\', \'traffic_light\', \'--default.lanenumber\', \'1\',\n            \'--default.speed\', \'13.89\', \'--no-turnarounds\', \'--lefthand\',\n            \'--tls.layout\', SETTINGS[\'phase_layout\'],\n            \'--tls.yellow.time\', str(SETTINGS[\'yellow_s\']),\n            \'--tls.allred.time\', str(SETTINGS[\'all_red_s\']),\n            \'--tls.green.time\', \'30\', \'--output-file\', str(netfile)]\n        p = subprocess.run(command, text=True, capture_output=True, check=True)\n        (folder / \'netgenerate.log\').write_text(p.stdout + p.stderr)\n    net = sumolib.net.readNet(str(netfile))\n    xml = ET.parse(netfile).getroot()\n    tls = {}\n    for logic in xml.findall(\'tlLogic\'):\n        phases = [{\'state\': p.attrib[\'state\'], \'duration\': float(p.attrib[\'duration\'])}\n                  for p in logic.findall(\'phase\')]\n        greens = [i for i,p in enumerate(phases) if any(c in \'Gg\' for c in p[\'state\'])]\n        if len(greens) != 4:\n            raise ValueError(\'Stage 1 requires four protected incoming-approach stages\')\n        assert not any(\'g\' in p[\'state\'] for p in phases), \'Unexpected permissive green\'\n        for i in greens:\n            yellow = phases[(i+1)%len(phases)]\n            red = phases[(i+2)%len(phases)]\n            assert \'y\' in yellow[\'state\'] and yellow[\'duration\'] == SETTINGS[\'yellow_s\']\n            assert set(red[\'state\']) == {\'r\'} and red[\'duration\'] == SETTINGS[\'all_red_s\']\n        tls[logic.attrib[\'id\']] = {\'phases\': phases, \'greens\': greens}\n    assert len(tls) == SETTINGS[\'grid\']**2\n    return netfile, net, tls\n\n\ndef generate_demand(folder, net, scenario, seed, demand_s, scale=1.0):\n    """Explicit immutable vehicle requests; routes and departures shared by controllers."""\n    rng = random.Random(seed)\n    entries = [e for e in net.getEdges() if len(e.getFromNode().getIncoming()) == 1]\n    entries.sort(key=lambda e:e.getID())\n    routes = []\n    root = ET.Element(\'routes\')\n    ET.SubElement(root, \'vType\', id=\'car\', length=\'5\', minGap=\'2.5\',\n                  accel=\'2.6\', decel=\'4.5\', sigma=\'0\', speedFactor=\'1.0\', maxSpeed=\'13.89\')\n    requests = []\n    for origin in entries:\n        x,y = origin.getFromNode().getCoord()\n        # Opposite boundary with turns through different streets where feasible.\n        if origin.getID().startswith(\'left\'): choices = [e for e in entries if e.getID().startswith(\'right\')]\n        elif origin.getID().startswith(\'right\'): choices = [e for e in entries if e.getID().startswith(\'left\')]\n        elif origin.getID().startswith(\'bottom\'): choices = [e for e in entries if e.getID().startswith(\'top\')]\n        else: choices = [e for e in entries if e.getID().startswith(\'bottom\')]\n        for dest_entry in choices:\n            exits = dest_entry.getFromNode().getIncoming()\n            target = exits[0]\n            path, _ = net.getShortestPath(origin, target)\n            if not path:\n                raise RuntimeError(\'Unreachable OD pair\')\n            routes.append({\'origin\': origin.getID(), \'target\': target.getID(),\n                           \'edges\': \' \'.join(e.getID() for e in path),\n                           \'axis\': \'horizontal\' if origin.getID().startswith((\'left\',\'right\')) else \'vertical\'})\n        group = routes[-len(choices):]\n        for second in range(demand_s):\n            frac = second / demand_s\n            axis = group[0][\'axis\']\n            if scenario == \'balanced\': rate = 0.105\n            elif scenario == \'directional_switch\':\n                dominant = \'horizontal\' if frac < 0.5 else \'vertical\'\n                rate = 0.19 if axis == dominant else 0.055\n            elif scenario == \'surge\':\n                rate = 0.25 if 0.30 <= frac < 0.65 else 0.085\n            else: raise ValueError(scenario)\n            if rng.random() < min(1.0,rate*scale):\n                route = rng.choice(group)\n                requests.append((second + rng.random(), route[\'edges\']))\n    requests.sort()\n    for i, (depart, edges) in enumerate(requests):\n        # Assign driver speeds explicitly before simulation: controllers cannot\n        # change which random speed factor a later-inserted vehicle receives.\n        veh = ET.SubElement(root, \'vehicle\', id=f\'v{i:06d}\', type=\'car\',\n            depart=f\'{depart:.3f}\', departLane=\'best\', departSpeed=\'max\',\n            speedFactor=f\'{rng.uniform(0.90,1.10):.5f}\')\n        ET.SubElement(veh, \'route\', edges=edges)\n    demand = folder / f\'{scenario}_seed{seed}_d{demand_s}_load{scale:g}.rou.xml\'\n    ET.indent(root)\n    ET.ElementTree(root).write(demand, encoding=\'utf-8\', xml_declaration=True)\n    schedule = {v.attrib[\'id\']:float(v.attrib[\'depart\']) for v in root.findall(\'vehicle\')}\n    return demand, schedule\n\n\ndef get_layout(tls):\n    lanes = set()\n    for tid, item in tls.items():\n        links = sim.trafficlight.getControlledLinks(tid)\n        maps = {}\n        for phase in item[\'greens\']:\n            served = defaultdict(set)\n            for color, group in zip(item[\'phases\'][phase][\'state\'], links):\n                if color in \'Gg\':\n                    for incoming, outgoing, via in group:\n                        served[incoming].add(outgoing)\n                        lanes.update([incoming, outgoing])\n            maps[phase] = {k:sorted(v) for k,v in served.items()}\n        item[\'maps\'] = maps\n    capacities = {lane:sim.lane.getLength(lane)/7.5 for lane in sorted(lanes)}\n    return sorted(lanes), capacities\n\n\nclass Observation:\n    """Counts are sample-and-hold, delayed, noisy and missing in degraded mode.\n\n    Detector noise uses a separate seeded RNG, identical draw order across\n    controllers. Evaluation metrics always use simulator truth.\n    """\n    def __init__(self, lanes, capacities, regime, seed, demand_s):\n        self.lanes, self.capacities, self.regime = lanes, capacities, regime\n        self.rng = random.Random(seed+950001)\n        self.history = deque()\n        self.current = {l:(0.0,0.0) for l in lanes} # queue, vehicle count\n        self.measured = self.current.copy()\n        self.demand_s = demand_s\n        self.outage_lanes = set(l for l in lanes if l.startswith(\'B1\'))\n        self.stats = {\'sample_lane_opportunities\':0, \'dropped_lane_samples\':0}\n\n    def update(self,t):\n        if self.regime == \'perfect\':\n            self.current = {l:(float(sim.lane.getLastStepHaltingNumber(l)),\n                               float(sim.lane.getLastStepVehicleNumber(l))) for l in self.lanes}\n            return self.current\n        if int(t) % SETTINGS[\'observation_sample_s\'] == 0:\n            for lane in self.lanes:\n                self.stats[\'sample_lane_opportunities\'] += 1\n                drop = self.rng.random() < SETTINGS[\'dropout_probability\']\n                zq, zn = self.rng.gauss(0,1), self.rng.gauss(0,1)\n                outage = (self.demand_s*0.40 <= t < self.demand_s*0.55 and lane in self.outage_lanes)\n                if drop or outage:\n                    self.stats[\'dropped_lane_samples\'] += 1\n                    continue\n                q = sim.lane.getLastStepHaltingNumber(lane)\n                n = sim.lane.getLastStepVehicleNumber(lane)\n                n = max(0.0,min(self.capacities[lane], n + (0.20*n+0.5)*zn))\n                q = max(0.0,min(n, q + (0.20*q+0.5)*zq))\n                self.measured[lane] = (q,n)\n            self.history.append((t+SETTINGS[\'observation_delay_s\'], self.measured.copy()))\n        while self.history and self.history[0][0] <= t:\n            _, self.current = self.history.popleft()\n        return self.current\n\n\ndef announcements(tls, states, obs):\n    """One-hop message: queued demand currently being served toward each lane."""\n    message = defaultdict(float)\n    for tid,item in tls.items():\n        phase = states[tid][\'phase\']\n        if phase not in item[\'maps\']:\n            continue\n        for incoming, outs in item[\'maps\'][phase].items():\n            for out in outs:\n                message[out] += obs[incoming][0] / len(outs)\n    return message\n\n\ndef phase_rows(mapping, obs, capacities, message, turn_weights=None):\n    rows = []\n    for incoming, outs in sorted(mapping.items()):\n        q,n = obs[incoming]\n        # Equal turning weights are an explicit approximation in this pilot.\n        weights=[(turn_weights or {}).get(incoming,{}).get(out,0.0) for out in outs]\n        if sum(weights)<=0:weights=[1/len(outs)]*len(outs)\n        else:weights=[w/sum(weights) for w in weights]\n        downstream_q = sum(w*obs[out][0] for w,out in zip(weights,outs))\n        downstream_fill = sum(w*min(1.0,obs[out][1]/capacities[out]) for w,out in zip(weights,outs))\n        rows.append((q, downstream_q, downstream_fill, message[incoming], max(0.0,n-q)))\n    return rows\n\n\ndef run_one(out, netfile, tls_template, demand, schedule, controller, regime,\n            seed, demand_s, drain_s, fixed_green, trace=True, config=None,\n            turn_weights=None, stop_when_clear=False, live_hook=None, observation_factory=None):\n    out.mkdir(parents=True, exist_ok=True)\n    # Copy structure because phase maps are attached by this run.\n    tls = json.loads(json.dumps(tls_template))\n    kernel = Kernel(config)\n    horizon = demand_s + drain_s\n    command = [binary(\'sumo\'), \'-n\', str(netfile), \'-r\', str(demand),\n        \'--seed\', str(seed), \'--step-length\', \'1\', \'--time-to-teleport\', \'-1\',\n        \'--max-depart-delay\', \'-1\', \'--collision.action\', \'warn\',\n        \'--collision.check-junctions\', \'true\', \'--no-step-log\', \'true\',\n        \'--duration-log.disable\', \'true\',\n        \'--tripinfo-output\', str(out/\'tripinfo.xml\'), \'--tripinfo-output.write-unfinished\', \'true\',\n        \'--error-log\', str(out/\'sumo.log\')]\n    started = time.monotonic()\n    sim.start(command)\n    departed, arrived = {}, {}\n    collisions = teleports = switches = 0\n    phases_trace, timeline, replay = [], [], []\n    try:\n        lanes, capacities = get_layout(tls)\n        metric_lanes=[lane for lane in sim.lane.getIDList() if not lane.startswith(\':\')]\n        observer = observation_factory(lanes,capacities) if observation_factory else Observation(lanes, capacities, regime, seed, demand_s)\n        native=controller==\'sumo_actuated\'\n        states = {tid:{\'phase\':0, \'start\':0.0,\'target\':None,\n                      \'last_end\':{g:0.0 for g in item[\'greens\']}} for tid,item in tls.items()}\n        for tid in tls:\n            if native:\n                logic=sim.trafficlight.getAllProgramLogics(tid)[0]\n                # A new ID is required: updating an existing static program\'s\n                # type does not replace its underlying SUMO controller class.\n                logic.programID=\'reference_actuated\'\n                logic.type=3 # SUMO actuated, built-in induction-loop gap control\n                for phase in logic.phases:\n                    if \'G\' in phase.state:\n                        phase.minDur=SETTINGS[\'minimum_green_s\'];phase.maxDur=SETTINGS[\'maximum_green_s\']\n                sim.trafficlight.setProgramLogic(tid,logic)\n                sim.trafficlight.setProgram(tid,\'reference_actuated\')\n                sim.trafficlight.setPhase(tid,0)\n                sim.trafficlight.setPhaseDuration(tid,SETTINGS[\'minimum_green_s\'])\n                continue\n            sim.trafficlight.setPhase(tid,0)\n            sim.trafficlight.setPhaseDuration(tid,1e7)\n        for t in range(horizon):\n            if live_hook:live_hook(\'before\',t,states,lanes,obs if t else {},None)\n            obs = observer.update(t)\n            effective_controller=\'fixed\' if getattr(observer,\'fallback\',False) else controller\n            message = announcements(tls,states,obs)\n            updates = []\n            for tid,item in tls.items():\n                st = states[tid]\n                if native:\n                    actual=sim.trafficlight.getPhase(tid)\n                    if actual!=st[\'phase\']:\n                        old=st[\'phase\'];duration=t-st[\'start\']\n                        assert actual==(old+1)%len(item[\'phases\'])\n                        if old in item[\'greens\']:assert 10<=duration<=50\n                        else:assert duration==item[\'phases\'][old][\'duration\']\n                        phases_trace.append([t,tid,old,actual,duration])\n                        st.update(phase=actual,start=float(t))\n                        switches+=int(actual in item[\'greens\'])\n                    continue\n                phase, elapsed = st[\'phase\'], t-st[\'start\']\n                p = item[\'phases\'][phase]\n                change = False\n                target = None\n                if phase not in item[\'greens\']:\n                    change = elapsed >= p[\'duration\']\n                elif effective_controller == \'fixed\':\n                    change = elapsed >= fixed_green\n                elif int(elapsed) % SETTINGS[\'decision_interval_s\'] == 0:\n                    mode = MODE[effective_controller]\n                    weights=turn_weights if effective_controller in [\'calibrated_pressure\',\'predictive\'] else None\n                    rows={g:phase_rows(item[\'maps\'][g],obs,capacities,message,weights) for g in item[\'greens\']}\n                    if effective_controller in [\'message_only\',\'moving_only\']:\n                        rows={g:[(q,d,f,a if effective_controller==\'message_only\' else 0,m if effective_controller==\'moving_only\' else 0)\n                                 for q,d,f,a,m in data] for g,data in rows.items()}\n                    if effective_controller==\'predictive\':\n                        scores={g:kernel.predictive(data,0 if g==phase else 5) for g,data in rows.items()}\n                    else:scores={g:kernel.score(mode,data) for g,data in rows.items()}\n                    others=[g for g in item[\'greens\'] if g!=phase]\n                    target=max(others,key=lambda g:(scores[g],t-st[\'last_end\'][g],-g))\n                    overdue=[g for g in others if t-st[\'last_end\'][g]>=SETTINGS[\'maximum_red_target_s\']]\n                    if overdue:\n                        target=max(overdue,key=lambda g:(t-st[\'last_end\'][g],-g))\n                        change=elapsed>=SETTINGS[\'minimum_green_s\']\n                    else:\n                        change=kernel.switch(scores[phase],scores[target],elapsed)\n                if change:\n                    if phase in item[\'greens\']:\n                        assert elapsed >= SETTINGS[\'minimum_green_s\']\n                        assert elapsed <= SETTINGS[\'maximum_green_s\']\n                    else:\n                        assert elapsed == p[\'duration\']\n                    if phase in item[\'greens\']:\n                        if effective_controller==\'fixed\':target=item[\'greens\'][(item[\'greens\'].index(phase)+1)%len(item[\'greens\'])]\n                        updates.append((tid,phase+1,elapsed,target))\n                    elif set(p[\'state\'])=={\'r\'}:\n                        assert st[\'target\'] in item[\'greens\']\n                        updates.append((tid,st[\'target\'],elapsed,None))\n                    else:\n                        updates.append((tid,phase+1,elapsed,None))\n            # All agents decide from the same pre-action snapshot.\n            for tid,new,elapsed,target in updates:\n                old = states[tid][\'phase\']\n                sim.trafficlight.setPhase(tid,new)\n                sim.trafficlight.setPhaseDuration(tid,1e7)\n                if old in tls[tid][\'greens\']:\n                    states[tid][\'last_end\'][old]=float(t)\n                    states[tid][\'target\']=target\n                states[tid].update(phase=new,start=float(t))\n                phases_trace.append([t,tid,old,new,elapsed])\n                switches += int(new in tls[tid][\'greens\'])\n            sim.simulationStep()\n            now = float(t+1)\n            # SUMO labels step events by the interval\'s starting time (t),\n            # whereas simulation.getTime() after stepping is t+1.\n            for vid in sim.simulation.getDepartedIDList(): departed[vid] = float(t)\n            for vid in sim.simulation.getArrivedIDList(): arrived[vid] = float(t)\n            collisions += sim.simulation.getCollidingVehiclesNumber()\n            teleports += sim.simulation.getStartingTeleportNumber()\n            if live_hook:live_hook(\'after\',t+1,states,lanes,obs,\n                {\'departed\':departed,\'arrived\':arrived,\'schedule\':schedule,\'collisions\':collisions,\'teleports\':teleports,\'tls\':tls,\n                 \'sensor_fallback\':getattr(observer,\'fallback\',False)})\n            if (t+1)%5 == 0:\n                requested = sum(d<=now for d in schedule.values())\n                pending = requested-len(departed)\n                assert pending >= 0\n                halted = sum(sim.lane.getLastStepHaltingNumber(l) for l in metric_lanes)\n                timeline.append([now,halted,pending,sim.vehicle.getIDCount(),len(arrived)])\n            if trace and (t+1)%5 == 0:\n                # Compact replay positions are measured SUMO output, not generated imagery.\n                vehicles = [[v,*sim.vehicle.getPosition(v),sim.vehicle.getAngle(v)] for v in sim.vehicle.getIDList()]\n                replay.append({\'t\':now,\'vehicles\':vehicles,\'phases\':{k:v[\'phase\'] for k,v in states.items()}})\n            if stop_when_clear and t+1>=demand_s and len(arrived)==len(schedule):break\n        active = set(sim.vehicle.getIDList())\n        pending_ids = set(schedule)-set(departed)\n        assert set(schedule) == set(arrived) | active | pending_ids\n        assert not (set(arrived)&active or active&pending_ids or set(arrived)&pending_ids)\n    finally:\n        sim.close()\n    trips = ET.parse(out/\'tripinfo.xml\').getroot().findall(\'tripinfo\')\n    inserted = [x for x in trips if float(x.attrib[\'depart\']) >= 0]\n    assert {x.attrib[\'id\'] for x in inserted} == set(departed), \'Trip ledger missing inserted vehicles\'\n    # Count every requested vehicle, including requests never inserted.\n    system_time = sum(max(0.0,arrived.get(v,horizon)-d) for v,d in schedule.items())\n    # Independent check against SUMO\'s XML ledger, not the online arrival map.\n    xml_arrivals = {x.attrib[\'id\']:float(x.attrib[\'arrival\']) for x in inserted\n                    if float(x.attrib[\'arrival\']) >= 0}\n    ledger_time = sum(xml_arrivals.get(v,horizon)-d for v,d in schedule.items())\n    assert math.isclose(system_time,ledger_time,abs_tol=1e-6), \'Arrival-clock mismatch\'\n    insertion_wait = sum(max(0.0,float(x.attrib[\'depart\'])-schedule[x.attrib[\'id\']]) for x in inserted)\n    insertion_wait += sum(horizon-schedule[v] for v in pending_ids)\n    time_loss = sum(float(x.attrib[\'timeLoss\']) for x in inserted)\n    completed_wait = [float(x.attrib[\'waitingTime\']) for x in inserted if float(x.attrib[\'arrival\']) >= 0]\n    all_wait = [float(x.attrib[\'waitingTime\']) for x in inserted]\n    audit = {\'conservation_pass\': True, \'signal_transition_pass\':True,\'metric_ledger_pass\':True,\n             \'colliding_vehicle_events\':collisions,\'teleport_vehicle_events\':teleports,\n             \'valid_run\':collisions == 0 and teleports == 0,\n             \'detector_stats\':observer.stats}\n    result = {\'controller\':controller,\'scenario\':demand.stem.split(\'_seed\')[0],\n        \'regime\':regime,\'seed\':seed,\'horizon_s\':horizon,\'demand_s\':demand_s,\n        \'fixed_green_s\':fixed_green,\'scheduled\':len(schedule),\'departed\':len(departed),\n        \'finished_at_s\':t+1,\'cleared\':len(arrived)==len(schedule),\n        \'controller_config\':kernel.config if controller==\'predictive\' else {},\n        \'arrived\':len(arrived),\'active\':len(active),\'pending\':len(pending_ids),\n        \'completion_fraction\':len(arrived)/len(schedule) if schedule else 1.0,\n        \'total_system_time_s\':round(system_time,3),\n        \'mean_system_time_s\':round(system_time/max(1,len(schedule)),3),\n        \'network_time_loss_plus_entry_wait_s\':round(time_loss+insertion_wait,3),\n        \'entry_wait_s\':round(insertion_wait,3),\n        \'p95_completed_stopped_wait_s\':round(float(np.percentile(completed_wait,95)),3) if completed_wait else None,\n        \'max_inserted_stopped_wait_s\':max(all_wait,default=0),\n        \'switches\':switches,\'wall_s\':round(time.monotonic()-started,3),\n        \'demand_sha256\':sha(demand),\'network_sha256\':sha(netfile), **audit}\n    write_json(out/\'result.json\',result)\n    for name,header,rows in [\n        (\'timeline.csv\',[\'time_s\',\'halted\',\'pending\',\'active\',\'arrived\'],timeline),\n        (\'signals.csv\',[\'time_s\',\'tls\',\'old_phase\',\'new_phase\',\'old_duration_s\'],phases_trace)]:\n        with (out/name).open(\'w\',newline=\'\') as f:\n            w=csv.writer(f);w.writerow(header);w.writerows(rows)\n    if replay: write_json(out/\'replay.json\',replay)\n    return result\n\n\ndef summarize(results,output):\n    output.mkdir(exist_ok=True,parents=True)\n    keys = list(results[0])\n    with (output/\'results.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=keys);w.writeheader()\n        w.writerows({k:json.dumps(v) if isinstance(v,dict) else v for k,v in row.items()} for row in results)\n    groups = defaultdict(list)\n    for r in results: groups[(r[\'scenario\'],r[\'regime\'],r[\'controller\'])].append(r)\n    aggregate=[]\n    for (scenario,regime,controller),rows in sorted(groups.items()):\n        refs = {r[\'seed\']:r for r in results if r[\'scenario\']==scenario and r[\'regime\']==regime and r[\'controller\']==\'max_pressure\'}\n        paired = [(refs[r[\'seed\']][\'total_system_time_s\']-r[\'total_system_time_s\'])/refs[r[\'seed\']][\'total_system_time_s\']*100\n                  for r in rows if r[\'seed\'] in refs]\n        aggregate.append({\'scenario\':scenario,\'regime\':regime,\'controller\':controller,\n            \'runs\':len(rows),\'mean_system_time_s\':round(statistics.mean(r[\'mean_system_time_s\'] for r in rows),3),\n            \'mean_completion\':round(statistics.mean(r[\'completion_fraction\'] for r in rows),4),\n            \'mean_paired_improvement_vs_pressure_pct\':round(statistics.mean(paired),3) if paired else None,\n            \'min_paired_improvement_vs_pressure_pct\':round(min(paired),3) if paired else None,\n            \'max_paired_improvement_vs_pressure_pct\':round(max(paired),3) if paired else None,\n            \'all_runs_valid\':all(r[\'valid_run\'] for r in rows)})\n    write_json(output/\'aggregate.json\',aggregate)\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    combinations=sorted({(r[\'scenario\'],r[\'regime\']) for r in results})\n    fig,axes=plt.subplots(len(combinations),1,figsize=(11,3.2*len(combinations)),squeeze=False)\n    colors=[\'#aab3c0\',\'#6197b7\',\'#173a58\',\'#36a79c\',\'#e69c3a\']\n    for ax,(scenario,regime) in zip(axes[:,0],combinations):\n        for j,c in enumerate(CONTROLLERS):\n            rows=groups.get((scenario,regime,c),[])\n            values=[r[\'mean_system_time_s\'] for r in rows]\n            if values:\n                ax.bar(j,statistics.mean(values),color=colors[j],width=0.65)\n                ax.scatter([j+(i-(len(values)-1)/2)*0.08 for i in range(len(values))],values,s=18,c=\'#132338\',zorder=3)\n        ax.set_xticks(range(5),[\'Fixed\\n(dev selected)\',\'Queue\\nactuated\',\'Max\\npressure\',\'Receiving\\naware\',\'Coordinated\'])\n        ax.set_ylabel(\'System time / request (s)\\nLower is better\')\n        ax.set_title(f\'{scenario.replace("_"," ").title()} | {regime} observations\',loc=\'left\',fontweight=\'bold\')\n        ax.spines[[\'top\',\'right\']].set_visible(False)\n        ax.grid(axis=\'y\',alpha=0.15);ax.set_axisbelow(True)\n    fig.suptitle(\'Adaptive Signals — Stage 1 synthetic-network pilot\\nDots are seeds; unfinished and uninserted vehicles remain in the metric\',fontsize=13)\n    fig.tight_layout(rect=(0,0,1,0.95))\n    fig.savefig(output/\'pilot_comparison.png\',dpi=160)\n    plt.close(fig)\n    return aggregate\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--profile\',choices=[\'smoke\',\'pilot\',\'extended\'],default=\'pilot\')\n    p.add_argument(\'--output\',type=Path,default=ROOT/\'runs/pilot\')\n    p.add_argument(\'--seeds\',type=int,nargs=\'+\')\n    p.add_argument(\'--demand-s\',type=int)\n    p.add_argument(\'--drain-s\',type=int)\n    p.add_argument(\'--resume\',action=\'store_true\')\n    p.add_argument(\'--fixed-green\',type=int,default=None,help=\'Override dev selection (10..50 s)\')\n    args=p.parse_args()\n    demand_s=args.demand_s or (120 if args.profile==\'smoke\' else 900)\n    drain_s=args.drain_s if args.drain_s is not None else (120 if args.profile==\'smoke\' else 600)\n    if demand_s<1 or drain_s<0:p.error(\'Invalid demand/drain duration\')\n    if args.fixed_green is not None and not 10<=args.fixed_green<=50:p.error(\'Fixed green must be 10..50\')\n    seeds=args.seeds or ([777] if args.profile==\'smoke\' else SETTINGS[\'evaluation_seeds\'])\n    if len(set(seeds))!=len(seeds):p.error(\'Seeds must be unique\')\n    if SETTINGS[\'development_seed\'] in seeds:p.error(\'Development seed cannot be an evaluation seed\')\n    scenarios=[\'balanced\'] if args.profile==\'smoke\' else SCENARIOS\n    output=args.output.resolve();output.mkdir(parents=True,exist_ok=True)\n    source_hashes={str(q.relative_to(ROOT)):sha(q) for q in [ROOT/\'experiment.py\',ROOT/\'src/controllers.cpp\',ROOT/\'requirements.txt\']}\n    signature={\'settings\':SETTINGS,\'source_sha256\':source_hashes,\'demand_s\':demand_s,\'drain_s\':drain_s,\'seeds\':seeds,\n        \'scenarios\':scenarios,\'profile\':args.profile,\'fixed_override\':args.fixed_green,\n        \'sumo_version\':sim.getVersion()[1]}\n    manifest_path=output/\'manifest.json\'\n    if manifest_path.exists():\n        old=json.loads(manifest_path.read_text())\n        if old[\'signature\']!=signature:raise RuntimeError(\'Output belongs to a different configuration/source version. Use a new --output.\')\n        if not args.resume:raise RuntimeError(\'Output already exists; use --resume or a new --output\')\n    else:\n        write_json(manifest_path,{\'signature\':signature,\'python\':sys.version,\'platform\':platform.platform(),\n            \'started_utc\':time.strftime(\'%Y-%m-%dT%H:%M:%SZ\',time.gmtime()),\n            \'metric\':\'sum(min(arrival,horizon)-requested_depart) over every requested vehicle\',\n            \'limitations\':[\'synthetic grid and demand\',\'no RL\',\'no CCTV\',\'no pedestrians/motorcycles\',\n                \'equal turning weights\',\'three-seed pilot is not a statistical superiority claim\']})\n    netfile,net,tls=build_network(output/\'inputs\')\n    def cached_run(folder,*params,**kwargs):\n        file=folder/\'result.json\'\n        if args.resume and file.exists():\n            r=json.loads(file.read_text())\n            # Input identity and controller configuration are verified on resume.\n            assert r[\'network_sha256\']==sha(netfile) and r[\'demand_sha256\']==sha(params[0])\n            return r\n        return run_one(folder,netfile,tls,*params,**kwargs)\n    # Hold-out development stream only: select fixed duration on identical scenarios.\n    if args.fixed_green is None:\n        tuning=[]\n        for scenario in scenarios:\n            d,s=generate_demand(output/\'inputs\',net,scenario,101,demand_s)\n            for green in SETTINGS[\'fixed_candidates_s\']:\n                row=cached_run(output/\'development\'/f\'{scenario}_g{green}\',d,s,\'fixed\',\'perfect\',101,demand_s,drain_s,green,False)\n                tuning.append(row)\n        fixed=min(SETTINGS[\'fixed_candidates_s\'],key=lambda g:statistics.mean(r[\'mean_system_time_s\'] for r in tuning if r[\'fixed_green_s\']==g))\n        write_json(output/\'fixed_selection.json\',{\'selected_green_s\':fixed,\'development_results\':tuning})\n    else:fixed=args.fixed_green\n    campaign=[]\n    for scenario in scenarios:\n        for regime in [\'perfect\',\'degraded\']:\n            # Pilot degrades the most transient scenario; extended degrades all.\n            if args.profile==\'pilot\' and regime==\'degraded\' and scenario!=\'directional_switch\':continue\n            for seed in seeds:\n                d,s=generate_demand(output/\'inputs\',net,scenario,seed,demand_s)\n                for c in CONTROLLERS:\n                    dest=output/\'evaluation\'/f\'{scenario}_{regime}_seed{seed}_{c}\'\n                    row=cached_run(dest,d,s,c,regime,seed,demand_s,drain_s,fixed)\n                    campaign.append(row)\n                    write_json(output/\'progress.json\',{\'completed\':len(campaign),\'last\':row})\n                    print(f\'{len(campaign):03d} {scenario:18s} {regime:8s} seed={seed} {c:16s} \'\n                          f\'system={row["mean_system_time_s"]:.1f}s arrived={row["arrived"]}/{row["scheduled"]} \'\n                          f\'valid={row["valid_run"]}\',flush=True)\n    aggregate=summarize(campaign,output)\n    assert all(r[\'valid_run\'] for r in campaign),\'Invalid runs retained in results; investigate SUMO logs\'\n    print(f\'Completed {len(campaign)} evaluation runs. Results: {output}\',flush=True)\n    return aggregate\n\n\nif __name__==\'__main__\': main()\n', 'stage2.py': '"""Stage-2 development selection and untouched-seed evaluation."""\nfrom __future__ import annotations\nimport argparse\nfrom collections import defaultdict\nimport csv\nimport json\nfrom pathlib import Path\nimport statistics\nimport sys\nimport time\nimport xml.etree.ElementTree as ET\nimport experiment as e\n\nCONTROLLERS=[\'fixed\',\'sumo_actuated\',\'max_pressure\',\'calibrated_pressure\',\n             \'receiving_aware\',\'message_only\',\'moving_only\',\'coordinated\',\'predictive\']\nLOADS=[0.3,0.6,1.0]\nDEV_SEEDS=[201,1201]\nEVAL_SEEDS=[12777,13777,14777,15777]\nCONFIGS=[{\'horizon_s\':h,\'receiving_power\':p,\'message_weight\':0.3,\'moving_weight\':0.2}\n         for h,p in [(10,1),(20,1),(30,1),(20,2)]]\nDEV_CASES=[(\'balanced\',0.3),(\'directional_switch\',0.6),(\'surge\',1.0)]\n\n\ndef learn_turns(demands):\n    counts=defaultdict(lambda:defaultdict(int))\n    for path in demands:\n        for v in ET.parse(path).getroot().findall(\'vehicle\'):\n            edges=v.find(\'route\').attrib[\'edges\'].split()\n            for incoming,outgoing in zip(edges,edges[1:]):\n                counts[incoming+\'_0\'][outgoing+\'_0\']+=1\n    return {lane:{out:n/sum(outs.values()) for out,n in sorted(outs.items())}\n            for lane,outs in sorted(counts.items())}\n\n\ndef aggregate(rows,folder):\n    with (folder/\'results.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader()\n        w.writerows({k:json.dumps(v) if isinstance(v,dict) else v for k,v in r.items()} for r in rows)\n    groups=defaultdict(list)\n    for r in rows:groups[(r[\'scenario\'],r[\'load_scale\'],r[\'regime\'],r[\'controller\'])].append(r)\n    output=[]\n    for (scenario,load,regime,controller),g in sorted(groups.items()):\n        refs={r[\'seed\']:r for r in rows if r[\'scenario\']==scenario and r[\'load_scale\']==load and r[\'regime\']==regime and r[\'controller\']==\'fixed\'}\n        change=[100*(refs[r[\'seed\']][\'total_system_time_s\']-r[\'total_system_time_s\'])/refs[r[\'seed\']][\'total_system_time_s\'] for r in g]\n        output.append({\'scenario\':scenario,\'load_scale\':load,\'regime\':regime,\'controller\':controller,\'runs\':len(g),\n            \'mean_system_time_s\':statistics.mean(r[\'mean_system_time_s\'] for r in g),\n            \'mean_completion\':statistics.mean(r[\'completion_fraction\'] for r in g),\n            \'cleared_runs\':sum(r[\'cleared\'] for r in g),\n            \'paired_reduction_vs_fixed_pct\':statistics.mean(change),\n            \'min_reduction_pct\':min(change),\'max_reduction_pct\':max(change),\n            \'mean_max_stopped_wait_s\':statistics.mean(r[\'max_inserted_stopped_wait_s\'] for r in g),\n            \'mean_switches\':statistics.mean(r[\'switches\'] for r in g),\n            \'all_valid\':all(r[\'valid_run\'] for r in g)})\n    e.write_json(folder/\'aggregate.json\',output)\n    plot(output,folder)\n    return output\n\n\ndef plot(agg,folder):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    chosen=[\'fixed\',\'sumo_actuated\',\'calibrated_pressure\',\'coordinated\',\'predictive\']\n    labels=[\'Fixed\',\'SUMO actuated\',\'Calibrated pressure\',\'Stage-1 coordinated\',\'Discharge lookahead\']\n    colors=[\'#8e9caf\',\'#5278ab\',\'#173a58\',\'#35a89a\',\'#e7a140\']\n    fig,axes=plt.subplots(2,3,figsize=(15,8),sharey=False)\n    for i,scenario in enumerate([\'balanced\',\'directional_switch\']):\n        for j,load in enumerate(LOADS):\n            ax=axes[i,j]\n            data=[next((r for r in agg if r[\'scenario\']==scenario and r[\'load_scale\']==load and r[\'regime\']==\'perfect\' and r[\'controller\']==c),None) for c in chosen]\n            for k,r in enumerate(data):\n                if r:ax.bar(k,r[\'mean_system_time_s\'],color=colors[k]);ax.text(k,r[\'mean_system_time_s\']+8,f"{r[\'mean_completion\']:.0%}",ha=\'center\',fontsize=8)\n            ax.set_title(f\'{scenario.replace("_"," ").title()} | load ×{load:g}\',loc=\'left\',fontsize=11,fontweight=\'bold\')\n            ax.set_xticks(range(len(chosen)),[\'Fixed\',\'SUMO\\nactuated\',\'Turn\\npressure\',\'Stage-1\\ncoord.\',\'Discharge\\nlookahead\'],fontsize=9)\n            ax.set_ylim(0,max((r[\'mean_system_time_s\'] for r in data if r),default=1)*1.2)\n            ax.spines[[\'top\',\'right\']].set_visible(False);ax.grid(axis=\'y\',alpha=.15);ax.set_axisbelow(True)\n            if j==0:ax.set_ylabel(\'System time / request (s)\')\n    fig.suptitle(\'Stage 2 — broader demand and longer draining\\nLabels show completion fraction; four fresh seeds; synthetic network\',fontsize=14)\n    fig.tight_layout(rect=[0,0,1,.92]);fig.savefig(folder/\'stage2_comparison.png\',dpi=150);plt.close(fig)\n\n\ndef campaign(folder,profile=\'full\',resume=False):\n    folder=Path(folder).resolve();folder.mkdir(parents=True,exist_ok=True)\n    demand_s,drain_s=(120,300) if profile==\'smoke\' else (900,1800)\n    seeds=EVAL_SEEDS[:1] if profile==\'smoke\' else EVAL_SEEDS\n    source={str(p.relative_to(e.ROOT)):e.sha(p) for p in [e.ROOT/\'experiment.py\',e.ROOT/\'stage2.py\',e.ROOT/\'src/controllers.cpp\']}\n    signature={\'source_sha256\':source,\'settings\':e.SETTINGS,\'dev_cases\':DEV_CASES,\'development_seeds\':DEV_SEEDS,\n        \'evaluation_seeds\':seeds,\'configs\':CONFIGS,\'fixed_candidates\':[10,20,30,45],\n        \'profile\':profile,\'demand_s\':demand_s,\'drain_s\':drain_s,\'sumo_version\':e.sim.getVersion()[1]}\n    # Normalize tuples exactly as JSON writes them for idempotent resume.\n    signature=json.loads(json.dumps(signature))\n    manifest=folder/\'manifest.json\'\n    if manifest.exists():\n        assert json.loads(manifest.read_text())[\'signature\']==signature,\'Use a new output folder after source/config changes\'\n        if not resume:raise RuntimeError(\'Use --resume or a new output folder\')\n    else:e.write_json(manifest,{\'signature\':signature,\'started_utc\':time.strftime(\'%Y-%m-%dT%H:%M:%SZ\',time.gmtime()),\n        \'plan\':\'development-only tuning, frozen before fresh-seed evaluation; 900 s demand / 1800 s max drain\',\n        \'limits\':[\'synthetic demand/network\',\'no RL\',\'no live CCTV\',\'approximate discharge and pressure models\']})\n    nf,net,tls=e.build_network(folder/\'inputs\')\n    dev=[]\n    for scenario,load in DEV_CASES:\n        for seed in DEV_SEEDS:\n            d,s=e.generate_demand(folder/\'inputs\',net,scenario,seed,demand_s,load)\n            dev.append((scenario,load,seed,d,s))\n    turns=learn_turns([d for _,_,_,d,_ in dev]);e.write_json(folder/\'turn_weights.json\',turns)\n    def run(dest,scenario,load,seed,d,s,c,regime=\'perfect\',green=20,config=None,trace=False):\n        result=dest/\'result.json\'\n        if resume and result.exists():\n            r=json.loads(result.read_text());assert r[\'demand_sha256\']==e.sha(d) and r[\'network_sha256\']==e.sha(nf)\n            return r\n        r=e.run_one(dest,nf,tls,d,s,c,regime,seed,demand_s,drain_s,green,trace,\n            config=config,turn_weights=turns,stop_when_clear=True)\n        r.update(load_scale=load)\n        e.write_json(result,r)\n        if not r[\'valid_run\']:raise RuntimeError(f\'Invalid run retained: {dest}\')\n        return r\n    fixed_results=[]\n    for scenario,load,seed,d,s in dev:\n        for green in [10,20,30,45]:\n            dest=folder/\'development\'/f\'{scenario}_l{load}_seed{seed}_fixed_g{green}\'\n            fixed_results.append(run(dest,scenario,load,seed,d,s,\'fixed\',green=green))\n    fixed={str(load):min([10,20,30,45],key=lambda g:statistics.mean(r[\'mean_system_time_s\'] for r in fixed_results if r[\'load_scale\']==load and r[\'fixed_green_s\']==g)) for load in LOADS}\n    scores=[];tuning=[]\n    for ci,config in enumerate(CONFIGS):\n        ratios=[]\n        for scenario,load,seed,d,s in dev:\n            dest=folder/\'development\'/f\'{scenario}_l{load}_seed{seed}_predictive_c{ci}\'\n            r=run(dest,scenario,load,seed,d,s,\'predictive\',green=fixed[str(load)],config=config)\n            baseline=next(x for x in fixed_results if x[\'scenario\']==scenario and x[\'load_scale\']==load and x[\'seed\']==seed and x[\'fixed_green_s\']==fixed[str(load)])\n            ratios.append(r[\'total_system_time_s\']/baseline[\'total_system_time_s\']);tuning.append(r)\n        scores.append(statistics.mean(ratios))\n        print(f\'DEV candidate={ci} relative_system_time={scores[-1]:.4f}\',flush=True)\n    chosen=min(range(len(CONFIGS)),key=lambda i:scores[i]);config=CONFIGS[chosen]\n    selection={\'fixed_green_by_load\':fixed,\'predictive_config\':config,\'candidate_scores\':scores,\n        \'objective\':\'mean ratio of total system time to dev-selected fixed timing across six dev streams\',\n        \'selected_before_evaluation\':True,\'turn_weight_sha256\':e.sha(folder/\'turn_weights.json\')}\n    e.write_json(folder/\'selection.json\',selection)\n    print(\'SELECTION FROZEN\',json.dumps(selection),flush=True)\n    cases=[(scenario,load,\'perfect\') for scenario in [\'balanced\',\'directional_switch\'] for load in LOADS]\n    cases += [(\'surge\',0.6,\'perfect\'),(\'directional_switch\',1.0,\'degraded\')]\n    if profile==\'smoke\':cases=[(\'balanced\',0.3,\'perfect\')]\n    results=[]\n    for scenario,load,regime in cases:\n        for seed in seeds:\n            d,s=e.generate_demand(folder/\'inputs\',net,scenario,seed,demand_s,load)\n            for c in CONTROLLERS:\n                # Built-in actuated uses its own loop detectors, not the camera model.\n                if regime==\'degraded\' and c==\'sumo_actuated\':continue\n                dest=folder/\'evaluation\'/f\'{scenario}_l{load}_{regime}_seed{seed}_{c}\'\n                r=run(dest,scenario,load,seed,d,s,c,regime=regime,green=fixed[str(load)],config=config)\n                results.append(r)\n                e.write_json(folder/\'progress.json\',{\'completed\':len(results),\'last\':r})\n                print(f"EVAL {len(results):03d} {scenario:18s} l={load:g} {regime:8s} seed={seed} {c:19s} "\n                      f"system={r[\'mean_system_time_s\']:.1f}s complete={r[\'completion_fraction\']:.1%} valid={r[\'valid_run\']}",flush=True)\n    agg=aggregate(results,folder)\n    expected=9 if profile==\'smoke\' else 284\n    assert len(results)==expected\n    groups=defaultdict(list)\n    for r in results:groups[(r[\'scenario\'],r[\'load_scale\'],r[\'regime\'],r[\'seed\'])].append(r)\n    for g in groups.values():\n        assert len({r[\'demand_sha256\'] for r in g})==len({r[\'network_sha256\'] for r in g})==1\n    e.write_json(folder/\'audit.json\',{\'evaluation_runs\':len(results),\'development_runs\':len(fixed_results)+len(tuning),\n        \'all_valid\':all(r[\'valid_run\'] for r in results),\'matched_input_groups\':len(groups),\n        \'collision_events\':sum(r[\'colliding_vehicle_events\'] for r in results),\'teleport_events\':sum(r[\'teleport_vehicle_events\'] for r in results),\n        \'all_metric_ledgers_agree\':all(r[\'metric_ledger_pass\'] for r in results),\'selected_before_evaluation\':True})\n    print(f\'CAMPAIGN_COMPLETE {len(results)} evaluation runs\',flush=True)\n    return agg\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--profile\',choices=[\'smoke\',\'full\'],default=\'full\')\n    p.add_argument(\'--output\',type=Path,default=e.ROOT/\'runs/stage2\')\n    p.add_argument(\'--resume\',action=\'store_true\')\n    a=p.parse_args();campaign(a.output,a.profile,a.resume)\n', 'live_server.py': '"""Local/Colab live SUMO viewer. One simulation-owning worker, HTTP snapshots.\n\nThe browser draws actual SUMO positions; it does not simulate vehicle motion.\nNo cloud hosting is implied. Bind only to loopback and use Colab\'s private port\nproxy or a local browser. Ctrl+C closes the server and simulation worker.\n"""\nfrom __future__ import annotations\nimport argparse\nimport copy\nimport json\nimport queue\nimport secrets\nimport signal\nimport threading\nimport time\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom pathlib import Path\nimport urllib.parse\nimport experiment as e\nfrom stage2 import CONTROLLERS, learn_turns\n\n\nclass CancelRun(Exception):pass\n\n\nclass LiveApp:\n    def __init__(self,folder,network_builder=None,demand_builder=None,turn_learner=None,observation_factory=None):\n        self.folder=Path(folder);self.folder.mkdir(parents=True,exist_ok=True)\n        self.demand_builder=demand_builder or e.generate_demand\n        self.turn_learner=turn_learner or learn_turns\n        self.observation_factory=observation_factory\n        self.netfile,self.net,self.tls=(network_builder or e.build_network)(self.folder/\'inputs\')\n        self.cv=threading.Condition();self.commands=queue.Queue()\n        self.paused=False;self.step_budget=0;self.cancel=False;self.closed=False;self.speed=10.0\n        self.snapshot={\'status\':\'ready\',\'time_s\':0,\'vehicles\':[],\'signals\':[],\n            \'metrics\':{},\'history\':[],\'message\':\'Choose a controller and start a live run.\'}\n        self.scene={\'bounds\':self.net.getBoundary(),\'lanes\':[\n            {\'id\':l.getID(),\'shape\':l.getShape(),\'length\':l.getLength(),\n             \'from\':edge.getFromNode().getID(),\'to\':edge.getToNode().getID()}\n            for edge in self.net.getEdges() for l in edge.getLanes()],\n            \'nodes\':[{\'id\':tid,\'xy\':self._signal_xy(tid)} for tid in self.tls]}\n        self.config={\'controller\':\'predictive\',\'scenario\':\'directional_switch\',\'scale\':0.6,\n                     \'regime\':\'perfect\',\'seed\':12777,\'demand_s\':600}\n        selection=e.ROOT/\'runs/stage2/selection.json\'\n        self.selection=json.loads(selection.read_text()) if selection.exists() else {\n            \'predictive_config\':{\'horizon_s\':20,\'receiving_power\':1,\'message_weight\':0.3,\'moving_weight\':0.2},\n            \'fixed_green_by_load\':{\'0.3\':10,\'0.6\':10,\'1.0\':20}}\n        self.turns=None\n        weights=e.ROOT/\'runs/stage2/turn_weights.json\'\n        if weights.exists():self.turns=json.loads(weights.read_text())\n        self.worker=threading.Thread(target=self._worker,name=\'sumo-owner\',daemon=True);self.worker.start()\n\n    def _signal_xy(self,tid):\n        try:return self.net.getNode(tid).getCoord()\n        except KeyError:\n            coords={tuple(c[0].getEdge().getToNode().getCoord()) for c in self.net.getTLS(tid).getConnections()}\n            return tuple(sum(p[i] for p in coords)/len(coords) for i in [0,1])\n\n    def control(self,action,values=None):\n        values=values or {}\n        with self.cv:\n            if action==\'start\':\n                c={**self.config,**values}\n                if c[\'controller\'] not in CONTROLLERS:raise ValueError(\'Unknown controller\')\n                if c[\'scenario\'] not in e.SCENARIOS:raise ValueError(\'Unknown traffic pattern\')\n                if c[\'regime\'] not in [\'perfect\',\'degraded\']:raise ValueError(\'Unknown sensor condition\')\n                if c[\'controller\']==\'sumo_actuated\' and c[\'regime\']==\'degraded\':\n                    raise ValueError(\'SUMO actuated uses loop detectors; select perfect sensors for that reference.\')\n                if not 0.1<=float(c[\'scale\'])<=1.5:raise ValueError(\'Demand multiplier must be 0.1–1.5\')\n                if not 0<=int(c[\'seed\'])<2**31:raise ValueError(\'Seed outside supported range\')\n                if not 60<=int(c[\'demand_s\'])<=3600:raise ValueError(\'Demand duration must be 60–3600 s\')\n                c.update(scale=float(c[\'scale\']),seed=int(c[\'seed\']),demand_s=int(c[\'demand_s\']))\n                self.config=c;self.cancel=True;self.paused=False;self.step_budget=0\n                self.snapshot[\'status\']=\'starting\';self.commands.put(c)\n            elif action==\'pause\':self.paused=True\n            elif action==\'resume\':self.paused=False\n            elif action==\'step\':self.paused=True;self.step_budget+=1\n            elif action==\'speed\':\n                speed=float(values.get(\'speed\',10))\n                if not 1<=speed<=60:raise ValueError(\'Speed must be 1–60×\')\n                self.speed=speed\n            elif action==\'stop\':self.cancel=True;self.paused=False\n            else:raise ValueError(\'Unknown action\')\n            self.cv.notify_all()\n\n    def state(self):\n        with self.cv:\n            data=copy.deepcopy(self.snapshot)\n            if data[\'status\']==\'running\' and self.paused:data[\'status\']=\'paused\'\n            data.update(speed=self.speed,config=self.config)\n            return data\n\n    def _worker(self):\n        while not self.closed:\n            c=self.commands.get()\n            if c is None:return\n            # Newer reset commands supersede queued, unstarted requests.\n            while not self.commands.empty():\n                later=self.commands.get_nowait()\n                if later is None:return\n                c=later\n            with self.cv:self.cancel=False\n            try:\n                d,s=self.demand_builder(self.folder/\'inputs\',self.net,c[\'scenario\'],c[\'seed\'],c[\'demand_s\'],c[\'scale\'])\n                turns=self.turns\n                if turns is None:\n                    # Default calibration is a development stream, never this live stream.\n                    dd,_=self.demand_builder(self.folder/\'inputs\',self.net,\'balanced\',201,900,0.6)\n                    turns=self.turn_learner([dd])\n                self.history=[]\n                with self.cv:self.snapshot.update(status=\'running\',message=\'\',history=[],metrics={},time_s=0,vehicles=[],signals=[])\n                closest=min(self.selection[\'fixed_green_by_load\'],key=lambda x:abs(float(x)-c[\'scale\']))\n                green=self.selection[\'fixed_green_by_load\'][closest]\n                result=e.run_one(self.folder/\'latest\',self.netfile,self.tls,d,s,c[\'controller\'],c[\'regime\'],\n                    c[\'seed\'],c[\'demand_s\'],1800,green,False,config=self.selection[\'predictive_config\'],\n                    turn_weights=turns,stop_when_clear=True,live_hook=self._hook,observation_factory=self.observation_factory)\n                with self.cv:self.snapshot.update(status=\'completed\',result=result,\n                    message=\'All requests cleared.\' if result[\'cleared\'] else \'Cutoff reached; unfinished vehicles remain counted.\')\n            except CancelRun:\n                with self.cv:self.snapshot.update(status=\'stopped\',message=\'Run stopped. Start resets the same seeded experiment.\')\n            except Exception as exc:\n                with self.cv:self.snapshot.update(status=\'error\',message=f\'{type(exc).__name__}: {exc}\')\n\n    def _hook(self,event,t,states,lanes,obs,ledger):\n        if event==\'before\':\n            with self.cv:\n                while self.paused and self.step_budget==0 and not self.cancel and not self.closed:self.cv.wait()\n                if self.cancel or self.closed:raise CancelRun()\n                if self.step_budget:self.step_budget-=1\n                # Event-based pacing keeps pause/reset responsive; no busy loop.\n                self.cv.wait(timeout=1.0/self.speed)\n                if self.cancel or self.closed:raise CancelRun()\n            return\n        vehicles=[]\n        for v in e.sim.vehicle.getIDList():\n            x,y=e.sim.vehicle.getPosition(v)\n            vehicles.append([v,round(x,2),round(y,2),round(e.sim.vehicle.getAngle(v),1),round(e.sim.vehicle.getSpeed(v),2)])\n        signals=[]\n        for tid,item in ledger[\'tls\'].items():\n            phase=e.sim.trafficlight.getPhase(tid)\n            for g,mapping in item[\'maps\'].items():\n                color=\'G\' if phase==g else \'y\' if phase==g+1 else \'r\'\n                for lane in mapping:signals.append([lane,color])\n        requested={v:d for v,d in ledger[\'schedule\'].items() if d<=t}\n        pending=len(requested)-len(ledger[\'departed\'])\n        halted=sum(e.sim.lane.getLastStepHaltingNumber(lane[\'id\']) for lane in self.scene[\'lanes\'])\n        system=sum(max(0,ledger[\'arrived\'].get(v,t)-d) for v,d in requested.items())\n        metrics={\'requested\':len(requested),\'total_requests\':len(ledger[\'schedule\']),\n            \'arrived\':len(ledger[\'arrived\']),\'active\':len(vehicles),\'pending\':pending,\'halted\':halted,\n            \'mean_system_so_far_s\':round(system/max(1,len(requested)),1),\n            \'collisions\':ledger[\'collisions\'],\'teleports\':ledger[\'teleports\']}\n        metrics[\'sensor_fallback\']=ledger.get(\'sensor_fallback\',False)\n        if t%5==0:self.history.append([t,halted,pending,len(ledger[\'arrived\'])])\n        sensors={l:[round(q,1),round(n,1)] for l,(q,n) in obs.items()}\n        with self.cv:self.snapshot.update(time_s=t,vehicles=vehicles,signals=signals,metrics=metrics,\n            history=self.history[-600:],sensors=sensors,status=\'running\')\n\n    def close(self):\n        with self.cv:self.closed=True;self.cancel=True;self.paused=False;self.cv.notify_all()\n        self.commands.put(None);self.worker.join(timeout=5)\n\n\ndef server(app,host=\'127.0.0.1\',port=8765):\n    access=secrets.token_urlsafe(24)\n    class Handler(BaseHTTPRequestHandler):\n        def log_message(self,*args):pass\n        def send(self,code,payload,kind=\'application/json\'):\n            data=payload.encode() if isinstance(payload,str) else json.dumps(payload,allow_nan=False).encode()\n            self.send_response(code);self.send_header(\'Content-Type\',kind+\'; charset=utf-8\')\n            self.send_header(\'Cache-Control\',\'no-store\');self.send_header(\'Content-Length\',str(len(data)))\n            self.end_headers();self.wfile.write(data)\n        def do_GET(self):\n            path=urllib.parse.urlparse(self.path).path\n            if path.endswith(\'/api/state\') or path.endswith(\'/api/scene\'):\n                if self.headers.get(\'X-Simulation-Key\')!=access:return self.send(403,{\'error\':\'Invalid simulation key\'})\n                return self.send(200,app.state() if path.endswith(\'/api/state\') else app.scene)\n            if path.endswith(\'/\') or path.endswith(\'/index.html\'):\n                html=(e.ROOT/\'viewer/index.html\').read_text().replace(\'__ACCESS_KEY__\',json.dumps(access))\n                return self.send(200,html,\'text/html\')\n            self.send(404,{\'error\':\'Not found\'})\n        def do_POST(self):\n            if not urllib.parse.urlparse(self.path).path.endswith(\'/api/control\'):return self.send(404,{\'error\':\'Not found\'})\n            if self.headers.get(\'X-Simulation-Key\')!=access:return self.send(403,{\'error\':\'Invalid simulation key\'})\n            length=int(self.headers.get(\'Content-Length\',\'0\'))\n            if not 0<length<=8192:return self.send(400,{\'error\':\'Invalid request length\'})\n            try:\n                body=json.loads(self.rfile.read(length));app.control(body[\'action\'],body.get(\'values\'))\n                self.send(200,{\'ok\':True})\n            except (KeyError,TypeError,ValueError) as exc:self.send(400,{\'error\':str(exc)})\n    http=ThreadingHTTPServer((host,port),Handler)\n    http.daemon_threads=True\n    return http\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'--port\',type=int,default=8765)\n    p.add_argument(\'--output\',type=Path,default=e.ROOT/\'runs/live\')\n    a=p.parse_args();app=LiveApp(a.output);http=server(app,port=a.port)\n    def interrupt(*_args):raise KeyboardInterrupt()\n    signal.signal(signal.SIGTERM,interrupt)\n    print(f\'LIVE_SIMULATION_READY http://127.0.0.1:{a.port}/\',flush=True)\n    try:http.serve_forever(poll_interval=.2)\n    except KeyboardInterrupt:pass\n    finally:http.server_close();app.close()\n', 'real_live.py': '"""Hat Yai real-road prototype. Generated signals and simulated traffic by default."""\nimport argparse\nimport json\nimport signal\nfrom pathlib import Path\nimport experiment as e\nfrom live_server import LiveApp,server\nfrom real_network import build_real_network,generate_real_demand,learn_real_turns\nfrom traffic_inputs import QueueTape,counts_demand\n\n\nclass RealRoadApp(LiveApp):\n    def __init__(self,folder,counts_csv=None,queues_csv=None):\n        self.counts_csv=Path(counts_csv) if counts_csv else None\n        self.queues_csv=Path(queues_csv) if queues_csv else None\n        def demand(folder,net,scenario,seed,demand_s,scale):\n            if self.counts_csv:\n                if scale!=1:raise ValueError(\'Measured entry totals cannot be scaled; set demand multiplier to 1.0\')\n                return counts_demand(self.counts_csv,folder,net,seed,demand_s)\n            return generate_real_demand(folder,net,scenario,seed,demand_s,scale)\n        factory=(lambda lanes,capacities:QueueTape(self.queues_csv,lanes,capacities)) if self.queues_csv else None\n        super().__init__(folder,network_builder=build_real_network,demand_builder=demand,\n            turn_learner=lambda demands:learn_real_turns(demands,self.net),observation_factory=factory)\n        audit=json.loads((e.ROOT/\'data/network_audit.json\').read_text())\n        labels={x[\'controller_id\']:x[\'label\'] for x in audit[\'signal_inventory\']}\n        for node in self.scene[\'nodes\']:node[\'label\']=labels[node[\'id\']]\n        controlled={incoming.getID():tid for tid in self.tls for incoming,outgoing,index in self.net.getTLS(tid).getConnections()}\n        for lane in self.scene[\'lanes\']:\n            lane[\'controller_id\']=controlled.get(lane[\'id\'])\n            lane[\'road_name\']=self.net.getLane(lane[\'id\']).getEdge().getName()\n        self.scene.update(attribution=\'© OpenStreetMap contributors\',place=\'Hat Yai\',\n            field_ready=False,model_notice=\'Lane/turn rules and signal timings remain unverified.\',\n            traffic_source=\'file entry totals; modeled destinations; provenance unverified\' if self.counts_csv else \'synthetic entry demand\',\n            sensor_source=\'queue CSV; provenance unverified\' if self.queues_csv else \'simulated detectors\',\n            file_entry_counts=bool(self.counts_csv),file_queue_observations=bool(self.queues_csv))\n        self.selection={\'predictive_config\':{\'horizon_s\':20,\'receiving_power\':1,\'message_weight\':.3,\'moving_weight\':.2},\n                        \'fixed_green_by_load\':{\'0.3\':20,\'0.6\':20,\'1.0\':20}}\n        self.turns=None\n        self.config.update(seed=22777,scale=1.0 if self.counts_csv else .6,\n                           controller=\'fixed\' if self.queues_csv else \'sumo_actuated\')\n\n    def control(self,action,values=None):\n        if action==\'start\' and self.queues_csv and (values or {}).get(\'controller\',self.config[\'controller\'])==\'sumo_actuated\':\n            raise ValueError(\'The native loop-detector reference does not consume the recorded queue CSV\')\n        return super().control(action,values)\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--port\',type=int,default=8765);p.add_argument(\'--counts-csv\',type=Path);p.add_argument(\'--queues-csv\',type=Path)\n    a=p.parse_args();app=RealRoadApp(e.ROOT/\'runs/live\',a.counts_csv,a.queues_csv);http=server(app,port=a.port)\n    def stop(*_):raise KeyboardInterrupt()\n    signal.signal(signal.SIGTERM,stop)\n    print(f\'HAT_YAI_LIVE_READY http://127.0.0.1:{a.port}/ — real roads, unverified simulation model\',flush=True)\n    try:http.serve_forever(poll_interval=.2)\n    except KeyboardInterrupt:pass\n    finally:http.server_close();app.close()\n', 'real_network.py': '"""Bounded Hat Yai OSM import. Real geometry; uncalibrated simulation inputs."""\nfrom __future__ import annotations\nfrom collections import defaultdict,Counter\nimport csv\nimport json\nimport math\nfrom pathlib import Path\nimport random\nimport subprocess\nimport xml.etree.ElementTree as ET\nimport experiment as e\n\nDATA=e.ROOT/\'data\'\n_OD_CACHE={}\n\n\ndef signal_xy(net,tid):\n    try:return net.getNode(tid).getCoord()\n    except KeyError:\n        coords={tuple(c[0].getEdge().getToNode().getCoord()) for c in net.getTLS(tid).getConnections()}\n        return tuple(sum(p[i] for p in coords)/len(coords) for i in [0,1])\n\n\ndef load_network(path):\n    net=e.sumolib.net.readNet(str(path));xml=ET.parse(path).getroot();tls={}\n    for logic in xml.findall(\'tlLogic\'):\n        phases=[{\'state\':p.get(\'state\'),\'duration\':float(p.get(\'duration\'))} for p in logic.findall(\'phase\')]\n        greens=[i for i,p in enumerate(phases) if \'G\' in p[\'state\']]\n        if not 2<=len(greens)<=8:raise ValueError(f\'Unsupported approach count: {logic.get("id")}\')\n        if any(\'g\' in p[\'state\'] for p in phases):raise ValueError(\'Permissive green needs separate validation\')\n        if len(phases)!=3*len(greens):raise ValueError(\'Unsupported clearance layout\')\n        for i in greens:\n            assert i%3==0\n            assert \'y\' in phases[i+1][\'state\'] and phases[i+1][\'duration\']==3\n            assert set(phases[i+2][\'state\'])=={\'r\'} and phases[i+2][\'duration\']==2\n        tls[logic.get(\'id\')]={\'phases\':phases,\'greens\':greens}\n    if not tls:raise ValueError(\'No supported signal controllers found\')\n    return Path(path),net,tls\n\n\ndef prepare(force=False):\n    DATA.mkdir(exist_ok=True)\n    source=DATA/\'hat_yai_roads.osm.xml\';raw=DATA/\'hat_yai.raw.net.xml\';final=DATA/\'hat_yai.net.xml\'\n    if not source.exists():raise FileNotFoundError(\'The bundled, provenance-tracked OSM road extract is missing\')\n    command=[e.binary(\'netconvert\'),\'--osm-files\',str(source),\'--output-file\',str(raw),\n        \'--lefthand\',\'--geometry.remove\',\'--junctions.join\',\'--tls.guess-signals\',\n        \'--tls.discard-simple\',\'--tls.join\',\'--tls.layout\',\'incoming\',\'--tls.yellow.time\',\'3\',\n        \'--tls.allred.time\',\'2\',\'--tls.green.time\',\'20\',\'--no-turnarounds\',\n        \'--keep-edges.by-vclass\',\'passenger\',\'--keep-edges.in-geo-boundary\',\'100.470,7.001,100.483,7.014\',\n        \'--osm.turn-lanes\',\'true\',\'--output.street-names\',\'true\',\'--output.original-names\',\'true\']\n    if force or not raw.exists():\n        p=subprocess.run(command,capture_output=True,text=True)\n        (DATA/\'netconvert.log\').write_text(p.stdout+p.stderr)\n        if p.returncode:raise RuntimeError(\'OSM conversion failed; see data/netconvert.log\')\n    # Explicit demonstration assumption, NOT a claimed legal speed limit.\n    tree=ET.parse(raw);capped=0\n    for lane in tree.getroot().findall(\'./edge/lane\'):\n        if float(lane.get(\'speed\'))>13.89:lane.set(\'speed\',\'13.89\');capped+=1\n    tree.getroot().insert(0,ET.Comment(\' © OpenStreetMap contributors, ODbL 1.0. Simulation lane speeds capped at 13.89 m/s; unverified signal/lane/turn model. \'))\n    ET.indent(tree);tree.write(final,encoding=\'utf-8\',xml_declaration=True)\n    nf,net,tls=load_network(final)\n    r=ET.parse(source).getroot();road_tags=[]\n    for w in r.findall(\'way\'):\n        tags={x.get(\'k\'):x.get(\'v\') for x in w.findall(\'tag\')}\n        if \'highway\' in tags:road_tags.append(tags)\n    warnings=[l for l in (DATA/\'netconvert.log\').read_text().splitlines() if \'Warning:\' in l]\n    inventory=[]\n    for i,(tid,item) in enumerate(sorted(tls.items()),1):\n        x,y=signal_xy(net,tid);lon,lat=net.convertXY2LonLat(x,y)\n        inventory.append({\'label\':f\'J{i:02d}\',\'controller_id\':tid,\'latitude\':lat,\'longitude\':lon,\n            \'green_stages\':len(item[\'greens\']),\'signal_plan_source\':\'SUMO-generated; not field verified\',\n            \'site_verified\':False})\n    audit={\'source_sha256\':e.sha(source),\'network_sha256\':e.sha(final),\'sumo_version\':e.sim.getVersion()[1],\n        \'road_edges\':len(net.getEdges()),\'network_nodes\':len(net.getNodes()),\'signal_controllers\':len(tls),\n        \'green_stage_counts\':sorted(len(x[\'greens\']) for x in tls.values()),\n        \'source_highway_ways\':len(road_tags),\'ways_with_explicit_maxspeed\':sum(\'maxspeed\' in t for t in road_tags),\n        \'ways_with_explicit_lanes\':sum(\'lanes\' in t for t in road_tags),\n        \'lanes_with_demo_speed_cap\':capped,\'demo_speed_cap_mps\':13.89,\n        \'warnings\':warnings,\'unresolved_restriction_warning_lines\':sum(\'restriction\' in w.lower() for w in warnings),\n        \'field_ready\':False,\'signal_inventory\':inventory,\n        \'status\':\'real road geometry; inferred lane/turn/signal model; synthetic traffic by default\',\n        \'attribution\':\'© OpenStreetMap contributors, ODbL 1.0\',\n        \'conversion_command\':command}\n    e.write_json(DATA/\'network_audit.json\',audit)\n    with (DATA/\'signal_inventory.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=list(inventory[0]));w.writeheader();w.writerows(inventory)\n    return nf,net,tls\n\n\ndef build_real_network(_folder=None):\n    final=DATA/\'hat_yai.net.xml\';audit=DATA/\'network_audit.json\'\n    if not final.exists() or not audit.exists():return prepare()\n    assert json.loads(audit.read_text())[\'network_sha256\']==e.sha(final),\'Road network changed; rerun prepare\'\n    return load_network(final)\n\n\ndef od_pool(net):\n    key=e.sha(DATA/\'hat_yai.net.xml\')\n    if key in _OD_CACHE:return _OD_CACHE[key]\n    edges=[x for x in net.getEdges() if x.allows(\'passenger\')]\n    sources=[x for x in edges if len([z for z in x.getFromNode().getIncoming() if z.allows(\'passenger\')])<=1]\n    sinks=[x for x in edges if len([z for z in x.getToNode().getOutgoing() if z.allows(\'passenger\')])<=1]\n    xmin,ymin,xmax,ymax=net.getBoundary();cx,cy=(xmin+xmax)/2,(ymin+ymax)/2\n    def choose(items,origin):\n        def point(edge):return (edge.getFromNode() if origin else edge.getToNode()).getCoord()\n        ranked=sorted(items,key=lambda x:-math.hypot(point(x)[0]-cx,point(x)[1]-cy))\n        ranked=ranked[:max(24,len(ranked)//2)]\n        ranked.sort(key=lambda x:math.atan2(point(x)[1]-cy,point(x)[0]-cx))\n        return [ranked[round(i*(len(ranked)-1)/min(23,len(ranked)-1))] for i in range(min(24,len(ranked)))]\n    pool=[]\n    for origin in choose(sources,True):\n        for dest in choose(sinks,False):\n            if origin.getID()==dest.getID():continue\n            path,cost=net.getShortestPath(origin,dest,vClass=\'passenger\')\n            if not path:continue\n            length=sum(x.getLength() for x in path)\n            if not 600<=length<=4000:continue\n            if not any(x.getToNode().getType().startswith(\'traffic_light\') for x in path):continue\n            ox,oy=origin.getFromNode().getCoord();dx,dy=dest.getToNode().getCoord()\n            pool.append({\'entry\':origin.getID(),\'exit\':dest.getID(),\'edges\':[x.getID() for x in path],\n                         \'axis\':\'horizontal\' if abs(dx-ox)>=abs(dy-oy) else \'vertical\',\'length_m\':length})\n    if len(pool)<10:raise ValueError(\'Insufficient connected cross-signal OD pairs\')\n    _OD_CACHE[key]=pool\n    return pool\n\n\ndef write_routes(folder,requests,name,seed):\n    folder.mkdir(parents=True,exist_ok=True);rng=random.Random(seed+330001)\n    root=ET.Element(\'routes\');ET.SubElement(root,\'vType\',id=\'car\',length=\'5\',minGap=\'2.5\',\n        accel=\'2.6\',decel=\'4.5\',sigma=\'0\',speedFactor=\'1.0\',maxSpeed=\'13.89\')\n    for i,(depart,edges) in enumerate(sorted(requests,key=lambda x:x[0])):\n        v=ET.SubElement(root,\'vehicle\',id=f\'v{i:06d}\',type=\'car\',depart=f\'{depart:.3f}\',\n                        departLane=\'best\',departSpeed=\'max\',speedFactor=f\'{rng.uniform(.9,1.1):.5f}\')\n        ET.SubElement(v,\'route\',edges=\' \'.join(edges))\n    ET.indent(root);path=folder/name;ET.ElementTree(root).write(path,encoding=\'utf-8\',xml_declaration=True)\n    return path,{v.get(\'id\'):float(v.get(\'depart\')) for v in root.findall(\'vehicle\')}\n\n\ndef generate_real_demand(folder,net,scenario,seed,demand_s,scale=1.0):\n    """Synthetic entry demand over real roads; not a measured Hat Yai demand model."""\n    pool=od_pool(net);rng=random.Random(seed);requests=[]\n    for t in range(demand_s):\n        rate=.5\n        if scenario==\'surge\':rate=1.2 if .3<=t/demand_s<.65 else .4\n        if rng.random()<min(1.0,rate*scale):\n            choices=pool\n            if scenario==\'directional_switch\':\n                dominant=\'horizontal\' if t/demand_s<.5 else \'vertical\'\n                if rng.random()<.8:choices=[p for p in pool if p[\'axis\']==dominant] or pool\n            route=rng.choice(choices);requests.append((t+rng.random(),route[\'edges\']))\n    return write_routes(folder,requests,f\'{scenario}_seed{seed}_d{demand_s}_load{scale:g}.rou.xml\',seed)\n\n\ndef learn_real_turns(demands,net):\n    counts=defaultdict(Counter)\n    for file in demands:\n        for v in ET.parse(file).getroot().findall(\'vehicle\'):\n            edges=v.find(\'route\').get(\'edges\').split()\n            for a,b in zip(edges,edges[1:]):counts[a][b]+=1\n    result={}\n    for edge_id,outcounts in counts.items():\n        edge=net.getEdge(edge_id)\n        for lane in edge.getLanes():\n            outs=defaultdict(list)\n            for conn in lane.getOutgoing():outs[conn.getToLane().getEdge().getID()].append(conn.getToLane().getID())\n            values={out:outcounts[edge]/len(lanes) for edge,lanes in outs.items() for out in lanes if outcounts[edge]>0}\n            if values:result[lane.getID()]={k:v/sum(values.values()) for k,v in values.items()}\n    return result\n\n\ndef export_lane_mapping(net,tls,folder):\n    pool=od_pool(net);entries=sorted({p[\'entry\'] for p in pool})\n    with (folder/\'entry_edges.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.writer(f);w.writerow([\'entry_edge_id\',\'road_name\',\'longitude\',\'latitude\',\'allowed_destination_count\'])\n        for edge_id in entries:\n            edge=net.getEdge(edge_id);lon,lat=net.convertXY2LonLat(*edge.getFromNode().getCoord())\n            w.writerow([edge_id,edge.getName(),lon,lat,sum(p[\'entry\']==edge_id for p in pool)])\n    with (folder/\'sensor_lane_mapping.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.writer(f);w.writerow([\'lane_id\',\'controller_id\',\'road_name\',\'longitude\',\'latitude\',\'length_m\'])\n        seen=set()\n        for tid in tls:\n            for incoming,outgoing,index in net.getTLS(tid).getConnections():\n                if incoming.getID() in seen:continue\n                seen.add(incoming.getID());lon,lat=net.convertXY2LonLat(*incoming.getShape()[-1])\n                w.writerow([incoming.getID(),tid,incoming.getEdge().getName(),lon,lat,incoming.getLength()])\n\n\nif __name__==\'__main__\':\n    nf,n,t=prepare(force=True);export_lane_mapping(n,t,DATA)\n    print(\'Prepared\',len(n.getEdges()),\'road edges,\',len(t),\'signal controllers,\',len(od_pool(n)),\'modeled OD routes\')\n', 'traffic_inputs.py': '"""Validated file interfaces for real counts; no camera access is assumed."""\nfrom collections import defaultdict\nimport csv\nimport math\nfrom pathlib import Path\nimport random\nfrom real_network import od_pool,write_routes\n\n\ndef number(value,name):\n    x=float(value)\n    if not math.isfinite(x) or x<0:raise ValueError(f\'{name} must be finite and nonnegative\')\n    return x\n\n\nclass QueueTape:\n    def __init__(self,file,lanes,capacities,stale_after_s=15,minimum_coverage=.8):\n        self.capacities=capacities;self.lanes=list(lanes);self.cursor=0;self.last={}\n        self.stale_after_s=stale_after_s;self.minimum_coverage=minimum_coverage\n        self.fallback=True;self.rows=[];seen=set()\n        with Path(file).open(newline=\'\') as f:\n            reader=csv.DictReader(f)\n            required={\'time_s\',\'lane_id\',\'queue_vehicles\',\'vehicles\'}\n            if not required.issubset(reader.fieldnames or []):raise ValueError(\'Queue CSV needs \'+\',\'.join(sorted(required)))\n            for row in reader:\n                t=number(row[\'time_s\'],\'time_s\');lane=row[\'lane_id\']\n                if lane not in capacities:raise ValueError(f\'Unknown sensor lane: {lane}\')\n                q=number(row[\'queue_vehicles\'],\'queue_vehicles\');n=number(row[\'vehicles\'],\'vehicles\')\n                if q>n or n>capacities[lane]+1:raise ValueError(f\'Invalid queue/vehicle count for {lane}\')\n                if (t,lane) in seen:raise ValueError(\'Duplicate queue sample\')\n                seen.add((t,lane));self.rows.append((t,lane,q,n))\n        if not self.rows:raise ValueError(\'Queue CSV has no observations\')\n        self.rows.sort();self.stats={\'source\':\'recorded_queue_csv\',\'sample_count\':len(self.rows),\n                                    \'fallback_steps\':0,\'last_coverage\':0.0}\n\n    def update(self,t):\n        while self.cursor<len(self.rows) and self.rows[self.cursor][0]<=t:\n            stamp,lane,q,n=self.rows[self.cursor];self.last[lane]=(stamp,q,n);self.cursor+=1\n        current={};fresh=0\n        for lane in self.lanes:\n            record=self.last.get(lane)\n            if record and t-record[0]<=self.stale_after_s:\n                current[lane]=record[1:];fresh+=1\n            else:current[lane]=(0.0,0.0) # no simulator-truth substitution\n        coverage=fresh/max(1,len(self.lanes));self.fallback=coverage<self.minimum_coverage\n        self.stats[\'last_coverage\']=round(coverage,4)\n        self.stats[\'fallback_steps\']+=int(self.fallback)\n        return current\n\n\ndef counts_demand(file,folder,net,seed,demand_s):\n    pool=od_pool(net);by_entry=defaultdict(list)\n    for route in pool:by_entry[route[\'entry\']].append(route)\n    rng=random.Random(seed);requests=[];intervals=defaultdict(list)\n    with Path(file).open(newline=\'\') as f:\n        reader=csv.DictReader(f)\n        fields={\'start_s\',\'end_s\',\'entry_edge_id\',\'vehicles\'}\n        if not fields.issubset(reader.fieldnames or []):raise ValueError(\'Entry counts CSV needs \'+\',\'.join(sorted(fields)))\n        rows=list(reader)\n    if not rows:raise ValueError(\'Entry counts CSV has no measurement intervals\')\n    for row in rows:\n        start=number(row[\'start_s\'],\'start_s\');end=number(row[\'end_s\'],\'end_s\')\n        count=number(row[\'vehicles\'],\'vehicles\');entry=row[\'entry_edge_id\']\n        if count!=int(count):raise ValueError(\'Entry counts must be integer totals\')\n        if not start<end<=demand_s:raise ValueError(\'Entry intervals must lie inside the demand period\')\n        if entry not in by_entry:raise ValueError(f\'Unknown or unroutable entry edge: {entry}\')\n        if any(start<b and end>a for a,b in intervals[entry]):raise ValueError(\'Overlapping count intervals for the same entry\')\n        intervals[entry].append((start,end))\n        # Total bin count is observed; within-bin timing and destination are modeled.\n        for i in range(int(count)):\n            depart=start+(i+.5)*(end-start)/max(1,int(count));route=rng.choice(by_entry[entry])\n            requests.append((depart,route[\'edges\']))\n    return write_routes(folder,requests,f\'entry_counts_seed{seed}.rou.xml\',seed)\n', 'real_pilot.py': '"""Real-road integration checks with SYNTHETIC demand; not field efficacy."""\nfrom pathlib import Path\nimport json\nimport csv\nimport statistics\nimport argparse\nimport experiment as e\nfrom real_network import build_real_network,generate_real_demand,learn_real_turns\n\n\ndef main(folder=None):\n    folder=Path(folder) if folder else e.ROOT/\'runs/real_pilot\';folder.mkdir(parents=True,exist_ok=True)\n    nf,net,tls=build_real_network();inputs=folder/\'inputs\';inputs.mkdir(exist_ok=True)\n    dev,_=generate_real_demand(inputs,net,\'balanced\',201,900,.6)\n    turns=learn_real_turns([dev],net);e.write_json(folder/\'modeled_turn_weights.json\',turns)\n    signature={\'sources\':{str(p.relative_to(e.ROOT)):e.sha(p) for p in [e.ROOT/\'experiment.py\',e.ROOT/\'real_network.py\',e.ROOT/\'real_pilot.py\',e.ROOT/\'src/controllers.cpp\']},\n        \'network_sha256\':e.sha(nf),\'seeds\':[32777,33777,34777],\'demand_s\':600,\'maximum_drain_s\':900,\n        \'load_scales\':[.6,1.0],\'scenarios\':[\'balanced\',\'directional_switch\'],\n        \'controllers\':[\'fixed\',\'sumo_actuated\',\'predictive\'],\'field_measured_demand\':False,\n        \'fixed_green_s\':20,\'fixed_timing_source\':\'nominal generated baseline; not the actual municipal program\',\n        \'controller_parameters\':\'frozen from synthetic Stage 2; not tuned to this topology\',\n        \'turn_weights\':\'modeled development routes, not observed field turns\'}\n    manifest=folder/\'manifest.json\'\n    if manifest.exists():assert json.loads(manifest.read_text())==signature,\'Pilot source/configuration changed; use a new output folder\'\n    else:e.write_json(manifest,signature)\n    rows=[]\n    for scenario in signature[\'scenarios\']:\n        for load in signature[\'load_scales\']:\n            for seed in signature[\'seeds\']:\n                d,s=generate_real_demand(inputs,net,scenario,seed,600,load)\n                for c in signature[\'controllers\']:\n                    dest=folder/\'evaluation\'/f\'{scenario}_l{load}_seed{seed}_{c}\'\n                    if (dest/\'result.json\').exists():\n                        r=json.loads((dest/\'result.json\').read_text());assert r[\'demand_sha256\']==e.sha(d)\n                    else:r=e.run_one(dest,nf,tls,d,s,c,\'perfect\',seed,600,900,20,\n                        c==\'predictive\' and seed==32777 and scenario==\'balanced\' and load==.6,\n                        turn_weights=turns,stop_when_clear=True)\n                    r[\'load_scale\']=load;e.write_json(dest/\'result.json\',r);rows.append(r)\n                    print(f"{len(rows):02d}/36 {scenario} load={load} seed={seed} {c} valid={r[\'valid_run\']} completion={r[\'completion_fraction\']:.1%}",flush=True)\n    with (folder/\'results.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows({k:json.dumps(v) if isinstance(v,dict) else v for k,v in r.items()} for r in rows)\n    report={\'runs\':len(rows),\'all_valid\':all(r[\'valid_run\'] for r in rows),\n        \'all_metric_ledgers_agree\':all(r[\'metric_ledger_pass\'] for r in rows),\n        \'collision_events\':sum(r[\'colliding_vehicle_events\'] for r in rows),\n        \'teleport_events\':sum(r[\'teleport_vehicle_events\'] for r in rows),\n        \'cleared_runs_by_controller\':{c:sum(r[\'cleared\'] for r in rows if r[\'controller\']==c) for c in signature[\'controllers\']},\n        \'field_ready\':False,\'demand_calibrated\':False,\'site_signal_plans_verified\':False,\n        \'interpretation\':\'software integration on real road geometry, not evidence of field effectiveness\'}\n    e.write_json(folder/\'audit.json\',report)\n    print(json.dumps(report),flush=True)\n    assert report[\'all_valid\'],\'Invalid integration runs retained; investigate before accepting model\'\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\',type=Path)\n    main(parser.parse_args().output)\n', 'src/controllers.cpp': '#include <algorithm>\n#include <cmath>\n#include <limits>\n\n// Stage-1 heuristics, not a claim of a novel max-pressure formulation.\n// Each row is one served incoming lane. Downstream values use equal weights\n// across its allowed outgoing lanes; turning fractions are not observed.\nextern "C" double phase_score(int mode, int n, const double* queue,\n    const double* downstream_queue, const double* downstream_fill,\n    const double* announced, const double* moving) {\n    double result = 0.0;\n    for (int i = 0; i < n; ++i) {\n        double pressure = queue[i] - downstream_queue[i];\n        if (mode == 0) { result += queue[i]; continue; }\n        if (mode == 1) { result += pressure; continue; }\n        double receiving = std::pow(std::clamp(1.0-downstream_fill[i], 0.0, 1.0), 2);\n        if (mode == 3) pressure += 0.30*announced[i] + 0.20*moving[i];\n        result += pressure * receiving;\n    }\n    return result;\n}\n\nextern "C" int should_switch(double current_score, double next_score,\n    double elapsed, double min_green, double max_green, double hysteresis) {\n    if (elapsed < min_green) return 0;\n    if (elapsed >= max_green) return 1;\n    return next_score > current_score + hysteresis;\n}\n\n// Approximate finite-horizon queue-discharge benefit. This is a heuristic,\n// not a calibrated microscopic model or a claim of optimal MPC.\nextern "C" double discharge_score(int n, const double* queue,\n    const double* downstream_queue, const double* downstream_fill,\n    const double* announced, const double* moving, double horizon,\n    double lost_time, double receiving_power, double message_weight,\n    double moving_weight) {\n    const double service_time = std::max(0.0, horizon-lost_time);\n    const double saturation = 0.45; // assumed vehicles/s on one approach\n    double result = 0;\n    for (int i=0;i<n;++i) {\n        const double demand = std::max(0.0,queue[i]+message_weight*announced[i]+moving_weight*moving[i]);\n        const double free_fraction = std::clamp(1-downstream_fill[i],0.0,1.0);\n        const double released = std::min(demand,saturation*service_time);\n        const double wait_saved = released*service_time-released*released/(2*saturation);\n        result += wait_saved*std::pow(free_fraction,receiving_power)\n                  - 0.25*released*downstream_queue[i];\n    }\n    return result;\n}\n', 'requirements.txt': 'eclipse-sumo==1.25.0\nlibsumo==1.25.0\ntraci==1.25.0\nsumolib==1.25.0\nnumpy>=1.26,<3\nmatplotlib>=3.8,<4\npyproj==3.7.2\n', 'viewer/index.html': '<!doctype html>\n<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>Adaptive Signals · Hat Yai prototype</title>\n<style>\n:root{color-scheme:dark;--bg:#09121e;--panel:#101e2e;--line:#25384c;--text:#e6eef8;--muted:#95a9bf;--teal:#54d5bd;--amber:#f7ba58;--red:#f17b7b}*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--text);font:14px system-ui,-apple-system,Segoe UI,sans-serif}header{padding:20px 28px;border-bottom:1px solid var(--line);display:flex;align-items:center;justify-content:space-between;gap:16px}h1{font-size:23px;letter-spacing:-.7px;margin:0}header p{margin:4px 0 0;color:var(--muted);font-size:12px}.brand{display:flex;gap:13px;align-items:center}.mark{display:grid;grid-template-columns:repeat(2,8px);gap:4px;padding:10px;border:1px solid var(--line);border-radius:12px}.mark i{width:8px;height:8px;background:var(--teal);border-radius:2px}.mark i:nth-child(2){opacity:.3}.mark i:nth-child(3){opacity:.5}.badge{border:1px solid var(--line);border-radius:20px;padding:7px 12px;font-size:11px;letter-spacing:.8px;color:var(--muted)}.status{color:var(--teal);display:flex;align-items:center;gap:8px}.dot{width:6px;height:6px;background:currentColor;border-radius:50%}.layout{display:grid;grid-template-columns:268px minmax(0,1fr);min-height:calc(100vh - 86px)}aside{padding:23px 20px;border-right:1px solid var(--line)}.eyebrow{font-size:10px;font-weight:700;letter-spacing:1.6px;color:var(--muted);text-transform:uppercase;margin:0 0 18px}label{display:block;font-size:12px;color:var(--muted);margin:18px 0 8px}select,input[type=number]{font:inherit;color:var(--text);width:100%;background:#142438;border:1px solid #30465e;padding:11px;border-radius:7px;outline:none}select:focus,input:focus{border-color:var(--teal)}input[type=range]{width:100%;accent-color:var(--teal);cursor:pointer}.sliderlabels{display:flex;justify-content:space-between;font-size:10px;color:var(--muted)}.row{display:flex;align-items:center;justify-content:space-between;gap:8px}.value{color:var(--teal);font-weight:650}button{font:inherit;border:1px solid #365069;background:#182b40;color:var(--text);padding:10px 14px;border-radius:7px;cursor:pointer;transition:background .15s}button:hover{background:#24415d}button:disabled{opacity:.4;cursor:default}.primary{width:100%;background:var(--teal);border-color:var(--teal);color:#06231f;font-weight:700;margin-top:22px}.primary:hover{background:#8be9d5}.note{color:var(--muted);font-size:11px;line-height:1.65}.divider{height:1px;background:var(--line);margin:24px 0}.inspector{font-size:12px;line-height:1.9}.approach{display:flex;justify-content:space-between;border-bottom:1px solid #1b2d40;padding:4px 0}.green{color:var(--teal)}main{padding:22px 26px;min-width:0}.topline{display:flex;justify-content:space-between;align-items:center;margin-bottom:18px}.topline h2{font-size:15px;margin:0}.clock{font-variant-numeric:tabular-nums;font-size:13px;color:var(--muted)}.kpis{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:16px}.kpi{background:var(--panel);border:1px solid var(--line);padding:15px 17px;border-radius:9px}.kpi .number{font-size:28px;letter-spacing:-1px;font-variant-numeric:tabular-nums;margin:4px 0}.kpi .caption{font-size:11px;color:var(--muted)}.kpi .detail{font-size:10px;color:var(--muted)}.mapcard{border:1px solid var(--line);background:#0c1827;border-radius:10px;overflow:hidden}.mapbar{padding:12px 16px;border-bottom:1px solid var(--line);display:flex;justify-content:space-between;align-items:center;gap:10px}.mapbar .title{font-size:12px}.legend{display:flex;align-items:center;gap:12px;font-size:10px;color:var(--muted)}.legend b{display:inline-block;width:6px;height:6px;border-radius:2px;margin-right:4px}.mapwrap{height:clamp(360px,49vh,650px);position:relative}#map{width:100%;height:100%;display:block;cursor:crosshair}.maphint{position:absolute;left:15px;bottom:12px;font-size:10px;color:#7790a9;pointer-events:none}.controls{display:flex;align-items:center;gap:7px;padding:12px 15px;border-top:1px solid var(--line);flex-wrap:wrap}.controls button{padding:8px 13px;font-size:12px}.speed{display:flex;align-items:center;gap:9px;margin-left:auto;color:var(--muted);font-size:11px}.speed select{width:76px;padding:6px;font-size:12px}.chartcard{margin-top:14px;padding:15px 16px;background:var(--panel);border:1px solid var(--line);border-radius:9px}.charthead{display:flex;align-items:center;justify-content:space-between;font-size:12px}.chartwrap{height:105px;margin-top:6px}#chart{display:block;width:100%;height:100%}.footer{display:flex;justify-content:space-between;font-size:10px;color:var(--muted);padding-top:12px;gap:15px}.error{display:none;background:#3b1c28;color:#ffc0c0;border:1px solid #9e4456;padding:12px;margin-bottom:14px;border-radius:7px;font-size:12px}.disclosure{font-size:10px;letter-spacing:.6px;color:var(--muted)}@media(max-width:950px){header{padding:17px}.layout{grid-template-columns:220px minmax(0,1fr)}aside{padding:20px 14px}main{padding:18px 14px}.kpis{gap:7px}.kpi{padding:12px}.kpi .number{font-size:23px}.legend{gap:7px}.badge{display:none}}@media(max-width:650px){.layout{display:flex;flex-direction:column}aside{border-right:0;border-bottom:1px solid var(--line);display:grid;grid-template-columns:1fr 1fr;gap:0 14px}aside>.eyebrow,aside>.primary,aside>.note,aside>.divider,aside>.inspector{grid-column:1/-1}aside>.divider,aside>.inspector{display:none}label{margin-top:12px}.kpis{grid-template-columns:repeat(2,1fr)}.mapwrap{height:420px}.mapbar{flex-wrap:wrap}.disclosure{display:none}}\n</style></head><body>\n<header><div class="brand"><div class="mark" aria-hidden="true"><i></i><i></i><i></i><i></i></div><div><h1>Adaptive Signals</h1><p>Hat Yai · real-road simulation prototype</p></div></div><span class="badge">REAL ROADS · SIMULATED TRAFFIC</span><div class="status" id="status"><i class="dot"></i><span>Connecting</span></div></header>\n<div class="layout"><aside>\n<p class="eyebrow">Experiment setup</p>\n<div><label for="controller">Signal controller</label><select id="controller"><option value="predictive">Discharge lookahead</option><option value="coordinated">Stage-1 coordinated</option><option value="calibrated_pressure">Turn-calibrated pressure</option><option value="sumo_actuated">SUMO actuated reference</option><option value="fixed">Development-tuned fixed</option><option value="receiving_aware">Receiving capacity only</option><option value="message_only">Upstream messages only</option><option value="moving_only">Moving vehicles only</option><option value="max_pressure">Equal-turn pressure</option></select></div>\n<div><label for="scenario">Traffic pattern</label><select id="scenario"><option value="directional_switch">Changing direction</option><option value="balanced">Balanced demand</option><option value="surge">Demand surge</option></select></div>\n<div><label class="row" for="load">Demand multiplier <span class="value" id="loadValue">0.6×</span></label><input type="range" id="load" min="0.1" max="1.5" value="0.6" step="0.1"><div class="sliderlabels"><span>Lighter</span><span>Heavier</span></div></div>\n<div><label for="regime">Controller sensors</label><select id="regime"><option value="perfect">Exact lane counts</option><option value="degraded">Delayed, noisy & missing</option></select></div>\n<div><label for="duration">Traffic arrival period</label><select id="duration"><option value="300">5 minutes</option><option value="600" selected>10 minutes</option><option value="900">15 minutes</option><option value="1800">30 minutes</option></select></div>\n<div><label for="seed">Traffic seed</label><input id="seed" type="number" min="0" max="2147483647" value="12777"></div>\n<button class="primary" id="start">Start / reset run</button><p class="note">Settings apply on reset. Reuse the seed to compare identical traffic requests. Sensor settings affect the controller; counters show simulation truth.</p>\n<div class="divider"></div><div class="inspector"><p class="eyebrow" id="nodeTitle">Intersection B1</p><div id="approaches">Select an intersection on the map.</div></div>\n</aside><main>\n<div class="topline"><h2>Network overview</h2><span class="clock" id="clock">00:00 simulation time</span></div>\n<div class="error" id="error" role="alert"></div>\n<div class="kpis"><div class="kpi"><div class="caption">Completed trips</div><div class="number" id="arrived">—</div><div class="detail" id="requested">Awaiting a live run</div></div><div class="kpi"><div class="caption">Vehicles in network</div><div class="number" id="active">—</div><div class="detail" id="pending">Includes internal junctions</div></div><div class="kpi"><div class="caption">Stopped vehicles</div><div class="number" id="halted">—</div><div class="detail">Speed below 0.1 m/s on roads</div></div><div class="kpi"><div class="caption">Mean system time so far</div><div class="number" id="system">—</div><div class="detail">All requests made so far</div></div></div>\n<section class="mapcard"><div class="mapbar"><span class="title">Hat Yai · OpenStreetMap road geometry</span><div class="legend"><span><b style="background:#54d5bd"></b>Moving</span><span><b style="background:#f7ba58"></b>Stopped</span><span><b style="background:#f17b7b"></b>Red signal</span></div><label style="margin:0;font-size:10px"><input type="checkbox" id="overlay" checked> Detector queues</label></div><div class="mapwrap"><canvas id="map" aria-label="Live traffic simulation map"></canvas><span class="maphint">Click a signal to inspect queues. Scroll to zoom; drag to pan.</span></div><div class="controls"><button id="pause" disabled>Pause</button><button id="resume" disabled>Resume</button><button id="step" disabled>Step 1 second</button><button id="stop" disabled>Stop</button><div class="speed"><label for="speed" style="margin:0">Playback rate</label><select id="speed"><option value="1">1×</option><option value="5">5×</option><option value="10" selected>10×</option><option value="30">30×</option><option value="60">60×</option></select></div></div></section>\n<section class="chartcard"><div class="charthead"><span>Queue and entry backlog</span><div class="legend"><span><b style="background:#f7ba58"></b>Stopped on roads</span><span><b style="background:#869eec"></b>Waiting to enter</span></div></div><div class="chartwrap"><canvas id="chart" aria-label="Queue and entry backlog over time"></canvas></div></section>\n<div class="footer"><span id="message">© OpenStreetMap contributors · lane/turn rules and signal plans unverified.</span><span id="audit">Collision / teleport monitoring active</span></div>\n</main></div>\n<script>\n\'use strict\';\nconst ACCESS_KEY=__ACCESS_KEY__;\nconst $=id=>document.getElementById(id);\nlet scene=null,state=null,selected=null,busy=false,connected=false,zoom=1,pan=[0,0],dragging=false,dragged=false,lastPointer=null,initialized=false;\nconst base=new URL(\'./\',location.href);\nasync function api(path,body){const response=await fetch(new URL(\'api/\'+path,base),{method:body?\'POST\':\'GET\',headers:{\'X-Simulation-Key\':ACCESS_KEY,...(body?{\'Content-Type\':\'application/json\'}:{})},...(body?{body:JSON.stringify(body)}:{})});const data=await response.json();if(!response.ok)throw new Error(data.error||\'The simulation request failed.\');return data;}\nfunction error(message){$(\'error\').textContent=message;$(\'error\').style.display=message?\'block\':\'none\';}\nasync function command(action,values){try{error(\'\');await api(\'control\',{action,values});}catch(e){error(e.message);}}\n$(\'load\').oninput=()=>{$(\'loadValue\').textContent=Number($(\'load\').value).toFixed(1)+\'×\';};\n$(\'start\').onclick=()=>command(\'start\',{controller:$(\'controller\').value,scenario:$(\'scenario\').value,scale:Number($(\'load\').value),regime:$(\'regime\').value,seed:Number($(\'seed\').value),demand_s:Number($(\'duration\').value)});\nfor(const id of [\'pause\',\'resume\',\'step\',\'stop\'])$(id).onclick=()=>command(id);\n$(\'speed\').onchange=()=>command(\'speed\',{speed:Number($(\'speed\').value)});\n$(\'overlay\').onchange=draw;\nfunction size(canvas){const r=canvas.getBoundingClientRect(),d=window.devicePixelRatio||1;canvas.width=Math.round(r.width*d);canvas.height=Math.round(r.height*d);const ctx=canvas.getContext(\'2d\');ctx.setTransform(d,0,0,d,0,0);return {ctx,w:r.width,h:r.height};}\nlet project=null;\nfunction draw(){const {ctx,w,h}=size($(\'map\'));ctx.fillStyle=\'#0c1827\';ctx.fillRect(0,0,w,h);if(!scene)return;if(!selected)selected=scene.nodes[0]?.id;const [xmin,ymin,xmax,ymax]=scene.bounds;const scale=Math.min((w-64)/(xmax-xmin),(h-46)/(ymax-ymin))*zoom;const ox=(w-(xmax-xmin)*scale)/2+pan[0],oy=(h-(ymax-ymin)*scale)/2;project=(x,y)=>[ox+(x-xmin)*scale,h-oy-(y-ymin)*scale+pan[1]];\n function line(points,color,width){ctx.beginPath();points.forEach((p,i)=>{const [x,y]=project(...p);if(i)ctx.lineTo(x,y);else ctx.moveTo(x,y);});ctx.strokeStyle=color;ctx.lineWidth=width;ctx.stroke();}\n ctx.lineCap=\'butt\';for(const lane of scene.lanes){line(lane.shape,\'#24384e\',Math.max(3,4.2*scale));}\n const signalMap=new Map(state?.signals||[]);\n for(const lane of scene.lanes){const points=lane.shape,last=points[points.length-1],prev=points[points.length-2];if(!prev)continue;const end=project(...last),p=project(...prev);const dx=end[0]-p[0],dy=end[1]-p[1],len=Math.hypot(dx,dy)||1,ux=dx/len,uy=dy/len;const observed=state?.sensors?.[lane.id]?.[0]||0;\n  if($(\'overlay\').checked&&observed>0){const length=Math.min(lane.length-10,observed*7.5)*scale;ctx.beginPath();ctx.moveTo(...end);ctx.lineTo(end[0]-ux*length,end[1]-uy*length);ctx.lineWidth=Math.max(4,5*scale);ctx.strokeStyle=\'rgba(247,186,88,.32)\';ctx.stroke();}\n  const s=signalMap.get(lane.id);if(s){ctx.beginPath();ctx.moveTo(end[0]-uy*4,end[1]+ux*4);ctx.lineTo(end[0]+uy*4,end[1]-ux*4);ctx.lineWidth=2.5;ctx.strokeStyle=s===\'G\'?\'#54d5bd\':s===\'y\'?\'#f7ba58\':\'#f17b7b\';ctx.stroke();}}\n for(const node of scene.nodes){const [x,y]=project(...node.xy);if(node.id===selected){ctx.strokeStyle=\'rgba(84,213,189,.6)\';ctx.lineWidth=1;ctx.beginPath();ctx.arc(x,y,17,0,Math.PI*2);ctx.stroke();}ctx.fillStyle=\'#a6bacf\';ctx.font=\'10px system-ui\';ctx.fillText(node.label||node.id,x+13,y-12);}\n for(const v of state?.vehicles||[]){const [id,x,y,angle,speed]=v,[px,py]=project(x,y);ctx.save();ctx.translate(px,py);ctx.rotate(angle*Math.PI/180-Math.PI/2);ctx.fillStyle=speed<.1?\'#f7ba58\':speed<5?\'#b6d784\':\'#54d5bd\';ctx.fillRect(-Math.max(4.5,5*scale)/2,-Math.max(2,2*scale)/2,Math.max(4.5,5*scale),Math.max(2,2*scale));ctx.restore();}\n drawChart();inspect();}\nfunction inspect(){if(!scene)return;$(\'nodeTitle\').textContent=\'Signal \'+(scene.nodes.find(n=>n.id===selected)?.label||selected);const incoming=scene.lanes.filter(l=>l.controller_id===selected||l.to===selected),signals=new Map(state?.signals||[]);$(\'approaches\').replaceChildren();for(const lane of incoming){const div=document.createElement(\'div\');div.className=\'approach\';const a=document.createElement(\'span\'),b=document.createElement(\'span\');a.textContent=lane.road_name||(\'Approach \'+(incoming.indexOf(lane)+1));a.style.maxWidth=\'135px\';a.style.overflowWrap=\'anywhere\';b.textContent=state?.sensors?((state.sensors[lane.id]?.[0]||0).toFixed(1)+\' vehicles\'):\'—\';if(signals.get(lane.id)===\'G\'){a.className=\'green\';b.className=\'green\';}div.append(a,b);$(\'approaches\').append(div);}}\nfunction drawChart(){const {ctx,w,h}=size($(\'chart\'));ctx.clearRect(0,0,w,h);const rows=state?.history||[];const left=30,top=8,bottom=h-18,maxT=Math.max(60,state?.time_s||0),maxY=Math.max(10,...rows.flatMap(r=>[r[1],r[2]]));ctx.font=\'9px system-ui\';ctx.strokeStyle=\'#26394c\';ctx.fillStyle=\'#8da4bb\';for(let i=0;i<3;i++){const val=maxY*i/2,y=bottom-(bottom-top)*i/2;ctx.beginPath();ctx.moveTo(left,y);ctx.lineTo(w-5,y);ctx.stroke();ctx.fillText(String(Math.round(val)),0,y+3);}for(const [idx,color]of [[1,\'#f7ba58\'],[2,\'#869eec\']]){ctx.beginPath();rows.forEach((r,i)=>{const x=left+r[0]/maxT*(w-left-5),y=bottom-r[idx]/maxY*(bottom-top);if(i)ctx.lineTo(x,y);else ctx.moveTo(x,y);});ctx.strokeStyle=color;ctx.lineWidth=1.8;ctx.stroke();}ctx.fillText(\'0 s\',left,h-3);ctx.fillText(Math.round(maxT)+\' s\',w-40,h-3);}\n$(\'map\').onclick=event=>{if(dragged||!scene||!project)return;const r=$(\'map\').getBoundingClientRect(),x=event.clientX-r.left,y=event.clientY-r.top;let best=null,d=30;for(const n of scene.nodes){const p=project(...n.xy),distance=Math.hypot(x-p[0],y-p[1]);if(distance<d){best=n;d=distance;}}if(best){selected=best.id;draw();}};\nfunction render(){if(!state)return;const m=state.metrics||{},started=state.time_s>0;const status=state.status;$(\'status\').lastElementChild.textContent=status[0].toUpperCase()+status.slice(1);$(\'status\').style.color=status===\'error\'?\'#f17b7b\':status===\'paused\'?\'#f7ba58\':\'#54d5bd\';const t=state.time_s||0;$(\'clock\').textContent=String(Math.floor(t/60)).padStart(2,\'0\')+\':\'+String(t%60).padStart(2,\'0\')+\' simulation time\';for(const id of [\'arrived\',\'active\',\'halted\'])$(id).textContent=started?(m[id]??0):\'—\';$(\'system\').textContent=started?(m.mean_system_so_far_s??0)+\' s\':\'—\';$(\'requested\').textContent=started?`${m.requested} requested so far / ${m.total_requests} total`:\'Awaiting a live run\';$(\'pending\').textContent=started?`${m.pending} waiting to enter`:\'Includes internal junctions\';$(\'message\').textContent=state.message||(m.sensor_fallback?\'Recorded detector data stale/incomplete: fixed-timing fallback in simulation.\':\'© OpenStreetMap contributors · unverified lane/turn/signal model · \'+(scene?.traffic_source||\'synthetic traffic\'));$(\'audit\').textContent=started?`${m.collisions||0} collision events · ${m.teleports||0} teleports`:\'Collision / teleport monitoring active\';$(\'pause\').disabled=status!==\'running\';$(\'resume\').disabled=status!==\'paused\';$(\'step\').disabled=![\'running\',\'paused\'].includes(status);$(\'stop\').disabled=![\'running\',\'paused\',\'starting\'].includes(status);if(status===\'error\')error(state.message);draw();}\nasync function poll(){if(busy)return;busy=true;try{if(!scene)scene=await api(\'scene\');state=await api(\'state\');if(!initialized){$(\'controller\').value=state.config.controller;$(\'scenario\').value=state.config.scenario;$(\'regime\').value=state.config.regime;$(\'load\').value=state.config.scale;$(\'loadValue\').textContent=state.config.scale.toFixed(1)+\'×\';$(\'seed\').value=state.config.seed;if(scene.file_entry_counts){$(\'load\').disabled=true;$(\'scenario\').disabled=true;}if(scene.file_queue_observations){$(\'regime\').disabled=true;$(\'regime\').options[0].textContent=\'Recorded queue CSV\';}initialized=true;}connected=true;render();}catch(e){$(\'status\').lastElementChild.textContent=\'Disconnected\';$(\'status\').style.color=\'#f17b7b\';error(\'Live connection unavailable. Keep the Python/Colab simulation session running. \'+e.message);}finally{busy=false;}}\n$(\'map\').style.touchAction=\'none\';\n$(\'map\').addEventListener(\'wheel\',event=>{event.preventDefault();zoom=Math.min(5,Math.max(1,zoom*(event.deltaY<0?1.15:1/1.15)));if(zoom===1)pan=[0,0];draw();},{passive:false});\n$(\'map\').onpointerdown=event=>{dragging=true;dragged=false;lastPointer=[event.clientX,event.clientY];$(\'map\').setPointerCapture(event.pointerId);};\n$(\'map\').onpointermove=event=>{if(!dragging)return;const dx=event.clientX-lastPointer[0],dy=event.clientY-lastPointer[1];if(Math.abs(dx)+Math.abs(dy)>2)dragged=true;pan[0]+=dx;pan[1]+=dy;lastPointer=[event.clientX,event.clientY];draw();};\n$(\'map\').onpointerup=()=>{dragging=false;};\nwindow.addEventListener(\'resize\',draw);poll();setInterval(poll,200);\n</script></body></html>\n', 'tests/test_real_integration.py': 'import csv\nimport json\nfrom pathlib import Path\nimport re\nimport sys\nimport tempfile\nimport threading\nimport time\nimport unittest\nimport urllib.error\nimport urllib.request\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nimport experiment as e\nfrom real_network import build_real_network,od_pool,generate_real_demand\nfrom real_live import RealRoadApp\nfrom live_server import server\nfrom traffic_inputs import QueueTape,counts_demand\n\n\nclass RealInputTests(unittest.TestCase):\n    def test_variable_stage_real_network_and_attribution(self):\n        nf,net,tls=build_real_network()\n        self.assertGreater(len(net.getEdges()),500)\n        self.assertEqual(len(tls),7)\n        self.assertIn(5,[len(t[\'greens\']) for t in tls.values()])\n        self.assertIn(\'OpenStreetMap\',nf.read_text())\n        audit=json.loads((e.ROOT/\'data/network_audit.json\').read_text())\n        self.assertFalse(audit[\'field_ready\'])\n        self.assertGreater(audit[\'unresolved_restriction_warning_lines\'],0)\n\n    def test_unknown_sensor_lane_and_bad_counts_rejected(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            p=Path(tmp)/\'q.csv\'\n            p.write_text(\'time_s,lane_id,queue_vehicles,vehicles\\n0,wrong,0,0\\n\')\n            with self.assertRaises(ValueError):QueueTape(p,[\'a\'],{\'a\':10})\n            p.write_text(\'time_s,lane_id,queue_vehicles,vehicles\\n0,a,9,2\\n\')\n            with self.assertRaises(ValueError):QueueTape(p,[\'a\'],{\'a\':10})\n\n    def test_stale_queue_samples_trigger_fallback_without_truth_substitution(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            p=Path(tmp)/\'q.csv\';p.write_text(\'time_s,lane_id,queue_vehicles,vehicles\\n0,a,3,4\\n\')\n            tape=QueueTape(p,[\'a\'],{\'a\':10},stale_after_s=2)\n            self.assertEqual(tape.update(0)[\'a\'],(3,4));self.assertFalse(tape.fallback)\n            self.assertEqual(tape.update(3)[\'a\'],(0,0));self.assertTrue(tape.fallback)\n\n    def test_entry_counts_conserved_and_overlap_rejected(self):\n        nf,net,tls=build_real_network();entry=od_pool(net)[0][\'entry\']\n        with tempfile.TemporaryDirectory() as tmp:\n            folder=Path(tmp);p=folder/\'c.csv\'\n            p.write_text(f\'start_s,end_s,entry_edge_id,vehicles\\n0,60,{entry},9\\n\')\n            route,schedule=counts_demand(p,folder,net,22777,120)\n            self.assertEqual(len(schedule),9);self.assertTrue(all(0<=t<60 for t in schedule.values()))\n            p.write_text(p.read_text()+f\'30,90,{entry},3\\n\')\n            with self.assertRaises(ValueError):counts_demand(p,folder,net,22777,120)\n\n    def test_recorded_sensor_fallback_executes_fixed_policy(self):\n        nf,net,tls=build_real_network()\n        with tempfile.TemporaryDirectory() as tmp:\n            folder=Path(tmp);d,s=generate_real_demand(folder,net,\'balanced\',201,120,.6)\n            # Discover exactly the observer\'s required lane set from the network.\n            lanes=set()\n            for tid in tls:\n                for incoming,outgoing,index in net.getTLS(tid).getConnections():lanes.update([incoming.getID(),outgoing.getID()])\n            p=folder/\'q.csv\'\n            with p.open(\'w\',newline=\'\') as f:\n                w=csv.writer(f);w.writerow([\'time_s\',\'lane_id\',\'queue_vehicles\',\'vehicles\'])\n                for lane in sorted(lanes):w.writerow([0,lane,0,0])\n            factory=lambda required,capacities:QueueTape(p,required,capacities,stale_after_s=0)\n            a=e.run_one(folder/\'fallback\',nf,tls,d,s,\'predictive\',\'perfect\',201,120,600,20,False,\n                        observation_factory=factory,stop_when_clear=True)\n            b=e.run_one(folder/\'fixed\',nf,tls,d,s,\'fixed\',\'perfect\',201,120,600,20,False,stop_when_clear=True)\n            self.assertTrue(a[\'valid_run\']);self.assertGreater(a[\'detector_stats\'][\'fallback_steps\'],0)\n            self.assertEqual(a[\'total_system_time_s\'],b[\'total_system_time_s\'])\n\n\nclass RealLiveTests(unittest.TestCase):\n    def test_real_road_server_start_pause_step_and_joined_signal_scene(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            app=RealRoadApp(tmp);http=server(app,port=0)\n            thread=threading.Thread(target=http.serve_forever,daemon=True);thread.start()\n            base=f\'http://127.0.0.1:{http.server_port}/\'\n            try:\n                html=urllib.request.urlopen(base).read().decode()\n                key=json.loads(re.search(r\'const ACCESS_KEY=("[^"]+");\',html).group(1))\n                def call(path,body=None):\n                    req=urllib.request.Request(base+\'api/\'+path,data=json.dumps(body).encode() if body else None,\n                        headers={\'X-Simulation-Key\':key,\'Content-Type\':\'application/json\'})\n                    with urllib.request.urlopen(req,timeout=5) as f:return json.load(f)\n                scene=call(\'scene\');self.assertEqual(len(scene[\'nodes\']),7)\n                self.assertTrue(all(n.get(\'label\') for n in scene[\'nodes\']))\n                self.assertFalse(scene[\'field_ready\']);self.assertGreater(len(scene[\'lanes\']),1000)\n                call(\'control\',{\'action\':\'speed\',\'values\':{\'speed\':60}})\n                call(\'control\',{\'action\':\'start\',\'values\':{\'demand_s\':60,\'seed\':22777,\'scale\':.6}})\n                for _ in range(150):\n                    state=call(\'state\')\n                    if state[\'status\']==\'error\':self.fail(state[\'message\'])\n                    if state[\'time_s\']>=8:break\n                    time.sleep(.03)\n                self.assertGreaterEqual(state[\'time_s\'],8)\n                call(\'control\',{\'action\':\'pause\'});time.sleep(.1);t=call(\'state\')[\'time_s\']\n                time.sleep(.1);self.assertEqual(call(\'state\')[\'time_s\'],t)\n                call(\'control\',{\'action\':\'step\'})\n                for _ in range(100):\n                    state=call(\'state\')\n                    if state[\'time_s\']==t+1:break\n                    time.sleep(.02)\n                self.assertEqual(state[\'time_s\'],t+1)\n                fixture={\'scene\':scene,\'state\':state}\n                (e.ROOT/\'tests/viewer_fixture.json\').write_text(json.dumps(fixture))\n            finally:\n                http.shutdown();http.server_close();app.close();thread.join(timeout=2)\n\n\nif __name__==\'__main__\':unittest.main(verbosity=2)\n', 'tests/check_viewer.cjs': '// Numerical/DOM smoke check with supplied SUMO-shaped data, not a browser test.\nconst fs=require(\'node:fs\'),vm=require(\'node:vm\');\nconst [htmlPath,fixturePath]=process.argv.slice(2);\nconst fixture=JSON.parse(fs.readFileSync(fixturePath,\'utf8\'));\nconst source=fs.readFileSync(htmlPath,\'utf8\').split(\'<script>\')[1].split(\'</script>\')[0].replace(\'__ACCESS_KEY__\',\'"test-key"\');\nlet drawCalls=0;\nconst numeric=(...args)=>{if(args.some(v=>typeof v!==\'number\'||!Number.isFinite(v)))throw Error(\'Nonfinite canvas coordinate\');drawCalls++;};\nconst ctx={setTransform:numeric,fillRect:numeric,clearRect:numeric,moveTo:numeric,lineTo:numeric,\n arc:numeric,translate:numeric,rotate:numeric,beginPath(){},stroke(){},save(){},restore(){},\n fillText(text,x,y){numeric(x,y);}};\nconst elements=new Map();\nfunction element(id){if(!elements.has(id))elements.set(id,{id,value:id===\'load\'?\'0.6\':\'10\',checked:true,\n style:{},lastElementChild:{textContent:\'\'},append(){},replaceChildren(){},getContext(){return ctx;},addEventListener(){},setPointerCapture(){},options:[{textContent:""}],\n getBoundingClientRect(){return {width:900,height:id===\'chart\'?105:600,left:0,top:0};}});return elements.get(id);}\nconst sandbox={console,URL,Math,Map,Number,String,JSON,Error,\n document:{getElementById:element,createElement(){return {style:{},append(){}};}},\n window:{devicePixelRatio:2,addEventListener(){}},location:{href:\'http://local.test/\'},\n fetch:async url=>({ok:true,json:async()=>String(url).endsWith(\'scene\')?fixture.scene:fixture.state}),\n setInterval(){},setTimeout};\nvm.createContext(sandbox);vm.runInContext(source,sandbox);\nsetTimeout(()=>{\n try{\n  vm.runInContext(\'draw(); inspect(); render();\',sandbox);\n  if(drawCalls<100)throw Error(\'Rendering did not exercise scene\');\n  if(element(\'arrived\').textContent!==fixture.state.metrics.arrived)throw Error(\'Metrics binding failed\');\n  console.log(\'VIEWER_NUMERICAL_SMOKE_PASS\',drawCalls,\'canvas calls\');\n }catch(e){console.error(e);process.exitCode=1;}\n},30);\n', 'README.md': "# Adaptive Signals — Hat Yai real-road prototype\n\n**Ready for an offline real-road prototype; not ready to control public signals.**\nThis package imports an actual OpenStreetMap extract around central Hat Yai,\ngeneralizes the controller/viewer to its variable signal layouts, and adds file\ninterfaces for entry counts and recorded queue observations. It contains no live\nCCTV connection, no physical signal-controller driver, and no measured Hat Yai\ntraffic dataset.\n\n## What is real, and what remains modeled\n\n| Component | Status |\n|---|---|\n| Road geometry and one-way/access tags | Imported from OpenStreetMap; provenance and source XML included |\n| Lane counts, lane connections and turning rules | Partly inferred; unresolved import warnings retained |\n| Signal locations | Based on OSM tags and SUMO grouping; not site verified |\n| Signal phases and timing | SUMO-generated protected approaches; not municipal timing plans |\n| Speeds | Demo cap 13.89 m/s (50 km/h), explicitly imposed; not a claimed legal limit |\n| Default vehicle arrivals and destinations | Synthetic |\n| Recorded count / queue files | Validated adapters implemented; no real observations supplied |\n| Live camera feed / public signal hardware | Not connected |\n\nThe road source includes 489 highway way objects; only seven have explicit\n`maxspeed` tags and 35 have `lanes` tags. These are object counts, not percentages\nof road distance. The imported network has 1,054 directed road edges and seven\nsignal controllers with 3–5 green stages. Conversion warnings include unresolved\nturn restrictions and junction geometry. `data/network_audit.json` deliberately\nsets `field_ready: false`.\n\n## Readiness finding\n\nThe initial Stage-2 synthetic-grid gains did not reliably transfer to this\nunvalidated imported model. In the 36-run integration pilot, all software checks\npassed and collision/teleport counts were zero, but complete clearance occurred\nin only 1/12 runs for our discharge candidate, versus 7/12 for each reference.\nThis is evidence to investigate, not a result to hide or a field-performance\nestimate. The real-road viewer defaults to native actuated control. Our candidate\nremains available for direct comparison with the same seeded requests.\n\n## Open the live Hat Yai prototype\n\nOpen `AdaptiveSignals_HatYai_RealRoad_Colab.ipynb` in Colab. Run the first two code\ncells, then press **Start / reset run**. The road network is bundled, so the\nnotebook does not redownload OSM on each run. A Linux Python 3.11–3.12 CPU runtime\nwith g++ is recommended. Internet is needed for pip installation only.\n\nThe viewer shows actual SUMO positions on the imported roads. Scroll to zoom,\ndrag to pan, and click a J01–J07 signal controller to inspect approach queues.\nStart/reset, pause, one-second step, speed and seeded controller comparisons are\nsupported. Keep the runtime connected. This is a local runtime application,\nnot a published website or real CCTV display.\n\nLocally:\n\n```bash\npython -m pip install -r requirements.txt\npython -m unittest discover -s tests -v\npython real_live.py --port 8765\n```\n\nOpen `http://127.0.0.1:8765/`. Ctrl+C closes the server and simulation. Colab uses\nits private iframe helper, with no public tunnel. HTTP runtime behavior and\nmocked-canvas checks are tested; browser visual QA and the actual Colab iframe\nare not claimed tested.\n\n## Integrate recorded observations\n\nUse `data/entry_edges.csv`, `sensor_lane_mapping.csv` and `signal_inventory.csv`\nto map a named road/approach to the exact imported identifiers. Verify those\nmappings against the actual site before calling the resulting data calibrated.\n\n### Entry counts\n\n`entry_counts_TEMPLATE.csv` has columns:\n\n```text\nstart_s,end_s,entry_edge_id,vehicles\n```\n\nTimes are seconds from the simulation/video origin. Counts are integer totals\nover nonoverlapping [start,end) intervals for a modeled entry edge. Unknown edges,\nnegative/nonfinite values, overlapping intervals and intervals beyond the demand\nperiod are rejected. Exact totals are preserved, but within-bin departure times\nare spread uniformly and destinations remain modeled. This does not identify a\nfull real-world origin–destination matrix.\n\n```bash\npython real_live.py --counts-csv /absolute/path/entry_counts.csv\n```\n\nThe demand multiplier is fixed at 1 for measured entry totals. Set the arrival\nduration in the viewer to cover the file. The clearly labeled\n`EXAMPLE_SYNTHETIC_entry_counts.csv` exists only to exercise the adapter.\n\n### Queue observations\n\n`queue_observations_TEMPLATE.csv` has columns:\n\n```text\ntime_s,lane_id,queue_vehicles,vehicles\n```\n\n`vehicles` includes moving and stopped vehicles on that modeled lane;\n`queue_vehicles <= vehicles`. Samples are ordered by simulation/video-relative\ntime. The detector model needs incoming and receiving-lane coverage. If less\nthan 80% of its required lane observations are fresh within 15 s, external\ncontrollers fall back to nominal fixed timing **in simulation**. Missing\nobservations are never silently filled with simulator truth. This prototype\nfallback is not a validated hardware fail-safe design.\n\n```bash\npython real_live.py --queues-csv /absolute/path/queues.csv\n```\n\nThe recorded-queue viewer defaults to fixed timing; select our candidate to test\nits recommendations. Native actuated control uses its own loop detectors and\nis rejected when a queue CSV is supplied, rather than pretending to consume it.\nRecorded queues and simulated traffic may be inconsistent unless demand/time\nalignment is calibrated. The adapter alone does not make the simulation real.\n\nBoth files can be supplied together. The notebook's second code cell exposes\n`COUNTS_CSV` and `QUEUES_CSV` paths.\n\n## Rerun / audit\n\n```bash\npython real_pilot.py --output runs/my_real_road_pilot\n```\n\nThe supplied accepted run is `runs/real_pilot_final/`. It uses 600 s synthetic\narrivals, up to 900 s draining, two demand patterns, two loads, three seeds and\nthree controllers = 36 runs. Results include every requested vehicle, not just\ncompleted trips. Fixed timing is nominal 20 s, not a calibrated municipal plan.\nThe candidate's coefficients are reused from Stage 2 without retuning to this\nnetwork. Development-route turning fractions are modeled, not field observations.\n\n`python real_network.py` reconstructs the network from the bundled OSM road\nextract and reapplies the explicitly documented speed cap. Rebuilding may change\nnetwork bytes; use a new pilot output folder after any source/network change.\nExisting outputs resume only with matching source/configuration and input hashes.\n\n## What we need before a real pilot\n\nChoose one actual intersection and verify its approach lanes, turn movements,\nsignal conflict groups, phase order, clearance intervals and pedestrian timings.\nObtain time-aligned entry/turn counts and queue measurements from permitted\nrecorded footage or site observations, then calibrate and test out-of-sample.\nCompare to the actual existing signal program. Run recommendations in shadow\nmode and review failures before considering a controlled hardware pilot with\nthe responsible traffic authority and qualified engineers.\n\n## Data attribution\n\n© OpenStreetMap contributors. Source data is available under ODbL 1.0:\nhttps://www.openstreetmap.org/copyright\n\n`data/provenance.json` records the bounded API request, retrieval timestamp,\nhashes and additional requests used to complete missing restriction members.\nFull member ways may extend outside the requested rectangle. The road-only\nextract is supplied; the larger raw map download is an intermediate file.\n\nSUMO and dependency licenses remain with their upstream projects. Primary docs:\nhttps://sumo.dlr.de/docs/Networks/Import/OpenStreetMap.html\n", 'READINESS_REPORT.md': "# Hat Yai integration — readiness assessment\n\n**The software is ready for an offline real-road prototype and recorded-data input. Our adaptive controller is not ready for physical signal deployment.**\n\nActual OpenStreetMap road geometry has been integrated into the same SUMO runner and live interface. The imported model contains 1,054 directed road edges, seven controller groups and three-to-five green stages per controller. The live interface handles joined signal groups, variable phases, road names, map zoom/pan and the existing controls.\n\n## The transfer test matters\n\nWe retained the Stage-2 discharge parameters and tested them on this new topology, using synthetic demand, not real Hat Yai counts. The 36-run integration experiment uses two demand patterns × two loads × three fresh seeds × three controllers. All 36 pass conservation and independent metric checks, with zero collision and teleport events.\n\n| Controller | Fully cleared runs | Mean completion fraction | Mean system time (s/request) |\n|---|---:|---:|---:|\n| Nominal fixed 20 s | 7/12 | 91.1% | 318.8 |\n| Native SUMO actuated | 7/12 | 92.1% | 287.4 |\n| Our Stage-2 discharge candidate | 1/12 | 82.0% | 375.6 |\n\nThis is a transfer failure on the imported model. It prevents us from treating the earlier synthetic-grid gains as deployment evidence. It does not measure actual Hat Yai travel-time savings, and it does not yet isolate whether geometry, lane/turn inference, demand modeling, saturation assumptions, score design or their interaction explains the loss. The viewer therefore defaults to the native actuated reference; our candidate remains selectable.\n\nFixed timing is a nominal generated reference, not the actual municipal signal schedule. The 1,500 s maximum horizon includes 600 s demand plus up to 900 s draining. Incomplete requests remain in the primary system-time score. These means are descriptive over matched synthetic test streams; they are not real-world effect estimates.\n\n## Why this is not a calibrated city model\n\nOnly 7 of 489 source highway way objects have explicit speed tags, and 35 have lane-count tags. These object counts do not represent road-distance coverage. Lane counts/links, turns and signal grouping involve inference. The conversion retains unresolved turn-restriction and junction-geometry warnings in data/netconvert.log. Signals are protected approach programs generated by SUMO, not surveyed signal schedules.\n\nA demonstration speed cap of 13.89 m/s is explicitly applied to the imported simulation lanes. This is not a claim about the roads' legal limits or measured operating speeds. OpenStreetMap geometry/access data and API provenance are preserved separately. Field readiness remains false in the audit.\n\n## Real inputs that can now be connected\n\n- Entry totals from video/manual counting can enter through the validated interval-count CSV. Exact counts are preserved; within-bin timing and destinations remain modeled. Entry-edge and GPS mapping files are provided.\n- Recorded per-lane queue/vehicle observations can enter through the queue CSV. Unknown lanes, impossible counts and duplicate samples are rejected. Stale/incomplete coverage triggers nominal fixed timing in simulation. No missing observations are replaced with simulator truth.\n- These adapters have been tested with explicitly synthetic fixtures. No actual camera feed, field measurements or hardware connection has been supplied or verified.\n\n## Verification\n\nSix integration tests exercise variable-stage real networks, input validation, stale-data fallback, measured-count conservation, actual fallback execution and live HTTP pause/step behavior on this road graph. JavaScript syntax and a mocked-canvas numerical check pass. The standalone notebook is checked in a fresh directory. Browser visual QA and the actual Colab iframe are not claimed tested.\n\n## Next useful evidence\n\nChoose one named Hat Yai intersection. Obtain a permitted recording or time-aligned manual entry/turn/queue counts, and the actual approach-lane and signal phase/clearance plan. Correct and validate that small network before adding more intersections. Compare recommendations against the real existing program in shadow mode, including worst-case queues, starvation and sensor failures. Only after site-specific validation should a controlled hardware pilot be considered with the responsible authority and qualified traffic engineers.\n\nWe should resolve the observed transfer failure and the model uncertainties before claiming field benefit. More synthetic optimization alone cannot supply that missing evidence.\n\nData: © OpenStreetMap contributors, ODbL 1.0. Source provenance and exact artifact hashes are included.\n", 'data/hat_yai.net.xml': '<?xml version=\'1.0\' encoding=\'utf-8\'?>\n<net xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" version="1.20" junctionCornerDetail="5" lefthand="true" limitTurnSpeed="5.50" xsi:noNamespaceSchemaLocation="http://sumo.dlr.de/xsd/net_file.xsd">\n  <!-- © OpenStreetMap contributors, ODbL 1.0. Simulation lane speeds capped at 13.89 m/s; unverified signal/lane/turn model. -->\n  <location netOffset="-662073.43,-773826.88" convBoundary="0.00,-0.00,1992.35,2036.67" origBoundary="100.467264,6.997910,100.487814,7.018679" projParameter="+proj=utm +zone=47 +ellps=WGS84 +datum=WGS84 +units=m +no_defs" />\n  <type id="highway.residential" priority="3" numLanes="1" speed="13.89" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" oneway="0" />\n  <type id="highway.secondary" priority="11" numLanes="1" speed="27.78" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" oneway="0" />\n  <type id="highway.secondary_link" priority="6" numLanes="1" speed="22.22" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" oneway="0" />\n  <type id="highway.tertiary" priority="10" numLanes="1" speed="22.22" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" oneway="0" />\n  <type id="highway.tertiary_link" priority="5" numLanes="1" speed="22.22" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" oneway="0" />\n  <type id="highway.unclassified" priority="4" numLanes="1" speed="13.89" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" oneway="0" />\n  <edge id=":1107784878_0" function="internal">\n    <lane id=":1107784878_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.20" shape="640.20,504.69 629.04,503.71" />\n  </edge>\n  <edge id=":1107784878_1" function="internal">\n    <lane id=":1107784878_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.48" length="2.43" shape="640.20,504.69 637.79,505.03" />\n  </edge>\n  <edge id=":1107784878_4" function="internal">\n    <lane id=":1107784878_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.48" length="9.35" shape="637.79,505.03 635.89,506.54 634.51,509.20 633.63,513.03" />\n  </edge>\n  <edge id=":1107784878_2" function="internal">\n    <lane id=":1107784878_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.39" length="9.04" shape="628.76,506.90 631.19,507.47 632.81,508.68 633.62,510.53 633.63,513.03" />\n  </edge>\n  <edge id=":1107784878_3" function="internal">\n    <lane id=":1107784878_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.20" shape="628.76,506.90 639.92,507.87" />\n  </edge>\n  <edge id=":1107784879_0" function="internal">\n    <lane id=":1107784879_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.02" shape="536.00,1000.74 533.92,1011.56" />\n  </edge>\n  <edge id=":1107784879_1" function="internal">\n    <lane id=":1107784879_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.45" length="8.88" shape="536.00,1000.74 535.83,1003.15 536.48,1005.04 537.95,1006.41 540.26,1007.25" />\n  </edge>\n  <edge id=":1107784879_2" function="internal">\n    <lane id=":1107784879_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.55" length="8.94" shape="529.49,1005.02 531.77,1005.83 533.27,1007.19 533.98,1009.10 533.92,1011.56" />\n  </edge>\n  <edge id=":1107784879_3" function="internal">\n    <lane id=":1107784879_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.99" shape="529.49,1005.02 540.26,1007.25" />\n  </edge>\n  <edge id=":1107784883_0" function="internal">\n    <lane id=":1107784883_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.81" length="9.54" shape="623.64,583.80 622.92,586.24 621.56,587.91 619.56,588.81 616.91,588.95" />\n  </edge>\n  <edge id=":1107784883_1" function="internal">\n    <lane id=":1107784883_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.62" shape="623.64,583.80 621.46,598.26" />\n  </edge>\n  <edge id=":1107784883_2" function="internal">\n    <lane id=":1107784883_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.22" length="11.82" shape="623.64,583.80 623.46,587.69 624.17,590.53 625.76,592.34 628.23,593.12" />\n  </edge>\n  <edge id=":1107784883_3" function="internal">\n    <lane id=":1107784883_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.64" shape="628.51,589.93 616.91,588.95" />\n  </edge>\n  <edge id=":1107784883_4" function="internal">\n    <lane id=":1107784883_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.63" length="2.68" shape="628.51,589.93 625.96,590.26 625.87,590.32" />\n  </edge>\n  <edge id=":1107784883_7" function="internal">\n    <lane id=":1107784883_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.63" length="9.43" shape="625.87,590.32 623.93,591.77 622.43,594.43 621.46,598.26" />\n  </edge>\n  <edge id=":1107784883_5" function="internal">\n    <lane id=":1107784883_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.34" length="9.04" shape="616.65,592.14 619.10,592.70 620.71,593.90 621.50,595.76 621.46,598.26" />\n  </edge>\n  <edge id=":1107784883_6" function="internal">\n    <lane id=":1107784883_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.62" shape="616.65,592.14 628.23,593.12" />\n  </edge>\n  <edge id=":1107784884_0" function="internal">\n    <lane id=":1107784884_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.79" shape="612.29,655.43 609.67,669.99" />\n  </edge>\n  <edge id=":1107784884_1" function="internal">\n    <lane id=":1107784884_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.40" length="9.04" shape="612.29,655.43 612.25,657.92 613.03,659.79 614.63,661.04 617.04,661.66" />\n  </edge>\n  <edge id=":1107784884_2" function="internal">\n    <lane id=":1107784884_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.18" length="9.05" shape="604.94,664.03 607.44,664.47 609.06,665.61 609.81,667.45 609.67,669.99" />\n  </edge>\n  <edge id=":1107784884_3" function="internal">\n    <lane id=":1107784884_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.91" shape="604.94,664.03 616.69,664.84" />\n    <lane id=":1107784884_3_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.91" shape="605.03,660.83 617.04,661.66" />\n  </edge>\n  <edge id=":1107784887_0" function="internal">\n    <lane id=":1107784887_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.20" length="9.05" shape="520.88,1087.81 520.05,1090.21 518.68,1091.64 516.74,1092.10 514.26,1091.60" />\n  </edge>\n  <edge id=":1107784887_1" function="internal">\n    <lane id=":1107784887_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.86" shape="520.88,1087.81 518.17,1102.41" />\n  </edge>\n  <edge id=":1107784887_2" function="internal">\n    <lane id=":1107784887_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.76" length="12.38" shape="520.88,1087.81 520.48,1091.81 520.99,1094.90 522.41,1097.07 524.74,1098.31" />\n  </edge>\n  <edge id=":1107784887_3" function="internal">\n    <lane id=":1107784887_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.92" shape="525.61,1095.23 514.26,1091.60" />\n  </edge>\n  <edge id=":1107784887_4" function="internal">\n    <lane id=":1107784887_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.14" length="2.66" shape="525.61,1095.23 522.95,1095.06" />\n  </edge>\n  <edge id=":1107784887_7" function="internal">\n    <lane id=":1107784887_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.14" length="9.25" shape="522.95,1095.06 520.83,1096.20 519.23,1098.65 518.17,1102.41" />\n  </edge>\n  <edge id=":1107784887_5" function="internal">\n    <lane id=":1107784887_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.31" length="10.21" shape="513.18,1094.62 515.75,1095.91 517.43,1097.64 518.24,1099.81 518.17,1102.41" />\n  </edge>\n  <edge id=":1107784887_6" function="internal">\n    <lane id=":1107784887_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.13" shape="513.18,1094.62 524.74,1098.31" />\n  </edge>\n  <edge id=":1107926433_0" function="internal">\n    <lane id=":1107926433_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.41" length="9.04" shape="1383.69,729.60 1383.18,732.04 1382.00,733.69 1380.17,734.56 1377.68,734.64" />\n  </edge>\n  <edge id=":1107926433_1" function="internal">\n    <lane id=":1107926433_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.78" length="9.33" shape="1391.93,736.43 1389.58,735.73 1387.95,734.39 1387.06,732.42 1386.89,729.82" />\n  </edge>\n  <edge id=":1107926433_2" function="internal">\n    <lane id=":1107926433_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.36" shape="1391.93,736.43 1377.68,734.64" />\n  </edge>\n  <edge id=":1107926467_0" function="internal">\n    <lane id=":1107926467_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.30" length="11.45" shape="1515.89,1683.28 1512.41,1683.59 1509.47,1683.28 1507.07,1682.35 1505.20,1680.79" />\n  </edge>\n  <edge id=":1107926467_1" function="internal">\n    <lane id=":1107926467_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.54" length="15.67" shape="1515.89,1683.28 1511.57,1684.64 1509.22,1686.96 1508.84,1690.24 1510.44,1694.48" />\n  </edge>\n  <edge id=":1107926467_2" function="internal">\n    <lane id=":1107926467_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.90" length="9.10" shape="1513.25,1692.94 1512.32,1690.48 1512.54,1688.57 1513.90,1687.22 1516.41,1686.43" />\n  </edge>\n  <edge id=":1107926467_3" function="internal">\n    <lane id=":1107926467_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.60" shape="1513.25,1692.94 1511.43,1689.73 1509.54,1686.65 1507.49,1683.68 1505.20,1680.79" />\n  </edge>\n  <edge id=":1107926467_4" function="internal">\n    <lane id=":1107926467_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.94" shape="1502.77,1682.87 1505.00,1685.67 1506.92,1688.46 1508.69,1691.35 1510.44,1694.48" />\n  </edge>\n  <edge id=":1107926467_5" function="internal">\n    <lane id=":1107926467_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.49" length="3.39" shape="1502.77,1682.87 1505.26,1685.01 1505.36,1685.05" />\n  </edge>\n  <edge id=":1107926467_6" function="internal">\n    <lane id=":1107926467_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.49" length="11.36" shape="1505.36,1685.05 1508.36,1686.31 1512.08,1686.79 1516.41,1686.43" />\n  </edge>\n  <edge id=":1107926481_0" function="internal">\n    <lane id=":1107926481_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.15" length="12.32" shape="1349.16,1564.24 1345.89,1562.56 1343.50,1560.55 1341.99,1558.21 1341.37,1555.54" />\n  </edge>\n  <edge id=":1107926481_1" function="internal">\n    <lane id=":1107926481_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.28" shape="1349.16,1564.24 1345.23,1563.02 1340.74,1562.29 1336.21,1562.13 1332.16,1562.66" />\n  </edge>\n  <edge id=":1107926481_2" function="internal">\n    <lane id=":1107926481_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.45" length="16.22" shape="1349.16,1564.24 1344.53,1563.05 1341.30,1563.85 1339.46,1566.63 1339.03,1571.39" />\n  </edge>\n  <edge id=":1107926481_3" function="internal">\n    <lane id=":1107926481_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.77" length="9.16" shape="1342.23,1571.24 1342.47,1568.55 1343.51,1566.98 1345.33,1566.53 1347.95,1567.20" />\n  </edge>\n  <edge id=":1107926481_4" function="internal">\n    <lane id=":1107926481_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.72" shape="1342.23,1571.24 1341.37,1555.54" />\n  </edge>\n  <edge id=":1107926481_5" function="internal">\n    <lane id=":1107926481_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.90" length="4.86" shape="1342.23,1571.24 1341.41,1566.60 1341.33,1566.48" />\n  </edge>\n  <edge id=":1107926481_12" function="internal">\n    <lane id=":1107926481_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.90" length="11.00" shape="1341.33,1566.48 1339.46,1563.62 1336.37,1562.30 1332.16,1562.66" />\n  </edge>\n  <edge id=":1107926481_6" function="internal">\n    <lane id=":1107926481_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.38" length="9.93" shape="1332.93,1565.76 1335.47,1565.58 1337.34,1566.45 1338.53,1568.39 1339.03,1571.39" />\n  </edge>\n  <edge id=":1107926481_7" function="internal">\n    <lane id=":1107926481_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.31" shape="1332.93,1565.76 1336.67,1565.21 1340.52,1565.32 1344.33,1566.01 1347.95,1567.20" />\n  </edge>\n  <edge id=":1107926481_8" function="internal">\n    <lane id=":1107926481_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.67" length="14.58" shape="1332.93,1565.76 1336.82,1564.29 1339.52,1562.10 1341.04,1559.18 1341.37,1555.54" />\n  </edge>\n  <edge id=":1107926481_9" function="internal">\n    <lane id=":1107926481_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.30" length="10.06" shape="1338.17,1555.75 1337.93,1558.18 1336.85,1560.14 1334.92,1561.63 1332.16,1562.66" />\n  </edge>\n  <edge id=":1107926481_10" function="internal">\n    <lane id=":1107926481_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.66" shape="1338.17,1555.75 1339.03,1571.39" />\n  </edge>\n  <edge id=":1107926481_11" function="internal">\n    <lane id=":1107926481_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.33" length="3.81" shape="1338.17,1555.75 1338.97,1559.34 1339.04,1559.45" />\n  </edge>\n  <edge id=":1107926481_13" function="internal">\n    <lane id=":1107926481_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.33" length="12.09" shape="1339.04,1559.45 1340.87,1562.44 1343.86,1565.06 1347.95,1567.20" />\n  </edge>\n  <edge id=":1107926513_0" function="internal">\n    <lane id=":1107926513_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.23" shape="1333.29,1126.92 1333.05,1141.15" />\n  </edge>\n  <edge id=":1107926513_1" function="internal">\n    <lane id=":1107926513_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="3.84" shape="1333.29,1126.92 1333.83,1130.72" />\n  </edge>\n  <edge id=":1107926513_6" function="internal">\n    <lane id=":1107926513_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="10.29" shape="1333.83,1130.72 1335.46,1133.47 1338.19,1135.18 1342.01,1135.83" />\n  </edge>\n  <edge id=":1107926513_2" function="internal">\n    <lane id=":1107926513_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.59" length="9.08" shape="1342.11,1132.63 1339.65,1132.21 1337.89,1131.12 1336.84,1129.36 1336.49,1126.92" />\n  </edge>\n  <edge id=":1107926513_3" function="internal">\n    <lane id=":1107926513_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="14.17" shape="1342.11,1132.63 1338.25,1133.07 1335.46,1134.63 1333.72,1137.33 1333.05,1141.15" />\n  </edge>\n  <edge id=":1107926513_4" function="internal">\n    <lane id=":1107926513_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="9.02" shape="1336.25,1141.25 1336.68,1138.82 1337.78,1137.10 1339.56,1136.11 1342.01,1135.83" />\n  </edge>\n  <edge id=":1107926513_5" function="internal">\n    <lane id=":1107926513_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.33" shape="1336.25,1141.25 1336.49,1126.92" />\n  </edge>\n  <edge id=":1107926537_0" function="internal">\n    <lane id=":1107926537_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.78" shape="1340.02,802.10 1337.24,803.75 1334.79,804.69 1332.18,805.14 1328.92,805.34" />\n  </edge>\n  <edge id=":1107926537_1" function="internal">\n    <lane id=":1107926537_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.07" length="3.21" shape="1340.02,802.10 1337.63,804.23" />\n  </edge>\n  <edge id=":1107926537_6" function="internal">\n    <lane id=":1107926537_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.07" length="10.24" shape="1337.63,804.23 1335.97,806.90 1335.05,810.12 1334.87,813.87" />\n  </edge>\n  <edge id=":1107926537_2" function="internal">\n    <lane id=":1107926537_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.81" length="10.17" shape="1338.07,813.72 1338.19,810.82 1338.85,808.35 1340.06,806.33 1341.81,804.75" />\n  </edge>\n  <edge id=":1107926537_3" function="internal">\n    <lane id=":1107926537_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="14.12" shape="1338.07,813.72 1337.34,809.92 1335.57,807.26 1332.77,805.73 1328.92,805.34" />\n  </edge>\n  <edge id=":1107926537_4" function="internal">\n    <lane id=":1107926537_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="8.99" shape="1329.05,808.54 1331.50,808.79 1333.28,809.76 1334.41,811.45 1334.87,813.87" />\n  </edge>\n  <edge id=":1107926537_5" function="internal">\n    <lane id=":1107926537_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.50" shape="1329.05,808.54 1332.57,808.25 1335.75,807.62 1338.77,806.49 1341.81,804.75" />\n  </edge>\n  <edge id=":1107926561_0" function="internal">\n    <lane id=":1107926561_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.11" length="9.06" shape="1293.99,1572.11 1291.45,1572.37 1289.59,1571.70 1288.43,1570.11 1287.97,1567.59" />\n  </edge>\n  <edge id=":1107926561_1" function="internal">\n    <lane id=":1107926561_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.49" shape="1293.99,1572.11 1280.00,1575.85" />\n  </edge>\n  <edge id=":1107926561_2" function="internal">\n    <lane id=":1107926561_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.36" shape="1280.89,1578.92 1294.76,1575.22" />\n  </edge>\n  <edge id=":1107926561_3" function="internal">\n    <lane id=":1107926561_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.91" length="3.58" shape="1280.89,1578.92 1284.13,1577.40" />\n  </edge>\n  <edge id=":1107926561_6" function="internal">\n    <lane id=":1107926561_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.91" length="10.96" shape="1284.13,1577.40 1286.38,1575.01 1287.66,1571.74 1287.97,1567.59" />\n  </edge>\n  <edge id=":1107926561_4" function="internal">\n    <lane id=":1107926561_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.56" length="10.25" shape="1284.77,1567.72 1284.57,1570.73 1283.71,1573.08 1282.19,1574.79 1280.00,1575.85" />\n  </edge>\n  <edge id=":1107926561_5" function="internal">\n    <lane id=":1107926561_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.69" length="15.02" shape="1284.77,1567.72 1285.54,1571.89 1287.47,1574.53 1290.54,1575.64 1294.76,1575.22" />\n  </edge>\n  <edge id=":1107926599_0" function="internal">\n    <lane id=":1107926599_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.60" length="8.61" shape="1136.42,1606.79 1134.16,1606.62 1132.40,1605.76 1131.17,1604.19 1130.44,1601.92" />\n  </edge>\n  <edge id=":1107926599_1" function="internal">\n    <lane id=":1107926599_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.40" shape="1136.42,1606.79 1126.07,1607.71" />\n  </edge>\n  <edge id=":1107926599_2" function="internal">\n    <lane id=":1107926599_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.31" shape="1126.39,1610.90 1136.66,1609.98" />\n  </edge>\n  <edge id=":1107926599_3" function="internal">\n    <lane id=":1107926599_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.01" length="2.46" shape="1126.39,1610.90 1128.73,1610.12" />\n  </edge>\n  <edge id=":1107926599_4" function="internal">\n    <lane id=":1107926599_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.01" length="8.80" shape="1128.73,1610.12 1130.18,1608.37 1130.75,1605.63 1130.44,1601.92" />\n  </edge>\n  <edge id=":1107926639_0" function="internal">\n    <lane id=":1107926639_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.80" length="9.33" shape="1334.24,874.55 1333.79,876.96 1332.64,878.72 1330.78,879.83 1328.21,880.28" />\n  </edge>\n  <edge id=":1107926639_1" function="internal">\n    <lane id=":1107926639_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.75" shape="1334.24,874.55 1334.07,889.30" />\n  </edge>\n  <edge id=":1107926639_2" function="internal">\n    <lane id=":1107926639_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.43" length="14.57" shape="1334.24,874.55 1334.68,878.30 1336.28,881.16 1339.06,883.14 1343.02,884.24" />\n  </edge>\n  <edge id=":1107926639_3" function="internal">\n    <lane id=":1107926639_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.02" length="9.80" shape="1343.47,881.07 1340.76,880.33 1338.85,879.02 1337.74,877.13 1337.44,874.66" />\n  </edge>\n  <edge id=":1107926639_4" function="internal">\n    <lane id=":1107926639_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.30" shape="1343.47,881.07 1339.70,880.61 1335.86,880.32 1332.01,880.21 1328.21,880.28" />\n  </edge>\n  <edge id=":1107926639_5" function="internal">\n    <lane id=":1107926639_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.75" length="4.41" shape="1343.47,881.07 1339.32,881.07 1339.09,881.18" />\n  </edge>\n  <edge id=":1107926639_12" function="internal">\n    <lane id=":1107926639_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.75" length="10.40" shape="1339.09,881.18 1336.37,882.44 1334.62,885.19 1334.07,889.30" />\n  </edge>\n  <edge id=":1107926639_6" function="internal">\n    <lane id=":1107926639_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.20" length="9.05" shape="1337.27,889.27 1337.61,886.75 1338.68,885.08 1340.48,884.24 1343.02,884.24" />\n  </edge>\n  <edge id=":1107926639_7" function="internal">\n    <lane id=":1107926639_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.60" shape="1337.27,889.27 1337.44,874.66" />\n  </edge>\n  <edge id=":1107926639_8" function="internal">\n    <lane id=":1107926639_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.03" length="14.66" shape="1337.27,889.27 1336.67,885.18 1334.96,882.33 1332.14,880.69 1328.21,880.28" />\n  </edge>\n  <edge id=":1107926639_9" function="internal">\n    <lane id=":1107926639_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.54" length="9.37" shape="1328.36,883.48 1330.83,883.75 1332.61,884.81 1333.69,886.66 1334.07,889.30" />\n  </edge>\n  <edge id=":1107926639_10" function="internal">\n    <lane id=":1107926639_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.70" shape="1328.36,883.48 1332.06,883.40 1335.70,883.51 1339.34,883.79 1343.02,884.24" />\n  </edge>\n  <edge id=":1107926639_11" function="internal">\n    <lane id=":1107926639_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.25" length="3.92" shape="1328.36,883.48 1332.22,882.78" />\n  </edge>\n  <edge id=":1107926639_13" function="internal">\n    <lane id=":1107926639_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.25" length="10.26" shape="1332.22,882.78 1335.02,881.08 1336.76,878.37 1337.44,874.66" />\n  </edge>\n  <edge id=":1107926734_0" function="internal">\n    <lane id=":1107926734_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.76" shape="1036.98,1618.65 1022.39,1620.89" />\n  </edge>\n  <edge id=":1107926734_1" function="internal">\n    <lane id=":1107926734_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.17" length="11.87" shape="1036.98,1618.65 1034.51,1619.57 1033.03,1621.47 1032.52,1624.35 1033.01,1628.22" />\n  </edge>\n  <edge id=":1107926734_2" function="internal">\n    <lane id=":1107926734_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.82" length="9.58" shape="1022.91,1624.04 1025.47,1624.00 1027.49,1624.80 1028.96,1626.44 1029.88,1628.92" />\n  </edge>\n  <edge id=":1107926734_3" function="internal">\n    <lane id=":1107926734_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.69" shape="1022.91,1624.04 1037.43,1621.82" />\n  </edge>\n  <edge id=":1107926734_4" function="internal">\n    <lane id=":1107926734_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.46" length="9.03" shape="1026.93,1614.44 1027.03,1616.92 1026.31,1618.82 1024.76,1620.14 1022.39,1620.89" />\n  </edge>\n  <edge id=":1107926734_5" function="internal">\n    <lane id=":1107926734_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.72" shape="1026.93,1614.44 1029.88,1628.92" />\n    <lane id=":1107926734_5_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.72" shape="1030.08,1613.86 1033.01,1628.22" />\n  </edge>\n  <edge id=":1107926734_7" function="internal">\n    <lane id=":1107926734_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.54" length="12.11" shape="1030.08,1613.86 1031.14,1617.65 1032.72,1620.24 1034.82,1621.63 1037.43,1621.82" />\n  </edge>\n  <edge id=":1107926940_0" function="internal">\n    <lane id=":1107926940_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.04" length="9.07" shape="1251.05,1584.27 1248.49,1584.63 1246.63,1584.03 1245.47,1582.45 1245.01,1579.91" />\n  </edge>\n  <edge id=":1107926940_1" function="internal">\n    <lane id=":1107926940_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.62" shape="1251.05,1584.27 1236.96,1588.21" />\n  </edge>\n  <edge id=":1107926940_2" function="internal">\n    <lane id=":1107926940_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.70" shape="1237.79,1591.30 1251.94,1587.34" />\n  </edge>\n  <edge id=":1107926940_3" function="internal">\n    <lane id=":1107926940_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.89" length="3.84" shape="1237.79,1591.30 1241.08,1589.83 1241.24,1589.66" />\n  </edge>\n  <edge id=":1107926940_6" function="internal">\n    <lane id=":1107926940_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.89" length="10.88" shape="1241.24,1589.66 1243.38,1587.45 1244.69,1584.14 1245.01,1579.91" />\n  </edge>\n  <edge id=":1107926940_4" function="internal">\n    <lane id=":1107926940_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.54" length="10.36" shape="1241.82,1580.03 1241.61,1583.10 1240.73,1585.48 1239.18,1587.19 1236.96,1588.21" />\n  </edge>\n  <edge id=":1107926940_5" function="internal">\n    <lane id=":1107926940_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.63" length="15.22" shape="1241.82,1580.03 1242.59,1584.29 1244.53,1586.93 1247.65,1587.95 1251.94,1587.34" />\n  </edge>\n  <edge id=":1107926968_0" function="internal">\n    <lane id=":1107926968_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="3.94" length="2.11" shape="1534.36,1729.06 1533.80,1729.12 1533.32,1729.01 1532.91,1728.72 1532.58,1728.27" />\n  </edge>\n  <edge id=":1107926968_1" function="internal">\n    <lane id=":1107926968_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.14" length="6.34" shape="1529.77,1729.81 1530.77,1731.17 1531.99,1732.01 1533.43,1732.35 1535.11,1732.17" />\n  </edge>\n  <edge id=":1107926977_0" function="internal">\n    <lane id=":1107926977_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.79" length="9.21" shape="1087.62,1611.66 1085.17,1611.55 1083.27,1610.68 1081.91,1609.04 1081.09,1606.64" />\n  </edge>\n  <edge id=":1107926977_1" function="internal">\n    <lane id=":1107926977_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="25.33" shape="1087.62,1611.66 1062.51,1615.02" />\n  </edge>\n  <edge id=":1107926977_2" function="internal">\n    <lane id=":1107926977_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="4.32" shape="1087.62,1611.66 1083.49,1612.93" />\n  </edge>\n  <edge id=":1107926977_9" function="internal">\n    <lane id=":1107926977_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="21.33" shape="1083.49,1612.93 1082.52,1613.23 1075.82,1616.38 1069.25,1619.91 1064.53,1622.63" />\n  </edge>\n  <edge id=":1107926977_3" function="internal">\n    <lane id=":1107926977_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="24.28" shape="1066.17,1625.38 1070.68,1622.72 1076.80,1619.34 1083.05,1616.34 1087.94,1614.84" />\n  </edge>\n  <edge id=":1107926977_4" function="internal">\n    <lane id=":1107926977_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="12.17" length="25.28" shape="1066.17,1625.38 1073.35,1620.49 1078.23,1615.73 1080.81,1611.12 1081.09,1606.64" />\n  </edge>\n  <edge id=":1107926977_5" function="internal">\n    <lane id=":1107926977_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.15" length="15.12" shape="1066.17,1625.38 1068.66,1622.32 1068.57,1618.71 1066.37,1615.84 1062.51,1615.02" />\n  </edge>\n  <edge id=":1107926977_6" function="internal">\n    <lane id=":1107926977_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.74" length="12.89" shape="1062.96,1618.19 1067.51,1617.83 1069.28,1618.45 1068.29,1620.05 1064.53,1622.63" />\n  </edge>\n  <edge id=":1107926977_7" function="internal">\n    <lane id=":1107926977_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="25.21" shape="1062.96,1618.19 1087.94,1614.84" />\n  </edge>\n  <edge id=":1107926977_8" function="internal">\n    <lane id=":1107926977_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.76" length="8.73" shape="1062.96,1618.19 1068.91,1616.57 1071.23,1615.47" />\n  </edge>\n  <edge id=":1107926977_10" function="internal">\n    <lane id=":1107926977_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.76" length="13.79" shape="1071.23,1615.47 1075.01,1613.67 1079.61,1610.14 1081.09,1606.64" />\n  </edge>\n  <edge id=":1110416410_0" function="internal">\n    <lane id=":1110416410_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.24" shape="750.65,600.55 738.45,599.46" />\n  </edge>\n  <edge id=":1110416410_1" function="internal">\n    <lane id=":1110416410_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.41" length="9.04" shape="744.43,608.88 744.90,606.43 746.05,604.76 747.86,603.86 750.35,603.73" />\n  </edge>\n  <edge id=":1110416410_2" function="internal">\n    <lane id=":1110416410_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.65" length="12.52" shape="744.43,608.88 744.22,604.96 743.15,602.09 741.23,600.26 738.45,599.46" />\n  </edge>\n  <edge id=":1110416410_3" function="internal">\n    <lane id=":1110416410_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.22" shape="738.18,602.65 750.35,603.73" />\n  </edge>\n  <edge id=":1110416410_4" function="internal">\n    <lane id=":1110416410_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.03" length="9.07" shape="743.48,594.00 743.55,596.58 742.73,598.35 741.04,599.31 738.45,599.46" />\n  </edge>\n  <edge id=":1110416410_5" function="internal">\n    <lane id=":1110416410_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.46" length="12.79" shape="743.48,594.00 744.50,598.07 745.99,601.05 747.94,602.94 750.35,603.73" />\n  </edge>\n  <edge id=":1110416444_0" function="internal">\n    <lane id=":1110416444_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="10.17" shape="651.74,1123.22 648.68,1122.47 646.73,1121.12 645.90,1119.18 646.18,1116.66" />\n  </edge>\n  <edge id=":1110416444_1" function="internal">\n    <lane id=":1110416444_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.13" shape="651.74,1123.22 637.78,1120.99" />\n  </edge>\n  <edge id=":1110416444_2" function="internal">\n    <lane id=":1110416444_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.82" length="3.94" shape="651.74,1123.22 647.80,1123.29" />\n  </edge>\n  <edge id=":1110416444_9" function="internal">\n    <lane id=":1110416444_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.82" length="10.67" shape="647.80,1123.29 647.67,1123.29 644.74,1124.67 642.94,1127.36 642.27,1131.37" />\n  </edge>\n  <edge id=":1110416444_3" function="internal">\n    <lane id=":1110416444_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.29" length="9.04" shape="645.47,1131.44 645.89,1128.96 647.00,1127.29 648.82,1126.44 651.33,1126.40" />\n  </edge>\n  <edge id=":1110416444_4" function="internal">\n    <lane id=":1110416444_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.85" shape="645.47,1131.44 645.43,1127.69 645.33,1123.97 645.48,1120.29 646.18,1116.66" />\n  </edge>\n  <edge id=":1110416444_5" function="internal">\n    <lane id=":1110416444_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.58" length="14.29" shape="645.47,1131.44 645.07,1127.45 643.65,1124.38 641.22,1122.22 637.78,1120.99" />\n  </edge>\n  <edge id=":1110416444_6" function="internal">\n    <lane id=":1110416444_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.18" length="9.75" shape="637.17,1124.13 639.45,1124.97 641.07,1126.45 642.01,1128.59 642.27,1131.37" />\n  </edge>\n  <edge id=":1110416444_7" function="internal">\n    <lane id=":1110416444_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.35" shape="637.17,1124.13 651.33,1126.40" />\n  </edge>\n  <edge id=":1110416444_8" function="internal">\n    <lane id=":1110416444_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.96" length="3.24" shape="637.17,1124.13 640.13,1124.13 640.37,1124.02" />\n  </edge>\n  <edge id=":1110416444_10" function="internal">\n    <lane id=":1110416444_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.96" length="9.77" shape="640.37,1124.02 642.63,1122.89 644.64,1120.40 646.18,1116.66" />\n  </edge>\n  <edge id=":1110416469_0" function="internal">\n    <lane id=":1110416469_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.97" shape="758.77,1660.10 755.09,1660.77 751.43,1661.54 747.80,1662.43 744.19,1663.43" />\n  </edge>\n  <edge id=":1110416469_1" function="internal">\n    <lane id=":1110416469_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.85" length="4.27" shape="758.77,1660.10 754.93,1661.32 754.76,1661.47" />\n  </edge>\n  <edge id=":1110416469_6" function="internal">\n    <lane id=":1110416469_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.85" length="10.28" shape="754.76,1661.47 752.51,1663.48 751.53,1666.56 751.98,1670.57" />\n  </edge>\n  <edge id=":1110416469_2" function="internal">\n    <lane id=":1110416469_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.32" length="9.04" shape="755.08,1669.76 754.80,1667.27 755.41,1665.35 756.91,1664.01 759.30,1663.25" />\n  </edge>\n  <edge id=":1110416469_3" function="internal">\n    <lane id=":1110416469_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="14.47" shape="755.08,1669.76 753.53,1666.05 751.20,1663.75 748.09,1662.88 744.19,1663.43" />\n  </edge>\n  <edge id=":1110416469_4" function="internal">\n    <lane id=":1110416469_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.49" length="9.20" shape="745.09,1666.50 747.55,1666.16 749.52,1666.72 751.00,1668.19 751.98,1670.57" />\n  </edge>\n  <edge id=":1110416469_5" function="internal">\n    <lane id=":1110416469_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.59" shape="745.09,1666.50 748.64,1665.51 752.14,1664.66 755.67,1663.91 759.30,1663.25" />\n  </edge>\n  <edge id=":1110416470_0" function="internal">\n    <lane id=":1110416470_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.46" length="10.23" shape="578.91,1384.65 576.26,1383.94 574.30,1382.58 573.02,1380.58 572.43,1377.92" />\n  </edge>\n  <edge id=":1110416470_1" function="internal">\n    <lane id=":1110416470_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.01" shape="578.91,1384.65 567.00,1383.12" />\n  </edge>\n  <edge id=":1110416470_2" function="internal">\n    <lane id=":1110416470_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.45" length="9.03" shape="572.33,1392.71 572.93,1390.30 574.16,1388.69 576.02,1387.86 578.50,1387.82" />\n  </edge>\n  <edge id=":1110416470_3" function="internal">\n    <lane id=":1110416470_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.82" shape="572.33,1392.71 572.62,1389.01 572.73,1385.32 572.66,1381.63 572.43,1377.92" />\n  </edge>\n  <edge id=":1110416470_4" function="internal">\n    <lane id=":1110416470_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.57" length="12.35" shape="572.33,1392.71 572.34,1388.82 571.45,1385.92 569.67,1384.02 567.00,1383.12" />\n  </edge>\n  <edge id=":1110416470_5" function="internal">\n    <lane id=":1110416470_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.01" shape="566.59,1386.29 578.50,1387.82" />\n  </edge>\n  <edge id=":1110416470_6" function="internal">\n    <lane id=":1110416470_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.95" length="12.15" shape="566.59,1386.29 569.44,1386.09 571.36,1384.62 572.36,1381.90 572.43,1377.92" />\n  </edge>\n  <edge id=":1110416526_0" function="internal">\n    <lane id=":1110416526_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.21" shape="871.63,1643.83 857.55,1645.70" />\n  </edge>\n  <edge id=":1110416526_1" function="internal">\n    <lane id=":1110416526_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.06" length="3.84" shape="871.63,1643.83 867.95,1644.93" />\n  </edge>\n  <edge id=":1110416526_6" function="internal">\n    <lane id=":1110416526_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.06" length="10.28" shape="867.95,1644.93 865.48,1646.95 864.20,1649.90 864.13,1653.77" />\n  </edge>\n  <edge id=":1110416526_2" function="internal">\n    <lane id=":1110416526_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.58" length="9.05" shape="867.30,1653.38 867.35,1650.89 868.17,1649.00 869.75,1647.70 872.10,1646.99" />\n  </edge>\n  <edge id=":1110416526_3" function="internal">\n    <lane id=":1110416526_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="14.14" shape="867.30,1653.38 866.30,1649.64 864.34,1647.11 861.42,1645.80 857.55,1645.70" />\n  </edge>\n  <edge id=":1110416526_4" function="internal">\n    <lane id=":1110416526_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="9.00" shape="857.92,1648.88 860.38,1648.94 862.24,1649.78 863.49,1651.39 864.13,1653.77" />\n  </edge>\n  <edge id=":1110416526_5" function="internal">\n    <lane id=":1110416526_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.31" shape="857.92,1648.88 872.10,1646.99" />\n  </edge>\n  <edge id=":1110416568_0" function="internal">\n    <lane id=":1110416568_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.30" length="10.96" shape="740.99,684.46 742.09,681.20 743.65,678.99 745.68,677.84 748.16,677.73" />\n  </edge>\n  <edge id=":1110416568_1" function="internal">\n    <lane id=":1110416568_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.12" shape="740.99,684.46 741.51,680.75 741.58,676.97 741.48,673.16 741.50,669.38" />\n  </edge>\n  <edge id=":1110416568_2" function="internal">\n    <lane id=":1110416568_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.24" length="9.05" shape="733.38,677.50 735.84,678.10 737.40,679.34 738.08,681.22 737.87,683.74" />\n  </edge>\n  <edge id=":1110416568_3" function="internal">\n    <lane id=":1110416568_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.77" shape="733.38,677.50 748.16,677.73" />\n  </edge>\n  <edge id=":1110416568_4" function="internal">\n    <lane id=":1110416568_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.88" length="10.59" shape="733.69,674.32 737.01,674.28 739.41,673.45 740.91,671.81 741.50,669.38" />\n  </edge>\n  <edge id=":1110416631_0" function="internal">\n    <lane id=":1110416631_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="1.70" shape="567.73,1497.58 566.07,1497.93" />\n  </edge>\n  <edge id=":1110416654_0" function="internal">\n    <lane id=":1110416654_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.84" shape="573.94,1108.42 563.39,1105.94" />\n  </edge>\n  <edge id=":1110416654_1" function="internal">\n    <lane id=":1110416654_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.25" length="9.05" shape="567.22,1116.24 567.75,1113.77 568.94,1112.17 570.80,1111.43 573.32,1111.56" />\n  </edge>\n  <edge id=":1110416654_2" function="internal">\n    <lane id=":1110416654_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.96" length="11.82" shape="567.22,1116.24 567.21,1112.21 566.57,1109.14 565.30,1107.05 563.39,1105.94" />\n  </edge>\n  <edge id=":1110416654_3" function="internal">\n    <lane id=":1110416654_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.10" shape="562.52,1109.02 573.32,1111.56" />\n  </edge>\n  <edge id=":1110416726_0" function="internal">\n    <lane id=":1110416726_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.85" length="15.28" shape="800.86,296.16 797.86,299.73 794.80,302.22 791.66,303.64 788.45,303.99" />\n  </edge>\n  <edge id=":1110416726_1" function="internal">\n    <lane id=":1110416726_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.06" shape="800.86,296.16 798.53,299.87 796.15,304.44 794.08,309.15 792.64,313.29" />\n  </edge>\n  <edge id=":1110416726_2" function="internal">\n    <lane id=":1110416726_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.37" length="18.82" shape="800.86,296.16 797.99,301.31 798.10,304.69 801.18,306.29 807.25,306.12" />\n  </edge>\n  <edge id=":1110416726_3" function="internal">\n    <lane id=":1110416726_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.47" length="9.60" shape="806.80,302.95 803.72,303.04 802.16,302.21 802.12,300.48 803.59,297.84" />\n  </edge>\n  <edge id=":1110416726_4" function="internal">\n    <lane id=":1110416726_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.40" shape="806.80,302.95 802.55,303.45 797.64,303.84 792.73,304.04 788.45,303.99" />\n  </edge>\n  <edge id=":1110416726_5" function="internal">\n    <lane id=":1110416726_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.87" length="5.44" shape="806.80,302.95 801.51,304.22" />\n  </edge>\n  <edge id=":1110416726_12" function="internal">\n    <lane id=":1110416726_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.87" length="13.17" shape="801.51,304.22 797.38,306.36 794.43,309.39 792.64,313.29" />\n  </edge>\n  <edge id=":1110416726_6" function="internal">\n    <lane id=":1110416726_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.77" length="14.89" shape="795.73,314.14 797.14,311.14 799.53,308.81 802.90,307.13 807.25,306.12" />\n  </edge>\n  <edge id=":1110416726_7" function="internal">\n    <lane id=":1110416726_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.16" shape="795.73,314.14 797.11,310.13 799.06,305.71 801.32,301.42 803.59,297.84" />\n  </edge>\n  <edge id=":1110416726_8" function="internal">\n    <lane id=":1110416726_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.67" length="15.07" shape="795.73,314.14 796.27,309.91 795.23,306.81 792.63,304.84 788.45,303.99" />\n  </edge>\n  <edge id=":1110416726_9" function="internal">\n    <lane id=":1110416726_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.09" length="9.06" shape="788.27,307.18 790.78,307.69 792.35,308.88 792.97,310.75 792.64,313.29" />\n  </edge>\n  <edge id=":1110416726_10" function="internal">\n    <lane id=":1110416726_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.03" shape="788.27,307.18 792.64,307.24 797.77,307.02 802.89,306.62 807.25,306.12" />\n  </edge>\n  <edge id=":1110416726_11" function="internal">\n    <lane id=":1110416726_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.97" length="4.56" shape="788.27,307.18 792.32,306.78 792.76,306.58" />\n  </edge>\n  <edge id=":1110416726_13" function="internal">\n    <lane id=":1110416726_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.97" length="14.15" shape="792.76,306.58 796.22,305.08 799.98,302.11 803.59,297.84" />\n  </edge>\n  <edge id=":1110416742_0" function="internal">\n    <lane id=":1110416742_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="1345.05,1638.51 1344.41,1624.10" />\n  </edge>\n  <edge id=":1110416742_1" function="internal">\n    <lane id=":1110416742_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.75" length="4.16" shape="1345.05,1638.51 1344.25,1634.43" />\n  </edge>\n  <edge id=":1110416742_6" function="internal">\n    <lane id=":1110416742_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.75" length="10.66" shape="1344.25,1634.43 1342.33,1631.80 1339.30,1630.62 1335.15,1630.88" />\n  </edge>\n  <edge id=":1110416742_2" function="internal">\n    <lane id=":1110416742_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.19" length="9.05" shape="1335.81,1634.02 1338.34,1633.86 1340.19,1634.58 1341.37,1636.18 1341.85,1638.67" />\n  </edge>\n  <edge id=":1110416742_3" function="internal">\n    <lane id=":1110416742_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.61" length="14.46" shape="1335.81,1634.02 1339.68,1632.70 1342.41,1630.60 1343.98,1627.74 1344.41,1624.10" />\n  </edge>\n  <edge id=":1110416742_4" function="internal">\n    <lane id=":1110416742_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.23" length="9.91" shape="1341.21,1624.23 1340.91,1626.65 1339.80,1628.56 1337.88,1629.98 1335.15,1630.88" />\n  </edge>\n  <edge id=":1110416742_5" function="internal">\n    <lane id=":1110416742_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.46" shape="1341.21,1624.23 1341.85,1638.67" />\n  </edge>\n  <edge id=":1110416754_0" function="internal">\n    <lane id=":1110416754_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.40" length="9.04" shape="616.11,1700.98 613.64,1701.33 611.69,1700.80 610.26,1699.37 609.34,1697.05" />\n  </edge>\n  <edge id=":1110416754_1" function="internal">\n    <lane id=":1110416754_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.41" shape="616.11,1700.98 602.29,1705.04" />\n  </edge>\n  <edge id=":1110416754_2" function="internal">\n    <lane id=":1110416754_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="603.20,1708.11 617.01,1704.05" />\n  </edge>\n  <edge id=":1110416754_3" function="internal">\n    <lane id=":1110416754_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.18" length="3.98" shape="603.20,1708.11 606.66,1706.52 606.77,1706.39" />\n  </edge>\n  <edge id=":1110416754_6" function="internal">\n    <lane id=":1110416754_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.18" length="10.29" shape="606.77,1706.39 608.83,1704.14 609.73,1700.98 609.34,1697.05" />\n  </edge>\n  <edge id=":1110416754_4" function="internal">\n    <lane id=":1110416754_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.72" length="9.30" shape="606.23,1697.79 606.49,1700.38 605.92,1702.45 604.52,1704.01 602.29,1705.04" />\n  </edge>\n  <edge id=":1110416754_5" function="internal">\n    <lane id=":1110416754_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.91" length="14.39" shape="606.23,1697.79 607.69,1701.48 609.97,1703.76 613.08,1704.62 617.01,1704.05" />\n  </edge>\n  <edge id=":1115585385_0" function="internal">\n    <lane id=":1115585385_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.04" length="11.63" shape="1035.39,977.05 1033.06,976.34 1030.99,974.78 1029.18,972.38 1027.64,969.13" />\n  </edge>\n  <edge id=":1115585385_1" function="internal">\n    <lane id=":1115585385_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.89" shape="1035.39,977.05 1020.54,976.04" />\n  </edge>\n  <edge id=":1115585385_2" function="internal">\n    <lane id=":1115585385_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.16" length="2.90" shape="1035.39,977.05 1032.74,977.30 1032.55,977.44" />\n  </edge>\n  <edge id=":1115585385_9" function="internal">\n    <lane id=":1115585385_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.16" length="9.03" shape="1032.55,977.44 1030.81,978.76 1029.61,981.43 1029.14,985.31" />\n  </edge>\n  <edge id=":1115585385_3" function="internal">\n    <lane id=":1115585385_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.41" length="11.92" shape="1020.43,979.24 1024.29,979.73 1027.03,980.91 1028.64,982.77 1029.14,985.31" />\n  </edge>\n  <edge id=":1115585385_4" function="internal">\n    <lane id=":1115585385_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.63" shape="1020.43,979.24 1035.02,980.22" />\n  </edge>\n  <edge id=":1115585385_5" function="internal">\n    <lane id=":1115585385_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.48" length="4.19" shape="1020.43,979.24 1024.59,978.81" />\n  </edge>\n  <edge id=":1115585385_10" function="internal">\n    <lane id=":1115585385_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.48" length="11.84" shape="1024.59,978.81 1025.09,978.75 1027.85,976.91 1028.70,973.70 1027.64,969.13" />\n  </edge>\n  <edge id=":1115585385_6" function="internal">\n    <lane id=":1115585385_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.81" length="9.14" shape="1024.65,970.27 1025.26,972.88 1024.77,974.71 1023.20,975.76 1020.54,976.04" />\n  </edge>\n  <edge id=":1115585385_7" function="internal">\n    <lane id=":1115585385_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.79" shape="1024.65,970.27 1026.13,973.89 1027.61,977.67 1028.74,981.51 1029.14,985.31" />\n  </edge>\n  <edge id=":1115585385_8" function="internal">\n    <lane id=":1115585385_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.24" length="15.12" shape="1024.65,970.27 1026.62,974.33 1029.00,977.34 1031.80,979.30 1035.02,980.22" />\n  </edge>\n  <edge id=":1115585403_0" function="internal">\n    <lane id=":1115585403_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.75" shape="1065.46,1075.03 1065.86,1072.16 1066.04,1070.19 1066.03,1068.20 1065.86,1065.32" />\n  </edge>\n  <edge id=":1115585403_1" function="internal">\n    <lane id=":1115585403_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.97" length="8.10" shape="1065.46,1075.03 1065.64,1072.20 1065.25,1070.16 1064.28,1068.92 1062.73,1068.47" />\n  </edge>\n  <edge id=":1115585403_2" function="internal">\n    <lane id=":1115585403_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.01" length="1.11" shape="1062.64,1071.67 1063.72,1071.42" />\n  </edge>\n  <edge id=":1115585403_3" function="internal">\n    <lane id=":1115585403_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.01" length="6.97" shape="1063.72,1071.42 1064.20,1071.31 1065.26,1070.13 1065.81,1068.14 1065.86,1065.32" />\n  </edge>\n  <edge id=":1115585416_0" function="internal">\n    <lane id=":1115585416_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.56" shape="961.11,838.39 958.83,841.08 956.62,843.48 954.28,845.74 951.61,848.04" />\n  </edge>\n  <edge id=":1115585416_1" function="internal">\n    <lane id=":1115585416_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.80" length="14.68" shape="961.11,838.39 958.99,841.89 958.61,845.11 959.98,848.06 963.09,850.72" />\n  </edge>\n  <edge id=":1115585416_2" function="internal">\n    <lane id=":1115585416_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.26" length="9.05" shape="964.80,848.02 962.89,846.38 962.04,844.56 962.28,842.58 963.58,840.42" />\n  </edge>\n  <edge id=":1115585416_3" function="internal">\n    <lane id=":1115585416_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.13" length="14.19" shape="964.80,848.02 961.20,846.26 957.80,845.67 954.60,846.27 951.61,848.04" />\n  </edge>\n  <edge id=":1115585416_4" function="internal">\n    <lane id=":1115585416_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.81" length="10.16" shape="953.63,850.52 955.72,849.31 957.99,848.94 960.45,849.41 963.09,850.72" />\n  </edge>\n  <edge id=":1115585416_5" function="internal">\n    <lane id=":1115585416_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.20" shape="953.63,850.52 956.38,848.15 958.87,845.73 961.23,843.18 963.58,840.42" />\n  </edge>\n  <edge id=":1115585463_0" function="internal">\n    <lane id=":1115585463_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.54" length="8.01" shape="1001.63,871.11 999.83,872.28 998.02,872.68 996.20,872.30 994.38,871.16" />\n  </edge>\n  <edge id=":1115585463_1" function="internal">\n    <lane id=":1115585463_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.59" length="12.34" shape="1001.63,871.11 999.38,873.58 998.14,876.25 997.90,879.12 998.67,882.20" />\n  </edge>\n  <edge id=":1115585463_2" function="internal">\n    <lane id=":1115585463_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.16" length="8.34" shape="1001.64,881.00 1001.14,878.96 1001.32,877.03 1002.18,875.22 1003.72,873.53" />\n  </edge>\n  <edge id=":1115585463_3" function="internal">\n    <lane id=":1115585463_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.40" shape="1001.64,881.00 1000.26,877.96 998.75,875.55 996.87,873.40 994.38,871.16" />\n  </edge>\n  <edge id=":1115585463_4" function="internal">\n    <lane id=":1115585463_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.85" shape="992.32,873.61 994.67,875.65 996.23,877.37 997.43,879.35 998.67,882.20" />\n  </edge>\n  <edge id=":1115585463_5" function="internal">\n    <lane id=":1115585463_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.05" length="3.51" shape="992.32,873.61 995.18,875.41 995.31,875.43" />\n  </edge>\n  <edge id=":1115585463_6" function="internal">\n    <lane id=":1115585463_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.05" length="9.08" shape="995.31,875.43 998.04,876.00 1000.88,875.37 1003.72,873.53" />\n  </edge>\n  <edge id=":1115585792_0" function="internal">\n    <lane id=":1115585792_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="8.64" shape="783.51,840.81 781.28,839.54 779.76,838.65 778.29,837.70 776.17,836.25" />\n  </edge>\n  <edge id=":1115585792_1" function="internal">\n    <lane id=":1115585792_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.47" length="2.53" shape="783.51,840.81 781.21,840.06 781.10,840.09" />\n  </edge>\n  <edge id=":1115585792_4" function="internal">\n    <lane id=":1115585792_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.47" length="8.00" shape="781.10,840.09 779.24,840.65 777.60,842.58 776.28,845.85" />\n  </edge>\n  <edge id=":1115585792_2" function="internal">\n    <lane id=":1115585792_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.92" length="7.78" shape="774.34,838.88 775.70,840.18 776.48,841.77 776.67,843.66 776.28,845.85" />\n  </edge>\n  <edge id=":1115585792_3" function="internal">\n    <lane id=":1115585792_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="8.92" shape="774.34,838.88 776.53,840.38 778.05,841.35 779.61,842.27 781.91,843.58" />\n  </edge>\n  <edge id=":1115585857_0" function="internal">\n    <lane id=":1115585857_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.58" length="6.10" shape="1025.76,1709.68 1024.62,1711.06 1024.01,1712.15 1023.72,1713.38 1023.55,1715.17" />\n  </edge>\n  <edge id=":1115585857_1" function="internal">\n    <lane id=":1115585857_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="12.69" length="6.77" shape="1021.69,1708.81 1022.59,1710.61 1023.23,1711.88 1023.56,1713.21 1023.55,1715.17" />\n  </edge>\n  <edge id=":1115585909_0" function="internal">\n    <lane id=":1115585909_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.83" shape="1011.55,927.13 1010.46,923.91 1009.79,920.98 1009.45,918.00 1009.30,914.58" />\n  </edge>\n  <edge id=":1115585909_1" function="internal">\n    <lane id=":1115585909_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.37" length="3.39" shape="1011.55,927.13 1009.89,924.18" />\n  </edge>\n  <edge id=":1115585909_6" function="internal">\n    <lane id=":1115585909_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.37" length="10.49" shape="1009.89,924.18 1007.51,922.03 1004.39,920.69 1000.54,920.17" />\n  </edge>\n  <edge id=":1115585909_2" function="internal">\n    <lane id=":1115585909_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="10.09" shape="1000.45,923.37 1003.32,923.74 1005.63,924.69 1007.37,926.20 1008.56,928.28" />\n  </edge>\n  <edge id=":1115585909_3" function="internal">\n    <lane id=":1115585909_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.91" length="14.39" shape="1000.45,923.37 1004.40,922.92 1007.19,921.30 1008.82,918.52 1009.30,914.58" />\n  </edge>\n  <edge id=":1115585909_4" function="internal">\n    <lane id=":1115585909_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.40" length="9.04" shape="1006.10,914.65 1005.80,917.13 1004.77,918.87 1003.02,919.88 1000.54,920.17" />\n  </edge>\n  <edge id=":1115585909_5" function="internal">\n    <lane id=":1115585909_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.92" shape="1006.10,914.65 1006.28,918.23 1006.70,921.59 1007.44,924.88 1008.56,928.28" />\n  </edge>\n  <edge id=":1122072673_0" function="internal">\n    <lane id=":1122072673_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.78" length="8.30" shape="540.31,292.62 539.34,294.45 537.91,295.67 536.02,296.28 533.67,296.27" />\n  </edge>\n  <edge id=":1122072673_1" function="internal">\n    <lane id=":1122072673_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.74" shape="540.31,292.62 537.15,301.83" />\n  </edge>\n  <edge id=":1122072673_2" function="internal">\n    <lane id=":1122072673_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.74" shape="540.18,302.87 543.34,293.66" />\n  </edge>\n  <edge id=":1122072673_3" function="internal">\n    <lane id=":1122072673_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.65" length="2.52" shape="540.18,302.87 540.49,300.37" />\n  </edge>\n  <edge id=":1122072673_4" function="internal">\n    <lane id=":1122072673_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.65" length="8.47" shape="540.49,300.37 539.50,298.43 537.23,297.07 533.67,296.27" />\n  </edge>\n  <edge id=":1122072674_0" function="internal">\n    <lane id=":1122072674_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.16" length="8.49" shape="589.76,389.09 588.28,390.65 586.53,391.55 584.51,391.78 582.22,391.35" />\n  </edge>\n  <edge id=":1122072674_1" function="internal">\n    <lane id=":1122072674_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.61" shape="589.76,389.09 587.93,391.93 586.66,394.56 585.78,397.35 585.11,400.66" />\n  </edge>\n  <edge id=":1122072674_2" function="internal">\n    <lane id=":1122072674_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.18" shape="588.27,401.18 588.85,398.08 589.54,395.74 590.63,393.57 592.37,390.94" />\n  </edge>\n  <edge id=":1122072674_3" function="internal">\n    <lane id=":1122072674_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.04" length="3.55" shape="588.27,401.18 588.36,397.74 588.32,397.64" />\n  </edge>\n  <edge id=":1122072674_6" function="internal">\n    <lane id=":1122072674_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.04" length="9.25" shape="588.32,397.64 587.38,394.95 585.33,392.82 582.22,391.35" />\n  </edge>\n  <edge id=":1122072674_4" function="internal">\n    <lane id=":1122072674_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.54" length="8.14" shape="581.26,394.41 583.24,395.34 584.55,396.70 585.17,398.47 585.11,400.66" />\n  </edge>\n  <edge id=":1122072674_5" function="internal">\n    <lane id=":1122072674_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.59" length="12.54" shape="581.26,394.41 584.61,395.02 587.58,394.65 590.17,393.29 592.37,390.94" />\n  </edge>\n  <edge id=":1122072681_0" function="internal">\n    <lane id=":1122072681_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.47" shape="493.81,483.69 493.62,487.35 493.73,490.92 493.82,494.50 493.58,498.15" />\n  </edge>\n  <edge id=":1122072681_1" function="internal">\n    <lane id=":1122072681_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="14.22" shape="493.81,483.69 493.91,487.59 495.23,490.51 497.77,492.47 501.55,493.45" />\n  </edge>\n  <edge id=":1122072681_2" function="internal">\n    <lane id=":1122072681_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.49" length="9.03" shape="501.90,490.27 499.51,489.64 497.89,488.40 497.05,486.55 496.99,484.07" />\n  </edge>\n  <edge id=":1122072681_3" function="internal">\n    <lane id=":1122072681_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.74" length="2.31" shape="501.90,490.27 499.60,490.40" />\n  </edge>\n  <edge id=":1122072681_4" function="internal">\n    <lane id=":1122072681_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.74" length="10.66" shape="499.60,490.40 498.71,490.46 496.25,491.84 494.54,494.40 493.58,498.15" />\n  </edge>\n  <edge id=":1123399483_0" function="internal">\n    <lane id=":1123399483_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.99" shape="1166.00,1422.62 1152.01,1422.96" />\n  </edge>\n  <edge id=":1123399483_1" function="internal">\n    <lane id=":1123399483_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.32" length="3.73" shape="1166.00,1422.62 1162.35,1423.42" />\n  </edge>\n  <edge id=":1123399483_6" function="internal">\n    <lane id=":1123399483_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.32" length="10.40" shape="1162.35,1423.42 1159.70,1425.23 1158.03,1428.05 1157.35,1431.90" />\n  </edge>\n  <edge id=":1123399483_2" function="internal">\n    <lane id=":1123399483_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.89" length="9.37" shape="1160.55,1432.03 1161.00,1429.45 1162.09,1427.56 1163.83,1426.34 1166.22,1425.81" />\n  </edge>\n  <edge id=":1123399483_3" function="internal">\n    <lane id=":1123399483_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.96" length="14.29" shape="1160.55,1432.03 1160.16,1428.12 1158.60,1425.31 1155.89,1423.59 1152.01,1422.96" />\n  </edge>\n  <edge id=":1123399483_4" function="internal">\n    <lane id=":1123399483_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.45" length="9.03" shape="1151.95,1426.16 1154.41,1426.56 1156.12,1427.64 1157.11,1429.42 1157.35,1431.90" />\n  </edge>\n  <edge id=":1123399483_5" function="internal">\n    <lane id=":1123399483_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.27" shape="1151.95,1426.16 1166.22,1425.81" />\n  </edge>\n  <edge id=":1123399486_0" function="internal">\n    <lane id=":1123399486_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.78" length="9.68" shape="512.26,1603.70 509.46,1603.83 507.30,1603.14 505.78,1601.64 504.91,1599.32" />\n  </edge>\n  <edge id=":1123399486_1" function="internal">\n    <lane id=":1123399486_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.69" length="15.00" shape="512.26,1603.70 508.24,1605.00 505.87,1607.25 505.15,1610.43 506.09,1614.56" />\n  </edge>\n  <edge id=":1123399486_2" function="internal">\n    <lane id=":1123399486_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.12" length="9.06" shape="509.08,1613.41 508.51,1610.92 508.94,1608.99 510.37,1607.64 512.81,1606.85" />\n  </edge>\n  <edge id=":1123399486_3" function="internal">\n    <lane id=":1123399486_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.71" shape="509.08,1613.41 507.81,1609.93 506.69,1606.46 505.73,1602.94 504.91,1599.32" />\n  </edge>\n  <edge id=":1123399486_4" function="internal">\n    <lane id=":1123399486_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.24" shape="501.78,1599.96 502.62,1603.65 503.63,1607.35 504.80,1611.00 506.09,1614.56" />\n  </edge>\n  <edge id=":1123399486_5" function="internal">\n    <lane id=":1123399486_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.23" length="4.11" shape="501.78,1599.96 503.11,1603.57 503.29,1603.76" />\n  </edge>\n  <edge id=":1123399486_6" function="internal">\n    <lane id=":1123399486_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.23" length="10.62" shape="503.29,1603.76 505.39,1605.93 508.62,1607.02 512.81,1606.85" />\n  </edge>\n  <edge id=":1123399489_0" function="internal">\n    <lane id=":1123399489_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.94" shape="1052.38,1402.41 1041.16,1398.34" />\n  </edge>\n  <edge id=":1123399489_1" function="internal">\n    <lane id=":1123399489_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.66" length="9.24" shape="1045.99,1409.67 1045.91,1406.89 1046.75,1405.26 1048.53,1404.76 1051.23,1405.40" />\n  </edge>\n  <edge id=":1123399489_2" function="internal">\n    <lane id=":1123399489_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.40" length="12.70" shape="1045.99,1409.67 1045.06,1405.10 1043.94,1401.69 1042.64,1399.44 1041.16,1398.34" />\n  </edge>\n  <edge id=":1123399489_3" function="internal">\n    <lane id=":1123399489_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.78" shape="1040.16,1401.38 1051.23,1405.40" />\n  </edge>\n  <edge id=":1123399502_0" function="internal">\n    <lane id=":1123399502_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.37" length="8.93" shape="564.48,1512.78 564.12,1515.19 563.05,1516.87 561.27,1517.81 558.79,1518.02" />\n  </edge>\n  <edge id=":1123399502_1" function="internal">\n    <lane id=":1123399502_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.92" shape="564.48,1512.78 564.52,1516.37 564.71,1519.74 565.13,1523.09 565.82,1526.60" />\n  </edge>\n  <edge id=":1123399502_2" function="internal">\n    <lane id=":1123399502_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.89" length="10.33" shape="573.56,1519.96 570.98,1519.54 569.14,1518.25 568.04,1516.10 567.68,1513.07" />\n  </edge>\n  <edge id=":1123399502_3" function="internal">\n    <lane id=":1123399502_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.91" shape="573.56,1519.96 569.84,1519.71 566.17,1519.10 562.50,1518.44 558.79,1518.02" />\n  </edge>\n  <edge id=":1123399502_4" function="internal">\n    <lane id=":1123399502_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.94" length="12.19" shape="573.56,1519.96 569.59,1520.40 566.98,1521.65 565.72,1523.72 565.82,1526.60" />\n  </edge>\n  <edge id=":1123399502_5" function="internal">\n    <lane id=":1123399502_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.90" shape="568.94,1525.87 568.25,1522.52 567.87,1519.52 567.71,1516.49 567.68,1513.07" />\n  </edge>\n  <edge id=":1123399502_6" function="internal">\n    <lane id=":1123399502_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.89" length="3.51" shape="568.94,1525.87 567.65,1522.61" />\n  </edge>\n  <edge id=":1123399502_9" function="internal">\n    <lane id=":1123399502_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.89" length="10.37" shape="567.65,1522.61 565.53,1520.21 562.57,1518.68 558.79,1518.02" />\n  </edge>\n  <edge id=":1123399502_7" function="internal">\n    <lane id=":1123399502_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.53" length="9.73" shape="558.61,1521.22 561.32,1521.68 563.42,1522.73 564.92,1524.37 565.82,1526.60" />\n  </edge>\n  <edge id=":1123399502_8" function="internal">\n    <lane id=":1123399502_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.83" length="14.06" shape="558.61,1521.22 562.57,1520.89 565.40,1519.42 567.10,1516.82 567.68,1513.07" />\n  </edge>\n  <edge id=":1123399513_0" function="internal">\n    <lane id=":1123399513_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.75" shape="1481.15,1699.51 1474.13,1686.54" />\n  </edge>\n  <edge id=":1123399513_1" function="internal">\n    <lane id=":1123399513_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.77" length="3.93" shape="1481.15,1699.51 1479.00,1696.58 1478.75,1696.43" />\n  </edge>\n  <edge id=":1123399513_6" function="internal">\n    <lane id=":1123399513_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.77" length="10.77" shape="1478.75,1696.43 1476.18,1694.85 1472.68,1694.33 1468.52,1695.02" />\n  </edge>\n  <edge id=":1123399513_2" function="internal">\n    <lane id=":1123399513_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.41" length="10.24" shape="1469.40,1698.10 1472.38,1697.58 1474.86,1697.89 1476.84,1699.02 1478.30,1700.97" />\n  </edge>\n  <edge id=":1123399513_3" function="internal">\n    <lane id=":1123399513_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.64" length="15.17" shape="1469.40,1698.10 1473.33,1696.33 1475.43,1693.82 1475.70,1690.55 1474.13,1686.54" />\n  </edge>\n  <edge id=":1123399513_4" function="internal">\n    <lane id=":1123399513_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.06" length="9.07" shape="1471.35,1688.11 1472.28,1690.51 1472.12,1692.46 1470.86,1693.97 1468.52,1695.02" />\n  </edge>\n  <edge id=":1123399513_5" function="internal">\n    <lane id=":1123399513_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.62" shape="1471.35,1688.11 1478.30,1700.97" />\n  </edge>\n  <edge id=":1123399516_0" function="internal">\n    <lane id=":1123399516_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.01" shape="1238.34,1382.75 1237.36,1386.78 1235.23,1391.04 1232.49,1395.02 1229.66,1398.21" />\n  </edge>\n  <edge id=":1123399516_1" function="internal">\n    <lane id=":1123399516_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.34" shape="1238.34,1382.75 1238.30,1387.01 1238.53,1391.90 1238.89,1396.79 1239.26,1401.05" />\n  </edge>\n  <edge id=":1123399516_2" function="internal">\n    <lane id=":1123399516_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.87" shape="1242.45,1400.77 1242.08,1396.59 1241.72,1391.86 1241.49,1387.13 1241.53,1382.94" />\n  </edge>\n  <edge id=":1123399516_3" function="internal">\n    <lane id=":1123399516_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.61" length="4.92" shape="1242.45,1400.77 1241.33,1395.98" />\n  </edge>\n  <edge id=":1123399516_6" function="internal">\n    <lane id=":1123399516_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.61" length="14.45" shape="1241.33,1395.98 1241.20,1395.43 1238.65,1393.22 1234.80,1394.15 1229.66,1398.21" />\n  </edge>\n  <edge id=":1123399516_4" function="internal">\n    <lane id=":1123399516_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.59" length="9.72" shape="1232.94,1399.48 1235.48,1397.50 1237.37,1397.10 1238.64,1398.28 1239.26,1401.05" />\n  </edge>\n  <edge id=":1123399516_5" function="internal">\n    <lane id=":1123399516_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.90" shape="1232.94,1399.48 1235.76,1396.13 1238.47,1391.77 1240.57,1387.14 1241.53,1382.94" />\n  </edge>\n  <edge id=":1123399535_0" function="internal">\n    <lane id=":1123399535_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.96" shape="1143.13,1234.86 1139.58,1234.42 1136.31,1233.59 1133.09,1232.46 1129.74,1231.14" />\n  </edge>\n  <edge id=":1123399535_1" function="internal">\n    <lane id=":1123399535_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.97" length="3.80" shape="1143.13,1234.86 1139.33,1235.12" />\n  </edge>\n  <edge id=":1123399535_6" function="internal">\n    <lane id=":1123399535_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.97" length="10.82" shape="1139.33,1235.12 1138.84,1235.16 1135.77,1236.60 1133.92,1239.17 1133.29,1242.89" />\n  </edge>\n  <edge id=":1123399535_2" function="internal">\n    <lane id=":1123399535_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.33" length="8.83" shape="1136.49,1242.90 1136.87,1240.65 1137.98,1239.08 1139.83,1238.21 1142.40,1238.02" />\n  </edge>\n  <edge id=":1123399535_3" function="internal">\n    <lane id=":1123399535_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.45" length="14.48" shape="1136.49,1242.90 1136.09,1238.76 1134.83,1235.42 1132.71,1232.88 1129.74,1231.14" />\n  </edge>\n  <edge id=":1123399535_4" function="internal">\n    <lane id=":1123399535_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.17" length="10.61" shape="1128.58,1234.12 1130.65,1235.37 1132.13,1237.25 1133.01,1239.75 1133.29,1242.89" />\n  </edge>\n  <edge id=":1123399535_5" function="internal">\n    <lane id=":1123399535_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.43" shape="1128.58,1234.12 1132.00,1235.47 1135.36,1236.66 1138.79,1237.56 1142.40,1238.02" />\n  </edge>\n  <edge id=":1123399561_0" function="internal">\n    <lane id=":1123399561_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.55" shape="1189.32,1229.39 1186.01,1230.59 1182.89,1231.50 1179.71,1232.20 1176.24,1232.82" />\n  </edge>\n  <edge id=":1123399561_1" function="internal">\n    <lane id=":1123399561_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.46" length="3.41" shape="1189.32,1229.39 1186.37,1231.11" />\n  </edge>\n  <edge id=":1123399561_6" function="internal">\n    <lane id=":1123399561_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.46" length="10.86" shape="1186.37,1231.11 1184.26,1233.61 1182.98,1236.88 1182.52,1240.93" />\n  </edge>\n  <edge id=":1123399561_2" function="internal">\n    <lane id=":1123399561_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.17" length="10.46" shape="1185.72,1240.96 1186.04,1237.90 1186.94,1235.45 1188.43,1233.61 1190.50,1232.37" />\n  </edge>\n  <edge id=":1123399561_3" function="internal">\n    <lane id=":1123399561_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.73" length="14.88" shape="1185.72,1240.96 1185.16,1236.82 1183.39,1234.08 1180.42,1232.74 1176.24,1232.82" />\n  </edge>\n  <edge id=":1123399561_4" function="internal">\n    <lane id=":1123399561_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.17" length="9.05" shape="1176.75,1235.98 1179.30,1235.93 1181.10,1236.74 1182.18,1238.41 1182.52,1240.93" />\n  </edge>\n  <edge id=":1123399561_5" function="internal">\n    <lane id=":1123399561_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.24" shape="1176.75,1235.98 1180.33,1235.33 1183.73,1234.56 1187.08,1233.60 1190.50,1232.37" />\n  </edge>\n  <edge id=":1123399562_0" function="internal">\n    <lane id=":1123399562_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.52" shape="830.36,867.58 817.71,860.45" />\n  </edge>\n  <edge id=":1123399562_1" function="internal">\n    <lane id=":1123399562_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.64" length="15.17" shape="830.36,867.58 826.33,866.05 823.07,866.34 820.57,868.46 818.84,872.40" />\n  </edge>\n  <edge id=":1123399562_2" function="internal">\n    <lane id=":1123399562_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.06" length="9.07" shape="821.92,873.26 822.96,870.90 824.45,869.64 826.40,869.46 828.81,870.38" />\n  </edge>\n  <edge id=":1123399562_3" function="internal">\n    <lane id=":1123399562_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.97" length="14.65" shape="821.92,873.26 822.59,869.10 822.12,865.58 820.49,862.70 817.71,860.45" />\n  </edge>\n  <edge id=":1123399562_4" function="internal">\n    <lane id=":1123399562_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.64" length="10.38" shape="816.12,863.23 817.99,864.78 819.07,866.83 819.35,869.37 818.84,872.40" />\n  </edge>\n  <edge id=":1123399562_5" function="internal">\n    <lane id=":1123399562_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.57" shape="816.12,863.23 828.81,870.38" />\n  </edge>\n  <edge id=":1123399569_0" function="internal">\n    <lane id=":1123399569_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.56" length="7.65" shape="885.54,1760.11 883.55,1759.66 882.04,1758.70 881.02,1757.24 880.48,1755.27" />\n  </edge>\n  <edge id=":1123399591_0" function="internal">\n    <lane id=":1123399591_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.40" length="15.19" shape="579.22,1711.85 575.77,1712.38 572.25,1711.98 568.66,1710.63 565.00,1708.35" />\n  </edge>\n  <edge id=":1123399591_1" function="internal">\n    <lane id=":1123399591_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.01" shape="579.22,1711.85 562.09,1717.41" />\n  </edge>\n  <edge id=":1123399591_2" function="internal">\n    <lane id=":1123399591_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.63" length="9.29" shape="578.69,1721.42 576.83,1719.31 576.45,1717.52 577.55,1716.06 580.12,1714.92" />\n  </edge>\n  <edge id=":1123399591_3" function="internal">\n    <lane id=":1123399591_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.93" shape="578.69,1721.42 565.00,1708.35" />\n  </edge>\n  <edge id=":1123399591_4" function="internal">\n    <lane id=":1123399591_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.34" length="17.94" shape="578.69,1721.42 574.66,1718.54 570.54,1716.91 566.35,1716.54 562.09,1717.41" />\n  </edge>\n  <edge id=":1123399591_5" function="internal">\n    <lane id=":1123399591_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.86" shape="563.14,1720.43 580.12,1714.92" />\n  </edge>\n  <edge id=":1123399591_6" function="internal">\n    <lane id=":1123399591_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.38" length="4.76" shape="563.14,1720.43 567.40,1718.31" />\n  </edge>\n  <edge id=":1123399591_9" function="internal">\n    <lane id=":1123399591_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.38" length="12.58" shape="567.40,1718.31 567.87,1718.07 569.76,1715.26 568.80,1712.02 565.00,1708.35" />\n  </edge>\n  <edge id=":1123399591_7" function="internal">\n    <lane id=":1123399591_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.60" length="9.33" shape="563.09,1710.91 565.13,1712.89 565.65,1714.63 564.63,1716.14 562.09,1717.41" />\n  </edge>\n  <edge id=":1123399591_8" function="internal">\n    <lane id=":1123399591_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="12.43" length="18.15" shape="563.09,1710.91 567.41,1713.59 571.68,1715.15 575.92,1715.59 580.12,1714.92" />\n  </edge>\n  <edge id=":1123399595_0" function="internal">\n    <lane id=":1123399595_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.29" shape="790.06,1036.84 787.86,1050.96" />\n  </edge>\n  <edge id=":1123399595_1" function="internal">\n    <lane id=":1123399595_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.95" length="3.93" shape="790.06,1036.84 790.06,1040.77" />\n  </edge>\n  <edge id=":1123399595_6" function="internal">\n    <lane id=":1123399595_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.95" length="10.36" shape="790.06,1040.77 791.33,1043.73 793.86,1045.71 797.66,1046.71" />\n  </edge>\n  <edge id=":1123399595_2" function="internal">\n    <lane id=":1123399595_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.45" length="9.03" shape="798.03,1043.53 795.63,1042.90 794.03,1041.65 793.23,1039.78 793.23,1037.30" />\n  </edge>\n  <edge id=":1123399595_3" function="internal">\n    <lane id=":1123399595_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.15" length="14.20" shape="798.03,1043.53 794.12,1043.62 791.12,1044.88 789.03,1047.33 787.86,1050.96" />\n  </edge>\n  <edge id=":1123399595_4" function="internal">\n    <lane id=":1123399595_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.69" length="9.23" shape="791.02,1051.49 791.78,1049.15 793.14,1047.57 795.10,1046.76 797.66,1046.71" />\n  </edge>\n  <edge id=":1123399595_5" function="internal">\n    <lane id=":1123399595_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.37" shape="791.02,1051.49 793.23,1037.30" />\n  </edge>\n  <edge id=":1123399600_0" function="internal">\n    <lane id=":1123399600_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.47" length="15.31" shape="1092.42,983.43 1088.93,982.93 1085.76,981.39 1082.91,978.81 1080.40,975.18" />\n  </edge>\n  <edge id=":1123399600_1" function="internal">\n    <lane id=":1123399600_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.47" shape="1092.42,983.43 1088.13,983.33 1083.20,983.02 1078.27,982.57 1074.01,982.06" />\n  </edge>\n  <edge id=":1123399600_2" function="internal">\n    <lane id=":1123399600_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.59" length="4.12" shape="1092.42,983.43 1088.35,984.04" />\n  </edge>\n  <edge id=":1123399600_12" function="internal">\n    <lane id=":1123399600_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.59" length="11.27" shape="1088.35,984.04 1088.06,984.08 1085.37,985.97 1084.35,989.10 1084.99,993.46" />\n  </edge>\n  <edge id=":1123399600_3" function="internal">\n    <lane id=":1123399600_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.99" length="9.08" shape="1088.06,992.55 1087.68,989.98 1088.28,988.13 1089.87,987.02 1092.44,986.63" />\n  </edge>\n  <edge id=":1123399600_4" function="internal">\n    <lane id=":1123399600_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.01" shape="1088.06,992.55 1086.61,988.42 1084.62,983.70 1082.43,979.05 1080.40,975.18" />\n  </edge>\n  <edge id=":1123399600_5" function="internal">\n    <lane id=":1123399600_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.85" length="18.18" shape="1088.06,992.55 1086.11,989.14 1082.68,985.93 1078.43,983.41 1074.01,982.06" />\n  </edge>\n  <edge id=":1123399600_6" function="internal">\n    <lane id=":1123399600_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.87" length="14.91" shape="1073.57,985.23 1077.80,986.23 1081.11,987.94 1083.51,990.35 1084.99,993.46" />\n  </edge>\n  <edge id=":1123399600_7" function="internal">\n    <lane id=":1123399600_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.94" shape="1073.57,985.23 1077.91,985.74 1082.99,986.20 1088.08,986.52 1092.44,986.63" />\n  </edge>\n  <edge id=":1123399600_8" function="internal">\n    <lane id=":1123399600_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.38" length="5.53" shape="1073.57,985.23 1079.10,985.31" />\n  </edge>\n  <edge id=":1123399600_13" function="internal">\n    <lane id=":1123399600_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.38" length="12.34" shape="1079.10,985.31 1082.08,983.66 1082.51,980.29 1080.40,975.18" />\n  </edge>\n  <edge id=":1123399600_9" function="internal">\n    <lane id=":1123399600_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.55" length="9.42" shape="1077.61,976.76 1078.73,979.45 1078.50,981.23 1076.93,982.10 1074.01,982.06" />\n  </edge>\n  <edge id=":1123399600_10" function="internal">\n    <lane id=":1123399600_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.28" shape="1077.61,976.76 1079.61,980.53 1081.69,984.94 1083.59,989.44 1084.99,993.46" />\n  </edge>\n  <edge id=":1123399600_11" function="internal">\n    <lane id=":1123399600_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.43" length="18.37" shape="1077.61,976.76 1080.47,980.40 1084.36,983.57 1088.58,985.80 1092.44,986.63" />\n  </edge>\n  <edge id=":1123399639_0" function="internal">\n    <lane id=":1123399639_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.10" length="9.83" shape="1231.09,1171.92 1230.14,1174.60 1228.69,1176.45 1226.74,1177.45 1224.30,1177.62" />\n  </edge>\n  <edge id=":1123399639_1" function="internal">\n    <lane id=":1123399639_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.74" length="14.84" shape="1231.09,1171.92 1230.76,1176.07 1231.89,1179.12 1234.50,1181.08 1238.57,1181.93" />\n  </edge>\n  <edge id=":1123399639_2" function="internal">\n    <lane id=":1123399639_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.19" length="9.05" shape="1238.77,1178.74 1236.29,1178.22 1234.70,1177.03 1234.01,1175.16 1234.21,1172.63" />\n  </edge>\n  <edge id=":1123399639_3" function="internal">\n    <lane id=":1123399639_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.51" shape="1238.77,1178.74 1224.30,1177.62" />\n  </edge>\n  <edge id=":1123399639_4" function="internal">\n    <lane id=":1123399639_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.60" shape="1224.01,1180.81 1238.57,1181.93" />\n  </edge>\n  <edge id=":1123399639_5" function="internal">\n    <lane id=":1123399639_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.50" length="3.96" shape="1224.01,1180.81 1227.71,1180.58 1227.94,1180.47" />\n  </edge>\n  <edge id=":1123399639_6" function="internal">\n    <lane id=":1123399639_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.50" length="10.55" shape="1227.94,1180.47 1230.65,1179.14 1232.81,1176.49 1234.21,1172.63" />\n  </edge>\n  <edge id=":1123399641_0" function="internal">\n    <lane id=":1123399641_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.04" length="9.07" shape="1205.32,1414.18 1202.95,1415.21 1201.00,1415.13 1199.46,1413.93 1198.33,1411.61" />\n  </edge>\n  <edge id=":1123399641_1" function="internal">\n    <lane id=":1123399641_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.07" shape="1205.32,1414.18 1202.03,1416.01 1198.58,1417.54 1195.00,1418.73 1191.33,1419.56" />\n  </edge>\n  <edge id=":1123399641_2" function="internal">\n    <lane id=":1123399641_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.94" length="15.88" shape="1205.32,1414.18 1202.51,1416.66 1200.07,1420.15 1198.35,1424.01 1197.71,1427.60" />\n  </edge>\n  <edge id=":1123399641_3" function="internal">\n    <lane id=":1123399641_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.90" length="12.89" shape="1200.91,1427.59 1201.28,1424.34 1202.42,1421.47 1204.33,1418.99 1207.00,1416.89" />\n  </edge>\n  <edge id=":1123399641_4" function="internal">\n    <lane id=":1123399641_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.24" shape="1200.91,1427.59 1200.71,1423.65 1200.18,1419.51 1199.37,1415.41 1198.33,1411.61" />\n  </edge>\n  <edge id=":1123399641_5" function="internal">\n    <lane id=":1123399641_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.73" length="4.62" shape="1200.91,1427.59 1200.30,1423.46 1200.05,1423.09" />\n  </edge>\n  <edge id=":1123399641_12" function="internal">\n    <lane id=":1123399641_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.73" length="10.25" shape="1200.05,1423.09 1198.50,1420.74 1195.51,1419.45 1191.33,1419.56" />\n  </edge>\n  <edge id=":1123399641_6" function="internal">\n    <lane id=":1123399641_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.17" length="9.05" shape="1191.88,1422.72 1194.42,1422.65 1196.24,1423.44 1197.34,1425.09 1197.71,1427.60" />\n  </edge>\n  <edge id=":1123399641_7" function="internal">\n    <lane id=":1123399641_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.29" shape="1191.88,1422.72 1195.72,1421.82 1199.70,1420.47 1203.54,1418.80 1207.00,1416.89" />\n  </edge>\n  <edge id=":1123399641_8" function="internal">\n    <lane id=":1123399641_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.59" length="14.02" shape="1191.88,1422.72 1194.85,1421.25 1197.18,1418.42 1198.47,1414.96 1198.33,1411.61" />\n  </edge>\n  <edge id=":1123399641_9" function="internal">\n    <lane id=":1123399641_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.35" length="9.46" shape="1195.28,1412.59 1195.75,1415.26 1195.25,1417.31 1193.78,1418.75 1191.33,1419.56" />\n  </edge>\n  <edge id=":1123399641_10" function="internal">\n    <lane id=":1123399641_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.27" shape="1195.28,1412.59 1196.30,1416.24 1197.06,1420.01 1197.54,1423.82 1197.71,1427.60" />\n  </edge>\n  <edge id=":1123399641_11" function="internal">\n    <lane id=":1123399641_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.63" length="3.88" shape="1195.28,1412.59 1196.98,1416.07" />\n  </edge>\n  <edge id=":1123399641_13" function="internal">\n    <lane id=":1123399641_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.63" length="11.34" shape="1196.98,1416.07 1197.18,1416.48 1199.76,1418.50 1203.04,1418.63 1207.00,1416.89" />\n  </edge>\n  <edge id=":1123399659_0" function="internal">\n    <lane id=":1123399659_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.23" shape="1106.91,1421.31 1103.90,1421.16 1101.80,1420.96 1099.75,1420.47 1096.93,1419.47" />\n  </edge>\n  <edge id=":1123399659_1" function="internal">\n    <lane id=":1123399659_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.12" length="9.06" shape="1101.66,1430.08 1101.65,1427.52 1102.50,1425.74 1104.19,1424.73 1106.74,1424.50" />\n  </edge>\n  <edge id=":1123399659_2" function="internal">\n    <lane id=":1123399659_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.34" length="12.02" shape="1101.66,1430.08 1100.85,1425.92 1099.79,1422.76 1098.48,1420.61 1096.93,1419.47" />\n  </edge>\n  <edge id=":1123399659_3" function="internal">\n    <lane id=":1123399659_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.22" shape="1095.78,1422.45 1098.76,1423.49 1101.14,1424.05 1103.58,1424.32 1106.74,1424.50" />\n  </edge>\n  <edge id=":1123399675_0" function="internal">\n    <lane id=":1123399675_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.16" length="9.90" shape="542.29,1669.46 539.48,1670.05 537.12,1669.79 535.22,1668.67 533.77,1666.70" />\n  </edge>\n  <edge id=":1123399675_1" function="internal">\n    <lane id=":1123399675_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.73" length="14.88" shape="542.29,1669.46 538.55,1671.34 536.54,1673.90 536.25,1677.15 537.68,1681.08" />\n  </edge>\n  <edge id=":1123399675_2" function="internal">\n    <lane id=":1123399675_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.17" length="9.05" shape="540.50,1679.56 539.63,1677.17 539.81,1675.20 541.03,1673.64 543.31,1672.50" />\n  </edge>\n  <edge id=":1123399675_3" function="internal">\n    <lane id=":1123399675_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.52" shape="540.50,1679.56 533.77,1666.70" />\n  </edge>\n  <edge id=":1123399675_4" function="internal">\n    <lane id=":1123399675_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.60" shape="530.92,1668.14 537.68,1681.08" />\n  </edge>\n  <edge id=":1123399675_5" function="internal">\n    <lane id=":1123399675_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.55" length="3.93" shape="530.92,1668.14 533.07,1671.14 533.27,1671.26" />\n  </edge>\n  <edge id=":1123399675_6" function="internal">\n    <lane id=":1123399675_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.55" length="10.60" shape="533.27,1671.26 535.85,1672.86 539.27,1673.32 543.31,1672.50" />\n  </edge>\n  <edge id=":1123399676_0" function="internal">\n    <lane id=":1123399676_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.65" shape="999.45,1675.14 997.21,1678.07 995.28,1681.15 993.40,1684.26 991.26,1687.28" />\n  </edge>\n  <edge id=":1123399676_1" function="internal">\n    <lane id=":1123399676_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.91" length="14.31" shape="999.45,1675.14 997.24,1678.46 996.65,1681.62 997.67,1684.65 1000.31,1687.53" />\n  </edge>\n  <edge id=":1123399676_2" function="internal">\n    <lane id=":1123399676_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.42" length="9.04" shape="1002.40,1685.10 1000.73,1683.28 1000.08,1681.37 1000.45,1679.36 1001.84,1677.27" />\n  </edge>\n  <edge id=":1123399676_3" function="internal">\n    <lane id=":1123399676_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.64" length="2.33" shape="1002.40,1685.10 1000.37,1683.95" />\n  </edge>\n  <edge id=":1123399676_4" function="internal">\n    <lane id=":1123399676_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.64" length="10.59" shape="1000.37,1683.95 999.58,1683.51 996.78,1683.34 994.01,1684.59 991.26,1687.28" />\n  </edge>\n  <edge id=":1123399683_0" function="internal">\n    <lane id=":1123399683_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.05" shape="1238.61,1315.01 1237.95,1300.98" />\n  </edge>\n  <edge id=":1123399683_1" function="internal">\n    <lane id=":1123399683_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.23" length="3.60" shape="1238.61,1315.01 1237.76,1311.52" />\n  </edge>\n  <edge id=":1123399683_6" function="internal">\n    <lane id=":1123399683_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.23" length="10.39" shape="1237.76,1311.52 1235.88,1309.01 1232.97,1307.49 1229.03,1306.95" />\n  </edge>\n  <edge id=":1123399683_2" function="internal">\n    <lane id=":1123399683_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.68" length="8.88" shape="1229.35,1310.15 1231.85,1310.49 1233.69,1311.46 1234.88,1313.06 1235.42,1315.28" />\n  </edge>\n  <edge id=":1123399683_3" function="internal">\n    <lane id=":1123399683_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="14.39" shape="1229.35,1310.15 1233.14,1309.61 1235.84,1307.90 1237.44,1305.02 1237.95,1300.98" />\n  </edge>\n  <edge id=":1123399683_4" function="internal">\n    <lane id=":1123399683_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.60" length="9.44" shape="1234.75,1301.01 1234.41,1303.63 1233.35,1305.49 1231.55,1306.60 1229.03,1306.95" />\n  </edge>\n  <edge id=":1123399683_5" function="internal">\n    <lane id=":1123399683_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.28" shape="1234.75,1301.01 1235.42,1315.28" />\n  </edge>\n  <edge id=":1123399700_0" function="internal">\n    <lane id=":1123399700_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.28" shape="1439.96,1648.74 1428.12,1639.08" />\n  </edge>\n  <edge id=":1123399700_1" function="internal">\n    <lane id=":1123399700_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.47" length="4.90" shape="1439.96,1648.74 1436.00,1646.16 1435.83,1646.14" />\n  </edge>\n  <edge id=":1123399700_6" function="internal">\n    <lane id=":1123399700_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.47" length="11.20" shape="1435.83,1646.14 1432.69,1645.87 1430.03,1647.86 1428.02,1652.14" />\n  </edge>\n  <edge id=":1123399700_2" function="internal">\n    <lane id=":1123399700_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.80" length="9.15" shape="1431.08,1653.09 1432.22,1650.65 1433.73,1649.52 1435.61,1649.69 1437.86,1651.16" />\n  </edge>\n  <edge id=":1123399700_3" function="internal">\n    <lane id=":1123399700_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.55" length="15.32" shape="1431.08,1653.09 1432.01,1648.62 1431.83,1644.79 1430.53,1641.62 1428.12,1639.08" />\n  </edge>\n  <edge id=":1123399700_4" function="internal">\n    <lane id=":1123399700_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.30" length="11.36" shape="1426.21,1641.65 1427.87,1643.46 1428.72,1645.81 1428.77,1648.70 1428.02,1652.14" />\n  </edge>\n  <edge id=":1123399700_5" function="internal">\n    <lane id=":1123399700_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.04" shape="1426.21,1641.65 1437.86,1651.16" />\n  </edge>\n  <edge id=":1123399706_0" function="internal">\n    <lane id=":1123399706_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.23" length="9.05" shape="804.49,937.65 803.71,940.06 802.36,941.53 800.43,942.05 797.94,941.64" />\n  </edge>\n  <edge id=":1123399706_1" function="internal">\n    <lane id=":1123399706_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.42" shape="804.49,937.65 802.09,952.88" />\n  </edge>\n  <edge id=":1123399706_2" function="internal">\n    <lane id=":1123399706_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.62" length="15.16" shape="804.49,937.65 804.36,941.74 805.52,945.02 807.97,947.51 811.70,949.20" />\n  </edge>\n  <edge id=":1123399706_3" function="internal">\n    <lane id=":1123399706_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.25" length="10.43" shape="812.61,946.13 810.04,944.97 808.36,943.26 807.56,941.00 807.64,938.19" />\n  </edge>\n  <edge id=":1123399706_4" function="internal">\n    <lane id=":1123399706_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.34" shape="812.61,946.13 797.94,941.64" />\n  </edge>\n  <edge id=":1123399706_5" function="internal">\n    <lane id=":1123399706_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.77" length="4.13" shape="812.61,946.13 808.52,945.54" />\n  </edge>\n  <edge id=":1123399706_12" function="internal">\n    <lane id=":1123399706_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.77" length="10.64" shape="808.52,945.54 805.40,946.47 803.26,948.92 802.09,952.88" />\n  </edge>\n  <edge id=":1123399706_6" function="internal">\n    <lane id=":1123399706_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.22" length="9.05" shape="805.26,953.34 805.98,950.91 807.29,949.41 809.20,948.84 811.70,949.20" />\n  </edge>\n  <edge id=":1123399706_7" function="internal">\n    <lane id=":1123399706_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.33" shape="805.26,953.34 807.64,938.19" />\n  </edge>\n  <edge id=":1123399706_8" function="internal">\n    <lane id=":1123399706_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.81" length="15.23" shape="805.26,953.34 805.29,949.25 804.07,945.93 801.62,943.40 797.94,941.64" />\n  </edge>\n  <edge id=":1123399706_9" function="internal">\n    <lane id=":1123399706_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.46" length="10.67" shape="796.97,944.69 799.55,945.92 801.26,947.70 802.11,950.02 802.09,952.88" />\n  </edge>\n  <edge id=":1123399706_10" function="internal">\n    <lane id=":1123399706_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.41" shape="796.97,944.69 811.70,949.20" />\n  </edge>\n  <edge id=":1123399706_11" function="internal">\n    <lane id=":1123399706_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.77" length="4.28" shape="796.97,944.69 801.04,945.36 801.19,945.32" />\n  </edge>\n  <edge id=":1123399706_13" function="internal">\n    <lane id=":1123399706_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.77" length="10.47" shape="801.19,945.32 804.17,944.51 806.37,942.12 807.64,938.19" />\n  </edge>\n  <edge id=":1123399713_0" function="internal">\n    <lane id=":1123399713_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.96" shape="1226.49,1652.24 1211.81,1655.08" />\n  </edge>\n  <edge id=":1123399713_1" function="internal">\n    <lane id=":1123399713_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.61" length="3.69" shape="1226.49,1652.24 1223.23,1653.42 1223.06,1653.57" />\n  </edge>\n  <edge id=":1123399713_6" function="internal">\n    <lane id=":1123399713_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.61" length="11.44" shape="1223.06,1653.57 1220.65,1655.64 1218.75,1658.91 1217.53,1663.22" />\n  </edge>\n  <edge id=":1123399713_2" function="internal">\n    <lane id=":1123399713_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.35" length="11.25" shape="1220.67,1663.80 1221.59,1660.46 1222.96,1657.95 1224.78,1656.25 1227.07,1655.38" />\n  </edge>\n  <edge id=":1123399713_3" function="internal">\n    <lane id=":1123399713_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.50" length="15.90" shape="1220.67,1663.80 1220.87,1659.16 1219.45,1656.17 1216.43,1654.81 1211.81,1655.08" />\n  </edge>\n  <edge id=":1123399713_4" function="internal">\n    <lane id=":1123399713_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.84" length="9.13" shape="1212.44,1658.22 1215.09,1658.06 1216.83,1658.84 1217.64,1660.56 1217.53,1663.22" />\n  </edge>\n  <edge id=":1123399713_5" function="internal">\n    <lane id=":1123399713_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.90" shape="1212.44,1658.22 1227.07,1655.38" />\n  </edge>\n  <edge id=":1123399740_0" function="internal">\n    <lane id=":1123399740_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.78" shape="1287.34,1640.92 1272.83,1643.74" />\n  </edge>\n  <edge id=":1123399740_1" function="internal">\n    <lane id=":1123399740_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.79" length="3.42" shape="1287.34,1640.92 1284.15,1642.16" />\n  </edge>\n  <edge id=":1123399740_6" function="internal">\n    <lane id=":1123399740_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.79" length="11.65" shape="1284.15,1642.16 1281.60,1644.42 1279.69,1647.69 1278.43,1651.97" />\n  </edge>\n  <edge id=":1123399740_2" function="internal">\n    <lane id=":1123399740_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.55" length="11.33" shape="1281.57,1652.59 1282.52,1649.24 1283.91,1646.70 1285.74,1644.97 1288.00,1644.05" />\n  </edge>\n  <edge id=":1123399740_3" function="internal">\n    <lane id=":1123399740_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.50" length="15.88" shape="1281.57,1652.59 1281.82,1647.97 1280.45,1644.95 1277.45,1643.54 1272.83,1643.74" />\n  </edge>\n  <edge id=":1123399740_4" function="internal">\n    <lane id=":1123399740_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.85" length="9.12" shape="1273.41,1646.88 1276.07,1646.77 1277.79,1647.58 1278.58,1649.31 1278.43,1651.97" />\n  </edge>\n  <edge id=":1123399740_5" function="internal">\n    <lane id=":1123399740_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.86" shape="1273.41,1646.88 1288.00,1644.05" />\n  </edge>\n  <edge id=":1123399771_0" function="internal">\n    <lane id=":1123399771_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.57" length="9.02" shape="1065.89,1110.48 1063.54,1109.72 1061.98,1108.39 1061.20,1106.51 1061.20,1104.06" />\n  </edge>\n  <edge id=":1123399771_1" function="internal">\n    <lane id=":1123399771_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="8.97" shape="1058.86,1118.13 1059.65,1115.80 1060.99,1114.28 1062.89,1113.56 1065.34,1113.64" />\n  </edge>\n  <edge id=":1123399771_2" function="internal">\n    <lane id=":1123399771_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.26" shape="1058.86,1118.13 1061.20,1104.06" />\n  </edge>\n  <edge id=":1123399863_0" function="internal">\n    <lane id=":1123399863_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.19" length="9.05" shape="712.88,1748.64 713.07,1751.18 712.36,1753.03 710.76,1754.21 708.28,1754.72" />\n  </edge>\n  <edge id=":1123399943_0" function="internal">\n    <lane id=":1123399943_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.19" length="10.10" shape="522.32,1629.48 519.36,1629.83 516.97,1629.37 515.15,1628.08 513.90,1625.97" />\n  </edge>\n  <edge id=":1123399943_1" function="internal">\n    <lane id=":1123399943_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.64" length="15.18" shape="522.32,1629.48 518.31,1631.06 516.09,1633.47 515.68,1636.72 517.06,1640.81" />\n  </edge>\n  <edge id=":1123399943_2" function="internal">\n    <lane id=":1123399943_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.05" length="9.07" shape="519.91,1639.36 519.09,1636.92 519.34,1634.98 520.66,1633.54 523.06,1632.59" />\n  </edge>\n  <edge id=":1123399943_3" function="internal">\n    <lane id=":1123399943_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.68" shape="519.91,1639.36 518.27,1636.04 516.73,1632.74 515.28,1629.41 513.90,1625.97" />\n  </edge>\n  <edge id=":1123399943_4" function="internal">\n    <lane id=":1123399943_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.01" shape="510.91,1627.12 512.31,1630.60 513.81,1634.04 515.40,1637.44 517.06,1640.81" />\n  </edge>\n  <edge id=":1123399943_5" function="internal">\n    <lane id=":1123399943_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.58" length="3.92" shape="510.91,1627.12 512.77,1630.33 512.94,1630.46" />\n  </edge>\n  <edge id=":1123399943_6" function="internal">\n    <lane id=":1123399943_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.58" length="10.85" shape="512.94,1630.46 515.42,1632.31 518.84,1633.07 523.06,1632.59" />\n  </edge>\n  <edge id=":11292971036_0" function="internal">\n    <lane id=":11292971036_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.40" shape="1563.16,1177.53 1562.77,1177.60" />\n  </edge>\n  <edge id=":1140280757_0" function="internal">\n    <lane id=":1140280757_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.91" length="9.63" shape="691.97,1122.16 691.05,1124.76 689.62,1126.52 687.68,1127.44 685.23,1127.51" />\n  </edge>\n  <edge id=":1140280757_1" function="internal">\n    <lane id=":1140280757_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.78" length="14.72" shape="691.97,1122.16 691.67,1126.26 692.82,1129.30 695.39,1131.28 699.40,1132.20" />\n  </edge>\n  <edge id=":1140280757_2" function="internal">\n    <lane id=":1140280757_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.24" length="9.05" shape="699.66,1129.01 697.20,1128.44 695.61,1127.23 694.91,1125.36 695.09,1122.84" />\n  </edge>\n  <edge id=":1140280757_3" function="internal">\n    <lane id=":1140280757_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.51" shape="699.66,1129.01 685.23,1127.51" />\n  </edge>\n  <edge id=":1140280757_4" function="internal">\n    <lane id=":1140280757_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.98" length="10.69" shape="691.44,1137.19 692.07,1134.78 693.61,1133.14 696.05,1132.28 699.40,1132.20" />\n  </edge>\n  <edge id=":1140280757_5" function="internal">\n    <lane id=":1140280757_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.84" shape="691.44,1137.19 692.00,1133.53 692.99,1129.98 694.12,1126.44 695.09,1122.84" />\n  </edge>\n  <edge id=":1140280757_6" function="internal">\n    <lane id=":1140280757_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.83" length="12.89" shape="691.44,1137.19 691.26,1133.28 690.17,1130.37 688.16,1128.44 685.23,1127.51" />\n  </edge>\n  <edge id=":1140280757_7" function="internal">\n    <lane id=":1140280757_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.66" shape="684.82,1130.69 699.40,1132.20" />\n  </edge>\n  <edge id=":1140280757_8" function="internal">\n    <lane id=":1140280757_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.34" length="4.09" shape="684.82,1130.69 688.59,1130.60 688.88,1130.46" />\n  </edge>\n  <edge id=":1140280757_9" function="internal">\n    <lane id=":1140280757_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.34" length="10.38" shape="688.88,1130.46 691.55,1129.26 693.72,1126.67 695.09,1122.84" />\n  </edge>\n  <edge id=":1140280758_0" function="internal">\n    <lane id=":1140280758_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.50" length="9.26" shape="683.05,1400.52 680.57,1400.23 678.81,1399.18 677.76,1397.36 677.42,1394.77" />\n  </edge>\n  <edge id=":1140280758_1" function="internal">\n    <lane id=":1140280758_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.54" shape="683.05,1400.52 679.84,1400.58 677.28,1400.47 674.74,1400.16 671.57,1399.61" />\n  </edge>\n  <edge id=":1140280758_2" function="internal">\n    <lane id=":1140280758_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.23" shape="670.99,1402.75 674.26,1403.31 677.04,1403.64 679.83,1403.77 683.15,1403.72" />\n  </edge>\n  <edge id=":1140280758_3" function="internal">\n    <lane id=":1140280758_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.00" length="2.80" shape="670.99,1402.75 673.78,1402.70" />\n  </edge>\n  <edge id=":1140280758_4" function="internal">\n    <lane id=":1140280758_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.00" length="9.28" shape="673.78,1402.70 675.79,1401.35 677.00,1398.71 677.42,1394.77" />\n  </edge>\n  <edge id=":1160612969_0" function="internal">\n    <lane id=":1160612969_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.49" length="9.03" shape="901.56,610.08 900.97,612.48 899.75,614.12 897.90,614.98 895.42,615.06" />\n  </edge>\n  <edge id=":1160612969_1" function="internal">\n    <lane id=":1160612969_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.52" shape="901.72,624.58 902.34,620.96 903.24,617.49 904.13,614.01 904.74,610.39" />\n  </edge>\n  <edge id=":1160612969_2" function="internal">\n    <lane id=":1160612969_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.96" length="10.40" shape="901.72,624.58 901.52,622.10 900.34,620.22 898.19,618.93 895.08,618.24" />\n  </edge>\n  <edge id=":1160612974_0" function="internal">\n    <lane id=":1160612974_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.74" shape="888.37,693.51 899.06,694.53" />\n  </edge>\n  <edge id=":1160612974_1" function="internal">\n    <lane id=":1160612974_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.56" length="8.80" shape="888.37,693.51 890.72,693.39 892.52,692.53 893.76,690.93 894.45,688.60" />\n  </edge>\n  <edge id=":1174378788_0" function="internal">\n    <lane id=":1174378788_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.90" shape="505.01,397.83 504.11,400.81 503.54,403.08 503.15,405.38 502.77,408.47" />\n  </edge>\n  <edge id=":1174378788_1" function="internal">\n    <lane id=":1174378788_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.30" shape="505.94,408.85 506.31,405.86 506.66,403.75 507.17,401.68 508.06,398.80" />\n  </edge>\n  <edge id=":1174378788_2" function="internal">\n    <lane id=":1174378788_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.50" length="9.03" shape="497.86,402.26 500.26,402.89 501.87,404.14 502.71,406.00 502.77,408.47" />\n  </edge>\n  <edge id=":1174378788_3" function="internal">\n    <lane id=":1174378788_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.85" length="11.63" shape="497.86,402.26 501.79,402.43 504.80,401.90 506.89,400.70 508.06,398.80" />\n  </edge>\n  <edge id=":1176926303_0" function="internal">\n    <lane id=":1176926303_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.28" length="9.04" shape="411.09,570.02 408.63,569.46 407.04,568.24 406.30,566.38 406.42,563.87" />\n  </edge>\n  <edge id=":1176926303_1" function="internal">\n    <lane id=":1176926303_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.91" length="9.57" shape="403.78,578.08 404.58,575.76 406.02,574.18 408.10,573.33 410.82,573.21" />\n  </edge>\n  <edge id=":1176926303_2" function="internal">\n    <lane id=":1176926303_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.51" shape="403.78,578.08 406.42,563.87" />\n    <lane id=":1176926303_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.51" shape="400.63,577.52 403.28,563.26" />\n    <lane id=":1176926303_2_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.51" shape="397.48,576.97 400.14,562.65" />\n  </edge>\n  <edge id=":1196113662_0" function="internal">\n    <lane id=":1196113662_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.43" shape="355.34,377.89 353.56,389.33" />\n    <lane id=":1196113662_0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.43" shape="358.49,378.45 356.73,389.75" />\n    <lane id=":1196113662_0_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.43" shape="361.64,379.01 359.90,390.17" />\n  </edge>\n  <edge id=":1196113662_3" function="internal">\n    <lane id=":1196113662_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.43" length="9.03" shape="361.64,379.01 361.56,381.50 362.30,383.38 363.87,384.67 366.26,385.36" />\n  </edge>\n  <edge id=":1196113662_4" function="internal">\n    <lane id=":1196113662_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.48" length="9.03" shape="348.72,383.08 351.12,383.72 352.72,384.98 353.54,386.85 353.56,389.33" />\n  </edge>\n  <edge id=":1196113662_5" function="internal">\n    <lane id=":1196113662_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.68" shape="348.72,383.08 366.26,385.36" />\n  </edge>\n  <edge id=":12339345632_0" function="internal">\n    <lane id=":12339345632_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.49" shape="761.47,1389.68 747.22,1392.31" />\n  </edge>\n  <edge id=":12339345632_1" function="internal">\n    <lane id=":12339345632_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.06" length="4.11" shape="761.47,1389.68 757.80,1390.89 757.62,1391.04" />\n  </edge>\n  <edge id=":12339345632_6" function="internal">\n    <lane id=":12339345632_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.06" length="10.16" shape="757.62,1391.04 755.37,1393.00 754.20,1396.03 754.28,1399.96" />\n  </edge>\n  <edge id=":12339345632_2" function="internal">\n    <lane id=":12339345632_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.58" length="9.15" shape="757.44,1399.45 757.38,1396.92 758.13,1394.97 759.68,1393.61 762.03,1392.83" />\n  </edge>\n  <edge id=":12339345632_3" function="internal">\n    <lane id=":12339345632_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.94" length="14.32" shape="757.44,1399.45 756.27,1395.69 754.17,1393.24 751.16,1392.12 747.22,1392.31" />\n  </edge>\n  <edge id=":12339345632_4" function="internal">\n    <lane id=":12339345632_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.44" length="9.03" shape="747.83,1395.45 750.32,1395.33 752.22,1396.04 753.54,1397.58 754.28,1399.96" />\n  </edge>\n  <edge id=":12339345632_5" function="internal">\n    <lane id=":12339345632_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.43" shape="747.83,1395.45 762.03,1392.83" />\n  </edge>\n  <edge id=":12344163778_0" function="internal">\n    <lane id=":12344163778_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.40" shape="529.73,1036.68 528.19,1045.96" />\n  </edge>\n  <edge id=":12344163778_1" function="internal">\n    <lane id=":12344163778_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.18" length="7.82" shape="529.73,1036.68 529.48,1039.41 529.74,1041.46 530.51,1042.81 531.80,1043.47" />\n  </edge>\n  <edge id=":12344163778_2" function="internal">\n    <lane id=":12344163778_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.04" length="1.02" shape="532.44,1040.34 531.43,1040.41" />\n  </edge>\n  <edge id=":12344163778_3" function="internal">\n    <lane id=":12344163778_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.04" length="6.84" shape="531.43,1040.41 530.97,1040.44 529.77,1041.41 528.85,1043.25 528.19,1045.96" />\n  </edge>\n  <edge id=":12346099051_0" function="internal">\n    <lane id=":12346099051_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.20" length="9.05" shape="572.13,1491.62 569.59,1491.54 567.82,1490.64 566.80,1488.93 566.55,1486.41" />\n  </edge>\n  <edge id=":12346099051_1" function="internal">\n    <lane id=":12346099051_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.49" shape="572.13,1491.62 568.94,1491.95 566.40,1492.10 563.86,1492.08 560.67,1491.91" />\n  </edge>\n  <edge id=":12346099051_2" function="internal">\n    <lane id=":12346099051_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.31" length="7.47" shape="566.08,1497.73 566.51,1496.71 567.72,1495.89 569.71,1495.25 572.48,1494.80" />\n  </edge>\n  <edge id=":12346099051_3" function="internal">\n    <lane id=":12346099051_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.33" shape="566.08,1497.73 566.55,1486.41" />\n  </edge>\n  <edge id=":12346099051_4" function="internal">\n    <lane id=":12346099051_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.57" length="9.01" shape="566.08,1497.73 565.82,1495.33 564.83,1493.55 563.11,1492.41 560.67,1491.91" />\n  </edge>\n  <edge id=":12346099051_5" function="internal">\n    <lane id=":12346099051_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.06" shape="560.45,1495.10 563.73,1495.27 566.47,1495.29 569.21,1495.13 572.48,1494.80" />\n  </edge>\n  <edge id=":12346099051_6" function="internal">\n    <lane id=":12346099051_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.34" length="11.99" shape="560.45,1495.10 562.97,1494.70 564.82,1493.12 566.02,1490.36 566.55,1486.41" />\n  </edge>\n  <edge id=":12346099054_0" function="internal">\n    <lane id=":12346099054_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.52" length="8.98" shape="1016.56,1353.31 1016.14,1355.63 1015.01,1357.27 1013.17,1358.24 1010.63,1358.54" />\n  </edge>\n  <edge id=":12346099054_1" function="internal">\n    <lane id=":12346099054_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.28" shape="1016.56,1353.31 1016.03,1364.58" />\n  </edge>\n  <edge id=":12346099054_2" function="internal">\n    <lane id=":12346099054_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.40" shape="1019.22,1364.79 1019.76,1353.41" />\n  </edge>\n  <edge id=":12346099054_3" function="internal">\n    <lane id=":12346099054_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.30" length="2.85" shape="1019.22,1364.79 1018.84,1362.11 1018.76,1362.00" />\n  </edge>\n  <edge id=":12346099054_4" function="internal">\n    <lane id=":12346099054_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.30" length="9.27" shape="1018.76,1362.00 1017.28,1360.17 1014.55,1358.98 1010.63,1358.54" />\n  </edge>\n  <edge id=":12351390200_0" function="internal">\n    <lane id=":12351390200_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.38" shape="1197.25,1408.26 1197.12,1407.91" />\n  </edge>\n  <edge id=":12351390200_1" function="internal">\n    <lane id=":12351390200_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.13" shape="1194.16,1409.12 1194.21,1409.24" />\n  </edge>\n  <edge id=":12649445299_0" function="internal">\n    <lane id=":12649445299_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.58" shape="908.61,873.44 904.99,874.26 901.45,875.04 897.89,875.53 894.24,875.48" />\n  </edge>\n  <edge id=":12649445299_1" function="internal">\n    <lane id=":12649445299_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.07" length="3.91" shape="908.61,873.44 905.29,874.70 905.02,874.94" />\n  </edge>\n  <edge id=":12649445299_6" function="internal">\n    <lane id=":12649445299_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.07" length="10.84" shape="905.02,874.94 902.82,876.94 901.22,880.16 900.48,884.36" />\n  </edge>\n  <edge id=":12649445299_2" function="internal">\n    <lane id=":12649445299_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.76" length="10.55" shape="903.67,884.58 904.19,881.47 905.30,879.10 906.98,877.47 909.25,876.58" />\n  </edge>\n  <edge id=":12649445299_3" function="internal">\n    <lane id=":12649445299_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.28" length="14.76" shape="903.67,884.58 903.28,881.01 901.58,878.30 898.56,876.46 894.24,875.48" />\n  </edge>\n  <edge id=":12649445299_4" function="internal">\n    <lane id=":12649445299_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.57" length="8.92" shape="894.88,878.77 897.45,879.37 899.24,880.50 900.25,882.16 900.48,884.36" />\n  </edge>\n  <edge id=":12649445299_5" function="internal">\n    <lane id=":12649445299_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.61" shape="894.88,878.77 898.53,878.80 902.09,878.26 905.64,877.42 909.25,876.58" />\n  </edge>\n  <edge id=":12649445300_0" function="internal">\n    <lane id=":12649445300_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.46" shape="810.34,903.69 807.85,917.93" />\n  </edge>\n  <edge id=":12649445300_1" function="internal">\n    <lane id=":12649445300_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.84" length="4.17" shape="810.34,903.69 810.21,907.73 810.26,907.85" />\n  </edge>\n  <edge id=":12649445300_6" function="internal">\n    <lane id=":12649445300_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.84" length="10.38" shape="810.26,907.85 811.43,910.72 814.01,912.67 817.95,913.58" />\n  </edge>\n  <edge id=":12649445300_2" function="internal">\n    <lane id=":12649445300_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.32" length="9.04" shape="818.22,910.39 815.77,909.83 814.17,908.62 813.41,906.76 813.49,904.25" />\n  </edge>\n  <edge id=":12649445300_3" function="internal">\n    <lane id=":12649445300_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.30" length="14.35" shape="818.22,910.39 814.21,910.58 811.15,911.90 809.03,914.35 807.85,917.93" />\n  </edge>\n  <edge id=":12649445300_4" function="internal">\n    <lane id=":12649445300_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.86" length="9.50" shape="811.01,918.47 811.79,916.14 813.20,914.55 815.26,913.70 817.95,913.58" />\n  </edge>\n  <edge id=":12649445300_5" function="internal">\n    <lane id=":12649445300_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.44" shape="811.01,918.47 813.49,904.25" />\n  </edge>\n  <edge id=":13279030938_0" function="internal">\n    <lane id=":13279030938_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.72" length="9.34" shape="1305.82,978.55 1305.34,981.13 1304.20,982.97 1302.42,984.06 1299.98,984.42" />\n  </edge>\n  <edge id=":13279030938_1" function="internal">\n    <lane id=":13279030938_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.89" length="14.45" shape="1305.82,978.55 1306.18,982.54 1307.74,985.36 1310.50,987.04 1314.46,987.56" />\n  </edge>\n  <edge id=":13279030938_2" function="internal">\n    <lane id=":13279030938_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.37" length="9.04" shape="1314.42,984.36 1311.94,984.03 1310.22,982.99 1309.24,981.22 1309.02,978.73" />\n  </edge>\n  <edge id=":13279030938_3" function="internal">\n    <lane id=":13279030938_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.44" shape="1314.42,984.36 1299.98,984.42" />\n  </edge>\n  <edge id=":13279030938_4" function="internal">\n    <lane id=":13279030938_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.49" shape="1299.97,987.62 1314.46,987.56" />\n  </edge>\n  <edge id=":13279030938_5" function="internal">\n    <lane id=":13279030938_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.18" length="4.04" shape="1299.97,987.62 1303.75,987.08 1303.94,986.96" />\n  </edge>\n  <edge id=":13279030938_6" function="internal">\n    <lane id=":13279030938_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.18" length="10.27" shape="1303.94,986.96 1306.52,985.42 1308.27,982.63 1309.02,978.73" />\n  </edge>\n  <edge id=":13339819885_0" function="internal">\n    <lane id=":13339819885_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.63" length="8.73" shape="906.55,529.49 905.16,530.30 903.29,530.79 900.95,530.96 898.12,530.81" />\n  </edge>\n  <edge id=":13339819885_1" function="internal">\n    <lane id=":13339819885_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.95" length="11.36" shape="906.55,529.49 903.98,532.15 902.76,534.63 902.89,536.93 904.38,539.05" />\n  </edge>\n  <edge id=":13339819885_2" function="internal">\n    <lane id=":13339819885_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.26" length="8.62" shape="897.79,533.99 900.14,534.56 901.50,535.62 902.65,537.12 904.38,539.05" />\n  </edge>\n  <edge id=":13339819888_0" function="internal">\n    <lane id=":13339819888_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.37" length="7.75" shape="911.28,523.55 910.82,525.68 909.92,526.94 908.56,527.96 906.70,529.36" />\n  </edge>\n  <edge id=":13339819888_1" function="internal">\n    <lane id=":13339819888_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.84" length="5.98" shape="911.94,529.27 914.47,523.86" />\n  </edge>\n  <edge id=":13339819888_2" function="internal">\n    <lane id=":13339819888_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.81" length="5.70" shape="911.94,529.27 910.57,528.53 909.25,528.29 907.96,528.57 906.70,529.36" />\n  </edge>\n  <edge id=":13339819893_0" function="internal">\n    <lane id=":13339819893_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="8.51" shape="910.99,547.03 911.52,544.64 912.41,543.15 913.48,541.71 914.61,539.45" />\n  </edge>\n  <edge id=":13339819893_1" function="internal">\n    <lane id=":13339819893_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.70" length="8.50" shape="904.53,539.19 906.11,540.95 907.19,542.79 907.75,544.71 907.81,546.70" />\n  </edge>\n  <edge id=":13339819893_2" function="internal">\n    <lane id=":13339819893_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.73" length="12.68" shape="904.53,539.19 907.99,541.85 910.82,542.78 913.03,541.98 914.61,539.45" />\n  </edge>\n  <edge id=":13339844003_0" function="internal">\n    <lane id=":13339844003_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.38" shape="921.46,533.40 918.47,532.96 916.43,532.29 914.57,531.18 912.10,529.39" />\n  </edge>\n  <edge id=":13339844003_1" function="internal">\n    <lane id=":13339844003_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.59" length="7.53" shape="914.68,539.26 915.54,537.88 916.89,536.97 918.75,536.54 921.11,536.58" />\n  </edge>\n  <edge id=":13339844003_2" function="internal">\n    <lane id=":13339844003_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.93" length="11.06" shape="914.68,539.26 915.40,536.28 915.21,533.65 914.11,531.35 912.10,529.39" />\n  </edge>\n  <edge id=":13362936997_0" function="internal">\n    <lane id=":13362936997_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.55" length="9.60" shape="1434.98,355.13 1435.16,352.59 1436.39,351.11 1438.17,349.74 1440.01,347.52" />\n  </edge>\n  <edge id=":13362936997_1" function="internal">\n    <lane id=":13362936997_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.96" length="4.94" shape="1433.27,345.22 1434.40,349.36 1434.82,349.85" />\n  </edge>\n  <edge id=":13362936997_2" function="internal">\n    <lane id=":13362936997_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.96" length="7.39" shape="1434.82,349.85 1435.90,351.13 1437.77,350.52 1440.01,347.52" />\n  </edge>\n  <edge id=":13449811511_0" function="internal">\n    <lane id=":13449811511_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.26" length="9.98" shape="1670.50,771.89 1671.54,769.18 1673.05,767.31 1675.02,766.27 1677.46,766.07" />\n  </edge>\n  <edge id=":13449811511_1" function="internal">\n    <lane id=":13449811511_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.16" length="9.05" shape="1663.00,764.90 1665.48,765.46 1667.05,766.68 1667.69,768.56 1667.40,771.09" />\n  </edge>\n  <edge id=":13449811511_2" function="internal">\n    <lane id=":13449811511_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.50" shape="1663.00,764.90 1677.46,766.07" />\n  </edge>\n  <edge id=":13545019681_0" function="internal">\n    <lane id=":13545019681_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.01" shape="1073.41,1037.81 1070.89,1034.59 1068.23,1030.99 1065.91,1027.23 1064.38,1023.51" />\n  </edge>\n  <edge id=":13545019681_1" function="internal">\n    <lane id=":13545019681_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.50" length="9.60" shape="1064.91,1041.28 1065.77,1038.37 1067.05,1037.15 1068.76,1037.62 1070.89,1039.78" />\n  </edge>\n  <edge id=":13545019681_2" function="internal">\n    <lane id=":13545019681_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.88" shape="1064.91,1041.28 1065.37,1037.11 1065.45,1032.36 1065.13,1027.63 1064.38,1023.51" />\n  </edge>\n  <edge id=":1369020939_0" function="internal">\n    <lane id=":1369020939_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.11" length="9.06" shape="1394.25,1577.86 1393.08,1580.14 1391.51,1581.33 1389.54,1581.43 1387.18,1580.45" />\n  </edge>\n  <edge id=":1369020939_1" function="internal">\n    <lane id=":1369020939_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.12" length="14.49" shape="1394.25,1577.86 1393.33,1581.91 1393.55,1585.43 1394.91,1588.44 1397.39,1590.92" />\n  </edge>\n  <edge id=":1369020939_2" function="internal">\n    <lane id=":1369020939_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.80" length="10.38" shape="1399.27,1588.33 1397.58,1586.60 1396.68,1584.45 1396.58,1581.88 1397.28,1578.90" />\n  </edge>\n  <edge id=":1369020939_3" function="internal">\n    <lane id=":1369020939_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.44" shape="1399.27,1588.33 1396.28,1586.22 1393.32,1584.25 1390.31,1582.35 1387.18,1580.45" />\n  </edge>\n  <edge id=":1369020939_4" function="internal">\n    <lane id=":1369020939_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.14" shape="1385.54,1583.20 1388.64,1585.07 1391.56,1586.92 1394.44,1588.83 1397.39,1590.92" />\n  </edge>\n  <edge id=":1369020939_5" function="internal">\n    <lane id=":1369020939_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.68" length="4.25" shape="1385.54,1583.20 1389.47,1584.83" />\n  </edge>\n  <edge id=":1369020939_6" function="internal">\n    <lane id=":1369020939_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.68" length="10.78" shape="1389.47,1584.83 1392.73,1584.66 1395.33,1582.68 1397.28,1578.90" />\n  </edge>\n  <edge id=":1369020942_0" function="internal">\n    <lane id=":1369020942_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.52" length="8.44" shape="1414.34,1541.28 1412.06,1541.08 1410.33,1540.22 1409.14,1538.70 1408.49,1536.50" />\n  </edge>\n  <edge id=":1369020942_1" function="internal">\n    <lane id=":1369020942_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.00" length="12.85" shape="1414.34,1541.28 1410.86,1541.86 1408.06,1543.21 1405.93,1545.32 1404.47,1548.19" />\n  </edge>\n  <edge id=":1369020942_2" function="internal">\n    <lane id=":1369020942_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.63" length="9.04" shape="1407.49,1549.23 1408.51,1547.26 1410.02,1545.81 1412.02,1544.88 1414.51,1544.47" />\n  </edge>\n  <edge id=":1369020942_3" function="internal">\n    <lane id=":1369020942_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.92" shape="1407.49,1549.23 1408.44,1545.95 1408.88,1542.95 1408.87,1539.91 1408.49,1536.50" />\n  </edge>\n  <edge id=":1369020942_4" function="internal">\n    <lane id=":1369020942_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.41" shape="1405.33,1536.97 1405.73,1540.15 1405.78,1542.66 1405.39,1545.14 1404.47,1548.19" />\n  </edge>\n  <edge id=":1369020942_5" function="internal">\n    <lane id=":1369020942_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.02" length="3.59" shape="1405.33,1536.97 1406.34,1540.42" />\n  </edge>\n  <edge id=":1369020942_6" function="internal">\n    <lane id=":1369020942_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.02" length="9.67" shape="1406.34,1540.42 1408.21,1542.81 1410.93,1544.17 1414.51,1544.47" />\n  </edge>\n  <edge id=":1369020943_0" function="internal">\n    <lane id=":1369020943_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.45" length="9.03" shape="1400.93,1343.98 1398.45,1343.81 1396.64,1342.89 1395.50,1341.20 1395.03,1338.76" />\n  </edge>\n  <edge id=":1369020943_1" function="internal">\n    <lane id=":1369020943_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.01" length="14.27" shape="1400.93,1343.98 1397.09,1344.82 1394.45,1346.68 1393.00,1349.55 1392.76,1353.44" />\n  </edge>\n  <edge id=":1369020943_2" function="internal">\n    <lane id=":1369020943_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.52" length="9.09" shape="1395.95,1353.18 1396.11,1350.71 1397.03,1348.89 1398.72,1347.70 1401.17,1347.17" />\n  </edge>\n  <edge id=":1369020943_3" function="internal">\n    <lane id=":1369020943_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.45" shape="1395.95,1353.18 1395.03,1338.76" />\n  </edge>\n  <edge id=":1369020943_4" function="internal">\n    <lane id=":1369020943_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.55" shape="1391.84,1338.91 1392.76,1353.44" />\n  </edge>\n  <edge id=":1369020943_5" function="internal">\n    <lane id=":1369020943_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.95" length="4.21" shape="1391.84,1338.91 1392.58,1342.78 1392.73,1343.00" />\n  </edge>\n  <edge id=":1369020943_6" function="internal">\n    <lane id=":1369020943_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.95" length="10.09" shape="1392.73,1343.00 1394.38,1345.44 1397.25,1346.90 1401.17,1347.17" />\n  </edge>\n  <edge id=":1369020950_0" function="internal">\n    <lane id=":1369020950_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.31" shape="1393.76,1188.25 1379.51,1187.03" />\n    <lane id=":1369020950_0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.31" shape="1393.50,1191.44 1379.22,1190.22" />\n  </edge>\n  <edge id=":1369020950_2" function="internal">\n    <lane id=":1369020950_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.59" length="3.71" shape="1393.50,1191.44 1389.81,1191.84" />\n  </edge>\n  <edge id=":1369020950_8" function="internal">\n    <lane id=":1369020950_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.59" length="13.41" shape="1389.81,1191.84 1389.47,1191.88 1386.53,1193.94 1384.69,1197.63 1383.94,1202.94" />\n  </edge>\n  <edge id=":1369020950_3" function="internal">\n    <lane id=":1369020950_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.40" length="9.04" shape="1387.14,1203.04 1387.58,1200.58 1388.69,1198.90 1390.50,1197.97 1392.98,1197.82" />\n  </edge>\n  <edge id=":1369020950_4" function="internal">\n    <lane id=":1369020950_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.85" length="16.82" shape="1387.14,1203.04 1386.79,1197.71 1385.36,1193.80 1382.83,1191.30 1379.22,1190.22" />\n  </edge>\n  <edge id=":1369020950_5" function="internal">\n    <lane id=":1369020950_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.73" length="9.31" shape="1378.64,1196.59 1381.03,1197.18 1382.71,1198.43 1383.68,1200.35 1383.94,1202.94" />\n  </edge>\n  <edge id=":1369020950_6" function="internal">\n    <lane id=":1369020950_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.38" shape="1378.64,1196.59 1392.98,1197.82" />\n    <lane id=":1369020950_6_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.38" shape="1378.93,1193.41 1393.24,1194.63" />\n  </edge>\n  <edge id=":14069947700_0" function="internal">\n    <lane id=":14069947700_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.28" length="9.99" shape="571.98,669.40 572.90,666.64 574.32,664.69 576.24,663.55 578.65,663.22" />\n  </edge>\n  <edge id=":14069947700_1" function="internal">\n    <lane id=":14069947700_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.15" length="9.06" shape="564.16,662.78 566.67,663.22 568.29,664.36 569.02,666.20 568.85,668.74" />\n  </edge>\n  <edge id=":14069947700_2" function="internal">\n    <lane id=":14069947700_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.50" shape="564.16,662.78 578.65,663.22" />\n    <lane id=":14069947700_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.50" shape="564.25,659.58 578.75,660.03" />\n  </edge>\n  <edge id=":14069981565_0" function="internal">\n    <lane id=":14069981565_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.56" length="9.44" shape="460.06,734.04 458.64,743.37" />\n  </edge>\n  <edge id=":14069981565_1" function="internal">\n    <lane id=":14069981565_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.98" length="8.06" shape="460.06,734.04 459.85,736.86 460.23,738.90 461.17,740.15 462.69,740.63" />\n  </edge>\n  <edge id=":14069981565_2" function="internal">\n    <lane id=":14069981565_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.38" length="7.95" shape="462.84,737.43 461.35,737.75 460.15,738.84 459.25,740.71 458.64,743.37" />\n  </edge>\n  <edge id=":14070071629_0" function="internal">\n    <lane id=":14070071629_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.08" length="7.65" shape="597.13,285.40 595.07,285.03 593.38,285.41 592.05,286.53 591.10,288.39" />\n  </edge>\n  <edge id=":14070071629_1" function="internal">\n    <lane id=":14070071629_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="3.91" length="2.55" shape="594.12,289.44 594.44,288.82 594.88,288.44 595.45,288.32 596.13,288.44" />\n  </edge>\n  <edge id=":14159239425_0" function="internal">\n    <lane id=":14159239425_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.92" length="9.10" shape="1137.58,722.31 1137.37,719.70 1138.11,717.92 1139.78,716.97 1142.40,716.86" />\n  </edge>\n  <edge id=":14159239425_1" function="internal">\n    <lane id=":14159239425_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.06" length="10.89" shape="1127.73,715.40 1130.08,716.07 1131.98,717.56 1133.44,719.88 1134.46,723.02" />\n  </edge>\n  <edge id=":14159239425_2" function="internal">\n    <lane id=":14159239425_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.74" shape="1127.73,715.40 1142.40,716.86" />\n  </edge>\n  <edge id=":14159239427_0" function="internal">\n    <lane id=":14159239427_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.74" shape="1082.25,723.51 1079.17,725.59 1076.15,727.66 1073.19,729.81 1070.31,732.14" />\n  </edge>\n  <edge id=":14159239427_1" function="internal">\n    <lane id=":14159239427_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.67" length="3.78" shape="1082.25,723.51 1079.58,726.02 1079.53,726.12" />\n  </edge>\n  <edge id=":14159239427_6" function="internal">\n    <lane id=":14159239427_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.67" length="10.87" shape="1079.53,726.12 1078.22,729.02 1078.19,732.52 1079.46,736.51" />\n  </edge>\n  <edge id=":14159239427_2" function="internal">\n    <lane id=":14159239427_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.29" length="10.11" shape="1082.38,735.19 1081.45,732.36 1081.42,729.92 1082.28,727.85 1084.05,726.15" />\n  </edge>\n  <edge id=":14159239427_3" function="internal">\n    <lane id=":14159239427_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.58" length="15.34" shape="1082.38,735.19 1079.97,731.38 1077.16,729.61 1073.94,729.86 1070.31,732.14" />\n  </edge>\n  <edge id=":14159239427_4" function="internal">\n    <lane id=":14159239427_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.01" length="9.19" shape="1072.23,734.71 1074.41,733.34 1076.34,733.18 1078.03,734.24 1079.46,736.51" />\n  </edge>\n  <edge id=":14159239427_5" function="internal">\n    <lane id=":14159239427_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.60" shape="1072.23,734.71 1075.10,732.39 1078.01,730.26 1080.99,728.22 1084.05,726.15" />\n  </edge>\n  <edge id=":14165547215_0" function="internal">\n    <lane id=":14165547215_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.51" shape="1387.99,1281.26 1385.51,1266.96" />\n  </edge>\n  <edge id=":14165547215_1" function="internal">\n    <lane id=":14165547215_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.70" length="3.80" shape="1387.99,1281.26 1386.82,1277.81 1386.71,1277.69" />\n  </edge>\n  <edge id=":14165547215_6" function="internal">\n    <lane id=":14165547215_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.70" length="10.74" shape="1386.71,1277.69 1384.67,1275.33 1381.54,1273.82 1377.44,1273.27" />\n  </edge>\n  <edge id=":14165547215_2" function="internal">\n    <lane id=":14165547215_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.33" length="10.05" shape="1377.39,1276.47 1380.30,1276.83 1382.52,1277.85 1384.03,1279.51 1384.84,1281.81" />\n  </edge>\n  <edge id=":14165547215_3" function="internal">\n    <lane id=":14165547215_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.71" length="14.95" shape="1377.39,1276.47 1381.57,1275.92 1384.31,1274.16 1385.63,1271.17 1385.51,1266.96" />\n  </edge>\n  <edge id=":14165547215_4" function="internal">\n    <lane id=":14165547215_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.14" length="9.06" shape="1382.35,1267.51 1382.43,1270.06 1381.63,1271.87 1379.97,1272.94 1377.44,1273.27" />\n  </edge>\n  <edge id=":14165547215_5" function="internal">\n    <lane id=":14165547215_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.51" shape="1382.35,1267.51 1384.84,1281.81" />\n  </edge>\n  <edge id=":14179759489_0" function="internal">\n    <lane id=":14179759489_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.29" length="12.73" shape="406.58,1432.83 402.73,1430.44 400.15,1428.14 398.84,1425.91 398.80,1423.75" />\n  </edge>\n  <edge id=":14179759489_1" function="internal">\n    <lane id=":14179759489_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.39" length="17.14" shape="406.58,1432.83 401.73,1430.97 398.39,1431.46 396.58,1434.30 396.29,1439.49" />\n  </edge>\n  <edge id=":14179759489_2" function="internal">\n    <lane id=":14179759489_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.62" length="9.29" shape="399.48,1439.26 399.64,1436.44 400.62,1434.90 402.43,1434.64 405.06,1435.65" />\n  </edge>\n  <edge id=":14179759489_3" function="internal">\n    <lane id=":14179759489_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.61" shape="399.48,1439.26 399.02,1435.41 398.49,1431.44 398.28,1427.51 398.80,1423.75" />\n  </edge>\n  <edge id=":14179759489_4" function="internal">\n    <lane id=":14179759489_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.64" shape="395.41,1423.97 394.92,1427.73 395.20,1431.67 395.80,1435.64 396.29,1439.49" />\n  </edge>\n  <edge id=":14179759489_5" function="internal">\n    <lane id=":14179759489_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.28" length="3.38" shape="395.41,1423.97 395.42,1426.80 395.68,1427.28" />\n  </edge>\n  <edge id=":14179759489_6" function="internal">\n    <lane id=":14179759489_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.28" length="12.80" shape="395.68,1427.28 397.03,1429.69 400.24,1432.64 405.06,1435.65" />\n  </edge>\n  <edge id=":14182422401_0" function="internal">\n    <lane id=":14182422401_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.49" length="9.03" shape="729.38,407.46 728.73,409.85 727.47,411.45 725.60,412.26 723.12,412.28" />\n  </edge>\n  <edge id=":14182422401_1" function="internal">\n    <lane id=":14182422401_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.03" length="14.20" shape="729.38,407.46 729.45,411.36 730.73,414.31 733.24,416.31 736.97,417.37" />\n  </edge>\n  <edge id=":14182422401_2" function="internal">\n    <lane id=":14182422401_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.54" length="9.07" shape="737.39,414.20 735.01,413.52 733.42,412.24 732.60,410.35 732.56,407.86" />\n  </edge>\n  <edge id=":14182422401_3" function="internal">\n    <lane id=":14182422401_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="737.39,414.20 723.12,412.28" />\n  </edge>\n  <edge id=":14182422401_4" function="internal">\n    <lane id=":14182422401_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="722.69,415.46 736.97,417.37" />\n  </edge>\n  <edge id=":14182422401_5" function="internal">\n    <lane id=":14182422401_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="4.07" shape="722.69,415.46 726.60,415.42 726.75,415.35" />\n  </edge>\n  <edge id=":14182422401_6" function="internal">\n    <lane id=":14182422401_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="10.15" shape="726.75,415.35 729.54,414.14 731.53,411.62 732.56,407.86" />\n  </edge>\n  <edge id=":14188374629_0" function="internal">\n    <lane id=":14188374629_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.41" length="9.04" shape="1221.26,1549.86 1218.77,1549.75 1216.95,1548.86 1215.80,1547.19 1215.31,1544.75" />\n  </edge>\n  <edge id=":14188374629_1" function="internal">\n    <lane id=":14188374629_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.41" shape="1221.26,1549.86 1206.92,1551.26" />\n  </edge>\n  <edge id=":14188374629_2" function="internal">\n    <lane id=":14188374629_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.41" shape="1207.23,1554.44 1221.57,1553.05" />\n  </edge>\n  <edge id=":14188374629_3" function="internal">\n    <lane id=":14188374629_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.15" length="4.00" shape="1207.23,1554.44 1210.94,1553.53 1211.08,1553.42" />\n  </edge>\n  <edge id=":14188374629_6" function="internal">\n    <lane id=":14188374629_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.15" length="10.26" shape="1211.08,1553.42 1213.53,1551.61 1214.98,1548.68 1215.31,1544.75" />\n  </edge>\n  <edge id=":14188374629_4" function="internal">\n    <lane id=":14188374629_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.68" length="9.26" shape="1212.12,1544.92 1211.91,1547.49 1210.97,1549.41 1209.31,1550.66 1206.92,1551.26" />\n  </edge>\n  <edge id=":14188374629_5" function="internal">\n    <lane id=":14188374629_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.92" length="14.37" shape="1212.12,1544.92 1212.89,1548.80 1214.72,1551.45 1217.61,1552.87 1221.57,1553.05" />\n  </edge>\n  <edge id=":14188764645_0" function="internal">\n    <lane id=":14188764645_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="807.82,570.82 806.44,585.15" />\n  </edge>\n  <edge id=":14188764645_1" function="internal">\n    <lane id=":14188764645_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="4.07" shape="807.82,570.82 808.00,574.72 808.07,574.87" />\n  </edge>\n  <edge id=":14188764645_6" function="internal">\n    <lane id=":14188764645_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="10.15" shape="808.07,574.87 809.39,577.62 811.98,579.51 815.78,580.40" />\n  </edge>\n  <edge id=":14188764645_2" function="internal">\n    <lane id=":14188764645_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.49" length="9.03" shape="816.06,577.21 813.65,576.64 812.00,575.44 811.12,573.60 811.01,571.13" />\n  </edge>\n  <edge id=":14188764645_3" function="internal">\n    <lane id=":14188764645_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.03" length="14.20" shape="816.06,577.21 812.17,577.42 809.26,578.81 807.35,581.39 806.44,585.15" />\n  </edge>\n  <edge id=":14188764645_4" function="internal">\n    <lane id=":14188764645_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.54" length="9.08" shape="809.62,585.46 810.21,583.06 811.43,581.42 813.29,580.53 815.78,580.40" />\n  </edge>\n  <edge id=":14188764645_5" function="internal">\n    <lane id=":14188764645_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="809.62,585.46 811.01,571.13" />\n  </edge>\n  <edge id=":14188764646_0" function="internal">\n    <lane id=":14188764646_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.62" length="9.13" shape="853.23,574.79 852.58,577.22 851.33,578.88 849.47,579.79 847.01,579.93" />\n  </edge>\n  <edge id=":14188764646_1" function="internal">\n    <lane id=":14188764646_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="14.20" shape="853.23,574.79 853.30,578.68 854.60,581.62 857.13,583.59 860.89,584.60" />\n  </edge>\n  <edge id=":14188764646_2" function="internal">\n    <lane id=":14188764646_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.50" length="9.03" shape="861.28,581.43 858.89,580.78 857.28,579.53 856.45,577.66 856.41,575.19" />\n  </edge>\n  <edge id=":14188764646_3" function="internal">\n    <lane id=":14188764646_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.35" shape="861.28,581.43 847.01,579.93" />\n  </edge>\n  <edge id=":14188764646_4" function="internal">\n    <lane id=":14188764646_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.24" shape="846.73,583.11 860.89,584.60" />\n  </edge>\n  <edge id=":14188764646_5" function="internal">\n    <lane id=":14188764646_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.10" length="3.83" shape="846.73,583.11 850.56,582.90" />\n  </edge>\n  <edge id=":14188764646_6" function="internal">\n    <lane id=":14188764646_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.10" length="10.32" shape="850.56,582.90 853.45,581.51 855.40,578.94 856.41,575.19" />\n  </edge>\n  <edge id=":14188764647_0" function="internal">\n    <lane id=":14188764647_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="867.85,527.69 853.52,526.21" />\n  </edge>\n  <edge id=":14188764647_1" function="internal">\n    <lane id=":14188764647_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="4.03" shape="867.85,527.69 864.00,527.85 863.84,527.92" />\n  </edge>\n  <edge id=":14188764647_6" function="internal">\n    <lane id=":14188764647_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="10.19" shape="863.84,527.92 861.09,529.21 859.12,531.79 858.11,535.57" />\n  </edge>\n  <edge id=":14188764647_2" function="internal">\n    <lane id=":14188764647_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.59" length="9.14" shape="861.28,535.96 861.94,533.52 863.19,531.86 865.06,530.98 867.52,530.87" />\n  </edge>\n  <edge id=":14188764647_3" function="internal">\n    <lane id=":14188764647_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.96" length="14.27" shape="861.28,535.96 861.21,532.04 859.89,529.11 857.33,527.17 853.52,526.21" />\n  </edge>\n  <edge id=":14188764647_4" function="internal">\n    <lane id=":14188764647_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.46" length="9.03" shape="853.20,529.40 855.60,530.00 857.23,531.23 858.06,533.08 858.11,535.57" />\n  </edge>\n  <edge id=":14188764647_5" function="internal">\n    <lane id=":14188764647_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="853.20,529.40 867.52,530.87" />\n  </edge>\n  <edge id=":1431715363_0" function="internal">\n    <lane id=":1431715363_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="8.04" shape="806.30,685.95 814.30,686.68" />\n  </edge>\n  <edge id=":1431715363_1" function="internal">\n    <lane id=":1431715363_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.03" length="2.65" shape="808.82,681.97 810.15,683.93 810.34,684.13" />\n  </edge>\n  <edge id=":1431715363_2" function="internal">\n    <lane id=":1431715363_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.03" length="4.83" shape="810.34,684.13 811.51,685.37 812.89,686.29 814.30,686.68" />\n  </edge>\n  <edge id=":1431715366_0" function="internal">\n    <lane id=":1431715366_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.65" length="8.58" shape="769.18,675.59 769.44,677.98 770.32,679.78 771.82,681.01 773.95,681.66" />\n  </edge>\n  <edge id=":1431715366_1" function="internal">\n    <lane id=":1431715366_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.28" shape="763.78,680.13 773.95,681.66" />\n  </edge>\n  <edge id=":1431715369_0" function="internal">\n    <lane id=":1431715369_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.19" shape="418.44,501.67 420.58,490.69" />\n    <lane id=":1431715369_0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.19" shape="415.30,501.06 417.44,490.07" />\n    <lane id=":1431715369_0_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.19" shape="412.16,500.45 414.30,489.45" />\n  </edge>\n  <edge id=":1431715369_3" function="internal">\n    <lane id=":1431715369_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.35" length="9.04" shape="412.16,500.45 412.28,497.95 411.55,496.07 409.98,494.81 407.56,494.16" />\n  </edge>\n  <edge id=":1431715372_0" function="internal">\n    <lane id=":1431715372_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.84" shape="789.16,325.88 785.73,340.32" />\n  </edge>\n  <edge id=":1431715372_1" function="internal">\n    <lane id=":1431715372_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.72" length="4.53" shape="789.16,325.88 788.62,330.05 788.73,330.37" />\n  </edge>\n  <edge id=":1431715372_6" function="internal">\n    <lane id=":1431715372_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.72" length="10.38" shape="788.73,330.37 789.63,333.15 792.17,335.19 796.25,336.17" />\n  </edge>\n  <edge id=":1431715372_2" function="internal">\n    <lane id=":1431715372_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.16" length="9.05" shape="796.55,332.99 794.07,332.39 792.53,331.15 791.92,329.27 792.24,326.74" />\n  </edge>\n  <edge id=":1431715372_3" function="internal">\n    <lane id=":1431715372_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.41" length="14.60" shape="796.55,332.99 792.41,333.12 789.23,334.39 787.00,336.79 785.73,340.32" />\n  </edge>\n  <edge id=":1431715372_4" function="internal">\n    <lane id=":1431715372_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.99" length="9.79" shape="788.87,340.93 789.72,338.63 791.23,337.07 793.41,336.25 796.25,336.17" />\n  </edge>\n  <edge id=":1431715372_5" function="internal">\n    <lane id=":1431715372_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.59" shape="788.87,340.93 792.24,326.74" />\n  </edge>\n  <edge id=":1431715392_0" function="internal">\n    <lane id=":1431715392_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.15" length="9.64" shape="775.91,602.85 773.34,602.28 771.44,601.08 770.20,599.24 769.62,596.76" />\n  </edge>\n  <edge id=":1431715392_1" function="internal">\n    <lane id=":1431715392_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.42" shape="775.91,602.85 764.53,601.84" />\n  </edge>\n  <edge id=":1431715392_2" function="internal">\n    <lane id=":1431715392_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.06" length="2.49" shape="775.91,602.85 773.44,603.18" />\n  </edge>\n  <edge id=":1431715392_6" function="internal">\n    <lane id=":1431715392_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.06" length="8.89" shape="773.44,603.18 771.67,604.65 770.58,607.25 770.17,610.99" />\n  </edge>\n  <edge id=":1431715392_3" function="internal">\n    <lane id=":1431715392_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.83" length="9.44" shape="764.23,605.03 766.86,605.61 768.73,606.80 769.83,608.59 770.17,610.99" />\n  </edge>\n  <edge id=":1431715392_4" function="internal">\n    <lane id=":1431715392_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.45" shape="764.23,605.03 775.64,606.03" />\n  </edge>\n  <edge id=":1431715392_5" function="internal">\n    <lane id=":1431715392_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.87" length="2.67" shape="764.23,605.03 766.89,604.72" />\n  </edge>\n  <edge id=":1431715392_7" function="internal">\n    <lane id=":1431715392_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.87" length="8.95" shape="766.89,604.72 768.67,603.25 769.58,600.59 769.62,596.76" />\n  </edge>\n  <edge id=":1431715395_0" function="internal">\n    <lane id=":1431715395_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.50" shape="515.74,364.57 511.20,378.34" />\n  </edge>\n  <edge id=":1431715395_1" function="internal">\n    <lane id=":1431715395_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.96" length="4.17" shape="515.74,364.57 515.01,368.43 515.06,368.66" />\n  </edge>\n  <edge id=":1431715395_6" function="internal">\n    <lane id=":1431715395_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.96" length="10.12" shape="515.06,368.66 515.71,371.57 517.83,373.99 521.38,375.68" />\n  </edge>\n  <edge id=":1431715395_2" function="internal">\n    <lane id=":1431715395_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.45" length="9.03" shape="522.33,372.63 520.09,371.56 518.75,370.03 518.31,368.04 518.77,365.60" />\n  </edge>\n  <edge id=":1431715395_3" function="internal">\n    <lane id=":1431715395_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.03" length="14.25" shape="522.33,372.63 518.46,371.99 515.31,372.73 512.89,374.85 511.20,378.34" />\n  </edge>\n  <edge id=":1431715395_4" function="internal">\n    <lane id=":1431715395_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.54" length="9.11" shape="514.25,379.31 515.33,377.09 516.88,375.74 518.89,375.27 521.38,375.68" />\n  </edge>\n  <edge id=":1431715395_5" function="internal">\n    <lane id=":1431715395_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.43" shape="514.25,379.31 518.77,365.60" />\n  </edge>\n  <edge id=":1431715406_0" function="internal">\n    <lane id=":1431715406_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="8.96" shape="805.02,599.72 804.45,602.10 803.24,603.74 801.42,604.63 798.98,604.77" />\n  </edge>\n  <edge id=":1431715406_1" function="internal">\n    <lane id=":1431715406_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.47" shape="805.02,599.72 804.93,603.38 805.19,606.94 805.47,610.51 805.47,614.17" />\n  </edge>\n  <edge id=":1431715406_2" function="internal">\n    <lane id=":1431715406_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="8.96" shape="813.18,606.14 810.80,605.53 809.18,604.31 808.32,602.48 808.21,600.03" />\n  </edge>\n  <edge id=":1431715406_3" function="internal">\n    <lane id=":1431715406_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.27" shape="813.18,606.14 798.98,604.77" />\n  </edge>\n  <edge id=":1431715406_4" function="internal">\n    <lane id=":1431715406_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.75" length="6.74" shape="812.83,609.32 809.75,609.34 807.49,610.15 806.66,611.09" />\n  </edge>\n  <edge id=":1431715406_7" function="internal">\n    <lane id=":1431715406_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.75" length="3.38" shape="806.66,611.09 806.06,611.76 805.47,614.17" />\n  </edge>\n  <edge id=":1431715406_5" function="internal">\n    <lane id=":1431715406_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.98" length="10.38" shape="798.71,607.96 801.80,608.57 803.96,609.81 805.18,611.68 805.47,614.17" />\n  </edge>\n  <edge id=":1431715406_6" function="internal">\n    <lane id=":1431715406_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.01" length="3.95" shape="798.71,607.96 802.55,607.74 802.64,607.69" />\n  </edge>\n  <edge id=":1431715406_8" function="internal">\n    <lane id=":1431715406_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.01" length="10.12" shape="802.64,607.69 805.41,606.34 807.30,603.77 808.21,600.03" />\n  </edge>\n  <edge id=":1431715409_0" function="internal">\n    <lane id=":1431715409_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.45" shape="738.46,513.08 729.03,512.44" />\n  </edge>\n  <edge id=":1431715409_1" function="internal">\n    <lane id=":1431715409_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.64" length="2.53" shape="738.46,513.08 735.96,513.50" />\n  </edge>\n  <edge id=":1431715409_4" function="internal">\n    <lane id=":1431715409_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.64" length="8.37" shape="735.96,513.50 734.36,514.96 733.68,517.49 733.90,521.07" />\n  </edge>\n  <edge id=":1431715409_2" function="internal">\n    <lane id=":1431715409_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.84" length="8.15" shape="728.75,515.63 730.69,516.11 732.20,517.18 733.27,518.83 733.90,521.07" />\n  </edge>\n  <edge id=":1431715409_3" function="internal">\n    <lane id=":1431715409_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.48" shape="728.75,515.63 738.20,516.28" />\n  </edge>\n  <edge id=":1431715415_0" function="internal">\n    <lane id=":1431715415_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.20" shape="821.32,522.90 807.23,521.15" />\n  </edge>\n  <edge id=":1431715415_1" function="internal">\n    <lane id=":1431715415_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="3.90" shape="821.32,522.90 817.42,523.05" />\n  </edge>\n  <edge id=":1431715415_6" function="internal">\n    <lane id=":1431715415_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="10.32" shape="817.42,523.05 814.52,524.42 812.60,526.99 811.68,530.78" />\n  </edge>\n  <edge id=":1431715415_2" function="internal">\n    <lane id=":1431715415_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.49" length="9.03" shape="814.87,531.09 815.45,528.68 816.67,527.05 818.51,526.18 820.99,526.08" />\n  </edge>\n  <edge id=":1431715415_3" function="internal">\n    <lane id=":1431715415_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.13" length="14.15" shape="814.87,531.09 814.71,527.21 813.38,524.25 810.89,522.23 807.23,521.15" />\n  </edge>\n  <edge id=":1431715415_4" function="internal">\n    <lane id=":1431715415_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.66" length="9.17" shape="806.76,524.32 809.12,525.02 810.73,526.33 811.58,528.25 811.68,530.78" />\n  </edge>\n  <edge id=":1431715415_5" function="internal">\n    <lane id=":1431715415_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.33" shape="806.76,524.32 820.99,526.08" />\n  </edge>\n  <edge id=":1431715421_0" function="internal">\n    <lane id=":1431715421_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.52" length="9.05" shape="500.72,286.60 500.07,289.00 498.81,290.61 496.94,291.45 494.47,291.50" />\n  </edge>\n  <edge id=":1431715421_1" function="internal">\n    <lane id=":1431715421_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.50" length="9.03" shape="508.76,293.24 506.37,292.60 504.76,291.34 503.94,289.48 503.90,287.00" />\n  </edge>\n  <edge id=":1431715421_2" function="internal">\n    <lane id=":1431715421_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="508.76,293.24 494.47,291.50" />\n  </edge>\n  <edge id=":1431867100_0" function="internal">\n    <lane id=":1431867100_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.07" length="8.21" shape="1633.90,282.32 1631.80,281.43 1629.98,281.46 1628.41,282.41 1627.12,284.28" />\n  </edge>\n  <edge id=":1431867100_1" function="internal">\n    <lane id=":1431867100_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="3.90" length="2.74" shape="1629.98,285.71 1630.41,285.09 1630.93,284.77 1631.54,284.76 1632.24,285.06" />\n  </edge>\n  <edge id=":1431867103_0" function="internal">\n    <lane id=":1431867103_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.23" length="9.05" shape="1717.93,443.08 1716.88,445.38 1715.37,446.69 1713.40,446.99 1710.97,446.30" />\n  </edge>\n  <edge id=":1431867103_1" function="internal">\n    <lane id=":1431867103_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.65" length="14.38" shape="1717.93,443.08 1717.27,447.08 1717.82,450.46 1719.58,453.21 1722.53,455.32" />\n  </edge>\n  <edge id=":1431867103_2" function="internal">\n    <lane id=":1431867103_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.27" length="9.88" shape="1723.98,452.47 1722.01,451.04 1720.86,449.14 1720.52,446.79 1721.00,443.97" />\n  </edge>\n  <edge id=":1431867103_3" function="internal">\n    <lane id=":1431867103_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="1723.98,452.47 1710.97,446.30" />\n  </edge>\n  <edge id=":1431867103_4" function="internal">\n    <lane id=":1431867103_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.24" shape="1709.66,449.23 1722.53,455.32" />\n  </edge>\n  <edge id=":1431867103_5" function="internal">\n    <lane id=":1431867103_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.78" length="4.12" shape="1709.66,449.23 1713.63,450.35" />\n  </edge>\n  <edge id=":1431867103_6" function="internal">\n    <lane id=":1431867103_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.78" length="10.62" shape="1713.63,450.35 1716.84,449.85 1719.30,447.72 1721.00,443.97" />\n  </edge>\n  <edge id=":1431867108_0" function="internal">\n    <lane id=":1431867108_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.78" length="9.42" shape="1442.93,359.75 1439.61,358.78 1437.18,357.73 1435.65,356.58 1435.01,355.33" />\n  </edge>\n  <edge id=":1431867108_1" function="internal">\n    <lane id=":1431867108_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.29" shape="1442.93,359.75 1439.52,358.98 1436.39,358.50 1433.24,358.38 1429.78,358.68" />\n  </edge>\n  <edge id=":1431867108_2" function="internal">\n    <lane id=":1431867108_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.81" length="9.14" shape="1437.00,367.68 1436.92,365.00 1437.76,363.30 1439.51,362.59 1442.18,362.85" />\n  </edge>\n  <edge id=":1431867108_3" function="internal">\n    <lane id=":1431867108_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.50" shape="1437.00,367.68 1435.01,355.33" />\n  </edge>\n  <edge id=":1431867108_4" function="internal">\n    <lane id=":1431867108_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.70" length="12.82" shape="1437.00,367.68 1435.93,363.44 1434.37,360.53 1432.32,358.94 1429.78,358.68" />\n  </edge>\n  <edge id=":1431867108_5" function="internal">\n    <lane id=":1431867108_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.19" shape="1430.13,361.86 1433.41,361.55 1436.19,361.65 1438.94,362.10 1442.18,362.85" />\n  </edge>\n  <edge id=":1431867108_6" function="internal">\n    <lane id=":1431867108_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.59" length="2.82" shape="1430.13,361.86 1432.58,361.15 1432.78,360.98" />\n  </edge>\n  <edge id=":1431867108_7" function="internal">\n    <lane id=":1431867108_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.59" length="6.49" shape="1432.78,360.98 1434.21,359.82 1435.02,357.88 1435.01,355.33" />\n  </edge>\n  <edge id=":1431867150_0" function="internal">\n    <lane id=":1431867150_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.37" shape="1674.25,304.28 1671.12,303.73 1668.72,303.04 1666.44,302.00 1663.68,300.41" />\n  </edge>\n  <edge id=":1431867150_1" function="internal">\n    <lane id=":1431867150_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.34" length="3.47" shape="1674.25,304.28 1670.97,304.25 1670.80,304.31" />\n  </edge>\n  <edge id=":1431867150_6" function="internal">\n    <lane id=":1431867150_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.34" length="9.09" shape="1670.80,304.31 1668.21,305.21 1665.98,307.15 1664.28,310.09" />\n  </edge>\n  <edge id=":1431867150_2" function="internal">\n    <lane id=":1431867150_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.88" length="8.28" shape="1667.24,311.32 1668.36,309.37 1669.83,308.07 1671.63,307.43 1673.78,307.45" />\n  </edge>\n  <edge id=":1431867150_3" function="internal">\n    <lane id=":1431867150_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.04" length="12.70" shape="1667.24,311.32 1668.11,308.02 1667.81,305.10 1666.33,302.56 1663.68,300.41" />\n  </edge>\n  <edge id=":1431867150_4" function="internal">\n    <lane id=":1431867150_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.54" length="8.08" shape="1662.02,303.14 1663.71,304.51 1664.65,306.13 1664.84,307.99 1664.28,310.09" />\n  </edge>\n  <edge id=":1431867150_5" function="internal">\n    <lane id=":1431867150_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.63" shape="1662.02,303.14 1664.97,304.81 1667.65,305.99 1670.46,306.83 1673.78,307.45" />\n  </edge>\n  <edge id=":1431867160_0" function="internal">\n    <lane id=":1431867160_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.56" length="9.04" shape="1425.62,359.27 1423.13,359.28 1421.23,358.51 1419.91,356.95 1419.17,354.61" />\n  </edge>\n  <edge id=":1431867160_1" function="internal">\n    <lane id=":1431867160_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.47" length="9.04" shape="1421.49,368.82 1421.44,366.32 1422.19,364.43 1423.76,363.13 1426.13,362.44" />\n  </edge>\n  <edge id=":1431867160_2" function="internal">\n    <lane id=":1431867160_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="1421.49,368.82 1419.17,354.61" />\n    <lane id=":1431867160_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="1418.33,369.33 1416.01,355.12" />\n  </edge>\n  <edge id=":1431867166_0" function="internal">\n    <lane id=":1431867166_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.66" length="10.74" shape="1556.92,381.55 1555.11,383.13 1552.85,384.05 1550.15,384.31 1547.00,383.89" />\n  </edge>\n  <edge id=":1431867166_1" function="internal">\n    <lane id=":1431867166_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.21" shape="1556.92,381.55 1554.74,384.44 1552.88,387.38 1551.25,390.45 1549.73,393.75" />\n  </edge>\n  <edge id=":1431867166_2" function="internal">\n    <lane id=":1431867166_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.33" shape="1552.66,395.03 1554.11,391.85 1555.58,389.04 1557.28,386.36 1559.39,383.59" />\n  </edge>\n  <edge id=":1431867166_3" function="internal">\n    <lane id=":1431867166_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.70" length="14.99" shape="1552.66,395.03 1553.79,390.96 1553.22,387.74 1550.95,385.39 1547.00,383.89" />\n  </edge>\n  <edge id=":1431867166_4" function="internal">\n    <lane id=":1431867166_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.13" length="9.06" shape="1546.31,387.02 1548.70,387.92 1550.07,389.34 1550.41,391.29 1549.73,393.75" />\n  </edge>\n  <edge id=":1431867166_5" function="internal">\n    <lane id=":1431867166_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.91" length="14.26" shape="1546.31,387.02 1550.37,387.52 1553.91,387.12 1556.91,385.81 1559.39,383.59" />\n  </edge>\n  <edge id=":1558314095_0" function="internal">\n    <lane id=":1558314095_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.72" length="9.84" shape="1698.04,302.37 1697.31,304.65 1695.83,306.14 1693.59,306.83 1690.59,306.72" />\n  </edge>\n  <edge id=":1558314095_1" function="internal">\n    <lane id=":1558314095_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.48" shape="1698.04,302.37 1697.37,305.96 1696.43,309.41 1695.37,312.82 1694.31,316.34" />\n  </edge>\n  <edge id=":1558314095_2" function="internal">\n    <lane id=":1558314095_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.53" shape="1697.39,317.20 1698.46,313.67 1699.54,310.24 1700.49,306.78 1701.16,303.18" />\n  </edge>\n  <edge id=":1558314095_3" function="internal">\n    <lane id=":1558314095_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.80" length="4.09" shape="1697.39,317.20 1697.91,313.14" />\n  </edge>\n  <edge id=":1558314095_6" function="internal">\n    <lane id=":1558314095_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.80" length="10.58" shape="1697.91,313.14 1696.95,310.04 1694.51,307.90 1690.59,306.72" />\n  </edge>\n  <edge id=":1558314095_4" function="internal">\n    <lane id=":1558314095_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.26" length="9.05" shape="1690.12,309.89 1692.54,310.61 1694.04,311.93 1694.63,313.84 1694.31,316.34" />\n  </edge>\n  <edge id=":1558314095_5" function="internal">\n    <lane id=":1558314095_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.09" length="14.75" shape="1690.12,309.89 1694.55,310.03 1697.86,308.95 1700.07,306.67 1701.16,303.18" />\n  </edge>\n  <edge id=":1558997825_0" function="internal">\n    <lane id=":1558997825_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.83" length="12.65" shape="1057.33,696.08 1054.75,699.19 1052.20,701.36 1049.70,702.58 1047.24,702.87" />\n  </edge>\n  <edge id=":1558997825_1" function="internal">\n    <lane id=":1558997825_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.64" length="9.27" shape="1062.72,704.05 1059.97,703.47 1058.59,702.27 1058.57,700.43 1059.92,697.97" />\n  </edge>\n  <edge id=":1558997825_2" function="internal">\n    <lane id=":1558997825_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.53" shape="1062.72,704.05 1047.24,702.87" />\n  </edge>\n  <edge id=":1559016903_0" function="internal">\n    <lane id=":1559016903_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.53" length="9.07" shape="878.70,335.07 878.10,337.48 876.88,339.12 875.02,339.99 872.52,340.11" />\n  </edge>\n  <edge id=":1559016903_1" function="internal">\n    <lane id=":1559016903_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="878.70,335.07 877.23,349.42" />\n  </edge>\n  <edge id=":1559016903_2" function="internal">\n    <lane id=":1559016903_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="4.04" shape="878.70,335.07 878.87,338.95 878.93,339.09" />\n  </edge>\n  <edge id=":1559016903_12" function="internal">\n    <lane id=":1559016903_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="10.20" shape="878.93,339.09 880.24,341.87 882.81,343.82 886.58,344.82" />\n  </edge>\n  <edge id=":1559016903_3" function="internal">\n    <lane id=":1559016903_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.59" length="9.15" shape="886.96,341.64 884.53,341.00 882.87,339.75 881.99,337.88 881.88,335.39" />\n  </edge>\n  <edge id=":1559016903_4" function="internal">\n    <lane id=":1559016903_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.52" shape="886.96,341.64 872.52,340.11" />\n  </edge>\n  <edge id=":1559016903_5" function="internal">\n    <lane id=":1559016903_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.98" length="14.24" shape="886.96,341.64 883.05,341.74 880.13,343.06 878.19,345.62 877.23,349.42" />\n  </edge>\n  <edge id=":1559016903_6" function="internal">\n    <lane id=":1559016903_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.48" length="9.03" shape="880.41,349.75 881.02,347.35 882.25,345.72 884.11,344.88 886.58,344.82" />\n  </edge>\n  <edge id=":1559016903_7" function="internal">\n    <lane id=":1559016903_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.43" shape="880.41,349.75 881.88,335.39" />\n  </edge>\n  <edge id=":1559016903_8" function="internal">\n    <lane id=":1559016903_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.98" length="4.10" shape="880.41,349.75 880.27,345.84 880.19,345.67" />\n  </edge>\n  <edge id=":1559016903_13" function="internal">\n    <lane id=":1559016903_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.98" length="10.14" shape="880.19,345.67 878.91,342.93 876.32,341.02 872.52,340.11" />\n  </edge>\n  <edge id=":1559016903_9" function="internal">\n    <lane id=":1559016903_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.48" length="9.04" shape="872.23,343.30 874.64,343.88 876.27,345.09 877.14,346.94 877.23,349.42" />\n  </edge>\n  <edge id=":1559016903_10" function="internal">\n    <lane id=":1559016903_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.44" shape="872.23,343.30 886.58,344.82" />\n  </edge>\n  <edge id=":1559016903_11" function="internal">\n    <lane id=":1559016903_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.02" length="14.21" shape="872.23,343.30 876.13,343.11 879.03,341.73 880.95,339.16 881.88,335.39" />\n  </edge>\n  <edge id=":1559016904_0" function="internal">\n    <lane id=":1559016904_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.39" shape="780.93,369.23 779.21,383.52" />\n  </edge>\n  <edge id=":1559016904_1" function="internal">\n    <lane id=":1559016904_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.91" length="4.21" shape="780.93,369.23 780.94,373.13 781.06,373.41" />\n  </edge>\n  <edge id=":1559016904_6" function="internal">\n    <lane id=":1559016904_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.91" length="10.07" shape="781.06,373.41 782.23,376.04 784.81,377.97 788.66,378.92" />\n  </edge>\n  <edge id=":1559016904_2" function="internal">\n    <lane id=":1559016904_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.42" length="9.03" shape="788.99,375.74 786.55,375.14 784.93,373.92 784.11,372.07 784.11,369.60" />\n  </edge>\n  <edge id=":1559016904_3" function="internal">\n    <lane id=":1559016904_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.05" length="14.21" shape="788.99,375.74 785.08,375.89 782.15,377.24 780.20,379.78 779.21,383.52" />\n  </edge>\n  <edge id=":1559016904_4" function="internal">\n    <lane id=":1559016904_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.56" length="9.10" shape="782.39,383.89 783.02,381.50 784.28,379.88 786.16,379.02 788.66,378.92" />\n  </edge>\n  <edge id=":1559016904_5" function="internal">\n    <lane id=":1559016904_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="782.39,383.89 784.11,369.60" />\n  </edge>\n  <edge id=":1559016905_0" function="internal">\n    <lane id=":1559016905_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.53" length="9.07" shape="874.15,378.75 873.54,381.15 872.31,382.78 870.44,383.65 867.95,383.75" />\n  </edge>\n  <edge id=":1559016905_1" function="internal">\n    <lane id=":1559016905_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.41" shape="874.15,378.75 872.58,393.08" />\n  </edge>\n  <edge id=":1559016905_2" function="internal">\n    <lane id=":1559016905_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.44" shape="875.76,393.44 877.33,379.09" />\n  </edge>\n  <edge id=":1559016905_3" function="internal">\n    <lane id=":1559016905_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.98" length="4.11" shape="875.76,393.44 875.65,389.54 875.56,389.36" />\n  </edge>\n  <edge id=":1559016905_6" function="internal">\n    <lane id=":1559016905_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.98" length="10.13" shape="875.56,389.36 874.31,386.62 871.74,384.69 867.95,383.75" />\n  </edge>\n  <edge id=":1559016905_4" function="internal">\n    <lane id=":1559016905_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.48" length="9.03" shape="867.63,386.93 870.03,387.53 871.66,388.75 872.51,390.60 872.58,393.08" />\n  </edge>\n  <edge id=":1559016905_5" function="internal">\n    <lane id=":1559016905_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.02" length="14.22" shape="867.63,386.93 871.53,386.78 874.45,385.42 876.38,382.85 877.33,379.09" />\n  </edge>\n  <edge id=":1559016909_0" function="internal">\n    <lane id=":1559016909_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.47" length="9.03" shape="775.72,413.65 775.09,416.05 773.84,417.65 771.98,418.47 769.50,418.49" />\n  </edge>\n  <edge id=":1559016909_1" function="internal">\n    <lane id=":1559016909_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="775.72,413.65 774.07,427.97" />\n  </edge>\n  <edge id=":1559016909_2" function="internal">\n    <lane id=":1559016909_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.98" length="4.09" shape="775.72,413.65 775.83,417.56 775.90,417.72" />\n  </edge>\n  <edge id=":1559016909_12" function="internal">\n    <lane id=":1559016909_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.98" length="10.17" shape="775.90,417.72 777.16,420.49 779.73,422.43 783.52,423.38" />\n  </edge>\n  <edge id=":1559016909_3" function="internal">\n    <lane id=":1559016909_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.49" length="9.05" shape="783.85,420.19 781.44,419.59 779.81,418.36 778.97,416.50 778.90,414.01" />\n  </edge>\n  <edge id=":1559016909_4" function="internal">\n    <lane id=":1559016909_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.45" shape="783.85,420.19 769.50,418.49" />\n  </edge>\n  <edge id=":1559016909_5" function="internal">\n    <lane id=":1559016909_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.04" length="14.21" shape="783.85,420.19 779.94,420.34 777.01,421.68 775.06,424.23 774.07,427.97" />\n  </edge>\n  <edge id=":1559016909_6" function="internal">\n    <lane id=":1559016909_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.55" length="9.09" shape="777.25,428.34 777.88,425.95 779.14,424.33 781.01,423.47 783.52,423.38" />\n  </edge>\n  <edge id=":1559016909_7" function="internal">\n    <lane id=":1559016909_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="777.25,428.34 778.90,414.01" />\n  </edge>\n  <edge id=":1559016909_8" function="internal">\n    <lane id=":1559016909_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="4.04" shape="777.25,428.34 777.14,424.47 777.07,424.31" />\n  </edge>\n  <edge id=":1559016909_13" function="internal">\n    <lane id=":1559016909_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="10.19" shape="777.07,424.31 775.81,421.54 773.26,419.55 769.50,418.49" />\n  </edge>\n  <edge id=":1559016909_9" function="internal">\n    <lane id=":1559016909_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.58" length="9.14" shape="769.07,421.66 771.50,422.34 773.14,423.62 774.00,425.50 774.07,427.97" />\n  </edge>\n  <edge id=":1559016909_10" function="internal">\n    <lane id=":1559016909_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.54" shape="769.07,421.66 783.52,423.38" />\n  </edge>\n  <edge id=":1559016909_11" function="internal">\n    <lane id=":1559016909_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.97" length="14.26" shape="769.07,421.66 772.99,421.63 775.94,420.34 777.91,417.80 778.90,414.01" />\n  </edge>\n  <edge id=":1559016916_0" function="internal">\n    <lane id=":1559016916_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.21" shape="771.92,459.30 771.16,470.48" />\n  </edge>\n  <edge id=":1559016916_1" function="internal">\n    <lane id=":1559016916_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.51" length="2.42" shape="771.92,459.30 772.31,461.69" />\n  </edge>\n  <edge id=":1559016916_4" function="internal">\n    <lane id=":1559016916_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.51" length="9.37" shape="772.31,461.69 773.86,463.55 776.55,464.90 780.39,465.73" />\n  </edge>\n  <edge id=":1559016916_2" function="internal">\n    <lane id=":1559016916_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.38" length="9.04" shape="774.36,470.69 774.88,468.25 776.06,466.61 777.90,465.77 780.39,465.73" />\n  </edge>\n  <edge id=":1559016916_3" function="internal">\n    <lane id=":1559016916_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.21" shape="774.36,470.69 775.11,459.51" />\n  </edge>\n  <edge id=":1559016918_0" function="internal">\n    <lane id=":1559016918_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="14.10" shape="864.62,469.39 864.76,473.25 866.10,476.15 868.63,478.08 872.37,479.06" />\n  </edge>\n  <edge id=":1559016918_1" function="internal">\n    <lane id=":1559016918_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="8.97" shape="872.73,475.88 870.36,475.26 868.74,474.03 867.89,472.18 867.80,469.73" />\n  </edge>\n  <edge id=":1559016918_2" function="internal">\n    <lane id=":1559016918_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.45" shape="858.32,475.74 861.91,476.45 865.34,477.43 868.78,478.39 872.37,479.06" />\n  </edge>\n  <edge id=":1559016918_3" function="internal">\n    <lane id=":1559016918_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.57" length="9.24" shape="858.32,475.74 862.18,475.79 865.05,474.80 866.64,473.09" />\n  </edge>\n  <edge id=":1559016918_4" function="internal">\n    <lane id=":1559016918_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.57" length="3.59" shape="866.64,473.09 866.92,472.78 867.80,469.73" />\n  </edge>\n  <edge id=":1559029797_0" function="internal">\n    <lane id=":1559029797_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.41" length="9.04" shape="382.63,742.11 380.22,741.50 378.61,740.26 377.82,738.40 377.84,735.90" />\n  </edge>\n  <edge id=":1559029797_1" function="internal">\n    <lane id=":1559029797_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.26" shape="376.22,747.01 377.84,735.90" />\n    <lane id=":1559029797_1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.26" shape="373.05,746.56 374.68,735.42" />\n    <lane id=":1559029797_1_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.26" shape="369.88,746.12 371.52,734.94" />\n  </edge>\n  <edge id=":1559029798_0" function="internal">\n    <lane id=":1559029798_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.55" length="8.83" shape="458.43,744.70 457.74,746.98 456.47,748.53 454.62,749.34 452.19,749.42" />\n  </edge>\n  <edge id=":1559029798_1" function="internal">\n    <lane id=":1559029798_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.56" length="10.82" shape="458.43,744.70 456.89,755.41" />\n  </edge>\n  <edge id=":1639088623_0" function="internal">\n    <lane id=":1639088623_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.48" length="9.03" shape="1536.44,960.60 1533.96,960.66 1532.07,959.91 1530.76,958.35 1530.03,955.98" />\n  </edge>\n  <edge id=":1639088623_1" function="internal">\n    <lane id=":1639088623_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.50" length="9.04" shape="1532.38,970.21 1532.31,967.74 1533.05,965.84 1534.60,964.51 1536.97,963.76" />\n  </edge>\n  <edge id=":1639088623_2" function="internal">\n    <lane id=":1639088623_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="1532.38,970.21 1530.03,955.98" />\n  </edge>\n  <edge id=":1639088658_0" function="internal">\n    <lane id=":1639088658_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.14" length="11.26" shape="1555.10,1061.11 1551.92,1060.33 1549.51,1058.97 1547.87,1057.02 1547.00,1054.50" />\n  </edge>\n  <edge id=":1639088658_1" function="internal">\n    <lane id=":1639088658_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.93" shape="1555.10,1061.11 1551.69,1060.70 1548.66,1060.46 1545.63,1060.41 1542.20,1060.53" />\n  </edge>\n  <edge id=":1639088658_2" function="internal">\n    <lane id=":1639088658_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.95" length="9.09" shape="1549.57,1069.53 1549.50,1066.92 1550.33,1065.18 1552.05,1064.30 1554.66,1064.28" />\n  </edge>\n  <edge id=":1639088658_3" function="internal">\n    <lane id=":1639088658_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.26" shape="1549.57,1069.53 1547.00,1054.50" />\n  </edge>\n  <edge id=":1639088658_4" function="internal">\n    <lane id=":1639088658_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.96" length="12.80" shape="1549.57,1069.53 1548.52,1065.48 1546.94,1062.62 1544.83,1060.97 1542.20,1060.53" />\n  </edge>\n  <edge id=":1639088658_5" function="internal">\n    <lane id=":1639088658_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.31" shape="1542.39,1063.72 1545.71,1063.60 1548.54,1063.65 1551.36,1063.87 1554.66,1064.28" />\n  </edge>\n  <edge id=":1639088658_6" function="internal">\n    <lane id=":1639088658_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.10" length="11.94" shape="1542.39,1063.72 1544.98,1063.02 1546.61,1061.25 1547.29,1058.41 1547.00,1054.50" />\n  </edge>\n  <edge id=":1680666322_0" function="internal">\n    <lane id=":1680666322_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.46" shape="935.02,307.21 933.54,321.59" />\n  </edge>\n  <edge id=":1680666322_1" function="internal">\n    <lane id=":1680666322_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.88" length="4.16" shape="935.02,307.21 935.17,311.21 935.23,311.35" />\n  </edge>\n  <edge id=":1680666322_6" function="internal">\n    <lane id=":1680666322_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.88" length="10.31" shape="935.23,311.35 936.58,314.11 939.25,315.92 943.19,316.63" />\n  </edge>\n  <edge id=":1680666322_2" function="internal">\n    <lane id=":1680666322_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.36" length="9.04" shape="943.31,313.43 940.85,312.99 939.17,311.86 938.29,310.05 938.20,307.55" />\n  </edge>\n  <edge id=":1680666322_3" function="internal">\n    <lane id=":1680666322_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.22" length="14.31" shape="943.31,313.43 939.35,313.82 936.40,315.31 934.47,317.90 933.54,321.59" />\n  </edge>\n  <edge id=":1680666322_4" function="internal">\n    <lane id=":1680666322_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.77" length="9.38" shape="936.73,321.90 937.33,319.52 938.61,317.84 940.57,316.88 943.19,316.63" />\n  </edge>\n  <edge id=":1680666322_5" function="internal">\n    <lane id=":1680666322_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.43" shape="936.73,321.90 938.20,307.55" />\n  </edge>\n  <edge id=":1680666463_0" function="internal">\n    <lane id=":1680666463_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.50" length="10.66" shape="1496.56,366.12 1495.03,368.66 1493.10,370.27 1490.77,370.95 1488.03,370.70" />\n  </edge>\n  <edge id=":1680666463_1" function="internal">\n    <lane id=":1680666463_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.47" shape="1496.56,366.12 1495.07,369.64 1493.64,373.28 1492.32,376.96 1491.14,380.60" />\n  </edge>\n  <edge id=":1680666463_2" function="internal">\n    <lane id=":1680666463_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.69" length="15.00" shape="1496.56,366.12 1495.42,370.20 1495.98,373.42 1498.24,375.77 1502.21,377.27" />\n  </edge>\n  <edge id=":1680666463_3" function="internal">\n    <lane id=":1680666463_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.12" length="9.06" shape="1502.90,374.14 1500.50,373.24 1499.14,371.82 1498.80,369.88 1499.49,367.41" />\n  </edge>\n  <edge id=":1680666463_4" function="internal">\n    <lane id=":1680666463_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.26" shape="1502.90,374.14 1488.03,370.70" />\n  </edge>\n  <edge id=":1680666463_5" function="internal">\n    <lane id=":1680666463_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.46" length="4.31" shape="1502.90,374.14 1498.72,373.79 1498.61,373.82" />\n  </edge>\n  <edge id=":1680666463_12" function="internal">\n    <lane id=":1680666463_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.46" length="10.73" shape="1498.61,373.82 1495.38,374.75 1492.85,377.02 1491.14,380.60" />\n  </edge>\n  <edge id=":1680666463_6" function="internal">\n    <lane id=":1680666463_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.06" length="10.16" shape="1494.21,381.54 1495.35,379.15 1497.07,377.64 1499.35,377.02 1502.21,377.27" />\n  </edge>\n  <edge id=":1680666463_7" function="internal">\n    <lane id=":1680666463_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.09" shape="1494.21,381.54 1495.36,377.95 1496.64,374.40 1498.02,370.88 1499.49,367.41" />\n  </edge>\n  <edge id=":1680666463_8" function="internal">\n    <lane id=":1680666463_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.90" length="14.42" shape="1494.21,381.54 1494.81,377.60 1493.99,374.48 1491.72,372.18 1488.03,370.70" />\n  </edge>\n  <edge id=":1680666463_9" function="internal">\n    <lane id=":1680666463_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.39" length="9.04" shape="1487.27,373.81 1489.59,374.74 1491.01,376.18 1491.53,378.13 1491.14,380.60" />\n  </edge>\n  <edge id=":1680666463_10" function="internal">\n    <lane id=":1680666463_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.33" shape="1487.27,373.81 1502.21,377.27" />\n  </edge>\n  <edge id=":1680666463_11" function="internal">\n    <lane id=":1680666463_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.84" length="4.22" shape="1487.27,373.81 1491.22,374.18 1491.46,374.11" />\n  </edge>\n  <edge id=":1680666463_13" function="internal">\n    <lane id=":1680666463_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.84" length="10.96" shape="1491.46,374.11 1494.57,373.24 1497.32,370.98 1499.49,367.41" />\n  </edge>\n  <edge id=":1680666468_0" function="internal">\n    <lane id=":1680666468_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.45" shape="659.70,364.38 648.67,373.71" />\n  </edge>\n  <edge id=":1680666468_1" function="internal">\n    <lane id=":1680666468_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.90" length="3.94" shape="659.70,364.38 657.20,367.43" />\n  </edge>\n  <edge id=":1680666468_6" function="internal">\n    <lane id=":1680666468_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.90" length="10.61" shape="657.20,367.43 657.10,367.55 656.18,370.69 656.94,373.82 659.38,376.94" />\n  </edge>\n  <edge id=":1680666468_2" function="internal">\n    <lane id=":1680666468_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.35" length="9.01" shape="661.58,374.61 660.06,372.68 659.58,370.74 660.15,368.80 661.75,366.84" />\n  </edge>\n  <edge id=":1680666468_3" function="internal">\n    <lane id=":1680666468_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.42" length="14.39" shape="661.58,374.61 658.31,372.23 655.07,371.29 651.86,371.78 648.67,373.71" />\n  </edge>\n  <edge id=":1680666468_4" function="internal">\n    <lane id=":1680666468_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.00" length="9.66" shape="650.72,376.17 652.83,374.91 654.97,374.62 657.15,375.29 659.38,376.94" />\n  </edge>\n  <edge id=":1680666468_5" function="internal">\n    <lane id=":1680666468_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.44" shape="650.72,376.17 661.75,366.84" />\n  </edge>\n  <edge id=":1680666546_0" function="internal">\n    <lane id=":1680666546_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.46" length="9.03" shape="562.69,379.34 561.67,381.60 560.18,382.98 558.20,383.47 555.75,383.07" />\n  </edge>\n  <edge id=":1680666546_1" function="internal">\n    <lane id=":1680666546_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.08" length="14.22" shape="562.69,379.34 562.14,383.21 562.93,386.36 565.07,388.76 568.54,390.43" />\n  </edge>\n  <edge id=":1680666546_2" function="internal">\n    <lane id=":1680666546_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.60" length="9.15" shape="569.50,387.37 567.27,386.30 565.91,384.75 565.41,382.72 565.77,380.22" />\n  </edge>\n  <edge id=":1680666546_3" function="internal">\n    <lane id=":1680666546_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="569.50,387.37 555.75,383.07" />\n  </edge>\n  <edge id=":1680666546_4" function="internal">\n    <lane id=":1680666546_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="554.80,386.13 568.54,390.43" />\n  </edge>\n  <edge id=":1680666546_5" function="internal">\n    <lane id=":1680666546_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.96" length="14.28" shape="554.80,386.13 558.67,386.75 561.79,385.98 564.16,383.80 565.77,380.22" />\n  </edge>\n  <edge id=":1680666686_0" function="internal">\n    <lane id=":1680666686_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.69" length="9.42" shape="923.85,407.49 923.18,410.01 921.89,411.74 919.98,412.66 917.45,412.79" />\n  </edge>\n  <edge id=":1680666686_1" function="internal">\n    <lane id=":1680666686_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.73" shape="923.85,407.49 922.10,422.11" />\n  </edge>\n  <edge id=":1680666686_2" function="internal">\n    <lane id=":1680666686_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.87" length="4.16" shape="923.85,407.49 923.93,411.50 923.99,411.63" />\n  </edge>\n  <edge id=":1680666686_12" function="internal">\n    <lane id=":1680666686_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.87" length="10.33" shape="923.99,411.63 925.30,414.43 927.95,416.27 931.89,417.03" />\n  </edge>\n  <edge id=":1680666686_3" function="internal">\n    <lane id=":1680666686_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.35" length="9.04" shape="932.04,413.83 929.58,413.36 927.93,412.21 927.07,410.38 927.03,407.88" />\n  </edge>\n  <edge id=":1680666686_4" function="internal">\n    <lane id=":1680666686_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.63" shape="932.04,413.83 917.45,412.79" />\n  </edge>\n  <edge id=":1680666686_5" function="internal">\n    <lane id=":1680666686_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.30" length="14.54" shape="932.04,413.83 928.07,414.19 925.09,415.69 923.10,418.33 922.10,422.11" />\n  </edge>\n  <edge id=":1680666686_6" function="internal">\n    <lane id=":1680666686_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.86" length="9.63" shape="925.28,422.47 925.94,419.99 927.26,418.25 929.25,417.26 931.89,417.03" />\n  </edge>\n  <edge id=":1680666686_7" function="internal">\n    <lane id=":1680666686_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.70" shape="925.28,422.47 927.03,407.88" />\n  </edge>\n  <edge id=":1680666686_8" function="internal">\n    <lane id=":1680666686_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.97" length="4.05" shape="925.28,422.47 925.17,418.56 925.12,418.44" />\n  </edge>\n  <edge id=":1680666686_13" function="internal">\n    <lane id=":1680666686_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.97" length="10.21" shape="925.12,418.44 923.83,415.64 921.26,413.71 917.45,412.79" />\n  </edge>\n  <edge id=":1680666686_9" function="internal">\n    <lane id=":1680666686_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.47" length="9.03" shape="917.14,415.97 919.55,416.56 921.18,417.78 922.03,419.63 922.10,422.11" />\n  </edge>\n  <edge id=":1680666686_10" function="internal">\n    <lane id=":1680666686_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.79" shape="917.14,415.97 931.89,417.03" />\n  </edge>\n  <edge id=":1680666686_11" function="internal">\n    <lane id=":1680666686_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.16" length="14.48" shape="917.14,415.97 921.05,415.79 924.00,414.37 925.99,411.74 927.03,407.88" />\n  </edge>\n  <edge id=":1680666775_0" function="internal">\n    <lane id=":1680666775_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.79" length="9.46" shape="1442.07,457.65 1439.40,457.57 1437.36,456.74 1435.96,455.14 1435.20,452.79" />\n  </edge>\n  <edge id=":1680666775_1" function="internal">\n    <lane id=":1680666775_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.39" length="9.06" shape="1437.53,467.03 1437.48,464.54 1438.27,462.68 1439.88,461.44 1442.32,460.84" />\n  </edge>\n  <edge id=":1680666775_2" function="internal">\n    <lane id=":1680666775_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="1437.53,467.03 1435.20,452.79" />\n    <lane id=":1680666775_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="1434.37,467.54 1432.04,453.31" />\n  </edge>\n  <edge id=":1680666827_0" function="internal">\n    <lane id=":1680666827_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="1553.27,470.13 1538.97,468.23" />\n  </edge>\n  <edge id=":1680666827_1" function="internal">\n    <lane id=":1680666827_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.87" length="4.14" shape="1553.27,470.13 1549.26,470.17 1549.15,470.22" />\n  </edge>\n  <edge id=":1680666827_6" function="internal">\n    <lane id=":1680666827_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.87" length="10.36" shape="1549.15,470.22 1546.32,471.51 1544.45,474.15 1543.65,478.09" />\n  </edge>\n  <edge id=":1680666827_2" function="internal">\n    <lane id=":1680666827_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.35" length="9.04" shape="1546.85,478.26 1547.34,475.81 1548.51,474.17 1550.34,473.33 1552.85,473.31" />\n  </edge>\n  <edge id=":1680666827_3" function="internal">\n    <lane id=":1680666827_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.26" length="14.31" shape="1546.85,478.26 1546.55,474.29 1545.13,471.29 1542.61,469.27 1538.97,468.23" />\n  </edge>\n  <edge id=":1680666827_4" function="internal">\n    <lane id=":1680666827_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.82" length="9.43" shape="1538.55,471.40 1540.91,472.09 1542.55,473.43 1543.46,475.43 1543.65,478.09" />\n  </edge>\n  <edge id=":1680666827_5" function="internal">\n    <lane id=":1680666827_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="1538.55,471.40 1552.85,473.31" />\n  </edge>\n  <edge id=":1680666891_0" function="internal">\n    <lane id=":1680666891_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.49" shape="1640.28,485.00 1626.13,481.88" />\n  </edge>\n  <edge id=":1680666891_1" function="internal">\n    <lane id=":1680666891_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.94" length="4.16" shape="1640.28,485.00 1636.35,484.67 1636.14,484.74" />\n  </edge>\n  <edge id=":1680666891_6" function="internal">\n    <lane id=":1680666891_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.94" length="10.15" shape="1636.14,484.74 1633.30,485.69 1631.12,488.06 1629.82,491.78" />\n  </edge>\n  <edge id=":1680666891_2" function="internal">\n    <lane id=":1680666891_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.44" length="9.03" shape="1632.96,492.40 1633.78,490.05 1635.15,488.55 1637.08,487.91 1639.56,488.12" />\n  </edge>\n  <edge id=":1680666891_3" function="internal">\n    <lane id=":1680666891_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.06" length="14.26" shape="1632.96,492.40 1633.18,488.47 1632.11,485.41 1629.76,483.21 1626.13,481.88" />\n  </edge>\n  <edge id=":1680666891_4" function="internal">\n    <lane id=":1680666891_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.58" length="9.15" shape="1625.47,485.01 1627.79,485.86 1629.28,487.28 1629.96,489.25 1629.82,491.78" />\n  </edge>\n  <edge id=":1680666891_5" function="internal">\n    <lane id=":1680666891_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.43" shape="1625.47,485.01 1639.56,488.12" />\n  </edge>\n  <edge id=":1680667029_0" function="internal">\n    <lane id=":1680667029_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.60" length="9.13" shape="980.33,502.08 979.70,504.52 978.46,506.19 976.61,507.10 974.14,507.23" />\n  </edge>\n  <edge id=":1680667029_1" function="internal">\n    <lane id=":1680667029_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.98" length="14.25" shape="980.33,502.08 980.44,505.99 981.77,508.91 984.34,510.84 988.14,511.78" />\n  </edge>\n  <edge id=":1680667029_2" function="internal">\n    <lane id=":1680667029_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.48" length="9.03" shape="988.46,508.60 986.05,508.00 984.43,506.78 983.58,504.93 983.51,502.45" />\n  </edge>\n  <edge id=":1680667029_3" function="internal">\n    <lane id=":1680667029_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.38" shape="988.46,508.60 974.14,507.23" />\n  </edge>\n  <edge id=":1680667029_4" function="internal">\n    <lane id=":1680667029_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.35" shape="973.86,510.42 988.14,511.78" />\n  </edge>\n  <edge id=":1680667029_5" function="internal">\n    <lane id=":1680667029_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.08" length="3.99" shape="973.86,510.42 977.70,510.21 977.82,510.15" />\n  </edge>\n  <edge id=":1680667029_6" function="internal">\n    <lane id=":1680667029_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.08" length="10.21" shape="977.82,510.15 980.59,508.81 982.53,506.22 983.51,502.45" />\n  </edge>\n  <edge id=":1680667079_0" function="internal">\n    <lane id=":1680667079_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.35" length="10.36" shape="1553.95,521.42 1551.23,520.96 1549.17,519.75 1547.77,517.80 1547.03,515.10" />\n  </edge>\n  <edge id=":1680667079_1" function="internal">\n    <lane id=":1680667079_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.11" shape="1553.95,521.42 1538.84,521.41" />\n  </edge>\n  <edge id=":1680667079_2" function="internal">\n    <lane id=":1680667079_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.86" length="4.36" shape="1553.95,521.42 1549.96,521.88 1549.67,522.05" />\n  </edge>\n  <edge id=":1680667079_12" function="internal">\n    <lane id=":1680667079_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.86" length="10.15" shape="1549.67,522.05 1547.17,523.52 1545.59,526.34 1545.23,530.34" />\n  </edge>\n  <edge id=":1680667079_3" function="internal">\n    <lane id=":1680667079_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.34" length="9.04" shape="1548.42,530.18 1548.65,527.68 1549.63,525.93 1551.37,524.91 1553.86,524.62" />\n  </edge>\n  <edge id=":1680667079_4" function="internal">\n    <lane id=":1680667079_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.15" shape="1548.42,530.18 1547.03,515.10" />\n  </edge>\n  <edge id=":1680667079_5" function="internal">\n    <lane id=":1680667079_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.21" length="14.73" shape="1548.42,530.18 1547.65,526.25 1545.79,523.48 1542.86,521.86 1538.84,521.41" />\n  </edge>\n  <edge id=":1680667079_6" function="internal">\n    <lane id=":1680667079_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.76" length="9.66" shape="1538.93,524.61 1541.57,524.90 1543.50,525.96 1544.72,527.77 1545.23,530.34" />\n  </edge>\n  <edge id=":1680667079_7" function="internal">\n    <lane id=":1680667079_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.93" shape="1538.93,524.61 1553.86,524.62" />\n  </edge>\n  <edge id=":1680667079_8" function="internal">\n    <lane id=":1680667079_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.82" length="3.96" shape="1538.93,524.61 1542.83,523.93" />\n  </edge>\n  <edge id=":1680667079_13" function="internal">\n    <lane id=":1680667079_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.82" length="10.65" shape="1542.83,523.93 1542.93,523.92 1545.62,522.10 1546.98,519.16 1547.03,515.10" />\n  </edge>\n  <edge id=":1680667079_9" function="internal">\n    <lane id=":1680667079_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.29" length="9.04" shape="1543.85,515.52 1543.83,518.04 1542.98,519.86 1541.32,520.98 1538.84,521.41" />\n  </edge>\n  <edge id=":1680667079_10" function="internal">\n    <lane id=":1680667079_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.88" shape="1543.85,515.52 1545.23,530.34" />\n  </edge>\n  <edge id=":1680667079_11" function="internal">\n    <lane id=":1680667079_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.72" length="14.93" shape="1543.85,515.52 1544.92,519.40 1546.95,522.22 1549.92,523.95 1553.86,524.62" />\n  </edge>\n  <edge id=":1680667164_0" function="internal">\n    <lane id=":1680667164_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.48" length="9.03" shape="1554.90,546.64 1552.42,546.45 1550.62,545.51 1549.47,543.83 1549.00,541.39" />\n  </edge>\n  <edge id=":1680667164_1" function="internal">\n    <lane id=":1680667164_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.37" shape="1554.90,546.64 1540.57,547.74" />\n  </edge>\n  <edge id=":1680667164_2" function="internal">\n    <lane id=":1680667164_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.29" shape="1540.85,550.92 1555.10,549.83" />\n  </edge>\n  <edge id=":1680667164_3" function="internal">\n    <lane id=":1680667164_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.11" length="3.83" shape="1540.85,550.92 1544.58,550.05" />\n  </edge>\n  <edge id=":1680667164_6" function="internal">\n    <lane id=":1680667164_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.11" length="10.35" shape="1544.58,550.05 1547.19,548.16 1548.66,545.28 1549.00,541.39" />\n  </edge>\n  <edge id=":1680667164_4" function="internal">\n    <lane id=":1680667164_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.63" length="9.16" shape="1545.80,541.56 1545.58,544.08 1544.64,545.95 1542.97,547.17 1540.57,547.74" />\n  </edge>\n  <edge id=":1680667164_5" function="internal">\n    <lane id=":1680667164_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.98" length="14.25" shape="1545.80,541.56 1546.55,545.39 1548.35,548.05 1551.20,549.53 1555.10,549.83" />\n  </edge>\n  <edge id=":1680667181_0" function="internal">\n    <lane id=":1680667181_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="9.01" shape="1080.84,540.96 1080.29,543.37 1079.10,545.03 1077.28,545.93 1074.81,546.08" />\n  </edge>\n  <edge id=":1680667181_1" function="internal">\n    <lane id=":1680667181_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.05" length="14.14" shape="1080.84,540.96 1081.06,544.83 1082.46,547.72 1085.02,549.64 1088.76,550.58" />\n  </edge>\n  <edge id=":1680667181_2" function="internal">\n    <lane id=":1680667181_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.56" length="9.05" shape="1089.09,547.39 1086.70,546.79 1085.06,545.57 1084.17,543.71 1084.03,541.23" />\n  </edge>\n  <edge id=":1680667181_3" function="internal">\n    <lane id=":1680667181_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.34" shape="1089.09,547.39 1074.81,546.08" />\n  </edge>\n  <edge id=":1680667181_4" function="internal">\n    <lane id=":1680667181_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.26" shape="1074.56,549.27 1088.76,550.58" />\n  </edge>\n  <edge id=":1680667181_5" function="internal">\n    <lane id=":1680667181_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="3.88" shape="1074.56,549.27 1078.43,549.03" />\n  </edge>\n  <edge id=":1680667181_6" function="internal">\n    <lane id=":1680667181_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="10.28" shape="1078.43,549.03 1081.30,547.61 1083.17,545.01 1084.03,541.23" />\n  </edge>\n  <edge id=":1680667730_0" function="internal">\n    <lane id=":1680667730_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.30" length="10.01" shape="1458.03,713.22 1460.92,713.62 1463.10,714.65 1464.59,716.32 1465.37,718.63" />\n  </edge>\n  <edge id=":1680667730_1" function="internal">\n    <lane id=":1680667730_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.14" length="9.05" shape="1463.04,704.31 1463.08,706.86 1462.27,708.66 1460.60,709.72 1458.08,710.03" />\n  </edge>\n  <edge id=":1680667730_2" function="internal">\n    <lane id=":1680667730_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.51" shape="1463.04,704.31 1465.37,718.63" />\n  </edge>\n  <edge id=":1680668223_0" function="internal">\n    <lane id=":1680668223_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.45" length="9.03" shape="1645.22,1703.12 1642.74,1703.24 1640.83,1702.53 1639.51,1700.99 1638.76,1698.62" />\n  </edge>\n  <edge id=":1680668223_1" function="internal">\n    <lane id=":1680668223_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.62" length="9.17" shape="1641.09,1712.83 1641.05,1710.36 1641.82,1708.45 1643.41,1707.08 1645.82,1706.27" />\n  </edge>\n  <edge id=":1680668223_2" function="internal">\n    <lane id=":1680668223_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="1641.09,1712.83 1638.76,1698.62" />\n    <lane id=":1680668223_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="1637.93,1713.35 1635.61,1699.13" />\n  </edge>\n  <edge id=":1680668226_0" function="internal">\n    <lane id=":1680668226_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.34" length="9.04" shape="1621.16,1711.24 1623.65,1710.99 1625.57,1711.62 1626.90,1713.13 1627.65,1715.52" />\n  </edge>\n  <edge id=":1680668226_1" function="internal">\n    <lane id=":1680668226_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.83" length="9.45" shape="1625.33,1701.29 1625.35,1703.75 1624.54,1705.70 1622.88,1707.17 1620.39,1708.13" />\n  </edge>\n  <edge id=":1680668226_2" function="internal">\n    <lane id=":1680668226_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="1625.33,1701.29 1627.65,1715.52" />\n  </edge>\n  <edge id=":1966386254_0" function="internal">\n    <lane id=":1966386254_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.93" length="10.08" shape="569.54,867.67 566.94,869.68 564.57,870.97 562.44,871.54 560.56,871.39" />\n  </edge>\n  <edge id=":1966386254_1" function="internal">\n    <lane id=":1966386254_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.55" length="12.29" shape="569.54,867.67 567.07,869.93 565.22,872.08 563.88,874.54 562.93,877.70" />\n  </edge>\n  <edge id=":1966386254_2" function="internal">\n    <lane id=":1966386254_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.01" length="11.08" shape="569.54,867.67 566.97,870.42 566.17,872.64 567.16,874.32 569.93,875.48" />\n  </edge>\n  <edge id=":1966386254_3" function="internal">\n    <lane id=":1966386254_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.22" shape="570.69,872.37 567.73,871.92 565.60,871.96 563.48,871.96 560.56,871.39" />\n  </edge>\n  <edge id=":1966386254_4" function="internal">\n    <lane id=":1966386254_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.99" length="10.78" shape="570.69,872.37 567.86,872.13 565.62,872.94 563.98,874.80 562.93,877.70" />\n  </edge>\n  <edge id=":1966399248_0" function="internal">\n    <lane id=":1966399248_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.88" length="9.55" shape="774.73,1129.10 773.95,1131.67 772.60,1133.44 770.68,1134.41 768.20,1134.59" />\n  </edge>\n  <edge id=":1966399248_1" function="internal">\n    <lane id=":1966399248_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.07" shape="774.73,1129.10 774.17,1132.82 773.74,1136.57 773.45,1140.33 773.28,1144.09" />\n  </edge>\n  <edge id=":1966399248_2" function="internal">\n    <lane id=":1966399248_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.86" length="14.53" shape="774.73,1129.10 774.64,1133.13 775.88,1136.12 778.47,1138.06 782.40,1138.95" />\n  </edge>\n  <edge id=":1966399248_3" function="internal">\n    <lane id=":1966399248_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.33" length="9.04" shape="782.66,1135.76 780.22,1135.21 778.61,1134.00 777.83,1132.14 777.89,1129.63" />\n  </edge>\n  <edge id=":1966399248_4" function="internal">\n    <lane id=":1966399248_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.51" shape="782.66,1135.76 768.20,1134.59" />\n  </edge>\n  <edge id=":1966399248_5" function="internal">\n    <lane id=":1966399248_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.92" length="4.16" shape="782.66,1135.76 778.64,1136.00 778.52,1136.07" />\n  </edge>\n  <edge id=":1966399248_12" function="internal">\n    <lane id=":1966399248_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.92" length="10.33" shape="778.52,1136.07 775.74,1137.47 773.96,1140.17 773.28,1144.09" />\n  </edge>\n  <edge id=":1966399248_6" function="internal">\n    <lane id=":1966399248_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.41" length="9.12" shape="776.48,1144.17 776.91,1141.71 778.03,1140.02 779.87,1139.10 782.40,1138.95" />\n  </edge>\n  <edge id=":1966399248_7" function="internal">\n    <lane id=":1966399248_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.62" shape="776.48,1144.17 776.64,1140.49 776.92,1136.88 777.34,1133.28 777.89,1129.63" />\n  </edge>\n  <edge id=":1966399248_8" function="internal">\n    <lane id=":1966399248_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.19" length="14.28" shape="776.48,1144.17 776.05,1140.24 774.52,1137.33 771.91,1135.44 768.20,1134.59" />\n  </edge>\n  <edge id=":1966399248_9" function="internal">\n    <lane id=":1966399248_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.73" length="9.32" shape="767.94,1137.78 770.33,1138.34 772.02,1139.58 773.01,1141.50 773.28,1144.09" />\n  </edge>\n  <edge id=":1966399248_10" function="internal">\n    <lane id=":1966399248_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.51" shape="767.94,1137.78 782.40,1138.95" />\n  </edge>\n  <edge id=":1966399248_11" function="internal">\n    <lane id=":1966399248_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.31" length="3.97" shape="767.94,1137.78 771.73,1137.52 771.89,1137.45" />\n  </edge>\n  <edge id=":1966399248_13" function="internal">\n    <lane id=":1966399248_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.31" length="10.42" shape="771.89,1137.45 774.66,1136.08 776.71,1133.45 777.89,1129.63" />\n  </edge>\n  <edge id=":1966404349_0" function="internal">\n    <lane id=":1966404349_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.33" length="9.04" shape="1027.36,1060.39 1026.95,1062.86 1025.83,1064.54 1024.03,1065.42 1021.52,1065.51" />\n  </edge>\n  <edge id=":1966404349_1" function="internal">\n    <lane id=":1966404349_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.70" shape="1027.36,1060.39 1027.09,1075.08" />\n  </edge>\n  <edge id=":1966404349_2" function="internal">\n    <lane id=":1966404349_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.69" length="12.22" shape="1027.36,1060.39 1027.63,1064.34 1028.64,1067.25 1030.39,1069.11 1032.89,1069.93" />\n  </edge>\n  <edge id=":1966404349_3" function="internal">\n    <lane id=":1966404349_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.76" shape="1033.21,1066.74 1021.52,1065.51" />\n  </edge>\n  <edge id=":1966404349_4" function="internal">\n    <lane id=":1966404349_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.16" length="2.65" shape="1033.21,1066.74 1030.58,1067.03" />\n  </edge>\n  <edge id=":1966404349_7" function="internal">\n    <lane id=":1966404349_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.16" length="9.24" shape="1030.58,1067.03 1028.68,1068.52 1027.52,1071.20 1027.09,1075.08" />\n  </edge>\n  <edge id=":1966404349_5" function="internal">\n    <lane id=":1966404349_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.96" length="9.74" shape="1021.18,1068.69 1023.79,1069.33 1025.65,1070.61 1026.75,1072.53 1027.09,1075.08" />\n  </edge>\n  <edge id=":1966404349_6" function="internal">\n    <lane id=":1966404349_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.78" shape="1021.18,1068.69 1032.89,1069.93" />\n  </edge>\n  <edge id=":1966412828_0" function="internal">\n    <lane id=":1966412828_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.75" length="9.15" shape="438.21,401.21 439.04,398.73 440.38,397.05 442.23,396.17 444.57,396.07" />\n  </edge>\n  <edge id=":1966412828_1" function="internal">\n    <lane id=":1966412828_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.05" shape="438.21,401.21 438.88,397.89 439.28,395.61 439.56,393.31 439.89,389.94" />\n    <lane id=":1966412828_1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.05" shape="435.07,400.56 435.72,397.35 436.11,395.14 436.39,392.91 436.71,389.64" />\n    <lane id=":1966412828_1_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.05" shape="431.94,399.92 432.57,396.81 432.94,394.66 433.21,392.50 433.52,389.34" />\n  </edge>\n  <edge id=":1966412828_4" function="internal">\n    <lane id=":1966412828_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.01" shape="427.69,393.92 444.57,396.07" />\n  </edge>\n  <edge id=":1966412828_5" function="internal">\n    <lane id=":1966412828_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.10" length="11.32" shape="427.69,393.92 431.44,394.09 434.19,393.44 435.95,391.96 436.71,389.64" />\n    <lane id=":1966412828_5_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.25" length="8.54" shape="427.69,393.92 430.05,393.91 431.81,393.14 432.97,391.62 433.52,389.34" />\n  </edge>\n  <edge id=":2052489649_0" function="internal">\n    <lane id=":2052489649_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.78" shape="737.01,805.13 734.75,802.77 733.02,800.75 731.66,798.51 730.48,795.51" />\n  </edge>\n  <edge id=":2052489649_1" function="internal">\n    <lane id=":2052489649_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.63" length="9.29" shape="728.26,809.04 729.29,806.43 730.70,805.27 732.51,805.57 734.70,807.34" />\n  </edge>\n  <edge id=":2052489649_2" function="internal">\n    <lane id=":2052489649_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.94" shape="728.26,809.04 729.33,805.55 730.39,802.24 730.94,798.95 730.48,795.51" />\n  </edge>\n  <edge id=":2052489649_3" function="internal">\n    <lane id=":2052489649_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.21" shape="727.43,796.47 728.72,799.67 730.34,802.34 732.32,804.79 734.70,807.34" />\n  </edge>\n  <edge id=":2052497441_0" function="internal">\n    <lane id=":2052497441_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.49" length="9.03" shape="510.66,1144.01 509.88,1146.37 508.53,1147.90 506.62,1148.60 504.14,1148.49" />\n  </edge>\n  <edge id=":2052497441_1" function="internal">\n    <lane id=":2052497441_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.18" shape="510.66,1144.01 508.74,1155.03" />\n  </edge>\n  <edge id=":2052522376_0" function="internal">\n    <lane id=":2052522376_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="8.91" shape="334.23,1117.06 331.92,1116.30 330.39,1114.97 329.65,1113.09 329.70,1110.66" />\n  </edge>\n  <edge id=":2052522376_1" function="internal">\n    <lane id=":2052522376_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.88" shape="327.75,1121.47 329.70,1110.66" />\n    <lane id=":2052522376_1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.88" shape="324.61,1120.85 326.54,1110.15" />\n    <lane id=":2052522376_1_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.88" shape="321.47,1120.23 323.38,1109.63" />\n  </edge>\n  <edge id=":2052547074_0" function="internal">\n    <lane id=":2052547074_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.76" length="9.42" shape="483.67,570.34 482.97,572.76 481.63,574.42 479.66,575.32 477.06,575.46" />\n  </edge>\n  <edge id=":2052547074_1" function="internal">\n    <lane id=":2052547074_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.55" shape="483.67,570.34 481.69,584.75" />\n  </edge>\n  <edge id=":2052547074_2" function="internal">\n    <lane id=":2052547074_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.26" length="11.79" shape="483.67,570.34 483.50,574.22 484.20,577.07 485.76,578.90 488.19,579.71" />\n  </edge>\n  <edge id=":2052547074_3" function="internal">\n    <lane id=":2052547074_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.51" shape="488.52,576.53 477.06,575.46" />\n  </edge>\n  <edge id=":2052547074_4" function="internal">\n    <lane id=":2052547074_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.47" length="11.94" shape="488.52,576.53 485.98,576.82 484.00,578.29 482.57,580.94 481.69,584.75" />\n  </edge>\n  <edge id=":2052547074_5" function="internal">\n    <lane id=":2052547074_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.39" length="9.04" shape="476.79,578.65 479.22,579.21 480.85,580.41 481.67,582.26 481.69,584.75" />\n  </edge>\n  <edge id=":2052547074_6" function="internal">\n    <lane id=":2052547074_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.45" shape="476.79,578.65 488.19,579.71" />\n  </edge>\n  <edge id=":2052547075_0" function="internal">\n    <lane id=":2052547075_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.40" length="9.04" shape="704.71,596.57 702.28,596.00 700.66,594.80 699.83,592.95 699.81,590.45" />\n  </edge>\n  <edge id=":2052547075_1" function="internal">\n    <lane id=":2052547075_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.26" shape="704.71,596.57 693.49,595.60" />\n  </edge>\n  <edge id=":2052547075_2" function="internal">\n    <lane id=":2052547075_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.47" length="9.09" shape="698.42,604.90 698.92,602.47 700.09,600.80 701.93,599.89 704.44,599.76" />\n  </edge>\n  <edge id=":2052547075_3" function="internal">\n    <lane id=":2052547075_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.52" shape="698.42,604.90 699.81,590.45" />\n  </edge>\n  <edge id=":2052547075_4" function="internal">\n    <lane id=":2052547075_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.41" length="11.75" shape="698.42,604.90 698.30,601.01 697.44,598.16 695.84,596.36 693.49,595.60" />\n  </edge>\n  <edge id=":2052547075_5" function="internal">\n    <lane id=":2052547075_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.27" shape="693.21,598.79 704.44,599.76" />\n  </edge>\n  <edge id=":2052547075_6" function="internal">\n    <lane id=":2052547075_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.48" length="2.47" shape="693.21,598.79 695.66,598.44" />\n  </edge>\n  <edge id=":2052547075_7" function="internal">\n    <lane id=":2052547075_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.48" length="9.35" shape="695.66,598.44 697.57,596.94 698.95,594.28 699.81,590.45" />\n  </edge>\n  <edge id=":2505176020_0" function="internal">\n    <lane id=":2505176020_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.57" length="9.11" shape="1627.77,752.38 1627.22,754.83 1626.03,756.53 1624.20,757.47 1621.74,757.67" />\n  </edge>\n  <edge id=":2505176020_1" function="internal">\n    <lane id=":2505176020_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.47" length="9.03" shape="1636.11,758.62 1633.68,758.10 1632.01,756.94 1631.11,755.11 1630.96,752.64" />\n  </edge>\n  <edge id=":2505176020_2" function="internal">\n    <lane id=":2505176020_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="1636.11,758.62 1621.74,757.67" />\n  </edge>\n  <edge id=":2505176027_0" function="internal">\n    <lane id=":2505176027_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="8.95" shape="1848.43,779.16 1850.82,779.68 1852.48,780.84 1853.41,782.64 1853.61,785.09" />\n  </edge>\n  <edge id=":2505176175_0" function="internal">\n    <lane id=":2505176175_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.92" length="9.51" shape="1540.58,1060.61 1537.90,1060.42 1535.85,1059.52 1534.45,1057.90 1533.69,1055.57" />\n  </edge>\n  <edge id=":2505176175_1" function="internal">\n    <lane id=":2505176175_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.27" length="9.01" shape="1536.04,1069.82 1535.98,1067.31 1536.76,1065.46 1538.38,1064.30 1540.83,1063.80" />\n  </edge>\n  <edge id=":2505176175_2" function="internal">\n    <lane id=":2505176175_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.44" shape="1536.04,1069.82 1533.69,1055.57" />\n    <lane id=":2505176175_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.44" shape="1532.88,1070.35 1530.54,1056.09" />\n  </edge>\n  <edge id=":2505176258_0" function="internal">\n    <lane id=":2505176258_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.77" length="9.45" shape="1562.99,1198.62 1560.39,1198.51 1558.41,1197.62 1557.06,1195.96 1556.32,1193.53" />\n  </edge>\n  <edge id=":2505176258_1" function="internal">\n    <lane id=":2505176258_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.96" shape="1562.99,1198.62 1549.11,1200.18" />\n  </edge>\n  <edge id=":2505176258_2" function="internal">\n    <lane id=":2505176258_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.37" length="9.04" shape="1558.60,1208.01 1558.55,1205.49 1559.32,1203.62 1560.92,1202.39 1563.34,1201.80" />\n  </edge>\n  <edge id=":2505176258_3" function="internal">\n    <lane id=":2505176258_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.67" shape="1558.60,1208.01 1556.32,1193.53" />\n    <lane id=":2505176258_3_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.67" shape="1555.44,1208.52 1553.16,1194.01" />\n  </edge>\n  <edge id=":2505176258_5" function="internal">\n    <lane id=":2505176258_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.40" length="11.57" shape="1555.44,1208.52 1554.50,1204.65 1553.13,1201.97 1551.33,1200.48 1549.11,1200.18" />\n  </edge>\n  <edge id=":2505176258_6" function="internal">\n    <lane id=":2505176258_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.94" shape="1549.49,1203.35 1563.34,1201.80" />\n  </edge>\n  <edge id=":2505176258_7" function="internal">\n    <lane id=":2505176258_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.12" length="11.27" shape="1549.49,1203.35 1551.59,1202.55 1552.90,1200.72 1553.42,1197.87 1553.16,1194.01" />\n  </edge>\n  <edge id=":2505176265_0" function="internal">\n    <lane id=":2505176265_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.39" shape="1571.58,1204.05 1570.29,1194.75" />\n  </edge>\n  <edge id=":2505176265_1" function="internal">\n    <lane id=":2505176265_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.19" length="7.81" shape="1571.58,1204.05 1571.00,1201.37 1570.13,1199.50 1568.99,1198.44 1567.57,1198.19" />\n  </edge>\n  <edge id=":2505176265_2" function="internal">\n    <lane id=":2505176265_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.05" length="1.01" shape="1567.88,1201.37 1568.83,1201.00" />\n  </edge>\n  <edge id=":2505176265_3" function="internal">\n    <lane id=":2505176265_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.05" length="6.84" shape="1568.83,1201.00 1569.25,1200.83 1570.11,1199.55 1570.45,1197.52 1570.29,1194.75" />\n  </edge>\n  <edge id=":2505176388_0" function="internal">\n    <lane id=":2505176388_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.51" length="10.44" shape="1612.39,1423.15 1609.44,1422.67 1607.21,1421.52 1605.69,1419.72 1604.90,1417.26" />\n  </edge>\n  <edge id=":2505176388_1" function="internal">\n    <lane id=":2505176388_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.27" shape="1612.39,1423.15 1609.07,1423.03 1606.26,1423.03 1603.45,1423.17 1600.14,1423.49" />\n  </edge>\n  <edge id=":2505176388_2" function="internal">\n    <lane id=":2505176388_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.11" length="9.06" shape="1607.20,1431.95 1607.17,1429.40 1608.00,1427.61 1609.69,1426.59 1612.24,1426.35" />\n  </edge>\n  <edge id=":2505176388_3" function="internal">\n    <lane id=":2505176388_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.87" shape="1607.20,1431.95 1604.90,1417.26" />\n  </edge>\n  <edge id=":2505176388_4" function="internal">\n    <lane id=":2505176388_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.60" length="12.27" shape="1607.20,1431.95 1606.23,1428.02 1604.73,1425.30 1602.70,1423.79 1600.14,1423.49" />\n  </edge>\n  <edge id=":2505176388_5" function="internal">\n    <lane id=":2505176388_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.77" shape="1600.49,1426.67 1603.72,1426.36 1606.36,1426.21 1608.99,1426.22 1612.24,1426.35" />\n  </edge>\n  <edge id=":2505176388_6" function="internal">\n    <lane id=":2505176388_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.23" length="11.81" shape="1600.49,1426.67 1602.94,1425.84 1604.49,1424.00 1605.14,1421.14 1604.90,1417.26" />\n  </edge>\n  <edge id=":2505176494_0" function="internal">\n    <lane id=":2505176494_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.90" length="9.53" shape="1607.87,1472.08 1605.17,1471.92 1603.11,1471.04 1601.70,1469.43 1600.94,1467.10" />\n  </edge>\n  <edge id=":2505176494_1" function="internal">\n    <lane id=":2505176494_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.31" length="9.04" shape="1603.26,1481.34 1603.21,1478.83 1604.00,1476.98 1605.62,1475.80 1608.08,1475.28" />\n  </edge>\n  <edge id=":2505176494_2" function="internal">\n    <lane id=":2505176494_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.43" shape="1603.26,1481.34 1600.94,1467.10" />\n    <lane id=":2505176494_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.43" shape="1600.10,1481.86 1597.78,1467.62" />\n  </edge>\n  <edge id=":2505176514_0" function="internal">\n    <lane id=":2505176514_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.99" length="9.63" shape="1611.31,1492.76 1608.57,1492.54 1606.49,1491.62 1605.06,1490.00 1604.30,1487.67" />\n  </edge>\n  <edge id=":2505176514_1" function="internal">\n    <lane id=":2505176514_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.26" length="9.04" shape="1606.63,1501.93 1606.58,1499.41 1607.37,1497.57 1609.01,1496.42 1611.49,1495.95" />\n  </edge>\n  <edge id=":2505176514_2" function="internal">\n    <lane id=":2505176514_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.45" shape="1606.63,1501.93 1604.30,1487.67" />\n    <lane id=":2505176514_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.45" shape="1603.47,1502.44 1601.14,1488.19" />\n  </edge>\n  <edge id=":2505176697_0" function="internal">\n    <lane id=":2505176697_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.70" length="10.47" shape="1349.87,1592.04 1346.91,1591.07 1344.75,1589.64 1343.42,1587.73 1342.89,1585.35" />\n  </edge>\n  <edge id=":2505176697_1" function="internal">\n    <lane id=":2505176697_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.63" length="15.25" shape="1349.87,1592.04 1345.54,1591.73 1342.50,1592.96 1340.75,1595.74 1340.30,1600.06" />\n  </edge>\n  <edge id=":2505176697_2" function="internal">\n    <lane id=":2505176697_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.03" length="9.07" shape="1343.50,1599.94 1343.77,1597.37 1344.80,1595.72 1346.61,1594.98 1349.19,1595.17" />\n  </edge>\n  <edge id=":2505176697_3" function="internal">\n    <lane id=":2505176697_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.60" shape="1343.50,1599.94 1342.89,1585.35" />\n  </edge>\n  <edge id=":2505176697_4" function="internal">\n    <lane id=":2505176697_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.57" shape="1339.69,1585.50 1340.30,1600.06" />\n  </edge>\n  <edge id=":2505176697_5" function="internal">\n    <lane id=":2505176697_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.03" length="3.69" shape="1339.69,1585.50 1340.42,1588.99 1340.49,1589.10" />\n  </edge>\n  <edge id=":2505176697_6" function="internal">\n    <lane id=":2505176697_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.03" length="11.00" shape="1340.49,1589.10 1342.25,1591.76 1345.17,1593.82 1349.19,1595.17" />\n  </edge>\n  <edge id=":2506085476_0" function="internal">\n    <lane id=":2506085476_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.34" shape="1276.66,565.41 1262.36,564.35" />\n  </edge>\n  <edge id=":2506085476_1" function="internal">\n    <lane id=":2506085476_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.92" length="3.96" shape="1276.66,565.41 1272.71,565.71" />\n  </edge>\n  <edge id=":2506085476_6" function="internal">\n    <lane id=":2506085476_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.92" length="10.41" shape="1272.71,565.71 1269.86,567.22 1268.12,569.93 1267.47,573.84" />\n  </edge>\n  <edge id=":2506085476_2" function="internal">\n    <lane id=":2506085476_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.41" length="9.03" shape="1270.67,573.90 1271.07,571.44 1272.17,569.74 1273.96,568.80 1276.45,568.61" />\n  </edge>\n  <edge id=":2506085476_3" function="internal">\n    <lane id=":2506085476_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.19" length="14.25" shape="1270.67,573.90 1270.22,569.99 1268.69,567.09 1266.07,565.21 1262.36,564.35" />\n  </edge>\n  <edge id=":2506085476_4" function="internal">\n    <lane id=":2506085476_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.74" length="9.31" shape="1262.10,567.54 1264.50,568.10 1266.19,569.34 1267.18,571.25 1267.47,573.84" />\n  </edge>\n  <edge id=":2506085476_5" function="internal">\n    <lane id=":2506085476_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.39" shape="1262.10,567.54 1276.45,568.61" />\n  </edge>\n  <edge id=":2506085493_0" function="internal">\n    <lane id=":2506085493_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.67" shape="1194.39,645.94 1193.39,660.58" />\n  </edge>\n  <edge id=":2506085493_1" function="internal">\n    <lane id=":2506085493_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.49" length="4.30" shape="1194.39,645.94 1194.81,650.21" />\n  </edge>\n  <edge id=":2506085493_6" function="internal">\n    <lane id=":2506085493_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.49" length="11.67" shape="1194.81,650.21 1194.85,650.59 1196.67,653.36 1199.86,654.26 1204.42,653.29" />\n  </edge>\n  <edge id=":2506085493_2" function="internal">\n    <lane id=":2506085493_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.83" length="9.14" shape="1203.33,650.28 1200.72,650.84 1198.89,650.32 1197.85,648.74 1197.59,646.08" />\n  </edge>\n  <edge id=":2506085493_3" function="internal">\n    <lane id=":2506085493_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.09" length="15.10" shape="1203.33,650.28 1199.26,652.18 1196.25,654.53 1194.29,657.33 1193.39,660.58" />\n  </edge>\n  <edge id=":2506085493_4" function="internal">\n    <lane id=":2506085493_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.88" length="11.53" shape="1196.57,660.92 1197.26,658.58 1198.80,656.53 1201.19,654.77 1204.42,653.29" />\n  </edge>\n  <edge id=":2506085493_5" function="internal">\n    <lane id=":2506085493_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.88" shape="1196.57,660.92 1197.59,646.08" />\n  </edge>\n  <edge id=":2506085496_0" function="internal">\n    <lane id=":2506085496_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.30" length="9.04" shape="657.25,669.29 659.68,669.92 661.25,671.18 661.95,673.06 661.78,675.57" />\n  </edge>\n  <edge id=":2506085496_1" function="internal">\n    <lane id=":2506085496_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.12" shape="657.25,669.29 668.37,670.34" />\n    <lane id=":2506085496_1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.12" shape="657.60,666.11 668.61,667.15" />\n  </edge>\n  <edge id=":2506085508_0" function="internal">\n    <lane id=":2506085508_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.28" length="9.84" shape="1187.26,681.66 1185.45,683.82 1183.45,685.09 1181.27,685.45 1178.90,684.91" />\n  </edge>\n  <edge id=":2506085508_1" function="internal">\n    <lane id=":2506085508_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.81" length="14.63" shape="1187.26,681.66 1185.54,685.35 1185.49,688.59 1187.14,691.38 1190.48,693.72" />\n  </edge>\n  <edge id=":2506085508_2" function="internal">\n    <lane id=":2506085508_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.28" length="9.04" shape="1191.92,690.87 1189.86,689.42 1188.85,687.70 1188.87,685.69 1189.94,683.41" />\n  </edge>\n  <edge id=":2506085508_3" function="internal">\n    <lane id=":2506085508_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.33" shape="1191.92,690.87 1178.90,684.91" />\n  </edge>\n  <edge id=":2506085508_4" function="internal">\n    <lane id=":2506085508_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.05" shape="1177.70,687.87 1190.48,693.72" />\n  </edge>\n  <edge id=":2506085508_5" function="internal">\n    <lane id=":2506085508_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.66" length="3.62" shape="1177.70,687.87 1181.22,688.71" />\n  </edge>\n  <edge id=":2506085508_6" function="internal">\n    <lane id=":2506085508_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.66" length="10.66" shape="1181.22,688.71 1184.44,688.24 1187.34,686.48 1189.94,683.41" />\n  </edge>\n  <edge id=":2506085525_0" function="internal">\n    <lane id=":2506085525_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.60" length="9.15" shape="1354.43,742.81 1355.07,740.36 1356.32,738.68 1358.18,737.78 1360.64,737.66" />\n  </edge>\n  <edge id=":2506085525_1" function="internal">\n    <lane id=":2506085525_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.45" length="9.03" shape="1346.30,736.30 1348.72,736.88 1350.35,738.10 1351.19,739.95 1351.25,742.43" />\n  </edge>\n  <edge id=":2506085525_2" function="internal">\n    <lane id=":2506085525_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="1346.30,736.30 1360.64,737.66" />\n  </edge>\n  <edge id=":2506085555_0" function="internal">\n    <lane id=":2506085555_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.46" shape="1221.24,809.71 1209.81,810.49" />\n  </edge>\n  <edge id=":2506085555_1" function="internal">\n    <lane id=":2506085555_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.25" length="2.73" shape="1221.24,809.71 1218.76,810.36 1218.64,810.48" />\n  </edge>\n  <edge id=":2506085555_4" function="internal">\n    <lane id=":2506085555_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.25" length="9.06" shape="1218.64,810.48 1217.08,812.09 1216.22,814.90 1216.16,818.78" />\n  </edge>\n  <edge id=":2506085555_2" function="internal">\n    <lane id=":2506085555_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.48" length="9.06" shape="1210.11,813.67 1212.59,813.79 1214.43,814.68 1215.61,816.34 1216.16,818.78" />\n  </edge>\n  <edge id=":2506085555_3" function="internal">\n    <lane id=":2506085555_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.29" shape="1210.11,813.67 1221.37,812.91" />\n  </edge>\n  <edge id=":2506085628_0" function="internal">\n    <lane id=":2506085628_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.40" length="9.04" shape="617.01,878.35 616.12,880.68 614.71,882.13 612.77,882.69 610.29,882.37" />\n  </edge>\n  <edge id=":2506085628_1" function="internal">\n    <lane id=":2506085628_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.49" length="11.78" shape="617.01,878.35 616.48,882.23 616.84,885.22 618.08,887.30 620.21,888.47" />\n  </edge>\n  <edge id=":2506085628_2" function="internal">\n    <lane id=":2506085628_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.20" shape="621.08,885.39 610.29,882.37" />\n  </edge>\n  <edge id=":2506085628_3" function="internal">\n    <lane id=":2506085628_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.19" shape="609.43,885.45 620.21,888.47" />\n  </edge>\n  <edge id=":2506085631_0" function="internal">\n    <lane id=":2506085631_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.60" shape="1227.10,886.48 1215.59,887.87" />\n  </edge>\n  <edge id=":2506085631_1" function="internal">\n    <lane id=":2506085631_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.39" length="2.71" shape="1227.10,886.48 1224.70,887.24 1224.57,887.38" />\n  </edge>\n  <edge id=":2506085631_7" function="internal">\n    <lane id=":2506085631_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.39" length="9.16" shape="1224.57,887.38 1223.08,889.05 1222.24,891.92 1222.18,895.86" />\n  </edge>\n  <edge id=":2506085631_2" function="internal">\n    <lane id=":2506085631_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.33" length="9.04" shape="1216.09,891.03 1218.59,890.99 1220.45,891.79 1221.64,893.41 1222.18,895.86" />\n  </edge>\n  <edge id=":2506085631_3" function="internal">\n    <lane id=":2506085631_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.36" shape="1216.09,891.03 1227.36,889.67" />\n  </edge>\n  <edge id=":2506085631_4" function="internal">\n    <lane id=":2506085631_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.86" length="9.56" shape="1221.08,881.35 1220.90,883.85 1219.92,885.77 1218.15,887.11 1215.59,887.87" />\n  </edge>\n  <edge id=":2506085631_5" function="internal">\n    <lane id=":2506085631_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.55" shape="1221.08,881.35 1222.18,895.86" />\n  </edge>\n  <edge id=":2506085631_6" function="internal">\n    <lane id=":2506085631_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.32" length="11.73" shape="1221.08,881.35 1221.73,885.16 1222.99,887.82 1224.87,889.32 1227.36,889.67" />\n  </edge>\n  <edge id=":2506085690_0" function="internal">\n    <lane id=":2506085690_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.96" length="10.31" shape="753.89,921.78 752.75,924.00 750.96,925.31 748.53,925.72 745.44,925.22" />\n  </edge>\n  <edge id=":2506085690_1" function="internal">\n    <lane id=":2506085690_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.80" shape="753.89,921.78 748.59,935.60" />\n  </edge>\n  <edge id=":2506085690_2" function="internal">\n    <lane id=":2506085690_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.76" length="13.16" shape="753.89,921.78 753.22,925.60 753.75,928.65 755.50,930.93 758.45,932.44" />\n  </edge>\n  <edge id=":2506085690_3" function="internal">\n    <lane id=":2506085690_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.59" shape="759.42,929.39 745.44,925.22" />\n  </edge>\n  <edge id=":2506085690_4" function="internal">\n    <lane id=":2506085690_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.85" length="4.28" shape="759.42,929.39 755.45,928.73 755.21,928.80" />\n  </edge>\n  <edge id=":2506085690_9" function="internal">\n    <lane id=":2506085690_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.85" length="10.25" shape="755.21,928.80 752.31,929.55 750.03,931.84 748.59,935.60" />\n  </edge>\n  <edge id=":2506085690_5" function="internal">\n    <lane id=":2506085690_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.33" length="9.04" shape="751.72,936.31 752.61,933.96 754.03,932.54 755.98,932.03 758.45,932.44" />\n  </edge>\n  <edge id=":2506085690_6" function="internal">\n    <lane id=":2506085690_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.19" length="14.37" shape="751.72,936.31 752.07,932.32 751.14,929.14 748.93,926.77 745.44,925.22" />\n  </edge>\n  <edge id=":2506085690_7" function="internal">\n    <lane id=":2506085690_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.74" length="9.39" shape="744.58,928.30 746.83,929.32 748.25,930.87 748.83,932.97 748.59,935.60" />\n  </edge>\n  <edge id=":2506085690_8" function="internal">\n    <lane id=":2506085690_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.47" shape="744.58,928.30 758.45,932.44" />\n  </edge>\n  <edge id=":2506085849_0" function="internal">\n    <lane id=":2506085849_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.54" shape="471.44,987.56 470.95,990.73 470.88,993.31 470.76,995.87 470.12,999.00" />\n  </edge>\n  <edge id=":2506085849_1" function="internal">\n    <lane id=":2506085849_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.27" length="11.78" shape="471.44,987.56 471.42,990.12 472.64,992.19 475.11,993.78 478.83,994.89" />\n  </edge>\n  <edge id=":2506085849_2" function="internal">\n    <lane id=":2506085849_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.65" length="10.16" shape="464.70,992.13 467.67,993.06 469.57,994.52 470.38,996.50 470.12,999.00" />\n  </edge>\n  <edge id=":2506085849_3" function="internal">\n    <lane id=":2506085849_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.39" shape="464.70,992.13 478.83,994.89" />\n  </edge>\n  <edge id=":2506085849_4" function="internal">\n    <lane id=":2506085849_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.45" length="11.81" shape="464.70,992.13 468.54,992.50 471.46,991.99 473.47,990.58 474.56,988.28" />\n  </edge>\n  <edge id=":2506086181_0" function="internal">\n    <lane id=":2506086181_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.29" length="9.04" shape="451.58,1063.19 450.54,1065.48 449.04,1066.81 447.07,1067.17 444.62,1066.57" />\n  </edge>\n  <edge id=":2506086181_1" function="internal">\n    <lane id=":2506086181_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.53" length="11.86" shape="451.58,1063.19 450.80,1067.08 450.96,1070.11 452.06,1072.28 454.09,1073.60" />\n  </edge>\n  <edge id=":2506086181_2" function="internal">\n    <lane id=":2506086181_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.28" shape="455.16,1070.59 444.62,1066.57" />\n  </edge>\n  <edge id=":2506086181_3" function="internal">\n    <lane id=":2506086181_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.41" shape="443.42,1069.54 454.09,1073.60" />\n  </edge>\n  <edge id=":2506086743_0" function="internal">\n    <lane id=":2506086743_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.42" shape="447.18,1137.62 437.93,1135.89" />\n  </edge>\n  <edge id=":2506086743_1" function="internal">\n    <lane id=":2506086743_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.13" length="7.80" shape="447.18,1137.62 444.45,1137.29 442.40,1137.51 441.04,1138.25 440.38,1139.53" />\n  </edge>\n  <edge id=":2506086743_2" function="internal">\n    <lane id=":2506086743_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.06" length="7.82" shape="443.51,1140.18 443.41,1138.73 442.44,1137.53 440.62,1136.59 437.93,1135.89" />\n  </edge>\n  <edge id=":2506086795_0" function="internal">\n    <lane id=":2506086795_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.50" shape="833.46,1140.00 819.01,1138.73" />\n  </edge>\n  <edge id=":2506086795_1" function="internal">\n    <lane id=":2506086795_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.80" length="4.22" shape="833.46,1140.00 829.37,1140.20 829.25,1140.26" />\n  </edge>\n  <edge id=":2506086795_6" function="internal">\n    <lane id=":2506086795_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.80" length="10.45" shape="829.25,1140.26 826.49,1141.69 824.81,1144.47 824.33,1148.53" />\n  </edge>\n  <edge id=":2506086795_2" function="internal">\n    <lane id=":2506086795_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.26" length="9.04" shape="827.53,1148.44 827.82,1145.94 828.86,1144.23 830.64,1143.31 833.16,1143.18" />\n  </edge>\n  <edge id=":2506086795_3" function="internal">\n    <lane id=":2506086795_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.39" length="14.41" shape="827.53,1148.44 826.91,1144.45 825.28,1141.49 822.65,1139.59 819.01,1138.73" />\n  </edge>\n  <edge id=":2506086795_4" function="internal">\n    <lane id=":2506086795_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.97" length="9.64" shape="818.75,1141.92 821.13,1142.50 822.85,1143.79 823.92,1145.80 824.33,1148.53" />\n  </edge>\n  <edge id=":2506086795_5" function="internal">\n    <lane id=":2506086795_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.46" shape="818.75,1141.92 833.16,1143.18" />\n  </edge>\n  <edge id=":2506086831_0" function="internal">\n    <lane id=":2506086831_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.49" length="9.03" shape="877.03,1138.45 876.48,1140.87 875.28,1142.52 873.45,1143.40 870.97,1143.52" />\n  </edge>\n  <edge id=":2506086831_1" function="internal">\n    <lane id=":2506086831_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.02" length="14.21" shape="877.03,1138.45 877.25,1142.35 878.66,1145.24 881.26,1147.13 885.04,1148.02" />\n  </edge>\n  <edge id=":2506086831_2" function="internal">\n    <lane id=":2506086831_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.53" length="9.07" shape="885.32,1144.83 882.91,1144.26 881.26,1143.06 880.36,1141.21 880.22,1138.72" />\n  </edge>\n  <edge id=":2506086831_3" function="internal">\n    <lane id=":2506086831_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.41" shape="885.32,1144.83 870.97,1143.52" />\n  </edge>\n  <edge id=":2506086831_4" function="internal">\n    <lane id=":2506086831_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.46" length="9.90" shape="878.48,1153.25 878.75,1150.73 879.94,1149.02 882.03,1148.12 885.04,1148.02" />\n  </edge>\n  <edge id=":2506086831_5" function="internal">\n    <lane id=":2506086831_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.66" shape="878.48,1153.25 878.56,1149.58 879.08,1145.98 879.73,1142.39 880.22,1138.72" />\n  </edge>\n  <edge id=":2506086831_6" function="internal">\n    <lane id=":2506086831_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.30" length="13.54" shape="878.48,1153.25 877.80,1149.24 876.32,1146.28 874.05,1144.37 870.97,1143.52" />\n  </edge>\n  <edge id=":2506086831_7" function="internal">\n    <lane id=":2506086831_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="870.67,1146.71 885.04,1148.02" />\n  </edge>\n  <edge id=":2506086831_8" function="internal">\n    <lane id=":2506086831_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.98" length="4.09" shape="870.67,1146.71 874.57,1146.52 874.74,1146.44" />\n  </edge>\n  <edge id=":2506086831_9" function="internal">\n    <lane id=":2506086831_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.98" length="10.14" shape="874.74,1146.44 877.46,1145.12 879.35,1142.52 880.22,1138.72" />\n  </edge>\n  <edge id=":2506087136_0" function="internal">\n    <lane id=":2506087136_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.37" length="9.04" shape="1284.02,1173.30 1283.61,1175.77 1282.50,1177.46 1280.70,1178.37 1278.21,1178.52" />\n  </edge>\n  <edge id=":2506087136_1" function="internal">\n    <lane id=":2506087136_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="21.15" shape="1284.02,1173.30 1283.43,1194.44" />\n  </edge>\n  <edge id=":2506087136_2" function="internal">\n    <lane id=":2506087136_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.98" length="17.37" shape="1284.02,1173.30 1284.46,1178.65 1286.06,1182.57 1288.81,1185.06 1292.73,1186.13" />\n  </edge>\n  <edge id=":2506087136_3" function="internal">\n    <lane id=":2506087136_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.91" length="9.87" shape="1293.23,1179.75 1290.55,1179.17 1288.65,1177.92 1287.54,1175.98 1287.22,1173.37" />\n  </edge>\n  <edge id=":2506087136_4" function="internal">\n    <lane id=":2506087136_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.08" shape="1293.23,1179.75 1278.21,1178.52" />\n    <lane id=":2506087136_4_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.08" shape="1292.98,1182.94 1277.93,1181.70" />\n  </edge>\n  <edge id=":2506087136_6" function="internal">\n    <lane id=":2506087136_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.60" length="3.73" shape="1292.98,1182.94 1289.27,1183.35" />\n  </edge>\n  <edge id=":2506087136_14" function="internal">\n    <lane id=":2506087136_14_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.60" length="13.38" shape="1289.27,1183.35 1288.95,1183.39 1286.02,1185.46 1284.18,1189.14 1283.43,1194.44" />\n  </edge>\n  <edge id=":2506087136_7" function="internal">\n    <lane id=":2506087136_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.41" length="9.04" shape="1286.63,1194.55 1287.07,1192.09 1288.19,1190.41 1289.99,1189.48 1292.48,1189.32" />\n  </edge>\n  <edge id=":2506087136_8" function="internal">\n    <lane id=":2506087136_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="21.18" shape="1286.63,1194.55 1287.22,1173.37" />\n  </edge>\n  <edge id=":2506087136_9" function="internal">\n    <lane id=":2506087136_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.98" length="17.40" shape="1286.63,1194.55 1286.24,1189.22 1284.66,1185.31 1281.89,1182.80 1277.93,1181.70" />\n  </edge>\n  <edge id=":2506087136_10" function="internal">\n    <lane id=":2506087136_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.91" length="9.90" shape="1277.39,1188.08 1280.10,1188.68 1282.02,1189.94 1283.13,1191.86 1283.43,1194.44" />\n  </edge>\n  <edge id=":2506087136_11" function="internal">\n    <lane id=":2506087136_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.13" shape="1277.39,1188.08 1292.48,1189.32" />\n    <lane id=":2506087136_11_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.13" shape="1277.66,1184.89 1292.73,1186.13" />\n  </edge>\n  <edge id=":2506087136_13" function="internal">\n    <lane id=":2506087136_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.57" length="3.87" shape="1277.66,1184.89 1281.51,1184.49" />\n  </edge>\n  <edge id=":2506087136_15" function="internal">\n    <lane id=":2506087136_15_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.57" length="13.34" shape="1281.51,1184.49 1281.74,1184.47 1284.69,1182.41 1286.52,1178.71 1287.22,1173.37" />\n  </edge>\n  <edge id=":2506087304_0" function="internal">\n    <lane id=":2506087304_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.59" length="9.11" shape="429.93,1190.14 429.09,1192.46 427.70,1193.97 425.75,1194.68 423.24,1194.58" />\n  </edge>\n  <edge id=":2506087304_1" function="internal">\n    <lane id=":2506087304_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.37" shape="429.93,1190.14 427.18,1204.24" />\n  </edge>\n  <edge id=":2506087304_2" function="internal">\n    <lane id=":2506087304_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.30" shape="430.32,1204.82 433.07,1190.78" />\n  </edge>\n  <edge id=":2506087304_3" function="internal">\n    <lane id=":2506087304_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="3.90" shape="430.32,1204.82 430.48,1200.92" />\n  </edge>\n  <edge id=":2506087304_6" function="internal">\n    <lane id=":2506087304_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="10.31" shape="430.48,1200.92 429.35,1197.92 426.94,1195.80 423.24,1194.58" />\n  </edge>\n  <edge id=":2506087304_4" function="internal">\n    <lane id=":2506087304_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.50" length="9.03" shape="422.67,1197.73 425.03,1198.51 426.56,1199.85 427.28,1201.76 427.18,1204.24" />\n  </edge>\n  <edge id=":2506087304_5" function="internal">\n    <lane id=":2506087304_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="14.17" shape="422.67,1197.73 426.56,1197.88 429.59,1196.77 431.76,1194.40 433.07,1190.78" />\n  </edge>\n  <edge id=":2506087432_0" function="internal">\n    <lane id=":2506087432_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.50" length="9.03" shape="640.36,1220.98 639.96,1223.42 638.87,1225.15 637.10,1226.16 634.64,1226.45" />\n  </edge>\n  <edge id=":2506087432_1" function="internal">\n    <lane id=":2506087432_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.48" shape="640.36,1220.98 640.54,1224.63 641.06,1228.18 641.60,1231.72 641.87,1235.37" />\n  </edge>\n  <edge id=":2506087432_2" function="internal">\n    <lane id=":2506087432_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.03" length="10.45" shape="634.57,1229.65 637.74,1230.07 640.02,1231.17 641.39,1232.93 641.87,1235.37" />\n  </edge>\n  <edge id=":2506087432_3" function="internal">\n    <lane id=":2506087432_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="14.20" shape="634.57,1229.65 638.43,1229.19 641.22,1227.60 642.93,1224.89 643.56,1221.05" />\n  </edge>\n  <edge id=":2506087522_0" function="internal">\n    <lane id=":2506087522_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.39" shape="366.48,1243.17 356.20,1241.67" />\n  </edge>\n  <edge id=":2506087522_1" function="internal">\n    <lane id=":2506087522_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.78" length="9.67" shape="366.48,1243.17 363.17,1243.09 361.15,1243.84 360.40,1245.41 360.95,1247.82" />\n  </edge>\n  <edge id=":2506087522_2" function="internal">\n    <lane id=":2506087522_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.68" length="1.64" shape="363.90,1246.58 363.03,1245.18" />\n  </edge>\n  <edge id=":2506087522_3" function="internal">\n    <lane id=":2506087522_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.68" length="7.87" shape="363.03,1245.18 362.81,1244.83 361.17,1243.43 358.96,1242.38 356.20,1241.67" />\n  </edge>\n  <edge id=":2506087563_0" function="internal">\n    <lane id=":2506087563_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.18" length="10.75" shape="420.16,1242.31 419.22,1245.52 417.76,1247.71 415.80,1248.89 413.32,1249.05" />\n  </edge>\n  <edge id=":2506087563_1" function="internal">\n    <lane id=":2506087563_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.87" length="14.50" shape="420.16,1242.31 420.00,1246.33 421.20,1249.33 423.74,1251.32 427.64,1252.30" />\n  </edge>\n  <edge id=":2506087563_2" function="internal">\n    <lane id=":2506087563_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.34" length="9.04" shape="427.97,1249.12 425.54,1248.51 423.95,1247.27 423.21,1245.39 423.30,1242.89" />\n  </edge>\n  <edge id=":2506087563_3" function="internal">\n    <lane id=":2506087563_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.66" shape="427.97,1249.12 424.28,1248.98 420.64,1249.13 417.00,1249.25 413.32,1249.05" />\n  </edge>\n  <edge id=":2506087709_0" function="internal">\n    <lane id=":2506087709_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.78" length="9.16" shape="301.36,1293.30 300.85,1290.66 301.42,1288.86 303.05,1287.91 305.74,1287.82" />\n  </edge>\n  <edge id=":2506087732_0" function="internal">\n    <lane id=":2506087732_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.07" length="8.40" shape="429.60,1302.75 429.77,1300.41 429.13,1298.67 427.67,1297.51 425.40,1296.94" />\n  </edge>\n  <edge id=":2506087732_1" function="internal">\n    <lane id=":2506087732_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="3.90" length="2.80" shape="425.08,1300.12 425.83,1300.31 426.32,1300.70 426.53,1301.28 426.47,1302.06" />\n  </edge>\n  <edge id=":2506087810_0" function="internal">\n    <lane id=":2506087810_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.14" shape="489.21,1292.04 487.84,1303.10" />\n  </edge>\n  <edge id=":2506087810_1" function="internal">\n    <lane id=":2506087810_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="9.00" shape="489.21,1292.04 489.27,1294.50 490.10,1296.36 491.70,1297.62 494.08,1298.27" />\n  </edge>\n  <edge id=":2506087863_0" function="internal">\n    <lane id=":2506087863_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.99" length="9.82" shape="572.02,1306.23 569.22,1305.65 567.22,1304.46 566.00,1302.66 565.58,1300.24" />\n  </edge>\n  <edge id=":2506087863_1" function="internal">\n    <lane id=":2506087863_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.13" length="9.06" shape="566.49,1314.83 566.57,1312.28 567.48,1310.52 569.20,1309.57 571.76,1309.42" />\n  </edge>\n  <edge id=":2506087863_2" function="internal">\n    <lane id=":2506087863_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.63" shape="566.49,1314.83 566.12,1311.16 565.84,1307.54 565.66,1303.93 565.58,1300.24" />\n  </edge>\n  <edge id=":2506087863_3" function="internal">\n    <lane id=":2506087863_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.25" shape="559.86,1306.57 563.11,1307.22 565.80,1308.07 568.49,1308.88 571.76,1309.42" />\n  </edge>\n  <edge id=":2506087863_4" function="internal">\n    <lane id=":2506087863_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.52" length="10.02" shape="559.86,1306.57 562.39,1306.45 564.19,1305.35 565.25,1303.28 565.58,1300.24" />\n  </edge>\n  <edge id=":2506087889_0" function="internal">\n    <lane id=":2506087889_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.78" length="9.44" shape="636.48,1314.82 639.09,1315.39 640.93,1316.57 642.02,1318.39 642.34,1320.83" />\n  </edge>\n  <edge id=":2506087889_1" function="internal">\n    <lane id=":2506087889_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.32" length="9.04" shape="642.43,1306.36 642.09,1308.85 641.03,1310.56 639.25,1311.48 636.74,1311.63" />\n  </edge>\n  <edge id=":2506087889_2" function="internal">\n    <lane id=":2506087889_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.47" shape="642.43,1306.36 642.34,1320.83" />\n  </edge>\n  <edge id=":2506087955_0" function="internal">\n    <lane id=":2506087955_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.21" length="9.87" shape="575.20,1336.49 572.39,1335.99 570.30,1334.88 568.92,1333.15 568.27,1330.80" />\n  </edge>\n  <edge id=":2506087955_1" function="internal">\n    <lane id=":2506087955_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.22" length="9.05" shape="569.67,1345.17 569.82,1342.64 570.76,1340.88 572.49,1339.90 575.01,1339.68" />\n  </edge>\n  <edge id=":2506087955_2" function="internal">\n    <lane id=":2506087955_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.43" shape="569.67,1345.17 568.27,1330.80" />\n  </edge>\n  <edge id=":2506087962_0" function="internal">\n    <lane id=":2506087962_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.31" length="9.04" shape="946.19,1339.58 943.70,1339.89 941.78,1339.30 940.43,1337.80 939.66,1335.41" />\n  </edge>\n  <edge id=":2506087962_1" function="internal">\n    <lane id=":2506087962_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.44" shape="946.19,1339.58 932.20,1343.19" />\n  </edge>\n  <edge id=":2506087962_2" function="internal">\n    <lane id=":2506087962_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.49" shape="933.00,1346.29 947.03,1342.67" />\n  </edge>\n  <edge id=":2506087962_3" function="internal">\n    <lane id=":2506087962_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.22" length="4.06" shape="933.00,1346.29 936.49,1344.89 936.71,1344.68" />\n  </edge>\n  <edge id=":2506087962_6" function="internal">\n    <lane id=":2506087962_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.22" length="10.28" shape="936.71,1344.68 938.76,1342.61 939.82,1339.46 939.66,1335.41" />\n  </edge>\n  <edge id=":2506087962_4" function="internal">\n    <lane id=":2506087962_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.79" length="9.47" shape="936.50,1335.96 936.62,1338.65 935.94,1340.76 934.47,1342.27 932.20,1343.19" />\n  </edge>\n  <edge id=":2506087962_5" function="internal">\n    <lane id=":2506087962_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.84" length="14.57" shape="936.50,1335.96 937.75,1339.81 939.92,1342.21 943.01,1343.16 947.03,1342.67" />\n  </edge>\n  <edge id=":2506087968_0" function="internal">\n    <lane id=":2506087968_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.48" length="9.19" shape="642.04,1334.74 641.62,1337.19 640.49,1338.90 638.64,1339.85 636.08,1340.06" />\n  </edge>\n  <edge id=":2506087968_1" function="internal">\n    <lane id=":2506087968_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.61" shape="642.04,1334.74 641.90,1338.43 641.64,1342.03 641.27,1345.63 640.76,1349.28" />\n  </edge>\n  <edge id=":2506087968_2" function="internal">\n    <lane id=":2506087968_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.31" length="9.04" shape="635.89,1343.25 638.35,1343.75 639.98,1344.93 640.78,1346.77 640.76,1349.28" />\n  </edge>\n  <edge id=":2506087987_0" function="internal">\n    <lane id=":2506087987_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.36" length="9.04" shape="415.92,1349.88 415.04,1352.22 413.62,1353.66 411.68,1354.20 409.21,1353.85" />\n  </edge>\n  <edge id=":2506087987_1" function="internal">\n    <lane id=":2506087987_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.56" shape="415.92,1349.88 412.37,1364.01" />\n  </edge>\n  <edge id=":2506087987_2" function="internal">\n    <lane id=":2506087987_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.29" length="14.50" shape="415.92,1349.88 415.60,1353.85 416.57,1357.03 418.83,1359.43 422.37,1361.06" />\n  </edge>\n  <edge id=":2506087987_3" function="internal">\n    <lane id=":2506087987_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.85" length="9.59" shape="423.27,1357.99 420.94,1356.92 419.46,1355.32 418.83,1353.21 419.04,1350.57" />\n  </edge>\n  <edge id=":2506087987_4" function="internal">\n    <lane id=":2506087987_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.66" shape="423.27,1357.99 409.21,1353.85" />\n  </edge>\n  <edge id=":2506087987_5" function="internal">\n    <lane id=":2506087987_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.98" length="4.07" shape="423.27,1357.99 419.41,1357.43 419.24,1357.47" />\n  </edge>\n  <edge id=":2506087987_12" function="internal">\n    <lane id=":2506087987_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.98" length="10.16" shape="419.24,1357.47 416.30,1358.24 413.96,1360.44 412.37,1364.01" />\n  </edge>\n  <edge id=":2506087987_6" function="internal">\n    <lane id=":2506087987_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.48" length="9.03" shape="415.45,1364.88 416.46,1362.61 417.94,1361.22 419.91,1360.70 422.37,1361.06" />\n  </edge>\n  <edge id=":2506087987_7" function="internal">\n    <lane id=":2506087987_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.75" shape="415.45,1364.88 419.04,1350.57" />\n  </edge>\n  <edge id=":2506087987_8" function="internal">\n    <lane id=":2506087987_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.09" length="14.42" shape="415.45,1364.88 415.96,1361.01 415.09,1357.88 412.84,1355.49 409.21,1353.85" />\n  </edge>\n  <edge id=":2506087987_9" function="internal">\n    <lane id=":2506087987_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.62" length="9.30" shape="408.30,1356.92 410.66,1357.98 412.12,1359.51 412.69,1361.52 412.37,1364.01" />\n  </edge>\n  <edge id=":2506087987_10" function="internal">\n    <lane id=":2506087987_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.66" shape="408.30,1356.92 422.37,1361.06" />\n  </edge>\n  <edge id=":2506087987_11" function="internal">\n    <lane id=":2506087987_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.88" length="4.13" shape="408.30,1356.92 412.26,1357.49 412.38,1357.45" />\n  </edge>\n  <edge id=":2506087987_13" function="internal">\n    <lane id=":2506087987_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.88" length="10.34" shape="412.38,1357.45 415.37,1356.62 417.63,1354.32 419.04,1350.57" />\n  </edge>\n  <edge id=":2506088002_0" function="internal">\n    <lane id=":2506088002_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.18" length="9.05" shape="840.83,1365.04 838.30,1365.28 836.43,1364.61 835.22,1363.04 834.67,1360.56" />\n  </edge>\n  <edge id=":2506088002_1" function="internal">\n    <lane id=":2506088002_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.33" shape="840.83,1365.04 837.30,1365.94 833.92,1366.91 830.56,1367.97 827.10,1369.13" />\n  </edge>\n  <edge id=":2506088002_2" function="internal">\n    <lane id=":2506088002_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.90" shape="828.27,1372.11 831.66,1370.97 834.88,1369.96 838.12,1369.03 841.59,1368.15" />\n  </edge>\n  <edge id=":2506088002_3" function="internal">\n    <lane id=":2506088002_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.91" length="14.33" shape="828.27,1372.11 831.33,1370.47 833.41,1367.99 834.53,1364.69 834.67,1360.56" />\n  </edge>\n  <edge id=":2506088002_4" function="internal">\n    <lane id=":2506088002_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.60" length="10.20" shape="831.48,1360.80 831.39,1363.81 830.64,1366.20 829.21,1367.98 827.10,1369.13" />\n  </edge>\n  <edge id=":2506088002_5" function="internal">\n    <lane id=":2506088002_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.74" length="14.85" shape="831.48,1360.80 832.38,1364.87 834.37,1367.45 837.44,1368.54 841.59,1368.15" />\n  </edge>\n  <edge id=":2506088023_0" function="internal">\n    <lane id=":2506088023_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.53" length="9.03" shape="1404.11,1385.38 1401.64,1385.20 1399.82,1384.27 1398.63,1382.61 1398.09,1380.20" />\n  </edge>\n  <edge id=":2506088023_1" function="internal">\n    <lane id=":2506088023_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="14.16" shape="1404.11,1385.38 1400.32,1386.19 1397.69,1388.01 1396.23,1390.86 1395.93,1394.73" />\n  </edge>\n  <edge id=":2506088023_2" function="internal">\n    <lane id=":2506088023_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="9.01" shape="1399.12,1394.52 1399.31,1392.06 1400.24,1390.25 1401.92,1389.08 1404.33,1388.57" />\n  </edge>\n  <edge id=":2506088023_3" function="internal">\n    <lane id=":2506088023_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.35" shape="1399.12,1394.52 1398.09,1380.20" />\n  </edge>\n  <edge id=":2506088023_4" function="internal">\n    <lane id=":2506088023_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.31" shape="1394.90,1380.46 1395.93,1394.73" />\n  </edge>\n  <edge id=":2506088023_5" function="internal">\n    <lane id=":2506088023_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.02" length="4.00" shape="1394.90,1380.46 1395.75,1384.23 1395.82,1384.34" />\n  </edge>\n  <edge id=":2506088023_6" function="internal">\n    <lane id=":2506088023_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.02" length="10.15" shape="1395.82,1384.34 1397.60,1386.84 1400.46,1388.29 1404.33,1388.57" />\n  </edge>\n  <edge id=":2506088024_0" function="internal">\n    <lane id=":2506088024_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.22" length="10.14" shape="783.76,1384.21 780.62,1384.91 778.36,1384.52 776.98,1383.05 776.49,1380.49" />\n  </edge>\n  <edge id=":2506088024_1" function="internal">\n    <lane id=":2506088024_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.17" shape="783.76,1384.21 779.87,1385.48 775.56,1386.74 771.20,1387.85 767.20,1388.68" />\n  </edge>\n  <edge id=":2506088024_2" function="internal">\n    <lane id=":2506088024_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.99" length="17.22" shape="783.76,1384.21 779.30,1386.32 776.14,1389.10 774.27,1392.54 773.71,1396.66" />\n  </edge>\n  <edge id=":2506088024_3" function="internal">\n    <lane id=":2506088024_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.46" length="9.87" shape="784.58,1393.12 781.82,1391.54 780.94,1390.03 781.94,1388.59 784.81,1387.23" />\n  </edge>\n  <edge id=":2506088024_4" function="internal">\n    <lane id=":2506088024_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.19" length="15.93" shape="784.58,1393.12 781.10,1391.03 778.58,1388.22 777.05,1384.71 776.49,1380.49" />\n  </edge>\n  <edge id=":2506088024_5" function="internal">\n    <lane id=":2506088024_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.13" shape="784.58,1393.12 780.63,1391.46 776.02,1389.83 771.35,1388.73 767.20,1388.68" />\n  </edge>\n  <edge id=":2506088024_6" function="internal">\n    <lane id=":2506088024_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.10" length="14.55" shape="784.58,1393.12 779.77,1391.53 776.35,1391.59 774.33,1393.30 773.71,1396.66" />\n  </edge>\n  <edge id=":2506088024_7" function="internal">\n    <lane id=":2506088024_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.78" length="13.12" shape="776.91,1396.53 777.36,1393.51 778.82,1390.95 781.31,1388.86 784.81,1387.23" />\n  </edge>\n  <edge id=":2506088024_8" function="internal">\n    <lane id=":2506088024_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.05" shape="776.91,1396.53 776.49,1380.49" />\n  </edge>\n  <edge id=":2506088024_9" function="internal">\n    <lane id=":2506088024_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.75" length="4.22" shape="776.91,1396.53 776.23,1392.47 776.17,1392.39" />\n  </edge>\n  <edge id=":2506088024_16" function="internal">\n    <lane id=":2506088024_16_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.75" length="10.58" shape="776.17,1392.39 774.39,1389.81 771.38,1388.55 767.20,1388.68" />\n  </edge>\n  <edge id=":2506088024_10" function="internal">\n    <lane id=":2506088024_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.21" length="9.09" shape="767.75,1391.83 770.32,1391.75 772.16,1392.53 773.30,1394.16 773.71,1396.66" />\n  </edge>\n  <edge id=":2506088024_11" function="internal">\n    <lane id=":2506088024_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.69" shape="767.75,1391.83 771.83,1390.98 776.36,1389.82 780.85,1388.52 784.81,1387.23" />\n  </edge>\n  <edge id=":2506088024_12" function="internal">\n    <lane id=":2506088024_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.94" length="15.82" shape="767.75,1391.83 771.63,1390.54 774.38,1388.22 776.00,1384.87 776.49,1380.49" />\n  </edge>\n  <edge id=":2506088024_13" function="internal">\n    <lane id=":2506088024_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.62" length="11.23" shape="773.29,1380.54 772.95,1383.69 771.82,1386.10 769.90,1387.77 767.20,1388.68" />\n  </edge>\n  <edge id=":2506088024_14" function="internal">\n    <lane id=":2506088024_14_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.13" shape="773.29,1380.54 773.71,1396.66" />\n  </edge>\n  <edge id=":2506088024_15" function="internal">\n    <lane id=":2506088024_15_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.78" length="4.36" shape="773.29,1380.54 774.05,1384.83" />\n  </edge>\n  <edge id=":2506088024_17" function="internal">\n    <lane id=":2506088024_17_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.78" length="12.25" shape="774.05,1384.83 774.07,1384.95 776.25,1387.53 779.83,1388.29 784.81,1387.23" />\n  </edge>\n  <edge id=":2506088084_0" function="internal">\n    <lane id=":2506088084_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.87" length="8.34" shape="726.41,1391.18 725.90,1393.53 724.92,1395.30 723.46,1396.48 721.52,1397.08" />\n  </edge>\n  <edge id=":2506088084_1" function="internal">\n    <lane id=":2506088084_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.52" length="11.09" shape="726.41,1391.18 726.42,1394.86 727.30,1397.33 729.06,1398.57 731.69,1398.58" />\n  </edge>\n  <edge id=":2506088084_2" function="internal">\n    <lane id=":2506088084_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.71" shape="731.08,1395.44 721.52,1397.08" />\n  </edge>\n  <edge id=":2506088084_3" function="internal">\n    <lane id=":2506088084_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.88" shape="721.96,1400.25 731.69,1398.58" />\n  </edge>\n  <edge id=":2506088243_0" function="internal">\n    <lane id=":2506088243_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.09" length="7.82" shape="774.26,1426.20 774.01,1428.47 773.32,1430.29 772.19,1431.65 770.62,1432.56" />\n  </edge>\n  <edge id=":2506088243_1" function="internal">\n    <lane id=":2506088243_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.70" length="9.75" shape="774.26,1426.20 774.71,1428.12 776.11,1429.49 778.47,1430.31 781.77,1430.56" />\n  </edge>\n  <edge id=":2506088274_0" function="internal">\n    <lane id=":2506088274_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.49" length="9.03" shape="1409.23,1463.75 1406.76,1463.58 1404.94,1462.65 1403.78,1460.98 1403.27,1458.55" />\n  </edge>\n  <edge id=":2506088274_1" function="internal">\n    <lane id=":2506088274_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.08" length="14.17" shape="1409.23,1463.75 1405.42,1464.57 1402.75,1466.38 1401.22,1469.20 1400.83,1473.03" />\n  </edge>\n  <edge id=":2506088274_2" function="internal">\n    <lane id=":2506088274_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.60" length="9.12" shape="1404.03,1472.90 1404.28,1470.44 1405.27,1468.63 1407.00,1467.47 1409.45,1466.95" />\n  </edge>\n  <edge id=":2506088274_3" function="internal">\n    <lane id=":2506088274_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.36" shape="1404.03,1472.90 1403.27,1458.55" />\n  </edge>\n  <edge id=":2506088274_4" function="internal">\n    <lane id=":2506088274_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.29" shape="1400.08,1458.76 1400.83,1473.03" />\n  </edge>\n  <edge id=":2506088274_5" function="internal">\n    <lane id=":2506088274_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="3.90" shape="1400.08,1458.76 1400.88,1462.58" />\n  </edge>\n  <edge id=":2506088274_6" function="internal">\n    <lane id=":2506088274_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="10.32" shape="1400.88,1462.58 1402.71,1465.21 1405.57,1466.67 1409.45,1466.95" />\n  </edge>\n  <edge id=":2506088476_0" function="internal">\n    <lane id=":2506088476_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.86" length="9.12" shape="466.20,1458.33 465.50,1460.89 464.20,1462.28 462.31,1462.52 459.82,1461.60" />\n  </edge>\n  <edge id=":2506088476_1" function="internal">\n    <lane id=":2506088476_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.44" length="15.11" shape="466.20,1458.33 466.07,1462.79 466.93,1466.44 468.78,1469.29 471.62,1471.33" />\n  </edge>\n  <edge id=":2506088476_2" function="internal">\n    <lane id=":2506088476_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.17" length="11.11" shape="473.03,1468.46 471.07,1467.00 469.81,1464.89 469.24,1462.14 469.38,1458.74" />\n  </edge>\n  <edge id=":2506088476_3" function="internal">\n    <lane id=":2506088476_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.88" shape="473.03,1468.46 459.82,1461.60" />\n  </edge>\n  <edge id=":2506088476_4" function="internal">\n    <lane id=":2506088476_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.01" shape="458.30,1464.41 471.62,1471.33" />\n  </edge>\n  <edge id=":2506088476_5" function="internal">\n    <lane id=":2506088476_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.51" length="4.61" shape="458.30,1464.41 462.62,1466.02" />\n  </edge>\n  <edge id=":2506088476_6" function="internal">\n    <lane id=":2506088476_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.51" length="11.23" shape="462.62,1466.02 465.90,1465.61 468.16,1463.18 469.38,1458.74" />\n  </edge>\n  <edge id=":2506088491_0" function="internal">\n    <lane id=":2506088491_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.99" length="9.42" shape="533.18,1484.34 532.63,1486.92 531.47,1488.80 529.70,1490.00 527.32,1490.50" />\n  </edge>\n  <edge id=":2506088491_1" function="internal">\n    <lane id=":2506088491_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.97" length="14.25" shape="533.18,1484.34 533.41,1488.25 534.84,1491.13 537.47,1492.97 541.30,1493.79" />\n  </edge>\n  <edge id=":2506088491_2" function="internal">\n    <lane id=":2506088491_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.47" length="9.03" shape="541.51,1490.60 539.09,1490.08 537.42,1488.91 536.52,1487.09 536.37,1484.61" />\n  </edge>\n  <edge id=":2506088491_3" function="internal">\n    <lane id=":2506088491_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.21" shape="541.51,1490.60 537.89,1490.40 534.42,1490.31 530.94,1490.35 527.32,1490.50" />\n  </edge>\n  <edge id=":2506088491_4" function="internal">\n    <lane id=":2506088491_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.80" shape="527.51,1493.70 531.06,1493.54 534.40,1493.51 537.74,1493.59 541.30,1493.79" />\n  </edge>\n  <edge id=":2506088491_5" function="internal">\n    <lane id=":2506088491_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.41" length="3.80" shape="527.51,1493.70 531.11,1492.95 531.21,1492.89" />\n  </edge>\n  <edge id=":2506088491_6" function="internal">\n    <lane id=":2506088491_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.41" length="10.27" shape="531.21,1492.89 533.79,1491.18 535.55,1488.40 536.37,1484.61" />\n  </edge>\n  <edge id=":2506088492_0" function="internal">\n    <lane id=":2506088492_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.17" length="9.78" shape="1241.60,1495.66 1241.08,1498.04 1239.82,1499.84 1237.82,1501.05 1235.09,1501.68" />\n  </edge>\n  <edge id=":2506088492_1" function="internal">\n    <lane id=":2506088492_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.39" shape="1241.60,1495.66 1241.14,1510.04" />\n  </edge>\n  <edge id=":2506088492_2" function="internal">\n    <lane id=":2506088492_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.25" shape="1244.34,1510.07 1244.79,1495.83" />\n  </edge>\n  <edge id=":2506088492_3" function="internal">\n    <lane id=":2506088492_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.80" length="4.09" shape="1244.34,1510.07 1243.80,1506.02" />\n  </edge>\n  <edge id=":2506088492_6" function="internal">\n    <lane id=":2506088492_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.80" length="10.58" shape="1243.80,1506.02 1242.08,1503.27 1239.17,1501.82 1235.09,1501.68" />\n  </edge>\n  <edge id=":2506088492_4" function="internal">\n    <lane id=":2506088492_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.26" length="9.04" shape="1235.43,1504.86 1237.95,1504.95 1239.74,1505.84 1240.80,1507.54 1241.14,1510.04" />\n  </edge>\n  <edge id=":2506088492_5" function="internal">\n    <lane id=":2506088492_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.56" length="14.34" shape="1235.43,1504.86 1239.36,1503.93 1242.23,1502.12 1244.04,1499.42 1244.79,1495.83" />\n  </edge>\n  <edge id=":2506088494_0" function="internal">\n    <lane id=":2506088494_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.73" length="10.29" shape="509.42,1491.57 507.05,1491.36 504.86,1490.45 502.84,1488.84 501.01,1486.53" />\n  </edge>\n  <edge id=":2506088494_1" function="internal">\n    <lane id=":2506088494_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.69" shape="509.42,1491.57 505.91,1492.01 502.74,1492.89 499.71,1494.21 496.60,1495.95" />\n  </edge>\n  <edge id=":2506088494_2" function="internal">\n    <lane id=":2506088494_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.16" shape="498.27,1498.68 501.14,1497.01 503.67,1495.86 506.33,1495.13 509.61,1494.76" />\n  </edge>\n  <edge id=":2506088494_3" function="internal">\n    <lane id=":2506088494_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.95" length="3.80" shape="498.27,1498.68 501.20,1496.26" />\n  </edge>\n  <edge id=":2506088494_6" function="internal">\n    <lane id=":2506088494_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.95" length="10.52" shape="501.20,1496.26 501.31,1496.17 502.78,1493.31 502.68,1490.09 501.01,1486.53" />\n  </edge>\n  <edge id=":2506088494_4" function="internal">\n    <lane id=":2506088494_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.44" length="9.03" shape="498.33,1488.28 499.38,1490.53 499.45,1492.56 498.52,1494.37 496.60,1495.95" />\n  </edge>\n  <edge id=":2506088494_5" function="internal">\n    <lane id=":2506088494_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.98" length="13.65" shape="498.33,1488.28 500.73,1491.27 503.40,1493.34 506.37,1494.51 509.61,1494.76" />\n  </edge>\n  <edge id=":2506088518_0" function="internal">\n    <lane id=":2506088518_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.74" length="9.47" shape="1154.33,1503.07 1153.88,1505.73 1152.76,1507.62 1150.98,1508.76 1148.54,1509.13" />\n  </edge>\n  <edge id=":2506088518_1" function="internal">\n    <lane id=":2506088518_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.80" length="14.66" shape="1154.33,1503.07 1154.75,1507.14 1156.38,1509.94 1159.24,1511.48 1163.31,1511.76" />\n  </edge>\n  <edge id=":2506088518_2" function="internal">\n    <lane id=":2506088518_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.26" length="9.04" shape="1163.07,1508.57 1160.55,1508.40 1158.79,1507.45 1157.78,1505.72 1157.53,1503.21" />\n  </edge>\n  <edge id=":2506088518_3" function="internal">\n    <lane id=":2506088518_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.54" shape="1163.07,1508.57 1148.54,1509.13" />\n  </edge>\n  <edge id=":2506088518_4" function="internal">\n    <lane id=":2506088518_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.79" shape="1148.54,1512.33 1163.31,1511.76" />\n  </edge>\n  <edge id=":2506088518_5" function="internal">\n    <lane id=":2506088518_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.20" length="4.26" shape="1148.54,1512.33 1152.33,1511.77 1152.70,1511.54" />\n  </edge>\n  <edge id=":2506088518_6" function="internal">\n    <lane id=":2506088518_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.20" length="10.21" shape="1152.70,1511.54 1155.09,1510.06 1156.82,1507.20 1157.53,1503.21" />\n  </edge>\n  <edge id=":2506088528_0" function="internal">\n    <lane id=":2506088528_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.57" length="9.28" shape="1411.79,1517.41 1409.19,1517.18 1407.29,1516.22 1406.11,1514.52 1405.64,1512.09" />\n  </edge>\n  <edge id=":2506088528_1" function="internal">\n    <lane id=":2506088528_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.83" length="14.58" shape="1411.79,1517.41 1407.81,1518.18 1405.16,1520.04 1403.84,1523.00 1403.86,1527.06" />\n  </edge>\n  <edge id=":2506088528_2" function="internal">\n    <lane id=":2506088528_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.30" length="9.04" shape="1407.03,1526.59 1407.02,1524.07 1407.83,1522.24 1409.47,1521.08 1411.94,1520.61" />\n  </edge>\n  <edge id=":2506088528_3" function="internal">\n    <lane id=":2506088528_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.57" shape="1407.03,1526.59 1406.53,1522.94 1406.14,1519.36 1405.84,1515.77 1405.64,1512.09" />\n  </edge>\n  <edge id=":2506088528_4" function="internal">\n    <lane id=":2506088528_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.91" shape="1402.44,1512.22 1402.65,1515.95 1402.95,1519.66 1403.36,1523.35 1403.86,1527.06" />\n  </edge>\n  <edge id=":2506088528_5" function="internal">\n    <lane id=":2506088528_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.05" length="4.13" shape="1402.44,1512.22 1403.17,1516.05 1403.30,1516.24" />\n  </edge>\n  <edge id=":2506088528_6" function="internal">\n    <lane id=":2506088528_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.05" length="10.34" shape="1403.30,1516.24 1405.00,1518.73 1407.92,1520.25 1411.94,1520.61" />\n  </edge>\n  <edge id=":2506088529_0" function="internal">\n    <lane id=":2506088529_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.11" shape="1116.60,1524.31 1114.72,1513.36" />\n  </edge>\n  <edge id=":2506088529_1" function="internal">\n    <lane id=":2506088529_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="4.10" shape="1110.15,1519.79 1112.48,1519.03 1113.73,1517.96" />\n  </edge>\n  <edge id=":2506088529_2" function="internal">\n    <lane id=":2506088529_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="4.88" shape="1113.73,1517.96 1114.03,1517.71 1114.77,1515.82 1114.72,1513.36" />\n  </edge>\n  <edge id=":2506088612_0" function="internal">\n    <lane id=":2506088612_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.46" length="9.03" shape="1066.53,1533.06 1066.40,1530.58 1067.10,1528.67 1068.63,1527.33 1071.00,1526.57" />\n  </edge>\n  <edge id=":2506088612_1" function="internal">\n    <lane id=":2506088612_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.25" shape="1066.53,1533.06 1064.52,1522.00" />\n  </edge>\n  <edge id=":2506088612_2" function="internal">\n    <lane id=":2506088612_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.25" shape="1059.94,1528.65 1071.00,1526.57" />\n  </edge>\n  <edge id=":2506088612_3" function="internal">\n    <lane id=":2506088612_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.62" length="4.21" shape="1059.94,1528.65 1062.29,1527.81 1063.57,1526.67" />\n  </edge>\n  <edge id=":2506088612_4" function="internal">\n    <lane id=":2506088612_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.62" length="4.92" shape="1063.57,1526.67 1063.83,1526.43 1064.58,1524.49 1064.52,1522.00" />\n  </edge>\n  <edge id=":2506088715_0" function="internal">\n    <lane id=":2506088715_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.28" shape="1011.97,1532.49 1013.88,1543.65" />\n    <lane id=":2506088715_0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.28" shape="1015.13,1531.99 1017.03,1543.07" />\n  </edge>\n  <edge id=":2506088715_2" function="internal">\n    <lane id=":2506088715_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.41" length="9.04" shape="1015.13,1531.99 1015.87,1534.37 1017.19,1535.91 1019.09,1536.60 1021.58,1536.46" />\n  </edge>\n  <edge id=":2506088938_0" function="internal">\n    <lane id=":2506088938_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.41" shape="586.35,1599.59 583.04,1585.57" />\n  </edge>\n  <edge id=":2506088938_1" function="internal">\n    <lane id=":2506088938_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.21" length="3.97" shape="586.35,1599.59 584.94,1596.06 584.82,1595.94" />\n  </edge>\n  <edge id=":2506088938_6" function="internal">\n    <lane id=":2506088938_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.21" length="10.32" shape="584.82,1595.94 582.69,1593.76 579.58,1592.67 575.61,1592.82" />\n  </edge>\n  <edge id=":2506088938_2" function="internal">\n    <lane id=":2506088938_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.76" length="9.35" shape="576.15,1595.97 578.78,1595.87 580.83,1596.56 582.32,1598.05 583.23,1600.33" />\n  </edge>\n  <edge id=":2506088938_3" function="internal">\n    <lane id=":2506088938_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.89" length="14.43" shape="576.15,1595.97 579.95,1594.74 582.36,1592.60 583.39,1589.54 583.04,1585.57" />\n  </edge>\n  <edge id=":2506088938_4" function="internal">\n    <lane id=":2506088938_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.38" length="9.04" shape="579.92,1586.30 580.14,1588.79 579.50,1590.71 577.99,1592.05 575.61,1592.82" />\n  </edge>\n  <edge id=":2506088938_5" function="internal">\n    <lane id=":2506088938_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.41" shape="579.92,1586.30 583.23,1600.33" />\n  </edge>\n  <edge id=":2506089058_0" function="internal">\n    <lane id=":2506089058_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.41" shape="591.66,1622.16 588.36,1608.14" />\n  </edge>\n  <edge id=":2506089058_1" function="internal">\n    <lane id=":2506089058_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="4.08" shape="591.66,1622.16 590.25,1618.54 590.12,1618.40" />\n  </edge>\n  <edge id=":2506089058_6" function="internal">\n    <lane id=":2506089058_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="10.12" shape="590.12,1618.40 588.01,1616.23 584.96,1615.26 581.08,1615.61" />\n  </edge>\n  <edge id=":2506089058_2" function="internal">\n    <lane id=":2506089058_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.50" length="9.03" shape="581.81,1618.72 584.28,1618.50 586.22,1619.12 587.64,1620.58 588.54,1622.89" />\n  </edge>\n  <edge id=":2506089058_3" function="internal">\n    <lane id=":2506089058_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="14.20" shape="581.81,1618.72 585.44,1617.30 587.74,1615.06 588.71,1612.01 588.36,1608.14" />\n  </edge>\n  <edge id=":2506089058_4" function="internal">\n    <lane id=":2506089058_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="9.04" shape="585.25,1608.87 585.47,1611.34 584.85,1613.28 583.39,1614.70 581.08,1615.61" />\n  </edge>\n  <edge id=":2506089058_5" function="internal">\n    <lane id=":2506089058_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="585.25,1608.87 588.54,1622.89" />\n  </edge>\n  <edge id=":2506089309_0" function="internal">\n    <lane id=":2506089309_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.11" shape="1039.45,1637.56 1036.74,1639.16 1034.70,1640.46 1032.78,1641.91 1030.45,1644.01" />\n  </edge>\n  <edge id=":2506089309_1" function="internal">\n    <lane id=":2506089309_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.41" shape="1032.65,1646.33 1034.89,1644.30 1036.62,1642.99 1038.47,1641.85 1041.09,1640.31" />\n  </edge>\n  <edge id=":2506089309_2" function="internal">\n    <lane id=":2506089309_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.94" length="8.23" shape="1031.50,1636.18 1031.91,1638.78 1031.87,1640.96 1031.38,1642.70 1030.45,1644.01" />\n  </edge>\n  <edge id=":2506089309_3" function="internal">\n    <lane id=":2506089309_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.15" length="9.89" shape="1034.62,1635.48 1035.67,1638.66 1037.10,1640.52 1038.90,1641.07 1041.09,1640.31" />\n  </edge>\n  <edge id=":2506089487_0" function="internal">\n    <lane id=":2506089487_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.41" shape="600.48,1659.75 597.17,1645.72" />\n  </edge>\n  <edge id=":2506089487_1" function="internal">\n    <lane id=":2506089487_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.85" length="4.04" shape="600.48,1659.75 599.00,1655.99" />\n  </edge>\n  <edge id=":2506089487_6" function="internal">\n    <lane id=":2506089487_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.85" length="10.51" shape="599.00,1655.99 596.68,1653.73 593.54,1652.96 589.57,1653.68" />\n  </edge>\n  <edge id=":2506089487_2" function="internal">\n    <lane id=":2506089487_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.32" length="9.04" shape="590.58,1656.71 593.05,1656.27 595.01,1656.75 596.44,1658.15 597.37,1660.48" />\n  </edge>\n  <edge id=":2506089487_3" function="internal">\n    <lane id=":2506089487_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.32" length="14.33" shape="590.58,1656.71 594.18,1654.96 596.48,1652.55 597.48,1649.47 597.17,1645.72" />\n  </edge>\n  <edge id=":2506089487_4" function="internal">\n    <lane id=":2506089487_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.88" length="9.51" shape="594.06,1646.45 594.24,1648.89 593.56,1650.91 592.00,1652.51 589.57,1653.68" />\n  </edge>\n  <edge id=":2506089487_5" function="internal">\n    <lane id=":2506089487_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="594.06,1646.45 597.37,1660.48" />\n  </edge>\n  <edge id=":2506089545_0" function="internal">\n    <lane id=":2506089545_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.73" shape="1177.46,1661.98 1163.03,1664.93" />\n  </edge>\n  <edge id=":2506089545_1" function="internal">\n    <lane id=":2506089545_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.23" length="3.71" shape="1177.46,1661.98 1174.16,1663.22 1174.02,1663.35" />\n  </edge>\n  <edge id=":2506089545_6" function="internal">\n    <lane id=":2506089545_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.23" length="11.15" shape="1174.02,1663.35 1171.67,1665.47 1169.99,1668.71 1169.12,1672.96" />\n  </edge>\n  <edge id=":2506089545_2" function="internal">\n    <lane id=":2506089545_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.93" length="10.76" shape="1172.30,1673.27 1172.93,1670.09 1174.10,1667.66 1175.82,1666.01 1178.09,1665.11" />\n  </edge>\n  <edge id=":2506089545_3" function="internal">\n    <lane id=":2506089545_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.57" length="15.50" shape="1172.30,1673.27 1172.11,1668.82 1170.50,1665.95 1167.47,1664.65 1163.03,1664.93" />\n  </edge>\n  <edge id=":2506089545_4" function="internal">\n    <lane id=":2506089545_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.95" length="9.09" shape="1163.68,1668.07 1166.28,1667.90 1168.06,1668.66 1169.01,1670.35 1169.12,1672.96" />\n  </edge>\n  <edge id=":2506089545_5" function="internal">\n    <lane id=":2506089545_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.71" shape="1163.68,1668.07 1178.09,1665.11" />\n  </edge>\n  <edge id=":2506089600_0" function="internal">\n    <lane id=":2506089600_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.77" shape="1123.26,1673.19 1108.84,1676.37" />\n  </edge>\n  <edge id=":2506089600_1" function="internal">\n    <lane id=":2506089600_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.11" length="3.79" shape="1123.26,1673.19 1119.95,1674.47 1119.78,1674.63" />\n  </edge>\n  <edge id=":2506089600_6" function="internal">\n    <lane id=":2506089600_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.11" length="11.06" shape="1119.78,1674.63 1117.49,1676.74 1115.89,1680.00 1115.15,1684.25" />\n  </edge>\n  <edge id=":2506089600_2" function="internal">\n    <lane id=":2506089600_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.80" length="10.65" shape="1118.34,1684.46 1118.86,1681.30 1119.97,1678.89 1121.65,1677.23 1123.91,1676.33" />\n  </edge>\n  <edge id=":2506089600_3" function="internal">\n    <lane id=":2506089600_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.58" length="15.44" shape="1118.34,1684.46 1118.00,1680.04 1116.31,1677.22 1113.25,1676.00 1108.84,1676.37" />\n  </edge>\n  <edge id=":2506089600_4" function="internal">\n    <lane id=":2506089600_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.97" length="9.09" shape="1109.56,1679.49 1112.15,1679.27 1113.95,1679.99 1114.95,1681.65 1115.15,1684.25" />\n  </edge>\n  <edge id=":2506089600_5" function="internal">\n    <lane id=":2506089600_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.70" shape="1109.56,1679.49 1123.91,1676.33" />\n  </edge>\n  <edge id=":2506089779_0" function="internal">\n    <lane id=":2506089779_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.97" shape="1025.36,1705.19 1022.50,1702.80 1019.85,1700.16 1017.23,1697.47 1014.47,1694.95" />\n  </edge>\n  <edge id=":2506089779_1" function="internal">\n    <lane id=":2506089779_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.49" length="4.57" shape="1025.36,1705.19 1022.18,1703.27 1021.33,1703.24" />\n  </edge>\n  <edge id=":2506089779_5" function="internal">\n    <lane id=":2506089779_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.49" length="6.48" shape="1021.33,1703.24 1020.50,1703.20 1020.31,1704.99 1021.61,1708.62" />\n  </edge>\n  <edge id=":2506089779_2" function="internal">\n    <lane id=":2506089779_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.16" length="11.53" shape="1013.73,1705.43 1015.64,1703.41 1017.59,1703.26 1019.58,1705.00 1021.61,1708.62" />\n  </edge>\n  <edge id=":2506089779_3" function="internal">\n    <lane id=":2506089779_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.59" length="11.52" shape="1013.73,1705.43 1015.52,1702.04 1016.24,1699.17 1015.89,1696.80 1014.47,1694.95" />\n  </edge>\n  <edge id=":2506089779_4" function="internal">\n    <lane id=":2506089779_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.59" shape="1012.44,1697.43 1015.22,1699.87 1017.71,1702.50 1019.86,1705.39 1021.61,1708.62" />\n  </edge>\n  <edge id=":2506089901_0" function="internal">\n    <lane id=":2506089901_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.69" shape="1412.13,1703.31 1407.53,1718.31" />\n  </edge>\n  <edge id=":2506089901_1" function="internal">\n    <lane id=":2506089901_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.39" length="5.24" shape="1412.13,1703.31 1411.23,1708.47" />\n  </edge>\n  <edge id=":2506089901_6" function="internal">\n    <lane id=":2506089901_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.39" length="11.99" shape="1411.23,1708.47 1412.36,1711.65 1415.51,1712.85 1420.69,1712.06" />\n  </edge>\n  <edge id=":2506089901_2" function="internal">\n    <lane id=":2506089901_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.61" length="9.31" shape="1419.81,1708.99 1417.01,1709.41 1415.31,1708.76 1414.70,1707.05 1415.18,1704.26" />\n  </edge>\n  <edge id=":2506089901_3" function="internal">\n    <lane id=":2506089901_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.12" length="16.04" shape="1419.81,1708.99 1415.15,1710.69 1411.55,1712.81 1409.01,1715.35 1407.53,1718.31" />\n  </edge>\n  <edge id=":2506089901_4" function="internal">\n    <lane id=":2506089901_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.00" length="12.85" shape="1410.60,1719.23 1411.75,1717.03 1413.82,1715.10 1416.80,1713.45 1420.69,1712.06" />\n  </edge>\n  <edge id=":2506089901_5" function="internal">\n    <lane id=":2506089901_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.66" shape="1410.60,1719.23 1415.18,1704.26" />\n  </edge>\n  <edge id=":2506090118_0" function="internal">\n    <lane id=":2506090118_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="3.91" length="2.54" shape="534.00,1727.22 533.32,1727.35 532.75,1727.24 532.30,1726.88 531.97,1726.27" />\n  </edge>\n  <edge id=":2506090118_1" function="internal">\n    <lane id=":2506090118_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.08" length="7.61" shape="528.97,1727.39 529.97,1729.22 531.32,1730.30 533.01,1730.64 535.05,1730.24" />\n  </edge>\n  <edge id=":2507400591_0" function="internal">\n    <lane id=":2507400591_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.64" length="10.69" shape="1499.81,718.64 1496.79,718.16 1494.49,717.00 1492.91,715.18 1492.05,712.68" />\n  </edge>\n  <edge id=":2507400591_1" function="internal">\n    <lane id=":2507400591_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.54" shape="1499.81,718.64 1487.27,718.69" />\n  </edge>\n  <edge id=":2507400591_2" function="internal">\n    <lane id=":2507400591_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.06" length="9.07" shape="1494.80,727.49 1494.67,724.92 1495.44,723.12 1497.11,722.09 1499.67,721.84" />\n  </edge>\n  <edge id=":2507400591_3" function="internal">\n    <lane id=":2507400591_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.07" shape="1494.80,727.49 1492.05,712.68" />\n  </edge>\n  <edge id=":2507400591_4" function="internal">\n    <lane id=":2507400591_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="12.66" shape="1494.80,727.49 1493.67,723.58 1492.04,720.80 1489.91,719.17 1487.27,718.69" />\n  </edge>\n  <edge id=":2507400591_5" function="internal">\n    <lane id=":2507400591_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.19" shape="1487.48,721.88 1499.67,721.84" />\n  </edge>\n  <edge id=":2507400591_6" function="internal">\n    <lane id=":2507400591_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.05" length="11.96" shape="1487.48,721.88 1490.07,721.24 1491.70,719.49 1492.36,716.64 1492.05,712.68" />\n  </edge>\n  <edge id=":2507400593_0" function="internal">\n    <lane id=":2507400593_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.91" length="8.10" shape="1502.98,748.30 1501.09,747.83 1499.59,746.76 1498.50,745.12 1497.81,742.89" />\n  </edge>\n  <edge id=":2507400593_1" function="internal">\n    <lane id=":2507400593_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.34" shape="1502.98,748.30 1493.67,747.51" />\n  </edge>\n  <edge id=":2507400604_0" function="internal">\n    <lane id=":2507400604_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.18" shape="1655.14,841.58 1652.96,855.59" />\n  </edge>\n  <edge id=":2507400604_1" function="internal">\n    <lane id=":2507400604_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.21" length="3.78" shape="1655.14,841.58 1655.19,845.36" />\n  </edge>\n  <edge id=":2507400604_6" function="internal">\n    <lane id=":2507400604_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.21" length="10.39" shape="1655.19,845.36 1656.46,848.31 1658.94,850.44 1662.63,851.74" />\n  </edge>\n  <edge id=":2507400604_2" function="internal">\n    <lane id=":2507400604_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.76" length="9.28" shape="1663.28,848.60 1660.84,847.75 1659.20,846.37 1658.35,844.45 1658.31,841.99" />\n  </edge>\n  <edge id=":2507400604_3" function="internal">\n    <lane id=":2507400604_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.95" length="14.29" shape="1663.28,848.60 1659.35,848.36 1656.33,849.45 1654.20,851.86 1652.96,855.59" />\n  </edge>\n  <edge id=":2507400604_4" function="internal">\n    <lane id=":2507400604_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.45" length="9.03" shape="1656.11,856.15 1656.89,853.79 1658.24,852.27 1660.15,851.59 1662.63,851.74" />\n  </edge>\n  <edge id=":2507400604_5" function="internal">\n    <lane id=":2507400604_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.33" shape="1656.11,856.15 1658.31,841.99" />\n  </edge>\n  <edge id=":2507400609_0" function="internal">\n    <lane id=":2507400609_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.13" shape="1748.58,865.75 1734.67,863.31" />\n  </edge>\n  <edge id=":2507400609_1" function="internal">\n    <lane id=":2507400609_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.95" length="3.93" shape="1748.58,865.75 1744.65,865.74" />\n  </edge>\n  <edge id=":2507400609_6" function="internal">\n    <lane id=":2507400609_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.95" length="10.36" shape="1744.65,865.74 1741.69,867.00 1739.70,869.53 1738.69,873.32" />\n  </edge>\n  <edge id=":2507400609_2" function="internal">\n    <lane id=":2507400609_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.45" length="9.03" shape="1741.86,873.70 1742.50,871.30 1743.76,869.71 1745.63,868.91 1748.12,868.92" />\n  </edge>\n  <edge id=":2507400609_3" function="internal">\n    <lane id=":2507400609_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.25" length="14.16" shape="1741.86,873.70 1741.80,869.79 1740.58,866.76 1738.20,864.60 1734.67,863.31" />\n  </edge>\n  <edge id=":2507400609_4" function="internal">\n    <lane id=":2507400609_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.80" length="9.31" shape="1734.02,866.44 1736.32,867.29 1737.86,868.72 1738.65,870.73 1738.69,873.32" />\n  </edge>\n  <edge id=":2507400609_5" function="internal">\n    <lane id=":2507400609_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.31" shape="1734.02,866.44 1748.12,868.92" />\n  </edge>\n  <edge id=":2507400638_0" function="internal">\n    <lane id=":2507400638_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.06" length="14.79" shape="1533.10,926.42 1529.97,927.84 1527.47,927.08 1525.60,924.14 1524.37,919.03" />\n  </edge>\n  <edge id=":2507400638_1" function="internal">\n    <lane id=":2507400638_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.03" shape="1533.10,926.42 1529.97,927.40 1526.65,926.73 1523.16,925.57 1519.50,925.12" />\n  </edge>\n  <edge id=":2507400638_2" function="internal">\n    <lane id=":2507400638_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.56" length="9.16" shape="1526.77,935.06 1526.68,932.22 1527.49,930.59 1529.20,930.18 1531.81,930.99" />\n  </edge>\n  <edge id=":2507400638_3" function="internal">\n    <lane id=":2507400638_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.21" shape="1526.77,935.06 1524.37,919.03" />\n  </edge>\n  <edge id=":2507400638_4" function="internal">\n    <lane id=":2507400638_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.05" length="13.52" shape="1526.77,935.06 1525.71,930.55 1524.14,927.39 1522.07,925.58 1519.50,925.12" />\n  </edge>\n  <edge id=":2507400638_5" function="internal">\n    <lane id=":2507400638_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.53" shape="1519.75,928.31 1523.10,928.27 1525.95,928.72 1528.72,929.63 1531.81,930.99" />\n  </edge>\n  <edge id=":2507400638_6" function="internal">\n    <lane id=":2507400638_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.19" length="11.86" shape="1519.75,928.31 1522.26,927.56 1523.87,925.76 1524.57,922.92 1524.37,919.03" />\n  </edge>\n  <edge id=":2507400639_0" function="internal">\n    <lane id=":2507400639_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.86" length="9.48" shape="1518.47,925.20 1515.79,925.06 1513.75,924.19 1512.35,922.59 1511.59,920.26" />\n  </edge>\n  <edge id=":2507400639_1" function="internal">\n    <lane id=":2507400639_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.33" length="9.04" shape="1513.92,934.49 1513.87,931.98 1514.65,930.13 1516.27,928.93 1518.72,928.39" />\n  </edge>\n  <edge id=":2507400639_2" function="internal">\n    <lane id=":2507400639_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="1513.92,934.49 1511.59,920.26" />\n    <lane id=":2507400639_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="1510.76,935.01 1508.43,920.77" />\n  </edge>\n  <edge id=":2507400689_0" function="internal">\n    <lane id=":2507400689_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.39" length="10.37" shape="1589.84,1280.87 1586.96,1280.46 1584.77,1279.35 1583.28,1277.53 1582.48,1275.00" />\n  </edge>\n  <edge id=":2507400689_1" function="internal">\n    <lane id=":2507400689_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.30" shape="1589.84,1280.87 1577.55,1280.87" />\n  </edge>\n  <edge id=":2507400689_2" function="internal">\n    <lane id=":2507400689_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.17" length="9.05" shape="1584.81,1289.80 1584.78,1287.26 1585.60,1285.45 1587.27,1284.39 1589.79,1284.07" />\n  </edge>\n  <edge id=":2507400689_3" function="internal">\n    <lane id=":2507400689_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.98" shape="1584.81,1289.80 1582.48,1275.00" />\n  </edge>\n  <edge id=":2507400689_4" function="internal">\n    <lane id=":2507400689_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="12.63" shape="1584.81,1289.80 1583.82,1285.86 1582.28,1283.05 1580.19,1281.39 1577.55,1280.87" />\n  </edge>\n  <edge id=":2507400689_5" function="internal">\n    <lane id=":2507400689_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.20" shape="1577.60,1284.07 1589.79,1284.07" />\n  </edge>\n  <edge id=":2507400689_6" function="internal">\n    <lane id=":2507400689_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.07" length="12.00" shape="1577.60,1284.07 1580.26,1283.46 1581.97,1281.75 1582.70,1278.93 1582.48,1275.00" />\n  </edge>\n  <edge id=":2507400690_0" function="internal">\n    <lane id=":2507400690_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.13" length="9.82" shape="1576.85,1280.88 1574.03,1280.59 1571.89,1279.63 1570.43,1277.98 1569.66,1275.67" />\n  </edge>\n  <edge id=":2507400690_1" function="internal">\n    <lane id=":2507400690_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.21" length="9.05" shape="1571.99,1289.95 1571.94,1287.41 1572.75,1285.59 1574.40,1284.48 1576.90,1284.08" />\n  </edge>\n  <edge id=":2507400690_2" function="internal">\n    <lane id=":2507400690_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.47" shape="1571.99,1289.95 1569.66,1275.67" />\n    <lane id=":2507400690_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.47" shape="1568.83,1290.46 1566.50,1276.18" />\n  </edge>\n  <edge id=":2507400696_0" function="internal">\n    <lane id=":2507400696_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.62" length="10.63" shape="1600.23,1345.54 1597.24,1345.00 1594.98,1343.80 1593.44,1341.95 1592.63,1339.45" />\n  </edge>\n  <edge id=":2507400696_1" function="internal">\n    <lane id=":2507400696_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.46" shape="1600.23,1345.54 1596.88,1345.36 1594.00,1345.28 1591.12,1345.32 1587.77,1345.47" />\n  </edge>\n  <edge id=":2507400696_2" function="internal">\n    <lane id=":2507400696_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.08" length="9.06" shape="1594.97,1354.28 1594.93,1351.71 1595.77,1349.93 1597.46,1348.94 1600.02,1348.74" />\n  </edge>\n  <edge id=":2507400696_3" function="internal">\n    <lane id=":2507400696_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.01" shape="1594.97,1354.28 1592.63,1339.45" />\n  </edge>\n  <edge id=":2507400696_4" function="internal">\n    <lane id=":2507400696_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.84" length="12.54" shape="1594.97,1354.28 1593.98,1350.30 1592.45,1347.51 1590.38,1345.90 1587.77,1345.47" />\n  </edge>\n  <edge id=":2507400696_5" function="internal">\n    <lane id=":2507400696_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.07" shape="1587.96,1348.67 1591.25,1348.51 1593.99,1348.47 1596.73,1348.55 1600.02,1348.74" />\n  </edge>\n  <edge id=":2507400696_6" function="internal">\n    <lane id=":2507400696_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.14" length="11.91" shape="1587.96,1348.67 1590.53,1347.96 1592.16,1346.19 1592.86,1343.35 1592.63,1339.45" />\n  </edge>\n  <edge id=":2507400698_0" function="internal">\n    <lane id=":2507400698_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.94" length="9.58" shape="1587.23,1345.51 1584.50,1345.33 1582.43,1344.43 1581.01,1342.82 1580.25,1340.49" />\n  </edge>\n  <edge id=":2507400698_1" function="internal">\n    <lane id=":2507400698_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.29" length="9.04" shape="1582.58,1354.74 1582.53,1352.22 1583.32,1350.38 1584.95,1349.20 1587.41,1348.70" />\n  </edge>\n  <edge id=":2507400698_2" function="internal">\n    <lane id=":2507400698_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.43" shape="1582.58,1354.74 1580.25,1340.49" />\n    <lane id=":2507400698_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.43" shape="1579.42,1355.25 1577.09,1341.01" />\n  </edge>\n  <edge id=":2507400748_0" function="internal">\n    <lane id=":2507400748_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.52" length="9.05" shape="1606.93,1623.24 1609.42,1623.20 1611.32,1623.96 1612.64,1625.52 1613.38,1627.88" />\n  </edge>\n  <edge id=":2507400748_1" function="internal">\n    <lane id=":2507400748_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.50" length="9.03" shape="1611.07,1613.67 1611.12,1616.14 1610.36,1618.04 1608.80,1619.35 1606.43,1620.08" />\n  </edge>\n  <edge id=":2507400748_2" function="internal">\n    <lane id=":2507400748_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="1611.07,1613.67 1613.38,1627.88" />\n  </edge>\n  <edge id=":2507400769_0" function="internal">\n    <lane id=":2507400769_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.64" shape="822.87,1649.75 808.38,1651.80" />\n  </edge>\n  <edge id=":2507400769_1" function="internal">\n    <lane id=":2507400769_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.90" length="4.30" shape="822.87,1649.75 819.02,1650.77 818.77,1650.97" />\n  </edge>\n  <edge id=":2507400769_6" function="internal">\n    <lane id=":2507400769_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.90" length="10.12" shape="818.77,1650.97 816.49,1652.78 815.30,1655.78 815.43,1659.76" />\n  </edge>\n  <edge id=":2507400769_2" function="internal">\n    <lane id=":2507400769_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.38" length="9.04" shape="818.58,1659.20 818.49,1656.71 819.25,1654.83 820.83,1653.57 823.24,1652.93" />\n  </edge>\n  <edge id=":2507400769_3" function="internal">\n    <lane id=":2507400769_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="14.34" shape="818.58,1659.20 817.35,1655.42 815.24,1652.93 812.25,1651.73 808.38,1651.80" />\n  </edge>\n  <edge id=":2507400769_4" function="internal">\n    <lane id=":2507400769_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.59" length="9.21" shape="808.90,1654.96 811.38,1654.91 813.29,1655.70 814.64,1657.32 815.43,1659.76" />\n  </edge>\n  <edge id=":2507400769_5" function="internal">\n    <lane id=":2507400769_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.48" shape="808.90,1654.96 823.24,1652.93" />\n  </edge>\n  <edge id=":2507400772_0" function="internal">\n    <lane id=":2507400772_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="705.32,1674.82 691.49,1678.88" />\n  </edge>\n  <edge id=":2507400772_1" function="internal">\n    <lane id=":2507400772_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.23" length="3.96" shape="705.32,1674.82 701.88,1676.41 701.77,1676.53" />\n  </edge>\n  <edge id=":2507400772_6" function="internal">\n    <lane id=":2507400772_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.23" length="10.34" shape="701.77,1676.53 699.70,1678.79 698.77,1681.96 699.10,1685.92" />\n  </edge>\n  <edge id=":2507400772_2" function="internal">\n    <lane id=":2507400772_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.79" length="9.39" shape="702.23,1685.23 702.00,1682.60 702.59,1680.50 704.00,1678.93 706.22,1677.89" />\n  </edge>\n  <edge id=":2507400772_3" function="internal">\n    <lane id=":2507400772_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.88" length="14.46" shape="702.23,1685.23 700.82,1681.49 698.56,1679.18 695.45,1678.31 691.49,1678.88" />\n  </edge>\n  <edge id=":2507400772_4" function="internal">\n    <lane id=":2507400772_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.36" length="9.04" shape="692.39,1681.95 694.87,1681.59 696.81,1682.14 698.22,1683.58 699.10,1685.92" />\n  </edge>\n  <edge id=":2507400772_5" function="internal">\n    <lane id=":2507400772_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="692.39,1681.95 706.22,1677.89" />\n  </edge>\n  <edge id=":2507400773_0" function="internal">\n    <lane id=":2507400773_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.81" length="10.51" shape="1698.44,1685.71 1697.86,1688.78 1696.71,1691.13 1695.01,1692.74 1692.74,1693.61" />\n  </edge>\n  <edge id=":2507400773_1" function="internal">\n    <lane id=":2507400773_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.63" length="15.22" shape="1698.44,1685.71 1698.68,1690.04 1700.28,1692.90 1703.24,1694.30 1707.57,1694.24" />\n  </edge>\n  <edge id=":2507400773_2" function="internal">\n    <lane id=":2507400773_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.04" length="9.07" shape="1707.07,1691.08 1704.49,1691.11 1702.72,1690.28 1701.77,1688.57 1701.63,1686.00" />\n  </edge>\n  <edge id=":2507400773_3" function="internal">\n    <lane id=":2507400773_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.55" shape="1707.07,1691.08 1692.74,1693.61" />\n  </edge>\n  <edge id=":2507400773_4" function="internal">\n    <lane id=":2507400773_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="1693.37,1696.75 1707.57,1694.24" />\n  </edge>\n  <edge id=":2507400773_5" function="internal">\n    <lane id=":2507400773_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.12" length="3.53" shape="1693.37,1696.75 1696.68,1695.52" />\n  </edge>\n  <edge id=":2507400773_6" function="internal">\n    <lane id=":2507400773_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.12" length="11.12" shape="1696.68,1695.52 1699.15,1693.31 1700.81,1690.14 1701.63,1686.00" />\n  </edge>\n  <edge id=":3125684630_0" function="internal">\n    <lane id=":3125684630_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.26" shape="346.75,440.38 345.20,451.50" />\n    <lane id=":3125684630_0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.26" shape="349.92,440.81 348.36,451.96" />\n    <lane id=":3125684630_0_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.26" shape="353.10,441.23 351.53,452.42" />\n  </edge>\n  <edge id=":3125684630_3" function="internal">\n    <lane id=":3125684630_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.49" length="9.03" shape="340.45,445.18 342.83,445.86 344.42,447.14 345.20,449.02 345.20,451.50" />\n  </edge>\n  <edge id=":3376091528_0" function="internal">\n    <lane id=":3376091528_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.21" shape="1015.80,1370.40 1015.80,1370.61" />\n  </edge>\n  <edge id=":3376091528_1" function="internal">\n    <lane id=":3376091528_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.30" shape="1018.99,1370.78 1019.00,1370.48" />\n  </edge>\n  <edge id=":4085816302_0" function="internal">\n    <lane id=":4085816302_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.47" length="10.80" shape="347.19,1023.78 344.04,1022.72 341.84,1021.20 340.60,1019.22 340.31,1016.77" />\n  </edge>\n  <edge id=":4085816302_1" function="internal">\n    <lane id=":4085816302_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.84" length="9.13" shape="341.26,1031.77 341.22,1029.11 342.07,1027.41 343.83,1026.68 346.48,1026.90" />\n  </edge>\n  <edge id=":4085816302_2" function="internal">\n    <lane id=":4085816302_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.81" shape="341.26,1031.77 340.55,1027.35 340.15,1024.30 340.07,1021.23 340.31,1016.77" />\n    <lane id=":4085816302_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.81" shape="338.10,1032.26 337.35,1027.63 336.94,1024.43 336.87,1021.22 337.11,1016.55" />\n    <lane id=":4085816302_2_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.81" shape="334.93,1032.76 334.16,1027.91 333.74,1024.57 333.66,1021.21 333.92,1016.33" />\n  </edge>\n  <edge id=":4085816302_5" function="internal">\n    <lane id=":4085816302_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.36" shape="327.82,1021.77 332.14,1022.71 337.18,1024.18 342.20,1025.73 346.48,1026.90" />\n  </edge>\n  <edge id=":4085816302_6" function="internal">\n    <lane id=":4085816302_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.49" length="9.47" shape="327.82,1021.77 330.33,1021.73 332.18,1020.81 333.38,1019.01 333.92,1016.33" />\n  </edge>\n  <edge id=":4488358856_0" function="internal">\n    <lane id=":4488358856_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.78" length="13.56" shape="1133.03,1302.24 1132.75,1306.84 1131.97,1310.43 1130.68,1313.02 1128.89,1314.61" />\n  </edge>\n  <edge id=":4488358856_1" function="internal">\n    <lane id=":4488358856_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.37" length="17.55" shape="1133.03,1302.24 1133.69,1307.52 1135.73,1310.16 1139.16,1310.19 1143.96,1307.58" />\n  </edge>\n  <edge id=":4488358856_2" function="internal">\n    <lane id=":4488358856_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.53" length="9.22" shape="1141.97,1305.07 1139.44,1306.44 1137.65,1306.43 1136.57,1305.03 1136.23,1302.26" />\n  </edge>\n  <edge id=":4488358856_3" function="internal">\n    <lane id=":4488358856_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.20" shape="1141.97,1305.07 1138.83,1307.46 1135.56,1310.05 1132.23,1312.54 1128.89,1314.61" />\n  </edge>\n  <edge id=":4488358856_4" function="internal">\n    <lane id=":4488358856_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.34" shape="1130.79,1317.22 1134.14,1315.13 1137.51,1312.61 1140.81,1309.98 1143.96,1307.58" />\n  </edge>\n  <edge id=":4488358856_5" function="internal">\n    <lane id=":4488358856_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.78" length="3.52" shape="1130.79,1317.22 1133.15,1315.22 1133.35,1314.84" />\n  </edge>\n  <edge id=":4488358856_6" function="internal">\n    <lane id=":4488358856_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.78" length="13.09" shape="1133.35,1314.84 1134.84,1312.05 1135.87,1307.73 1136.23,1302.26" />\n  </edge>\n  <edge id=":4488358857_0" function="internal">\n    <lane id=":4488358857_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.48" shape="1097.46,1209.96 1094.32,1207.49 1090.91,1204.92 1087.37,1202.57 1083.84,1200.75" />\n  </edge>\n  <edge id=":4488358857_1" function="internal">\n    <lane id=":4488358857_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.38" length="5.25" shape="1097.46,1209.96 1092.95,1207.28" />\n  </edge>\n  <edge id=":4488358857_6" function="internal">\n    <lane id=":4488358857_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.38" length="12.29" shape="1092.95,1207.28 1092.84,1207.21 1089.45,1207.14 1087.30,1209.76 1086.38,1215.06" />\n  </edge>\n  <edge id=":4488358857_2" function="internal">\n    <lane id=":4488358857_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.58" length="9.36" shape="1089.58,1215.20 1090.07,1212.37 1091.22,1210.98 1093.02,1211.01 1095.49,1212.48" />\n  </edge>\n  <edge id=":4488358857_3" function="internal">\n    <lane id=":4488358857_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.96" length="16.51" shape="1089.58,1215.20 1089.42,1209.86 1088.41,1205.67 1086.55,1202.64 1083.84,1200.75" />\n  </edge>\n  <edge id=":4488358857_4" function="internal">\n    <lane id=":4488358857_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.84" length="12.86" shape="1082.34,1203.59 1084.27,1205.01 1085.59,1207.39 1086.29,1210.74 1086.38,1215.06" />\n  </edge>\n  <edge id=":4488358857_5" function="internal">\n    <lane id=":4488358857_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.92" shape="1082.34,1203.59 1085.79,1205.36 1089.17,1207.60 1092.42,1210.07 1095.49,1212.48" />\n  </edge>\n  <edge id=":4488358858_0" function="internal">\n    <lane id=":4488358858_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.47" length="9.77" shape="1082.97,1290.30 1082.48,1293.39 1081.38,1294.76 1079.66,1294.39 1077.33,1292.30" />\n  </edge>\n  <edge id=":4488358858_1" function="internal">\n    <lane id=":4488358858_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="12.76" length="17.75" shape="1082.97,1290.30 1083.01,1296.07 1083.70,1300.79 1085.04,1304.47 1087.02,1307.10" />\n  </edge>\n  <edge id=":4488358858_2" function="internal">\n    <lane id=":4488358858_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.74" length="14.96" shape="1089.18,1304.73 1087.67,1302.64 1086.67,1299.56 1086.16,1295.49 1086.17,1290.44" />\n  </edge>\n  <edge id=":4488358858_3" function="internal">\n    <lane id=":4488358858_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.17" shape="1089.18,1304.73 1086.23,1301.89 1083.12,1298.65 1080.07,1295.34 1077.33,1292.30" />\n  </edge>\n  <edge id=":4488358858_4" function="internal">\n    <lane id=":4488358858_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.51" shape="1074.94,1294.43 1077.72,1297.51 1080.85,1300.90 1084.04,1304.22 1087.02,1307.10" />\n  </edge>\n  <edge id=":4488358858_5" function="internal">\n    <lane id=":4488358858_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.43" length="6.24" shape="1074.94,1294.43 1079.59,1298.60" />\n  </edge>\n  <edge id=":4488358858_6" function="internal">\n    <lane id=":4488358858_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.43" length="13.22" shape="1079.59,1298.60 1083.00,1299.32 1085.20,1296.60 1086.17,1290.44" />\n  </edge>\n  <edge id=":4488366488_0" function="internal">\n    <lane id=":4488366488_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.20" shape="1238.71,1204.30 1235.79,1206.73 1232.78,1209.09 1229.64,1211.25 1226.35,1213.09" />\n  </edge>\n  <edge id=":4488366488_1" function="internal">\n    <lane id=":4488366488_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="12.04" length="3.56" shape="1238.71,1204.30 1236.61,1206.73 1236.46,1207.05" />\n  </edge>\n  <edge id=":4488366488_6" function="internal">\n    <lane id=":4488366488_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="12.04" length="12.29" shape="1236.46,1207.05 1235.13,1209.98 1234.26,1214.05 1234.00,1218.95" />\n  </edge>\n  <edge id=":4488366488_2" function="internal">\n    <lane id=":4488366488_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.95" length="13.04" shape="1237.20,1218.92 1237.39,1214.75 1238.05,1211.33 1239.17,1208.66 1240.76,1206.76" />\n  </edge>\n  <edge id=":4488366488_3" function="internal">\n    <lane id=":4488366488_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.43" length="16.12" shape="1237.20,1218.92 1236.48,1214.44 1234.44,1211.97 1231.06,1211.52 1226.35,1213.09" />\n  </edge>\n  <edge id=":4488366488_4" function="internal">\n    <lane id=":4488366488_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.65" length="8.74" shape="1228.12,1215.78 1230.67,1214.94 1232.50,1215.18 1233.61,1216.52 1234.00,1218.95" />\n  </edge>\n  <edge id=":4488366488_5" function="internal">\n    <lane id=":4488366488_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.57" shape="1228.12,1215.78 1231.46,1213.92 1234.69,1211.67 1237.80,1209.22 1240.76,1206.76" />\n  </edge>\n  <edge id=":4488368090_0" function="internal">\n    <lane id=":4488368090_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.45" length="9.03" shape="1182.03,1296.56 1181.65,1299.02 1180.58,1300.75 1178.80,1301.74 1176.33,1302.00" />\n  </edge>\n  <edge id=":4488368090_1" function="internal">\n    <lane id=":4488368090_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.38" length="14.11" shape="1182.03,1296.56 1182.50,1300.44 1184.00,1303.37 1186.53,1305.36 1190.08,1306.40" />\n  </edge>\n  <edge id=":4488368090_2" function="internal">\n    <lane id=":4488368090_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.96" length="9.43" shape="1190.52,1303.23 1188.19,1302.54 1186.52,1301.20 1185.54,1299.22 1185.22,1296.59" />\n  </edge>\n  <edge id=":4488368090_3" function="internal">\n    <lane id=":4488368090_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.25" shape="1190.52,1303.23 1186.92,1302.77 1183.44,1302.43 1179.96,1302.17 1176.33,1302.00" />\n  </edge>\n  <edge id=":4488368090_4" function="internal">\n    <lane id=":4488368090_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.92" shape="1176.21,1305.20 1179.79,1305.37 1183.16,1305.61 1186.52,1305.95 1190.08,1306.40" />\n  </edge>\n  <edge id=":4488368090_5" function="internal">\n    <lane id=":4488368090_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.95" length="3.93" shape="1176.21,1305.20 1180.13,1304.78" />\n  </edge>\n  <edge id=":4488368090_6" function="internal">\n    <lane id=":4488368090_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.95" length="10.37" shape="1180.13,1304.78 1182.93,1303.21 1184.63,1300.48 1185.22,1296.59" />\n  </edge>\n  <edge id=":485580842_0" function="internal">\n    <lane id=":485580842_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.44" length="9.03" shape="369.27,837.28 366.86,836.67 365.25,835.43 364.45,833.57 364.44,831.08" />\n  </edge>\n  <edge id=":485580842_1" function="internal">\n    <lane id=":485580842_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.72" shape="369.27,837.28 351.66,835.33" />\n  </edge>\n  <edge id=":485580842_2" function="internal">\n    <lane id=":485580842_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.54" shape="362.76,842.54 364.44,831.08" />\n    <lane id=":485580842_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.54" shape="359.60,842.05 361.27,830.64" />\n    <lane id=":485580842_2_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.54" shape="356.44,841.57 358.10,830.19" />\n  </edge>\n  <edge id=":485580842_5" function="internal">\n    <lane id=":485580842_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.43" length="9.03" shape="356.44,841.57 356.46,839.08 355.67,837.21 354.07,835.96 351.66,835.33" />\n  </edge>\n  <edge id=":4888861605_0" function="internal">\n    <lane id=":4888861605_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.30" shape="1049.23,1700.52 1049.02,1700.73" />\n  </edge>\n  <edge id=":4888861605_1" function="internal">\n    <lane id=":4888861605_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.31" shape="1051.16,1703.08 1051.39,1702.88" />\n  </edge>\n  <edge id=":4888861608_0" function="internal">\n    <lane id=":4888861608_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.94" length="6.02" shape="1031.11,1707.12 1029.85,1707.19 1028.51,1706.91 1027.06,1706.29 1025.53,1705.31" />\n  </edge>\n  <edge id=":4888861608_1" function="internal">\n    <lane id=":4888861608_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.42" length="5.93" shape="1031.11,1707.12 1029.37,1707.44 1028.15,1707.73 1027.11,1708.33 1025.89,1709.53" />\n  </edge>\n  <edge id=":4888861612_0" function="internal">\n    <lane id=":4888861612_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="6.79" shape="1029.03,1726.57 1030.45,1725.14 1031.44,1724.12 1032.57,1723.34 1034.40,1722.59" />\n  </edge>\n  <edge id=":4888861612_1" function="internal">\n    <lane id=":4888861612_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.31" length="6.06" shape="1028.54,1722.50 1030.23,1723.07 1031.46,1723.27 1032.70,1723.11 1034.40,1722.59" />\n  </edge>\n  <edge id=":4888861618_0" function="internal">\n    <lane id=":4888861618_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.70" length="6.44" shape="1024.13,1718.27 1024.62,1720.11 1024.86,1721.43 1024.75,1722.74 1024.23,1724.55" />\n  </edge>\n  <edge id=":4888861618_1" function="internal">\n    <lane id=":4888861618_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.23" length="6.17" shape="1024.13,1718.27 1024.68,1719.63 1025.57,1720.78 1026.79,1721.71 1028.35,1722.44" />\n  </edge>\n  <edge id=":4888861620_0" function="internal">\n    <lane id=":4888861620_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="6.98" shape="1039.25,1717.45 1039.85,1715.51 1040.54,1714.28 1041.48,1713.16 1042.79,1711.55" />\n  </edge>\n  <edge id=":4888861620_1" function="internal">\n    <lane id=":4888861620_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.23" length="4.76" shape="1039.25,1717.45 1039.63,1716.08 1039.87,1715.10 1039.82,1714.14 1039.35,1712.85" />\n  </edge>\n  <edge id=":4888862222_0" function="internal">\n    <lane id=":4888862222_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.49" length="8.72" shape="1042.18,1706.93 1040.46,1708.07 1038.52,1708.54 1036.38,1708.34 1034.02,1707.48" />\n  </edge>\n  <edge id=":4888862222_1" function="internal">\n    <lane id=":4888862222_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.26" length="7.63" shape="1039.26,1712.67 1038.24,1710.68 1037.29,1709.43 1036.02,1708.50 1034.02,1707.48" />\n  </edge>\n  <edge id=":4900464580_0" function="internal">\n    <lane id=":4900464580_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.26" length="10.52" shape="977.17,1060.70 974.33,1060.00 972.31,1058.63 971.09,1056.58 970.68,1053.84" />\n  </edge>\n  <edge id=":4900464580_1" function="internal">\n    <lane id=":4900464580_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.53" shape="977.17,1060.70 961.73,1058.99" />\n  </edge>\n  <edge id=":4900464580_2" function="internal">\n    <lane id=":4900464580_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.68" length="4.26" shape="977.17,1060.70 972.91,1060.85" />\n  </edge>\n  <edge id=":4900464580_12" function="internal">\n    <lane id=":4900464580_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.68" length="10.80" shape="972.91,1060.85 970.02,1062.36 968.48,1065.25 968.31,1069.50" />\n  </edge>\n  <edge id=":4900464580_3" function="internal">\n    <lane id=":4900464580_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.10" length="9.06" shape="971.49,1069.18 971.59,1066.62 972.52,1064.88 974.26,1063.97 976.82,1063.88" />\n  </edge>\n  <edge id=":4900464580_4" function="internal">\n    <lane id=":4900464580_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.37" shape="971.49,1069.18 971.15,1065.39 970.90,1061.52 970.74,1057.65 970.68,1053.84" />\n  </edge>\n  <edge id=":4900464580_5" function="internal">\n    <lane id=":4900464580_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.05" length="15.44" shape="971.49,1069.18 970.53,1065.09 968.58,1062.03 965.65,1060.00 961.73,1058.99" />\n  </edge>\n  <edge id=":4900464580_6" function="internal">\n    <lane id=":4900464580_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.74" length="11.05" shape="961.37,1062.17 964.15,1062.89 966.23,1064.35 967.62,1066.56 968.31,1069.50" />\n  </edge>\n  <edge id=":4900464580_7" function="internal">\n    <lane id=":4900464580_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.55" shape="961.37,1062.17 976.82,1063.88" />\n  </edge>\n  <edge id=":4900464580_8" function="internal">\n    <lane id=":4900464580_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.81" length="4.19" shape="961.37,1062.17 965.45,1062.04 965.55,1061.99" />\n  </edge>\n  <edge id=":4900464580_13" function="internal">\n    <lane id=":4900464580_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.81" length="10.45" shape="965.55,1061.99 968.36,1060.61 970.10,1057.88 970.68,1053.84" />\n  </edge>\n  <edge id=":4900464580_9" function="internal">\n    <lane id=":4900464580_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.27" length="9.04" shape="967.48,1053.84 967.12,1056.34 966.05,1058.03 964.25,1058.91 961.73,1058.99" />\n  </edge>\n  <edge id=":4900464580_10" function="internal">\n    <lane id=":4900464580_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.69" shape="967.48,1053.84 967.54,1057.70 967.70,1061.68 967.96,1065.66 968.31,1069.50" />\n  </edge>\n  <edge id=":4900464580_11" function="internal">\n    <lane id=":4900464580_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.63" length="15.28" shape="967.48,1053.84 968.07,1057.85 969.82,1060.86 972.74,1062.87 976.82,1063.88" />\n  </edge>\n  <edge id=":4900464584_0" function="internal">\n    <lane id=":4900464584_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.52" shape="1057.87,979.82 1046.45,978.33" />\n  </edge>\n  <edge id=":4900464584_1" function="internal">\n    <lane id=":4900464584_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.96" length="9.09" shape="1052.38,988.30 1052.32,985.66 1053.14,983.90 1054.84,983.01 1057.43,982.99" />\n  </edge>\n  <edge id=":4900464584_2" function="internal">\n    <lane id=":4900464584_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.45" shape="1046.08,981.51 1057.43,982.99" />\n  </edge>\n  <edge id=":4900464587_0" function="internal">\n    <lane id=":4900464587_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.96" length="9.66" shape="1241.06,1127.38 1240.21,1129.68 1238.72,1131.24 1236.59,1132.06 1233.82,1132.14" />\n  </edge>\n  <edge id=":4900464587_1" function="internal">\n    <lane id=":4900464587_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.49" shape="1241.06,1127.38 1238.02,1141.55" />\n  </edge>\n  <edge id=":4900464587_2" function="internal">\n    <lane id=":4900464587_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.58" shape="1241.14,1142.26 1244.20,1128.01" />\n  </edge>\n  <edge id=":4900464587_3" function="internal">\n    <lane id=":4900464587_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.79" length="4.29" shape="1241.14,1142.26 1241.47,1138.17 1241.41,1138.00" />\n  </edge>\n  <edge id=":4900464587_6" function="internal">\n    <lane id=":4900464587_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.79" length="10.42" shape="1241.41,1138.00 1240.36,1135.11 1237.81,1133.10 1233.82,1132.14" />\n  </edge>\n  <edge id=":4900464587_4" function="internal">\n    <lane id=":4900464587_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.24" length="9.05" shape="1233.52,1135.32 1235.97,1135.92 1237.54,1137.15 1238.22,1139.03 1238.02,1141.55" />\n  </edge>\n  <edge id=":4900464587_5" function="internal">\n    <lane id=":4900464587_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.38" length="14.44" shape="1233.52,1135.32 1237.58,1135.19 1240.71,1133.92 1242.92,1131.53 1244.20,1128.01" />\n  </edge>\n  <edge id=":4900464597_0" function="internal">\n    <lane id=":4900464597_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.39" length="9.04" shape="968.50,1146.51 968.07,1148.97 966.94,1150.65 965.14,1151.56 962.65,1151.69" />\n  </edge>\n  <edge id=":4900464597_1" function="internal">\n    <lane id=":4900464597_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.51" length="14.15" shape="968.50,1146.51 968.87,1150.43 970.26,1153.46 972.68,1155.59 976.12,1156.83" />\n  </edge>\n  <edge id=":4900464597_2" function="internal">\n    <lane id=":4900464597_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.11" length="9.59" shape="976.75,1153.69 974.47,1152.85 972.86,1151.40 971.94,1149.31 971.70,1146.61" />\n  </edge>\n  <edge id=":4900464597_3" function="internal">\n    <lane id=":4900464597_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.26" shape="976.75,1153.69 973.18,1153.03 969.73,1152.49 966.26,1152.05 962.65,1151.69" />\n  </edge>\n  <edge id=":4900464597_4" function="internal">\n    <lane id=":4900464597_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.90" shape="962.36,1154.88 965.93,1155.23 969.27,1155.65 972.61,1156.17 976.12,1156.83" />\n  </edge>\n  <edge id=":4900464597_5" function="internal">\n    <lane id=":4900464597_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.90" length="3.98" shape="962.36,1154.88 966.34,1154.67" />\n  </edge>\n  <edge id=":4900464597_6" function="internal">\n    <lane id=":4900464597_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.90" length="10.43" shape="966.34,1154.67 969.22,1153.21 971.01,1150.53 971.70,1146.61" />\n  </edge>\n  <edge id=":4900464598_0" function="internal">\n    <lane id=":4900464598_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.28" shape="976.34,974.46 962.08,973.77" />\n  </edge>\n  <edge id=":4900464598_1" function="internal">\n    <lane id=":4900464598_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.94" length="3.95" shape="976.34,974.46 972.42,974.88" />\n  </edge>\n  <edge id=":4900464598_6" function="internal">\n    <lane id=":4900464598_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.94" length="10.39" shape="972.42,974.88 969.61,976.47 967.94,979.21 967.38,983.12" />\n  </edge>\n  <edge id=":4900464598_2" function="internal">\n    <lane id=":4900464598_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.43" length="9.03" shape="970.58,983.12 970.93,980.65 971.99,978.92 973.75,977.93 976.23,977.66" />\n  </edge>\n  <edge id=":4900464598_3" function="internal">\n    <lane id=":4900464598_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.20" length="14.21" shape="970.58,983.12 970.04,979.23 968.45,976.37 965.79,974.55 962.08,973.77" />\n  </edge>\n  <edge id=":4900464598_4" function="internal">\n    <lane id=":4900464598_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.74" length="9.29" shape="961.88,976.96 964.28,977.48 966.00,978.67 967.03,980.56 967.38,983.12" />\n  </edge>\n  <edge id=":4900464598_5" function="internal">\n    <lane id=":4900464598_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.36" shape="961.88,976.96 976.23,977.66" />\n  </edge>\n  <edge id=":4900465031_0" function="internal">\n    <lane id=":4900465031_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.54" shape="1730.09,401.75 1725.91,415.67" />\n  </edge>\n  <edge id=":4900465031_1" function="internal">\n    <lane id=":4900465031_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.44" length="3.94" shape="1730.09,401.75 1729.54,405.43 1729.59,405.64" />\n  </edge>\n  <edge id=":4900465031_6" function="internal">\n    <lane id=":4900465031_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.44" length="10.50" shape="1729.59,405.64 1730.33,408.59 1732.46,411.23 1735.94,413.34" />\n  </edge>\n  <edge id=":4900465031_2" function="internal">\n    <lane id=":4900465031_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.03" length="9.72" shape="1737.25,410.42 1734.85,408.98 1733.37,407.21 1732.80,405.12 1733.14,402.69" />\n  </edge>\n  <edge id=":4900465031_3" function="internal">\n    <lane id=":4900465031_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.78" length="14.74" shape="1737.25,410.42 1733.28,409.29 1730.07,409.79 1727.61,411.92 1725.91,415.67" />\n  </edge>\n  <edge id=":4900465031_4" function="internal">\n    <lane id=":4900465031_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.23" length="9.05" shape="1728.98,416.56 1730.03,414.26 1731.54,412.96 1733.51,412.65 1735.94,413.34" />\n  </edge>\n  <edge id=":4900465031_5" function="internal">\n    <lane id=":4900465031_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.48" shape="1728.98,416.56 1733.14,402.69" />\n  </edge>\n  <edge id=":4900465052_0" function="internal">\n    <lane id=":4900465052_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.50" length="9.03" shape="1614.54,1670.12 1617.02,1670.06 1618.91,1670.81 1620.23,1672.37 1620.97,1674.73" />\n  </edge>\n  <edge id=":4900465052_1" function="internal">\n    <lane id=":4900465052_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.52" length="9.05" shape="1618.66,1660.52 1618.71,1662.99 1617.95,1664.89 1616.38,1666.21 1614.02,1666.96" />\n  </edge>\n  <edge id=":4900465052_2" function="internal">\n    <lane id=":4900465052_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="1618.66,1660.52 1620.97,1674.73" />\n  </edge>\n  <edge id=":4900465069_0" function="internal">\n    <lane id=":4900465069_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.50" shape="1347.83,1696.36 1347.25,1681.87" />\n  </edge>\n  <edge id=":4900465069_1" function="internal">\n    <lane id=":4900465069_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="4.15" shape="1347.83,1696.36 1347.16,1692.52 1347.03,1692.31" />\n  </edge>\n  <edge id=":4900465069_6" function="internal">\n    <lane id=":4900465069_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="10.09" shape="1347.03,1692.31 1345.42,1689.82 1342.61,1688.26 1338.72,1687.84" />\n  </edge>\n  <edge id=":4900465069_2" function="internal">\n    <lane id=":4900465069_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="9.06" shape="1338.83,1691.04 1341.30,1691.30 1343.10,1692.29 1344.21,1694.01 1344.63,1696.45" />\n  </edge>\n  <edge id=":4900465069_3" function="internal">\n    <lane id=":4900465069_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.97" length="14.26" shape="1338.83,1691.04 1342.68,1690.35 1345.37,1688.59 1346.89,1685.76 1347.25,1681.87" />\n  </edge>\n  <edge id=":4900465069_4" function="internal">\n    <lane id=":4900465069_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.47" length="9.03" shape="1344.05,1682.03 1343.83,1684.50 1342.86,1686.29 1341.16,1687.40 1338.72,1687.84" />\n  </edge>\n  <edge id=":4900465069_5" function="internal">\n    <lane id=":4900465069_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.43" shape="1344.05,1682.03 1344.63,1696.45" />\n  </edge>\n  <edge id=":4900465071_0" function="internal">\n    <lane id=":4900465071_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.55" shape="1624.45,519.04 1621.86,533.36" />\n  </edge>\n  <edge id=":4900465071_1" function="internal">\n    <lane id=":4900465071_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.14" length="4.11" shape="1624.45,519.04 1624.27,522.87 1624.36,523.12" />\n  </edge>\n  <edge id=":4900465071_6" function="internal">\n    <lane id=":4900465071_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.14" length="10.22" shape="1624.36,523.12 1625.38,525.90 1627.79,528.13 1631.51,529.55" />\n  </edge>\n  <edge id=":4900465071_2" function="internal">\n    <lane id=":4900465071_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.68" length="9.30" shape="1632.24,526.44 1629.80,525.51 1628.22,524.07 1627.48,522.12 1627.59,519.66" />\n  </edge>\n  <edge id=":4900465071_3" function="internal">\n    <lane id=":4900465071_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.89" length="14.45" shape="1632.24,526.44 1628.26,526.09 1625.20,527.12 1623.07,529.55 1621.86,533.36" />\n  </edge>\n  <edge id=":4900465071_4" function="internal">\n    <lane id=":4900465071_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.37" length="9.04" shape="1625.01,533.88 1625.77,531.50 1627.10,529.98 1629.02,529.33 1631.51,529.55" />\n  </edge>\n  <edge id=":4900465071_5" function="internal">\n    <lane id=":4900465071_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.46" shape="1625.01,533.88 1627.59,519.66" />\n  </edge>\n  <edge id=":4900465090_0" function="internal">\n    <lane id=":4900465090_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.03" length="1.03" shape="314.47,1657.91 314.56,1658.93" />\n  </edge>\n  <edge id=":4900465090_3" function="internal">\n    <lane id=":4900465090_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.03" length="6.86" shape="314.56,1658.93 314.61,1659.39 315.60,1660.57 317.47,1661.45 320.19,1662.04" />\n  </edge>\n  <edge id=":4900465090_1" function="internal">\n    <lane id=":4900465090_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.41" shape="310.87,1660.76 320.19,1662.04" />\n  </edge>\n  <edge id=":4900465090_2" function="internal">\n    <lane id=":4900465090_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.20" length="7.83" shape="310.87,1660.76 313.61,1660.94 315.65,1660.61 316.98,1659.80 317.62,1658.48" />\n  </edge>\n  <edge id=":4900465099_0" function="internal">\n    <lane id=":4900465099_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.90" length="8.41" shape="581.18,1011.91 580.94,1013.68 581.71,1015.10 583.51,1016.16 586.34,1016.87" />\n  </edge>\n  <edge id=":4900465099_1" function="internal">\n    <lane id=":4900465099_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.60" shape="576.90,1015.12 586.34,1016.87" />\n  </edge>\n  <edge id=":4900465099_2" function="internal">\n    <lane id=":4900465099_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.62" length="8.22" shape="576.90,1015.12 579.59,1015.47 581.70,1015.25 583.22,1014.45 584.17,1013.07" />\n  </edge>\n  <edge id=":4900465166_0" function="internal">\n    <lane id=":4900465166_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.49" length="9.09" shape="898.66,911.44 898.15,913.89 896.98,915.58 895.15,916.50 892.67,916.65" />\n  </edge>\n  <edge id=":4900465166_1" function="internal">\n    <lane id=":4900465166_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.45" shape="898.66,911.44 897.77,925.86" />\n  </edge>\n  <edge id=":4900465166_2" function="internal">\n    <lane id=":4900465166_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.93" length="4.13" shape="898.66,911.44 898.96,915.37 899.05,915.54" />\n  </edge>\n  <edge id=":4900465166_12" function="internal">\n    <lane id=":4900465166_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.93" length="10.21" shape="899.05,915.54 900.46,918.22 903.15,919.99 907.04,920.66" />\n  </edge>\n  <edge id=":4900465166_3" function="internal">\n    <lane id=":4900465166_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.43" length="9.03" shape="907.14,917.46 904.68,917.04 902.99,915.93 902.04,914.13 901.86,911.65" />\n  </edge>\n  <edge id=":4900465166_4" function="internal">\n    <lane id=":4900465166_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.49" shape="907.14,917.46 892.67,916.65" />\n  </edge>\n  <edge id=":4900465166_5" function="internal">\n    <lane id=":4900465166_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.10" length="14.26" shape="907.14,917.46 903.22,917.89 900.36,919.43 898.54,922.09 897.77,925.86" />\n  </edge>\n  <edge id=":4900465166_6" function="internal">\n    <lane id=":4900465166_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.63" length="9.21" shape="900.96,926.05 901.46,923.63 902.64,921.92 904.50,920.94 907.04,920.66" />\n  </edge>\n  <edge id=":4900465166_7" function="internal">\n    <lane id=":4900465166_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="900.96,926.05 901.86,911.65" />\n  </edge>\n  <edge id=":4900465166_8" function="internal">\n    <lane id=":4900465166_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.08" length="3.99" shape="900.96,926.05 900.63,922.21 900.56,922.09" />\n  </edge>\n  <edge id=":4900465166_13" function="internal">\n    <lane id=":4900465166_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.08" length="10.21" shape="900.56,922.09 899.14,919.37 896.49,917.51 892.67,916.65" />\n  </edge>\n  <edge id=":4900465166_9" function="internal">\n    <lane id=":4900465166_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.60" length="9.14" shape="892.41,919.84 894.87,920.39 896.59,921.58 897.55,923.41 897.77,925.86" />\n  </edge>\n  <edge id=":4900465166_10" function="internal">\n    <lane id=":4900465166_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.66" shape="892.41,919.84 907.04,920.66" />\n  </edge>\n  <edge id=":4900465166_11" function="internal">\n    <lane id=":4900465166_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="14.31" shape="892.41,919.84 896.32,919.60 899.19,918.16 901.04,915.51 901.86,911.65" />\n  </edge>\n  <edge id=":4900465208_0" function="internal">\n    <lane id=":4900465208_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.34" shape="594.11,500.55 579.85,499.04" />\n  </edge>\n  <edge id=":4900465208_1" function="internal">\n    <lane id=":4900465208_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="3.98" shape="594.11,500.55 590.26,500.72 590.14,500.78" />\n  </edge>\n  <edge id=":4900465208_6" function="internal">\n    <lane id=":4900465208_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="10.21" shape="590.14,500.78 587.35,502.09 585.39,504.66 584.38,508.42" />\n  </edge>\n  <edge id=":4900465208_2" function="internal">\n    <lane id=":4900465208_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.59" length="9.12" shape="587.55,508.82 588.21,506.39 589.46,504.73 591.32,503.85 593.79,503.74" />\n  </edge>\n  <edge id=":4900465208_3" function="internal">\n    <lane id=":4900465208_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.98" length="14.24" shape="587.55,508.82 587.48,504.91 586.17,501.98 583.63,500.02 579.85,499.04" />\n  </edge>\n  <edge id=":4900465208_4" function="internal">\n    <lane id=":4900465208_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.48" length="9.03" shape="579.49,502.22 581.89,502.84 583.50,504.09 584.33,505.95 584.38,508.42" />\n  </edge>\n  <edge id=":4900465208_5" function="internal">\n    <lane id=":4900465208_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.38" shape="579.49,502.22 593.79,503.74" />\n  </edge>\n  <edge id=":4900465209_0" function="internal">\n    <lane id=":4900465209_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.61" length="9.20" shape="575.49,579.78 574.83,582.25 573.57,583.93 571.71,584.82 569.24,584.93" />\n  </edge>\n  <edge id=":4900465209_1" function="internal">\n    <lane id=":4900465209_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.93" length="14.36" shape="575.49,579.78 575.56,583.74 576.90,586.67 579.50,588.57 583.36,589.45" />\n  </edge>\n  <edge id=":4900465209_2" function="internal">\n    <lane id=":4900465209_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.42" length="9.04" shape="583.62,586.26 581.19,585.71 579.55,584.51 578.71,582.66 578.67,580.17" />\n  </edge>\n  <edge id=":4900465209_3" function="internal">\n    <lane id=":4900465209_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.44" shape="583.62,586.26 569.24,584.93" />\n  </edge>\n  <edge id=":4900465209_4" function="internal">\n    <lane id=":4900465209_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.51" shape="568.91,588.11 583.36,589.45" />\n  </edge>\n  <edge id=":4900465209_5" function="internal">\n    <lane id=":4900465209_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.08" length="4.11" shape="568.91,588.11 572.76,587.95 572.99,587.85" />\n  </edge>\n  <edge id=":4900465209_6" function="internal">\n    <lane id=":4900465209_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.08" length="10.17" shape="572.99,587.85 575.67,586.58 577.64,583.98 578.67,580.17" />\n  </edge>\n  <edge id=":4900465219_0" function="internal">\n    <lane id=":4900465219_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.38" shape="1157.36,554.43 1143.06,552.99" />\n  </edge>\n  <edge id=":4900465219_1" function="internal">\n    <lane id=":4900465219_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.96" length="4.07" shape="1157.36,554.43 1153.44,554.60 1153.31,554.67" />\n  </edge>\n  <edge id=":4900465219_6" function="internal">\n    <lane id=":4900465219_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.96" length="10.21" shape="1153.31,554.67 1150.55,556.00 1148.67,558.61 1147.82,562.44" />\n  </edge>\n  <edge id=":4900465219_2" function="internal">\n    <lane id=":4900465219_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.46" length="9.03" shape="1151.01,562.68 1151.55,560.26 1152.73,558.61 1154.57,557.73 1157.05,557.62" />\n  </edge>\n  <edge id=":4900465219_3" function="internal">\n    <lane id=":4900465219_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.09" length="14.22" shape="1151.01,562.68 1150.77,558.77 1149.36,555.86 1146.79,553.93 1143.06,552.99" />\n  </edge>\n  <edge id=":4900465219_4" function="internal">\n    <lane id=":4900465219_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.61" length="9.16" shape="1142.73,556.17 1145.12,556.77 1146.76,558.02 1147.66,559.91 1147.82,562.44" />\n  </edge>\n  <edge id=":4900465219_5" function="internal">\n    <lane id=":4900465219_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.39" shape="1142.73,556.17 1157.05,557.62" />\n  </edge>\n  <edge id=":4900465244_0" function="internal">\n    <lane id=":4900465244_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.41" shape="1379.04,647.73 1364.63,647.84" />\n  </edge>\n  <edge id=":4900465244_1" function="internal">\n    <lane id=":4900465244_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.39" length="3.88" shape="1379.04,647.73 1375.36,648.33 1375.23,648.40" />\n  </edge>\n  <edge id=":4900465244_6" function="internal">\n    <lane id=":4900465244_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.39" length="10.48" shape="1375.23,648.40 1372.60,650.03 1370.76,652.84 1369.84,656.76" />\n  </edge>\n  <edge id=":4900465244_2" function="internal">\n    <lane id=":4900465244_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.97" length="9.61" shape="1373.03,657.09 1373.64,654.41 1374.85,652.50 1376.66,651.33 1379.08,650.93" />\n  </edge>\n  <edge id=":4900465244_3" function="internal">\n    <lane id=":4900465244_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.82" length="14.62" shape="1373.03,657.09 1372.86,653.02 1371.40,650.13 1368.66,648.40 1364.63,647.84" />\n  </edge>\n  <edge id=":4900465244_4" function="internal">\n    <lane id=":4900465244_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.29" length="9.04" shape="1364.65,651.04 1367.14,651.39 1368.84,652.46 1369.74,654.25 1369.84,656.76" />\n  </edge>\n  <edge id=":4900465244_5" function="internal">\n    <lane id=":4900465244_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.43" shape="1364.65,651.04 1379.08,650.93" />\n  </edge>\n  <edge id=":4900465288_0" function="internal">\n    <lane id=":4900465288_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.91" length="9.47" shape="1240.78,1542.53 1240.38,1544.95 1239.25,1546.75 1237.38,1547.95 1234.78,1548.55" />\n  </edge>\n  <edge id=":4900465288_1" function="internal">\n    <lane id=":4900465288_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.34" shape="1240.78,1542.53 1240.97,1556.87" />\n  </edge>\n  <edge id=":4900465288_2" function="internal">\n    <lane id=":4900465288_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.19" shape="1244.17,1556.76 1243.98,1542.57" />\n  </edge>\n  <edge id=":4900465288_3" function="internal">\n    <lane id=":4900465288_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.90" length="3.99" shape="1244.17,1556.76 1243.46,1552.83" />\n  </edge>\n  <edge id=":4900465288_6" function="internal">\n    <lane id=":4900465288_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.90" length="10.44" shape="1243.46,1552.83 1241.66,1550.16 1238.76,1548.73 1234.78,1548.55" />\n  </edge>\n  <edge id=":4900465288_4" function="internal">\n    <lane id=":4900465288_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.38" length="9.04" shape="1235.09,1551.73 1237.58,1551.84 1239.40,1552.74 1240.53,1554.42 1240.97,1556.87" />\n  </edge>\n  <edge id=":4900465288_5" function="internal">\n    <lane id=":4900465288_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.34" length="14.23" shape="1235.09,1551.73 1238.94,1550.84 1241.71,1549.01 1243.39,1546.25 1243.98,1542.57" />\n  </edge>\n  <edge id=":4900465363_0" function="internal">\n    <lane id=":4900465363_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.60" length="9.18" shape="570.49,1224.97 567.98,1224.57 566.18,1223.48 565.11,1221.72 564.75,1219.27" />\n  </edge>\n  <edge id=":4900465363_1" function="internal">\n    <lane id=":4900465363_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.55" shape="570.49,1224.97 567.29,1225.14 564.75,1225.57 562.20,1225.94 559.01,1225.91" />\n  </edge>\n  <edge id=":4900465363_2" function="internal">\n    <lane id=":4900465363_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.43" length="9.03" shape="564.83,1233.70 565.16,1231.23 566.19,1229.49 567.95,1228.47 570.42,1228.17" />\n  </edge>\n  <edge id=":4900465363_3" function="internal">\n    <lane id=":4900465363_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.43" shape="564.83,1233.70 564.75,1219.27" />\n  </edge>\n  <edge id=":4900465363_4" function="internal">\n    <lane id=":4900465363_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.33" length="10.81" shape="564.83,1233.70 564.44,1230.49 563.33,1228.12 561.52,1226.60 559.01,1225.91" />\n  </edge>\n  <edge id=":4900465364_0" function="internal">\n    <lane id=":4900465364_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.06" shape="498.76,1214.73 497.18,1225.68" />\n  </edge>\n  <edge id=":4900465364_1" function="internal">\n    <lane id=":4900465364_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.56" length="3.96" shape="503.42,1220.79 500.93,1220.90 499.59,1221.51" />\n  </edge>\n  <edge id=":4900465364_2" function="internal">\n    <lane id=":4900465364_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.56" length="5.02" shape="499.59,1221.51 499.07,1221.75 497.81,1223.35 497.18,1225.68" />\n  </edge>\n  <edge id=":4900465377_0" function="internal">\n    <lane id=":4900465377_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.14" length="9.06" shape="1736.33,893.02 1735.66,895.48 1734.38,896.99 1732.48,897.53 1729.97,897.12" />\n  </edge>\n  <edge id=":4900465377_1" function="internal">\n    <lane id=":4900465377_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.81" length="14.26" shape="1736.33,893.02 1736.23,897.11 1737.00,900.57 1738.64,903.41 1741.14,905.62" />\n  </edge>\n  <edge id=":4900465377_2" function="internal">\n    <lane id=":4900465377_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.55" length="10.68" shape="1742.88,902.93 1741.10,901.32 1739.94,899.19 1739.41,896.55 1739.50,893.40" />\n  </edge>\n  <edge id=":4900465377_3" function="internal">\n    <lane id=":4900465377_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.21" shape="1742.88,902.93 1739.76,901.09 1736.64,899.57 1733.40,898.28 1729.97,897.12" />\n  </edge>\n  <edge id=":4900465377_4" function="internal">\n    <lane id=":4900465377_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.36" shape="1729.00,900.17 1732.32,901.27 1735.28,902.44 1738.15,903.84 1741.14,905.62" />\n  </edge>\n  <edge id=":4900465377_5" function="internal">\n    <lane id=":4900465377_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.70" length="4.09" shape="1729.00,900.17 1733.03,900.83" />\n  </edge>\n  <edge id=":4900465377_6" function="internal">\n    <lane id=":4900465377_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.70" length="10.87" shape="1733.03,900.83 1733.16,900.85 1736.29,899.96 1738.41,897.47 1739.50,893.40" />\n  </edge>\n  <edge id=":4900465441_0" function="internal">\n    <lane id=":4900465441_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.50" length="9.04" shape="1634.21,1342.09 1633.72,1344.53 1632.57,1346.22 1630.76,1347.16 1628.29,1347.35" />\n  </edge>\n  <edge id=":4900465441_1" function="internal">\n    <lane id=":4900465441_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.98" length="14.24" shape="1634.21,1342.09 1634.54,1345.99 1636.04,1348.83 1638.71,1350.61 1642.55,1351.34" />\n  </edge>\n  <edge id=":4900465441_2" function="internal">\n    <lane id=":4900465441_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.48" length="9.03" shape="1642.69,1348.15 1640.26,1347.68 1638.56,1346.55 1637.61,1344.75 1637.40,1342.28" />\n  </edge>\n  <edge id=":4900465441_3" function="internal">\n    <lane id=":4900465441_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.43" shape="1642.69,1348.15 1628.29,1347.35" />\n  </edge>\n  <edge id=":4900465441_4" function="internal">\n    <lane id=":4900465441_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.49" shape="1628.08,1350.54 1642.55,1351.34" />\n  </edge>\n  <edge id=":4900465441_5" function="internal">\n    <lane id=":4900465441_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="4.14" shape="1628.08,1350.54 1631.97,1350.24 1632.18,1350.13" />\n  </edge>\n  <edge id=":4900465441_6" function="internal">\n    <lane id=":4900465441_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="10.09" shape="1632.18,1350.13 1634.82,1348.76 1636.63,1346.11 1637.40,1342.28" />\n  </edge>\n  <edge id=":4900465471_0" function="internal">\n    <lane id=":4900465471_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.44" shape="601.00,713.57 599.20,722.84" />\n  </edge>\n  <edge id=":4900465471_1" function="internal">\n    <lane id=":4900465471_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.99" length="8.04" shape="601.00,713.57 600.67,716.38 600.94,718.43 601.81,719.73 603.29,720.28" />\n  </edge>\n  <edge id=":4900465471_2" function="internal">\n    <lane id=":4900465471_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.34" length="1.08" shape="603.61,717.10 602.55,717.27" />\n  </edge>\n  <edge id=":4900465471_3" function="internal">\n    <lane id=":4900465471_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.34" length="6.86" shape="602.55,717.27 602.11,717.34 600.88,718.37 599.90,720.21 599.20,722.84" />\n  </edge>\n  <edge id=":4900465472_0" function="internal">\n    <lane id=":4900465472_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.04" length="9.86" shape="653.07,716.37 652.17,718.77 650.65,720.39 648.50,721.24 645.74,721.32" />\n  </edge>\n  <edge id=":4900465472_1" function="internal">\n    <lane id=":4900465472_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.72" shape="653.07,716.37 649.94,730.75" />\n  </edge>\n  <edge id=":4900465472_2" function="internal">\n    <lane id=":4900465472_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.21" length="11.84" shape="653.07,716.37 652.62,720.23 653.14,723.12 654.61,725.03 657.04,725.96" />\n  </edge>\n  <edge id=":4900465472_3" function="internal">\n    <lane id=":4900465472_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.87" shape="657.52,722.80 645.74,721.32" />\n  </edge>\n  <edge id=":4900465472_4" function="internal">\n    <lane id=":4900465472_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.65" length="2.76" shape="657.52,722.80 654.94,722.98 654.79,723.07" />\n  </edge>\n  <edge id=":4900465472_7" function="internal">\n    <lane id=":4900465472_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.65" length="9.43" shape="654.79,723.07 652.81,724.36 651.15,726.95 649.94,730.75" />\n  </edge>\n  <edge id=":4900465472_5" function="internal">\n    <lane id=":4900465472_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.26" length="9.04" shape="645.42,724.50 647.87,725.11 649.44,726.35 650.13,728.23 649.94,730.75" />\n  </edge>\n  <edge id=":4900465472_6" function="internal">\n    <lane id=":4900465472_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.70" shape="645.42,724.50 657.04,725.96" />\n  </edge>\n  <edge id=":4900465473_0" function="internal">\n    <lane id=":4900465473_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.03" length="9.65" shape="698.68,723.14 697.70,725.38 696.12,726.87 693.96,727.61 691.21,727.60" />\n  </edge>\n  <edge id=":4900465473_1" function="internal">\n    <lane id=":4900465473_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="698.68,723.14 695.22,737.12" />\n  </edge>\n  <edge id=":4900465473_2" function="internal">\n    <lane id=":4900465473_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.31" shape="698.34,737.85 701.77,723.95" />\n  </edge>\n  <edge id=":4900465473_3" function="internal">\n    <lane id=":4900465473_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.82" length="4.06" shape="698.34,737.85 698.69,733.80" />\n  </edge>\n  <edge id=":4900465473_6" function="internal">\n    <lane id=":4900465473_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.82" length="10.54" shape="698.69,733.80 697.62,730.74 695.12,728.68 691.21,727.60" />\n  </edge>\n  <edge id=":4900465473_4" function="internal">\n    <lane id=":4900465473_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.29" length="9.04" shape="690.81,730.78 693.23,731.44 694.78,732.72 695.44,734.62 695.22,737.12" />\n  </edge>\n  <edge id=":4900465473_5" function="internal">\n    <lane id=":4900465473_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.45" length="14.33" shape="690.81,730.78 694.82,730.77 697.99,729.63 700.31,727.36 701.77,723.95" />\n  </edge>\n  <edge id=":4900465513_0" function="internal">\n    <lane id=":4900465513_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.46" length="4.09" shape="438.25,981.32 438.08,983.80 438.61,985.32" />\n  </edge>\n  <edge id=":4900465513_2" function="internal">\n    <lane id=":4900465513_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.46" length="4.94" shape="438.61,985.32 438.75,985.72 440.26,987.08 442.61,987.89" />\n  </edge>\n  <edge id=":4900465513_1" function="internal">\n    <lane id=":4900465513_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.23" shape="431.60,985.69 442.61,987.89" />\n  </edge>\n  <edge id=":4900465517_0" function="internal">\n    <lane id=":4900465517_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.58" shape="550.93,1381.06 536.43,1379.48" />\n  </edge>\n  <edge id=":4900465517_1" function="internal">\n    <lane id=":4900465517_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.92" length="4.25" shape="550.93,1381.06 546.97,1381.12 546.71,1381.24" />\n  </edge>\n  <edge id=":4900465517_6" function="internal">\n    <lane id=":4900465517_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.92" length="10.11" shape="546.71,1381.24 544.03,1382.44 542.12,1385.03 541.23,1388.89" />\n  </edge>\n  <edge id=":4900465517_2" function="internal">\n    <lane id=":4900465517_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.41" length="9.04" shape="544.42,1389.16 544.98,1386.73 546.18,1385.10 548.03,1384.27 550.52,1384.23" />\n  </edge>\n  <edge id=":4900465517_3" function="internal">\n    <lane id=":4900465517_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.05" length="14.30" shape="544.42,1389.16 544.20,1385.22 542.80,1382.29 540.20,1380.37 536.43,1379.48" />\n  </edge>\n  <edge id=":4900465517_4" function="internal">\n    <lane id=":4900465517_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.56" length="9.16" shape="536.14,1382.66 538.55,1383.24 540.20,1384.47 541.09,1386.35 541.23,1388.89" />\n  </edge>\n  <edge id=":4900465517_5" function="internal">\n    <lane id=":4900465517_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.46" shape="536.14,1382.66 550.52,1384.23" />\n  </edge>\n  <edge id=":4900465706_0" function="internal">\n    <lane id=":4900465706_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.92" length="9.56" shape="408.85,1492.91 406.44,1492.46 404.66,1491.28 403.50,1489.35 402.96,1486.69" />\n  </edge>\n  <edge id=":4900465706_1" function="internal">\n    <lane id=":4900465706_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.43" shape="408.85,1492.91 394.43,1492.52" />\n  </edge>\n  <edge id=":4900465706_2" function="internal">\n    <lane id=":4900465706_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.43" shape="394.34,1495.72 408.77,1496.11" />\n  </edge>\n  <edge id=":4900465706_3" function="internal">\n    <lane id=":4900465706_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.83" length="4.05" shape="394.34,1495.72 398.37,1495.25" />\n  </edge>\n  <edge id=":4900465706_6" function="internal">\n    <lane id=":4900465706_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.83" length="10.53" shape="398.37,1495.25 401.15,1493.59 402.68,1490.73 402.96,1486.69" />\n  </edge>\n  <edge id=":4900465706_4" function="internal">\n    <lane id=":4900465706_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.30" length="9.04" shape="399.77,1486.92 399.60,1489.43 398.65,1491.20 396.92,1492.23 394.43,1492.52" />\n  </edge>\n  <edge id=":4900465706_5" function="internal">\n    <lane id=":4900465706_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.35" length="14.35" shape="399.77,1486.92 400.58,1490.86 402.35,1493.70 405.08,1495.45 408.77,1496.11" />\n  </edge>\n  <edge id=":4900465707_0" function="internal">\n    <lane id=":4900465707_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.23" shape="1234.14,984.16 1222.92,984.06" />\n  </edge>\n  <edge id=":4900465707_1" function="internal">\n    <lane id=":4900465707_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.26" shape="1222.87,987.26 1234.13,987.36" />\n  </edge>\n  <edge id=":4900465707_2" function="internal">\n    <lane id=":4900465707_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.33" length="9.04" shape="1228.24,978.41 1228.07,980.91 1227.12,982.69 1225.40,983.74 1222.92,984.06" />\n  </edge>\n  <edge id=":4900465707_3" function="internal">\n    <lane id=":4900465707_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.56" length="11.82" shape="1228.24,978.41 1228.86,982.32 1230.04,985.11 1231.80,986.79 1234.13,987.36" />\n  </edge>\n  <edge id=":4900465949_0" function="internal">\n    <lane id=":4900465949_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.06" length="11.11" shape="1459.06,457.63 1455.93,456.87 1453.56,455.53 1451.95,453.60 1451.09,451.09" />\n  </edge>\n  <edge id=":4900465949_1" function="internal">\n    <lane id=":4900465949_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.22" shape="1459.06,457.63 1455.76,457.23 1452.98,457.00 1450.18,456.98 1446.87,457.15" />\n  </edge>\n  <edge id=":4900465949_2" function="internal">\n    <lane id=":4900465949_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.99" length="9.08" shape="1453.47,466.05 1453.45,463.45 1454.31,461.71 1456.04,460.83 1458.64,460.80" />\n  </edge>\n  <edge id=":4900465949_3" function="internal">\n    <lane id=":4900465949_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.15" shape="1453.47,466.05 1451.09,451.09" />\n  </edge>\n  <edge id=":4900465949_4" function="internal">\n    <lane id=":4900465949_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.66" length="12.20" shape="1453.47,466.05 1452.54,462.01 1451.13,459.18 1449.24,457.56 1446.87,457.15" />\n  </edge>\n  <edge id=":4900465949_5" function="internal">\n    <lane id=":4900465949_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.57" shape="1447.11,460.34 1450.32,460.17 1452.88,460.19 1455.45,460.40 1458.64,460.80" />\n  </edge>\n  <edge id=":4900465949_6" function="internal">\n    <lane id=":4900465949_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.04" length="2.42" shape="1447.11,460.34 1449.41,459.61" />\n  </edge>\n  <edge id=":4900465949_7" function="internal">\n    <lane id=":4900465949_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.04" length="9.09" shape="1449.41,459.61 1450.84,457.83 1451.40,454.98 1451.09,451.09" />\n  </edge>\n  <edge id=":4900465989_0" function="internal">\n    <lane id=":4900465989_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.98" length="11.85" shape="387.17,1292.99 384.82,1292.29 382.74,1290.70 380.93,1288.22 379.39,1284.85" />\n  </edge>\n  <edge id=":4900465989_1" function="internal">\n    <lane id=":4900465989_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.12" shape="387.17,1292.99 372.12,1291.44" />\n  </edge>\n  <edge id=":4900465989_2" function="internal">\n    <lane id=":4900465989_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.12" shape="371.80,1294.63 386.84,1296.18" />\n  </edge>\n  <edge id=":4900465989_3" function="internal">\n    <lane id=":4900465989_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.44" length="4.84" shape="371.80,1294.63 376.63,1294.46" />\n  </edge>\n  <edge id=":4900465989_6" function="internal">\n    <lane id=":4900465989_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.44" length="11.51" shape="376.63,1294.46 379.51,1292.78 380.43,1289.58 379.39,1284.85" />\n  </edge>\n  <edge id=":4900465989_4" function="internal">\n    <lane id=":4900465989_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.75" length="9.18" shape="376.39,1285.95 376.97,1288.61 376.45,1290.41 374.84,1291.35 372.12,1291.44" />\n  </edge>\n  <edge id=":4900465989_5" function="internal">\n    <lane id=":4900465989_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.18" length="15.42" shape="376.39,1285.95 378.35,1290.16 380.75,1293.26 383.58,1295.27 386.84,1296.18" />\n  </edge>\n  <edge id=":4900465999_0" function="internal">\n    <lane id=":4900465999_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.04" length="10.49" shape="1054.88,913.33 1051.79,913.52 1049.23,913.01 1047.19,911.81 1045.69,909.92" />\n  </edge>\n  <edge id=":4900465999_1" function="internal">\n    <lane id=":4900465999_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.69" length="15.01" shape="1054.88,913.33 1050.83,914.59 1048.43,916.80 1047.68,919.98 1048.58,924.12" />\n  </edge>\n  <edge id=":4900465999_2" function="internal">\n    <lane id=":4900465999_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.12" length="9.06" shape="1051.58,923.01 1051.04,920.51 1051.49,918.59 1052.94,917.25 1055.38,916.49" />\n  </edge>\n  <edge id=":4900465999_3" function="internal">\n    <lane id=":4900465999_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.36" shape="1051.58,923.01 1050.26,919.60 1048.89,916.34 1047.39,913.15 1045.69,909.92" />\n  </edge>\n  <edge id=":4900465999_4" function="internal">\n    <lane id=":4900465999_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.88" shape="1042.89,911.48 1044.56,914.64 1045.99,917.68 1047.30,920.78 1048.58,924.12" />\n  </edge>\n  <edge id=":4900465999_5" function="internal">\n    <lane id=":4900465999_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.33" length="14.41" shape="1042.89,911.48 1045.01,914.20 1047.80,915.94 1051.26,916.71 1055.38,916.49" />\n  </edge>\n  <edge id=":4900466295_0" function="internal">\n    <lane id=":4900466295_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.88" length="9.11" shape="376.58,1576.54 377.18,1579.11 376.72,1580.97 375.18,1582.10 372.56,1582.51" />\n  </edge>\n  <edge id=":4900466295_1" function="internal">\n    <lane id=":4900466295_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.43" shape="376.58,1576.54 380.62,1587.24" />\n  </edge>\n  <edge id=":4900466297_0" function="internal">\n    <lane id=":4900466297_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.35" length="7.71" shape="406.78,1672.81 409.05,1673.26 410.85,1674.05 412.16,1675.18 412.99,1676.65" />\n  </edge>\n  <edge id=":4900466297_1" function="internal">\n    <lane id=":4900466297_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="8.56" shape="410.10,1668.60 412.99,1676.65" />\n  </edge>\n  <edge id=":4900466448_0" function="internal">\n    <lane id=":4900466448_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.88" length="9.61" shape="885.21,1045.71 884.50,1048.23 883.17,1049.98 881.21,1050.97 878.62,1051.19" />\n  </edge>\n  <edge id=":4900466448_1" function="internal">\n    <lane id=":4900466448_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.70" shape="885.21,1045.71 883.60,1060.32" />\n  </edge>\n  <edge id=":4900466448_2" function="internal">\n    <lane id=":4900466448_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.93" length="14.35" shape="885.21,1045.71 885.24,1049.66 886.54,1052.61 889.12,1054.54 892.96,1055.46" />\n  </edge>\n  <edge id=":4900466448_3" function="internal">\n    <lane id=":4900466448_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.42" length="9.04" shape="893.26,1052.28 890.84,1051.70 889.22,1050.48 888.40,1048.63 888.38,1046.14" />\n  </edge>\n  <edge id=":4900466448_4" function="internal">\n    <lane id=":4900466448_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.68" shape="893.26,1052.28 878.62,1051.19" />\n  </edge>\n  <edge id=":4900466448_5" function="internal">\n    <lane id=":4900466448_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.02" length="4.26" shape="893.26,1052.28 889.32,1052.47 889.04,1052.60" />\n  </edge>\n  <edge id=":4900466448_12" function="internal">\n    <lane id=":4900466448_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.02" length="10.10" shape="889.04,1052.60 886.39,1053.87 884.49,1056.49 883.60,1060.32" />\n  </edge>\n  <edge id=":4900466448_6" function="internal">\n    <lane id=":4900466448_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.53" length="9.17" shape="886.79,1060.59 887.36,1058.15 888.57,1056.48 890.44,1055.59 892.96,1055.46" />\n  </edge>\n  <edge id=":4900466448_7" function="internal">\n    <lane id=":4900466448_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.54" shape="886.79,1060.59 888.38,1046.14" />\n  </edge>\n  <edge id=":4900466448_8" function="internal">\n    <lane id=":4900466448_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.95" length="14.30" shape="886.79,1060.59 886.56,1056.66 885.12,1053.79 882.48,1051.96 878.62,1051.19" />\n  </edge>\n  <edge id=":4900466448_9" function="internal">\n    <lane id=":4900466448_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.45" length="9.03" shape="878.44,1054.38 880.88,1054.87 882.55,1056.02 883.46,1057.84 883.60,1060.32" />\n  </edge>\n  <edge id=":4900466448_10" function="internal">\n    <lane id=":4900466448_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.56" shape="878.44,1054.38 892.96,1055.46" />\n  </edge>\n  <edge id=":4900466448_11" function="internal">\n    <lane id=":4900466448_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.32" length="3.92" shape="878.44,1054.38 882.35,1054.05" />\n  </edge>\n  <edge id=":4900466448_13" function="internal">\n    <lane id=":4900466448_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.32" length="10.56" shape="882.35,1054.05 885.30,1052.56 887.31,1049.93 888.38,1046.14" />\n  </edge>\n  <edge id=":5149825984_0" function="internal">\n    <lane id=":5149825984_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.46" length="14.31" shape="1257.97,1170.62 1256.66,1174.63 1254.64,1177.42 1251.90,1179.00 1248.44,1179.36" />\n  </edge>\n  <edge id=":5149825984_1" function="internal">\n    <lane id=":5149825984_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.73" shape="1257.97,1170.62 1256.85,1174.97 1255.18,1180.08 1253.25,1185.10 1251.33,1189.16" />\n  </edge>\n  <edge id=":5149825984_2" function="internal">\n    <lane id=":5149825984_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.49" length="17.53" shape="1257.97,1170.62 1257.47,1176.08 1258.50,1180.10 1261.07,1182.68 1265.17,1183.82" />\n  </edge>\n  <edge id=":5149825984_3" function="internal">\n    <lane id=":5149825984_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.27" length="9.04" shape="1265.71,1177.44 1263.26,1176.87 1261.67,1175.65 1260.95,1173.78 1261.10,1171.26" />\n  </edge>\n  <edge id=":5149825984_4" function="internal">\n    <lane id=":5149825984_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.45" shape="1265.71,1177.44 1260.06,1177.47 1255.70,1178.24 1252.03,1179.09 1248.44,1179.36" />\n  </edge>\n  <edge id=":5149825984_5" function="internal">\n    <lane id=":5149825984_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.60" length="4.49" shape="1265.44,1180.63 1261.19,1180.87 1260.97,1180.96" />\n  </edge>\n  <edge id=":5149825984_12" function="internal">\n    <lane id=":5149825984_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.60" length="13.01" shape="1260.97,1180.96 1257.42,1182.38 1254.14,1185.14 1251.33,1189.16" />\n  </edge>\n  <edge id=":5149825984_6" function="internal">\n    <lane id=":5149825984_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.86" length="12.01" shape="1254.14,1190.70 1255.72,1188.82 1258.04,1187.57 1261.10,1186.97 1264.89,1187.01" />\n  </edge>\n  <edge id=":5149825984_7" function="internal">\n    <lane id=":5149825984_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="20.69" shape="1254.14,1190.70 1256.10,1186.51 1258.15,1181.18 1259.93,1175.75 1261.10,1171.26" />\n  </edge>\n  <edge id=":5149825984_8" function="internal">\n    <lane id=":5149825984_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.53" length="16.63" shape="1254.14,1190.70 1255.97,1186.02 1255.63,1182.57 1253.12,1180.35 1248.44,1179.36" />\n  </edge>\n  <edge id=":5149825984_9" function="internal">\n    <lane id=":5149825984_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.87" length="9.55" shape="1248.24,1182.56 1250.87,1183.13 1252.26,1184.42 1252.42,1186.43 1251.33,1189.16" />\n  </edge>\n  <edge id=":5149825984_10" function="internal">\n    <lane id=":5149825984_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.98" shape="1248.24,1182.56 1265.17,1183.82" />\n  </edge>\n  <edge id=":5149825984_11" function="internal">\n    <lane id=":5149825984_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.67" length="6.85" shape="1248.24,1182.56 1252.94,1182.10 1254.80,1181.09" />\n  </edge>\n  <edge id=":5149825984_13" function="internal">\n    <lane id=":5149825984_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.67" length="12.11" shape="1254.80,1181.09 1256.66,1180.07 1259.38,1176.46 1261.10,1171.26" />\n  </edge>\n  <edge id=":617234280_0" function="internal">\n    <lane id=":617234280_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.75" shape="312.50,636.93 310.36,651.78" />\n    <lane id=":617234280_0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.75" shape="315.65,637.51 313.55,652.11" />\n    <lane id=":617234280_0_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.75" shape="318.79,638.09 316.73,652.44" />\n  </edge>\n  <edge id=":617234280_3" function="internal">\n    <lane id=":617234280_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.36" length="9.02" shape="318.79,638.09 318.70,640.58 319.44,642.45 321.02,643.70 323.43,644.33" />\n  </edge>\n  <edge id=":617234280_4" function="internal">\n    <lane id=":617234280_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.52" length="9.06" shape="305.39,645.57 307.78,646.18 309.41,647.42 310.27,649.28 310.36,651.78" />\n    <lane id=":617234280_4_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.02" length="14.21" shape="305.73,642.39 309.49,643.34 312.05,645.28 313.40,648.20 313.55,652.11" />\n  </edge>\n  <edge id=":617234280_6" function="internal">\n    <lane id=":617234280_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.99" shape="305.73,642.39 310.97,643.45 314.40,644.93 317.84,646.42 323.07,647.51" />\n    <lane id=":617234280_6_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.99" shape="305.73,642.39 323.43,644.33" />\n  </edge>\n  <edge id=":617234281_0" function="internal">\n    <lane id=":617234281_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.74" shape="472.60,650.99 470.49,665.58" />\n  </edge>\n  <edge id=":617234281_1" function="internal">\n    <lane id=":617234281_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.29" length="9.04" shape="472.60,650.99 472.62,653.51 473.46,655.33 475.12,656.46 477.59,656.89" />\n  </edge>\n  <edge id=":617234281_2" function="internal">\n    <lane id=":617234281_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.32" length="9.04" shape="465.62,659.55 468.07,660.06 469.71,661.23 470.51,663.07 470.49,665.58" />\n  </edge>\n  <edge id=":617234281_3" function="internal">\n    <lane id=":617234281_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.84" shape="465.62,659.55 477.49,660.09" />\n    <lane id=":617234281_3_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.84" shape="465.81,656.35 477.59,656.89" />\n  </edge>\n  <edge id=":617234535_0" function="internal">\n    <lane id=":617234535_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.62" length="9.19" shape="340.94,480.53 340.23,482.96 338.93,484.61 337.03,485.46 334.55,485.52" />\n  </edge>\n  <edge id=":617234535_1" function="internal">\n    <lane id=":617234535_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.43" shape="340.94,480.53 339.10,491.70" />\n    <lane id=":617234535_1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.43" shape="344.11,481.00 342.25,492.27" />\n    <lane id=":617234535_1_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.43" shape="347.27,481.46 345.39,492.85" />\n  </edge>\n  <edge id=":617234535_4" function="internal">\n    <lane id=":617234535_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.55" shape="351.97,487.60 334.55,485.52" />\n  </edge>\n  <edge id=":617234535_5" function="internal">\n    <lane id=":617234535_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.58" length="12.02" shape="351.97,487.60 348.11,487.50 345.20,488.25 343.25,489.84 342.25,492.27" />\n    <lane id=":617234535_5_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.80" length="9.47" shape="351.97,487.60 349.50,487.68 347.58,488.58 346.21,490.31 345.39,492.85" />\n  </edge>\n  <edge id=":617234576_0" function="internal">\n    <lane id=":617234576_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.88" length="9.51" shape="389.34,660.61 390.08,658.16 391.43,656.46 393.40,655.51 395.98,655.31" />\n  </edge>\n  <edge id=":617234576_1" function="internal">\n    <lane id=":617234576_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.47" shape="389.34,660.61 391.69,646.42" />\n    <lane id=":617234576_1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.47" shape="386.18,660.13 388.53,645.86" />\n    <lane id=":617234576_1_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.47" shape="383.01,659.65 385.38,645.30" />\n  </edge>\n  <edge id=":617234576_4" function="internal">\n    <lane id=":617234576_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.43" shape="378.53,653.79 395.98,655.31" />\n    <lane id=":617234576_4_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.43" shape="378.89,650.61 396.17,652.11" />\n  </edge>\n  <edge id=":617234576_6" function="internal">\n    <lane id=":617234576_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.57" length="11.98" shape="378.89,650.61 382.73,650.68 385.62,649.91 387.55,648.31 388.53,645.86" />\n    <lane id=":617234576_6_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.78" length="9.43" shape="378.89,650.61 381.34,650.51 383.24,649.59 384.58,647.86 385.38,645.30" />\n  </edge>\n  <edge id=":617234843_0" function="internal">\n    <lane id=":617234843_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="8.85" shape="445.35,839.93 444.69,842.26 443.44,843.84 441.61,844.67 439.20,844.75" />\n  </edge>\n  <edge id=":617234843_1" function="internal">\n    <lane id=":617234843_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="7.08" shape="446.23,845.59 439.20,844.75" />\n  </edge>\n  <edge id=":6634630594_0" function="internal">\n    <lane id=":6634630594_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.64" length="12.47" shape="350.82,1340.87 348.55,1339.90 346.53,1338.03 344.77,1335.26 343.27,1331.59" />\n  </edge>\n  <edge id=":6634630594_1" function="internal">\n    <lane id=":6634630594_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.40" shape="350.82,1340.87 347.09,1340.02 343.28,1339.19 339.45,1338.57 335.66,1338.33" />\n  </edge>\n  <edge id=":6634630594_2" function="internal">\n    <lane id=":6634630594_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.40" shape="334.98,1341.53 338.77,1341.76 342.61,1342.36 346.42,1343.16 350.15,1344.00" />\n  </edge>\n  <edge id=":6634630594_3" function="internal">\n    <lane id=":6634630594_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.68" length="3.91" shape="334.98,1341.53 338.86,1341.02" />\n  </edge>\n  <edge id=":6634630594_6" function="internal">\n    <lane id=":6634630594_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.68" length="12.21" shape="338.86,1341.02 339.84,1340.88 342.84,1339.01 343.98,1335.91 343.27,1331.59" />\n  </edge>\n  <edge id=":6634630594_4" function="internal">\n    <lane id=":6634630594_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.94" length="9.13" shape="340.23,1332.60 340.66,1335.09 340.04,1336.88 338.37,1337.96 335.66,1338.33" />\n  </edge>\n  <edge id=":6634630594_5" function="internal">\n    <lane id=":6634630594_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.79" length="15.78" shape="340.23,1332.60 342.10,1337.06 344.38,1340.44 347.06,1342.75 350.15,1344.00" />\n  </edge>\n  <edge id=":6634630596_0" function="internal">\n    <lane id=":6634630596_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.43" length="13.93" shape="336.25,1287.75 332.66,1286.93 329.78,1285.28 327.59,1282.81 326.09,1279.50" />\n  </edge>\n  <edge id=":6634630596_1" function="internal">\n    <lane id=":6634630596_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.80" shape="336.25,1287.75 318.54,1285.92" />\n  </edge>\n  <edge id=":6634630596_2" function="internal">\n    <lane id=":6634630596_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.46" length="4.75" shape="336.25,1287.75 331.49,1287.91" />\n  </edge>\n  <edge id=":6634630596_12" function="internal">\n    <lane id=":6634630596_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.46" length="11.41" shape="331.49,1287.91 328.62,1289.58 327.62,1292.75 328.49,1297.43" />\n  </edge>\n  <edge id=":6634630596_3" function="internal">\n    <lane id=":6634630596_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.78" length="9.15" shape="331.52,1296.41 331.03,1293.76 331.60,1291.97 333.23,1291.02 335.92,1290.93" />\n  </edge>\n  <edge id=":6634630596_4" function="internal">\n    <lane id=":6634630596_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.76" shape="331.52,1296.41 326.09,1279.50" />\n  </edge>\n  <edge id=":6634630596_5" function="internal">\n    <lane id=":6634630596_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.59" length="17.33" shape="331.52,1296.41 329.52,1292.85 326.28,1289.58 322.42,1287.11 318.54,1285.92" />\n  </edge>\n  <edge id=":6634630596_6" function="internal">\n    <lane id=":6634630596_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.58" length="14.04" shape="318.21,1289.11 321.77,1289.93 324.67,1291.59 326.91,1294.09 328.49,1297.43" />\n  </edge>\n  <edge id=":6634630596_7" function="internal">\n    <lane id=":6634630596_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.80" shape="318.21,1289.11 335.92,1290.93" />\n  </edge>\n  <edge id=":6634630596_8" function="internal">\n    <lane id=":6634630596_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.48" length="4.69" shape="318.21,1289.11 322.91,1288.94" />\n  </edge>\n  <edge id=":6634630596_13" function="internal">\n    <lane id=":6634630596_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.48" length="11.33" shape="322.91,1288.94 325.78,1287.28 326.84,1284.14 326.09,1279.50" />\n  </edge>\n  <edge id=":6634630596_9" function="internal">\n    <lane id=":6634630596_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.81" length="9.14" shape="323.03,1280.45 323.46,1283.09 322.86,1284.88 321.22,1285.83 318.54,1285.92" />\n  </edge>\n  <edge id=":6634630596_10" function="internal">\n    <lane id=":6634630596_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.83" shape="323.03,1280.45 328.49,1297.43" />\n  </edge>\n  <edge id=":6634630596_11" function="internal">\n    <lane id=":6634630596_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.45" length="17.28" shape="323.03,1280.45 324.96,1284.01 328.15,1287.27 332.01,1289.74 335.92,1290.93" />\n  </edge>\n  <edge id=":796747634_0" function="internal">\n    <lane id=":796747634_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.49" length="10.25" shape="1488.81,903.42 1491.80,903.91 1494.05,905.01 1495.57,906.72 1496.36,909.02" />\n  </edge>\n  <edge id=":796747634_1" function="internal">\n    <lane id=":796747634_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.08" length="9.06" shape="1494.02,894.65 1494.07,897.22 1493.25,899.00 1491.56,900.01 1489.00,900.22" />\n  </edge>\n  <edge id=":796747634_2" function="internal">\n    <lane id=":796747634_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.56" shape="1494.02,894.65 1496.36,909.02" />\n  </edge>\n  <edge id=":796747714_0" function="internal">\n    <lane id=":796747714_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.15" length="9.85" shape="1447.86,650.20 1450.69,650.51 1452.84,651.48 1454.30,653.13 1455.07,655.44" />\n  </edge>\n  <edge id=":796747714_1" function="internal">\n    <lane id=":796747714_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.20" length="9.05" shape="1452.75,641.16 1452.79,643.69 1451.99,645.51 1450.33,646.62 1447.83,647.00" />\n  </edge>\n  <edge id=":796747714_2" function="internal">\n    <lane id=":796747714_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.47" shape="1452.75,641.16 1455.07,655.44" />\n  </edge>\n  <edge id=":809537690_0" function="internal">\n    <lane id=":809537690_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.30" length="9.04" shape="1568.63,1182.37 1568.65,1179.85 1569.49,1178.03 1571.15,1176.90 1573.63,1176.46" />\n  </edge>\n  <edge id=":809537690_1" function="internal">\n    <lane id=":809537690_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.31" shape="1568.63,1182.37 1567.05,1171.17" />\n  </edge>\n  <edge id=":809537690_2" function="internal">\n    <lane id=":809537690_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.74" shape="1562.96,1177.56 1566.00,1177.03 1568.26,1176.73 1570.54,1176.57 1573.63,1176.46" />\n  </edge>\n  <edge id=":809537690_3" function="internal">\n    <lane id=":809537690_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.44" length="3.39" shape="1562.96,1177.56 1565.05,1176.82 1565.90,1176.01" />\n  </edge>\n  <edge id=":809537690_4" function="internal">\n    <lane id=":809537690_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.44" length="5.19" shape="1565.90,1176.01 1566.43,1175.51 1567.10,1173.62 1567.05,1171.17" />\n  </edge>\n  <edge id=":809537820_0" function="internal">\n    <lane id=":809537820_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.75" length="8.74" shape="1571.37,1164.60 1569.01,1164.36 1567.20,1163.46 1565.94,1161.90 1565.21,1159.68" />\n  </edge>\n  <edge id=":809537820_1" function="internal">\n    <lane id=":809537820_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.44" shape="1571.37,1164.60 1568.33,1164.68 1566.15,1164.74 1563.98,1164.95 1560.98,1165.46" />\n  </edge>\n  <edge id=":809537820_2" function="internal">\n    <lane id=":809537820_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.44" shape="1566.88,1169.99 1565.21,1159.68" />\n  </edge>\n  <edge id=":809537820_3" function="internal">\n    <lane id=":809537820_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.27" length="8.56" shape="1566.88,1169.99 1566.21,1167.65 1565.00,1166.11 1563.26,1165.38 1560.98,1165.46" />\n  </edge>\n  <edge id=":8348961008_0" function="internal">\n    <lane id=":8348961008_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.54" length="9.08" shape="318.69,1200.55 316.32,1199.77 314.76,1198.43 314.01,1196.52 314.08,1194.05" />\n  </edge>\n  <edge id=":8348961008_1" function="internal">\n    <lane id=":8348961008_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.49" length="9.03" shape="311.66,1208.25 312.42,1205.89 313.75,1204.34 315.65,1203.62 318.13,1203.70" />\n  </edge>\n  <edge id=":8348961008_2" function="internal">\n    <lane id=":8348961008_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="311.66,1208.25 314.08,1194.05" />\n    <lane id=":8348961008_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="308.50,1207.71 310.92,1193.51" />\n    <lane id=":8348961008_2_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="305.35,1207.17 307.77,1192.98" />\n  </edge>\n  <edge id=":8348961009_0" function="internal">\n    <lane id=":8348961009_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.53" length="9.06" shape="425.37,1214.01 424.58,1216.36 423.23,1217.89 421.31,1218.61 418.82,1218.52" />\n  </edge>\n  <edge id=":8348961009_1" function="internal">\n    <lane id=":8348961009_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="425.37,1214.01 422.76,1228.18" />\n  </edge>\n  <edge id=":8348961009_2" function="internal">\n    <lane id=":8348961009_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="425.91,1228.75 428.52,1214.60" />\n  </edge>\n  <edge id=":8348961009_3" function="internal">\n    <lane id=":8348961009_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="4.07" shape="425.91,1228.75 426.06,1224.86 426.00,1224.70" />\n  </edge>\n  <edge id=":8348961009_6" function="internal">\n    <lane id=":8348961009_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="10.14" shape="426.00,1224.70 424.93,1221.86 422.52,1219.74 418.82,1218.52" />\n  </edge>\n  <edge id=":8348961009_4" function="internal">\n    <lane id=":8348961009_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.50" length="9.03" shape="418.26,1221.67 420.61,1222.45 422.14,1223.79 422.86,1225.70 422.76,1228.18" />\n  </edge>\n  <edge id=":8348961009_5" function="internal">\n    <lane id=":8348961009_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.02" length="14.20" shape="418.26,1221.67 422.15,1221.81 425.16,1220.68 427.28,1218.27 428.52,1214.60" />\n  </edge>\n  <edge id=":8363308288_0" function="internal">\n    <lane id=":8363308288_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="8.80" shape="1361.52,1127.81 1361.03,1130.15 1359.90,1131.81 1358.15,1132.77 1355.77,1133.04" />\n  </edge>\n  <edge id=":8363308288_1" function="internal">\n    <lane id=":8363308288_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.14" length="13.76" shape="1361.52,1127.81 1361.79,1131.56 1363.15,1134.41 1365.60,1136.37 1369.14,1137.44" />\n  </edge>\n  <edge id=":8363308288_2" function="internal">\n    <lane id=":8363308288_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.66" length="8.91" shape="1369.62,1134.27 1367.34,1133.58 1365.76,1132.31 1364.88,1130.46 1364.71,1128.02" />\n  </edge>\n  <edge id=":8363308288_3" function="internal">\n    <lane id=":8363308288_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.92" shape="1369.62,1134.27 1366.07,1133.79 1362.71,1133.43 1359.34,1133.19 1355.77,1133.04" />\n  </edge>\n  <edge id=":8363308288_4" function="internal">\n    <lane id=":8363308288_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.53" shape="1355.67,1136.23 1359.19,1136.38 1362.43,1136.61 1365.66,1136.96 1369.14,1137.44" />\n  </edge>\n  <edge id=":8363308288_5" function="internal">\n    <lane id=":8363308288_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.01" length="3.89" shape="1355.67,1136.23 1359.42,1135.81 1359.53,1135.76" />\n  </edge>\n  <edge id=":8363308288_6" function="internal">\n    <lane id=":8363308288_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.01" length="9.93" shape="1359.53,1135.76 1362.17,1134.31 1363.94,1131.71 1364.71,1128.02" />\n  </edge>\n  <edge id=":8523326825_0" function="internal">\n    <lane id=":8523326825_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.30" shape="1596.41,880.48 1582.25,878.49" />\n  </edge>\n  <edge id=":8523326825_1" function="internal">\n    <lane id=":8523326825_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="3.90" shape="1596.41,880.48 1592.51,880.53" />\n  </edge>\n  <edge id=":8523326825_6" function="internal">\n    <lane id=":8523326825_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.99" length="10.32" shape="1592.51,880.53 1589.57,881.82 1587.59,884.35 1586.58,888.12" />\n  </edge>\n  <edge id=":8523326825_2" function="internal">\n    <lane id=":8523326825_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.49" length="9.03" shape="1589.76,888.51 1590.40,886.11 1591.66,884.51 1593.52,883.69 1596.00,883.65" />\n  </edge>\n  <edge id=":8523326825_3" function="internal">\n    <lane id=":8523326825_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.08" length="14.18" shape="1589.76,888.51 1589.68,884.61 1588.41,881.65 1585.93,879.61 1582.25,878.49" />\n  </edge>\n  <edge id=":8523326825_4" function="internal">\n    <lane id=":8523326825_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.60" length="9.12" shape="1581.77,881.66 1584.13,882.37 1585.72,883.69 1586.54,885.60 1586.58,888.12" />\n  </edge>\n  <edge id=":8523326825_5" function="internal">\n    <lane id=":8523326825_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.37" shape="1581.77,881.66 1596.00,883.65" />\n  </edge>\n  <edge id=":8523326826_0" function="internal">\n    <lane id=":8523326826_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="3.90" length="2.57" shape="1583.09,916.80 1582.90,917.48 1582.55,917.94 1582.02,918.18 1581.31,918.20" />\n  </edge>\n  <edge id=":8523326826_1" function="internal">\n    <lane id=":8523326826_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.08" length="7.71" shape="1580.95,921.38 1583.06,921.32 1584.65,920.60 1585.71,919.22 1586.26,917.19" />\n  </edge>\n  <edge id=":8523328986_0" function="internal">\n    <lane id=":8523328986_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.01" length="9.68" shape="1560.84,1099.21 1558.08,1099.01 1555.97,1098.11 1554.53,1096.50 1553.75,1094.18" />\n  </edge>\n  <edge id=":8523328986_1" function="internal">\n    <lane id=":8523328986_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.25" length="9.05" shape="1556.21,1108.42 1556.13,1105.90 1556.90,1104.06 1558.53,1102.89 1561.00,1102.41" />\n  </edge>\n  <edge id=":8523328986_2" function="internal">\n    <lane id=":8523328986_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.46" shape="1556.21,1108.42 1553.75,1094.18" />\n  </edge>\n  <edge id=":8523334139_0" function="internal">\n    <lane id=":8523334139_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.17" shape="622.79,1022.70 632.79,1024.55" />\n  </edge>\n  <edge id=":8523334139_1" function="internal">\n    <lane id=":8523334139_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.63" length="8.49" shape="622.79,1022.70 625.01,1022.73 626.81,1022.03 628.19,1020.60 629.14,1018.44" />\n  </edge>\n  <edge id=":8523344319_0" function="internal">\n    <lane id=":8523344319_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="7.33" shape="453.66,846.84 446.43,845.62" />\n  </edge>\n  <edge id=":8523344319_1" function="internal">\n    <lane id=":8523344319_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.52" length="9.01" shape="453.66,846.84 451.20,846.77 449.31,847.51 447.97,849.06 447.19,851.41" />\n  </edge>\n  <edge id=":8523431062_0" function="internal">\n    <lane id=":8523431062_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.69" length="10.46" shape="987.09,1322.93 986.65,1326.01 985.61,1328.38 983.96,1330.02 981.71,1330.95" />\n  </edge>\n  <edge id=":8523431062_1" function="internal">\n    <lane id=":8523431062_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.63" length="15.24" shape="987.09,1322.93 987.54,1327.25 989.28,1330.03 992.32,1331.27 996.65,1330.97" />\n  </edge>\n  <edge id=":8523431062_2" function="internal">\n    <lane id=":8523431062_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.03" length="9.07" shape="995.98,1327.84 993.40,1328.02 991.59,1327.28 990.55,1325.63 990.29,1323.05" />\n  </edge>\n  <edge id=":8523431062_3" function="internal">\n    <lane id=":8523431062_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.60" shape="995.98,1327.84 981.71,1330.95" />\n  </edge>\n  <edge id=":8523431062_4" function="internal">\n    <lane id=":8523431062_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.58" shape="982.40,1334.07 996.65,1330.97" />\n  </edge>\n  <edge id=":8523431062_5" function="internal">\n    <lane id=":8523431062_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.02" length="3.70" shape="982.40,1334.07 985.72,1332.76 985.82,1332.66" />\n  </edge>\n  <edge id=":8523431062_6" function="internal">\n    <lane id=":8523431062_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.02" length="10.99" shape="985.82,1332.66 988.13,1330.48 989.66,1327.25 990.29,1323.05" />\n  </edge>\n  <edge id=":8523431064_0" function="internal">\n    <lane id=":8523431064_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.44" length="9.03" shape="968.09,1333.98 965.61,1334.16 963.69,1333.51 962.32,1332.00 961.51,1329.65" />\n  </edge>\n  <edge id=":8523431064_1" function="internal">\n    <lane id=":8523431064_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.33" shape="968.09,1333.98 954.17,1337.41" />\n  </edge>\n  <edge id=":8523431064_2" function="internal">\n    <lane id=":8523431064_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.18" shape="955.01,1340.49 968.78,1337.10" />\n  </edge>\n  <edge id=":8523431064_3" function="internal">\n    <lane id=":8523431064_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.23" length="3.78" shape="955.01,1340.49 958.47,1338.98" />\n  </edge>\n  <edge id=":8523431064_6" function="internal">\n    <lane id=":8523431064_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.23" length="10.40" shape="958.47,1338.98 960.71,1336.67 961.72,1333.56 961.51,1329.65" />\n  </edge>\n  <edge id=":8523431064_4" function="internal">\n    <lane id=":8523431064_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.78" length="9.30" shape="958.37,1330.25 958.52,1332.84 957.87,1334.90 956.42,1336.42 954.17,1337.41" />\n  </edge>\n  <edge id=":8523431064_5" function="internal">\n    <lane id=":8523431064_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.95" length="14.31" shape="958.37,1330.25 959.65,1333.97 961.81,1336.36 964.86,1337.40 968.78,1337.10" />\n  </edge>\n  <edge id=":8523449430_0" function="internal">\n    <lane id=":8523449430_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.61" length="9.17" shape="1681.53,1344.20 1681.00,1346.69 1679.83,1348.43 1678.01,1349.42 1675.56,1349.66" />\n  </edge>\n  <edge id=":8523449430_1" function="internal">\n    <lane id=":8523449430_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.95" length="14.30" shape="1681.53,1344.20 1681.80,1348.13 1683.27,1350.99 1685.93,1352.79 1689.80,1353.52" />\n  </edge>\n  <edge id=":8523449430_2" function="internal">\n    <lane id=":8523449430_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.45" length="9.03" shape="1689.94,1350.33 1687.50,1349.86 1685.82,1348.72 1684.89,1346.92 1684.72,1344.44" />\n  </edge>\n  <edge id=":8523449430_3" function="internal">\n    <lane id=":8523449430_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="1689.94,1350.33 1675.56,1349.66" />\n  </edge>\n  <edge id=":8523449430_4" function="internal">\n    <lane id=":8523449430_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="1675.41,1352.86 1689.80,1353.52" />\n  </edge>\n  <edge id=":8523449430_5" function="internal">\n    <lane id=":8523449430_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.09" length="4.02" shape="1675.41,1352.86 1679.24,1352.48 1679.39,1352.40" />\n  </edge>\n  <edge id=":8523449430_6" function="internal">\n    <lane id=":8523449430_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.09" length="10.21" shape="1679.39,1352.40 1682.07,1350.95 1683.89,1348.27 1684.72,1344.44" />\n  </edge>\n  <edge id=":856214470_0" function="internal">\n    <lane id=":856214470_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.69" shape="667.97,1037.41 670.71,1027.08" />\n  </edge>\n  <edge id=":856214470_1" function="internal">\n    <lane id=":856214470_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.57" length="3.96" shape="664.00,1031.04 666.41,1031.21 667.88,1030.75" />\n  </edge>\n  <edge id=":856214470_2" function="internal">\n    <lane id=":856214470_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.57" length="4.81" shape="667.88,1030.75 668.34,1030.60 669.77,1029.23 670.71,1027.08" />\n  </edge>\n  <edge id=":856214503_0" function="internal">\n    <lane id=":856214503_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.34" length="9.05" shape="708.21,912.80 705.82,912.04 704.31,910.71 703.69,908.79 703.94,906.30" />\n  </edge>\n  <edge id=":856214503_1" function="internal">\n    <lane id=":856214503_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.46" shape="708.21,912.80 705.06,912.25 702.59,911.71 700.15,911.01 697.12,909.99" />\n  </edge>\n  <edge id=":856214503_2" function="internal">\n    <lane id=":856214503_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.96" length="9.59" shape="700.25,920.28 701.27,918.02 702.85,916.55 704.99,915.86 707.70,915.95" />\n  </edge>\n  <edge id=":856214503_3" function="internal">\n    <lane id=":856214503_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.46" shape="700.25,920.28 703.94,906.30" />\n  </edge>\n  <edge id=":856214503_4" function="internal">\n    <lane id=":856214503_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.56" length="11.98" shape="700.25,920.28 700.93,916.49 700.64,913.51 699.37,911.35 697.12,909.99" />\n  </edge>\n  <edge id=":856214503_5" function="internal">\n    <lane id=":856214503_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.01" shape="696.07,913.02 699.19,914.05 701.81,914.80 704.46,915.39 707.70,915.95" />\n  </edge>\n  <edge id=":856214503_6" function="internal">\n    <lane id=":856214503_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.16" length="2.65" shape="696.07,913.02 698.70,913.34" />\n  </edge>\n  <edge id=":856214503_7" function="internal">\n    <lane id=":856214503_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.16" length="9.24" shape="698.70,913.34 700.89,912.32 702.64,909.98 703.94,906.30" />\n  </edge>\n  <edge id=":856214524_0" function="internal">\n    <lane id=":856214524_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.75" length="9.37" shape="1344.05,984.42 1341.53,983.95 1339.72,982.80 1338.62,980.98 1338.23,978.48" />\n  </edge>\n  <edge id=":856214524_1" function="internal">\n    <lane id=":856214524_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.67" shape="1344.05,984.42 1329.38,984.18" />\n  </edge>\n  <edge id=":856214524_2" function="internal">\n    <lane id=":856214524_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.97" length="4.30" shape="1344.05,984.42 1340.14,984.80 1339.81,984.98" />\n  </edge>\n  <edge id=":856214524_12" function="internal">\n    <lane id=":856214524_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.97" length="10.01" shape="1339.81,984.98 1337.32,986.35 1335.59,989.08 1334.95,992.96" />\n  </edge>\n  <edge id=":856214524_3" function="internal">\n    <lane id=":856214524_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.47" length="9.06" shape="1338.15,993.04 1338.55,990.57 1339.65,988.84 1341.43,987.86 1343.90,987.62" />\n  </edge>\n  <edge id=":856214524_4" function="internal">\n    <lane id=":856214524_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.56" shape="1338.15,993.04 1338.23,978.48" />\n  </edge>\n  <edge id=":856214524_5" function="internal">\n    <lane id=":856214524_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.94" length="14.33" shape="1338.15,993.04 1337.67,989.12 1336.05,986.34 1333.29,984.69 1329.38,984.18" />\n  </edge>\n  <edge id=":856214524_6" function="internal">\n    <lane id=":856214524_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.43" length="9.03" shape="1329.42,987.38 1331.88,987.70 1333.63,988.74 1334.65,990.49 1334.95,992.96" />\n  </edge>\n  <edge id=":856214524_7" function="internal">\n    <lane id=":856214524_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.49" shape="1329.42,987.38 1343.90,987.62" />\n  </edge>\n  <edge id=":856214524_8" function="internal">\n    <lane id=":856214524_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.03" length="3.94" shape="1329.42,987.38 1333.31,986.78" />\n  </edge>\n  <edge id=":856214524_13" function="internal">\n    <lane id=":856214524_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.03" length="10.34" shape="1333.31,986.78 1336.08,985.10 1337.72,982.33 1338.23,978.48" />\n  </edge>\n  <edge id=":856214524_9" function="internal">\n    <lane id=":856214524_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.54" length="9.13" shape="1335.03,978.51 1334.70,980.97 1333.65,982.73 1331.88,983.80 1329.38,984.18" />\n  </edge>\n  <edge id=":856214524_10" function="internal">\n    <lane id=":856214524_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.45" shape="1335.03,978.51 1334.95,992.96" />\n  </edge>\n  <edge id=":856214524_11" function="internal">\n    <lane id=":856214524_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.20" length="14.32" shape="1335.03,978.51 1335.62,982.35 1337.30,985.14 1340.06,986.90 1343.90,987.62" />\n  </edge>\n  <edge id=":856214538_0" function="internal">\n    <lane id=":856214538_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.07" length="9.07" shape="895.56,964.48 895.05,967.00 893.86,968.56 891.99,969.15 889.45,968.78" />\n  </edge>\n  <edge id=":856214538_1" function="internal">\n    <lane id=":856214538_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.76" shape="895.56,964.48 894.20,979.18" />\n  </edge>\n  <edge id=":856214538_2" function="internal">\n    <lane id=":856214538_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.09" length="14.91" shape="895.56,964.48 895.88,968.72 897.39,971.78 900.09,973.66 903.97,974.36" />\n  </edge>\n  <edge id=":856214538_3" function="internal">\n    <lane id=":856214538_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.62" length="9.62" shape="904.05,971.16 901.60,970.70 899.90,969.47 898.95,967.45 898.76,964.66" />\n  </edge>\n  <edge id=":856214538_4" function="internal">\n    <lane id=":856214538_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.83" shape="904.05,971.16 900.33,970.94 896.67,970.46 893.05,969.73 889.45,968.78" />\n  </edge>\n  <edge id=":856214538_5" function="internal">\n    <lane id=":856214538_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.33" length="3.92" shape="904.05,971.16 900.15,971.58" />\n  </edge>\n  <edge id=":856214538_12" function="internal">\n    <lane id=":856214538_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.33" length="10.24" shape="900.15,971.58 897.21,973.06 895.23,975.59 894.20,979.18" />\n  </edge>\n  <edge id=":856214538_6" function="internal">\n    <lane id=":856214538_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.90" length="9.40" shape="897.37,979.60 898.05,977.26 899.38,975.60 901.36,974.63 903.97,974.36" />\n  </edge>\n  <edge id=":856214538_7" function="internal">\n    <lane id=":856214538_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.01" shape="897.37,979.60 898.76,964.66" />\n  </edge>\n  <edge id=":856214538_8" function="internal">\n    <lane id=":856214538_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.65" length="14.82" shape="897.37,979.60 897.29,975.87 895.94,972.82 893.32,970.46 889.45,968.78" />\n  </edge>\n  <edge id=":856214538_9" function="internal">\n    <lane id=":856214538_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.28" length="10.21" shape="888.54,971.85 891.29,973.02 893.15,974.63 894.12,976.68 894.20,979.18" />\n  </edge>\n  <edge id=":856214538_10" function="internal">\n    <lane id=":856214538_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.67" shape="888.54,971.85 892.27,972.82 896.18,973.59 900.13,974.12 903.97,974.36" />\n  </edge>\n  <edge id=":856214538_11" function="internal">\n    <lane id=":856214538_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.65" length="4.59" shape="888.54,971.85 892.80,972.47 893.08,972.38" />\n  </edge>\n  <edge id=":856214538_13" function="internal">\n    <lane id=":856214538_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.65" length="10.55" shape="893.08,972.38 895.92,971.48 897.90,968.87 898.76,964.66" />\n  </edge>\n  <edge id=":856214571_0" function="internal">\n    <lane id=":856214571_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.28" shape="1548.91,1200.20 1545.83,1200.07 1543.46,1199.30 1540.97,1198.62 1537.50,1198.78" />\n    <lane id=":856214571_0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.28" shape="1548.91,1200.20 1538.12,1201.92" />\n  </edge>\n  <edge id=":856214571_2" function="internal">\n    <lane id=":856214571_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.87" length="14.14" shape="1548.91,1200.20 1546.82,1201.21 1545.61,1203.62 1545.27,1207.45 1545.80,1212.70" />\n  </edge>\n  <edge id=":856214571_3" function="internal">\n    <lane id=":856214571_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.44" length="9.03" shape="1539.34,1208.20 1541.83,1208.08 1543.73,1208.79 1545.06,1210.33 1545.80,1212.70" />\n  </edge>\n  <edge id=":856214571_4" function="internal">\n    <lane id=":856214571_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.69" shape="1538.73,1205.06 1549.29,1203.38" />\n  </edge>\n  <edge id=":856214571_5" function="internal">\n    <lane id=":856214571_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.65" length="9.32" shape="1542.38,1192.16 1542.42,1194.63 1541.63,1196.56 1539.99,1197.94 1537.50,1198.78" />\n  </edge>\n  <edge id=":856214571_6" function="internal">\n    <lane id=":856214571_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="20.82" shape="1542.38,1192.16 1545.80,1212.70" />\n  </edge>\n  <edge id=":856214571_7" function="internal">\n    <lane id=":856214571_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.17" length="14.26" shape="1542.38,1192.16 1543.56,1197.29 1545.11,1200.87 1547.02,1202.89 1549.29,1203.38" />\n  </edge>\n  <edge id=":856214578_0" function="internal">\n    <lane id=":856214578_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.23" length="9.05" shape="343.35,973.21 343.89,970.74 345.09,969.14 346.95,968.42 349.47,968.58" />\n  </edge>\n  <edge id=":856214578_1" function="internal">\n    <lane id=":856214578_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.86" shape="343.35,973.21 344.61,961.68" />\n    <lane id=":856214578_1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.86" shape="340.16,972.98 341.44,961.20" />\n    <lane id=":856214578_1_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.86" shape="336.97,972.76 338.28,960.72" />\n  </edge>\n  <edge id=":856214578_4" function="internal">\n    <lane id=":856214578_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.85" shape="331.90,965.41 349.47,968.58" />\n  </edge>\n  <edge id=":856214578_5" function="internal">\n    <lane id=":856214578_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.33" length="11.72" shape="331.90,965.41 335.76,965.64 338.64,965.02 340.53,963.54 341.44,961.20" />\n    <lane id=":856214578_5_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="9.03" shape="331.90,965.41 334.38,965.43 336.26,964.66 337.56,963.09 338.28,960.72" />\n  </edge>\n  <edge id=":8642733850_0" function="internal">\n    <lane id=":8642733850_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="1652.66,420.29 1639.51,414.42" />\n  </edge>\n  <edge id=":8642733850_1" function="internal">\n    <lane id=":8642733850_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.93" length="4.10" shape="1652.66,420.29 1648.86,419.21 1648.72,419.23" />\n  </edge>\n  <edge id=":8642733850_6" function="internal">\n    <lane id=":8642733850_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.93" length="10.24" shape="1648.72,419.23 1645.67,419.63 1643.09,421.55 1641.11,424.97" />\n  </edge>\n  <edge id=":8642733850_2" function="internal">\n    <lane id=":8642733850_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.43" length="9.03" shape="1644.08,426.16 1645.32,424.01 1646.95,422.80 1648.96,422.53 1651.36,423.22" />\n  </edge>\n  <edge id=":8642733850_3" function="internal">\n    <lane id=":8642733850_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.13" length="14.25" shape="1644.08,426.16 1645.03,422.34 1644.59,419.12 1642.74,416.48 1639.51,414.42" />\n  </edge>\n  <edge id=":8642733850_4" function="internal">\n    <lane id=":8642733850_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.66" length="9.23" shape="1638.20,417.35 1640.28,418.67 1641.46,420.38 1641.73,422.48 1641.11,424.97" />\n  </edge>\n  <edge id=":8642733850_5" function="internal">\n    <lane id=":8642733850_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="1638.20,417.35 1651.36,423.22" />\n  </edge>\n  <edge id=":8642866006_0" function="internal">\n    <lane id=":8642866006_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.08" length="9.80" shape="1485.00,718.75 1482.19,718.50 1480.05,717.56 1478.59,715.93 1477.82,713.61" />\n  </edge>\n  <edge id=":8642866006_1" function="internal">\n    <lane id=":8642866006_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.24" length="9.06" shape="1480.15,727.88 1480.10,725.35 1480.91,723.52 1482.56,722.39 1485.06,721.95" />\n  </edge>\n  <edge id=":8642866006_2" function="internal">\n    <lane id=":8642866006_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.46" shape="1480.15,727.88 1477.82,713.61" />\n    <lane id=":8642866006_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.46" shape="1476.99,728.39 1474.66,714.13" />\n  </edge>\n  <edge id=":8642866010_0" function="internal">\n    <lane id=":8642866010_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.25" shape="1446.67,457.17 1446.93,457.15" />\n  </edge>\n  <edge id=":8642866010_1" function="internal">\n    <lane id=":8642866010_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.34" shape="1447.25,460.34 1446.91,460.36" />\n  </edge>\n  <edge id=":8976428985_0" function="internal">\n    <lane id=":8976428985_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.11" length="9.73" shape="898.79,685.16 897.20,682.78 896.34,680.95 896.06,678.97 896.25,676.13" />\n  </edge>\n  <edge id=":8976428985_1" function="internal">\n    <lane id=":8976428985_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.55" shape="894.90,685.59 896.25,676.13" />\n  </edge>\n  <edge id=":8976429008_0" function="internal">\n    <lane id=":8976429008_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.07" length="9.07" shape="1500.48,757.52 1500.47,754.95 1501.32,753.18 1503.03,752.22 1505.60,752.06" />\n  </edge>\n  <edge id=":8976429008_1" function="internal">\n    <lane id=":8976429008_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.32" shape="1494.30,751.19 1505.60,752.06" />\n  </edge>\n  <edge id=":8976429009_0" function="internal">\n    <lane id=":8976429009_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.96" length="9.64" shape="1380.03,745.70 1380.95,743.10 1382.38,741.33 1384.32,740.38 1386.76,740.27" />\n  </edge>\n  <edge id=":8976429009_1" function="internal">\n    <lane id=":8976429009_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.26" length="9.05" shape="1372.37,738.78 1374.83,739.38 1376.40,740.62 1377.09,742.49 1376.90,745.01" />\n  </edge>\n  <edge id=":8976429009_2" function="internal">\n    <lane id=":8976429009_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.47" shape="1372.37,738.78 1386.76,740.27" />\n  </edge>\n  <edge id=":916327686_0" function="internal">\n    <lane id=":916327686_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="25.09" shape="450.77,1697.29 444.81,1694.20 439.71,1691.75 435.60,1688.13 432.63,1681.53" />\n    <lane id=":916327686_0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="25.49" shape="450.77,1697.29 444.43,1694.70 438.48,1693.36 433.53,1690.84 430.17,1684.71" />\n    <lane id=":916327686_0_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="24.91" shape="450.77,1697.29 445.08,1694.50 440.03,1693.06 434.64,1691.31 427.96,1687.58" />\n  </edge>\n  <edge id=":916327686_3" function="internal">\n    <lane id=":916327686_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="25.15" shape="450.77,1697.29 444.56,1694.11 438.34,1692.47 432.12,1692.37 425.89,1693.81" />\n    <lane id=":916327686_3_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="25.15" shape="449.05,1699.99 443.44,1697.17 437.68,1695.78 431.80,1695.80 425.78,1697.24" />\n  </edge>\n  <edge id=":916327702_0" function="internal">\n    <lane id=":916327702_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.52" length="8.78" shape="347.80,1517.98 350.55,1518.24 352.71,1518.92 354.27,1520.04 355.25,1521.59" />\n  </edge>\n  <edge id=":916327702_1" function="internal">\n    <lane id=":916327702_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.10" shape="351.28,1510.16 355.25,1521.59" />\n  </edge>\n  <edge id=":916327810_0" function="internal">\n    <lane id=":916327810_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.95" length="9.09" shape="321.33,1415.54 321.80,1418.11 321.25,1419.96 319.68,1421.09 317.10,1421.50" />\n    <lane id=":916327810_0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.83" length="12.37" shape="321.33,1415.54 322.20,1419.50 321.79,1422.35 320.11,1424.08 317.16,1424.70" />\n  </edge>\n  <edge id=":916327810_2" function="internal">\n    <lane id=":916327810_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.72" shape="321.33,1415.54 325.87,1429.54" />\n  </edge>\n  <edge id=":916327839_0" function="internal">\n    <lane id=":916327839_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.61" length="8.50" shape="407.76,1711.74 406.95,1713.87 405.63,1715.31 403.82,1716.06 401.51,1716.13" />\n  </edge>\n  <edge id=":916327839_1" function="internal">\n    <lane id=":916327839_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.05" shape="407.76,1711.74 406.81,1715.60 406.24,1718.30 406.12,1721.02 406.53,1724.94" />\n    <lane id=":916327839_1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.05" shape="410.88,1712.46 409.99,1716.12 409.47,1718.68 409.37,1721.26 409.77,1724.98" />\n  </edge>\n  <edge id=":916327839_3" function="internal">\n    <lane id=":916327839_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.94" length="8.45" shape="401.17,1719.31 403.24,1719.85 404.83,1720.97 405.92,1722.66 406.53,1724.94" />\n  </edge>\n  <edge id=":916327888_0" function="internal">\n    <lane id=":916327888_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.80" length="6.38" shape="467.12,1725.55 465.78,1724.19 464.85,1723.22 464.21,1722.10 463.76,1720.31" />\n  </edge>\n  <edge id=":916327888_1" function="internal">\n    <lane id=":916327888_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="6.32" shape="462.98,1726.87 463.56,1724.96 463.91,1723.62 464.00,1722.26 463.76,1720.31" />\n    <lane id=":916327888_1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.63" length="6.32" shape="459.73,1726.63 460.26,1724.94 460.65,1723.76 460.79,1722.57 460.61,1720.85" />\n  </edge>\n  <edge id=":916327912_0" function="internal">\n    <lane id=":916327912_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.30" length="11.23" shape="374.75,1492.00 371.33,1491.61 368.64,1490.64 366.71,1489.09 365.52,1486.96" />\n  </edge>\n  <edge id=":916327912_1" function="internal">\n    <lane id=":916327912_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.85" length="9.12" shape="370.50,1501.00 369.94,1498.40 370.45,1496.57 372.03,1495.50 374.67,1495.20" />\n  </edge>\n  <edge id=":916327912_2" function="internal">\n    <lane id=":916327912_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.98" shape="370.50,1501.00 365.52,1486.96" />\n    <lane id=":916327912_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.98" shape="367.49,1502.10 362.48,1487.98" />\n    <lane id=":916327912_2_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.98" shape="364.49,1503.20 359.45,1489.00" />\n  </edge>\n  <edge id=":916327921_0" function="internal">\n    <lane id=":916327921_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.71" shape="416.98,1690.08 416.63,1693.79 415.11,1697.73 412.66,1702.33 409.47,1708.05" />\n  </edge>\n  <edge id=":916327921_1" function="internal">\n    <lane id=":916327921_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.66" shape="423.81,1694.55 418.31,1696.83 414.92,1699.29 412.38,1702.75 409.47,1708.05" />\n    <lane id=":916327921_1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.66" shape="425.59,1697.31 420.52,1699.38 417.35,1701.59 414.98,1704.73 412.28,1709.58" />\n  </edge>\n  <edge id=":947896950_0" function="internal">\n    <lane id=":947896950_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.27" length="8.95" shape="899.63,1350.84 897.18,1351.05 895.32,1350.39 894.05,1348.84 893.37,1346.42" />\n  </edge>\n  <edge id=":947896950_1" function="internal">\n    <lane id=":947896950_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.08" shape="899.63,1350.84 888.84,1353.34" />\n  </edge>\n  <edge id=":947896950_2" function="internal">\n    <lane id=":947896950_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.81" shape="889.60,1356.45 900.13,1354.01" />\n  </edge>\n  <edge id=":947896950_3" function="internal">\n    <lane id=":947896950_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.62" length="2.36" shape="889.60,1356.45 891.69,1355.37" />\n  </edge>\n  <edge id=":947896950_4" function="internal">\n    <lane id=":947896950_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.62" length="9.40" shape="891.69,1355.37 893.02,1353.34 893.58,1350.36 893.37,1346.42" />\n  </edge>\n  <edge id=":947896956_0" function="internal">\n    <lane id=":947896956_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.33" length="9.52" shape="479.43,1368.47 478.71,1370.90 477.31,1372.38 475.23,1372.90 472.46,1372.46" />\n  </edge>\n  <edge id=":947896956_1" function="internal">\n    <lane id=":947896956_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.67" shape="479.43,1368.47 478.71,1372.09 477.70,1375.59 476.70,1379.09 475.98,1382.71" />\n  </edge>\n  <edge id=":947896956_2" function="internal">\n    <lane id=":947896956_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.58" length="9.43" shape="479.43,1368.47 479.36,1372.45 480.33,1375.39 482.04,1377.00" />\n  </edge>\n  <edge id=":947896956_9" function="internal">\n    <lane id=":947896956_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.58" length="3.56" shape="482.04,1377.00 482.34,1377.28 485.38,1378.12" />\n  </edge>\n  <edge id=":947896956_3" function="internal">\n    <lane id=":947896956_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.45" shape="485.66,1374.94 482.17,1374.56 478.99,1374.06 475.85,1373.38 472.46,1372.46" />\n  </edge>\n  <edge id=":947896956_4" function="internal">\n    <lane id=":947896956_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.08" length="14.04" shape="485.66,1374.94 481.84,1375.13 478.95,1376.50 477.00,1379.02 475.98,1382.71" />\n  </edge>\n  <edge id=":947896956_5" function="internal">\n    <lane id=":947896956_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.60" length="9.03" shape="479.15,1383.12 479.81,1380.75 481.06,1379.13 482.92,1378.25 485.38,1378.12" />\n  </edge>\n  <edge id=":947896956_6" function="internal">\n    <lane id=":947896956_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.41" length="3.59" shape="479.15,1383.12 479.14,1379.54" />\n  </edge>\n  <edge id=":947896956_10" function="internal">\n    <lane id=":947896956_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.41" length="10.31" shape="479.14,1379.54 479.14,1379.32 478.02,1376.27 475.79,1373.99 472.46,1372.46" />\n  </edge>\n  <edge id=":947896956_7" function="internal">\n    <lane id=":947896956_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.99" length="9.31" shape="471.56,1375.53 473.77,1376.55 475.24,1378.09 475.97,1380.14 475.98,1382.71" />\n  </edge>\n  <edge id=":947896956_8" function="internal">\n    <lane id=":947896956_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.08" shape="471.56,1375.53 475.04,1376.47 478.40,1377.19 481.79,1377.73 485.38,1378.12" />\n  </edge>\n  <edge id=":947896958_0" function="internal">\n    <lane id=":947896958_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.20" length="9.05" shape="1575.33,1229.65 1575.30,1227.11 1576.11,1225.30 1577.77,1224.20 1580.28,1223.83" />\n  </edge>\n  <edge id=":947896958_1" function="internal">\n    <lane id=":947896958_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.26" shape="1575.33,1229.65 1573.65,1218.52" />\n  </edge>\n  <edge id=":947896964_0" function="internal">\n    <lane id=":947896964_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.60" length="10.30" shape="1017.79,1318.37 1017.24,1320.74 1015.90,1322.61 1013.77,1323.98 1010.86,1324.85" />\n  </edge>\n  <edge id=":947896964_1" function="internal">\n    <lane id=":947896964_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.52" shape="1017.79,1318.37 1017.16,1332.87" />\n  </edge>\n  <edge id=":947896964_2" function="internal">\n    <lane id=":947896964_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.42" shape="1020.36,1332.97 1020.98,1318.56" />\n  </edge>\n  <edge id=":947896964_3" function="internal">\n    <lane id=":947896964_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.67" length="15.08" shape="1020.36,1332.97 1019.87,1328.73 1018.13,1325.96 1015.12,1324.67 1010.86,1324.85" />\n  </edge>\n  <edge id=":947896964_4" function="internal">\n    <lane id=":947896964_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.09" length="9.06" shape="1011.45,1328.00 1014.01,1327.89 1015.82,1328.66 1016.87,1330.32 1017.16,1332.87" />\n  </edge>\n  <edge id=":947896964_5" function="internal">\n    <lane id=":947896964_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.94" length="14.58" shape="1011.45,1328.00 1015.44,1326.77 1018.36,1324.79 1020.21,1322.05 1020.98,1318.56" />\n  </edge>\n  <edge id=":947896972_0" function="internal">\n    <lane id=":947896972_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.46" length="10.22" shape="1636.35,1568.02 1633.38,1567.55 1631.13,1566.46 1629.61,1564.77 1628.81,1562.46" />\n  </edge>\n  <edge id=":947896972_1" function="internal">\n    <lane id=":947896972_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.09" length="9.06" shape="1631.20,1576.82 1631.14,1574.26 1631.95,1572.47 1633.64,1571.45 1636.19,1571.22" />\n  </edge>\n  <edge id=":947896972_2" function="internal">\n    <lane id=":947896972_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.55" shape="1631.20,1576.82 1628.81,1562.46" />\n  </edge>\n  <edge id=":947896979_0" function="internal">\n    <lane id=":947896979_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.39" shape="1614.50,1477.76 1612.92,1468.50" />\n  </edge>\n  <edge id=":947896979_1" function="internal">\n    <lane id=":947896979_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.39" length="7.90" shape="1614.50,1477.76 1613.81,1475.13 1612.86,1473.28 1611.65,1472.21 1610.18,1471.93" />\n  </edge>\n  <edge id=":947896979_2" function="internal">\n    <lane id=":947896979_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.99" length="8.01" shape="1610.39,1475.12 1611.87,1474.62 1612.79,1473.35 1613.14,1471.31 1612.92,1468.50" />\n  </edge>\n  <edge id=":947896981_0" function="internal">\n    <lane id=":947896981_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.44" length="9.03" shape="1657.60,1700.69 1655.12,1700.83 1653.21,1700.14 1651.88,1698.60 1651.13,1696.23" />\n  </edge>\n  <edge id=":947896981_1" function="internal">\n    <lane id=":947896981_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.38" shape="1657.60,1700.69 1646.44,1702.89" />\n  </edge>\n  <edge id=":947896981_2" function="internal">\n    <lane id=":947896981_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.67" length="9.29" shape="1653.50,1710.54 1653.45,1708.03 1654.22,1706.08 1655.82,1704.68 1658.23,1703.83" />\n  </edge>\n  <edge id=":947896981_3" function="internal">\n    <lane id=":947896981_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.50" shape="1653.50,1710.54 1651.13,1696.23" />\n  </edge>\n  <edge id=":947896981_4" function="internal">\n    <lane id=":947896981_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.27" length="11.77" shape="1653.50,1710.54 1652.52,1706.79 1651.01,1704.26 1648.99,1702.96 1646.44,1702.89" />\n  </edge>\n  <edge id=":947896981_5" function="internal">\n    <lane id=":947896981_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.41" shape="1647.04,1706.03 1658.23,1703.83" />\n  </edge>\n  <edge id=":947896981_6" function="internal">\n    <lane id=":947896981_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.43" length="2.64" shape="1647.04,1706.03 1649.37,1705.02 1649.43,1704.94" />\n  </edge>\n  <edge id=":947896981_7" function="internal">\n    <lane id=":947896981_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.43" length="9.24" shape="1649.43,1704.94 1650.83,1703.05 1651.42,1700.12 1651.13,1696.23" />\n  </edge>\n  <edge id=":947896989_0" function="internal">\n    <lane id=":947896989_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.84" length="10.90" shape="1624.07,1492.98 1621.01,1492.40 1618.68,1491.16 1617.07,1489.29 1616.17,1486.76" />\n  </edge>\n  <edge id=":947896989_1" function="internal">\n    <lane id=":947896989_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.67" shape="1624.07,1492.98 1620.69,1492.75 1617.74,1492.64 1614.79,1492.64 1611.41,1492.75" />\n  </edge>\n  <edge id=":947896989_2" function="internal">\n    <lane id=":947896989_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.05" length="9.07" shape="1618.79,1501.67 1618.74,1499.09 1619.56,1497.32 1621.25,1496.34 1623.83,1496.17" />\n  </edge>\n  <edge id=":947896989_3" function="internal">\n    <lane id=":947896989_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.14" shape="1618.79,1501.67 1616.17,1486.76" />\n  </edge>\n  <edge id=":947896989_4" function="internal">\n    <lane id=":947896989_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.96" length="12.74" shape="1618.79,1501.67 1617.76,1497.67 1616.19,1494.86 1614.07,1493.21 1611.41,1492.75" />\n  </edge>\n  <edge id=":947896989_5" function="internal">\n    <lane id=":947896989_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.25" shape="1611.59,1495.95 1614.91,1495.84 1617.71,1495.84 1620.51,1495.94 1623.83,1496.17" />\n  </edge>\n  <edge id=":947896989_6" function="internal">\n    <lane id=":947896989_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.06" length="11.97" shape="1611.59,1495.95 1614.22,1495.27 1615.86,1493.52 1616.51,1490.68 1616.17,1486.76" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_0" function="internal">\n    <lane id=":cluster_1107773211_1110416556_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.65" length="27.48" shape="768.49,510.23 766.95,516.28 763.47,517.56 758.04,514.07 750.67,505.81" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_1" function="internal">\n    <lane id=":cluster_1107773211_1110416556_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.37" length="24.04" shape="768.49,510.23 766.94,511.94 762.57,513.04 755.38,513.51 745.37,513.36" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_2" function="internal">\n    <lane id=":cluster_1107773211_1110416556_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.15" length="13.90" shape="768.49,510.23 768.76,513.99 770.15,516.85 772.66,518.81 776.28,519.87" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_3" function="internal">\n    <lane id=":cluster_1107773211_1110416556_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.69" length="9.03" shape="776.74,516.70 774.39,516.01 772.76,514.74 771.86,512.88 771.68,510.44" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_4" function="internal">\n    <lane id=":cluster_1107773211_1110416556_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="28.46" shape="776.74,516.70 771.01,515.25 763.01,512.46 755.35,509.07 750.67,505.81" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_5" function="internal">\n    <lane id=":cluster_1107773211_1110416556_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.55" shape="776.74,516.70 770.45,515.90 761.07,514.84 751.68,513.87 745.37,513.36" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_6" function="internal">\n    <lane id=":cluster_1107773211_1110416556_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.95" length="19.79" shape="759.40,523.32 759.46,520.74 762.29,519.31 767.90,519.01 776.28,519.87" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_7" function="internal">\n    <lane id=":cluster_1107773211_1110416556_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.42" shape="759.40,523.32 760.45,520.06 764.69,517.02 769.35,513.91 771.68,510.44" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_8" function="internal">\n    <lane id=":cluster_1107773211_1110416556_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.60" shape="759.40,523.32 757.65,519.19 755.46,514.31 753.06,509.56 750.67,505.81" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_9" function="internal">\n    <lane id=":cluster_1107773211_1110416556_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.94" length="18.24" shape="759.40,523.32 757.01,519.11 753.88,516.05 750.00,514.13 745.37,513.36" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_10" function="internal">\n    <lane id=":cluster_1107773211_1110416556_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.08" shape="745.37,516.56 751.61,517.06 760.84,518.02 770.06,519.08 776.28,519.87" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_11" function="internal">\n    <lane id=":cluster_1107773211_1110416556_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.26" length="8.18" shape="745.37,516.56 753.56,516.57" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_12" function="internal">\n    <lane id=":cluster_1107773211_1110416556_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.36" length="4.39" shape="745.37,516.56 749.75,516.21" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_16" function="internal">\n    <lane id=":cluster_1107773211_1110416556_16_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.26" length="20.34" shape="753.56,516.57 756.70,516.57 764.87,515.55 769.86,513.51 771.68,510.44" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_17" function="internal">\n    <lane id=":cluster_1107773211_1110416556_17_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.36" length="13.84" shape="749.75,516.21 751.02,516.11 753.78,514.16 753.67,510.73 750.67,505.81" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_13" function="internal">\n    <lane id=":cluster_1107773211_1110416556_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.50" length="9.48" shape="748.15,507.79 749.71,510.34 749.76,512.12 748.31,513.12 745.37,513.36" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_14" function="internal">\n    <lane id=":cluster_1107773211_1110416556_14_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.80" shape="748.15,507.79 753.15,511.23 761.52,515.03 770.23,518.23 776.28,519.87" />\n  </edge>\n  <edge id=":cluster_1107773211_1110416556_15" function="internal">\n    <lane id=":cluster_1107773211_1110416556_15_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.20" length="31.30" shape="748.15,507.79 755.13,514.41 762.63,517.92 768.78,517.02 771.68,510.44" />\n  </edge>\n  <edge id=":cluster_1110416379_1680666461_0" function="internal">\n    <lane id=":cluster_1110416379_1680666461_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.62" length="9.19" shape="931.51,341.32 930.80,343.68 929.48,345.27 927.55,346.09 924.99,346.14" />\n  </edge>\n  <edge id=":cluster_1110416379_1680666461_1" function="internal">\n    <lane id=":cluster_1110416379_1680666461_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.26" shape="931.51,341.32 927.39,375.33" />\n  </edge>\n  <edge id=":cluster_1110416379_1680666461_2" function="internal">\n    <lane id=":cluster_1110416379_1680666461_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.15" length="12.69" shape="931.51,341.32 931.19,349.55 931.59,353.99" />\n  </edge>\n  <edge id=":cluster_1110416379_1680666461_12" function="internal">\n    <lane id=":cluster_1110416379_1680666461_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.15" length="17.73" shape="931.59,353.99 932.05,359.12 933.93,367.08 936.69,370.51" />\n  </edge>\n  <edge id=":cluster_1110416379_1680666461_3" function="internal">\n    <lane id=":cluster_1110416379_1680666461_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.51" length="26.91" shape="936.87,367.31 934.54,365.59 933.40,360.76 933.44,352.82 934.67,341.78" />\n  </edge>\n  <edge id=":cluster_1110416379_1680666461_4" function="internal">\n    <lane id=":cluster_1110416379_1680666461_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="24.74" shape="936.87,367.31 933.61,363.96 930.92,356.84 928.24,349.66 924.99,346.14" />\n  </edge>\n  <edge id=":cluster_1110416379_1680666461_5" function="internal">\n    <lane id=":cluster_1110416379_1680666461_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.10" length="14.00" shape="936.87,367.31 933.07,367.63 930.23,369.07 928.34,371.63 927.39,375.33" />\n  </edge>\n  <edge id=":cluster_1110416379_1680666461_6" function="internal">\n    <lane id=":cluster_1110416379_1680666461_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.62" length="9.03" shape="930.58,375.67 931.18,373.29 932.40,371.64 934.24,370.71 936.69,370.51" />\n  </edge>\n  <edge id=":cluster_1110416379_1680666461_7" function="internal">\n    <lane id=":cluster_1110416379_1680666461_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.15" shape="930.58,375.67 934.67,341.78" />\n  </edge>\n  <edge id=":cluster_1110416379_1680666461_8" function="internal">\n    <lane id=":cluster_1110416379_1680666461_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.59" length="11.28" shape="930.58,375.67 930.65,367.40 930.33,364.41" />\n  </edge>\n  <edge id=":cluster_1110416379_1680666461_13" function="internal">\n    <lane id=":cluster_1110416379_1680666461_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.59" length="19.42" shape="930.33,364.41 929.59,357.77 927.63,349.71 924.99,346.14" />\n  </edge>\n  <edge id=":cluster_1110416379_1680666461_9" function="internal">\n    <lane id=":cluster_1110416379_1680666461_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.94" length="27.25" shape="924.62,349.32 926.87,351.19 928.09,356.15 928.26,364.19 927.39,375.33" />\n  </edge>\n  <edge id=":cluster_1110416379_1680666461_10" function="internal">\n    <lane id=":cluster_1110416379_1680666461_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="24.84" shape="924.62,349.32 927.90,352.84 930.64,360.02 933.39,367.16 936.69,370.51" />\n  </edge>\n  <edge id=":cluster_1110416379_1680666461_11" function="internal">\n    <lane id=":cluster_1110416379_1680666461_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.09" length="14.25" shape="924.62,349.32 928.55,349.24 931.54,347.95 933.58,345.47 934.67,341.78" />\n  </edge>\n  <edge id=":cluster_1115585393_13472440789_947896971_0" function="internal">\n    <lane id=":cluster_1115585393_13472440789_947896971_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.06" length="11.43" shape="1026.00,1151.31 1025.60,1155.05 1024.49,1157.65 1022.68,1159.11 1020.18,1159.43" />\n  </edge>\n  <edge id=":cluster_1115585393_13472440789_947896971_1" function="internal">\n    <lane id=":cluster_1115585393_13472440789_947896971_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.43" shape="1026.00,1151.31 1026.71,1155.51 1028.30,1160.25 1029.80,1164.98 1030.24,1169.17" />\n  </edge>\n  <edge id=":cluster_1115585393_13472440789_947896971_2" function="internal">\n    <lane id=":cluster_1115585393_13472440789_947896971_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="12.80" length="35.69" shape="1026.00,1151.31 1027.83,1156.42 1033.47,1160.47 1042.93,1163.46 1056.19,1165.38" />\n  </edge>\n  <edge id=":cluster_1115585393_13472440789_947896971_3" function="internal">\n    <lane id=":cluster_1115585393_13472440789_947896971_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.31" length="9.04" shape="1056.49,1162.20 1054.04,1161.61 1052.45,1160.39 1051.70,1158.52 1051.81,1156.02" />\n  </edge>\n  <edge id=":cluster_1115585393_13472440789_947896971_4" function="internal">\n    <lane id=":cluster_1115585393_13472440789_947896971_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="36.41" shape="1056.49,1162.20 1020.18,1159.43" />\n  </edge>\n  <edge id=":cluster_1115585393_13472440789_947896971_5" function="internal">\n    <lane id=":cluster_1115585393_13472440789_947896971_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.12" length="28.06" shape="1056.49,1162.20 1048.72,1162.30 1040.23,1163.67 1033.31,1166.04 1030.24,1169.17" />\n  </edge>\n  <edge id=":cluster_1115585393_13472440789_947896971_6" function="internal">\n    <lane id=":cluster_1115585393_13472440789_947896971_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.41" length="24.46" shape="1033.43,1169.51 1035.10,1166.94 1039.45,1165.40 1046.48,1164.88 1056.19,1165.38" />\n  </edge>\n  <edge id=":cluster_1115585393_13472440789_947896971_7" function="internal">\n    <lane id=":cluster_1115585393_13472440789_947896971_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="23.16" shape="1033.43,1169.51 1036.39,1165.99 1042.47,1162.74 1048.62,1159.50 1051.81,1156.02" />\n  </edge>\n  <edge id=":cluster_1115585393_13472440789_947896971_8" function="internal">\n    <lane id=":cluster_1115585393_13472440789_947896971_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.06" length="18.76" shape="1033.43,1169.51 1032.95,1165.56 1030.59,1162.57 1026.33,1160.52 1020.18,1159.43" />\n  </edge>\n  <edge id=":cluster_1115585393_13472440789_947896971_9" function="internal">\n    <lane id=":cluster_1115585393_13472440789_947896971_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.79" length="13.60" shape="1019.91,1162.62 1024.65,1163.38 1027.96,1164.73 1029.82,1166.66 1030.24,1169.17" />\n  </edge>\n  <edge id=":cluster_1115585393_13472440789_947896971_10" function="internal">\n    <lane id=":cluster_1115585393_13472440789_947896971_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="36.39" shape="1019.91,1162.62 1056.19,1165.38" />\n  </edge>\n  <edge id=":cluster_1115585393_13472440789_947896971_11" function="internal">\n    <lane id=":cluster_1115585393_13472440789_947896971_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="12.44" length="33.12" shape="1019.91,1162.62 1028.53,1162.44 1038.91,1161.02 1047.77,1158.75 1051.81,1156.02" />\n  </edge>\n  <edge id=":cluster_1122072658_14138648129_0" function="internal">\n    <lane id=":cluster_1122072658_14138648129_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="12.90" length="23.04" shape="715.88,511.27 708.44,510.14 702.61,508.23 698.39,505.52 695.76,502.02" />\n  </edge>\n  <edge id=":cluster_1122072658_14138648129_1" function="internal">\n    <lane id=":cluster_1122072658_14138648129_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="26.99" shape="715.88,511.27 688.99,508.93" />\n  </edge>\n  <edge id=":cluster_1122072658_14138648129_2" function="internal">\n    <lane id=":cluster_1122072658_14138648129_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.65" length="9.15" shape="709.33,519.60 710.01,517.17 711.27,515.50 713.14,514.59 715.59,514.46" />\n  </edge>\n  <edge id=":cluster_1122072658_14138648129_3" function="internal">\n    <lane id=":cluster_1122072658_14138648129_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="22.48" shape="709.33,519.60 707.79,515.40 703.59,510.65 698.87,505.98 695.76,502.02" />\n  </edge>\n  <edge id=":cluster_1122072658_14138648129_4" function="internal">\n    <lane id=":cluster_1122072658_14138648129_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.08" length="23.96" shape="709.33,519.60 707.47,516.17 702.24,512.86 695.47,510.26 688.99,508.93" />\n  </edge>\n  <edge id=":cluster_1122072658_14138648129_5" function="internal">\n    <lane id=":cluster_1122072658_14138648129_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="26.99" shape="688.71,512.12 715.59,514.46" />\n  </edge>\n  <edge id=":cluster_1122072658_14138648129_6" function="internal">\n    <lane id=":cluster_1122072658_14138648129_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.40" length="4.94" shape="688.71,512.12 693.64,511.89" />\n  </edge>\n  <edge id=":cluster_1122072658_14138648129_9" function="internal">\n    <lane id=":cluster_1122072658_14138648129_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.40" length="11.87" shape="693.64,511.89 693.75,511.89 696.61,510.13 697.28,506.84 695.76,502.02" />\n  </edge>\n  <edge id=":cluster_1122072658_14138648129_7" function="internal">\n    <lane id=":cluster_1122072658_14138648129_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.67" length="9.24" shape="692.87,503.38 693.70,506.03 693.33,507.84 691.76,508.80 688.99,508.93" />\n  </edge>\n  <edge id=":cluster_1122072658_14138648129_8" function="internal">\n    <lane id=":cluster_1122072658_14138648129_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.65" length="25.90" shape="692.87,503.38 696.17,507.00 701.91,510.41 708.81,513.08 715.59,514.46" />\n  </edge>\n  <edge id=":cluster_1123399587_8976429007_0" function="internal">\n    <lane id=":cluster_1123399587_8976429007_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.10" shape="1108.46,708.23 1090.45,706.42" />\n  </edge>\n  <edge id=":cluster_1123399587_8976429007_1" function="internal">\n    <lane id=":cluster_1123399587_8976429007_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="7.11" shape="1108.46,708.23 1104.26,709.04 1101.76,710.37" />\n  </edge>\n  <edge id=":cluster_1123399587_8976429007_6" function="internal">\n    <lane id=":cluster_1123399587_8976429007_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.55" shape="1101.76,710.37 1099.58,711.53 1095.05,714.64 1091.33,717.32" />\n  </edge>\n  <edge id=":cluster_1123399587_8976429007_2" function="internal">\n    <lane id=":cluster_1123399587_8976429007_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.46" shape="1093.13,719.97 1096.49,717.69 1100.22,715.46 1104.11,713.85 1107.94,713.44" />\n  </edge>\n  <edge id=":cluster_1123399587_8976429007_3" function="internal">\n    <lane id=":cluster_1123399587_8976429007_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.13" length="18.42" shape="1093.13,719.97 1096.46,716.14 1097.03,711.86 1094.98,708.26 1090.45,706.42" />\n  </edge>\n  <edge id=":cluster_1123399587_8976429007_4" function="internal">\n    <lane id=":cluster_1123399587_8976429007_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.45" length="10.12" shape="1089.91,711.82 1093.17,712.43 1094.49,713.55 1093.88,715.18 1091.33,717.32" />\n  </edge>\n  <edge id=":cluster_1123399587_8976429007_5" function="internal">\n    <lane id=":cluster_1123399587_8976429007_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.10" shape="1089.91,711.82 1107.94,713.44" />\n  </edge>\n  <edge id=":cluster_1123399715_3376084911_3376091818_0" function="internal">\n    <lane id=":cluster_1123399715_3376084911_3376091818_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="22.77" shape="1015.15,1382.82 1013.65,1386.61 1010.37,1390.79 1006.53,1395.81 1003.35,1402.11" />\n  </edge>\n  <edge id=":cluster_1123399715_3376084911_3376091818_1" function="internal">\n    <lane id=":cluster_1123399715_3376084911_3376091818_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.16" length="17.79" shape="1028.28,1395.35 1023.72,1394.07 1020.54,1391.58 1018.75,1387.88 1018.35,1382.98" />\n  </edge>\n  <edge id=":cluster_1123399715_3376084911_3376091818_2" function="internal">\n    <lane id=":cluster_1123399715_3376084911_3376091818_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.91" length="27.67" shape="1028.28,1395.35 1018.61,1394.72 1011.23,1395.63 1006.14,1398.10 1003.35,1402.11" />\n  </edge>\n  <edge id=":cluster_1123399715_3376084911_3376091818_3" function="internal">\n    <lane id=":cluster_1123399715_3376084911_3376091818_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.00" length="9.08" shape="996.21,1394.93 998.75,1395.45 1000.29,1396.64 1000.81,1398.51 1000.32,1401.06" />\n  </edge>\n  <edge id=":cluster_11292971032_2406954578_2406954579_2406954580_0" function="internal">\n    <lane id=":cluster_11292971032_2406954578_2406954579_2406954580_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.45" length="9.02" shape="1557.66,1166.08 1555.18,1166.19 1553.28,1165.48 1551.96,1163.94 1551.21,1161.57" />\n  </edge>\n  <edge id=":cluster_11292971032_2406954578_2406954579_2406954580_1" function="internal">\n    <lane id=":cluster_11292971032_2406954578_2406954579_2406954580_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.50" length="30.01" shape="1557.66,1166.08 1549.48,1168.75 1544.06,1173.12 1541.40,1179.21 1541.49,1187.00" />\n  </edge>\n  <edge id=":cluster_11292971032_2406954578_2406954579_2406954580_2" function="internal">\n    <lane id=":cluster_11292971032_2406954578_2406954579_2406954580_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.65" length="9.24" shape="1554.99,1184.70 1554.98,1182.22 1555.79,1180.30 1557.41,1178.94 1559.84,1178.14" />\n  </edge>\n  <edge id=":cluster_11292971032_2406954578_2406954579_2406954580_3" function="internal">\n    <lane id=":cluster_11292971032_2406954578_2406954579_2406954580_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="23.44" shape="1554.99,1184.70 1551.21,1161.57" />\n    <lane id=":cluster_11292971032_2406954578_2406954579_2406954580_3_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="23.44" shape="1551.83,1185.24 1548.06,1162.09" />\n  </edge>\n  <edge id=":cluster_11292971032_2406954578_2406954579_2406954580_5" function="internal">\n    <lane id=":cluster_11292971032_2406954578_2406954579_2406954580_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="23.51" shape="1537.83,1163.78 1541.49,1187.00" />\n  </edge>\n  <edge id=":cluster_11292971032_2406954578_2406954579_2406954580_6" function="internal">\n    <lane id=":cluster_11292971032_2406954578_2406954579_2406954580_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.36" length="17.76" shape="1537.83,1163.78 1540.30,1171.38 1544.79,1176.30 1547.73,1177.32" />\n  </edge>\n  <edge id=":cluster_11292971032_2406954578_2406954579_2406954580_7" function="internal">\n    <lane id=":cluster_11292971032_2406954578_2406954579_2406954580_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.36" length="12.33" shape="1547.73,1177.32 1551.31,1178.56 1559.84,1178.14" />\n  </edge>\n  <edge id=":cluster_13424220808_2506085647_0" function="internal">\n    <lane id=":cluster_13424220808_2506085647_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.34" length="9.04" shape="659.46,890.95 658.55,893.28 657.12,894.70 655.17,895.20 652.70,894.79" />\n  </edge>\n  <edge id=":cluster_13424220808_2506085647_1" function="internal">\n    <lane id=":cluster_13424220808_2506085647_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.65" length="19.41" shape="659.46,890.95 660.15,894.64 663.09,898.68 667.31,902.27 671.84,904.64" />\n  </edge>\n  <edge id=":cluster_13424220808_2506085647_2" function="internal">\n    <lane id=":cluster_13424220808_2506085647_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.69" length="15.58" shape="672.88,901.62 667.86,899.48 664.47,897.11 662.71,894.51 662.58,891.67" />\n  </edge>\n  <edge id=":cluster_13424220808_2506085647_3" function="internal">\n    <lane id=":cluster_13424220808_2506085647_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="21.30" shape="672.88,901.62 652.70,894.79" />\n  </edge>\n  <edge id=":cluster_13424220808_2506085647_4" function="internal">\n    <lane id=":cluster_13424220808_2506085647_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.39" length="9.04" shape="664.91,908.26 665.92,905.98 667.41,904.62 669.38,904.17 671.84,904.64" />\n  </edge>\n  <edge id=":cluster_13424220808_2506085647_5" function="internal">\n    <lane id=":cluster_13424220808_2506085647_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.89" shape="664.91,908.26 664.96,904.32 663.83,899.99 662.65,895.64 662.58,891.67" />\n  </edge>\n  <edge id=":cluster_13424220808_2506085647_6" function="internal">\n    <lane id=":cluster_13424220808_2506085647_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.34" length="19.22" shape="664.91,908.26 664.33,904.56 661.43,900.57 657.22,897.05 652.70,894.79" />\n  </edge>\n  <edge id=":cluster_13424220808_2506085647_7" function="internal">\n    <lane id=":cluster_13424220808_2506085647_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="21.23" shape="651.73,897.84 671.84,904.64" />\n  </edge>\n  <edge id=":cluster_13424220808_2506085647_8" function="internal">\n    <lane id=":cluster_13424220808_2506085647_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.86" length="4.02" shape="651.73,897.84 655.70,898.50" />\n  </edge>\n  <edge id=":cluster_13424220808_2506085647_9" function="internal">\n    <lane id=":cluster_13424220808_2506085647_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.86" length="10.48" shape="655.70,898.50 658.82,897.69 661.12,895.41 662.58,891.67" />\n  </edge>\n  <edge id=":cluster_13472440775_1666900747_0" function="internal">\n    <lane id=":cluster_13472440775_1666900747_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.41" length="9.04" shape="1331.91,1177.09 1331.47,1179.54 1330.36,1181.23 1328.55,1182.16 1326.07,1182.32" />\n  </edge>\n  <edge id=":cluster_13472440775_1666900747_1" function="internal">\n    <lane id=":cluster_13472440775_1666900747_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="21.59" shape="1331.91,1177.09 1332.36,1181.83 1333.50,1187.76 1334.62,1193.69 1335.01,1198.43" />\n  </edge>\n  <edge id=":cluster_13472440775_1666900747_2" function="internal">\n    <lane id=":cluster_13472440775_1666900747_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.72" length="20.47" shape="1331.91,1177.09 1332.55,1182.41 1334.85,1186.39 1338.81,1189.00 1344.43,1190.27" />\n  </edge>\n  <edge id=":cluster_13472440775_1666900747_3" function="internal">\n    <lane id=":cluster_13472440775_1666900747_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.91" length="13.22" shape="1345.01,1183.89 1340.61,1183.13 1337.49,1181.76 1335.66,1179.78 1335.11,1177.19" />\n  </edge>\n  <edge id=":cluster_13472440775_1666900747_4" function="internal">\n    <lane id=":cluster_13472440775_1666900747_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.99" shape="1345.01,1183.89 1326.07,1182.32" />\n    <lane id=":cluster_13472440775_1666900747_4_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.99" shape="1344.72,1187.08 1325.82,1185.51" />\n  </edge>\n  <edge id=":cluster_13472440775_1666900747_6" function="internal">\n    <lane id=":cluster_13472440775_1666900747_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.61" length="3.84" shape="1344.72,1187.08 1340.90,1187.46" />\n  </edge>\n  <edge id=":cluster_13472440775_1666900747_14" function="internal">\n    <lane id=":cluster_13472440775_1666900747_14_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.61" length="13.23" shape="1340.90,1187.46 1340.71,1187.48 1337.76,1189.51 1335.86,1193.16 1335.01,1198.43" />\n  </edge>\n  <edge id=":cluster_13472440775_1666900747_7" function="internal">\n    <lane id=":cluster_13472440775_1666900747_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.42" length="9.03" shape="1338.21,1198.60 1338.69,1196.16 1339.84,1194.48 1341.66,1193.58 1344.14,1193.45" />\n  </edge>\n  <edge id=":cluster_13472440775_1666900747_8" function="internal">\n    <lane id=":cluster_13472440775_1666900747_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="21.66" shape="1338.21,1198.60 1337.81,1193.85 1336.70,1187.90 1335.56,1181.94 1335.11,1177.19" />\n  </edge>\n  <edge id=":cluster_13472440775_1666900747_9" function="internal">\n    <lane id=":cluster_13472440775_1666900747_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.59" length="20.43" shape="1338.21,1198.60 1337.67,1193.25 1335.43,1189.29 1331.48,1186.71 1325.82,1185.51" />\n  </edge>\n  <edge id=":cluster_13472440775_1666900747_10" function="internal">\n    <lane id=":cluster_13472440775_1666900747_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.76" length="13.00" shape="1325.32,1191.89 1329.67,1192.59 1332.74,1193.92 1334.52,1195.86 1335.01,1198.43" />\n  </edge>\n  <edge id=":cluster_13472440775_1666900747_11" function="internal">\n    <lane id=":cluster_13472440775_1666900747_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.91" shape="1325.32,1191.89 1344.14,1193.45" />\n    <lane id=":cluster_13472440775_1666900747_11_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.91" shape="1325.57,1188.70 1344.43,1190.27" />\n  </edge>\n  <edge id=":cluster_13472440775_1666900747_13" function="internal">\n    <lane id=":cluster_13472440775_1666900747_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.59" length="3.71" shape="1325.57,1188.70 1329.26,1188.29" />\n  </edge>\n  <edge id=":cluster_13472440775_1666900747_15" function="internal">\n    <lane id=":cluster_13472440775_1666900747_15_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.59" length="13.40" shape="1329.26,1188.29 1329.59,1188.25 1332.53,1186.18 1334.36,1182.49 1335.11,1177.19" />\n  </edge>\n  <edge id=":cluster_14188374630_2506088509_0" function="internal">\n    <lane id=":cluster_14188374630_2506088509_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.01" length="9.72" shape="1196.68,1499.58 1196.25,1502.34 1195.18,1504.37 1193.45,1505.69 1191.07,1506.27" />\n  </edge>\n  <edge id=":cluster_14188374630_2506088509_1" function="internal">\n    <lane id=":cluster_14188374630_2506088509_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="20.02" shape="1196.68,1499.58 1198.73,1503.16 1203.41,1506.56 1208.17,1509.95 1210.46,1513.54" />\n  </edge>\n  <edge id=":cluster_14188374630_2506088509_2" function="internal">\n    <lane id=":cluster_14188374630_2506088509_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.85" length="24.41" shape="1196.68,1499.58 1198.99,1502.81 1204.80,1505.22 1212.09,1506.58 1218.82,1506.66" />\n  </edge>\n  <edge id=":cluster_14188374630_2506088509_3" function="internal">\n    <lane id=":cluster_14188374630_2506088509_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.06" length="20.49" shape="1218.47,1503.48 1210.27,1504.01 1204.43,1503.55 1200.97,1502.11 1199.88,1499.68" />\n  </edge>\n  <edge id=":cluster_14188374630_2506088509_4" function="internal">\n    <lane id=":cluster_14188374630_2506088509_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="27.54" shape="1218.47,1503.48 1191.07,1506.27" />\n  </edge>\n  <edge id=":cluster_14188374630_2506088509_5" function="internal">\n    <lane id=":cluster_14188374630_2506088509_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.24" length="3.81" shape="1218.47,1503.48 1214.79,1504.46" />\n  </edge>\n  <edge id=":cluster_14188374630_2506088509_12" function="internal">\n    <lane id=":cluster_14188374630_2506088509_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.24" length="10.67" shape="1214.79,1504.46 1212.22,1506.46 1210.78,1509.48 1210.46,1513.54" />\n  </edge>\n  <edge id=":cluster_14188374630_2506088509_6" function="internal">\n    <lane id=":cluster_14188374630_2506088509_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.80" length="9.53" shape="1213.66,1513.37 1213.86,1510.66 1214.79,1508.64 1216.44,1507.31 1218.82,1506.66" />\n  </edge>\n  <edge id=":cluster_14188374630_2506088509_7" function="internal">\n    <lane id=":cluster_14188374630_2506088509_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.83" shape="1213.66,1513.37 1211.37,1509.83 1206.61,1506.53 1201.93,1503.23 1199.88,1499.68" />\n  </edge>\n  <edge id=":cluster_14188374630_2506088509_8" function="internal">\n    <lane id=":cluster_14188374630_2506088509_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.28" length="24.65" shape="1213.66,1513.37 1211.13,1510.28 1205.22,1507.90 1197.88,1506.48 1191.07,1506.27" />\n  </edge>\n  <edge id=":cluster_14188374630_2506088509_9" function="internal">\n    <lane id=":cluster_14188374630_2506088509_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.48" length="20.84" shape="1191.34,1509.46 1199.59,1509.13 1205.53,1509.70 1209.15,1511.17 1210.46,1513.54" />\n  </edge>\n  <edge id=":cluster_14188374630_2506088509_10" function="internal">\n    <lane id=":cluster_14188374630_2506088509_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="27.62" shape="1191.34,1509.46 1218.82,1506.66" />\n  </edge>\n  <edge id=":cluster_14188374630_2506088509_11" function="internal">\n    <lane id=":cluster_14188374630_2506088509_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.43" length="4.01" shape="1191.34,1509.46 1194.96,1508.59 1195.19,1508.43" />\n  </edge>\n  <edge id=":cluster_14188374630_2506088509_13" function="internal">\n    <lane id=":cluster_14188374630_2506088509_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.43" length="10.44" shape="1195.19,1508.43 1197.60,1506.67 1199.23,1503.70 1199.88,1499.68" />\n  </edge>\n  <edge id=":cluster_1558314142_1558314144_0" function="internal">\n    <lane id=":cluster_1558314142_1558314144_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.81" length="9.02" shape="1365.75,696.97 1365.17,699.33 1363.99,701.04 1362.19,702.09 1359.79,702.48" />\n  </edge>\n  <edge id=":cluster_1558314142_1558314144_1" function="internal">\n    <lane id=":cluster_1558314142_1558314144_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="27.77" shape="1365.75,696.97 1368.52,701.49 1375.08,706.96 1381.67,712.43 1384.54,716.96" />\n  </edge>\n  <edge id=":cluster_1558314142_1558314144_2" function="internal">\n    <lane id=":cluster_1558314142_1558314144_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.60" length="5.46" shape="1365.75,696.97 1368.53,701.66" />\n  </edge>\n  <edge id=":cluster_1558314142_1558314144_12" function="internal">\n    <lane id=":cluster_1558314142_1558314144_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.60" length="27.39" shape="1368.53,701.66 1368.60,701.78 1376.13,706.54 1385.43,710.22 1393.62,711.78" />\n  </edge>\n  <edge id=":cluster_1558314142_1558314144_3" function="internal">\n    <lane id=":cluster_1558314142_1558314144_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.10" length="29.99" shape="1393.70,708.58 1382.45,707.66 1374.58,705.47 1370.07,702.01 1368.93,697.29" />\n  </edge>\n  <edge id=":cluster_1558314142_1558314144_4" function="internal">\n    <lane id=":cluster_1558314142_1558314144_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.46" shape="1393.70,708.58 1386.99,707.56 1376.74,705.44 1366.49,703.37 1359.79,702.48" />\n  </edge>\n  <edge id=":cluster_1558314142_1558314144_5" function="internal">\n    <lane id=":cluster_1558314142_1558314144_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.09" length="14.03" shape="1393.70,708.58 1389.90,709.03 1387.11,710.58 1385.32,713.22 1384.54,716.96" />\n  </edge>\n  <edge id=":cluster_1558314142_1558314144_6" function="internal">\n    <lane id=":cluster_1558314142_1558314144_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.61" length="9.03" shape="1387.73,717.17 1388.23,714.77 1389.39,713.06 1391.18,712.07 1393.62,711.78" />\n  </edge>\n  <edge id=":cluster_1558314142_1558314144_7" function="internal">\n    <lane id=":cluster_1558314142_1558314144_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="27.70" shape="1387.73,717.17 1384.86,712.66 1378.26,707.23 1371.70,701.80 1368.93,697.29" />\n  </edge>\n  <edge id=":cluster_1558314142_1558314144_8" function="internal">\n    <lane id=":cluster_1558314142_1558314144_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.58" length="5.57" shape="1387.73,717.17 1384.94,712.35" />\n  </edge>\n  <edge id=":cluster_1558314142_1558314144_13" function="internal">\n    <lane id=":cluster_1558314142_1558314144_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.58" length="27.44" shape="1384.94,712.35 1384.79,712.11 1377.25,707.31 1367.97,703.78 1359.79,702.48" />\n  </edge>\n  <edge id=":cluster_1558314142_1558314144_9" function="internal">\n    <lane id=":cluster_1558314142_1558314144_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.11" length="30.26" shape="1359.87,705.68 1370.96,706.16 1378.76,708.20 1383.29,711.80 1384.54,716.96" />\n  </edge>\n  <edge id=":cluster_1558314142_1558314144_10" function="internal">\n    <lane id=":cluster_1558314142_1558314144_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.32" shape="1359.87,705.68 1366.55,706.57 1376.75,708.64 1386.94,710.76 1393.62,711.78" />\n  </edge>\n  <edge id=":cluster_1558314142_1558314144_11" function="internal">\n    <lane id=":cluster_1558314142_1558314144_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.26" length="13.73" shape="1359.87,705.68 1363.52,705.08 1366.25,703.49 1368.05,700.89 1368.93,697.29" />\n  </edge>\n  <edge id=":cluster_1559016910_4900465210_0" function="internal">\n    <lane id=":cluster_1559016910_4900465210_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.13" length="28.19" shape="871.49,402.63 869.83,414.10 867.85,422.22 865.56,426.97 862.94,428.36" />\n  </edge>\n  <edge id=":cluster_1559016910_4900465210_1" function="internal">\n    <lane id=":cluster_1559016910_4900465210_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="35.28" shape="871.49,402.63 867.95,437.73" />\n  </edge>\n  <edge id=":cluster_1559016910_4900465210_2" function="internal">\n    <lane id=":cluster_1559016910_4900465210_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="4.16" shape="871.49,402.63 871.61,406.55 871.71,406.76" />\n  </edge>\n  <edge id=":cluster_1559016910_4900465210_12" function="internal">\n    <lane id=":cluster_1559016910_4900465210_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="10.20" shape="871.71,406.76 872.98,409.47 875.59,411.39 879.45,412.32" />\n  </edge>\n  <edge id=":cluster_1559016910_4900465210_3" function="internal">\n    <lane id=":cluster_1559016910_4900465210_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.50" length="9.13" shape="879.76,409.14 877.29,408.55 875.62,407.33 874.75,405.47 874.67,402.99" />\n  </edge>\n  <edge id=":cluster_1559016910_4900465210_4" function="internal">\n    <lane id=":cluster_1559016910_4900465210_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="25.98" shape="879.76,409.14 875.73,412.01 871.35,418.76 866.97,425.51 862.94,428.36" />\n  </edge>\n  <edge id=":cluster_1559016910_4900465210_5" function="internal">\n    <lane id=":cluster_1559016910_4900465210_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.74" length="31.53" shape="879.76,409.14 876.46,412.21 872.79,419.98 869.65,429.48 867.95,437.73" />\n  </edge>\n  <edge id=":cluster_1559016910_4900465210_6" function="internal">\n    <lane id=":cluster_1559016910_4900465210_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.11" length="28.13" shape="871.13,438.06 872.68,426.60 874.59,418.49 876.85,413.73 879.45,412.32" />\n  </edge>\n  <edge id=":cluster_1559016910_4900465210_7" function="internal">\n    <lane id=":cluster_1559016910_4900465210_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="35.25" shape="871.13,438.06 874.67,402.99" />\n  </edge>\n  <edge id=":cluster_1559016910_4900465210_8" function="internal">\n    <lane id=":cluster_1559016910_4900465210_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="4.05" shape="871.13,438.06 870.96,434.17 870.89,434.03" />\n  </edge>\n  <edge id=":cluster_1559016910_4900465210_13" function="internal">\n    <lane id=":cluster_1559016910_4900465210_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="10.43" shape="870.89,434.03 869.54,431.26 866.87,429.32 862.94,428.36" />\n  </edge>\n  <edge id=":cluster_1559016910_4900465210_9" function="internal">\n    <lane id=":cluster_1559016910_4900465210_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.59" length="9.31" shape="862.61,431.55 865.16,432.16 866.90,433.40 867.83,435.25 867.95,437.73" />\n  </edge>\n  <edge id=":cluster_1559016910_4900465210_10" function="internal">\n    <lane id=":cluster_1559016910_4900465210_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="25.99" shape="862.61,431.55 866.64,428.69 871.03,421.95 875.42,415.20 879.45,412.32" />\n  </edge>\n  <edge id=":cluster_1559016910_4900465210_11" function="internal">\n    <lane id=":cluster_1559016910_4900465210_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.76" length="31.58" shape="862.61,431.55 865.93,428.49 869.67,420.73 872.89,411.23 874.67,402.99" />\n  </edge>\n  <edge id=":cluster_1559016919_1680666950_0" function="internal">\n    <lane id=":cluster_1559016919_1680666950_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.51" length="9.03" shape="915.98,475.18 915.35,477.57 914.10,479.19 912.25,480.03 909.77,480.10" />\n  </edge>\n  <edge id=":cluster_1559016919_1680666950_1" function="internal">\n    <lane id=":cluster_1559016919_1680666950_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="35.52" shape="915.98,475.18 912.53,510.53" />\n  </edge>\n  <edge id=":cluster_1559016919_1680666950_2" function="internal">\n    <lane id=":cluster_1559016919_1680666950_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.64" length="12.57" shape="915.98,475.18 915.90,483.63 916.36,487.73" />\n  </edge>\n  <edge id=":cluster_1559016919_1680666950_12" function="internal">\n    <lane id=":cluster_1559016919_1680666950_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.64" length="19.22" shape="916.36,487.73 917.02,493.65 919.08,502.08 921.81,505.76" />\n  </edge>\n  <edge id=":cluster_1559016919_1680666950_3" function="internal">\n    <lane id=":cluster_1559016919_1680666950_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.02" length="28.43" shape="922.10,502.58 919.66,500.69 918.36,495.55 918.19,487.17 919.15,475.54" />\n  </edge>\n  <edge id=":cluster_1559016919_1680666950_4" function="internal">\n    <lane id=":cluster_1559016919_1680666950_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="26.07" shape="922.10,502.58 918.77,498.96 915.93,491.39 913.09,483.79 909.77,480.10" />\n  </edge>\n  <edge id=":cluster_1559016919_1680666950_5" function="internal">\n    <lane id=":cluster_1559016919_1680666950_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.01" length="14.17" shape="922.10,502.58 918.22,502.78 915.33,504.17 913.44,506.76 912.53,510.53" />\n  </edge>\n  <edge id=":cluster_1559016919_1680666950_6" function="internal">\n    <lane id=":cluster_1559016919_1680666950_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.52" length="9.03" shape="915.71,510.83 916.29,508.43 917.50,506.78 919.34,505.89 921.81,505.76" />\n  </edge>\n  <edge id=":cluster_1559016919_1680666950_7" function="internal">\n    <lane id=":cluster_1559016919_1680666950_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="35.45" shape="915.71,510.83 919.15,475.54" />\n  </edge>\n  <edge id=":cluster_1559016919_1680666950_8" function="internal">\n    <lane id=":cluster_1559016919_1680666950_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.84" length="11.87" shape="915.71,510.83 915.66,502.36 915.26,498.98" />\n  </edge>\n  <edge id=":cluster_1559016919_1680666950_13" function="internal">\n    <lane id=":cluster_1559016919_1680666950_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.84" length="20.00" shape="915.26,498.98 914.46,492.31 912.40,483.84 909.77,480.10" />\n  </edge>\n  <edge id=":cluster_1559016919_1680666950_9" function="internal">\n    <lane id=":cluster_1559016919_1680666950_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.22" length="28.51" shape="909.41,483.28 911.72,485.23 913.01,490.42 913.28,498.85 912.53,510.53" />\n  </edge>\n  <edge id=":cluster_1559016919_1680666950_10" function="internal">\n    <lane id=":cluster_1559016919_1680666950_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="26.11" shape="909.41,483.28 912.74,486.97 915.61,494.57 918.47,502.14 921.81,505.76" />\n  </edge>\n  <edge id=":cluster_1559016919_1680666950_11" function="internal">\n    <lane id=":cluster_1559016919_1680666950_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.00" length="14.19" shape="909.41,483.28 913.29,483.17 916.21,481.85 918.17,479.30 919.15,475.54" />\n  </edge>\n  <edge id=":cluster_1680667233_2506085475_0" function="internal">\n    <lane id=":cluster_1680667233_2506085475_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.58" length="9.03" shape="1218.63,554.92 1217.96,557.29 1216.70,558.90 1214.84,559.77 1212.38,559.87" />\n  </edge>\n  <edge id=":cluster_1680667233_2506085475_1" function="internal">\n    <lane id=":cluster_1680667233_2506085475_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="21.06" shape="1218.63,554.92 1220.55,558.80 1225.34,562.87 1230.07,566.93 1231.79,570.78" />\n  </edge>\n  <edge id=":cluster_1680667233_2506085475_2" function="internal">\n    <lane id=":cluster_1680667233_2506085475_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.54" length="26.39" shape="1218.63,554.92 1220.77,558.42 1226.71,561.81 1234.34,564.49 1241.56,565.86" />\n  </edge>\n  <edge id=":cluster_1680667233_2506085475_3" function="internal">\n    <lane id=":cluster_1680667233_2506085475_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.79" length="22.61" shape="1241.83,562.67 1232.78,561.58 1226.43,559.99 1222.77,557.91 1221.80,555.34" />\n  </edge>\n  <edge id=":cluster_1680667233_2506085475_4" function="internal">\n    <lane id=":cluster_1680667233_2506085475_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.58" shape="1241.83,562.67 1212.38,559.87" />\n  </edge>\n  <edge id=":cluster_1680667233_2506085475_5" function="internal">\n    <lane id=":cluster_1680667233_2506085475_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.42" length="3.72" shape="1241.83,562.67 1238.12,562.93" />\n  </edge>\n  <edge id=":cluster_1680667233_2506085475_12" function="internal">\n    <lane id=":cluster_1680667233_2506085475_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.42" length="10.63" shape="1238.12,562.93 1235.21,564.36 1233.10,566.98 1231.79,570.78" />\n  </edge>\n  <edge id=":cluster_1680667233_2506085475_6" function="internal">\n    <lane id=":cluster_1680667233_2506085475_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.01" length="9.63" shape="1234.92,571.43 1235.80,568.83 1237.21,567.03 1239.13,566.04 1241.56,565.86" />\n  </edge>\n  <edge id=":cluster_1680667233_2506085475_7" function="internal">\n    <lane id=":cluster_1680667233_2506085475_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="21.21" shape="1234.92,571.43 1233.21,567.55 1228.49,563.41 1223.72,559.26 1221.80,555.34" />\n  </edge>\n  <edge id=":cluster_1680667233_2506085475_8" function="internal">\n    <lane id=":cluster_1680667233_2506085475_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.34" length="26.39" shape="1234.92,571.43 1232.98,567.81 1227.16,564.25 1219.58,561.38 1212.38,559.87" />\n  </edge>\n  <edge id=":cluster_1680667233_2506085475_9" function="internal">\n    <lane id=":cluster_1680667233_2506085475_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.60" length="22.57" shape="1212.06,563.05 1221.13,564.31 1227.44,566.02 1230.99,568.17 1231.79,570.78" />\n  </edge>\n  <edge id=":cluster_1680667233_2506085475_10" function="internal">\n    <lane id=":cluster_1680667233_2506085475_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.63" shape="1212.06,563.05 1241.56,565.86" />\n  </edge>\n  <edge id=":cluster_1680667233_2506085475_11" function="internal">\n    <lane id=":cluster_1680667233_2506085475_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="4.07" shape="1212.06,563.05 1215.90,562.89 1216.11,562.79" />\n  </edge>\n  <edge id=":cluster_1680667233_2506085475_13" function="internal">\n    <lane id=":cluster_1680667233_2506085475_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.07" length="9.99" shape="1216.11,562.79 1218.80,561.55 1220.77,559.03 1221.80,555.34" />\n  </edge>\n  <edge id=":cluster_1966399241_8523333972_0" function="internal">\n    <lane id=":cluster_1966399241_8523333972_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.85" length="16.57" shape="731.75,1125.65 730.49,1128.16 727.66,1129.79 723.24,1130.55 717.24,1130.44" />\n  </edge>\n  <edge id=":cluster_1966399241_8523333972_1" function="internal">\n    <lane id=":cluster_1966399241_8523333972_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.17" shape="731.75,1125.65 729.97,1129.24 726.81,1132.72 723.81,1136.21 722.52,1139.84" />\n  </edge>\n  <edge id=":cluster_1966399241_8523333972_2" function="internal">\n    <lane id=":cluster_1966399241_8523333972_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.87" length="14.49" shape="731.75,1125.65 731.69,1129.66 732.96,1132.63 735.55,1134.57 739.46,1135.46" />\n  </edge>\n  <edge id=":cluster_1966399241_8523333972_3" function="internal">\n    <lane id=":cluster_1966399241_8523333972_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.35" length="9.04" shape="739.72,1132.27 737.28,1131.71 735.66,1130.51 734.88,1128.65 734.91,1126.15" />\n  </edge>\n  <edge id=":cluster_1966399241_8523333972_4" function="internal">\n    <lane id=":cluster_1966399241_8523333972_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="22.56" shape="739.72,1132.27 717.24,1130.44" />\n  </edge>\n  <edge id=":cluster_1966399241_8523333972_5" function="internal">\n    <lane id=":cluster_1966399241_8523333972_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.07" length="7.74" shape="739.72,1132.27 734.39,1132.61 732.08,1133.25" />\n  </edge>\n  <edge id=":cluster_1966399241_8523333972_9" function="internal">\n    <lane id=":cluster_1966399241_8523333972_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.07" length="12.18" shape="732.08,1133.25 728.75,1134.18 724.29,1136.69 722.52,1139.84" />\n  </edge>\n  <edge id=":cluster_1966399241_8523333972_6" function="internal">\n    <lane id=":cluster_1966399241_8523333972_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.83" length="9.29" shape="716.98,1133.63 719.36,1134.18 721.08,1135.40 722.13,1137.29 722.52,1139.84" />\n  </edge>\n  <edge id=":cluster_1966399241_8523333972_7" function="internal">\n    <lane id=":cluster_1966399241_8523333972_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="22.56" shape="716.98,1133.63 739.46,1135.46" />\n  </edge>\n  <edge id=":cluster_1966399241_8523333972_8" function="internal">\n    <lane id=":cluster_1966399241_8523333972_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.77" length="7.72" shape="716.98,1133.63 722.41,1133.30 724.61,1132.70" />\n  </edge>\n  <edge id=":cluster_1966399241_8523333972_10" function="internal">\n    <lane id=":cluster_1966399241_8523333972_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.77" length="12.66" shape="724.61,1132.70 728.10,1131.74 732.72,1129.25 734.91,1126.15" />\n  </edge>\n  <edge id=":cluster_2052547073_2506085499_0" function="internal">\n    <lane id=":cluster_2052547073_2506085499_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.90" length="9.02" shape="713.07,680.85 714.00,678.61 715.43,677.09 717.35,676.28 719.77,676.18" />\n  </edge>\n  <edge id=":cluster_2052547073_2506085499_1" function="internal">\n    <lane id=":cluster_2052547073_2506085499_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.72" length="24.38" shape="688.82,671.92 698.70,673.09 705.52,674.83 709.28,677.15 709.98,680.04" />\n  </edge>\n  <edge id=":cluster_2052547073_2506085499_2" function="internal">\n    <lane id=":cluster_2052547073_2506085499_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.27" shape="688.82,671.92 719.77,676.18" />\n    <lane id=":cluster_2052547073_2506085499_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.27" shape="689.07,668.73 720.08,673.00" />\n  </edge>\n  <edge id=":cluster_2052547073_2506085499_4" function="internal">\n    <lane id=":cluster_2052547073_2506085499_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.47" length="9.03" shape="689.07,668.73 691.55,668.57 693.36,667.65 694.52,665.97 695.01,663.54" />\n  </edge>\n  <edge id=":cluster_2505176169_856214432_0" function="internal">\n    <lane id=":cluster_2505176169_856214432_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.66" shape="1524.30,998.04 1521.93,983.55" />\n    <lane id=":cluster_2505176169_856214432_0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.66" shape="1521.13,998.52 1518.77,984.06" />\n  </edge>\n  <edge id=":cluster_2505176169_856214432_2" function="internal">\n    <lane id=":cluster_2505176169_856214432_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.14" length="11.10" shape="1521.13,998.52 1519.65,995.59 1516.27,993.42 1512.60,992.44" />\n  </edge>\n  <edge id=":cluster_2505176169_856214432_7" function="internal">\n    <lane id=":cluster_2505176169_856214432_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.14" length="8.84" shape="1512.60,992.44 1511.00,992.01 1503.84,991.36" />\n  </edge>\n  <edge id=":cluster_2505176169_856214432_3" function="internal">\n    <lane id=":cluster_2505176169_856214432_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.33" length="10.08" shape="1503.74,994.56 1506.66,994.98 1508.86,996.04 1510.34,997.73 1511.10,1000.05" />\n  </edge>\n  <edge id=":cluster_2505176169_856214432_4" function="internal">\n    <lane id=":cluster_2505176169_856214432_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.13" length="21.60" shape="1503.74,994.56 1510.99,994.12 1515.92,992.22 1518.51,988.87 1518.77,984.06" />\n  </edge>\n  <edge id=":cluster_2505176169_856214432_5" function="internal">\n    <lane id=":cluster_2505176169_856214432_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.13" length="9.06" shape="1508.83,985.69 1508.88,988.25 1508.06,990.04 1506.38,991.08 1503.84,991.36" />\n  </edge>\n  <edge id=":cluster_2505176169_856214432_6" function="internal">\n    <lane id=":cluster_2505176169_856214432_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.54" shape="1508.83,985.69 1511.10,1000.05" />\n  </edge>\n  <edge id=":cluster_2505176422_2505176434_0" function="internal">\n    <lane id=":cluster_2505176422_2505176434_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.71" length="9.30" shape="1599.74,1423.54 1597.14,1423.48 1595.16,1422.66 1593.79,1421.08 1593.03,1418.73" />\n  </edge>\n  <edge id=":cluster_2505176422_2505176434_1" function="internal">\n    <lane id=":cluster_2505176422_2505176434_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.99" length="23.60" shape="1599.74,1423.54 1591.40,1425.06 1585.67,1427.50 1582.53,1430.86 1582.00,1435.13" />\n  </edge>\n  <edge id=":cluster_2505176422_2505176434_2" function="internal">\n    <lane id=":cluster_2505176422_2505176434_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.40" length="9.04" shape="1595.36,1432.95 1595.31,1430.46 1596.08,1428.59 1597.68,1427.34 1600.09,1426.72" />\n  </edge>\n  <edge id=":cluster_2505176422_2505176434_3" function="internal">\n    <lane id=":cluster_2505176422_2505176434_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.41" shape="1595.36,1432.95 1593.03,1418.73" />\n    <lane id=":cluster_2505176422_2505176434_3_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.41" shape="1592.20,1433.47 1589.88,1419.25" />\n  </edge>\n  <edge id=":cluster_2505176422_2505176434_5" function="internal">\n    <lane id=":cluster_2505176422_2505176434_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.40" shape="1579.69,1420.92 1582.00,1435.13" />\n  </edge>\n  <edge id=":cluster_2505176422_2505176434_6" function="internal">\n    <lane id=":cluster_2505176422_2505176434_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.24" length="11.22" shape="1579.69,1420.92 1581.45,1424.25 1585.43,1426.33 1588.36,1426.71" />\n  </edge>\n  <edge id=":cluster_2505176422_2505176434_7" function="internal">\n    <lane id=":cluster_2505176422_2505176434_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.24" length="11.77" shape="1588.36,1426.71 1591.65,1427.15 1600.09,1426.72" />\n  </edge>\n  <edge id=":cluster_2506087462_8523307708_0" function="internal">\n    <lane id=":cluster_2506087462_8523307708_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.46" length="15.94" shape="318.52,1235.09 313.75,1233.72 310.55,1231.56 308.91,1228.61 308.83,1224.87" />\n  </edge>\n  <edge id=":cluster_2506087462_8523307708_1" function="internal">\n    <lane id=":cluster_2506087462_8523307708_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.64" length="12.85" shape="318.52,1235.09 314.28,1234.81 311.60,1235.74 310.47,1237.87 310.91,1241.20" />\n  </edge>\n  <edge id=":cluster_2506087462_8523307708_2" function="internal">\n    <lane id=":cluster_2506087462_8523307708_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.76" shape="313.97,1240.25 312.35,1236.96 310.33,1233.77 308.85,1229.97 308.83,1224.87" />\n    <lane id=":cluster_2506087462_8523307708_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.76" shape="313.97,1240.25 311.84,1236.99 308.70,1233.82 306.13,1229.88 305.68,1224.33" />\n    <lane id=":cluster_2506087462_8523307708_2_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.76" shape="313.97,1240.25 311.32,1237.06 307.06,1233.94 303.39,1229.86 302.52,1223.80" />\n  </edge>\n  <edge id=":cluster_2506087462_8523307708_5" function="internal">\n    <lane id=":cluster_2506087462_8523307708_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.41" length="14.37" shape="295.67,1235.11 300.71,1234.81 304.59,1233.00 307.30,1229.69 307.52,1228.97" />\n    <lane id=":cluster_2506087462_8523307708_5_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.17" length="13.83" shape="295.92,1231.92 299.66,1231.70 302.53,1230.36 304.54,1227.90 305.68,1224.33" />\n    <lane id=":cluster_2506087462_8523307708_5_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.70" length="8.99" shape="296.18,1228.73 298.61,1228.59 300.48,1227.71 301.78,1226.12 302.52,1223.80" />\n  </edge>\n  <edge id=":cluster_2506087462_8523307708_8" function="internal">\n    <lane id=":cluster_2506087462_8523307708_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.41" length="4.30" shape="307.52,1228.97 308.83,1224.87" />\n  </edge>\n  <edge id=":cluster_2506088039_8523406713_0" function="internal">\n    <lane id=":cluster_2506088039_8523406713_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.56" length="9.03" shape="635.08,1386.27 634.36,1388.63 633.07,1390.22 631.18,1391.03 628.72,1391.06" />\n  </edge>\n  <edge id=":cluster_2506088039_8523406713_1" function="internal">\n    <lane id=":cluster_2506088039_8523406713_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="22.98" shape="635.08,1386.27 637.05,1390.47 642.04,1395.26 646.98,1400.04 648.79,1404.21" />\n  </edge>\n  <edge id=":cluster_2506088039_8523406713_2" function="internal">\n    <lane id=":cluster_2506088039_8523406713_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.24" length="28.34" shape="635.08,1386.27 637.21,1390.22 643.26,1394.50 651.08,1398.21 658.54,1400.45" />\n  </edge>\n  <edge id=":cluster_2506088039_8523406713_3" function="internal">\n    <lane id=":cluster_2506088039_8523406713_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.04" shape="659.12,1397.30 628.72,1391.06" />\n  </edge>\n  <edge id=":cluster_2506088039_8523406713_4" function="internal">\n    <lane id=":cluster_2506088039_8523406713_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.05" length="3.85" shape="659.12,1397.30 655.27,1397.14" />\n  </edge>\n  <edge id=":cluster_2506088039_8523406713_9" function="internal">\n    <lane id=":cluster_2506088039_8523406713_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.05" length="10.24" shape="655.27,1397.14 652.27,1398.24 650.11,1400.60 648.79,1404.21" />\n  </edge>\n  <edge id=":cluster_2506088039_8523406713_5" function="internal">\n    <lane id=":cluster_2506088039_8523406713_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.57" length="9.03" shape="651.92,1404.87 652.76,1402.56 654.15,1401.05 656.07,1400.35 658.54,1400.45" />\n  </edge>\n  <edge id=":cluster_2506088039_8523406713_6" function="internal">\n    <lane id=":cluster_2506088039_8523406713_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.77" length="28.22" shape="651.92,1404.87 650.00,1400.68 644.04,1396.44 636.22,1392.96 628.72,1391.06" />\n  </edge>\n  <edge id=":cluster_2506088039_8523406713_7" function="internal">\n    <lane id=":cluster_2506088039_8523406713_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.08" length="24.59" shape="628.31,1394.23 637.84,1395.92 644.43,1398.14 648.08,1400.91 648.79,1404.21" />\n  </edge>\n  <edge id=":cluster_2506088039_8523406713_8" function="internal">\n    <lane id=":cluster_2506088039_8523406713_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.86" shape="628.31,1394.23 658.54,1400.45" />\n  </edge>\n  <edge id=":cluster_2507400612_2507400618_0" function="internal">\n    <lane id=":cluster_2507400612_2507400618_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.17" length="22.72" shape="1650.96,866.78 1649.03,875.62 1646.87,881.83 1644.48,885.41 1641.85,886.37" />\n  </edge>\n  <edge id=":cluster_2507400612_2507400618_1" function="internal">\n    <lane id=":cluster_2507400612_2507400618_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.35" shape="1650.96,866.78 1646.74,895.82" />\n  </edge>\n  <edge id=":cluster_2507400612_2507400618_2" function="internal">\n    <lane id=":cluster_2507400612_2507400618_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.34" length="4.20" shape="1650.96,866.78 1650.84,870.48 1651.01,870.95" />\n  </edge>\n  <edge id=":cluster_2507400612_2507400618_12" function="internal">\n    <lane id=":cluster_2507400612_2507400618_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.34" length="9.75" shape="1651.01,870.95 1651.90,873.48 1654.17,875.81 1657.62,877.44" />\n  </edge>\n  <edge id=":cluster_2507400612_2507400618_3" function="internal">\n    <lane id=":cluster_2507400612_2507400618_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.91" length="9.27" shape="1658.59,874.39 1656.28,873.30 1654.76,871.76 1654.04,869.78 1654.11,867.34" />\n  </edge>\n  <edge id=":cluster_2507400612_2507400618_4" function="internal">\n    <lane id=":cluster_2507400612_2507400618_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="21.02" shape="1658.59,874.39 1654.66,875.71 1650.30,880.05 1645.89,884.55 1641.85,886.37" />\n  </edge>\n  <edge id=":cluster_2507400612_2507400618_5" function="internal">\n    <lane id=":cluster_2507400612_2507400618_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.88" length="25.74" shape="1658.59,874.39 1654.85,875.96 1651.19,881.49 1648.26,888.83 1646.74,895.82" />\n  </edge>\n  <edge id=":cluster_2507400612_2507400618_6" function="internal">\n    <lane id=":cluster_2507400612_2507400618_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.13" length="21.83" shape="1649.92,896.13 1651.15,887.27 1652.85,881.21 1655.00,877.93 1657.62,877.44" />\n  </edge>\n  <edge id=":cluster_2507400612_2507400618_7" function="internal">\n    <lane id=":cluster_2507400612_2507400618_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.09" shape="1649.92,896.13 1654.11,867.34" />\n  </edge>\n  <edge id=":cluster_2507400612_2507400618_8" function="internal">\n    <lane id=":cluster_2507400612_2507400618_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.13" length="3.68" shape="1649.92,896.13 1649.74,892.46" />\n  </edge>\n  <edge id=":cluster_2507400612_2507400618_13" function="internal">\n    <lane id=":cluster_2507400612_2507400618_13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.13" length="10.67" shape="1649.74,892.46 1649.73,892.29 1648.33,889.39 1645.70,887.41 1641.85,886.37" />\n  </edge>\n  <edge id=":cluster_2507400612_2507400618_9" function="internal">\n    <lane id=":cluster_2507400612_2507400618_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.67" length="9.30" shape="1641.44,889.54 1643.96,890.22 1645.68,891.49 1646.61,893.36 1646.74,895.82" />\n  </edge>\n  <edge id=":cluster_2507400612_2507400618_10" function="internal">\n    <lane id=":cluster_2507400612_2507400618_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="20.67" shape="1641.44,889.54 1645.39,887.71 1649.60,883.16 1653.78,878.78 1657.62,877.44" />\n  </edge>\n  <edge id=":cluster_2507400612_2507400618_11" function="internal">\n    <lane id=":cluster_2507400612_2507400618_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.93" length="26.39" shape="1641.44,889.54 1645.01,887.49 1648.81,881.79 1652.10,874.43 1654.11,867.34" />\n  </edge>\n  <edge id=":cluster_620881097_8523329816_0" function="internal">\n    <lane id=":cluster_620881097_8523329816_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.60" length="9.05" shape="326.19,1158.11 323.88,1157.21 322.39,1155.79 321.71,1153.85 321.84,1151.39" />\n  </edge>\n  <edge id=":cluster_620881097_8523329816_1" function="internal">\n    <lane id=":cluster_620881097_8523329816_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.72" length="17.90" shape="317.24,1175.52 318.75,1168.77 320.62,1164.14 322.85,1161.62 325.45,1161.22" />\n  </edge>\n  <edge id=":cluster_620881097_8523329816_2" function="internal">\n    <lane id=":cluster_620881097_8523329816_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="24.64" shape="317.24,1175.52 321.84,1151.39" />\n    <lane id=":cluster_620881097_8523329816_2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="24.64" shape="314.08,1174.98 318.69,1150.78" />\n    <lane id=":cluster_620881097_8523329816_2_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="24.64" shape="310.93,1174.44 315.55,1150.17" />\n  </edge>\n  <edge id=":cluster_620881097_8523329816_5" function="internal">\n    <lane id=":cluster_620881097_8523329816_5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.41" length="9.04" shape="310.93,1174.44 310.99,1171.95 310.23,1170.07 308.65,1168.81 306.25,1168.16" />\n  </edge>\n  <edge id=":cluster_849403384_849403500_8976428987_8976428988_0" function="internal">\n    <lane id=":cluster_849403384_849403500_8976428987_8976428988_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.68" length="13.87" shape="1490.12,747.25 1487.01,746.46 1484.58,744.59 1482.82,741.64 1481.74,737.62" />\n  </edge>\n  <edge id=":cluster_849403384_849403500_8976428987_8976428988_1" function="internal">\n    <lane id=":cluster_849403384_849403500_8976428987_8976428988_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="26.48" shape="1490.12,747.25 1463.72,745.14" />\n  </edge>\n  <edge id=":cluster_849403384_849403500_8976428987_8976428988_2" function="internal">\n    <lane id=":cluster_849403384_849403500_8976428987_8976428988_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.36" length="23.35" shape="1490.12,747.25 1484.11,747.89 1477.92,750.28 1473.27,753.97 1471.85,758.49" />\n  </edge>\n  <edge id=":cluster_849403384_849403500_8976428987_8976428988_3" function="internal">\n    <lane id=":cluster_849403384_849403500_8976428987_8976428988_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.06" length="9.07" shape="1484.81,756.38 1484.76,753.80 1485.59,752.03 1487.28,751.05 1489.85,750.87" />\n  </edge>\n  <edge id=":cluster_849403384_849403500_8976428987_8976428988_4" function="internal">\n    <lane id=":cluster_849403384_849403500_8976428987_8976428988_4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.01" shape="1484.81,756.38 1481.74,737.62" />\n    <lane id=":cluster_849403384_849403500_8976428987_8976428988_4_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.01" shape="1481.65,756.89 1478.58,738.14" />\n  </edge>\n  <edge id=":cluster_849403384_849403500_8976428987_8976428988_6" function="internal">\n    <lane id=":cluster_849403384_849403500_8976428987_8976428988_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.03" length="22.35" shape="1481.65,756.89 1479.50,753.02 1475.04,749.38 1469.41,746.56 1463.72,745.14" />\n  </edge>\n  <edge id=":cluster_849403384_849403500_8976428987_8976428988_7" function="internal">\n    <lane id=":cluster_849403384_849403500_8976428987_8976428988_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.85" length="13.92" shape="1463.36,748.82 1466.53,749.72 1469.01,751.63 1470.78,754.55 1471.85,758.49" />\n  </edge>\n  <edge id=":cluster_849403384_849403500_8976428987_8976428988_8" function="internal">\n    <lane id=":cluster_849403384_849403500_8976428987_8976428988_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="26.57" shape="1463.36,748.82 1489.85,750.87" />\n  </edge>\n  <edge id=":cluster_849403384_849403500_8976428987_8976428988_9" function="internal">\n    <lane id=":cluster_849403384_849403500_8976428987_8976428988_9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.70" length="20.51" shape="1463.36,748.82 1468.48,748.30 1473.66,746.03 1477.49,742.48 1478.58,738.14" />\n  </edge>\n  <edge id=":cluster_849403384_849403500_8976428987_8976428988_10" function="internal">\n    <lane id=":cluster_849403384_849403500_8976428987_8976428988_10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="6.02" length="9.08" shape="1468.80,739.74 1468.84,742.33 1468.01,744.09 1466.31,745.03 1463.72,745.14" />\n  </edge>\n  <edge id=":cluster_849403384_849403500_8976428987_8976428988_11" function="internal">\n    <lane id=":cluster_849403384_849403500_8976428987_8976428988_11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.00" shape="1468.80,739.74 1471.85,758.49" />\n  </edge>\n  <edge id=":cluster_849403384_849403500_8976428987_8976428988_12" function="internal">\n    <lane id=":cluster_849403384_849403500_8976428987_8976428988_12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.43" length="24.69" shape="1468.80,739.74 1471.30,743.44 1476.65,746.91 1483.33,749.57 1489.85,750.87" />\n  </edge>\n  <edge id=":cluster_916327730_916327962_985092239_985092700_0" function="internal">\n    <lane id=":cluster_916327730_916327962_985092239_985092700_0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.68" length="9.84" shape="322.66,1338.41 319.88,1338.12 317.66,1337.22 316.00,1335.70 314.90,1333.56" />\n  </edge>\n  <edge id=":cluster_916327730_916327962_985092239_985092700_1" function="internal">\n    <lane id=":cluster_916327730_916327962_985092239_985092700_1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="11.53" length="37.97" shape="322.66,1338.41 315.94,1342.19 310.05,1351.09 306.35,1361.55 306.21,1370.04" />\n  </edge>\n  <edge id=":cluster_916327730_916327962_985092239_985092700_2" function="internal">\n    <lane id=":cluster_916327730_916327962_985092239_985092700_2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="9.30" length="24.30" shape="324.05,1364.11 321.12,1354.28 319.91,1347.26 320.43,1343.03 322.68,1341.61" />\n  </edge>\n  <edge id=":cluster_916327730_916327962_985092239_985092700_3" function="internal">\n    <lane id=":cluster_916327730_916327962_985092239_985092700_3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.88" shape="324.05,1364.11 314.90,1333.56" />\n    <lane id=":cluster_916327730_916327962_985092239_985092700_3_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.88" shape="321.01,1365.12 311.86,1334.58" />\n    <lane id=":cluster_916327730_916327962_985092239_985092700_3_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.88" shape="317.97,1366.13 308.83,1335.60" />\n  </edge>\n  <edge id=":cluster_916327730_916327962_985092239_985092700_6" function="internal">\n    <lane id=":cluster_916327730_916327962_985092239_985092700_6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="7.23" length="9.01" shape="299.36,1365.35 301.74,1365.55 303.67,1366.40 305.16,1367.90 306.21,1370.04" />\n  </edge>\n  <edge id=":cluster_916327730_916327962_985092239_985092700_7" function="internal">\n    <lane id=":cluster_916327730_916327962_985092239_985092700_7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="33.87" shape="299.36,1365.35 306.51,1361.44 312.89,1353.31 318.33,1345.27 322.68,1341.61" />\n  </edge>\n  <edge id=":cluster_916327730_916327962_985092239_985092700_8" function="internal">\n    <lane id=":cluster_916327730_916327962_985092239_985092700_8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="10.54" length="30.79" shape="299.20,1362.15 304.74,1359.01 308.31,1351.88 309.74,1343.25 308.83,1335.60" />\n  </edge>\n  <edge id="-100401320" from="1123399492" to="1110416469" priority="10" type="highway.tertiary">\n    <lane id="-100401320_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="85.60" shape="776.67,1752.60 755.08,1669.76">\n      <param key="origId" value="100401320" />\n    </lane>\n  </edge>\n  <edge id="-1204844051" from="1115585385" to="4900464584" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary">\n    <lane id="-1204844051_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.13" shape="1035.02,980.22 1046.08,981.51">\n      <param key="origId" value="1204844051" />\n    </lane>\n  </edge>\n  <edge id="-1204844052" from="4900464584" to="1123399600" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="1050.82,980.45 1055.67,981.13 1067.01,982.70 1084.18,985.09">\n    <lane id="-1204844052_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.30" shape="1057.43,982.99 1066.79,984.28 1073.57,985.23">\n      <param key="origId" value="1204844052 969757850 185951074" />\n    </lane>\n  </edge>\n  <edge id="-1218747872" from="2505176175" to="1639088658" priority="6" type="highway.secondary_link" shape="1533.23,1062.66 1535.13,1062.51 1545.68,1061.94 1548.26,1061.79">\n    <lane id="-1218747872_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="1.56" shape="1540.83,1063.80 1542.39,1063.72">\n      <param key="origId" value="1218747872 243314101 1218747873" />\n    </lane>\n  </edge>\n  <edge id="-1218747879" from="2505176265" to="2505176258" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="1570.92,1199.46 1566.49,1199.90 1561.19,1200.40 1555.83,1200.99">\n    <lane id="-1218747879_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="4.60" shape="1567.57,1198.19 1566.34,1198.31 1562.99,1198.62">\n      <param key="origId" value="1218747879 1218747878 66984610" />\n    </lane>\n  </edge>\n  <edge id="-122540020#0" from="14165547215" to="1369020950" name="ถนนเฉลียวอุทิศ" priority="3" type="highway.residential" shape="1385.27,1274.98 1383.56,1265.10 1385.88,1192.43">\n    <lane id="-122540020#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="63.99" shape="1385.51,1266.96 1385.16,1264.99 1387.14,1203.04">\n      <param key="origId" value="122540020" />\n    </lane>\n  </edge>\n  <edge id="-122540020#2" from="1369020943" to="14165547215" name="ถนนเฉลียวอุทิศ" priority="3" type="highway.residential" shape="1393.78,1346.12 1392.32,1315.41 1385.27,1274.98">\n    <lane id="-122540020#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="58.04" shape="1395.03,1338.76 1393.91,1315.23 1387.99,1281.26">\n      <param key="origId" value="122540020" />\n    </lane>\n  </edge>\n  <edge id="-122540020#3" from="2506088023" to="1369020943" name="ถนนเฉลียวอุทิศ" priority="3" type="highway.residential">\n    <lane id="-122540020#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="27.11" shape="1398.09,1380.20 1395.95,1353.18">\n      <param key="origId" value="122540020" />\n    </lane>\n  </edge>\n  <edge id="-122540020#4" from="2506088274" to="2506088023" name="ถนนเฉลียวอุทิศ" priority="3" type="highway.residential">\n    <lane id="-122540020#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="64.17" shape="1403.27,1458.55 1399.12,1394.52">\n      <param key="origId" value="122540020" />\n    </lane>\n  </edge>\n  <edge id="-122540020#5" from="2506088528" to="2506088274" name="ถนนเฉลียวอุทิศ" priority="3" type="highway.residential">\n    <lane id="-122540020#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.23" shape="1405.64,1512.09 1404.03,1472.90">\n      <param key="origId" value="122540020" />\n    </lane>\n  </edge>\n  <edge id="-122540020#6" from="1369020942" to="2506088528" name="ถนนเฉลียวอุทิศ" priority="3" type="highway.residential">\n    <lane id="-122540020#6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.02" shape="1408.49,1536.50 1407.03,1526.59">\n      <param key="origId" value="122540020" />\n    </lane>\n  </edge>\n  <edge id="-122540020#7" from="1369020939" to="1369020942" name="ถนนเฉลียวอุทิศ" priority="3" type="highway.residential">\n    <lane id="-122540020#7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.38" shape="1397.28,1578.90 1407.49,1549.23">\n      <param key="origId" value="122540020" />\n    </lane>\n  </edge>\n  <edge id="-1270957863" from="856214524" to="cluster_2505176169_856214432" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="1336.71,985.69 1480.29,992.16 1510.05,993.17">\n    <lane id="-1270957863_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="159.98" shape="1343.90,987.62 1480.23,993.76 1503.74,994.56">\n      <param key="origId" value="1270957863" />\n    </lane>\n    <param key="origTo" value="856214432" />\n  </edge>\n  <edge id="-129827552#0" from="1680666546" to="1431715395" priority="3" type="highway.residential" shape="562.22,386.77 546.67,381.91 514.90,371.98">\n    <lane id="-129827552#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="35.02" shape="555.75,383.07 547.15,380.38 522.33,372.63">\n      <param key="origId" value="129827552" />\n    </lane>\n  </edge>\n  <edge id="-129827552#1" from="1122072674" to="1680666546" priority="3" type="highway.residential">\n    <lane id="-129827552#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.33" shape="582.22,391.35 569.50,387.37">\n      <param key="origId" value="129827552" />\n    </lane>\n  </edge>\n  <edge id="-129827554#0" from="1680666468" to="1110416726" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary" shape="654.95,370.55 659.35,366.88 686.80,341.59 722.47,312.70 743.50,302.79 772.69,304.69 796.31,306.04">\n    <lane id="-129827554#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="148.32" shape="661.75,366.84 687.85,342.80 723.33,314.06 743.81,304.41 772.59,306.29 788.27,307.18">\n      <param key="origId" value="129827554" />\n    </lane>\n  </edge>\n  <edge id="-129827554#4" from="cluster_1107773211_1110416556" to="1680666468" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary" shape="755.95,515.14 746.90,503.59 735.81,491.86 708.30,469.47 688.34,452.63 637.72,421.66 626.12,414.63 622.96,408.83 622.57,400.82 627.89,393.90 636.28,386.15 654.95,370.55">\n    <lane id="-129827554#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="204.43" shape="750.67,505.81 748.11,502.54 736.90,490.68 709.32,468.24 689.28,451.33 638.55,420.29 627.32,413.49 624.54,408.39 624.20,401.33 629.08,394.98 637.34,387.35 650.72,376.17">\n      <param key="origId" value="129827554" />\n    </lane>\n    <param key="origFrom" value="1110416556" />\n  </edge>\n  <edge id="-129827558" from="1431715359" to="1122072673" name="ถนนเสน่หานุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="-129827558_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="78.65" shape="565.88,218.24 540.31,292.62">\n      <param key="origId" value="129827558" />\n    </lane>\n  </edge>\n  <edge id="-129827561#0" from="14188764645" to="1431715415" name="ถนนหอมนวล" priority="10" type="highway.tertiary">\n    <lane id="-129827561#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.22" shape="811.01,571.13 814.87,531.09">\n      <param key="origId" value="129827561" />\n    </lane>\n  </edge>\n  <edge id="-129827561#1" from="1431715406" to="14188764645" name="ถนนหอมนวล" priority="10" type="highway.tertiary">\n    <lane id="-129827561#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.64" shape="808.21,600.03 809.62,585.46">\n      <param key="origId" value="129827561" />\n    </lane>\n  </edge>\n  <edge id="-129827562" from="1431715421" to="1431715380" name="ถนนอำนวยวิทย์" priority="10" type="highway.tertiary" shape="501.61,292.37 507.54,245.23 511.42,218.11 530.77,114.96">\n    <lane id="-129827562_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="174.15" shape="503.90,287.00 509.13,245.44 513.00,218.37 532.34,115.25">\n      <param key="origId" value="129827562" />\n    </lane>\n  </edge>\n  <edge id="-129842388#0" from="1431867108" to="1431867160" name="ถนนราษฎร์ยินดี ซอย 9" priority="10" type="highway.tertiary" shape="1435.64,359.62 1430.93,360.13 1423.82,361.15 1418.75,361.96">\n    <lane id="-129842388#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="4.20" shape="1429.78,358.68 1425.62,359.27">\n      <param key="origId" value="129842388 932339003 932339004" />\n    </lane>\n  </edge>\n  <edge id="-129842388#1" from="1680666463" to="1431867108" name="ถนนราษฎร์ยินดี ซอย 9" priority="10" type="highway.tertiary">\n    <lane id="-129842388#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="46.41" shape="1488.03,370.70 1442.93,359.75">\n      <param key="origId" value="129842388" />\n    </lane>\n  </edge>\n  <edge id="-129842388#2" from="1431867166" to="1680666463" name="ถนนราษฎร์ยินดี ซอย 9" priority="10" type="highway.tertiary">\n    <lane id="-129842388#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="45.17" shape="1547.00,383.89 1502.90,374.14">\n      <param key="origId" value="129842388" />\n    </lane>\n  </edge>\n  <edge id="-129842391" from="1431867112" to="13362936997" name="ถนนพัฒโนอุทิศ" priority="3" type="highway.residential">\n    <lane id="-129842391_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="218.25" shape="1546.59,157.03 1437.24,345.92">\n      <param key="origId" value="129842391" />\n    </lane>\n  </edge>\n  <edge id="-129842397#0" from="1431867103" to="1431867141" name="Thanon Klongren 2 Soi 7" priority="3" type="highway.residential">\n    <lane id="-129842397#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="133.72" shape="1722.53,455.32 1841.83,515.74">\n      <param key="origId" value="129842397" />\n    </lane>\n  </edge>\n  <edge id="-129842397#1" from="8642733850" to="1431867103" name="Thanon Klongren 2 Soi 7" priority="3" type="highway.residential">\n    <lane id="-129842397#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="63.85" shape="1651.36,423.22 1709.66,449.23">\n      <param key="origId" value="129842397" />\n    </lane>\n  </edge>\n  <edge id="-129842397#3" from="1431867150" to="8642733850" name="Thanon Klongren 2 Soi 7" priority="3" type="highway.residential" shape="1668.15,304.99 1626.18,405.38 1626.23,406.69 1626.46,407.86 1626.80,408.70 1627.26,409.47 1627.85,410.40 1628.63,411.16 1629.62,411.86 1630.95,412.36 1645.32,418.77">\n    <lane id="-129842397#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="122.89" shape="1664.28,310.09 1624.57,405.09 1624.64,406.88 1624.92,408.32 1625.36,409.41 1625.90,410.31 1626.60,411.42 1627.60,412.39 1628.86,413.29 1630.34,413.84 1638.20,417.35">\n      <param key="origId" value="129842397" />\n    </lane>\n  </edge>\n  <edge id="-129842397#4" from="1431867100" to="1431867150" name="Thanon Klongren 2 Soi 7" priority="3" type="highway.residential">\n    <lane id="-129842397#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.84" shape="1632.24,285.06 1662.02,303.14">\n      <param key="origId" value="129842397" />\n    </lane>\n  </edge>\n  <edge id="-1316153439" from="6634655051" to="2506087563" priority="3" type="highway.residential">\n    <lane id="-1316153439_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="1.35" shape="429.32,1249.26 427.97,1249.12">\n      <param key="origId" value="1316153439" />\n    </lane>\n  </edge>\n  <edge id="-1334274432" from="12344163777" to="12344163778" priority="3" type="highway.residential">\n    <lane id="-1334274432_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="52.92" shape="584.29,1050.93 532.44,1040.34">\n      <param key="origId" value="1334274432" />\n    </lane>\n  </edge>\n  <edge id="-1365892748#1" from="2506087432" to="1110416444" name="Thanon Pracharak" priority="3" type="highway.residential" shape="641.81,1228.22 642.75,1182.15 644.04,1123.85">\n    <lane id="-1365892748#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="89.63" shape="643.56,1221.05 644.35,1182.18 645.47,1131.44">\n      <param key="origId" value="1365892748" />\n    </lane>\n  </edge>\n  <edge id="-1365935945#0" from="4900465166" to="856214538" priority="3" type="highway.residential">\n    <lane id="-1365935945#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="38.68" shape="897.77,925.86 895.56,964.48">\n      <param key="origId" value="1365935945" />\n    </lane>\n  </edge>\n  <edge id="-1365935945#1" from="12649445299" to="4900465166" priority="3" type="highway.residential">\n    <lane id="-1365935945#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="27.13" shape="900.48,884.36 898.66,911.44">\n      <param key="origId" value="1365935945" />\n    </lane>\n  </edge>\n  <edge id="-142395239#0" from="1558314095" to="1431867150" priority="3" type="highway.residential">\n    <lane id="-142395239#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.52" shape="1690.59,306.72 1674.25,304.28">\n      <param key="origId" value="142395239" />\n    </lane>\n  </edge>\n  <edge id="-142453602#0" from="1115585909" to="1115585385" priority="10" type="highway.tertiary">\n    <lane id="-142453602#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="44.97" shape="1008.56,928.28 1024.65,970.27">\n      <param key="origId" value="142453602" />\n    </lane>\n  </edge>\n  <edge id="-142453602#1" from="1115585463" to="1115585909" priority="10" type="highway.tertiary" shape="998.04,876.32 1007.38,899.61 1007.86,921.98">\n    <lane id="-142453602#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="33.83" shape="998.67,882.20 1005.79,899.94 1006.10,914.65">\n      <param key="origId" value="142453602" />\n    </lane>\n  </edge>\n  <edge id="-142453602#2" from="1115585416" to="1115585463" priority="10" type="highway.tertiary" shape="957.51,845.29 972.02,854.49 998.04,876.32">\n    <lane id="-142453602#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="37.19" shape="963.09,850.72 971.07,855.78 992.32,873.61">\n      <param key="origId" value="142453602" />\n    </lane>\n  </edge>\n  <edge id="-142455590" from="cluster_1559016910_4900465210" to="1559016909" name="ถนนฉัยยากุลอุทิศ ซอย 2" priority="3" type="highway.residential" shape="870.29,430.73 776.48,421.04">\n    <lane id="-142455590_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="79.52" shape="862.94,428.36 783.85,420.19">\n      <param key="origId" value="142455590" />\n    </lane>\n    <param key="origFrom" value="1559016910" />\n  </edge>\n  <edge id="-142455592#1" from="1559016911" to="1431867166" name="Rajyindee 9 Road" priority="3" type="highway.residential" shape="1532.91,436.17 1549.79,397.60 1554.36,387.16">\n    <lane id="-142455592#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="45.60" shape="1534.38,436.81 1551.26,398.24 1552.66,395.03">\n      <param key="origId" value="142455592" />\n    </lane>\n  </edge>\n  <edge id="-142455595" from="1559016905" to="1559016904" name="ถนนฉัยยากุลอุทิศ ซอย 3" priority="10" type="highway.tertiary">\n    <lane id="-142455595_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="79.37" shape="867.95,383.75 788.99,375.74">\n      <param key="origId" value="142455595" />\n    </lane>\n  </edge>\n  <edge id="-142455600#0" from="cluster_1559016910_4900465210" to="1559016918" priority="10" type="highway.tertiary" shape="870.29,430.73 865.46,476.66">\n    <lane id="-142455600#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.84" shape="867.95,437.73 864.62,469.39">\n      <param key="origId" value="142455600" />\n    </lane>\n    <param key="origFrom" value="1559016910" />\n  </edge>\n  <edge id="-142455600#2" from="1559016905" to="cluster_1559016910_4900465210" priority="10" type="highway.tertiary" shape="874.99,386.07 872.26,410.02">\n    <lane id="-142455600#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.61" shape="872.58,393.08 871.49,402.63">\n      <param key="origId" value="142455600" />\n    </lane>\n    <param key="origTo" value="4900465210" />\n  </edge>\n  <edge id="-142455600#3" from="1559016903" to="1559016905" priority="10" type="highway.tertiary">\n    <lane id="-142455600#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.50" shape="877.23,349.42 874.15,378.75">\n      <param key="origId" value="142455600" />\n    </lane>\n  </edge>\n  <edge id="-142455604#1" from="1110416726" to="1559016903" name="ถนนละม้ายสงเคราะห์" priority="10" type="highway.tertiary" shape="796.31,306.04 825.48,301.95 848.07,290.44 863.26,283.28 883.27,271.94 885.61,281.61 879.58,342.38">\n    <lane id="-142455604#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="143.44" shape="807.25,306.12 825.97,303.50 848.77,291.88 864.00,284.70 882.21,274.38 883.99,281.72 878.70,335.07">\n      <param key="origId" value="142455604 142455600" />\n    </lane>\n  </edge>\n  <edge id="-1447303746" from="13279030940" to="13279030938" priority="3" type="highway.residential" shape="1278.57,948.71 1293.26,948.79 1309.04,948.88 1307.02,986.05">\n    <lane id="-1447303746_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="56.92" shape="1278.56,950.31 1293.25,950.39 1307.35,950.47 1305.82,978.55">\n      <param key="origId" value="1447303746" />\n    </lane>\n  </edge>\n  <edge id="-1454593202#0" from="14188764647" to="13339819885" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary">\n    <lane id="-1454593202#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.43" shape="867.52,530.87 897.79,533.99">\n      <param key="origId" value="1454593202" />\n    </lane>\n  </edge>\n  <edge id="-1454593202#1" from="1431715415" to="14188764647" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary">\n    <lane id="-1454593202#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.38" shape="820.99,526.08 853.20,529.40">\n      <param key="origId" value="1454593202" />\n    </lane>\n  </edge>\n  <edge id="-1454593202#2" from="cluster_1107773211_1110416556" to="1431715415" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary" shape="769.62,517.28 813.97,523.75">\n    <lane id="-1454593202#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.81" shape="776.28,519.87 806.76,524.32">\n      <param key="origId" value="1454593202" />\n    </lane>\n    <param key="origFrom" value="1107773211" />\n  </edge>\n  <edge id="-1457057266#1" from="1431867100" to="1431867166" name="Rajyindee 9 Road" priority="3" type="highway.residential" shape="1630.10,281.89 1614.61,312.84 1595.26,337.69 1569.64,368.67 1554.36,387.16">\n    <lane id="-1457057266#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="120.50" shape="1627.12,284.28 1613.25,311.98 1594.01,336.69 1568.41,367.65 1556.92,381.55">\n      <param key="origId" value="1457057266" />\n    </lane>\n  </edge>\n  <edge id="-1476708031" from="13545019681" to="4900464587" priority="3" type="highway.residential" shape="1066.57,1031.65 1082.32,1051.80 1084.98,1054.78 1091.91,1065.39 1106.24,1078.99 1120.89,1086.83 1132.57,1091.43 1137.55,1092.40 1142.37,1093.01 1146.91,1093.09 1151.10,1092.82 1163.63,1091.56 1168.74,1090.52 1200.01,1086.64 1230.26,1080.85 1243.18,1079.53 1245.84,1079.75 1248.23,1080.23 1250.27,1081.43 1250.98,1083.23 1250.93,1086.01 1241.28,1134.45">\n    <lane id="-1476708031_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="247.14" shape="1070.89,1039.78 1081.09,1052.83 1083.71,1055.76 1090.67,1066.42 1105.29,1080.30 1120.22,1088.28 1132.12,1092.97 1137.30,1093.98 1142.26,1094.61 1146.95,1094.69 1151.23,1094.41 1163.87,1093.14 1169.00,1092.10 1200.26,1088.22 1230.49,1082.43 1243.20,1081.14 1245.62,1081.34 1247.65,1081.75 1248.98,1082.53 1249.37,1083.52 1249.33,1085.84 1241.06,1127.38">\n      <param key="origId" value="1476708031 530314558" />\n    </lane>\n  </edge>\n  <edge id="-148941242#0" from="1110416726" to="1110416683" name="ถนนฉัยยากุลอุทิศ" priority="10" type="highway.tertiary" shape="796.31,306.04 799.75,300.43 814.75,279.64">\n    <lane id="-148941242#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="21.29" shape="803.59,297.84 816.05,280.58">\n      <param key="origId" value="148941242 148941245" />\n    </lane>\n  </edge>\n  <edge id="-148941242#1" from="1431715372" to="1110416726" name="ถนนฉัยยากุลอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="-148941242#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.07" shape="792.24,326.74 795.73,314.14">\n      <param key="origId" value="148941242" />\n    </lane>\n  </edge>\n  <edge id="-148941242#2" from="1559016904" to="1431715372" name="ถนนฉัยยากุลอุทิศ" priority="10" type="highway.tertiary" shape="781.62,376.60 782.15,371.97 784.78,353.67 788.61,333.85">\n    <lane id="-148941242#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.07" shape="784.11,369.60 786.36,353.94 788.87,340.93">\n      <param key="origId" value="148941242" />\n    </lane>\n  </edge>\n  <edge id="-148941242#3" from="1559016909" to="1559016904" name="ถนนฉัยยากุลอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="-148941242#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.32" shape="778.90,414.01 782.39,383.89">\n      <param key="origId" value="148941242" />\n    </lane>\n  </edge>\n  <edge id="-148941242#4" from="1559016916" to="1559016909" name="ถนนฉัยยากุลอุทิศ" priority="10" type="highway.tertiary" shape="773.15,464.80 775.55,429.12 776.48,421.04">\n    <lane id="-148941242#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.25" shape="775.11,459.51 777.14,429.27 777.25,428.34">\n      <param key="origId" value="148941242" />\n    </lane>\n  </edge>\n  <edge id="-148941242#5" from="cluster_1107773211_1110416556" to="1559016916" name="ถนนฉัยยากุลอุทิศ" priority="10" type="highway.tertiary" shape="769.62,517.28 773.15,464.80">\n    <lane id="-148941242#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.84" shape="771.68,510.44 774.36,470.69">\n      <param key="origId" value="148941242" />\n    </lane>\n    <param key="origFrom" value="1107773211" />\n  </edge>\n  <edge id="-151072451#2" from="1639088636" to="1639088623" name="ถนนราษฎร์ยินดี ซอย 7" priority="10" type="highway.tertiary" shape="1964.80,1000.49 1913.65,1010.39 1873.92,1015.70 1795.52,1018.16 1766.55,1009.20 1752.45,1006.11 1742.34,1003.89 1735.79,1002.49 1692.60,993.21 1543.17,961.10 1531.15,963.11">\n    <lane id="-151072451#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="435.96" shape="1964.50,998.92 1913.39,1008.81 1873.79,1014.10 1795.74,1016.55 1766.96,1007.65 1752.79,1004.55 1742.68,1002.33 1736.13,1000.93 1692.94,991.65 1543.21,959.47 1536.44,960.60">\n      <param key="origId" value="151072451" />\n    </lane>\n  </edge>\n  <edge id="-154097280" from="1966404349" to="1115585403" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="1027.19,1067.73 1044.91,1069.56 1066.17,1070.17">\n    <lane id="-154097280_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.82" shape="1032.89,1069.93 1044.80,1071.16 1062.64,1071.67">\n      <param key="origId" value="154097280" />\n    </lane>\n  </edge>\n  <edge id="-1546894905" from="14069967501" to="14069947700" priority="3" type="highway.residential" shape="564.06,698.75 570.57,668.33 572.02,661.42">\n    <lane id="-1546894905_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.36" shape="565.62,699.08 571.98,669.40">\n      <param key="origId" value="1546894905" />\n    </lane>\n  </edge>\n  <edge id="-1546896992" from="14069981568" to="14069981565" priority="3" type="highway.residential" shape="514.36,707.90 510.01,735.67 503.03,740.89 500.47,740.77 473.16,739.51 459.32,738.87">\n    <lane id="-1546896992_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="74.69" shape="512.78,707.65 508.53,734.78 502.53,739.26 500.54,739.17 473.23,737.91 462.84,737.43">\n      <param key="origId" value="1546896992" />\n    </lane>\n  </edge>\n  <edge id="-1546907364" from="14070071630" to="14070071629" priority="3" type="highway.residential">\n    <lane id="-1546907364_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="63.63" shape="657.61,305.18 597.13,285.40">\n      <param key="origId" value="1546907364" />\n    </lane>\n  </edge>\n  <edge id="-1552748146" from="14127615694" to="2506088518" priority="3" type="highway.residential">\n    <lane id="-1552748146_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="5.47" shape="1143.07,1512.33 1148.54,1512.33">\n      <param key="origId" value="1552748146" />\n    </lane>\n  </edge>\n  <edge id="-1552748564#0" from="4900465706" to="14127602686" priority="3" type="highway.residential">\n    <lane id="-1552748564#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.29" shape="408.77,1496.11 439.05,1496.94">\n      <param key="origId" value="1552748564" />\n    </lane>\n  </edge>\n  <edge id="-1552748564#1" from="916327912" to="4900465706" priority="3" type="highway.residential">\n    <lane id="-1552748564#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.68" shape="374.67,1495.20 394.34,1495.72">\n      <param key="origId" value="1552748564" />\n    </lane>\n  </edge>\n  <edge id="-1556742612" from="14159239426" to="14159239425" priority="3" type="highway.residential">\n    <lane id="-1556742612_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="49.63" shape="1148.47,770.74 1137.58,722.31">\n      <param key="origId" value="1556742612" />\n    </lane>\n  </edge>\n  <edge id="-1556742613" from="14159239429" to="14159239427" priority="3" type="highway.residential" shape="1096.62,765.79 1091.68,759.42 1077.61,728.60">\n    <lane id="-1556742613_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="33.60" shape="1097.88,764.81 1093.06,758.59 1082.38,735.19">\n      <param key="origId" value="1556742613" />\n    </lane>\n  </edge>\n  <edge id="-155690340" from="1680666699" to="1680666686" name="ซอยละม้ายสงเคราะห์ 1" priority="3" type="highway.residential">\n    <lane id="-155690340_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="158.64" shape="1090.51,421.27 932.04,413.83">\n      <param key="origId" value="155690340" />\n    </lane>\n  </edge>\n  <edge id="-155690480#0" from="4900465244" to="1680667509" priority="3" type="highway.residential">\n    <lane id="-155690480#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="22.78" shape="1364.63,647.84 1341.85,647.97">\n      <param key="origId" value="155690480" />\n    </lane>\n  </edge>\n  <edge id="-155690480#1" from="796747714" to="4900465244" priority="3" type="highway.residential">\n    <lane id="-155690480#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="68.79" shape="1447.83,647.00 1379.04,647.73">\n      <param key="origId" value="155690480" />\n    </lane>\n  </edge>\n  <edge id="-155690513" from="1680667730" to="cluster_1558314142_1558314144" name="Soi 14 Rajyindee Road" priority="3" type="highway.residential" shape="1464.25,711.76 1463.62,711.75 1453.36,711.52 1440.07,711.22 1418.62,710.74 1386.60,710.02">\n    <lane id="-155690513_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="64.40" shape="1458.08,710.03 1453.40,709.92 1440.11,709.62 1418.66,709.14 1393.70,708.58">\n      <param key="origId" value="155690513" />\n    </lane>\n    <param key="origTo" value="1558314144" />\n  </edge>\n  <edge id="-155690528" from="1680667181" to="1680666789" priority="3" type="highway.residential">\n    <lane id="-155690528_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="80.06" shape="1084.03,541.23 1090.67,461.44">\n      <param key="origId" value="155690528" />\n    </lane>\n  </edge>\n  <edge id="-155690734#0" from="1680667079" to="1680667083" priority="3" type="highway.residential">\n    <lane id="-155690734#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="36.52" shape="1538.84,521.41 1502.34,522.40">\n      <param key="origId" value="155690734" />\n    </lane>\n  </edge>\n  <edge id="-155690734#1" from="1680667090" to="1680667079" priority="3" type="highway.residential">\n    <lane id="-155690734#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="36.86" shape="1590.80,522.52 1553.95,521.42">\n      <param key="origId" value="155690734" />\n    </lane>\n  </edge>\n  <edge id="-155690741#1" from="14182422401" to="1680666468" name="Thanon Chaiyakuluthit Soi 2" priority="3" type="highway.residential" shape="730.07,414.83 707.09,411.75 697.89,410.52 685.55,399.46 654.95,370.55">\n    <lane id="-155690741#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="75.28" shape="723.12,412.28 707.30,410.16 698.59,409.00 686.63,398.28 661.58,374.61">\n      <param key="origId" value="155690741" />\n    </lane>\n  </edge>\n  <edge id="-155690741#2" from="1559016909" to="14182422401" name="Thanon Chaiyakuluthit Soi 2" priority="3" type="highway.residential">\n    <lane id="-155690741#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.39" shape="769.50,418.49 737.39,414.20">\n      <param key="origId" value="155690741" />\n    </lane>\n  </edge>\n  <edge id="-155690766" from="1680667686" to="14124372093" name="ถนนคลองเรียน 2 ซอย 9" priority="10" type="highway.tertiary" shape="1803.64,705.64 1681.35,669.43 1689.38,642.59">\n    <lane id="-155690766_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="152.34" shape="1804.09,704.11 1683.34,668.35 1690.91,643.05">\n      <param key="origId" value="155690766" />\n    </lane>\n  </edge>\n  <edge id="-155690791#0" from="14070071629" to="1680666546" name="ซอยสเน่หานุสรณ์" priority="10" type="highway.tertiary" shape="593.64,285.94 579.09,328.01 562.22,386.77">\n    <lane id="-155690791#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="95.31" shape="591.10,288.39 577.56,327.53 562.69,379.34">\n      <param key="origId" value="155690791" />\n    </lane>\n  </edge>\n  <edge id="-155690812#0" from="1680667164" to="13351099349" priority="3" type="highway.residential">\n    <lane id="-155690812#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="26.12" shape="1540.57,547.74 1514.55,550.03">\n      <param key="origId" value="155690812" />\n    </lane>\n  </edge>\n  <edge id="-155690812#1" from="1680667158" to="1680667164" priority="3" type="highway.residential">\n    <lane id="-155690812#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="37.29" shape="1592.11,544.21 1554.90,546.64">\n      <param key="origId" value="155690812" />\n    </lane>\n  </edge>\n  <edge id="-155690875" from="cluster_1680667233_2506085475" to="1680666683" priority="3" type="highway.residential" shape="1219.28,562.16 1227.11,503.25 1221.95,408.43">\n    <lane id="-155690875_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="147.59" shape="1221.80,555.34 1228.72,503.31 1223.55,408.34">\n      <param key="origId" value="155690875" />\n    </lane>\n    <param key="origFrom" value="1680667233" />\n  </edge>\n  <edge id="-155690878#0" from="1680667029" to="cluster_1559016919_1680666950" priority="3" type="highway.residential" shape="981.09,509.46 914.80,503.53">\n    <lane id="-155690878#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="52.25" shape="974.14,507.23 922.10,502.58">\n      <param key="origId" value="155690878" />\n    </lane>\n    <param key="origTo" value="1680666950" />\n  </edge>\n  <edge id="-155690878#1" from="1680667034" to="1680667029" priority="3" type="highway.residential">\n    <lane id="-155690878#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="44.48" shape="1032.71,513.08 988.46,508.60">\n      <param key="origId" value="155690878" />\n    </lane>\n  </edge>\n  <edge id="-155690968" from="1680666294" to="1680666322" priority="3" type="highway.residential" shape="1170.59,306.94 1043.25,303.54 1041.24,303.89 1039.62,304.98 1038.19,306.58 1037.27,314.84 1036.36,316.57 1034.94,317.65 1032.83,318.20 935.82,314.77">\n    <lane id="-155690968_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="237.39" shape="1170.63,305.34 1043.13,301.94 1040.63,302.37 1038.56,303.76 1036.66,305.89 1035.71,314.36 1035.11,315.51 1034.23,316.18 1032.65,316.59 943.31,313.43">\n      <param key="origId" value="155690968" />\n    </lane>\n  </edge>\n  <edge id="-155690981" from="1680666534" to="cluster_1110416379_1680666461" name="Soi 2 Lamaisongkraw" priority="3" type="highway.residential" shape="1109.81,386.07 1065.79,387.10 1065.76,376.41 929.75,368.50">\n    <lane id="-155690981_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="183.66" shape="1109.77,384.47 1067.39,385.46 1067.36,374.90 936.87,367.31">\n      <param key="origId" value="155690981" />\n    </lane>\n    <param key="origTo" value="1680666461" />\n  </edge>\n  <edge id="-155691015#0" from="1680667079" to="1680666827" priority="3" type="highway.residential" shape="1546.44,522.80 1543.84,503.33 1545.67,470.73">\n    <lane id="-155691015#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="36.98" shape="1547.03,515.10 1545.45,503.27 1546.85,478.26">\n      <param key="origId" value="155691015" />\n    </lane>\n  </edge>\n  <edge id="-155691015#1" from="1680667164" to="1680667079" priority="3" type="highway.residential">\n    <lane id="-155691015#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.23" shape="1549.00,541.39 1548.42,530.18">\n      <param key="origId" value="155691015" />\n    </lane>\n  </edge>\n  <edge id="-155691022#0" from="4900465071" to="1680666891" name="Soi 8 Rajyindee" priority="3" type="highway.residential">\n    <lane id="-155691022#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="27.78" shape="1627.59,519.66 1632.96,492.40">\n      <param key="origId" value="155691022" />\n    </lane>\n  </edge>\n  <edge id="-155691022#1" from="1431867140" to="4900465071" name="Soi 8 Rajyindee" priority="3" type="highway.residential" shape="1823.78,612.25 1779.79,609.07 1684.84,605.32 1614.53,587.23 1624.65,526.30">\n    <lane id="-155691022#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="262.85" shape="1823.90,610.65 1779.88,607.47 1685.07,603.73 1616.35,586.05 1625.01,533.88">\n      <param key="origId" value="155691022" />\n    </lane>\n  </edge>\n  <edge id="-155691032" from="1680666823" to="1680667029" priority="3" type="highway.residential" shape="1032.81,467.36 986.39,463.57 981.09,509.46">\n    <lane id="-155691032_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="88.62" shape="1032.94,465.77 984.98,461.85 980.33,502.08">\n      <param key="origId" value="155691032" />\n    </lane>\n  </edge>\n  <edge id="-155691084#0" from="4900465949" to="8642866010" name="ซอย 8 ราษฎร์ยินดี" priority="10" type="highway.tertiary">\n    <lane id="-155691084#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.20" shape="1446.87,457.15 1446.67,457.17">\n      <param key="origId" value="155691084" />\n    </lane>\n  </edge>\n  <edge id="-155691084#1" from="1680666827" to="4900465949" name="ซอย 8 ราษฎร์ยินดี" priority="10" type="highway.tertiary">\n    <lane id="-155691084#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="80.62" shape="1538.97,468.23 1459.06,457.63">\n      <param key="origId" value="155691084" />\n    </lane>\n  </edge>\n  <edge id="-155691084#2" from="1680666891" to="1680666827" name="ซอย 8 ราษฎร์ยินดี" priority="10" type="highway.tertiary" shape="1632.80,484.92 1600.10,478.02 1545.67,470.73">\n    <lane id="-155691084#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="73.84" shape="1626.13,481.88 1600.37,476.44 1553.27,470.13">\n      <param key="origId" value="155691084" />\n    </lane>\n  </edge>\n  <edge id="-155691084#3" from="8523321638" to="1680666891" name="ซอย 8 ราษฎร์ยินดี" priority="10" type="highway.tertiary" shape="1836.34,543.00 1713.76,503.59 1632.80,484.92">\n    <lane id="-155691084#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="204.67" shape="1836.83,541.48 1714.19,502.05 1640.28,485.00">\n      <param key="origId" value="155691084" />\n    </lane>\n  </edge>\n  <edge id="-155691088#0" from="1680666463" to="1431867172" name="Soi Vachira-Anan 1" priority="3" type="highway.residential" shape="1494.83,374.00 1523.62,308.81 1566.82,230.20 1600.88,173.25">\n    <lane id="-155691088#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="219.22" shape="1499.49,367.41 1525.06,309.52 1568.21,231.00 1602.25,174.07">\n      <param key="origId" value="155691088" />\n    </lane>\n  </edge>\n  <edge id="-155691088#1" from="1680666716" to="1680666463" name="Soi Vachira-Anan 1" priority="3" type="highway.residential">\n    <lane id="-155691088#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="46.75" shape="1480.57,426.26 1494.21,381.54">\n      <param key="origId" value="155691088" />\n    </lane>\n  </edge>\n  <edge id="-1557537524" from="14165547216" to="14165547215" priority="3" type="highway.residential">\n    <lane id="-1557537524_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="27.78" shape="1349.62,1276.06 1377.39,1276.47">\n      <param key="origId" value="1557537524" />\n    </lane>\n  </edge>\n  <edge id="-1558734968" from="2506088476" to="14179759489" priority="3" type="highway.residential">\n    <lane id="-1558734968_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="60.52" shape="459.82,1461.60 406.58,1432.83">\n      <param key="origId" value="1558734968" />\n    </lane>\n  </edge>\n  <edge id="-1559020882" from="14182422402" to="14182422401" priority="3" type="highway.residential">\n    <lane id="-1559020882_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="26.65" shape="732.70,381.02 729.38,407.46">\n      <param key="origId" value="1559020882" />\n    </lane>\n  </edge>\n  <edge id="-1559802495#0" from="14188374629" to="4900465288" priority="3" type="highway.residential">\n    <lane id="-1559802495#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.59" shape="1221.57,1553.05 1235.09,1551.73">\n      <param key="origId" value="1559802495" />\n    </lane>\n  </edge>\n  <edge id="-1559802495#1" from="14188374628" to="14188374629" priority="3" type="highway.residential">\n    <lane id="-1559802495#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.22" shape="1194.06,1555.72 1207.23,1554.44">\n      <param key="origId" value="1559802495" />\n    </lane>\n  </edge>\n  <edge id="-1559802496" from="cluster_14188374630_2506088509" to="14188374629" priority="3" type="highway.residential" shape="1211.66,1505.83 1212.12,1514.65 1213.86,1547.57 1214.10,1552.17">\n    <lane id="-1559802496_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.42" shape="1210.46,1513.54 1210.52,1514.73 1212.12,1544.92">\n      <param key="origId" value="1559802496" />\n    </lane>\n    <param key="origFrom" value="14188374630" />\n  </edge>\n  <edge id="-1559841160#0" from="14188764646" to="14188764645" priority="3" type="highway.residential">\n    <lane id="-1559841160#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.07" shape="847.01,579.93 816.06,577.21">\n      <param key="origId" value="1559841160" />\n    </lane>\n  </edge>\n  <edge id="-1559841160#1" from="14188764647" to="14188764646" priority="3" type="highway.residential">\n    <lane id="-1559841160#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.53" shape="858.11,535.57 853.23,574.79">\n      <param key="origId" value="1559841160" />\n    </lane>\n  </edge>\n  <edge id="-1559841161" from="14188764648" to="14188764646" priority="3" type="highway.residential">\n    <lane id="-1559841161_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.30" shape="878.45,583.52 861.28,581.43">\n      <param key="origId" value="1559841161" />\n    </lane>\n  </edge>\n  <edge id="-185951074#0" from="13279030938" to="856214524" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary">\n    <lane id="-185951074#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.96" shape="1314.46,987.56 1329.42,987.38">\n      <param key="origId" value="185951074" />\n    </lane>\n  </edge>\n  <edge id="-185951074#1" from="4900465707" to="13279030938" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary">\n    <lane id="-185951074#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="65.84" shape="1234.13,987.36 1299.97,987.62">\n      <param key="origId" value="185951074" />\n    </lane>\n  </edge>\n  <edge id="-185951074#3" from="1123399600" to="4900465707" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="1084.18,985.09 1136.16,984.71 1173.33,984.99 1228.78,985.74">\n    <lane id="-185951074#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="130.44" shape="1092.44,986.63 1136.16,986.31 1173.31,986.59 1222.87,987.26">\n      <param key="origId" value="185951074" />\n    </lane>\n  </edge>\n  <edge id="-243079673" from="2505176494" to="947896979" priority="4" type="highway.unclassified">\n    <lane id="-243079673_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="2.32" shape="1608.08,1475.28 1610.39,1475.12">\n      <param key="origId" value="243079673" />\n    </lane>\n  </edge>\n  <edge id="-243079723" from="947896972" to="2505176686" name="ถนนราษฏร์ยินดี ซอย 2" priority="3" type="highway.residential" shape="1629.94,1569.30 1795.57,1577.62 1803.25,1578.19 1929.69,1587.64 1953.55,1589.43 1986.72,1591.26">\n    <lane id="-243079723_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="351.13" shape="1636.19,1571.22 1795.47,1579.22 1803.13,1579.79 1929.57,1589.24 1953.45,1591.03 1986.63,1592.86">\n      <param key="origId" value="243079723" />\n    </lane>\n  </edge>\n  <edge id="-243079757#3" from="1639088658" to="8523328988" name="Soi Rajyindee Road 6/ 1" priority="3" type="highway.residential" shape="1548.26,1061.79 1669.41,1078.37 1673.44,1078.96 1682.83,1080.32 1707.65,1084.84 1738.77,1093.07 1746.95,1096.63 1752.94,1098.57 1763.96,1102.14">\n    <lane id="-243079757#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="213.10" shape="1554.66,1064.28 1669.19,1079.96 1673.21,1080.54 1682.57,1081.90 1707.30,1086.40 1738.24,1094.59 1746.38,1098.13 1752.45,1100.09 1763.47,1103.66">\n      <param key="origId" value="243079757" />\n    </lane>\n  </edge>\n  <edge id="-243079775" from="947896989" to="2505176514" priority="3" type="highway.residential">\n    <lane id="-243079775_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.10" shape="1611.41,1492.75 1611.31,1492.76">\n      <param key="origId" value="243079775" />\n    </lane>\n  </edge>\n  <edge id="-243188024" from="2506085525" to="1115585463" priority="3" type="highway.residential" shape="1353.51,736.98 1352.58,744.79 1352.13,748.64 1347.02,759.73 1343.94,766.43 1325.25,778.71 1304.91,788.15 1282.42,790.45 1184.64,787.30 1166.48,787.71 1120.06,798.73 1087.97,813.59 1032.16,846.90 998.04,876.32">\n    <lane id="-243188024_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="394.82" shape="1351.25,742.43 1350.99,744.60 1350.57,748.20 1345.57,759.06 1342.68,765.34 1324.47,777.31 1304.48,786.59 1282.36,788.85 1184.65,785.70 1166.27,786.11 1119.53,797.21 1087.22,812.17 1031.22,845.60 1001.63,871.11">\n      <param key="origId" value="243188024" />\n    </lane>\n  </edge>\n  <edge id="-243188056" from="2506088695" to="1369020942" name="Soi Boonraj" priority="3" type="highway.residential">\n    <lane id="-243188056_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="123.42" shape="1537.57,1534.57 1414.34,1541.28">\n      <param key="origId" value="243188056" />\n    </lane>\n  </edge>\n  <edge id="-243188085" from="2506088938" to="1123399486" priority="3" type="highway.residential">\n    <lane id="-243188085_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="64.27" shape="575.61,1592.82 512.26,1603.70">\n      <param key="origId" value="243188085" />\n    </lane>\n  </edge>\n  <edge id="-243188092" from="2506088507" to="2506088528" name="Soi Boonraj 1" priority="3" type="highway.residential">\n    <lane id="-243188092_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="81.08" shape="1492.77,1513.51 1411.79,1517.41">\n      <param key="origId" value="243188092" />\n    </lane>\n  </edge>\n  <edge id="-243188101" from="12351390200" to="2506088019" priority="3" type="highway.residential" shape="1195.69,1408.63 1189.35,1393.17 1186.69,1386.25">\n    <lane id="-243188101_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="23.96" shape="1197.12,1407.91 1190.84,1392.58 1188.18,1385.68">\n      <param key="origId" value="243188101 1335076429" />\n    </lane>\n  </edge>\n  <edge id="-243188103" from="2506087252" to="2506087304" priority="3" type="highway.residential">\n    <lane id="-243188103_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.49" shape="392.67,1192.34 422.67,1197.73">\n      <param key="origId" value="243188103" />\n    </lane>\n  </edge>\n  <edge id="-243188115" from="2506090451" to="1123399713" name="Soi Thongdee 3" priority="3" type="highway.residential">\n    <lane id="-243188115_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="104.21" shape="1201.90,1766.30 1220.67,1663.80">\n      <param key="origId" value="243188115" />\n    </lane>\n  </edge>\n  <edge id="-243188139" from="2506088011" to="2506088023" name="Soi Boonraj 3" priority="3" type="highway.residential">\n    <lane id="-243188139_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="88.51" shape="1492.42,1379.39 1404.11,1385.38">\n      <param key="origId" value="243188139" />\n    </lane>\n  </edge>\n  <edge id="-243188161" from="2506089487" to="1123399675" priority="3" type="highway.residential">\n    <lane id="-243188161_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="49.84" shape="589.57,1653.68 542.29,1669.46">\n      <param key="origId" value="243188161" />\n    </lane>\n  </edge>\n  <edge id="-243188167" from="2506088265" to="2506088274" name="Soi Boonraj 2" priority="3" type="highway.residential">\n    <lane id="-243188167_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="112.69" shape="1521.64,1455.80 1409.23,1463.75">\n      <param key="origId" value="243188167" />\n    </lane>\n  </edge>\n  <edge id="-243188174#0" from="6634630596" to="2506087709" name="ถนนทิพรัตน์" priority="10" type="highway.tertiary">\n    <lane id="-243188174#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.54" shape="318.54,1285.92 306.07,1284.64">\n      <param key="origId" value="243188174" />\n    </lane>\n  </edge>\n  <edge id="-243188174#1" from="4900465989" to="6634630596" name="ถนนทิพรัตน์" priority="10" type="highway.tertiary">\n    <lane id="-243188174#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="36.07" shape="372.12,1291.44 336.25,1287.75">\n      <param key="origId" value="243188174" />\n    </lane>\n  </edge>\n  <edge id="-243188174#2" from="2506087732" to="4900465989" name="ถนนทิพรัตน์" priority="10" type="highway.tertiary">\n    <lane id="-243188174#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="38.44" shape="425.40,1296.94 387.17,1292.99">\n      <param key="origId" value="243188174" />\n    </lane>\n  </edge>\n  <edge id="-243188196" from="2506089901" to="1123399513" priority="3" type="highway.residential">\n    <lane id="-243188196_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="50.67" shape="1420.69,1712.06 1469.40,1698.10">\n      <param key="origId" value="243188196" />\n    </lane>\n  </edge>\n  <edge id="-243188208" from="2506087962" to="2506087592" priority="3" type="highway.residential">\n    <lane id="-243188208_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="86.17" shape="939.66,1335.41 925.09,1250.49">\n      <param key="origId" value="243188208" />\n    </lane>\n  </edge>\n  <edge id="-243188212#0" from="1123399641" to="12351390200" name="ถนนศุภสารรังสรรค์ ซอย 4/2" priority="3" type="highway.residential">\n    <lane id="-243188212#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="3.52" shape="1198.33,1411.61 1197.25,1408.26">\n      <param key="origId" value="243188212" />\n    </lane>\n  </edge>\n  <edge id="-243188212#1" from="cluster_14188374630_2506088509" to="1123399641" name="ถนนศุภสารรังสรรค์ ซอย 4/2" priority="3" type="highway.residential" shape="1198.03,1507.30 1198.68,1487.25 1199.46,1465.00 1199.28,1419.81">\n    <lane id="-243188212#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="72.11" shape="1199.88,1499.68 1200.28,1487.30 1201.06,1465.02 1200.91,1427.59">\n      <param key="origId" value="243188212" />\n    </lane>\n    <param key="origFrom" value="2506088509" />\n  </edge>\n  <edge id="-243188225" from="4488366488" to="1123399683" name="ถนนศุภสารรังสรรค์ ซอย 4" priority="10" type="highway.tertiary" shape="1235.51,1209.07 1235.99,1262.05 1236.41,1307.92">\n    <lane id="-243188225_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="82.06" shape="1234.00,1218.95 1234.39,1262.06 1234.75,1301.01">\n      <param key="origId" value="243188225" />\n    </lane>\n  </edge>\n  <edge id="-243188243" from="1123399591" to="2506090118" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="-243188243_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.75" shape="562.09,1717.41 534.00,1727.22">\n      <param key="origId" value="243188243" />\n    </lane>\n  </edge>\n  <edge id="-243188273#1" from="1123399569" to="1110416526" name="ถนนจุติอุทิศ 2" priority="10" type="highway.tertiary" shape="879.44,1759.57 869.56,1685.12 864.85,1646.46">\n    <lane id="-243188273#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="102.74" shape="880.48,1755.27 871.15,1684.92 867.30,1653.38">\n      <param key="origId" value="243188273" />\n    </lane>\n  </edge>\n  <edge id="-243188283" from="2506087957" to="1369020943" name="Soi Boonraj 4" priority="3" type="highway.residential">\n    <lane id="-243188283_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="87.07" shape="1487.75,1337.43 1400.93,1343.98">\n      <param key="origId" value="243188283" />\n    </lane>\n  </edge>\n  <edge id="-243188318" from="2506085476" to="2506085486" priority="3" type="highway.residential" shape="1269.22,566.52 1269.08,573.30 1268.68,591.83 1268.62,594.28 1268.26,611.86 1268.19,614.82 1267.81,632.68 1267.64,640.76">\n    <lane id="-243188318_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="66.90" shape="1267.47,573.84 1267.08,591.79 1267.02,594.25 1266.66,611.83 1266.59,614.79 1266.21,632.65 1266.04,640.73">\n      <param key="origId" value="243188318" />\n    </lane>\n  </edge>\n  <edge id="-243188351#1" from="1107926639" to="2506085631" name="ถนนประชาธิปัตย์ ซอย 3" priority="10" type="highway.tertiary" shape="1335.59,881.55 1271.06,884.49 1221.64,888.53">\n    <lane id="-243188351#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="101.32" shape="1328.21,880.28 1270.96,882.89 1227.10,886.48">\n      <param key="origId" value="243188351" />\n    </lane>\n  </edge>\n  <edge id="-243188361" from="2506089058" to="1123399943" priority="3" type="highway.residential">\n    <lane id="-243188361_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="60.37" shape="581.08,1615.61 522.32,1629.48">\n      <param key="origId" value="243188361" />\n    </lane>\n  </edge>\n  <edge id="-243188389#0" from="12346099051" to="12339345632" priority="3" type="highway.residential" shape="566.22,1493.89 623.24,1487.51 642.40,1482.58 667.30,1472.52 687.10,1460.41 699.39,1451.16 724.09,1434.01 752.03,1411.12 756.61,1404.38 754.70,1392.49">\n    <lane id="-243188389#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="218.22" shape="572.48,1494.80 623.53,1489.09 642.90,1484.10 668.02,1473.95 688.00,1461.73 700.33,1452.46 725.05,1435.29 753.22,1412.21 758.29,1404.75 757.44,1399.45">\n      <param key="origId" value="243188389 498671702" />\n    </lane>\n  </edge>\n  <edge id="-243188389#1" from="2506088491" to="12346099051" priority="3" type="highway.residential">\n    <lane id="-243188389#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.20" shape="541.30,1493.79 560.45,1495.10">\n      <param key="origId" value="243188389" />\n    </lane>\n  </edge>\n  <edge id="-243188389#2" from="2506088494" to="2506088491" priority="3" type="highway.residential">\n    <lane id="-243188389#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.93" shape="509.61,1494.76 527.51,1493.70">\n      <param key="origId" value="243188389" />\n    </lane>\n  </edge>\n  <edge id="-243188389#3" from="2506090118" to="2506088494" priority="3" type="highway.residential" shape="531.57,1729.76 519.79,1698.32 505.76,1676.19 486.29,1632.43 465.91,1577.02 464.79,1553.03 471.77,1533.82 479.25,1514.13 486.21,1504.18 503.67,1493.51">\n    <lane id="-243188389#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="252.17" shape="531.97,1726.27 521.23,1697.60 507.17,1675.43 487.77,1631.83 467.50,1576.70 466.40,1553.28 473.27,1534.38 480.68,1514.88 487.33,1505.37 498.27,1498.68">\n      <param key="origId" value="243188389" />\n    </lane>\n  </edge>\n  <edge id="-243188445" from="2506090385" to="1123399740" name="Soi Thongdee 1" priority="3" type="highway.residential" shape="1253.06,1765.60 1268.98,1708.27 1281.68,1643.74">\n    <lane id="-243188445_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="116.69" shape="1254.60,1766.03 1270.54,1708.64 1281.57,1652.59">\n      <param key="origId" value="243188445" />\n    </lane>\n  </edge>\n  <edge id="-243188447" from="2506085849" to="2506085670" name="Soi Duangchan" priority="3" type="highway.residential">\n    <lane id="-243188447_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="79.03" shape="474.56,988.28 492.28,911.26">\n      <param key="origId" value="243188447" />\n    </lane>\n  </edge>\n  <edge id="-243188458" from="2506085493" to="cluster_1680667233_2506085475" priority="3" type="highway.residential" shape="1195.62,654.79 1217.65,646.78 1218.81,641.19 1224.47,613.92 1225.62,608.38 1229.31,590.55 1234.89,563.71">\n    <lane id="-243188458_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="93.73" shape="1204.42,653.29 1219.04,647.98 1220.38,641.52 1226.04,614.25 1227.19,608.70 1230.88,590.88 1234.92,571.43">\n      <param key="origId" value="243188458" />\n    </lane>\n    <param key="origTo" value="2506085475" />\n  </edge>\n  <edge id="-243188465#0" from="4900465473" to="cluster_13424220808_2506085647" name="ถนนประชารมย์" priority="3" type="highway.residential" shape="698.50,730.13 687.90,775.48 659.33,898.58">\n    <lane id="-243188465#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="157.93" shape="695.22,737.12 686.34,775.12 659.46,890.95">\n      <param key="origId" value="243188465" />\n    </lane>\n    <param key="origTo" value="2506085647" />\n  </edge>\n  <edge id="-243188465#1" from="cluster_2052547073_2506085499" to="4900465473" name="ถนนประชารมย์" priority="3" type="highway.residential" shape="713.23,673.94 698.50,730.13">\n    <lane id="-243188465#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="44.56" shape="709.98,680.04 698.68,723.14">\n      <param key="origId" value="243188465" />\n    </lane>\n    <param key="origFrom" value="2506085499" />\n  </edge>\n  <edge id="-243188491" from="1966399248" to="2506088024" name="ถนนประชาธิปัตย์ 1" priority="10" type="highway.tertiary" shape="775.07,1136.75 772.64,1232.89 775.02,1388.94">\n    <lane id="-243188491_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="236.49" shape="773.28,1144.09 771.04,1232.88 773.29,1380.54">\n      <param key="origId" value="243188491" />\n    </lane>\n  </edge>\n  <edge id="-243188503#0" from="2506089901" to="1123399700" priority="3" type="highway.residential">\n    <lane id="-243188503#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="53.58" shape="1415.18,1704.26 1431.08,1653.09">\n      <param key="origId" value="243188503" />\n    </lane>\n  </edge>\n  <edge id="-243188503#1" from="1110416342" to="2506089901" priority="3" type="highway.residential">\n    <lane id="-243188503#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.76" shape="1398.93,1758.29 1410.60,1719.23">\n      <param key="origId" value="243188503" />\n    </lane>\n  </edge>\n  <edge id="-243188505" from="2506087968" to="2506087955" priority="3" type="highway.residential">\n    <lane id="-243188505_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="60.98" shape="636.08,1340.06 575.20,1336.49">\n      <param key="origId" value="243188505" />\n    </lane>\n  </edge>\n  <edge id="-243188508#1" from="cluster_14188374630_2506088509" to="2506088518" priority="3" type="highway.residential" shape="1198.03,1507.30 1169.35,1509.69 1155.61,1510.74">\n    <lane id="-243188508#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="28.09" shape="1191.07,1506.27 1169.22,1508.10 1163.07,1508.57">\n      <param key="origId" value="243188508" />\n    </lane>\n    <param key="origFrom" value="2506088509" />\n  </edge>\n  <edge id="-243188508#3" from="2506088492" to="cluster_14188374630_2506088509" priority="3" type="highway.residential" shape="1242.82,1502.45 1211.66,1505.83">\n    <lane id="-243188508#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.71" shape="1235.09,1501.68 1218.47,1503.48">\n      <param key="origId" value="243188508" />\n    </lane>\n    <param key="origTo" value="14188374630" />\n  </edge>\n  <edge id="-243188536#0" from="8348961009" to="2506087563" priority="3" type="highway.residential">\n    <lane id="-243188536#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.37" shape="422.76,1228.18 420.16,1242.31">\n      <param key="origId" value="243188536" />\n    </lane>\n  </edge>\n  <edge id="-243188536#1" from="2506087304" to="8348961009" priority="3" type="highway.residential">\n    <lane id="-243188536#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.94" shape="427.18,1204.24 425.37,1214.01">\n      <param key="origId" value="243188536" />\n    </lane>\n  </edge>\n  <edge id="-243188536#2" from="2506086743" to="2506087304" priority="3" type="highway.residential">\n    <lane id="-243188536#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="51.67" shape="440.38,1139.53 429.93,1190.14">\n      <param key="origId" value="243188536" />\n    </lane>\n  </edge>\n  <edge id="-243188552#0" from="2506088492" to="1123399516" name="ถนนกุลวัฒน์อุทิศ" priority="10" type="highway.tertiary" shape="1242.82,1502.45 1243.94,1482.31 1244.44,1442.03 1239.67,1387.29">\n    <lane id="-243188552#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="95.24" shape="1244.79,1495.83 1245.54,1482.36 1246.04,1441.97 1242.45,1400.77">\n      <param key="origId" value="243188552" />\n    </lane>\n  </edge>\n  <edge id="-243188552#1" from="4900465288" to="2506088492" name="ถนนกุลวัฒน์อุทิศ" priority="10" type="highway.tertiary">\n    <lane id="-243188552#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.50" shape="1243.98,1542.57 1244.34,1510.07">\n      <param key="origId" value="243188552" />\n    </lane>\n  </edge>\n  <edge id="-243188552#2" from="1107926940" to="4900465288" name="ถนนกุลวัฒน์อุทิศ" priority="10" type="highway.tertiary">\n    <lane id="-243188552#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="23.17" shape="1245.01,1579.91 1244.17,1556.76">\n      <param key="origId" value="243188552" />\n    </lane>\n  </edge>\n  <edge id="-243188561#1" from="2506086151" to="2506085690" priority="3" type="highway.residential" shape="718.23,1077.60 740.35,979.48 751.80,928.65">\n    <lane id="-243188561#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="145.20" shape="719.79,1077.95 741.91,979.83 751.72,936.31">\n      <param key="origId" value="243188561" />\n    </lane>\n  </edge>\n  <edge id="-243188563" from="2506088002" to="2506086795" name="ถนนแสงศรี ซอย 6" priority="10" type="highway.tertiary" shape="833.66,1368.43 830.68,1329.02 825.73,1140.88">\n    <lane id="-243188563_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="212.27" shape="834.67,1360.56 832.28,1328.94 827.53,1148.44">\n      <param key="origId" value="243188563" />\n    </lane>\n  </edge>\n  <edge id="-243188565" from="1123399483" to="2506088518" name="ถนนศุภสารรังสรรค์ ซอย 4/3" priority="3" type="highway.residential">\n    <lane id="-243188565_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="71.24" shape="1157.35,1431.90 1154.33,1503.07">\n      <param key="origId" value="243188565" />\n    </lane>\n  </edge>\n  <edge id="-243188576" from="2506087987" to="14179759489" name="ถนนแสงศรี ซอย 1" priority="3" type="highway.residential" shape="415.88,1357.48 396.81,1424.90 397.16,1429.56">\n    <lane id="-243188576_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="62.32" shape="412.37,1364.01 395.41,1423.97">\n      <param key="origId" value="243188576 498671328" />\n    </lane>\n  </edge>\n  <edge id="-243188585" from="1107926561" to="2506087136" name="ถนนศุภสารรังสรรค์ ซอย 5" priority="10" type="highway.tertiary" shape="1286.69,1575.57 1282.91,1482.86 1280.79,1323.87 1285.38,1183.95">\n    <lane id="-243188585_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="373.19" shape="1287.97,1567.59 1284.51,1482.82 1282.39,1323.89 1286.63,1194.55">\n      <param key="origId" value="243188585" />\n    </lane>\n  </edge>\n  <edge id="-243188607#0" from="2506085493" to="2506085508" priority="3" type="highway.residential" shape="1195.62,654.79 1195.02,660.35 1193.41,675.21 1184.44,688.87">\n    <lane id="-243188607#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="22.54" shape="1193.39,660.58 1191.86,674.65 1187.26,681.66">\n      <param key="origId" value="243188607" />\n    </lane>\n  </edge>\n  <edge id="-243188607#1" from="2506085487" to="2506085493" priority="3" type="highway.residential" shape="1196.15,642.20 1195.78,651.18 1195.62,654.79">\n    <lane id="-243188607#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="3.81" shape="1194.55,642.13 1194.39,645.94">\n      <param key="origId" value="243188607" />\n    </lane>\n  </edge>\n  <edge id="-243188627" from="2506089600" to="2506090779" name="Soi Thongdee 4" priority="3" type="highway.residential" shape="1117.29,1676.07 1112.62,1746.83 1110.54,1782.99 1094.06,1836.35">\n    <lane id="-243188627_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="154.30" shape="1115.15,1684.25 1111.02,1746.73 1108.95,1782.70 1092.53,1835.88">\n      <param key="origId" value="243188627" />\n    </lane>\n  </edge>\n  <edge id="-243188629" from="796747634" to="1107926639" name="ถนนประชาธิปัตย์ ซอย 3" priority="10" type="highway.tertiary" shape="1495.25,902.19 1449.80,899.54 1371.81,886.77 1335.59,881.55">\n    <lane id="-243188629_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="146.93" shape="1489.00,900.22 1449.98,897.95 1372.05,885.19 1343.47,881.07">\n      <param key="origId" value="243188629" />\n    </lane>\n  </edge>\n  <edge id="-243188644" from="2506090932" to="2506089545" name="Soi Thongdee 3" priority="3" type="highway.residential" shape="1132.87,1853.87 1151.92,1783.68 1161.22,1769.38 1171.53,1664.80">\n    <lane id="-243188644_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="186.80" shape="1134.41,1854.29 1153.40,1784.34 1162.77,1769.93 1172.30,1673.27">\n      <param key="origId" value="243188644" />\n    </lane>\n  </edge>\n  <edge id="-243188705" from="1123399516" to="1123399683" name="ถนนศุภสารรังสรรค์ ซอย 4" priority="10" type="highway.tertiary" shape="1239.67,1387.29 1240.19,1378.58 1240.21,1369.87 1239.97,1360.17 1239.41,1351.13 1238.95,1340.28 1238.15,1331.09 1237.36,1319.25 1236.41,1307.92">\n    <lane id="-243188705_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="68.04" shape="1241.53,1382.94 1241.79,1378.63 1241.81,1369.85 1241.57,1360.10 1241.01,1351.05 1240.55,1340.18 1239.75,1330.97 1238.96,1319.13 1238.61,1315.01">\n      <param key="origId" value="243188705" />\n    </lane>\n  </edge>\n  <edge id="-243188710#0" from="1107926467" to="1107926968" name="ซอยบุญราช" priority="10" type="highway.tertiary">\n    <lane id="-243188710#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.27" shape="1510.44,1694.48 1529.77,1729.81">\n      <param key="origId" value="243188710" />\n    </lane>\n  </edge>\n  <edge id="-243188710#1" from="1369020939" to="1107926467" name="ซอยบุญราช" priority="10" type="highway.tertiary" shape="1393.18,1585.88 1439.20,1619.29 1455.76,1632.51 1470.17,1645.72 1480.32,1655.39 1489.65,1665.05 1507.76,1686.25">\n    <lane id="-243188710#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="140.38" shape="1397.39,1590.92 1438.23,1620.56 1454.72,1633.73 1469.08,1646.89 1479.19,1656.53 1488.47,1666.13 1502.77,1682.87">\n      <param key="origId" value="243188710" />\n    </lane>\n  </edge>\n  <edge id="-243188710#2" from="1107926481" to="1369020939" name="ซอยบุญราช" priority="10" type="highway.tertiary" shape="1340.21,1562.31 1380.63,1578.83 1386.23,1581.75 1393.18,1585.88">\n    <lane id="-243188710#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.88" shape="1347.95,1567.20 1379.96,1580.28 1385.45,1583.15 1385.54,1583.20">\n      <param key="origId" value="243188710" />\n    </lane>\n  </edge>\n  <edge id="-243188741#0" from="2506085555" to="1107926537" priority="3" type="highway.residential">\n    <lane id="-243188741#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="107.76" shape="1221.37,812.91 1329.05,808.54">\n      <param key="origId" value="243188741" />\n    </lane>\n  </edge>\n  <edge id="-243188765#0" from="1110416742" to="2505176697" name="ถนนสามชัย" priority="10" type="highway.tertiary">\n    <lane id="-243188765#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="24.18" shape="1344.41,1624.10 1343.50,1599.94">\n      <param key="origId" value="243188765" />\n    </lane>\n  </edge>\n  <edge id="-243188765#1" from="4900465069" to="1110416742" name="ถนนสามชัย" priority="10" type="highway.tertiary">\n    <lane id="-243188765#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="43.41" shape="1347.25,1681.87 1345.05,1638.51">\n      <param key="origId" value="243188765" />\n    </lane>\n  </edge>\n  <edge id="-243188765#2" from="2935421242" to="4900465069" name="ถนนสามชัย" priority="10" type="highway.tertiary" shape="1359.74,1915.00 1357.54,1886.11 1356.01,1864.68 1352.64,1817.30 1350.13,1805.95 1348.32,1768.47 1346.02,1689.19">\n    <lane id="-243188765#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="219.13" shape="1361.34,1914.88 1359.14,1885.99 1357.61,1864.57 1354.23,1817.07 1351.72,1805.74 1349.92,1768.41 1347.83,1696.36">\n      <param key="origId" value="243188765" />\n    </lane>\n  </edge>\n  <edge id="-243188786#0" from="1123399502" to="1110416631" name="ถนนสวนศิริ" priority="10" type="highway.tertiary" shape="565.91,1520.01 566.06,1518.32 566.14,1496.10">\n    <lane id="-243188786#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.49" shape="567.68,1513.07 567.73,1497.58">\n      <param key="origId" value="243188786 243188785" />\n    </lane>\n  </edge>\n  <edge id="-243188786#1" from="2506088938" to="1123399502" name="ถนนสวนศิริ" priority="10" type="highway.tertiary">\n    <lane id="-243188786#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="61.34" shape="583.04,1585.57 568.94,1525.87">\n      <param key="origId" value="243188786" />\n    </lane>\n  </edge>\n  <edge id="-243188786#2" from="2506089058" to="2506088938" name="ถนนสวนศิริ" priority="10" type="highway.tertiary">\n    <lane id="-243188786#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="8.78" shape="588.36,1608.14 586.35,1599.59">\n      <param key="origId" value="243188786" />\n    </lane>\n  </edge>\n  <edge id="-243188786#3" from="2506089487" to="2506089058" name="ถนนสวนศิริ" priority="10" type="highway.tertiary">\n    <lane id="-243188786#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="24.19" shape="597.17,1645.72 591.66,1622.16">\n      <param key="origId" value="243188786" />\n    </lane>\n  </edge>\n  <edge id="-243188786#4" from="1110416754" to="2506089487" name="ถนนสวนศิริ" priority="10" type="highway.tertiary">\n    <lane id="-243188786#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="38.34" shape="609.34,1697.05 600.48,1659.75">\n      <param key="origId" value="243188786" />\n    </lane>\n  </edge>\n  <edge id="-243314073" from="2507400690" to="2507400689" priority="3" type="highway.residential">\n    <lane id="-243314073_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.70" shape="1576.90,1284.08 1577.60,1284.07">\n      <param key="origId" value="243314073" />\n    </lane>\n  </edge>\n  <edge id="-243314103" from="2507400748" to="2507400752" name="ซอยยินดี" priority="3" type="highway.residential">\n    <lane id="-243314103_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="84.14" shape="1606.43,1620.08 1523.33,1633.24">\n      <param key="origId" value="243314103" />\n    </lane>\n  </edge>\n  <edge id="-243314112" from="2507400689" to="12518064113" name="Soi Charoensook" priority="3" type="highway.residential">\n    <lane id="-243314112_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="200.78" shape="1589.79,1284.07 1790.55,1287.17">\n      <param key="origId" value="243314112" />\n    </lane>\n  </edge>\n  <edge id="-243314137" from="2507400792" to="2507400769" priority="10" type="highway.tertiary" shape="832.42,1757.05 830.97,1738.28 815.71,1652.20">\n    <lane id="-243314137_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="99.01" shape="834.02,1756.93 832.56,1738.08 818.58,1659.20">\n      <param key="origId" value="243314137" />\n    </lane>\n  </edge>\n  <edge id="-243314160" from="2507400772" to="1123399863" priority="3" type="highway.residential">\n    <lane id="-243314160_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="64.22" shape="699.10,1685.92 712.88,1748.64">\n      <param key="origId" value="243314160" />\n    </lane>\n  </edge>\n  <edge id="-243314165#0" from="8523326825" to="2507400614" name="ซอยเตชะโต" priority="3" type="highway.residential">\n    <lane id="-243314165#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="27.30" shape="1582.25,878.49 1555.26,874.41">\n      <param key="origId" value="243314165" />\n    </lane>\n  </edge>\n  <edge id="-243314165#1" from="cluster_2507400612_2507400618" to="8523326825" name="ซอยเตชะโต" priority="3" type="highway.residential" shape="1649.02,888.91 1589.04,881.14">\n    <lane id="-243314165#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="45.82" shape="1641.85,886.37 1596.41,880.48">\n      <param key="origId" value="243314165" />\n    </lane>\n    <param key="origFrom" value="2507400618" />\n  </edge>\n  <edge id="-243314177" from="2507400773" to="2507400742" priority="3" type="highway.residential">\n    <lane id="-243314177_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="79.62" shape="1701.63,1686.00 1708.67,1606.69">\n      <param key="origId" value="243314177" />\n    </lane>\n  </edge>\n  <edge id="-243314186" from="cluster_2505176422_2505176434" to="2505176388" priority="3" type="highway.residential" shape="1592.59,1425.94 1606.03,1424.45">\n    <lane id="-243314186_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.40" shape="1600.09,1426.72 1600.49,1426.67">\n      <param key="origId" value="243314186" />\n    </lane>\n    <param key="origFrom" value="2505176422" />\n  </edge>\n  <edge id="-243314192" from="2507400638" to="2507400639" priority="5" type="highway.tertiary_link">\n    <lane id="-243314192_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="1.03" shape="1519.50,925.12 1518.47,925.20">\n      <param key="origId" value="243314192" />\n    </lane>\n  </edge>\n  <edge id="-243314213" from="2507400698" to="2507400696" priority="3" type="highway.residential">\n    <lane id="-243314213_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.55" shape="1587.41,1348.70 1587.96,1348.67">\n      <param key="origId" value="243314213" />\n    </lane>\n  </edge>\n  <edge id="-243314248#0" from="4900465377" to="cluster_2507400612_2507400618" priority="3" type="highway.residential" shape="1736.98,901.02 1705.03,890.88 1651.34,873.76">\n    <lane id="-243314248#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="74.90" shape="1729.97,897.12 1705.52,889.36 1658.59,874.39">\n      <param key="origId" value="243314248" />\n    </lane>\n    <param key="origTo" value="2507400612" />\n  </edge>\n  <edge id="-243314248#1" from="2507400625" to="4900465377" priority="3" type="highway.residential">\n    <lane id="-243314248#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="5.62" shape="1747.60,905.99 1742.88,902.93">\n      <param key="origId" value="243314248" />\n    </lane>\n  </edge>\n  <edge id="-243314310#0" from="2507400591" to="8642866006" priority="3" type="highway.residential" shape="1493.33,719.96 1489.18,720.23 1479.79,720.51 1477.33,720.55">\n    <lane id="-243314310#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="2.27" shape="1487.27,718.69 1485.00,718.75">\n      <param key="origId" value="243314310 155690866 932339001" />\n    </lane>\n  </edge>\n  <edge id="-243314310#1" from="2505176020" to="2507400591" priority="3" type="highway.residential" shape="1628.90,758.14 1631.53,725.98 1493.33,719.96">\n    <lane id="-243314310#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="161.86" shape="1630.96,752.64 1633.26,724.45 1499.81,718.64">\n      <param key="origId" value="243314310" />\n    </lane>\n  </edge>\n  <edge id="-243314312" from="8523326826" to="2507400638" priority="3" type="highway.residential" shape="1584.29,920.15 1549.87,916.18 1531.90,929.27 1525.44,926.27">\n    <lane id="-243314312_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="52.30" shape="1581.31,918.20 1549.43,914.52 1533.10,926.42">\n      <param key="origId" value="243314312" />\n    </lane>\n  </edge>\n  <edge id="-243314378#1" from="8523449430" to="2507400702" name="Thanon Rajyindee Soi 5" priority="3" type="highway.residential" shape="1682.59,1351.59 1704.79,1352.61 1718.36,1352.72 1777.01,1353.17 1890.27,1360.87">\n    <lane id="-243314378#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="200.64" shape="1689.80,1353.52 1704.75,1354.21 1718.35,1354.32 1776.95,1354.77 1890.16,1362.47">\n      <param key="origId" value="243314378" />\n    </lane>\n  </edge>\n  <edge id="-243314378#2" from="4900465441" to="8523449430" name="Thanon Rajyindee Soi 5" priority="3" type="highway.residential">\n    <lane id="-243314378#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.90" shape="1642.55,1351.34 1675.41,1352.86">\n      <param key="origId" value="243314378" />\n    </lane>\n  </edge>\n  <edge id="-243314378#3" from="2507400696" to="4900465441" name="Thanon Rajyindee Soi 5" priority="3" type="highway.residential">\n    <lane id="-243314378#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="28.12" shape="1600.02,1348.74 1628.08,1350.54">\n      <param key="origId" value="243314378" />\n    </lane>\n  </edge>\n  <edge id="-243314395" from="1107926968" to="1680668226" name="ถนนสามชัย ซอย 3" priority="10" type="highway.tertiary" shape="1532.35,1731.19 1590.76,1717.09 1626.47,1708.28">\n    <lane id="-243314395_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="88.56" shape="1535.11,1732.17 1591.14,1718.64 1621.16,1711.24">\n      <param key="origId" value="243314395" />\n    </lane>\n  </edge>\n  <edge id="-243314436#0" from="cluster_2507400612_2507400618" to="2507400624" name="Soi Te Ta Cho" priority="3" type="highway.residential" shape="1649.02,888.91 1647.33,906.25">\n    <lane id="-243314436#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.32" shape="1646.74,895.82 1645.74,906.09">\n      <param key="origId" value="243314436" />\n    </lane>\n    <param key="origFrom" value="2507400618" />\n  </edge>\n  <edge id="-243314436#2" from="2507400604" to="cluster_2507400612_2507400618" name="Soi Te Ta Cho" priority="3" type="highway.residential" shape="1655.82,848.70 1651.34,873.76">\n    <lane id="-243314436#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.37" shape="1652.96,855.59 1650.96,866.78">\n      <param key="origId" value="243314436" />\n    </lane>\n    <param key="origTo" value="2507400612" />\n  </edge>\n  <edge id="-243314436#3" from="13449811511" to="2507400604" name="Soi Te Ta Cho" priority="3" type="highway.residential" shape="1670.51,765.50 1662.85,794.93 1655.82,848.70">\n    <lane id="-243314436#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="71.67" shape="1667.40,771.09 1661.28,794.62 1655.14,841.58">\n      <param key="origId" value="243314436" />\n    </lane>\n  </edge>\n  <edge id="-243314440" from="2505176598" to="947896989" name="Thanon Rajyindee Soi 3" priority="3" type="highway.residential" shape="1988.69,1551.38 1877.94,1534.67 1774.26,1514.87 1687.60,1499.36 1617.54,1494.09">\n    <lane id="-243314440_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="369.55" shape="1988.93,1549.80 1878.21,1533.09 1774.55,1513.30 1687.80,1497.77 1624.07,1492.98">\n      <param key="origId" value="243314440" />\n    </lane>\n  </edge>\n  <edge id="-243314467#0" from="2507400609" to="2507400604" priority="3" type="highway.residential" shape="1741.14,866.28 1714.77,860.84 1655.82,848.70">\n    <lane id="-243314467#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="72.89" shape="1734.67,863.31 1715.09,859.27 1663.28,848.60">\n      <param key="origId" value="243314467" />\n    </lane>\n  </edge>\n  <edge id="-243314467#1" from="2505176027" to="2507400609" priority="3" type="highway.residential" shape="1855.54,779.68 1852.06,837.51 1850.81,854.34 1848.56,861.70 1844.41,868.54 1831.37,877.68 1807.27,875.42 1797.49,874.51 1757.60,868.69 1741.14,866.28">\n    <lane id="-243314467#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="181.90" shape="1853.61,785.09 1850.46,837.40 1849.23,854.04 1847.09,861.04 1843.22,867.42 1830.93,876.03 1807.42,873.83 1797.68,872.92 1757.83,867.11 1748.58,865.75">\n      <param key="origId" value="243314467" />\n    </lane>\n  </edge>\n  <edge id="-324267161#1" from="916327839" to="3309949761" name="Thanon Prathanuthit" priority="10" type="highway.tertiary" shape="407.87,1718.42 394.42,1716.98 342.85,1710.33 273.34,1701.37 235.21,1696.46 167.86,1687.78 165.86,1687.52 108.39,1679.00 0.00,1666.14">\n    <lane id="-324267161#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="404.64" shape="401.51,1716.13 394.61,1715.39 343.05,1708.74 273.54,1699.78 235.41,1694.87 168.06,1686.19 166.08,1685.94 108.60,1677.41 0.19,1664.55">\n      <param key="origId" value="324267161" />\n    </lane>\n  </edge>\n  <edge id="-330615760" from="2506088024" to="2506088243" priority="3" type="highway.residential" shape="775.02,1388.94 775.30,1396.15 775.46,1404.60 775.89,1422.93 775.81,1430.60">\n    <lane id="-330615760_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.33" length="29.54" shape="773.71,1396.66 773.86,1404.64 774.29,1422.94 774.26,1426.20">\n      <param key="origId" value="330615760 330615766 1257722737" />\n    </lane>\n  </edge>\n  <edge id="-330615829#0" from="12346099054" to="947896964" name="ถนนศุภสารรังสรรค์ ซอย 3" priority="10" type="highway.tertiary">\n    <lane id="-330615829#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="20.45" shape="1019.76,1353.41 1020.36,1332.97">\n      <param key="origId" value="330615829" />\n    </lane>\n  </edge>\n  <edge id="-330615829#2" from="3376091528" to="12346099054" name="ถนนศุภสารรังสรรค์ ซอย 3" priority="10" type="highway.tertiary" shape="1017.40,1370.57 1017.49,1366.73 1017.99,1359.35">\n    <lane id="-330615829#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="5.69" shape="1019.00,1370.48 1019.09,1366.80 1019.22,1364.79">\n      <param key="origId" value="330615829" />\n    </lane>\n  </edge>\n  <edge id="-330615844" from="cluster_1123399715_3376084911_3376091818" to="3376091528" priority="10" type="highway.tertiary" shape="1016.75,1382.90 1017.40,1370.57">\n    <lane id="-330615844_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.22" shape="1018.35,1382.98 1018.99,1370.78">\n      <param key="origId" value="330615844" />\n    </lane>\n    <param key="origFrom" value="3376091818" />\n  </edge>\n  <edge id="-452036922" from="4488368090" to="1123399561" name="ถนนศุภสารรังสรรค์ ซอย 6" priority="10" type="highway.tertiary">\n    <lane id="-452036922_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="55.64" shape="1185.22,1296.59 1185.72,1240.96">\n      <param key="origId" value="452036922" />\n    </lane>\n  </edge>\n  <edge id="-452036923#0" from="4488366488" to="5149825984" priority="10" type="highway.tertiary" shape="1235.51,1209.07 1243.53,1202.35 1250.19,1194.55 1257.36,1181.54">\n    <lane id="-452036923#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="21.07" shape="1240.76,1206.76 1244.66,1203.49 1251.51,1195.47 1254.14,1190.70">\n      <param key="origId" value="452036923" />\n    </lane>\n  </edge>\n  <edge id="-452036923#1" from="1123399561" to="4488366488" priority="10" type="highway.tertiary" shape="1184.19,1233.15 1199.91,1226.91 1219.15,1218.30 1227.66,1214.23 1235.51,1209.07">\n    <lane id="-452036923#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="41.12" shape="1190.50,1232.37 1200.53,1228.38 1219.82,1219.75 1228.12,1215.78">\n      <param key="origId" value="452036923" />\n    </lane>\n  </edge>\n  <edge id="-452036923#2" from="1123399535" to="1123399561" priority="10" type="highway.tertiary" shape="1134.93,1234.89 1141.31,1236.36 1146.98,1236.67 1152.72,1236.78 1158.18,1236.60 1169.12,1235.59 1184.19,1233.15">\n    <lane id="-452036923#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.51" shape="1142.40,1238.02 1146.92,1238.27 1152.73,1238.38 1158.28,1238.20 1169.32,1237.18 1176.75,1235.98">\n      <param key="origId" value="452036923" />\n    </lane>\n  </edge>\n  <edge id="-452036923#3" from="4488358857" to="1123399535" priority="10" type="highway.tertiary" shape="1088.44,1204.94 1111.51,1222.97 1117.75,1226.81 1121.95,1229.39 1126.28,1231.49 1130.73,1233.25 1134.93,1234.89">\n    <lane id="-452036923#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.71" shape="1095.49,1212.48 1110.60,1224.29 1116.91,1228.17 1121.18,1230.79 1125.64,1232.96 1128.58,1234.12">\n      <param key="origId" value="452036923" />\n    </lane>\n  </edge>\n  <edge id="-452036923#4" from="4488358858" to="4488358857" priority="10" type="highway.tertiary" shape="1084.03,1302.21 1072.41,1289.19 1066.44,1280.27 1061.68,1271.35 1058.40,1261.02 1055.94,1249.27 1055.17,1220.66 1055.22,1213.45 1055.31,1210.02 1056.13,1206.60 1057.13,1204.39 1059.02,1202.36 1061.61,1200.21 1064.74,1199.20 1068.20,1198.97 1071.48,1198.93 1075.33,1199.49 1078.66,1200.32 1083.69,1202.42 1088.44,1204.94">\n    <lane id="-452036923#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="118.31" shape="1077.33,1292.30 1073.68,1288.21 1067.81,1279.45 1063.16,1270.73 1059.95,1260.61 1057.54,1249.08 1056.77,1220.64 1056.82,1213.47 1056.91,1210.23 1057.65,1207.12 1058.48,1205.29 1060.12,1203.53 1062.39,1201.64 1065.04,1200.78 1068.26,1200.57 1071.37,1200.53 1075.02,1201.06 1078.16,1201.84 1082.34,1203.59">\n      <param key="origId" value="452036923" />\n    </lane>\n  </edge>\n  <edge id="-452036923#5" from="4488358856" to="4488358858" priority="10" type="highway.tertiary" shape="1134.58,1312.92 1132.15,1314.69 1128.49,1316.63 1124.39,1317.88 1119.94,1318.25 1113.13,1317.98 1107.47,1317.15 1101.99,1315.60 1096.93,1313.11 1091.32,1308.84 1084.03,1302.21">\n    <lane id="-452036923#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="43.36" shape="1128.89,1314.61 1127.88,1315.14 1124.09,1316.30 1119.91,1316.65 1113.28,1316.38 1107.81,1315.58 1102.57,1314.10 1097.78,1311.74 1092.34,1307.61 1089.18,1304.73">\n      <param key="origId" value="452036923" />\n    </lane>\n  </edge>\n  <edge id="-452036923#6" from="4488368090" to="4488358856" priority="10" type="highway.tertiary" shape="1183.56,1303.87 1151.97,1302.69 1148.45,1303.06 1145.79,1304.31 1142.13,1306.92 1134.58,1312.92">\n    <lane id="-452036923#6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="35.41" shape="1176.33,1302.00 1151.92,1301.09 1148.01,1301.50 1144.98,1302.92 1141.97,1305.07">\n      <param key="origId" value="452036923" />\n    </lane>\n  </edge>\n  <edge id="-452036923#7" from="1123399683" to="4488368090" priority="10" type="highway.tertiary" shape="1236.41,1307.92 1229.97,1308.56 1223.52,1308.50 1210.44,1307.65 1183.56,1303.87">\n    <lane id="-452036923#7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="38.74" shape="1229.03,1306.95 1223.58,1306.90 1210.60,1306.06 1190.52,1303.23">\n      <param key="origId" value="452036923" />\n    </lane>\n  </edge>\n  <edge id="-452037066" from="4488358856" to="1123399535" name="ถนนศุภสารรังสรรค์ ซอย 7" priority="10" type="highway.tertiary">\n    <lane id="-452037066_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="59.36" shape="1136.23,1302.26 1136.49,1242.90">\n      <param key="origId" value="452037066" />\n    </lane>\n  </edge>\n  <edge id="-452037067" from="4488358858" to="4488358857" name="Thanon Supasarnrangsan Soi 8" priority="3" type="highway.residential">\n    <lane id="-452037067_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="75.32" shape="1086.17,1290.44 1089.58,1215.20">\n      <param key="origId" value="452037067" />\n    </lane>\n  </edge>\n  <edge id="-497290263#0" from="2506089600" to="4888861605" name="Thanon Jirnakorn" priority="10" type="highway.tertiary" shape="1117.29,1676.07 1065.23,1688.05 1050.15,1701.84">\n    <lane id="-497290263#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="66.18" shape="1108.84,1676.37 1064.46,1686.59 1049.23,1700.52">\n      <param key="origId" value="497290263" />\n    </lane>\n  </edge>\n  <edge id="-497290263#1" from="2506089545" to="2506089600" name="Thanon Jirnakorn" priority="10" type="highway.tertiary">\n    <lane id="-497290263#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.61" shape="1163.03,1664.93 1123.26,1673.19">\n      <param key="origId" value="497290263" />\n    </lane>\n  </edge>\n  <edge id="-497290263#2" from="1123399713" to="2506089545" name="Thanon Jirnakorn" priority="10" type="highway.tertiary">\n    <lane id="-497290263#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="35.03" shape="1211.81,1655.08 1177.46,1661.98">\n      <param key="origId" value="497290263" />\n    </lane>\n  </edge>\n  <edge id="-497290263#3" from="1123399740" to="1123399713" name="Thanon Jirnakorn" priority="10" type="highway.tertiary">\n    <lane id="-497290263#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="47.12" shape="1272.83,1643.74 1226.49,1652.24">\n      <param key="origId" value="497290263" />\n    </lane>\n  </edge>\n  <edge id="-497290263#4" from="1110416742" to="1123399740" name="Thanon Jirnakorn" priority="10" type="highway.tertiary">\n    <lane id="-497290263#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="48.86" shape="1335.15,1630.88 1287.34,1640.92">\n      <param key="origId" value="497290263" />\n    </lane>\n  </edge>\n  <edge id="-498671328#0" from="14179759489" to="2506087987" name="ถนนแสงศรี ซอย 1" priority="3" type="highway.residential" shape="397.16,1429.56 396.81,1424.90 415.88,1357.48">\n    <lane id="-498671328#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="61.19" shape="398.80,1423.75 415.45,1364.88">\n      <param key="origId" value="498671328 243188576" />\n    </lane>\n  </edge>\n  <edge id="-498671328#1" from="4900465706" to="14179759489" name="ถนนแสงศรี ซอย 1" priority="3" type="highway.residential">\n    <lane id="-498671328#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="47.56" shape="402.96,1486.69 399.48,1439.26">\n      <param key="origId" value="498671328" />\n    </lane>\n  </edge>\n  <edge id="-498671331" from="2506088476" to="2506088494" priority="3" type="highway.residential" shape="466.68,1467.12 495.75,1481.42 503.67,1493.51">\n    <lane id="-498671331_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.38" shape="471.62,1471.33 494.65,1482.66 498.33,1488.28">\n      <param key="origId" value="498671331" />\n    </lane>\n  </edge>\n  <edge id="-498671388#0" from="4900466448" to="2506086831" name="Thanon Prachathipat 2" priority="3" type="highway.residential">\n    <lane id="-498671388#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="78.40" shape="883.60,1060.32 877.03,1138.45">\n      <param key="origId" value="498671388" />\n    </lane>\n  </edge>\n  <edge id="-498671388#1" from="856214538" to="4900466448" name="Thanon Prachathipat 2" priority="3" type="highway.residential">\n    <lane id="-498671388#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="67.14" shape="894.20,979.18 885.21,1045.71">\n      <param key="origId" value="498671388" />\n    </lane>\n  </edge>\n  <edge id="-498671389#1" from="4900464580" to="4900464597" name="Thanon Boonrong Uthit" priority="3" type="highway.residential" shape="969.09,1061.43 971.51,1085.16 971.45,1088.48 970.92,1121.40 969.86,1153.94">\n    <lane id="-498671389#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="77.10" shape="968.31,1069.50 969.91,1085.23 969.85,1088.45 969.32,1121.36 968.50,1146.51">\n      <param key="origId" value="498671389" />\n    </lane>\n  </edge>\n  <edge id="-498671389#2" from="4900464598" to="4900464580" name="Thanon Boonrong Uthit" priority="3" type="highway.residential">\n    <lane id="-498671389#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="70.72" shape="967.38,983.12 967.48,1053.84">\n      <param key="origId" value="498671389" />\n    </lane>\n  </edge>\n  <edge id="-498671391" from="1123399771" to="4900464587" priority="3" type="highway.residential" shape="1060.17,1111.10 1136.28,1124.51 1241.28,1134.45">\n    <lane id="-498671391_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="169.70" shape="1065.34,1113.64 1136.07,1126.10 1233.52,1135.32">\n      <param key="origId" value="498671391" />\n    </lane>\n  </edge>\n  <edge id="-498671393" from="1115585909" to="4900465166" priority="3" type="highway.residential">\n    <lane id="-498671393_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="93.45" shape="1000.54,920.17 907.14,917.46">\n      <param key="origId" value="498671393" />\n    </lane>\n  </edge>\n  <edge id="-498671427" from="4900465052" to="1107926467" name="ซอยทินกร" priority="3" type="highway.residential">\n    <lane id="-498671427_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="99.47" shape="1614.02,1666.96 1515.89,1683.28">\n      <param key="origId" value="498671427" />\n    </lane>\n  </edge>\n  <edge id="-498671432" from="4900465069" to="4900465067" priority="3" type="highway.residential" shape="1346.02,1689.19 1316.22,1690.21 1318.32,1734.52">\n    <lane id="-498671432_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="70.17" shape="1338.72,1687.84 1314.55,1688.67 1316.72,1734.60">\n      <param key="origId" value="498671432" />\n    </lane>\n  </edge>\n  <edge id="-498671433" from="4900465071" to="4900465070" priority="3" type="highway.residential">\n    <lane id="-498671433_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.86" shape="1631.51,529.55 1670.31,538.66">\n      <param key="origId" value="498671433" />\n    </lane>\n  </edge>\n  <edge id="-498671442" from="4900465099" to="4900465098" priority="3" type="highway.residential">\n    <lane id="-498671442_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="95.51" shape="584.17,1013.07 618.93,924.11">\n      <param key="origId" value="498671442" />\n    </lane>\n  </edge>\n  <edge id="-498671465" from="12649445300" to="4900465166" priority="3" type="highway.residential">\n    <lane id="-498671465_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="74.72" shape="817.95,913.58 892.41,919.84">\n      <param key="origId" value="498671465" />\n    </lane>\n  </edge>\n  <edge id="-498671482" from="4900465209" to="4900465208" priority="3" type="highway.residential">\n    <lane id="-498671482_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="71.90" shape="578.67,580.17 587.55,508.82">\n      <param key="origId" value="498671482" />\n    </lane>\n  </edge>\n  <edge id="-498671483" from="1680666686" to="cluster_1559016910_4900465210" priority="3" type="highway.residential" shape="924.52,415.08 872.26,410.02">\n    <lane id="-498671483_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="37.86" shape="917.45,412.79 879.76,409.14">\n      <param key="origId" value="498671483" />\n    </lane>\n    <param key="origTo" value="4900465210" />\n  </edge>\n  <edge id="-498671497" from="4900465244" to="cluster_1558314142_1558314144" priority="3" type="highway.residential" shape="1372.20,649.40 1366.65,703.92">\n    <lane id="-498671497_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.42" shape="1369.84,656.76 1365.75,696.97">\n      <param key="origId" value="498671497" />\n    </lane>\n    <param key="origTo" value="1558314142" />\n  </edge>\n  <edge id="-498671536" from="4900465363" to="2506087432" name="Soi 1/1 Sophasarnrungsen" priority="3" type="highway.residential">\n    <lane id="-498671536_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="64.17" shape="570.42,1228.17 634.57,1229.65">\n      <param key="origId" value="498671536" />\n    </lane>\n  </edge>\n  <edge id="-498671539" from="4900465370" to="4900465090" priority="3" type="highway.residential" shape="352.76,1624.28 322.01,1625.68 315.46,1661.39">\n    <lane id="-498671539_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="66.38" shape="352.69,1622.68 320.67,1624.14 314.47,1657.91">\n      <param key="origId" value="498671539" />\n    </lane>\n  </edge>\n  <edge id="-498671542" from="4900465377" to="2507400609" priority="3" type="highway.residential">\n    <lane id="-498671542_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.84" shape="1739.50,893.40 1741.86,873.70">\n      <param key="origId" value="498671542" />\n    </lane>\n  </edge>\n  <edge id="-498671544#1" from="13424210990" to="1140280757" priority="3" type="highway.residential" shape="709.31,1046.68 704.66,1072.26 700.57,1090.15 691.90,1129.98">\n    <lane id="-498671544#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="77.40" shape="707.74,1046.39 703.09,1071.94 699.01,1089.80 691.97,1122.16">\n      <param key="origId" value="498671544" />\n    </lane>\n  </edge>\n  <edge id="-498671564" from="4900465441" to="4900465440" priority="3" type="highway.residential">\n    <lane id="-498671564_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.04" shape="1637.40,1342.28 1639.08,1313.28">\n      <param key="origId" value="498671564" />\n    </lane>\n  </edge>\n  <edge id="-498671569" from="2506087136" to="4900465451" priority="3" type="highway.residential">\n    <lane id="-498671569_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="43.44" shape="1287.22,1173.37 1288.20,1129.95">\n      <param key="origId" value="498671569" />\n    </lane>\n  </edge>\n  <edge id="-498671576#0" from="4900465472" to="4900465471" priority="3" type="highway.residential">\n    <lane id="-498671576#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="42.34" shape="645.74,721.32 603.61,717.10">\n      <param key="origId" value="498671576" />\n    </lane>\n  </edge>\n  <edge id="-498671576#1" from="4900465473" to="4900465472" priority="3" type="highway.residential" shape="698.50,730.13 678.85,727.66 651.55,723.51">\n    <lane id="-498671576#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.03" shape="691.21,727.60 679.07,726.08 657.52,722.80">\n      <param key="origId" value="498671576" />\n    </lane>\n  </edge>\n  <edge id="-498671589" from="2506088491" to="4900465517" name="Thanon Sangsri Soi 3" priority="3" type="highway.residential">\n    <lane id="-498671589_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="95.79" shape="536.37,1484.61 544.42,1389.16">\n      <param key="origId" value="498671589" />\n    </lane>\n  </edge>\n  <edge id="-498671600" from="4900465641" to="1558314095" priority="3" type="highway.residential" shape="1675.64,380.29 1683.14,362.56 1697.89,309.43">\n    <lane id="-498671600_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="66.98" shape="1677.11,380.91 1684.65,363.09 1697.39,317.20">\n      <param key="origId" value="498671600" />\n    </lane>\n  </edge>\n  <edge id="-498671615" from="1123399600" to="4900465999" priority="3" type="highway.residential" shape="1084.18,985.09 1051.62,927.71 1047.32,916.15">\n    <lane id="-498671615_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="59.65" shape="1080.40,975.18 1053.07,927.03 1051.58,923.01">\n      <param key="origId" value="498671615" />\n    </lane>\n  </edge>\n  <edge id="-498671616" from="2506085631" to="4900465999" name="ถนนประชาธิปัตย์ ซอย 3" priority="10" type="highway.tertiary">\n    <lane id="-498671616_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="162.71" shape="1215.59,887.87 1054.88,913.33">\n      <param key="origId" value="498671616" />\n    </lane>\n  </edge>\n  <edge id="-498671673" from="2506087522" to="4900465989" priority="3" type="highway.residential" shape="360.41,1242.41 373.69,1273.98 381.04,1293.97">\n    <lane id="-498671673_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="41.15" shape="360.95,1247.82 372.20,1274.57 376.39,1285.95">\n      <param key="origId" value="498671673" />\n    </lane>\n  </edge>\n  <edge id="-498671678" from="4900465999" to="2506085555" priority="3" type="highway.residential" shape="1047.32,916.15 1043.25,908.83 1040.61,897.73 1042.43,887.44 1046.40,878.30 1064.65,856.51 1097.70,837.85 1129.52,825.19 1176.89,815.26 1215.59,811.54">\n    <lane id="-498671678_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="213.01" shape="1045.69,909.92 1044.76,908.25 1042.24,897.68 1043.97,887.90 1047.78,879.15 1065.69,857.76 1098.39,839.30 1129.98,826.73 1177.13,816.84 1210.11,813.67">\n      <param key="origId" value="498671678 243188741" />\n    </lane>\n  </edge>\n  <edge id="-498671679" from="4900466002" to="4900465219" priority="3" type="highway.residential" shape="1197.37,611.04 1146.01,606.92 1149.97,555.31">\n    <lane id="-498671679_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="92.83" shape="1197.50,609.45 1147.73,605.45 1151.01,562.68">\n      <param key="origId" value="498671679" />\n    </lane>\n  </edge>\n  <edge id="-498671694" from="4900465031" to="4900465030" priority="3" type="highway.residential">\n    <lane id="-498671694_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.07" shape="1735.94,413.34 1772.52,429.70">\n      <param key="origId" value="498671694" />\n    </lane>\n  </edge>\n  <edge id="-530314558#0" from="4900464587" to="13545019681" priority="3" type="highway.residential" shape="1241.28,1134.45 1250.93,1086.01 1250.98,1083.23 1250.27,1081.43 1248.23,1080.23 1245.84,1079.75 1243.18,1079.53 1230.26,1080.85 1200.01,1086.64 1168.74,1090.52 1163.63,1091.56 1151.10,1092.82 1146.91,1093.09 1142.37,1093.01 1137.55,1092.40 1132.57,1091.43 1120.89,1086.83 1106.24,1078.99 1091.91,1065.39 1084.98,1054.78 1082.32,1051.80 1066.57,1031.65">\n    <lane id="-530314558#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="250.00" shape="1244.20,1128.01 1252.53,1086.18 1252.59,1082.94 1251.56,1080.33 1248.81,1078.71 1246.06,1078.16 1243.16,1077.92 1230.03,1079.27 1199.76,1085.06 1168.48,1088.94 1163.39,1089.98 1150.97,1091.23 1146.87,1091.49 1142.48,1091.41 1137.80,1090.82 1133.02,1089.89 1121.56,1085.38 1107.19,1077.68 1093.15,1064.36 1086.25,1053.80 1083.55,1050.77 1073.41,1037.81">\n      <param key="origId" value="530314558 1476708031" />\n    </lane>\n  </edge>\n  <edge id="-530314558#1" from="1123399639" to="4900464587" priority="3" type="highway.residential">\n    <lane id="-530314558#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.15" shape="1234.21,1172.63 1241.14,1142.26">\n      <param key="origId" value="530314558" />\n    </lane>\n  </edge>\n  <edge id="-66984610#0" from="2505176258" to="856214571" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="-66984610#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.20" shape="1549.11,1200.18 1548.91,1200.20">\n      <param key="origId" value="66984610" />\n    </lane>\n  </edge>\n  <edge id="-684741141#0" from="1123399675" to="1123399591" priority="3" type="highway.residential" shape="535.40,1673.45 543.92,1689.30 557.04,1704.41 572.08,1715.62">\n    <lane id="-684741141#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.64" shape="537.68,1681.08 542.60,1690.21 555.95,1705.59 563.09,1710.91">\n      <param key="origId" value="684741141" />\n    </lane>\n  </edge>\n  <edge id="-684741141#1" from="1123399943" to="1123399675" priority="3" type="highway.residential">\n    <lane id="-684741141#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.65" shape="517.06,1640.81 530.92,1668.14">\n      <param key="origId" value="684741141" />\n    </lane>\n  </edge>\n  <edge id="-684741141#2" from="1123399486" to="1123399943" priority="3" type="highway.residential">\n    <lane id="-684741141#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.46" shape="506.09,1614.56 510.91,1627.12">\n      <param key="origId" value="684741141" />\n    </lane>\n  </edge>\n  <edge id="-684741141#3" from="1123399502" to="1123399486" priority="3" type="highway.residential" shape="565.91,1520.01 530.57,1518.11 513.77,1518.80 503.98,1523.23 499.02,1530.34 496.34,1551.16 498.85,1577.34 504.75,1606.61">\n    <lane id="-684741141#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="137.40" shape="558.79,1518.02 530.58,1516.51 513.39,1517.21 502.92,1521.95 497.48,1529.75 494.73,1551.13 497.27,1577.58 501.78,1599.96">\n      <param key="origId" value="684741141" />\n    </lane>\n  </edge>\n  <edge id="-684741338" from="cluster_1123399715_3376084911_3376091818" to="1123399489" priority="3" type="highway.residential" shape="1028.07,1396.94 1034.14,1397.73 1044.60,1401.15">\n    <lane id="-684741338_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.68" shape="1027.86,1398.53 1033.79,1399.30 1040.16,1401.38">\n      <param key="origId" value="684741338 684740716" />\n    </lane>\n    <param key="origFrom" value="3376084911" />\n  </edge>\n  <edge id="-706087699#0" from="4900465031" to="1558314095" priority="3" type="highway.residential" shape="1729.59,408.75 1750.84,340.22 1745.44,336.33 1742.99,333.67 1742.39,331.89 1742.33,329.88 1748.43,261.40 1705.02,261.15 1699.57,302.99 1697.89,309.43">\n    <lane id="-706087699#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="239.94" shape="1733.14,402.69 1752.71,339.59 1746.51,335.13 1744.40,332.84 1743.98,331.60 1743.93,329.93 1750.18,259.81 1703.62,259.54 1698.04,302.37">\n      <param key="origId" value="706087699 498671695 142395239" />\n    </lane>\n  </edge>\n  <edge id="-706087699#1" from="1431867103" to="4900465031" priority="3" type="highway.residential">\n    <lane id="-706087699#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="28.55" shape="1721.00,443.97 1728.98,416.56">\n      <param key="origId" value="706087699" />\n    </lane>\n  </edge>\n  <edge id="-706094740#0" from="cluster_1558314142_1558314144" to="1107926433" name="ถนนสามชัย" priority="3" type="highway.residential" shape="1386.60,710.02 1384.91,735.44">\n    <lane id="-706094740#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.67" shape="1384.54,716.96 1383.69,729.60">\n      <param key="origId" value="706094740" />\n    </lane>\n    <param key="origFrom" value="1558314144" />\n  </edge>\n  <edge id="-706094740#2" from="2506085508" to="cluster_1558314142_1558314144" name="ถนนสามชัย" priority="3" type="highway.residential" shape="1184.44,688.87 1196.41,694.94 1231.94,700.19 1302.34,705.45 1366.65,703.92">\n    <lane id="-706094740#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="170.65" shape="1190.48,693.72 1195.92,696.48 1231.76,701.78 1302.30,707.05 1359.87,705.68">\n      <param key="origId" value="706094740" />\n    </lane>\n    <param key="origTo" value="1558314142" />\n  </edge>\n  <edge id="-706094741#0" from="1123399641" to="1123399516" name="Thanon Supasarnrangsan Soi 4/1" priority="3" type="highway.residential" shape="1199.28,1419.81 1225.96,1403.23 1230.06,1400.05 1233.61,1396.60 1237.18,1392.48 1238.72,1389.96 1239.67,1387.29">\n    <lane id="-706094741#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.31" shape="1207.00,1416.89 1226.88,1404.55 1231.11,1401.26 1232.94,1399.48">\n      <param key="origId" value="706094741" />\n    </lane>\n  </edge>\n  <edge id="-706094741#1" from="1123399483" to="1123399641" name="Thanon Supasarnrangsan Soi 4/1" priority="3" type="highway.residential" shape="1159.26,1424.69 1178.99,1423.33 1199.28,1419.81">\n    <lane id="-706094741#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="25.88" shape="1166.22,1425.81 1179.18,1424.92 1191.88,1422.72">\n      <param key="origId" value="706094741" />\n    </lane>\n  </edge>\n  <edge id="-706094741#2" from="1123399659" to="1123399483" name="Thanon Supasarnrangsan Soi 4/1" priority="3" type="highway.residential" shape="1100.56,1422.57 1130.75,1424.18 1159.26,1424.69">\n    <lane id="-706094741#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="45.25" shape="1106.74,1424.50 1130.69,1425.78 1151.95,1426.16">\n      <param key="origId" value="706094741" />\n    </lane>\n  </edge>\n  <edge id="-706094741#3" from="1123399489" to="1123399659" name="Thanon Supasarnrangsan Soi 4/1" priority="3" type="highway.residential">\n    <lane id="-706094741#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="47.70" shape="1051.23,1405.40 1095.78,1422.45">\n      <param key="origId" value="706094741" />\n    </lane>\n  </edge>\n  <edge id="-706094742" from="2506085508" to="1558997825" name="ถนนสามชัย" priority="3" type="highway.residential" shape="1184.44,688.87 1171.54,683.66 1152.06,679.26 1116.01,679.94 1079.31,683.08 1062.24,692.08 1053.99,703.38">\n    <lane id="-706094742_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="127.72" shape="1178.90,684.91 1172.02,682.13 1152.22,677.66 1115.93,678.34 1078.85,681.51 1061.17,690.84 1057.33,696.08">\n      <param key="origId" value="706094742" />\n    </lane>\n  </edge>\n  <edge id="-706095305" from="2506088476" to="947896956" name="Thanon Sangchan" priority="3" type="highway.residential">\n    <lane id="-706095305_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="76.24" shape="469.38,1458.74 479.15,1383.12">\n      <param key="origId" value="706095305" />\n    </lane>\n  </edge>\n  <edge id="-706095306" from="2506087732" to="2506087987" priority="3" type="highway.residential">\n    <lane id="-706095306_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="48.97" shape="426.47,1302.06 415.92,1349.88">\n      <param key="origId" value="706095306" />\n    </lane>\n  </edge>\n  <edge id="-706095308#0" from="6634630596" to="6634630594" priority="3" type="highway.residential">\n    <lane id="-706095308#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="37.09" shape="328.49,1297.43 340.23,1332.60">\n      <param key="origId" value="706095308" />\n    </lane>\n  </edge>\n  <edge id="-706095308#1" from="cluster_2506087462_8523307708" to="6634630596" priority="3" type="highway.residential" shape="310.25,1233.64 327.17,1288.42">\n    <lane id="-706095308#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="41.08" shape="310.91,1241.20 323.03,1280.45">\n      <param key="origId" value="706095308" />\n    </lane>\n    <param key="origFrom" value="8523307708" />\n  </edge>\n  <edge id="-706096644#1" from="1122072681" to="1174378788" name="ถนนเสน่หานุสรณ์" priority="10" type="highway.tertiary" shape="494.55,491.05 500.61,440.08 505.02,403.09">\n    <lane id="-706096644#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="75.75" shape="496.99,484.07 502.20,440.27 505.94,408.85">\n      <param key="origId" value="706096644" />\n    </lane>\n  </edge>\n  <edge id="-706103887" from="2506087889" to="2506087863" name="Soi Watthayorn" priority="3" type="highway.residential">\n    <lane id="-706103887_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="64.95" shape="636.74,1311.63 572.02,1306.23">\n      <param key="origId" value="706103887" />\n    </lane>\n  </edge>\n  <edge id="-706104217#1" from="8523431062" to="947896964" name="ถนนเเสงศรี" priority="3" type="highway.residential" shape="988.37,1331.11 1003.24,1327.91 1018.99,1324.95">\n    <lane id="-706104217#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.10" shape="996.65,1330.97 1003.56,1329.48 1011.45,1328.00">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="-706104217#10" from="2506088024" to="2506088002" name="ถนนเเสงศรี" priority="3" type="highway.residential" shape="775.02,1388.94 831.06,1369.45 833.66,1368.43">\n    <lane id="-706104217#10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="46.01" shape="784.81,1387.23 828.27,1372.11">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="-706104217#11" from="12339345632" to="2506088024" name="ถนนเเสงศรี" priority="3" type="highway.residential">\n    <lane id="-706104217#11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="5.81" shape="762.03,1392.83 767.75,1391.83">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="-706104217#12" from="2506088084" to="12339345632" name="ถนนเเสงศรี" priority="3" type="highway.residential">\n    <lane id="-706104217#12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.44" shape="731.69,1398.58 747.83,1395.45">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="-706104217#13" from="1140280758" to="2506088084" name="ถนนเเสงศรี" priority="3" type="highway.residential" shape="677.38,1402.31 701.16,1401.53 725.72,1398.11">\n    <lane id="-706104217#13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.02" shape="683.15,1403.72 701.30,1403.13 721.96,1400.25">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="-706104217#14" from="cluster_2506088039_8523406713" to="1140280758" name="ถนนเเสงศรี" priority="3" type="highway.residential" shape="651.83,1397.58 677.38,1402.31">\n    <lane id="-706104217#14_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.66" shape="658.54,1400.45 670.99,1402.75">\n      <param key="origId" value="706104217" />\n    </lane>\n    <param key="origFrom" value="8523406713" />\n  </edge>\n  <edge id="-706104217#16" from="1110416470" to="cluster_2506088039_8523406713" name="ถนนเเสงศรี" priority="3" type="highway.residential" shape="573.07,1385.51 633.99,1393.35">\n    <lane id="-706104217#16_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="50.22" shape="578.50,1387.82 628.31,1394.23">\n      <param key="origId" value="706104217" />\n    </lane>\n    <param key="origTo" value="2506088039" />\n  </edge>\n  <edge id="-706104217#17" from="4900465517" to="1110416470" name="ถนนเเสงศรี" priority="3" type="highway.residential">\n    <lane id="-706104217#17_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.21" shape="550.52,1384.23 566.59,1386.29">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="-706104217#18" from="947896956" to="4900465517" name="ถนนเเสงศรี" priority="3" type="highway.residential">\n    <lane id="-706104217#18_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="50.97" shape="485.38,1378.12 536.14,1382.66">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="-706104217#19" from="2506087987" to="947896956" name="ถนนเเสงศรี" priority="3" type="highway.residential">\n    <lane id="-706104217#19_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="51.28" shape="422.37,1361.06 471.56,1375.53">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="-706104217#2" from="8523431064" to="8523431062" name="ถนนเเสงศรี" priority="3" type="highway.residential">\n    <lane id="-706104217#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.96" shape="968.78,1337.10 982.40,1334.07">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="-706104217#20" from="6634630594" to="2506087987" name="ถนนเเสงศรี" priority="3" type="highway.residential" shape="344.79,1341.21 403.15,1353.73 415.88,1357.48">\n    <lane id="-706104217#20_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="59.58" shape="350.15,1344.00 402.76,1355.28 408.30,1356.92">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="-706104217#21" from="cluster_916327730_916327962_985092239_985092700" to="6634630594" name="ถนนเเสงศรี" priority="3" type="highway.residential" shape="313.71,1340.07 338.73,1339.91 344.79,1341.21">\n    <lane id="-706104217#21_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.31" shape="322.68,1341.61 334.98,1341.53">\n      <param key="origId" value="706104217" />\n    </lane>\n    <param key="origFrom" value="985092700" />\n  </edge>\n  <edge id="-706104217#3" from="2506087962" to="8523431064" name="ถนนเเสงศรี" priority="3" type="highway.residential">\n    <lane id="-706104217#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="8.28" shape="947.03,1342.67 955.01,1340.49">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="-706104217#7" from="947896950" to="2506087962" name="ถนนเเสงศรี" priority="3" type="highway.residential" shape="894.34,1353.65 895.42,1353.48 916.10,1348.60 936.76,1343.77 939.35,1343.10">\n    <lane id="-706104217#7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="33.77" shape="900.13,1354.01 916.47,1350.16 933.00,1346.29">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="-706104217#8" from="2506088002" to="947896950" name="ถนนเเสงศรี" priority="3" type="highway.residential">\n    <lane id="-706104217#8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="49.41" shape="841.59,1368.15 889.60,1356.45">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="-706104218" from="1123399639" to="5149825984" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="-706104218_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.69" shape="1238.57,1181.93 1248.24,1182.56">\n      <param key="origId" value="706104218" />\n    </lane>\n  </edge>\n  <edge id="-706104219#0" from="cluster_1115585393_13472440789_947896971" to="1123399639" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="1050.46,1163.25 1230.93,1179.84">\n    <lane id="-706104219#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="168.53" shape="1056.19,1165.38 1224.01,1180.81">\n      <param key="origId" value="706104219" />\n    </lane>\n    <param key="origFrom" value="1115585393" />\n  </edge>\n  <edge id="-706104219#11" from="1140280757" to="cluster_1966399241_8523333972" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="691.90,1129.98 722.38,1132.46">\n    <lane id="-706104219#11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.64" shape="699.40,1132.20 716.98,1133.63">\n      <param key="origId" value="706104219" />\n    </lane>\n    <param key="origTo" value="1966399241" />\n  </edge>\n  <edge id="-706104219#12" from="1110416444" to="1140280757" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="-706104219#12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="33.76" shape="651.33,1126.40 684.82,1130.69">\n      <param key="origId" value="706104219" />\n    </lane>\n  </edge>\n  <edge id="-706104219#13" from="1110416654" to="1110416444" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="-706104219#13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="65.07" shape="573.32,1111.56 637.17,1124.13">\n      <param key="origId" value="706104219" />\n    </lane>\n  </edge>\n  <edge id="-706104219#14" from="1107784887" to="1110416654" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="-706104219#14_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.27" shape="524.74,1098.31 562.52,1109.02">\n      <param key="origId" value="706104219" />\n    </lane>\n  </edge>\n  <edge id="-706104219#15" from="2506086181" to="1107784887" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="-706104219#15_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="62.72" shape="454.09,1073.60 513.18,1094.62">\n      <param key="origId" value="706104219" />\n    </lane>\n  </edge>\n  <edge id="-706104219#16" from="4085816302" to="2506086181" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="336.66,1023.03 365.48,1029.57 393.30,1039.01 399.77,1047.10 407.39,1053.24 449.53,1070.28">\n    <lane id="-706104219#16_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="107.88" shape="346.48,1026.90 365.04,1031.11 392.34,1040.37 398.63,1048.24 406.57,1054.63 443.42,1069.54">\n      <param key="origId" value="706104219" />\n    </lane>\n  </edge>\n  <edge id="-706104219#4" from="4900464597" to="cluster_1115585393_13472440789_947896971" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="969.86,1153.94 993.81,1158.75 1025.86,1161.53">\n    <lane id="-706104219#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="44.23" shape="976.12,1156.83 993.58,1160.34 1019.91,1162.62">\n      <param key="origId" value="706104219" />\n    </lane>\n    <param key="origTo" value="13472440789" />\n  </edge>\n  <edge id="-706104219#6" from="2506086831" to="4900464597" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="878.02,1145.79 909.53,1148.59 969.86,1153.94">\n    <lane id="-706104219#6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="77.63" shape="885.04,1148.02 909.39,1150.18 962.36,1154.88">\n      <param key="origId" value="706104219" />\n    </lane>\n  </edge>\n  <edge id="-706104219#7" from="2506086795" to="2506086831" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="-706104219#7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="37.68" shape="833.16,1143.18 870.67,1146.71">\n      <param key="origId" value="706104219" />\n    </lane>\n  </edge>\n  <edge id="-706104219#8" from="1966399248" to="2506086795" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="-706104219#8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="36.47" shape="782.40,1138.95 818.75,1141.92">\n      <param key="origId" value="706104219" />\n    </lane>\n  </edge>\n  <edge id="-706104219#9" from="cluster_1966399241_8523333972" to="1966399248" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="732.16,1133.26 775.07,1136.75">\n    <lane id="-706104219#9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="28.57" shape="739.46,1135.46 767.94,1137.78">\n      <param key="origId" value="706104219" />\n    </lane>\n    <param key="origFrom" value="8523333972" />\n  </edge>\n  <edge id="-706105950" from="8976429009" to="1107926537" name="ถนนสามชัย" priority="3" type="highway.residential" shape="1379.76,739.49 1376.45,754.50 1373.59,766.33 1363.49,785.54 1353.53,794.92 1336.13,806.65">\n    <lane id="-706105950_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="70.91" shape="1376.90,745.01 1374.89,754.14 1372.08,765.76 1362.20,784.56 1352.53,793.67 1340.02,802.10">\n      <param key="origId" value="706105950" />\n    </lane>\n  </edge>\n  <edge id="-706105951#0" from="1107926513" to="cluster_13472440775_1666900747" name="ถนนสามชัย" priority="3" type="highway.residential" shape="1334.88,1134.02 1333.17,1187.69">\n    <lane id="-706105951#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="35.96" shape="1333.05,1141.15 1331.91,1177.09">\n      <param key="origId" value="706105951" />\n    </lane>\n    <param key="origTo" value="13472440775" />\n  </edge>\n  <edge id="-706105951#2" from="856214524" to="1107926513" name="ถนนสามชัย" priority="3" type="highway.residential" shape="1336.71,985.69 1334.96,1064.21 1334.88,1134.02">\n    <lane id="-706105951#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="133.97" shape="1334.95,992.96 1333.36,1064.19 1333.29,1126.92">\n      <param key="origId" value="706105951" />\n    </lane>\n  </edge>\n  <edge id="-706105951#3" from="1107926639" to="856214524" name="ถนนสามชัย" priority="3" type="highway.residential">\n    <lane id="-706105951#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="89.22" shape="1334.07,889.30 1335.03,978.51">\n      <param key="origId" value="706105951" />\n    </lane>\n  </edge>\n  <edge id="-706105951#5" from="1107926537" to="1107926639" name="ถนนสามชัย" priority="3" type="highway.residential" shape="1336.13,806.65 1337.36,832.67 1335.59,881.55">\n    <lane id="-706105951#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="60.73" shape="1334.87,813.87 1335.76,832.68 1334.24,874.55">\n      <param key="origId" value="706105951" />\n    </lane>\n  </edge>\n  <edge id="-706256377#0" from="2052547075" to="1110416410" name="ถนนชีอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="-706256377#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="33.87" shape="704.44,599.76 738.18,602.65">\n      <param key="origId" value="706256377" />\n    </lane>\n  </edge>\n  <edge id="-706256377#1" from="1107784883" to="2052547075" name="ถนนชีอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="-706256377#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="65.23" shape="628.23,593.12 693.21,598.79">\n      <param key="origId" value="706256377" />\n    </lane>\n  </edge>\n  <edge id="-706256377#2" from="4900465209" to="1107784883" name="ถนนชีอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="-706256377#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="33.41" shape="583.36,589.45 616.65,592.14">\n      <param key="origId" value="706256377" />\n    </lane>\n  </edge>\n  <edge id="-706256377#3" from="2052547074" to="4900465209" name="ถนนชีอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="-706256377#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="81.15" shape="488.19,579.71 568.91,588.11">\n      <param key="origId" value="706256377" />\n    </lane>\n  </edge>\n  <edge id="-706256377#4" from="1176926303" to="2052547074" name="ถนนชีอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="-706256377#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="66.19" shape="410.82,573.21 476.79,578.65">\n      <param key="origId" value="706256377" />\n    </lane>\n  </edge>\n  <edge id="-706256379#0" from="1431715409" to="cluster_1107773211_1110416556" name="ถนนกิมประดิษฐ์" priority="10" type="highway.tertiary" shape="732.80,514.38 734.81,514.54 750.02,515.15 755.95,515.14">\n    <lane id="-706256379#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="7.18" shape="738.20,516.28 745.37,516.56">\n      <param key="origId" value="706256379 1374050352 1374050353" />\n    </lane>\n    <param key="origTo" value="1110416556" />\n  </edge>\n  <edge id="-706256379#1" from="cluster_1122072658_14138648129" to="1431715409" name="ถนนกิมประดิษฐ์" priority="10" type="highway.tertiary" shape="710.30,512.38 732.80,514.38">\n    <lane id="-706256379#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.20" shape="715.59,514.46 728.75,515.63">\n      <param key="origId" value="706256379" />\n    </lane>\n    <param key="origFrom" value="14138648129" />\n  </edge>\n  <edge id="-706256379#3" from="1107784878" to="cluster_1122072658_14138648129" name="ถนนกิมประดิษฐ์" priority="10" type="highway.tertiary" shape="634.65,505.81 698.36,511.35">\n    <lane id="-706256379#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="48.97" shape="639.92,507.87 688.71,512.12">\n      <param key="origId" value="706256379" />\n    </lane>\n    <param key="origTo" value="1122072658" />\n  </edge>\n  <edge id="-706256379#4" from="4900465208" to="1107784878" name="ถนนกิมประดิษฐ์" priority="10" type="highway.tertiary" shape="586.86,501.44 602.22,502.97 634.65,505.81">\n    <lane id="-706256379#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="35.12" shape="593.79,503.74 602.07,504.56 628.76,506.90">\n      <param key="origId" value="706256379" />\n    </lane>\n  </edge>\n  <edge id="-706256379#6" from="1122072681" to="4900465208" name="ถนนกิมประดิษฐ์" priority="10" type="highway.tertiary" shape="494.55,491.05 525.51,494.53 586.86,501.44">\n    <lane id="-706256379#6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="78.43" shape="501.55,493.45 525.33,496.12 579.49,502.22">\n      <param key="origId" value="706256379" />\n    </lane>\n  </edge>\n  <edge id="-706256380#0" from="1680666322" to="1110416677" name="ถนนละม้ายสงเคราะห์" priority="10" type="highway.tertiary" shape="935.82,314.77 938.88,286.13 945.08,227.47 946.62,206.52">\n    <lane id="-706256380#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="101.41" shape="938.20,307.55 940.47,286.30 946.67,227.61 948.22,206.64">\n      <param key="origId" value="706256380" />\n    </lane>\n  </edge>\n  <edge id="-706256380#1" from="cluster_1110416379_1680666461" to="1680666322" name="ถนนละม้ายสงเคราะห์" priority="10" type="highway.tertiary" shape="932.07,348.59 933.40,339.41 935.82,314.77">\n    <lane id="-706256380#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.98" shape="934.67,341.78 934.99,339.60 936.73,321.90">\n      <param key="origId" value="706256380" />\n    </lane>\n    <param key="origFrom" value="1110416379" />\n  </edge>\n  <edge id="-706256380#3" from="1680666686" to="cluster_1110416379_1680666461" name="ถนนละม้ายสงเคราะห์" priority="10" type="highway.tertiary" shape="924.52,415.08 925.67,405.81 929.75,368.50">\n    <lane id="-706256380#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.40" shape="927.03,407.88 927.26,406.00 930.58,375.67">\n      <param key="origId" value="706256380" />\n    </lane>\n    <param key="origTo" value="1680666461" />\n  </edge>\n  <edge id="-706256380#4" from="cluster_1559016919_1680666950" to="1680666686" name="ถนนละม้ายสงเคราะห์" priority="10" type="highway.tertiary" shape="916.74,482.51 924.52,415.08">\n    <lane id="-706256380#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="53.42" shape="919.15,475.54 925.28,422.47">\n      <param key="origId" value="706256380" />\n    </lane>\n    <param key="origFrom" value="1559016919" />\n  </edge>\n  <edge id="-706256380#6" from="13339819888" to="cluster_1559016919_1680666950" name="ถนนละม้ายสงเคราะห์" priority="10" type="highway.tertiary" shape="912.32,529.55 914.80,503.53">\n    <lane id="-706256380#6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.08" shape="914.47,523.86 915.71,510.83">\n      <param key="origId" value="706256380" />\n    </lane>\n    <param key="origTo" value="1680666950" />\n  </edge>\n  <edge id="-706256381" from="13339819893" to="1160612969" name="ถนนเทพสงเคราะห์" priority="10" type="highway.tertiary" shape="909.94,541.55 909.56,545.24 902.44,617.43">\n    <lane id="-706256381_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="63.68" shape="907.81,546.70 901.56,610.08">\n      <param key="origId" value="706256381" />\n    </lane>\n  </edge>\n  <edge id="-706256386" from="cluster_1559016919_1680666950" to="1559016918" name="ถนนฉัยยากุลอุทิศ ซอย 1" priority="10" type="highway.tertiary" shape="916.74,482.51 865.46,476.66">\n    <lane id="-706256386_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="37.28" shape="909.77,480.10 872.73,475.88">\n      <param key="origId" value="706256386" />\n    </lane>\n    <param key="origFrom" value="1559016919" />\n  </edge>\n  <edge id="-706520617#0" from="12649445299" to="1115585416" name="Thumnoonvithi Road" priority="3" type="highway.residential" shape="902.62,876.30 910.16,874.76 935.89,862.92 957.51,845.29">\n    <lane id="-706520617#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="51.95" shape="909.25,876.58 910.66,876.29 936.74,864.29 953.63,850.52">\n      <param key="origId" value="706520617" />\n    </lane>\n  </edge>\n  <edge id="-706520617#1" from="1123399562" to="12649445299" name="Thumnoonvithi Road" priority="3" type="highway.residential" shape="822.54,865.07 837.89,873.59 882.87,875.84 897.13,877.41 902.62,876.30">\n    <lane id="-706520617#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="67.44" shape="828.81,870.38 837.44,875.17 882.74,877.44 894.88,878.77">\n      <param key="origId" value="706520617" />\n    </lane>\n  </edge>\n  <edge id="-706520617#2" from="1115585792" to="1123399562" name="Thumnoonvithi Road" priority="3" type="highway.residential">\n    <lane id="-706520617#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.44" shape="781.91,843.58 816.12,863.23">\n      <param key="origId" value="706520617" />\n    </lane>\n  </edge>\n  <edge id="-706520617#3" from="2052489649" to="1115585792" name="Thumnoonvithi Road" priority="3" type="highway.residential" shape="730.39,800.51 746.91,817.81 778.12,839.56">\n    <lane id="-706520617#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="50.88" shape="734.70,807.34 745.86,819.03 774.34,838.88">\n      <param key="origId" value="706520617" />\n    </lane>\n  </edge>\n  <edge id="-706520617#4" from="1110416568" to="2052489649" name="Thumnoonvithi Road" priority="3" type="highway.residential" shape="741.15,676.65 723.19,754.34 723.06,777.36 730.39,800.51">\n    <lane id="-706520617#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="115.51" shape="737.87,683.74 721.59,754.15 721.46,777.60 727.43,796.47">\n      <param key="origId" value="706520617" />\n    </lane>\n  </edge>\n  <edge id="-71380788" from="2505176747" to="823954361" name="Thanon Rajyindee Soi 1/1" priority="3" type="highway.residential" shape="1734.25,1636.75 1782.01,1634.57 1850.43,1645.23 1949.93,1657.34 1983.68,1660.37">\n    <lane id="-71380788_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="250.96" shape="1734.32,1638.35 1781.92,1636.18 1850.21,1646.82 1949.76,1658.93 1983.54,1661.96">\n      <param key="origId" value="71380788" />\n    </lane>\n  </edge>\n  <edge id="-72016745#1" from="1369020950" to="856214571" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="1385.88,1192.43 1476.45,1199.74 1532.40,1204.67 1544.12,1202.38">\n    <lane id="-72016745#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="146.54" shape="1392.98,1197.82 1476.05,1204.52 1532.65,1209.51 1539.34,1208.20">\n      <param key="origId" value="72016745" />\n    </lane>\n    <lane id="-72016745#1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="146.54" shape="1393.24,1194.63 1476.32,1201.33 1532.48,1206.28 1538.73,1205.06">\n      <param key="origId" value="72016745" />\n    </lane>\n  </edge>\n  <edge id="-72016745#2" from="cluster_13472440775_1666900747" to="1369020950" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="1337.16,1188.00 1385.88,1192.43">\n    <lane id="-72016745#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.64" shape="1344.14,1193.45 1378.64,1196.59">\n      <param key="origId" value="72016745" />\n    </lane>\n    <lane id="-72016745#2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.64" shape="1344.43,1190.27 1378.93,1193.41">\n      <param key="origId" value="72016745" />\n    </lane>\n    <param key="origFrom" value="1666900747" />\n  </edge>\n  <edge id="-72016745#4" from="2506087136" to="cluster_13472440775_1666900747" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="1285.38,1183.95 1333.17,1187.69">\n    <lane id="-72016745#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.94" shape="1292.48,1189.32 1325.32,1191.89">\n      <param key="origId" value="72016745" />\n    </lane>\n    <lane id="-72016745#4_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.94" shape="1292.73,1186.13 1325.57,1188.70">\n      <param key="origId" value="72016745" />\n    </lane>\n    <param key="origTo" value="13472440775" />\n  </edge>\n  <edge id="-72016745#5" from="5149825984" to="2506087136" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="-72016745#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.54" shape="1264.89,1187.01 1277.39,1188.08">\n      <param key="origId" value="72016745" />\n    </lane>\n    <lane id="-72016745#5_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.54" shape="1265.17,1183.82 1277.66,1184.89">\n      <param key="origId" value="72016745" />\n    </lane>\n  </edge>\n  <edge id="-81350005#2" from="947896964" to="cluster_1115585393_13472440789_947896971" name="Tiparat Road" priority="10" type="highway.tertiary" shape="1018.99,1324.95 1023.66,1248.28 1024.18,1247.20 1025.14,1232.01 1032.62,1162.00">\n    <lane id="-81350005#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="149.69" shape="1020.98,1318.56 1025.24,1248.69 1025.76,1247.61 1026.73,1232.15 1033.43,1169.51">\n      <param key="origId" value="81350005" />\n    </lane>\n    <param key="origTo" value="947896971" />\n  </edge>\n  <edge id="-81350006#1" from="2507400773" to="7691192946" name="Thanon Rajyindee Soi 1" priority="10" type="highway.tertiary" shape="1699.32,1693.92 1725.14,1689.85 1781.33,1686.14 1826.63,1692.44 1854.69,1696.27 1971.98,1710.95">\n    <lane id="-81350006#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="266.16" shape="1707.57,1694.24 1725.32,1691.44 1781.27,1687.75 1826.41,1694.02 1854.48,1697.86 1971.78,1712.54">\n      <param key="origId" value="81350006" />\n    </lane>\n  </edge>\n  <edge id="-81350006#2" from="947896981" to="2507400773" name="Thanon Rajyindee Soi 1" priority="10" type="highway.tertiary">\n    <lane id="-81350006#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="35.85" shape="1658.23,1703.83 1693.37,1696.75">\n      <param key="origId" value="81350006" />\n    </lane>\n  </edge>\n  <edge id="-898440288" from="8348961009" to="8348961008" priority="3" type="highway.residential">\n    <lane id="-898440288_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="101.73" shape="418.82,1218.52 318.69,1200.55">\n      <param key="origId" value="898440288" />\n    </lane>\n  </edge>\n  <edge id="-900244003#0" from="8363308288" to="1107926513" priority="3" type="highway.residential">\n    <lane id="-900244003#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.66" shape="1355.77,1133.04 1342.11,1132.63">\n      <param key="origId" value="900244003" />\n    </lane>\n  </edge>\n  <edge id="-900244003#1" from="8363308289" to="8363308288" priority="3" type="highway.residential">\n    <lane id="-900244003#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="59.68" shape="1428.63,1143.19 1369.62,1134.27">\n      <param key="origId" value="900244003" />\n    </lane>\n  </edge>\n  <edge id="-900244004" from="8363308291" to="8363308288" priority="3" type="highway.residential" shape="1437.69,1124.41 1364.26,1110.52 1362.66,1134.84">\n    <lane id="-900244004_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="95.77" shape="1437.99,1122.84 1362.78,1108.61 1361.52,1127.81">\n      <param key="origId" value="900244004" />\n    </lane>\n  </edge>\n  <edge id="-917695557" from="8523328981" to="8523328986" priority="3" type="highway.residential" shape="1638.21,1116.72 1624.61,1106.37 1610.70,1101.62 1595.00,1100.24 1576.33,1100.04 1554.92,1101.11">\n    <lane id="-917695557_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="82.77" shape="1639.18,1115.45 1625.37,1104.94 1611.03,1100.04 1595.08,1098.64 1576.30,1098.44 1560.84,1099.21">\n      <param key="origId" value="917695557" />\n    </lane>\n  </edge>\n  <edge id="-917695719" from="8523326825" to="8523326826" name="ซอยเตชะโต" priority="3" type="highway.residential">\n    <lane id="-917695719_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="28.89" shape="1586.58,888.12 1583.09,916.80">\n      <param key="origId" value="917695719" />\n    </lane>\n  </edge>\n  <edge id="-917696759" from="8523333971" to="cluster_1966399241_8523333972" priority="3" type="highway.residential" shape="740.12,1083.30 732.16,1133.26">\n    <lane id="-917696759_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="43.13" shape="738.54,1083.05 731.75,1125.65">\n      <param key="origId" value="917696759" />\n    </lane>\n    <param key="origTo" value="8523333972" />\n  </edge>\n  <edge id="-917697169" from="8523329815" to="cluster_620881097_8523329816" name="Thanon Lamaipradit" priority="3" type="highway.residential" shape="373.17,1170.89 317.35,1157.66">\n    <lane id="-917697169_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="48.66" shape="373.54,1169.33 326.19,1158.11">\n      <param key="origId" value="917697169" />\n    </lane>\n    <param key="origTo" value="8523329816" />\n  </edge>\n  <edge id="-917704692" from="8523406712" to="cluster_2506088039_8523406713" priority="3" type="highway.residential" shape="634.98,1476.99 651.83,1397.58">\n    <lane id="-917704692_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="74.06" shape="636.55,1477.32 651.92,1404.87">\n      <param key="origId" value="917704692" />\n    </lane>\n    <param key="origTo" value="8523406713" />\n  </edge>\n  <edge id="-917705248" from="8523431061" to="8523431062" priority="3" type="highway.residential">\n    <lane id="-917705248_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="54.24" shape="989.20,1268.73 987.09,1322.93">\n      <param key="origId" value="917705248" />\n    </lane>\n  </edge>\n  <edge id="-917705249" from="8523431063" to="8523431064" priority="3" type="highway.residential">\n    <lane id="-917705249_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="67.65" shape="945.70,1263.80 958.37,1330.25">\n      <param key="origId" value="917705249" />\n    </lane>\n  </edge>\n  <edge id="-917706996" from="8523449429" to="8523449430" priority="3" type="highway.residential">\n    <lane id="-917706996_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="27.56" shape="1683.54,1316.71 1681.53,1344.20">\n      <param key="origId" value="917706996" />\n    </lane>\n  </edge>\n  <edge id="-931354569#0" from="856214503" to="2506085690" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="702.16,913.43 708.72,914.50 727.59,921.92 751.80,928.65">\n    <lane id="-931354569#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="38.96" shape="707.70,915.95 708.29,916.05 727.08,923.44 744.58,928.30">\n      <param key="origId" value="931354569" />\n    </lane>\n  </edge>\n  <edge id="-931354569#1" from="cluster_13424220808_2506085647" to="856214503" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="666.89,901.24 702.16,913.43">\n    <lane id="-931354569#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="25.64" shape="671.84,904.64 696.07,913.02">\n      <param key="origId" value="931354569" />\n    </lane>\n    <param key="origFrom" value="13424220808" />\n  </edge>\n  <edge id="-931354569#3" from="2506085628" to="cluster_13424220808_2506085647" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="615.42,885.46 638.61,891.99 659.33,898.58">\n    <lane id="-931354569#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.89" shape="620.21,888.47 638.15,893.52 651.73,897.84">\n      <param key="origId" value="931354569" />\n    </lane>\n    <param key="origTo" value="2506085647" />\n  </edge>\n  <edge id="-931354569#4" from="1966386254" to="2506085628" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="564.09,872.41 577.84,875.77 595.79,879.99 615.42,885.46">\n    <lane id="-931354569#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.75" shape="569.93,875.48 577.47,877.33 595.39,881.54 609.43,885.45">\n      <param key="origId" value="931354569" />\n    </lane>\n  </edge>\n  <edge id="-932324369" from="8642733849" to="8642733850" priority="3" type="highway.residential">\n    <lane id="-932324369_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.98" shape="1637.39,442.86 1644.08,426.16">\n      <param key="origId" value="932324369" />\n    </lane>\n  </edge>\n  <edge id="-932339005" from="8642866010" to="1680666775" priority="3" type="highway.residential" shape="1447.22,458.73 1438.77,459.59 1434.74,459.90">\n    <lane id="-932339005_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="4.88" shape="1446.93,457.15 1442.07,457.65">\n      <param key="origId" value="932339005 932339006" />\n    </lane>\n  </edge>\n  <edge id="-95533432#0" from="2506085476" to="1110416454" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary">\n    <lane id="-95533432#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="26.49" shape="1276.45,568.61 1302.88,570.38">\n      <param key="origId" value="95533432" />\n    </lane>\n  </edge>\n  <edge id="-95533432#1" from="cluster_1680667233_2506085475" to="2506085476" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary" shape="1234.89,563.71 1269.22,566.52">\n    <lane id="-95533432#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="20.61" shape="1241.56,565.86 1262.10,567.54">\n      <param key="origId" value="95533432" />\n    </lane>\n    <param key="origFrom" value="2506085475" />\n  </edge>\n  <edge id="-95533432#3" from="4900465219" to="cluster_1680667233_2506085475" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary" shape="1149.97,555.31 1219.28,562.16">\n    <lane id="-95533432#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="55.28" shape="1157.05,557.62 1212.06,563.05">\n      <param key="origId" value="95533432" />\n    </lane>\n    <param key="origTo" value="1680667233" />\n  </edge>\n  <edge id="-95533432#4" from="1680667181" to="4900465219" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary">\n    <lane id="-95533432#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="54.26" shape="1088.76,550.58 1142.73,556.17">\n      <param key="origId" value="95533432" />\n    </lane>\n  </edge>\n  <edge id="-95533432#5" from="13339844003" to="1680667181" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary" shape="915.79,534.39 927.61,535.69 973.52,539.57 1081.84,548.25">\n    <lane id="-95533432#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="153.98" shape="921.11,536.58 927.46,537.28 973.39,541.16 1074.56,549.27">\n      <param key="origId" value="95533432" />\n    </lane>\n  </edge>\n  <edge id="-95542480" from="1680668223" to="947896981" priority="10" type="highway.tertiary">\n    <lane id="-95542480_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="1.24" shape="1645.82,1706.27 1647.04,1706.03">\n      <param key="origId" value="95542480" />\n    </lane>\n  </edge>\n  <edge id="-95542482#0" from="1107926561" to="1107926481" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="-95542482#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.32" shape="1294.76,1575.22 1332.93,1565.76">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="-95542482#1" from="1107926940" to="1107926561" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="-95542482#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.15" shape="1251.94,1587.34 1280.89,1578.92">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="-95542482#10" from="2507400769" to="1110416526" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="-95542482#10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.91" shape="823.24,1652.93 857.92,1648.88">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="-95542482#11" from="1110416469" to="2507400769" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="-95542482#11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="50.29" shape="759.30,1663.25 808.90,1654.96">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="-95542482#12" from="2507400772" to="1110416469" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="-95542482#12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.50" shape="706.22,1677.89 745.09,1666.50">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="-95542482#13" from="1110416754" to="2507400772" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="-95542482#13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="78.55" shape="617.01,1704.05 692.39,1681.95">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="-95542482#14" from="1123399591" to="1110416754" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="-95542482#14_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="24.05" shape="580.12,1714.92 603.20,1708.11">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="-95542482#3" from="1107926599" to="1107926940" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary" shape="1131.66,1608.75 1158.83,1606.70 1169.20,1605.36 1179.04,1604.10 1196.94,1600.50 1243.71,1588.07">\n    <lane id="-95542482#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="103.12" shape="1136.66,1609.98 1158.99,1608.29 1169.40,1606.95 1179.30,1605.68 1197.30,1602.06 1237.79,1591.30">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="-95542482#4" from="1107926977" to="1107926599" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="-95542482#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="38.65" shape="1087.94,1614.84 1126.39,1610.90">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="-95542482#5" from="1107926734" to="1107926977" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="-95542482#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="25.79" shape="1037.43,1621.82 1062.96,1618.19">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="-95542482#9" from="1110416526" to="1107926734" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary" shape="864.85,1646.46 882.92,1643.76 886.75,1643.20 980.57,1629.40 989.17,1627.95 1029.82,1621.29">\n    <lane id="-95542482#9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="152.55" shape="872.10,1646.99 883.16,1645.34 886.98,1644.78 980.82,1630.98 989.43,1629.53 1022.91,1624.04">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="-95542485#3" from="cluster_13472440775_1666900747" to="1107926481" name="ถนนสามชัย" priority="10" type="highway.tertiary" shape="1337.16,1188.00 1335.39,1221.92 1333.87,1308.83 1333.52,1328.97 1333.40,1335.98 1332.74,1374.07 1332.12,1439.67 1340.21,1562.31">\n    <lane id="-95542485#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="357.63" shape="1335.01,1198.43 1333.79,1221.86 1332.27,1308.80 1331.92,1328.94 1331.80,1335.95 1331.14,1374.05 1330.52,1439.72 1338.17,1555.75">\n      <param key="origId" value="95542485" />\n    </lane>\n    <param key="origFrom" value="1666900747" />\n  </edge>\n  <edge id="-95809258" from="2505176697" to="1107926481" name="ถนนสามชัย" priority="10" type="highway.tertiary">\n    <lane id="-95809258_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.13" shape="1342.89,1585.35 1342.23,1571.24">\n      <param key="origId" value="95809258" />\n    </lane>\n  </edge>\n  <edge id="-95809262#0" from="1559016903" to="cluster_1110416379_1680666461" name="ถนนฉัยยากุลอุทิศ ซอย 4" priority="10" type="highway.tertiary" shape="879.58,342.38 932.07,348.59">\n    <lane id="-95809262#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="38.30" shape="886.58,344.82 924.62,349.32">\n      <param key="origId" value="95809262" />\n    </lane>\n    <param key="origTo" value="1110416379" />\n  </edge>\n  <edge id="-95809262#1" from="1431715372" to="1559016903" name="ถนนฉัยยากุลอุทิศ ซอย 4" priority="10" type="highway.tertiary">\n    <lane id="-95809262#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="76.31" shape="796.25,336.17 872.23,343.30">\n      <param key="origId" value="95809262" />\n    </lane>\n  </edge>\n  <edge id="-96713525" from="2505176388" to="2505176470" name="ถนนราฎร์ยินดี ซอย 4" priority="3" type="highway.residential" shape="1606.03,1424.45 1740.65,1430.88 1746.91,1430.90 1758.26,1431.12 1759.14,1427.29 1759.19,1422.07 1761.09,1418.23 1763.55,1414.75 1768.30,1412.57 1775.69,1411.99 1870.66,1421.13 1964.46,1441.79 1991.67,1447.43">\n    <lane id="-96713525_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="395.80" shape="1612.24,1426.35 1740.61,1432.48 1746.89,1432.50 1759.53,1432.74 1760.74,1427.48 1760.79,1422.45 1762.47,1419.05 1764.61,1416.02 1768.71,1414.14 1775.68,1413.60 1870.41,1422.71 1964.13,1443.35 1991.35,1449.00">\n      <param key="origId" value="96713525" />\n    </lane>\n  </edge>\n  <edge id="-96837170#2" from="1122072674" to="cluster_1122072658_14138648129" name="ถนนชนตารา" priority="10" type="highway.tertiary" shape="587.71,394.75 585.50,408.14 586.87,419.02 595.95,432.07 602.26,438.87 621.25,452.16 634.67,457.85 651.08,463.22 665.22,468.29 677.86,477.88 687.78,488.73 698.36,511.35">\n    <lane id="-96837170#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="160.46" shape="585.11,400.66 583.88,408.11 585.33,419.61 594.70,433.08 601.20,440.08 620.47,453.57 634.11,459.35 650.56,464.73 664.45,469.71 676.78,479.07 686.43,489.63 692.87,503.38">\n      <param key="origId" value="96837170" />\n    </lane>\n    <param key="origTo" value="1122072658" />\n  </edge>\n  <edge id="-96837170#4" from="14063214200" to="1122072674" name="ถนนชนตารา" priority="10" type="highway.tertiary" shape="717.14,283.47 712.14,286.10 697.30,298.24 671.09,319.68 638.40,344.49 600.18,377.18 587.71,394.75">\n    <lane id="-96837170#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="166.51" shape="716.40,282.05 711.25,284.76 696.29,297.00 670.10,318.42 637.40,343.24 598.99,376.09 589.76,389.09">\n      <param key="origId" value="96837170" />\n    </lane>\n  </edge>\n  <edge id="-96837172#0" from="1431715395" to="1174378788" name="ถนนเสน่หานุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="-96837172#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="20.45" shape="511.20,378.34 505.01,397.83">\n      <param key="origId" value="96837172" />\n    </lane>\n  </edge>\n  <edge id="-96837172#1" from="1122072673" to="1431715395" name="ถนนเสน่หานุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="-96837172#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="66.29" shape="537.15,301.83 515.74,364.57">\n      <param key="origId" value="96837172" />\n    </lane>\n  </edge>\n  <edge id="-96941841#0" from="1123399700" to="2505176697" name="ซอยชุมอุทิศ 1" priority="10" type="highway.tertiary" shape="1432.19,1644.11 1405.51,1624.24 1391.95,1614.51 1379.04,1605.89 1374.63,1603.37 1370.80,1601.10 1366.43,1599.00 1360.98,1596.67 1352.05,1594.15 1341.59,1591.88">\n    <lane id="-96941841#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="92.06" shape="1428.12,1639.08 1406.45,1622.95 1392.86,1613.19 1379.88,1604.53 1375.43,1601.99 1371.56,1599.69 1367.09,1597.54 1361.51,1595.16 1352.44,1592.60 1349.87,1592.04">\n      <param key="origId" value="96941841" />\n    </lane>\n  </edge>\n  <edge id="-96941841#1" from="1123399513" to="1123399700" name="ซอยชุมอุทิศ 1" priority="10" type="highway.tertiary" shape="1476.70,1694.34 1468.76,1680.28 1463.81,1672.96 1460.87,1669.38 1457.62,1666.20 1432.19,1644.11">\n    <lane id="-96941841#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="51.44" shape="1474.13,1686.54 1470.12,1679.44 1465.09,1672.00 1462.05,1668.30 1458.71,1665.02 1439.96,1648.74">\n      <param key="origId" value="96941841" />\n    </lane>\n  </edge>\n  <edge id="-96941841#2" from="1110416567" to="1123399513" name="ซอยชุมอุทิศ 1" priority="10" type="highway.tertiary">\n    <lane id="-96941841#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="43.61" shape="1501.04,1738.32 1481.15,1699.51">\n      <param key="origId" value="96941841" />\n    </lane>\n  </edge>\n  <edge id="-96941842#0" from="12649445300" to="1123399562" name="ถนนประชาธิปัตย์ 1" priority="10" type="highway.tertiary" shape="810.60,911.36 812.26,902.01 822.54,865.07">\n    <lane id="-96941842#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.12" shape="813.49,904.25 813.82,902.37 821.92,873.26">\n      <param key="origId" value="96941842" />\n    </lane>\n  </edge>\n  <edge id="-96941842#1" from="1123399706" to="12649445300" name="ถนนประชาธิปัตย์ 1" priority="10" type="highway.tertiary">\n    <lane id="-96941842#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="20.01" shape="807.64,938.19 811.01,918.47">\n      <param key="origId" value="96941842" />\n    </lane>\n  </edge>\n  <edge id="-96941842#2" from="1123399595" to="1123399706" name="ถนนประชาธิปัตย์ 1" priority="10" type="highway.tertiary">\n    <lane id="-96941842#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="84.82" shape="793.23,1037.30 805.26,953.34">\n      <param key="origId" value="96941842" />\n    </lane>\n  </edge>\n  <edge id="-96941842#3" from="1966399248" to="1123399595" name="ถนนประชาธิปัตย์ 1" priority="10" type="highway.tertiary">\n    <lane id="-96941842#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="79.24" shape="777.89,1129.63 791.02,1051.49">\n      <param key="origId" value="96941842" />\n    </lane>\n  </edge>\n  <edge id="-96941849" from="5149825984" to="1123399600" name="ถนนศุภสารรังสรรค์ ซอย 9" priority="3" type="highway.residential" shape="1257.36,1181.54 1274.73,1096.98 1276.73,1086.74 1276.93,1084.28 1276.93,1082.33 1276.72,1079.73 1276.47,1077.41 1275.35,1073.39 1273.78,1069.30 1271.57,1065.56 1268.49,1062.04 1264.99,1059.50 1260.20,1056.96 1250.01,1055.45 1226.74,1058.06 1156.64,1067.35 1135.66,1065.63 1119.55,1057.96 1107.81,1043.83 1096.73,1029.41 1090.48,1011.70 1086.82,994.00 1084.18,985.09">\n    <lane id="-96941849_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="341.17" shape="1261.10,1171.26 1276.30,1097.29 1278.32,1086.96 1278.53,1084.34 1278.53,1082.27 1278.31,1079.58 1278.05,1077.11 1276.87,1072.89 1275.23,1068.60 1272.87,1064.62 1269.58,1060.85 1265.84,1058.14 1260.71,1055.42 1250.04,1053.84 1226.55,1056.47 1156.60,1065.74 1136.08,1064.06 1120.56,1056.67 1109.06,1042.83 1098.15,1028.64 1092.02,1011.27 1088.37,993.61 1088.06,992.55">\n      <param key="origId" value="96941849" />\n    </lane>\n  </edge>\n  <edge id="-96941857#0" from="4900466448" to="1123399595" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="885.81,1053.19 844.49,1050.89 820.81,1047.80 790.61,1044.28">\n    <lane id="-96941857#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="81.00" shape="878.62,1051.19 844.64,1049.30 821.01,1046.21 798.03,1043.53">\n      <param key="origId" value="96941857" />\n    </lane>\n  </edge>\n  <edge id="-96941857#1" from="4900464580" to="4900466448" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="969.09,1061.43 945.80,1058.79 885.81,1053.19">\n    <lane id="-96941857#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="68.80" shape="961.73,1058.99 945.96,1057.20 893.26,1052.28">\n      <param key="origId" value="96941857" />\n    </lane>\n  </edge>\n  <edge id="-96941857#2" from="1966404349" to="4900464580" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="-96941857#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="44.61" shape="1021.52,1065.51 977.17,1060.70">\n      <param key="origId" value="96941857" />\n    </lane>\n  </edge>\n  <edge id="-969757849#0" from="4900464598" to="1115585385" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="968.97,975.80 997.07,976.80 1009.74,977.26 1029.31,977.95">\n    <lane id="-969757849#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="44.22" shape="976.23,977.66 997.01,978.40 1009.68,978.86 1020.43,979.24">\n      <param key="origId" value="969757849" />\n    </lane>\n  </edge>\n  <edge id="-969757849#1" from="856214538" to="4900464598" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="896.70,972.59 929.25,973.33 968.97,975.80">\n    <lane id="-969757849#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="57.98" shape="903.97,974.36 929.18,974.93 961.88,976.96">\n      <param key="origId" value="969757849" />\n    </lane>\n  </edge>\n  <edge id="-969757849#2" from="1123399706" to="856214538" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary">\n    <lane id="-969757849#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="80.11" shape="811.70,949.20 888.54,971.85">\n      <param key="origId" value="969757849" />\n    </lane>\n  </edge>\n  <edge id="-969757849#3" from="2506085690" to="1123399706" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary">\n    <lane id="-969757849#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.42" shape="758.45,932.44 796.97,944.69">\n      <param key="origId" value="969757849" />\n    </lane>\n  </edge>\n  <edge id="-969765780#1" from="14159239427" to="cluster_1123399587_8976429007" name="Thumnoonvithi Road" priority="3" type="highway.residential" shape="1077.61,728.60 1100.90,712.74">\n    <lane id="-969765780#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.99" shape="1084.05,726.15 1093.13,719.97">\n      <param key="origId" value="969765780" />\n    </lane>\n    <param key="origTo" value="1123399587" />\n  </edge>\n  <edge id="-969765780#2" from="1115585416" to="14159239427" name="Thumnoonvithi Road" priority="3" type="highway.residential" shape="957.51,845.29 996.73,797.55 1072.06,732.75 1077.61,728.60">\n    <lane id="-969765780#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="152.11" shape="963.58,840.42 997.88,798.67 1072.23,734.71">\n      <param key="origId" value="969765780" />\n    </lane>\n  </edge>\n  <edge id="-969765781" from="1431715392" to="1431715406" name="ถนนชีอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="-969765781_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="23.15" shape="775.64,606.03 798.71,607.96">\n      <param key="origId" value="969765781" />\n    </lane>\n  </edge>\n  <edge id="-969765782" from="1110416410" to="1431715392" name="ถนนชีอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="-969765782_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.94" shape="750.35,603.73 764.23,605.03">\n      <param key="origId" value="969765782" />\n    </lane>\n  </edge>\n  <edge id="-980451721#0" from="2506089309" to="1123399676" priority="10" type="highway.tertiary" shape="1034.52,1642.36 1011.08,1664.51 995.81,1681.63">\n    <lane id="-980451721#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="43.97" shape="1030.45,1644.01 1009.93,1663.39 999.45,1675.14">\n      <param key="origId" value="980451721" />\n    </lane>\n  </edge>\n  <edge id="-980451721#1" from="1107926977" to="2506089309" priority="10" type="highway.tertiary">\n    <lane id="-980451721#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.19" shape="1064.53,1622.63 1039.45,1637.56">\n      <param key="origId" value="980451721" />\n    </lane>\n  </edge>\n  <edge id="100401250#0" from="1160612974" to="8976428985" name="ถนนเทพสงเคราะห์" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="100401250#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="3.05" shape="894.45,688.60 894.90,685.59">\n      <param key="origId" value="100401250" />\n    </lane>\n  </edge>\n  <edge id="100401250#1" from="8976428985" to="1160612969" name="ถนนเทพสงเคราะห์" priority="10" type="highway.tertiary" spreadType="center" shape="896.05,678.00 900.21,639.65 902.44,617.43">\n    <lane id="100401250#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="51.84" shape="896.25,676.13 900.21,639.65 901.72,624.58">\n      <param key="origId" value="100401250" />\n    </lane>\n  </edge>\n  <edge id="100401320" from="1110416469" to="1123399492" priority="10" type="highway.tertiary">\n    <lane id="100401320_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="85.60" shape="751.98,1670.57 773.57,1753.40">\n      <param key="origId" value="100401320" />\n    </lane>\n  </edge>\n  <edge id="101706795#0" from="1196113662" to="1966412828" name="ถนนปรีดารมย์" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="101706795#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="62.03" shape="366.26,385.36 427.69,393.92">\n      <param key="origId" value="101706795" />\n    </lane>\n  </edge>\n  <edge id="101706795#1" from="1966412828" to="1174378788" name="ถนนปรีดารมย์" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="101706795#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="53.65" shape="444.57,396.07 497.86,402.26">\n      <param key="origId" value="101706795" />\n    </lane>\n  </edge>\n  <edge id="101939921" from="1160612969" to="1431715406" name="ถนนชีอุทิศ" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="101939921_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="82.73" shape="895.42,615.06 813.18,606.14">\n      <param key="origId" value="101939921" />\n    </lane>\n    <lane id="101939921_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="82.73" shape="895.08,618.24 812.83,609.32">\n      <param key="origId" value="101939921" />\n    </lane>\n  </edge>\n  <edge id="1168168079" from="4900465363" to="4900465364" name="Soi 1/1 Sophasarnrungsen" priority="3" type="highway.residential" spreadType="center">\n    <lane id="1168168079_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="55.83" shape="559.01,1225.91 503.42,1220.79">\n      <param key="origId" value="1168168079" />\n    </lane>\n  </edge>\n  <edge id="1204844051" from="4900464584" to="1115585385" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary">\n    <lane id="1204844051_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.13" shape="1046.45,978.33 1035.39,977.05">\n      <param key="origId" value="1204844051" />\n    </lane>\n  </edge>\n  <edge id="1218225518#0" from="cluster_11292971032_2406954578_2406954579_2406954580" to="2505176175" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1550.56,1167.43 1533.23,1062.66">\n    <lane id="1218225518#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="92.99" shape="1551.21,1161.57 1536.04,1069.82">\n      <param key="origId" value="1218225518" />\n    </lane>\n    <lane id="1218225518#0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="92.99" shape="1548.06,1162.09 1532.88,1070.35">\n      <param key="origId" value="1218225518" />\n    </lane>\n    <param key="origFrom" value="2406954578" />\n  </edge>\n  <edge id="1218225518#1" from="2505176175" to="cluster_2505176169_856214432" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1533.23,1062.66 1521.92,993.40">\n    <lane id="1218225518#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="58.31" shape="1533.69,1055.57 1524.30,998.04">\n      <param key="origId" value="1218225518" />\n    </lane>\n    <lane id="1218225518#1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="58.31" shape="1530.54,1056.09 1521.13,998.52">\n      <param key="origId" value="1218225518" />\n    </lane>\n    <param key="origTo" value="2505176169" />\n  </edge>\n  <edge id="1218225522#0" from="cluster_849403384_849403500_8976428987_8976428988" to="796747634" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1470.39,749.50 1484.48,836.06 1495.25,902.19">\n    <lane id="1218225522#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="137.96" shape="1471.85,758.49 1484.48,836.06 1494.02,894.65">\n      <param key="origId" value="1218225522" />\n    </lane>\n    <param key="origFrom" value="849403384" />\n  </edge>\n  <edge id="1218225524#1" from="cluster_11292971032_2406954578_2406954579_2406954580" to="856214571" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1540.50,1181.18 1544.12,1202.38">\n    <lane id="1218225524#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="5.24" shape="1541.49,1187.00 1542.38,1192.16">\n      <param key="origId" value="1218225524" />\n    </lane>\n    <param key="origFrom" value="11292971032" />\n  </edge>\n  <edge id="1218747873" from="1639088658" to="2505176175" priority="6" type="highway.secondary_link" shape="1548.26,1061.79 1545.68,1061.94 1535.13,1062.51 1533.23,1062.66">\n    <lane id="1218747873_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="1.62" shape="1542.20,1060.53 1540.58,1060.61">\n      <param key="origId" value="1218747873 243314101 1218747872" />\n    </lane>\n  </edge>\n  <edge id="1218747874" from="cluster_11292971032_2406954578_2406954579_2406954580" to="11292971036" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" spreadType="center" shape="1552.58,1179.46 1557.16,1178.63 1563.31,1177.50">\n    <lane id="1218747874_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.33" length="3.37" shape="1559.84,1178.14 1563.16,1177.53">\n      <param key="origId" value="1218747874 1218747875" />\n    </lane>\n    <param key="origFrom" value="2406954580" />\n  </edge>\n  <edge id="122540020#0" from="1369020950" to="14165547215" name="ถนนเฉลียวอุทิศ" priority="3" type="highway.residential" shape="1385.88,1192.43 1383.56,1265.10 1385.27,1274.98">\n    <lane id="122540020#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="64.64" shape="1383.94,1202.94 1381.96,1265.21 1382.35,1267.51">\n      <param key="origId" value="122540020" />\n    </lane>\n  </edge>\n  <edge id="122540020#1" from="14165547215" to="1369020943" name="ถนนเฉลียวอุทิศ" priority="3" type="highway.residential" shape="1385.27,1274.98 1392.32,1315.41 1393.78,1346.12">\n    <lane id="122540020#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="57.64" shape="1384.84,1281.81 1390.73,1315.59 1391.84,1338.91">\n      <param key="origId" value="122540020" />\n    </lane>\n  </edge>\n  <edge id="122540020#3" from="1369020943" to="2506088023" name="ถนนเฉลียวอุทิศ" priority="3" type="highway.residential">\n    <lane id="122540020#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="27.11" shape="1392.76,1353.44 1394.90,1380.46">\n      <param key="origId" value="122540020" />\n    </lane>\n  </edge>\n  <edge id="122540020#4" from="2506088023" to="2506088274" name="ถนนเฉลียวอุทิศ" priority="3" type="highway.residential">\n    <lane id="122540020#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="64.17" shape="1395.93,1394.73 1400.08,1458.76">\n      <param key="origId" value="122540020" />\n    </lane>\n  </edge>\n  <edge id="122540020#5" from="2506088274" to="2506088528" name="ถนนเฉลียวอุทิศ" priority="3" type="highway.residential">\n    <lane id="122540020#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.23" shape="1400.83,1473.03 1402.44,1512.22">\n      <param key="origId" value="122540020" />\n    </lane>\n  </edge>\n  <edge id="122540020#6" from="2506088528" to="1369020942" name="ถนนเฉลียวอุทิศ" priority="3" type="highway.residential">\n    <lane id="122540020#6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.02" shape="1403.86,1527.06 1405.33,1536.97">\n      <param key="origId" value="122540020" />\n    </lane>\n  </edge>\n  <edge id="122540020#7" from="1369020942" to="1369020939" name="ถนนเฉลียวอุทิศ" priority="3" type="highway.residential">\n    <lane id="122540020#7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.38" shape="1404.47,1548.19 1394.25,1577.86">\n      <param key="origId" value="122540020" />\n    </lane>\n  </edge>\n  <edge id="1227721991#0" from="cluster_2505176169_856214432" to="cluster_11292971032_2406954578_2406954579_2406954580" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1510.05,993.17 1519.48,1055.14 1520.82,1064.00 1533.72,1138.75 1538.72,1169.24">\n    <lane id="1227721991#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="165.91" shape="1511.10,1000.05 1519.48,1055.14 1520.82,1064.00 1533.72,1138.75 1537.83,1163.78">\n      <param key="origId" value="1227721991" />\n    </lane>\n    <param key="origFrom" value="856214432" />\n    <param key="origTo" value="2406954579" />\n  </edge>\n  <edge id="1257722736#0" from="2506088243" to="cluster_1123399715_3376084911_3376091818" priority="3" type="highway.residential" spreadType="center" shape="775.81,1430.60 788.75,1430.52 798.57,1432.46 805.40,1437.29 808.77,1440.78 813.32,1447.20 820.99,1461.08 826.55,1469.37 840.03,1486.03 851.11,1497.68 864.16,1506.37 873.88,1511.62 882.22,1515.25 892.31,1516.95 904.16,1514.25 908.36,1511.86 913.11,1508.38 918.67,1503.30 925.21,1496.93 933.94,1485.41 936.82,1480.35 938.32,1476.79 938.94,1473.23 938.61,1459.19 939.86,1439.42 940.72,1434.42 942.45,1429.42 944.42,1424.66 947.11,1419.90 953.80,1410.64 957.07,1407.53 960.91,1404.41 964.42,1402.41 968.16,1400.76 976.82,1397.99 986.21,1395.87 993.87,1394.80 1003.97,1395.38">\n    <lane id="1257722736#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="327.41" shape="781.77,1430.56 788.75,1430.52 798.57,1432.46 805.40,1437.29 808.77,1440.78 813.32,1447.20 820.99,1461.08 826.55,1469.37 840.03,1486.03 851.11,1497.68 864.16,1506.37 873.88,1511.62 882.22,1515.25 892.31,1516.95 904.16,1514.25 908.36,1511.86 913.11,1508.38 918.67,1503.30 925.21,1496.93 933.94,1485.41 936.82,1480.35 938.32,1476.79 938.94,1473.23 938.61,1459.19 939.86,1439.42 940.72,1434.42 942.45,1429.42 944.42,1424.66 947.11,1419.90 953.80,1410.64 957.07,1407.53 960.91,1404.41 964.42,1402.41 968.16,1400.76 976.82,1397.99 986.21,1395.87 993.87,1394.80 996.21,1394.93">\n      <param key="origId" value="1257722736" />\n    </lane>\n    <param key="origTo" value="1123399715" />\n  </edge>\n  <edge id="1257722737" from="2506088243" to="2506088024" priority="3" type="highway.residential" shape="775.81,1430.60 775.89,1422.93 775.46,1404.60 775.30,1396.15 775.02,1388.94">\n    <lane id="1257722737_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="8.33" length="29.70" shape="777.46,1426.23 777.49,1422.92 777.06,1404.56 776.91,1396.53">\n      <param key="origId" value="1257722737 330615766 330615760" />\n    </lane>\n  </edge>\n  <edge id="1257722738#0" from="2506088243" to="1123399502" priority="3" type="highway.residential" spreadType="center" shape="775.81,1430.60 758.43,1437.16 704.60,1490.00 688.12,1499.56 687.43,1499.90 678.94,1504.01 650.77,1510.80 612.21,1516.84 594.09,1519.81 565.91,1520.01">\n    <lane id="1257722738#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="224.62" shape="770.62,1432.56 758.43,1437.16 704.60,1490.00 688.12,1499.56 687.43,1499.90 678.94,1504.01 650.77,1510.80 612.21,1516.84 594.09,1519.81 573.56,1519.96">\n      <param key="origId" value="1257722738" />\n    </lane>\n  </edge>\n  <edge id="1257722739#2" from="1123399863" to="1123399591" priority="10" type="highway.tertiary" spreadType="center" shape="715.76,1754.29 646.60,1758.26 616.02,1743.98 593.07,1734.05 572.08,1715.62">\n    <lane id="1257722739#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="139.67" shape="708.28,1754.72 646.60,1758.26 616.02,1743.98 593.07,1734.05 578.69,1721.42">\n      <param key="origId" value="1257722739" />\n    </lane>\n  </edge>\n  <edge id="1270957860#0" from="796747634" to="cluster_2505176169_856214432" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1495.25,902.19 1500.37,933.70 1510.05,993.17">\n    <lane id="1270957860#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="77.67" shape="1496.36,909.02 1500.37,933.70 1508.83,985.69">\n      <param key="origId" value="1270957860" />\n    </lane>\n    <param key="origTo" value="856214432" />\n  </edge>\n  <edge id="1270957861" from="2507400639" to="cluster_849403384_849403500_8976428987_8976428988" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1511.13,927.36 1505.80,894.77 1484.92,766.97 1482.20,750.31">\n    <lane id="1270957861_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="166.05" shape="1511.59,920.26 1507.38,894.51 1486.50,766.71 1484.81,756.38">\n      <param key="origId" value="1270957861" />\n    </lane>\n    <lane id="1270957861_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="166.05" shape="1508.43,920.77 1504.22,895.03 1483.34,767.23 1481.65,756.89">\n      <param key="origId" value="1270957861" />\n    </lane>\n    <param key="origTo" value="849403500" />\n  </edge>\n  <edge id="1270957862" from="cluster_2505176169_856214432" to="2507400639" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1521.92,993.40 1511.13,927.36">\n    <lane id="1270957862_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="49.71" shape="1521.93,983.55 1513.92,934.49">\n      <param key="origId" value="1270957862" />\n    </lane>\n    <lane id="1270957862_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="49.71" shape="1518.77,984.06 1510.76,935.01">\n      <param key="origId" value="1270957862" />\n    </lane>\n    <param key="origFrom" value="2505176169" />\n  </edge>\n  <edge id="1270957863" from="cluster_2505176169_856214432" to="856214524" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="1510.05,993.17 1480.29,992.16 1336.71,985.69">\n    <lane id="1270957863_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="159.95" shape="1503.84,991.36 1480.35,990.56 1344.05,984.42">\n      <param key="origId" value="1270957863" />\n    </lane>\n    <param key="origFrom" value="856214432" />\n  </edge>\n  <edge id="129827552#0" from="1431715395" to="1680666546" priority="3" type="highway.residential" shape="514.90,371.98 546.67,381.91 562.22,386.77">\n    <lane id="129827552#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="35.02" shape="521.38,375.68 546.19,383.44 554.80,386.13">\n      <param key="origId" value="129827552" />\n    </lane>\n  </edge>\n  <edge id="129827552#1" from="1680666546" to="1122072674" priority="3" type="highway.residential">\n    <lane id="129827552#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.33" shape="568.54,390.43 581.26,394.41">\n      <param key="origId" value="129827552" />\n    </lane>\n  </edge>\n  <edge id="129827554#0" from="1110416726" to="1680666468" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary" shape="796.31,306.04 772.69,304.69 743.50,302.79 722.47,312.70 686.80,341.59 659.35,366.88 654.95,370.55">\n    <lane id="129827554#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="150.77" shape="788.45,303.99 772.79,303.09 743.19,301.17 721.61,311.34 685.75,340.38 659.70,364.38">\n      <param key="origId" value="129827554" />\n    </lane>\n  </edge>\n  <edge id="129827554#1" from="1680666468" to="cluster_1107773211_1110416556" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary" shape="654.95,370.55 636.28,386.15 627.89,393.90 622.57,400.82 622.96,408.83 626.12,414.63 637.72,421.66 688.34,452.63 708.30,469.47 735.81,491.86 746.90,503.59 755.95,515.14">\n    <lane id="129827554#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="209.52" shape="648.67,373.71 635.22,384.95 626.70,392.82 620.94,400.31 621.38,409.27 624.92,415.77 636.89,423.03 687.40,453.93 707.28,470.70 734.72,493.04 745.69,504.64 748.15,507.79">\n      <param key="origId" value="129827554" />\n    </lane>\n    <param key="origTo" value="1110416556" />\n  </edge>\n  <edge id="129827558" from="1122072673" to="1431715359" name="ถนนเสน่หานุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="129827558_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="78.65" shape="543.34,293.66 568.90,219.28">\n      <param key="origId" value="129827558" />\n    </lane>\n  </edge>\n  <edge id="129827561#0" from="1431715415" to="14188764645" name="ถนนหอมนวล" priority="10" type="highway.tertiary">\n    <lane id="129827561#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.22" shape="811.68,530.78 807.82,570.82">\n      <param key="origId" value="129827561" />\n    </lane>\n  </edge>\n  <edge id="129827561#1" from="14188764645" to="1431715406" name="ถนนหอมนวล" priority="10" type="highway.tertiary">\n    <lane id="129827561#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.64" shape="806.44,585.15 805.02,599.72">\n      <param key="origId" value="129827561" />\n    </lane>\n  </edge>\n  <edge id="129827562" from="1431715380" to="1431715421" name="ถนนอำนวยวิทย์" priority="10" type="highway.tertiary" shape="530.77,114.96 511.42,218.11 507.54,245.23 501.61,292.37">\n    <lane id="129827562_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="174.34" shape="529.20,114.67 509.84,217.85 505.95,245.02 500.72,286.60">\n      <param key="origId" value="129827562" />\n    </lane>\n  </edge>\n  <edge id="129842388#1" from="1431867108" to="1680666463" name="ถนนราษฎร์ยินดี ซอย 9" priority="10" type="highway.tertiary">\n    <lane id="129842388#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="46.41" shape="1442.18,362.85 1487.27,373.81">\n      <param key="origId" value="129842388" />\n    </lane>\n  </edge>\n  <edge id="129842388#2" from="1680666463" to="1431867166" name="ถนนราษฎร์ยินดี ซอย 9" priority="10" type="highway.tertiary">\n    <lane id="129842388#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="45.17" shape="1502.21,377.27 1546.31,387.02">\n      <param key="origId" value="129842388" />\n    </lane>\n  </edge>\n  <edge id="129842391" from="13362936997" to="1431867112" name="ถนนพัฒโนอุทิศ" priority="3" type="highway.residential">\n    <lane id="129842391_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="218.25" shape="1440.01,347.52 1549.35,158.63">\n      <param key="origId" value="129842391" />\n    </lane>\n  </edge>\n  <edge id="129842397#0" from="1431867141" to="1431867103" name="Thanon Klongren 2 Soi 7" priority="3" type="highway.residential">\n    <lane id="129842397#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="133.72" shape="1843.27,512.88 1723.98,452.47">\n      <param key="origId" value="129842397" />\n    </lane>\n  </edge>\n  <edge id="129842397#1" from="1431867103" to="8642733850" name="Thanon Klongren 2 Soi 7" priority="3" type="highway.residential">\n    <lane id="129842397#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="63.85" shape="1710.97,446.30 1652.66,420.29">\n      <param key="origId" value="129842397" />\n    </lane>\n  </edge>\n  <edge id="129842397#2" from="8642733850" to="1431867150" name="Thanon Klongren 2 Soi 7" priority="3" type="highway.residential" shape="1645.32,418.77 1630.95,412.36 1629.62,411.86 1628.63,411.16 1627.85,410.40 1627.26,409.47 1626.80,408.70 1626.46,407.86 1626.23,406.69 1626.18,405.38 1668.15,304.99">\n    <lane id="129842397#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="117.91" shape="1639.51,414.42 1631.59,410.90 1630.38,410.43 1629.66,409.93 1629.10,409.38 1628.62,408.63 1628.24,407.99 1628.00,407.40 1627.82,406.50 1627.79,405.67 1667.24,311.32">\n      <param key="origId" value="129842397" />\n    </lane>\n  </edge>\n  <edge id="129842397#4" from="1431867150" to="1431867100" name="Thanon Klongren 2 Soi 7" priority="3" type="highway.residential">\n    <lane id="129842397#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.84" shape="1663.68,300.41 1633.90,282.32">\n      <param key="origId" value="129842397" />\n    </lane>\n  </edge>\n  <edge id="1316153439" from="2506087563" to="6634655051" priority="3" type="highway.residential">\n    <lane id="1316153439_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="1.35" shape="427.64,1252.30 428.98,1252.44">\n      <param key="origId" value="1316153439" />\n    </lane>\n  </edge>\n  <edge id="1334274432" from="12344163778" to="12344163777" priority="3" type="highway.residential">\n    <lane id="1334274432_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="52.92" shape="531.80,1043.47 583.65,1054.07">\n      <param key="origId" value="1334274432" />\n    </lane>\n  </edge>\n  <edge id="1335076429" from="2506088019" to="12351390200" priority="3" type="highway.residential" shape="1186.69,1386.25 1189.35,1393.17 1195.69,1408.63">\n    <lane id="1335076429_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="24.03" shape="1185.20,1386.82 1187.86,1393.76 1194.16,1409.12">\n      <param key="origId" value="1335076429 243188101" />\n    </lane>\n  </edge>\n  <edge id="1365892748#0" from="1110416444" to="2506087432" name="Thanon Pracharak" priority="3" type="highway.residential" shape="644.04,1123.85 642.75,1182.15 641.81,1228.22">\n    <lane id="1365892748#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="89.63" shape="642.27,1131.37 641.15,1182.12 640.36,1220.98">\n      <param key="origId" value="1365892748" />\n    </lane>\n  </edge>\n  <edge id="1365892989" from="1431715392" to="1431715366" priority="3" type="highway.residential" spreadType="center" shape="770.27,603.98 769.48,661.36 769.07,680.94">\n    <lane id="1365892989_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="64.60" shape="770.17,610.99 769.48,661.36 769.18,675.59">\n      <param key="origId" value="1365892989" />\n    </lane>\n  </edge>\n  <edge id="1365892991" from="1110416568" to="1110416410" priority="3" type="highway.residential" spreadType="center">\n    <lane id="1365892991_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="60.57" shape="741.50,669.38 744.43,608.88">\n      <param key="origId" value="1365892991" />\n    </lane>\n  </edge>\n  <edge id="1365935945#0" from="856214538" to="4900465166" priority="3" type="highway.residential">\n    <lane id="1365935945#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="38.68" shape="898.76,964.66 900.96,926.05">\n      <param key="origId" value="1365935945" />\n    </lane>\n  </edge>\n  <edge id="1365935945#1" from="4900465166" to="12649445299" priority="3" type="highway.residential">\n    <lane id="1365935945#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="27.13" shape="901.86,911.65 903.67,884.58">\n      <param key="origId" value="1365935945" />\n    </lane>\n  </edge>\n  <edge id="1374050353" from="cluster_1107773211_1110416556" to="1431715409" name="ถนนกิมประดิษฐ์" priority="10" type="highway.tertiary" shape="755.95,515.14 750.02,515.15 734.81,514.54 732.80,514.38">\n    <lane id="1374050353_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="6.92" shape="745.37,513.36 738.46,513.08">\n      <param key="origId" value="1374050353 1374050352 706256379" />\n    </lane>\n    <param key="origFrom" value="1110416556" />\n  </edge>\n  <edge id="142395239#0" from="1431867150" to="1558314095" priority="3" type="highway.residential">\n    <lane id="142395239#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.52" shape="1673.78,307.45 1690.12,309.89">\n      <param key="origId" value="142395239" />\n    </lane>\n  </edge>\n  <edge id="142395239#1" from="1558314095" to="4900465031" priority="3" type="highway.residential" shape="1697.89,309.43 1699.57,302.99 1705.02,261.15 1748.43,261.40 1742.33,329.88 1742.39,331.89 1742.99,333.67 1745.44,336.33 1750.84,340.22 1729.59,408.75">\n    <lane id="142395239#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="226.47" shape="1701.16,303.18 1706.42,262.76 1746.68,262.99 1740.73,329.83 1740.80,332.18 1741.58,334.50 1744.37,337.53 1748.97,340.85 1730.09,401.75">\n      <param key="origId" value="142395239 498671695 706087699" />\n    </lane>\n  </edge>\n  <edge id="142453602#0" from="1115585385" to="1115585909" priority="10" type="highway.tertiary">\n    <lane id="142453602#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="44.97" shape="1027.64,969.13 1011.55,927.13">\n      <param key="origId" value="142453602" />\n    </lane>\n  </edge>\n  <edge id="142453602#1" from="1115585909" to="1115585463" priority="10" type="highway.tertiary" shape="1007.86,921.98 1007.38,899.61 998.04,876.32">\n    <lane id="142453602#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="35.00" shape="1009.30,914.58 1008.97,899.28 1001.64,881.00">\n      <param key="origId" value="142453602" />\n    </lane>\n  </edge>\n  <edge id="142453602#2" from="1115585463" to="1115585416" priority="10" type="highway.tertiary" shape="998.04,876.32 972.02,854.49 957.51,845.29">\n    <lane id="142453602#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="37.62" shape="994.38,871.16 972.97,853.20 964.80,848.02">\n      <param key="origId" value="142453602" />\n    </lane>\n  </edge>\n  <edge id="142455590" from="1559016909" to="cluster_1559016910_4900465210" name="ถนนฉัยยากุลอุทิศ ซอย 2" priority="3" type="highway.residential" shape="776.48,421.04 870.29,430.73">\n    <lane id="142455590_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="79.52" shape="783.52,423.38 862.61,431.55">\n      <param key="origId" value="142455590" />\n    </lane>\n    <param key="origTo" value="1559016910" />\n  </edge>\n  <edge id="142455592#0" from="1431867166" to="1559016911" name="Rajyindee 9 Road" priority="3" type="highway.residential" shape="1554.36,387.16 1549.79,397.60 1532.91,436.17">\n    <lane id="142455592#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="45.60" shape="1549.73,393.75 1548.32,396.96 1531.44,435.53">\n      <param key="origId" value="142455592" />\n    </lane>\n  </edge>\n  <edge id="142455595" from="1559016904" to="1559016905" name="ถนนฉัยยากุลอุทิศ ซอย 3" priority="10" type="highway.tertiary">\n    <lane id="142455595_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="79.37" shape="788.66,378.92 867.63,386.93">\n      <param key="origId" value="142455595" />\n    </lane>\n  </edge>\n  <edge id="142455599" from="1559016916" to="1559016918" name="ถนนฉัยยากุลอุทิศ ซอย 1" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="142455599_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="78.56" shape="780.39,465.73 858.32,475.74">\n      <param key="origId" value="142455599" />\n    </lane>\n  </edge>\n  <edge id="142455600#0" from="1559016918" to="cluster_1559016910_4900465210" priority="10" type="highway.tertiary" shape="865.46,476.66 870.29,430.73">\n    <lane id="142455600#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.84" shape="867.80,469.73 871.13,438.06">\n      <param key="origId" value="142455600" />\n    </lane>\n    <param key="origTo" value="1559016910" />\n  </edge>\n  <edge id="142455600#2" from="cluster_1559016910_4900465210" to="1559016905" priority="10" type="highway.tertiary" shape="872.26,410.02 874.99,386.07">\n    <lane id="142455600#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.61" shape="874.67,402.99 875.76,393.44">\n      <param key="origId" value="142455600" />\n    </lane>\n    <param key="origFrom" value="4900465210" />\n  </edge>\n  <edge id="142455600#3" from="1559016905" to="1559016903" priority="10" type="highway.tertiary">\n    <lane id="142455600#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.50" shape="877.33,379.09 880.41,349.75">\n      <param key="origId" value="142455600" />\n    </lane>\n  </edge>\n  <edge id="142455600#4" from="1559016903" to="1110416726" name="ถนนละม้ายสงเคราะห์" priority="10" type="highway.tertiary" shape="879.58,342.38 885.61,281.61 883.27,271.94 863.26,283.28 848.07,290.44 825.48,301.95 796.31,306.04">\n    <lane id="142455600#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="151.80" shape="881.88,335.39 887.23,281.50 884.33,269.50 862.52,281.86 847.37,289.00 824.99,300.40 806.80,302.95">\n      <param key="origId" value="142455600 142455604" />\n    </lane>\n  </edge>\n  <edge id="142456544" from="1559029798" to="1559029797" name="ตรอกสุคนธหงส์" priority="3" type="highway.residential" spreadType="center">\n    <lane id="142456544_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="69.94" shape="452.19,749.42 382.63,742.11">\n      <param key="origId" value="142456544" />\n    </lane>\n  </edge>\n  <edge id="1447303746" from="13279030938" to="13279030940" priority="3" type="highway.residential" shape="1307.02,986.05 1309.04,948.88 1293.26,948.79 1278.57,948.71">\n    <lane id="1447303746_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="63.64" shape="1309.02,978.73 1310.73,947.29 1293.27,947.19 1278.58,947.11">\n      <param key="origId" value="1447303746" />\n    </lane>\n  </edge>\n  <edge id="1454593202#0" from="13339819885" to="14188764647" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary">\n    <lane id="1454593202#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.43" shape="898.12,530.81 867.85,527.69">\n      <param key="origId" value="1454593202" />\n    </lane>\n  </edge>\n  <edge id="1454593202#1" from="14188764647" to="1431715415" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary">\n    <lane id="1454593202#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.38" shape="853.52,526.21 821.32,522.90">\n      <param key="origId" value="1454593202" />\n    </lane>\n  </edge>\n  <edge id="1454593202#2" from="1431715415" to="cluster_1107773211_1110416556" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary" shape="813.97,523.75 769.62,517.28">\n    <lane id="1454593202#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.81" shape="807.23,521.15 776.74,516.70">\n      <param key="origId" value="1454593202" />\n    </lane>\n    <param key="origTo" value="1107773211" />\n  </edge>\n  <edge id="1454593203#1" from="13339844003" to="13339819888" priority="10" type="highway.tertiary" spreadType="center" shape="915.79,534.39 915.19,532.43 913.99,530.76 912.32,529.55">\n    <lane id="1454593203#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.20" shape="912.10,529.39 911.94,529.27">\n      <param key="origId" value="1454593203" />\n    </lane>\n  </edge>\n  <edge id="1454593203#2" from="13339819888" to="13339819885" priority="10" type="highway.tertiary" spreadType="center" shape="912.32,529.55 910.34,528.95 908.21,529.02 906.22,529.77 904.59,531.15 903.50,532.97">\n    <lane id="1454593203#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.20" shape="906.70,529.36 906.55,529.49">\n      <param key="origId" value="1454593203" />\n    </lane>\n  </edge>\n  <edge id="1454593203#3" from="13339819885" to="13339819893" priority="10" type="highway.tertiary" spreadType="center" shape="903.50,532.97 903.09,535.06 903.39,537.16 904.38,539.05 905.93,540.50 907.88,541.37 909.94,541.55">\n    <lane id="1454593203#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.20" shape="904.38,539.05 904.53,539.19">\n      <param key="origId" value="1454593203" />\n    </lane>\n  </edge>\n  <edge id="1454593203#4" from="13339819893" to="13339844003" priority="10" type="highway.tertiary" spreadType="center" shape="909.94,541.55 911.95,541.07 913.70,539.97 915.01,538.38 915.73,536.45 915.79,534.39">\n    <lane id="1454593203#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.20" shape="914.61,539.45 914.68,539.26">\n      <param key="origId" value="1454593203" />\n    </lane>\n  </edge>\n  <edge id="1457057266#0" from="1431867166" to="1431867100" name="Rajyindee 9 Road" priority="3" type="highway.residential" shape="1554.36,387.16 1569.64,368.67 1595.26,337.69 1614.61,312.84 1630.10,281.89">\n    <lane id="1457057266#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="121.23" shape="1559.39,383.59 1570.87,369.69 1596.51,338.69 1615.97,313.70 1629.98,285.71">\n      <param key="origId" value="1457057266" />\n    </lane>\n  </edge>\n  <edge id="1457057267" from="1431867108" to="13362936997" name="Thanon Pattano Uthit" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="1457057267_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.20" shape="1435.01,355.33 1434.98,355.13">\n      <param key="origId" value="1457057267" />\n    </lane>\n  </edge>\n  <edge id="148941242#1" from="1110416726" to="1431715372" name="ถนนฉัยยากุลอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="148941242#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.07" shape="792.64,313.29 789.16,325.88">\n      <param key="origId" value="148941242" />\n    </lane>\n  </edge>\n  <edge id="148941242#2" from="1431715372" to="1559016904" name="ถนนฉัยยากุลอุทิศ" priority="10" type="highway.tertiary" shape="788.61,333.85 784.78,353.67 782.15,371.97 781.62,376.60">\n    <lane id="148941242#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.31" shape="785.73,340.32 783.20,353.40 780.93,369.23">\n      <param key="origId" value="148941242" />\n    </lane>\n  </edge>\n  <edge id="148941242#3" from="1559016904" to="1559016909" name="ถนนฉัยยากุลอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="148941242#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.32" shape="779.21,383.52 775.72,413.65">\n      <param key="origId" value="148941242" />\n    </lane>\n  </edge>\n  <edge id="148941242#4" from="1559016909" to="1559016916" name="ถนนฉัยยากุลอุทิศ" priority="10" type="highway.tertiary" shape="776.48,421.04 775.55,429.12 773.15,464.80">\n    <lane id="148941242#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.40" shape="774.07,427.97 773.96,428.97 771.92,459.30">\n      <param key="origId" value="148941242" />\n    </lane>\n  </edge>\n  <edge id="148941242#5" from="1559016916" to="cluster_1107773211_1110416556" name="ถนนฉัยยากุลอุทิศ" priority="10" type="highway.tertiary" shape="773.15,464.80 769.62,517.28">\n    <lane id="148941242#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.84" shape="771.16,470.48 768.49,510.23">\n      <param key="origId" value="148941242" />\n    </lane>\n    <param key="origTo" value="1107773211" />\n  </edge>\n  <edge id="148941245" from="1110416683" to="1110416726" name="ถนนฉัยยากุลอุทิศ" priority="10" type="highway.tertiary" shape="814.75,279.64 799.75,300.43 796.31,306.04">\n    <lane id="148941245_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="21.53" shape="813.45,278.70 800.86,296.16">\n      <param key="origId" value="148941245 148941242" />\n    </lane>\n  </edge>\n  <edge id="151072451#0" from="1639088623" to="1639088636" name="ถนนราษฎร์ยินดี ซอย 7" priority="10" type="highway.tertiary" shape="1531.15,963.11 1543.17,961.10 1692.60,993.21 1735.79,1002.49 1742.34,1003.89 1752.45,1006.11 1766.55,1009.20 1795.52,1018.16 1873.92,1015.70 1913.65,1010.39 1964.80,1000.49">\n    <lane id="151072451#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="436.04" shape="1536.97,963.76 1543.13,962.73 1692.26,994.77 1735.45,1004.05 1742.00,1005.45 1752.11,1007.67 1766.14,1010.75 1795.30,1019.77 1874.05,1017.30 1913.91,1011.97 1965.10,1002.06">\n      <param key="origId" value="151072451" />\n    </lane>\n  </edge>\n  <edge id="151072453#0" from="809537820" to="8523328986" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="151072453#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="52.05" shape="1565.21,1159.68 1556.21,1108.42">\n      <param key="origId" value="151072453" />\n    </lane>\n  </edge>\n  <edge id="151072453#1" from="8523328986" to="1639088658" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="151072453#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="24.99" shape="1553.75,1094.18 1549.57,1069.53">\n      <param key="origId" value="151072453" />\n    </lane>\n  </edge>\n  <edge id="151072453#2" from="1639088658" to="1639088623" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="151072453#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="85.54" shape="1547.00,1054.50 1532.38,970.21">\n      <param key="origId" value="151072453" />\n    </lane>\n  </edge>\n  <edge id="151072453#3" from="1639088623" to="2507400638" priority="10" type="highway.tertiary" spreadType="center" shape="1531.15,963.11 1530.38,958.22 1526.54,933.61 1525.44,926.27">\n    <lane id="151072453#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="21.17" shape="1530.03,955.98 1526.77,935.06">\n      <param key="origId" value="151072453 969757856 969757855" />\n    </lane>\n  </edge>\n  <edge id="154097280" from="1115585403" to="1966404349" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="1066.17,1070.17 1044.91,1069.56 1027.19,1067.73">\n    <lane id="154097280_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.59" shape="1062.73,1068.47 1045.02,1067.96 1033.21,1066.74">\n      <param key="origId" value="154097280" />\n    </lane>\n  </edge>\n  <edge id="1546894905" from="14069947700" to="14069967501" priority="3" type="highway.residential" shape="572.02,661.42 570.57,668.33 564.06,698.75">\n    <lane id="1546894905_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.34" shape="568.85,668.74 562.50,698.42">\n      <param key="origId" value="1546894905" />\n    </lane>\n  </edge>\n  <edge id="1546896992" from="14069981565" to="14069981568" priority="3" type="highway.residential" shape="459.32,738.87 473.16,739.51 500.47,740.77 503.03,740.89 510.01,735.67 514.36,707.90">\n    <lane id="1546896992_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="79.58" shape="462.69,740.63 473.09,741.11 500.40,742.37 503.53,742.52 511.49,736.56 515.94,708.15">\n      <param key="origId" value="1546896992" />\n    </lane>\n  </edge>\n  <edge id="1546907364" from="14070071629" to="14070071630" priority="3" type="highway.residential">\n    <lane id="1546907364_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="63.63" shape="596.13,288.44 656.61,308.22">\n      <param key="origId" value="1546907364" />\n    </lane>\n  </edge>\n  <edge id="154707523#0" from="cluster_2052547073_2506085499" to="2052547075" priority="10" type="highway.tertiary" spreadType="center" shape="694.59,670.76 698.84,597.67">\n    <lane id="154707523#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="58.74" shape="695.01,663.54 698.42,604.90">\n      <param key="origId" value="154707523" />\n    </lane>\n    <param key="origFrom" value="2052547073" />\n  </edge>\n  <edge id="154707523#1" from="2052547075" to="cluster_1122072658_14138648129" priority="10" type="highway.tertiary" spreadType="center" shape="698.84,597.67 710.30,512.38">\n    <lane id="154707523#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="71.49" shape="699.81,590.45 709.33,519.60">\n      <param key="origId" value="154707523" />\n    </lane>\n    <param key="origTo" value="14138648129" />\n  </edge>\n  <edge id="1547102487#0" from="2507400593" to="2507400591" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="1547102487#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.69" shape="1497.81,742.89 1494.80,727.49">\n      <param key="origId" value="1547102487" />\n    </lane>\n  </edge>\n  <edge id="1547102487#1" from="2507400591" to="4900465949" priority="10" type="highway.tertiary" spreadType="center" shape="1493.33,719.96 1480.17,645.35 1473.44,600.14 1467.54,560.58 1452.32,458.35">\n    <lane id="1547102487#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="249.64" shape="1492.05,712.68 1480.17,645.35 1473.44,600.14 1467.54,560.58 1453.47,466.05">\n      <param key="origId" value="1547102487 1547102487 1547102487 705855017" />\n    </lane>\n  </edge>\n  <edge id="1552748146" from="2506088518" to="14127615694" priority="3" type="highway.residential">\n    <lane id="1552748146_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="5.47" shape="1148.54,1509.13 1143.07,1509.13">\n      <param key="origId" value="1552748146" />\n    </lane>\n  </edge>\n  <edge id="1552748564#0" from="14127602686" to="4900465706" priority="3" type="highway.residential">\n    <lane id="1552748564#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.29" shape="439.13,1493.74 408.85,1492.91">\n      <param key="origId" value="1552748564" />\n    </lane>\n  </edge>\n  <edge id="1552748564#1" from="4900465706" to="916327912" priority="3" type="highway.residential">\n    <lane id="1552748564#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.68" shape="394.43,1492.52 374.75,1492.00">\n      <param key="origId" value="1552748564" />\n    </lane>\n  </edge>\n  <edge id="1556742612" from="14159239425" to="14159239426" priority="3" type="highway.residential">\n    <lane id="1556742612_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="49.63" shape="1134.46,723.02 1145.35,771.44">\n      <param key="origId" value="1556742612" />\n    </lane>\n  </edge>\n  <edge id="1556742613" from="14159239427" to="14159239429" priority="3" type="highway.residential" shape="1077.61,728.60 1091.68,759.42 1096.62,765.79">\n    <lane id="1556742613_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.34" shape="1079.46,736.51 1090.30,760.25 1095.36,766.77">\n      <param key="origId" value="1556742613" />\n    </lane>\n  </edge>\n  <edge id="155690340" from="1680666686" to="1680666699" name="ซอยละม้ายสงเคราะห์ 1" priority="3" type="highway.residential">\n    <lane id="155690340_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="158.64" shape="931.89,417.03 1090.35,424.47">\n      <param key="origId" value="155690340" />\n    </lane>\n  </edge>\n  <edge id="155690480#0" from="1680667509" to="4900465244" priority="3" type="highway.residential">\n    <lane id="155690480#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="22.78" shape="1341.87,651.17 1364.65,651.04">\n      <param key="origId" value="155690480" />\n    </lane>\n  </edge>\n  <edge id="155690480#1" from="4900465244" to="796747714" priority="3" type="highway.residential">\n    <lane id="155690480#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="68.79" shape="1379.08,650.93 1447.86,650.20">\n      <param key="origId" value="155690480" />\n    </lane>\n  </edge>\n  <edge id="155690513" from="cluster_1558314142_1558314144" to="1680667730" name="Soi 14 Rajyindee Road" priority="3" type="highway.residential" shape="1386.60,710.02 1418.62,710.74 1440.07,711.22 1453.36,711.52 1463.62,711.75 1464.25,711.76">\n    <lane id="155690513_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="64.42" shape="1393.62,711.78 1418.58,712.34 1440.03,712.82 1453.32,713.12 1458.03,713.22">\n      <param key="origId" value="155690513" />\n    </lane>\n    <param key="origFrom" value="1558314144" />\n  </edge>\n  <edge id="155690528" from="1680666789" to="1680667181" priority="3" type="highway.residential">\n    <lane id="155690528_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="80.06" shape="1087.49,461.18 1080.84,540.96">\n      <param key="origId" value="155690528" />\n    </lane>\n  </edge>\n  <edge id="155690734#0" from="1680667083" to="1680667079" priority="3" type="highway.residential">\n    <lane id="155690734#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="36.52" shape="1502.42,525.60 1538.93,524.61">\n      <param key="origId" value="155690734" />\n    </lane>\n  </edge>\n  <edge id="155690734#1" from="1680667079" to="1680667090" priority="3" type="highway.residential">\n    <lane id="155690734#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="36.86" shape="1553.86,524.62 1590.70,525.72">\n      <param key="origId" value="155690734" />\n    </lane>\n  </edge>\n  <edge id="155690741#0" from="1680666468" to="14182422401" name="Thanon Chaiyakuluthit Soi 2" priority="3" type="highway.residential" shape="654.95,370.55 685.55,399.46 697.89,410.52 707.09,411.75 730.07,414.83">\n    <lane id="155690741#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="77.33" shape="659.38,376.94 684.47,400.64 697.19,412.04 706.88,413.34 722.69,415.46">\n      <param key="origId" value="155690741" />\n    </lane>\n  </edge>\n  <edge id="155690741#2" from="14182422401" to="1559016909" name="Thanon Chaiyakuluthit Soi 2" priority="3" type="highway.residential">\n    <lane id="155690741#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.39" shape="736.97,417.37 769.07,421.66">\n      <param key="origId" value="155690741" />\n    </lane>\n  </edge>\n  <edge id="155690766" from="14124372093" to="1680667686" name="ถนนคลองเรียน 2 ซอย 9" priority="10" type="highway.tertiary" shape="1689.38,642.59 1681.35,669.43 1803.64,705.64">\n    <lane id="155690766_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="158.76" shape="1687.85,642.13 1679.36,670.51 1803.19,707.17">\n      <param key="origId" value="155690766" />\n    </lane>\n  </edge>\n  <edge id="155690791#0" from="1680666546" to="14070071629" name="ซอยสเน่หานุสรณ์" priority="10" type="highway.tertiary" shape="562.22,386.77 579.09,328.01 593.64,285.94">\n    <lane id="155690791#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="95.14" shape="565.77,380.22 580.62,328.49 594.12,289.44">\n      <param key="origId" value="155690791" />\n    </lane>\n  </edge>\n  <edge id="155690812#0" from="13351099349" to="1680667164" priority="3" type="highway.residential">\n    <lane id="155690812#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="26.12" shape="1514.83,553.21 1540.85,550.92">\n      <param key="origId" value="155690812" />\n    </lane>\n  </edge>\n  <edge id="155690812#1" from="1680667164" to="1680667158" priority="3" type="highway.residential">\n    <lane id="155690812#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="37.29" shape="1555.10,549.83 1592.31,547.41">\n      <param key="origId" value="155690812" />\n    </lane>\n  </edge>\n  <edge id="155690875" from="1680666683" to="cluster_1680667233_2506085475" priority="3" type="highway.residential" shape="1221.95,408.43 1227.11,503.25 1219.28,562.16">\n    <lane id="155690875_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="146.99" shape="1220.35,408.52 1225.50,503.19 1218.63,554.92">\n      <param key="origId" value="155690875" />\n    </lane>\n    <param key="origTo" value="1680667233" />\n  </edge>\n  <edge id="155690878#0" from="cluster_1559016919_1680666950" to="1680667029" priority="3" type="highway.residential" shape="914.80,503.53 981.09,509.46">\n    <lane id="155690878#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="52.25" shape="921.81,505.76 973.86,510.42">\n      <param key="origId" value="155690878" />\n    </lane>\n    <param key="origFrom" value="1680666950" />\n  </edge>\n  <edge id="155690878#1" from="1680667029" to="1680667034" priority="3" type="highway.residential">\n    <lane id="155690878#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="44.48" shape="988.14,511.78 1032.39,516.26">\n      <param key="origId" value="155690878" />\n    </lane>\n  </edge>\n  <edge id="155690968" from="1680666322" to="1680666294" priority="3" type="highway.residential" shape="935.82,314.77 1032.83,318.20 1034.94,317.65 1036.36,316.57 1037.27,314.84 1038.19,306.58 1039.62,304.98 1041.24,303.89 1043.25,303.54 1170.59,306.94">\n    <lane id="155690968_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="237.39" shape="943.19,316.63 1033.01,319.81 1035.65,319.12 1037.61,317.63 1038.83,315.32 1039.72,307.27 1040.68,306.20 1041.85,305.41 1043.37,305.14 1170.55,308.54">\n      <param key="origId" value="155690968" />\n    </lane>\n  </edge>\n  <edge id="155690981" from="cluster_1110416379_1680666461" to="1680666534" name="Soi 2 Lamaisongkraw" priority="3" type="highway.residential" shape="929.75,368.50 1065.76,376.41 1065.79,387.10 1109.81,386.07">\n    <lane id="155690981_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="184.18" shape="936.69,370.51 1064.16,377.92 1064.19,388.74 1109.85,387.67">\n      <param key="origId" value="155690981" />\n    </lane>\n    <param key="origFrom" value="1680666461" />\n  </edge>\n  <edge id="155691015#0" from="1680666827" to="1680667079" priority="3" type="highway.residential" shape="1545.67,470.73 1543.84,503.33 1546.44,522.80">\n    <lane id="155691015#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="37.58" shape="1543.65,478.09 1542.23,503.39 1543.85,515.52">\n      <param key="origId" value="155691015" />\n    </lane>\n  </edge>\n  <edge id="155691015#1" from="1680667079" to="1680667164" priority="3" type="highway.residential">\n    <lane id="155691015#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.23" shape="1545.23,530.34 1545.80,541.56">\n      <param key="origId" value="155691015" />\n    </lane>\n  </edge>\n  <edge id="155691022#0" from="1680666891" to="4900465071" name="Soi 8 Rajyindee" priority="3" type="highway.residential">\n    <lane id="155691022#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="27.78" shape="1629.82,491.78 1624.45,519.04">\n      <param key="origId" value="155691022" />\n    </lane>\n  </edge>\n  <edge id="155691022#1" from="4900465071" to="1431867140" name="Soi 8 Rajyindee" priority="3" type="highway.residential" shape="1624.65,526.30 1614.53,587.23 1684.84,605.32 1779.79,609.07 1823.78,612.25">\n    <lane id="155691022#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="269.30" shape="1621.86,533.36 1612.71,588.41 1684.61,606.91 1779.70,610.67 1823.66,613.85">\n      <param key="origId" value="155691022" />\n    </lane>\n  </edge>\n  <edge id="155691032" from="1680667029" to="1680666823" priority="3" type="highway.residential" shape="981.09,509.46 986.39,463.57 1032.81,467.36">\n    <lane id="155691032_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="82.43" shape="983.51,502.45 987.80,465.29 1032.68,468.95">\n      <param key="origId" value="155691032" />\n    </lane>\n  </edge>\n  <edge id="155691084#0" from="8642866010" to="4900465949" name="ซอย 8 ราษฎร์ยินดี" priority="10" type="highway.tertiary">\n    <lane id="155691084#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.20" shape="1446.91,460.36 1447.11,460.34">\n      <param key="origId" value="155691084" />\n    </lane>\n  </edge>\n  <edge id="155691084#1" from="4900465949" to="1680666827" name="ซอย 8 ราษฎร์ยินดี" priority="10" type="highway.tertiary">\n    <lane id="155691084#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="80.62" shape="1458.64,460.80 1538.55,471.40">\n      <param key="origId" value="155691084" />\n    </lane>\n  </edge>\n  <edge id="155691084#2" from="1680666827" to="1680666891" name="ซอย 8 ราษฎร์ยินดี" priority="10" type="highway.tertiary" shape="1545.67,470.73 1600.10,478.02 1632.80,484.92">\n    <lane id="155691084#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="73.60" shape="1552.85,473.31 1599.83,479.60 1625.47,485.01">\n      <param key="origId" value="155691084" />\n    </lane>\n  </edge>\n  <edge id="155691084#3" from="1680666891" to="8523321638" name="ซอย 8 ราษฎร์ยินดี" priority="10" type="highway.tertiary" shape="1632.80,484.92 1713.76,503.59 1836.34,543.00">\n    <lane id="155691084#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="204.40" shape="1639.56,488.12 1713.33,505.13 1835.85,544.52">\n      <param key="origId" value="155691084" />\n    </lane>\n  </edge>\n  <edge id="155691088#0" from="1431867172" to="1680666463" name="Soi Vachira-Anan 1" priority="3" type="highway.residential" shape="1600.88,173.25 1566.82,230.20 1523.62,308.81 1494.83,374.00">\n    <lane id="155691088#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="219.61" shape="1599.51,172.43 1565.43,229.40 1522.18,308.10 1496.56,366.12">\n      <param key="origId" value="155691088" />\n    </lane>\n  </edge>\n  <edge id="155691088#1" from="1680666463" to="1680666716" name="Soi Vachira-Anan 1" priority="3" type="highway.residential">\n    <lane id="155691088#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="46.75" shape="1491.14,380.60 1477.51,425.32">\n      <param key="origId" value="155691088" />\n    </lane>\n  </edge>\n  <edge id="1557537524" from="14165547215" to="14165547216" priority="3" type="highway.residential">\n    <lane id="1557537524_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="27.78" shape="1377.44,1273.27 1349.66,1272.86">\n      <param key="origId" value="1557537524" />\n    </lane>\n  </edge>\n  <edge id="1558734968" from="14179759489" to="2506088476" priority="3" type="highway.residential">\n    <lane id="1558734968_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="60.52" shape="405.06,1435.65 458.30,1464.41">\n      <param key="origId" value="1558734968" />\n    </lane>\n  </edge>\n  <edge id="1559020882" from="14182422401" to="14182422402" priority="3" type="highway.residential">\n    <lane id="1559020882_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="26.65" shape="732.56,407.86 735.88,381.42">\n      <param key="origId" value="1559020882" />\n    </lane>\n  </edge>\n  <edge id="1559802495#0" from="4900465288" to="14188374629" priority="3" type="highway.residential">\n    <lane id="1559802495#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.59" shape="1234.78,1548.55 1221.26,1549.86">\n      <param key="origId" value="1559802495" />\n    </lane>\n  </edge>\n  <edge id="1559802495#1" from="14188374629" to="14188374628" priority="3" type="highway.residential">\n    <lane id="1559802495#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.22" shape="1206.92,1551.26 1193.76,1552.54">\n      <param key="origId" value="1559802495" />\n    </lane>\n  </edge>\n  <edge id="1559802496" from="14188374629" to="cluster_14188374630_2506088509" priority="3" type="highway.residential" shape="1214.10,1552.17 1213.86,1547.57 1212.12,1514.65 1211.66,1505.83">\n    <lane id="1559802496_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.42" shape="1215.31,1544.75 1213.72,1514.57 1213.66,1513.37">\n      <param key="origId" value="1559802496" />\n    </lane>\n    <param key="origTo" value="14188374630" />\n  </edge>\n  <edge id="1559841160#0" from="14188764645" to="14188764646" priority="3" type="highway.residential">\n    <lane id="1559841160#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.07" shape="815.78,580.40 846.73,583.11">\n      <param key="origId" value="1559841160" />\n    </lane>\n  </edge>\n  <edge id="1559841160#1" from="14188764646" to="14188764647" priority="3" type="highway.residential">\n    <lane id="1559841160#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.53" shape="856.41,575.19 861.28,535.96">\n      <param key="origId" value="1559841160" />\n    </lane>\n  </edge>\n  <edge id="1559841161" from="14188764646" to="14188764648" priority="3" type="highway.residential">\n    <lane id="1559841161_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.30" shape="860.89,584.60 878.07,586.70">\n      <param key="origId" value="1559841161" />\n    </lane>\n  </edge>\n  <edge id="185951074#0" from="856214524" to="13279030938" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary">\n    <lane id="185951074#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.96" shape="1329.38,984.18 1314.42,984.36">\n      <param key="origId" value="185951074" />\n    </lane>\n  </edge>\n  <edge id="185951074#1" from="13279030938" to="4900465707" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary">\n    <lane id="185951074#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="65.84" shape="1299.98,984.42 1234.14,984.16">\n      <param key="origId" value="185951074" />\n    </lane>\n  </edge>\n  <edge id="185951074#2" from="4900465707" to="1123399600" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="1228.78,985.74 1173.33,984.99 1136.16,984.71 1084.18,985.09">\n    <lane id="185951074#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="130.50" shape="1222.92,984.06 1173.35,983.39 1136.16,983.11 1092.42,983.43">\n      <param key="origId" value="185951074" />\n    </lane>\n  </edge>\n  <edge id="185951074#4" from="1123399600" to="4900464584" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="1084.18,985.09 1067.01,982.70 1055.67,981.13 1050.82,980.45">\n    <lane id="185951074#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.30" shape="1074.01,982.06 1067.23,981.12 1057.87,979.82">\n      <param key="origId" value="185951074 969757850 1204844052" />\n    </lane>\n  </edge>\n  <edge id="194743656#0" from="2052497441" to="2506086743" name="ถนนแสงอาทิตย์" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="194743656#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="57.99" shape="504.14,1148.49 447.18,1137.62">\n      <param key="origId" value="194743656" />\n    </lane>\n  </edge>\n  <edge id="194743656#1" from="2506086743" to="2052522376" name="ถนนแสงอาทิตย์" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="194743656#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="105.39" shape="437.93,1135.89 334.23,1117.06">\n      <param key="origId" value="194743656" />\n    </lane>\n  </edge>\n  <edge id="243079656" from="2506089779" to="1115585857" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="243079656_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.20" shape="1021.61,1708.62 1021.69,1708.81">\n      <param key="origId" value="243079656" />\n    </lane>\n  </edge>\n  <edge id="243079673" from="947896979" to="2505176494" priority="4" type="highway.unclassified">\n    <lane id="243079673_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="2.32" shape="1610.18,1471.93 1607.87,1472.08">\n      <param key="origId" value="243079673" />\n    </lane>\n  </edge>\n  <edge id="243079723" from="2505176686" to="947896972" name="ถนนราษฏร์ยินดี ซอย 2" priority="3" type="highway.residential" shape="1986.72,1591.26 1953.55,1589.43 1929.69,1587.64 1803.25,1578.19 1795.57,1577.62 1629.94,1569.30">\n    <lane id="243079723_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="351.15" shape="1986.81,1589.66 1953.65,1587.83 1929.81,1586.04 1803.37,1576.59 1795.67,1576.02 1636.35,1568.02">\n      <param key="origId" value="243079723" />\n    </lane>\n  </edge>\n  <edge id="243079738" from="947896958" to="849403490" name="Thanon Rajyindee Soi 6" priority="3" type="highway.residential" spreadType="center" shape="1574.42,1223.86 1777.03,1222.70 1817.02,1222.00 1851.75,1217.58 1992.35,1212.03">\n    <lane id="243079738_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="412.47" shape="1580.28,1223.83 1777.03,1222.70 1817.02,1222.00 1851.75,1217.58 1992.35,1212.03">\n      <param key="origId" value="243079738" />\n    </lane>\n  </edge>\n  <edge id="243079757#1" from="8523328988" to="1639088658" name="Soi Rajyindee Road 6/ 1" priority="3" type="highway.residential" shape="1763.96,1102.14 1752.94,1098.57 1746.95,1096.63 1738.77,1093.07 1707.65,1084.84 1682.83,1080.32 1673.44,1078.96 1669.41,1078.37 1548.26,1061.79">\n    <lane id="243079757#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="213.66" shape="1764.45,1100.62 1753.43,1097.05 1747.52,1095.13 1739.30,1091.55 1708.00,1083.28 1683.09,1078.74 1673.67,1077.38 1669.63,1076.79 1555.10,1061.11">\n      <param key="origId" value="243079757" />\n    </lane>\n  </edge>\n  <edge id="243079775" from="2505176514" to="947896989" priority="3" type="highway.residential">\n    <lane id="243079775_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.10" shape="1611.49,1495.95 1611.59,1495.95">\n      <param key="origId" value="243079775" />\n    </lane>\n  </edge>\n  <edge id="243079805" from="1110416568" to="1431715366" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="243079805_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.81" shape="748.16,677.73 763.78,680.13">\n      <param key="origId" value="243079805" />\n    </lane>\n  </edge>\n  <edge id="243079844#0" from="11292971036" to="809537690" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="243079844#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.20" shape="1562.77,1177.60 1562.96,1177.56">\n      <param key="origId" value="243079844" />\n    </lane>\n  </edge>\n  <edge id="243079844#1" from="809537690" to="2406954577" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" spreadType="center" shape="1567.86,1176.65 1859.58,1166.87 1973.62,1165.64 1989.32,1164.50">\n    <lane id="243079844#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="415.90" shape="1573.63,1176.46 1859.58,1166.87 1973.62,1165.64 1989.32,1164.50">\n      <param key="origId" value="243079844" />\n    </lane>\n  </edge>\n  <edge id="243188024" from="1115585463" to="2506085525" priority="3" type="highway.residential" shape="998.04,876.32 1032.16,846.90 1087.97,813.59 1120.06,798.73 1166.48,787.71 1184.64,787.30 1282.42,790.45 1304.91,788.15 1325.25,778.71 1343.94,766.43 1347.02,759.73 1352.13,748.64 1352.58,744.79 1353.51,736.98">\n    <lane id="243188024_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="397.25" shape="1003.72,873.53 1033.10,848.20 1088.72,815.01 1120.59,800.25 1166.69,789.31 1184.63,788.90 1282.48,792.05 1305.34,789.71 1326.03,780.11 1345.20,767.52 1348.47,760.40 1353.69,749.08 1354.17,744.98 1354.43,742.81">\n      <param key="origId" value="243188024" />\n    </lane>\n  </edge>\n  <edge id="243188056" from="1369020942" to="2506088695" name="Soi Boonraj" priority="3" type="highway.residential">\n    <lane id="243188056_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="123.42" shape="1414.51,1544.47 1537.75,1537.77">\n      <param key="origId" value="243188056" />\n    </lane>\n  </edge>\n  <edge id="243188064#0" from="2506085496" to="4900465472" name="ถนนภราดร" priority="3" type="highway.residential" spreadType="center">\n    <lane id="243188064#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="41.72" shape="661.78,675.57 653.07,716.37">\n      <param key="origId" value="243188064" />\n    </lane>\n  </edge>\n  <edge id="243188064#1" from="4900465472" to="2506085628" name="ถนนภราดร" priority="3" type="highway.residential" spreadType="center">\n    <lane id="243188064#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="151.23" shape="649.94,730.75 617.01,878.35">\n      <param key="origId" value="243188064" />\n    </lane>\n  </edge>\n  <edge id="243188085" from="1123399486" to="2506088938" priority="3" type="highway.residential">\n    <lane id="243188085_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="64.27" shape="512.81,1606.85 576.15,1595.97">\n      <param key="origId" value="243188085" />\n    </lane>\n  </edge>\n  <edge id="243188092" from="2506088528" to="2506088507" name="Soi Boonraj 1" priority="3" type="highway.residential">\n    <lane id="243188092_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="81.08" shape="1411.94,1520.61 1492.93,1516.71">\n      <param key="origId" value="243188092" />\n    </lane>\n  </edge>\n  <edge id="243188103" from="2506087304" to="2506087252" priority="3" type="highway.residential">\n    <lane id="243188103_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.49" shape="423.24,1194.58 393.23,1189.20">\n      <param key="origId" value="243188103" />\n    </lane>\n  </edge>\n  <edge id="243188115" from="1123399713" to="2506090451" name="Soi Thongdee 3" priority="3" type="highway.residential">\n    <lane id="243188115_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="104.21" shape="1217.53,1663.22 1198.76,1765.72">\n      <param key="origId" value="243188115" />\n    </lane>\n  </edge>\n  <edge id="243188120" from="cluster_620881097_8523329816" to="2506087028" priority="10" type="highway.tertiary" spreadType="center" shape="315.06,1169.24 276.72,1164.52 253.86,1161.70">\n    <lane id="243188120_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="52.78" shape="306.25,1168.16 276.72,1164.52 253.86,1161.70">\n      <param key="origId" value="243188120" />\n    </lane>\n    <param key="origFrom" value="620881097" />\n  </edge>\n  <edge id="243188139" from="2506088023" to="2506088011" name="Soi Boonraj 3" priority="3" type="highway.residential">\n    <lane id="243188139_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="88.51" shape="1404.33,1388.57 1492.64,1382.59">\n      <param key="origId" value="243188139" />\n    </lane>\n  </edge>\n  <edge id="243188161" from="1123399675" to="2506089487" priority="3" type="highway.residential">\n    <lane id="243188161_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="49.84" shape="543.31,1672.50 590.58,1656.71">\n      <param key="origId" value="243188161" />\n    </lane>\n  </edge>\n  <edge id="243188167" from="2506088274" to="2506088265" name="Soi Boonraj 2" priority="3" type="highway.residential">\n    <lane id="243188167_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="112.69" shape="1409.45,1466.95 1521.86,1459.00">\n      <param key="origId" value="243188167" />\n    </lane>\n  </edge>\n  <edge id="243188174#0" from="2506087709" to="6634630596" name="ถนนทิพรัตน์" priority="10" type="highway.tertiary">\n    <lane id="243188174#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.54" shape="305.74,1287.82 318.21,1289.11">\n      <param key="origId" value="243188174" />\n    </lane>\n  </edge>\n  <edge id="243188174#1" from="6634630596" to="4900465989" name="ถนนทิพรัตน์" priority="10" type="highway.tertiary">\n    <lane id="243188174#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="36.07" shape="335.92,1290.93 371.80,1294.63">\n      <param key="origId" value="243188174" />\n    </lane>\n  </edge>\n  <edge id="243188174#2" from="4900465989" to="2506087732" name="ถนนทิพรัตน์" priority="10" type="highway.tertiary">\n    <lane id="243188174#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="38.44" shape="386.84,1296.18 425.08,1300.12">\n      <param key="origId" value="243188174" />\n    </lane>\n  </edge>\n  <edge id="243188196" from="1123399513" to="2506089901" priority="3" type="highway.residential">\n    <lane id="243188196_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="50.67" shape="1468.52,1695.02 1419.81,1708.99">\n      <param key="origId" value="243188196" />\n    </lane>\n  </edge>\n  <edge id="243188208" from="2506087592" to="2506087962" priority="3" type="highway.residential">\n    <lane id="243188208_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="86.17" shape="921.93,1251.03 936.50,1335.96">\n      <param key="origId" value="243188208" />\n    </lane>\n  </edge>\n  <edge id="243188212#0" from="12351390200" to="1123399641" name="ถนนศุภสารรังสรรค์ ซอย 4/2" priority="3" type="highway.residential">\n    <lane id="243188212#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="3.52" shape="1194.21,1409.24 1195.28,1412.59">\n      <param key="origId" value="243188212" />\n    </lane>\n  </edge>\n  <edge id="243188212#1" from="1123399641" to="cluster_14188374630_2506088509" name="ถนนศุภสารรังสรรค์ ซอย 4/2" priority="3" type="highway.residential" shape="1199.28,1419.81 1199.46,1465.00 1198.68,1487.25 1198.03,1507.30">\n    <lane id="243188212#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="71.99" shape="1197.71,1427.60 1197.86,1464.98 1197.08,1487.20 1196.68,1499.58">\n      <param key="origId" value="243188212" />\n    </lane>\n    <param key="origTo" value="2506088509" />\n  </edge>\n  <edge id="243188225" from="1123399683" to="4488366488" name="ถนนศุภสารรังสรรค์ ซอย 4" priority="10" type="highway.tertiary" shape="1236.41,1307.92 1235.99,1262.05 1235.51,1209.07">\n    <lane id="243188225_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="82.06" shape="1237.95,1300.98 1237.59,1262.04 1237.20,1218.92">\n      <param key="origId" value="243188225" />\n    </lane>\n  </edge>\n  <edge id="243188243" from="2506090118" to="1123399591" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="243188243_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.75" shape="535.05,1730.24 563.14,1720.43">\n      <param key="origId" value="243188243" />\n    </lane>\n  </edge>\n  <edge id="243188273#0" from="1110416526" to="1123399569" name="ถนนจุติอุทิศ 2" priority="10" type="highway.tertiary" shape="864.85,1646.46 869.56,1685.12 879.44,1759.57">\n    <lane id="243188273#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="102.78" shape="864.13,1653.77 867.97,1685.32 877.31,1755.69">\n      <param key="origId" value="243188273" />\n    </lane>\n  </edge>\n  <edge id="243188283" from="1369020943" to="2506087957" name="Soi Boonraj 4" priority="3" type="highway.residential">\n    <lane id="243188283_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="87.07" shape="1401.17,1347.17 1487.99,1340.63">\n      <param key="origId" value="243188283" />\n    </lane>\n  </edge>\n  <edge id="243188298" from="2506089779" to="1123399676" priority="10" type="highway.tertiary" shape="1017.51,1699.50 999.80,1685.05 997.91,1683.43 995.81,1681.63">\n    <lane id="243188298_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.58" shape="1014.47,1694.95 1002.40,1685.10">\n      <param key="origId" value="243188298 497290269" />\n    </lane>\n  </edge>\n  <edge id="243188318" from="2506085486" to="2506085476" priority="3" type="highway.residential" shape="1267.64,640.76 1267.81,632.68 1268.19,614.82 1268.26,611.86 1268.62,594.28 1268.68,591.83 1269.08,573.30 1269.22,566.52">\n    <lane id="243188318_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="66.91" shape="1269.24,640.79 1269.41,632.71 1269.79,614.85 1269.86,611.89 1270.22,594.31 1270.28,591.87 1270.67,573.90">\n      <param key="origId" value="243188318" />\n    </lane>\n  </edge>\n  <edge id="243188351#0" from="2506085631" to="1107926639" name="ถนนประชาธิปัตย์ ซอย 3" priority="10" type="highway.tertiary" shape="1221.64,888.53 1271.06,884.49 1335.59,881.55">\n    <lane id="243188351#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="101.20" shape="1227.36,889.67 1271.16,886.09 1328.36,883.48">\n      <param key="origId" value="243188351" />\n    </lane>\n  </edge>\n  <edge id="243188361" from="1123399943" to="2506089058" priority="3" type="highway.residential">\n    <lane id="243188361_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="60.37" shape="523.06,1632.59 581.81,1618.72">\n      <param key="origId" value="243188361" />\n    </lane>\n  </edge>\n  <edge id="243188389#1" from="12346099051" to="2506088491" priority="3" type="highway.residential">\n    <lane id="243188389#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.20" shape="560.67,1491.91 541.51,1490.60">\n      <param key="origId" value="243188389" />\n    </lane>\n  </edge>\n  <edge id="243188389#2" from="2506088491" to="2506088494" priority="3" type="highway.residential">\n    <lane id="243188389#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.93" shape="527.32,1490.50 509.42,1491.57">\n      <param key="origId" value="243188389" />\n    </lane>\n  </edge>\n  <edge id="243188389#3" from="2506088494" to="2506090118" priority="3" type="highway.residential" shape="503.67,1493.51 486.21,1504.18 479.25,1514.13 471.77,1533.82 464.79,1553.03 465.91,1577.02 486.29,1632.43 505.76,1676.19 519.79,1698.32 531.57,1729.76">\n    <lane id="243188389#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="256.64" shape="496.60,1495.95 485.09,1502.99 477.82,1513.38 470.27,1533.26 463.18,1552.78 464.32,1577.34 484.81,1633.03 504.35,1676.95 518.35,1699.04 528.97,1727.39">\n      <param key="origId" value="243188389" />\n    </lane>\n  </edge>\n  <edge id="243188432#0" from="947896950" to="2506086831" name="ถนนแสงศรี ซอย 8" priority="10" type="highway.tertiary" spreadType="center" shape="894.34,1353.65 892.10,1336.95 888.46,1305.42 881.61,1204.04 878.02,1145.79">\n    <lane id="243188432#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="193.79" shape="893.37,1346.42 892.10,1336.95 888.46,1305.42 881.61,1204.04 878.48,1153.25">\n      <param key="origId" value="243188432" />\n    </lane>\n  </edge>\n  <edge id="243188445" from="1123399740" to="2506090385" name="Soi Thongdee 1" priority="3" type="highway.residential" shape="1281.68,1643.74 1268.98,1708.27 1253.06,1765.60">\n    <lane id="243188445_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="116.44" shape="1278.43,1651.97 1267.42,1707.90 1251.52,1765.17">\n      <param key="origId" value="243188445" />\n    </lane>\n  </edge>\n  <edge id="243188447" from="2506085670" to="2506085849" name="Soi Duangchan" priority="3" type="highway.residential">\n    <lane id="243188447_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="79.03" shape="489.16,910.54 471.44,987.56">\n      <param key="origId" value="243188447" />\n    </lane>\n  </edge>\n  <edge id="243188452#0" from="2506088715" to="2506088612" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="243188452#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.15" shape="1021.58,1536.46 1059.94,1528.65">\n      <param key="origId" value="243188452" />\n    </lane>\n  </edge>\n  <edge id="243188452#1" from="2506088612" to="2506088529" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="243188452#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.73" shape="1071.00,1526.57 1110.15,1519.79">\n      <param key="origId" value="243188452" />\n    </lane>\n  </edge>\n  <edge id="243188458" from="cluster_1680667233_2506085475" to="2506085493" priority="3" type="highway.residential" shape="1234.89,563.71 1229.31,590.55 1225.62,608.38 1224.47,613.92 1218.81,641.19 1217.65,646.78 1195.62,654.79">\n    <lane id="243188458_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="90.16" shape="1231.79,570.78 1227.74,590.22 1224.05,608.06 1222.90,613.59 1217.24,640.86 1216.26,645.58 1203.33,650.28">\n      <param key="origId" value="243188458" />\n    </lane>\n    <param key="origFrom" value="2506085475" />\n  </edge>\n  <edge id="243188465#0" from="cluster_13424220808_2506085647" to="4900465473" name="ถนนประชารมย์" priority="3" type="highway.residential" shape="659.33,898.58 687.90,775.48 698.50,730.13">\n    <lane id="243188465#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="157.92" shape="662.58,891.67 689.46,775.84 698.34,737.85">\n      <param key="origId" value="243188465" />\n    </lane>\n    <param key="origFrom" value="2506085647" />\n  </edge>\n  <edge id="243188465#1" from="4900465473" to="cluster_2052547073_2506085499" name="ถนนประชารมย์" priority="3" type="highway.residential" shape="698.50,730.13 713.23,673.94">\n    <lane id="243188465#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="44.56" shape="701.77,723.95 713.07,680.85">\n      <param key="origId" value="243188465" />\n    </lane>\n    <param key="origTo" value="2506085499" />\n  </edge>\n  <edge id="243188491" from="2506088024" to="1966399248" name="ถนนประชาธิปัตย์ 1" priority="10" type="highway.tertiary" shape="775.02,1388.94 772.64,1232.89 775.07,1136.75">\n    <lane id="243188491_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="236.36" shape="776.49,1380.49 774.24,1232.90 776.48,1144.17">\n      <param key="origId" value="243188491" />\n    </lane>\n  </edge>\n  <edge id="243188499" from="2506085555" to="2506085631" priority="3" type="highway.residential" spreadType="center">\n    <lane id="243188499_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="62.76" shape="1216.16,818.78 1221.08,881.35">\n      <param key="origId" value="243188499" />\n    </lane>\n  </edge>\n  <edge id="243188503#0" from="1123399700" to="2506089901" priority="3" type="highway.residential">\n    <lane id="243188503#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="53.58" shape="1428.02,1652.14 1412.13,1703.31">\n      <param key="origId" value="243188503" />\n    </lane>\n  </edge>\n  <edge id="243188503#1" from="2506089901" to="1110416342" priority="3" type="highway.residential">\n    <lane id="243188503#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.76" shape="1407.53,1718.31 1395.87,1757.37">\n      <param key="origId" value="243188503" />\n    </lane>\n  </edge>\n  <edge id="243188505" from="2506087955" to="2506087968" priority="3" type="highway.residential">\n    <lane id="243188505_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="60.98" shape="575.01,1339.68 635.89,1343.25">\n      <param key="origId" value="243188505" />\n    </lane>\n  </edge>\n  <edge id="243188508#0" from="2506088518" to="cluster_14188374630_2506088509" priority="3" type="highway.residential" shape="1155.61,1510.74 1169.35,1509.69 1198.03,1507.30">\n    <lane id="243188508#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="28.12" shape="1163.31,1511.76 1169.48,1511.28 1191.34,1509.46">\n      <param key="origId" value="243188508" />\n    </lane>\n    <param key="origTo" value="2506088509" />\n  </edge>\n  <edge id="243188508#3" from="cluster_14188374630_2506088509" to="2506088492" priority="3" type="highway.residential" shape="1211.66,1505.83 1242.82,1502.45">\n    <lane id="243188508#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.71" shape="1218.82,1506.66 1235.43,1504.86">\n      <param key="origId" value="243188508" />\n    </lane>\n    <param key="origFrom" value="14188374630" />\n  </edge>\n  <edge id="243188527" from="2506087810" to="2506087863" name="Soi Watthayorn" priority="3" type="highway.residential" spreadType="center">\n    <lane id="243188527_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="66.31" shape="494.08,1298.27 559.86,1306.57">\n      <param key="origId" value="243188527" />\n    </lane>\n  </edge>\n  <edge id="243188536#0" from="2506087563" to="8348961009" priority="3" type="highway.residential">\n    <lane id="243188536#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.37" shape="423.30,1242.89 425.91,1228.75">\n      <param key="origId" value="243188536" />\n    </lane>\n  </edge>\n  <edge id="243188536#1" from="8348961009" to="2506087304" priority="3" type="highway.residential">\n    <lane id="243188536#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.94" shape="428.52,1214.60 430.32,1204.82">\n      <param key="origId" value="243188536" />\n    </lane>\n  </edge>\n  <edge id="243188536#2" from="2506087304" to="2506086743" priority="3" type="highway.residential">\n    <lane id="243188536#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="51.67" shape="433.07,1190.78 443.51,1140.18">\n      <param key="origId" value="243188536" />\n    </lane>\n  </edge>\n  <edge id="243188552#0" from="1123399516" to="2506088492" name="ถนนกุลวัฒน์อุทิศ" priority="10" type="highway.tertiary" shape="1239.67,1387.29 1244.44,1442.03 1243.94,1482.31 1242.82,1502.45">\n    <lane id="243188552#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="94.79" shape="1239.26,1401.05 1242.84,1442.09 1242.34,1482.26 1241.60,1495.66">\n      <param key="origId" value="243188552" />\n    </lane>\n  </edge>\n  <edge id="243188552#1" from="2506088492" to="4900465288" name="ถนนกุลวัฒน์อุทิศ" priority="10" type="highway.tertiary">\n    <lane id="243188552#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.50" shape="1241.14,1510.04 1240.78,1542.53">\n      <param key="origId" value="243188552" />\n    </lane>\n  </edge>\n  <edge id="243188552#2" from="4900465288" to="1107926940" name="ถนนกุลวัฒน์อุทิศ" priority="10" type="highway.tertiary">\n    <lane id="243188552#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="23.17" shape="1240.97,1556.87 1241.82,1580.03">\n      <param key="origId" value="243188552" />\n    </lane>\n  </edge>\n  <edge id="243188561#0" from="2506085690" to="2506086151" priority="3" type="highway.residential" shape="751.80,928.65 740.35,979.48 718.23,1077.60">\n    <lane id="243188561#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="145.20" shape="748.59,935.60 738.79,979.13 716.67,1077.25">\n      <param key="origId" value="243188561" />\n    </lane>\n  </edge>\n  <edge id="243188563" from="2506086795" to="2506088002" name="ถนนแสงศรี ซอย 6" priority="10" type="highway.tertiary" shape="825.73,1140.88 830.68,1329.02 833.66,1368.43">\n    <lane id="243188563_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="212.43" shape="824.33,1148.53 829.08,1329.10 831.48,1360.80">\n      <param key="origId" value="243188563" />\n    </lane>\n  </edge>\n  <edge id="243188565" from="2506088518" to="1123399483" name="ถนนศุภสารรังสรรค์ ซอย 4/3" priority="3" type="highway.residential">\n    <lane id="243188565_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="71.24" shape="1157.53,1503.21 1160.55,1432.03">\n      <param key="origId" value="243188565" />\n    </lane>\n  </edge>\n  <edge id="243188585" from="2506087136" to="1107926561" name="ถนนศุภสารรังสรรค์ ซอย 5" priority="10" type="highway.tertiary" shape="1285.38,1183.95 1280.79,1323.87 1282.91,1482.86 1286.69,1575.57">\n    <lane id="243188585_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="373.43" shape="1283.43,1194.44 1279.19,1323.85 1281.31,1482.90 1284.77,1567.72">\n      <param key="origId" value="243188585" />\n    </lane>\n  </edge>\n  <edge id="243188594#0" from="1107926977" to="2506088612" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="243188594#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="75.00" shape="1081.09,1606.64 1066.53,1533.06">\n      <param key="origId" value="243188594" />\n    </lane>\n  </edge>\n  <edge id="243188594#1" from="2506088612" to="1123399489" priority="10" type="highway.tertiary" spreadType="center" shape="1065.44,1527.53 1055.93,1470.61 1044.60,1401.15">\n    <lane id="243188594#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="113.84" shape="1064.52,1522.00 1055.93,1470.61 1045.99,1409.67">\n      <param key="origId" value="243188594" />\n    </lane>\n  </edge>\n  <edge id="243188607#0" from="2506085508" to="2506085493" priority="3" type="highway.residential" shape="1184.44,688.87 1193.41,675.21 1195.02,660.35 1195.62,654.79">\n    <lane id="243188607#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="24.08" shape="1189.94,683.41 1194.96,675.77 1196.57,660.92">\n      <param key="origId" value="243188607" />\n    </lane>\n  </edge>\n  <edge id="243188607#1" from="2506085493" to="2506085487" priority="3" type="highway.residential" shape="1195.62,654.79 1195.78,651.18 1196.15,642.20">\n    <lane id="243188607#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="3.82" shape="1197.59,646.08 1197.75,642.27">\n      <param key="origId" value="243188607" />\n    </lane>\n  </edge>\n  <edge id="243188627" from="2506090779" to="2506089600" name="Soi Thongdee 4" priority="3" type="highway.residential" shape="1094.06,1836.35 1110.54,1782.99 1112.62,1746.83 1117.29,1676.07">\n    <lane id="243188627_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="155.05" shape="1095.59,1836.82 1112.13,1783.28 1114.22,1746.93 1118.34,1684.46">\n      <param key="origId" value="243188627" />\n    </lane>\n  </edge>\n  <edge id="243188629" from="1107926639" to="796747634" name="ถนนประชาธิปัตย์ ซอย 3" priority="10" type="highway.tertiary" shape="1335.59,881.55 1371.81,886.77 1449.80,899.54 1495.25,902.19">\n    <lane id="243188629_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="147.20" shape="1343.02,884.24 1371.57,888.35 1449.62,901.13 1488.81,903.42">\n      <param key="origId" value="243188629" />\n    </lane>\n  </edge>\n  <edge id="243188641#0" from="1107926599" to="2506088529" name="Thanon Suvanvong" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="243188641#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="78.83" shape="1130.44,1601.92 1116.60,1524.31">\n      <param key="origId" value="243188641" />\n    </lane>\n  </edge>\n  <edge id="243188641#1" from="2506088529" to="1123399659" name="Thanon Suvanvong" priority="10" type="highway.tertiary" spreadType="center" shape="1115.62,1518.84 1106.49,1463.17 1100.56,1422.57">\n    <lane id="243188641#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="84.31" shape="1114.72,1513.36 1106.49,1463.17 1101.66,1430.08">\n      <param key="origId" value="243188641" />\n    </lane>\n  </edge>\n  <edge id="243188644" from="2506089545" to="2506090932" name="Soi Thongdee 3" priority="3" type="highway.residential" shape="1171.53,1664.80 1161.22,1769.38 1151.92,1783.68 1132.87,1853.87">\n    <lane id="243188644_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="186.24" shape="1169.12,1672.96 1159.67,1768.83 1150.44,1783.02 1131.33,1853.45">\n      <param key="origId" value="243188644" />\n    </lane>\n  </edge>\n  <edge id="243188646#0" from="2506087563" to="2506087522" priority="3" type="highway.residential" spreadType="center">\n    <lane id="243188646#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="47.21" shape="413.32,1249.05 366.48,1243.17">\n      <param key="origId" value="243188646" />\n    </lane>\n  </edge>\n  <edge id="243188646#1" from="2506087522" to="cluster_2506087462_8523307708" priority="3" type="highway.residential" spreadType="center" shape="360.41,1242.41 310.25,1233.64">\n    <lane id="243188646#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="38.25" shape="356.20,1241.67 318.52,1235.09">\n      <param key="origId" value="243188646" />\n    </lane>\n    <param key="origTo" value="8523307708" />\n  </edge>\n  <edge id="243188670#0" from="cluster_1123399715_3376084911_3376091818" to="2506088715" priority="10" type="highway.tertiary" spreadType="center" shape="1003.97,1395.38 995.67,1419.48 1014.45,1537.91">\n    <lane id="243188670#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="133.10" shape="1000.32,1401.06 994.03,1419.34 1011.97,1532.49">\n      <param key="origId" value="243188670" />\n    </lane>\n    <lane id="243188670#0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="133.10" shape="1003.35,1402.11 997.31,1419.62 1015.13,1531.99">\n      <param key="origId" value="243188670" />\n    </lane>\n    <param key="origFrom" value="1123399715" />\n  </edge>\n  <edge id="243188670#1" from="2506088715" to="1107926734" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="243188670#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="71.98" shape="1013.88,1543.65 1026.93,1614.44">\n      <param key="origId" value="243188670" />\n    </lane>\n    <lane id="243188670#1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="71.98" shape="1017.03,1543.07 1030.08,1613.86">\n      <param key="origId" value="243188670" />\n    </lane>\n  </edge>\n  <edge id="243188670#2" from="1107926734" to="2506089309" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="243188670#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="7.44" shape="1029.88,1628.92 1031.50,1636.18">\n      <param key="origId" value="243188670" />\n    </lane>\n    <lane id="243188670#2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="7.44" shape="1033.01,1628.22 1034.62,1635.48">\n      <param key="origId" value="243188670" />\n    </lane>\n  </edge>\n  <edge id="243188705" from="1123399683" to="1123399516" name="ถนนศุภสารรังสรรค์ ซอย 4" priority="10" type="highway.tertiary" shape="1236.41,1307.92 1237.36,1319.25 1238.15,1331.09 1238.95,1340.28 1239.41,1351.13 1239.97,1360.17 1240.21,1369.87 1240.19,1378.58 1239.67,1387.29">\n    <lane id="243188705_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="67.58" shape="1235.42,1315.28 1235.76,1319.37 1236.55,1331.21 1237.35,1340.38 1237.81,1351.21 1238.37,1360.24 1238.61,1369.89 1238.59,1378.53 1238.34,1382.75">\n      <param key="origId" value="243188705" />\n    </lane>\n  </edge>\n  <edge id="243188710#0" from="1107926968" to="1107926467" name="ซอยบุญราช" priority="10" type="highway.tertiary">\n    <lane id="243188710#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.27" shape="1532.58,1728.27 1513.25,1692.94">\n      <param key="origId" value="243188710" />\n    </lane>\n  </edge>\n  <edge id="243188710#1" from="1107926467" to="1369020939" name="ซอยบุญราช" priority="10" type="highway.tertiary" shape="1507.76,1686.25 1489.65,1665.05 1480.32,1655.39 1470.17,1645.72 1455.76,1632.51 1439.20,1619.29 1393.18,1585.88">\n    <lane id="243188710#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="141.14" shape="1505.20,1680.79 1490.83,1663.97 1481.45,1654.25 1471.26,1644.55 1456.80,1631.29 1440.17,1618.02 1399.27,1588.33">\n      <param key="origId" value="243188710" />\n    </lane>\n  </edge>\n  <edge id="243188710#2" from="1369020939" to="1107926481" name="ซอยบุญราช" priority="10" type="highway.tertiary" shape="1393.18,1585.88 1386.23,1581.75 1380.63,1578.83 1340.21,1562.31">\n    <lane id="243188710#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="41.36" shape="1387.18,1580.45 1387.01,1580.35 1381.30,1577.38 1349.16,1564.24">\n      <param key="origId" value="243188710" />\n    </lane>\n  </edge>\n  <edge id="243188741#0" from="1107926537" to="2506085555" priority="3" type="highway.residential">\n    <lane id="243188741#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="107.76" shape="1328.92,805.34 1221.24,809.71">\n      <param key="origId" value="243188741" />\n    </lane>\n  </edge>\n  <edge id="243188741#1" from="2506085555" to="4900465999" priority="3" type="highway.residential" shape="1215.59,811.54 1176.89,815.26 1129.52,825.19 1097.70,837.85 1064.65,856.51 1046.40,878.30 1042.43,887.44 1040.61,897.73 1043.25,908.83 1047.32,916.15">\n    <lane id="243188741#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="219.40" shape="1209.81,810.49 1176.65,813.68 1129.06,823.65 1097.01,836.40 1063.61,855.26 1045.02,877.45 1040.89,886.98 1038.98,897.78 1041.74,909.41 1042.89,911.48">\n      <param key="origId" value="243188741 498671678" />\n    </lane>\n  </edge>\n  <edge id="243188765#0" from="2505176697" to="1110416742" name="ถนนสามชัย" priority="10" type="highway.tertiary">\n    <lane id="243188765#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="24.18" shape="1340.30,1600.06 1341.21,1624.23">\n      <param key="origId" value="243188765" />\n    </lane>\n  </edge>\n  <edge id="243188765#1" from="1110416742" to="4900465069" name="ถนนสามชัย" priority="10" type="highway.tertiary">\n    <lane id="243188765#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="43.41" shape="1341.85,1638.67 1344.05,1682.03">\n      <param key="origId" value="243188765" />\n    </lane>\n  </edge>\n  <edge id="243188765#2" from="4900465069" to="2935421242" name="ถนนสามชัย" priority="10" type="highway.tertiary" shape="1346.02,1689.19 1348.32,1768.47 1350.13,1805.95 1352.64,1817.30 1356.01,1864.68 1357.54,1886.11 1359.74,1915.00">\n    <lane id="243188765#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="219.28" shape="1344.63,1696.45 1346.72,1768.53 1348.54,1806.16 1351.05,1817.53 1354.41,1864.79 1355.94,1886.23 1358.14,1915.12">\n      <param key="origId" value="243188765" />\n    </lane>\n  </edge>\n  <edge id="243188785" from="1110416631" to="1123399502" name="ถนนสวนศิริ" priority="10" type="highway.tertiary" shape="566.14,1496.10 566.06,1518.32 565.91,1520.01">\n    <lane id="243188785_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.22" shape="564.53,1497.57 564.48,1512.78">\n      <param key="origId" value="243188785 243188786" />\n    </lane>\n  </edge>\n  <edge id="243188786#1" from="1123399502" to="2506088938" name="ถนนสวนศิริ" priority="10" type="highway.tertiary">\n    <lane id="243188786#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="61.34" shape="565.82,1526.60 579.92,1586.30">\n      <param key="origId" value="243188786" />\n    </lane>\n  </edge>\n  <edge id="243188786#2" from="2506088938" to="2506089058" name="ถนนสวนศิริ" priority="10" type="highway.tertiary">\n    <lane id="243188786#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="8.78" shape="583.23,1600.33 585.25,1608.87">\n      <param key="origId" value="243188786" />\n    </lane>\n  </edge>\n  <edge id="243188786#3" from="2506089058" to="2506089487" name="ถนนสวนศิริ" priority="10" type="highway.tertiary">\n    <lane id="243188786#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="24.19" shape="588.54,1622.89 594.06,1646.45">\n      <param key="origId" value="243188786" />\n    </lane>\n  </edge>\n  <edge id="243188786#4" from="2506089487" to="1110416754" name="ถนนสวนศิริ" priority="10" type="highway.tertiary">\n    <lane id="243188786#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="38.34" shape="597.37,1660.48 606.23,1697.79">\n      <param key="origId" value="243188786" />\n    </lane>\n  </edge>\n  <edge id="243314073" from="2507400689" to="2507400690" priority="3" type="highway.residential">\n    <lane id="243314073_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.70" shape="1577.55,1280.87 1576.85,1280.88">\n      <param key="origId" value="243314073" />\n    </lane>\n  </edge>\n  <edge id="243314103" from="2507400752" to="2507400748" name="ซอยยินดี" priority="3" type="highway.residential">\n    <lane id="243314103_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="84.14" shape="1523.83,1636.40 1606.93,1623.24">\n      <param key="origId" value="243314103" />\n    </lane>\n  </edge>\n  <edge id="243314112" from="12518064113" to="2507400689" name="Soi Charoensook" priority="3" type="highway.residential">\n    <lane id="243314112_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="200.78" shape="1790.59,1283.97 1589.84,1280.87">\n      <param key="origId" value="243314112" />\n    </lane>\n  </edge>\n  <edge id="243314137" from="2507400769" to="2507400792" priority="10" type="highway.tertiary" shape="815.71,1652.20 830.97,1738.28 832.42,1757.05">\n    <lane id="243314137_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="98.70" shape="815.43,1659.76 829.38,1738.48 830.82,1757.17">\n      <param key="origId" value="243314137" />\n    </lane>\n  </edge>\n  <edge id="243314160" from="1123399863" to="2507400772" priority="3" type="highway.residential">\n    <lane id="243314160_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="64.22" shape="716.01,1747.96 702.23,1685.23">\n      <param key="origId" value="243314160" />\n    </lane>\n  </edge>\n  <edge id="243314165#0" from="2507400614" to="8523326825" name="ซอยเตชะโต" priority="3" type="highway.residential">\n    <lane id="243314165#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="27.30" shape="1554.78,877.57 1581.77,881.66">\n      <param key="origId" value="243314165" />\n    </lane>\n  </edge>\n  <edge id="243314165#1" from="8523326825" to="cluster_2507400612_2507400618" name="ซอยเตชะโต" priority="3" type="highway.residential" shape="1589.04,881.14 1649.02,888.91">\n    <lane id="243314165#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="45.82" shape="1596.00,883.65 1641.44,889.54">\n      <param key="origId" value="243314165" />\n    </lane>\n    <param key="origTo" value="2507400618" />\n  </edge>\n  <edge id="243314177" from="2507400742" to="2507400773" priority="3" type="highway.residential">\n    <lane id="243314177_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="79.62" shape="1705.49,1606.41 1698.44,1685.71">\n      <param key="origId" value="243314177" />\n    </lane>\n  </edge>\n  <edge id="243314186" from="2505176388" to="cluster_2505176422_2505176434" priority="3" type="highway.residential" shape="1606.03,1424.45 1592.59,1425.94">\n    <lane id="243314186_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.40" shape="1600.14,1423.49 1599.74,1423.54">\n      <param key="origId" value="243314186" />\n    </lane>\n    <param key="origTo" value="2505176422" />\n  </edge>\n  <edge id="243314192" from="2507400639" to="2507400638" priority="5" type="highway.tertiary_link">\n    <lane id="243314192_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="1.03" shape="1518.72,928.39 1519.75,928.31">\n      <param key="origId" value="243314192" />\n    </lane>\n  </edge>\n  <edge id="243314213" from="2507400696" to="2507400698" priority="3" type="highway.residential">\n    <lane id="243314213_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.55" shape="1587.77,1345.47 1587.23,1345.51">\n      <param key="origId" value="243314213" />\n    </lane>\n  </edge>\n  <edge id="243314248#0" from="cluster_2507400612_2507400618" to="4900465377" priority="3" type="highway.residential" shape="1651.34,873.76 1705.03,890.88 1736.98,901.02">\n    <lane id="243314248#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="74.91" shape="1657.62,877.44 1704.54,892.40 1729.00,900.17">\n      <param key="origId" value="243314248" />\n    </lane>\n    <param key="origFrom" value="2507400612" />\n  </edge>\n  <edge id="243314248#1" from="4900465377" to="2507400625" priority="3" type="highway.residential">\n    <lane id="243314248#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="5.62" shape="1741.14,905.62 1745.86,908.67">\n      <param key="origId" value="243314248" />\n    </lane>\n  </edge>\n  <edge id="243314310#1" from="2507400591" to="2505176020" priority="3" type="highway.residential" shape="1493.33,719.96 1631.53,725.98 1628.90,758.14">\n    <lane id="243314310#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="155.21" shape="1499.67,721.84 1629.80,727.51 1627.77,752.38">\n      <param key="origId" value="243314310" />\n    </lane>\n  </edge>\n  <edge id="243314312" from="2507400638" to="8523326826" priority="3" type="highway.residential" shape="1525.44,926.27 1531.90,929.27 1549.87,916.18 1584.29,920.15">\n    <lane id="243314312_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="53.69" shape="1531.81,930.99 1532.08,931.12 1550.31,917.84 1580.95,921.38">\n      <param key="origId" value="243314312" />\n    </lane>\n  </edge>\n  <edge id="243314378#0" from="2507400702" to="8523449430" name="Thanon Rajyindee Soi 5" priority="3" type="highway.residential" shape="1890.27,1360.87 1777.01,1353.17 1718.36,1352.72 1704.79,1352.61 1682.59,1351.59">\n    <lane id="243314378#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="200.71" shape="1890.38,1359.27 1777.07,1351.57 1718.37,1351.12 1704.83,1351.01 1689.94,1350.33">\n      <param key="origId" value="243314378" />\n    </lane>\n  </edge>\n  <edge id="243314378#2" from="8523449430" to="4900465441" name="Thanon Rajyindee Soi 5" priority="3" type="highway.residential">\n    <lane id="243314378#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.90" shape="1675.56,1349.66 1642.69,1348.15">\n      <param key="origId" value="243314378" />\n    </lane>\n  </edge>\n  <edge id="243314378#3" from="4900465441" to="2507400696" name="Thanon Rajyindee Soi 5" priority="3" type="highway.residential">\n    <lane id="243314378#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="28.12" shape="1628.29,1347.35 1600.23,1345.54">\n      <param key="origId" value="243314378" />\n    </lane>\n  </edge>\n  <edge id="243314395" from="1680668226" to="1107926968" name="ถนนสามชัย ซอย 3" priority="10" type="highway.tertiary" shape="1626.47,1708.28 1590.76,1717.09 1532.35,1731.19">\n    <lane id="243314395_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="88.55" shape="1620.39,1708.13 1590.38,1715.54 1534.36,1729.06">\n      <param key="origId" value="243314395" />\n    </lane>\n  </edge>\n  <edge id="243314436#0" from="2507400624" to="cluster_2507400612_2507400618" name="Soi Te Ta Cho" priority="3" type="highway.residential" shape="1647.33,906.25 1649.02,888.91">\n    <lane id="243314436#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.32" shape="1648.92,906.41 1649.92,896.13">\n      <param key="origId" value="243314436" />\n    </lane>\n    <param key="origTo" value="2507400618" />\n  </edge>\n  <edge id="243314436#2" from="cluster_2507400612_2507400618" to="2507400604" name="Soi Te Ta Cho" priority="3" type="highway.residential" shape="1651.34,873.76 1655.82,848.70">\n    <lane id="243314436#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.37" shape="1654.11,867.34 1656.11,856.15">\n      <param key="origId" value="243314436" />\n    </lane>\n    <param key="origFrom" value="2507400612" />\n  </edge>\n  <edge id="243314436#3" from="2507400604" to="13449811511" name="Soi Te Ta Cho" priority="3" type="highway.residential" shape="1655.82,848.70 1662.85,794.93 1670.51,765.50">\n    <lane id="243314436#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="71.27" shape="1658.31,841.99 1664.42,795.24 1670.50,771.89">\n      <param key="origId" value="243314436" />\n    </lane>\n  </edge>\n  <edge id="243314440" from="947896989" to="2505176598" name="Thanon Rajyindee Soi 3" priority="3" type="highway.residential" shape="1617.54,1494.09 1687.60,1499.36 1774.26,1514.87 1877.94,1534.67 1988.69,1551.38">\n    <lane id="243314440_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="369.31" shape="1623.83,1496.17 1687.40,1500.95 1773.97,1516.44 1877.67,1536.25 1988.45,1552.96">\n      <param key="origId" value="243314440" />\n    </lane>\n  </edge>\n  <edge id="243314467#0" from="2507400604" to="2507400609" priority="3" type="highway.residential" shape="1655.82,848.70 1714.77,860.84 1741.14,866.28">\n    <lane id="243314467#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="72.89" shape="1662.63,851.74 1714.45,862.41 1734.02,866.44">\n      <param key="origId" value="243314467" />\n    </lane>\n  </edge>\n  <edge id="243314467#1" from="2507400609" to="2505176027" priority="3" type="highway.residential" shape="1741.14,866.28 1757.60,868.69 1797.49,874.51 1807.27,875.42 1831.37,877.68 1844.41,868.54 1848.56,861.70 1850.81,854.34 1852.06,837.51 1855.54,779.68">\n    <lane id="243314467#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="187.32" shape="1748.12,868.92 1757.37,870.27 1797.30,876.10 1807.12,877.01 1831.81,879.33 1845.60,869.66 1850.03,862.36 1852.39,854.64 1853.66,837.62 1856.81,785.28">\n      <param key="origId" value="243314467" />\n    </lane>\n  </edge>\n  <edge id="307427662" from="3125684629" to="3125684630" name="ตรอกจิระพัทธ์" priority="3" type="highway.residential" spreadType="center">\n    <lane id="307427662_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="67.51" shape="273.57,435.95 340.45,445.18">\n      <param key="origId" value="307427662" />\n    </lane>\n  </edge>\n  <edge id="324267161#0" from="3309949761" to="916327839" name="Thanon Prathanuthit" priority="10" type="highway.tertiary" shape="0.00,1666.14 108.39,1679.00 165.86,1687.52 167.86,1687.78 235.21,1696.46 273.34,1701.37 342.85,1710.33 394.42,1716.98 407.87,1718.42">\n    <lane id="324267161#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="404.68" shape="-0.19,1667.73 108.18,1680.59 165.63,1689.10 167.66,1689.37 235.01,1698.05 273.14,1702.96 342.65,1711.92 394.23,1718.57 401.17,1719.31">\n      <param key="origId" value="324267161" />\n    </lane>\n  </edge>\n  <edge id="330615829#0" from="947896964" to="12346099054" name="ถนนศุภสารรังสรรค์ ซอย 3" priority="10" type="highway.tertiary">\n    <lane id="330615829#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="20.45" shape="1017.16,1332.87 1016.56,1353.31">\n      <param key="origId" value="330615829" />\n    </lane>\n  </edge>\n  <edge id="330615829#1" from="12346099054" to="3376091528" name="ถนนศุภสารรังสรรค์ ซอย 3" priority="10" type="highway.tertiary" shape="1017.99,1359.35 1017.49,1366.73 1017.40,1370.57">\n    <lane id="330615829#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="5.83" shape="1016.03,1364.58 1015.89,1366.66 1015.80,1370.40">\n      <param key="origId" value="330615829" />\n    </lane>\n  </edge>\n  <edge id="330615844" from="3376091528" to="cluster_1123399715_3376084911_3376091818" priority="10" type="highway.tertiary" shape="1017.40,1370.57 1016.75,1382.90">\n    <lane id="330615844_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.22" shape="1015.80,1370.61 1015.15,1382.82">\n      <param key="origId" value="330615844" />\n    </lane>\n    <param key="origTo" value="3376091818" />\n  </edge>\n  <edge id="40264696#1" from="1122072640" to="1196113662" name="ถนนนิพัทธ์อุทิศ 2" priority="10" type="highway.tertiary" spreadType="center" shape="372.96,277.33 365.25,340.28 357.48,384.14">\n    <lane id="40264696#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="102.18" shape="369.78,276.94 362.08,339.81 355.34,377.89">\n      <param key="origId" value="40264696" />\n    </lane>\n    <lane id="40264696#1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="102.18" shape="372.96,277.33 365.25,340.28 358.49,378.45">\n      <param key="origId" value="40264696" />\n    </lane>\n    <lane id="40264696#1_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="102.18" shape="376.14,277.72 368.42,340.75 361.64,379.01">\n      <param key="origId" value="40264696" />\n    </lane>\n  </edge>\n  <edge id="40264696#2" from="1196113662" to="3125684630" name="ถนนนิพัทธ์อุทิศ 2" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="40264696#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="51.51" shape="353.56,389.33 346.75,440.38">\n      <param key="origId" value="40264696" />\n    </lane>\n    <lane id="40264696#2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="51.51" shape="356.73,389.75 349.92,440.81">\n      <param key="origId" value="40264696" />\n    </lane>\n    <lane id="40264696#2_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="51.51" shape="359.90,390.17 353.10,441.23">\n      <param key="origId" value="40264696" />\n    </lane>\n  </edge>\n  <edge id="40264696#3" from="3125684630" to="617234535" name="ถนนนิพัทธ์อุทิศ 2" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="40264696#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.35" shape="345.20,451.50 340.94,480.53">\n      <param key="origId" value="40264696" />\n    </lane>\n    <lane id="40264696#3_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.35" shape="348.36,451.96 344.11,481.00">\n      <param key="origId" value="40264696" />\n    </lane>\n    <lane id="40264696#3_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.35" shape="351.53,452.42 347.27,481.46">\n      <param key="origId" value="40264696" />\n    </lane>\n  </edge>\n  <edge id="40264696#4" from="617234535" to="617234280" name="ถนนนิพัทธ์อุทิศ 2" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="40264696#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="147.65" shape="339.10,491.70 312.50,636.93">\n      <param key="origId" value="40264696" />\n    </lane>\n    <lane id="40264696#4_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="147.65" shape="342.25,492.27 315.65,637.51">\n      <param key="origId" value="40264696" />\n    </lane>\n    <lane id="40264696#4_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="147.65" shape="345.39,492.85 318.79,638.09">\n      <param key="origId" value="40264696" />\n    </lane>\n  </edge>\n  <edge id="40264696#5" from="617234280" to="617233014" name="ถนนนิพัทธ์อุทิศ 2" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="40264696#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="174.66" shape="310.36,651.78 292.39,825.51">\n      <param key="origId" value="40264696" />\n    </lane>\n    <lane id="40264696#5_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="174.66" shape="313.55,652.11 295.57,825.84">\n      <param key="origId" value="40264696" />\n    </lane>\n    <lane id="40264696#5_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="174.66" shape="316.73,652.44 298.76,826.17">\n      <param key="origId" value="40264696" />\n    </lane>\n  </edge>\n  <edge id="40264697#1" from="620881566" to="617234280" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="40264697#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="60.23" shape="245.50,639.19 305.39,645.57">\n      <param key="origId" value="40264697" />\n    </lane>\n    <lane id="40264697#1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="60.23" shape="245.84,636.01 305.73,642.39">\n      <param key="origId" value="40264697" />\n    </lane>\n  </edge>\n  <edge id="40264697#2" from="617234280" to="617234576" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="40264697#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="55.82" shape="323.07,647.51 378.53,653.79">\n      <param key="origId" value="40264697" />\n    </lane>\n    <lane id="40264697#2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="55.82" shape="323.43,644.33 378.89,650.61">\n      <param key="origId" value="40264697" />\n    </lane>\n  </edge>\n  <edge id="40264697#3" from="617234576" to="617234281" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="40264697#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="69.77" shape="395.98,655.31 465.62,659.55">\n      <param key="origId" value="40264697" />\n    </lane>\n    <lane id="40264697#3_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="69.77" shape="396.17,652.11 465.81,656.35">\n      <param key="origId" value="40264697" />\n    </lane>\n  </edge>\n  <edge id="40264697#4" from="617234281" to="14069947700" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="40264697#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="86.70" shape="477.49,660.09 564.16,662.78">\n      <param key="origId" value="40264697" />\n    </lane>\n    <lane id="40264697#4_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="86.70" shape="477.59,656.89 564.25,659.58">\n      <param key="origId" value="40264697" />\n    </lane>\n  </edge>\n  <edge id="40264697#5" from="14069947700" to="1107784884" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="40264697#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="26.30" shape="578.65,663.22 604.94,664.03">\n      <param key="origId" value="40264697" />\n    </lane>\n    <lane id="40264697#5_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="26.30" shape="578.75,660.03 605.03,660.83">\n      <param key="origId" value="40264697" />\n    </lane>\n  </edge>\n  <edge id="40264697#6" from="1107784884" to="2506085496" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="40264697#6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.80" shape="616.69,664.84 657.25,669.29">\n      <param key="origId" value="40264697" />\n    </lane>\n    <lane id="40264697#6_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.80" shape="617.04,661.66 657.60,666.11">\n      <param key="origId" value="40264697" />\n    </lane>\n  </edge>\n  <edge id="40264697#7" from="2506085496" to="cluster_2052547073_2506085499" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center" shape="663.32,668.35 694.59,670.76">\n    <lane id="40264697#7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="20.52" shape="668.37,670.34 688.82,671.92">\n      <param key="origId" value="40264697" />\n    </lane>\n    <lane id="40264697#7_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="20.52" shape="668.61,667.15 689.07,668.73">\n      <param key="origId" value="40264697" />\n    </lane>\n    <param key="origTo" value="2052547073" />\n  </edge>\n  <edge id="40264697#9" from="cluster_2052547073_2506085499" to="1110416568" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center" shape="713.23,673.94 741.15,676.65">\n    <lane id="40264697#9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.68" shape="719.77,676.18 733.38,677.50">\n      <param key="origId" value="40264697" />\n    </lane>\n    <lane id="40264697#9_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.68" shape="720.08,673.00 733.69,674.32">\n      <param key="origId" value="40264697" />\n    </lane>\n    <param key="origFrom" value="2506085499" />\n  </edge>\n  <edge id="452036922" from="1123399561" to="4488368090" name="ถนนศุภสารรังสรรค์ ซอย 6" priority="10" type="highway.tertiary">\n    <lane id="452036922_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="55.64" shape="1182.52,1240.93 1182.03,1296.56">\n      <param key="origId" value="452036922" />\n    </lane>\n  </edge>\n  <edge id="452036923#0" from="5149825984" to="4488366488" priority="10" type="highway.tertiary" shape="1257.36,1181.54 1250.19,1194.55 1243.53,1202.35 1235.51,1209.07">\n    <lane id="452036923#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.89" shape="1251.33,1189.16 1248.87,1193.63 1242.40,1201.21 1238.71,1204.30">\n      <param key="origId" value="452036923" />\n    </lane>\n  </edge>\n  <edge id="452036923#1" from="4488366488" to="1123399561" priority="10" type="highway.tertiary" shape="1235.51,1209.07 1227.66,1214.23 1219.15,1218.30 1199.91,1226.91 1184.19,1233.15">\n    <lane id="452036923#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.47" shape="1226.35,1213.09 1218.48,1216.85 1199.29,1225.44 1189.32,1229.39">\n      <param key="origId" value="452036923" />\n    </lane>\n  </edge>\n  <edge id="452036923#2" from="1123399561" to="1123399535" priority="10" type="highway.tertiary" shape="1184.19,1233.15 1169.12,1235.59 1158.18,1236.60 1152.72,1236.78 1146.98,1236.67 1141.31,1236.36 1134.93,1234.89">\n    <lane id="452036923#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="33.27" shape="1176.24,1232.82 1168.92,1234.00 1158.08,1235.00 1152.71,1235.18 1147.04,1235.07 1143.13,1234.86">\n      <param key="origId" value="452036923" />\n    </lane>\n  </edge>\n  <edge id="452036923#3" from="1123399535" to="4488358857" priority="10" type="highway.tertiary" shape="1134.93,1234.89 1130.73,1233.25 1126.28,1231.49 1121.95,1229.39 1117.75,1226.81 1111.51,1222.97 1088.44,1204.94">\n    <lane id="452036923#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="38.78" shape="1129.74,1231.14 1126.92,1230.02 1122.72,1227.99 1118.59,1225.45 1112.42,1221.65 1097.46,1209.96">\n      <param key="origId" value="452036923" />\n    </lane>\n  </edge>\n  <edge id="452036923#4" from="4488358857" to="4488358858" priority="10" type="highway.tertiary" shape="1088.44,1204.94 1083.69,1202.42 1078.66,1200.32 1075.33,1199.49 1071.48,1198.93 1068.20,1198.97 1064.74,1199.20 1061.61,1200.21 1059.02,1202.36 1057.13,1204.39 1056.13,1206.60 1055.31,1210.02 1055.22,1213.45 1055.17,1220.66 1055.94,1249.27 1058.40,1261.02 1061.68,1271.35 1066.44,1280.27 1072.41,1289.19 1084.03,1302.21">\n    <lane id="452036923#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="127.27" shape="1083.84,1200.75 1079.16,1198.80 1075.64,1197.92 1071.59,1197.33 1068.14,1197.37 1064.44,1197.62 1060.83,1198.78 1057.92,1201.19 1055.78,1203.49 1054.61,1206.08 1053.71,1209.81 1053.62,1213.42 1053.57,1220.68 1054.34,1249.46 1056.85,1261.43 1060.20,1271.97 1065.07,1281.09 1071.14,1290.17 1074.94,1294.43">\n      <param key="origId" value="452036923" />\n    </lane>\n  </edge>\n  <edge id="452036923#5" from="4488358858" to="4488358856" priority="10" type="highway.tertiary" shape="1084.03,1302.21 1091.32,1308.84 1096.93,1313.11 1101.99,1315.60 1107.47,1317.15 1113.13,1317.98 1119.94,1318.25 1124.39,1317.88 1128.49,1316.63 1132.15,1314.69 1134.58,1312.92">\n    <lane id="452036923#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="47.75" shape="1087.02,1307.10 1090.30,1310.07 1096.08,1314.48 1101.41,1317.10 1107.13,1318.72 1112.98,1319.58 1119.97,1319.85 1124.69,1319.46 1129.10,1318.12 1130.79,1317.22">\n      <param key="origId" value="452036923" />\n    </lane>\n  </edge>\n  <edge id="452036923#6" from="4488358856" to="4488368090" priority="10" type="highway.tertiary" shape="1134.58,1312.92 1142.13,1306.92 1145.79,1304.31 1148.45,1303.06 1151.97,1302.69 1183.56,1303.87">\n    <lane id="452036923#6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="33.13" shape="1143.96,1307.58 1146.60,1305.70 1148.89,1304.62 1152.02,1304.29 1176.21,1305.20">\n      <param key="origId" value="452036923" />\n    </lane>\n  </edge>\n  <edge id="452036923#7" from="4488368090" to="1123399683" priority="10" type="highway.tertiary" shape="1183.56,1303.87 1210.44,1307.65 1223.52,1308.50 1229.97,1308.56 1236.41,1307.92">\n    <lane id="452036923#7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.50" shape="1190.08,1306.40 1210.28,1309.24 1223.46,1310.10 1229.35,1310.15">\n      <param key="origId" value="452036923" />\n    </lane>\n  </edge>\n  <edge id="452037066" from="1123399535" to="4488358856" name="ถนนศุภสารรังสรรค์ ซอย 7" priority="10" type="highway.tertiary">\n    <lane id="452037066_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="59.36" shape="1133.29,1242.89 1133.03,1302.24">\n      <param key="origId" value="452037066" />\n    </lane>\n  </edge>\n  <edge id="452037067" from="4488358857" to="4488358858" name="Thanon Supasarnrangsan Soi 8" priority="3" type="highway.residential">\n    <lane id="452037067_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="75.32" shape="1086.38,1215.06 1082.97,1290.30">\n      <param key="origId" value="452037067" />\n    </lane>\n  </edge>\n  <edge id="497290260" from="4888861605" to="4888862222" priority="10" type="highway.tertiary" spreadType="center" shape="1049.21,1700.55 1038.24,1710.51">\n    <lane id="497290260_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.23" shape="1049.02,1700.73 1042.18,1706.93">\n      <param key="origId" value="497290260" />\n    </lane>\n  </edge>\n  <edge id="497290261#1" from="4888862222" to="4888861608" priority="10" type="highway.tertiary" spreadType="center" shape="1038.24,1710.51 1036.31,1708.59 1033.86,1707.40 1031.27,1707.09 1028.70,1707.61">\n    <lane id="497290261#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="2.95" shape="1034.02,1707.48 1033.86,1707.40 1031.27,1707.09 1031.11,1707.12">\n      <param key="origId" value="497290261" />\n    </lane>\n  </edge>\n  <edge id="497290261#2" from="4888861608" to="1115585857" priority="10" type="highway.tertiary" spreadType="center" shape="1028.70,1707.61 1026.43,1708.92 1024.71,1710.87 1023.70,1713.27">\n    <lane id="497290261#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.20" shape="1025.89,1709.53 1025.76,1709.68">\n      <param key="origId" value="497290261" />\n    </lane>\n  </edge>\n  <edge id="497290261#3" from="1115585857" to="4888861618" priority="10" type="highway.tertiary" spreadType="center" shape="1023.70,1713.27 1023.50,1715.86 1024.16,1718.38 1025.58,1720.56">\n    <lane id="497290261#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="3.18" shape="1023.55,1715.17 1023.50,1715.86 1024.13,1718.27">\n      <param key="origId" value="497290261" />\n    </lane>\n  </edge>\n  <edge id="497290261#4" from="4888861618" to="4888861612" priority="10" type="highway.tertiary" spreadType="center" shape="1025.58,1720.56 1027.63,1722.18 1030.09,1723.06 1032.65,1723.18">\n    <lane id="497290261#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.20" shape="1028.35,1722.44 1028.54,1722.50">\n      <param key="origId" value="497290261" />\n    </lane>\n  </edge>\n  <edge id="497290261#5" from="4888861612" to="4888861620" priority="10" type="highway.tertiary" spreadType="center" shape="1032.65,1723.18 1035.42,1722.25 1037.57,1720.59 1039.05,1718.30 1039.68,1715.67">\n    <lane id="497290261#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="7.39" shape="1034.40,1722.59 1035.42,1722.25 1037.57,1720.59 1039.05,1718.30 1039.25,1717.45">\n      <param key="origId" value="497290261" />\n    </lane>\n  </edge>\n  <edge id="497290261#7" from="4888861620" to="4888862222" priority="10" type="highway.tertiary" spreadType="center" shape="1039.68,1715.67 1039.40,1712.97 1038.24,1710.51">\n    <lane id="497290261#7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.20" shape="1039.35,1712.85 1039.26,1712.67">\n      <param key="origId" value="497290261" />\n    </lane>\n  </edge>\n  <edge id="497290262" from="4888861608" to="2506089779" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="497290262_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.20" shape="1025.53,1705.31 1025.36,1705.19">\n      <param key="origId" value="497290262" />\n    </lane>\n  </edge>\n  <edge id="497290263#0" from="4888861605" to="2506089600" name="Thanon Jirnakorn" priority="10" type="highway.tertiary" shape="1050.15,1701.84 1065.23,1688.05 1117.29,1676.07">\n    <lane id="497290263#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="64.50" shape="1051.39,1702.88 1066.00,1689.51 1109.56,1679.49">\n      <param key="origId" value="497290263" />\n    </lane>\n  </edge>\n  <edge id="497290263#1" from="2506089600" to="2506089545" name="Thanon Jirnakorn" priority="10" type="highway.tertiary">\n    <lane id="497290263#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.61" shape="1123.91,1676.33 1163.68,1668.07">\n      <param key="origId" value="497290263" />\n    </lane>\n  </edge>\n  <edge id="497290263#2" from="2506089545" to="1123399713" name="Thanon Jirnakorn" priority="10" type="highway.tertiary">\n    <lane id="497290263#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="35.03" shape="1178.09,1665.11 1212.44,1658.22">\n      <param key="origId" value="497290263" />\n    </lane>\n  </edge>\n  <edge id="497290263#3" from="1123399713" to="1123399740" name="Thanon Jirnakorn" priority="10" type="highway.tertiary">\n    <lane id="497290263#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="47.12" shape="1227.07,1655.38 1273.41,1646.88">\n      <param key="origId" value="497290263" />\n    </lane>\n  </edge>\n  <edge id="497290263#4" from="1123399740" to="1110416742" name="Thanon Jirnakorn" priority="10" type="highway.tertiary">\n    <lane id="497290263#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="48.86" shape="1288.00,1644.05 1335.81,1634.02">\n      <param key="origId" value="497290263" />\n    </lane>\n  </edge>\n  <edge id="497290264" from="4888861620" to="4888861605" priority="10" type="highway.tertiary" spreadType="center" shape="1039.68,1715.67 1043.24,1710.96 1051.43,1702.81">\n    <lane id="497290264_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.91" shape="1042.79,1711.55 1043.24,1710.96 1051.16,1703.08">\n      <param key="origId" value="497290264" />\n    </lane>\n  </edge>\n  <edge id="497290265" from="4888861618" to="4888861612" priority="10" type="highway.tertiary" spreadType="center" shape="1025.58,1720.56 1020.98,1734.11 1032.65,1723.18">\n    <lane id="497290265_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="21.13" shape="1024.23,1724.55 1020.98,1734.11 1029.03,1726.57">\n      <param key="origId" value="497290265 497290266" />\n    </lane>\n  </edge>\n  <edge id="497290269" from="1123399676" to="2506089779" priority="10" type="highway.tertiary" shape="995.81,1681.63 997.91,1683.43 999.80,1685.05 1017.51,1699.50">\n    <lane id="497290269_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.65" shape="1000.31,1687.53 1012.44,1697.43">\n      <param key="origId" value="497290269 243188298" />\n    </lane>\n  </edge>\n  <edge id="497371052#0" from="4889402217" to="13362936997" priority="10" type="highway.tertiary" spreadType="center" shape="1380.35,-0.00 1397.50,133.22 1405.24,180.33 1427.80,314.39 1434.73,353.45">\n    <lane id="497371052#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="349.31" shape="1380.35,-0.00 1397.50,133.22 1405.24,180.33 1427.80,314.39 1433.27,345.22">\n      <param key="origId" value="497371052" />\n    </lane>\n  </edge>\n  <edge id="498671328#1" from="14179759489" to="4900465706" name="ถนนแสงศรี ซอย 1" priority="3" type="highway.residential">\n    <lane id="498671328#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="47.56" shape="396.29,1439.49 399.77,1486.92">\n      <param key="origId" value="498671328" />\n    </lane>\n  </edge>\n  <edge id="498671329" from="4900466295" to="2506088792" name="ถนนพูลสุวรรณ" priority="3" type="highway.residential" spreadType="center" shape="378.86,1582.41 313.73,1583.47 229.94,1577.75 206.91,1577.60 165.48,1577.99 13.76,1577.09">\n    <lane id="498671329_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="359.01" shape="372.56,1582.51 313.73,1583.47 229.94,1577.75 206.91,1577.60 165.48,1577.99 13.76,1577.09">\n      <param key="origId" value="498671329" />\n    </lane>\n  </edge>\n  <edge id="498671330#0" from="4900465092" to="4900465090" name="ถนนมุสลิม" priority="3" type="highway.residential" spreadType="center" shape="6.18,1622.70 155.37,1640.04 159.87,1640.82 174.08,1642.50 199.78,1645.55 315.46,1661.39">\n    <lane id="498671330#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="307.08" shape="6.18,1622.70 155.37,1640.04 159.87,1640.82 174.08,1642.50 199.78,1645.55 310.87,1660.76">\n      <param key="origId" value="498671330" />\n    </lane>\n  </edge>\n  <edge id="498671330#1" from="4900465090" to="4900466297" name="ถนนมุสลิม" priority="3" type="highway.residential" spreadType="center" shape="315.46,1661.39 367.85,1668.53 411.81,1673.36">\n    <lane id="498671330#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="87.26" shape="320.19,1662.04 367.85,1668.53 406.78,1672.81">\n      <param key="origId" value="498671330" />\n    </lane>\n  </edge>\n  <edge id="498671331" from="2506088494" to="2506088476" priority="3" type="highway.residential" shape="503.67,1493.51 495.75,1481.42 466.68,1467.12">\n    <lane id="498671331_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.13" shape="501.01,1486.53 496.85,1480.18 473.03,1468.46">\n      <param key="origId" value="498671331" />\n    </lane>\n  </edge>\n  <edge id="498671388#0" from="2506086831" to="4900466448" name="Thanon Prachathipat 2" priority="3" type="highway.residential">\n    <lane id="498671388#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="78.40" shape="880.22,1138.72 886.79,1060.59">\n      <param key="origId" value="498671388" />\n    </lane>\n  </edge>\n  <edge id="498671388#1" from="4900466448" to="856214538" name="Thanon Prachathipat 2" priority="3" type="highway.residential">\n    <lane id="498671388#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="67.14" shape="888.38,1046.14 897.37,979.60">\n      <param key="origId" value="498671388" />\n    </lane>\n  </edge>\n  <edge id="498671389#0" from="4900464597" to="4900464580" name="Thanon Boonrong Uthit" priority="3" type="highway.residential" shape="969.86,1153.94 970.92,1121.40 971.45,1088.48 971.51,1085.16 969.09,1061.43">\n    <lane id="498671389#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="77.53" shape="971.70,1146.61 972.52,1121.44 973.05,1088.51 973.11,1085.09 971.49,1069.18">\n      <param key="origId" value="498671389" />\n    </lane>\n  </edge>\n  <edge id="498671389#2" from="4900464580" to="4900464598" name="Thanon Boonrong Uthit" priority="3" type="highway.residential">\n    <lane id="498671389#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="70.72" shape="970.68,1053.84 970.58,983.12">\n      <param key="origId" value="498671389" />\n    </lane>\n  </edge>\n  <edge id="498671391" from="4900464587" to="1123399771" priority="3" type="highway.residential" shape="1241.28,1134.45 1136.28,1124.51 1060.17,1111.10">\n    <lane id="498671391_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="169.45" shape="1233.82,1132.14 1136.49,1122.92 1065.89,1110.48">\n      <param key="origId" value="498671391" />\n    </lane>\n  </edge>\n  <edge id="498671393" from="4900465166" to="1115585909" priority="3" type="highway.residential">\n    <lane id="498671393_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="93.45" shape="907.04,920.66 1000.45,923.37">\n      <param key="origId" value="498671393" />\n    </lane>\n  </edge>\n  <edge id="498671395#0" from="1115585385" to="1966404349" priority="3" type="highway.residential" spreadType="center">\n    <lane id="498671395#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="75.10" shape="1029.14,985.31 1027.36,1060.39">\n      <param key="origId" value="498671395" />\n    </lane>\n  </edge>\n  <edge id="498671395#1" from="1966404349" to="cluster_1115585393_13472440789_947896971" priority="3" type="highway.residential" spreadType="center" shape="1027.19,1067.73 1025.86,1161.53">\n    <lane id="498671395#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="76.23" shape="1027.09,1075.08 1026.00,1151.31">\n      <param key="origId" value="498671395" />\n    </lane>\n    <param key="origTo" value="13472440789" />\n  </edge>\n  <edge id="498671427" from="1107926467" to="4900465052" name="ซอยทินกร" priority="3" type="highway.residential">\n    <lane id="498671427_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="99.47" shape="1516.41,1686.43 1614.54,1670.12">\n      <param key="origId" value="498671427" />\n    </lane>\n  </edge>\n  <edge id="498671432" from="4900465067" to="4900465069" priority="3" type="highway.residential" shape="1318.32,1734.52 1316.22,1690.21 1346.02,1689.19">\n    <lane id="498671432_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="63.68" shape="1319.92,1734.44 1317.89,1691.75 1338.83,1691.04">\n      <param key="origId" value="498671432" />\n    </lane>\n  </edge>\n  <edge id="498671433" from="4900465070" to="4900465071" priority="3" type="highway.residential">\n    <lane id="498671433_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.86" shape="1671.05,535.54 1632.24,526.44">\n      <param key="origId" value="498671433" />\n    </lane>\n  </edge>\n  <edge id="498671442" from="4900465098" to="4900465099" priority="3" type="highway.residential">\n    <lane id="498671442_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="95.51" shape="615.95,922.95 581.18,1011.91">\n      <param key="origId" value="498671442" />\n    </lane>\n  </edge>\n  <edge id="498671465" from="4900465166" to="12649445300" priority="3" type="highway.residential">\n    <lane id="498671465_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="74.72" shape="892.67,916.65 818.22,910.39">\n      <param key="origId" value="498671465" />\n    </lane>\n  </edge>\n  <edge id="498671482" from="4900465208" to="4900465209" priority="3" type="highway.residential">\n    <lane id="498671482_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="71.90" shape="584.38,508.42 575.49,579.78">\n      <param key="origId" value="498671482" />\n    </lane>\n  </edge>\n  <edge id="498671483" from="cluster_1559016910_4900465210" to="1680666686" priority="3" type="highway.residential" shape="872.26,410.02 924.52,415.08">\n    <lane id="498671483_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="37.86" shape="879.45,412.32 917.14,415.97">\n      <param key="origId" value="498671483" />\n    </lane>\n    <param key="origFrom" value="4900465210" />\n  </edge>\n  <edge id="498671497" from="cluster_1558314142_1558314144" to="4900465244" priority="3" type="highway.residential" shape="1366.65,703.92 1372.20,649.40">\n    <lane id="498671497_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.42" shape="1368.93,697.29 1373.03,657.09">\n      <param key="origId" value="498671497" />\n    </lane>\n    <param key="origFrom" value="1558314142" />\n  </edge>\n  <edge id="498671536" from="2506087432" to="4900465363" name="Soi 1/1 Sophasarnrungsen" priority="3" type="highway.residential">\n    <lane id="498671536_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="64.17" shape="634.64,1226.45 570.49,1224.97">\n      <param key="origId" value="498671536" />\n    </lane>\n  </edge>\n  <edge id="498671539" from="4900465090" to="4900465370" priority="3" type="highway.residential" shape="315.46,1661.39 322.01,1625.68 352.76,1624.28">\n    <lane id="498671539_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="61.29" shape="317.62,1658.48 323.35,1627.22 352.83,1625.88">\n      <param key="origId" value="498671539" />\n    </lane>\n  </edge>\n  <edge id="498671542" from="2507400609" to="4900465377" priority="3" type="highway.residential">\n    <lane id="498671542_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.84" shape="1738.69,873.32 1736.33,893.02">\n      <param key="origId" value="498671542" />\n    </lane>\n  </edge>\n  <edge id="498671544#0" from="1140280757" to="13424210990" priority="3" type="highway.residential" shape="691.90,1129.98 700.57,1090.15 704.66,1072.26 709.31,1046.68">\n    <lane id="498671544#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="77.51" shape="695.09,1122.84 702.13,1090.50 706.23,1072.58 710.88,1046.97">\n      <param key="origId" value="498671544" />\n    </lane>\n  </edge>\n  <edge id="498671564" from="4900465440" to="4900465441" priority="3" type="highway.residential">\n    <lane id="498671564_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.04" shape="1635.88,1313.10 1634.21,1342.09">\n      <param key="origId" value="498671564" />\n    </lane>\n  </edge>\n  <edge id="498671569" from="4900465451" to="2506087136" priority="3" type="highway.residential">\n    <lane id="498671569_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="43.44" shape="1285.00,1129.87 1284.02,1173.30">\n      <param key="origId" value="498671569" />\n    </lane>\n  </edge>\n  <edge id="498671576#0" from="4900465471" to="4900465472" priority="3" type="highway.residential">\n    <lane id="498671576#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="42.34" shape="603.29,720.28 645.42,724.50">\n      <param key="origId" value="498671576" />\n    </lane>\n  </edge>\n  <edge id="498671576#1" from="4900465472" to="4900465473" priority="3" type="highway.residential" shape="651.55,723.51 678.85,727.66 698.50,730.13">\n    <lane id="498671576#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.11" shape="657.04,725.96 678.63,729.24 690.81,730.78">\n      <param key="origId" value="498671576" />\n    </lane>\n  </edge>\n  <edge id="498671586" from="8523344319" to="4900465513" priority="10" type="highway.tertiary" spreadType="center" shape="448.19,845.90 440.34,889.17 457.32,892.46 445.63,946.95 437.07,986.83">\n    <lane id="498671586_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="146.56" shape="447.19,851.41 440.34,889.17 457.32,892.46 445.63,946.95 438.25,981.32">\n      <param key="origId" value="498671586" />\n    </lane>\n  </edge>\n  <edge id="498671589" from="4900465517" to="2506088491" name="Thanon Sangsri Soi 3" priority="3" type="highway.residential">\n    <lane id="498671589_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="95.79" shape="541.23,1388.89 533.18,1484.34">\n      <param key="origId" value="498671589" />\n    </lane>\n  </edge>\n  <edge id="498671600" from="1558314095" to="4900465641" priority="3" type="highway.residential" shape="1697.89,309.43 1683.14,362.56 1675.64,380.29">\n    <lane id="498671600_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="66.56" shape="1694.31,316.34 1681.63,362.03 1674.17,379.67">\n      <param key="origId" value="498671600" />\n    </lane>\n  </edge>\n  <edge id="498671615" from="4900465999" to="1123399600" priority="3" type="highway.residential" shape="1047.32,916.15 1051.62,927.71 1084.18,985.09">\n    <lane id="498671615_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="60.17" shape="1048.58,924.12 1050.17,928.39 1077.61,976.76">\n      <param key="origId" value="498671615" />\n    </lane>\n  </edge>\n  <edge id="498671616" from="4900465999" to="2506085631" name="ถนนประชาธิปัตย์ ซอย 3" priority="10" type="highway.tertiary">\n    <lane id="498671616_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="162.71" shape="1055.38,916.49 1216.09,891.03">\n      <param key="origId" value="498671616" />\n    </lane>\n  </edge>\n  <edge id="498671639" from="2506085631" to="4900465707" priority="3" type="highway.residential" spreadType="center">\n    <lane id="498671639_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="82.77" shape="1222.18,895.86 1228.24,978.41">\n      <param key="origId" value="498671639" />\n    </lane>\n  </edge>\n  <edge id="498671673" from="4900465989" to="2506087522" priority="3" type="highway.residential" shape="381.04,1293.97 373.69,1273.98 360.41,1242.41">\n    <lane id="498671673_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="41.30" shape="379.39,1284.85 375.18,1273.39 363.90,1246.58">\n      <param key="origId" value="498671673" />\n    </lane>\n  </edge>\n  <edge id="498671676#0" from="12346099054" to="2506088024" priority="3" type="highway.residential" spreadType="center" shape="1017.99,1359.35 1012.74,1358.57 1007.68,1358.50 1000.35,1359.07 990.07,1360.35 983.05,1361.87 973.76,1364.64 965.18,1368.16 956.24,1372.26 946.66,1377.02 939.55,1381.65 934.07,1385.27 931.70,1387.11 929.61,1389.25 927.72,1391.91 926.45,1394.74 922.55,1407.94 916.38,1430.86 913.56,1457.00 912.16,1463.97 911.11,1467.54 909.42,1471.99 907.24,1475.44 904.44,1479.77 902.39,1482.31 899.45,1484.83 895.54,1486.74 891.65,1487.77 882.60,1489.78 872.63,1486.27 865.11,1478.52 856.54,1466.94 849.31,1455.88 830.55,1428.97 819.72,1414.43 814.19,1409.11 810.50,1406.51 803.69,1402.31 793.04,1396.82 775.02,1388.94">\n    <lane id="498671676#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="349.48" shape="1010.63,1358.54 1007.68,1358.50 1000.35,1359.07 990.07,1360.35 983.05,1361.87 973.76,1364.64 965.18,1368.16 956.24,1372.26 946.66,1377.02 939.55,1381.65 934.07,1385.27 931.70,1387.11 929.61,1389.25 927.72,1391.91 926.45,1394.74 922.55,1407.94 916.38,1430.86 913.56,1457.00 912.16,1463.97 911.11,1467.54 909.42,1471.99 907.24,1475.44 904.44,1479.77 902.39,1482.31 899.45,1484.83 895.54,1486.74 891.65,1487.77 882.60,1489.78 872.63,1486.27 865.11,1478.52 856.54,1466.94 849.31,1455.88 830.55,1428.97 819.72,1414.43 814.19,1409.11 810.50,1406.51 803.69,1402.31 793.04,1396.82 784.58,1393.12">\n      <param key="origId" value="498671676" />\n    </lane>\n  </edge>\n  <edge id="498671679" from="4900465219" to="4900466002" priority="3" type="highway.residential" shape="1149.97,555.31 1146.01,606.92 1197.37,611.04">\n    <lane id="498671679_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="99.20" shape="1147.82,562.44 1144.29,608.39 1197.24,612.63">\n      <param key="origId" value="498671679" />\n    </lane>\n  </edge>\n  <edge id="498671694" from="4900465030" to="4900465031" priority="3" type="highway.residential">\n    <lane id="498671694_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.07" shape="1773.82,426.78 1737.25,410.42">\n      <param key="origId" value="498671694" />\n    </lane>\n  </edge>\n  <edge id="498671702" from="12339345632" to="12346099051" priority="3" type="highway.residential" shape="754.70,1392.49 756.61,1404.38 752.03,1411.12 724.09,1434.01 699.39,1451.16 687.10,1460.41 667.30,1472.52 642.40,1482.58 623.24,1487.51 566.22,1493.89">\n    <lane id="498671702_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="212.91" shape="754.28,1399.96 754.93,1404.01 750.84,1410.03 723.13,1432.73 698.45,1449.86 686.20,1459.09 666.58,1471.09 641.90,1481.06 622.95,1485.93 572.13,1491.62">\n      <param key="origId" value="498671702 243188389" />\n    </lane>\n  </edge>\n  <edge id="530314558#1" from="4900464587" to="1123399639" priority="3" type="highway.residential">\n    <lane id="530314558#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.15" shape="1238.02,1141.55 1231.09,1171.92">\n      <param key="origId" value="530314558" />\n    </lane>\n  </edge>\n  <edge id="65080358#0" from="856214571" to="cluster_2505176422_2505176434" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1544.12,1202.38 1580.76,1427.50">\n    <lane id="65080358#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="210.96" shape="1545.80,1212.70 1579.69,1420.92">\n      <param key="origId" value="65080358" />\n    </lane>\n    <param key="origTo" value="2505176434" />\n  </edge>\n  <edge id="65080358#1" from="cluster_2505176422_2505176434" to="2507400748" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1580.76,1427.50 1609.91,1606.57 1612.23,1620.78">\n    <lane id="65080358#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="180.89" shape="1582.00,1435.13 1609.91,1606.57 1611.07,1613.67">\n      <param key="origId" value="65080358" />\n    </lane>\n    <param key="origFrom" value="2505176434" />\n  </edge>\n  <edge id="65080358#2" from="2507400748" to="4900465052" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center">\n    <lane id="65080358#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="33.06" shape="1613.38,1627.88 1618.66,1660.52">\n      <param key="origId" value="65080358" />\n    </lane>\n  </edge>\n  <edge id="65080358#3" from="4900465052" to="1680668226" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center">\n    <lane id="65080358#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="26.92" shape="1620.97,1674.73 1625.33,1701.29">\n      <param key="origId" value="65080358" />\n    </lane>\n  </edge>\n  <edge id="65080358#4" from="1680668226" to="1680668239" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1626.47,1708.28 1656.62,1893.50 1671.32,1970.82">\n    <lane id="65080358#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="259.02" shape="1627.65,1715.52 1656.62,1893.50 1671.32,1970.82">\n      <param key="origId" value="65080358" />\n    </lane>\n  </edge>\n  <edge id="66984604#1" from="2406954575" to="809537820" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" spreadType="center" shape="1989.24,1153.73 1973.63,1154.76 1935.52,1155.38 1850.45,1156.53 1802.10,1156.89 1566.11,1164.78">\n    <lane id="66984604#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="418.04" shape="1989.24,1153.73 1973.63,1154.76 1935.52,1155.38 1850.45,1156.53 1802.10,1156.89 1571.37,1164.60">\n      <param key="origId" value="66984604" />\n    </lane>\n  </edge>\n  <edge id="66984604#2" from="809537820" to="cluster_11292971032_2406954578_2406954579_2406954580" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" spreadType="center" shape="1566.11,1164.78 1562.23,1165.22 1555.46,1166.50 1550.56,1167.43">\n    <lane id="66984604#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="3.38" shape="1560.98,1165.46 1557.66,1166.08">\n      <param key="origId" value="66984604 1218747877 1218747876" />\n    </lane>\n    <param key="origTo" value="2406954578" />\n  </edge>\n  <edge id="66984610#0" from="856214571" to="2505176258" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="66984610#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.20" shape="1549.29,1203.38 1549.49,1203.35">\n      <param key="origId" value="66984610" />\n    </lane>\n  </edge>\n  <edge id="66984610#1" from="2505176258" to="2505176265" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="1555.83,1200.99 1561.19,1200.40 1566.49,1199.90 1570.92,1199.46">\n    <lane id="66984610#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="4.57" shape="1563.34,1201.80 1566.64,1201.49 1567.88,1201.37">\n      <param key="origId" value="66984610 1218747878 1218747879" />\n    </lane>\n  </edge>\n  <edge id="66984613#0" from="947896981" to="947896972" name="ถนนราษฎร์ยินดี 2" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="66984613#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="121.06" shape="1651.13,1696.23 1631.20,1576.82">\n      <param key="origId" value="66984613" />\n    </lane>\n  </edge>\n  <edge id="66984613#1" from="947896972" to="947896989" name="ถนนราษฎร์ยินดี 2" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="66984613#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="61.62" shape="1628.81,1562.46 1618.79,1501.67">\n      <param key="origId" value="66984613" />\n    </lane>\n  </edge>\n  <edge id="66984613#2" from="947896989" to="947896979" name="ถนนราษฎร์ยินดี 2" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="66984613#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.15" shape="1616.17,1486.76 1614.50,1477.76">\n      <param key="origId" value="66984613" />\n    </lane>\n  </edge>\n  <edge id="66984613#3" from="947896979" to="2505176388" name="ถนนราษฎร์ยินดี 2" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="66984613#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="36.99" shape="1612.92,1468.50 1607.20,1431.95">\n      <param key="origId" value="66984613" />\n    </lane>\n  </edge>\n  <edge id="66984613#4" from="2505176388" to="2507400696" name="ถนนราษฎร์ยินดี 2" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="66984613#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="63.76" shape="1604.90,1417.26 1594.97,1354.28">\n      <param key="origId" value="66984613" />\n    </lane>\n  </edge>\n  <edge id="66984613#5" from="2507400696" to="2507400689" name="ถนนราษฎร์ยินดี 2" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="66984613#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="50.25" shape="1592.63,1339.45 1584.81,1289.80">\n      <param key="origId" value="66984613" />\n    </lane>\n  </edge>\n  <edge id="66984613#6" from="2507400689" to="947896958" name="ถนนราษฎร์ยินดี 2" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="66984613#6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="45.91" shape="1582.48,1275.00 1575.33,1229.65">\n      <param key="origId" value="66984613" />\n    </lane>\n  </edge>\n  <edge id="66984613#7" from="947896958" to="2505176265" name="ถนนราษฎร์ยินดี 2" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="66984613#7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.62" shape="1573.65,1218.52 1571.58,1204.05">\n      <param key="origId" value="66984613" />\n    </lane>\n  </edge>\n  <edge id="684704215#0" from="1431715369" to="617234535" name="Thanon Manasruedee" priority="3" type="highway.residential" spreadType="center">\n    <lane id="684704215#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="55.97" shape="407.56,494.16 351.97,487.60">\n      <param key="origId" value="684704215" />\n    </lane>\n  </edge>\n  <edge id="684704215#1" from="617234535" to="620881544" name="Thanon Manasruedee" priority="3" type="highway.residential" spreadType="center">\n    <lane id="684704215#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="68.06" shape="334.55,485.52 266.97,477.39">\n      <param key="origId" value="684704215" />\n    </lane>\n  </edge>\n  <edge id="684714761#0" from="2506085784" to="856214578" name="Duangchan Road" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="684714761#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="54.31" shape="278.20,957.28 331.90,965.41">\n      <param key="origId" value="684714761" />\n    </lane>\n  </edge>\n  <edge id="684714761#1" from="856214578" to="4900465513" name="Duangchan Road" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="684714761#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="83.89" shape="349.47,968.58 431.60,985.69">\n      <param key="origId" value="684714761" />\n    </lane>\n  </edge>\n  <edge id="684714761#2" from="4900465513" to="2506085849" name="Duangchan Road" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="684714761#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="22.50" shape="442.61,987.89 464.70,992.13">\n      <param key="origId" value="684714761" />\n    </lane>\n  </edge>\n  <edge id="684714761#3" from="2506085849" to="1107784879" name="Duangchan Road" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="684714761#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="51.67" shape="478.83,994.89 529.49,1005.02">\n      <param key="origId" value="684714761" />\n    </lane>\n  </edge>\n  <edge id="684740716" from="1123399489" to="cluster_1123399715_3376084911_3376091818" priority="3" type="highway.residential" shape="1044.60,1401.15 1034.14,1397.73 1028.07,1396.94">\n    <lane id="684740716_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.28" shape="1041.16,1398.34 1034.49,1396.16 1028.28,1395.35">\n      <param key="origId" value="684740716 684741338" />\n    </lane>\n    <param key="origTo" value="3376084911" />\n  </edge>\n  <edge id="684741141#0" from="1123399591" to="1123399675" priority="3" type="highway.residential" shape="572.08,1715.62 557.04,1704.41 543.92,1689.30 535.40,1673.45">\n    <lane id="684741141#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="38.24" shape="565.00,1708.35 558.13,1703.23 545.24,1688.39 540.50,1679.56">\n      <param key="origId" value="684741141" />\n    </lane>\n  </edge>\n  <edge id="684741141#1" from="1123399675" to="1123399943" priority="3" type="highway.residential">\n    <lane id="684741141#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.65" shape="533.77,1666.70 519.91,1639.36">\n      <param key="origId" value="684741141" />\n    </lane>\n  </edge>\n  <edge id="684741141#2" from="1123399943" to="1123399486" priority="3" type="highway.residential">\n    <lane id="684741141#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.46" shape="513.90,1625.97 509.08,1613.41">\n      <param key="origId" value="684741141" />\n    </lane>\n  </edge>\n  <edge id="684741141#3" from="1123399486" to="1123399502" priority="3" type="highway.residential" shape="504.75,1606.61 498.85,1577.34 496.34,1551.16 499.02,1530.34 503.98,1523.23 513.77,1518.80 530.57,1518.11 565.91,1520.01">\n    <lane id="684741141#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="131.48" shape="504.91,1599.32 500.43,1577.10 497.95,1551.19 500.56,1530.93 505.04,1524.51 514.15,1520.39 530.56,1519.71 558.61,1521.22">\n      <param key="origId" value="684741141" />\n    </lane>\n  </edge>\n  <edge id="705855017#1" from="4900465949" to="1431867108" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="705855017#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="84.60" shape="1451.09,451.09 1437.00,367.68">\n      <param key="origId" value="705855017" />\n    </lane>\n  </edge>\n  <edge id="706087699#1" from="4900465031" to="1431867103" priority="3" type="highway.residential">\n    <lane id="706087699#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="28.55" shape="1725.91,415.67 1717.93,443.08">\n      <param key="origId" value="706087699" />\n    </lane>\n  </edge>\n  <edge id="706092523#0" from="1110416444" to="856214470" name="Thanon Pracharak" priority="10" type="highway.tertiary" spreadType="center" shape="644.04,1123.85 656.53,1081.86 669.32,1032.15">\n    <lane id="706092523#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="82.20" shape="646.18,1116.66 656.53,1081.86 667.97,1037.41">\n      <param key="origId" value="706092523" />\n    </lane>\n  </edge>\n  <edge id="706092523#2" from="856214470" to="856214503" name="Thanon Pracharak" priority="10" type="highway.tertiary" spreadType="center" shape="669.32,1032.15 683.43,980.68 688.39,962.84 702.16,913.43">\n    <lane id="706092523#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="110.81" shape="670.71,1027.08 683.43,980.68 688.39,962.84 700.25,920.28">\n      <param key="origId" value="706092523" />\n    </lane>\n  </edge>\n  <edge id="706092523#4" from="856214503" to="2052489649" name="Thanon Pracharak" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="706092523#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="100.25" shape="703.94,906.30 728.26,809.04">\n      <param key="origId" value="706092523" />\n    </lane>\n  </edge>\n  <edge id="706093504#0" from="1966386254" to="1107784879" name="Thanon Sangchan" priority="3" type="highway.residential" spreadType="center" shape="564.09,872.41 542.54,970.86 534.83,1006.09">\n    <lane id="706093504#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="125.95" shape="562.93,877.70 542.54,970.86 536.00,1000.74">\n      <param key="origId" value="706093504" />\n    </lane>\n  </edge>\n  <edge id="706093504#1" from="1107784879" to="12344163778" name="Thanon Sangchan" priority="3" type="highway.residential" spreadType="center">\n    <lane id="706093504#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="25.46" shape="533.92,1011.56 529.73,1036.68">\n      <param key="origId" value="706093504" />\n    </lane>\n  </edge>\n  <edge id="706093504#2" from="12344163778" to="1107784887" name="Thanon Sangchan" priority="3" type="highway.residential" spreadType="center" shape="528.97,1041.26 523.64,1073.32 519.48,1095.16">\n    <lane id="706093504#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="42.49" shape="528.19,1045.96 523.64,1073.32 520.88,1087.81">\n      <param key="origId" value="706093504" />\n    </lane>\n  </edge>\n  <edge id="706093505#0" from="1107784879" to="4900465099" name="Duangchan Road" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="706093505#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="37.48" shape="540.26,1007.25 576.90,1015.12">\n      <param key="origId" value="706093505" />\n    </lane>\n  </edge>\n  <edge id="706093505#1" from="4900465099" to="8523334139" name="Duangchan Road" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="706093505#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="36.92" shape="586.34,1016.87 622.79,1022.70">\n      <param key="origId" value="706093505" />\n    </lane>\n  </edge>\n  <edge id="706093505#2" from="8523334139" to="856214470" name="Duangchan Road" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="706093505#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="31.88" shape="632.79,1024.55 664.00,1031.04">\n      <param key="origId" value="706093505" />\n    </lane>\n  </edge>\n  <edge id="706093506#0" from="1107784887" to="2052497441" name="Thanon Sangchan" priority="3" type="highway.residential" spreadType="center">\n    <lane id="706093506#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="42.27" shape="518.17,1102.41 510.66,1144.01">\n      <param key="origId" value="706093506" />\n    </lane>\n  </edge>\n  <edge id="706093506#1" from="2052497441" to="4900465364" name="Thanon Sangchan" priority="3" type="highway.residential" spreadType="center">\n    <lane id="706093506#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="60.53" shape="508.74,1155.03 498.76,1214.73">\n      <param key="origId" value="706093506" />\n    </lane>\n  </edge>\n  <edge id="706093506#2" from="4900465364" to="2506087810" name="Thanon Sangchan" priority="3" type="highway.residential" spreadType="center" shape="497.83,1220.27 493.11,1259.61 488.55,1297.57">\n    <lane id="706093506#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="66.84" shape="497.18,1225.68 493.11,1259.61 489.21,1292.04">\n      <param key="origId" value="706093506" />\n    </lane>\n  </edge>\n  <edge id="706093506#3" from="2506087810" to="947896956" name="Thanon Sangchan" priority="3" type="highway.residential" spreadType="center">\n    <lane id="706093506#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="65.91" shape="487.84,1303.10 479.43,1368.47">\n      <param key="origId" value="706093506" />\n    </lane>\n  </edge>\n  <edge id="706093508#0" from="1966386254" to="8523344319" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" spreadType="center" shape="564.09,872.41 507.86,856.14 448.19,845.90">\n    <lane id="706093508#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="109.85" shape="560.56,871.39 507.86,856.14 453.66,846.84">\n      <param key="origId" value="706093508" />\n    </lane>\n  </edge>\n  <edge id="706093508#1" from="8523344319" to="617234843" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="706093508#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.20" shape="446.43,845.62 446.23,845.59">\n      <param key="origId" value="706093508" />\n    </lane>\n  </edge>\n  <edge id="706093508#2" from="617234843" to="485580842" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="706093508#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="70.33" shape="439.20,844.75 369.27,837.28">\n      <param key="origId" value="706093508" />\n    </lane>\n  </edge>\n  <edge id="706093508#3" from="485580842" to="617233014" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="706093508#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="50.49" shape="351.66,835.33 301.50,829.59">\n      <param key="origId" value="706093508" />\n    </lane>\n  </edge>\n  <edge id="706093509#0" from="1107784884" to="4900465471" name="Thanon Sangchan" priority="3" type="highway.residential" spreadType="center">\n    <lane id="706093509#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="44.44" shape="609.67,669.99 601.00,713.57">\n      <param key="origId" value="706093509" />\n    </lane>\n  </edge>\n  <edge id="706093509#1" from="4900465471" to="1966386254" name="Thanon Sangchan" priority="3" type="highway.residential" spreadType="center" shape="600.05,718.35 581.17,817.52 572.03,865.50 564.09,872.41">\n    <lane id="706093509#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="148.52" shape="599.20,722.84 581.17,817.52 572.03,865.50 569.54,867.67">\n      <param key="origId" value="706093509" />\n    </lane>\n  </edge>\n  <edge id="706094740#0" from="1107926433" to="cluster_1558314142_1558314144" name="ถนนสามชัย" priority="3" type="highway.residential" shape="1384.91,735.44 1386.60,710.02">\n    <lane id="706094740#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.67" shape="1386.89,729.82 1387.73,717.17">\n      <param key="origId" value="706094740" />\n    </lane>\n    <param key="origTo" value="1558314144" />\n  </edge>\n  <edge id="706094740#2" from="cluster_1558314142_1558314144" to="2506085508" name="ถนนสามชัย" priority="3" type="highway.residential" shape="1366.65,703.92 1302.34,705.45 1231.94,700.19 1196.41,694.94 1184.44,688.87">\n    <lane id="706094740#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="169.07" shape="1359.79,702.48 1302.38,703.85 1232.12,698.60 1196.90,693.40 1191.92,690.87">\n      <param key="origId" value="706094740" />\n    </lane>\n    <param key="origFrom" value="1558314142" />\n  </edge>\n  <edge id="706094741#0" from="1123399516" to="1123399641" name="Thanon Supasarnrangsan Soi 4/1" priority="3" type="highway.residential" shape="1239.67,1387.29 1238.72,1389.96 1237.18,1392.48 1233.61,1396.60 1230.06,1400.05 1225.96,1403.23 1199.28,1419.81">\n    <lane id="706094741#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.15" shape="1229.66,1398.21 1229.01,1398.84 1225.04,1401.91 1205.32,1414.18">\n      <param key="origId" value="706094741" />\n    </lane>\n  </edge>\n  <edge id="706094741#1" from="1123399641" to="1123399483" name="Thanon Supasarnrangsan Soi 4/1" priority="3" type="highway.residential" shape="1199.28,1419.81 1178.99,1423.33 1159.26,1424.69">\n    <lane id="706094741#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="25.55" shape="1191.33,1419.56 1178.80,1421.74 1166.00,1422.62">\n      <param key="origId" value="706094741" />\n    </lane>\n  </edge>\n  <edge id="706094741#2" from="1123399483" to="1123399659" name="Thanon Supasarnrangsan Soi 4/1" priority="3" type="highway.residential" shape="1159.26,1424.69 1130.75,1424.18 1100.56,1422.57">\n    <lane id="706094741#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="45.14" shape="1152.01,1422.96 1130.81,1422.58 1106.91,1421.31">\n      <param key="origId" value="706094741" />\n    </lane>\n  </edge>\n  <edge id="706094741#3" from="1123399659" to="1123399489" name="Thanon Supasarnrangsan Soi 4/1" priority="3" type="highway.residential">\n    <lane id="706094741#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="47.70" shape="1096.93,1419.47 1052.38,1402.41">\n      <param key="origId" value="706094741" />\n    </lane>\n  </edge>\n  <edge id="706094742" from="1558997825" to="2506085508" name="ถนนสามชัย" priority="3" type="highway.residential" shape="1053.99,703.38 1062.24,692.08 1079.31,683.08 1116.01,679.94 1152.06,679.26 1171.54,683.66 1184.44,688.87">\n    <lane id="706094742_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="123.43" shape="1059.92,697.97 1063.31,693.32 1079.77,684.65 1116.09,681.54 1151.90,680.86 1171.06,685.19 1177.70,687.87">\n      <param key="origId" value="706094742" />\n    </lane>\n  </edge>\n  <edge id="706095305" from="947896956" to="2506088476" name="Thanon Sangchan" priority="3" type="highway.residential">\n    <lane id="706095305_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="76.24" shape="475.98,1382.71 466.20,1458.33">\n      <param key="origId" value="706095305" />\n    </lane>\n  </edge>\n  <edge id="706095306" from="2506087987" to="2506087732" priority="3" type="highway.residential">\n    <lane id="706095306_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="48.97" shape="419.04,1350.57 429.60,1302.75">\n      <param key="origId" value="706095306" />\n    </lane>\n  </edge>\n  <edge id="706095308#0" from="6634630594" to="6634630596" priority="3" type="highway.residential">\n    <lane id="706095308#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="37.09" shape="343.27,1331.59 331.52,1296.41">\n      <param key="origId" value="706095308" />\n    </lane>\n  </edge>\n  <edge id="706095308#1" from="6634630596" to="cluster_2506087462_8523307708" priority="3" type="highway.residential" shape="327.17,1288.42 310.25,1233.64">\n    <lane id="706095308#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="41.08" shape="326.09,1279.50 313.97,1240.25">\n      <param key="origId" value="706095308" />\n    </lane>\n    <param key="origTo" value="8523307708" />\n  </edge>\n  <edge id="706096637" from="1174378892" to="1196113662" name="ถนนปรีดารมย์" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="706096637_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="67.36" shape="281.85,374.98 348.72,383.08">\n      <param key="origId" value="706096637" />\n    </lane>\n  </edge>\n  <edge id="706096640#0" from="1122072681" to="2052547074" name="ถนนเสน่หานุสรณ์" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="706096640#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="72.87" shape="493.58,498.15 483.67,570.34">\n      <param key="origId" value="706096640" />\n    </lane>\n  </edge>\n  <edge id="706096640#1" from="2052547074" to="617234281" name="ถนนเสน่หานุสรณ์" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="706096640#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="66.86" shape="481.69,584.75 472.60,650.99">\n      <param key="origId" value="706096640" />\n    </lane>\n  </edge>\n  <edge id="706096642" from="485580845" to="4085816302" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" spreadType="center" shape="269.45,1011.91 281.39,1015.13 336.66,1023.03">\n    <lane id="706096642_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="59.27" shape="269.45,1011.91 281.39,1015.13 327.82,1021.77">\n      <param key="origId" value="706096642" />\n    </lane>\n  </edge>\n  <edge id="706096643#0" from="1122072673" to="1431715421" name="ถนนนิยมรัฐ" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="706096643#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="25.09" shape="533.67,296.27 508.76,293.24">\n      <param key="origId" value="706096643" />\n    </lane>\n  </edge>\n  <edge id="706096643#1" from="1431715421" to="1122072645" name="ถนนนิยมรัฐ" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="706096643#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="41.95" shape="494.47,291.50 452.83,286.43">\n      <param key="origId" value="706096643" />\n    </lane>\n  </edge>\n  <edge id="706096644#0" from="1174378788" to="1122072681" name="ถนนเสน่หานุสรณ์" priority="10" type="highway.tertiary" shape="505.02,403.09 500.61,440.08 494.55,491.05">\n    <lane id="706096644#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="75.75" shape="502.77,408.47 499.02,439.89 493.81,483.69">\n      <param key="origId" value="706096644" />\n    </lane>\n  </edge>\n  <edge id="706097788#0" from="617234281" to="14069981565" name="ถนนเสน่หานุสรณ์" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="706097788#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.56" length="69.25" shape="470.49,665.58 460.06,734.04">\n      <param key="origId" value="706097788" />\n    </lane>\n  </edge>\n  <edge id="706097788#1" from="14069981565" to="1559029798" name="ถนนเสน่หานุสรณ์" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="706097788#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.56" length="1.35" shape="458.64,743.37 458.43,744.70">\n      <param key="origId" value="706097788" />\n    </lane>\n  </edge>\n  <edge id="706097788#2" from="1559029798" to="617234843" name="ถนนเสน่หานุสรณ์" priority="10" type="highway.tertiary" spreadType="center" shape="457.63,749.99 453.55,779.83 444.61,845.33">\n    <lane id="706097788#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="5.56" length="85.30" shape="456.89,755.41 453.55,779.83 445.35,839.93">\n      <param key="origId" value="706097788" />\n    </lane>\n  </edge>\n  <edge id="706102885" from="1160617590" to="2506089779" priority="3" type="highway.residential" spreadType="center" shape="770.08,1791.30 839.14,1794.09 901.45,1794.01 928.19,1793.79 934.31,1793.36 938.39,1792.71 942.15,1792.00 945.91,1790.78 951.93,1787.85 959.06,1782.72 966.46,1775.95 971.87,1769.88 976.77,1763.42 985.35,1750.02 1017.51,1699.50">\n    <lane id="706102885_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="286.73" shape="770.08,1791.30 839.14,1794.09 901.45,1794.01 928.19,1793.79 934.31,1793.36 938.39,1792.71 942.15,1792.00 945.91,1790.78 951.93,1787.85 959.06,1782.72 966.46,1775.95 971.87,1769.88 976.77,1763.42 985.35,1750.02 1013.73,1705.43">\n      <param key="origId" value="706102885" />\n    </lane>\n  </edge>\n  <edge id="706103887" from="2506087863" to="2506087889" name="Soi Watthayorn" priority="3" type="highway.residential">\n    <lane id="706103887_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="64.95" shape="571.76,1309.42 636.48,1314.82">\n      <param key="origId" value="706103887" />\n    </lane>\n  </edge>\n  <edge id="706104217#0" from="947896964" to="8523431062" name="ถนนเเสงศรี" priority="3" type="highway.residential" shape="1018.99,1324.95 1003.24,1327.91 988.37,1331.11">\n    <lane id="706104217#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="15.18" shape="1010.86,1324.85 1002.92,1326.34 995.98,1327.84">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="706104217#11" from="2506088024" to="12339345632" name="ถนนเเสงศรี" priority="3" type="highway.residential">\n    <lane id="706104217#11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="5.81" shape="767.20,1388.68 761.47,1389.68">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="706104217#12" from="12339345632" to="2506088084" name="ถนนเเสงศรี" priority="3" type="highway.residential">\n    <lane id="706104217#12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.44" shape="747.22,1392.31 731.08,1395.44">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="706104217#13" from="2506088084" to="1140280758" name="ถนนเเสงศรี" priority="3" type="highway.residential" shape="725.72,1398.11 701.16,1401.53 677.38,1402.31">\n    <lane id="706104217#13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="38.68" shape="721.52,1397.08 701.02,1399.93 683.05,1400.52">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="706104217#14" from="1140280758" to="cluster_2506088039_8523406713" name="ถนนเเสงศรี" priority="3" type="highway.residential" shape="677.38,1402.31 651.83,1397.58">\n    <lane id="706104217#14_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.66" shape="671.57,1399.61 659.12,1397.30">\n      <param key="origId" value="706104217" />\n    </lane>\n    <param key="origTo" value="8523406713" />\n  </edge>\n  <edge id="706104217#16" from="cluster_2506088039_8523406713" to="1110416470" name="ถนนเเสงศรี" priority="3" type="highway.residential" shape="633.99,1393.35 573.07,1385.51">\n    <lane id="706104217#16_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="50.22" shape="628.72,1391.06 578.91,1384.65">\n      <param key="origId" value="706104217" />\n    </lane>\n    <param key="origFrom" value="2506088039" />\n  </edge>\n  <edge id="706104217#17" from="1110416470" to="4900465517" name="ถนนเเสงศรี" priority="3" type="highway.residential">\n    <lane id="706104217#17_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.21" shape="567.00,1383.12 550.93,1381.06">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="706104217#18" from="4900465517" to="947896956" name="ถนนเเสงศรี" priority="3" type="highway.residential">\n    <lane id="706104217#18_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="50.97" shape="536.43,1379.48 485.66,1374.94">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="706104217#19" from="947896956" to="2506087987" name="ถนนเเสงศรี" priority="3" type="highway.residential">\n    <lane id="706104217#19_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="51.28" shape="472.46,1372.46 423.27,1357.99">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="706104217#2" from="8523431062" to="8523431064" name="ถนนเเสงศรี" priority="3" type="highway.residential">\n    <lane id="706104217#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.96" shape="981.71,1330.95 968.09,1333.98">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="706104217#20" from="2506087987" to="6634630594" name="ถนนเเสงศรี" priority="3" type="highway.residential" shape="415.88,1357.48 403.15,1353.73 344.79,1341.21">\n    <lane id="706104217#20_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="59.82" shape="409.21,1353.85 403.54,1352.18 350.82,1340.87">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="706104217#21" from="6634630594" to="cluster_916327730_916327962_985092239_985092700" name="ถนนเเสงศรี" priority="3" type="highway.residential" shape="344.79,1341.21 338.73,1339.91 313.71,1340.07">\n    <lane id="706104217#21_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.01" shape="335.66,1338.33 322.66,1338.41">\n      <param key="origId" value="706104217" />\n    </lane>\n    <param key="origTo" value="985092700" />\n  </edge>\n  <edge id="706104217#3" from="8523431064" to="2506087962" name="ถนนเเสงศรี" priority="3" type="highway.residential">\n    <lane id="706104217#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="8.28" shape="954.17,1337.41 946.19,1339.58">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="706104217#4" from="2506087962" to="947896950" name="ถนนเเสงศรี" priority="3" type="highway.residential" shape="939.35,1343.10 936.76,1343.77 916.10,1348.60 895.42,1353.48 894.34,1353.65">\n    <lane id="706104217#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="33.46" shape="932.20,1343.19 915.73,1347.04 899.63,1350.84">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="706104217#8" from="947896950" to="2506088002" name="ถนนเเสงศรี" priority="3" type="highway.residential">\n    <lane id="706104217#8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="49.41" shape="888.84,1353.34 840.83,1365.04">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="706104217#9" from="2506088002" to="2506088024" name="ถนนเเสงศรี" priority="3" type="highway.residential" shape="833.66,1368.43 831.06,1369.45 775.02,1388.94">\n    <lane id="706104217#9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="45.89" shape="827.10,1369.13 783.76,1384.21">\n      <param key="origId" value="706104217" />\n    </lane>\n  </edge>\n  <edge id="706104218" from="5149825984" to="1123399639" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="706104218_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.69" shape="1248.44,1179.36 1238.77,1178.74">\n      <param key="origId" value="706104218" />\n    </lane>\n  </edge>\n  <edge id="706104219#0" from="1123399639" to="cluster_1115585393_13472440789_947896971" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="1230.93,1179.84 1050.46,1163.25">\n    <lane id="706104219#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="168.53" shape="1224.30,1177.62 1056.49,1162.20">\n      <param key="origId" value="706104219" />\n    </lane>\n    <param key="origTo" value="1115585393" />\n  </edge>\n  <edge id="706104219#11" from="cluster_1966399241_8523333972" to="1140280757" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="722.38,1132.46 691.90,1129.98">\n    <lane id="706104219#11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.64" shape="717.24,1130.44 699.66,1129.01">\n      <param key="origId" value="706104219" />\n    </lane>\n    <param key="origFrom" value="1966399241" />\n  </edge>\n  <edge id="706104219#12" from="1140280757" to="1110416444" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="706104219#12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="33.76" shape="685.23,1127.51 651.74,1123.22">\n      <param key="origId" value="706104219" />\n    </lane>\n  </edge>\n  <edge id="706104219#13" from="1110416444" to="1110416654" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="706104219#13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="65.07" shape="637.78,1120.99 573.94,1108.42">\n      <param key="origId" value="706104219" />\n    </lane>\n  </edge>\n  <edge id="706104219#14" from="1110416654" to="1107784887" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="706104219#14_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.27" shape="563.39,1105.94 525.61,1095.23">\n      <param key="origId" value="706104219" />\n    </lane>\n  </edge>\n  <edge id="706104219#15" from="1107784887" to="2506086181" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="706104219#15_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="62.72" shape="514.26,1091.60 455.16,1070.59">\n      <param key="origId" value="706104219" />\n    </lane>\n  </edge>\n  <edge id="706104219#16" from="2506086181" to="4085816302" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="449.53,1070.28 407.39,1053.24 399.77,1047.10 393.30,1039.01 365.48,1029.57 336.66,1023.03">\n    <lane id="706104219#16_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="108.44" shape="444.62,1066.57 408.21,1051.85 400.91,1045.96 394.26,1037.65 365.92,1028.03 347.19,1023.78">\n      <param key="origId" value="706104219" />\n    </lane>\n  </edge>\n  <edge id="706104219#3" from="cluster_1115585393_13472440789_947896971" to="4900464597" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="1025.86,1161.53 993.81,1158.75 969.86,1153.94">\n    <lane id="706104219#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="43.87" shape="1020.18,1159.43 994.04,1157.16 976.75,1153.69">\n      <param key="origId" value="706104219" />\n    </lane>\n    <param key="origFrom" value="13472440789" />\n  </edge>\n  <edge id="706104219#5" from="4900464597" to="2506086831" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="969.86,1153.94 909.53,1148.59 878.02,1145.79">\n    <lane id="706104219#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="77.63" shape="962.65,1151.69 909.67,1147.00 885.32,1144.83">\n      <param key="origId" value="706104219" />\n    </lane>\n  </edge>\n  <edge id="706104219#7" from="2506086831" to="2506086795" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="706104219#7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="37.68" shape="870.97,1143.52 833.46,1140.00">\n      <param key="origId" value="706104219" />\n    </lane>\n  </edge>\n  <edge id="706104219#8" from="2506086795" to="1966399248" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="706104219#8_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="36.47" shape="819.01,1138.73 782.66,1135.76">\n      <param key="origId" value="706104219" />\n    </lane>\n  </edge>\n  <edge id="706104219#9" from="1966399248" to="cluster_1966399241_8523333972" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="775.07,1136.75 732.16,1133.26">\n    <lane id="706104219#9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="28.57" shape="768.20,1134.59 739.72,1132.27">\n      <param key="origId" value="706104219" />\n    </lane>\n    <param key="origTo" value="8523333972" />\n  </edge>\n  <edge id="706105950" from="1107926537" to="8976429009" name="ถนนสามชัย" priority="3" type="highway.residential" shape="1336.13,806.65 1353.53,794.92 1363.49,785.54 1373.59,766.33 1376.45,754.50 1379.76,739.49">\n    <lane id="706105950_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="73.36" shape="1341.81,804.75 1354.53,796.17 1364.78,786.52 1375.10,766.90 1378.01,754.86 1380.03,745.70">\n      <param key="origId" value="706105950" />\n    </lane>\n  </edge>\n  <edge id="706105951#0" from="cluster_13472440775_1666900747" to="1107926513" name="ถนนสามชัย" priority="3" type="highway.residential" shape="1333.17,1187.69 1334.88,1134.02">\n    <lane id="706105951#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="35.96" shape="1335.11,1177.19 1336.25,1141.25">\n      <param key="origId" value="706105951" />\n    </lane>\n    <param key="origFrom" value="13472440775" />\n  </edge>\n  <edge id="706105951#1" from="1107926513" to="856214524" name="ถนนสามชัย" priority="3" type="highway.residential" shape="1334.88,1134.02 1334.96,1064.21 1336.71,985.69">\n    <lane id="706105951#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="133.90" shape="1336.49,1126.92 1336.56,1064.23 1338.15,993.04">\n      <param key="origId" value="706105951" />\n    </lane>\n  </edge>\n  <edge id="706105951#3" from="856214524" to="1107926639" name="ถนนสามชัย" priority="3" type="highway.residential">\n    <lane id="706105951#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="89.22" shape="1338.23,978.48 1337.27,889.27">\n      <param key="origId" value="706105951" />\n    </lane>\n  </edge>\n  <edge id="706105951#4" from="1107926639" to="1107926537" name="ถนนสามชัย" priority="3" type="highway.residential" shape="1335.59,881.55 1337.36,832.67 1336.13,806.65">\n    <lane id="706105951#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="60.99" shape="1337.44,874.66 1338.96,832.66 1338.07,813.72">\n      <param key="origId" value="706105951" />\n    </lane>\n  </edge>\n  <edge id="706105952#0" from="2505176265" to="809537690" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="706105952#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.49" shape="1570.29,1194.75 1568.63,1182.37">\n      <param key="origId" value="706105952" />\n    </lane>\n  </edge>\n  <edge id="706105952#1" from="809537690" to="809537820" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="706105952#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="1.19" shape="1567.05,1171.17 1566.88,1169.99">\n      <param key="origId" value="706105952" />\n    </lane>\n  </edge>\n  <edge id="706256377#0" from="1110416410" to="2052547075" name="ถนนชีอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="706256377#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="33.87" shape="738.45,599.46 704.71,596.57">\n      <param key="origId" value="706256377" />\n    </lane>\n  </edge>\n  <edge id="706256377#1" from="2052547075" to="1107784883" name="ถนนชีอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="706256377#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="65.23" shape="693.49,595.60 628.51,589.93">\n      <param key="origId" value="706256377" />\n    </lane>\n  </edge>\n  <edge id="706256377#2" from="1107784883" to="4900465209" name="ถนนชีอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="706256377#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="33.41" shape="616.91,588.95 583.62,586.26">\n      <param key="origId" value="706256377" />\n    </lane>\n  </edge>\n  <edge id="706256377#3" from="4900465209" to="2052547074" name="ถนนชีอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="706256377#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="81.15" shape="569.24,584.93 488.52,576.53">\n      <param key="origId" value="706256377" />\n    </lane>\n  </edge>\n  <edge id="706256377#4" from="2052547074" to="1176926303" name="ถนนชีอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="706256377#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="66.19" shape="477.06,575.46 411.09,570.02">\n      <param key="origId" value="706256377" />\n    </lane>\n  </edge>\n  <edge id="706256379#1" from="1431715409" to="cluster_1122072658_14138648129" name="ถนนกิมประดิษฐ์" priority="10" type="highway.tertiary" shape="732.80,514.38 710.30,512.38">\n    <lane id="706256379#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.20" shape="729.03,512.44 715.88,511.27">\n      <param key="origId" value="706256379" />\n    </lane>\n    <param key="origTo" value="14138648129" />\n  </edge>\n  <edge id="706256379#3" from="cluster_1122072658_14138648129" to="1107784878" name="ถนนกิมประดิษฐ์" priority="10" type="highway.tertiary" shape="698.36,511.35 634.65,505.81">\n    <lane id="706256379#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="48.97" shape="688.99,508.93 640.20,504.69">\n      <param key="origId" value="706256379" />\n    </lane>\n    <param key="origFrom" value="1122072658" />\n  </edge>\n  <edge id="706256379#4" from="1107784878" to="4900465208" name="ถนนกิมประดิษฐ์" priority="10" type="highway.tertiary" shape="634.65,505.81 602.22,502.97 586.86,501.44">\n    <lane id="706256379#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="35.08" shape="629.04,503.71 602.37,501.38 594.11,500.55">\n      <param key="origId" value="706256379" />\n    </lane>\n  </edge>\n  <edge id="706256379#5" from="4900465208" to="1122072681" name="ถนนกิมประดิษฐ์" priority="10" type="highway.tertiary" shape="586.86,501.44 525.51,494.53 494.55,491.05">\n    <lane id="706256379#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="78.43" shape="579.85,499.04 525.69,492.94 501.90,490.27">\n      <param key="origId" value="706256379" />\n    </lane>\n  </edge>\n  <edge id="706256380#0" from="1110416677" to="1680666322" name="ถนนละม้ายสงเคราะห์" priority="10" type="highway.tertiary" shape="946.62,206.52 945.08,227.47 938.88,286.13 935.82,314.77">\n    <lane id="706256380#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="101.31" shape="945.02,206.40 943.49,227.33 937.29,285.96 935.02,307.21">\n      <param key="origId" value="706256380" />\n    </lane>\n  </edge>\n  <edge id="706256380#1" from="1680666322" to="cluster_1110416379_1680666461" name="ถนนละม้ายสงเคราะห์" priority="10" type="highway.tertiary" shape="935.82,314.77 933.40,339.41 932.07,348.59">\n    <lane id="706256380#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.84" shape="933.54,321.59 931.81,339.22 931.51,341.32">\n      <param key="origId" value="706256380" />\n    </lane>\n    <param key="origTo" value="1110416379" />\n  </edge>\n  <edge id="706256380#3" from="cluster_1110416379_1680666461" to="1680666686" name="ถนนละม้ายสงเคราะห์" priority="10" type="highway.tertiary" shape="929.75,368.50 925.67,405.81 924.52,415.08">\n    <lane id="706256380#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.35" shape="927.39,375.33 924.08,405.62 923.85,407.49">\n      <param key="origId" value="706256380" />\n    </lane>\n    <param key="origFrom" value="1680666461" />\n  </edge>\n  <edge id="706256380#4" from="1680666686" to="cluster_1559016919_1680666950" name="ถนนละม้ายสงเคราะห์" priority="10" type="highway.tertiary" shape="924.52,415.08 916.74,482.51">\n    <lane id="706256380#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="53.42" shape="922.10,422.11 915.98,475.18">\n      <param key="origId" value="706256380" />\n    </lane>\n    <param key="origTo" value="1559016919" />\n  </edge>\n  <edge id="706256380#6" from="cluster_1559016919_1680666950" to="13339819888" name="ถนนละม้ายสงเคราะห์" priority="10" type="highway.tertiary" shape="914.80,503.53 912.32,529.55">\n    <lane id="706256380#6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.08" shape="912.53,510.53 911.28,523.55">\n      <param key="origId" value="706256380" />\n    </lane>\n    <param key="origFrom" value="1680666950" />\n  </edge>\n  <edge id="706256381" from="1160612969" to="13339819893" name="ถนนเทพสงเคราะห์" priority="10" type="highway.tertiary" shape="902.44,617.43 909.56,545.24 909.94,541.55">\n    <lane id="706256381_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="63.67" shape="904.74,610.39 910.99,547.03">\n      <param key="origId" value="706256381" />\n    </lane>\n  </edge>\n  <edge id="706256383" from="1431715392" to="cluster_1107773211_1110416556" priority="3" type="highway.residential" spreadType="center" shape="770.27,603.98 766.61,563.65 763.74,533.62 755.95,515.14">\n    <lane id="706256383_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="74.60" shape="769.62,596.76 766.61,563.65 763.74,533.62 759.40,523.32">\n      <param key="origId" value="706256383" />\n    </lane>\n    <param key="origTo" value="1110416556" />\n  </edge>\n  <edge id="706256385" from="1431715406" to="1431715363" name="ถนนหอมนวล" priority="10" type="highway.tertiary" spreadType="center" shape="805.93,606.96 801.88,670.20 811.45,686.42">\n    <lane id="706256385_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="69.80" shape="805.47,614.17 801.88,670.20 808.82,681.97">\n      <param key="origId" value="706256385" />\n    </lane>\n  </edge>\n  <edge id="706256386" from="1559016918" to="cluster_1559016919_1680666950" name="ถนนฉัยยากุลอุทิศ ซอย 1" priority="10" type="highway.tertiary" shape="865.46,476.66 916.74,482.51">\n    <lane id="706256386_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="37.28" shape="872.37,479.06 909.41,483.28">\n      <param key="origId" value="706256386" />\n    </lane>\n    <param key="origTo" value="1559016919" />\n  </edge>\n  <edge id="706520617#0" from="1115585416" to="12649445299" name="Thumnoonvithi Road" priority="3" type="highway.residential" shape="957.51,845.29 935.89,862.92 910.16,874.76 902.62,876.30">\n    <lane id="706520617#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="50.39" shape="951.61,848.04 935.04,861.55 909.66,873.23 908.61,873.44">\n      <param key="origId" value="706520617" />\n    </lane>\n  </edge>\n  <edge id="706520617#1" from="12649445299" to="1123399562" name="Thumnoonvithi Road" priority="3" type="highway.residential" shape="902.62,876.30 897.13,877.41 882.87,875.84 837.89,873.59 822.54,865.07">\n    <lane id="706520617#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="65.14" shape="894.24,875.48 883.00,874.24 838.34,872.01 830.36,867.58">\n      <param key="origId" value="706520617" />\n    </lane>\n  </edge>\n  <edge id="706520617#2" from="1123399562" to="1115585792" name="Thumnoonvithi Road" priority="3" type="highway.residential">\n    <lane id="706520617#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.44" shape="817.71,860.45 783.51,840.81">\n      <param key="origId" value="706520617" />\n    </lane>\n  </edge>\n  <edge id="706520617#3" from="1115585792" to="2052489649" name="Thumnoonvithi Road" priority="3" type="highway.residential" shape="778.12,839.56 746.91,817.81 730.39,800.51">\n    <lane id="706520617#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="50.24" shape="776.17,836.25 747.96,816.59 737.01,805.13">\n      <param key="origId" value="706520617" />\n    </lane>\n  </edge>\n  <edge id="706520617#4" from="2052489649" to="1110416568" name="Thumnoonvithi Road" priority="3" type="highway.residential" shape="730.39,800.51 723.06,777.36 723.19,754.34 741.15,676.65">\n    <lane id="706520617#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="113.79" shape="730.48,795.51 724.66,777.12 724.79,754.53 740.99,684.46">\n      <param key="origId" value="706520617" />\n    </lane>\n  </edge>\n  <edge id="71380773#0" from="1431715366" to="1431715363" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center" shape="769.07,680.94 797.17,685.11 811.45,686.42">\n    <lane id="71380773#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.64" shape="773.95,681.66 797.17,685.11 806.30,685.95">\n      <param key="origId" value="71380773" />\n    </lane>\n  </edge>\n  <edge id="71380773#1" from="1431715363" to="1160612974" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="71380773#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="74.38" shape="814.30,686.68 888.37,693.51">\n      <param key="origId" value="71380773" />\n    </lane>\n  </edge>\n  <edge id="71380788" from="823954361" to="2505176747" name="Thanon Rajyindee Soi 1/1" priority="3" type="highway.residential" shape="1983.68,1660.37 1949.93,1657.34 1850.43,1645.23 1782.01,1634.57 1734.25,1636.75">\n    <lane id="71380788_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="251.39" shape="1983.82,1658.78 1950.10,1655.75 1850.65,1643.64 1782.10,1632.96 1734.18,1635.15">\n      <param key="origId" value="71380788" />\n    </lane>\n  </edge>\n  <edge id="72016745#0" from="856214571" to="1369020950" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="1544.12,1202.38 1532.40,1204.67 1476.45,1199.74 1385.88,1192.43">\n    <lane id="72016745#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="144.77" shape="1537.50,1198.78 1532.15,1199.83 1476.85,1194.96 1393.76,1188.25">\n      <param key="origId" value="72016745" />\n    </lane>\n    <lane id="72016745#0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="144.77" shape="1538.12,1201.92 1532.32,1203.06 1476.58,1198.15 1393.50,1191.44">\n      <param key="origId" value="72016745" />\n    </lane>\n  </edge>\n  <edge id="72016745#2" from="1369020950" to="cluster_13472440775_1666900747" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="1385.88,1192.43 1337.16,1188.00">\n    <lane id="72016745#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.64" shape="1379.51,1187.03 1345.01,1183.89">\n      <param key="origId" value="72016745" />\n    </lane>\n    <lane id="72016745#2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.64" shape="1379.22,1190.22 1344.72,1187.08">\n      <param key="origId" value="72016745" />\n    </lane>\n    <param key="origTo" value="1666900747" />\n  </edge>\n  <edge id="72016745#4" from="cluster_13472440775_1666900747" to="2506087136" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="1333.17,1187.69 1285.38,1183.95">\n    <lane id="72016745#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.94" shape="1326.07,1182.32 1293.23,1179.75">\n      <param key="origId" value="72016745" />\n    </lane>\n    <lane id="72016745#4_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.94" shape="1325.82,1185.51 1292.98,1182.94">\n      <param key="origId" value="72016745" />\n    </lane>\n    <param key="origFrom" value="13472440775" />\n  </edge>\n  <edge id="72016745#5" from="2506087136" to="5149825984" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="72016745#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.54" shape="1278.21,1178.52 1265.71,1177.44">\n      <param key="origId" value="72016745" />\n    </lane>\n    <lane id="72016745#5_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="12.54" shape="1277.93,1181.70 1265.44,1180.63">\n      <param key="origId" value="72016745" />\n    </lane>\n  </edge>\n  <edge id="77967082#0" from="916327777" to="cluster_916327730_916327962_985092239_985092700" name="ถนนเชื่อมรัฐอุทิศ" priority="10" type="highway.tertiary" spreadType="center" shape="39.51,1375.45 137.37,1371.50 304.04,1363.52">\n    <lane id="77967082#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="260.04" shape="39.57,1377.05 137.44,1373.10 299.36,1365.35">\n      <param key="origId" value="77967082" />\n    </lane>\n    <lane id="77967082#0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="260.04" shape="39.45,1373.85 137.30,1369.90 299.20,1362.15">\n      <param key="origId" value="77967082" />\n    </lane>\n    <param key="origTo" value="916327730" />\n  </edge>\n  <edge id="77967084#2" from="cluster_916327730_916327962_985092239_985092700" to="916327810" name="ถนนเพชรเกษม" priority="11" type="highway.secondary" spreadType="center" shape="304.04,1363.52 323.81,1422.99">\n    <lane id="77967084#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="47.94" shape="306.21,1370.04 321.33,1415.54">\n      <param key="origId" value="77967084" />\n    </lane>\n    <param key="origFrom" value="916327730" />\n  </edge>\n  <edge id="77967084#3" from="916327810" to="916327702" name="ถนนเพชรเกษม" priority="11" type="highway.secondary" spreadType="center">\n    <lane id="77967084#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="84.53" shape="325.87,1429.54 351.28,1510.16">\n      <param key="origId" value="77967084" />\n    </lane>\n  </edge>\n  <edge id="77967084#4" from="916327702" to="4900466295" name="ถนนเพชรเกษม" priority="11" type="highway.secondary" spreadType="center">\n    <lane id="77967084#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="58.95" shape="355.25,1521.59 376.58,1576.54">\n      <param key="origId" value="77967084" />\n    </lane>\n  </edge>\n  <edge id="77967084#5" from="4900466295" to="4900466297" name="ถนนเพชรเกษม" priority="11" type="highway.secondary" spreadType="center" shape="378.86,1582.41 396.96,1631.95 408.85,1665.11 411.81,1673.36">\n    <lane id="77967084#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="86.54" shape="380.62,1587.24 396.96,1631.95 408.85,1665.11 410.10,1668.60">\n      <param key="origId" value="77967084 931348309" />\n    </lane>\n  </edge>\n  <edge id="77967085" from="916327824" to="916327702" name="ถนนมนตรี 2" priority="10" type="highway.tertiary" spreadType="center" shape="24.84,1511.03 179.84,1512.52 230.99,1513.01 284.87,1515.63 353.26,1516.45">\n    <lane id="77967085_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="323.06" shape="24.82,1512.63 179.82,1514.12 230.94,1514.61 284.82,1517.23 347.80,1517.98">\n      <param key="origId" value="77967085" />\n    </lane>\n    <lane id="77967085_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="323.06" shape="24.86,1509.43 179.86,1510.92 231.04,1511.41 284.92,1514.03 347.84,1514.78">\n      <param key="origId" value="77967085" />\n    </lane>\n  </edge>\n  <edge id="77967090#0" from="916327686" to="916327912" name="ถนนเพชรเกษม" priority="11" type="highway.secondary" spreadType="center" shape="446.23,1696.30 430.34,1685.33 418.94,1643.06 364.29,1493.33">\n    <lane id="77967090#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="193.15" shape="432.63,1681.53 421.99,1642.09 370.50,1501.00">\n      <param key="origId" value="77967090" />\n    </lane>\n    <lane id="77967090#0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="193.15" shape="430.17,1684.71 418.94,1643.06 367.49,1502.10">\n      <param key="origId" value="77967090" />\n    </lane>\n    <lane id="77967090#0_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="193.15" shape="427.96,1687.58 427.56,1687.30 415.89,1644.03 364.49,1503.20">\n      <param key="origId" value="77967090" />\n    </lane>\n  </edge>\n  <edge id="77967090#1" from="916327912" to="cluster_916327730_916327962_985092239_985092700" name="ถนนเพชรเกษม" priority="11" type="highway.secondary" spreadType="center" shape="364.29,1493.33 319.37,1360.27">\n    <lane id="77967090#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="129.67" shape="365.52,1486.96 324.05,1364.11">\n      <param key="origId" value="77967090" />\n    </lane>\n    <lane id="77967090#1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="129.67" shape="362.48,1487.98 321.01,1365.12">\n      <param key="origId" value="77967090" />\n    </lane>\n    <lane id="77967090#1_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="129.67" shape="359.45,1489.00 317.97,1366.13">\n      <param key="origId" value="77967090" />\n    </lane>\n    <param key="origTo" value="916327962" />\n  </edge>\n  <edge id="77967090#3" from="cluster_916327730_916327962_985092239_985092700" to="2506087709" name="ถนนเพชรเกษม" priority="11" type="highway.secondary" spreadType="center" shape="313.71,1340.07 295.23,1285.13">\n    <lane id="77967090#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="42.48" shape="314.90,1333.56 301.36,1293.30">\n      <param key="origId" value="77967090" />\n    </lane>\n    <lane id="77967090#3_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="42.48" shape="311.86,1334.58 298.32,1294.32">\n      <param key="origId" value="77967090" />\n    </lane>\n    <lane id="77967090#3_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="42.48" shape="308.83,1335.60 295.29,1295.34">\n      <param key="origId" value="77967090" />\n    </lane>\n    <param key="origFrom" value="985092700" />\n  </edge>\n  <edge id="77967092" from="916327810" to="916327792" name="ถนนมนตรี 1" priority="10" type="highway.tertiary" spreadType="center" shape="323.81,1422.99 285.17,1423.65 177.44,1425.50 29.35,1428.06">\n    <lane id="77967092_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="287.82" shape="317.10,1421.50 285.14,1422.05 177.41,1423.90 29.32,1426.46">\n      <param key="origId" value="77967092" />\n    </lane>\n    <lane id="77967092_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="287.82" shape="317.16,1424.70 285.20,1425.25 177.47,1427.10 29.38,1429.66">\n      <param key="origId" value="77967092" />\n    </lane>\n  </edge>\n  <edge id="77967093#0" from="916327888" to="916327686" priority="11" type="highway.secondary" spreadType="center" shape="462.10,1720.09 460.84,1712.70 457.59,1705.95 452.59,1700.34 446.23,1696.30">\n    <lane id="77967093#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="26.17" shape="463.76,1720.31 462.38,1712.21 458.93,1705.05 453.64,1699.11 450.77,1697.29">\n      <param key="origId" value="77967093" />\n    </lane>\n    <lane id="77967093#0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="26.17" shape="460.61,1720.85 459.30,1713.19 456.25,1706.85 451.54,1701.57 449.05,1699.99">\n      <param key="origId" value="77967093" />\n    </lane>\n  </edge>\n  <edge id="77967093#1" from="916327686" to="916327921" priority="11" type="highway.secondary" spreadType="center" shape="446.23,1696.30 441.85,1694.75 437.27,1693.96 432.61,1693.96 428.03,1694.75 423.65,1696.30 419.60,1698.57 416.00,1701.49 412.95,1704.99">\n    <lane id="77967093#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="1.21" shape="425.89,1693.81 423.81,1694.55">\n      <param key="origId" value="77967093" />\n    </lane>\n    <lane id="77967093#1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="1.21" shape="425.78,1697.24 425.59,1697.31">\n      <param key="origId" value="77967093" />\n    </lane>\n  </edge>\n  <edge id="77967093#2" from="916327921" to="916327839" priority="11" type="highway.secondary" spreadType="center" shape="412.95,1704.99 409.49,1711.37 407.87,1718.42">\n    <lane id="77967093#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="3.65" shape="409.47,1708.05 407.98,1710.80 407.76,1711.74">\n      <param key="origId" value="77967093" />\n    </lane>\n    <lane id="77967093#2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="3.65" shape="412.28,1709.58 411.00,1711.94 410.88,1712.46">\n      <param key="origId" value="77967093" />\n    </lane>\n  </edge>\n  <edge id="77967093#3" from="916327839" to="916327888" priority="11" type="highway.secondary" spreadType="center" shape="407.87,1718.42 407.81,1722.57 408.39,1726.67 409.61,1730.64 411.42,1734.38 413.77,1737.80 416.63,1740.82 419.93,1743.37 423.58,1745.38 427.50,1746.82 436.22,1747.46 445.63,1745.68 450.36,1743.08 454.48,1739.61 457.83,1735.41 460.27,1730.62 461.72,1725.44 462.10,1720.09">\n    <lane id="77967093#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="74.73" shape="406.53,1724.94 406.82,1727.02 408.12,1731.23 410.03,1735.19 412.52,1738.81 415.55,1742.01 419.05,1744.71 422.91,1746.84 427.16,1748.40 436.31,1749.07 446.18,1747.20 451.27,1744.41 455.63,1740.73 459.18,1736.28 461.77,1731.20 462.98,1726.87">\n      <param key="origId" value="77967093" />\n    </lane>\n    <lane id="77967093#3_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="74.73" shape="409.77,1724.98 409.96,1726.32 411.10,1730.05 412.81,1733.57 415.02,1736.79 417.71,1739.63 420.81,1742.03 424.25,1743.92 427.84,1745.24 436.13,1745.85 445.08,1744.16 449.45,1741.75 453.33,1738.49 456.48,1734.54 458.77,1730.04 459.73,1726.63">\n      <param key="origId" value="77967093" />\n    </lane>\n  </edge>\n  <edge id="81350005#0" from="cluster_1115585393_13472440789_947896971" to="947896964" name="Tiparat Road" priority="10" type="highway.tertiary" shape="1032.62,1162.00 1025.14,1232.01 1024.18,1247.20 1023.66,1248.28 1018.99,1324.95">\n    <lane id="81350005#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="149.84" shape="1030.24,1169.17 1023.55,1231.87 1022.60,1246.79 1022.08,1247.87 1017.79,1318.37">\n      <param key="origId" value="81350005" />\n    </lane>\n    <param key="origFrom" value="947896971" />\n  </edge>\n  <edge id="81350006#1" from="7691192946" to="2507400773" name="Thanon Rajyindee Soi 1" priority="10" type="highway.tertiary" shape="1971.98,1710.95 1854.69,1696.27 1826.63,1692.44 1781.33,1686.14 1725.14,1689.85 1699.32,1693.92">\n    <lane id="81350006#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="267.06" shape="1972.18,1709.36 1854.90,1694.68 1826.85,1690.85 1781.39,1684.53 1724.96,1688.26 1707.07,1691.08">\n      <param key="origId" value="81350006" />\n    </lane>\n  </edge>\n  <edge id="81350006#2" from="2507400773" to="947896981" name="Thanon Rajyindee Soi 1" priority="10" type="highway.tertiary">\n    <lane id="81350006#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="35.85" shape="1692.74,1693.61 1657.60,1700.69">\n      <param key="origId" value="81350006" />\n    </lane>\n  </edge>\n  <edge id="84842974#0" from="2506087509" to="cluster_2506087462_8523307708" name="ถนนนิพัทธ์อุทิศ 3" priority="10" type="highway.tertiary" spreadType="center" shape="277.85,1230.47 304.27,1232.59">\n    <lane id="84842974#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.13" shape="277.59,1233.66 295.67,1235.11">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.13" shape="277.85,1230.47 295.92,1231.92">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#0_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.13" shape="278.11,1227.28 296.18,1228.73">\n      <param key="origId" value="84842974" />\n    </lane>\n    <param key="origTo" value="2506087462" />\n  </edge>\n  <edge id="84842974#1" from="cluster_2506087462_8523307708" to="8348961008" name="ถนนนิพัทธ์อุทิศ 3" priority="10" type="highway.tertiary" spreadType="center" shape="304.27,1232.59 309.72,1200.57">\n    <lane id="84842974#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.86" shape="308.83,1224.87 311.66,1208.25">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.86" shape="305.68,1224.33 308.50,1207.71">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#1_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.86" shape="302.52,1223.80 305.35,1207.17">\n      <param key="origId" value="84842974" />\n    </lane>\n    <param key="origFrom" value="2506087462" />\n  </edge>\n  <edge id="84842974#10" from="856214578" to="485580842" name="ถนนนิพัทธ์อุทิศ 3" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="84842974#10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="120.52" shape="344.61,961.68 362.76,842.54">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#10_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="120.52" shape="341.44,961.20 359.60,842.05">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#10_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="120.52" shape="338.28,960.72 356.44,841.57">\n      <param key="origId" value="84842974" />\n    </lane>\n  </edge>\n  <edge id="84842974#11" from="485580842" to="1559029797" name="ถนนนิพัทธ์อุทิศ 3" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="84842974#11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="84.90" shape="364.44,831.08 376.22,747.01">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#11_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="84.90" shape="361.27,830.64 373.05,746.56">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#11_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="84.90" shape="358.10,830.19 369.88,746.12">\n      <param key="origId" value="84842974" />\n    </lane>\n  </edge>\n  <edge id="84842974#12" from="1559029797" to="617234576" name="ถนนนิพัทธ์อุทิศ 3" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="84842974#12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="76.16" shape="377.84,735.90 389.34,660.61">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#12_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="76.16" shape="374.68,735.42 386.18,660.13">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#12_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="76.16" shape="371.52,734.94 383.01,659.65">\n      <param key="origId" value="84842974" />\n    </lane>\n  </edge>\n  <edge id="84842974#13" from="617234576" to="1176926303" name="ถนนนิพัทธ์อุทิศ 3" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="84842974#13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="69.40" shape="391.69,646.42 403.78,578.08">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#13_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="69.40" shape="388.53,645.86 400.63,577.52">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#13_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="69.40" shape="385.38,645.30 397.48,576.97">\n      <param key="origId" value="84842974" />\n    </lane>\n  </edge>\n  <edge id="84842974#14" from="1176926303" to="1431715369" name="ถนนนิพัทธ์อุทิศ 3" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="84842974#14_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="63.35" shape="406.42,563.87 418.44,501.67">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#14_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="63.35" shape="403.28,563.26 415.30,501.06">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#14_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="63.35" shape="400.14,562.65 412.16,500.45">\n      <param key="origId" value="84842974" />\n    </lane>\n  </edge>\n  <edge id="84842974#15" from="1431715369" to="1966412828" name="ถนนนิพัทธ์อุทิศ 3" priority="10" type="highway.tertiary" spreadType="center" shape="416.43,495.21 434.30,404.33 436.19,395.10">\n    <lane id="84842974#15_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="91.23" shape="420.58,490.69 437.44,404.95 438.21,401.21">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#15_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="91.23" shape="417.44,490.07 434.30,404.33 435.07,400.56">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#15_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="91.23" shape="414.30,489.45 431.16,403.71 431.94,399.92">\n      <param key="origId" value="84842974" />\n    </lane>\n  </edge>\n  <edge id="84842974#17" from="1966412828" to="1122072645" name="ถนนนิพัทธ์อุทิศ 3" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="84842974#17_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="101.21" shape="439.89,389.94 449.41,289.18">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#17_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="101.21" shape="436.71,389.64 446.23,288.88">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#17_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="101.21" shape="433.52,389.34 443.04,288.58">\n      <param key="origId" value="84842974" />\n    </lane>\n  </edge>\n  <edge id="84842974#2" from="8348961008" to="cluster_620881097_8523329816" name="ถนนนิพัทธ์อุทิศ 3" priority="10" type="highway.tertiary" spreadType="center" shape="309.72,1200.57 315.06,1169.24">\n    <lane id="84842974#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.80" shape="314.08,1194.05 317.24,1175.52">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.80" shape="310.92,1193.51 314.08,1174.98">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#2_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="18.80" shape="307.77,1192.98 310.93,1174.44">\n      <param key="origId" value="84842974" />\n    </lane>\n    <param key="origTo" value="620881097" />\n  </edge>\n  <edge id="84842974#4" from="cluster_620881097_8523329816" to="2052522376" name="ถนนนิพัทธ์อุทิศ 3" priority="10" type="highway.tertiary" spreadType="center" shape="317.35,1157.66 318.32,1152.68 324.07,1123.56 325.67,1115.51">\n    <lane id="84842974#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.51" shape="321.84,1151.39 327.21,1124.18 327.75,1121.47">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#4_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.51" shape="318.69,1150.78 324.07,1123.56 324.61,1120.85">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#4_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.51" shape="315.55,1150.17 320.93,1122.94 321.47,1120.23">\n      <param key="origId" value="84842974" />\n    </lane>\n    <param key="origFrom" value="8523329816" />\n  </edge>\n  <edge id="84842974#7" from="2052522376" to="4085816302" name="ถนนนิพัทธ์อุทิศ 3" priority="10" type="highway.tertiary" spreadType="center" shape="325.67,1115.51 333.63,1066.39 338.62,1035.64 336.66,1023.03">\n    <lane id="84842974#7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="78.90" shape="329.70,1110.66 336.79,1066.90 341.86,1035.65 341.26,1031.77">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#7_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="78.90" shape="326.54,1110.15 333.63,1066.39 338.62,1035.64 338.10,1032.26">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#7_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="78.90" shape="323.38,1109.63 330.47,1065.88 335.38,1035.63 334.93,1032.76">\n      <param key="origId" value="84842974" />\n    </lane>\n  </edge>\n  <edge id="84842974#9" from="4085816302" to="856214578" name="ถนนนิพัทธ์อุทิศ 3" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="84842974#9_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="43.67" shape="340.31,1016.77 343.35,973.21">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#9_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="43.67" shape="337.11,1016.55 340.16,972.98">\n      <param key="origId" value="84842974" />\n    </lane>\n    <lane id="84842974#9_2" index="2" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="43.67" shape="333.92,1016.33 336.97,972.76">\n      <param key="origId" value="84842974" />\n    </lane>\n  </edge>\n  <edge id="898440288" from="8348961008" to="8348961009" priority="3" type="highway.residential">\n    <lane id="898440288_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="101.73" shape="318.13,1203.70 418.26,1221.67">\n      <param key="origId" value="898440288" />\n    </lane>\n  </edge>\n  <edge id="900244003#0" from="1107926513" to="8363308288" priority="3" type="highway.residential">\n    <lane id="900244003#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.66" shape="1342.01,1135.83 1355.67,1136.23">\n      <param key="origId" value="900244003" />\n    </lane>\n  </edge>\n  <edge id="900244003#1" from="8363308288" to="8363308289" priority="3" type="highway.residential">\n    <lane id="900244003#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="59.68" shape="1369.14,1137.44 1428.15,1146.35">\n      <param key="origId" value="900244003" />\n    </lane>\n  </edge>\n  <edge id="900244004" from="8363308288" to="8363308291" priority="3" type="highway.residential" shape="1362.66,1134.84 1364.26,1110.52 1437.69,1124.41">\n    <lane id="900244004_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="88.55" shape="1364.71,1128.02 1365.74,1112.43 1437.39,1125.98">\n      <param key="origId" value="900244004" />\n    </lane>\n  </edge>\n  <edge id="911123067" from="2506085849" to="2506086181" name="Soi Duangchan" priority="3" type="highway.residential" spreadType="center">\n    <lane id="911123067_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="66.81" shape="470.12,999.00 451.58,1063.19">\n      <param key="origId" value="911123067" />\n    </lane>\n  </edge>\n  <edge id="917695557" from="8523328986" to="8523328981" priority="3" type="highway.residential" shape="1554.92,1101.11 1576.33,1100.04 1595.00,1100.24 1610.70,1101.62 1624.61,1106.37 1638.21,1116.72">\n    <lane id="917695557_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="80.52" shape="1561.00,1102.41 1576.36,1101.64 1594.92,1101.84 1610.37,1103.20 1623.85,1107.80 1637.24,1117.99">\n      <param key="origId" value="917695557" />\n    </lane>\n  </edge>\n  <edge id="917695719" from="8523326826" to="8523326825" name="ซอยเตชะโต" priority="3" type="highway.residential">\n    <lane id="917695719_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="28.89" shape="1586.26,917.19 1589.76,888.51">\n      <param key="origId" value="917695719" />\n    </lane>\n  </edge>\n  <edge id="917696759" from="cluster_1966399241_8523333972" to="8523333971" priority="3" type="highway.residential" shape="732.16,1133.26 740.12,1083.30">\n    <lane id="917696759_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="43.13" shape="734.91,1126.15 741.70,1083.55">\n      <param key="origId" value="917696759" />\n    </lane>\n    <param key="origFrom" value="8523333972" />\n  </edge>\n  <edge id="917696883" from="8523334139" to="cluster_13424220808_2506085647" priority="3" type="highway.residential" spreadType="center" shape="627.65,1023.48 638.42,987.13 642.44,980.05 649.89,954.12 659.17,928.56 666.89,901.24">\n    <lane id="917696883_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="116.07" shape="629.14,1018.44 638.42,987.13 642.44,980.05 649.89,954.12 659.17,928.56 664.91,908.26">\n      <param key="origId" value="917696883" />\n    </lane>\n    <param key="origTo" value="13424220808" />\n  </edge>\n  <edge id="917697169" from="cluster_620881097_8523329816" to="8523329815" name="Thanon Lamaipradit" priority="3" type="highway.residential" shape="317.35,1157.66 373.17,1170.89">\n    <lane id="917697169_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="48.66" shape="325.45,1161.22 372.80,1172.45">\n      <param key="origId" value="917697169" />\n    </lane>\n    <param key="origFrom" value="8523329816" />\n  </edge>\n  <edge id="917704692" from="cluster_2506088039_8523406713" to="8523406712" priority="3" type="highway.residential" shape="651.83,1397.58 634.98,1476.99">\n    <lane id="917704692_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="74.06" shape="648.79,1404.21 633.41,1476.66">\n      <param key="origId" value="917704692" />\n    </lane>\n    <param key="origFrom" value="8523406713" />\n  </edge>\n  <edge id="917705248" from="8523431062" to="8523431061" priority="3" type="highway.residential">\n    <lane id="917705248_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="54.24" shape="990.29,1323.05 992.40,1268.85">\n      <param key="origId" value="917705248" />\n    </lane>\n  </edge>\n  <edge id="917705249" from="8523431064" to="8523431063" priority="3" type="highway.residential">\n    <lane id="917705249_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="67.65" shape="961.51,1329.65 948.84,1263.20">\n      <param key="origId" value="917705249" />\n    </lane>\n  </edge>\n  <edge id="917706996" from="8523449430" to="8523449429" priority="3" type="highway.residential">\n    <lane id="917706996_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="27.56" shape="1684.72,1344.44 1686.74,1316.95">\n      <param key="origId" value="917706996" />\n    </lane>\n  </edge>\n  <edge id="931348308" from="4900466297" to="916327921" name="ถนนเพชรเกษม" priority="11" type="highway.secondary" spreadType="center" shape="411.81,1673.36 416.75,1687.14 417.43,1695.74 412.95,1704.99">\n    <lane id="931348308_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.09" shape="412.99,1676.65 416.75,1687.14 416.98,1690.08">\n      <param key="origId" value="931348308" />\n    </lane>\n  </edge>\n  <edge id="931354569#0" from="2506085690" to="856214503" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="751.80,928.65 727.59,921.92 708.72,914.50 702.16,913.43">\n    <lane id="931354569#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.31" shape="745.44,925.22 728.10,920.40 709.15,912.95 708.21,912.80">\n      <param key="origId" value="931354569" />\n    </lane>\n  </edge>\n  <edge id="931354569#1" from="856214503" to="cluster_13424220808_2506085647" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="702.16,913.43 666.89,901.24">\n    <lane id="931354569#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="25.64" shape="697.12,909.99 672.88,901.62">\n      <param key="origId" value="931354569" />\n    </lane>\n    <param key="origTo" value="13424220808" />\n  </edge>\n  <edge id="931354569#3" from="cluster_13424220808_2506085647" to="2506085628" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="659.33,898.58 638.61,891.99 615.42,885.46">\n    <lane id="931354569#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="33.00" shape="652.70,894.79 639.07,890.46 621.08,885.39">\n      <param key="origId" value="931354569" />\n    </lane>\n    <param key="origFrom" value="2506085647" />\n  </edge>\n  <edge id="931354569#4" from="2506085628" to="1966386254" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="615.42,885.46 595.79,879.99 577.84,875.77 564.09,872.41">\n    <lane id="931354569#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.85" shape="610.29,882.37 596.19,878.44 578.21,874.21 570.69,872.37">\n      <param key="origId" value="931354569" />\n    </lane>\n  </edge>\n  <edge id="931354570" from="1115585792" to="2506085690" priority="3" type="highway.residential" spreadType="center" shape="778.12,839.56 758.80,905.62 751.80,928.65">\n    <lane id="931354570_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="79.16" shape="776.28,845.85 758.80,905.62 753.89,921.78">\n      <param key="origId" value="931354570" />\n    </lane>\n  </edge>\n  <edge id="932324369" from="8642733850" to="8642733849" priority="3" type="highway.residential">\n    <lane id="932324369_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="17.98" shape="1641.11,424.97 1634.41,441.66">\n      <param key="origId" value="932324369" />\n    </lane>\n  </edge>\n  <edge id="932339001" from="8642866006" to="2507400591" priority="3" type="highway.residential" shape="1477.33,720.55 1479.79,720.51 1489.18,720.23 1493.33,719.96">\n    <lane id="932339001_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="2.43" shape="1485.06,721.95 1487.48,721.88">\n      <param key="origId" value="932339001 155690866 243314310" />\n    </lane>\n  </edge>\n  <edge id="932339002#0" from="8642866006" to="1680666775" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1477.33,720.55 1469.33,671.59 1465.44,647.78 1458.07,602.67 1451.65,563.41 1434.74,459.90">\n    <lane id="932339002#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="249.85" shape="1477.82,713.61 1470.91,671.33 1467.02,647.52 1459.65,602.41 1453.23,563.15 1437.53,467.03">\n      <param key="origId" value="932339002" />\n    </lane>\n    <lane id="932339002#0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="249.85" shape="1474.66,714.13 1467.75,671.85 1463.86,648.04 1456.49,602.93 1450.07,563.67 1434.37,467.54">\n      <param key="origId" value="932339002" />\n    </lane>\n  </edge>\n  <edge id="932339002#4" from="1680666775" to="1431867160" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center">\n    <lane id="932339002#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="85.09" shape="1435.20,452.79 1421.49,368.82">\n      <param key="origId" value="932339002" />\n    </lane>\n    <lane id="932339002#4_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="85.09" shape="1432.04,453.31 1418.33,369.33">\n      <param key="origId" value="932339002" />\n    </lane>\n  </edge>\n  <edge id="932339002#5" from="1431867160" to="809537730" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1418.75,361.96 1411.39,316.91 1390.33,182.86 1371.46,72.58 1363.43,23.45 1361.22,9.89 1361.73,0.45">\n    <lane id="932339002#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="359.01" shape="1419.17,354.61 1412.97,316.66 1391.91,182.60 1373.04,72.32 1365.01,23.19 1362.83,9.80 1363.33,0.54">\n      <param key="origId" value="932339002" />\n    </lane>\n    <lane id="932339002#5_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="359.01" shape="1416.01,355.12 1409.81,317.16 1388.75,183.12 1369.88,72.84 1361.85,23.71 1359.61,9.98 1360.13,0.36">\n      <param key="origId" value="932339002" />\n    </lane>\n  </edge>\n  <edge id="932339004" from="1431867160" to="1431867108" name="ถนนราษฎร์ยินดี ซอย 9" priority="10" type="highway.tertiary" shape="1418.75,361.96 1423.82,361.15 1430.93,360.13 1435.64,359.62">\n    <lane id="932339004_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="4.04" shape="1426.13,362.44 1430.13,361.86">\n      <param key="origId" value="932339004 932339003 129842388" />\n    </lane>\n  </edge>\n  <edge id="932339006" from="1680666775" to="8642866010" priority="3" type="highway.residential" shape="1434.74,459.90 1438.77,459.59 1447.22,458.73">\n    <lane id="932339006_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="4.96" shape="1442.32,460.84 1447.25,460.34">\n      <param key="origId" value="932339006 932339005" />\n    </lane>\n  </edge>\n  <edge id="95533432#0" from="1110416454" to="2506085476" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary">\n    <lane id="95533432#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="26.49" shape="1303.10,567.18 1276.66,565.41">\n      <param key="origId" value="95533432" />\n    </lane>\n  </edge>\n  <edge id="95533432#1" from="2506085476" to="cluster_1680667233_2506085475" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary" shape="1269.22,566.52 1234.89,563.71">\n    <lane id="95533432#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="20.61" shape="1262.36,564.35 1241.83,562.67">\n      <param key="origId" value="95533432" />\n    </lane>\n    <param key="origTo" value="2506085475" />\n  </edge>\n  <edge id="95533432#3" from="cluster_1680667233_2506085475" to="4900465219" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary" shape="1219.28,562.16 1149.97,555.31">\n    <lane id="95533432#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="55.28" shape="1212.38,559.87 1157.36,554.43">\n      <param key="origId" value="95533432" />\n    </lane>\n    <param key="origFrom" value="1680667233" />\n  </edge>\n  <edge id="95533432#4" from="4900465219" to="1680667181" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary">\n    <lane id="95533432#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="54.26" shape="1143.06,552.99 1089.09,547.39">\n      <param key="origId" value="95533432" />\n    </lane>\n  </edge>\n  <edge id="95533432#5" from="1680667181" to="13339844003" name="ถนนหอยมุกต์" priority="10" type="highway.tertiary" shape="1081.84,548.25 973.52,539.57 927.61,535.69 915.79,534.39">\n    <lane id="95533432#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="153.88" shape="1074.81,546.08 973.65,537.98 927.76,534.10 921.46,533.40">\n      <param key="origId" value="95533432" />\n    </lane>\n  </edge>\n  <edge id="95533433#0" from="1107784878" to="1107784883" name="Thanon Sangchan" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="95533433#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="71.48" shape="633.63,513.03 623.64,583.80">\n      <param key="origId" value="95533433" />\n    </lane>\n  </edge>\n  <edge id="95533433#1" from="1107784883" to="1107784884" name="Thanon Sangchan" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="95533433#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="57.90" shape="621.46,598.26 612.29,655.43">\n      <param key="origId" value="95533433" />\n    </lane>\n  </edge>\n  <edge id="95542480" from="947896981" to="1680668223" priority="10" type="highway.tertiary">\n    <lane id="95542480_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="1.24" shape="1646.44,1702.89 1645.22,1703.12">\n      <param key="origId" value="95542480" />\n    </lane>\n  </edge>\n  <edge id="95542482#0" from="1107926481" to="1107926561" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="95542482#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="39.32" shape="1332.16,1562.66 1293.99,1572.11">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="95542482#1" from="1107926561" to="1107926940" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="95542482#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="30.15" shape="1280.00,1575.85 1251.05,1584.27">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="95542482#10" from="1110416526" to="2507400769" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="95542482#10_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.91" shape="857.55,1645.70 822.87,1649.75">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="95542482#11" from="2507400769" to="1110416469" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="95542482#11_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="50.29" shape="808.38,1651.80 758.77,1660.10">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="95542482#12" from="1110416469" to="2507400772" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="95542482#12_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.50" shape="744.19,1663.43 705.32,1674.82">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="95542482#13" from="2507400772" to="1110416754" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="95542482#13_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="78.55" shape="691.49,1678.88 616.11,1700.98">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="95542482#14" from="1110416754" to="1123399591" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="95542482#14_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="24.05" shape="602.29,1705.04 579.22,1711.85">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="95542482#2" from="1107926940" to="1107926599" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary" shape="1243.71,1588.07 1196.94,1600.50 1179.04,1604.10 1169.20,1605.36 1158.83,1606.70 1131.66,1608.75">\n    <lane id="95542482#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="102.53" shape="1236.96,1588.21 1196.58,1598.94 1178.78,1602.52 1169.00,1603.77 1158.67,1605.11 1136.42,1606.79">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="95542482#4" from="1107926599" to="1107926977" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="95542482#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="38.65" shape="1126.07,1607.71 1087.62,1611.66">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="95542482#5" from="1107926977" to="1107926734" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="95542482#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="25.79" shape="1062.51,1615.02 1036.98,1618.65">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="95542482#6" from="1107926734" to="1110416526" name="ถนนจุติอนุสรณ์" priority="10" type="highway.tertiary" shape="1029.82,1621.29 989.17,1627.95 980.57,1629.40 886.75,1643.20 882.92,1643.76 864.85,1646.46">\n    <lane id="95542482#6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="152.50" shape="1022.39,1620.89 988.91,1626.37 980.32,1627.82 886.52,1641.62 882.68,1642.18 871.63,1643.83">\n      <param key="origId" value="95542482" />\n    </lane>\n  </edge>\n  <edge id="95542485#0" from="1107926481" to="cluster_13472440775_1666900747" name="ถนนสามชัย" priority="10" type="highway.tertiary" shape="1340.21,1562.31 1332.12,1439.67 1332.74,1374.07 1333.40,1335.98 1333.52,1328.97 1333.87,1308.83 1335.39,1221.92 1337.16,1188.00">\n    <lane id="95542485#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="357.25" shape="1341.37,1555.54 1333.72,1439.62 1334.34,1374.09 1335.00,1336.01 1335.12,1329.00 1335.47,1308.86 1336.99,1221.98 1338.21,1198.60">\n      <param key="origId" value="95542485" />\n    </lane>\n    <param key="origTo" value="1666900747" />\n  </edge>\n  <edge id="95809258" from="1107926481" to="2505176697" name="ถนนสามชัย" priority="10" type="highway.tertiary">\n    <lane id="95809258_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="14.13" shape="1339.03,1571.39 1339.69,1585.50">\n      <param key="origId" value="95809258" />\n    </lane>\n  </edge>\n  <edge id="95809261#0" from="2506087432" to="2506087889" name="Thanon Pracharak" priority="3" type="highway.residential" spreadType="center">\n    <lane id="95809261#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="70.99" shape="641.87,1235.37 642.43,1306.36">\n      <param key="origId" value="95809261" />\n    </lane>\n  </edge>\n  <edge id="95809261#1" from="2506087889" to="2506087968" name="Thanon Pracharak" priority="3" type="highway.residential" spreadType="center">\n    <lane id="95809261#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.92" shape="642.34,1320.83 642.04,1334.74">\n      <param key="origId" value="95809261" />\n    </lane>\n  </edge>\n  <edge id="95809261#2" from="2506087968" to="cluster_2506088039_8523406713" name="Thanon Pracharak" priority="3" type="highway.residential" spreadType="center" shape="641.88,1342.00 633.99,1393.35">\n    <lane id="95809261#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="37.43" shape="640.76,1349.28 635.08,1386.27">\n      <param key="origId" value="95809261" />\n    </lane>\n    <param key="origTo" value="2506088039" />\n  </edge>\n  <edge id="95809262#0" from="cluster_1110416379_1680666461" to="1559016903" name="ถนนฉัยยากุลอุทิศ ซอย 4" priority="10" type="highway.tertiary" shape="932.07,348.59 879.58,342.38">\n    <lane id="95809262#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="38.30" shape="924.99,346.14 886.96,341.64">\n      <param key="origId" value="95809262" />\n    </lane>\n    <param key="origFrom" value="1110416379" />\n  </edge>\n  <edge id="95809262#1" from="1559016903" to="1431715372" name="ถนนฉัยยากุลอุทิศ ซอย 4" priority="10" type="highway.tertiary">\n    <lane id="95809262#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="76.31" shape="872.52,340.11 796.55,332.99">\n      <param key="origId" value="95809262" />\n    </lane>\n  </edge>\n  <edge id="95809267#0" from="1110416631" to="12346099051" name="ถนนสวนศิริ" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="95809267#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="0.20" shape="566.07,1497.93 566.08,1497.73">\n      <param key="origId" value="95809267" />\n    </lane>\n  </edge>\n  <edge id="95809267#1" from="12346099051" to="1110416470" name="ถนนสวนศิริ" priority="10" type="highway.tertiary" spreadType="center" shape="566.22,1493.89 567.76,1459.14 569.54,1419.93 573.07,1385.51">\n    <lane id="95809267#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="93.90" shape="566.55,1486.41 567.76,1459.14 569.54,1419.93 572.33,1392.71">\n      <param key="origId" value="95809267" />\n    </lane>\n  </edge>\n  <edge id="95809268#0" from="1110416470" to="2506087955" name="ถนนศุภสารรังสรรค์ ซอย 1" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="95809268#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.87" shape="572.43,1377.92 569.67,1345.17">\n      <param key="origId" value="95809268" />\n    </lane>\n  </edge>\n  <edge id="95809268#1" from="2506087955" to="2506087863" name="ถนนศุภสารรังสรรค์ ซอย 1" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="95809268#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="16.07" shape="568.27,1330.80 566.49,1314.83">\n      <param key="origId" value="95809268" />\n    </lane>\n  </edge>\n  <edge id="95809268#2" from="2506087863" to="4900465363" name="ถนนศุภสารรังสรรค์ ซอย 1" priority="10" type="highway.tertiary" spreadType="center" shape="565.65,1307.30 565.14,1257.21 564.74,1226.44">\n    <lane id="95809268#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="66.54" shape="565.58,1300.24 565.14,1257.21 564.83,1233.70">\n      <param key="origId" value="95809268" />\n    </lane>\n  </edge>\n  <edge id="95809268#3" from="4900465363" to="1110416654" name="ถนนศุภสารรังสรรค์ ซอย 1" priority="10" type="highway.tertiary" spreadType="center" shape="564.74,1226.44 564.87,1150.09 567.73,1108.83">\n    <lane id="95809268#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="103.12" shape="564.75,1219.27 564.87,1150.09 567.22,1116.24">\n      <param key="origId" value="95809268" />\n    </lane>\n  </edge>\n  <edge id="96242763#0" from="cluster_1115585393_13472440789_947896971" to="1123399771" name="ถนนบุญรองอุทิศ" priority="10" type="highway.tertiary" spreadType="center" shape="1050.46,1163.25 1060.17,1111.10">\n    <lane id="96242763#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="38.54" shape="1051.81,1156.02 1058.86,1118.13">\n      <param key="origId" value="96242763" />\n    </lane>\n    <param key="origFrom" value="1115585393" />\n  </edge>\n  <edge id="96242763#1" from="1123399771" to="1115585403" name="ถนนบุญรองอุทิศ" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="96242763#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.34" shape="1061.20,1104.06 1065.46,1075.03">\n      <param key="origId" value="96242763" />\n    </lane>\n  </edge>\n  <edge id="96242763#2" from="1115585403" to="13545019681" name="ถนนบุญรองอุทิศ" priority="10" type="highway.tertiary" spreadType="center" shape="1066.17,1070.17 1064.48,1043.80 1066.57,1031.65">\n    <lane id="96242763#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="24.12" shape="1065.86,1065.32 1064.48,1043.80 1064.91,1041.28">\n      <param key="origId" value="96242763" />\n    </lane>\n  </edge>\n  <edge id="96242763#3" from="13545019681" to="4900464584" name="ถนนบุญรองอุทิศ" priority="10" type="highway.tertiary" spreadType="center" shape="1066.57,1031.65 1062.80,1017.65 1056.06,1002.66 1053.42,994.72 1051.92,985.46 1050.82,980.45">\n    <lane id="96242763#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="37.37" shape="1064.38,1023.51 1062.80,1017.65 1056.06,1002.66 1053.42,994.72 1052.38,988.30">\n      <param key="origId" value="96242763" />\n    </lane>\n  </edge>\n  <edge id="96713525" from="2505176470" to="2505176388" name="ถนนราฎร์ยินดี ซอย 4" priority="3" type="highway.residential" shape="1991.67,1447.43 1964.46,1441.79 1870.66,1421.13 1775.69,1411.99 1768.30,1412.57 1763.55,1414.75 1761.09,1418.23 1759.19,1422.07 1759.14,1427.29 1758.26,1431.12 1746.91,1430.90 1740.65,1430.88 1606.03,1424.45">\n    <lane id="96713525_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="395.55" shape="1991.99,1445.86 1964.79,1440.23 1870.91,1419.55 1775.70,1410.38 1767.89,1411.00 1762.49,1413.48 1759.71,1417.41 1757.59,1421.69 1757.54,1427.10 1756.99,1429.50 1746.93,1429.30 1740.69,1429.28 1612.39,1423.15">\n      <param key="origId" value="96713525" />\n    </lane>\n  </edge>\n  <edge id="96713729" from="cluster_1966399241_8523333972" to="2506088084" name="ถนนแสงศรี ซอย 4" priority="10" type="highway.tertiary" spreadType="center" shape="722.38,1132.46 727.12,1383.99 725.72,1398.11">\n    <lane id="96713729_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="251.42" shape="722.52,1139.84 727.12,1383.99 726.41,1391.18">\n      <param key="origId" value="96713729" />\n    </lane>\n    <param key="origFrom" value="1966399241" />\n  </edge>\n  <edge id="96837170#0" from="cluster_1122072658_14138648129" to="1122072674" name="ถนนชนตารา" priority="10" type="highway.tertiary" shape="698.36,511.35 687.78,488.73 677.86,477.88 665.22,468.29 651.08,463.22 634.67,457.85 621.25,452.16 602.26,438.87 595.95,432.07 586.87,419.02 585.50,408.14 587.71,394.75">\n    <lane id="96837170#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="158.51" shape="695.76,502.02 689.13,487.83 678.94,476.69 665.99,466.87 651.60,461.71 635.23,456.35 622.03,450.75 603.32,437.66 597.20,431.06 588.41,418.43 587.12,408.17 588.27,401.18">\n      <param key="origId" value="96837170" />\n    </lane>\n    <param key="origFrom" value="1122072658" />\n  </edge>\n  <edge id="96837170#3" from="1122072674" to="14063214200" name="ถนนชนตารา" priority="10" type="highway.tertiary" shape="587.71,394.75 600.18,377.18 638.40,344.49 671.09,319.68 697.30,298.24 712.14,286.10 717.14,283.47">\n    <lane id="96837170#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="165.01" shape="592.37,390.94 601.37,378.27 639.40,345.74 672.08,320.94 698.31,299.48 713.03,287.44 717.88,284.89">\n      <param key="origId" value="96837170" />\n    </lane>\n  </edge>\n  <edge id="96837172#0" from="1174378788" to="1431715395" name="ถนนเสน่หานุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="96837172#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="20.45" shape="508.06,398.80 514.25,379.31">\n      <param key="origId" value="96837172" />\n    </lane>\n  </edge>\n  <edge id="96837172#1" from="1431715395" to="1122072673" name="ถนนเสน่หานุสรณ์" priority="10" type="highway.tertiary">\n    <lane id="96837172#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="66.29" shape="518.77,365.60 540.18,302.87">\n      <param key="origId" value="96837172" />\n    </lane>\n  </edge>\n  <edge id="96941841#0" from="2505176697" to="1123399700" name="ซอยชุมอุทิศ 1" priority="10" type="highway.tertiary" shape="1341.59,1591.88 1352.05,1594.15 1360.98,1596.67 1366.43,1599.00 1370.80,1601.10 1374.63,1603.37 1379.04,1605.89 1391.95,1614.51 1405.51,1624.24 1432.19,1644.11">\n    <lane id="96941841#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="90.70" shape="1349.19,1595.17 1351.66,1595.70 1360.45,1598.18 1365.77,1600.46 1370.04,1602.51 1373.83,1604.75 1378.20,1607.25 1391.04,1615.83 1404.57,1625.53 1426.21,1641.65">\n      <param key="origId" value="96941841" />\n    </lane>\n  </edge>\n  <edge id="96941841#1" from="1123399700" to="1123399513" name="ซอยชุมอุทิศ 1" priority="10" type="highway.tertiary" shape="1432.19,1644.11 1457.62,1666.20 1460.87,1669.38 1463.81,1672.96 1468.76,1680.28 1476.70,1694.34">\n    <lane id="96941841#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="50.34" shape="1437.86,1651.16 1456.53,1667.38 1459.69,1670.46 1462.53,1673.92 1467.40,1681.12 1471.35,1688.11">\n      <param key="origId" value="96941841" />\n    </lane>\n  </edge>\n  <edge id="96941841#2" from="1123399513" to="1110416567" name="ซอยชุมอุทิศ 1" priority="10" type="highway.tertiary">\n    <lane id="96941841#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="43.61" shape="1478.30,1700.97 1498.20,1739.78">\n      <param key="origId" value="96941841" />\n    </lane>\n  </edge>\n  <edge id="96941842#0" from="1123399562" to="12649445300" name="ถนนประชาธิปัตย์ 1" priority="10" type="highway.tertiary" shape="822.54,865.07 812.26,902.01 810.60,911.36">\n    <lane id="96941842#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="32.43" shape="818.84,872.40 810.70,901.65 810.34,903.69">\n      <param key="origId" value="96941842" />\n    </lane>\n  </edge>\n  <edge id="96941842#1" from="12649445300" to="1123399706" name="ถนนประชาธิปัตย์ 1" priority="10" type="highway.tertiary">\n    <lane id="96941842#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="20.01" shape="807.85,917.93 804.49,937.65">\n      <param key="origId" value="96941842" />\n    </lane>\n  </edge>\n  <edge id="96941842#2" from="1123399706" to="1123399595" name="ถนนประชาธิปัตย์ 1" priority="10" type="highway.tertiary">\n    <lane id="96941842#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="84.82" shape="802.09,952.88 790.06,1036.84">\n      <param key="origId" value="96941842" />\n    </lane>\n  </edge>\n  <edge id="96941842#3" from="1123399595" to="1966399248" name="ถนนประชาธิปัตย์ 1" priority="10" type="highway.tertiary">\n    <lane id="96941842#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="79.24" shape="787.86,1050.96 774.73,1129.10">\n      <param key="origId" value="96941842" />\n    </lane>\n  </edge>\n  <edge id="96941844" from="1431715409" to="1110416410" priority="3" type="highway.residential" spreadType="center" shape="732.80,514.38 737.02,539.97 738.38,564.10 744.78,601.61">\n    <lane id="96941844_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="73.65" shape="733.90,521.07 737.02,539.97 738.38,564.10 743.48,594.00">\n      <param key="origId" value="96941844" />\n    </lane>\n  </edge>\n  <edge id="96941849" from="1123399600" to="5149825984" name="ถนนศุภสารรังสรรค์ ซอย 9" priority="3" type="highway.residential" shape="1084.18,985.09 1086.82,994.00 1090.48,1011.70 1096.73,1029.41 1107.81,1043.83 1119.55,1057.96 1135.66,1065.63 1156.64,1067.35 1226.74,1058.06 1250.01,1055.45 1260.20,1056.96 1264.99,1059.50 1268.49,1062.04 1271.57,1065.56 1273.78,1069.30 1275.35,1073.39 1276.47,1077.41 1276.72,1079.73 1276.93,1082.33 1276.93,1084.28 1276.73,1086.74 1274.73,1096.98 1257.36,1181.54">\n    <lane id="96941849_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="339.62" shape="1084.99,993.46 1085.27,994.39 1088.94,1012.13 1095.31,1030.18 1106.56,1044.83 1118.54,1059.25 1135.24,1067.20 1156.68,1068.96 1226.93,1059.65 1249.98,1057.06 1259.69,1058.50 1264.14,1060.86 1267.40,1063.23 1270.27,1066.50 1272.33,1070.00 1273.83,1073.89 1274.89,1077.71 1275.13,1079.88 1275.33,1082.39 1275.33,1084.22 1275.14,1086.52 1273.16,1096.67 1257.97,1170.62">\n      <param key="origId" value="96941849" />\n    </lane>\n  </edge>\n  <edge id="96941857#0" from="1123399595" to="4900466448" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="790.61,1044.28 820.81,1047.80 844.49,1050.89 885.81,1053.19">\n    <lane id="96941857#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="81.19" shape="797.66,1046.71 820.61,1049.39 844.34,1052.48 878.44,1054.38">\n      <param key="origId" value="96941857" />\n    </lane>\n  </edge>\n  <edge id="96941857#1" from="4900466448" to="4900464580" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary" shape="885.81,1053.19 945.80,1058.79 969.09,1061.43">\n    <lane id="96941857#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="68.74" shape="892.96,1055.46 945.64,1060.38 961.37,1062.17">\n      <param key="origId" value="96941857" />\n    </lane>\n  </edge>\n  <edge id="96941857#2" from="4900464580" to="1966404349" name="ถนนศุภสารรังสรรค์" priority="10" type="highway.tertiary">\n    <lane id="96941857#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="44.61" shape="976.82,1063.88 1021.18,1068.69">\n      <param key="origId" value="96941857" />\n    </lane>\n  </edge>\n  <edge id="969757849#0" from="1115585385" to="4900464598" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="1029.31,977.95 1009.74,977.26 997.07,976.80 968.97,975.80">\n    <lane id="969757849#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="44.22" shape="1020.54,976.04 1009.80,975.66 997.13,975.20 976.34,974.46">\n      <param key="origId" value="969757849" />\n    </lane>\n  </edge>\n  <edge id="969757849#1" from="4900464598" to="856214538" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary" shape="968.97,975.80 929.25,973.33 896.70,972.59">\n    <lane id="969757849#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="58.10" shape="962.08,973.77 929.32,971.73 904.05,971.16">\n      <param key="origId" value="969757849" />\n    </lane>\n  </edge>\n  <edge id="969757849#2" from="856214538" to="1123399706" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary">\n    <lane id="969757849#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="80.11" shape="889.45,968.78 812.61,946.13">\n      <param key="origId" value="969757849" />\n    </lane>\n  </edge>\n  <edge id="969757849#3" from="1123399706" to="2506085690" name="ถนนประชาธิปัตย์" priority="10" type="highway.tertiary">\n    <lane id="969757849#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="40.42" shape="797.94,941.64 759.42,929.39">\n      <param key="origId" value="969757849" />\n    </lane>\n  </edge>\n  <edge id="969757851#0" from="1160612974" to="cluster_1123399587_8976429007" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center" shape="893.63,694.00 924.41,697.03 1051.85,708.64 1100.90,712.74">\n    <lane id="969757851#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="191.63" shape="899.06,694.53 924.41,697.03 1051.85,708.64 1089.91,711.82">\n      <param key="origId" value="969757851" />\n    </lane>\n    <param key="origTo" value="1123399587" />\n  </edge>\n  <edge id="969757851#1" from="cluster_1123399587_8976429007" to="14159239425" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center" shape="1100.90,712.74 1134.54,716.08">\n    <lane id="969757851#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="19.89" shape="1107.94,713.44 1127.73,715.40">\n      <param key="origId" value="969757851" />\n    </lane>\n    <param key="origFrom" value="1123399587" />\n  </edge>\n  <edge id="969757851#2" from="14159239425" to="2506085525" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center" shape="1134.54,716.08 1185.65,721.18 1353.51,736.98">\n    <lane id="969757851#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="204.82" shape="1142.40,716.86 1185.65,721.18 1346.30,736.30">\n      <param key="origId" value="969757851" />\n    </lane>\n  </edge>\n  <edge id="969757851#3" from="2506085525" to="8976429009" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="969757851#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="11.79" shape="1360.64,737.66 1372.37,738.78">\n      <param key="origId" value="969757851" />\n    </lane>\n  </edge>\n  <edge id="969757851#4" from="8976429009" to="cluster_849403384_849403500_8976428987_8976428988" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center" shape="1379.76,739.49 1465.60,749.07 1467.33,749.27 1470.39,749.50">\n    <lane id="969757851#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="77.07" shape="1386.76,740.27 1463.36,748.82">\n      <param key="origId" value="969757851" />\n    </lane>\n    <param key="origTo" value="849403384" />\n  </edge>\n  <edge id="969757852" from="cluster_849403384_849403500_8976428987_8976428988" to="8976429008" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center" shape="1482.20,750.31 1499.05,751.54 1499.60,751.57">\n    <lane id="969757852_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="4.46" shape="1489.85,750.87 1494.30,751.19">\n      <param key="origId" value="969757852 969757853" />\n    </lane>\n    <param key="origFrom" value="849403500" />\n  </edge>\n  <edge id="969757853#1" from="8976429008" to="13449811511" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center" shape="1499.60,751.57 1628.82,762.17 1670.51,765.50">\n    <lane id="969757853#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="157.93" shape="1505.60,752.06 1628.82,762.17 1663.00,764.90">\n      <param key="origId" value="969757853" />\n    </lane>\n  </edge>\n  <edge id="969757853#2" from="13449811511" to="2505176027" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center" shape="1670.51,765.50 1790.52,775.42 1792.69,775.56 1828.67,777.73 1855.54,779.68">\n    <lane id="969757853#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="171.48" shape="1677.46,766.07 1790.52,775.42 1792.69,775.56 1828.67,777.73 1848.43,779.16">\n      <param key="origId" value="969757853" />\n    </lane>\n  </edge>\n  <edge id="969757855#1" from="2507400638" to="8976429008" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="969757855#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="163.26" shape="1524.37,919.03 1500.48,757.52">\n      <param key="origId" value="969757855" />\n    </lane>\n  </edge>\n  <edge id="969765780#1" from="cluster_1123399587_8976429007" to="14159239427" name="Thumnoonvithi Road" priority="3" type="highway.residential" shape="1100.90,712.74 1077.61,728.60">\n    <lane id="969765780#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="10.99" shape="1091.33,717.32 1082.25,723.51">\n      <param key="origId" value="969765780" />\n    </lane>\n    <param key="origFrom" value="1123399587" />\n  </edge>\n  <edge id="969765780#2" from="14159239427" to="1115585416" name="Thumnoonvithi Road" priority="3" type="highway.residential" shape="1077.61,728.60 1072.06,732.75 996.73,797.55 957.51,845.29">\n    <lane id="969765780#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="152.88" shape="1070.31,732.14 995.58,796.43 961.11,838.39">\n      <param key="origId" value="969765780" />\n    </lane>\n  </edge>\n  <edge id="969765781" from="1431715406" to="1431715392" name="ถนนชีอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="969765781_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="23.15" shape="798.98,604.77 775.91,602.85">\n      <param key="origId" value="969765781" />\n    </lane>\n  </edge>\n  <edge id="969765782" from="1431715392" to="1110416410" name="ถนนชีอุทิศ" priority="10" type="highway.tertiary">\n    <lane id="969765782_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="13.94" shape="764.53,601.84 750.65,600.55">\n      <param key="origId" value="969765782" />\n    </lane>\n  </edge>\n  <edge id="970010765#0" from="cluster_849403384_849403500_8976428987_8976428988" to="1107926433" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center" shape="1469.77,745.74 1430.81,741.89 1384.91,735.44">\n    <lane id="970010765#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="72.33" shape="1463.72,745.14 1430.81,741.89 1391.93,736.43">\n      <param key="origId" value="970010765" />\n    </lane>\n    <param key="origFrom" value="8976428988" />\n  </edge>\n  <edge id="970010765#1" from="1107926433" to="cluster_1123399587_8976429007" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center" shape="1384.91,735.44 1349.01,731.47 1216.53,719.07 1101.79,707.56">\n    <lane id="970010765#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="270.52" shape="1377.68,734.64 1349.01,731.47 1216.53,719.07 1108.46,708.23">\n      <param key="origId" value="970010765" />\n    </lane>\n    <param key="origTo" value="8976429007" />\n  </edge>\n  <edge id="970010765#2" from="cluster_1123399587_8976429007" to="1558997825" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center" shape="1101.79,707.56 1079.32,705.31 1053.99,703.38">\n    <lane id="970010765#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="27.83" shape="1090.45,706.42 1079.32,705.31 1062.72,704.05">\n      <param key="origId" value="970010765" />\n    </lane>\n    <param key="origFrom" value="8976429007" />\n  </edge>\n  <edge id="970010765#3" from="1558997825" to="8976428985" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center" shape="1053.99,703.38 951.03,695.54 906.41,691.59 902.89,689.83 900.73,687.97 898.63,684.92 896.05,678.00">\n    <lane id="970010765#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="151.48" shape="1047.24,702.87 951.03,695.54 906.41,691.59 902.89,689.83 900.73,687.97 898.79,685.16">\n      <param key="origId" value="970010765" />\n    </lane>\n  </edge>\n  <edge id="970010767#0" from="8976428996" to="2505176020" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center" shape="1855.51,772.01 1850.51,771.78 1804.07,768.58 1793.36,768.00 1711.93,763.62 1628.90,758.14">\n    <lane id="970010767#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="219.81" shape="1855.51,772.01 1850.51,771.78 1804.07,768.58 1793.36,768.00 1711.93,763.62 1636.11,758.62">\n      <param key="origId" value="970010767" />\n    </lane>\n  </edge>\n  <edge id="970010767#1" from="2505176020" to="2507400593" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center" shape="1628.90,758.14 1611.06,756.96 1522.71,750.07 1498.79,747.93">\n    <lane id="970010767#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="119.13" shape="1621.74,757.67 1611.06,756.96 1522.71,750.07 1502.98,748.30">\n      <param key="origId" value="970010767" />\n    </lane>\n  </edge>\n  <edge id="970010767#2" from="2507400593" to="cluster_849403384_849403500_8976428987_8976428988" name="ถนนธรรมนูญวิถี" priority="10" type="highway.tertiary" spreadType="center" shape="1498.79,747.93 1495.99,747.68 1482.91,746.72 1481.60,746.67">\n    <lane id="970010767#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="3.57" shape="1493.67,747.51 1490.12,747.25">\n      <param key="origId" value="970010767 970010766 990212970" />\n    </lane>\n    <param key="origTo" value="8976428987" />\n  </edge>\n  <edge id="970311985#1" from="916327945" to="916327888" name="ถนนเพชรเกษม" priority="11" type="highway.secondary" spreadType="center" shape="1084.32,2036.67 1071.79,2031.05 1031.40,2006.19 988.50,1982.70 944.42,1962.95 900.58,1943.74 858.46,1924.13 747.14,1869.85 713.18,1852.97 619.17,1806.28 598.16,1796.77 496.83,1746.02 473.02,1731.97 462.10,1720.09">\n    <lane id="970311985#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="692.20" shape="1084.32,2036.67 1071.79,2031.05 1031.40,2006.19 988.50,1982.70 944.42,1962.95 900.58,1943.74 858.46,1924.13 747.14,1869.85 713.18,1852.97 619.17,1806.28 598.16,1796.77 496.83,1746.02 473.02,1731.97 467.12,1725.55">\n      <param key="origId" value="970311985" />\n    </lane>\n  </edge>\n  <edge id="970323545" from="2507400847" to="947896981" priority="10" type="highway.tertiary" spreadType="center">\n    <lane id="970323545_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="170.90" shape="1681.29,1879.16 1653.50,1710.54">\n      <param key="origId" value="970323545" />\n    </lane>\n  </edge>\n  <edge id="970323546#0" from="2505176985" to="1680668223" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center">\n    <lane id="970323546#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="169.88" shape="1668.48,1880.49 1641.09,1712.83">\n      <param key="origId" value="970323546" />\n    </lane>\n    <lane id="970323546#0_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="169.88" shape="1665.32,1881.01 1637.93,1713.35">\n      <param key="origId" value="970323546" />\n    </lane>\n  </edge>\n  <edge id="970323546#1" from="1680668223" to="2505176514" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center">\n    <lane id="970323546#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="199.30" shape="1638.76,1698.62 1606.63,1501.93">\n      <param key="origId" value="970323546" />\n    </lane>\n    <lane id="970323546#1_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="199.30" shape="1635.61,1699.13 1603.47,1502.44">\n      <param key="origId" value="970323546" />\n    </lane>\n  </edge>\n  <edge id="970323546#2" from="2505176514" to="2505176494" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center">\n    <lane id="970323546#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="6.41" shape="1604.30,1487.67 1603.26,1481.34">\n      <param key="origId" value="970323546" />\n    </lane>\n    <lane id="970323546#2_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="6.41" shape="1601.14,1488.19 1600.10,1481.86">\n      <param key="origId" value="970323546" />\n    </lane>\n  </edge>\n  <edge id="970323546#3" from="2505176494" to="cluster_2505176422_2505176434" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1600.47,1474.18 1598.00,1459.01 1592.59,1425.94">\n    <lane id="970323546#3_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.60" shape="1600.94,1467.10 1599.58,1458.75 1595.36,1432.95">\n      <param key="origId" value="970323546" />\n    </lane>\n    <lane id="970323546#3_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="34.60" shape="1597.78,1467.62 1596.42,1459.27 1592.20,1433.47">\n      <param key="origId" value="970323546" />\n    </lane>\n    <param key="origTo" value="2505176422" />\n  </edge>\n  <edge id="970323546#4" from="cluster_2505176422_2505176434" to="2507400698" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1592.59,1425.94 1579.78,1347.54">\n    <lane id="970323546#4_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="64.85" shape="1593.03,1418.73 1582.58,1354.74">\n      <param key="origId" value="970323546" />\n    </lane>\n    <lane id="970323546#4_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="64.85" shape="1589.88,1419.25 1579.42,1355.25">\n      <param key="origId" value="970323546" />\n    </lane>\n    <param key="origFrom" value="2505176422" />\n  </edge>\n  <edge id="970323546#5" from="2507400698" to="2507400690" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center">\n    <lane id="970323546#5_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="51.21" shape="1580.25,1340.49 1571.99,1289.95">\n      <param key="origId" value="970323546" />\n    </lane>\n    <lane id="970323546#5_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="51.21" shape="1577.09,1341.01 1568.83,1290.46">\n      <param key="origId" value="970323546" />\n    </lane>\n  </edge>\n  <edge id="970323546#6" from="2507400690" to="2505176258" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center">\n    <lane id="970323546#6_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="68.56" shape="1569.66,1275.67 1558.60,1208.01">\n      <param key="origId" value="970323546" />\n    </lane>\n    <lane id="970323546#6_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="68.56" shape="1566.50,1276.18 1555.44,1208.52">\n      <param key="origId" value="970323546" />\n    </lane>\n  </edge>\n  <edge id="970323546#7" from="2505176258" to="cluster_11292971032_2406954578_2406954579_2406954580" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1555.83,1200.99 1552.58,1179.46">\n    <lane id="970323546#7_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="8.90" shape="1556.32,1193.53 1554.99,1184.70">\n      <param key="origId" value="970323546" />\n    </lane>\n    <lane id="970323546#7_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="8.90" shape="1553.16,1194.01 1551.83,1185.24">\n      <param key="origId" value="970323546" />\n    </lane>\n    <param key="origTo" value="2406954580" />\n  </edge>\n  <edge id="980451721#0" from="1123399676" to="2506089309" priority="10" type="highway.tertiary" shape="995.81,1681.63 1011.08,1664.51 1034.52,1642.36">\n    <lane id="980451721#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="43.70" shape="1001.84,1677.27 1012.23,1665.63 1032.65,1646.33">\n      <param key="origId" value="980451721" />\n    </lane>\n  </edge>\n  <edge id="980451721#1" from="2506089309" to="1107926977" priority="10" type="highway.tertiary">\n    <lane id="980451721#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="29.19" shape="1041.09,1640.31 1066.17,1625.38">\n      <param key="origId" value="980451721" />\n    </lane>\n  </edge>\n  <edge id="980451722#0" from="1123399676" to="1123399569" priority="10" type="highway.tertiary" spreadType="center" shape="995.81,1681.63 939.79,1751.21 935.41,1755.41 931.03,1758.36 926.66,1759.99 921.91,1760.88 916.85,1761.30 911.05,1761.46 895.68,1761.01 879.44,1759.57">\n    <lane id="980451722#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="139.36" shape="991.26,1687.28 939.79,1751.21 935.41,1755.41 931.03,1758.36 926.66,1759.99 921.91,1760.88 916.85,1761.30 911.05,1761.46 895.68,1761.01 885.54,1760.11">\n      <param key="origId" value="980451722" />\n    </lane>\n  </edge>\n  <edge id="98567749" from="1140280758" to="1140280757" name="ถนนแสงศรี ซอย 2" priority="10" type="highway.tertiary" spreadType="center" shape="677.38,1402.31 677.60,1355.90 691.90,1129.98">\n    <lane id="98567749_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="258.01" shape="677.42,1394.77 677.60,1355.90 691.44,1137.19">\n      <param key="origId" value="98567749" />\n    </lane>\n  </edge>\n  <edge id="990212966#0" from="796720075" to="796747714" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1350.52,1.19 1350.06,10.33 1357.23,54.43 1364.83,102.30 1368.80,126.31 1375.95,169.63 1377.00,175.95 1378.15,182.93 1382.65,210.53 1385.13,225.82 1395.08,286.90 1453.95,648.54">\n    <lane id="990212966#0_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="648.28" shape="1350.52,1.19 1350.06,10.33 1357.23,54.43 1364.83,102.30 1368.80,126.31 1375.95,169.63 1377.00,175.95 1378.15,182.93 1382.65,210.53 1385.13,225.82 1395.08,286.90 1452.75,641.16">\n      <param key="origId" value="990212966" />\n    </lane>\n  </edge>\n  <edge id="990212966#1" from="796747714" to="1680667730" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center">\n    <lane id="990212966#1_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="49.51" shape="1455.07,655.44 1463.04,704.31">\n      <param key="origId" value="990212966" />\n    </lane>\n  </edge>\n  <edge id="990212966#2" from="1680667730" to="cluster_849403384_849403500_8976428987_8976428988" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1464.25,711.76 1466.50,725.60 1469.77,745.74">\n    <lane id="990212966#2_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="21.38" shape="1465.37,718.63 1466.50,725.60 1468.80,739.74">\n      <param key="origId" value="990212966" />\n    </lane>\n    <param key="origTo" value="8976428988" />\n  </edge>\n  <edge id="990212968" from="cluster_849403384_849403500_8976428987_8976428988" to="8642866006" name="ถนนราษฎร์ยินดี" priority="11" type="highway.secondary" spreadType="center" shape="1481.60,746.67 1477.33,720.55">\n    <lane id="990212968_0" index="0" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.87" shape="1481.74,737.62 1480.15,727.88">\n      <param key="origId" value="990212968" />\n    </lane>\n    <lane id="990212968_1" index="1" disallow="tram rail_urban rail rail_electric rail_fast ship container cable_car subway aircraft wheelchair scooter drone" speed="13.89" length="9.87" shape="1478.58,738.14 1476.99,728.39">\n      <param key="origId" value="990212968" />\n    </lane>\n    <param key="origFrom" value="8976428987" />\n  </edge>\n  <tlLogic id="2052547074" type="static" programID="0" offset="0">\n    <phase duration="20" state="GGGrrrr" />\n    <phase duration="3" state="yyyrrrr" />\n    <phase duration="2" state="rrrrrrr" />\n    <phase duration="20" state="rrrGGrr" />\n    <phase duration="3" state="rrryyrr" />\n    <phase duration="2" state="rrrrrrr" />\n    <phase duration="20" state="rrrrrGG" />\n    <phase duration="3" state="rrrrryy" />\n    <phase duration="2" state="rrrrrrr" />\n  </tlLogic>\n  <tlLogic id="4085816302" type="static" programID="0" offset="0">\n    <phase duration="20" state="Grrrrrr" />\n    <phase duration="3" state="yrrrrrr" />\n    <phase duration="2" state="rrrrrrr" />\n    <phase duration="20" state="rGGGGrr" />\n    <phase duration="3" state="ryyyyrr" />\n    <phase duration="2" state="rrrrrrr" />\n    <phase duration="20" state="rrrrrGG" />\n    <phase duration="3" state="rrrrryy" />\n    <phase duration="2" state="rrrrrrr" />\n  </tlLogic>\n  <tlLogic id="947896964" type="static" programID="0" offset="0">\n    <phase duration="20" state="GGrrrr" />\n    <phase duration="3" state="yyrrrr" />\n    <phase duration="2" state="rrrrrr" />\n    <phase duration="20" state="rrGGrr" />\n    <phase duration="3" state="rryyrr" />\n    <phase duration="2" state="rrrrrr" />\n    <phase duration="20" state="rrrrGG" />\n    <phase duration="3" state="rrrryy" />\n    <phase duration="2" state="rrrrrr" />\n  </tlLogic>\n  <tlLogic id="cluster_1115585393_13472440789_947896971" type="static" programID="0" offset="0">\n    <phase duration="20" state="GGGrrrrrrrrr" />\n    <phase duration="3" state="yyyrrrrrrrrr" />\n    <phase duration="2" state="rrrrrrrrrrrr" />\n    <phase duration="20" state="rrrGGGrrrrrr" />\n    <phase duration="3" state="rrryyyrrrrrr" />\n    <phase duration="2" state="rrrrrrrrrrrr" />\n    <phase duration="20" state="rrrrrrGGGrrr" />\n    <phase duration="3" state="rrrrrryyyrrr" />\n    <phase duration="2" state="rrrrrrrrrrrr" />\n    <phase duration="20" state="rrrrrrrrrGGG" />\n    <phase duration="3" state="rrrrrrrrryyy" />\n    <phase duration="2" state="rrrrrrrrrrrr" />\n  </tlLogic>\n  <tlLogic id="cluster_849403384_849403500_8976428987_8976428988" type="static" programID="0" offset="0">\n    <phase duration="20" state="GGGrrrrrrrrrr" />\n    <phase duration="3" state="yyyrrrrrrrrrr" />\n    <phase duration="2" state="rrrrrrrrrrrrr" />\n    <phase duration="20" state="rrrGGGGrrrrrr" />\n    <phase duration="3" state="rrryyyyrrrrrr" />\n    <phase duration="2" state="rrrrrrrrrrrrr" />\n    <phase duration="20" state="rrrrrrrGGGrrr" />\n    <phase duration="3" state="rrrrrrryyyrrr" />\n    <phase duration="2" state="rrrrrrrrrrrrr" />\n    <phase duration="20" state="rrrrrrrrrrGGG" />\n    <phase duration="3" state="rrrrrrrrrryyy" />\n    <phase duration="2" state="rrrrrrrrrrrrr" />\n  </tlLogic>\n  <tlLogic id="cluster_916327730_916327962_985092239_985092700" type="static" programID="0" offset="0">\n    <phase duration="20" state="GGrrrrrrr" />\n    <phase duration="3" state="yyrrrrrrr" />\n    <phase duration="2" state="rrrrrrrrr" />\n    <phase duration="20" state="rrGGGGrrr" />\n    <phase duration="3" state="rryyyyrrr" />\n    <phase duration="2" state="rrrrrrrrr" />\n    <phase duration="20" state="rrrrrrGGG" />\n    <phase duration="3" state="rrrrrryyy" />\n    <phase duration="2" state="rrrrrrrrr" />\n  </tlLogic>\n  <tlLogic id="joinedS_809537820_856214571" type="static" programID="0" offset="0">\n    <phase duration="20" state="GGrrrrrrrrrr" />\n    <phase duration="3" state="yyrrrrrrrrrr" />\n    <phase duration="2" state="rrrrrrrrrrrr" />\n    <phase duration="20" state="rrGGrrrrrrrr" />\n    <phase duration="3" state="rryyrrrrrrrr" />\n    <phase duration="2" state="rrrrrrrrrrrr" />\n    <phase duration="20" state="rrrrGGGrrrrr" />\n    <phase duration="3" state="rrrryyyrrrrr" />\n    <phase duration="2" state="rrrrrrrrrrrr" />\n    <phase duration="20" state="rrrrrrrGGrrr" />\n    <phase duration="3" state="rrrrrrryyrrr" />\n    <phase duration="2" state="rrrrrrrrrrrr" />\n    <phase duration="20" state="rrrrrrrrrGGG" />\n    <phase duration="3" state="rrrrrrrrryyy" />\n    <phase duration="2" state="rrrrrrrrrrrr" />\n  </tlLogic>\n  <junction id="1107784878" type="priority" x="634.65" y="505.81" incLanes="706256379#3_0 -706256379#4_0" intLanes=":1107784878_0_0 :1107784878_4_0 :1107784878_2_0 :1107784878_3_0" shape="640.34,503.09 639.78,509.47 637.50,509.73 636.65,510.24 635.98,511.00 635.50,512.00 635.22,513.25 632.05,512.80 631.91,510.56 631.47,509.75 630.77,509.13 629.82,508.71 628.62,508.49 629.18,502.12">\n    <request index="0" response="0000" foes="0000" cont="0" />\n    <request index="1" response="1100" foes="1100" cont="1" />\n    <request index="2" response="0000" foes="0010" cont="0" />\n    <request index="3" response="0000" foes="0010" cont="0" />\n  </junction>\n  <junction id="1107784879" type="priority" x="534.83" y="1006.09" incLanes="706093504#0_0 684714761#3_0" intLanes=":1107784879_0_0 :1107784879_1_0 :1107784879_2_0 :1107784879_3_0" shape="534.44,1000.40 537.56,1001.08 537.53,1003.27 537.91,1004.12 538.54,1004.81 539.44,1005.33 540.59,1005.69 539.92,1008.82 537.75,1008.79 536.92,1009.16 536.27,1009.79 535.80,1010.68 535.50,1011.83 532.34,1011.30 532.28,1009.02 531.88,1008.15 531.23,1007.45 530.33,1006.93 529.18,1006.59 529.81,1003.45 532.01,1003.45 532.86,1003.07 533.55,1002.43 534.08,1001.54">\n    <request index="0" response="1100" foes="1100" cont="0" />\n    <request index="1" response="1000" foes="1000" cont="0" />\n    <request index="2" response="0000" foes="0001" cont="0" />\n    <request index="3" response="0000" foes="0011" cont="0" />\n  </junction>\n  <junction id="1107784883" type="priority" x="622.62" y="591.02" incLanes="95533433#0_0 706256377#1_0 -706256377#2_0" intLanes=":1107784883_0_0 :1107784883_1_0 :1107784883_2_0 :1107784883_3_0 :1107784883_7_0 :1107784883_5_0 :1107784883_6_0" shape="622.05,583.58 625.22,584.03 625.36,586.26 625.80,587.08 626.50,587.70 627.45,588.12 628.65,588.33 628.09,594.71 625.58,594.96 624.64,595.47 623.90,596.23 623.37,597.25 623.04,598.52 619.88,598.01 619.79,595.78 619.36,594.97 618.67,594.36 617.73,593.95 616.53,593.74 617.04,587.36 619.57,587.10 620.51,586.59 621.24,585.84 621.75,584.83">\n    <request index="0" response="0001000" foes="0001000" cont="0" />\n    <request index="1" response="1111000" foes="1111000" cont="0" />\n    <request index="2" response="1011000" foes="1011000" cont="0" />\n    <request index="3" response="0000000" foes="0000111" cont="0" />\n    <request index="4" response="1100000" foes="1100110" cont="1" />\n    <request index="5" response="0000000" foes="0010010" cont="0" />\n    <request index="6" response="0000000" foes="0010110" cont="0" />\n  </junction>\n  <junction id="1107784884" type="priority" x="611.14" y="662.62" incLanes="95533433#1_0 40264697#5_0 40264697#5_1" intLanes=":1107784884_0_0 :1107784884_1_0 :1107784884_2_0 :1107784884_3_0 :1107784884_3_1" shape="610.71,655.18 613.87,655.68 613.96,657.92 614.39,658.75 615.08,659.39 616.02,659.82 617.22,660.07 616.52,666.43 613.97,666.64 612.99,667.16 612.21,667.93 611.63,668.98 611.24,670.30 608.10,669.68 608.09,667.48 607.69,666.70 607.02,666.13 606.09,665.78 604.89,665.63 605.08,659.23 607.91,658.85 608.97,658.29 609.79,657.50 610.37,656.46">\n    <request index="0" response="11100" foes="11100" cont="0" />\n    <request index="1" response="11000" foes="11000" cont="0" />\n    <request index="2" response="00000" foes="00001" cont="0" />\n    <request index="3" response="00000" foes="00011" cont="0" />\n    <request index="4" response="00000" foes="00011" cont="0" />\n  </junction>\n  <junction id="1107784887" type="priority" x="519.48" y="1095.16" incLanes="706093504#2_0 706104219#14_0 -706104219#15_0" intLanes=":1107784887_0_0 :1107784887_1_0 :1107784887_2_0 :1107784887_3_0 :1107784887_7_0 :1107784887_5_0 :1107784887_6_0" shape="519.31,1087.51 522.45,1088.11 522.48,1090.65 522.94,1091.67 523.68,1092.52 524.72,1093.19 526.05,1093.69 524.30,1099.85 522.09,1099.68 521.24,1100.02 520.57,1100.63 520.07,1101.53 519.74,1102.70 516.60,1102.13 516.50,1099.54 516.00,1098.47 515.19,1097.54 514.07,1096.76 512.65,1096.13 514.79,1090.10 516.97,1090.40 517.81,1090.12 518.47,1089.54 518.98,1088.67">\n    <request index="0" response="0001000" foes="0001000" cont="0" />\n    <request index="1" response="1111000" foes="1111000" cont="0" />\n    <request index="2" response="1011000" foes="1011000" cont="0" />\n    <request index="3" response="0000000" foes="0000111" cont="0" />\n    <request index="4" response="1100000" foes="1100110" cont="1" />\n    <request index="5" response="0000000" foes="0010010" cont="0" />\n    <request index="6" response="0000000" foes="0010110" cont="0" />\n  </junction>\n  <junction id="1107926433" type="priority" x="1384.91" y="735.44" incLanes="-706094740#0_0 970010765#0_0" intLanes=":1107926433_0_0 :1107926433_1_0 :1107926433_2_0" shape="1382.10,729.50 1388.48,729.92 1388.76,732.41 1389.26,733.33 1389.99,734.05 1390.95,734.55 1392.15,734.84 1391.71,738.01 1377.51,736.23 1377.86,733.05 1380.10,732.85 1380.90,732.38 1381.51,731.67 1381.91,730.71">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="1107926467" type="priority" x="1507.76" y="1686.25" incLanes="-498671427_0 243188710#0_0 -243188710#1_0" intLanes=":1107926467_0_0 :1107926467_1_0 :1107926467_2_0 :1107926467_3_0 :1107926467_4_0 :1107926467_6_0" shape="1515.63,1681.70 1516.68,1688.01 1514.70,1688.77 1514.20,1689.38 1514.02,1690.15 1514.17,1691.09 1514.65,1692.18 1509.04,1695.25 1507.66,1692.70 1506.63,1690.81 1505.73,1689.28 1504.73,1687.82 1503.41,1686.13 1501.55,1683.91 1506.42,1679.76 1508.60,1681.32 1510.02,1681.76 1511.67,1681.97 1513.54,1681.95">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="1107926481" type="priority" x="1340.21" y="1562.31" incLanes="243188710#2_0 -95809258_0 -95542482#0_0 -95542485#3_0" intLanes=":1107926481_0_0 :1107926481_1_0 :1107926481_2_0 :1107926481_3_0 :1107926481_4_0 :1107926481_12_0 :1107926481_6_0 :1107926481_7_0 :1107926481_8_0 :1107926481_9_0 :1107926481_10_0 :1107926481_13_0" shape="1349.76,1562.76 1347.34,1568.68 1345.31,1568.29 1344.61,1568.55 1344.13,1569.11 1343.87,1569.99 1343.83,1571.16 1337.43,1571.46 1336.87,1568.73 1336.28,1567.87 1335.50,1567.35 1334.51,1567.16 1333.31,1567.32 1331.77,1561.10 1334.56,1559.96 1335.51,1559.16 1336.16,1558.21 1336.52,1557.11 1336.58,1555.86 1342.96,1555.44 1343.86,1558.32 1344.82,1559.60 1346.12,1560.76 1347.77,1561.82">\n    <request index="0" response="000000010000" foes="000100010000" cont="0" />\n    <request index="1" response="111000110000" foes="111100110000" cont="0" />\n    <request index="2" response="110011110000" foes="110111110000" cont="0" />\n    <request index="3" response="000000000000" foes="100010000000" cont="0" />\n    <request index="4" response="000000000000" foes="100110000111" cont="0" />\n    <request index="5" response="011000000000" foes="111110000110" cont="1" />\n    <request index="6" response="010000000000" foes="010000000100" cont="0" />\n    <request index="7" response="110000111000" foes="110000111100" cont="0" />\n    <request index="8" response="110000110111" foes="110000110111" cont="0" />\n    <request index="9" response="000000000000" foes="000000100010" cont="0" />\n    <request index="10" response="000000000000" foes="000111100110" cont="0" />\n    <request index="11" response="000000111000" foes="000110111110" cont="1" />\n  </junction>\n  <junction id="1107926513" type="priority" x="1334.88" y="1134.02" incLanes="-706105951#2_0 -900244003#0_0 706105951#0_0" intLanes=":1107926513_0_0 :1107926513_6_0 :1107926513_2_0 :1107926513_3_0 :1107926513_4_0 :1107926513_5_0" shape="1331.69,1126.92 1338.09,1126.92 1338.54,1129.15 1339.10,1129.95 1339.89,1130.52 1340.91,1130.89 1342.15,1131.03 1341.97,1137.43 1339.74,1137.81 1338.94,1138.34 1338.36,1139.10 1338.00,1140.09 1337.85,1141.30 1331.45,1141.10">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1107926537" type="priority" x="1336.13" y="806.65" incLanes="-706105950_0 706105951#4_0 -243188741#0_0" intLanes=":1107926537_0_0 :1107926537_6_0 :1107926537_2_0 :1107926537_3_0 :1107926537_4_0 :1107926537_5_0" shape="1339.13,800.77 1342.71,806.08 1340.90,807.90 1340.30,809.08 1339.89,810.43 1339.68,811.95 1339.66,813.64 1333.27,813.95 1332.73,811.76 1332.14,811.01 1331.34,810.49 1330.33,810.20 1329.11,810.14 1328.85,803.74 1331.16,803.65 1332.91,803.54 1334.33,803.30 1335.68,802.82 1337.20,802.01">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1107926561" type="priority" x="1286.69" y="1575.57" incLanes="95542482#0_0 -95542482#1_0 243188585_0" intLanes=":1107926561_0_0 :1107926561_1_0 :1107926561_2_0 :1107926561_6_0 :1107926561_4_0 :1107926561_5_0" shape="1293.61,1570.56 1295.15,1576.77 1281.34,1580.46 1279.55,1574.31 1281.66,1573.09 1282.38,1572.12 1282.87,1570.92 1283.13,1569.47 1283.17,1567.78 1289.56,1567.52 1290.09,1569.64 1290.66,1570.28 1291.44,1570.65 1292.42,1570.74">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="1107926599" type="priority" x="1131.66" y="1608.75" incLanes="95542482#2_0 -95542482#4_0" intLanes=":1107926599_0_0 :1107926599_1_0 :1107926599_2_0 :1107926599_4_0" shape="1136.30,1605.19 1136.78,1611.57 1126.56,1612.49 1125.90,1606.12 1127.83,1605.52 1128.44,1604.96 1128.82,1604.21 1128.96,1603.29 1128.87,1602.20 1132.02,1601.63 1132.80,1603.73 1133.43,1604.44 1134.22,1604.92 1135.18,1605.17">\n    <request index="0" response="0000" foes="1000" cont="0" />\n    <request index="1" response="0000" foes="1000" cont="0" />\n    <request index="2" response="0000" foes="0000" cont="0" />\n    <request index="3" response="0011" foes="0011" cont="1" />\n  </junction>\n  <junction id="1107926639" type="priority" x="1335.59" y="881.55" incLanes="-706105951#5_0 -243188629_0 706105951#3_0 243188351#0_0" intLanes=":1107926639_0_0 :1107926639_1_0 :1107926639_2_0 :1107926639_3_0 :1107926639_4_0 :1107926639_12_0 :1107926639_6_0 :1107926639_7_0 :1107926639_8_0 :1107926639_9_0 :1107926639_10_0 :1107926639_13_0" shape="1332.64,874.49 1339.04,874.72 1339.49,877.06 1340.13,877.95 1341.05,878.65 1342.24,879.16 1343.70,879.49 1342.79,885.82 1340.59,885.95 1339.83,886.39 1339.29,887.09 1338.97,888.04 1338.87,889.25 1332.47,889.32 1332.00,886.88 1331.44,886.05 1330.66,885.47 1329.65,885.15 1328.43,885.08 1328.14,878.69 1330.58,878.13 1331.44,877.54 1332.08,876.73 1332.48,875.72">\n    <request index="0" response="000000010000" foes="000100010000" cont="0" />\n    <request index="1" response="111000110000" foes="111100110000" cont="0" />\n    <request index="2" response="110111110000" foes="110111110000" cont="0" />\n    <request index="3" response="000000000000" foes="100010000000" cont="0" />\n    <request index="4" response="000000000000" foes="100110000111" cont="0" />\n    <request index="5" response="111000000000" foes="111110000110" cont="1" />\n    <request index="6" response="010000000000" foes="010000000100" cont="0" />\n    <request index="7" response="110000111000" foes="110000111100" cont="0" />\n    <request index="8" response="110000110011" foes="110000110111" cont="0" />\n    <request index="9" response="000000000000" foes="000000100010" cont="0" />\n    <request index="10" response="000000000000" foes="000111100110" cont="0" />\n    <request index="11" response="000000011000" foes="000110111110" cont="1" />\n  </junction>\n  <junction id="1107926734" type="priority" x="1029.82" y="1621.29" incLanes="95542482#5_0 -95542482#9_0 243188670#1_0 243188670#1_1" intLanes=":1107926734_0_0 :1107926734_1_0 :1107926734_2_0 :1107926734_3_0 :1107926734_4_0 :1107926734_5_0 :1107926734_5_1 :1107926734_7_0" shape="1036.75,1617.07 1037.66,1623.41 1035.55,1624.15 1034.90,1624.81 1034.52,1625.64 1034.41,1626.67 1034.57,1627.87 1028.32,1629.27 1027.32,1626.94 1026.55,1626.19 1025.60,1625.72 1024.47,1625.53 1023.17,1625.62 1022.13,1619.31 1024.25,1618.51 1024.91,1617.84 1025.32,1616.98 1025.47,1615.95 1025.36,1614.73 1031.65,1613.57 1032.56,1615.79 1033.31,1616.50 1034.26,1616.96 1035.41,1617.14">\n    <request index="0" response="11110000" foes="11110000" cont="0" />\n    <request index="1" response="11101000" foes="11101000" cont="0" />\n    <request index="2" response="01100000" foes="01100000" cont="0" />\n    <request index="3" response="11100000" foes="11100010" cont="0" />\n    <request index="4" response="00000000" foes="00000001" cont="0" />\n    <request index="5" response="00000000" foes="00001111" cont="0" />\n    <request index="6" response="00000000" foes="00001111" cont="0" />\n    <request index="7" response="00000000" foes="00001011" cont="0" />\n  </junction>\n  <junction id="1107926940" type="priority" x="1243.71" y="1588.07" incLanes="95542482#1_0 -95542482#3_0 243188552#2_0" intLanes=":1107926940_0_0 :1107926940_1_0 :1107926940_2_0 :1107926940_6_0 :1107926940_4_0 :1107926940_5_0" shape="1250.60,1582.73 1252.39,1588.88 1238.20,1592.85 1236.55,1586.66 1238.68,1585.47 1239.40,1584.50 1239.90,1583.28 1240.17,1581.81 1240.22,1580.09 1246.61,1579.85 1247.12,1581.95 1247.68,1582.57 1248.45,1582.91 1249.42,1582.96">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="1107926968" type="priority" x="1532.35" y="1731.19" incLanes="243314395_0 -243188710#0_0" intLanes=":1107926968_0_0 :1107926968_1_0" shape="1533.98,1727.50 1535.48,1733.73 1532.57,1733.89 1531.32,1733.51 1530.20,1732.84 1529.22,1731.86 1528.37,1730.58">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="1107926977" type="priority" x="1082.50" y="1613.79" incLanes="95542482#4_0 980451721#1_0 -95542482#5_0" intLanes=":1107926977_0_0 :1107926977_1_0 :1107926977_9_0 :1107926977_3_0 :1107926977_4_0 :1107926977_5_0 :1107926977_6_0 :1107926977_7_0 :1107926977_10_0" shape="1087.45,1610.07 1088.11,1616.43 1083.18,1617.20 1079.68,1618.30 1076.95,1619.79 1074.35,1621.67 1071.24,1623.98 1066.98,1626.75 1063.71,1621.25 1063.19,1619.77 1062.28,1613.44 1067.76,1612.54 1072.20,1611.58 1075.60,1610.53 1077.95,1609.41 1079.25,1608.22 1079.52,1606.95 1082.66,1606.33 1083.55,1608.59 1084.26,1609.33 1085.15,1609.83 1086.22,1610.07">\n    <request index="0" response="000000000" foes="100010000" cont="0" />\n    <request index="1" response="000000000" foes="100110000" cont="0" />\n    <request index="2" response="011000000" foes="011110000" cont="1" />\n    <request index="3" response="010000000" foes="010000000" cont="0" />\n    <request index="4" response="110000111" foes="110000111" cont="0" />\n    <request index="5" response="110000110" foes="110000110" cont="0" />\n    <request index="6" response="000000000" foes="000000100" cont="0" />\n    <request index="7" response="000000000" foes="000111100" cont="0" />\n    <request index="8" response="000000011" foes="000110011" cont="1" />\n  </junction>\n  <junction id="1110416342" type="dead_end" x="1397.40" y="1757.83" incLanes="243188503#1_0" intLanes="" shape="1397.40,1757.83 1394.33,1756.91 1397.40,1757.83" />\n  <junction id="1110416410" type="priority" x="744.78" y="601.61" incLanes="969765782_0 1365892991_0 -706256377#0_0 96941844_0" intLanes=":1110416410_0_0 :1110416410_1_0 :1110416410_2_0 :1110416410_3_0 :1110416410_4_0 :1110416410_5_0" shape="750.80,598.96 750.20,605.33 747.97,605.57 747.17,606.05 746.58,606.77 746.20,607.74 746.03,608.95 742.83,608.80 742.39,606.46 741.73,605.60 740.79,604.94 739.56,604.49 738.04,604.24 738.59,597.87 740.73,597.62 741.41,597.14 741.83,596.42 742.00,595.46 741.90,594.27 745.06,593.73 746.06,596.43 746.90,597.42 747.97,598.17 749.27,598.68">\n    <request index="0" response="000000" foes="110100" cont="0" />\n    <request index="1" response="001000" foes="101000" cont="0" />\n    <request index="2" response="011001" foes="011001" cont="0" />\n    <request index="3" response="000000" foes="100110" cont="0" />\n    <request index="4" response="000001" foes="000101" cont="0" />\n    <request index="5" response="001011" foes="001011" cont="0" />\n  </junction>\n  <junction id="1110416444" type="priority" x="644.04" y="1123.85" incLanes="706104219#12_0 -1365892748#1_0 -706104219#13_0" intLanes=":1110416444_0_0 :1110416444_1_0 :1110416444_9_0 :1110416444_3_0 :1110416444_4_0 :1110416444_5_0 :1110416444_6_0 :1110416444_7_0 :1110416444_10_0" shape="644.65,1116.20 647.71,1117.11 647.68,1119.32 648.20,1120.16 649.09,1120.83 650.33,1121.32 651.94,1121.64 651.13,1127.98 648.91,1128.15 648.13,1128.60 647.56,1129.31 647.21,1130.27 647.07,1131.47 640.67,1131.33 640.30,1128.55 639.77,1127.49 639.02,1126.67 638.05,1126.07 636.86,1125.70 638.09,1119.42 641.17,1119.52 642.37,1119.13 643.35,1118.45 644.11,1117.47">\n    <request index="0" response="000000000" foes="100010000" cont="0" />\n    <request index="1" response="000000000" foes="100110000" cont="0" />\n    <request index="2" response="011000000" foes="011110000" cont="1" />\n    <request index="3" response="010000000" foes="010000000" cont="0" />\n    <request index="4" response="110000111" foes="110000111" cont="0" />\n    <request index="5" response="110000110" foes="110000110" cont="0" />\n    <request index="6" response="000000000" foes="000000100" cont="0" />\n    <request index="7" response="000000000" foes="000111100" cont="0" />\n    <request index="8" response="000000011" foes="000110011" cont="1" />\n  </junction>\n  <junction id="1110416454" type="dead_end" x="1302.99" y="568.78" incLanes="-95533432#0_0" intLanes="" shape="1302.99,568.78 1302.78,571.97 1302.99,568.78" />\n  <junction id="1110416469" type="priority" x="751.64" y="1662.91" incLanes="95542482#11_0 -100401320_0 -95542482#12_0" intLanes=":1110416469_0_0 :1110416469_6_0 :1110416469_2_0 :1110416469_3_0 :1110416469_4_0 :1110416469_5_0" shape="758.51,1658.52 759.56,1664.83 757.48,1665.63 756.86,1666.29 756.51,1667.14 756.43,1668.16 756.63,1669.36 750.43,1670.97 749.42,1668.84 748.68,1668.21 747.79,1667.86 746.74,1667.80 745.54,1668.03 743.74,1661.89 746.84,1660.99 749.14,1660.35 751.05,1659.87 752.97,1659.47 755.33,1659.05">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1110416470" type="priority" x="573.07" y="1385.51" incLanes="706104217#16_0 95809267#1_0 -706104217#17_0" intLanes=":1110416470_0_0 :1110416470_1_0 :1110416470_2_0 :1110416470_3_0 :1110416470_4_0 :1110416470_5_0 :1110416470_6_0" shape="579.12,1383.06 578.30,1389.41 576.05,1389.57 575.22,1390.02 574.59,1390.72 574.16,1391.68 573.92,1392.88 570.74,1392.55 570.44,1390.23 569.86,1389.35 568.99,1388.67 567.83,1388.18 566.39,1387.88 567.20,1381.53 569.37,1381.37 570.10,1380.92 570.58,1380.21 570.83,1379.26 570.84,1378.05 574.02,1377.79 574.76,1380.45 575.49,1381.44 576.46,1382.21 577.67,1382.75">\n    <request index="0" response="0001000" foes="1001000" cont="0" />\n    <request index="1" response="0011000" foes="1011000" cont="0" />\n    <request index="2" response="0000000" foes="0100000" cont="0" />\n    <request index="3" response="0000000" foes="1100011" cont="0" />\n    <request index="4" response="0000000" foes="1100010" cont="0" />\n    <request index="5" response="0011100" foes="0011100" cont="0" />\n    <request index="6" response="0011011" foes="0011011" cont="0" />\n  </junction>\n  <junction id="1110416526" type="priority" x="864.85" y="1646.46" incLanes="95542482#6_0 -243188273#1_0 -95542482#10_0" intLanes=":1110416526_0_0 :1110416526_6_0 :1110416526_2_0 :1110416526_3_0 :1110416526_4_0 :1110416526_5_0" shape="871.39,1642.25 872.34,1648.58 870.21,1649.35 869.51,1650.02 869.06,1650.89 868.85,1651.94 868.89,1653.18 862.54,1653.96 861.83,1651.82 861.19,1651.11 860.36,1650.65 859.33,1650.44 858.11,1650.47 857.36,1644.11">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1110416567" type="dead_end" x="1499.62" y="1739.05" incLanes="96941841#2_0" intLanes="" shape="1499.62,1739.05 1496.77,1740.51 1499.62,1739.05" />\n  <junction id="1110416568" type="priority" x="741.15" y="676.65" incLanes="706520617#4_0 40264697#9_0 40264697#9_1" intLanes=":1110416568_0_0 :1110416568_1_0 :1110416568_2_0 :1110416568_3_0 :1110416568_4_0" shape="739.90,669.30 743.10,669.45 743.56,672.79 744.28,674.04 745.33,675.02 746.70,675.72 748.40,676.15 747.91,679.31 745.56,679.65 744.60,680.38 743.77,681.49 743.09,682.97 742.55,684.82 736.31,683.38 736.37,681.17 735.99,680.36 735.34,679.74 734.42,679.32 733.23,679.10 733.85,672.73 737.13,672.60 738.29,672.15 739.14,671.46 739.68,670.50">\n    <request index="0" response="01000" foes="01000" cont="0" />\n    <request index="1" response="11000" foes="11000" cont="0" />\n    <request index="2" response="00000" foes="00000" cont="0" />\n    <request index="3" response="00000" foes="00011" cont="0" />\n    <request index="4" response="00000" foes="00010" cont="0" />\n  </junction>\n  <junction id="1110416631" type="priority" x="566.14" y="1496.10" incLanes="-243188786#0_0" intLanes=":1110416631_0_0" shape="564.59,1494.57 567.79,1494.69 568.17,1495.80 568.54,1496.13 568.92,1496.47 569.22,1496.91 569.33,1497.58 562.93,1497.56 563.35,1496.41 563.74,1496.07 564.14,1495.73 564.45,1495.27">\n    <request index="0" response="0" foes="0" cont="0" />\n  </junction>\n  <junction id="1110416654" type="priority" x="567.73" y="1108.83" incLanes="706104219#13_0 95809268#3_0 -706104219#14_0" intLanes=":1110416654_0_0 :1110416654_1_0 :1110416654_2_0 :1110416654_3_0" shape="574.25,1106.85 573.01,1113.13 570.80,1113.15 570.00,1113.55 569.40,1114.22 569.01,1115.15 568.81,1116.35 565.62,1116.13 565.37,1113.52 564.89,1112.49 564.19,1111.66 563.25,1111.01 562.08,1110.56 563.83,1104.40">\n    <request index="0" response="0000" foes="0100" cont="0" />\n    <request index="1" response="1000" foes="1000" cont="0" />\n    <request index="2" response="1001" foes="1001" cont="0" />\n    <request index="3" response="0000" foes="0110" cont="0" />\n  </junction>\n  <junction id="1110416677" type="dead_end" x="946.62" y="206.52" incLanes="-706256380#0_0" intLanes="" shape="946.62,206.52 949.81,206.75 946.62,206.52" />\n  <junction id="1110416683" type="dead_end" x="814.75" y="279.64" incLanes="-148941242#0_0" intLanes="" shape="814.75,279.64 817.35,281.51 814.75,279.64" />\n  <junction id="1110416726" type="priority" x="796.31" y="306.04" incLanes="148941245_0 142455600#4_0 -148941242#1_0 -129827554#0_0" intLanes=":1110416726_0_0 :1110416726_1_0 :1110416726_2_0 :1110416726_3_0 :1110416726_4_0 :1110416726_12_0 :1110416726_6_0 :1110416726_7_0 :1110416726_8_0 :1110416726_9_0 :1110416726_10_0 :1110416726_13_0" shape="799.49,295.32 804.96,298.68 804.10,300.42 804.19,300.97 804.64,301.32 805.43,301.45 806.58,301.37 807.47,307.71 804.79,308.23 802.50,309.01 800.60,310.03 799.10,311.30 797.99,312.81 797.27,314.57 791.10,312.86 791.25,310.70 790.90,309.92 790.27,309.34 789.36,308.96 788.17,308.78 788.54,302.39 792.25,301.75 794.09,300.78 795.90,299.39 797.70,297.57">\n    <request index="0" response="000000010000" foes="000100010000" cont="0" />\n    <request index="1" response="111000110000" foes="111100110000" cont="0" />\n    <request index="2" response="110011110000" foes="110011110000" cont="0" />\n    <request index="3" response="000000000000" foes="100010000000" cont="0" />\n    <request index="4" response="000000000000" foes="100110000111" cont="0" />\n    <request index="5" response="011000000000" foes="111110000110" cont="1" />\n    <request index="6" response="010000000000" foes="010000000100" cont="0" />\n    <request index="7" response="110000111000" foes="110000111100" cont="0" />\n    <request index="8" response="110000110011" foes="110000110011" cont="0" />\n    <request index="9" response="000000000000" foes="000000100010" cont="0" />\n    <request index="10" response="000000000000" foes="000111100110" cont="0" />\n    <request index="11" response="000000111000" foes="000110111110" cont="1" />\n  </junction>\n  <junction id="1110416742" type="priority" x="1343.06" y="1630.86" incLanes="-243188765#1_0 497290263#4_0 243188765#0_0" intLanes=":1110416742_0_0 :1110416742_6_0 :1110416742_2_0 :1110416742_3_0 :1110416742_4_0 :1110416742_5_0" shape="1346.65,1638.43 1340.26,1638.76 1339.71,1636.63 1339.13,1635.96 1338.34,1635.57 1337.34,1635.44 1336.14,1635.58 1334.82,1629.32 1337.55,1628.30 1338.49,1627.54 1339.15,1626.62 1339.52,1625.54 1339.61,1624.29 1346.01,1624.04">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1110416754" type="priority" x="609.49" y="1704.59" incLanes="95542482#13_0 -95542482#14_0 243188786#4_0" intLanes=":1110416754_0_0 :1110416754_1_0 :1110416754_2_0 :1110416754_6_0 :1110416754_4_0 :1110416754_5_0" shape="615.66,1699.45 617.46,1705.59 603.65,1709.65 601.84,1703.51 603.86,1702.41 604.47,1701.61 604.80,1700.63 604.87,1699.48 604.67,1698.16 610.90,1696.68 611.84,1698.72 612.55,1699.32 613.43,1699.64 614.46,1699.68">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="1115585385" type="priority" x="1029.31" y="977.95" incLanes="1204844051_0 -969757849#0_0 -142453602#0_0" intLanes=":1115585385_0_0 :1115585385_1_0 :1115585385_9_0 :1115585385_3_0 :1115585385_4_0 :1115585385_10_0 :1115585385_6_0 :1115585385_7_0 :1115585385_8_0" shape="1035.58,975.46 1034.84,981.81 1032.60,982.00 1031.81,982.46 1031.23,983.18 1030.88,984.14 1030.74,985.35 1027.54,985.27 1026.78,982.92 1025.79,982.07 1024.39,981.44 1022.59,981.03 1020.37,980.84 1020.59,974.44 1022.66,974.10 1023.23,973.61 1023.51,972.91 1023.49,971.98 1023.16,970.85 1029.14,968.56 1030.00,970.54 1030.95,972.18 1031.98,973.50 1033.10,974.48 1034.30,975.14">\n    <request index="0" response="000000000" foes="000100000" cont="0" />\n    <request index="1" response="000000000" foes="111100000" cont="0" />\n    <request index="2" response="000011000" foes="110011000" cont="1" />\n    <request index="3" response="000000000" foes="010000100" cont="0" />\n    <request index="4" response="000000000" foes="110000100" cont="0" />\n    <request index="5" response="000000011" foes="110000011" cont="1" />\n    <request index="6" response="000000010" foes="000000010" cont="0" />\n    <request index="7" response="000111110" foes="000111110" cont="0" />\n    <request index="8" response="000110110" foes="000110110" cont="0" />\n  </junction>\n  <junction id="1115585403" type="priority" x="1066.17" y="1070.17" incLanes="96242763#1_0 -154097280_0" intLanes=":1115585403_0_0 :1115585403_1_0 :1115585403_3_0" shape="1067.04,1075.26 1063.87,1074.80 1063.83,1073.97 1063.66,1073.67 1063.40,1073.46 1063.05,1073.32 1062.59,1073.27 1062.78,1066.87 1063.64,1066.73 1063.94,1066.53 1064.14,1066.25 1064.25,1065.88 1064.26,1065.42 1067.46,1065.22 1067.59,1067.36 1067.65,1068.94 1067.64,1070.25 1067.54,1071.56 1067.35,1073.14">\n    <request index="0" response="000" foes="100" cont="0" />\n    <request index="1" response="000" foes="100" cont="0" />\n    <request index="2" response="011" foes="011" cont="1" />\n  </junction>\n  <junction id="1115585416" type="priority" x="957.51" y="845.29" incLanes="969765780#2_0 142453602#2_0 -706520617#0_0" intLanes=":1115585416_0_0 :1115585416_1_0 :1115585416_2_0 :1115585416_3_0 :1115585416_4_0 :1115585416_5_0" shape="959.87,837.37 964.82,841.43 963.78,843.39 963.76,844.29 964.06,845.13 964.70,845.93 965.66,846.67 962.23,852.07 959.39,850.77 958.09,850.57 956.87,850.67 955.72,851.06 954.64,851.76 950.60,846.80 952.77,845.00 954.35,843.63 955.58,842.43 956.74,841.16 958.09,839.55">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="0" />\n  </junction>\n  <junction id="1115585463" type="priority" x="998.04" y="876.32" incLanes="-243188024_0 142453602#1_0 -142453602#2_0" intLanes=":1115585463_0_0 :1115585463_1_0 :1115585463_2_0 :1115585463_3_0 :1115585463_4_0 :1115585463_6_0" shape="1000.59,869.90 1004.77,874.74 1003.29,876.49 1002.90,877.41 1002.75,878.38 1002.82,879.37 1003.13,880.41 997.19,882.79 996.37,880.83 995.68,879.39 994.97,878.28 994.10,877.29 992.92,876.21 991.29,874.84 995.41,869.93 997.14,870.90 998.00,871.02 998.87,870.89 999.73,870.52">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="1115585792" type="priority" x="778.12" y="839.56" incLanes="706520617#2_0 -706520617#3_0" intLanes=":1115585792_0_0 :1115585792_4_0 :1115585792_2_0 :1115585792_3_0" shape="784.30,839.42 781.12,844.97 779.64,844.48 779.04,844.59 778.54,844.93 778.13,845.50 777.82,846.30 774.74,845.40 775.06,843.24 774.94,842.32 774.62,841.50 774.12,840.79 773.43,840.19 777.09,834.94 779.69,836.71 780.62,837.30 781.57,837.86">\n    <request index="0" response="0000" foes="0000" cont="0" />\n    <request index="1" response="1100" foes="1100" cont="1" />\n    <request index="2" response="0000" foes="0010" cont="0" />\n    <request index="3" response="0000" foes="0010" cont="0" />\n  </junction>\n  <junction id="1115585857" type="priority" x="1023.70" y="1713.27" incLanes="497290261#2_0 243079656_0" intLanes=":1115585857_0_0 :1115585857_1_0" shape="1024.76,1708.39 1027.98,1709.88 1026.54,1711.12 1026.01,1711.96 1025.60,1712.93 1025.31,1714.04 1025.15,1715.30 1021.96,1715.05 1021.88,1712.90 1021.63,1712.16 1021.26,1711.45 1020.79,1710.60 1020.23,1709.46 1023.15,1708.15 1023.61,1708.79 1023.86,1708.90 1024.14,1708.87 1024.44,1708.70">\n    <request index="0" response="00" foes="10" cont="0" />\n    <request index="1" response="01" foes="01" cont="0" />\n  </junction>\n  <junction id="1115585909" type="priority" x="1007.86" y="921.98" incLanes="142453602#0_0 498671393_0 -142453602#1_0" intLanes=":1115585909_0_0 :1115585909_6_0 :1115585909_2_0 :1115585909_3_0 :1115585909_4_0 :1115585909_5_0" shape="1013.04,926.56 1007.07,928.85 1005.69,926.76 1004.69,926.01 1003.47,925.46 1002.04,925.12 1000.40,924.97 1000.59,918.57 1002.80,918.19 1003.57,917.66 1004.11,916.89 1004.42,915.90 1004.50,914.68 1010.90,914.55 1010.95,917.16 1011.03,919.11 1011.20,920.71 1011.55,922.27 1012.14,924.12">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1122072640" type="dead_end" x="372.96" y="277.33" incLanes="" intLanes="" shape="377.72,277.91 368.20,276.75" />\n  <junction id="1122072645" type="dead_end" x="446.53" y="285.66" incLanes="706096643#1_0 84842974#17_0 84842974#17_1 84842974#17_2" intLanes="" shape="453.02,284.84 452.63,288.02 451.79,288.08 451.48,288.25 451.25,288.52 451.09,288.88 451.00,289.33 441.45,288.42 442.94,285.83 444.57,285.06 446.80,284.64 449.61,284.56" />\n  <junction id="1122072673" type="priority" x="540.46" y="297.10" incLanes="-129827558_0 96837172#1_0" intLanes=":1122072673_0_0 :1122072673_1_0 :1122072673_2_0 :1122072673_4_0" shape="538.80,292.10 544.85,294.18 541.70,303.38 535.64,301.32 535.86,299.57 535.62,298.92 535.14,298.42 534.42,298.07 533.47,297.86 533.86,294.68 536.13,294.61 537.04,294.27 537.78,293.74 538.37,293.02">\n    <request index="0" response="0000" foes="1000" cont="0" />\n    <request index="1" response="0000" foes="1000" cont="0" />\n    <request index="2" response="0000" foes="0000" cont="0" />\n    <request index="3" response="0011" foes="0011" cont="1" />\n  </junction>\n  <junction id="1122072674" type="priority" x="587.71" y="394.75" incLanes="-96837170#4_0 96837170#0_0 129827552#1_0" intLanes=":1122072674_0_0 :1122072674_1_0 :1122072674_2_0 :1122072674_6_0 :1122072674_4_0 :1122072674_5_0" shape="588.46,388.17 593.68,391.87 592.42,393.69 591.55,395.09 590.96,396.33 590.55,397.64 590.22,399.26 589.85,401.45 583.53,400.40 583.48,398.38 583.13,397.57 582.56,396.89 581.78,396.35 580.78,395.93 582.69,389.83 585.00,390.16 586.01,390.00 586.92,389.61 587.74,389.00">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="1122072681" type="priority" x="494.55" y="491.05" incLanes="706096644#0_0 706256379#5_0" intLanes=":1122072681_0_0 :1122072681_1_0 :1122072681_2_0 :1122072681_4_0" shape="492.22,483.50 498.58,484.26 498.76,486.51 499.22,487.35 499.93,487.99 500.88,488.43 502.08,488.68 501.37,495.04 498.16,495.12 496.98,495.55 496.09,496.23 495.48,497.17 495.16,498.36 491.99,497.93 492.29,494.88 492.30,492.60 492.16,490.72 492.00,488.84 491.98,486.56">\n    <request index="0" response="0000" foes="1000" cont="0" />\n    <request index="1" response="0000" foes="1000" cont="0" />\n    <request index="2" response="0000" foes="0000" cont="0" />\n    <request index="3" response="0011" foes="0011" cont="1" />\n  </junction>\n  <junction id="1123399483" type="priority" x="1159.26" y="1424.69" incLanes="706094741#1_0 243188565_0 -706094741#2_0" intLanes=":1123399483_0_0 :1123399483_6_0 :1123399483_2_0 :1123399483_3_0 :1123399483_4_0 :1123399483_5_0" shape="1165.89,1421.03 1166.33,1427.41 1164.09,1428.05 1163.29,1428.72 1162.70,1429.62 1162.32,1430.74 1162.15,1432.10 1155.75,1431.83 1155.40,1429.60 1154.88,1428.81 1154.13,1428.24 1153.14,1427.89 1151.92,1427.76 1152.04,1421.36">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1123399486" type="priority" x="504.75" y="1606.61" incLanes="-243188085_0 684741141#2_0 -684741141#3_0" intLanes=":1123399486_0_0 :1123399486_1_0 :1123399486_2_0 :1123399486_3_0 :1123399486_4_0 :1123399486_6_0" shape="511.99,1602.12 513.08,1608.43 511.05,1609.22 510.48,1609.87 510.21,1610.69 510.24,1611.68 510.57,1612.84 504.59,1615.13 503.41,1612.05 502.56,1609.75 501.93,1607.84 501.41,1605.89 500.87,1603.51 500.21,1600.27 506.48,1599.01 507.45,1601.10 508.26,1601.75 509.28,1602.13 510.53,1602.26">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="1123399489" type="priority" x="1044.60" y="1401.15" incLanes="706094741#3_0 243188594#1_0 -684741338_0" intLanes=":1123399489_0_0 :1123399489_1_0 :1123399489_2_0 :1123399489_3_0" shape="1052.95,1400.92 1050.66,1406.90 1048.66,1406.54 1048.02,1406.81 1047.63,1407.38 1047.48,1408.25 1047.57,1409.41 1044.41,1409.93 1043.46,1406.58 1042.75,1405.28 1041.88,1404.24 1040.85,1403.44 1039.67,1402.90 1041.65,1396.82">\n    <request index="0" response="0100" foes="0100" cont="0" />\n    <request index="1" response="0000" foes="1000" cont="0" />\n    <request index="2" response="0000" foes="1001" cont="0" />\n    <request index="3" response="0110" foes="0110" cont="0" />\n  </junction>\n  <junction id="1123399492" type="dead_end" x="775.12" y="1753.00" incLanes="100401320_0" intLanes="" shape="775.12,1753.00 772.02,1753.81 775.12,1753.00" />\n  <junction id="1123399502" type="priority" x="565.91" y="1520.01" incLanes="243188785_0 1257722738#0_0 -243188786#1_0 684741141#3_0" intLanes=":1123399502_0_0 :1123399502_1_0 :1123399502_2_0 :1123399502_3_0 :1123399502_4_0 :1123399502_5_0 :1123399502_9_0 :1123399502_7_0 :1123399502_8_0" shape="562.88,1512.64 569.28,1513.21 569.74,1516.08 570.34,1517.08 571.17,1517.80 572.24,1518.22 573.55,1518.36 573.57,1521.56 571.45,1522.01 570.80,1522.56 570.43,1523.32 570.32,1524.30 570.49,1525.50 564.27,1526.97 563.22,1524.78 562.37,1523.99 561.31,1523.39 560.03,1523.01 558.53,1522.82 558.87,1516.43 561.09,1516.10 561.87,1515.59 562.43,1514.84 562.77,1513.86">\n    <request index="0" response="000000000" foes="001001000" cont="0" />\n    <request index="1" response="000000000" foes="111011000" cont="0" />\n    <request index="2" response="000100000" foes="100100000" cont="0" />\n    <request index="3" response="001100011" foes="101100011" cont="0" />\n    <request index="4" response="011100010" foes="011100010" cont="0" />\n    <request index="5" response="000000000" foes="100011100" cont="0" />\n    <request index="6" response="000000011" foes="100011011" cont="1" />\n    <request index="7" response="000000010" foes="000010010" cont="0" />\n    <request index="8" response="001101110" foes="001101110" cont="0" />\n  </junction>\n  <junction id="1123399513" type="priority" x="1476.70" y="1694.34" incLanes="-96941841#2_0 -243188196_0 96941841#1_0" intLanes=":1123399513_0_0 :1123399513_6_0 :1123399513_2_0 :1123399513_3_0 :1123399513_4_0 :1123399513_5_0" shape="1482.57,1698.78 1476.88,1701.70 1475.28,1699.89 1474.21,1699.40 1472.94,1699.20 1471.48,1699.28 1469.84,1699.64 1468.07,1693.48 1469.99,1692.48 1470.47,1691.79 1470.62,1690.96 1470.45,1689.99 1469.95,1688.90 1475.53,1685.75">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1123399516" type="priority" x="1239.67" y="1387.29" incLanes="243188705_0 -243188552#0_0 -706094741#0_0" intLanes=":1123399516_0_0 :1123399516_1_0 :1123399516_2_0 :1123399516_6_0 :1123399516_4_0 :1123399516_5_0" shape="1236.74,1382.65 1243.13,1383.03 1242.96,1386.78 1242.96,1389.56 1243.10,1391.84 1243.35,1394.12 1243.67,1396.89 1244.04,1400.63 1237.67,1401.19 1237.18,1399.40 1236.73,1399.04 1236.15,1399.04 1235.44,1399.40 1234.58,1400.11 1228.02,1397.57 1230.50,1395.43 1232.61,1393.12 1234.29,1390.68 1235.54,1388.12 1236.36,1385.45">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="1123399535" type="priority" x="1134.93" y="1234.89" incLanes="452036923#2_0 -452037066_0 -452036923#3_0" intLanes=":1123399535_0_0 :1123399535_6_0 :1123399535_2_0 :1123399535_3_0 :1123399535_4_0 :1123399535_5_0" shape="1143.49,1233.27 1142.04,1239.60 1139.85,1239.88 1139.09,1240.32 1138.54,1240.98 1138.21,1241.84 1138.09,1242.91 1131.69,1242.88 1131.29,1239.49 1130.78,1238.16 1130.06,1237.07 1129.13,1236.22 1128.00,1235.61 1130.32,1229.65 1133.03,1230.76 1135.04,1231.62 1136.73,1232.26 1138.46,1232.73 1140.59,1233.05">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1123399561" type="priority" x="1184.19" y="1233.15" incLanes="452036923#1_0 -452036922_0 -452036923#2_0" intLanes=":1123399561_0_0 :1123399561_6_0 :1123399561_2_0 :1123399561_3_0 :1123399561_4_0 :1123399561_5_0" shape="1188.73,1227.91 1191.09,1233.85 1189.02,1235.30 1188.29,1236.37 1187.76,1237.67 1187.44,1239.21 1187.32,1240.97 1180.92,1240.92 1180.50,1238.76 1179.96,1238.08 1179.20,1237.64 1178.21,1237.47 1177.01,1237.55 1175.98,1231.24 1178.76,1230.79 1180.82,1230.45 1182.50,1230.09 1184.15,1229.62 1186.11,1228.92">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1123399562" type="priority" x="822.54" y="865.07" incLanes="706520617#1_0 -96941842#0_0 -706520617#2_0" intLanes=":1123399562_0_0 :1123399562_1_0 :1123399562_2_0 :1123399562_3_0 :1123399562_4_0 :1123399562_5_0" shape="831.14,866.18 828.03,871.78 825.97,871.13 825.14,871.29 824.45,871.77 823.89,872.57 823.46,873.69 817.30,871.97 817.74,868.77 817.55,867.45 817.08,866.32 816.34,865.37 815.32,864.61 818.51,859.07">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="0" />\n  </junction>\n  <junction id="1123399569" type="priority" x="879.44" y="1759.57" incLanes="980451722#0_0 243188273#0_0" intLanes=":1123399569_0_0" shape="885.68,1758.52 885.40,1761.70 882.63,1761.32 880.32,1760.70 878.48,1759.85 877.10,1758.77 876.18,1757.45 875.73,1755.90 882.07,1755.06 882.66,1756.85 883.18,1757.51 883.86,1758.01 884.69,1758.34">\n    <request index="0" response="0" foes="0" cont="0" />\n  </junction>\n  <junction id="1123399591" type="priority" x="572.08" y="1715.62" incLanes="95542482#14_0 1257722739#2_0 243188243_0 -684741141#0_0" intLanes=":1123399591_0_0 :1123399591_1_0 :1123399591_2_0 :1123399591_3_0 :1123399591_4_0 :1123399591_5_0 :1123399591_9_0 :1123399591_7_0 :1123399591_8_0" shape="578.77,1710.31 580.58,1716.45 578.78,1717.37 578.45,1717.96 578.50,1718.63 578.93,1719.38 579.75,1720.22 577.63,1722.63 574.07,1720.78 571.88,1720.47 569.42,1720.56 566.68,1721.05 563.67,1721.95 561.56,1715.90 563.30,1714.90 563.59,1714.32 563.49,1713.67 563.00,1712.96 562.13,1712.19 565.95,1707.06 568.19,1708.55 570.38,1709.66 572.53,1710.39 574.65,1710.74 576.73,1710.72">\n    <request index="0" response="000000000" foes="001001000" cont="0" />\n    <request index="1" response="000000000" foes="111011000" cont="0" />\n    <request index="2" response="000100000" foes="100100000" cont="0" />\n    <request index="3" response="001100011" foes="101100011" cont="0" />\n    <request index="4" response="011100010" foes="111100010" cont="0" />\n    <request index="5" response="000000000" foes="100011100" cont="0" />\n    <request index="6" response="000000011" foes="100011011" cont="1" />\n    <request index="7" response="000000010" foes="000010010" cont="0" />\n    <request index="8" response="001111110" foes="001111110" cont="0" />\n  </junction>\n  <junction id="1123399595" type="priority" x="790.61" y="1044.28" incLanes="96941842#2_0 -96941857#0_0 -96941842#3_0" intLanes=":1123399595_0_0 :1123399595_6_0 :1123399595_2_0 :1123399595_3_0 :1123399595_4_0 :1123399595_5_0" shape="788.48,1036.61 794.81,1037.52 794.94,1039.77 795.38,1040.61 796.07,1041.25 797.02,1041.69 798.22,1041.94 797.48,1048.30 795.06,1048.47 794.15,1048.92 793.44,1049.62 792.92,1050.56 792.60,1051.76 786.29,1050.69">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1123399600" type="priority" x="1084.18" y="985.09" incLanes="185951074#2_0 -96941849_0 -1204844052_0 498671615_0" intLanes=":1123399600_0_0 :1123399600_1_0 :1123399600_12_0 :1123399600_3_0 :1123399600_4_0 :1123399600_5_0 :1123399600_6_0 :1123399600_7_0 :1123399600_13_0 :1123399600_9_0 :1123399600_10_0 :1123399600_11_0" shape="1092.41,981.83 1092.46,988.23 1090.36,988.67 1089.74,989.21 1089.41,989.96 1089.36,990.92 1089.59,992.09 1083.46,993.91 1082.69,992.07 1081.55,990.48 1080.05,989.17 1078.18,988.12 1075.95,987.33 1073.35,986.81 1074.23,980.47 1076.21,980.39 1076.71,980.02 1076.88,979.42 1076.71,978.59 1076.22,977.55 1081.79,974.39 1083.26,976.68 1084.86,978.54 1086.57,979.99 1088.40,981.02 1090.34,981.64">\n    <request index="0" response="000000000000" foes="000100010000" cont="0" />\n    <request index="1" response="000000000000" foes="111100110000" cont="0" />\n    <request index="2" response="000011000000" foes="110011110000" cont="1" />\n    <request index="3" response="000010000000" foes="100010000000" cont="0" />\n    <request index="4" response="000110000111" foes="100110000111" cont="0" />\n    <request index="5" response="011110000110" foes="011110000110" cont="0" />\n    <request index="6" response="000000000000" foes="010000000100" cont="0" />\n    <request index="7" response="000000000000" foes="110000111100" cont="0" />\n    <request index="8" response="000000000011" foes="110000110011" cont="1" />\n    <request index="9" response="000000000010" foes="000000100010" cont="0" />\n    <request index="10" response="000111000110" foes="000111100110" cont="0" />\n    <request index="11" response="000110011110" foes="000110011110" cont="0" />\n  </junction>\n  <junction id="1123399639" type="priority" x="1230.93" y="1179.84" incLanes="530314558#1_0 706104218_0 -706104219#0_0" intLanes=":1123399639_0_0 :1123399639_1_0 :1123399639_2_0 :1123399639_3_0 :1123399639_4_0 :1123399639_6_0" shape="1229.53,1171.57 1235.77,1172.99 1235.72,1175.18 1236.11,1175.98 1236.76,1176.57 1237.68,1176.96 1238.88,1177.14 1238.47,1183.53 1223.86,1182.40 1224.45,1176.03 1226.79,1175.70 1227.71,1175.10 1228.48,1174.21 1229.09,1173.03">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="1123399641" type="priority" x="1199.28" y="1419.81" incLanes="706094741#0_0 -243188212#1_0 -706094741#1_0 243188212#0_0" intLanes=":1123399641_0_0 :1123399641_1_0 :1123399641_2_0 :1123399641_3_0 :1123399641_4_0 :1123399641_12_0 :1123399641_6_0 :1123399641_7_0 :1123399641_8_0 :1123399641_9_0 :1123399641_10_0 :1123399641_13_0" shape="1204.47,1412.82 1207.85,1418.25 1206.21,1419.44 1204.87,1420.77 1203.83,1422.25 1203.09,1423.88 1202.65,1425.66 1202.51,1427.58 1196.11,1427.61 1195.66,1425.46 1195.11,1424.78 1194.35,1424.36 1193.36,1424.20 1192.15,1424.29 1191.06,1417.99 1193.16,1417.13 1193.76,1416.41 1194.06,1415.49 1194.06,1414.38 1193.76,1413.08 1199.85,1411.12 1200.91,1413.00 1201.62,1413.45 1202.45,1413.57 1203.40,1413.36">\n    <request index="0" response="000000010000" foes="000100010000" cont="0" />\n    <request index="1" response="111000110000" foes="111100110000" cont="0" />\n    <request index="2" response="110011110000" foes="110011110000" cont="0" />\n    <request index="3" response="000000000000" foes="100010000000" cont="0" />\n    <request index="4" response="000000000000" foes="100110000111" cont="0" />\n    <request index="5" response="011000000000" foes="011110000110" cont="1" />\n    <request index="6" response="010000000000" foes="010000000100" cont="0" />\n    <request index="7" response="110000111000" foes="110000111100" cont="0" />\n    <request index="8" response="110000110011" foes="110000110011" cont="0" />\n    <request index="9" response="000000000000" foes="000000100010" cont="0" />\n    <request index="10" response="000000000000" foes="000111100110" cont="0" />\n    <request index="11" response="000000011000" foes="000110011110" cont="1" />\n  </junction>\n  <junction id="1123399659" type="priority" x="1100.56" y="1422.57" incLanes="706094741#2_0 243188641#1_0 -706094741#3_0" intLanes=":1123399659_0_0 :1123399659_1_0 :1123399659_2_0 :1123399659_3_0" shape="1107.00,1419.71 1106.66,1426.10 1104.50,1426.42 1103.80,1426.93 1103.36,1427.67 1103.17,1428.64 1103.24,1429.84 1100.07,1430.31 1099.22,1427.48 1098.51,1426.34 1097.60,1425.36 1096.50,1424.57 1095.21,1423.95 1097.50,1417.97 1099.43,1418.68 1100.89,1419.13 1102.13,1419.39 1103.39,1419.53 1104.93,1419.61">\n    <request index="0" response="0100" foes="0100" cont="0" />\n    <request index="1" response="0000" foes="1000" cont="0" />\n    <request index="2" response="0000" foes="1001" cont="0" />\n    <request index="3" response="0110" foes="0110" cont="0" />\n  </junction>\n  <junction id="1123399675" type="priority" x="535.40" y="1673.45" incLanes="-243188161_0 684741141#0_0 -684741141#1_0" intLanes=":1123399675_0_0 :1123399675_1_0 :1123399675_2_0 :1123399675_3_0 :1123399675_4_0 :1123399675_6_0" shape="541.79,1667.94 543.81,1674.02 541.91,1675.11 541.44,1675.85 541.28,1676.71 541.44,1677.69 541.91,1678.81 536.27,1681.84 529.49,1668.87 535.20,1665.97 536.73,1667.78 537.75,1668.25 538.93,1668.43 540.27,1668.33">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="1123399676" type="priority" x="995.81" y="1681.63" incLanes="-980451721#0_0 243188298_0" intLanes=":1123399676_0_0 :1123399676_1_0 :1123399676_2_0 :1123399676_4_0" shape="998.26,1674.08 1003.03,1678.34 1001.88,1680.30 1001.79,1681.24 1002.02,1682.15 1002.57,1683.03 1003.44,1683.89 999.27,1688.75 996.61,1687.14 995.43,1686.87 994.35,1686.98 993.38,1687.45 992.51,1688.28 990.02,1686.27">\n    <request index="0" response="0000" foes="1000" cont="0" />\n    <request index="1" response="0000" foes="1000" cont="0" />\n    <request index="2" response="0000" foes="0000" cont="0" />\n    <request index="3" response="0011" foes="0011" cont="1" />\n  </junction>\n  <junction id="1123399683" type="priority" x="1236.41" y="1307.92" incLanes="-243188705_0 452036923#7_0 -243188225_0" intLanes=":1123399683_0_0 :1123399683_6_0 :1123399683_2_0 :1123399683_3_0 :1123399683_4_0 :1123399683_5_0" shape="1240.20,1314.88 1233.83,1315.41 1233.21,1313.40 1232.60,1312.69 1231.77,1312.18 1230.74,1311.87 1229.51,1311.76 1228.87,1305.35 1231.27,1304.89 1232.10,1304.29 1232.69,1303.45 1233.04,1302.36 1233.15,1301.03 1239.55,1300.97">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1123399700" type="priority" x="1432.19" y="1644.11" incLanes="-96941841#1_0 -243188503#0_0 96941841#0_0" intLanes=":1123399700_0_0 :1123399700_6_0 :1123399700_2_0 :1123399700_3_0 :1123399700_4_0 :1123399700_5_0" shape="1441.01,1647.53 1436.81,1652.36 1435.00,1651.33 1434.25,1651.35 1433.60,1651.73 1433.05,1652.47 1432.61,1653.56 1426.49,1651.66 1427.01,1649.66 1427.23,1647.88 1427.17,1646.31 1426.82,1644.97 1426.18,1643.84 1425.25,1642.93 1429.08,1637.80">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1123399706" type="priority" x="804.77" y="945.49" incLanes="96941842#1_0 969757849#2_0 -96941842#2_0 -969757849#3_0" intLanes=":1123399706_0_0 :1123399706_1_0 :1123399706_2_0 :1123399706_3_0 :1123399706_4_0 :1123399706_12_0 :1123399706_6_0 :1123399706_7_0 :1123399706_8_0 :1123399706_9_0 :1123399706_10_0 :1123399706_13_0" shape="802.91,937.38 809.22,938.46 809.28,941.26 809.77,942.38 810.56,943.31 811.66,944.05 813.06,944.60 811.25,950.74 809.06,950.55 808.23,950.88 807.59,951.49 807.13,952.39 806.85,953.56 800.51,952.66 800.38,949.75 799.86,948.58 799.04,947.60 797.91,946.81 796.48,946.21 798.42,940.11 800.62,940.35 801.45,940.04 802.11,939.44 802.60,938.55">\n    <request index="0" response="000000010000" foes="000100010000" cont="0" />\n    <request index="1" response="111000110000" foes="111100110000" cont="0" />\n    <request index="2" response="110011110000" foes="110111110000" cont="0" />\n    <request index="3" response="000000000000" foes="100010000000" cont="0" />\n    <request index="4" response="000000000000" foes="100110000111" cont="0" />\n    <request index="5" response="011000000000" foes="111110000110" cont="1" />\n    <request index="6" response="010000000000" foes="010000000100" cont="0" />\n    <request index="7" response="110000111000" foes="110000111100" cont="0" />\n    <request index="8" response="110000110111" foes="110000110111" cont="0" />\n    <request index="9" response="000000000000" foes="000000100010" cont="0" />\n    <request index="10" response="000000000000" foes="000111100110" cont="0" />\n    <request index="11" response="000000111000" foes="000110111110" cont="1" />\n  </junction>\n  <junction id="1123399713" type="priority" x="1220.67" y="1654.93" incLanes="-497290263#3_0 -243188115_0 497290263#2_0" intLanes=":1123399713_0_0 :1123399713_6_0 :1123399713_2_0 :1123399713_3_0 :1123399713_4_0 :1123399713_5_0" shape="1226.20,1650.66 1227.35,1656.96 1225.04,1658.07 1224.11,1659.10 1223.34,1660.45 1222.71,1662.11 1222.25,1664.08 1215.95,1662.93 1215.92,1660.83 1215.51,1660.18 1214.85,1659.78 1213.93,1659.65 1212.75,1659.78 1211.49,1653.51">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1123399740" type="priority" x="1281.68" y="1643.74" incLanes="-497290263#4_0 -243188445_0 497290263#3_0" intLanes=":1123399740_0_0 :1123399740_6_0 :1123399740_2_0 :1123399740_3_0 :1123399740_4_0 :1123399740_5_0" shape="1287.01,1639.35 1288.33,1645.62 1286.01,1646.79 1285.07,1647.85 1284.28,1649.22 1283.64,1650.90 1283.14,1652.90 1276.86,1651.66 1276.85,1649.56 1276.46,1648.90 1275.80,1648.49 1274.88,1648.35 1273.70,1648.46 1272.54,1642.16">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1123399771" type="priority" x="1060.17" y="1111.10" incLanes="498671391_0 96242763#0_0" intLanes=":1123399771_0_0 :1123399771_1_0 :1123399771_2_0" shape="1059.62,1103.83 1062.79,1104.29 1062.91,1106.55 1063.34,1107.40 1064.04,1108.08 1064.98,1108.59 1066.17,1108.91 1065.06,1115.21 1062.81,1115.26 1061.95,1115.67 1061.27,1116.33 1060.76,1117.25 1060.43,1118.42 1057.29,1117.83">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="1123399863" type="priority" x="715.76" y="1754.29" incLanes="-243314160_0" intLanes=":1123399863_0_0" shape="708.37,1756.32 708.18,1753.12 710.31,1752.56 710.96,1751.97 711.35,1751.18 711.47,1750.19 711.32,1748.99 717.57,1747.61 717.81,1750.10 717.34,1752.17 716.16,1753.83 714.27,1755.07 711.67,1755.90">\n    <request index="0" response="0" foes="0" cont="0" />\n  </junction>\n  <junction id="1123399943" type="priority" x="514.84" y="1632.89" incLanes="-243188361_0 684741141#1_0 -684741141#2_0" intLanes=":1123399943_0_0 :1123399943_1_0 :1123399943_2_0 :1123399943_3_0 :1123399943_4_0 :1123399943_6_0" shape="521.96,1627.92 523.43,1634.15 521.46,1635.06 520.96,1635.73 520.77,1636.55 520.90,1637.52 521.34,1638.64 515.63,1641.53 514.17,1638.65 513.10,1636.51 512.26,1634.73 511.47,1632.92 510.58,1630.71 509.42,1627.69 515.39,1625.40 516.76,1627.34 517.75,1627.90 518.95,1628.18 520.35,1628.19">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="11292971036" type="priority" x="1563.31" y="1177.50" incLanes="1218747874_0" intLanes=":11292971036_0_0" shape="1563.16,1175.90 1563.75,1179.05 1563.45,1179.10 1562.87,1175.95">\n    <request index="0" response="0" foes="0" cont="0" />\n  </junction>\n  <junction id="1140280757" type="priority" x="691.90" y="1129.98" incLanes="-498671544#1_0 706104219#11_0 98567749_0 -706104219#12_0" intLanes=":1140280757_0_0 :1140280757_1_0 :1140280757_2_0 :1140280757_3_0 :1140280757_4_0 :1140280757_5_0 :1140280757_6_0 :1140280757_7_0 :1140280757_9_0" shape="690.40,1121.82 696.66,1123.18 696.63,1125.39 697.01,1126.19 697.67,1126.80 698.60,1127.20 699.79,1127.41 699.27,1133.79 695.92,1133.96 694.72,1134.42 693.84,1135.13 693.28,1136.09 693.04,1137.30 689.85,1137.09 689.38,1134.73 688.67,1133.83 687.64,1133.12 686.29,1132.60 684.62,1132.27 685.43,1125.93 687.75,1125.69 688.66,1125.15 689.40,1124.33 689.98,1123.21">\n    <request index="0" response="000001000" foes="001001000" cont="0" />\n    <request index="1" response="110111000" foes="110111000" cont="0" />\n    <request index="2" response="000000000" foes="100100000" cont="0" />\n    <request index="3" response="000000000" foes="101100011" cont="0" />\n    <request index="4" response="010000000" foes="010000010" cont="0" />\n    <request index="5" response="110001100" foes="110001110" cont="0" />\n    <request index="6" response="110001001" foes="110001001" cont="0" />\n    <request index="7" response="000000000" foes="001110010" cont="0" />\n    <request index="8" response="000001100" foes="001101110" cont="1" />\n  </junction>\n  <junction id="1140280758" type="priority" x="677.38" y="1402.31" incLanes="706104217#13_0 -706104217#14_0" intLanes=":1140280758_0_0 :1140280758_1_0 :1140280758_2_0 :1140280758_4_0" shape="675.82,1394.76 679.02,1394.78 679.45,1397.14 680.00,1397.95 680.77,1398.52 681.77,1398.85 682.99,1398.92 683.20,1405.32 680.53,1405.40 678.55,1405.41 676.91,1405.33 675.29,1405.14 673.33,1404.81 670.70,1404.33 671.86,1398.03 674.05,1397.99 674.82,1397.58 675.37,1396.90 675.70,1395.96">\n    <request index="0" response="0000" foes="1000" cont="0" />\n    <request index="1" response="0000" foes="1000" cont="0" />\n    <request index="2" response="0000" foes="0000" cont="0" />\n    <request index="3" response="0011" foes="0011" cont="1" />\n  </junction>\n  <junction id="1160612969" type="priority" x="902.44" y="617.43" incLanes="-706256381_0 100401250#1_0" intLanes=":1160612969_0_0 :1160612969_1_0 :1160612969_2_0" shape="899.97,609.92 906.33,610.55 905.91,613.60 905.40,615.83 904.26,619.46 903.74,621.68 903.31,624.74 900.13,624.42 899.73,622.14 899.02,621.28 897.99,620.61 896.61,620.13 894.91,619.83 895.60,613.47 897.85,613.27 898.68,612.80 899.31,612.08 899.73,611.12">\n    <request index="0" response="000" foes="000" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="000" cont="0" />\n  </junction>\n  <junction id="1160612974" type="priority" x="893.63" y="694.00" incLanes="71380773#1_0" intLanes=":1160612974_0_0 :1160612974_1_0" shape="892.86,688.36 896.03,688.84 896.13,690.96 896.54,691.73 897.20,692.32 898.09,692.73 899.22,692.94 898.91,696.13 888.22,695.11 888.52,691.92 890.67,691.68 891.48,691.21 892.12,690.49 892.58,689.55">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="1160617590" type="dead_end" x="770.08" y="1791.30" incLanes="" intLanes="" shape="770.14,1789.70 770.02,1792.90" />\n  <junction id="1174378788" type="priority" x="505.02" y="403.09" incLanes="-96837172#0_0 -706096644#1_0 101706795#1_0" intLanes=":1174378788_0_0 :1174378788_1_0 :1174378788_2_0 :1174378788_3_0" shape="503.49,397.35 509.59,399.29 508.95,401.31 508.51,402.83 508.22,404.09 507.99,405.37 507.79,406.93 507.53,409.04 501.18,408.28 501.00,406.02 500.54,405.19 499.83,404.55 498.88,404.10 497.68,403.85 498.05,400.67 500.53,400.52 501.52,400.08 502.34,399.41 503.00,398.50">\n    <request index="0" response="0000" foes="1100" cont="0" />\n    <request index="1" response="0000" foes="1000" cont="0" />\n    <request index="2" response="0001" foes="0001" cont="0" />\n    <request index="3" response="0011" foes="0011" cont="0" />\n  </junction>\n  <junction id="1174378892" type="dead_end" x="281.85" y="374.98" incLanes="" intLanes="" shape="282.04,373.39 281.66,376.57" />\n  <junction id="1176926303" type="priority" x="401.81" y="570.86" incLanes="706256377#4_0 84842974#13_0 84842974#13_1 84842974#13_2" intLanes=":1176926303_0_0 :1176926303_1_0 :1176926303_2_0 :1176926303_2_1 :1176926303_2_2" shape="398.57,562.35 407.99,564.17 408.01,566.39 408.42,567.20 409.09,567.81 410.02,568.22 411.22,568.43 410.69,574.80 408.04,575.03 407.04,575.50 406.26,576.21 405.70,577.17 405.36,578.36 395.90,576.69">\n    <request index="0" response="11100" foes="11100" cont="0" />\n    <request index="1" response="00000" foes="00000" cont="0" />\n    <request index="2" response="00000" foes="00001" cont="0" />\n    <request index="3" response="00000" foes="00001" cont="0" />\n    <request index="4" response="00000" foes="00001" cont="0" />\n  </junction>\n  <junction id="1196113662" type="priority" x="357.48" y="384.14" incLanes="40264696#1_0 40264696#1_1 40264696#1_2 706096637_0" intLanes=":1196113662_0_0 :1196113662_0_1 :1196113662_0_2 :1196113662_3_0 :1196113662_4_0 :1196113662_5_0" shape="353.76,377.61 363.21,379.29 363.27,381.54 363.68,382.38 364.35,383.03 365.29,383.50 366.48,383.78 366.04,386.95 363.75,387.08 362.89,387.53 362.23,388.23 361.76,389.18 361.49,390.38 351.97,389.11 351.83,386.86 351.38,386.02 350.68,385.38 349.73,384.93 348.53,384.67 348.92,381.49 351.26,381.28 352.16,380.77 352.88,379.99 353.41,378.93">\n    <request index="0" response="000000" foes="110000" cont="0" />\n    <request index="1" response="000000" foes="110000" cont="0" />\n    <request index="2" response="000000" foes="110000" cont="0" />\n    <request index="3" response="000000" foes="100000" cont="0" />\n    <request index="4" response="000111" foes="000111" cont="0" />\n    <request index="5" response="001111" foes="001111" cont="0" />\n  </junction>\n  <junction id="12339345632" type="priority" x="754.70" y="1392.49" incLanes="706104217#11_0 -243188389#0_0 -706104217#12_0" intLanes=":12339345632_0_0 :12339345632_6_0 :12339345632_2_0 :12339345632_3_0 :12339345632_4_0 :12339345632_5_0" shape="761.20,1388.11 762.30,1394.41 760.18,1395.25 759.51,1395.95 759.09,1396.84 758.93,1397.92 759.02,1399.19 752.70,1400.21 751.91,1398.10 751.24,1397.44 750.39,1397.04 749.36,1396.90 748.14,1397.02 746.92,1390.74">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="12344163777" type="dead_end" x="583.97" y="1052.50" incLanes="1334274432_0" intLanes="" shape="583.97,1052.50 583.33,1055.64 583.97,1052.50" />\n  <junction id="12344163778" type="priority" x="528.97" y="1041.26" incLanes="706093504#1_0 -1334274432_0" intLanes=":12344163778_0_0 :12344163778_1_0 :12344163778_3_0" shape="528.15,1036.42 531.31,1036.94 531.36,1037.80 531.55,1038.14 531.85,1038.41 532.25,1038.62 532.76,1038.77 531.48,1045.04 530.64,1045.04 530.32,1045.18 530.07,1045.43 529.88,1045.78 529.77,1046.22 526.61,1045.69">\n    <request index="0" response="000" foes="100" cont="0" />\n    <request index="1" response="000" foes="100" cont="0" />\n    <request index="2" response="011" foes="011" cont="1" />\n  </junction>\n  <junction id="12346099051" type="priority" x="566.22" y="1493.89" incLanes="498671702_0 95809267#0_0 -243188389#1_0" intLanes=":12346099051_0_0 :12346099051_1_0 :12346099051_2_0 :12346099051_3_0 :12346099051_4_0 :12346099051_5_0 :12346099051_6_0" shape="564.95,1486.34 568.15,1486.48 568.49,1488.65 569.01,1489.36 569.76,1489.83 570.74,1490.05 571.95,1490.03 572.66,1496.39 567.56,1501.09 564.36,1500.98 560.34,1496.70 560.77,1490.31 563.01,1489.99 563.81,1489.45 564.41,1488.67 564.78,1487.63">\n    <request index="0" response="0001000" foes="1001000" cont="0" />\n    <request index="1" response="0011000" foes="1011000" cont="0" />\n    <request index="2" response="0000000" foes="0100000" cont="0" />\n    <request index="3" response="0000000" foes="1100011" cont="0" />\n    <request index="4" response="0000000" foes="1100010" cont="0" />\n    <request index="5" response="0011100" foes="0011100" cont="0" />\n    <request index="6" response="0011011" foes="0011011" cont="0" />\n  </junction>\n  <junction id="12346099054" type="priority" x="1017.99" y="1359.35" incLanes="330615829#0_0 -330615829#2_0" intLanes=":12346099054_0_0 :12346099054_1_0 :12346099054_2_0 :12346099054_4_0" shape="1014.97,1353.27 1021.36,1353.45 1020.82,1364.90 1014.44,1364.47 1014.12,1362.09 1013.57,1361.25 1012.77,1360.65 1011.71,1360.27 1010.40,1360.14 1010.87,1356.94 1013.10,1356.56 1013.89,1356.05 1014.46,1355.33 1014.82,1354.41">\n    <request index="0" response="0000" foes="1000" cont="0" />\n    <request index="1" response="0000" foes="1000" cont="0" />\n    <request index="2" response="0000" foes="0000" cont="0" />\n    <request index="3" response="0011" foes="0011" cont="1" />\n  </junction>\n  <junction id="12351390200" type="priority" x="1195.69" y="1408.63" incLanes="-243188212#0_0 1335076429_0" intLanes=":12351390200_0_0 :12351390200_1_0" shape="1198.78,1407.77 1192.68,1409.73 1198.60,1407.30">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="12518064113" type="dead_end" x="1790.57" y="1285.57" incLanes="-243314112_0" intLanes="" shape="1790.57,1285.57 1790.52,1288.77 1790.57,1285.57" />\n  <junction id="12649445299" type="priority" x="902.62" y="876.30" incLanes="706520617#0_0 1365935945#1_0 -706520617#1_0" intLanes=":12649445299_0_0 :12649445299_6_0 :12649445299_2_0 :12649445299_3_0 :12649445299_4_0 :12649445299_5_0" shape="908.29,871.88 909.57,878.15 907.35,879.23 906.53,880.18 905.91,881.41 905.49,882.91 905.27,884.69 898.88,884.26 898.57,882.32 898.07,881.59 897.34,881.03 896.39,880.64 895.20,880.42 893.92,873.84 896.99,874.07 899.27,874.00 901.14,873.70 903.01,873.21 905.26,872.58">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="12649445300" type="priority" x="810.60" y="911.36" incLanes="96941842#0_0 498671465_0 -96941842#1_0" intLanes=":12649445300_0_0 :12649445300_6_0 :12649445300_2_0 :12649445300_3_0 :12649445300_4_0 :12649445300_5_0" shape="808.76,903.41 815.06,904.53 815.12,906.75 815.54,907.56 816.21,908.18 817.15,908.59 818.35,908.80 817.81,915.18 815.21,915.40 814.23,915.88 813.47,916.59 812.92,917.54 812.59,918.74 806.28,917.66">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="13279030938" type="priority" x="1307.02" y="986.05" incLanes="-1447303746_0 185951074#0_0 -185951074#1_0" intLanes=":13279030938_0_0 :13279030938_1_0 :13279030938_2_0 :13279030938_3_0 :13279030938_4_0 :13279030938_6_0" shape="1304.23,978.47 1310.62,978.81 1310.94,981.03 1311.46,981.80 1312.20,982.34 1313.18,982.66 1314.40,982.76 1314.48,989.16 1299.96,989.22 1299.99,982.82 1302.24,982.35 1303.05,981.74 1303.65,980.89 1304.04,979.80">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="13279030940" type="dead_end" x="1278.57" y="948.71" incLanes="1447303746_0" intLanes="" shape="1278.57,948.71 1278.59,945.51 1278.57,948.71" />\n  <junction id="13339819885" type="priority" x="903.50" y="532.97" incLanes="1454593203#2_0 -1454593202#0_0" intLanes=":13339819885_0_0 :13339819885_1_0 :13339819885_2_0" shape="907.10,527.73 910.37,530.63 908.12,530.62 906.40,531.79 905.34,533.70 905.08,535.91 905.77,537.98 907.54,539.47 902.92,539.71 901.45,537.57 900.60,536.78 899.69,536.19 898.69,535.79 897.63,535.58 898.28,529.22 901.62,529.38 902.79,529.23 903.94,528.92 905.31,528.42">\n    <request index="0" response="000" foes="000" cont="0" />\n    <request index="1" response="000" foes="100" cont="0" />\n    <request index="2" response="010" foes="010" cont="0" />\n  </junction>\n  <junction id="13339819888" type="priority" x="912.32" y="529.55" incLanes="706256380#6_0 1454593203#1_0" intLanes=":13339819888_0_0 :13339819888_1_0 :13339819888_2_0" shape="909.69,523.40 916.06,524.01 915.89,526.07 915.85,527.62 915.94,528.89 916.20,530.15 916.62,531.65 917.23,533.63 914.78,536.57 905.32,533.05 905.48,528.34 907.56,526.87 908.37,526.32 909.00,525.67 909.44,524.76">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="13339819893" type="priority" x="909.94" y="541.55" incLanes="706256381_0 1454593203#3_0" intLanes=":13339819893_0_0 :13339819893_1_0 :13339819893_2_0" shape="914.15,535.62 916.05,539.64 914.07,542.10 913.48,543.07 913.08,544.11 912.80,545.42 912.58,547.19 906.22,546.54 906.04,544.39 905.67,543.34 905.10,542.31 904.34,541.30 903.38,540.31 904.84,535.96 905.77,538.08 907.54,539.30 909.71,539.66 911.83,539.15 913.46,537.80">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="13339844003" type="priority" x="915.79" y="534.39" incLanes="95533432#5_0 1454593203#4_0" intLanes=":13339844003_0_0 :13339844003_1_0 :13339844003_2_0" shape="921.63,531.81 920.93,538.17 918.76,538.33 917.73,538.72 916.73,539.30 915.78,540.09 914.86,541.08 910.38,539.80 909.94,529.80 913.50,528.43 916.22,530.35 917.26,530.91 918.37,531.30 919.76,531.58">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="13351099349" type="dead_end" x="1514.69" y="551.62" incLanes="-155690812#0_0" intLanes="" shape="1514.69,551.62 1514.41,548.43 1514.69,551.62" />\n  <junction id="13362936997" type="priority" x="1434.73" y="353.45" incLanes="-129842391_0 1457057267_0 497371052#0_0" intLanes=":13362936997_0_0 :13362936997_2_0" shape="1435.86,345.11 1441.40,348.32 1440.11,350.23 1437.92,352.82 1437.21,354.05 1436.87,355.61 1436.99,357.80 1433.82,358.26 1431.69,345.49 1434.84,344.94 1435.07,345.61 1435.23,345.72 1435.41,345.67 1435.62,345.47">\n    <request index="0" response="00" foes="10" cont="0" />\n    <request index="1" response="01" foes="01" cont="1" />\n  </junction>\n  <junction id="13424210990" type="dead_end" x="709.31" y="1046.68" incLanes="498671544#0_0" intLanes="" shape="709.31,1046.68 712.46,1047.25 709.31,1046.68" />\n  <junction id="13449811511" type="priority" x="1670.51" y="765.50" incLanes="243314436#3_0 969757853#1_0" intLanes=":13449811511_0_0 :13449811511_1_0 :13449811511_2_0" shape="1677.59,764.48 1677.32,767.67 1674.97,768.04 1674.01,768.66 1673.21,769.58 1672.55,770.79 1672.05,772.30 1665.85,770.68 1665.97,768.50 1665.61,767.70 1664.98,767.10 1664.06,766.70 1662.87,766.50 1663.13,763.31">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="13545019681" type="priority" x="1066.57" y="1031.65" incLanes="-530314558#0_0 96242763#2_0" intLanes=":13545019681_0_0 :13545019681_1_0 :13545019681_2_0" shape="1074.67,1036.82 1069.63,1040.76 1068.19,1039.45 1067.61,1039.38 1067.14,1039.71 1066.77,1040.44 1066.49,1041.55 1063.34,1041.01 1063.96,1037.39 1064.34,1034.67 1064.46,1032.40 1064.27,1030.15 1063.74,1027.47 1062.83,1023.93 1065.92,1023.09 1066.96,1026.39 1068.02,1028.72 1069.21,1030.50 1070.65,1032.16 1072.43,1034.12">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="1369020939" type="priority" x="1393.18" y="1585.88" incLanes="122540020#7_0 243188710#1_0 -243188710#2_0" intLanes=":1369020939_0_0 :1369020939_1_0 :1369020939_2_0 :1369020939_3_0 :1369020939_4_0 :1369020939_6_0" shape="1392.74,1577.34 1398.79,1579.42 1398.14,1582.61 1398.24,1583.95 1398.62,1585.14 1399.28,1586.17 1400.21,1587.03 1396.45,1592.21 1394.04,1590.44 1392.26,1589.11 1390.77,1588.05 1389.23,1587.06 1387.31,1585.96 1384.73,1584.58 1388.00,1579.08 1390.05,1579.79 1390.90,1579.66 1391.63,1579.21 1392.24,1578.44">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="1369020942" type="priority" x="1407.87" y="1543.23" incLanes="-243188056_0 -122540020#7_0 122540020#6_0" intLanes=":1369020942_0_0 :1369020942_1_0 :1369020942_2_0 :1369020942_3_0 :1369020942_4_0 :1369020942_6_0" shape="1414.25,1539.68 1414.60,1546.07 1412.02,1546.58 1411.00,1547.11 1410.16,1547.81 1409.49,1548.70 1409.01,1549.76 1402.96,1547.67 1403.66,1545.53 1404.09,1543.91 1404.28,1542.52 1404.26,1541.12 1404.07,1539.44 1403.74,1537.21 1410.08,1536.27 1410.78,1538.25 1411.39,1538.93 1412.17,1539.39 1413.12,1539.64">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="1369020943" type="priority" x="1393.78" y="1346.12" incLanes="-243188283_0 -122540020#3_0 122540020#1_0" intLanes=":1369020943_0_0 :1369020943_1_0 :1369020943_2_0 :1369020943_3_0 :1369020943_4_0 :1369020943_6_0" shape="1400.81,1342.38 1401.29,1348.76 1399.07,1349.38 1398.32,1349.99 1397.82,1350.81 1397.56,1351.83 1397.54,1353.06 1391.16,1353.56 1390.24,1338.99 1396.63,1338.69 1397.18,1340.87 1397.77,1341.61 1398.57,1342.10 1399.58,1342.36">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="1369020950" type="priority" x="1385.88" y="1192.43" incLanes="72016745#0_0 72016745#0_1 -122540020#0_0 -72016745#2_0 -72016745#2_1" intLanes=":1369020950_0_0 :1369020950_0_1 :1369020950_8_0 :1369020950_3_0 :1369020950_4_0 :1369020950_5_0 :1369020950_6_0 :1369020950_6_1" shape="1393.89,1186.66 1392.86,1199.41 1390.63,1199.68 1389.83,1200.17 1389.26,1200.90 1388.89,1201.88 1388.74,1203.09 1382.34,1202.89 1381.98,1200.44 1381.45,1199.54 1380.70,1198.87 1379.71,1198.42 1378.50,1198.19 1379.66,1185.44">\n    <request index="0" response="00000000" foes="00010000" cont="0" />\n    <request index="1" response="00000000" foes="00010000" cont="0" />\n    <request index="2" response="11100000" foes="11110000" cont="1" />\n    <request index="3" response="11000000" foes="11000000" cont="0" />\n    <request index="4" response="11000111" foes="11000111" cont="0" />\n    <request index="5" response="00000000" foes="00000100" cont="0" />\n    <request index="6" response="00000000" foes="00011100" cont="0" />\n    <request index="7" response="00000000" foes="00011100" cont="0" />\n  </junction>\n  <junction id="14063214200" type="dead_end" x="717.14" y="283.47" incLanes="96837170#3_0" intLanes="" shape="717.14,283.47 718.63,286.30 717.14,283.47" />\n  <junction id="14069947700" type="priority" x="572.02" y="661.42" incLanes="-1546894905_0 40264697#4_0 40264697#4_1" intLanes=":14069947700_0_0 :14069947700_1_0 :14069947700_2_0 :14069947700_2_1" shape="578.80,658.43 578.60,664.82 576.27,665.31 575.35,665.99 574.58,666.95 573.98,668.20 573.54,669.73 567.28,668.41 567.30,666.23 566.90,665.45 566.24,664.88 565.31,664.52 564.11,664.38 564.30,657.98">\n    <request index="0" response="1100" foes="1100" cont="0" />\n    <request index="1" response="0000" foes="0000" cont="0" />\n    <request index="2" response="0000" foes="0001" cont="0" />\n    <request index="3" response="0000" foes="0001" cont="0" />\n  </junction>\n  <junction id="14069967501" type="dead_end" x="564.06" y="698.75" incLanes="1546894905_0" intLanes="" shape="564.06,698.75 560.93,698.08 564.06,698.75" />\n  <junction id="14069981565" type="priority" x="459.32" y="738.87" incLanes="706097788#0_0 -1546896992_0" intLanes=":14069981565_0_0 :14069981565_1_0 :14069981565_2_0" shape="458.47,733.80 461.64,734.28 461.68,735.11 461.84,735.41 462.10,735.63 462.46,735.77 462.91,735.83 462.61,742.23 461.38,742.33 460.93,742.52 460.58,742.80 460.35,743.16 460.22,743.61 457.05,743.13">\n    <request index="0" response="000" foes="100" cont="0" />\n    <request index="1" response="000" foes="100" cont="0" />\n    <request index="2" response="011" foes="011" cont="0" />\n  </junction>\n  <junction id="14069981568" type="dead_end" x="514.36" y="707.90" incLanes="1546896992_0" intLanes="" shape="514.36,707.90 517.52,708.40 514.36,707.90" />\n  <junction id="14070071629" type="priority" x="593.64" y="285.94" incLanes="-1546907364_0 155690791#0_0" intLanes=":14070071629_0_0 :14070071629_1_0" shape="597.63,283.88 595.64,289.96 589.59,287.87 591.39,284.78 592.63,283.90 594.07,283.45 595.74,283.44">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="14070071630" type="dead_end" x="657.11" y="306.70" incLanes="1546907364_0" intLanes="" shape="657.11,306.70 656.12,309.74 657.11,306.70" />\n  <junction id="14124372093" type="dead_end" x="1689.38" y="642.59" incLanes="-155690766_0" intLanes="" shape="1689.38,642.59 1692.45,643.51 1689.38,642.59" />\n  <junction id="14127602686" type="dead_end" x="439.09" y="1495.34" incLanes="-1552748564#0_0" intLanes="" shape="439.09,1495.34 439.00,1498.54 439.09,1495.34" />\n  <junction id="14127615694" type="dead_end" x="1143.07" y="1510.73" incLanes="1552748146_0" intLanes="" shape="1143.07,1510.73 1143.07,1507.53 1143.07,1510.73" />\n  <junction id="14159239425" type="priority" x="1134.54" y="716.08" incLanes="-1556742612_0 969757851#1_0" intLanes=":14159239425_0_0 :14159239425_1_0 :14159239425_2_0" shape="1142.56,715.27 1142.25,718.46 1140.13,718.67 1139.48,719.13 1139.10,719.84 1138.99,720.78 1139.14,721.96 1132.90,723.37 1131.71,720.00 1130.90,718.79 1129.94,717.88 1128.83,717.28 1127.57,717.00 1127.89,713.81">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="14159239426" type="dead_end" x="1146.91" y="771.09" incLanes="1556742612_0" intLanes="" shape="1146.91,771.09 1143.79,771.79 1146.91,771.09" />\n  <junction id="14159239427" type="priority" x="1077.61" y="728.60" incLanes="969765780#1_0 -1556742613_0 -969765780#2_0" intLanes=":14159239427_0_0 :14159239427_6_0 :14159239427_2_0 :14159239427_3_0 :14159239427_4_0 :14159239427_5_0" shape="1081.35,722.18 1084.95,727.47 1083.35,729.26 1083.02,730.36 1082.98,731.61 1083.25,732.99 1083.83,734.52 1078.01,737.18 1076.70,735.35 1075.93,734.97 1075.09,734.95 1074.18,735.30 1073.19,736.00 1069.35,730.86 1071.76,728.83 1073.60,727.39 1075.16,726.27 1076.76,725.21 1078.72,723.94">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="14159239429" type="dead_end" x="1096.62" y="765.79" incLanes="1556742613_0" intLanes="" shape="1096.62,765.79 1094.09,767.75 1096.62,765.79" />\n  <junction id="14165547215" type="priority" x="1385.27" y="1274.98" incLanes="-122540020#2_0 -1557537524_0 122540020#0_0" intLanes=":14165547215_0_0 :14165547215_6_0 :14165547215_2_0 :14165547215_3_0 :14165547215_4_0 :14165547215_5_0" shape="1389.56,1280.98 1383.26,1282.08 1382.30,1279.88 1381.44,1279.11 1380.34,1278.55 1378.98,1278.20 1377.37,1278.06 1377.46,1271.67 1379.61,1271.26 1380.29,1270.72 1380.71,1269.97 1380.87,1268.99 1380.78,1267.78 1387.08,1266.69">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="14165547216" type="dead_end" x="1349.64" y="1274.46" incLanes="1557537524_0" intLanes="" shape="1349.64,1274.46 1349.69,1271.26 1349.64,1274.46" />\n  <junction id="14179759489" type="priority" x="397.16" y="1429.56" incLanes="-1558734968_0 -498671328#1_0 -243188576_0" intLanes=":14179759489_0_0 :14179759489_1_0 :14179759489_2_0 :14179759489_3_0 :14179759489_4_0 :14179759489_6_0" shape="407.34,1431.42 404.30,1437.05 402.38,1436.44 401.73,1436.63 401.30,1437.14 401.08,1437.98 401.07,1439.14 394.69,1439.61 394.27,1436.29 393.71,1433.80 393.19,1431.75 392.90,1429.71 393.01,1427.29 393.72,1424.08 400.49,1423.64 400.81,1426.08 401.70,1427.36 403.09,1428.67 404.97,1430.03">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="14182422401" type="priority" x="730.07" y="414.83" incLanes="-1559020882_0 -155690741#2_0 155690741#0_0" intLanes=":14182422401_0_0 :14182422401_1_0 :14182422401_2_0 :14182422401_3_0 :14182422401_4_0 :14182422401_6_0" shape="727.80,407.26 734.15,408.06 734.31,410.35 734.76,411.21 735.46,411.87 736.41,412.34 737.61,412.61 736.76,418.95 722.48,417.04 723.33,410.70 725.59,410.55 726.43,410.10 727.08,409.41 727.53,408.46">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="14182422402" type="dead_end" x="734.29" y="381.22" incLanes="1559020882_0" intLanes="" shape="734.29,381.22 737.47,381.62 734.29,381.22" />\n  <junction id="14188374628" type="dead_end" x="1193.91" y="1554.13" incLanes="1559802495#1_0" intLanes="" shape="1193.91,1554.13 1193.60,1550.94 1193.91,1554.13" />\n  <junction id="14188374629" type="priority" x="1214.10" y="1552.17" incLanes="1559802495#0_0 -1559802495#1_0 -1559802496_0" intLanes=":14188374629_0_0 :14188374629_1_0 :14188374629_2_0 :14188374629_6_0 :14188374629_4_0 :14188374629_5_0" shape="1221.10,1548.27 1221.72,1554.64 1207.38,1556.04 1206.76,1549.67 1208.95,1548.98 1209.69,1548.31 1210.20,1547.42 1210.48,1546.32 1210.52,1545.00 1216.91,1544.67 1217.47,1546.84 1218.06,1547.56 1218.87,1548.04 1219.88,1548.28">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="14188764645" type="priority" x="808.72" y="578.17" incLanes="129827561#0_0 -1559841160#0_0 -129827561#1_0" intLanes=":14188764645_0_0 :14188764645_6_0 :14188764645_2_0 :14188764645_3_0 :14188764645_4_0 :14188764645_5_0" shape="806.23,570.67 812.60,571.28 812.83,573.53 813.31,574.36 814.03,574.98 814.99,575.40 816.20,575.61 815.64,581.99 813.35,582.24 812.51,582.72 811.88,583.44 811.44,584.41 811.21,585.62 804.84,585.00">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="14188764646" type="priority" x="853.93" y="582.14" incLanes="-1559841160#1_0 -1559841161_0 1559841160#0_0" intLanes=":14188764646_0_0 :14188764646_1_0 :14188764646_2_0 :14188764646_3_0 :14188764646_4_0 :14188764646_6_0" shape="851.64,574.59 857.99,575.38 858.16,577.64 858.62,578.48 859.32,579.13 860.27,579.58 861.47,579.84 860.70,586.19 846.59,584.71 847.15,578.33 849.42,578.07 850.27,577.57 850.92,576.83 851.38,575.83">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="14188764647" type="priority" x="860.59" y="528.55" incLanes="1454593202#0_0 1559841160#1_0 -1454593202#1_0" intLanes=":14188764647_0_0 :14188764647_6_0 :14188764647_2_0 :14188764647_3_0 :14188764647_4_0 :14188764647_5_0" shape="868.01,526.10 867.36,532.46 865.09,532.69 864.25,533.18 863.59,533.92 863.14,534.91 862.87,536.16 856.52,535.37 856.35,533.12 855.89,532.29 855.19,531.66 854.23,531.22 853.03,530.99 853.69,524.62">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="14188764648" type="dead_end" x="878.26" y="585.11" incLanes="1559841161_0" intLanes="" shape="878.26,585.11 877.87,588.29 878.26,585.11" />\n  <junction id="1431715359" type="dead_end" x="567.39" y="218.76" incLanes="129827558_0" intLanes="" shape="567.39,218.76 570.42,219.80 567.39,218.76" />\n  <junction id="1431715363" type="priority" x="811.45" y="686.42" incLanes="71380773#0_0 706256385_0" intLanes=":1431715363_0_0 :1431715363_2_0" shape="814.45,685.09 814.16,688.28 806.15,687.54 806.45,684.35 807.46,684.26 807.71,684.05 807.80,683.74 807.70,683.31 807.44,682.78 810.20,681.15 811.66,683.26 812.37,684.01 813.07,684.57 813.77,684.93">\n    <request index="0" response="00" foes="10" cont="0" />\n    <request index="1" response="01" foes="01" cont="1" />\n  </junction>\n  <junction id="1431715366" type="priority" x="769.07" y="680.94" incLanes="1365892989_0 243079805_0" intLanes=":1431715366_0_0 :1431715366_1_0" shape="767.58,675.56 770.78,675.63 771.12,677.87 771.59,678.71 772.26,679.36 773.12,679.81 774.18,680.08 773.72,683.25 763.54,681.71 764.03,678.55 765.97,678.45 766.66,678.07 767.15,677.46 767.46,676.62">\n    <request index="0" response="10" foes="10" cont="0" />\n    <request index="1" response="00" foes="01" cont="0" />\n  </junction>\n  <junction id="1431715369" type="priority" x="416.43" y="495.21" incLanes="84842974#14_0 84842974#14_1 84842974#14_2" intLanes=":1431715369_0_0 :1431715369_0_1 :1431715369_0_2 :1431715369_3_0" shape="412.73,489.15 422.15,491.00 420.01,501.97 410.59,500.15 410.57,497.91 410.16,497.09 409.50,496.45 408.57,496.00 407.37,495.75 407.75,492.57 410.17,492.41 411.10,491.97 411.83,491.27 412.38,490.33">\n    <request index="0" response="0000" foes="0000" cont="0" />\n    <request index="1" response="0000" foes="0000" cont="0" />\n    <request index="2" response="0000" foes="0000" cont="0" />\n    <request index="3" response="0000" foes="0000" cont="0" />\n  </junction>\n  <junction id="1431715372" type="priority" x="788.61" y="333.85" incLanes="148941242#1_0 95809262#1_0 -148941242#2_0" intLanes=":1431715372_0_0 :1431715372_6_0 :1431715372_2_0 :1431715372_3_0 :1431715372_4_0 :1431715372_5_0" shape="787.61,325.46 793.78,327.17 793.63,329.35 793.98,330.15 794.60,330.76 795.51,331.17 796.70,331.39 796.10,337.77 793.29,337.95 792.24,338.40 791.41,339.10 790.81,340.05 790.44,341.23 784.16,340.02">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1431715380" type="dead_end" x="530.77" y="114.96" incLanes="-129827562_0" intLanes="" shape="530.77,114.96 533.92,115.55 530.77,114.96" />\n  <junction id="1431715392" type="priority" x="770.27" y="603.98" incLanes="969765781_0 -969765782_0" intLanes=":1431715392_0_0 :1431715392_1_0 :1431715392_6_0 :1431715392_3_0 :1431715392_4_0 :1431715392_7_0" shape="776.04,601.25 775.51,607.63 773.46,607.87 772.73,608.32 772.21,609.00 771.89,609.90 771.77,611.02 768.57,610.97 768.10,608.74 767.48,607.92 766.60,607.29 765.47,606.86 764.08,606.62 764.68,600.25 766.68,600.03 767.35,599.58 767.80,598.91 768.02,598.02 768.02,596.91 771.21,596.62 771.92,599.03 772.61,599.91 773.53,600.57 774.67,601.02">\n    <request index="0" response="000000" foes="100000" cont="0" />\n    <request index="1" response="000000" foes="100000" cont="0" />\n    <request index="2" response="011000" foes="011000" cont="1" />\n    <request index="3" response="000000" foes="000100" cont="0" />\n    <request index="4" response="000000" foes="000100" cont="0" />\n    <request index="5" response="000011" foes="000011" cont="1" />\n  </junction>\n  <junction id="1431715395" type="priority" x="514.90" y="371.98" incLanes="-96837172#1_0 -129827552#0_0 96837172#0_0" intLanes=":1431715395_0_0 :1431715395_6_0 :1431715395_2_0 :1431715395_3_0 :1431715395_4_0 :1431715395_5_0" shape="514.22,364.05 520.28,366.12 519.99,368.36 520.27,369.26 520.83,370.02 521.68,370.63 522.81,371.10 520.90,377.21 518.59,376.95 517.66,377.24 516.88,377.81 516.25,378.67 515.77,379.80 509.68,377.86">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1431715406" type="priority" x="805.93" y="606.96" incLanes="129827561#1_0 101939921_0 101939921_1 -969765781_0" intLanes=":1431715406_0_0 :1431715406_1_0 :1431715406_2_0 :1431715406_3_0 :1431715406_7_0 :1431715406_5_0 :1431715406_8_0" shape="803.43,599.57 809.80,600.18 810.03,602.42 810.50,603.24 811.21,603.87 812.16,604.31 813.35,604.55 812.66,610.91 809.66,611.03 808.59,611.46 807.80,612.15 807.29,613.09 807.06,614.28 803.87,614.07 803.40,611.77 802.68,610.92 801.63,610.26 800.27,609.81 798.58,609.56 799.11,603.18 801.34,602.92 802.16,602.44 802.78,601.72 803.21,600.76">\n    <request index="0" response="0001000" foes="0001000" cont="0" />\n    <request index="1" response="1111000" foes="1111000" cont="0" />\n    <request index="2" response="0000000" foes="1000000" cont="0" />\n    <request index="3" response="0000000" foes="1000011" cont="0" />\n    <request index="4" response="0100000" foes="0100010" cont="1" />\n    <request index="5" response="0000000" foes="0010010" cont="0" />\n    <request index="6" response="0001100" foes="0001110" cont="1" />\n  </junction>\n  <junction id="1431715409" type="priority" x="732.80" y="514.38" incLanes="1374050353_0 -706256379#1_0" intLanes=":1431715409_0_0 :1431715409_4_0 :1431715409_2_0 :1431715409_3_0" shape="738.58,511.49 738.08,517.87 736.41,518.14 735.88,518.55 735.55,519.12 735.41,519.88 735.48,520.81 732.32,521.33 731.63,519.17 731.08,518.38 730.39,517.80 729.56,517.41 728.60,517.22 729.17,510.84">\n    <request index="0" response="0000" foes="0000" cont="0" />\n    <request index="1" response="1100" foes="1100" cont="1" />\n    <request index="2" response="0000" foes="0010" cont="0" />\n    <request index="3" response="0000" foes="0010" cont="0" />\n  </junction>\n  <junction id="1431715415" type="priority" x="813.97" y="523.75" incLanes="1454593202#1_0 -129827561#0_0 -1454593202#2_0" intLanes=":1431715415_0_0 :1431715415_6_0 :1431715415_2_0 :1431715415_3_0 :1431715415_4_0 :1431715415_5_0" shape="821.48,521.31 820.82,527.67 818.57,527.89 817.74,528.36 817.12,529.08 816.69,530.04 816.46,531.24 810.09,530.63 809.87,528.26 809.40,527.37 808.69,526.68 807.73,526.19 806.53,525.90 807.46,519.57">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1431715421" type="priority" x="501.61" y="292.37" incLanes="129827562_0 706096643#0_0" intLanes=":1431715421_0_0 :1431715421_1_0 :1431715421_2_0" shape="499.14,286.40 505.49,287.20 505.65,289.46 506.10,290.30 506.81,290.94 507.76,291.39 508.96,291.65 508.57,294.83 494.28,293.09 494.66,289.91 496.92,289.74 497.77,289.28 498.42,288.57 498.87,287.61">\n    <request index="0" response="000" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="001" foes="001" cont="0" />\n  </junction>\n  <junction id="1431867100" type="priority" x="1630.10" y="281.89" incLanes="129842397#4_0 1457057266#0_0" intLanes=":1431867100_0_0 :1431867100_1_0" shape="1634.73,280.96 1631.41,286.43 1625.68,283.56 1626.80,281.77 1628.07,280.51 1629.50,279.81 1631.09,279.64 1632.83,280.03">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="1431867103" type="priority" x="1717.32" y="450.89" incLanes="706087699#1_0 129842397#0_0 -129842397#1_0" intLanes=":1431867103_0_0 :1431867103_1_0 :1431867103_2_0 :1431867103_3_0 :1431867103_4_0 :1431867103_6_0" shape="1716.39,442.63 1722.54,444.42 1722.15,447.30 1722.38,448.48 1722.88,449.50 1723.65,450.36 1724.70,451.04 1721.81,456.75 1709.01,450.69 1711.62,444.84 1713.77,445.32 1714.64,445.11 1715.36,444.58 1715.95,443.76">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="1431867108" type="priority" x="1435.64" y="359.62" incLanes="-129842388#1_0 705855017#1_0 932339004_0" intLanes=":1431867108_0_0 :1431867108_1_0 :1431867108_2_0 :1431867108_3_0 :1431867108_4_0 :1431867108_5_0 :1431867108_7_0" shape="1443.31,358.19 1441.80,364.41 1439.71,364.32 1439.05,364.69 1438.64,365.32 1438.48,366.23 1438.58,367.41 1435.42,367.94 1434.47,365.18 1433.71,364.27 1432.76,363.68 1431.62,363.41 1430.30,363.45 1429.61,357.09 1433.00,352.67 1436.16,352.20">\n    <request index="0" response="0000000" foes="1001000" cont="0" />\n    <request index="1" response="0000000" foes="1011000" cont="0" />\n    <request index="2" response="0100000" foes="0100000" cont="0" />\n    <request index="3" response="1100011" foes="1100011" cont="0" />\n    <request index="4" response="1100010" foes="1100010" cont="0" />\n    <request index="5" response="0000000" foes="0011100" cont="0" />\n    <request index="6" response="0000011" foes="0011011" cont="1" />\n  </junction>\n  <junction id="1431867112" type="dead_end" x="1547.97" y="157.83" incLanes="129842391_0" intLanes="" shape="1547.97,157.83 1550.74,159.43 1547.97,157.83" />\n  <junction id="1431867140" type="dead_end" x="1823.78" y="612.25" incLanes="155691022#1_0" intLanes="" shape="1823.78,612.25 1823.55,615.44 1823.78,612.25" />\n  <junction id="1431867141" type="dead_end" x="1842.55" y="514.31" incLanes="-129842397#0_0" intLanes="" shape="1842.55,514.31 1841.10,517.16 1842.55,514.31" />\n  <junction id="1431867150" type="priority" x="1668.15" y="304.99" incLanes="-142395239#0_0 129842397#2_0 -129842397#4_0" intLanes=":1431867150_0_0 :1431867150_6_0 :1431867150_2_0 :1431867150_3_0 :1431867150_4_0 :1431867150_5_0" shape="1674.49,302.70 1673.55,309.03 1671.50,309.13 1670.64,309.50 1669.88,310.10 1669.24,310.91 1668.71,311.94 1662.81,309.47 1663.22,307.51 1663.07,306.64 1662.68,305.86 1662.06,305.14 1661.19,304.51 1664.51,299.04 1666.47,300.21 1667.95,301.04 1669.24,301.61 1670.58,302.02 1672.24,302.35">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1431867160" type="priority" x="1418.75" y="361.96" incLanes="-129842388#0_0 932339002#4_0 932339002#4_1" intLanes=":1431867160_0_0 :1431867160_1_0 :1431867160_2_0 :1431867160_2_1" shape="1425.37,357.69 1426.38,364.02 1424.25,364.77 1423.57,365.44 1423.15,366.29 1422.98,367.33 1423.07,368.56 1416.75,369.59 1414.43,355.38 1420.75,354.35 1421.55,356.46 1422.22,357.14 1423.09,357.58 1424.14,357.76">\n    <request index="0" response="1100" foes="1100" cont="0" />\n    <request index="1" response="0000" foes="0000" cont="0" />\n    <request index="2" response="0000" foes="0001" cont="0" />\n    <request index="3" response="0000" foes="0001" cont="0" />\n  </junction>\n  <junction id="1431867166" type="priority" x="1554.36" y="387.16" incLanes="-1457057266#1_0 -142455592#1_0 129842388#2_0" intLanes=":1431867166_0_0 :1431867166_1_0 :1431867166_2_0 :1431867166_3_0 :1431867166_4_0 :1431867166_5_0" shape="1555.69,380.53 1560.62,384.61 1558.90,386.74 1557.69,388.37 1556.81,389.79 1556.05,391.29 1555.22,393.16 1554.13,395.67 1548.26,393.11 1548.72,390.98 1548.49,390.14 1547.95,389.47 1547.11,388.95 1545.96,388.58 1547.34,382.33 1550.85,382.70 1552.33,382.52 1553.63,382.10 1554.75,381.44">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="0" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1431867172" type="dead_end" x="1600.88" y="173.25" incLanes="-155691088#0_0" intLanes="" shape="1600.88,173.25 1603.63,174.89 1600.88,173.25" />\n  <junction id="1558314095" type="priority" x="1697.89" y="309.43" incLanes="-706087699#0_0 -498671600_0 142395239#0_0" intLanes=":1558314095_0_0 :1558314095_1_0 :1558314095_2_0 :1558314095_6_0 :1558314095_4_0 :1558314095_5_0" shape="1696.48,301.96 1702.72,303.59 1702.25,306.66 1701.75,308.88 1701.20,310.69 1700.57,312.48 1699.82,314.66 1698.94,317.63 1692.77,315.91 1692.92,313.71 1692.58,312.88 1691.96,312.23 1691.06,311.76 1689.88,311.47 1690.83,305.14 1693.74,305.13 1694.81,304.73 1695.62,304.07 1696.18,303.14">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="1558997825" type="priority" x="1053.99" y="703.38" incLanes="-706094742_0 970010765#2_0" intLanes=":1558997825_0_0 :1558997825_1_0 :1558997825_2_0" shape="1056.04,695.14 1061.21,698.92 1060.34,700.74 1060.44,701.41 1060.89,701.92 1061.69,702.27 1062.84,702.45 1062.60,705.64 1047.11,704.46 1047.36,701.27 1050.10,700.72 1051.52,699.89 1052.99,698.68 1054.50,697.10">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="1559016903" type="priority" x="879.58" y="342.38" incLanes="-142455604#1_0 95809262#0_0 142455600#3_0 -95809262#1_0" intLanes=":1559016903_0_0 :1559016903_1_0 :1559016903_12_0 :1559016903_3_0 :1559016903_4_0 :1559016903_5_0 :1559016903_6_0 :1559016903_7_0 :1559016903_13_0 :1559016903_9_0 :1559016903_10_0 :1559016903_11_0" shape="877.10,334.92 883.47,335.55 883.70,337.84 884.19,338.69 884.93,339.34 885.91,339.79 887.15,340.05 886.40,346.41 884.14,346.59 883.31,347.05 882.68,347.76 882.24,348.71 882.01,349.92 875.64,349.25 875.43,346.99 874.96,346.17 874.24,345.54 873.28,345.12 872.08,344.89 872.67,338.52 874.96,338.29 875.80,337.81 876.44,337.09 876.87,336.12">\n    <request index="0" response="000000000000" foes="000100010000" cont="0" />\n    <request index="1" response="000000000000" foes="111100110000" cont="0" />\n    <request index="2" response="000111000000" foes="110111110000" cont="1" />\n    <request index="3" response="000010000000" foes="100010000000" cont="0" />\n    <request index="4" response="000110000111" foes="100110000111" cont="0" />\n    <request index="5" response="011110000110" foes="111110000110" cont="0" />\n    <request index="6" response="000000000000" foes="010000000100" cont="0" />\n    <request index="7" response="000000000000" foes="110000111100" cont="0" />\n    <request index="8" response="000000000011" foes="110000110111" cont="1" />\n    <request index="9" response="000000000010" foes="000000100010" cont="0" />\n    <request index="10" response="000111000110" foes="000111100110" cont="0" />\n    <request index="11" response="000110111110" foes="000110111110" cont="0" />\n  </junction>\n  <junction id="1559016904" type="priority" x="781.62" y="376.60" incLanes="148941242#2_0 -142455595_0 -148941242#3_0" intLanes=":1559016904_0_0 :1559016904_6_0 :1559016904_2_0 :1559016904_3_0 :1559016904_4_0 :1559016904_5_0" shape="779.34,369.05 785.70,369.78 785.83,372.02 786.28,372.85 786.98,373.48 787.94,373.91 789.15,374.15 788.50,380.51 786.19,380.73 785.34,381.20 784.68,381.91 784.23,382.87 783.98,384.08 777.62,383.34">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1559016905" type="priority" x="874.99" y="386.07" incLanes="-142455600#3_0 142455600#2_0 142455595_0" intLanes=":1559016905_0_0 :1559016905_1_0 :1559016905_2_0 :1559016905_6_0 :1559016905_4_0 :1559016905_5_0" shape="872.56,378.59 878.92,379.26 877.35,393.63 870.99,392.90 870.80,390.65 870.34,389.82 869.62,389.19 868.67,388.76 867.46,388.52 868.11,382.16 870.40,381.94 871.24,381.47 871.88,380.75 872.32,379.79">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="1559016909" type="priority" x="776.48" y="421.04" incLanes="148941242#3_0 -142455590_0 -148941242#4_0 155690741#2_0" intLanes=":1559016909_0_0 :1559016909_1_0 :1559016909_12_0 :1559016909_3_0 :1559016909_4_0 :1559016909_5_0 :1559016909_6_0 :1559016909_7_0 :1559016909_13_0 :1559016909_9_0 :1559016909_10_0 :1559016909_11_0" shape="774.14,413.46 780.49,414.20 780.68,416.46 781.14,417.29 781.85,417.93 782.81,418.36 784.01,418.60 783.35,424.97 781.05,425.18 780.20,425.65 779.54,426.36 779.09,427.32 778.84,428.52 772.48,427.79 772.28,425.51 771.81,424.66 771.08,424.00 770.10,423.53 768.86,423.25 769.71,416.91 771.96,416.76 772.80,416.31 773.44,415.61 773.88,414.66">\n    <request index="0" response="000000000000" foes="000100010000" cont="0" />\n    <request index="1" response="000000000000" foes="111100110000" cont="0" />\n    <request index="2" response="000011000000" foes="110111110000" cont="1" />\n    <request index="3" response="000010000000" foes="100010000000" cont="0" />\n    <request index="4" response="000110000111" foes="100110000111" cont="0" />\n    <request index="5" response="111110000110" foes="111110000110" cont="0" />\n    <request index="6" response="000000000000" foes="010000000100" cont="0" />\n    <request index="7" response="000000000000" foes="110000111100" cont="0" />\n    <request index="8" response="000000000111" foes="110000110111" cont="1" />\n    <request index="9" response="000000000010" foes="000000100010" cont="0" />\n    <request index="10" response="000111000110" foes="000111100110" cont="0" />\n    <request index="11" response="000110011110" foes="000110111110" cont="0" />\n  </junction>\n  <junction id="1559016911" type="dead_end" x="1532.91" y="436.17" incLanes="142455592#0_0" intLanes="" shape="1532.91,436.17 1529.98,434.89 1532.91,436.17" />\n  <junction id="1559016916" type="priority" x="773.15" y="464.80" incLanes="148941242#4_0 -148941242#5_0" intLanes=":1559016916_0_0 :1559016916_4_0 :1559016916_2_0 :1559016916_3_0" shape="770.32,459.19 776.71,459.62 777.02,461.89 777.54,462.75 778.32,463.40 779.33,463.87 780.60,464.14 780.19,467.32 777.96,467.48 777.15,467.93 776.54,468.64 776.15,469.59 775.95,470.80 769.57,470.37">\n    <request index="0" response="0000" foes="0000" cont="0" />\n    <request index="1" response="1100" foes="1100" cont="1" />\n    <request index="2" response="0000" foes="0010" cont="0" />\n    <request index="3" response="0000" foes="0010" cont="0" />\n  </junction>\n  <junction id="1559016918" type="priority" x="865.46" y="476.66" incLanes="-142455600#0_0 -706256386_0 142455599_0" intLanes=":1559016918_0_0 :1559016918_1_0 :1559016918_2_0 :1559016918_4_0" shape="863.02,469.23 869.39,469.90 869.60,472.14 870.06,472.97 870.77,473.60 871.72,474.04 872.91,474.29 872.19,480.65 869.15,480.18 866.94,479.63 865.15,479.03 863.35,478.42 861.15,477.84 858.11,477.33 858.52,474.16 860.77,473.83 861.61,473.18 862.27,472.19 862.74,470.87">\n    <request index="0" response="1100" foes="1100" cont="0" />\n    <request index="1" response="0000" foes="1000" cont="0" />\n    <request index="2" response="0000" foes="0001" cont="0" />\n    <request index="3" response="0010" foes="0011" cont="1" />\n  </junction>\n  <junction id="1559029797" type="priority" x="373.80" y="741.18" incLanes="142456544_0 84842974#11_0 84842974#11_1 84842974#11_2" intLanes=":1559029797_0_0 :1559029797_1_0 :1559029797_1_1 :1559029797_1_2" shape="369.93,734.70 379.42,736.14 379.53,738.39 379.97,739.21 380.66,739.85 381.60,740.28 382.80,740.52 382.46,743.70 380.12,743.90 379.24,744.37 378.56,745.08 378.08,746.03 377.80,747.23 368.29,745.90">\n    <request index="0" response="1110" foes="1110" cont="0" />\n    <request index="1" response="0000" foes="0001" cont="0" />\n    <request index="2" response="0000" foes="0001" cont="0" />\n    <request index="3" response="0000" foes="0001" cont="0" />\n  </junction>\n  <junction id="1559029798" type="priority" x="457.63" y="749.99" incLanes="706097788#1_0" intLanes=":1559029798_0_0 :1559029798_1_0" shape="456.85,744.46 460.02,744.94 458.47,755.63 455.30,755.19 455.17,753.05 454.74,752.26 454.08,751.65 453.17,751.24 452.02,751.01 452.36,747.83 454.60,747.64 455.44,747.19 456.10,746.51 456.57,745.61">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="1639088623" type="priority" x="1531.15" y="963.11" incLanes="-151072451#2_0 151072453#2_0" intLanes=":1639088623_0_0 :1639088623_1_0 :1639088623_2_0" shape="1536.18,959.03 1537.23,965.34 1535.11,966.14 1534.43,966.82 1534.02,967.68 1533.86,968.72 1533.96,969.94 1530.81,970.49 1528.45,956.23 1531.61,955.73 1532.39,957.86 1533.06,958.53 1533.91,958.95 1534.95,959.12">\n    <request index="0" response="000" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="001" foes="001" cont="0" />\n  </junction>\n  <junction id="1639088636" type="dead_end" x="1964.80" y="1000.49" incLanes="151072451#0_0" intLanes="" shape="1964.80,1000.49 1965.41,1003.63 1964.80,1000.49" />\n  <junction id="1639088658" type="priority" x="1548.26" y="1061.79" incLanes="243079757#1_0 151072453#1_0 -1218747872_0" intLanes=":1639088658_0_0 :1639088658_1_0 :1639088658_2_0 :1639088658_3_0 :1639088658_4_0 :1639088658_5_0 :1639088658_6_0" shape="1555.31,1059.53 1554.44,1065.87 1552.32,1066.00 1551.64,1066.45 1551.22,1067.14 1551.06,1068.08 1551.15,1069.27 1547.99,1069.80 1547.02,1067.20 1546.21,1066.31 1545.19,1065.70 1543.94,1065.37 1542.48,1065.32 1542.11,1058.93 1544.25,1058.37 1544.93,1057.78 1545.35,1056.98 1545.52,1055.98 1545.42,1054.77 1548.57,1054.22 1549.67,1056.81 1550.65,1057.79 1551.91,1058.57 1553.47,1059.15">\n    <request index="0" response="0001000" foes="1001000" cont="0" />\n    <request index="1" response="0011000" foes="1011000" cont="0" />\n    <request index="2" response="0000000" foes="0100000" cont="0" />\n    <request index="3" response="0000000" foes="1100011" cont="0" />\n    <request index="4" response="0000000" foes="1100010" cont="0" />\n    <request index="5" response="0011100" foes="0011100" cont="0" />\n    <request index="6" response="0011011" foes="0011011" cont="0" />\n  </junction>\n  <junction id="1680666294" type="dead_end" x="1170.59" y="306.94" incLanes="155690968_0" intLanes="" shape="1170.59,306.94 1170.50,310.14 1170.59,306.94" />\n  <junction id="1680666322" type="priority" x="935.82" y="314.77" incLanes="706256380#0_0 -155690968_0 -706256380#1_0" intLanes=":1680666322_0_0 :1680666322_6_0 :1680666322_2_0 :1680666322_3_0 :1680666322_4_0 :1680666322_5_0" shape="933.43,307.04 939.79,307.72 940.00,309.94 940.47,310.73 941.19,311.31 942.15,311.68 943.36,311.83 943.14,318.23 940.64,318.59 939.72,319.11 939.03,319.86 938.56,320.84 938.32,322.06 931.95,321.43">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1680666463" type="priority" x="1494.83" y="374.00" incLanes="155691088#0_0 -129842388#2_0 -155691088#1_0 129842388#1_0" intLanes=":1680666463_0_0 :1680666463_1_0 :1680666463_2_0 :1680666463_3_0 :1680666463_4_0 :1680666463_12_0 :1680666463_6_0 :1680666463_7_0 :1680666463_8_0 :1680666463_9_0 :1680666463_10_0 :1680666463_13_0" shape="1495.10,365.47 1500.95,368.06 1500.49,370.19 1500.72,371.02 1501.25,371.70 1502.09,372.22 1503.24,372.58 1501.86,378.83 1499.03,378.71 1497.91,379.09 1496.99,379.77 1496.26,380.74 1495.74,382.00 1489.61,380.14 1489.83,377.90 1489.52,377.03 1488.92,376.31 1488.05,375.76 1486.89,375.36 1488.40,369.15 1491.18,369.23 1492.37,368.78 1493.41,368.01 1494.32,366.91">\n    <request index="0" response="000000010000" foes="000100010000" cont="0" />\n    <request index="1" response="111000110000" foes="111100110000" cont="0" />\n    <request index="2" response="110011110000" foes="110111110000" cont="0" />\n    <request index="3" response="000000000000" foes="100010000000" cont="0" />\n    <request index="4" response="000000000000" foes="100110000111" cont="0" />\n    <request index="5" response="111000000000" foes="111110000110" cont="1" />\n    <request index="6" response="010000000000" foes="010000000100" cont="0" />\n    <request index="7" response="110000111000" foes="110000111100" cont="0" />\n    <request index="8" response="110000110111" foes="110000110111" cont="0" />\n    <request index="9" response="000000000000" foes="000000100010" cont="0" />\n    <request index="10" response="000000000000" foes="000111100110" cont="0" />\n    <request index="11" response="000000011000" foes="000110111110" cont="1" />\n  </junction>\n  <junction id="1680666468" type="priority" x="654.95" y="370.55" incLanes="129827554#0_0 -155690741#1_0 -129827554#4_0" intLanes=":1680666468_0_0 :1680666468_6_0 :1680666468_2_0 :1680666468_3_0 :1680666468_4_0 :1680666468_5_0" shape="658.68,363.15 662.78,368.07 661.46,369.87 661.29,370.77 661.43,371.66 661.89,372.55 662.67,373.45 658.28,378.10 656.02,376.57 654.92,376.29 653.84,376.33 652.78,376.70 651.75,377.39 647.65,372.48">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1680666534" type="dead_end" x="1109.81" y="386.07" incLanes="155690981_0" intLanes="" shape="1109.81,386.07 1109.88,389.27 1109.81,386.07" />\n  <junction id="1680666546" type="priority" x="562.22" y="386.77" incLanes="-155690791#0_0 -129827552#1_0 129827552#0_0" intLanes=":1680666546_0_0 :1680666546_1_0 :1680666546_2_0 :1680666546_3_0 :1680666546_4_0 :1680666546_5_0" shape="561.15,378.89 567.30,380.66 567.09,383.01 567.40,383.95 567.98,384.74 568.84,385.37 569.98,385.84 568.06,391.95 554.32,387.65 556.23,381.55 558.47,381.78 559.37,381.48 560.11,380.90 560.71,380.04">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="0" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="1680666683" type="dead_end" x="1221.95" y="408.43" incLanes="-155690875_0" intLanes="" shape="1221.95,408.43 1225.15,408.26 1221.95,408.43" />\n  <junction id="1680666686" type="priority" x="924.52" y="415.08" incLanes="706256380#3_0 -155690340_0 -706256380#4_0 498671483_0" intLanes=":1680666686_0_0 :1680666686_1_0 :1680666686_12_0 :1680666686_3_0 :1680666686_4_0 :1680666686_5_0 :1680666686_6_0 :1680666686_7_0 :1680666686_13_0 :1680666686_9_0 :1680666686_10_0 :1680666686_11_0" shape="922.26,407.29 928.61,408.08 928.78,410.30 929.24,411.10 929.95,411.69 930.91,412.07 932.12,412.23 931.82,418.63 929.28,418.98 928.35,419.53 927.63,420.33 927.14,421.37 926.87,422.66 920.51,421.92 920.32,419.67 919.86,418.85 919.15,418.22 918.19,417.79 916.99,417.57 917.60,411.20 919.95,410.94 920.83,410.42 921.51,409.64 921.98,408.59">\n    <request index="0" response="000000000000" foes="000100010000" cont="0" />\n    <request index="1" response="000000000000" foes="111100110000" cont="0" />\n    <request index="2" response="000011000000" foes="110111110000" cont="1" />\n    <request index="3" response="000010000000" foes="100010000000" cont="0" />\n    <request index="4" response="000110000111" foes="100110000111" cont="0" />\n    <request index="5" response="111110000110" foes="111110000110" cont="0" />\n    <request index="6" response="000000000000" foes="010000000100" cont="0" />\n    <request index="7" response="000000000000" foes="110000111100" cont="0" />\n    <request index="8" response="000000000111" foes="110000110111" cont="1" />\n    <request index="9" response="000000000010" foes="000000100010" cont="0" />\n    <request index="10" response="000111000110" foes="000111100110" cont="0" />\n    <request index="11" response="000110011110" foes="000110111110" cont="0" />\n  </junction>\n  <junction id="1680666699" type="dead_end" x="1090.43" y="422.87" incLanes="155690340_0" intLanes="" shape="1090.43,422.87 1090.28,426.07 1090.43,422.87" />\n  <junction id="1680666716" type="dead_end" x="1479.04" y="425.79" incLanes="155691088#1_0" intLanes="" shape="1479.04,425.79 1475.98,424.86 1479.04,425.79" />\n  <junction id="1680666775" type="priority" x="1434.74" y="459.90" incLanes="-932339005_0 932339002#0_0 932339002#0_1" intLanes=":1680666775_0_0 :1680666775_1_0 :1680666775_2_0 :1680666775_2_1" shape="1441.95,456.05 1442.44,462.43 1440.30,463.09 1439.62,463.72 1439.19,464.54 1439.02,465.56 1439.10,466.77 1432.79,467.80 1430.46,453.57 1436.78,452.54 1437.64,454.69 1438.40,455.40 1439.37,455.86 1440.55,456.08">\n    <request index="0" response="1100" foes="1100" cont="0" />\n    <request index="1" response="0000" foes="0000" cont="0" />\n    <request index="2" response="0000" foes="0001" cont="0" />\n    <request index="3" response="0000" foes="0001" cont="0" />\n  </junction>\n  <junction id="1680666789" type="dead_end" x="1089.08" y="461.31" incLanes="-155690528_0" intLanes="" shape="1089.08,461.31 1092.27,461.58 1089.08,461.31" />\n  <junction id="1680666823" type="dead_end" x="1032.81" y="467.36" incLanes="155691032_0" intLanes="" shape="1032.81,467.36 1032.55,470.55 1032.81,467.36" />\n  <junction id="1680666827" type="priority" x="1545.67" y="470.73" incLanes="-155691084#2_0 -155691015#0_0 155691084#1_0" intLanes=":1680666827_0_0 :1680666827_6_0 :1680666827_2_0 :1680666827_3_0 :1680666827_4_0 :1680666827_5_0" shape="1553.49,468.55 1552.64,474.89 1550.41,475.04 1549.61,475.49 1549.01,476.19 1548.63,477.15 1548.45,478.35 1542.06,478.00 1541.76,475.45 1541.25,474.50 1540.52,473.78 1539.55,473.27 1538.34,472.99 1539.18,466.64">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1680666891" type="priority" x="1632.80" y="484.92" incLanes="-155691084#3_0 -155691022#0_0 155691084#2_0" intLanes=":1680666891_0_0 :1680666891_6_0 :1680666891_2_0 :1680666891_3_0 :1680666891_4_0 :1680666891_5_0" shape="1640.64,483.44 1639.20,489.68 1636.95,489.62 1636.08,489.99 1635.39,490.62 1634.87,491.53 1634.53,492.71 1628.25,491.47 1628.26,489.12 1627.87,488.21 1627.22,487.48 1626.31,486.94 1625.14,486.57 1626.46,480.31">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="1680667029" type="priority" x="981.09" y="509.46" incLanes="-155691032_0 -155690878#1_0 155690878#0_0" intLanes=":1680667029_0_0 :1680667029_1_0 :1680667029_2_0 :1680667029_3_0 :1680667029_4_0 :1680667029_6_0" shape="978.74,501.90 985.10,502.63 985.29,504.88 985.75,505.71 986.46,506.34 987.42,506.77 988.62,507.01 987.98,513.37 973.71,512.01 974.28,505.64 976.55,505.38 977.39,504.88 978.04,504.13 978.49,503.14">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="1680667034" type="dead_end" x="1032.55" y="514.67" incLanes="155690878#1_0" intLanes="" shape="1032.55,514.67 1032.23,517.85 1032.55,514.67" />\n  <junction id="1680667079" type="priority" x="1546.44" y="522.80" incLanes="-155690734#1_0 -155691015#1_0 155690734#0_0 155691015#0_0" intLanes=":1680667079_0_0 :1680667079_1_0 :1680667079_12_0 :1680667079_3_0 :1680667079_4_0 :1680667079_5_0 :1680667079_6_0 :1680667079_7_0 :1680667079_13_0 :1680667079_9_0 :1680667079_10_0 :1680667079_11_0" shape="1554.00,519.82 1553.81,526.22 1551.61,526.60 1550.86,527.13 1550.35,527.89 1550.07,528.88 1550.02,530.10 1543.63,530.42 1543.01,528.03 1542.35,527.20 1541.46,526.62 1540.33,526.29 1538.97,526.20 1538.80,519.81 1540.96,519.31 1541.67,518.73 1542.12,517.95 1542.32,516.95 1542.27,515.73 1548.61,514.89 1549.50,517.57 1550.28,518.52 1551.29,519.21 1552.53,519.65">\n    <request index="0" response="000000000000" foes="000100010000" cont="0" />\n    <request index="1" response="000000000000" foes="111100110000" cont="0" />\n    <request index="2" response="000111000000" foes="110111110000" cont="1" />\n    <request index="3" response="000010000000" foes="100010000000" cont="0" />\n    <request index="4" response="000110000111" foes="100110000111" cont="0" />\n    <request index="5" response="111110000110" foes="111110000110" cont="0" />\n    <request index="6" response="000000000000" foes="010000000100" cont="0" />\n    <request index="7" response="000000000000" foes="110000111100" cont="0" />\n    <request index="8" response="000000000011" foes="110000110111" cont="1" />\n    <request index="9" response="000000000010" foes="000000100010" cont="0" />\n    <request index="10" response="000111000110" foes="000111100110" cont="0" />\n    <request index="11" response="000110011110" foes="000110111110" cont="0" />\n  </junction>\n  <junction id="1680667083" type="dead_end" x="1502.38" y="524.00" incLanes="-155690734#0_0" intLanes="" shape="1502.38,524.00 1502.29,520.80 1502.38,524.00" />\n  <junction id="1680667090" type="dead_end" x="1590.75" y="524.12" incLanes="155690734#1_0" intLanes="" shape="1590.75,524.12 1590.65,527.32 1590.75,524.12" />\n  <junction id="1680667158" type="dead_end" x="1592.21" y="545.81" incLanes="155690812#1_0" intLanes="" shape="1592.21,545.81 1592.42,549.00 1592.21,545.81" />\n  <junction id="1680667164" type="priority" x="1547.77" y="548.71" incLanes="-155690812#1_0 155690812#0_0 155691015#1_0" intLanes=":1680667164_0_0 :1680667164_1_0 :1680667164_2_0 :1680667164_6_0 :1680667164_4_0 :1680667164_5_0" shape="1554.79,545.05 1555.21,551.43 1540.99,552.52 1540.43,546.14 1542.62,545.49 1543.37,544.84 1543.88,543.99 1544.16,542.92 1544.20,541.64 1550.59,541.31 1551.15,543.50 1551.75,544.24 1552.55,544.75 1553.57,545.01">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="1680667181" type="priority" x="1081.84" y="548.25" incLanes="155690528_0 95533432#4_0 -95533432#5_0" intLanes=":1680667181_0_0 :1680667181_1_0 :1680667181_2_0 :1680667181_3_0 :1680667181_4_0 :1680667181_6_0" shape="1079.25,540.83 1085.62,541.36 1085.88,543.65 1086.36,544.49 1087.09,545.13 1088.05,545.56 1089.26,545.80 1088.60,552.17 1074.43,550.87 1074.94,544.49 1077.19,544.22 1078.00,543.73 1078.62,543.00 1079.04,542.04">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="1680667509" type="dead_end" x="1341.86" y="649.57" incLanes="-155690480#0_0" intLanes="" shape="1341.86,649.57 1341.84,646.37 1341.86,649.57" />\n  <junction id="1680667686" type="dead_end" x="1803.64" y="705.64" incLanes="155690766_0" intLanes="" shape="1803.64,705.64 1802.73,708.71 1803.64,705.64" />\n  <junction id="1680667730" type="priority" x="1464.25" y="711.76" incLanes="155690513_0 990212966#1_0" intLanes=":1680667730_0_0 :1680667730_1_0 :1680667730_2_0" shape="1466.95,718.37 1463.79,718.89 1462.86,716.68 1462.02,715.90 1460.93,715.33 1459.59,714.97 1458.00,714.82 1458.10,708.43 1460.25,708.04 1460.94,707.51 1461.37,706.75 1461.54,705.77 1461.46,704.57 1464.62,704.05">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="1680668223" type="priority" x="1638.36" y="1706.07" incLanes="95542480_0 970323546#0_0 970323546#0_1" intLanes=":1680668223_0_0 :1680668223_1_0 :1680668223_2_0 :1680668223_2_1" shape="1644.92,1701.55 1646.12,1707.84 1643.91,1708.71 1643.21,1709.42 1642.76,1710.29 1642.58,1711.35 1642.66,1712.57 1636.35,1713.61 1634.03,1699.39 1640.34,1698.36 1641.14,1700.47 1641.81,1701.13 1642.66,1701.53 1643.70,1701.67">\n    <request index="0" response="1100" foes="1100" cont="0" />\n    <request index="1" response="0000" foes="0000" cont="0" />\n    <request index="2" response="0000" foes="0001" cont="0" />\n    <request index="3" response="0000" foes="0001" cont="0" />\n  </junction>\n  <junction id="1680668226" type="priority" x="1626.47" y="1708.28" incLanes="-243314395_0 65080358#3_0" intLanes=":1680668226_0_0 :1680668226_1_0 :1680668226_2_0" shape="1629.23,1715.27 1626.07,1715.78 1625.28,1713.69 1624.62,1713.06 1623.77,1712.70 1622.75,1712.61 1621.54,1712.79 1620.01,1706.58 1622.37,1705.54 1623.14,1704.78 1623.62,1703.86 1623.82,1702.79 1623.75,1701.55 1626.90,1701.03">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="1680668239" type="dead_end" x="1671.32" y="1970.82" incLanes="65080358#4_0" intLanes="" shape="1669.75,1971.12 1672.89,1970.52" />\n  <junction id="1966386254" type="priority" x="564.09" y="872.41" incLanes="706093509#1_0 931354569#4_0" intLanes=":1966386254_0_0 :1966386254_1_0 :1966386254_2_0 :1966386254_3_0 :1966386254_4_0" shape="568.49,866.46 570.59,868.87 569.91,869.72 569.89,870.07 570.07,870.37 570.46,870.62 571.07,870.82 569.55,877.04 566.95,876.65 565.99,876.73 565.26,876.99 564.76,877.43 564.49,878.05 561.37,877.36 561.60,875.17 561.47,874.34 561.18,873.69 560.73,873.22 560.11,872.92 561.00,869.85 563.03,869.82 564.22,869.39 565.53,868.69 566.95,867.71">\n    <request index="0" response="01000" foes="01000" cont="0" />\n    <request index="1" response="11000" foes="11000" cont="0" />\n    <request index="2" response="11000" foes="11000" cont="0" />\n    <request index="3" response="00000" foes="00111" cont="0" />\n    <request index="4" response="00000" foes="00110" cont="0" />\n  </junction>\n  <junction id="1966399248" type="priority" x="775.07" y="1136.75" incLanes="96941842#3_0 706104219#8_0 243188491_0 -706104219#9_0" intLanes=":1966399248_0_0 :1966399248_1_0 :1966399248_2_0 :1966399248_3_0 :1966399248_4_0 :1966399248_12_0 :1966399248_6_0 :1966399248_7_0 :1966399248_8_0 :1966399248_9_0 :1966399248_10_0 :1966399248_13_0" shape="773.15,1128.84 779.47,1129.90 779.54,1132.13 779.97,1132.94 780.65,1133.55 781.59,1133.96 782.79,1134.17 782.27,1140.55 779.99,1140.81 779.18,1141.30 778.59,1142.03 778.23,1143.00 778.08,1144.21 771.68,1144.05 771.30,1141.59 770.77,1140.70 770.01,1140.03 769.02,1139.59 767.81,1139.37 768.33,1132.99 770.67,1132.68 771.57,1132.12 772.28,1131.29 772.81,1130.20">\n    <request index="0" response="000000010000" foes="000100010000" cont="0" />\n    <request index="1" response="111000110000" foes="111100110000" cont="0" />\n    <request index="2" response="110011110000" foes="110111110000" cont="0" />\n    <request index="3" response="000000000000" foes="100010000000" cont="0" />\n    <request index="4" response="000000000000" foes="100110000111" cont="0" />\n    <request index="5" response="011000000000" foes="111110000110" cont="1" />\n    <request index="6" response="010000000000" foes="010000000100" cont="0" />\n    <request index="7" response="110000111000" foes="110000111100" cont="0" />\n    <request index="8" response="110000110111" foes="110000110111" cont="0" />\n    <request index="9" response="000000000000" foes="000000100010" cont="0" />\n    <request index="10" response="000000000000" foes="000111100110" cont="0" />\n    <request index="11" response="000000111000" foes="000110111110" cont="1" />\n  </junction>\n  <junction id="1966404349" type="priority" x="1027.19" y="1067.73" incLanes="498671395#0_0 154097280_0 96941857#2_0" intLanes=":1966404349_0_0 :1966404349_1_0 :1966404349_2_0 :1966404349_3_0 :1966404349_7_0 :1966404349_5_0 :1966404349_6_0" shape="1025.76,1060.35 1028.96,1060.42 1029.41,1062.84 1030.02,1063.74 1030.88,1064.42 1032.00,1064.89 1033.38,1065.15 1032.72,1071.52 1030.50,1071.73 1029.72,1072.21 1029.16,1072.93 1028.81,1073.90 1028.69,1075.11 1025.49,1075.06 1025.01,1072.62 1024.40,1071.72 1023.52,1071.03 1022.39,1070.55 1021.00,1070.28 1021.69,1063.92 1023.91,1063.71 1024.70,1063.24 1025.27,1062.52 1025.62,1061.56">\n    <request index="0" response="0001000" foes="0001000" cont="0" />\n    <request index="1" response="1111000" foes="1111000" cont="0" />\n    <request index="2" response="1011000" foes="1011000" cont="0" />\n    <request index="3" response="0000000" foes="0000111" cont="0" />\n    <request index="4" response="1100000" foes="1100110" cont="1" />\n    <request index="5" response="0000000" foes="0010010" cont="0" />\n    <request index="6" response="0000000" foes="0010110" cont="0" />\n  </junction>\n  <junction id="1966412828" type="priority" x="436.19" y="395.10" incLanes="84842974#15_0 84842974#15_1 84842974#15_2 101706795#0_0" intLanes=":1966412828_0_0 :1966412828_1_0 :1966412828_1_1 :1966412828_1_2 :1966412828_4_0 :1966412828_5_0 :1966412828_5_1" shape="431.93,389.19 441.48,390.09 441.68,392.34 442.11,393.17 442.77,393.81 443.65,394.24 444.75,394.48 444.38,397.66 442.21,397.90 441.37,398.41 440.68,399.19 440.15,400.23 439.77,401.53 430.37,399.60 430.37,397.55 430.01,396.78 429.41,396.18 428.56,395.76 427.47,395.50 427.91,392.33 429.99,392.21 430.75,391.80 431.33,391.16 431.72,390.29">\n    <request index="0" response="0000000" foes="0010000" cont="0" />\n    <request index="1" response="0000000" foes="1110000" cont="0" />\n    <request index="2" response="0000000" foes="1110000" cont="0" />\n    <request index="3" response="0000000" foes="1110000" cont="0" />\n    <request index="4" response="0001111" foes="0001111" cont="0" />\n    <request index="5" response="0001110" foes="0001110" cont="0" />\n    <request index="6" response="0001110" foes="0001110" cont="0" />\n  </junction>\n  <junction id="2052489649" type="priority" x="730.39" y="800.51" incLanes="706520617#3_0 706092523#4_0 -706520617#4_0" intLanes=":2052489649_0_0 :2052489649_1_0 :2052489649_2_0 :2052489649_3_0" shape="738.17,804.02 733.54,808.44 731.90,807.27 731.23,807.24 730.65,807.60 730.18,808.33 729.81,809.43 726.70,808.66 727.26,806.20 727.52,804.34 727.50,802.77 727.22,801.21 726.68,799.37 725.91,796.96 732.01,795.02 732.78,797.22 733.50,798.79 734.29,799.99 735.26,801.09 736.52,802.35">\n    <request index="0" response="0100" foes="0100" cont="0" />\n    <request index="1" response="1000" foes="1000" cont="0" />\n    <request index="2" response="0000" foes="1001" cont="0" />\n    <request index="3" response="0100" foes="0110" cont="0" />\n  </junction>\n  <junction id="2052497441" type="priority" x="509.66" y="1149.54" incLanes="706093506#0_0" intLanes=":2052497441_0_0 :2052497441_1_0" shape="509.08,1143.73 512.23,1144.30 510.32,1155.29 507.16,1154.76 507.09,1152.49 506.66,1151.61 505.98,1150.92 505.04,1150.40 503.84,1150.06 504.44,1146.92 506.71,1146.89 507.57,1146.49 508.25,1145.83 508.76,1144.91">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="2052522376" type="priority" x="325.67" y="1115.51" incLanes="194743656#1_0 84842974#4_0 84842974#4_1 84842974#4_2" intLanes=":2052522376_0_0 :2052522376_1_0 :2052522376_1_1 :2052522376_1_2" shape="321.80,1109.38 331.28,1110.91 331.36,1113.14 331.77,1113.99 332.44,1114.67 333.35,1115.17 334.51,1115.49 333.94,1118.64 331.71,1118.68 330.86,1119.07 330.17,1119.73 329.66,1120.63 329.32,1121.79 319.90,1119.92">\n    <request index="0" response="1110" foes="1110" cont="0" />\n    <request index="1" response="0000" foes="0001" cont="0" />\n    <request index="2" response="0000" foes="0001" cont="0" />\n    <request index="3" response="0000" foes="0001" cont="0" />\n  </junction>\n  <junction id="2052547074" type="traffic_light" x="482.68" y="577.53" incLanes="706096640#0_0 706256377#3_0 -706256377#4_0" intLanes=":2052547074_0_0 :2052547074_1_0 :2052547074_2_0 :2052547074_3_0 :2052547074_4_0 :2052547074_5_0 :2052547074_6_0" shape="482.08,570.13 485.25,570.56 485.39,572.81 485.84,573.64 486.54,574.27 487.49,574.70 488.69,574.94 488.02,581.30 485.64,581.52 484.74,582.00 484.05,582.74 483.56,583.73 483.27,584.97 480.10,584.54 479.96,582.30 479.52,581.48 478.82,580.87 477.86,580.45 476.66,580.24 477.19,573.87 479.66,573.61 480.58,573.11 481.29,572.36 481.79,571.37">\n    <request index="0" response="0001000" foes="0001000" cont="0" />\n    <request index="1" response="1111000" foes="1111000" cont="0" />\n    <request index="2" response="1011000" foes="1011000" cont="0" />\n    <request index="3" response="0000000" foes="0000111" cont="0" />\n    <request index="4" response="1100000" foes="1100110" cont="0" />\n    <request index="5" response="0000000" foes="0010010" cont="0" />\n    <request index="6" response="0000000" foes="0010110" cont="0" />\n  </junction>\n  <junction id="2052547075" type="priority" x="698.84" y="597.67" incLanes="706256377#0_0 154707523#0_0 -706256377#1_0" intLanes=":2052547075_0_0 :2052547075_1_0 :2052547075_2_0 :2052547075_3_0 :2052547075_4_0 :2052547075_5_0 :2052547075_7_0" shape="698.22,590.24 701.40,590.67 701.54,592.91 701.99,593.73 702.69,594.34 703.65,594.76 704.85,594.97 704.30,601.35 702.02,601.60 701.20,602.09 700.60,602.82 700.20,603.79 700.02,605.00 696.82,604.81 696.51,602.50 696.00,601.66 695.26,601.03 694.29,600.60 693.07,600.38 693.63,594.00 695.94,593.74 696.80,593.24 697.47,592.49 697.94,591.49">\n    <request index="0" response="0000000" foes="1001000" cont="0" />\n    <request index="1" response="0000000" foes="1011000" cont="0" />\n    <request index="2" response="0100000" foes="0100000" cont="0" />\n    <request index="3" response="1100011" foes="1100011" cont="0" />\n    <request index="4" response="1100010" foes="1100010" cont="0" />\n    <request index="5" response="0000000" foes="0011100" cont="0" />\n    <request index="6" response="0000011" foes="0011011" cont="1" />\n  </junction>\n  <junction id="2406954575" type="dead_end" x="1989.24" y="1153.73" incLanes="" intLanes="" shape="1989.35,1155.33 1989.13,1152.13" />\n  <junction id="2406954577" type="dead_end" x="1989.32" y="1164.50" incLanes="243079844#1_0" intLanes="" shape="1989.44,1166.10 1989.20,1162.90" />\n  <junction id="2505176020" type="priority" x="1628.90" y="758.14" incLanes="243314310#1_0 970010767#0_0" intLanes=":2505176020_0_0 :2505176020_1_0 :2505176020_2_0" shape="1626.17,752.25 1632.55,752.77 1632.81,755.01 1633.30,755.83 1634.03,756.43 1635.00,756.83 1636.22,757.02 1636.00,760.21 1621.63,759.26 1621.85,756.07 1624.10,755.76 1624.92,755.25 1625.54,754.49 1625.96,753.49">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="2505176027" type="priority" x="1855.54" y="779.68" incLanes="243314467#1_0 969757853#2_0" intLanes=":2505176027_0_0" shape="1858.40,785.37 1852.01,784.99 1851.71,782.77 1851.21,781.96 1850.47,781.36 1849.51,780.96 1848.31,780.76 1848.54,777.57 1851.67,777.99 1854.21,778.77 1856.15,779.89 1857.50,781.37 1858.25,783.20">\n    <request index="0" response="0" foes="0" cont="0" />\n  </junction>\n  <junction id="2505176175" type="priority" x="1533.23" y="1062.66" incLanes="1218747873_0 1218225518#0_0 1218225518#0_1" intLanes=":2505176175_0_0 :2505176175_1_0 :2505176175_2_0 :2505176175_2_1" shape="1540.45,1059.02 1540.96,1065.40 1538.81,1065.96 1538.12,1066.55 1537.70,1067.35 1537.53,1068.35 1537.62,1069.56 1531.30,1070.61 1528.96,1056.35 1535.27,1055.32 1536.14,1057.48 1536.89,1058.21 1537.86,1058.72 1539.05,1058.98">\n    <request index="0" response="1100" foes="1100" cont="0" />\n    <request index="1" response="0000" foes="0000" cont="0" />\n    <request index="2" response="0000" foes="0001" cont="0" />\n    <request index="3" response="0000" foes="0001" cont="0" />\n  </junction>\n  <junction id="2505176258" type="priority" x="1555.83" y="1200.99" incLanes="-1218747879_0 970323546#6_0 970323546#6_1 66984610#0_0" intLanes=":2505176258_0_0 :2505176258_1_0 :2505176258_2_0 :2505176258_3_0 :2505176258_3_1 :2505176258_5_0 :2505176258_6_0 :2505176258_7_0" shape="1562.81,1197.03 1563.51,1203.40 1561.37,1204.05 1560.69,1204.67 1560.26,1205.50 1560.09,1206.52 1560.18,1207.75 1553.86,1208.78 1553.01,1206.47 1552.29,1205.72 1551.37,1205.22 1550.26,1204.99 1548.96,1205.03 1548.20,1198.67 1550.34,1197.97 1551.03,1197.33 1551.47,1196.50 1551.65,1195.47 1551.58,1194.25 1557.90,1193.29 1558.73,1195.55 1559.44,1196.30 1560.36,1196.80 1561.49,1197.04">\n    <request index="0" response="00011000" foes="00011000" cont="0" />\n    <request index="1" response="00111000" foes="10111000" cont="0" />\n    <request index="2" response="00000000" foes="01000000" cont="0" />\n    <request index="3" response="00000000" foes="11000011" cont="0" />\n    <request index="4" response="00000000" foes="11000011" cont="0" />\n    <request index="5" response="00000000" foes="11000010" cont="0" />\n    <request index="6" response="00111100" foes="00111100" cont="0" />\n    <request index="7" response="00111010" foes="00111010" cont="0" />\n  </junction>\n  <junction id="2505176265" type="priority" x="1570.92" y="1199.46" incLanes="66984613#7_0 66984610#1_0" intLanes=":2505176265_0_0 :2505176265_1_0 :2505176265_3_0" shape="1573.16,1203.82 1569.99,1204.27 1569.68,1203.47 1569.40,1203.20 1569.03,1203.03 1568.58,1202.95 1568.04,1202.96 1567.41,1196.59 1568.22,1196.35 1568.48,1196.11 1568.65,1195.80 1568.72,1195.42 1568.70,1194.96 1571.87,1194.53">\n    <request index="0" response="000" foes="100" cont="0" />\n    <request index="1" response="000" foes="100" cont="0" />\n    <request index="2" response="011" foes="011" cont="1" />\n  </junction>\n  <junction id="2505176388" type="priority" x="1606.03" y="1424.45" incLanes="96713525_0 66984613#3_0 -243314186_0" intLanes=":2505176388_0_0 :2505176388_1_0 :2505176388_2_0 :2505176388_3_0 :2505176388_4_0 :2505176388_5_0 :2505176388_6_0" shape="1612.47,1421.55 1612.16,1427.95 1610.01,1428.28 1609.32,1428.79 1608.89,1429.53 1608.71,1430.51 1608.78,1431.71 1605.62,1432.20 1604.77,1429.80 1604.04,1429.01 1603.11,1428.49 1601.99,1428.24 1600.67,1428.26 1599.96,1421.90 1602.10,1421.22 1602.79,1420.58 1603.22,1419.76 1603.40,1418.73 1603.32,1417.51 1606.48,1417.01 1607.44,1419.42 1608.31,1420.29 1609.44,1420.94 1610.82,1421.36">\n    <request index="0" response="0001000" foes="1001000" cont="0" />\n    <request index="1" response="0011000" foes="1011000" cont="0" />\n    <request index="2" response="0000000" foes="0100000" cont="0" />\n    <request index="3" response="0000000" foes="1100011" cont="0" />\n    <request index="4" response="0000000" foes="1100010" cont="0" />\n    <request index="5" response="0011100" foes="0011100" cont="0" />\n    <request index="6" response="0011011" foes="0011011" cont="0" />\n  </junction>\n  <junction id="2505176470" type="dead_end" x="1991.67" y="1447.43" incLanes="-96713525_0" intLanes="" shape="1991.67,1447.43 1991.02,1450.56 1991.67,1447.43" />\n  <junction id="2505176494" type="priority" x="1600.47" y="1474.18" incLanes="243079673_0 970323546#2_0 970323546#2_1" intLanes=":2505176494_0_0 :2505176494_1_0 :2505176494_2_0 :2505176494_2_1" shape="1607.76,1470.49 1608.19,1476.87 1606.04,1477.46 1605.36,1478.06 1604.93,1478.86 1604.75,1479.87 1604.84,1481.09 1598.52,1482.12 1596.20,1467.87 1602.52,1466.85 1603.39,1469.00 1604.15,1469.73 1605.13,1470.22 1606.34,1470.47">\n    <request index="0" response="1100" foes="1100" cont="0" />\n    <request index="1" response="0000" foes="0000" cont="0" />\n    <request index="2" response="0000" foes="0001" cont="0" />\n    <request index="3" response="0000" foes="0001" cont="0" />\n  </junction>\n  <junction id="2505176514" type="priority" x="1603.82" y="1494.68" incLanes="-243079775_0 970323546#1_0 970323546#1_1" intLanes=":2505176514_0_0 :2505176514_1_0 :2505176514_2_0 :2505176514_2_1" shape="1611.28,1491.16 1611.56,1497.55 1609.41,1498.08 1608.72,1498.67 1608.29,1499.46 1608.12,1500.46 1608.20,1501.67 1601.89,1502.70 1599.56,1488.45 1605.87,1487.41 1606.76,1489.58 1607.55,1490.32 1608.56,1490.83 1609.81,1491.11">\n    <request index="0" response="1100" foes="1100" cont="0" />\n    <request index="1" response="0000" foes="0000" cont="0" />\n    <request index="2" response="0000" foes="0001" cont="0" />\n    <request index="3" response="0000" foes="0001" cont="0" />\n  </junction>\n  <junction id="2505176598" type="dead_end" x="1988.69" y="1551.38" incLanes="243314440_0" intLanes="" shape="1988.69,1551.38 1988.21,1554.54 1988.69,1551.38" />\n  <junction id="2505176686" type="dead_end" x="1986.72" y="1591.26" incLanes="-243079723_0" intLanes="" shape="1986.72,1591.26 1986.54,1594.46 1986.72,1591.26" />\n  <junction id="2505176697" type="priority" x="1341.59" y="1591.88" incLanes="-96941841#0_0 -243188765#0_0 95809258_0" intLanes=":2505176697_0_0 :2505176697_1_0 :2505176697_2_0 :2505176697_3_0 :2505176697_4_0 :2505176697_6_0" shape="1350.21,1590.48 1348.85,1596.73 1346.70,1596.70 1345.96,1597.09 1345.44,1597.75 1345.16,1598.68 1345.09,1599.88 1338.70,1600.12 1338.09,1585.58 1344.49,1585.28 1345.20,1587.63 1346.01,1588.58 1347.11,1589.36 1348.51,1590.00">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="2505176747" type="dead_end" x="1734.25" y="1636.75" incLanes="71380788_0" intLanes="" shape="1734.25,1636.75 1734.10,1633.55 1734.25,1636.75" />\n  <junction id="2505176985" type="dead_end" x="1666.90" y="1880.75" incLanes="" intLanes="" shape="1663.74,1881.27 1670.06,1880.23" />\n  <junction id="2506085476" type="priority" x="1269.22" y="566.52" incLanes="95533432#0_0 243188318_0 -95533432#1_0" intLanes=":2506085476_0_0 :2506085476_6_0 :2506085476_2_0 :2506085476_3_0 :2506085476_4_0 :2506085476_5_0" shape="1276.77,563.82 1276.34,570.20 1274.12,570.50 1273.33,571.00 1272.76,571.74 1272.40,572.72 1272.27,573.94 1265.87,573.80 1265.48,571.36 1264.94,570.47 1264.18,569.80 1263.19,569.36 1261.97,569.14 1262.49,562.76">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="2506085486" type="dead_end" x="1267.64" y="640.76" incLanes="-243188318_0" intLanes="" shape="1267.64,640.76 1264.44,640.69 1267.64,640.76" />\n  <junction id="2506085487" type="dead_end" x="1196.15" y="642.20" incLanes="243188607#1_0" intLanes="" shape="1196.15,642.20 1199.35,642.33 1196.15,642.20" />\n  <junction id="2506085493" type="priority" x="1195.62" y="654.79" incLanes="-243188607#1_0 243188458_0 243188607#0_0" intLanes=":2506085493_0_0 :2506085493_6_0 :2506085493_2_0 :2506085493_3_0 :2506085493_4_0 :2506085493_5_0" shape="1192.80,645.87 1199.19,646.15 1199.52,648.22 1200.01,648.81 1200.71,649.10 1201.64,649.09 1202.78,648.78 1204.97,654.79 1203.01,655.61 1201.38,656.53 1200.08,657.53 1199.11,658.63 1198.47,659.81 1198.16,661.09 1191.80,660.41">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="2506085496" type="priority" x="663.32" y="668.35" incLanes="40264697#6_0 40264697#6_1" intLanes=":2506085496_0_0 :2506085496_1_0 :2506085496_1_1" shape="668.74,665.56 668.24,671.94 665.93,672.24 665.03,672.78 664.29,673.56 663.73,674.60 663.35,675.90 660.22,675.23 660.24,673.01 659.85,672.19 659.19,671.56 658.27,671.13 657.07,670.88 657.77,664.52">\n    <request index="0" response="000" foes="000" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="000" cont="0" />\n  </junction>\n  <junction id="2506085508" type="priority" x="1184.44" y="688.87" incLanes="-243188607#0_0 706094740#2_0 706094742_0" intLanes=":2506085508_0_0 :2506085508_1_0 :2506085508_2_0 :2506085508_3_0 :2506085508_4_0 :2506085508_6_0" shape="1185.93,680.78 1191.28,684.29 1190.45,686.35 1190.52,687.25 1190.91,688.07 1191.62,688.80 1192.65,689.44 1189.75,695.15 1177.10,689.36 1179.50,683.42 1181.86,683.79 1182.96,683.51 1184.00,682.91 1184.99,682.00">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="2506085525" type="priority" x="1353.51" y="736.98" incLanes="243188024_0 969757851#2_0" intLanes=":2506085525_0_0 :2506085525_1_0 :2506085525_2_0" shape="1360.79,736.07 1360.49,739.25 1358.22,739.50 1357.38,740.00 1356.73,740.75 1356.28,741.75 1356.02,743.00 1349.66,742.24 1349.48,739.99 1349.02,739.17 1348.31,738.54 1347.36,738.12 1346.15,737.89 1346.45,734.71">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="2506085555" type="priority" x="1215.59" y="811.54" incLanes="243188741#0_0 -498671678_0" intLanes=":2506085555_0_0 :2506085555_4_0 :2506085555_2_0 :2506085555_3_0" shape="1221.18,808.11 1221.44,814.51 1219.25,815.04 1218.52,815.62 1218.02,816.42 1217.77,817.43 1217.75,818.66 1214.56,818.91 1213.95,816.71 1213.33,815.98 1212.51,815.50 1211.49,815.26 1210.27,815.27 1209.65,808.90">\n    <request index="0" response="0000" foes="0000" cont="0" />\n    <request index="1" response="1100" foes="1100" cont="1" />\n    <request index="2" response="0000" foes="0010" cont="0" />\n    <request index="3" response="0000" foes="0010" cont="0" />\n  </junction>\n  <junction id="2506085628" type="priority" x="615.42" y="885.46" incLanes="243188064#1_0 931354569#3_0 -931354569#4_0" intLanes=":2506085628_0_0 :2506085628_1_0 :2506085628_2_0 :2506085628_3_0" shape="615.45,878.00 618.57,878.70 618.49,881.08 618.85,882.02 619.47,882.80 620.36,883.41 621.51,883.85 619.78,890.01 609.00,886.99 610.72,880.83 612.96,880.99 613.83,880.66 614.53,880.05 615.07,879.16">\n    <request index="0" response="0100" foes="0100" cont="0" />\n    <request index="1" response="1100" foes="1100" cont="0" />\n    <request index="2" response="0000" foes="0011" cont="0" />\n    <request index="3" response="0000" foes="0010" cont="0" />\n  </junction>\n  <junction id="2506085631" type="priority" x="1221.64" y="888.53" incLanes="-243188351#1_0 498671616_0 243188499_0" intLanes=":2506085631_0_0 :2506085631_7_0 :2506085631_2_0 :2506085631_3_0 :2506085631_4_0 :2506085631_5_0 :2506085631_6_0" shape="1226.97,884.88 1227.49,891.26 1225.29,891.91 1224.55,892.55 1224.05,893.40 1223.79,894.46 1223.77,895.74 1220.58,895.97 1219.98,893.83 1219.37,893.14 1218.57,892.71 1217.55,892.53 1216.34,892.61 1215.34,886.29 1217.78,885.44 1218.61,884.73 1219.16,883.83 1219.46,882.75 1219.48,881.47 1222.67,881.22 1223.29,883.40 1223.90,884.13 1224.72,884.62 1225.74,884.87">\n    <request index="0" response="0000000" foes="1110000" cont="0" />\n    <request index="1" response="0001100" foes="1101100" cont="1" />\n    <request index="2" response="0000000" foes="0100010" cont="0" />\n    <request index="3" response="0000000" foes="1100010" cont="0" />\n    <request index="4" response="0000001" foes="0000001" cont="0" />\n    <request index="5" response="0001111" foes="0001111" cont="0" />\n    <request index="6" response="0001011" foes="0001011" cont="0" />\n  </junction>\n  <junction id="2506085670" type="dead_end" x="490.72" y="910.90" incLanes="-243188447_0" intLanes="" shape="490.72,910.90 493.84,911.62 490.72,910.90" />\n  <junction id="2506085690" type="priority" x="751.80" y="928.65" incLanes="931354570_0 969757849#3_0 -243188561#1_0 -931354569#0_0" intLanes=":2506085690_0_0 :2506085690_1_0 :2506085690_2_0 :2506085690_3_0 :2506085690_9_0 :2506085690_5_0 :2506085690_6_0 :2506085690_7_0 :2506085690_8_0" shape="752.36,921.32 755.42,922.25 755.40,924.57 755.96,925.57 756.90,926.45 758.21,927.21 759.91,927.87 757.97,933.97 755.75,933.73 754.89,934.04 754.19,934.63 753.65,935.50 753.28,936.66 747.03,935.25 747.15,932.73 746.80,931.73 746.19,930.92 745.31,930.29 744.16,929.85 745.87,923.68 748.95,924.07 750.15,923.83 751.12,923.29 751.85,922.45">\n    <request index="0" response="000001000" foes="001001000" cont="0" />\n    <request index="1" response="110011000" foes="111011000" cont="0" />\n    <request index="2" response="100111000" foes="100111000" cont="0" />\n    <request index="3" response="000000000" foes="001000111" cont="0" />\n    <request index="4" response="110000000" foes="111000110" cont="1" />\n    <request index="5" response="100000000" foes="100000100" cont="0" />\n    <request index="6" response="100011011" foes="100011011" cont="0" />\n    <request index="7" response="000000000" foes="000010010" cont="0" />\n    <request index="8" response="000000000" foes="001110110" cont="0" />\n  </junction>\n  <junction id="2506085784" type="dead_end" x="278.20" y="957.28" incLanes="" intLanes="" shape="278.44,955.70 277.96,958.86" />\n  <junction id="2506085849" type="priority" x="471.72" y="993.47" incLanes="243188447_0 684714761#2_0" intLanes=":2506085849_0_0 :2506085849_1_0 :2506085849_2_0 :2506085849_3_0 :2506085849_4_0" shape="469.88,987.20 476.12,988.64 476.05,990.89 476.42,991.76 477.06,992.45 477.97,992.97 479.14,993.32 478.51,996.46 475.23,996.29 473.97,996.64 472.95,997.28 472.18,998.22 471.66,999.44 468.59,998.55 468.61,996.31 468.10,995.42 467.22,994.69 465.99,994.11 464.40,993.70 465.01,990.55 467.29,990.52 468.19,990.09 468.92,989.40 469.48,988.44">\n    <request index="0" response="11100" foes="11100" cont="0" />\n    <request index="1" response="11000" foes="11000" cont="0" />\n    <request index="2" response="00000" foes="00001" cont="0" />\n    <request index="3" response="00000" foes="00011" cont="0" />\n    <request index="4" response="00000" foes="00011" cont="0" />\n  </junction>\n  <junction id="2506086151" type="dead_end" x="718.23" y="1077.60" incLanes="243188561#0_0" intLanes="" shape="718.23,1077.60 715.11,1076.90 718.23,1077.60" />\n  <junction id="2506086181" type="priority" x="449.53" y="1070.28" incLanes="911123067_0 706104219#15_0 -706104219#16_0" intLanes=":2506086181_0_0 :2506086181_1_0 :2506086181_2_0 :2506086181_3_0" shape="450.04,1062.74 453.12,1063.63 452.88,1066.06 453.17,1067.05 453.74,1067.88 454.58,1068.55 455.70,1069.08 453.55,1075.11 442.82,1071.02 445.22,1065.09 447.41,1065.49 448.28,1065.25 449.01,1064.71 449.60,1063.88">\n    <request index="0" response="0100" foes="0100" cont="0" />\n    <request index="1" response="1100" foes="1100" cont="0" />\n    <request index="2" response="0000" foes="0011" cont="0" />\n    <request index="3" response="0000" foes="0010" cont="0" />\n  </junction>\n  <junction id="2506086743" type="priority" x="442.59" y="1136.74" incLanes="194743656#0_0 243188536#2_0" intLanes=":2506086743_0_0 :2506086743_1_0 :2506086743_2_0" shape="447.48,1136.04 446.88,1139.19 446.03,1139.21 445.70,1139.38 445.43,1139.65 445.22,1140.02 445.08,1140.50 438.81,1139.20 438.82,1138.36 438.67,1138.04 438.43,1137.78 438.08,1137.59 437.64,1137.47 438.21,1134.32">\n    <request index="0" response="000" foes="100" cont="0" />\n    <request index="1" response="000" foes="100" cont="0" />\n    <request index="2" response="011" foes="011" cont="0" />\n  </junction>\n  <junction id="2506086795" type="priority" x="825.73" y="1140.88" incLanes="706104219#7_0 -243188563_0 -706104219#8_0" intLanes=":2506086795_0_0 :2506086795_6_0 :2506086795_2_0 :2506086795_3_0 :2506086795_4_0 :2506086795_5_0" shape="833.60,1138.41 833.01,1144.78 830.81,1145.01 830.05,1145.50 829.51,1146.22 829.21,1147.19 829.13,1148.40 822.73,1148.57 822.22,1145.90 821.64,1144.94 820.85,1144.22 819.84,1143.74 818.62,1143.51 819.14,1137.13">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="2506086831" type="priority" x="878.02" y="1145.79" incLanes="-498671388#0_0 706104219#5_0 243188432#0_0 -706104219#7_0" intLanes=":2506086831_0_0 :2506086831_1_0 :2506086831_2_0 :2506086831_3_0 :2506086831_4_0 :2506086831_5_0 :2506086831_6_0 :2506086831_7_0 :2506086831_9_0" shape="875.44,1138.32 881.81,1138.85 882.07,1141.13 882.56,1141.96 883.28,1142.59 884.25,1143.02 885.46,1143.24 884.89,1149.61 882.11,1149.81 881.16,1150.27 880.50,1150.99 880.14,1151.95 880.08,1153.16 876.88,1153.35 876.05,1150.80 875.15,1149.85 873.93,1149.12 872.39,1148.60 870.52,1148.30 871.12,1141.93 873.37,1141.69 874.19,1141.21 874.81,1140.49 875.22,1139.52">\n    <request index="0" response="000001000" foes="001001000" cont="0" />\n    <request index="1" response="110111000" foes="110111000" cont="0" />\n    <request index="2" response="000000000" foes="100100000" cont="0" />\n    <request index="3" response="000000000" foes="101100011" cont="0" />\n    <request index="4" response="010000000" foes="010000010" cont="0" />\n    <request index="5" response="110001100" foes="110001110" cont="0" />\n    <request index="6" response="110001001" foes="110001001" cont="0" />\n    <request index="7" response="000000000" foes="001110010" cont="0" />\n    <request index="8" response="000001100" foes="001101110" cont="1" />\n  </junction>\n  <junction id="2506087028" type="dead_end" x="253.86" y="1161.70" incLanes="243188120_0" intLanes="" shape="254.06,1160.11 253.66,1163.29" />\n  <junction id="2506087136" type="priority" x="1285.38" y="1183.95" incLanes="498671569_0 72016745#4_0 72016745#4_1 -243188585_0 -72016745#5_0 -72016745#5_1" intLanes=":2506087136_0_0 :2506087136_1_0 :2506087136_2_0 :2506087136_3_0 :2506087136_4_0 :2506087136_4_1 :2506087136_14_0 :2506087136_7_0 :2506087136_8_0 :2506087136_9_0 :2506087136_10_0 :2506087136_11_0 :2506087136_11_1 :2506087136_15_0" shape="1282.42,1173.27 1288.82,1173.41 1289.28,1175.88 1289.90,1176.79 1290.79,1177.47 1291.94,1177.92 1293.35,1178.15 1292.35,1190.92 1290.12,1191.19 1289.33,1191.68 1288.75,1192.41 1288.38,1193.39 1288.23,1194.60 1281.84,1194.39 1281.39,1191.95 1280.76,1191.06 1279.86,1190.38 1278.69,1189.92 1277.25,1189.67 1278.34,1176.92 1280.57,1176.67 1281.36,1176.18 1281.93,1175.45 1282.28,1174.48">\n    <request index="0" response="00000000110000" foes="00000000110000" cont="0" />\n    <request index="1" response="11110001110000" foes="11111001110000" cont="0" />\n    <request index="2" response="11100101110000" foes="11100101110000" cont="0" />\n    <request index="3" response="00000000000000" foes="10000100000000" cont="0" />\n    <request index="4" response="00000000000000" foes="10001100000111" cont="0" />\n    <request index="5" response="00000000000000" foes="10001100000111" cont="0" />\n    <request index="6" response="01110000000000" foes="01111100000110" cont="1" />\n    <request index="7" response="01100000000000" foes="01100000000000" cont="0" />\n    <request index="8" response="11100001111000" foes="11100001111100" cont="0" />\n    <request index="9" response="11100001110010" foes="11100001110010" cont="0" />\n    <request index="10" response="00000000000000" foes="00000001000010" cont="0" />\n    <request index="11" response="00000000000000" foes="00001111000110" cont="0" />\n    <request index="12" response="00000000000000" foes="00001111000110" cont="0" />\n    <request index="13" response="00000000111000" foes="00001100111110" cont="1" />\n  </junction>\n  <junction id="2506087252" type="dead_end" x="392.95" y="1190.77" incLanes="243188103_0" intLanes="" shape="392.95,1190.77 393.52,1187.62 392.95,1190.77" />\n  <junction id="2506087304" type="priority" x="430.06" y="1197.43" incLanes="-243188536#2_0 243188536#1_0 -243188103_0" intLanes=":2506087304_0_0 :2506087304_1_0 :2506087304_2_0 :2506087304_6_0 :2506087304_4_0 :2506087304_5_0" shape="428.37,1189.81 434.63,1191.11 431.90,1205.11 425.60,1203.95 425.57,1201.68 425.16,1200.82 424.50,1200.13 423.58,1199.63 422.39,1199.31 423.52,1193.01 425.85,1192.97 426.75,1192.57 427.47,1191.91 428.01,1190.99">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="2506087432" type="priority" x="641.81" y="1228.22" incLanes="1365892748#0_0 -498671536_0" intLanes=":2506087432_0_0 :2506087432_1_0 :2506087432_2_0 :2506087432_3_0" shape="638.76,1220.95 645.16,1221.08 644.98,1224.13 644.64,1226.38 644.24,1228.22 643.85,1230.06 643.56,1232.30 643.47,1235.36 640.27,1235.38 639.62,1233.15 638.82,1232.35 637.70,1231.77 636.27,1231.40 634.53,1231.25 634.68,1224.85 636.91,1224.46 637.70,1223.92 638.27,1223.16 638.62,1222.17">\n    <request index="0" response="0000" foes="0000" cont="0" />\n    <request index="1" response="1100" foes="1100" cont="0" />\n    <request index="2" response="0000" foes="0010" cont="0" />\n    <request index="3" response="0000" foes="0010" cont="0" />\n  </junction>\n  <junction id="2506087509" type="dead_end" x="277.85" y="1230.47" incLanes="" intLanes="" shape="278.23,1225.69 277.47,1235.25" />\n  <junction id="2506087522" type="priority" x="360.41" y="1242.41" incLanes="243188646#0_0 498671673_0" intLanes=":2506087522_0_0 :2506087522_1_0 :2506087522_3_0" shape="366.68,1241.58 366.28,1244.76 365.52,1244.81 365.31,1244.97 365.22,1245.21 365.24,1245.54 365.37,1245.96 359.47,1248.44 358.16,1245.67 357.55,1244.68 356.97,1243.94 356.43,1243.47 355.92,1243.25 356.47,1240.10">\n    <request index="0" response="000" foes="100" cont="0" />\n    <request index="1" response="000" foes="100" cont="0" />\n    <request index="2" response="011" foes="011" cont="1" />\n  </junction>\n  <junction id="2506087563" type="priority" x="420.38" y="1249.94" incLanes="-243188536#0_0 -1316153439_0" intLanes=":2506087563_0_0 :2506087563_1_0 :2506087563_2_0 :2506087563_3_0" shape="418.58,1242.02 424.88,1243.18 424.92,1245.41 425.33,1246.23 426.00,1246.86 426.94,1247.29 428.13,1247.53 427.47,1253.89 424.38,1253.45 422.12,1252.92 420.29,1252.33 418.46,1251.72 416.21,1251.15 413.12,1250.64 413.52,1247.47 415.85,1247.08 416.77,1246.35 417.54,1245.27 418.14,1243.82">\n    <request index="0" response="0000" foes="1000" cont="0" />\n    <request index="1" response="0000" foes="1000" cont="0" />\n    <request index="2" response="0000" foes="0000" cont="0" />\n    <request index="3" response="0011" foes="0011" cont="0" />\n  </junction>\n  <junction id="2506087592" type="dead_end" x="923.51" y="1250.76" incLanes="-243188208_0" intLanes="" shape="923.51,1250.76 926.66,1250.22 923.51,1250.76" />\n  <junction id="2506087709" type="priority" x="295.23" y="1285.13" incLanes="-243188174#0_0 77967090#3_0 77967090#3_1 77967090#3_2" intLanes=":2506087709_0_0" shape="306.23,1283.05 305.58,1289.41 303.51,1289.61 302.91,1290.05 302.61,1290.73 302.59,1291.65 302.87,1292.79 293.77,1295.85 292.76,1291.44 292.98,1287.94 294.43,1285.35 297.13,1283.67 301.06,1282.91">\n    <request index="0" response="0" foes="0" cont="0" />\n  </junction>\n  <junction id="2506087732" type="priority" x="428.81" y="1298.90" incLanes="706095306_0 243188174#2_0" intLanes=":2506087732_0_0 :2506087732_1_0" shape="431.16,1303.09 424.91,1301.71 425.57,1295.35 427.71,1295.77 429.36,1296.54 430.54,1297.65 431.23,1299.12 431.44,1300.93">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="2506087810" type="priority" x="488.55" y="1297.57" incLanes="706093506#2_0" intLanes=":2506087810_0_0 :2506087810_1_0" shape="487.63,1291.85 490.80,1292.23 490.98,1294.48 491.43,1295.32 492.14,1295.96 493.08,1296.42 494.28,1296.68 493.88,1299.86 491.63,1300.02 490.79,1300.47 490.15,1301.17 489.69,1302.11 489.43,1303.30 486.25,1302.89">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="2506087863" type="priority" x="565.65" y="1307.30" incLanes="-706103887_0 95809268#1_0 243188527_0" intLanes=":2506087863_0_0 :2506087863_1_0 :2506087863_2_0 :2506087863_3_0 :2506087863_4_0" shape="572.16,1304.63 571.62,1311.01 569.46,1311.27 568.74,1311.75 568.28,1312.48 568.05,1313.45 568.08,1314.65 564.90,1315.01 564.01,1311.46 563.24,1310.16 562.26,1309.17 561.07,1308.51 559.66,1308.16 560.06,1304.98 562.26,1304.68 563.03,1304.05 563.57,1303.10 563.88,1301.84 563.98,1300.25 567.18,1300.22 567.75,1302.49 568.44,1303.32 569.41,1303.96 570.65,1304.40">\n    <request index="0" response="00100" foes="10100" cont="0" />\n    <request index="1" response="00000" foes="01000" cont="0" />\n    <request index="2" response="00000" foes="11001" cont="0" />\n    <request index="3" response="00110" foes="00110" cont="0" />\n    <request index="4" response="00101" foes="00101" cont="0" />\n  </junction>\n  <junction id="2506087889" type="priority" x="642.49" y="1313.72" incLanes="706103887_0 95809261#0_0" intLanes=":2506087889_0_0 :2506087889_1_0 :2506087889_2_0" shape="643.94,1320.86 640.74,1320.79 640.29,1318.53 639.68,1317.70 638.82,1317.07 637.71,1316.64 636.34,1316.42 636.88,1310.04 639.09,1309.78 639.86,1309.29 640.41,1308.56 640.73,1307.59 640.83,1306.37 644.03,1306.35">\n    <request index="0" response="000" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="001" foes="001" cont="0" />\n  </junction>\n  <junction id="2506087955" type="priority" x="569.04" y="1337.73" incLanes="-243188505_0 95809268#0_0" intLanes=":2506087955_0_0 :2506087955_1_0 :2506087955_2_0" shape="575.29,1334.89 574.92,1341.28 572.74,1341.59 572.01,1342.10 571.52,1342.84 571.27,1343.82 571.26,1345.03 568.07,1345.30 566.68,1330.98 569.86,1330.62 570.66,1332.86 571.44,1333.68 572.47,1334.29 573.76,1334.69">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="2506087957" type="dead_end" x="1487.87" y="1339.03" incLanes="243188283_0" intLanes="" shape="1487.87,1339.03 1488.11,1342.22 1487.87,1339.03" />\n  <junction id="2506087962" type="priority" x="939.35" y="1343.10" incLanes="706104217#3_0 -706104217#7_0 243188208_0" intLanes=":2506087962_0_0 :2506087962_1_0 :2506087962_2_0 :2506087962_6_0 :2506087962_4_0 :2506087962_5_0" shape="945.77,1338.04 947.45,1344.21 933.41,1347.84 931.80,1341.64 933.88,1340.64 934.53,1339.83 934.92,1338.83 935.05,1337.63 934.92,1336.23 941.23,1335.14 942.04,1337.22 942.70,1337.84 943.55,1338.18 944.57,1338.25">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="2506087968" type="priority" x="641.88" y="1342.00" incLanes="95809261#1_0 243188505_0" intLanes=":2506087968_0_0 :2506087968_1_0 :2506087968_2_0" shape="640.44,1334.71 643.64,1334.77 643.56,1337.93 643.48,1340.26 643.35,1342.18 643.14,1344.09 642.82,1346.40 642.34,1349.52 639.18,1349.04 639.07,1346.81 638.64,1346.01 637.95,1345.42 637.00,1345.03 635.79,1344.85 636.17,1338.46 638.50,1338.15 639.33,1337.64 639.92,1336.90 640.29,1335.92">\n    <request index="0" response="000" foes="000" cont="0" />\n    <request index="1" response="100" foes="100" cont="0" />\n    <request index="2" response="000" foes="010" cont="0" />\n  </junction>\n  <junction id="2506087987" type="priority" x="415.88" y="1357.48" incLanes="-706095306_0 706104217#19_0 -498671328#0_0 -706104217#20_0" intLanes=":2506087987_0_0 :2506087987_1_0 :2506087987_2_0 :2506087987_3_0 :2506087987_4_0 :2506087987_12_0 :2506087987_6_0 :2506087987_7_0 :2506087987_8_0 :2506087987_9_0 :2506087987_10_0 :2506087987_13_0" shape="414.36,1349.54 420.61,1350.91 420.53,1353.46 420.91,1354.47 421.57,1355.30 422.50,1355.97 423.72,1356.45 421.92,1362.59 419.66,1362.39 418.77,1362.71 418.02,1363.30 417.43,1364.17 416.99,1365.31 410.83,1363.57 410.99,1361.26 410.64,1360.33 410.00,1359.55 409.07,1358.93 407.85,1358.45 409.66,1352.31 411.88,1352.50 412.75,1352.18 413.45,1351.58 413.99,1350.70">\n    <request index="0" response="000000010000" foes="000100010000" cont="0" />\n    <request index="1" response="111000110000" foes="111100110000" cont="0" />\n    <request index="2" response="110011110000" foes="110111110000" cont="0" />\n    <request index="3" response="000000000000" foes="100010000000" cont="0" />\n    <request index="4" response="000000000000" foes="100110000111" cont="0" />\n    <request index="5" response="011000000000" foes="111110000110" cont="1" />\n    <request index="6" response="010000000000" foes="010000000100" cont="0" />\n    <request index="7" response="110000111000" foes="110000111100" cont="0" />\n    <request index="8" response="110000110111" foes="110000110111" cont="0" />\n    <request index="9" response="000000000000" foes="000000100010" cont="0" />\n    <request index="10" response="000000000000" foes="000111100110" cont="0" />\n    <request index="11" response="000000111000" foes="000110111110" cont="1" />\n  </junction>\n  <junction id="2506088002" type="priority" x="833.66" y="1368.43" incLanes="706104217#8_0 -706104217#10_0 243188563_0" intLanes=":2506088002_0_0 :2506088002_1_0 :2506088002_2_0 :2506088002_3_0 :2506088002_4_0 :2506088002_5_0" shape="840.45,1363.48 841.97,1369.70 839.14,1370.40 837.05,1370.93 835.34,1371.42 833.65,1371.95 831.61,1372.65 828.86,1373.60 826.52,1367.64 828.57,1366.31 829.25,1365.31 829.69,1364.07 829.90,1362.61 829.88,1360.92 836.26,1360.44 836.86,1362.55 837.46,1363.19 838.26,1363.56 839.26,1363.66">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="0" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="2506088011" type="dead_end" x="1492.53" y="1380.99" incLanes="243188139_0" intLanes="" shape="1492.53,1380.99 1492.75,1384.18 1492.53,1380.99" />\n  <junction id="2506088019" type="dead_end" x="1186.69" y="1386.25" incLanes="-243188101_0" intLanes="" shape="1186.69,1386.25 1189.68,1385.10 1186.69,1386.25" />\n  <junction id="2506088023" type="priority" x="1397.06" y="1387.46" incLanes="-243188139_0 -122540020#4_0 122540020#3_0" intLanes=":2506088023_0_0 :2506088023_1_0 :2506088023_2_0 :2506088023_3_0 :2506088023_4_0 :2506088023_6_0" shape="1404.01,1383.78 1404.44,1390.17 1402.26,1390.76 1401.52,1391.36 1401.02,1392.18 1400.75,1393.19 1400.72,1394.42 1394.33,1394.83 1393.30,1380.58 1399.68,1380.08 1400.30,1382.26 1400.92,1382.99 1401.75,1383.49 1402.77,1383.75">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="2506088024" type="priority" x="775.02" y="1388.94" incLanes="706104217#9_0 498671676#0_0 1257722737_0 -706104217#11_0 -243188491_0" intLanes=":2506088024_0_0 :2506088024_1_0 :2506088024_2_0 :2506088024_3_0 :2506088024_4_0 :2506088024_5_0 :2506088024_6_0 :2506088024_7_0 :2506088024_8_0 :2506088024_16_0 :2506088024_10_0 :2506088024_11_0 :2506088024_12_0 :2506088024_13_0 :2506088024_14_0 :2506088024_17_0" shape="783.24,1382.69 785.34,1388.74 783.65,1389.65 783.42,1390.13 783.61,1390.62 784.21,1391.13 785.22,1391.66 783.94,1394.59 780.89,1393.73 779.82,1393.85 779.07,1394.35 778.64,1395.23 778.51,1396.47 772.11,1396.72 771.62,1394.57 771.05,1393.89 770.26,1393.47 769.25,1393.31 768.02,1393.41 766.92,1387.11 769.61,1386.00 770.54,1385.05 771.20,1383.82 771.58,1382.33 771.69,1380.56 778.09,1380.46 778.69,1382.49 779.41,1383.02 780.40,1383.23 781.68,1383.12">\n    <request index="0" response="0000000100000000" foes="0001000100010000" cont="0" />\n    <request index="1" response="1110001100000000" foes="1111001100110000" cont="0" />\n    <request index="2" response="1100111100000000" foes="1101111101110000" cont="0" />\n    <request index="3" response="1000000010000000" foes="1000100010000000" cont="0" />\n    <request index="4" response="1000000110000111" foes="1001100110000111" cont="0" />\n    <request index="5" response="0110001110000110" foes="0111101110000110" cont="0" />\n    <request index="6" response="0100011110000100" foes="0100011110000100" cont="0" />\n    <request index="7" response="0000000000000000" foes="1000100001111000" cont="0" />\n    <request index="8" response="0000000000000000" foes="1001100001110111" cont="0" />\n    <request index="9" response="0110000000000000" foes="0111100001100110" cont="1" />\n    <request index="10" response="0100000000000000" foes="0100000001000100" cont="0" />\n    <request index="11" response="1100001110111000" foes="1100001110111100" cont="0" />\n    <request index="12" response="1100001100110111" foes="1100001100110111" cont="0" />\n    <request index="13" response="0000000000000000" foes="0000001000100010" cont="0" />\n    <request index="14" response="0000000000000000" foes="0001111001100110" cont="0" />\n    <request index="15" response="0000000110000000" foes="0001100110011110" cont="1" />\n  </junction>\n  <junction id="2506088084" type="priority" x="725.72" y="1398.11" incLanes="96713729_0 706104217#12_0 -706104217#13_0" intLanes=":2506088084_0_0 :2506088084_1_0 :2506088084_2_0 :2506088084_3_0" shape="724.82,1391.02 728.00,1391.34 728.17,1393.01 728.54,1393.54 729.10,1393.85 729.84,1393.97 730.78,1393.87 732.00,1400.15 722.18,1401.83 721.30,1395.50 723.07,1394.80 723.74,1394.16 724.25,1393.31 724.61,1392.27">\n    <request index="0" response="0000" foes="0100" cont="0" />\n    <request index="1" response="0000" foes="1100" cont="0" />\n    <request index="2" response="0011" foes="0011" cont="0" />\n    <request index="3" response="0010" foes="0010" cont="0" />\n  </junction>\n  <junction id="2506088243" type="priority" x="775.81" y="1430.60" incLanes="-330615760_0" intLanes=":2506088243_0_0 :2506088243_1_0" shape="772.66,1426.18 779.06,1426.25 779.34,1427.76 779.72,1428.29 780.25,1428.67 780.93,1428.89 781.76,1428.96 781.78,1432.16 779.48,1432.18 777.76,1432.25 776.35,1432.41 774.98,1432.73 773.35,1433.26 771.19,1434.06 770.06,1431.06 771.48,1430.09 771.99,1429.36 772.35,1428.46 772.57,1427.40">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="2506088265" type="dead_end" x="1521.75" y="1457.40" incLanes="243188167_0" intLanes="" shape="1521.75,1457.40 1521.98,1460.59 1521.75,1457.40" />\n  <junction id="2506088274" type="priority" x="1402.14" y="1465.86" incLanes="-243188167_0 -122540020#5_0 122540020#4_0" intLanes=":2506088274_0_0 :2506088274_1_0 :2506088274_2_0 :2506088274_3_0 :2506088274_4_0 :2506088274_6_0" shape="1409.12,1462.16 1409.57,1468.54 1407.31,1469.15 1406.53,1469.76 1405.99,1470.58 1405.69,1471.60 1405.63,1472.83 1399.23,1473.09 1398.48,1458.86 1404.87,1458.45 1405.45,1460.64 1406.06,1461.37 1406.87,1461.87 1407.89,1462.13">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="2506088476" type="priority" x="466.68" y="1467.12" incLanes="706095305_0 498671331_0 1558734968_0" intLanes=":2506088476_0_0 :2506088476_1_0 :2506088476_2_0 :2506088476_3_0 :2506088476_4_0 :2506088476_6_0" shape="464.62,1458.12 470.96,1458.94 470.91,1462.65 471.25,1464.12 471.83,1465.34 472.66,1466.31 473.74,1467.02 470.91,1472.77 457.54,1465.82 460.58,1460.19 462.60,1460.80 463.35,1460.62 463.94,1460.12 464.36,1459.28">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="2506088491" type="priority" x="534.17" y="1491.70" incLanes="498671589_0 243188389#1_0 -243188389#2_0" intLanes=":2506088491_0_0 :2506088491_1_0 :2506088491_2_0 :2506088491_3_0 :2506088491_4_0 :2506088491_6_0" shape="531.59,1484.21 537.97,1484.74 538.22,1486.99 538.71,1487.80 539.44,1488.41 540.41,1488.81 541.62,1489.00 541.19,1495.39 538.30,1495.19 536.16,1495.07 534.39,1495.02 532.63,1495.04 530.49,1495.13 527.60,1495.30 527.22,1488.91 529.48,1488.28 530.31,1487.61 530.94,1486.71 531.36,1485.58">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="2506088492" type="priority" x="1242.82" y="1502.45" incLanes="243188552#0_0 -243188552#1_0 243188508#3_0" intLanes=":2506088492_0_0 :2506088492_1_0 :2506088492_2_0 :2506088492_6_0 :2506088492_4_0 :2506088492_5_0" shape="1240.00,1495.57 1246.39,1495.92 1245.94,1510.09 1239.54,1510.02 1239.12,1507.85 1238.58,1507.13 1237.81,1506.66 1236.82,1506.43 1235.60,1506.45 1234.91,1500.09 1237.64,1499.35 1238.62,1498.69 1239.33,1497.85 1239.79,1496.80">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="2506088494" type="priority" x="503.67" y="1493.51" incLanes="243188389#2_0 -243188389#3_0 -498671331_0" intLanes=":2506088494_0_0 :2506088494_1_0 :2506088494_2_0 :2506088494_6_0 :2506088494_4_0 :2506088494_5_0" shape="509.32,1489.97 509.70,1496.36 507.31,1496.54 505.53,1496.81 504.10,1497.23 502.74,1497.87 501.17,1498.78 499.11,1500.05 495.77,1494.59 497.42,1493.06 497.78,1492.19 497.83,1491.25 497.57,1490.24 496.99,1489.16 502.35,1485.65 504.45,1488.15 505.58,1489.01 506.77,1489.59 508.02,1489.91">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="2506088507" type="dead_end" x="1492.85" y="1515.11" incLanes="243188092_0" intLanes="" shape="1492.85,1515.11 1493.00,1518.31 1492.85,1515.11" />\n  <junction id="2506088518" type="priority" x="1155.61" y="1510.74" incLanes="-243188565_0 -243188508#1_0 -1552748146_0" intLanes=":2506088518_0_0 :2506088518_1_0 :2506088518_2_0 :2506088518_3_0 :2506088518_4_0 :2506088518_6_0" shape="1152.74,1503.01 1159.13,1503.28 1159.48,1505.46 1160.00,1506.20 1160.75,1506.70 1161.73,1506.95 1162.95,1506.97 1163.44,1513.35 1148.54,1513.93 1148.54,1507.53 1150.79,1507.03 1151.59,1506.40 1152.18,1505.52 1152.57,1504.39">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="2506088528" type="priority" x="1404.34" y="1519.37" incLanes="-243188092_0 -122540020#6_0 122540020#5_0" intLanes=":2506088528_0_0 :2506088528_1_0 :2506088528_2_0 :2506088528_3_0 :2506088528_4_0 :2506088528_6_0" shape="1411.71,1515.81 1412.02,1522.20 1409.86,1522.75 1409.17,1523.34 1408.73,1524.13 1408.54,1525.14 1408.61,1526.35 1402.28,1527.29 1401.81,1524.11 1401.48,1521.76 1401.26,1519.82 1401.11,1517.87 1400.98,1515.49 1400.85,1512.29 1407.24,1512.02 1407.81,1514.22 1408.44,1514.97 1409.30,1515.48 1410.39,1515.76">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="2506088529" type="priority" x="1115.62" y="1518.84" incLanes="243188641#0_0 243188452#1_0" intLanes=":2506088529_0_0 :2506088529_2_0" shape="1118.17,1524.03 1115.02,1524.59 1114.20,1522.50 1113.52,1521.83 1112.67,1521.42 1111.63,1521.27 1110.42,1521.36 1109.87,1518.21 1111.97,1517.40 1112.65,1516.73 1113.06,1515.87 1113.23,1514.84 1113.14,1513.62 1116.30,1513.10">\n    <request index="0" response="00" foes="10" cont="0" />\n    <request index="1" response="01" foes="01" cont="1" />\n  </junction>\n  <junction id="2506088612" type="priority" x="1065.44" y="1527.53" incLanes="243188594#0_0 243188452#0_0" intLanes=":2506088612_0_0 :2506088612_1_0 :2506088612_2_0 :2506088612_4_0" shape="1070.72,1524.99 1071.27,1528.14 1069.17,1528.96 1068.51,1529.64 1068.11,1530.50 1067.98,1531.53 1068.10,1532.75 1064.96,1533.37 1064.09,1531.26 1063.40,1530.61 1062.53,1530.21 1061.48,1530.08 1060.26,1530.22 1059.62,1527.08 1061.76,1526.19 1062.44,1525.47 1062.87,1524.58 1063.03,1523.51 1062.94,1522.26 1066.09,1521.73 1066.90,1523.85 1067.58,1524.52 1068.44,1524.93 1069.49,1525.09">\n    <request index="0" response="0000" foes="0100" cont="0" />\n    <request index="1" response="0000" foes="1100" cont="0" />\n    <request index="2" response="0011" foes="0011" cont="0" />\n    <request index="3" response="0010" foes="0010" cont="1" />\n  </junction>\n  <junction id="2506088695" type="dead_end" x="1537.66" y="1536.17" incLanes="243188056_0" intLanes="" shape="1537.66,1536.17 1537.83,1539.37 1537.66,1536.17" />\n  <junction id="2506088715" type="priority" x="1014.45" y="1537.91" incLanes="243188670#0_0 243188670#0_1" intLanes=":2506088715_0_0 :2506088715_0_1 :2506088715_2_0" shape="1021.26,1534.89 1021.90,1538.03 1019.74,1538.92 1019.06,1539.62 1018.65,1540.50 1018.49,1541.55 1018.60,1542.78 1012.31,1543.94 1010.39,1532.74 1016.71,1531.74 1017.49,1533.84 1018.16,1534.50 1019.01,1534.90 1020.04,1535.03">\n    <request index="0" response="000" foes="000" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="000" cont="0" />\n  </junction>\n  <junction id="2506088792" type="dead_end" x="13.76" y="1577.09" incLanes="498671329_0" intLanes="" shape="13.77,1575.49 13.75,1578.69" />\n  <junction id="2506088938" type="priority" x="583.18" y="1593.14" incLanes="-243188786#2_0 243188085_0 243188786#1_0" intLanes=":2506088938_0_0 :2506088938_6_0 :2506088938_2_0 :2506088938_3_0 :2506088938_4_0 :2506088938_5_0" shape="587.90,1599.23 581.68,1600.70 580.68,1598.62 579.90,1597.96 578.93,1597.57 577.77,1597.43 576.42,1597.55 575.34,1591.24 577.43,1590.43 578.07,1589.76 578.44,1588.91 578.54,1587.88 578.36,1586.67 584.59,1585.20">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="2506089058" type="priority" x="588.46" y="1615.51" incLanes="-243188786#3_0 243188361_0 243188786#2_0" intLanes=":2506089058_0_0 :2506089058_6_0 :2506089058_2_0 :2506089058_3_0 :2506089058_4_0 :2506089058_5_0" shape="593.22,1621.80 586.99,1623.26 586.05,1621.19 585.33,1620.56 584.45,1620.20 583.40,1620.11 582.18,1620.28 580.71,1614.05 582.78,1613.11 583.41,1612.39 583.77,1611.50 583.86,1610.45 583.69,1609.24 589.92,1607.77">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="2506089309" type="priority" x="1034.52" y="1642.36" incLanes="-980451721#1_0 980451721#0_0 243188670#2_0 243188670#2_1" intLanes=":2506089309_0_0 :2506089309_1_0 :2506089309_2_0 :2506089309_3_0" shape="1038.63,1636.19 1041.91,1641.69 1040.06,1642.77 1038.69,1643.57 1037.58,1644.26 1036.52,1645.03 1035.32,1646.05 1033.75,1647.49 1029.35,1642.84 1030.15,1641.45 1030.32,1640.48 1030.35,1639.34 1030.22,1638.02 1029.94,1636.52 1036.19,1635.13 1036.68,1636.23 1037.05,1636.50 1037.49,1636.59 1038.02,1636.48">\n    <request index="0" response="1100" foes="1100" cont="0" />\n    <request index="1" response="1000" foes="1000" cont="0" />\n    <request index="2" response="0000" foes="0001" cont="0" />\n    <request index="3" response="0000" foes="0011" cont="0" />\n  </junction>\n  <junction id="2506089487" type="priority" x="597.19" y="1652.82" incLanes="-243188786#4_0 243188161_0 243188786#3_0" intLanes=":2506089487_0_0 :2506089487_6_0 :2506089487_2_0 :2506089487_3_0 :2506089487_4_0 :2506089487_5_0" shape="602.04,1659.38 595.81,1660.85 594.87,1658.83 594.17,1658.25 593.30,1657.96 592.28,1657.95 591.09,1658.23 589.06,1652.16 591.38,1650.92 592.10,1650.10 592.52,1649.14 592.66,1648.04 592.50,1646.81 598.73,1645.36">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="2506089545" type="priority" x="1171.53" y="1664.80" incLanes="-497290263#2_0 -243188644_0 497290263#1_0" intLanes=":2506089545_0_0 :2506089545_6_0 :2506089545_2_0 :2506089545_3_0 :2506089545_4_0 :2506089545_5_0" shape="1177.14,1660.41 1178.40,1666.68 1176.16,1667.78 1175.32,1668.76 1174.66,1670.03 1174.18,1671.59 1173.89,1673.43 1167.53,1672.80 1167.31,1670.68 1166.84,1670.02 1166.13,1669.62 1165.19,1669.49 1164.00,1669.63 1162.70,1663.37">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="2506089600" type="priority" x="1117.29" y="1676.07" incLanes="-497290263#1_0 243188627_0 497290263#0_0" intLanes=":2506089600_0_0 :2506089600_6_0 :2506089600_2_0 :2506089600_3_0 :2506089600_4_0 :2506089600_5_0" shape="1122.94,1671.63 1124.24,1677.89 1122.02,1679.00 1121.21,1679.97 1120.59,1681.22 1120.16,1682.75 1119.94,1684.57 1113.55,1684.14 1113.26,1682.03 1112.77,1681.38 1112.05,1681.00 1111.10,1680.89 1109.92,1681.05 1108.48,1674.81">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="2506089779" type="priority" x="1017.51" y="1699.50" incLanes="497290262_0 706102885_0 497290269_0" intLanes=":2506089779_0_0 :2506089779_5_0 :2506089779_2_0 :2506089779_3_0 :2506089779_4_0" shape="1026.55,1704.07 1024.67,1706.66 1023.07,1707.97 1020.15,1709.28 1019.21,1707.41 1018.31,1706.09 1017.44,1705.32 1016.62,1705.10 1015.83,1705.42 1015.08,1706.29 1012.38,1704.57 1013.23,1702.42 1013.22,1701.41 1012.92,1700.45 1012.32,1699.54 1011.43,1698.67 1015.48,1693.71">\n    <request index="0" response="00000" foes="01000" cont="0" />\n    <request index="1" response="10000" foes="10100" cont="1" />\n    <request index="2" response="10010" foes="10010" cont="0" />\n    <request index="3" response="10001" foes="10001" cont="0" />\n    <request index="4" response="00000" foes="01110" cont="0" />\n  </junction>\n  <junction id="2506089901" type="priority" x="1410.71" y="1713.26" incLanes="243188503#0_0 243188196_0 -243188503#1_0" intLanes=":2506089901_0_0 :2506089901_6_0 :2506089901_2_0 :2506089901_3_0 :2506089901_4_0 :2506089901_5_0" shape="1410.60,1702.83 1416.71,1704.73 1416.48,1706.73 1416.78,1707.32 1417.36,1707.64 1418.23,1707.68 1419.37,1707.45 1421.13,1713.60 1418.70,1714.40 1416.64,1715.28 1414.95,1716.25 1413.64,1717.31 1412.70,1718.45 1412.13,1719.69 1406.00,1717.85">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="2506090118" type="priority" x="531.57" y="1729.76" incLanes="-243188243_0 243188389#3_0" intLanes=":2506090118_0_0 :2506090118_1_0" shape="533.47,1725.71 535.58,1731.75 532.05,1732.24 530.60,1731.83 529.35,1730.98 528.31,1729.69 527.48,1727.95">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="2506090385" type="dead_end" x="1253.06" y="1765.60" incLanes="243188445_0" intLanes="" shape="1253.06,1765.60 1249.98,1764.74 1253.06,1765.60" />\n  <junction id="2506090451" type="dead_end" x="1200.33" y="1766.01" incLanes="243188115_0" intLanes="" shape="1200.33,1766.01 1197.18,1765.43 1200.33,1766.01" />\n  <junction id="2506090779" type="dead_end" x="1094.06" y="1836.35" incLanes="-243188627_0" intLanes="" shape="1094.06,1836.35 1091.00,1835.41 1094.06,1836.35" />\n  <junction id="2506090932" type="dead_end" x="1132.87" y="1853.87" incLanes="243188644_0" intLanes="" shape="1132.87,1853.87 1129.78,1853.03 1132.87,1853.87" />\n  <junction id="2507400591" type="priority" x="1493.33" y="719.96" incLanes="-243314310#1_0 1547102487#0_0 932339001_0" intLanes=":2507400591_0_0 :2507400591_1_0 :2507400591_2_0 :2507400591_3_0 :2507400591_4_0 :2507400591_5_0 :2507400591_6_0" shape="1499.88,717.04 1499.60,723.44 1497.47,723.78 1496.80,724.29 1496.39,725.03 1496.25,725.99 1496.37,727.19 1493.23,727.80 1492.22,725.34 1491.38,724.49 1490.33,723.90 1489.07,723.56 1487.59,723.48 1487.17,717.09 1489.32,716.58 1490.00,716.00 1490.42,715.20 1490.57,714.18 1490.47,712.95 1493.62,712.40 1494.66,714.87 1495.57,715.76 1496.75,716.42 1498.18,716.85">\n    <request index="0" response="0001000" foes="1001000" cont="0" />\n    <request index="1" response="0011000" foes="1011000" cont="0" />\n    <request index="2" response="0000000" foes="0100000" cont="0" />\n    <request index="3" response="0000000" foes="1100011" cont="0" />\n    <request index="4" response="0000000" foes="1100010" cont="0" />\n    <request index="5" response="0011100" foes="0011100" cont="0" />\n    <request index="6" response="0011011" foes="0011011" cont="0" />\n  </junction>\n  <junction id="2507400593" type="priority" x="1498.79" y="747.93" incLanes="970010767#1_0" intLanes=":2507400593_0_0 :2507400593_1_0" shape="1503.12,746.71 1502.84,749.90 1493.53,749.10 1493.82,745.92 1495.41,745.71 1495.92,745.35 1496.22,744.81 1496.33,744.09 1496.24,743.20 1499.38,742.58 1500.13,744.76 1500.69,745.55 1501.38,746.13 1502.19,746.52">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="2507400604" type="priority" x="1655.82" y="848.70" incLanes="-243314436#3_0 -243314467#0_0 243314436#2_0" intLanes=":2507400604_0_0 :2507400604_6_0 :2507400604_2_0 :2507400604_3_0 :2507400604_4_0 :2507400604_5_0" shape="1653.55,841.37 1659.90,842.20 1660.08,844.50 1660.56,845.39 1661.31,846.11 1662.32,846.66 1663.60,847.04 1662.31,853.30 1660.05,853.29 1659.20,853.68 1658.51,854.34 1658.01,855.25 1657.69,856.43 1651.39,855.31">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="2507400609" type="priority" x="1741.14" y="866.28" incLanes="-243314467#1_0 -498671542_0 243314467#0_0" intLanes=":2507400609_0_0 :2507400609_6_0 :2507400609_2_0 :2507400609_3_0 :2507400609_4_0 :2507400609_5_0" shape="1748.81,864.17 1747.88,870.50 1745.63,870.62 1744.80,871.06 1744.16,871.75 1743.71,872.70 1743.45,873.89 1737.10,873.13 1736.95,870.65 1736.50,869.70 1735.81,868.94 1734.88,868.38 1733.70,868.01 1734.99,861.74">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="2507400614" type="dead_end" x="1555.02" y="875.99" incLanes="-243314165#0_0" intLanes="" shape="1555.02,875.99 1555.50,872.83 1555.02,875.99" />\n  <junction id="2507400624" type="dead_end" x="1647.33" y="906.25" incLanes="-243314436#0_0" intLanes="" shape="1647.33,906.25 1644.15,905.94 1647.33,906.25" />\n  <junction id="2507400625" type="dead_end" x="1746.73" y="907.33" incLanes="243314248#1_0" intLanes="" shape="1746.73,907.33 1744.99,910.02 1746.73,907.33" />\n  <junction id="2507400638" type="priority" x="1525.44" y="926.27" incLanes="-243314312_0 151072453#3_0 243314192_0" intLanes=":2507400638_0_0 :2507400638_1_0 :2507400638_2_0 :2507400638_3_0 :2507400638_4_0 :2507400638_5_0 :2507400638_6_0" shape="1533.80,923.93 1531.38,932.55 1529.42,932.05 1528.80,932.28 1528.41,932.81 1528.26,933.66 1528.35,934.82 1525.18,935.30 1524.20,932.15 1523.41,931.08 1522.42,930.35 1521.24,929.96 1519.87,929.90 1519.38,923.52 1521.53,922.91 1522.23,922.31 1522.67,921.49 1522.85,920.48 1522.79,919.26 1525.95,918.79 1526.57,921.66 1527.47,923.72 1528.64,924.98 1530.08,925.43 1531.80,925.08">\n    <request index="0" response="0001000" foes="1001000" cont="0" />\n    <request index="1" response="0011000" foes="1011000" cont="0" />\n    <request index="2" response="0000000" foes="0100000" cont="0" />\n    <request index="3" response="0000000" foes="1100011" cont="0" />\n    <request index="4" response="0000000" foes="1100010" cont="0" />\n    <request index="5" response="0011100" foes="0011100" cont="0" />\n    <request index="6" response="0011011" foes="0011011" cont="0" />\n  </junction>\n  <junction id="2507400639" type="priority" x="1511.13" y="927.36" incLanes="-243314192_0 1270957862_0 1270957862_1" intLanes=":2507400639_0_0 :2507400639_1_0 :2507400639_2_0 :2507400639_2_1" shape="1518.35,923.60 1518.84,929.98 1516.69,930.59 1516.01,931.20 1515.58,932.01 1515.41,933.02 1515.50,934.23 1509.18,935.27 1506.85,921.03 1513.17,920.00 1514.03,922.15 1514.79,922.87 1515.76,923.35 1516.95,923.60">\n    <request index="0" response="1100" foes="1100" cont="0" />\n    <request index="1" response="0000" foes="0000" cont="0" />\n    <request index="2" response="0000" foes="0001" cont="0" />\n    <request index="3" response="0000" foes="0001" cont="0" />\n  </junction>\n  <junction id="2507400689" type="priority" x="1583.64" y="1282.37" incLanes="243314112_0 66984613#5_0 -243314073_0" intLanes=":2507400689_0_0 :2507400689_1_0 :2507400689_2_0 :2507400689_3_0 :2507400689_4_0 :2507400689_5_0 :2507400689_6_0" shape="1589.87,1279.27 1589.77,1285.67 1587.62,1286.07 1586.92,1286.61 1586.49,1287.37 1586.31,1288.35 1586.39,1289.55 1583.23,1290.05 1582.30,1287.58 1581.48,1286.72 1580.43,1286.12 1579.14,1285.77 1577.62,1285.67 1577.52,1279.27 1579.67,1278.79 1580.37,1278.23 1580.80,1277.45 1580.98,1276.46 1580.90,1275.25 1584.06,1274.75 1585.02,1277.23 1585.86,1278.10 1586.95,1278.73 1588.29,1279.12">\n    <request index="0" response="0001000" foes="1001000" cont="0" />\n    <request index="1" response="0011000" foes="1011000" cont="0" />\n    <request index="2" response="0000000" foes="0100000" cont="0" />\n    <request index="3" response="0000000" foes="1100011" cont="0" />\n    <request index="4" response="0000000" foes="1100010" cont="0" />\n    <request index="5" response="0011100" foes="0011100" cont="0" />\n    <request index="6" response="0011011" foes="0011011" cont="0" />\n  </junction>\n  <junction id="2507400690" type="priority" x="1569.17" y="1282.60" incLanes="243314073_0 970323546#5_0 970323546#5_1" intLanes=":2507400690_0_0 :2507400690_1_0 :2507400690_2_0 :2507400690_2_1" shape="1576.82,1279.28 1576.93,1285.68 1574.77,1286.15 1574.09,1286.71 1573.66,1287.49 1573.48,1288.48 1573.57,1289.69 1567.25,1290.72 1564.92,1276.44 1571.24,1275.41 1572.14,1277.59 1572.96,1278.35 1574.01,1278.88 1575.30,1279.19">\n    <request index="0" response="1100" foes="1100" cont="0" />\n    <request index="1" response="0000" foes="0000" cont="0" />\n    <request index="2" response="0000" foes="0001" cont="0" />\n    <request index="3" response="0000" foes="0001" cont="0" />\n  </junction>\n  <junction id="2507400696" type="priority" x="1593.78" y="1346.73" incLanes="243314378#3_0 66984613#4_0 -243314213_0" intLanes=":2507400696_0_0 :2507400696_1_0 :2507400696_2_0 :2507400696_3_0 :2507400696_4_0 :2507400696_5_0 :2507400696_6_0" shape="1600.33,1343.95 1599.92,1350.33 1597.77,1350.63 1597.08,1351.13 1596.65,1351.86 1596.47,1352.83 1596.55,1354.03 1593.39,1354.52 1592.48,1352.04 1591.70,1351.20 1590.70,1350.62 1589.48,1350.31 1588.05,1350.27 1587.68,1343.88 1589.83,1343.31 1590.52,1342.72 1590.95,1341.92 1591.13,1340.91 1591.05,1339.70 1594.21,1339.20 1595.20,1341.68 1596.09,1342.58 1597.24,1343.26 1598.65,1343.72">\n    <request index="0" response="0001000" foes="1001000" cont="0" />\n    <request index="1" response="0011000" foes="1011000" cont="0" />\n    <request index="2" response="0000000" foes="0100000" cont="0" />\n    <request index="3" response="0000000" foes="1100011" cont="0" />\n    <request index="4" response="0000000" foes="1100010" cont="0" />\n    <request index="5" response="0011100" foes="0011100" cont="0" />\n    <request index="6" response="0011011" foes="0011011" cont="0" />\n  </junction>\n  <junction id="2507400698" type="priority" x="1579.78" y="1347.54" incLanes="243314213_0 970323546#4_0 970323546#4_1" intLanes=":2507400698_0_0 :2507400698_1_0 :2507400698_2_0 :2507400698_2_1" shape="1587.13,1343.91 1587.50,1350.30 1585.36,1350.87 1584.67,1351.46 1584.24,1352.26 1584.07,1353.26 1584.16,1354.48 1577.84,1355.51 1575.51,1341.26 1581.83,1340.23 1582.70,1342.39 1583.48,1343.12 1584.47,1343.62 1585.69,1343.88">\n    <request index="0" response="1100" foes="1100" cont="0" />\n    <request index="1" response="0000" foes="0000" cont="0" />\n    <request index="2" response="0000" foes="0001" cont="0" />\n    <request index="3" response="0000" foes="0001" cont="0" />\n  </junction>\n  <junction id="2507400702" type="dead_end" x="1890.27" y="1360.87" incLanes="-243314378#1_0" intLanes="" shape="1890.27,1360.87 1890.05,1364.06 1890.27,1360.87" />\n  <junction id="2507400742" type="dead_end" x="1707.08" y="1606.55" incLanes="-243314177_0" intLanes="" shape="1707.08,1606.55 1710.27,1606.83 1707.08,1606.55" />\n  <junction id="2507400748" type="priority" x="1612.23" y="1620.78" incLanes="243314103_0 65080358#1_0" intLanes=":2507400748_0_0 :2507400748_1_0 :2507400748_2_0" shape="1614.96,1627.63 1611.80,1628.14 1611.00,1626.01 1610.33,1625.33 1609.46,1624.91 1608.42,1624.74 1607.18,1624.82 1606.18,1618.50 1608.31,1617.71 1608.99,1617.04 1609.41,1616.19 1609.58,1615.15 1609.49,1613.92 1612.65,1613.41">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="2507400752" type="dead_end" x="1523.58" y="1634.82" incLanes="-243314103_0" intLanes="" shape="1523.58,1634.82 1523.08,1631.66 1523.58,1634.82" />\n  <junction id="2507400769" type="priority" x="815.71" y="1652.20" incLanes="95542482#10_0 -243314137_0 -95542482#11_0" intLanes=":2507400769_0_0 :2507400769_6_0 :2507400769_2_0 :2507400769_3_0 :2507400769_4_0 :2507400769_5_0" shape="822.68,1648.16 823.43,1654.52 821.30,1655.22 820.62,1655.85 820.21,1656.68 820.05,1657.71 820.15,1658.92 813.85,1660.04 813.00,1657.80 812.31,1657.08 811.44,1656.63 810.39,1656.45 809.17,1656.54 808.11,1650.23">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="2507400772" type="priority" x="699.07" y="1678.32" incLanes="95542482#12_0 243314160_0 -95542482#13_0" intLanes=":2507400772_0_0 :2507400772_6_0 :2507400772_2_0 :2507400772_3_0 :2507400772_4_0 :2507400772_5_0" shape="704.87,1673.28 706.67,1679.43 704.65,1680.53 704.03,1681.35 703.69,1682.35 703.60,1683.53 703.79,1684.89 697.54,1686.26 696.64,1684.22 695.94,1683.61 695.07,1683.29 694.04,1683.25 692.84,1683.48 691.04,1677.34">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="2507400773" type="priority" x="1699.32" y="1693.92" incLanes="243314177_0 81350006#1_0 -81350006#2_0" intLanes=":2507400773_0_0 :2507400773_1_0 :2507400773_2_0 :2507400773_3_0 :2507400773_4_0 :2507400773_6_0" shape="1696.85,1685.57 1703.22,1686.14 1703.47,1688.28 1703.95,1688.97 1704.67,1689.40 1705.62,1689.58 1706.82,1689.50 1707.82,1695.82 1693.69,1698.32 1692.42,1692.05 1694.66,1690.97 1695.49,1690.03 1696.13,1688.82 1696.59,1687.33">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="2507400792" type="dead_end" x="832.42" y="1757.05" incLanes="243314137_0" intLanes="" shape="832.42,1757.05 829.23,1757.30 832.42,1757.05" />\n  <junction id="2507400847" type="dead_end" x="1681.29" y="1879.16" incLanes="" intLanes="" shape="1679.71,1879.42 1682.87,1878.90" />\n  <junction id="2935421242" type="dead_end" x="1359.74" y="1915.00" incLanes="243188765#2_0" intLanes="" shape="1359.74,1915.00 1356.55,1915.24 1359.74,1915.00" />\n  <junction id="3125684629" type="dead_end" x="273.57" y="435.95" incLanes="" intLanes="" shape="273.79,434.37 273.35,437.53" />\n  <junction id="3125684630" type="priority" x="349.18" y="446.38" incLanes="40264696#2_0 40264696#2_1 40264696#2_2 307427662_0" intLanes=":3125684630_0_0 :3125684630_0_1 :3125684630_0_2 :3125684630_3_0" shape="345.17,440.17 354.68,441.44 353.11,452.66 343.61,451.26 343.49,449.01 343.06,448.16 342.37,447.50 341.43,447.04 340.23,446.76 340.67,443.59 342.93,443.45 343.78,443.01 344.43,442.32 344.89,441.37">\n    <request index="0" response="0000" foes="1000" cont="0" />\n    <request index="1" response="0000" foes="1000" cont="0" />\n    <request index="2" response="0000" foes="1000" cont="0" />\n    <request index="3" response="0111" foes="0111" cont="0" />\n  </junction>\n  <junction id="3309949761" type="dead_end" x="0.00" y="1666.14" incLanes="-324267161#1_0" intLanes="" shape="-0.00,1666.14 0.38,1662.96 0.00,1666.14" />\n  <junction id="3376091528" type="priority" x="1017.40" y="1370.57" incLanes="330615829#1_0 -330615844_0" intLanes=":3376091528_0_0 :3376091528_1_0" shape="1014.20,1370.37 1020.60,1370.52 1020.59,1370.87 1014.20,1370.53">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="4085816302" type="traffic_light" x="336.66" y="1023.03" incLanes="706104219#16_0 84842974#7_0 84842974#7_1 84842974#7_2 706096642_0" intLanes=":4085816302_0_0 :4085816302_1_0 :4085816302_2_0 :4085816302_2_1 :4085816302_2_2 :4085816302_5_0 :4085816302_6_0" shape="332.33,1016.21 341.90,1016.88 342.40,1019.25 343.17,1020.21 344.28,1021.03 345.74,1021.70 347.54,1022.22 346.12,1028.46 344.03,1028.41 343.35,1028.78 342.93,1029.43 342.76,1030.34 342.84,1031.53 333.35,1033.00 332.80,1030.23 332.09,1027.92 331.21,1026.07 330.17,1024.70 328.96,1023.79 327.60,1023.35 328.05,1020.18 330.28,1019.99 331.10,1019.47 331.71,1018.67 332.12,1017.58">\n    <request index="0" response="0011100" foes="0011100" cont="0" />\n    <request index="1" response="0000000" foes="0100000" cont="0" />\n    <request index="2" response="0000000" foes="1100001" cont="0" />\n    <request index="3" response="0000000" foes="1100001" cont="0" />\n    <request index="4" response="0000000" foes="1100001" cont="0" />\n    <request index="5" response="0011110" foes="0011110" cont="0" />\n    <request index="6" response="0011100" foes="0011100" cont="0" />\n  </junction>\n  <junction id="4488358856" type="priority" x="1134.58" y="1312.92" incLanes="452037066_0 -452036923#6_0 452036923#5_0" intLanes=":4488358856_0_0 :4488358856_1_0 :4488358856_2_0 :4488358856_3_0 :4488358856_4_0 :4488358856_6_0" shape="1131.43,1302.24 1137.83,1302.27 1138.17,1304.13 1138.61,1304.56 1139.22,1304.65 1140.01,1304.40 1140.97,1303.82 1144.96,1308.83 1142.17,1310.94 1140.18,1312.61 1138.56,1314.03 1136.90,1315.37 1134.77,1316.81 1131.74,1318.53 1127.94,1313.30 1129.86,1311.26 1130.54,1309.62 1131.02,1307.57 1131.32,1305.11">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="4488358857" type="priority" x="1088.44" y="1204.94" incLanes="452036923#3_0 -452037067_0 -452036923#4_0" intLanes=":4488358857_0_0 :4488358857_6_0 :4488358857_2_0 :4488358857_3_0 :4488358857_4_0 :4488358857_5_0" shape="1098.45,1208.70 1094.51,1213.74 1092.74,1212.82 1092.10,1212.89 1091.63,1213.33 1091.32,1214.12 1091.17,1215.28 1084.78,1214.99 1084.80,1212.35 1084.60,1210.11 1084.17,1208.25 1083.53,1206.78 1082.67,1205.70 1081.59,1205.01 1084.60,1199.33 1087.85,1200.79 1090.15,1202.05 1091.95,1203.29 1093.64,1204.70 1095.67,1206.44">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="4488358858" type="priority" x="1084.03" y="1302.21" incLanes="452037067_0 -452036923#5_0 452036923#4_0" intLanes=":4488358858_0_0 :4488358858_1_0 :4488358858_2_0 :4488358858_3_0 :4488358858_4_0 :4488358858_6_0" shape="1081.37,1290.22 1087.76,1290.51 1087.70,1293.75 1087.83,1296.56 1088.15,1298.94 1088.66,1300.90 1089.36,1302.44 1090.25,1303.55 1085.95,1308.28 1083.18,1305.73 1081.19,1303.79 1079.60,1302.13 1078.07,1300.42 1076.24,1298.32 1073.75,1295.50 1078.53,1291.24 1080.03,1292.45 1080.57,1292.48 1080.97,1292.11 1081.24,1291.36">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="4488366488" type="priority" x="1235.51" y="1209.07" incLanes="452036923#0_0 243188225_0 -452036923#1_0" intLanes=":4488366488_0_0 :4488366488_6_0 :4488366488_2_0 :4488366488_3_0 :4488366488_4_0 :4488366488_5_0" shape="1237.68,1203.08 1241.79,1207.98 1240.09,1210.34 1239.51,1212.00 1239.10,1213.98 1238.86,1216.28 1238.80,1218.91 1232.40,1218.97 1232.01,1217.23 1231.53,1216.79 1230.88,1216.62 1230.03,1216.74 1229.00,1217.13 1225.46,1211.74 1228.31,1210.31 1230.35,1209.12 1231.95,1208.00 1233.46,1206.74 1235.25,1205.16">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="4488368090" type="priority" x="1183.56" y="1303.87" incLanes="452036922_0 -452036923#7_0 452036923#6_0" intLanes=":4488368090_0_0 :4488368090_1_0 :4488368090_2_0 :4488368090_3_0 :4488368090_4_0 :4488368090_6_0" shape="1180.43,1296.55 1186.82,1296.61 1187.24,1299.16 1187.79,1300.11 1188.55,1300.84 1189.54,1301.35 1190.75,1301.65 1189.85,1307.99 1186.95,1307.58 1184.80,1307.31 1183.03,1307.13 1181.25,1307.01 1179.08,1306.91 1176.15,1306.80 1176.39,1300.40 1178.62,1300.04 1179.40,1299.51 1179.96,1298.76 1180.30,1297.77">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="485580842" type="priority" x="360.47" y="836.34" incLanes="706093508#2_0 84842974#10_0 84842974#10_1 84842974#10_2" intLanes=":485580842_0_0 :485580842_1_0 :485580842_2_0 :485580842_2_1 :485580842_2_2 :485580842_5_0" shape="356.52,829.97 366.02,831.30 366.16,833.55 366.60,834.38 367.30,835.01 368.24,835.45 369.44,835.69 369.10,838.87 366.75,839.11 365.87,839.63 365.17,840.41 364.66,841.46 364.34,842.78 354.85,841.33 354.75,839.08 354.31,838.25 353.62,837.61 352.68,837.17 351.48,836.92 351.85,833.74 354.18,833.53 355.05,833.03 355.73,832.27 356.22,831.25">\n    <request index="0" response="011100" foes="011100" cont="0" />\n    <request index="1" response="111100" foes="111100" cont="0" />\n    <request index="2" response="000000" foes="000011" cont="0" />\n    <request index="3" response="000000" foes="000011" cont="0" />\n    <request index="4" response="000000" foes="000011" cont="0" />\n    <request index="5" response="000000" foes="000010" cont="0" />\n  </junction>\n  <junction id="485580845" type="dead_end" x="269.45" y="1011.91" incLanes="" intLanes="" shape="269.87,1010.37 269.03,1013.45" />\n  <junction id="4888861605" type="priority" x="1050.15" y="1701.84" incLanes="-497290263#0_0 497290264_0" intLanes=":4888861605_0_0 :4888861605_1_0" shape="1048.15,1699.33 1052.47,1704.06 1052.23,1704.27 1047.94,1699.54">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="4888861608" type="priority" x="1028.70" y="1707.61" incLanes="497290261#1_0" intLanes=":4888861608_0_0 :4888861608_1_0" shape="1030.79,1705.55 1031.43,1708.72 1029.73,1708.90 1028.89,1709.28 1028.02,1709.86 1027.14,1710.62 1026.22,1711.57 1024.52,1708.67 1025.00,1707.85 1025.04,1707.46 1024.95,1707.10 1024.71,1706.75 1024.34,1706.43 1026.22,1703.84 1027.99,1704.95 1028.79,1705.30 1029.52,1705.52 1030.19,1705.60">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="4888861612" type="priority" x="1032.65" y="1723.18" incLanes="497290265_0 497290261#4_0" intLanes=":4888861612_0_0 :4888861612_1_0" shape="1033.89,1721.07 1034.91,1724.11 1032.89,1725.02 1032.27,1725.51 1031.70,1726.10 1031.04,1726.82 1030.12,1727.74 1027.93,1725.40 1028.39,1724.78 1028.40,1724.51 1028.28,1724.27 1028.02,1724.06 1027.62,1723.87 1027.90,1720.35 1029.64,1721.29 1030.61,1721.49 1031.64,1721.53 1032.74,1721.39">\n    <request index="0" response="10" foes="10" cont="0" />\n    <request index="1" response="00" foes="01" cont="0" />\n  </junction>\n  <junction id="4888861618" type="priority" x="1025.58" y="1720.56" incLanes="497290261#3_0" intLanes=":4888861618_0_0 :4888861618_1_0" shape="1030.17,1721.39 1028.10,1724.05 1027.04,1723.87 1026.61,1723.98 1026.25,1724.21 1025.96,1724.57 1025.74,1725.06 1022.71,1724.03 1023.21,1722.05 1023.27,1721.19 1023.22,1720.42 1023.05,1719.74 1022.75,1719.15 1025.55,1717.36 1026.37,1718.97 1027.05,1719.68 1027.91,1720.32 1028.95,1720.89">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="4888861620" type="priority" x="1039.68" y="1715.67" incLanes="497290261#5_0" intLanes=":4888861620_0_0 :4888861620_1_0" shape="1041.51,1710.59 1044.07,1712.52 1042.55,1714.33 1042.01,1714.97 1041.56,1715.66 1041.17,1716.55 1040.81,1717.83 1037.70,1717.08 1036.61,1710.81 1039.97,1710.43 1040.43,1711.05 1040.68,1711.15 1040.94,1711.10 1041.22,1710.92">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="4888862222" type="priority" x="1038.24" y="1710.51" incLanes="497290260_0 497290261#7_0" intLanes=":4888862222_0_0 :4888862222_1_0" shape="1041.11,1705.75 1043.26,1708.12 1041.91,1709.84 1041.48,1710.86 1041.21,1711.98 1041.11,1713.20 1041.17,1714.52 1038.10,1715.93 1032.73,1708.88 1035.15,1706.25 1037.38,1706.95 1038.40,1706.98 1039.36,1706.79 1040.26,1706.38">\n    <request index="0" response="10" foes="10" cont="0" />\n    <request index="1" response="00" foes="01" cont="0" />\n  </junction>\n  <junction id="4889402217" type="dead_end" x="1380.35" y="-0.00" incLanes="" intLanes="" shape="1381.94,-0.20 1378.76,0.20" />\n  <junction id="4900464580" type="priority" x="969.09" y="1061.43" incLanes="-96941857#2_0 498671389#0_0 96941857#1_0 -498671389#2_0" intLanes=":4900464580_0_0 :4900464580_1_0 :4900464580_12_0 :4900464580_3_0 :4900464580_4_0 :4900464580_5_0 :4900464580_6_0 :4900464580_7_0 :4900464580_13_0 :4900464580_9_0 :4900464580_10_0 :4900464580_11_0" shape="977.34,1059.11 976.65,1065.47 974.49,1065.67 973.77,1066.14 973.30,1066.85 973.07,1067.81 973.08,1069.02 966.71,1069.67 965.86,1066.63 965.06,1065.51 964.02,1064.66 962.73,1064.08 961.19,1063.75 961.91,1057.40 964.12,1057.20 964.89,1056.73 965.44,1056.02 965.77,1055.06 965.88,1053.85 972.28,1053.84 972.84,1056.52 973.55,1057.51 974.53,1058.28 975.80,1058.81">\n    <request index="0" response="000000000000" foes="000100010000" cont="0" />\n    <request index="1" response="000000000000" foes="111100110000" cont="0" />\n    <request index="2" response="000111000000" foes="110111110000" cont="1" />\n    <request index="3" response="000010000000" foes="100010000000" cont="0" />\n    <request index="4" response="000110000111" foes="100110000111" cont="0" />\n    <request index="5" response="011110000110" foes="111110000110" cont="0" />\n    <request index="6" response="000000000000" foes="010000000100" cont="0" />\n    <request index="7" response="000000000000" foes="110000111100" cont="0" />\n    <request index="8" response="000000000011" foes="110000110111" cont="1" />\n    <request index="9" response="000000000010" foes="000000100010" cont="0" />\n    <request index="10" response="000111000110" foes="000111100110" cont="0" />\n    <request index="11" response="000110111110" foes="000110111110" cont="0" />\n  </junction>\n  <junction id="4900464584" type="priority" x="1050.82" y="980.45" incLanes="185951074#4_0 96242763#3_0 -1204844051_0" intLanes=":4900464584_0_0 :4900464584_1_0 :4900464584_2_0" shape="1058.09,978.23 1057.20,984.57 1055.11,984.71 1054.44,985.15 1054.02,985.84 1053.86,986.77 1053.95,987.96 1050.81,988.64 1049.90,985.78 1049.17,984.72 1048.26,983.93 1047.17,983.39 1045.89,983.10 1046.63,976.74">\n    <request index="0" response="000" foes="000" cont="0" />\n    <request index="1" response="100" foes="100" cont="0" />\n    <request index="2" response="000" foes="010" cont="0" />\n  </junction>\n  <junction id="4900464587" type="priority" x="1241.28" y="1134.45" incLanes="-1476708031_0 -530314558#1_0 -498671391_0" intLanes=":4900464587_0_0 :4900464587_1_0 :4900464587_2_0 :4900464587_6_0 :4900464587_4_0 :4900464587_5_0" shape="1239.49,1127.07 1245.76,1128.32 1242.70,1142.61 1236.46,1141.19 1236.51,1138.98 1236.13,1138.17 1235.48,1137.56 1234.56,1137.14 1233.37,1136.92 1233.97,1130.54 1236.69,1130.36 1237.72,1129.90 1238.53,1129.20 1239.12,1128.26">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="4900464597" type="priority" x="969.86" y="1153.94" incLanes="-498671389#1_0 706104219#3_0 -706104219#6_0" intLanes=":4900464597_0_0 :4900464597_1_0 :4900464597_2_0 :4900464597_3_0 :4900464597_4_0 :4900464597_6_0" shape="966.90,1146.46 973.30,1146.66 973.65,1149.35 974.16,1150.37 974.91,1151.17 975.87,1151.75 977.07,1152.12 975.81,1158.40 972.94,1157.83 970.81,1157.43 969.06,1157.16 967.29,1156.94 965.13,1156.73 962.22,1156.48 962.79,1150.10 965.02,1149.85 965.81,1149.37 966.39,1148.64 966.75,1147.67">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="4900464598" type="priority" x="968.97" y="975.80" incLanes="969757849#0_0 498671389#2_0 -969757849#1_0" intLanes=":4900464598_0_0 :4900464598_6_0 :4900464598_2_0 :4900464598_3_0 :4900464598_4_0 :4900464598_5_0" shape="976.40,972.86 976.17,979.26 973.95,979.62 973.18,980.15 972.62,980.91 972.29,981.90 972.18,983.12 965.78,983.13 965.33,980.70 964.78,979.82 964.00,979.18 963.00,978.75 961.78,978.56 962.18,972.17">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="4900465030" type="dead_end" x="1773.17" y="428.24" incLanes="-498671694_0" intLanes="" shape="1773.17,428.24 1771.86,431.16 1773.17,428.24" />\n  <junction id="4900465031" type="priority" x="1729.59" y="408.75" incLanes="142395239#1_0 498671694_0 -706087699#1_0" intLanes=":4900465031_0_0 :4900465031_6_0 :4900465031_2_0 :4900465031_3_0 :4900465031_4_0 :4900465031_5_0" shape="1728.56,401.27 1734.67,403.17 1734.50,405.51 1734.89,406.53 1735.58,407.44 1736.58,408.25 1737.90,408.96 1735.29,414.80 1733.13,414.32 1732.27,414.54 1731.54,415.06 1730.96,415.88 1730.52,417.01 1724.37,415.22">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="4900465052" type="priority" x="1619.81" y="1667.62" incLanes="498671427_0 65080358#2_0" intLanes=":4900465052_0_0 :4900465052_1_0 :4900465052_2_0" shape="1622.55,1674.47 1619.40,1674.99 1618.60,1672.87 1617.92,1672.19 1617.07,1671.77 1616.03,1671.61 1614.80,1671.70 1613.75,1665.38 1615.89,1664.58 1616.57,1663.90 1617.00,1663.04 1617.17,1662.00 1617.08,1660.77 1620.24,1660.26">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="4900465067" type="dead_end" x="1318.32" y="1734.52" incLanes="-498671432_0" intLanes="" shape="1318.32,1734.52 1315.12,1734.67 1318.32,1734.52" />\n  <junction id="4900465069" type="priority" x="1346.02" y="1689.19" incLanes="-243188765#2_0 498671432_0 243188765#1_0" intLanes=":4900465069_0_0 :4900465069_6_0 :4900465069_2_0 :4900465069_3_0 :4900465069_4_0 :4900465069_5_0" shape="1349.43,1696.31 1343.03,1696.50 1342.52,1694.29 1341.94,1693.53 1341.13,1693.00 1340.12,1692.71 1338.88,1692.64 1338.66,1686.24 1340.86,1685.72 1341.61,1685.14 1342.12,1684.34 1342.41,1683.33 1342.46,1682.11 1348.85,1681.78">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="4900465070" type="dead_end" x="1670.68" y="537.10" incLanes="-498671433_0" intLanes="" shape="1670.68,537.10 1669.95,540.22 1670.68,537.10" />\n  <junction id="4900465071" type="priority" x="1624.65" y="526.30" incLanes="155691022#0_0 498671433_0 -155691022#1_0" intLanes=":4900465071_0_0 :4900465071_6_0 :4900465071_2_0 :4900465071_3_0 :4900465071_4_0 :4900465071_5_0" shape="1622.88,518.73 1629.16,519.97 1629.20,522.26 1629.63,523.16 1630.35,523.89 1631.34,524.47 1632.60,524.88 1631.14,531.11 1628.90,531.04 1628.06,531.41 1627.39,532.05 1626.90,532.96 1626.59,534.14 1620.28,533.09">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="4900465090" type="priority" x="315.46" y="1661.39" incLanes="-498671539_0 498671330#0_0" intLanes=":4900465090_3_0 :4900465090_1_0 :4900465090_2_0" shape="312.90,1657.62 319.19,1658.77 319.21,1659.61 319.36,1659.93 319.61,1660.17 319.96,1660.35 320.41,1660.45 319.98,1663.62 310.66,1662.35 311.09,1659.18 311.95,1659.09 312.28,1658.89 312.55,1658.57 312.76,1658.15">\n    <request index="0" response="110" foes="110" cont="1" />\n    <request index="1" response="000" foes="001" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="4900465092" type="dead_end" x="6.18" y="1622.70" incLanes="" intLanes="" shape="6.36,1621.11 6.00,1624.29" />\n  <junction id="4900465098" type="dead_end" x="617.44" y="923.53" incLanes="-498671442_0" intLanes="" shape="617.44,923.53 620.42,924.69 617.44,923.53" />\n  <junction id="4900465099" type="priority" x="581.28" y="1016.06" incLanes="498671442_0 706093505#0_0" intLanes=":4900465099_0_0 :4900465099_1_0 :4900465099_2_0" shape="579.69,1011.33 585.66,1013.66 585.52,1014.46 585.62,1014.76 585.83,1015.00 586.15,1015.18 586.59,1015.29 586.08,1018.45 576.56,1016.68 577.23,1013.56 578.16,1013.45 578.58,1013.16 578.98,1012.70 579.35,1012.09">\n    <request index="0" response="110" foes="110" cont="0" />\n    <request index="1" response="000" foes="001" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="4900465166" type="priority" x="899.77" y="918.85" incLanes="-1365935945#1_0 -498671393_0 1365935945#0_0 -498671465_0" intLanes=":4900465166_0_0 :4900465166_1_0 :4900465166_12_0 :4900465166_3_0 :4900465166_4_0 :4900465166_5_0 :4900465166_6_0 :4900465166_7_0 :4900465166_13_0 :4900465166_9_0 :4900465166_10_0 :4900465166_11_0" shape="897.07,911.33 903.45,911.76 903.75,913.99 904.25,914.78 904.99,915.36 905.97,915.72 907.18,915.86 907.00,922.26 904.63,922.64 903.78,923.17 903.15,923.93 902.75,924.92 902.56,926.14 896.17,925.77 895.84,923.51 895.31,922.69 894.54,922.07 893.53,921.65 892.27,921.43 892.81,915.05 895.05,914.79 895.87,914.29 896.47,913.55 896.87,912.56">\n    <request index="0" response="000000000000" foes="000100010000" cont="0" />\n    <request index="1" response="000000000000" foes="111100110000" cont="0" />\n    <request index="2" response="000111000000" foes="110111110000" cont="1" />\n    <request index="3" response="000010000000" foes="100010000000" cont="0" />\n    <request index="4" response="000110000111" foes="100110000111" cont="0" />\n    <request index="5" response="111110000110" foes="111110000110" cont="0" />\n    <request index="6" response="000000000000" foes="010000000100" cont="0" />\n    <request index="7" response="000000000000" foes="110000111100" cont="0" />\n    <request index="8" response="000000000011" foes="110000110111" cont="1" />\n    <request index="9" response="000000000010" foes="000000100010" cont="0" />\n    <request index="10" response="000111000110" foes="000111100110" cont="0" />\n    <request index="11" response="000110011110" foes="000110111110" cont="0" />\n  </junction>\n  <junction id="4900465208" type="priority" x="586.86" y="501.44" incLanes="706256379#4_0 -498671482_0 -706256379#6_0" intLanes=":4900465208_0_0 :4900465208_6_0 :4900465208_2_0 :4900465208_3_0 :4900465208_4_0 :4900465208_5_0" shape="594.26,498.96 593.63,505.33 591.36,505.56 590.52,506.05 589.87,506.79 589.41,507.78 589.14,509.02 582.79,508.23 582.62,505.97 582.17,505.14 581.46,504.50 580.51,504.06 579.31,503.81 580.03,497.45">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="4900465209" type="priority" x="576.17" y="587.26" incLanes="498671482_0 706256377#2_0 -706256377#3_0" intLanes=":4900465209_0_0 :4900465209_1_0 :4900465209_2_0 :4900465209_3_0 :4900465209_4_0 :4900465209_6_0" shape="573.90,579.58 580.25,580.37 580.42,582.61 580.88,583.43 581.59,584.04 582.54,584.45 583.75,584.66 583.23,591.04 568.74,589.70 569.40,583.34 571.67,583.10 572.52,582.61 573.17,581.85 573.63,580.84">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="4900465219" type="priority" x="1149.97" y="555.31" incLanes="95533432#3_0 -498671679_0 -95533432#4_0" intLanes=":4900465219_0_0 :4900465219_6_0 :4900465219_2_0 :4900465219_3_0 :4900465219_4_0 :4900465219_5_0" shape="1157.52,552.84 1156.89,559.21 1154.65,559.43 1153.83,559.91 1153.22,560.63 1152.81,561.60 1152.60,562.80 1146.22,562.31 1145.96,559.97 1145.47,559.10 1144.74,558.45 1143.77,558.00 1142.56,557.76 1143.22,551.39">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="4900465244" type="priority" x="1372.20" y="649.40" incLanes="-155690480#1_0 498671497_0 155690480#0_0" intLanes=":4900465244_0_0 :4900465244_6_0 :4900465244_2_0 :4900465244_3_0 :4900465244_4_0 :4900465244_5_0" shape="1379.03,646.13 1379.09,652.53 1376.82,653.07 1375.97,653.73 1375.33,654.64 1374.87,655.82 1374.62,657.25 1368.25,656.60 1368.03,654.39 1367.55,653.62 1366.83,653.07 1365.87,652.75 1364.66,652.64 1364.62,646.24">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="4900465288" type="priority" x="1242.30" y="1549.42" incLanes="243188552#1_0 -243188552#2_0 -1559802495#0_0" intLanes=":4900465288_0_0 :4900465288_1_0 :4900465288_2_0 :4900465288_6_0 :4900465288_4_0 :4900465288_5_0" shape="1239.18,1542.51 1245.58,1542.58 1245.77,1556.70 1239.37,1556.93 1238.85,1554.75 1238.27,1554.03 1237.47,1553.55 1236.47,1553.32 1235.24,1553.32 1234.62,1546.95 1237.13,1546.26 1238.02,1545.62 1238.65,1544.78 1239.04,1543.75">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="4900465363" type="priority" x="564.74" y="1226.44" incLanes="498671536_0 95809268#2_0" intLanes=":4900465363_0_0 :4900465363_1_0 :4900465363_2_0 :4900465363_3_0 :4900465363_4_0" shape="563.15,1219.27 566.35,1219.28 566.81,1221.51 567.39,1222.30 568.21,1222.87 569.25,1223.23 570.53,1223.37 570.38,1229.77 568.17,1230.16 567.39,1230.70 566.85,1231.47 566.53,1232.46 566.43,1233.68 563.23,1233.72 562.72,1230.44 562.10,1229.26 561.26,1228.37 560.18,1227.79 558.87,1227.51 559.16,1224.32 561.37,1223.92 562.15,1223.24 562.70,1222.24 563.04,1220.92">\n    <request index="0" response="01000" foes="01000" cont="0" />\n    <request index="1" response="11000" foes="11000" cont="0" />\n    <request index="2" response="00000" foes="00000" cont="0" />\n    <request index="3" response="00000" foes="00011" cont="0" />\n    <request index="4" response="00000" foes="00010" cont="0" />\n  </junction>\n  <junction id="4900465364" type="priority" x="497.83" y="1220.27" incLanes="706093506#1_0 1168168079_0" intLanes=":4900465364_0_0 :4900465364_2_0" shape="497.18,1214.47 500.33,1215.00 500.41,1217.17 500.82,1217.96 501.49,1218.57 502.40,1218.98 503.57,1219.19 503.27,1222.38 500.98,1222.60 500.13,1223.07 499.48,1223.76 499.02,1224.70 498.77,1225.87 495.59,1225.49">\n    <request index="0" response="00" foes="10" cont="0" />\n    <request index="1" response="01" foes="01" cont="1" />\n  </junction>\n  <junction id="4900465370" type="dead_end" x="352.76" y="1624.28" incLanes="498671539_0" intLanes="" shape="352.76,1624.28 352.91,1627.48 352.76,1624.28" />\n  <junction id="4900465377" type="priority" x="1736.98" y="901.02" incLanes="498671542_0 -243314248#1_0 243314248#0_0" intLanes=":4900465377_0_0 :4900465377_1_0 :4900465377_2_0 :4900465377_3_0 :4900465377_4_0 :4900465377_6_0" shape="1734.74,892.83 1741.09,893.59 1741.08,897.07 1741.41,898.50 1741.96,899.74 1742.75,900.77 1743.75,901.59 1740.28,906.96 1737.95,905.49 1736.19,904.47 1734.67,903.75 1733.09,903.15 1731.14,902.51 1728.51,901.69 1730.45,895.59 1732.62,895.82 1733.43,895.50 1734.05,894.90 1734.49,894.01">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="4900465440" type="dead_end" x="1637.48" y="1313.19" incLanes="-498671564_0" intLanes="" shape="1637.48,1313.19 1640.67,1313.37 1637.48,1313.19" />\n  <junction id="4900465441" type="priority" x="1635.39" y="1349.41" incLanes="498671564_0 243314378#2_0 -243314378#3_0" intLanes=":4900465441_0_0 :4900465441_1_0 :4900465441_2_0 :4900465441_3_0 :4900465441_4_0 :4900465441_6_0" shape="1632.61,1342.00 1639.00,1342.37 1639.32,1344.61 1639.83,1345.41 1640.57,1346.00 1641.55,1346.38 1642.77,1346.55 1642.47,1352.94 1627.98,1352.14 1628.39,1345.75 1630.63,1345.45 1631.44,1344.94 1632.04,1344.20 1632.43,1343.22">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="4900465451" type="dead_end" x="1286.60" y="1129.91" incLanes="-498671569_0" intLanes="" shape="1286.60,1129.91 1289.80,1129.98 1286.60,1129.91" />\n  <junction id="4900465471" type="priority" x="600.05" y="718.35" incLanes="706093509#0_0 -498671576#0_0" intLanes=":4900465471_0_0 :4900465471_1_0 :4900465471_3_0" shape="599.43,713.26 602.57,713.89 602.57,714.72 602.72,715.03 602.97,715.26 603.32,715.42 603.77,715.51 603.13,721.87 601.94,721.92 601.50,722.09 601.15,722.34 600.91,722.70 600.77,723.14 597.62,722.54">\n    <request index="0" response="000" foes="100" cont="0" />\n    <request index="1" response="000" foes="100" cont="0" />\n    <request index="2" response="011" foes="011" cont="1" />\n  </junction>\n  <junction id="4900465472" type="priority" x="651.55" y="723.51" incLanes="243188064#0_0 -498671576#1_0 498671576#0_0" intLanes=":4900465472_0_0 :4900465472_1_0 :4900465472_2_0 :4900465472_3_0 :4900465472_7_0 :4900465472_5_0 :4900465472_6_0" shape="651.51,716.04 654.64,716.70 654.61,718.94 655.00,719.79 655.65,720.45 656.57,720.92 657.76,721.22 656.80,727.54 654.27,727.64 653.29,728.10 652.50,728.83 651.91,729.83 651.50,731.10 648.37,730.40 648.42,728.19 648.03,727.37 647.38,726.75 646.46,726.33 645.26,726.10 645.90,719.73 648.62,719.53 649.66,719.04 650.49,718.30 651.11,717.30">\n    <request index="0" response="0001000" foes="0001000" cont="0" />\n    <request index="1" response="1111000" foes="1111000" cont="0" />\n    <request index="2" response="1011000" foes="1011000" cont="0" />\n    <request index="3" response="0000000" foes="0000111" cont="0" />\n    <request index="4" response="1100000" foes="1100110" cont="1" />\n    <request index="5" response="0000000" foes="0010010" cont="0" />\n    <request index="6" response="0000000" foes="0010110" cont="0" />\n  </junction>\n  <junction id="4900465473" type="priority" x="698.50" y="730.13" incLanes="-243188465#1_0 243188465#0_0 498671576#1_0" intLanes=":4900465473_0_0 :4900465473_1_0 :4900465473_2_0 :4900465473_6_0 :4900465473_4_0 :4900465473_5_0" shape="697.13,722.74 703.32,724.36 699.90,738.21 693.66,736.76 693.73,734.54 693.36,733.71 692.71,733.07 691.79,732.62 690.61,732.36 691.40,726.01 694.13,725.91 695.19,725.49 696.04,724.82 696.69,723.90">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="4900465513" type="priority" x="437.07" y="986.83" incLanes="498671586_0 684714761#1_0" intLanes=":4900465513_2_0 :4900465513_1_0" shape="436.69,980.98 439.82,981.66 439.79,983.91 440.17,984.78 440.82,985.47 441.73,985.98 442.91,986.32 442.30,989.46 431.27,987.26 431.92,984.12 434.19,984.14 435.07,983.75 435.78,983.09 436.32,982.17">\n    <request index="0" response="10" foes="10" cont="1" />\n    <request index="1" response="00" foes="01" cont="0" />\n  </junction>\n  <junction id="4900465517" type="priority" x="543.44" y="1381.71" incLanes="706104217#17_0 -498671589_0 -706104217#18_0" intLanes=":4900465517_0_0 :4900465517_6_0 :4900465517_2_0 :4900465517_3_0 :4900465517_4_0 :4900465517_5_0" shape="551.13,1379.47 550.32,1385.82 548.07,1385.98 547.26,1386.43 546.64,1387.14 546.22,1388.09 546.01,1389.30 539.63,1388.76 539.39,1386.42 538.90,1385.56 538.17,1384.92 537.21,1384.48 536.00,1384.26 536.57,1377.88">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="4900465641" type="dead_end" x="1675.64" y="380.29" incLanes="498671600_0" intLanes="" shape="1675.64,380.29 1672.69,379.04 1675.64,380.29" />\n  <junction id="4900465706" type="priority" x="401.92" y="1494.32" incLanes="1552748564#0_0 -1552748564#1_0 498671328#1_0" intLanes=":4900465706_0_0 :4900465706_1_0 :4900465706_2_0 :4900465706_6_0 :4900465706_4_0 :4900465706_5_0" shape="408.90,1491.31 408.72,1497.71 394.30,1497.32 394.47,1490.92 396.66,1490.54 397.40,1490.00 397.89,1489.24 398.15,1488.25 398.18,1487.04 404.56,1486.57 405.19,1489.15 405.81,1490.07 406.64,1490.73 407.67,1491.15">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="4900465707" type="priority" x="1228.78" y="985.74" incLanes="185951074#1_0 -185951074#3_0 498671639_0" intLanes=":4900465707_0_0 :4900465707_1_0 :4900465707_2_0 :4900465707_3_0" shape="1234.15,982.56 1234.12,988.96 1222.85,988.86 1222.94,982.46 1225.13,982.05 1225.87,981.50 1226.36,980.74 1226.62,979.74 1226.65,978.53 1229.84,978.29 1230.46,980.66 1231.07,981.49 1231.89,982.08 1232.92,982.44">\n    <request index="0" response="0000" foes="1100" cont="0" />\n    <request index="1" response="0000" foes="1000" cont="0" />\n    <request index="2" response="0001" foes="0001" cont="0" />\n    <request index="3" response="0011" foes="0011" cont="0" />\n  </junction>\n  <junction id="4900465949" type="priority" x="1452.32" y="458.35" incLanes="-155691084#1_0 1547102487#1_0 155691084#0_0" intLanes=":4900465949_0_0 :4900465949_1_0 :4900465949_2_0 :4900465949_3_0 :4900465949_4_0 :4900465949_5_0 :4900465949_7_0" shape="1459.27,456.04 1458.43,462.39 1456.29,462.54 1455.60,462.98 1455.16,463.68 1454.98,464.62 1455.05,465.82 1451.88,466.29 1446.67,461.98 1446.19,455.60 1449.52,451.36 1452.67,450.82 1453.74,453.38 1454.70,454.35 1455.94,455.12 1457.46,455.68">\n    <request index="0" response="0000000" foes="1001000" cont="0" />\n    <request index="1" response="0000000" foes="1011000" cont="0" />\n    <request index="2" response="0100000" foes="0100000" cont="0" />\n    <request index="3" response="1100011" foes="1100011" cont="0" />\n    <request index="4" response="1100010" foes="1100010" cont="0" />\n    <request index="5" response="0000000" foes="0011100" cont="0" />\n    <request index="6" response="0000011" foes="0011011" cont="1" />\n  </junction>\n  <junction id="4900465989" type="priority" x="381.04" y="1293.97" incLanes="-243188174#2_0 243188174#1_0 -498671673_0" intLanes=":4900465989_0_0 :4900465989_1_0 :4900465989_2_0 :4900465989_6_0 :4900465989_4_0 :4900465989_5_0" shape="387.33,1291.40 386.68,1297.77 371.63,1296.22 372.29,1289.85 374.35,1289.66 374.93,1289.22 375.21,1288.55 375.20,1287.64 374.89,1286.51 380.89,1284.30 381.76,1286.36 382.70,1288.06 383.73,1289.42 384.85,1290.43 386.05,1291.09">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="4900465999" type="priority" x="1047.32" y="916.15" incLanes="-498671616_0 -498671615_0 243188741#1_0" intLanes=":4900465999_0_0 :4900465999_1_0 :4900465999_2_0 :4900465999_3_0 :4900465999_4_0 :4900465999_5_0" shape="1054.63,911.75 1055.63,918.07 1053.59,918.84 1053.02,919.48 1052.74,920.30 1052.76,921.28 1053.08,922.45 1047.08,924.68 1046.07,921.96 1045.31,919.94 1044.64,918.30 1043.88,916.70 1042.89,914.80 1041.49,912.25 1047.09,909.14 1048.79,910.99 1049.94,911.54 1051.30,911.86 1052.87,911.93">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="0" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="4900466002" type="dead_end" x="1197.37" y="611.04" incLanes="498671679_0" intLanes="" shape="1197.37,611.04 1197.11,614.23 1197.37,611.04" />\n  <junction id="4900466295" type="priority" x="378.86" y="1582.41" incLanes="77967084#4_0" intLanes=":4900466295_0_0 :4900466295_1_0" shape="382.13,1586.69 379.12,1587.79 377.78,1585.71 376.80,1584.99 375.61,1584.48 374.20,1584.19 372.59,1584.11 372.54,1580.91 374.60,1580.46 375.18,1579.93 375.45,1579.20 375.42,1578.26 375.09,1577.12 378.07,1575.96">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="4900466297" type="priority" x="411.81" y="1673.36" incLanes="498671330#1_0 77967084#5_0" intLanes=":4900466297_0_0 :4900466297_1_0" shape="414.50,1676.11 411.48,1677.19 410.57,1675.84 409.84,1675.32 408.94,1674.91 407.86,1674.60 406.60,1674.40 406.95,1671.22 408.24,1671.11 408.61,1670.83 408.79,1670.41 408.78,1669.85 408.59,1669.14 411.61,1668.06">\n    <request index="0" response="10" foes="10" cont="0" />\n    <request index="1" response="00" foes="01" cont="0" />\n  </junction>\n  <junction id="4900466448" type="priority" x="885.81" y="1053.19" incLanes="-498671388#1_0 -96941857#1_0 498671388#0_0 96941857#0_0" intLanes=":4900466448_0_0 :4900466448_1_0 :4900466448_2_0 :4900466448_3_0 :4900466448_4_0 :4900466448_12_0 :4900466448_6_0 :4900466448_7_0 :4900466448_8_0 :4900466448_9_0 :4900466448_10_0 :4900466448_13_0" shape="883.62,1045.49 889.96,1046.35 890.11,1048.59 890.56,1049.42 891.26,1050.04 892.21,1050.46 893.41,1050.69 892.82,1057.06 890.51,1057.30 889.66,1057.78 889.03,1058.52 888.61,1059.50 888.39,1060.72 882.01,1060.19 881.75,1057.95 881.26,1057.14 880.53,1056.55 879.56,1056.16 878.35,1055.98 878.71,1049.59 881.18,1049.24 882.10,1048.69 882.81,1047.88 883.32,1046.81">\n    <request index="0" response="000000010000" foes="000100010000" cont="0" />\n    <request index="1" response="111000110000" foes="111100110000" cont="0" />\n    <request index="2" response="110111110000" foes="110111110000" cont="0" />\n    <request index="3" response="000000000000" foes="100010000000" cont="0" />\n    <request index="4" response="000000000000" foes="100110000111" cont="0" />\n    <request index="5" response="111000000000" foes="111110000110" cont="1" />\n    <request index="6" response="010000000000" foes="010000000100" cont="0" />\n    <request index="7" response="110000111000" foes="110000111100" cont="0" />\n    <request index="8" response="110000110011" foes="110000110111" cont="0" />\n    <request index="9" response="000000000000" foes="000000100010" cont="0" />\n    <request index="10" response="000000000000" foes="000111100110" cont="0" />\n    <request index="11" response="000000011000" foes="000110111110" cont="1" />\n  </junction>\n  <junction id="5149825984" type="priority" x="1257.36" y="1181.54" incLanes="96941849_0 72016745#5_0 72016745#5_1 -452036923#0_0 -706104218_0" intLanes=":5149825984_0_0 :5149825984_1_0 :5149825984_2_0 :5149825984_3_0 :5149825984_4_0 :5149825984_12_0 :5149825984_6_0 :5149825984_7_0 :5149825984_8_0 :5149825984_9_0 :5149825984_10_0 :5149825984_13_0" shape="1256.40,1170.30 1262.67,1171.59 1262.67,1173.80 1263.06,1174.61 1263.73,1175.22 1264.66,1175.63 1265.85,1175.85 1264.76,1188.60 1262.48,1188.51 1260.49,1188.64 1258.81,1189.01 1257.42,1189.60 1256.33,1190.42 1255.54,1191.48 1249.93,1188.39 1250.71,1186.15 1250.58,1185.34 1250.11,1184.74 1249.29,1184.34 1248.13,1184.15 1248.54,1177.77 1252.19,1177.11 1253.63,1176.10 1254.81,1174.63 1255.74,1172.69">\n    <request index="0" response="000000010000" foes="000100010000" cont="0" />\n    <request index="1" response="111000110000" foes="111100110000" cont="0" />\n    <request index="2" response="110010110000" foes="110010110000" cont="0" />\n    <request index="3" response="000000000000" foes="100010000000" cont="0" />\n    <request index="4" response="000000000000" foes="100110000111" cont="0" />\n    <request index="5" response="011000000000" foes="111110000110" cont="1" />\n    <request index="6" response="000000000000" foes="000000000000" cont="0" />\n    <request index="7" response="110000111000" foes="110000111100" cont="0" />\n    <request index="8" response="110000110011" foes="110000110011" cont="0" />\n    <request index="9" response="000000000000" foes="000000100010" cont="0" />\n    <request index="10" response="000000000000" foes="000110100110" cont="0" />\n    <request index="11" response="000000111000" foes="000110111110" cont="1" />\n  </junction>\n  <junction id="617233014" type="dead_end" x="295.26" y="828.87" incLanes="40264696#5_0 40264696#5_1 40264696#5_2 706093508#3_0" intLanes="" shape="290.80,825.34 300.35,826.33 300.43,827.18 300.60,827.49 300.87,827.73 301.23,827.90 301.68,828.00 301.32,831.17 297.97,830.66 295.26,829.97 293.19,829.09 291.76,828.02 290.96,826.78" />\n  <junction id="617234280" type="priority" x="314.29" y="644.91" incLanes="40264696#4_0 40264696#4_1 40264696#4_2 40264697#1_0 40264697#1_1" intLanes=":617234280_0_0 :617234280_0_1 :617234280_0_2 :617234280_3_0 :617234280_4_0 :617234280_4_1 :617234280_6_0 :617234280_6_1" shape="310.92,636.64 320.37,638.37 320.41,640.60 320.82,641.43 321.49,642.06 322.42,642.50 323.61,642.74 322.89,649.10 320.53,649.28 319.67,649.74 319.01,650.45 318.56,651.40 318.32,652.60 308.77,651.61 308.56,649.33 308.09,648.49 307.38,647.84 306.42,647.40 305.22,647.16 305.90,640.80 308.32,640.53 309.25,639.98 309.99,639.15 310.55,638.04">\n    <request index="0" response="00000000" foes="11110000" cont="0" />\n    <request index="1" response="00000000" foes="11110000" cont="0" />\n    <request index="2" response="00000000" foes="11110000" cont="0" />\n    <request index="3" response="00000000" foes="11000000" cont="0" />\n    <request index="4" response="00000111" foes="00000111" cont="0" />\n    <request index="5" response="00000111" foes="00000111" cont="0" />\n    <request index="6" response="00001111" foes="00001111" cont="0" />\n    <request index="7" response="00001111" foes="00001111" cont="0" />\n  </junction>\n  <junction id="617234281" type="priority" x="471.60" y="658.31" incLanes="706096640#1_0 40264697#3_0 40264697#3_1" intLanes=":617234281_0_0 :617234281_1_0 :617234281_2_0 :617234281_3_0 :617234281_3_1" shape="471.02,650.77 474.19,651.21 474.33,653.42 474.78,654.21 475.48,654.79 476.44,655.15 477.64,655.30 477.45,661.69 474.75,662.09 473.74,662.65 472.96,663.46 472.40,664.52 472.07,665.83 468.91,665.34 468.80,663.12 468.36,662.32 467.67,661.72 466.72,661.33 465.52,661.15 465.91,654.76 468.49,654.44 469.45,653.90 470.19,653.11 470.71,652.07">\n    <request index="0" response="11100" foes="11100" cont="0" />\n    <request index="1" response="11000" foes="11000" cont="0" />\n    <request index="2" response="00000" foes="00001" cont="0" />\n    <request index="3" response="00000" foes="00011" cont="0" />\n    <request index="4" response="00000" foes="00011" cont="0" />\n  </junction>\n  <junction id="617234535" type="priority" x="343.29" y="486.57" incLanes="40264696#3_0 40264696#3_1 40264696#3_2 684704215#0_0" intLanes=":617234535_0_0 :617234535_1_0 :617234535_1_1 :617234535_1_2 :617234535_4_0 :617234535_5_0 :617234535_5_1" shape="339.36,480.30 348.86,481.69 348.97,483.89 349.40,484.70 350.07,485.32 350.99,485.76 352.16,486.01 351.79,489.18 349.47,489.41 348.58,489.94 347.86,490.73 347.33,491.80 346.97,493.14 337.52,491.41 337.48,489.22 337.08,488.41 336.43,487.79 335.52,487.36 334.36,487.11 334.74,483.93 337.04,483.74 337.90,483.26 338.58,482.53 339.06,481.54">\n    <request index="0" response="0000000" foes="0010000" cont="0" />\n    <request index="1" response="0000000" foes="1110000" cont="0" />\n    <request index="2" response="0000000" foes="1110000" cont="0" />\n    <request index="3" response="0000000" foes="1110000" cont="0" />\n    <request index="4" response="0001111" foes="0001111" cont="0" />\n    <request index="5" response="0001110" foes="0001110" cont="0" />\n    <request index="6" response="0001110" foes="0001110" cont="0" />\n  </junction>\n  <junction id="617234576" type="priority" x="387.24" y="653.17" incLanes="84842974#12_0 84842974#12_1 84842974#12_2 40264697#2_0 40264697#2_1" intLanes=":617234576_0_0 :617234576_1_0 :617234576_1_1 :617234576_1_2 :617234576_4_0 :617234576_4_1 :617234576_6_0 :617234576_6_1" shape="383.81,645.02 393.26,646.70 393.31,648.72 393.69,649.45 394.32,649.99 395.17,650.35 396.27,650.51 395.88,656.90 393.41,657.22 392.48,657.76 391.76,658.54 391.24,659.57 390.92,660.86 381.43,659.41 381.34,657.35 380.94,656.60 380.31,656.01 379.45,655.61 378.35,655.38 379.07,649.02 381.35,648.78 382.23,648.25 382.93,647.44 383.46,646.37">\n    <request index="0" response="00000000" foes="00110000" cont="0" />\n    <request index="1" response="00000000" foes="11110000" cont="0" />\n    <request index="2" response="00000000" foes="11110000" cont="0" />\n    <request index="3" response="00000000" foes="11110000" cont="0" />\n    <request index="4" response="00001111" foes="00001111" cont="0" />\n    <request index="5" response="00001111" foes="00001111" cont="0" />\n    <request index="6" response="00001110" foes="00001110" cont="0" />\n    <request index="7" response="00001110" foes="00001110" cont="0" />\n  </junction>\n  <junction id="617234843" type="priority" x="444.61" y="845.33" incLanes="706097788#2_0 706093508#1_0" intLanes=":617234843_0_0 :617234843_1_0" shape="443.76,839.71 446.93,840.15 450.24,844.61 449.74,847.77 439.03,846.34 439.37,843.16 441.58,842.96 442.40,842.51 443.04,841.81 443.49,840.88">\n    <request index="0" response="10" foes="10" cont="0" />\n    <request index="1" response="00" foes="01" cont="0" />\n  </junction>\n  <junction id="620881544" type="dead_end" x="266.97" y="477.39" incLanes="684704215#1_0" intLanes="" shape="267.16,475.80 266.78,478.98" />\n  <junction id="620881566" type="dead_end" x="245.67" y="637.60" incLanes="" intLanes="" shape="246.01,634.42 245.33,640.78" />\n  <junction id="6634630594" type="priority" x="344.79" y="1341.21" incLanes="706104217#20_0 -706104217#21_0 -706095308#0_0" intLanes=":6634630594_0_0 :6634630594_1_0 :6634630594_2_0 :6634630594_6_0 :6634630594_4_0 :6634630594_5_0" shape="351.16,1339.30 349.82,1345.56 346.62,1344.82 344.25,1344.21 342.29,1343.73 340.32,1343.38 337.91,1343.19 334.64,1343.14 336.00,1336.73 338.04,1336.31 338.63,1335.81 338.94,1335.11 338.97,1334.21 338.71,1333.11 344.78,1331.08 345.65,1333.36 346.59,1335.28 347.61,1336.83 348.71,1338.02 349.89,1338.84">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="6634630596" type="priority" x="327.17" y="1288.42" incLanes="-243188174#1_0 706095308#0_0 243188174#0_0 -706095308#1_0" intLanes=":6634630596_0_0 :6634630596_1_0 :6634630596_12_0 :6634630596_3_0 :6634630596_4_0 :6634630596_5_0 :6634630596_6_0 :6634630596_7_0 :6634630596_13_0 :6634630596_9_0 :6634630596_10_0 :6634630596_11_0" shape="336.41,1286.16 335.75,1292.52 333.68,1292.72 333.09,1293.16 332.78,1293.84 332.77,1294.76 333.04,1295.91 326.97,1297.93 326.12,1295.91 325.01,1294.22 323.65,1292.85 322.04,1291.81 320.17,1291.09 318.05,1290.70 318.71,1284.33 320.78,1284.13 321.39,1283.68 321.72,1283.00 321.75,1282.08 321.50,1280.92 327.62,1279.03 328.41,1281.01 329.48,1282.68 330.81,1284.02 332.41,1285.05 334.28,1285.76">\n    <request index="0" response="000000000000" foes="000100010000" cont="0" />\n    <request index="1" response="000000000000" foes="111100110000" cont="0" />\n    <request index="2" response="000011000000" foes="110011110000" cont="1" />\n    <request index="3" response="000010000000" foes="100010000000" cont="0" />\n    <request index="4" response="000110000111" foes="100110000111" cont="0" />\n    <request index="5" response="011110000110" foes="011110000110" cont="0" />\n    <request index="6" response="000000000000" foes="010000000100" cont="0" />\n    <request index="7" response="000000000000" foes="110000111100" cont="0" />\n    <request index="8" response="000000000011" foes="110000110011" cont="1" />\n    <request index="9" response="000000000010" foes="000000100010" cont="0" />\n    <request index="10" response="000111000110" foes="000111100110" cont="0" />\n    <request index="11" response="000110011110" foes="000110011110" cont="0" />\n  </junction>\n  <junction id="6634655051" type="dead_end" x="429.15" y="1250.85" incLanes="1316153439_0" intLanes="" shape="429.15,1250.85 428.82,1254.03 429.15,1250.85" />\n  <junction id="7691192946" type="dead_end" x="1971.98" y="1710.95" incLanes="-81350006#1_0" intLanes="" shape="1971.98,1710.95 1971.58,1714.13 1971.98,1710.95" />\n  <junction id="796720075" type="dead_end" x="1350.52" y="1.19" incLanes="" intLanes="" shape="1352.12,1.27 1348.92,1.11" />\n  <junction id="796747634" type="priority" x="1495.25" y="902.19" incLanes="243188629_0 1218225522#0_0" intLanes=":796747634_0_0 :796747634_1_0 :796747634_2_0" shape="1497.94,908.77 1494.78,909.28 1493.82,907.05 1492.95,906.24 1491.80,905.63 1490.39,905.22 1488.72,905.01 1489.09,898.63 1491.24,898.32 1491.93,897.81 1492.36,897.08 1492.53,896.11 1492.44,894.91 1495.60,894.40">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="796747714" type="priority" x="1453.95" y="648.54" incLanes="155690480#1_0 990212966#0_0" intLanes=":796747714_0_0 :796747714_1_0 :796747714_2_0" shape="1456.65,655.19 1453.50,655.70 1452.59,653.51 1451.77,652.75 1450.71,652.21 1449.42,651.90 1447.88,651.80 1447.81,645.40 1449.96,644.94 1450.65,644.39 1451.08,643.61 1451.25,642.62 1451.17,641.41 1454.33,640.90">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="809537690" type="priority" x="1567.86" y="1176.65" incLanes="706105952#0_0 243079844#0_0" intLanes=":809537690_0_0 :809537690_1_0 :809537690_2_0 :809537690_4_0" shape="1573.57,1174.86 1573.68,1178.06 1571.52,1178.57 1570.81,1179.15 1570.36,1179.94 1570.16,1180.94 1570.21,1182.15 1567.04,1182.58 1562.57,1179.27 1561.98,1176.12 1565.47,1171.40 1568.63,1170.94 1569.45,1173.18 1570.17,1173.95 1571.10,1174.49 1572.23,1174.79">\n    <request index="0" response="0000" foes="0100" cont="0" />\n    <request index="1" response="0000" foes="1100" cont="0" />\n    <request index="2" response="0011" foes="0011" cont="0" />\n    <request index="3" response="0010" foes="0010" cont="1" />\n  </junction>\n  <junction id="809537730" type="dead_end" x="1361.73" y="0.45" incLanes="932339002#5_0 932339002#5_1" intLanes="" shape="1364.93,0.62 1358.53,0.28" />\n  <junction id="809537820" type="traffic_light" x="1566.11" y="1164.78" incLanes="66984604#1_0 706105952#1_0" intLanes=":809537820_0_0 :809537820_1_0 :809537820_2_0 :809537820_3_0" shape="1571.32,1163.00 1571.43,1166.20 1569.55,1166.65 1568.95,1167.15 1568.57,1167.83 1568.40,1168.70 1568.46,1169.76 1565.30,1170.22 1564.58,1168.16 1563.98,1167.51 1563.21,1167.10 1562.27,1166.95 1561.16,1167.05 1560.80,1163.86 1562.63,1163.14 1563.22,1162.56 1563.58,1161.84 1563.72,1160.97 1563.64,1159.96 1566.79,1159.41 1567.58,1161.46 1568.25,1162.17 1569.09,1162.66 1570.12,1162.94">\n    <request index="0" response="0000" foes="0100" cont="0" />\n    <request index="1" response="0000" foes="1100" cont="0" />\n    <request index="2" response="0011" foes="0011" cont="0" />\n    <request index="3" response="0010" foes="0010" cont="0" />\n  </junction>\n  <junction id="823954361" type="dead_end" x="1983.68" y="1660.37" incLanes="-71380788_0" intLanes="" shape="1983.68,1660.37 1983.39,1663.56 1983.68,1660.37" />\n  <junction id="8348961008" type="priority" x="309.72" y="1200.57" incLanes="-898440288_0 84842974#1_0 84842974#1_1 84842974#1_2" intLanes=":8348961008_0_0 :8348961008_1_0 :8348961008_2_0 :8348961008_2_1 :8348961008_2_2" shape="306.19,1192.71 315.65,1194.32 315.72,1196.59 316.15,1197.46 316.83,1198.14 317.77,1198.65 318.98,1198.98 317.84,1205.28 315.58,1205.32 314.72,1205.73 314.05,1206.40 313.55,1207.33 313.24,1208.52 303.77,1206.91">\n    <request index="0" response="11100" foes="11100" cont="0" />\n    <request index="1" response="00000" foes="00000" cont="0" />\n    <request index="2" response="00000" foes="00001" cont="0" />\n    <request index="3" response="00000" foes="00001" cont="0" />\n    <request index="4" response="00000" foes="00001" cont="0" />\n  </junction>\n  <junction id="8348961009" type="priority" x="425.64" y="1221.37" incLanes="-243188536#1_0 243188536#0_0 898440288_0" intLanes=":8348961009_0_0 :8348961009_1_0 :8348961009_2_0 :8348961009_6_0 :8348961009_4_0 :8348961009_5_0" shape="423.80,1213.72 430.09,1214.89 427.48,1229.04 421.19,1227.89 421.15,1225.62 420.75,1224.76 420.08,1224.08 419.16,1223.57 417.97,1223.25 419.10,1216.95 421.39,1216.90 422.26,1216.50 422.95,1215.83 423.47,1214.91">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="8363308288" type="priority" x="1362.66" y="1134.84" incLanes="-900244004_0 -900244003#1_0 900244003#0_0" intLanes=":8363308288_0_0 :8363308288_1_0 :8363308288_2_0 :8363308288_3_0 :8363308288_4_0 :8363308288_6_0" shape="1359.92,1127.70 1366.31,1128.12 1366.59,1130.40 1367.07,1131.26 1367.77,1131.93 1368.70,1132.40 1369.86,1132.69 1368.90,1139.02 1366.09,1138.60 1364.01,1138.31 1362.29,1138.13 1360.56,1138.01 1358.46,1137.92 1355.62,1137.83 1355.81,1131.44 1357.98,1131.07 1358.77,1130.56 1359.35,1129.83 1359.74,1128.87">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="8363308289" type="dead_end" x="1428.39" y="1144.77" incLanes="900244003#1_0" intLanes="" shape="1428.39,1144.77 1427.91,1147.93 1428.39,1144.77" />\n  <junction id="8363308291" type="dead_end" x="1437.69" y="1124.41" incLanes="900244004_0" intLanes="" shape="1437.69,1124.41 1437.10,1127.55 1437.69,1124.41" />\n  <junction id="849403490" type="dead_end" x="1992.35" y="1212.03" incLanes="243079738_0" intLanes="" shape="1992.41,1213.63 1992.29,1210.43" />\n  <junction id="8523321638" type="dead_end" x="1836.34" y="543.00" incLanes="155691084#3_0" intLanes="" shape="1836.34,543.00 1835.36,546.05 1836.34,543.00" />\n  <junction id="8523326825" type="priority" x="1589.04" y="881.14" incLanes="-243314165#1_0 917695719_0 243314165#0_0" intLanes=":8523326825_0_0 :8523326825_6_0 :8523326825_2_0 :8523326825_3_0 :8523326825_4_0 :8523326825_5_0" shape="1596.62,878.89 1595.79,885.24 1593.54,885.40 1592.70,885.85 1592.05,886.55 1591.60,887.50 1591.34,888.70 1584.99,887.92 1584.83,885.59 1584.37,884.71 1583.67,884.03 1582.73,883.54 1581.53,883.24 1582.49,876.91">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="8523326826" type="priority" x="1584.29" y="920.15" incLanes="-917695719_0 243314312_0" intLanes=":8523326826_0_0 :8523326826_1_0" shape="1581.50,916.61 1587.85,917.38 1586.72,920.81 1585.69,921.93 1584.36,922.67 1582.72,923.01 1580.76,922.96">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="8523328981" type="dead_end" x="1638.21" y="1116.72" incLanes="917695557_0" intLanes="" shape="1638.21,1116.72 1636.27,1119.27 1638.21,1116.72" />\n  <junction id="8523328986" type="priority" x="1554.92" y="1101.11" incLanes="-917695557_0 151072453#0_0" intLanes=":8523328986_0_0 :8523328986_1_0 :8523328986_2_0" shape="1560.76,1097.61 1561.08,1104.01 1558.94,1104.55 1558.26,1105.14 1557.84,1105.93 1557.68,1106.94 1557.78,1108.15 1554.63,1108.70 1552.17,1094.44 1555.32,1093.91 1556.22,1096.07 1557.02,1096.81 1558.04,1097.31 1559.29,1097.58">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="8523328988" type="dead_end" x="1763.96" y="1102.14" incLanes="-243079757#3_0" intLanes="" shape="1763.96,1102.14 1762.97,1105.18 1763.96,1102.14" />\n  <junction id="8523329815" type="dead_end" x="373.17" y="1170.89" incLanes="917697169_0" intLanes="" shape="373.17,1170.89 372.43,1174.00 373.17,1170.89" />\n  <junction id="8523333971" type="dead_end" x="740.12" y="1083.30" incLanes="917696759_0" intLanes="" shape="740.12,1083.30 743.28,1083.80 740.12,1083.30" />\n  <junction id="8523334139" type="priority" x="627.65" y="1023.48" incLanes="706093505#1_0" intLanes=":8523334139_0_0 :8523334139_1_0" shape="627.61,1017.99 630.68,1018.90 630.50,1020.85 630.79,1021.60 631.32,1022.21 632.09,1022.67 633.12,1022.98 632.47,1026.12 622.54,1024.28 623.05,1021.12 625.09,1021.02 625.92,1020.62 626.62,1019.98 627.18,1019.10">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="8523344319" type="priority" x="448.19" y="845.90" incLanes="706093508#0_0" intLanes=":8523344319_0_0 :8523344319_1_0" shape="453.93,845.26 453.39,848.42 451.14,848.48 450.28,848.90 449.60,849.57 449.09,850.51 448.77,851.69 445.62,851.12 442.41,846.60 442.91,843.44">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="8523406712" type="dead_end" x="634.98" y="1476.99" incLanes="917704692_0" intLanes="" shape="634.98,1476.99 631.85,1476.33 634.98,1476.99" fringe="inner">\n    <param key="buffer_stop" value="true" />\n  </junction>\n  <junction id="8523431061" type="dead_end" x="990.80" y="1268.79" incLanes="917705248_0" intLanes="" shape="990.80,1268.79 994.00,1268.91 990.80,1268.79" />\n  <junction id="8523431062" type="priority" x="988.37" y="1331.11" incLanes="-917705248_0 706104217#0_0 -706104217#2_0" intLanes=":8523431062_0_0 :8523431062_1_0 :8523431062_2_0 :8523431062_3_0 :8523431062_4_0 :8523431062_6_0" shape="985.49,1322.87 991.88,1323.12 992.23,1325.24 992.74,1325.90 993.48,1326.30 994.45,1326.42 995.64,1326.27 996.99,1332.53 982.75,1335.64 981.36,1329.39 983.56,1328.28 984.35,1327.32 984.93,1326.10 985.31,1324.62">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="8523431063" type="dead_end" x="947.27" y="1263.50" incLanes="917705249_0" intLanes="" shape="947.27,1263.50 950.41,1262.90 947.27,1263.50" />\n  <junction id="8523431064" type="priority" x="961.31" y="1337.12" incLanes="706104217#2_0 -706104217#3_0 -917705249_0" intLanes=":8523431064_0_0 :8523431064_1_0 :8523431064_2_0 :8523431064_6_0 :8523431064_4_0 :8523431064_5_0" shape="967.74,1332.41 969.13,1338.66 955.43,1342.04 953.75,1335.86 955.81,1334.80 956.44,1334.01 956.82,1333.03 956.94,1331.88 956.80,1330.55 963.09,1329.35 963.94,1331.44 964.62,1332.08 965.49,1332.46 966.53,1332.57">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="1" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="8523449429" type="dead_end" x="1685.14" y="1316.83" incLanes="917706996_0" intLanes="" shape="1685.14,1316.83 1688.33,1317.06 1685.14,1316.83" />\n  <junction id="8523449430" type="priority" x="1682.59" y="1351.59" incLanes="-917706996_0 243314378#0_0 -243314378#2_0" intLanes=":8523449430_0_0 :8523449430_1_0 :8523449430_2_0 :8523449430_3_0 :8523449430_4_0 :8523449430_6_0" shape="1679.93,1344.09 1686.31,1344.56 1686.60,1346.79 1687.09,1347.59 1687.83,1348.18 1688.81,1348.56 1690.02,1348.73 1689.72,1355.12 1675.34,1354.46 1675.63,1348.07 1677.88,1347.71 1678.70,1347.16 1679.32,1346.38 1679.73,1345.35">\n    <request index="0" response="001000" foes="001000" cont="0" />\n    <request index="1" response="111000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000000" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="001100" foes="001110" cont="1" />\n  </junction>\n  <junction id="856214470" type="priority" x="669.32" y="1032.15" incLanes="706092523#0_0 706093505#2_0" intLanes=":856214470_0_0 :856214470_2_0" shape="669.17,1026.65 672.25,1027.50 669.52,1037.81 666.42,1037.01 666.53,1034.91 666.20,1034.09 665.61,1033.44 664.77,1032.94 663.67,1032.61 664.33,1029.48 666.57,1029.52 667.46,1029.17 668.19,1028.58 668.76,1027.74">\n    <request index="0" response="00" foes="10" cont="0" />\n    <request index="1" response="01" foes="01" cont="1" />\n  </junction>\n  <junction id="856214503" type="priority" x="702.16" y="913.43" incLanes="931354569#0_0 706092523#2_0 -931354569#1_0" intLanes=":856214503_0_0 :856214503_1_0 :856214503_2_0 :856214503_3_0 :856214503_4_0 :856214503_5_0 :856214503_7_0" shape="702.39,905.91 705.50,906.68 705.39,908.92 705.75,909.76 706.39,910.43 707.29,910.91 708.47,911.22 707.44,917.53 704.79,917.56 703.75,917.96 702.91,918.61 702.25,919.53 701.79,920.71 698.71,919.85 698.84,917.54 698.46,916.59 697.78,915.77 696.81,915.09 695.55,914.53 697.64,908.48 699.85,908.78 700.72,908.49 701.43,907.92 701.99,907.06">\n    <request index="0" response="0000000" foes="1001000" cont="0" />\n    <request index="1" response="0000000" foes="1011000" cont="0" />\n    <request index="2" response="0100000" foes="0100000" cont="0" />\n    <request index="3" response="1100011" foes="1100011" cont="0" />\n    <request index="4" response="1100010" foes="1100010" cont="0" />\n    <request index="5" response="0000000" foes="0011100" cont="0" />\n    <request index="6" response="0000011" foes="0011011" cont="1" />\n  </junction>\n  <junction id="856214524" type="priority" x="1336.71" y="985.69" incLanes="1270957863_0 706105951#1_0 -185951074#0_0 -706105951#3_0" intLanes=":856214524_0_0 :856214524_1_0 :856214524_12_0 :856214524_3_0 :856214524_4_0 :856214524_5_0 :856214524_6_0 :856214524_7_0 :856214524_13_0 :856214524_9_0 :856214524_10_0 :856214524_11_0" shape="1344.12,982.82 1343.83,989.21 1341.60,989.56 1340.81,990.09 1340.24,990.85 1339.88,991.84 1339.75,993.07 1333.35,992.93 1332.95,990.71 1332.41,989.94 1331.65,989.40 1330.66,989.07 1329.44,988.98 1329.36,982.58 1331.64,982.11 1332.44,981.54 1333.00,980.76 1333.33,979.75 1333.43,978.53 1339.83,978.46 1340.33,980.80 1340.93,981.64 1341.76,982.25 1342.82,982.65">\n    <request index="0" response="000000000000" foes="000100010000" cont="0" />\n    <request index="1" response="000000000000" foes="111100110000" cont="0" />\n    <request index="2" response="000011000000" foes="110111110000" cont="1" />\n    <request index="3" response="000010000000" foes="100010000000" cont="0" />\n    <request index="4" response="000110000111" foes="100110000111" cont="0" />\n    <request index="5" response="011110000110" foes="111110000110" cont="0" />\n    <request index="6" response="000000000000" foes="010000000100" cont="0" />\n    <request index="7" response="000000000000" foes="110000111100" cont="0" />\n    <request index="8" response="000000000111" foes="110000110111" cont="1" />\n    <request index="9" response="000000000010" foes="000000100010" cont="0" />\n    <request index="10" response="000111000110" foes="000111100110" cont="0" />\n    <request index="11" response="000110111110" foes="000110111110" cont="0" />\n  </junction>\n  <junction id="856214538" type="priority" x="896.70" y="972.59" incLanes="-1365935945#0_0 969757849#1_0 498671388#1_0 -969757849#2_0" intLanes=":856214538_0_0 :856214538_1_0 :856214538_2_0 :856214538_3_0 :856214538_4_0 :856214538_12_0 :856214538_6_0 :856214538_7_0 :856214538_8_0 :856214538_9_0 :856214538_10_0 :856214538_13_0" shape="893.96,964.39 900.35,964.75 900.65,967.38 901.15,968.31 901.89,968.98 902.87,969.40 904.08,969.56 903.94,975.96 901.41,976.34 900.47,976.87 899.74,977.63 899.24,978.61 898.95,979.82 892.61,978.96 892.35,976.53 891.76,975.53 890.85,974.67 889.62,973.96 888.09,973.39 889.90,967.25 892.06,967.43 892.83,967.10 893.41,966.48 893.79,965.58">\n    <request index="0" response="000000010000" foes="000100010000" cont="0" />\n    <request index="1" response="111000110000" foes="111100110000" cont="0" />\n    <request index="2" response="110111110000" foes="110111110000" cont="0" />\n    <request index="3" response="000000000000" foes="100010000000" cont="0" />\n    <request index="4" response="000000000000" foes="100110000111" cont="0" />\n    <request index="5" response="011000000000" foes="111110000110" cont="1" />\n    <request index="6" response="010000000000" foes="010000000100" cont="0" />\n    <request index="7" response="110000111000" foes="110000111100" cont="0" />\n    <request index="8" response="110000110011" foes="110000110111" cont="0" />\n    <request index="9" response="000000000000" foes="000000100010" cont="0" />\n    <request index="10" response="000000000000" foes="000111100110" cont="0" />\n    <request index="11" response="000000111000" foes="000110111110" cont="1" />\n  </junction>\n  <junction id="856214571" type="traffic_light" x="1544.12" y="1202.38" incLanes="-66984610#0_0 -72016745#1_0 -72016745#1_1 1218225524#1_0" intLanes=":856214571_0_0 :856214571_0_1 :856214571_2_0 :856214571_3_0 :856214571_4_0 :856214571_5_0 :856214571_6_0 :856214571_7_0" shape="1549.44,1198.53 1550.20,1204.88 1548.12,1205.93 1547.51,1207.01 1547.18,1208.45 1547.14,1210.26 1547.38,1212.44 1544.22,1212.96 1543.43,1210.85 1542.76,1210.19 1541.90,1209.79 1540.87,1209.65 1539.65,1209.77 1537.20,1197.21 1539.50,1196.31 1540.23,1195.60 1540.70,1194.72 1540.89,1193.66 1540.80,1192.43 1543.95,1191.89 1544.44,1194.06 1545.10,1195.80 1545.93,1197.12 1546.93,1198.01 1548.10,1198.48">\n    <request index="0" response="11100000" foes="11100000" cont="0" />\n    <request index="1" response="11100000" foes="11100000" cont="0" />\n    <request index="2" response="11011000" foes="11011000" cont="0" />\n    <request index="3" response="01000000" foes="01000100" cont="0" />\n    <request index="4" response="11000000" foes="11000100" cont="0" />\n    <request index="5" response="00000000" foes="00000011" cont="0" />\n    <request index="6" response="00000000" foes="00011111" cont="0" />\n    <request index="7" response="00000000" foes="00010111" cont="0" />\n  </junction>\n  <junction id="856214578" type="priority" x="340.60" y="966.73" incLanes="84842974#9_0 84842974#9_1 84842974#9_2 684714761#0_0" intLanes=":856214578_0_0 :856214578_1_0 :856214578_1_1 :856214578_1_2 :856214578_4_0 :856214578_5_0 :856214578_5_1" shape="336.70,960.48 346.19,961.92 346.30,964.36 346.77,965.30 347.51,966.05 348.52,966.62 349.80,967.01 349.15,970.14 346.94,970.13 346.14,970.53 345.54,971.19 345.14,972.12 344.95,973.32 335.37,972.65 335.12,969.78 334.62,968.72 333.88,967.90 332.89,967.32 331.66,966.99 332.14,963.83 334.40,963.72 335.26,963.29 335.92,962.61 336.40,961.67">\n    <request index="0" response="0000000" foes="0010000" cont="0" />\n    <request index="1" response="0000000" foes="1110000" cont="0" />\n    <request index="2" response="0000000" foes="1110000" cont="0" />\n    <request index="3" response="0000000" foes="1110000" cont="0" />\n    <request index="4" response="0001111" foes="0001111" cont="0" />\n    <request index="5" response="0001110" foes="0001110" cont="0" />\n    <request index="6" response="0001110" foes="0001110" cont="0" />\n  </junction>\n  <junction id="8642733849" type="dead_end" x="1635.90" y="442.26" incLanes="932324369_0" intLanes="" shape="1635.90,442.26 1632.93,441.07 1635.90,442.26" />\n  <junction id="8642733850" type="priority" x="1645.32" y="418.77" incLanes="129842397#1_0 -932324369_0 -129842397#3_0" intLanes=":8642733850_0_0 :8642733850_6_0 :8642733850_2_0 :8642733850_3_0 :8642733850_4_0 :8642733850_5_0" shape="1653.31,418.83 1650.71,424.68 1648.51,424.18 1647.59,424.38 1646.80,424.88 1646.12,425.67 1645.56,426.76 1639.62,424.38 1640.10,422.01 1639.90,421.02 1639.40,420.15 1638.62,419.42 1637.55,418.81 1640.16,412.96">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n  </junction>\n  <junction id="8642866006" type="priority" x="1477.33" y="720.55" incLanes="-243314310#0_0 990212968_0 990212968_1" intLanes=":8642866006_0_0 :8642866006_1_0 :8642866006_2_0 :8642866006_2_1" shape="1484.98,717.15 1485.08,723.55 1482.93,724.06 1482.24,724.63 1481.81,725.41 1481.64,726.41 1481.73,727.62 1475.41,728.65 1473.08,714.39 1479.40,713.35 1480.30,715.53 1481.11,716.28 1482.16,716.80 1483.45,717.09">\n    <request index="0" response="1100" foes="1100" cont="0" />\n    <request index="1" response="0000" foes="0000" cont="0" />\n    <request index="2" response="0000" foes="0001" cont="0" />\n    <request index="3" response="0000" foes="0001" cont="0" />\n  </junction>\n  <junction id="8642866010" type="priority" x="1447.22" y="458.73" incLanes="-155691084#0_0 932339006_0" intLanes=":8642866010_0_0 :8642866010_1_0" shape="1447.11,455.53 1447.59,461.91 1447.41,461.93 1446.76,455.56">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="8976428985" type="priority" x="896.05" y="678.00" incLanes="970010765#3_0 100401250#0_0" intLanes=":8976428985_0_0 :8976428985_1_0" shape="894.66,675.96 897.84,676.30 897.74,679.57 898.00,680.72 898.50,681.81 899.27,683.03 900.34,684.58 897.25,685.74 896.87,685.32 896.74,685.29 896.63,685.37 896.55,685.55 896.48,685.83 893.32,685.35">\n    <request index="0" response="00" foes="10" cont="0" />\n    <request index="1" response="01" foes="01" cont="0" />\n  </junction>\n  <junction id="8976428996" type="dead_end" x="1855.51" y="772.01" incLanes="" intLanes="" shape="1855.44,773.61 1855.58,770.41" />\n  <junction id="8976429008" type="priority" x="1499.60" y="751.57" incLanes="969757855#1_0 969757852_0" intLanes=":8976429008_0_0 :8976429008_1_0" shape="1505.73,750.47 1505.46,753.66 1503.31,753.91 1502.62,754.40 1502.18,755.12 1501.99,756.09 1502.06,757.29 1498.90,757.75 1498.07,755.13 1497.38,754.18 1496.51,753.47 1495.46,753.01 1494.22,752.79 1494.39,749.60">\n    <request index="0" response="00" foes="10" cont="0" />\n    <request index="1" response="01" foes="01" cont="0" />\n  </junction>\n  <junction id="8976429009" type="priority" x="1379.76" y="739.49" incLanes="706105950_0 969757851#3_0" intLanes=":8976429009_0_0 :8976429009_1_0 :8976429009_2_0" shape="1386.94,738.68 1386.59,741.86 1384.27,742.13 1383.35,742.68 1382.60,743.52 1382.01,744.64 1381.59,746.04 1375.34,744.66 1375.38,742.45 1374.99,741.64 1374.34,741.02 1373.41,740.60 1372.22,740.38 1372.53,737.19">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="916327686" type="priority" x="446.23" y="1696.30" incLanes="77967093#0_0 77967093#0_1" intLanes=":916327686_0_0 :916327686_0_1 :916327686_0_2 :916327686_3_0 :916327686_3_1" shape="451.63,1695.94 448.20,1701.34 444.02,1699.06 439.93,1697.57 435.93,1696.87 432.01,1696.96 428.18,1697.85 424.43,1699.53 426.37,1691.94 427.94,1691.07 428.21,1690.57 428.14,1690.04 427.73,1689.46 426.98,1688.84 433.86,1679.94 435.55,1684.69 437.61,1687.79 440.13,1689.86 443.23,1691.51 447.02,1693.33">\n    <request index="0" response="00000" foes="00000" cont="0" />\n    <request index="1" response="00000" foes="00000" cont="0" />\n    <request index="2" response="00000" foes="00000" cont="0" />\n    <request index="3" response="00000" foes="00000" cont="0" />\n    <request index="4" response="00000" foes="00000" cont="0" />\n  </junction>\n  <junction id="916327702" type="priority" x="353.26" y="1516.45" incLanes="77967085_0 77967085_1 77967084#3_0" intLanes=":916327702_0_0 :916327702_1_0" shape="356.74,1521.01 353.76,1522.16 352.66,1520.76 351.78,1520.26 350.67,1519.90 349.34,1519.67 347.78,1519.58 347.86,1513.19 349.27,1512.92 349.68,1512.56 349.90,1512.07 349.92,1511.43 349.75,1510.64 352.80,1509.68">\n    <request index="0" response="10" foes="10" cont="0" />\n    <request index="1" response="00" foes="01" cont="0" />\n  </junction>\n  <junction id="916327777" type="dead_end" x="39.51" y="1375.45" incLanes="" intLanes="" shape="39.38,1372.25 39.64,1378.65" />\n  <junction id="916327792" type="dead_end" x="29.35" y="1428.06" incLanes="77967092_0 77967092_1" intLanes="" shape="29.29,1424.86 29.41,1431.26" />\n  <junction id="916327810" type="priority" x="323.81" y="1422.99" incLanes="77967084#2_0" intLanes=":916327810_0_0 :916327810_0_1 :916327810_2_0" shape="327.40,1429.06 324.35,1430.02 323.02,1427.91 321.96,1427.18 320.63,1426.67 319.04,1426.38 317.19,1426.30 317.08,1419.90 319.16,1419.44 319.76,1418.90 320.07,1418.16 320.09,1417.20 319.81,1416.04 322.85,1415.03">\n    <request index="0" response="000" foes="000" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="000" cont="0" />\n  </junction>\n  <junction id="916327824" type="dead_end" x="24.84" y="1511.03" incLanes="" intLanes="" shape="24.87,1507.83 24.81,1514.23" />\n  <junction id="916327839" type="priority" x="407.87" y="1718.42" incLanes="77967093#2_0 77967093#2_1 324267161#0_0" intLanes=":916327839_0_0 :916327839_1_0 :916327839_1_1 :916327839_3_0" shape="406.20,1711.38 412.44,1712.81 411.37,1717.28 411.08,1718.87 410.97,1720.48 411.07,1722.41 411.39,1725.00 404.91,1724.92 404.25,1722.85 403.67,1722.09 402.94,1721.51 402.05,1721.12 401.00,1720.90 401.68,1714.54 403.83,1714.36 404.67,1713.95 405.34,1713.31 405.85,1712.46">\n    <request index="0" response="0000" foes="0000" cont="0" />\n    <request index="1" response="0000" foes="1000" cont="0" />\n    <request index="2" response="0000" foes="0000" cont="0" />\n    <request index="3" response="0010" foes="0010" cont="0" />\n  </junction>\n  <junction id="916327888" type="priority" x="462.10" y="1720.09" incLanes="970311985#1_0 77967093#3_0 77967093#3_1" intLanes=":916327888_0_0 :916327888_1_0 :916327888_1_1" shape="468.29,1724.46 465.94,1726.63 465.37,1726.21 465.14,1726.21 464.93,1726.33 464.75,1726.59 464.61,1726.99 458.10,1726.51 458.77,1724.54 459.01,1723.84 459.16,1723.13 459.17,1722.27 459.03,1721.12 465.34,1720.04 465.90,1721.92 466.31,1722.50 466.83,1723.03 467.48,1723.64">\n    <request index="0" response="010" foes="010" cont="0" />\n    <request index="1" response="000" foes="001" cont="0" />\n    <request index="2" response="000" foes="000" cont="0" />\n  </junction>\n  <junction id="916327912" type="priority" x="364.29" y="1493.33" incLanes="1552748564#1_0 77967090#0_0 77967090#0_1 77967090#0_2" intLanes=":916327912_0_0 :916327912_1_0 :916327912_2_0 :916327912_2_1 :916327912_2_2" shape="374.80,1490.41 374.63,1496.80 372.56,1497.16 371.97,1497.66 371.68,1498.38 371.69,1499.31 372.00,1500.46 362.98,1503.75 357.94,1489.52 367.03,1486.44 368.46,1488.57 369.61,1489.33 371.05,1489.89 372.78,1490.25">\n    <request index="0" response="11100" foes="11100" cont="0" />\n    <request index="1" response="00000" foes="00000" cont="0" />\n    <request index="2" response="00000" foes="00001" cont="0" />\n    <request index="3" response="00000" foes="00001" cont="0" />\n    <request index="4" response="00000" foes="00001" cont="0" />\n  </junction>\n  <junction id="916327921" type="priority" x="412.95" y="1704.99" incLanes="931348308_0 77967093#1_0 77967093#1_1" intLanes=":916327921_0_0 :916327921_1_0 :916327921_1_1" shape="415.32,1689.34 418.65,1690.82 419.21,1692.76 419.76,1693.32 420.49,1693.60 421.40,1693.59 422.50,1693.31 427.74,1698.25 423.97,1699.65 421.20,1700.95 419.09,1702.40 417.35,1704.27 415.66,1706.83 413.69,1710.34 408.06,1707.29 410.25,1703.68 412.16,1701.00 413.71,1698.74 414.80,1696.39 415.37,1693.43">\n    <request index="0" response="010" foes="010" cont="0" />\n    <request index="1" response="000" foes="001" cont="0" />\n    <request index="2" response="000" foes="000" cont="0" />\n  </junction>\n  <junction id="916327945" type="dead_end" x="1084.32" y="2036.67" incLanes="" intLanes="" shape="1083.67,2038.13 1084.97,2035.21" />\n  <junction id="947896950" type="priority" x="894.34" y="1353.65" incLanes="706104217#4_0 -706104217#8_0" intLanes=":947896950_0_0 :947896950_1_0 :947896950_2_0 :947896950_4_0" shape="899.38,1349.26 900.38,1355.60 889.98,1358.01 888.46,1351.79 890.56,1350.79 891.24,1350.03 891.67,1349.08 891.85,1347.95 891.78,1346.64 894.96,1346.21 895.68,1348.31 896.33,1348.96 897.16,1349.33 898.18,1349.43">\n    <request index="0" response="0000" foes="1000" cont="0" />\n    <request index="1" response="0000" foes="1000" cont="0" />\n    <request index="2" response="0000" foes="0000" cont="0" />\n    <request index="3" response="0011" foes="0011" cont="1" />\n  </junction>\n  <junction id="947896956" type="priority" x="478.47" y="1375.90" incLanes="706093506#3_0 706104217#18_0 -706095305_0 -706104217#19_0" intLanes=":947896956_0_0 :947896956_1_0 :947896956_9_0 :947896956_3_0 :947896956_4_0 :947896956_5_0 :947896956_10_0 :947896956_7_0 :947896956_8_0" shape="477.84,1368.26 481.01,1368.67 481.31,1371.05 481.94,1371.94 482.90,1372.61 484.19,1373.08 485.80,1373.34 485.23,1379.72 482.96,1379.96 482.12,1380.44 481.46,1381.16 481.00,1382.13 480.74,1383.33 474.39,1382.51 474.27,1379.99 473.85,1378.99 473.18,1378.18 472.27,1377.53 471.11,1377.07 472.92,1370.93 475.42,1371.21 476.35,1370.91 477.07,1370.32 477.56,1369.44">\n    <request index="0" response="000000000" foes="001001000" cont="0" />\n    <request index="1" response="000000000" foes="111011000" cont="0" />\n    <request index="2" response="000100000" foes="101111000" cont="1" />\n    <request index="3" response="001000111" foes="001000111" cont="0" />\n    <request index="4" response="111000110" foes="111000110" cont="0" />\n    <request index="5" response="000000000" foes="100000100" cont="0" />\n    <request index="6" response="000000111" foes="100011111" cont="1" />\n    <request index="7" response="000000010" foes="000010010" cont="0" />\n    <request index="8" response="001100110" foes="001110110" cont="0" />\n  </junction>\n  <junction id="947896958" type="priority" x="1574.42" y="1223.86" incLanes="66984613#6_0" intLanes=":947896958_0_0 :947896958_1_0" shape="1580.27,1222.23 1580.29,1225.43 1578.14,1225.88 1577.45,1226.43 1577.01,1227.20 1576.83,1228.19 1576.91,1229.40 1573.75,1229.90 1572.07,1218.75 1575.24,1218.29 1576.05,1220.49 1576.78,1221.26 1577.73,1221.80 1578.89,1222.12">\n    <request index="0" response="00" foes="00" cont="0" />\n    <request index="1" response="00" foes="00" cont="0" />\n  </junction>\n  <junction id="947896964" type="traffic_light" x="1018.99" y="1324.95" incLanes="81350005#0_0 -330615829#0_0 -706104217#1_0" intLanes=":947896964_0_0 :947896964_1_0 :947896964_2_0 :947896964_3_0 :947896964_4_0 :947896964_5_0" shape="1016.19,1318.27 1022.58,1318.66 1021.96,1333.01 1015.56,1332.83 1015.19,1330.69 1014.66,1330.01 1013.92,1329.60 1012.94,1329.45 1011.74,1329.57 1010.56,1323.28 1013.58,1322.27 1014.66,1321.52 1015.46,1320.60 1015.97,1319.52">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="000000" foes="111000" cont="0" />\n    <request index="2" response="000000" foes="100000" cont="0" />\n    <request index="3" response="000011" foes="100011" cont="0" />\n    <request index="4" response="000010" foes="000010" cont="0" />\n    <request index="5" response="001110" foes="001110" cont="0" />\n  </junction>\n  <junction id="947896972" type="priority" x="1629.94" y="1569.30" incLanes="243079723_0 66984613#0_0" intLanes=":947896972_0_0 :947896972_1_0 :947896972_2_0" shape="1636.43,1566.42 1636.11,1572.81 1633.96,1573.14 1633.28,1573.65 1632.85,1574.39 1632.68,1575.36 1632.77,1576.56 1629.62,1577.09 1627.23,1562.73 1630.39,1562.20 1631.35,1564.43 1632.23,1565.23 1633.37,1565.83 1634.77,1566.23">\n    <request index="0" response="100" foes="100" cont="0" />\n    <request index="1" response="000" foes="000" cont="0" />\n    <request index="2" response="000" foes="001" cont="0" />\n  </junction>\n  <junction id="947896979" type="priority" x="1613.67" y="1473.30" incLanes="66984613#2_0 -243079673_0" intLanes=":947896979_0_0 :947896979_1_0 :947896979_2_0" shape="1616.07,1477.46 1612.93,1478.05 1612.54,1477.25 1612.18,1476.98 1611.73,1476.80 1611.17,1476.72 1610.50,1476.72 1610.07,1470.33 1610.88,1470.11 1611.14,1469.89 1611.30,1469.59 1611.37,1469.21 1611.34,1468.75 1614.50,1468.26">\n    <request index="0" response="000" foes="100" cont="0" />\n    <request index="1" response="000" foes="100" cont="0" />\n    <request index="2" response="011" foes="011" cont="0" />\n  </junction>\n  <junction id="947896981" type="priority" x="1652.32" y="1703.39" incLanes="81350006#2_0 970323545_0 -95542480_0" intLanes=":947896981_0_0 :947896981_1_0 :947896981_2_0 :947896981_3_0 :947896981_4_0 :947896981_5_0 :947896981_7_0" shape="1657.28,1699.13 1658.55,1705.40 1656.32,1706.31 1655.61,1707.04 1655.17,1707.94 1654.99,1709.02 1655.08,1710.28 1651.92,1710.80 1651.12,1708.69 1650.45,1708.03 1649.60,1707.62 1648.56,1707.48 1647.34,1707.60 1646.13,1701.32 1648.33,1700.43 1649.03,1699.72 1649.47,1698.83 1649.64,1697.75 1649.55,1696.50 1652.70,1695.97 1653.51,1698.07 1654.18,1698.73 1655.03,1699.13 1656.07,1699.26">\n    <request index="0" response="0000000" foes="1001000" cont="0" />\n    <request index="1" response="0000000" foes="1011000" cont="0" />\n    <request index="2" response="0100000" foes="0100000" cont="0" />\n    <request index="3" response="1100011" foes="1100011" cont="0" />\n    <request index="4" response="1100010" foes="1100010" cont="0" />\n    <request index="5" response="0000000" foes="0011100" cont="0" />\n    <request index="6" response="0000011" foes="0011011" cont="1" />\n  </junction>\n  <junction id="947896989" type="priority" x="1617.54" y="1494.09" incLanes="-243314440_0 66984613#1_0 243079775_0" intLanes=":947896989_0_0 :947896989_1_0 :947896989_2_0 :947896989_3_0 :947896989_4_0 :947896989_5_0 :947896989_6_0" shape="1624.19,1491.38 1623.71,1497.76 1621.56,1498.03 1620.88,1498.52 1620.45,1499.25 1620.28,1500.21 1620.37,1501.41 1617.21,1501.93 1616.25,1499.40 1615.43,1498.54 1614.39,1497.94 1613.11,1497.61 1611.61,1497.55 1611.34,1491.15 1613.48,1490.62 1614.15,1490.04 1614.56,1489.25 1614.71,1488.26 1614.60,1487.05 1617.75,1486.46 1618.84,1489.01 1619.78,1489.94 1620.98,1490.65 1622.45,1491.13">\n    <request index="0" response="0001000" foes="1001000" cont="0" />\n    <request index="1" response="0011000" foes="1011000" cont="0" />\n    <request index="2" response="0000000" foes="0100000" cont="0" />\n    <request index="3" response="0000000" foes="1100011" cont="0" />\n    <request index="4" response="0000000" foes="1100010" cont="0" />\n    <request index="5" response="0011100" foes="0011100" cont="0" />\n    <request index="6" response="0011011" foes="0011011" cont="0" />\n  </junction>\n  <junction id="cluster_1107773211_1110416556" type="priority" x="762.78" y="516.21" incLanes="148941242#5_0 1454593202#2_0 706256383_0 -706256379#0_0 129827554#1_0" intLanes=":cluster_1107773211_1110416556_0_0 :cluster_1107773211_1110416556_1_0 :cluster_1107773211_1110416556_2_0 :cluster_1107773211_1110416556_3_0 :cluster_1107773211_1110416556_4_0 :cluster_1107773211_1110416556_5_0 :cluster_1107773211_1110416556_6_0 :cluster_1107773211_1110416556_7_0 :cluster_1107773211_1110416556_8_0 :cluster_1107773211_1110416556_9_0 :cluster_1107773211_1110416556_10_0 :cluster_1107773211_1110416556_16_0 :cluster_1107773211_1110416556_17_0 :cluster_1107773211_1110416556_13_0 :cluster_1107773211_1110416556_14_0 :cluster_1107773211_1110416556_15_0" shape="766.89,510.12 773.28,510.55 773.57,512.83 774.07,513.69 774.80,514.35 775.77,514.83 776.97,515.12 776.05,521.45 770.98,520.81 766.93,520.51 763.89,520.54 761.87,520.92 760.86,521.64 760.87,522.70 757.92,523.94 755.52,520.92 753.66,519.82 751.34,518.99 748.58,518.44 745.38,518.17 745.37,511.76 747.31,511.50 747.75,511.09 747.82,510.50 747.54,509.73 746.90,508.77 751.93,504.83 754.89,508.24 758.10,511.24 761.23,513.37 763.96,514.20 765.96,513.26">\n    <request index="0" response="0001000100010000" foes="0001000100010000" cont="0" />\n    <request index="1" response="1111000100110000" foes="1111001100110000" cont="0" />\n    <request index="2" response="1100110011110000" foes="1100110011110000" cont="0" />\n    <request index="3" response="0000000000000000" foes="1000100010000000" cont="0" />\n    <request index="4" response="0000000000000000" foes="1001100110000111" cont="0" />\n    <request index="5" response="0000000000000000" foes="0111101110000110" cont="0" />\n    <request index="6" response="0100010000000000" foes="0100010000000100" cont="0" />\n    <request index="7" response="0100110000111000" foes="1100110000111100" cont="0" />\n    <request index="8" response="0001110000110000" foes="1101110000110011" cont="0" />\n    <request index="9" response="0011110000100010" foes="1011110000100010" cont="0" />\n    <request index="10" response="0000000000000000" foes="0100001111000100" cont="0" />\n    <request index="11" response="0000000000111000" foes="1100001110111100" cont="1" />\n    <request index="12" response="0000000000110000" foes="1100001100110011" cont="1" />\n    <request index="13" response="0000000000100000" foes="0000001000100010" cont="0" />\n    <request index="14" response="0001110100100000" foes="0001110111100110" cont="0" />\n    <request index="15" response="0001101110011000" foes="0001101110011110" cont="0" />\n    <param key="origEdgeIds" value="-1454593202#3 1454593202#3" />\n    <param key="origId" value="1107773211 1110416556" />\n  </junction>\n  <junction id="cluster_1110416379_1680666461" type="priority" x="930.91" y="358.54" incLanes="706256380#1_0 -155690981_0 -706256380#3_0 -95809262#0_0" intLanes=":cluster_1110416379_1680666461_0_0 :cluster_1110416379_1680666461_1_0 :cluster_1110416379_1680666461_12_0 :cluster_1110416379_1680666461_3_0 :cluster_1110416379_1680666461_4_0 :cluster_1110416379_1680666461_5_0 :cluster_1110416379_1680666461_6_0 :cluster_1110416379_1680666461_7_0 :cluster_1110416379_1680666461_13_0 :cluster_1110416379_1680666461_9_0 :cluster_1110416379_1680666461_10_0 :cluster_1110416379_1680666461_11_0" shape="929.92,341.09 936.26,342.00 935.33,349.18 934.82,355.07 934.73,359.67 935.06,362.97 935.80,364.99 936.97,365.71 936.59,372.10 934.33,372.42 933.49,372.92 932.85,373.66 932.41,374.64 932.17,375.85 925.80,375.15 926.49,367.87 926.81,361.89 926.76,357.20 926.35,353.81 925.57,351.71 924.43,350.91 925.18,344.55 927.56,344.39 928.45,343.93 929.14,343.23 929.63,342.28">\n    <request index="0" response="000000000000" foes="000100010000" cont="0" />\n    <request index="1" response="000000000000" foes="111100110000" cont="0" />\n    <request index="2" response="000011000000" foes="110111110000" cont="1" />\n    <request index="3" response="000010000000" foes="100010000000" cont="0" />\n    <request index="4" response="000110000111" foes="100110000111" cont="0" />\n    <request index="5" response="011110000110" foes="011110000110" cont="0" />\n    <request index="6" response="000000000000" foes="010000000100" cont="0" />\n    <request index="7" response="000000000000" foes="110000111100" cont="0" />\n    <request index="8" response="000000000111" foes="110000110111" cont="1" />\n    <request index="9" response="000000000010" foes="000000100010" cont="0" />\n    <request index="10" response="000111000110" foes="000111100110" cont="0" />\n    <request index="11" response="000110011110" foes="000110011110" cont="0" />\n    <param key="origEdgeIds" value="-706256380#2 706256380#2" />\n    <param key="origId" value="1110416379 1680666461" />\n  </junction>\n  <junction id="cluster_1115585393_13472440789_947896971" type="traffic_light" x="1036.31" y="1162.26" incLanes="498671395#1_0 706104219#0_0 -81350005#2_0 -706104219#4_0" intLanes=":cluster_1115585393_13472440789_947896971_0_0 :cluster_1115585393_13472440789_947896971_1_0 :cluster_1115585393_13472440789_947896971_2_0 :cluster_1115585393_13472440789_947896971_3_0 :cluster_1115585393_13472440789_947896971_4_0 :cluster_1115585393_13472440789_947896971_5_0 :cluster_1115585393_13472440789_947896971_6_0 :cluster_1115585393_13472440789_947896971_7_0 :cluster_1115585393_13472440789_947896971_8_0 :cluster_1115585393_13472440789_947896971_9_0 :cluster_1115585393_13472440789_947896971_10_0 :cluster_1115585393_13472440789_947896971_11_0" shape="1024.41,1151.02 1053.38,1156.31 1053.42,1158.53 1053.83,1159.35 1054.50,1159.96 1055.43,1160.38 1056.63,1160.60 1056.05,1166.98 1049.76,1166.53 1044.58,1166.44 1040.52,1166.71 1037.57,1167.34 1035.74,1168.33 1035.02,1169.68 1028.65,1169.00 1027.86,1166.70 1026.65,1165.81 1024.89,1165.10 1022.60,1164.57 1019.77,1164.21 1020.32,1157.84 1022.55,1157.23 1023.34,1156.31 1023.91,1154.96 1024.27,1153.20">\n    <request index="0" response="000000010000" foes="000100010000" cont="0" />\n    <request index="1" response="111000110000" foes="111100110000" cont="0" />\n    <request index="2" response="110011110000" foes="110011110000" cont="0" />\n    <request index="3" response="000000000000" foes="100010000000" cont="0" />\n    <request index="4" response="000000000000" foes="100110000111" cont="0" />\n    <request index="5" response="011000000000" foes="011110000110" cont="0" />\n    <request index="6" response="010000000000" foes="010000000100" cont="0" />\n    <request index="7" response="110000111000" foes="110000111100" cont="0" />\n    <request index="8" response="110000110011" foes="110000110011" cont="0" />\n    <request index="9" response="000000000000" foes="000000100010" cont="0" />\n    <request index="10" response="000000000000" foes="000111100110" cont="0" />\n    <request index="11" response="000000011000" foes="000110011110" cont="0" />\n    <param key="origEdgeIds" value="-706104219#1 -706104219#2 706104219#1 706104219#2" />\n    <param key="origId" value="1115585393 13472440789 947896971" />\n  </junction>\n  <junction id="cluster_1122072658_14138648129" type="priority" x="704.33" y="511.87" incLanes="706256379#1_0 154707523#1_0 -706256379#3_0 -96837170#2_0" intLanes=":cluster_1122072658_14138648129_0_0 :cluster_1122072658_14138648129_1_0 :cluster_1122072658_14138648129_2_0 :cluster_1122072658_14138648129_3_0 :cluster_1122072658_14138648129_4_0 :cluster_1122072658_14138648129_5_0 :cluster_1122072658_14138648129_9_0 :cluster_1122072658_14138648129_7_0 :cluster_1122072658_14138648129_8_0" shape="716.02,509.68 715.45,516.05 713.18,516.31 712.33,516.81 711.67,517.57 711.20,518.57 710.92,519.81 707.74,519.39 705.85,517.00 703.22,515.99 699.46,515.10 694.58,514.34 688.57,513.71 689.13,507.34 691.15,507.12 691.69,506.69 691.92,506.03 691.82,505.16 691.42,504.06 697.21,501.35 698.64,503.51 700.75,505.36 703.54,506.90 707.02,508.14 711.18,509.06">\n    <request index="0" response="000000000" foes="001001000" cont="0" />\n    <request index="1" response="000000000" foes="111011000" cont="0" />\n    <request index="2" response="000100000" foes="100100000" cont="0" />\n    <request index="3" response="001100011" foes="101100011" cont="0" />\n    <request index="4" response="011100010" foes="111100010" cont="0" />\n    <request index="5" response="000000000" foes="100011100" cont="0" />\n    <request index="6" response="000000011" foes="100011011" cont="1" />\n    <request index="7" response="000000010" foes="000010010" cont="0" />\n    <request index="8" response="001111110" foes="001111110" cont="0" />\n    <param key="origEdgeIds" value="-706256379#2 706256379#2" />\n    <param key="origId" value="1122072658 14138648129" />\n  </junction>\n  <junction id="cluster_1123399587_8976429007" type="priority" x="1101.34" y="710.15" incLanes="970010765#1_0 -969765780#1_0 969757851#0_0" intLanes=":cluster_1123399587_8976429007_0_0 :cluster_1123399587_8976429007_6_0 :cluster_1123399587_8976429007_2_0 :cluster_1123399587_8976429007_3_0 :cluster_1123399587_8976429007_4_0 :cluster_1123399587_8976429007_5_0" shape="1108.62,706.64 1107.78,715.03 1104.59,714.94 1102.26,715.38 1100.43,716.29 1098.70,717.61 1096.70,719.30 1094.03,721.29 1090.43,716.00 1091.82,714.71 1091.91,714.23 1091.60,713.85 1090.88,713.58 1089.75,713.41 1090.61,704.83">\n    <request index="0" response="000000" foes="001000" cont="0" />\n    <request index="1" response="110000" foes="111000" cont="1" />\n    <request index="2" response="100000" foes="100000" cont="0" />\n    <request index="3" response="100011" foes="100011" cont="0" />\n    <request index="4" response="000000" foes="000010" cont="0" />\n    <request index="5" response="000000" foes="001110" cont="0" />\n    <param key="origEdgeIds" value="-969765780#0 969765780#0" />\n    <param key="origId" value="1123399587 8976429007" />\n  </junction>\n  <junction id="cluster_1123399715_3376084911_3376091818" type="priority" x="1016.26" y="1391.74" incLanes="330615844_0 684740716_0 1257722736#0_0" intLanes=":cluster_1123399715_3376084911_3376091818_0_0 :cluster_1123399715_3376084911_3376091818_1_0 :cluster_1123399715_3376084911_3376091818_2_0 :cluster_1123399715_3376084911_3376091818_3_0" shape="1013.55,1382.73 1019.95,1383.07 1020.04,1386.01 1020.67,1388.49 1021.83,1390.50 1023.52,1392.06 1025.73,1393.14 1028.48,1393.77 1027.66,1400.11 1021.19,1399.42 1015.79,1399.18 1011.46,1399.38 1008.19,1400.02 1005.99,1401.10 1004.86,1402.63 998.81,1400.54 999.09,1398.42 998.79,1397.65 998.19,1397.08 997.30,1396.71 996.12,1396.53 996.30,1393.34 1001.40,1393.31 1005.61,1392.58 1008.94,1391.16 1011.37,1389.05 1012.91,1386.24">\n    <request index="0" response="0000" foes="0100" cont="0" />\n    <request index="1" response="0000" foes="0000" cont="0" />\n    <request index="2" response="0001" foes="0001" cont="0" />\n    <request index="3" response="0000" foes="0000" cont="0" />\n    <param key="origEdgeIds" value="1554703880 330615834 330615837" />\n    <param key="origId" value="1123399715 3376084911 3376091818" />\n  </junction>\n  <junction id="cluster_11292971032_2406954578_2406954579_2406954580" type="priority" x="1545.59" y="1174.33" incLanes="66984604#2_0 970323546#7_0 970323546#7_1 1227721991#0_0" intLanes=":cluster_11292971032_2406954578_2406954579_2406954580_0_0 :cluster_11292971032_2406954578_2406954579_2406954580_1_0 :cluster_11292971032_2406954578_2406954579_2406954580_2_0 :cluster_11292971032_2406954578_2406954579_2406954580_3_0 :cluster_11292971032_2406954578_2406954579_2406954580_3_1 :cluster_11292971032_2406954578_2406954579_2406954580_5_0 :cluster_11292971032_2406954578_2406954579_2406954580_7_0" shape="1557.37,1164.51 1560.13,1179.71 1557.88,1180.57 1557.16,1181.27 1556.70,1182.15 1556.50,1183.20 1556.57,1184.43 1539.92,1187.27 1536.25,1164.04 1552.79,1161.31 1553.59,1163.42 1554.26,1164.08 1555.12,1164.48 1556.15,1164.63">\n    <request index="0" response="0011000" foes="0011000" cont="0" />\n    <request index="1" response="1111000" foes="1111000" cont="0" />\n    <request index="2" response="0000000" foes="1000000" cont="0" />\n    <request index="3" response="0000000" foes="1000011" cont="0" />\n    <request index="4" response="0000000" foes="1000011" cont="0" />\n    <request index="5" response="0000000" foes="0000010" cont="0" />\n    <request index="6" response="0011100" foes="0011110" cont="1" />\n    <param key="origEdgeIds" value="1218225520 1218225521 1218225523 1218225524#0" />\n    <param key="origId" value="11292971032 2406954578 2406954579 2406954580" />\n  </junction>\n  <junction id="cluster_13424220808_2506085647" type="priority" x="663.11" y="899.91" incLanes="-243188465#0_0 931354569#1_0 917696883_0 -931354569#3_0" intLanes=":cluster_13424220808_2506085647_0_0 :cluster_13424220808_2506085647_1_0 :cluster_13424220808_2506085647_2_0 :cluster_13424220808_2506085647_3_0 :cluster_13424220808_2506085647_4_0 :cluster_13424220808_2506085647_5_0 :cluster_13424220808_2506085647_6_0 :cluster_13424220808_2506085647_7_0 :cluster_13424220808_2506085647_9_0" shape="657.90,890.58 664.14,892.03 664.70,894.93 665.93,896.30 667.79,897.62 670.28,898.89 673.40,900.11 671.31,906.15 669.09,905.86 668.21,906.14 667.47,906.70 666.88,907.56 666.45,908.70 663.37,907.83 662.55,905.01 660.93,903.60 658.51,902.19 655.28,900.78 651.24,899.37 653.18,893.27 655.40,893.51 656.27,893.20 656.97,892.61 657.52,891.74">\n    <request index="0" response="000001000" foes="001001000" cont="0" />\n    <request index="1" response="110111000" foes="110111000" cont="0" />\n    <request index="2" response="000000000" foes="100100000" cont="0" />\n    <request index="3" response="000000000" foes="101100011" cont="0" />\n    <request index="4" response="010000000" foes="010000010" cont="0" />\n    <request index="5" response="110001100" foes="110001110" cont="0" />\n    <request index="6" response="110001001" foes="110001001" cont="0" />\n    <request index="7" response="000000000" foes="001110010" cont="0" />\n    <request index="8" response="000001100" foes="001101110" cont="1" />\n    <param key="origEdgeIds" value="-931354569#2 931354569#2" />\n    <param key="origId" value="13424220808 2506085647" />\n  </junction>\n  <junction id="cluster_13472440775_1666900747" type="priority" x="1335.16" y="1187.84" incLanes="-706105951#0_0 72016745#2_0 72016745#2_1 95542485#0_0 -72016745#4_0 -72016745#4_1" intLanes=":cluster_13472440775_1666900747_0_0 :cluster_13472440775_1666900747_1_0 :cluster_13472440775_1666900747_2_0 :cluster_13472440775_1666900747_3_0 :cluster_13472440775_1666900747_4_0 :cluster_13472440775_1666900747_4_1 :cluster_13472440775_1666900747_14_0 :cluster_13472440775_1666900747_7_0 :cluster_13472440775_1666900747_8_0 :cluster_13472440775_1666900747_9_0 :cluster_13472440775_1666900747_10_0 :cluster_13472440775_1666900747_11_0 :cluster_13472440775_1666900747_11_1 :cluster_13472440775_1666900747_15_0" shape="1330.31,1177.04 1336.70,1177.24 1337.58,1179.71 1338.75,1180.65 1340.40,1181.39 1342.54,1181.94 1345.16,1182.30 1344.00,1195.05 1341.76,1195.29 1340.96,1195.78 1340.37,1196.50 1339.98,1197.47 1339.81,1198.68 1333.42,1198.35 1332.60,1195.94 1331.47,1195.03 1329.86,1194.32 1327.77,1193.80 1325.19,1193.49 1326.19,1180.72 1328.42,1180.45 1329.22,1179.96 1329.79,1179.22 1330.16,1178.25">\n    <request index="0" response="00000000110000" foes="00000000110000" cont="0" />\n    <request index="1" response="11110001110000" foes="11111001110000" cont="0" />\n    <request index="2" response="11101101110000" foes="11101101110000" cont="0" />\n    <request index="3" response="00000000000000" foes="10000100000000" cont="0" />\n    <request index="4" response="00000000000000" foes="10001100000111" cont="0" />\n    <request index="5" response="00000000000000" foes="10001100000111" cont="0" />\n    <request index="6" response="01110000000000" foes="01111100000110" cont="1" />\n    <request index="7" response="01100000000000" foes="01100000000000" cont="0" />\n    <request index="8" response="11100001111000" foes="11100001111100" cont="0" />\n    <request index="9" response="11100001110010" foes="11100001110110" cont="0" />\n    <request index="10" response="00000000000000" foes="00000001000010" cont="0" />\n    <request index="11" response="00000000000000" foes="00001111000110" cont="0" />\n    <request index="12" response="00000000000000" foes="00001111000110" cont="0" />\n    <request index="13" response="00000000111000" foes="00001100111110" cont="1" />\n    <param key="origEdgeIds" value="-72016745#3 72016745#3" />\n    <param key="origId" value="13472440775 1666900747" />\n  </junction>\n  <junction id="cluster_14188374630_2506088509" type="priority" x="1204.85" y="1506.56" incLanes="243188212#1_0 -243188508#3_0 1559802496_0 243188508#0_0" intLanes=":cluster_14188374630_2506088509_0_0 :cluster_14188374630_2506088509_1_0 :cluster_14188374630_2506088509_2_0 :cluster_14188374630_2506088509_3_0 :cluster_14188374630_2506088509_4_0 :cluster_14188374630_2506088509_12_0 :cluster_14188374630_2506088509_6_0 :cluster_14188374630_2506088509_7_0 :cluster_14188374630_2506088509_8_0 :cluster_14188374630_2506088509_9_0 :cluster_14188374630_2506088509_10_0 :cluster_14188374630_2506088509_13_0" shape="1195.08,1499.52 1201.48,1499.73 1203.29,1501.75 1205.62,1502.27 1208.90,1502.47 1213.12,1502.34 1218.30,1501.89 1218.99,1508.25 1216.81,1509.00 1216.07,1509.73 1215.56,1510.68 1215.29,1511.87 1215.25,1513.29 1208.86,1513.62 1206.84,1511.56 1204.41,1510.98 1201.04,1510.71 1196.73,1510.73 1191.47,1511.06 1190.94,1504.68 1193.17,1503.96 1193.97,1503.22 1194.55,1502.24 1194.92,1501.01">\n    <request index="0" response="000000010000" foes="000100010000" cont="0" />\n    <request index="1" response="111000110000" foes="111100110000" cont="0" />\n    <request index="2" response="110011110000" foes="110011110000" cont="0" />\n    <request index="3" response="000000000000" foes="100010000000" cont="0" />\n    <request index="4" response="000000000000" foes="100110000111" cont="0" />\n    <request index="5" response="011000000000" foes="011110000110" cont="1" />\n    <request index="6" response="010000000000" foes="010000000100" cont="0" />\n    <request index="7" response="110000111000" foes="110000111100" cont="0" />\n    <request index="8" response="110000110011" foes="110000110011" cont="0" />\n    <request index="9" response="000000000000" foes="000000100010" cont="0" />\n    <request index="10" response="000000000000" foes="000111100110" cont="0" />\n    <request index="11" response="000000011000" foes="000110011110" cont="1" />\n    <param key="origEdgeIds" value="-243188508#2 243188508#2" />\n    <param key="origId" value="14188374630 2506088509" />\n  </junction>\n  <junction id="cluster_1558314142_1558314144" type="priority" x="1376.62" y="706.97" incLanes="-498671497_0 -155690513_0 706094740#0_0 -706094740#2_0" intLanes=":cluster_1558314142_1558314144_0_0 :cluster_1558314142_1558314144_1_0 :cluster_1558314142_1558314144_12_0 :cluster_1558314142_1558314144_3_0 :cluster_1558314142_1558314144_4_0 :cluster_1558314142_1558314144_5_0 :cluster_1558314142_1558314144_6_0 :cluster_1558314142_1558314144_7_0 :cluster_1558314142_1558314144_13_0 :cluster_1558314142_1558314144_9_0 :cluster_1558314142_1558314144_10_0 :cluster_1558314142_1558314144_11_0" shape="1364.16,696.81 1370.52,697.46 1370.92,700.22 1372.70,702.51 1375.87,704.33 1380.43,705.68 1386.39,706.56 1393.73,706.98 1393.59,713.38 1391.34,713.77 1390.52,714.31 1389.92,715.07 1389.52,716.06 1389.32,717.28 1382.94,716.85 1382.39,713.19 1380.30,710.54 1376.83,708.78 1372.16,707.74 1366.46,707.29 1359.91,707.28 1359.75,700.88 1362.02,700.39 1362.85,699.82 1363.49,699.03 1363.92,698.03">\n    <request index="0" response="000000000000" foes="000100010000" cont="0" />\n    <request index="1" response="000000000000" foes="111100110000" cont="0" />\n    <request index="2" response="000111000000" foes="110111110000" cont="1" />\n    <request index="3" response="000010000000" foes="100010000000" cont="0" />\n    <request index="4" response="000110000111" foes="100110000111" cont="0" />\n    <request index="5" response="011110000110" foes="011110000110" cont="0" />\n    <request index="6" response="000000000000" foes="010000000100" cont="0" />\n    <request index="7" response="000000000000" foes="110000111100" cont="0" />\n    <request index="8" response="000000000011" foes="110000110111" cont="1" />\n    <request index="9" response="000000000010" foes="000000100010" cont="0" />\n    <request index="10" response="000111000110" foes="000111100110" cont="0" />\n    <request index="11" response="000110011110" foes="000110011110" cont="0" />\n    <param key="origEdgeIds" value="-706094740#1 706094740#1" />\n    <param key="origId" value="1558314142 1558314144" />\n  </junction>\n  <junction id="cluster_1559016910_4900465210" type="priority" x="871.27" y="420.38" incLanes="-142455600#2_0 -498671483_0 142455600#0_0 142455590_0" intLanes=":cluster_1559016910_4900465210_0_0 :cluster_1559016910_4900465210_1_0 :cluster_1559016910_4900465210_12_0 :cluster_1559016910_4900465210_3_0 :cluster_1559016910_4900465210_4_0 :cluster_1559016910_4900465210_5_0 :cluster_1559016910_4900465210_6_0 :cluster_1559016910_4900465210_7_0 :cluster_1559016910_4900465210_13_0 :cluster_1559016910_4900465210_9_0 :cluster_1559016910_4900465210_10_0 :cluster_1559016910_4900465210_11_0" shape="869.90,402.45 876.26,403.17 876.47,405.43 876.95,406.25 877.68,406.88 878.67,407.31 879.91,407.55 879.30,413.92 876.80,416.45 875.66,419.80 874.60,424.55 873.62,430.69 872.72,438.23 866.35,437.56 866.11,435.30 865.59,434.47 864.80,433.83 863.76,433.38 862.45,433.14 863.10,426.77 865.63,424.25 866.79,420.90 867.89,416.14 868.93,410.00">\n    <request index="0" response="000000000000" foes="000100010000" cont="0" />\n    <request index="1" response="000000000000" foes="111100110000" cont="0" />\n    <request index="2" response="000011000000" foes="110011110000" cont="1" />\n    <request index="3" response="000010000000" foes="100010000000" cont="0" />\n    <request index="4" response="000110000111" foes="100110000111" cont="0" />\n    <request index="5" response="011110000110" foes="011110000110" cont="0" />\n    <request index="6" response="000000000000" foes="010000000100" cont="0" />\n    <request index="7" response="000000000000" foes="110000111100" cont="0" />\n    <request index="8" response="000000000011" foes="110000110011" cont="1" />\n    <request index="9" response="000000000010" foes="000000100010" cont="0" />\n    <request index="10" response="000111000110" foes="000111100110" cont="0" />\n    <request index="11" response="000110011110" foes="000110011110" cont="0" />\n    <param key="origEdgeIds" value="-142455600#1 142455600#1" />\n    <param key="origId" value="1559016910 4900465210" />\n  </junction>\n  <junction id="cluster_1559016919_1680666950" type="priority" x="915.77" y="493.02" incLanes="706256380#4_0 -155690878#0_0 -706256380#6_0 706256386_0" intLanes=":cluster_1559016919_1680666950_0_0 :cluster_1559016919_1680666950_1_0 :cluster_1559016919_1680666950_12_0 :cluster_1559016919_1680666950_3_0 :cluster_1559016919_1680666950_4_0 :cluster_1559016919_1680666950_5_0 :cluster_1559016919_1680666950_6_0 :cluster_1559016919_1680666950_7_0 :cluster_1559016919_1680666950_13_0 :cluster_1559016919_1680666950_9_0 :cluster_1559016919_1680666950_10_0 :cluster_1559016919_1680666950_11_0" shape="914.39,474.99 920.74,475.73 919.99,483.34 919.63,489.58 919.68,494.47 920.13,498.00 920.98,500.17 922.24,500.98 921.67,507.36 919.41,507.60 918.58,508.09 917.96,508.81 917.53,509.78 917.30,510.98 910.93,510.38 911.55,502.71 911.80,496.42 911.70,491.48 911.24,487.91 910.41,485.71 909.23,484.87 909.95,478.52 912.21,478.33 913.05,477.86 913.69,477.15 914.14,476.20">\n    <request index="0" response="000000000000" foes="000100010000" cont="0" />\n    <request index="1" response="000000000000" foes="111100110000" cont="0" />\n    <request index="2" response="000011000000" foes="110111110000" cont="1" />\n    <request index="3" response="000010000000" foes="100010000000" cont="0" />\n    <request index="4" response="000110000111" foes="100110000111" cont="0" />\n    <request index="5" response="011110000110" foes="011110000110" cont="0" />\n    <request index="6" response="000000000000" foes="010000000100" cont="0" />\n    <request index="7" response="000000000000" foes="110000111100" cont="0" />\n    <request index="8" response="000000000111" foes="110000110111" cont="1" />\n    <request index="9" response="000000000010" foes="000000100010" cont="0" />\n    <request index="10" response="000111000110" foes="000111100110" cont="0" />\n    <request index="11" response="000110011110" foes="000110011110" cont="0" />\n    <param key="origEdgeIds" value="-706256380#5 706256380#5" />\n    <param key="origId" value="1559016919 1680666950" />\n  </junction>\n  <junction id="cluster_1680667233_2506085475" type="priority" x="1227.09" y="562.94" incLanes="155690875_0 95533432#1_0 -243188458_0 -95533432#3_0" intLanes=":cluster_1680667233_2506085475_0_0 :cluster_1680667233_2506085475_1_0 :cluster_1680667233_2506085475_2_0 :cluster_1680667233_2506085475_3_0 :cluster_1680667233_2506085475_4_0 :cluster_1680667233_2506085475_12_0 :cluster_1680667233_2506085475_6_0 :cluster_1680667233_2506085475_7_0 :cluster_1680667233_2506085475_8_0 :cluster_1680667233_2506085475_9_0 :cluster_1680667233_2506085475_10_0 :cluster_1680667233_2506085475_13_0" shape="1217.04,554.71 1223.39,555.55 1225.22,557.93 1227.77,558.91 1231.41,559.77 1236.14,560.49 1241.96,561.08 1241.43,567.46 1239.11,567.79 1238.20,568.37 1237.46,569.22 1236.89,570.35 1236.49,571.76 1230.22,570.45 1228.55,568.07 1226.05,567.04 1222.44,566.13 1217.73,565.33 1211.91,564.65 1212.53,558.28 1214.80,558.06 1215.65,557.58 1216.31,556.87 1216.77,555.91">\n    <request index="0" response="000000010000" foes="000100010000" cont="0" />\n    <request index="1" response="111000110000" foes="111100110000" cont="0" />\n    <request index="2" response="110011110000" foes="110111110000" cont="0" />\n    <request index="3" response="000000000000" foes="100010000000" cont="0" />\n    <request index="4" response="000000000000" foes="100110000111" cont="0" />\n    <request index="5" response="011000000000" foes="011110000110" cont="1" />\n    <request index="6" response="010000000000" foes="010000000100" cont="0" />\n    <request index="7" response="110000111000" foes="110000111100" cont="0" />\n    <request index="8" response="110000110111" foes="110000110111" cont="0" />\n    <request index="9" response="000000000000" foes="000000100010" cont="0" />\n    <request index="10" response="000000000000" foes="000111100110" cont="0" />\n    <request index="11" response="000000011000" foes="000110011110" cont="1" />\n    <param key="origEdgeIds" value="-95533432#2 95533432#2" />\n    <param key="origId" value="1680667233 2506085475" />\n  </junction>\n  <junction id="cluster_1966399241_8523333972" type="priority" x="727.27" y="1132.86" incLanes="-917696759_0 706104219#9_0 -706104219#11_0" intLanes=":cluster_1966399241_8523333972_0_0 :cluster_1966399241_8523333972_1_0 :cluster_1966399241_8523333972_2_0 :cluster_1966399241_8523333972_3_0 :cluster_1966399241_8523333972_4_0 :cluster_1966399241_8523333972_9_0 :cluster_1966399241_8523333972_6_0 :cluster_1966399241_8523333972_7_0 :cluster_1966399241_8523333972_10_0" shape="730.17,1125.39 736.49,1126.40 736.59,1128.63 737.02,1129.44 737.71,1130.06 738.65,1130.47 739.85,1130.67 739.33,1137.05 734.66,1136.79 730.85,1136.81 727.88,1137.12 725.78,1137.73 724.52,1138.62 724.12,1139.81 720.92,1139.87 720.43,1137.43 719.86,1136.55 719.07,1135.88 718.07,1135.44 716.85,1135.22 717.37,1128.84 721.09,1129.02 724.17,1128.90 726.62,1128.47 728.44,1127.75 729.62,1126.72">\n    <request index="0" response="000010000" foes="000010000" cont="0" />\n    <request index="1" response="111110000" foes="111110000" cont="0" />\n    <request index="2" response="110110000" foes="110110000" cont="0" />\n    <request index="3" response="000000000" foes="100000000" cont="0" />\n    <request index="4" response="000000000" foes="100000111" cont="0" />\n    <request index="5" response="011000000" foes="011000110" cont="1" />\n    <request index="6" response="000000000" foes="000100010" cont="0" />\n    <request index="7" response="000000000" foes="000100110" cont="0" />\n    <request index="8" response="000011000" foes="000011110" cont="1" />\n    <param key="origEdgeIds" value="-706104219#10 706104219#10" />\n    <param key="origId" value="1966399241 8523333972" />\n  </junction>\n  <junction id="cluster_2052547073_2506085499" type="priority" x="703.91" y="672.35" incLanes="243188465#1_0 40264697#7_0 40264697#7_1" intLanes=":cluster_2052547073_2506085499_0_0 :cluster_2052547073_2506085499_1_0 :cluster_2052547073_2506085499_2_0 :cluster_2052547073_2506085499_2_1 :cluster_2052547073_2506085499_4_0" shape="693.41,663.45 696.61,663.63 699.09,666.92 702.36,668.30 706.97,669.51 712.93,670.54 720.23,671.40 719.62,677.77 717.29,677.99 716.38,678.45 715.63,679.15 715.04,680.09 714.62,681.26 708.43,679.63 706.29,676.57 703.22,675.50 699.12,674.66 694.20,674.01 688.70,673.52 689.19,667.13 691.43,666.86 692.24,666.37 692.84,665.63 693.23,664.66">\n    <request index="0" response="01100" foes="01100" cont="0" />\n    <request index="1" response="00000" foes="00000" cont="0" />\n    <request index="2" response="00000" foes="00001" cont="0" />\n    <request index="3" response="00000" foes="00001" cont="0" />\n    <request index="4" response="00000" foes="00000" cont="0" />\n    <param key="origEdgeIds" value="40264697#8" />\n    <param key="origId" value="2052547073 2506085499" />\n  </junction>\n  <junction id="cluster_2505176169_856214432" type="priority" x="1515.98" y="993.29" incLanes="1218225518#1_0 1218225518#1_1 -1270957863_0 1270957860#0_0" intLanes=":cluster_2505176169_856214432_0_0 :cluster_2505176169_856214432_0_1 :cluster_2505176169_856214432_7_0 :cluster_2505176169_856214432_3_0 :cluster_2505176169_856214432_4_0 :cluster_2505176169_856214432_5_0 :cluster_2505176169_856214432_6_0" shape="1525.88,997.80 1509.51,1000.29 1508.60,998.07 1507.76,997.28 1506.65,996.69 1505.30,996.32 1503.68,996.16 1503.90,989.76 1506.05,989.40 1506.74,988.87 1507.17,988.13 1507.34,987.15 1507.25,985.95 1523.51,983.29">\n    <request index="0" response="0000000" foes="0010000" cont="0" />\n    <request index="1" response="0000000" foes="0010000" cont="0" />\n    <request index="2" response="1100000" foes="1110000" cont="1" />\n    <request index="3" response="1000000" foes="1000000" cont="0" />\n    <request index="4" response="1000111" foes="1000111" cont="0" />\n    <request index="5" response="0000000" foes="0000100" cont="0" />\n    <request index="6" response="0000000" foes="0011100" cont="0" />\n    <param key="origEdgeIds" value="-1227721990 1227721990" />\n    <param key="origId" value="2505176169 856214432" />\n  </junction>\n  <junction id="cluster_2505176422_2505176434" type="priority" x="1586.68" y="1426.72" incLanes="243314186_0 970323546#3_0 970323546#3_1 65080358#0_0" intLanes=":cluster_2505176422_2505176434_0_0 :cluster_2505176422_2505176434_1_0 :cluster_2505176422_2505176434_2_0 :cluster_2505176422_2505176434_3_0 :cluster_2505176422_2505176434_3_1 :cluster_2505176422_2505176434_5_0 :cluster_2505176422_2505176434_7_0" shape="1599.56,1421.95 1600.27,1428.31 1598.13,1428.99 1597.45,1429.63 1597.02,1430.45 1596.85,1431.48 1596.94,1432.70 1580.42,1435.39 1578.11,1421.17 1594.61,1418.48 1595.45,1420.62 1596.17,1421.32 1597.10,1421.77 1598.23,1421.98">\n    <request index="0" response="0011000" foes="0011000" cont="0" />\n    <request index="1" response="1111000" foes="1111000" cont="0" />\n    <request index="2" response="0000000" foes="1000000" cont="0" />\n    <request index="3" response="0000000" foes="1000011" cont="0" />\n    <request index="4" response="0000000" foes="1000011" cont="0" />\n    <request index="5" response="0000000" foes="0000010" cont="0" />\n    <request index="6" response="0011100" foes="0011110" cont="1" />\n    <param key="origEdgeIds" value="-243314123 243314123" />\n    <param key="origId" value="2505176422 2505176434" />\n  </junction>\n  <junction id="cluster_2506087462_8523307708" type="priority" x="307.26" y="1233.11" incLanes="243188646#1_0 706095308#1_0 84842974#0_0 84842974#0_1 84842974#0_2" intLanes=":cluster_2506087462_8523307708_0_0 :cluster_2506087462_8523307708_1_0 :cluster_2506087462_8523307708_2_0 :cluster_2506087462_8523307708_2_1 :cluster_2506087462_8523307708_2_2 :cluster_2506087462_8523307708_8_0 :cluster_2506087462_8523307708_5_1 :cluster_2506087462_8523307708_5_2" shape="300.94,1223.53 310.41,1225.14 310.32,1227.23 310.83,1229.05 311.93,1230.58 313.63,1231.84 315.91,1232.81 318.79,1233.51 318.24,1236.66 316.19,1236.70 315.60,1237.10 315.28,1237.74 315.25,1238.64 315.50,1239.78 309.38,1241.67 307.30,1239.36 305.31,1238.45 302.69,1237.71 299.43,1237.12 295.54,1236.70 296.31,1227.14 298.59,1226.88 299.45,1226.39 300.13,1225.67 300.63,1224.72">\n    <request index="0" response="00000000" foes="11111100" cont="0" />\n    <request index="1" response="00000000" foes="00011100" cont="0" />\n    <request index="2" response="11100011" foes="11100011" cont="0" />\n    <request index="3" response="11100011" foes="11100011" cont="0" />\n    <request index="4" response="11100011" foes="11100011" cont="0" />\n    <request index="5" response="00000001" foes="00011101" cont="1" />\n    <request index="6" response="00000001" foes="00011101" cont="0" />\n    <request index="7" response="00000001" foes="00011101" cont="0" />\n    <param key="origEdgeIds" value="243188646#2" />\n    <param key="origId" value="2506087462 8523307708" />\n  </junction>\n  <junction id="cluster_2506088039_8523406713" type="priority" x="642.91" y="1395.46" incLanes="95809261#2_0 706104217#14_0 -917704692_0 -706104217#16_0" intLanes=":cluster_2506088039_8523406713_0_0 :cluster_2506088039_8523406713_1_0 :cluster_2506088039_8523406713_2_0 :cluster_2506088039_8523406713_3_0 :cluster_2506088039_8523406713_9_0 :cluster_2506088039_8523406713_5_0 :cluster_2506088039_8523406713_6_0 :cluster_2506088039_8523406713_7_0 :cluster_2506088039_8523406713_8_0" shape="633.50,1386.03 636.66,1386.51 638.85,1389.70 641.97,1391.25 646.44,1392.77 652.25,1394.26 659.41,1395.73 658.25,1402.02 655.97,1402.05 655.09,1402.45 654.38,1403.11 653.85,1404.03 653.48,1405.21 647.22,1403.88 645.61,1400.56 643.02,1399.14 639.24,1397.87 634.27,1396.77 628.10,1395.82 628.92,1389.47 631.19,1389.32 632.05,1388.87 632.72,1388.17 633.20,1387.22">\n    <request index="0" response="000001000" foes="001001000" cont="0" />\n    <request index="1" response="110011000" foes="111011000" cont="0" />\n    <request index="2" response="100111000" foes="101111000" cont="0" />\n    <request index="3" response="000000000" foes="001000111" cont="0" />\n    <request index="4" response="110000000" foes="111000110" cont="1" />\n    <request index="5" response="100000000" foes="100000100" cont="0" />\n    <request index="6" response="100011111" foes="100011111" cont="0" />\n    <request index="7" response="000000000" foes="000010010" cont="0" />\n    <request index="8" response="000000000" foes="001110110" cont="0" />\n    <param key="origEdgeIds" value="-706104217#15 706104217#15" />\n    <param key="origId" value="2506088039 8523406713" />\n  </junction>\n  <junction id="cluster_2507400612_2507400618" type="priority" x="1650.18" y="881.33" incLanes="-243314436#2_0 -243314248#0_0 243314436#0_0 243314165#1_0" intLanes=":cluster_2507400612_2507400618_0_0 :cluster_2507400612_2507400618_1_0 :cluster_2507400612_2507400618_12_0 :cluster_2507400612_2507400618_3_0 :cluster_2507400612_2507400618_4_0 :cluster_2507400612_2507400618_5_0 :cluster_2507400612_2507400618_6_0 :cluster_2507400612_2507400618_7_0 :cluster_2507400612_2507400618_13_0 :cluster_2507400612_2507400618_9_0 :cluster_2507400612_2507400618_10_0 :cluster_2507400612_2507400618_11_0" shape="1649.39,866.50 1655.69,867.63 1655.75,869.96 1656.18,870.91 1656.88,871.71 1657.85,872.36 1659.08,872.87 1657.13,878.97 1654.82,880.35 1653.82,882.69 1652.94,886.12 1652.17,890.66 1651.52,896.29 1645.15,895.67 1644.88,893.39 1644.36,892.54 1643.58,891.88 1642.54,891.41 1641.24,891.13 1642.06,884.78 1644.64,882.98 1645.88,880.47 1647.08,876.88 1648.25,872.23">\n    <request index="0" response="000000000000" foes="000100010000" cont="0" />\n    <request index="1" response="000000000000" foes="111100110000" cont="0" />\n    <request index="2" response="000011000000" foes="110011110000" cont="1" />\n    <request index="3" response="000010000000" foes="100010000000" cont="0" />\n    <request index="4" response="000110000111" foes="100110000111" cont="0" />\n    <request index="5" response="111110000110" foes="111110000110" cont="0" />\n    <request index="6" response="000000000000" foes="010000000100" cont="0" />\n    <request index="7" response="000000000000" foes="110000111100" cont="0" />\n    <request index="8" response="000000000011" foes="110000110011" cont="1" />\n    <request index="9" response="000000000010" foes="000000100010" cont="0" />\n    <request index="10" response="000111000110" foes="000111100110" cont="0" />\n    <request index="11" response="000110011110" foes="000110111110" cont="0" />\n    <param key="origEdgeIds" value="-243314436#1 243314436#1" />\n    <param key="origId" value="2507400612 2507400618" />\n  </junction>\n  <junction id="cluster_620881097_8523329816" type="priority" x="316.20" y="1163.45" incLanes="-917697169_0 84842974#2_0 84842974#2_1 84842974#2_2" intLanes=":cluster_620881097_8523329816_0_0 :cluster_620881097_8523329816_1_0 :cluster_620881097_8523329816_2_0 :cluster_620881097_8523329816_2_1 :cluster_620881097_8523329816_2_2 :cluster_620881097_8523329816_5_0" shape="313.98,1149.86 323.41,1151.70 323.41,1153.98 323.81,1154.87 324.46,1155.60 325.38,1156.16 326.56,1156.55 325.08,1162.78 322.65,1163.82 321.57,1165.57 320.56,1168.15 319.65,1171.56 318.81,1175.79 309.35,1174.17 309.28,1171.93 308.86,1171.10 308.18,1170.45 307.25,1170.00 306.05,1169.74 306.44,1166.57 309.12,1164.94 310.40,1162.65 311.63,1159.37 312.83,1155.11">\n    <request index="0" response="011100" foes="011100" cont="0" />\n    <request index="1" response="000000" foes="000000" cont="0" />\n    <request index="2" response="000000" foes="000001" cont="0" />\n    <request index="3" response="000000" foes="000001" cont="0" />\n    <request index="4" response="000000" foes="000001" cont="0" />\n    <request index="5" response="000000" foes="000000" cont="0" />\n    <param key="origEdgeIds" value="84842974#3" />\n    <param key="origId" value="620881097 8523329816" />\n  </junction>\n  <junction id="cluster_849403384_849403500_8976428987_8976428988" type="traffic_light" x="1475.99" y="748.05" incLanes="970010767#2_0 1270957861_0 1270957861_1 969757851#4_0 990212966#2_0" intLanes=":cluster_849403384_849403500_8976428987_8976428988_0_0 :cluster_849403384_849403500_8976428987_8976428988_1_0 :cluster_849403384_849403500_8976428987_8976428988_2_0 :cluster_849403384_849403500_8976428987_8976428988_3_0 :cluster_849403384_849403500_8976428987_8976428988_4_0 :cluster_849403384_849403500_8976428987_8976428988_4_1 :cluster_849403384_849403500_8976428987_8976428988_6_0 :cluster_849403384_849403500_8976428987_8976428988_7_0 :cluster_849403384_849403500_8976428987_8976428988_8_0 :cluster_849403384_849403500_8976428987_8976428988_9_0 :cluster_849403384_849403500_8976428987_8976428988_10_0 :cluster_849403384_849403500_8976428987_8976428988_11_0 :cluster_849403384_849403500_8976428987_8976428988_12_0" shape="1490.23,745.65 1489.74,752.46 1487.59,752.74 1486.91,753.23 1486.48,753.96 1486.30,754.92 1486.39,756.12 1470.27,758.74 1469.73,756.38 1468.93,754.40 1467.88,752.82 1466.57,751.63 1465.01,750.82 1463.20,750.41 1463.88,743.55 1466.02,743.33 1466.70,742.86 1467.13,742.14 1467.30,741.19 1467.22,739.99 1483.32,737.36 1483.87,739.78 1484.66,741.78 1485.69,743.37 1486.97,744.55 1488.48,745.31">\n    <request index="0" response="0000000110000" foes="0000000110000" cont="0" />\n    <request index="1" response="1110001110000" foes="1111001110000" cont="0" />\n    <request index="2" response="1100111110000" foes="1100111110000" cont="0" />\n    <request index="3" response="0000000000000" foes="1000100000000" cont="0" />\n    <request index="4" response="0000000000000" foes="1001100000111" cont="0" />\n    <request index="5" response="0000000000000" foes="1001100000111" cont="0" />\n    <request index="6" response="0110000000000" foes="0111100000110" cont="0" />\n    <request index="7" response="0100000000000" foes="0100000000100" cont="0" />\n    <request index="8" response="1100001111000" foes="1100001111100" cont="0" />\n    <request index="9" response="1100001110010" foes="1100001110010" cont="0" />\n    <request index="10" response="0000000000000" foes="0000001000010" cont="0" />\n    <request index="11" response="0000000000000" foes="0001111000110" cont="0" />\n    <request index="12" response="0000000111000" foes="0001100111110" cont="0" />\n    <param key="origEdgeIds" value="990212967 990212971 990212972 990212973" />\n    <param key="origId" value="849403384 849403500 8976428987 8976428988" />\n  </junction>\n  <junction id="cluster_916327730_916327962_985092239_985092700" type="traffic_light" x="308.99" y="1351.85" incLanes="706104217#21_0 77967090#1_0 77967090#1_1 77967090#1_2 77967082#0_0 77967082#0_1" intLanes=":cluster_916327730_916327962_985092239_985092700_0_0 :cluster_916327730_916327962_985092239_985092700_1_0 :cluster_916327730_916327962_985092239_985092700_2_0 :cluster_916327730_916327962_985092239_985092700_3_0 :cluster_916327730_916327962_985092239_985092700_3_1 :cluster_916327730_916327962_985092239_985092700_3_2 :cluster_916327730_916327962_985092239_985092700_6_0 :cluster_916327730_916327962_985092239_985092700_7_0 :cluster_916327730_916327962_985092239_985092700_8_0" shape="322.65,1336.81 322.69,1343.21 321.14,1346.10 321.50,1349.35 322.45,1353.51 323.85,1358.34 325.56,1363.61 304.69,1370.55 303.55,1368.46 302.75,1367.75 301.79,1367.26 300.69,1366.99 299.43,1366.94 299.13,1360.55 304.72,1359.46 308.25,1356.95 309.98,1353.18 310.21,1348.34 309.23,1342.59 307.32,1336.11 316.41,1333.05 317.67,1335.16 318.61,1335.89 319.75,1336.41 321.10,1336.72">\n    <request index="0" response="000111000" foes="000111000" cont="0" />\n    <request index="1" response="111111000" foes="111111000" cont="0" />\n    <request index="2" response="000000000" foes="010000000" cont="0" />\n    <request index="3" response="000000000" foes="110000011" cont="0" />\n    <request index="4" response="000000000" foes="110000011" cont="0" />\n    <request index="5" response="000000000" foes="110000011" cont="0" />\n    <request index="6" response="000000000" foes="000000010" cont="0" />\n    <request index="7" response="000111100" foes="000111110" cont="0" />\n    <request index="8" response="000111000" foes="000111010" cont="0" />\n    <param key="origEdgeIds" value="-706104217#22 706104217#22 77967082#1 77967084#1 77967090#2" />\n    <param key="origId" value="916327730 916327962 985092239 985092700" />\n  </junction>\n  <junction id=":1107784878_4_0" type="internal" x="637.79" y="505.03" incLanes=":1107784878_1_0 -706256379#4_0" intLanes=":1107784878_2_0 :1107784878_3_0" />\n  <junction id=":1107784883_7_0" type="internal" x="625.87" y="590.32" incLanes=":1107784883_4_0 -706256377#2_0" intLanes=":1107784883_1_0 :1107784883_2_0 :1107784883_5_0 :1107784883_6_0" />\n  <junction id=":1107784887_7_0" type="internal" x="522.95" y="1095.06" incLanes=":1107784887_4_0 -706104219#15_0" intLanes=":1107784887_1_0 :1107784887_2_0 :1107784887_5_0 :1107784887_6_0" />\n  <junction id=":1107926467_6_0" type="internal" x="1505.36" y="1685.05" incLanes=":1107926467_5_0 243188710#0_0" intLanes=":1107926467_1_0 :1107926467_2_0 :1107926467_3_0" />\n  <junction id=":1107926481_12_0" type="internal" x="1341.33" y="1566.48" incLanes=":1107926481_5_0 -95542485#3_0" intLanes=":1107926481_1_0 :1107926481_2_0 :1107926481_7_0 :1107926481_8_0 :1107926481_9_0 :1107926481_10_0" />\n  <junction id=":1107926481_13_0" type="internal" x="1339.04" y="1559.45" incLanes=":1107926481_11_0 -95809258_0" intLanes=":1107926481_1_0 :1107926481_2_0 :1107926481_3_0 :1107926481_4_0 :1107926481_7_0 :1107926481_8_0" />\n  <junction id=":1107926513_6_0" type="internal" x="1333.83" y="1130.72" incLanes=":1107926513_1_0 706105951#0_0" intLanes=":1107926513_3_0 :1107926513_4_0 :1107926513_5_0" />\n  <junction id=":1107926537_6_0" type="internal" x="1337.63" y="804.23" incLanes=":1107926537_1_0 -243188741#0_0" intLanes=":1107926537_3_0 :1107926537_4_0 :1107926537_5_0" />\n  <junction id=":1107926561_6_0" type="internal" x="1284.13" y="1577.40" incLanes=":1107926561_3_0 95542482#0_0" intLanes=":1107926561_0_0 :1107926561_1_0 :1107926561_5_0" />\n  <junction id=":1107926599_4_0" type="internal" x="1128.73" y="1610.12" incLanes=":1107926599_3_0 95542482#2_0" intLanes=":1107926599_0_0 :1107926599_1_0" />\n  <junction id=":1107926639_12_0" type="internal" x="1339.09" y="881.18" incLanes=":1107926639_5_0 243188351#0_0" intLanes=":1107926639_1_0 :1107926639_2_0 :1107926639_7_0 :1107926639_8_0 :1107926639_9_0 :1107926639_10_0" />\n  <junction id=":1107926639_13_0" type="internal" x="1332.22" y="882.78" incLanes=":1107926639_11_0 -243188629_0" intLanes=":1107926639_1_0 :1107926639_2_0 :1107926639_3_0 :1107926639_4_0 :1107926639_7_0 :1107926639_8_0" />\n  <junction id=":1107926940_6_0" type="internal" x="1241.24" y="1589.66" incLanes=":1107926940_3_0 95542482#1_0" intLanes=":1107926940_0_0 :1107926940_1_0 :1107926940_5_0" />\n  <junction id=":1107926977_9_0" type="internal" x="1083.49" y="1612.93" incLanes=":1107926977_2_0 -95542482#5_0" intLanes=":1107926977_4_0 :1107926977_5_0 :1107926977_6_0 :1107926977_7_0" />\n  <junction id=":1107926977_10_0" type="internal" x="1071.23" y="1615.47" incLanes=":1107926977_8_0 95542482#4_0" intLanes=":1107926977_0_0 :1107926977_1_0 :1107926977_4_0 :1107926977_5_0" />\n  <junction id=":1110416444_9_0" type="internal" x="647.80" y="1123.29" incLanes=":1110416444_2_0 -706104219#13_0" intLanes=":1110416444_4_0 :1110416444_5_0 :1110416444_6_0 :1110416444_7_0" />\n  <junction id=":1110416444_10_0" type="internal" x="640.37" y="1124.02" incLanes=":1110416444_8_0 706104219#12_0" intLanes=":1110416444_0_0 :1110416444_1_0 :1110416444_4_0 :1110416444_5_0" />\n  <junction id=":1110416469_6_0" type="internal" x="754.76" y="1661.47" incLanes=":1110416469_1_0 -95542482#12_0" intLanes=":1110416469_3_0 :1110416469_4_0 :1110416469_5_0" />\n  <junction id=":1110416526_6_0" type="internal" x="867.95" y="1644.93" incLanes=":1110416526_1_0 -95542482#10_0" intLanes=":1110416526_3_0 :1110416526_4_0 :1110416526_5_0" />\n  <junction id=":1110416726_12_0" type="internal" x="801.51" y="304.22" incLanes=":1110416726_5_0 -129827554#0_0" intLanes=":1110416726_1_0 :1110416726_2_0 :1110416726_7_0 :1110416726_8_0 :1110416726_9_0 :1110416726_10_0" />\n  <junction id=":1110416726_13_0" type="internal" x="792.76" y="306.58" incLanes=":1110416726_11_0 142455600#4_0" intLanes=":1110416726_1_0 :1110416726_2_0 :1110416726_3_0 :1110416726_4_0 :1110416726_7_0 :1110416726_8_0" />\n  <junction id=":1110416742_6_0" type="internal" x="1344.25" y="1634.43" incLanes=":1110416742_1_0 243188765#0_0" intLanes=":1110416742_3_0 :1110416742_4_0 :1110416742_5_0" />\n  <junction id=":1110416754_6_0" type="internal" x="606.77" y="1706.39" incLanes=":1110416754_3_0 95542482#13_0" intLanes=":1110416754_0_0 :1110416754_1_0 :1110416754_5_0" />\n  <junction id=":1115585385_9_0" type="internal" x="1032.55" y="977.44" incLanes=":1115585385_2_0 -969757849#0_0" intLanes=":1115585385_3_0 :1115585385_4_0 :1115585385_7_0 :1115585385_8_0" />\n  <junction id=":1115585385_10_0" type="internal" x="1024.59" y="978.81" incLanes=":1115585385_5_0 1204844051_0" intLanes=":1115585385_0_0 :1115585385_1_0 :1115585385_7_0 :1115585385_8_0" />\n  <junction id=":1115585403_3_0" type="internal" x="1063.72" y="1071.42" incLanes=":1115585403_2_0 96242763#1_0" intLanes=":1115585403_0_0 :1115585403_1_0" />\n  <junction id=":1115585463_6_0" type="internal" x="995.31" y="875.43" incLanes=":1115585463_5_0 142453602#1_0" intLanes=":1115585463_1_0 :1115585463_2_0 :1115585463_3_0" />\n  <junction id=":1115585792_4_0" type="internal" x="781.10" y="840.09" incLanes=":1115585792_1_0 -706520617#3_0" intLanes=":1115585792_2_0 :1115585792_3_0" />\n  <junction id=":1115585909_6_0" type="internal" x="1009.89" y="924.18" incLanes=":1115585909_1_0 -142453602#1_0" intLanes=":1115585909_3_0 :1115585909_4_0 :1115585909_5_0" />\n  <junction id=":1122072673_4_0" type="internal" x="540.49" y="300.37" incLanes=":1122072673_3_0 -129827558_0" intLanes=":1122072673_0_0 :1122072673_1_0" />\n  <junction id=":1122072674_6_0" type="internal" x="588.32" y="397.64" incLanes=":1122072674_3_0 -96837170#4_0" intLanes=":1122072674_0_0 :1122072674_1_0 :1122072674_5_0" />\n  <junction id=":1122072681_4_0" type="internal" x="499.60" y="490.40" incLanes=":1122072681_3_0 706096644#0_0" intLanes=":1122072681_0_0 :1122072681_1_0" />\n  <junction id=":1123399483_6_0" type="internal" x="1162.35" y="1423.42" incLanes=":1123399483_1_0 -706094741#2_0" intLanes=":1123399483_3_0 :1123399483_4_0 :1123399483_5_0" />\n  <junction id=":1123399486_6_0" type="internal" x="503.29" y="1603.76" incLanes=":1123399486_5_0 684741141#2_0" intLanes=":1123399486_1_0 :1123399486_2_0 :1123399486_3_0" />\n  <junction id=":1123399502_9_0" type="internal" x="567.65" y="1522.61" incLanes=":1123399502_6_0 243188785_0" intLanes=":1123399502_0_0 :1123399502_1_0 :1123399502_3_0 :1123399502_4_0 :1123399502_8_0" />\n  <junction id=":1123399513_6_0" type="internal" x="1478.75" y="1696.43" incLanes=":1123399513_1_0 96941841#1_0" intLanes=":1123399513_3_0 :1123399513_4_0 :1123399513_5_0" />\n  <junction id=":1123399516_6_0" type="internal" x="1241.33" y="1395.98" incLanes=":1123399516_3_0 243188705_0" intLanes=":1123399516_0_0 :1123399516_1_0 :1123399516_5_0" />\n  <junction id=":1123399535_6_0" type="internal" x="1139.33" y="1235.12" incLanes=":1123399535_1_0 -452036923#3_0" intLanes=":1123399535_3_0 :1123399535_4_0 :1123399535_5_0" />\n  <junction id=":1123399561_6_0" type="internal" x="1186.37" y="1231.11" incLanes=":1123399561_1_0 -452036923#2_0" intLanes=":1123399561_3_0 :1123399561_4_0 :1123399561_5_0" />\n  <junction id=":1123399591_9_0" type="internal" x="567.40" y="1718.31" incLanes=":1123399591_6_0 95542482#14_0" intLanes=":1123399591_0_0 :1123399591_1_0 :1123399591_3_0 :1123399591_4_0 :1123399591_8_0" />\n  <junction id=":1123399595_6_0" type="internal" x="790.06" y="1040.77" incLanes=":1123399595_1_0 -96941842#3_0" intLanes=":1123399595_3_0 :1123399595_4_0 :1123399595_5_0" />\n  <junction id=":1123399600_12_0" type="internal" x="1088.35" y="984.04" incLanes=":1123399600_2_0 -1204844052_0" intLanes=":1123399600_4_0 :1123399600_5_0 :1123399600_6_0 :1123399600_7_0 :1123399600_10_0 :1123399600_11_0" />\n  <junction id=":1123399600_13_0" type="internal" x="1079.10" y="985.31" incLanes=":1123399600_8_0 185951074#2_0" intLanes=":1123399600_0_0 :1123399600_1_0 :1123399600_4_0 :1123399600_5_0 :1123399600_10_0 :1123399600_11_0" />\n  <junction id=":1123399639_6_0" type="internal" x="1227.94" y="1180.47" incLanes=":1123399639_5_0 706104218_0" intLanes=":1123399639_1_0 :1123399639_2_0 :1123399639_3_0" />\n  <junction id=":1123399641_12_0" type="internal" x="1200.05" y="1423.09" incLanes=":1123399641_5_0 243188212#0_0" intLanes=":1123399641_1_0 :1123399641_2_0 :1123399641_7_0 :1123399641_8_0 :1123399641_9_0 :1123399641_10_0" />\n  <junction id=":1123399641_13_0" type="internal" x="1196.98" y="1416.07" incLanes=":1123399641_11_0 -243188212#1_0" intLanes=":1123399641_1_0 :1123399641_2_0 :1123399641_3_0 :1123399641_4_0 :1123399641_7_0 :1123399641_8_0" />\n  <junction id=":1123399675_6_0" type="internal" x="533.27" y="1671.26" incLanes=":1123399675_5_0 684741141#0_0" intLanes=":1123399675_1_0 :1123399675_2_0 :1123399675_3_0" />\n  <junction id=":1123399676_4_0" type="internal" x="1000.37" y="1683.95" incLanes=":1123399676_3_0 -980451721#0_0" intLanes=":1123399676_0_0 :1123399676_1_0" />\n  <junction id=":1123399683_6_0" type="internal" x="1237.76" y="1311.52" incLanes=":1123399683_1_0 -243188225_0" intLanes=":1123399683_3_0 :1123399683_4_0 :1123399683_5_0" />\n  <junction id=":1123399700_6_0" type="internal" x="1435.83" y="1646.14" incLanes=":1123399700_1_0 96941841#0_0" intLanes=":1123399700_3_0 :1123399700_4_0 :1123399700_5_0" />\n  <junction id=":1123399706_12_0" type="internal" x="808.52" y="945.54" incLanes=":1123399706_5_0 -969757849#3_0" intLanes=":1123399706_1_0 :1123399706_2_0 :1123399706_7_0 :1123399706_8_0 :1123399706_9_0 :1123399706_10_0" />\n  <junction id=":1123399706_13_0" type="internal" x="801.19" y="945.32" incLanes=":1123399706_11_0 969757849#2_0" intLanes=":1123399706_1_0 :1123399706_2_0 :1123399706_3_0 :1123399706_4_0 :1123399706_7_0 :1123399706_8_0" />\n  <junction id=":1123399713_6_0" type="internal" x="1223.06" y="1653.57" incLanes=":1123399713_1_0 497290263#2_0" intLanes=":1123399713_3_0 :1123399713_4_0 :1123399713_5_0" />\n  <junction id=":1123399740_6_0" type="internal" x="1284.15" y="1642.16" incLanes=":1123399740_1_0 497290263#3_0" intLanes=":1123399740_3_0 :1123399740_4_0 :1123399740_5_0" />\n  <junction id=":1123399943_6_0" type="internal" x="512.94" y="1630.46" incLanes=":1123399943_5_0 684741141#1_0" intLanes=":1123399943_1_0 :1123399943_2_0 :1123399943_3_0" />\n  <junction id=":1140280757_9_0" type="internal" x="688.88" y="1130.46" incLanes=":1140280757_8_0 706104219#11_0" intLanes=":1140280757_1_0 :1140280757_2_0 :1140280757_3_0 :1140280757_5_0 :1140280757_6_0" />\n  <junction id=":1140280758_4_0" type="internal" x="673.78" y="1402.70" incLanes=":1140280758_3_0 706104217#13_0" intLanes=":1140280758_0_0 :1140280758_1_0" />\n  <junction id=":12339345632_6_0" type="internal" x="757.62" y="1391.04" incLanes=":12339345632_1_0 -706104217#12_0" intLanes=":12339345632_3_0 :12339345632_4_0 :12339345632_5_0" />\n  <junction id=":12344163778_3_0" type="internal" x="531.43" y="1040.41" incLanes=":12344163778_2_0 706093504#1_0" intLanes=":12344163778_0_0 :12344163778_1_0" />\n  <junction id=":12346099054_4_0" type="internal" x="1018.76" y="1362.00" incLanes=":12346099054_3_0 330615829#0_0" intLanes=":12346099054_0_0 :12346099054_1_0" />\n  <junction id=":12649445299_6_0" type="internal" x="905.02" y="874.94" incLanes=":12649445299_1_0 -706520617#1_0" intLanes=":12649445299_3_0 :12649445299_4_0 :12649445299_5_0" />\n  <junction id=":12649445300_6_0" type="internal" x="810.26" y="907.85" incLanes=":12649445300_1_0 -96941842#1_0" intLanes=":12649445300_3_0 :12649445300_4_0 :12649445300_5_0" />\n  <junction id=":13279030938_6_0" type="internal" x="1303.94" y="986.96" incLanes=":13279030938_5_0 185951074#0_0" intLanes=":13279030938_1_0 :13279030938_2_0 :13279030938_3_0" />\n  <junction id=":13362936997_2_0" type="internal" x="1434.82" y="349.85" incLanes=":13362936997_1_0 1457057267_0" intLanes=":13362936997_0_0" />\n  <junction id=":1369020939_6_0" type="internal" x="1389.47" y="1584.83" incLanes=":1369020939_5_0 243188710#1_0" intLanes=":1369020939_1_0 :1369020939_2_0 :1369020939_3_0" />\n  <junction id=":1369020942_6_0" type="internal" x="1406.34" y="1540.42" incLanes=":1369020942_5_0 -122540020#7_0" intLanes=":1369020942_1_0 :1369020942_2_0 :1369020942_3_0" />\n  <junction id=":1369020943_6_0" type="internal" x="1392.73" y="1343.00" incLanes=":1369020943_5_0 -122540020#3_0" intLanes=":1369020943_1_0 :1369020943_2_0 :1369020943_3_0" />\n  <junction id=":1369020950_8_0" type="internal" x="1389.81" y="1191.84" incLanes=":1369020950_2_0 -72016745#2_0 -72016745#2_1" intLanes=":1369020950_4_0 :1369020950_5_0 :1369020950_6_0 :1369020950_6_1" />\n  <junction id=":14159239427_6_0" type="internal" x="1079.53" y="726.12" incLanes=":14159239427_1_0 -969765780#2_0" intLanes=":14159239427_3_0 :14159239427_4_0 :14159239427_5_0" />\n  <junction id=":14165547215_6_0" type="internal" x="1386.71" y="1277.69" incLanes=":14165547215_1_0 122540020#0_0" intLanes=":14165547215_3_0 :14165547215_4_0 :14165547215_5_0" />\n  <junction id=":14179759489_6_0" type="internal" x="395.68" y="1427.28" incLanes=":14179759489_5_0 -498671328#1_0" intLanes=":14179759489_1_0 :14179759489_2_0 :14179759489_3_0" />\n  <junction id=":14182422401_6_0" type="internal" x="726.75" y="415.35" incLanes=":14182422401_5_0 -155690741#2_0" intLanes=":14182422401_1_0 :14182422401_2_0 :14182422401_3_0" />\n  <junction id=":14188374629_6_0" type="internal" x="1211.08" y="1553.42" incLanes=":14188374629_3_0 1559802495#0_0" intLanes=":14188374629_0_0 :14188374629_1_0 :14188374629_5_0" />\n  <junction id=":14188764645_6_0" type="internal" x="808.07" y="574.87" incLanes=":14188764645_1_0 -129827561#1_0" intLanes=":14188764645_3_0 :14188764645_4_0 :14188764645_5_0" />\n  <junction id=":14188764646_6_0" type="internal" x="850.56" y="582.90" incLanes=":14188764646_5_0 -1559841161_0" intLanes=":14188764646_1_0 :14188764646_2_0 :14188764646_3_0" />\n  <junction id=":14188764647_6_0" type="internal" x="863.84" y="527.92" incLanes=":14188764647_1_0 -1454593202#1_0" intLanes=":14188764647_3_0 :14188764647_4_0 :14188764647_5_0" />\n  <junction id=":1431715363_2_0" type="internal" x="810.34" y="684.13" incLanes=":1431715363_1_0 71380773#0_0" intLanes=":1431715363_0_0" />\n  <junction id=":1431715372_6_0" type="internal" x="788.73" y="330.37" incLanes=":1431715372_1_0 -148941242#2_0" intLanes=":1431715372_3_0 :1431715372_4_0 :1431715372_5_0" />\n  <junction id=":1431715392_6_0" type="internal" x="773.44" y="603.18" incLanes=":1431715392_2_0 -969765782_0" intLanes=":1431715392_3_0 :1431715392_4_0" />\n  <junction id=":1431715392_7_0" type="internal" x="766.89" y="604.72" incLanes=":1431715392_5_0 969765781_0" intLanes=":1431715392_0_0 :1431715392_1_0" />\n  <junction id=":1431715395_6_0" type="internal" x="515.06" y="368.66" incLanes=":1431715395_1_0 96837172#0_0" intLanes=":1431715395_3_0 :1431715395_4_0 :1431715395_5_0" />\n  <junction id=":1431715406_7_0" type="internal" x="806.66" y="611.09" incLanes=":1431715406_4_0 -969765781_0" intLanes=":1431715406_1_0 :1431715406_5_0" />\n  <junction id=":1431715406_8_0" type="internal" x="802.64" y="607.69" incLanes=":1431715406_6_0 101939921_0" intLanes=":1431715406_1_0 :1431715406_2_0 :1431715406_3_0" />\n  <junction id=":1431715409_4_0" type="internal" x="735.96" y="513.50" incLanes=":1431715409_1_0 -706256379#1_0" intLanes=":1431715409_2_0 :1431715409_3_0" />\n  <junction id=":1431715415_6_0" type="internal" x="817.42" y="523.05" incLanes=":1431715415_1_0 -1454593202#2_0" intLanes=":1431715415_3_0 :1431715415_4_0 :1431715415_5_0" />\n  <junction id=":1431867103_6_0" type="internal" x="1713.63" y="450.35" incLanes=":1431867103_5_0 129842397#0_0" intLanes=":1431867103_1_0 :1431867103_2_0 :1431867103_3_0" />\n  <junction id=":1431867108_7_0" type="internal" x="1432.78" y="360.98" incLanes=":1431867108_6_0 -129842388#1_0" intLanes=":1431867108_0_0 :1431867108_1_0 :1431867108_3_0 :1431867108_4_0" />\n  <junction id=":1431867150_6_0" type="internal" x="1670.80" y="304.31" incLanes=":1431867150_1_0 -129842397#4_0" intLanes=":1431867150_3_0 :1431867150_4_0 :1431867150_5_0" />\n  <junction id=":1558314095_6_0" type="internal" x="1697.91" y="313.14" incLanes=":1558314095_3_0 -706087699#0_0" intLanes=":1558314095_0_0 :1558314095_1_0 :1558314095_5_0" />\n  <junction id=":1559016903_12_0" type="internal" x="878.93" y="339.09" incLanes=":1559016903_2_0 142455600#3_0" intLanes=":1559016903_4_0 :1559016903_5_0 :1559016903_6_0 :1559016903_7_0 :1559016903_10_0 :1559016903_11_0" />\n  <junction id=":1559016903_13_0" type="internal" x="880.19" y="345.67" incLanes=":1559016903_8_0 -142455604#1_0" intLanes=":1559016903_0_0 :1559016903_1_0 :1559016903_4_0 :1559016903_5_0 :1559016903_10_0 :1559016903_11_0" />\n  <junction id=":1559016904_6_0" type="internal" x="781.06" y="373.41" incLanes=":1559016904_1_0 -148941242#3_0" intLanes=":1559016904_3_0 :1559016904_4_0 :1559016904_5_0" />\n  <junction id=":1559016905_6_0" type="internal" x="875.56" y="389.36" incLanes=":1559016905_3_0 -142455600#3_0" intLanes=":1559016905_0_0 :1559016905_1_0 :1559016905_5_0" />\n  <junction id=":1559016909_12_0" type="internal" x="775.90" y="417.72" incLanes=":1559016909_2_0 -148941242#4_0" intLanes=":1559016909_4_0 :1559016909_5_0 :1559016909_6_0 :1559016909_7_0 :1559016909_10_0 :1559016909_11_0" />\n  <junction id=":1559016909_13_0" type="internal" x="777.07" y="424.31" incLanes=":1559016909_8_0 148941242#3_0" intLanes=":1559016909_0_0 :1559016909_1_0 :1559016909_4_0 :1559016909_5_0 :1559016909_10_0 :1559016909_11_0" />\n  <junction id=":1559016916_4_0" type="internal" x="772.31" y="461.69" incLanes=":1559016916_1_0 -148941242#5_0" intLanes=":1559016916_2_0 :1559016916_3_0" />\n  <junction id=":1559016918_4_0" type="internal" x="866.64" y="473.09" incLanes=":1559016918_3_0 -706256386_0" intLanes=":1559016918_0_0 :1559016918_1_0" />\n  <junction id=":1680666322_6_0" type="internal" x="935.23" y="311.35" incLanes=":1680666322_1_0 -706256380#1_0" intLanes=":1680666322_3_0 :1680666322_4_0 :1680666322_5_0" />\n  <junction id=":1680666463_12_0" type="internal" x="1498.61" y="373.82" incLanes=":1680666463_5_0 129842388#1_0" intLanes=":1680666463_1_0 :1680666463_2_0 :1680666463_7_0 :1680666463_8_0 :1680666463_9_0 :1680666463_10_0" />\n  <junction id=":1680666463_13_0" type="internal" x="1491.46" y="374.11" incLanes=":1680666463_11_0 -129842388#2_0" intLanes=":1680666463_1_0 :1680666463_2_0 :1680666463_3_0 :1680666463_4_0 :1680666463_7_0 :1680666463_8_0" />\n  <junction id=":1680666468_6_0" type="internal" x="657.20" y="367.43" incLanes=":1680666468_1_0 -129827554#4_0" intLanes=":1680666468_3_0 :1680666468_4_0 :1680666468_5_0" />\n  <junction id=":1680666686_12_0" type="internal" x="923.99" y="411.63" incLanes=":1680666686_2_0 -706256380#4_0" intLanes=":1680666686_4_0 :1680666686_5_0 :1680666686_6_0 :1680666686_7_0 :1680666686_10_0 :1680666686_11_0" />\n  <junction id=":1680666686_13_0" type="internal" x="925.12" y="418.44" incLanes=":1680666686_8_0 706256380#3_0" intLanes=":1680666686_0_0 :1680666686_1_0 :1680666686_4_0 :1680666686_5_0 :1680666686_10_0 :1680666686_11_0" />\n  <junction id=":1680666827_6_0" type="internal" x="1549.15" y="470.22" incLanes=":1680666827_1_0 155691084#1_0" intLanes=":1680666827_3_0 :1680666827_4_0 :1680666827_5_0" />\n  <junction id=":1680666891_6_0" type="internal" x="1636.14" y="484.74" incLanes=":1680666891_1_0 155691084#2_0" intLanes=":1680666891_3_0 :1680666891_4_0 :1680666891_5_0" />\n  <junction id=":1680667029_6_0" type="internal" x="977.82" y="510.15" incLanes=":1680667029_5_0 -155690878#1_0" intLanes=":1680667029_1_0 :1680667029_2_0 :1680667029_3_0" />\n  <junction id=":1680667079_12_0" type="internal" x="1549.67" y="522.05" incLanes=":1680667079_2_0 155690734#0_0" intLanes=":1680667079_4_0 :1680667079_5_0 :1680667079_6_0 :1680667079_7_0 :1680667079_10_0 :1680667079_11_0" />\n  <junction id=":1680667079_13_0" type="internal" x="1542.83" y="523.93" incLanes=":1680667079_8_0 -155690734#1_0" intLanes=":1680667079_0_0 :1680667079_1_0 :1680667079_4_0 :1680667079_5_0 :1680667079_10_0 :1680667079_11_0" />\n  <junction id=":1680667164_6_0" type="internal" x="1544.58" y="550.05" incLanes=":1680667164_3_0 -155690812#1_0" intLanes=":1680667164_0_0 :1680667164_1_0 :1680667164_5_0" />\n  <junction id=":1680667181_6_0" type="internal" x="1078.43" y="549.03" incLanes=":1680667181_5_0 95533432#4_0" intLanes=":1680667181_1_0 :1680667181_2_0 :1680667181_3_0" />\n  <junction id=":1966399248_12_0" type="internal" x="778.52" y="1136.07" incLanes=":1966399248_5_0 -706104219#9_0" intLanes=":1966399248_1_0 :1966399248_2_0 :1966399248_7_0 :1966399248_8_0 :1966399248_9_0 :1966399248_10_0" />\n  <junction id=":1966399248_13_0" type="internal" x="771.89" y="1137.45" incLanes=":1966399248_11_0 706104219#8_0" intLanes=":1966399248_1_0 :1966399248_2_0 :1966399248_3_0 :1966399248_4_0 :1966399248_7_0 :1966399248_8_0" />\n  <junction id=":1966404349_7_0" type="internal" x="1030.58" y="1067.03" incLanes=":1966404349_4_0 96941857#2_0" intLanes=":1966404349_1_0 :1966404349_2_0 :1966404349_5_0 :1966404349_6_0" />\n  <junction id=":2052547075_7_0" type="internal" x="695.66" y="598.44" incLanes=":2052547075_6_0 706256377#0_0" intLanes=":2052547075_0_0 :2052547075_1_0 :2052547075_3_0 :2052547075_4_0" />\n  <junction id=":2505176265_3_0" type="internal" x="1568.83" y="1201.00" incLanes=":2505176265_2_0 66984613#7_0" intLanes=":2505176265_0_0 :2505176265_1_0" />\n  <junction id=":2505176697_6_0" type="internal" x="1340.49" y="1589.10" incLanes=":2505176697_5_0 -243188765#0_0" intLanes=":2505176697_1_0 :2505176697_2_0 :2505176697_3_0" />\n  <junction id=":2506085476_6_0" type="internal" x="1272.71" y="565.71" incLanes=":2506085476_1_0 -95533432#1_0" intLanes=":2506085476_3_0 :2506085476_4_0 :2506085476_5_0" />\n  <junction id=":2506085493_6_0" type="internal" x="1194.81" y="650.21" incLanes=":2506085493_1_0 243188607#0_0" intLanes=":2506085493_3_0 :2506085493_4_0 :2506085493_5_0" />\n  <junction id=":2506085508_6_0" type="internal" x="1181.22" y="688.71" incLanes=":2506085508_5_0 706094740#2_0" intLanes=":2506085508_1_0 :2506085508_2_0 :2506085508_3_0" />\n  <junction id=":2506085555_4_0" type="internal" x="1218.64" y="810.48" incLanes=":2506085555_1_0 -498671678_0" intLanes=":2506085555_2_0 :2506085555_3_0" />\n  <junction id=":2506085631_7_0" type="internal" x="1224.57" y="887.38" incLanes=":2506085631_1_0 498671616_0" intLanes=":2506085631_2_0 :2506085631_3_0 :2506085631_5_0 :2506085631_6_0" />\n  <junction id=":2506085690_9_0" type="internal" x="755.21" y="928.80" incLanes=":2506085690_4_0 -931354569#0_0" intLanes=":2506085690_1_0 :2506085690_2_0 :2506085690_6_0 :2506085690_7_0 :2506085690_8_0" />\n  <junction id=":2506086795_6_0" type="internal" x="829.25" y="1140.26" incLanes=":2506086795_1_0 -706104219#8_0" intLanes=":2506086795_3_0 :2506086795_4_0 :2506086795_5_0" />\n  <junction id=":2506086831_9_0" type="internal" x="874.74" y="1146.44" incLanes=":2506086831_8_0 706104219#5_0" intLanes=":2506086831_1_0 :2506086831_2_0 :2506086831_3_0 :2506086831_5_0 :2506086831_6_0" />\n  <junction id=":2506087136_14_0" type="internal" x="1289.27" y="1183.35" incLanes=":2506087136_6_0 -72016745#5_0 -72016745#5_1" intLanes=":2506087136_1_0 :2506087136_2_0 :2506087136_8_0 :2506087136_9_0 :2506087136_10_0 :2506087136_11_0 :2506087136_11_1" />\n  <junction id=":2506087136_15_0" type="internal" x="1281.51" y="1184.49" incLanes=":2506087136_13_0 72016745#4_0 72016745#4_1" intLanes=":2506087136_1_0 :2506087136_2_0 :2506087136_3_0 :2506087136_4_0 :2506087136_4_1 :2506087136_8_0 :2506087136_9_0" />\n  <junction id=":2506087304_6_0" type="internal" x="430.48" y="1200.92" incLanes=":2506087304_3_0 -243188536#2_0" intLanes=":2506087304_0_0 :2506087304_1_0 :2506087304_5_0" />\n  <junction id=":2506087522_3_0" type="internal" x="363.03" y="1245.18" incLanes=":2506087522_2_0 243188646#0_0" intLanes=":2506087522_0_0 :2506087522_1_0" />\n  <junction id=":2506087962_6_0" type="internal" x="936.71" y="1344.68" incLanes=":2506087962_3_0 706104217#3_0" intLanes=":2506087962_0_0 :2506087962_1_0 :2506087962_5_0" />\n  <junction id=":2506087987_12_0" type="internal" x="419.24" y="1357.47" incLanes=":2506087987_5_0 -706104217#20_0" intLanes=":2506087987_1_0 :2506087987_2_0 :2506087987_7_0 :2506087987_8_0 :2506087987_9_0 :2506087987_10_0" />\n  <junction id=":2506087987_13_0" type="internal" x="412.38" y="1357.45" incLanes=":2506087987_11_0 706104217#19_0" intLanes=":2506087987_1_0 :2506087987_2_0 :2506087987_3_0 :2506087987_4_0 :2506087987_7_0 :2506087987_8_0" />\n  <junction id=":2506088023_6_0" type="internal" x="1395.82" y="1384.34" incLanes=":2506088023_5_0 -122540020#4_0" intLanes=":2506088023_1_0 :2506088023_2_0 :2506088023_3_0" />\n  <junction id=":2506088024_16_0" type="internal" x="776.17" y="1392.39" incLanes=":2506088024_9_0 -243188491_0" intLanes=":2506088024_1_0 :2506088024_2_0 :2506088024_5_0 :2506088024_6_0 :2506088024_11_0 :2506088024_12_0 :2506088024_13_0 :2506088024_14_0" />\n  <junction id=":2506088024_17_0" type="internal" x="774.05" y="1384.83" incLanes=":2506088024_15_0 1257722737_0" intLanes=":2506088024_1_0 :2506088024_2_0 :2506088024_3_0 :2506088024_4_0 :2506088024_7_0 :2506088024_8_0 :2506088024_11_0 :2506088024_12_0" />\n  <junction id=":2506088274_6_0" type="internal" x="1400.88" y="1462.58" incLanes=":2506088274_5_0 -122540020#5_0" intLanes=":2506088274_1_0 :2506088274_2_0 :2506088274_3_0" />\n  <junction id=":2506088476_6_0" type="internal" x="462.62" y="1466.02" incLanes=":2506088476_5_0 498671331_0" intLanes=":2506088476_1_0 :2506088476_2_0 :2506088476_3_0" />\n  <junction id=":2506088491_6_0" type="internal" x="531.21" y="1492.89" incLanes=":2506088491_5_0 243188389#1_0" intLanes=":2506088491_1_0 :2506088491_2_0 :2506088491_3_0" />\n  <junction id=":2506088492_6_0" type="internal" x="1243.80" y="1506.02" incLanes=":2506088492_3_0 243188552#0_0" intLanes=":2506088492_0_0 :2506088492_1_0 :2506088492_5_0" />\n  <junction id=":2506088494_6_0" type="internal" x="501.20" y="1496.26" incLanes=":2506088494_3_0 243188389#2_0" intLanes=":2506088494_0_0 :2506088494_1_0 :2506088494_5_0" />\n  <junction id=":2506088518_6_0" type="internal" x="1152.70" y="1511.54" incLanes=":2506088518_5_0 -243188508#1_0" intLanes=":2506088518_1_0 :2506088518_2_0 :2506088518_3_0" />\n  <junction id=":2506088528_6_0" type="internal" x="1403.30" y="1516.24" incLanes=":2506088528_5_0 -122540020#6_0" intLanes=":2506088528_1_0 :2506088528_2_0 :2506088528_3_0" />\n  <junction id=":2506088529_2_0" type="internal" x="1113.73" y="1517.96" incLanes=":2506088529_1_0 243188641#0_0" intLanes=":2506088529_0_0" />\n  <junction id=":2506088612_4_0" type="internal" x="1063.57" y="1526.67" incLanes=":2506088612_3_0 243188594#0_0" intLanes=":2506088612_1_0" />\n  <junction id=":2506088938_6_0" type="internal" x="584.82" y="1595.94" incLanes=":2506088938_1_0 243188786#1_0" intLanes=":2506088938_3_0 :2506088938_4_0 :2506088938_5_0" />\n  <junction id=":2506089058_6_0" type="internal" x="590.12" y="1618.40" incLanes=":2506089058_1_0 243188786#2_0" intLanes=":2506089058_3_0 :2506089058_4_0 :2506089058_5_0" />\n  <junction id=":2506089487_6_0" type="internal" x="599.00" y="1655.99" incLanes=":2506089487_1_0 243188786#3_0" intLanes=":2506089487_3_0 :2506089487_4_0 :2506089487_5_0" />\n  <junction id=":2506089545_6_0" type="internal" x="1174.02" y="1663.35" incLanes=":2506089545_1_0 497290263#1_0" intLanes=":2506089545_3_0 :2506089545_4_0 :2506089545_5_0" />\n  <junction id=":2506089600_6_0" type="internal" x="1119.78" y="1674.63" incLanes=":2506089600_1_0 497290263#0_0" intLanes=":2506089600_3_0 :2506089600_4_0 :2506089600_5_0" />\n  <junction id=":2506089779_5_0" type="internal" x="1021.33" y="1703.24" incLanes=":2506089779_1_0 497290269_0" intLanes=":2506089779_2_0 :2506089779_4_0" />\n  <junction id=":2506089901_6_0" type="internal" x="1411.23" y="1708.47" incLanes=":2506089901_1_0 -243188503#1_0" intLanes=":2506089901_3_0 :2506089901_4_0 :2506089901_5_0" />\n  <junction id=":2507400604_6_0" type="internal" x="1655.19" y="845.36" incLanes=":2507400604_1_0 243314436#2_0" intLanes=":2507400604_3_0 :2507400604_4_0 :2507400604_5_0" />\n  <junction id=":2507400609_6_0" type="internal" x="1744.65" y="865.74" incLanes=":2507400609_1_0 243314467#0_0" intLanes=":2507400609_3_0 :2507400609_4_0 :2507400609_5_0" />\n  <junction id=":2507400769_6_0" type="internal" x="818.77" y="1650.97" incLanes=":2507400769_1_0 -95542482#11_0" intLanes=":2507400769_3_0 :2507400769_4_0 :2507400769_5_0" />\n  <junction id=":2507400772_6_0" type="internal" x="701.77" y="1676.53" incLanes=":2507400772_1_0 -95542482#13_0" intLanes=":2507400772_3_0 :2507400772_4_0 :2507400772_5_0" />\n  <junction id=":2507400773_6_0" type="internal" x="1696.68" y="1695.52" incLanes=":2507400773_5_0 81350006#1_0" intLanes=":2507400773_1_0 :2507400773_2_0 :2507400773_3_0" />\n  <junction id=":4488358856_6_0" type="internal" x="1133.35" y="1314.84" incLanes=":4488358856_5_0 -452036923#6_0" intLanes=":4488358856_1_0 :4488358856_2_0 :4488358856_3_0" />\n  <junction id=":4488358857_6_0" type="internal" x="1092.95" y="1207.28" incLanes=":4488358857_1_0 -452036923#4_0" intLanes=":4488358857_3_0 :4488358857_4_0 :4488358857_5_0" />\n  <junction id=":4488358858_6_0" type="internal" x="1079.59" y="1298.60" incLanes=":4488358858_5_0 -452036923#5_0" intLanes=":4488358858_1_0 :4488358858_2_0 :4488358858_3_0" />\n  <junction id=":4488366488_6_0" type="internal" x="1236.46" y="1207.05" incLanes=":4488366488_1_0 -452036923#1_0" intLanes=":4488366488_3_0 :4488366488_4_0 :4488366488_5_0" />\n  <junction id=":4488368090_6_0" type="internal" x="1180.13" y="1304.78" incLanes=":4488368090_5_0 -452036923#7_0" intLanes=":4488368090_1_0 :4488368090_2_0 :4488368090_3_0" />\n  <junction id=":4900464580_12_0" type="internal" x="972.91" y="1060.85" incLanes=":4900464580_2_0 96941857#1_0" intLanes=":4900464580_4_0 :4900464580_5_0 :4900464580_6_0 :4900464580_7_0 :4900464580_10_0 :4900464580_11_0" />\n  <junction id=":4900464580_13_0" type="internal" x="965.55" y="1061.99" incLanes=":4900464580_8_0 -96941857#2_0" intLanes=":4900464580_0_0 :4900464580_1_0 :4900464580_4_0 :4900464580_5_0 :4900464580_10_0 :4900464580_11_0" />\n  <junction id=":4900464587_6_0" type="internal" x="1241.41" y="1138.00" incLanes=":4900464587_3_0 -1476708031_0" intLanes=":4900464587_0_0 :4900464587_1_0 :4900464587_5_0" />\n  <junction id=":4900464597_6_0" type="internal" x="966.34" y="1154.67" incLanes=":4900464597_5_0 706104219#3_0" intLanes=":4900464597_1_0 :4900464597_2_0 :4900464597_3_0" />\n  <junction id=":4900464598_6_0" type="internal" x="972.42" y="974.88" incLanes=":4900464598_1_0 -969757849#1_0" intLanes=":4900464598_3_0 :4900464598_4_0 :4900464598_5_0" />\n  <junction id=":4900465031_6_0" type="internal" x="1729.59" y="405.64" incLanes=":4900465031_1_0 -706087699#1_0" intLanes=":4900465031_3_0 :4900465031_4_0 :4900465031_5_0" />\n  <junction id=":4900465069_6_0" type="internal" x="1347.03" y="1692.31" incLanes=":4900465069_1_0 243188765#1_0" intLanes=":4900465069_3_0 :4900465069_4_0 :4900465069_5_0" />\n  <junction id=":4900465071_6_0" type="internal" x="1624.36" y="523.12" incLanes=":4900465071_1_0 -155691022#1_0" intLanes=":4900465071_3_0 :4900465071_4_0 :4900465071_5_0" />\n  <junction id=":4900465090_3_0" type="internal" x="314.56" y="1658.93" incLanes=":4900465090_0_0 498671330#0_0" intLanes=":4900465090_1_0 :4900465090_2_0" />\n  <junction id=":4900465166_12_0" type="internal" x="899.05" y="915.54" incLanes=":4900465166_2_0 1365935945#0_0" intLanes=":4900465166_4_0 :4900465166_5_0 :4900465166_6_0 :4900465166_7_0 :4900465166_10_0 :4900465166_11_0" />\n  <junction id=":4900465166_13_0" type="internal" x="900.56" y="922.09" incLanes=":4900465166_8_0 -1365935945#1_0" intLanes=":4900465166_0_0 :4900465166_1_0 :4900465166_4_0 :4900465166_5_0 :4900465166_10_0 :4900465166_11_0" />\n  <junction id=":4900465208_6_0" type="internal" x="590.14" y="500.78" incLanes=":4900465208_1_0 -706256379#6_0" intLanes=":4900465208_3_0 :4900465208_4_0 :4900465208_5_0" />\n  <junction id=":4900465209_6_0" type="internal" x="572.99" y="587.85" incLanes=":4900465209_5_0 706256377#2_0" intLanes=":4900465209_1_0 :4900465209_2_0 :4900465209_3_0" />\n  <junction id=":4900465219_6_0" type="internal" x="1153.31" y="554.67" incLanes=":4900465219_1_0 -95533432#4_0" intLanes=":4900465219_3_0 :4900465219_4_0 :4900465219_5_0" />\n  <junction id=":4900465244_6_0" type="internal" x="1375.23" y="648.40" incLanes=":4900465244_1_0 155690480#0_0" intLanes=":4900465244_3_0 :4900465244_4_0 :4900465244_5_0" />\n  <junction id=":4900465288_6_0" type="internal" x="1243.46" y="1552.83" incLanes=":4900465288_3_0 243188552#1_0" intLanes=":4900465288_0_0 :4900465288_1_0 :4900465288_5_0" />\n  <junction id=":4900465364_2_0" type="internal" x="499.59" y="1221.51" incLanes=":4900465364_1_0 706093506#1_0" intLanes=":4900465364_0_0" />\n  <junction id=":4900465377_6_0" type="internal" x="1733.03" y="900.83" incLanes=":4900465377_5_0 -243314248#1_0" intLanes=":4900465377_1_0 :4900465377_2_0 :4900465377_3_0" />\n  <junction id=":4900465441_6_0" type="internal" x="1632.18" y="1350.13" incLanes=":4900465441_5_0 243314378#2_0" intLanes=":4900465441_1_0 :4900465441_2_0 :4900465441_3_0" />\n  <junction id=":4900465471_3_0" type="internal" x="602.55" y="717.27" incLanes=":4900465471_2_0 706093509#0_0" intLanes=":4900465471_0_0 :4900465471_1_0" />\n  <junction id=":4900465472_7_0" type="internal" x="654.79" y="723.07" incLanes=":4900465472_4_0 498671576#0_0" intLanes=":4900465472_1_0 :4900465472_2_0 :4900465472_5_0 :4900465472_6_0" />\n  <junction id=":4900465473_6_0" type="internal" x="698.69" y="733.80" incLanes=":4900465473_3_0 -243188465#1_0" intLanes=":4900465473_0_0 :4900465473_1_0 :4900465473_5_0" />\n  <junction id=":4900465513_2_0" type="internal" x="438.61" y="985.32" incLanes=":4900465513_0_0 684714761#1_0" intLanes=":4900465513_1_0" />\n  <junction id=":4900465517_6_0" type="internal" x="546.71" y="1381.24" incLanes=":4900465517_1_0 -706104217#18_0" intLanes=":4900465517_3_0 :4900465517_4_0 :4900465517_5_0" />\n  <junction id=":4900465706_6_0" type="internal" x="398.37" y="1495.25" incLanes=":4900465706_3_0 1552748564#0_0" intLanes=":4900465706_0_0 :4900465706_1_0 :4900465706_5_0" />\n  <junction id=":4900465949_7_0" type="internal" x="1449.41" y="459.61" incLanes=":4900465949_6_0 -155691084#1_0" intLanes=":4900465949_0_0 :4900465949_1_0 :4900465949_3_0 :4900465949_4_0" />\n  <junction id=":4900465989_6_0" type="internal" x="376.63" y="1294.46" incLanes=":4900465989_3_0 -243188174#2_0" intLanes=":4900465989_0_0 :4900465989_1_0 :4900465989_5_0" />\n  <junction id=":4900466448_12_0" type="internal" x="889.04" y="1052.60" incLanes=":4900466448_5_0 96941857#0_0" intLanes=":4900466448_1_0 :4900466448_2_0 :4900466448_7_0 :4900466448_8_0 :4900466448_9_0 :4900466448_10_0" />\n  <junction id=":4900466448_13_0" type="internal" x="882.35" y="1054.05" incLanes=":4900466448_11_0 -96941857#1_0" intLanes=":4900466448_1_0 :4900466448_2_0 :4900466448_3_0 :4900466448_4_0 :4900466448_7_0 :4900466448_8_0" />\n  <junction id=":5149825984_12_0" type="internal" x="1260.97" y="1180.96" incLanes=":5149825984_5_0 -706104218_0" intLanes=":5149825984_1_0 :5149825984_2_0 :5149825984_7_0 :5149825984_8_0 :5149825984_9_0 :5149825984_10_0" />\n  <junction id=":5149825984_13_0" type="internal" x="1254.80" y="1181.09" incLanes=":5149825984_11_0 72016745#5_0" intLanes=":5149825984_1_0 :5149825984_2_0 :5149825984_3_0 :5149825984_4_0 :5149825984_7_0 :5149825984_8_0" />\n  <junction id=":6634630594_6_0" type="internal" x="338.86" y="1341.02" incLanes=":6634630594_3_0 706104217#20_0" intLanes=":6634630594_0_0 :6634630594_1_0 :6634630594_5_0" />\n  <junction id=":6634630596_12_0" type="internal" x="331.49" y="1287.91" incLanes=":6634630596_2_0 243188174#0_0" intLanes=":6634630596_4_0 :6634630596_5_0 :6634630596_6_0 :6634630596_7_0 :6634630596_10_0 :6634630596_11_0" />\n  <junction id=":6634630596_13_0" type="internal" x="322.91" y="1288.94" incLanes=":6634630596_8_0 -243188174#1_0" intLanes=":6634630596_0_0 :6634630596_1_0 :6634630596_4_0 :6634630596_5_0 :6634630596_10_0 :6634630596_11_0" />\n  <junction id=":809537690_4_0" type="internal" x="1565.90" y="1176.01" incLanes=":809537690_3_0 706105952#0_0" intLanes=":809537690_1_0" />\n  <junction id=":8348961009_6_0" type="internal" x="426.00" y="1224.70" incLanes=":8348961009_3_0 -243188536#1_0" intLanes=":8348961009_0_0 :8348961009_1_0 :8348961009_5_0" />\n  <junction id=":8363308288_6_0" type="internal" x="1359.53" y="1135.76" incLanes=":8363308288_5_0 -900244003#1_0" intLanes=":8363308288_1_0 :8363308288_2_0 :8363308288_3_0" />\n  <junction id=":8523326825_6_0" type="internal" x="1592.51" y="880.53" incLanes=":8523326825_1_0 243314165#0_0" intLanes=":8523326825_3_0 :8523326825_4_0 :8523326825_5_0" />\n  <junction id=":8523431062_6_0" type="internal" x="985.82" y="1332.66" incLanes=":8523431062_5_0 706104217#0_0" intLanes=":8523431062_1_0 :8523431062_2_0 :8523431062_3_0" />\n  <junction id=":8523431064_6_0" type="internal" x="958.47" y="1338.98" incLanes=":8523431064_3_0 706104217#2_0" intLanes=":8523431064_0_0 :8523431064_1_0 :8523431064_5_0" />\n  <junction id=":8523449430_6_0" type="internal" x="1679.39" y="1352.40" incLanes=":8523449430_5_0 243314378#0_0" intLanes=":8523449430_1_0 :8523449430_2_0 :8523449430_3_0" />\n  <junction id=":856214470_2_0" type="internal" x="667.88" y="1030.75" incLanes=":856214470_1_0 706092523#0_0" intLanes=":856214470_0_0" />\n  <junction id=":856214503_7_0" type="internal" x="698.70" y="913.34" incLanes=":856214503_6_0 931354569#0_0" intLanes=":856214503_0_0 :856214503_1_0 :856214503_3_0 :856214503_4_0" />\n  <junction id=":856214524_12_0" type="internal" x="1339.81" y="984.98" incLanes=":856214524_2_0 -185951074#0_0" intLanes=":856214524_4_0 :856214524_5_0 :856214524_6_0 :856214524_7_0 :856214524_10_0 :856214524_11_0" />\n  <junction id=":856214524_13_0" type="internal" x="1333.31" y="986.78" incLanes=":856214524_8_0 1270957863_0" intLanes=":856214524_0_0 :856214524_1_0 :856214524_4_0 :856214524_5_0 :856214524_10_0 :856214524_11_0" />\n  <junction id=":856214538_12_0" type="internal" x="900.15" y="971.58" incLanes=":856214538_5_0 -969757849#2_0" intLanes=":856214538_1_0 :856214538_2_0 :856214538_7_0 :856214538_8_0 :856214538_9_0 :856214538_10_0" />\n  <junction id=":856214538_13_0" type="internal" x="893.08" y="972.38" incLanes=":856214538_11_0 969757849#1_0" intLanes=":856214538_1_0 :856214538_2_0 :856214538_3_0 :856214538_4_0 :856214538_7_0 :856214538_8_0" />\n  <junction id=":8642733850_6_0" type="internal" x="1648.72" y="419.23" incLanes=":8642733850_1_0 -129842397#3_0" intLanes=":8642733850_3_0 :8642733850_4_0 :8642733850_5_0" />\n  <junction id=":947896950_4_0" type="internal" x="891.69" y="1355.37" incLanes=":947896950_3_0 706104217#4_0" intLanes=":947896950_0_0 :947896950_1_0" />\n  <junction id=":947896956_9_0" type="internal" x="482.04" y="1377.00" incLanes=":947896956_2_0 -706095305_0" intLanes=":947896956_3_0 :947896956_4_0 :947896956_5_0 :947896956_8_0" />\n  <junction id=":947896956_10_0" type="internal" x="479.14" y="1379.54" incLanes=":947896956_6_0 706093506#3_0" intLanes=":947896956_0_0 :947896956_1_0 :947896956_3_0 :947896956_4_0 :947896956_8_0" />\n  <junction id=":947896981_7_0" type="internal" x="1649.43" y="1704.94" incLanes=":947896981_6_0 81350006#2_0" intLanes=":947896981_0_0 :947896981_1_0 :947896981_3_0 :947896981_4_0" />\n  <junction id=":cluster_1107773211_1110416556_16_0" type="internal" x="753.56" y="516.57" incLanes=":cluster_1107773211_1110416556_11_0 1454593202#2_0" intLanes=":cluster_1107773211_1110416556_2_0 :cluster_1107773211_1110416556_3_0 :cluster_1107773211_1110416556_4_0 :cluster_1107773211_1110416556_5_0 :cluster_1107773211_1110416556_7_0 :cluster_1107773211_1110416556_8_0 :cluster_1107773211_1110416556_9_0 :cluster_1107773211_1110416556_14_0 :cluster_1107773211_1110416556_15_0" />\n  <junction id=":cluster_1107773211_1110416556_17_0" type="internal" x="749.75" y="516.21" incLanes=":cluster_1107773211_1110416556_12_0 1454593202#2_0" intLanes=":cluster_1107773211_1110416556_0_0 :cluster_1107773211_1110416556_1_0 :cluster_1107773211_1110416556_4_0 :cluster_1107773211_1110416556_5_0 :cluster_1107773211_1110416556_8_0 :cluster_1107773211_1110416556_9_0 :cluster_1107773211_1110416556_14_0 :cluster_1107773211_1110416556_15_0" />\n  <junction id=":cluster_1110416379_1680666461_12_0" type="internal" x="931.59" y="353.99" incLanes=":cluster_1110416379_1680666461_2_0 -706256380#3_0" intLanes=":cluster_1110416379_1680666461_4_0 :cluster_1110416379_1680666461_5_0 :cluster_1110416379_1680666461_6_0 :cluster_1110416379_1680666461_7_0 :cluster_1110416379_1680666461_10_0 :cluster_1110416379_1680666461_11_0" />\n  <junction id=":cluster_1110416379_1680666461_13_0" type="internal" x="930.33" y="364.41" incLanes=":cluster_1110416379_1680666461_8_0 706256380#1_0" intLanes=":cluster_1110416379_1680666461_0_0 :cluster_1110416379_1680666461_1_0 :cluster_1110416379_1680666461_4_0 :cluster_1110416379_1680666461_5_0 :cluster_1110416379_1680666461_10_0 :cluster_1110416379_1680666461_11_0" />\n  <junction id=":cluster_1122072658_14138648129_9_0" type="internal" x="693.64" y="511.89" incLanes=":cluster_1122072658_14138648129_6_0 706256379#1_0" intLanes=":cluster_1122072658_14138648129_0_0 :cluster_1122072658_14138648129_1_0 :cluster_1122072658_14138648129_3_0 :cluster_1122072658_14138648129_4_0 :cluster_1122072658_14138648129_8_0" />\n  <junction id=":cluster_1123399587_8976429007_6_0" type="internal" x="1101.76" y="710.37" incLanes=":cluster_1123399587_8976429007_1_0 969757851#0_0" intLanes=":cluster_1123399587_8976429007_3_0 :cluster_1123399587_8976429007_4_0 :cluster_1123399587_8976429007_5_0" />\n  <junction id=":cluster_11292971032_2406954578_2406954579_2406954580_7_0" type="internal" x="1547.73" y="1177.32" incLanes=":cluster_11292971032_2406954578_2406954579_2406954580_6_0 970323546#7_0 970323546#7_1" intLanes=":cluster_11292971032_2406954578_2406954579_2406954580_1_0 :cluster_11292971032_2406954578_2406954579_2406954580_2_0 :cluster_11292971032_2406954578_2406954579_2406954580_3_0 :cluster_11292971032_2406954578_2406954579_2406954580_3_1" />\n  <junction id=":cluster_13424220808_2506085647_9_0" type="internal" x="655.70" y="898.50" incLanes=":cluster_13424220808_2506085647_8_0 931354569#1_0" intLanes=":cluster_13424220808_2506085647_1_0 :cluster_13424220808_2506085647_2_0 :cluster_13424220808_2506085647_3_0 :cluster_13424220808_2506085647_5_0 :cluster_13424220808_2506085647_6_0" />\n  <junction id=":cluster_13472440775_1666900747_14_0" type="internal" x="1340.90" y="1187.46" incLanes=":cluster_13472440775_1666900747_6_0 -72016745#4_0 -72016745#4_1" intLanes=":cluster_13472440775_1666900747_1_0 :cluster_13472440775_1666900747_2_0 :cluster_13472440775_1666900747_8_0 :cluster_13472440775_1666900747_9_0 :cluster_13472440775_1666900747_10_0 :cluster_13472440775_1666900747_11_0 :cluster_13472440775_1666900747_11_1" />\n  <junction id=":cluster_13472440775_1666900747_15_0" type="internal" x="1329.26" y="1188.29" incLanes=":cluster_13472440775_1666900747_13_0 72016745#2_0 72016745#2_1" intLanes=":cluster_13472440775_1666900747_1_0 :cluster_13472440775_1666900747_2_0 :cluster_13472440775_1666900747_3_0 :cluster_13472440775_1666900747_4_0 :cluster_13472440775_1666900747_4_1 :cluster_13472440775_1666900747_8_0 :cluster_13472440775_1666900747_9_0" />\n  <junction id=":cluster_14188374630_2506088509_12_0" type="internal" x="1214.79" y="1504.46" incLanes=":cluster_14188374630_2506088509_5_0 243188508#0_0" intLanes=":cluster_14188374630_2506088509_1_0 :cluster_14188374630_2506088509_2_0 :cluster_14188374630_2506088509_7_0 :cluster_14188374630_2506088509_8_0 :cluster_14188374630_2506088509_9_0 :cluster_14188374630_2506088509_10_0" />\n  <junction id=":cluster_14188374630_2506088509_13_0" type="internal" x="1195.19" y="1508.43" incLanes=":cluster_14188374630_2506088509_11_0 -243188508#3_0" intLanes=":cluster_14188374630_2506088509_1_0 :cluster_14188374630_2506088509_2_0 :cluster_14188374630_2506088509_3_0 :cluster_14188374630_2506088509_4_0 :cluster_14188374630_2506088509_7_0 :cluster_14188374630_2506088509_8_0" />\n  <junction id=":cluster_1558314142_1558314144_12_0" type="internal" x="1368.53" y="701.66" incLanes=":cluster_1558314142_1558314144_2_0 706094740#0_0 :cluster_1558314142_1558314144_8_0" intLanes=":cluster_1558314142_1558314144_13_0 :cluster_1558314142_1558314144_4_0 :cluster_1558314142_1558314144_5_0 :cluster_1558314142_1558314144_6_0 :cluster_1558314142_1558314144_7_0 :cluster_1558314142_1558314144_8_0 :cluster_1558314142_1558314144_10_0 :cluster_1558314142_1558314144_11_0" />\n  <junction id=":cluster_1558314142_1558314144_13_0" type="internal" x="1384.94" y="712.35" incLanes=":cluster_1558314142_1558314144_8_0 -498671497_0" intLanes=":cluster_1558314142_1558314144_0_0 :cluster_1558314142_1558314144_1_0 :cluster_1558314142_1558314144_2_0 :cluster_1558314142_1558314144_4_0 :cluster_1558314142_1558314144_5_0 :cluster_1558314142_1558314144_10_0 :cluster_1558314142_1558314144_11_0" />\n  <junction id=":cluster_1559016910_4900465210_12_0" type="internal" x="871.71" y="406.76" incLanes=":cluster_1559016910_4900465210_2_0 142455600#0_0" intLanes=":cluster_1559016910_4900465210_4_0 :cluster_1559016910_4900465210_5_0 :cluster_1559016910_4900465210_6_0 :cluster_1559016910_4900465210_7_0 :cluster_1559016910_4900465210_10_0 :cluster_1559016910_4900465210_11_0" />\n  <junction id=":cluster_1559016910_4900465210_13_0" type="internal" x="870.89" y="434.03" incLanes=":cluster_1559016910_4900465210_8_0 -142455600#2_0" intLanes=":cluster_1559016910_4900465210_0_0 :cluster_1559016910_4900465210_1_0 :cluster_1559016910_4900465210_4_0 :cluster_1559016910_4900465210_5_0 :cluster_1559016910_4900465210_10_0 :cluster_1559016910_4900465210_11_0" />\n  <junction id=":cluster_1559016919_1680666950_12_0" type="internal" x="916.36" y="487.73" incLanes=":cluster_1559016919_1680666950_2_0 -706256380#6_0" intLanes=":cluster_1559016919_1680666950_4_0 :cluster_1559016919_1680666950_5_0 :cluster_1559016919_1680666950_6_0 :cluster_1559016919_1680666950_7_0 :cluster_1559016919_1680666950_10_0 :cluster_1559016919_1680666950_11_0" />\n  <junction id=":cluster_1559016919_1680666950_13_0" type="internal" x="915.26" y="498.98" incLanes=":cluster_1559016919_1680666950_8_0 706256380#4_0" intLanes=":cluster_1559016919_1680666950_0_0 :cluster_1559016919_1680666950_1_0 :cluster_1559016919_1680666950_4_0 :cluster_1559016919_1680666950_5_0 :cluster_1559016919_1680666950_10_0 :cluster_1559016919_1680666950_11_0" />\n  <junction id=":cluster_1680667233_2506085475_12_0" type="internal" x="1238.12" y="562.93" incLanes=":cluster_1680667233_2506085475_5_0 -95533432#3_0" intLanes=":cluster_1680667233_2506085475_1_0 :cluster_1680667233_2506085475_2_0 :cluster_1680667233_2506085475_7_0 :cluster_1680667233_2506085475_8_0 :cluster_1680667233_2506085475_9_0 :cluster_1680667233_2506085475_10_0" />\n  <junction id=":cluster_1680667233_2506085475_13_0" type="internal" x="1216.11" y="562.79" incLanes=":cluster_1680667233_2506085475_11_0 95533432#1_0" intLanes=":cluster_1680667233_2506085475_1_0 :cluster_1680667233_2506085475_2_0 :cluster_1680667233_2506085475_3_0 :cluster_1680667233_2506085475_4_0 :cluster_1680667233_2506085475_7_0 :cluster_1680667233_2506085475_8_0" />\n  <junction id=":cluster_1966399241_8523333972_9_0" type="internal" x="732.08" y="1133.25" incLanes=":cluster_1966399241_8523333972_5_0 -706104219#11_0" intLanes=":cluster_1966399241_8523333972_1_0 :cluster_1966399241_8523333972_2_0 :cluster_1966399241_8523333972_6_0 :cluster_1966399241_8523333972_7_0" />\n  <junction id=":cluster_1966399241_8523333972_10_0" type="internal" x="724.61" y="1132.70" incLanes=":cluster_1966399241_8523333972_8_0 706104219#9_0" intLanes=":cluster_1966399241_8523333972_1_0 :cluster_1966399241_8523333972_2_0 :cluster_1966399241_8523333972_3_0 :cluster_1966399241_8523333972_4_0" />\n  <junction id=":cluster_2505176169_856214432_7_0" type="internal" x="1512.60" y="992.44" incLanes=":cluster_2505176169_856214432_2_0 1270957860#0_0" intLanes=":cluster_2505176169_856214432_4_0 :cluster_2505176169_856214432_5_0 :cluster_2505176169_856214432_6_0" />\n  <junction id=":cluster_2505176422_2505176434_7_0" type="internal" x="1588.36" y="1426.71" incLanes=":cluster_2505176422_2505176434_6_0 970323546#3_0 970323546#3_1" intLanes=":cluster_2505176422_2505176434_1_0 :cluster_2505176422_2505176434_2_0 :cluster_2505176422_2505176434_3_0 :cluster_2505176422_2505176434_3_1" />\n  <junction id=":cluster_2506087462_8523307708_8_0" type="internal" x="307.52" y="1228.97" incLanes=":cluster_2506087462_8523307708_5_0 243188646#1_0" intLanes=":cluster_2506087462_8523307708_0_0 :cluster_2506087462_8523307708_2_0 :cluster_2506087462_8523307708_2_1 :cluster_2506087462_8523307708_2_2" />\n  <junction id=":cluster_2506088039_8523406713_9_0" type="internal" x="655.27" y="1397.14" incLanes=":cluster_2506088039_8523406713_4_0 -706104217#16_0" intLanes=":cluster_2506088039_8523406713_1_0 :cluster_2506088039_8523406713_2_0 :cluster_2506088039_8523406713_6_0 :cluster_2506088039_8523406713_7_0 :cluster_2506088039_8523406713_8_0" />\n  <junction id=":cluster_2507400612_2507400618_12_0" type="internal" x="1651.01" y="870.95" incLanes=":cluster_2507400612_2507400618_2_0 243314436#0_0" intLanes=":cluster_2507400612_2507400618_4_0 :cluster_2507400612_2507400618_5_0 :cluster_2507400612_2507400618_6_0 :cluster_2507400612_2507400618_7_0 :cluster_2507400612_2507400618_10_0 :cluster_2507400612_2507400618_11_0" />\n  <junction id=":cluster_2507400612_2507400618_13_0" type="internal" x="1649.74" y="892.46" incLanes=":cluster_2507400612_2507400618_8_0 -243314436#2_0" intLanes=":cluster_2507400612_2507400618_0_0 :cluster_2507400612_2507400618_1_0 :cluster_2507400612_2507400618_4_0 :cluster_2507400612_2507400618_5_0 :cluster_2507400612_2507400618_10_0 :cluster_2507400612_2507400618_11_0" />\n  <connection from="-100401320" to="-95542482#11" fromLane="0" toLane="0" via=":1110416469_2_0" dir="l" state="m" />\n  <connection from="-100401320" to="95542482#12" fromLane="0" toLane="0" via=":1110416469_3_0" dir="r" state="m" />\n  <connection from="-1204844051" to="-1204844052" fromLane="0" toLane="0" via=":4900464584_2_0" dir="s" state="M" />\n  <connection from="-1204844052" to="96941849" fromLane="0" toLane="0" via=":1123399600_6_0" dir="l" state="M" />\n  <connection from="-1204844052" to="-185951074#3" fromLane="0" toLane="0" via=":1123399600_7_0" dir="s" state="M" />\n  <connection from="-1204844052" to="-498671615" fromLane="0" toLane="0" via=":1123399600_8_0" dir="r" state="m" />\n  <connection from="-1218747872" to="-243079757#3" fromLane="0" toLane="0" via=":1639088658_5_0" dir="s" state="m" />\n  <connection from="-1218747872" to="151072453#2" fromLane="0" toLane="0" via=":1639088658_6_0" dir="r" state="m" />\n  <connection from="-1218747879" to="970323546#7" fromLane="0" toLane="0" via=":2505176258_0_0" dir="l" state="m" />\n  <connection from="-1218747879" to="-66984610#0" fromLane="0" toLane="0" via=":2505176258_1_0" dir="s" state="m" />\n  <connection from="-122540020#0" to="-72016745#1" fromLane="0" toLane="0" via=":1369020950_3_0" dir="l" state="m" />\n  <connection from="-122540020#0" to="72016745#2" fromLane="0" toLane="1" via=":1369020950_4_0" dir="r" state="m" />\n  <connection from="-122540020#2" to="-122540020#0" fromLane="0" toLane="0" via=":14165547215_0_0" dir="s" state="M" />\n  <connection from="-122540020#2" to="1557537524" fromLane="0" toLane="0" via=":14165547215_1_0" dir="r" state="m" />\n  <connection from="-122540020#3" to="243188283" fromLane="0" toLane="0" via=":1369020943_2_0" dir="l" state="M" />\n  <connection from="-122540020#3" to="-122540020#2" fromLane="0" toLane="0" via=":1369020943_3_0" dir="s" state="M" />\n  <connection from="-122540020#4" to="243188139" fromLane="0" toLane="0" via=":2506088023_2_0" dir="l" state="M" />\n  <connection from="-122540020#4" to="-122540020#3" fromLane="0" toLane="0" via=":2506088023_3_0" dir="s" state="M" />\n  <connection from="-122540020#5" to="243188167" fromLane="0" toLane="0" via=":2506088274_2_0" dir="l" state="M" />\n  <connection from="-122540020#5" to="-122540020#4" fromLane="0" toLane="0" via=":2506088274_3_0" dir="s" state="M" />\n  <connection from="-122540020#6" to="243188092" fromLane="0" toLane="0" via=":2506088528_2_0" dir="l" state="M" />\n  <connection from="-122540020#6" to="-122540020#5" fromLane="0" toLane="0" via=":2506088528_3_0" dir="s" state="M" />\n  <connection from="-122540020#7" to="243188056" fromLane="0" toLane="0" via=":1369020942_2_0" dir="l" state="M" />\n  <connection from="-122540020#7" to="-122540020#6" fromLane="0" toLane="0" via=":1369020942_3_0" dir="s" state="M" />\n  <connection from="-1270957863" to="1227721991#0" fromLane="0" toLane="0" via=":cluster_2505176169_856214432_3_0" dir="l" state="m" />\n  <connection from="-1270957863" to="1270957862" fromLane="0" toLane="1" via=":cluster_2505176169_856214432_4_0" dir="r" state="m" />\n  <connection from="-129827552#0" to="96837172#1" fromLane="0" toLane="0" via=":1431715395_2_0" dir="l" state="m" />\n  <connection from="-129827552#0" to="-96837172#0" fromLane="0" toLane="0" via=":1431715395_3_0" dir="r" state="m" />\n  <connection from="-129827552#1" to="155690791#0" fromLane="0" toLane="0" via=":1680666546_2_0" dir="l" state="M" />\n  <connection from="-129827552#1" to="-129827552#0" fromLane="0" toLane="0" via=":1680666546_3_0" dir="s" state="m" />\n  <connection from="-129827554#0" to="148941242#1" fromLane="0" toLane="0" via=":1110416726_9_0" dir="l" state="M" />\n  <connection from="-129827554#0" to="-142455604#1" fromLane="0" toLane="0" via=":1110416726_10_0" dir="s" state="M" />\n  <connection from="-129827554#0" to="-148941242#0" fromLane="0" toLane="0" via=":1110416726_11_0" dir="r" state="m" />\n  <connection from="-129827554#4" to="155690741#0" fromLane="0" toLane="0" via=":1680666468_4_0" dir="l" state="M" />\n  <connection from="-129827554#4" to="-129827554#0" fromLane="0" toLane="0" via=":1680666468_5_0" dir="s" state="M" />\n  <connection from="-129827558" to="706096643#0" fromLane="0" toLane="0" via=":1122072673_0_0" dir="l" state="M" />\n  <connection from="-129827558" to="-96837172#1" fromLane="0" toLane="0" via=":1122072673_1_0" dir="s" state="M" />\n  <connection from="-129827561#0" to="-1454593202#1" fromLane="0" toLane="0" via=":1431715415_2_0" dir="l" state="m" />\n  <connection from="-129827561#0" to="1454593202#2" fromLane="0" toLane="0" via=":1431715415_3_0" dir="r" state="m" />\n  <connection from="-129827561#1" to="1559841160#0" fromLane="0" toLane="0" via=":14188764645_4_0" dir="l" state="M" />\n  <connection from="-129827561#1" to="-129827561#0" fromLane="0" toLane="0" via=":14188764645_5_0" dir="s" state="M" />\n  <connection from="-129842388#0" to="932339002#5" fromLane="0" toLane="0" via=":1431867160_0_0" dir="l" state="m" />\n  <connection from="-129842388#1" to="1457057267" fromLane="0" toLane="0" via=":1431867108_0_0" dir="l" state="M" />\n  <connection from="-129842388#1" to="-129842388#0" fromLane="0" toLane="0" via=":1431867108_1_0" dir="s" state="M" />\n  <connection from="-129842388#2" to="-155691088#0" fromLane="0" toLane="0" via=":1680666463_3_0" dir="l" state="M" />\n  <connection from="-129842388#2" to="-129842388#1" fromLane="0" toLane="0" via=":1680666463_4_0" dir="s" state="M" />\n  <connection from="-129842388#2" to="155691088#1" fromLane="0" toLane="0" via=":1680666463_5_0" dir="r" state="m" />\n  <connection from="-129842397#1" to="-129842397#0" fromLane="0" toLane="0" via=":1431867103_4_0" dir="s" state="M" />\n  <connection from="-129842397#1" to="-706087699#1" fromLane="0" toLane="0" via=":1431867103_5_0" dir="r" state="m" />\n  <connection from="-129842397#3" to="932324369" fromLane="0" toLane="0" via=":8642733850_4_0" dir="l" state="M" />\n  <connection from="-129842397#3" to="-129842397#1" fromLane="0" toLane="0" via=":8642733850_5_0" dir="s" state="M" />\n  <connection from="-129842397#4" to="-129842397#3" fromLane="0" toLane="0" via=":1431867150_4_0" dir="l" state="M" />\n  <connection from="-129842397#4" to="142395239#0" fromLane="0" toLane="0" via=":1431867150_5_0" dir="s" state="M" />\n  <connection from="-1316153439" to="243188536#0" fromLane="0" toLane="0" via=":2506087563_2_0" dir="l" state="M" />\n  <connection from="-1316153439" to="243188646#0" fromLane="0" toLane="0" via=":2506087563_3_0" dir="s" state="m" />\n  <connection from="-1334274432" to="706093504#2" fromLane="0" toLane="0" via=":12344163778_2_0" dir="r" state="m" />\n  <connection from="-1365892748#1" to="-706104219#12" fromLane="0" toLane="0" via=":1110416444_3_0" dir="l" state="m" />\n  <connection from="-1365892748#1" to="706092523#0" fromLane="0" toLane="0" via=":1110416444_4_0" dir="s" state="m" />\n  <connection from="-1365892748#1" to="706104219#13" fromLane="0" toLane="0" via=":1110416444_5_0" dir="r" state="m" />\n  <connection from="-1365935945#0" to="969757849#2" fromLane="0" toLane="0" via=":856214538_0_0" dir="l" state="m" />\n  <connection from="-1365935945#0" to="-498671388#1" fromLane="0" toLane="0" via=":856214538_1_0" dir="s" state="m" />\n  <connection from="-1365935945#0" to="-969757849#1" fromLane="0" toLane="0" via=":856214538_2_0" dir="r" state="m" />\n  <connection from="-1365935945#1" to="498671465" fromLane="0" toLane="0" via=":4900465166_0_0" dir="l" state="M" />\n  <connection from="-1365935945#1" to="-1365935945#0" fromLane="0" toLane="0" via=":4900465166_1_0" dir="s" state="M" />\n  <connection from="-1365935945#1" to="498671393" fromLane="0" toLane="0" via=":4900465166_2_0" dir="r" state="m" />\n  <connection from="-142395239#0" to="129842397#4" fromLane="0" toLane="0" via=":1431867150_0_0" dir="s" state="M" />\n  <connection from="-142395239#0" to="-129842397#3" fromLane="0" toLane="0" via=":1431867150_1_0" dir="r" state="m" />\n  <connection from="-142453602#0" to="969757849#0" fromLane="0" toLane="0" via=":1115585385_6_0" dir="l" state="m" />\n  <connection from="-142453602#0" to="498671395#0" fromLane="0" toLane="0" via=":1115585385_7_0" dir="s" state="m" />\n  <connection from="-142453602#0" to="-1204844051" fromLane="0" toLane="0" via=":1115585385_8_0" dir="r" state="m" />\n  <connection from="-142453602#1" to="-498671393" fromLane="0" toLane="0" via=":1115585909_4_0" dir="l" state="M" />\n  <connection from="-142453602#1" to="-142453602#0" fromLane="0" toLane="0" via=":1115585909_5_0" dir="s" state="M" />\n  <connection from="-142453602#2" to="-142453602#1" fromLane="0" toLane="0" via=":1115585463_4_0" dir="s" state="M" />\n  <connection from="-142453602#2" to="243188024" fromLane="0" toLane="0" via=":1115585463_5_0" dir="r" state="m" />\n  <connection from="-142455590" to="-148941242#3" fromLane="0" toLane="0" via=":1559016909_3_0" dir="l" state="m" />\n  <connection from="-142455590" to="-155690741#2" fromLane="0" toLane="0" via=":1559016909_4_0" dir="s" state="m" />\n  <connection from="-142455590" to="148941242#4" fromLane="0" toLane="0" via=":1559016909_5_0" dir="r" state="m" />\n  <connection from="-142455592#1" to="1457057266#0" fromLane="0" toLane="0" via=":1431867166_2_0" dir="s" state="m" />\n  <connection from="-142455592#1" to="-129842388#2" fromLane="0" toLane="0" via=":1431867166_3_0" dir="r" state="m" />\n  <connection from="-142455595" to="-148941242#2" fromLane="0" toLane="0" via=":1559016904_2_0" dir="l" state="m" />\n  <connection from="-142455595" to="148941242#3" fromLane="0" toLane="0" via=":1559016904_3_0" dir="r" state="m" />\n  <connection from="-142455600#0" to="706256386" fromLane="0" toLane="0" via=":1559016918_0_0" dir="r" state="m" />\n  <connection from="-142455600#2" to="-142455590" fromLane="0" toLane="0" via=":cluster_1559016910_4900465210_0_0" dir="l" state="M" />\n  <connection from="-142455600#2" to="-142455600#0" fromLane="0" toLane="0" via=":cluster_1559016910_4900465210_1_0" dir="s" state="M" />\n  <connection from="-142455600#2" to="498671483" fromLane="0" toLane="0" via=":cluster_1559016910_4900465210_2_0" dir="r" state="m" />\n  <connection from="-142455600#3" to="-142455595" fromLane="0" toLane="0" via=":1559016905_0_0" dir="l" state="M" />\n  <connection from="-142455600#3" to="-142455600#2" fromLane="0" toLane="0" via=":1559016905_1_0" dir="s" state="M" />\n  <connection from="-142455604#1" to="95809262#1" fromLane="0" toLane="0" via=":1559016903_0_0" dir="l" state="M" />\n  <connection from="-142455604#1" to="-142455600#3" fromLane="0" toLane="0" via=":1559016903_1_0" dir="s" state="M" />\n  <connection from="-142455604#1" to="-95809262#0" fromLane="0" toLane="0" via=":1559016903_2_0" dir="r" state="m" />\n  <connection from="-1447303746" to="185951074#1" fromLane="0" toLane="0" via=":13279030938_0_0" dir="l" state="m" />\n  <connection from="-1447303746" to="-185951074#0" fromLane="0" toLane="0" via=":13279030938_1_0" dir="r" state="m" />\n  <connection from="-1454593202#0" to="1454593203#3" fromLane="0" toLane="0" via=":13339819885_2_0" dir="l" state="m" visibility="9.00" />\n  <connection from="-1454593202#1" to="-1559841160#1" fromLane="0" toLane="0" via=":14188764647_4_0" dir="l" state="M" />\n  <connection from="-1454593202#1" to="-1454593202#0" fromLane="0" toLane="0" via=":14188764647_5_0" dir="s" state="M" />\n  <connection from="-1454593202#2" to="129827561#0" fromLane="0" toLane="0" via=":1431715415_4_0" dir="l" state="M" />\n  <connection from="-1454593202#2" to="-1454593202#1" fromLane="0" toLane="0" via=":1431715415_5_0" dir="s" state="M" />\n  <connection from="-1457057266#1" to="-129842388#2" fromLane="0" toLane="0" via=":1431867166_0_0" dir="l" state="M" />\n  <connection from="-1457057266#1" to="142455592#0" fromLane="0" toLane="0" via=":1431867166_1_0" dir="s" state="m" />\n  <connection from="-1476708031" to="498671391" fromLane="0" toLane="0" via=":4900464587_0_0" dir="l" state="M" />\n  <connection from="-1476708031" to="530314558#1" fromLane="0" toLane="0" via=":4900464587_1_0" dir="s" state="M" />\n  <connection from="-148941242#1" to="-142455604#1" fromLane="0" toLane="0" via=":1110416726_6_0" dir="l" state="m" />\n  <connection from="-148941242#1" to="-148941242#0" fromLane="0" toLane="0" via=":1110416726_7_0" dir="s" state="m" />\n  <connection from="-148941242#1" to="129827554#0" fromLane="0" toLane="0" via=":1110416726_8_0" dir="r" state="m" />\n  <connection from="-148941242#2" to="-95809262#1" fromLane="0" toLane="0" via=":1431715372_4_0" dir="l" state="M" />\n  <connection from="-148941242#2" to="-148941242#1" fromLane="0" toLane="0" via=":1431715372_5_0" dir="s" state="M" />\n  <connection from="-148941242#3" to="142455595" fromLane="0" toLane="0" via=":1559016904_4_0" dir="l" state="M" />\n  <connection from="-148941242#3" to="-148941242#2" fromLane="0" toLane="0" via=":1559016904_5_0" dir="s" state="M" />\n  <connection from="-148941242#4" to="142455590" fromLane="0" toLane="0" via=":1559016909_6_0" dir="l" state="M" />\n  <connection from="-148941242#4" to="-148941242#3" fromLane="0" toLane="0" via=":1559016909_7_0" dir="s" state="M" />\n  <connection from="-148941242#4" to="-155690741#2" fromLane="0" toLane="0" via=":1559016909_8_0" dir="r" state="m" />\n  <connection from="-148941242#5" to="142455599" fromLane="0" toLane="0" via=":1559016916_2_0" dir="l" state="M" />\n  <connection from="-148941242#5" to="-148941242#4" fromLane="0" toLane="0" via=":1559016916_3_0" dir="s" state="M" />\n  <connection from="-151072451#2" to="151072453#3" fromLane="0" toLane="0" via=":1639088623_0_0" dir="l" state="M" />\n  <connection from="-154097280" to="96242763#2" fromLane="0" toLane="0" via=":1115585403_2_0" dir="r" state="m" />\n  <connection from="-1546894905" to="40264697#5" fromLane="0" toLane="0" via=":14069947700_0_0" dir="l" state="m" />\n  <connection from="-1546896992" to="706097788#1" fromLane="0" toLane="0" via=":14069981565_2_0" dir="r" state="m" />\n  <connection from="-1546907364" to="-155690791#0" fromLane="0" toLane="0" via=":14070071629_0_0" dir="r" state="M" />\n  <connection from="-1552748146" to="243188508#0" fromLane="0" toLane="0" via=":2506088518_4_0" dir="s" state="M" />\n  <connection from="-1552748146" to="243188565" fromLane="0" toLane="0" via=":2506088518_5_0" dir="r" state="m" />\n  <connection from="-1552748564#1" to="-1552748564#0" fromLane="0" toLane="0" via=":4900465706_2_0" dir="s" state="M" />\n  <connection from="-1552748564#1" to="-498671328#1" fromLane="0" toLane="0" via=":4900465706_3_0" dir="r" state="m" />\n  <connection from="-1556742612" to="969757851#2" fromLane="0" toLane="0" via=":14159239425_0_0" dir="l" state="m" />\n  <connection from="-1556742613" to="-969765780#1" fromLane="0" toLane="0" via=":14159239427_2_0" dir="l" state="m" />\n  <connection from="-1556742613" to="969765780#2" fromLane="0" toLane="0" via=":14159239427_3_0" dir="r" state="m" />\n  <connection from="-155690340" to="-706256380#3" fromLane="0" toLane="0" via=":1680666686_3_0" dir="l" state="m" />\n  <connection from="-155690340" to="-498671483" fromLane="0" toLane="0" via=":1680666686_4_0" dir="s" state="m" />\n  <connection from="-155690340" to="706256380#4" fromLane="0" toLane="0" via=":1680666686_5_0" dir="r" state="m" />\n  <connection from="-155690480#1" to="-155690480#0" fromLane="0" toLane="0" via=":4900465244_0_0" dir="s" state="M" />\n  <connection from="-155690480#1" to="-498671497" fromLane="0" toLane="0" via=":4900465244_1_0" dir="r" state="m" />\n  <connection from="-155690513" to="498671497" fromLane="0" toLane="0" via=":cluster_1558314142_1558314144_3_0" dir="l" state="m" />\n  <connection from="-155690513" to="706094740#2" fromLane="0" toLane="0" via=":cluster_1558314142_1558314144_4_0" dir="s" state="m" />\n  <connection from="-155690513" to="-706094740#0" fromLane="0" toLane="0" via=":cluster_1558314142_1558314144_5_0" dir="r" state="m" />\n  <connection from="-155690734#1" to="-155691015#0" fromLane="0" toLane="0" via=":1680667079_0_0" dir="l" state="M" />\n  <connection from="-155690734#1" to="-155690734#0" fromLane="0" toLane="0" via=":1680667079_1_0" dir="s" state="M" />\n  <connection from="-155690734#1" to="155691015#1" fromLane="0" toLane="0" via=":1680667079_2_0" dir="r" state="m" />\n  <connection from="-155690741#1" to="-129827554#0" fromLane="0" toLane="0" via=":1680666468_2_0" dir="l" state="m" />\n  <connection from="-155690741#1" to="129827554#1" fromLane="0" toLane="0" via=":1680666468_3_0" dir="r" state="m" />\n  <connection from="-155690741#2" to="1559020882" fromLane="0" toLane="0" via=":14182422401_2_0" dir="l" state="M" />\n  <connection from="-155690741#2" to="-155690741#1" fromLane="0" toLane="0" via=":14182422401_3_0" dir="s" state="M" />\n  <connection from="-155690791#0" to="-129827552#0" fromLane="0" toLane="0" via=":1680666546_0_0" dir="l" state="M" />\n  <connection from="-155690791#0" to="129827552#1" fromLane="0" toLane="0" via=":1680666546_1_0" dir="r" state="M" />\n  <connection from="-155690812#1" to="-155691015#1" fromLane="0" toLane="0" via=":1680667164_0_0" dir="l" state="M" />\n  <connection from="-155690812#1" to="-155690812#0" fromLane="0" toLane="0" via=":1680667164_1_0" dir="s" state="M" />\n  <connection from="-155690878#0" to="-706256380#4" fromLane="0" toLane="0" via=":cluster_1559016919_1680666950_3_0" dir="l" state="m" />\n  <connection from="-155690878#0" to="-706256386" fromLane="0" toLane="0" via=":cluster_1559016919_1680666950_4_0" dir="s" state="m" />\n  <connection from="-155690878#0" to="706256380#6" fromLane="0" toLane="0" via=":cluster_1559016919_1680666950_5_0" dir="r" state="m" />\n  <connection from="-155690878#1" to="155691032" fromLane="0" toLane="0" via=":1680667029_2_0" dir="l" state="M" />\n  <connection from="-155690878#1" to="-155690878#0" fromLane="0" toLane="0" via=":1680667029_3_0" dir="s" state="M" />\n  <connection from="-155690968" to="-706256380#0" fromLane="0" toLane="0" via=":1680666322_2_0" dir="l" state="m" />\n  <connection from="-155690968" to="706256380#1" fromLane="0" toLane="0" via=":1680666322_3_0" dir="r" state="m" />\n  <connection from="-155690981" to="-706256380#1" fromLane="0" toLane="0" via=":cluster_1110416379_1680666461_3_0" dir="l" state="m" />\n  <connection from="-155690981" to="95809262#0" fromLane="0" toLane="0" via=":cluster_1110416379_1680666461_4_0" dir="s" state="m" />\n  <connection from="-155690981" to="706256380#3" fromLane="0" toLane="0" via=":cluster_1110416379_1680666461_5_0" dir="r" state="m" />\n  <connection from="-155691015#0" to="155691084#2" fromLane="0" toLane="0" via=":1680666827_2_0" dir="l" state="m" />\n  <connection from="-155691015#0" to="-155691084#1" fromLane="0" toLane="0" via=":1680666827_3_0" dir="r" state="m" />\n  <connection from="-155691015#1" to="155690734#1" fromLane="0" toLane="0" via=":1680667079_3_0" dir="l" state="m" />\n  <connection from="-155691015#1" to="-155691015#0" fromLane="0" toLane="0" via=":1680667079_4_0" dir="s" state="m" />\n  <connection from="-155691015#1" to="-155690734#0" fromLane="0" toLane="0" via=":1680667079_5_0" dir="r" state="m" />\n  <connection from="-155691022#0" to="155691084#3" fromLane="0" toLane="0" via=":1680666891_2_0" dir="l" state="m" />\n  <connection from="-155691022#0" to="-155691084#2" fromLane="0" toLane="0" via=":1680666891_3_0" dir="r" state="m" />\n  <connection from="-155691022#1" to="-498671433" fromLane="0" toLane="0" via=":4900465071_4_0" dir="l" state="M" />\n  <connection from="-155691022#1" to="-155691022#0" fromLane="0" toLane="0" via=":4900465071_5_0" dir="s" state="M" />\n  <connection from="-155691032" to="-155690878#0" fromLane="0" toLane="0" via=":1680667029_0_0" dir="l" state="m" />\n  <connection from="-155691032" to="155690878#1" fromLane="0" toLane="0" via=":1680667029_1_0" dir="r" state="m" />\n  <connection from="-155691084#0" to="-932339005" fromLane="0" toLane="0" via=":8642866010_0_0" dir="s" state="M" />\n  <connection from="-155691084#1" to="705855017#1" fromLane="0" toLane="0" via=":4900465949_0_0" dir="l" state="M" />\n  <connection from="-155691084#1" to="-155691084#0" fromLane="0" toLane="0" via=":4900465949_1_0" dir="s" state="M" />\n  <connection from="-155691084#2" to="-155691084#1" fromLane="0" toLane="0" via=":1680666827_0_0" dir="s" state="M" />\n  <connection from="-155691084#2" to="155691015#0" fromLane="0" toLane="0" via=":1680666827_1_0" dir="r" state="m" />\n  <connection from="-155691084#3" to="-155691084#2" fromLane="0" toLane="0" via=":1680666891_0_0" dir="s" state="M" />\n  <connection from="-155691084#3" to="155691022#0" fromLane="0" toLane="0" via=":1680666891_1_0" dir="r" state="m" />\n  <connection from="-155691088#1" to="129842388#2" fromLane="0" toLane="0" via=":1680666463_6_0" dir="l" state="m" />\n  <connection from="-155691088#1" to="-155691088#0" fromLane="0" toLane="0" via=":1680666463_7_0" dir="s" state="m" />\n  <connection from="-155691088#1" to="-129842388#1" fromLane="0" toLane="0" via=":1680666463_8_0" dir="r" state="m" />\n  <connection from="-1557537524" to="122540020#1" fromLane="0" toLane="0" via=":14165547215_2_0" dir="l" state="m" />\n  <connection from="-1557537524" to="-122540020#0" fromLane="0" toLane="0" via=":14165547215_3_0" dir="r" state="m" />\n  <connection from="-1558734968" to="-498671328#0" fromLane="0" toLane="0" via=":14179759489_0_0" dir="l" state="m" />\n  <connection from="-1558734968" to="498671328#1" fromLane="0" toLane="0" via=":14179759489_1_0" dir="r" state="m" />\n  <connection from="-1559020882" to="-155690741#1" fromLane="0" toLane="0" via=":14182422401_0_0" dir="l" state="m" />\n  <connection from="-1559020882" to="155690741#2" fromLane="0" toLane="0" via=":14182422401_1_0" dir="r" state="m" />\n  <connection from="-1559802495#0" to="243188552#2" fromLane="0" toLane="0" via=":4900465288_4_0" dir="l" state="m" />\n  <connection from="-1559802495#0" to="-243188552#1" fromLane="0" toLane="0" via=":4900465288_5_0" dir="r" state="m" />\n  <connection from="-1559802495#1" to="-1559802495#0" fromLane="0" toLane="0" via=":14188374629_2_0" dir="s" state="M" />\n  <connection from="-1559802495#1" to="1559802496" fromLane="0" toLane="0" via=":14188374629_3_0" dir="r" state="m" />\n  <connection from="-1559802496" to="1559802495#1" fromLane="0" toLane="0" via=":14188374629_4_0" dir="l" state="m" />\n  <connection from="-1559802496" to="-1559802495#0" fromLane="0" toLane="0" via=":14188374629_5_0" dir="r" state="m" />\n  <connection from="-1559841160#0" to="-129827561#0" fromLane="0" toLane="0" via=":14188764645_2_0" dir="l" state="m" />\n  <connection from="-1559841160#0" to="129827561#1" fromLane="0" toLane="0" via=":14188764645_3_0" dir="r" state="m" />\n  <connection from="-1559841160#1" to="-1559841160#0" fromLane="0" toLane="0" via=":14188764646_0_0" dir="l" state="m" />\n  <connection from="-1559841160#1" to="1559841161" fromLane="0" toLane="0" via=":14188764646_1_0" dir="r" state="m" />\n  <connection from="-1559841161" to="1559841160#1" fromLane="0" toLane="0" via=":14188764646_2_0" dir="l" state="M" />\n  <connection from="-1559841161" to="-1559841160#0" fromLane="0" toLane="0" via=":14188764646_3_0" dir="s" state="M" />\n  <connection from="-185951074#0" to="-706105951#2" fromLane="0" toLane="0" via=":856214524_6_0" dir="l" state="M" />\n  <connection from="-185951074#0" to="-1270957863" fromLane="0" toLane="0" via=":856214524_7_0" dir="s" state="M" />\n  <connection from="-185951074#0" to="706105951#3" fromLane="0" toLane="0" via=":856214524_8_0" dir="r" state="m" />\n  <connection from="-185951074#1" to="-185951074#0" fromLane="0" toLane="0" via=":13279030938_4_0" dir="s" state="M" />\n  <connection from="-185951074#1" to="1447303746" fromLane="0" toLane="0" via=":13279030938_5_0" dir="r" state="m" />\n  <connection from="-185951074#3" to="-185951074#1" fromLane="0" toLane="0" via=":4900465707_1_0" dir="s" state="M" />\n  <connection from="-243079673" to="66984613#3" fromLane="0" toLane="0" via=":947896979_2_0" dir="r" state="m" />\n  <connection from="-243079775" to="970323546#2" fromLane="0" toLane="0" via=":2505176514_0_0" dir="l" state="m" />\n  <connection from="-243188024" to="142453602#2" fromLane="0" toLane="0" via=":1115585463_0_0" dir="l" state="m" />\n  <connection from="-243188024" to="-142453602#1" fromLane="0" toLane="0" via=":1115585463_1_0" dir="r" state="m" />\n  <connection from="-243188056" to="-122540020#6" fromLane="0" toLane="0" via=":1369020942_0_0" dir="l" state="m" />\n  <connection from="-243188056" to="122540020#7" fromLane="0" toLane="0" via=":1369020942_1_0" dir="r" state="m" />\n  <connection from="-243188085" to="684741141#3" fromLane="0" toLane="0" via=":1123399486_0_0" dir="l" state="m" />\n  <connection from="-243188085" to="-684741141#2" fromLane="0" toLane="0" via=":1123399486_1_0" dir="r" state="m" />\n  <connection from="-243188092" to="-122540020#5" fromLane="0" toLane="0" via=":2506088528_0_0" dir="l" state="m" />\n  <connection from="-243188092" to="122540020#6" fromLane="0" toLane="0" via=":2506088528_1_0" dir="r" state="m" />\n  <connection from="-243188103" to="-243188536#1" fromLane="0" toLane="0" via=":2506087304_4_0" dir="l" state="m" />\n  <connection from="-243188103" to="243188536#2" fromLane="0" toLane="0" via=":2506087304_5_0" dir="r" state="m" />\n  <connection from="-243188115" to="497290263#3" fromLane="0" toLane="0" via=":1123399713_2_0" dir="l" state="m" />\n  <connection from="-243188115" to="-497290263#2" fromLane="0" toLane="0" via=":1123399713_3_0" dir="r" state="m" />\n  <connection from="-243188139" to="-122540020#3" fromLane="0" toLane="0" via=":2506088023_0_0" dir="l" state="m" />\n  <connection from="-243188139" to="122540020#4" fromLane="0" toLane="0" via=":2506088023_1_0" dir="r" state="m" />\n  <connection from="-243188161" to="684741141#1" fromLane="0" toLane="0" via=":1123399675_0_0" dir="l" state="m" />\n  <connection from="-243188161" to="-684741141#0" fromLane="0" toLane="0" via=":1123399675_1_0" dir="r" state="m" />\n  <connection from="-243188167" to="-122540020#4" fromLane="0" toLane="0" via=":2506088274_0_0" dir="l" state="m" />\n  <connection from="-243188167" to="122540020#5" fromLane="0" toLane="0" via=":2506088274_1_0" dir="r" state="m" />\n  <connection from="-243188174#1" to="706095308#1" fromLane="0" toLane="0" via=":6634630596_0_0" dir="l" state="M" />\n  <connection from="-243188174#1" to="-243188174#0" fromLane="0" toLane="0" via=":6634630596_1_0" dir="s" state="M" />\n  <connection from="-243188174#1" to="-706095308#0" fromLane="0" toLane="0" via=":6634630596_2_0" dir="r" state="m" />\n  <connection from="-243188174#2" to="498671673" fromLane="0" toLane="0" via=":4900465989_0_0" dir="l" state="M" />\n  <connection from="-243188174#2" to="-243188174#1" fromLane="0" toLane="0" via=":4900465989_1_0" dir="s" state="M" />\n  <connection from="-243188196" to="96941841#2" fromLane="0" toLane="0" via=":1123399513_2_0" dir="l" state="m" />\n  <connection from="-243188196" to="-96941841#1" fromLane="0" toLane="0" via=":1123399513_3_0" dir="r" state="m" />\n  <connection from="-243188212#0" to="-243188101" fromLane="0" toLane="0" via=":12351390200_0_0" dir="s" state="M" />\n  <connection from="-243188212#1" to="-706094741#0" fromLane="0" toLane="0" via=":1123399641_3_0" dir="l" state="M" />\n  <connection from="-243188212#1" to="-243188212#0" fromLane="0" toLane="0" via=":1123399641_4_0" dir="s" state="M" />\n  <connection from="-243188212#1" to="706094741#1" fromLane="0" toLane="0" via=":1123399641_5_0" dir="r" state="m" />\n  <connection from="-243188225" to="-452036923#7" fromLane="0" toLane="0" via=":1123399683_4_0" dir="l" state="M" />\n  <connection from="-243188225" to="243188705" fromLane="0" toLane="0" via=":1123399683_5_0" dir="s" state="M" />\n  <connection from="-243188243" to="-243188389#3" fromLane="0" toLane="0" via=":2506090118_0_0" dir="l" state="M" />\n  <connection from="-243188273#1" to="-95542482#9" fromLane="0" toLane="0" via=":1110416526_2_0" dir="l" state="m" />\n  <connection from="-243188273#1" to="95542482#10" fromLane="0" toLane="0" via=":1110416526_3_0" dir="r" state="m" />\n  <connection from="-243188283" to="-122540020#2" fromLane="0" toLane="0" via=":1369020943_0_0" dir="l" state="m" />\n  <connection from="-243188283" to="122540020#3" fromLane="0" toLane="0" via=":1369020943_1_0" dir="r" state="m" />\n  <connection from="-243188351#1" to="-498671616" fromLane="0" toLane="0" via=":2506085631_0_0" dir="s" state="M" />\n  <connection from="-243188351#1" to="498671639" fromLane="0" toLane="0" via=":2506085631_1_0" dir="r" state="m" />\n  <connection from="-243188361" to="684741141#2" fromLane="0" toLane="0" via=":1123399943_0_0" dir="l" state="m" />\n  <connection from="-243188361" to="-684741141#1" fromLane="0" toLane="0" via=":1123399943_1_0" dir="r" state="m" />\n  <connection from="-243188389#0" to="-706104217#11" fromLane="0" toLane="0" via=":12339345632_2_0" dir="l" state="m" />\n  <connection from="-243188389#0" to="706104217#12" fromLane="0" toLane="0" via=":12339345632_3_0" dir="r" state="m" />\n  <connection from="-243188389#1" to="-243188389#0" fromLane="0" toLane="0" via=":12346099051_5_0" dir="s" state="m" />\n  <connection from="-243188389#1" to="95809267#1" fromLane="0" toLane="0" via=":12346099051_6_0" dir="r" state="m" />\n  <connection from="-243188389#2" to="-243188389#1" fromLane="0" toLane="0" via=":2506088491_4_0" dir="s" state="M" />\n  <connection from="-243188389#2" to="-498671589" fromLane="0" toLane="0" via=":2506088491_5_0" dir="r" state="m" />\n  <connection from="-243188389#3" to="-243188389#2" fromLane="0" toLane="0" via=":2506088494_2_0" dir="s" state="M" />\n  <connection from="-243188389#3" to="498671331" fromLane="0" toLane="0" via=":2506088494_3_0" dir="r" state="m" />\n  <connection from="-243188445" to="497290263#4" fromLane="0" toLane="0" via=":1123399740_2_0" dir="l" state="m" />\n  <connection from="-243188445" to="-497290263#3" fromLane="0" toLane="0" via=":1123399740_3_0" dir="r" state="m" />\n  <connection from="-243188458" to="-95533432#1" fromLane="0" toLane="0" via=":cluster_1680667233_2506085475_6_0" dir="l" state="m" />\n  <connection from="-243188458" to="-155690875" fromLane="0" toLane="0" via=":cluster_1680667233_2506085475_7_0" dir="s" state="m" />\n  <connection from="-243188458" to="95533432#3" fromLane="0" toLane="0" via=":cluster_1680667233_2506085475_8_0" dir="r" state="m" />\n  <connection from="-243188465#0" to="931354569#3" fromLane="0" toLane="0" via=":cluster_13424220808_2506085647_0_0" dir="l" state="m" />\n  <connection from="-243188465#0" to="-931354569#1" fromLane="0" toLane="0" via=":cluster_13424220808_2506085647_1_0" dir="r" state="m" />\n  <connection from="-243188465#1" to="-498671576#1" fromLane="0" toLane="0" via=":4900465473_0_0" dir="l" state="M" />\n  <connection from="-243188465#1" to="-243188465#0" fromLane="0" toLane="0" via=":4900465473_1_0" dir="s" state="M" />\n  <connection from="-243188491" to="706104217#11" fromLane="0" toLane="0" via=":2506088024_13_0" dir="l" state="M" />\n  <connection from="-243188491" to="-330615760" fromLane="0" toLane="0" via=":2506088024_14_0" dir="s" state="M" />\n  <connection from="-243188491" to="-706104217#10" fromLane="0" toLane="0" via=":2506088024_15_0" dir="r" state="m" />\n  <connection from="-243188503#0" to="96941841#1" fromLane="0" toLane="0" via=":1123399700_2_0" dir="l" state="m" />\n  <connection from="-243188503#0" to="-96941841#0" fromLane="0" toLane="0" via=":1123399700_3_0" dir="r" state="m" />\n  <connection from="-243188503#1" to="-243188196" fromLane="0" toLane="0" via=":2506089901_4_0" dir="l" state="M" />\n  <connection from="-243188503#1" to="-243188503#0" fromLane="0" toLane="0" via=":2506089901_5_0" dir="s" state="M" />\n  <connection from="-243188505" to="95809268#1" fromLane="0" toLane="0" via=":2506087955_0_0" dir="l" state="m" />\n  <connection from="-243188508#1" to="243188565" fromLane="0" toLane="0" via=":2506088518_2_0" dir="l" state="M" />\n  <connection from="-243188508#1" to="1552748146" fromLane="0" toLane="0" via=":2506088518_3_0" dir="s" state="M" />\n  <connection from="-243188508#3" to="-243188212#1" fromLane="0" toLane="0" via=":cluster_14188374630_2506088509_3_0" dir="l" state="M" />\n  <connection from="-243188508#3" to="-243188508#1" fromLane="0" toLane="0" via=":cluster_14188374630_2506088509_4_0" dir="s" state="M" />\n  <connection from="-243188508#3" to="-1559802496" fromLane="0" toLane="0" via=":cluster_14188374630_2506088509_5_0" dir="r" state="m" />\n  <connection from="-243188536#0" to="243188646#0" fromLane="0" toLane="0" via=":2506087563_0_0" dir="l" state="M" />\n  <connection from="-243188536#0" to="1316153439" fromLane="0" toLane="0" via=":2506087563_1_0" dir="r" state="M" />\n  <connection from="-243188536#1" to="-898440288" fromLane="0" toLane="0" via=":8348961009_0_0" dir="l" state="M" />\n  <connection from="-243188536#1" to="-243188536#0" fromLane="0" toLane="0" via=":8348961009_1_0" dir="s" state="M" />\n  <connection from="-243188536#2" to="243188103" fromLane="0" toLane="0" via=":2506087304_0_0" dir="l" state="M" />\n  <connection from="-243188536#2" to="-243188536#1" fromLane="0" toLane="0" via=":2506087304_1_0" dir="s" state="M" />\n  <connection from="-243188552#0" to="-243188705" fromLane="0" toLane="0" via=":1123399516_2_0" dir="s" state="M" />\n  <connection from="-243188552#0" to="706094741#0" fromLane="0" toLane="0" via=":1123399516_3_0" dir="r" state="m" />\n  <connection from="-243188552#1" to="-243188552#0" fromLane="0" toLane="0" via=":2506088492_2_0" dir="s" state="M" />\n  <connection from="-243188552#1" to="-243188508#3" fromLane="0" toLane="0" via=":2506088492_3_0" dir="r" state="m" />\n  <connection from="-243188552#2" to="-243188552#1" fromLane="0" toLane="0" via=":4900465288_2_0" dir="s" state="M" />\n  <connection from="-243188552#2" to="1559802495#0" fromLane="0" toLane="0" via=":4900465288_3_0" dir="r" state="m" />\n  <connection from="-243188561#1" to="-969757849#3" fromLane="0" toLane="0" via=":2506085690_5_0" dir="l" state="m" />\n  <connection from="-243188561#1" to="931354569#0" fromLane="0" toLane="0" via=":2506085690_6_0" dir="r" state="m" />\n  <connection from="-243188563" to="-706104219#7" fromLane="0" toLane="0" via=":2506086795_2_0" dir="l" state="m" />\n  <connection from="-243188563" to="706104219#8" fromLane="0" toLane="0" via=":2506086795_3_0" dir="r" state="m" />\n  <connection from="-243188565" to="1552748146" fromLane="0" toLane="0" via=":2506088518_0_0" dir="l" state="m" />\n  <connection from="-243188565" to="243188508#0" fromLane="0" toLane="0" via=":2506088518_1_0" dir="r" state="m" />\n  <connection from="-243188576" to="498671328#1" fromLane="0" toLane="0" via=":14179759489_4_0" dir="s" state="M" />\n  <connection from="-243188576" to="1558734968" fromLane="0" toLane="0" via=":14179759489_5_0" dir="r" state="m" />\n  <connection from="-243188585" to="-72016745#4" fromLane="0" toLane="0" via=":2506087136_7_0" dir="l" state="m" />\n  <connection from="-243188585" to="-498671569" fromLane="0" toLane="0" via=":2506087136_8_0" dir="s" state="m" />\n  <connection from="-243188585" to="72016745#5" fromLane="0" toLane="1" via=":2506087136_9_0" dir="r" state="m" />\n  <connection from="-243188607#0" to="-706094742" fromLane="0" toLane="0" via=":2506085508_0_0" dir="l" state="m" />\n  <connection from="-243188607#0" to="-706094740#2" fromLane="0" toLane="0" via=":2506085508_1_0" dir="r" state="m" />\n  <connection from="-243188607#1" to="-243188607#0" fromLane="0" toLane="0" via=":2506085493_0_0" dir="s" state="M" />\n  <connection from="-243188607#1" to="-243188458" fromLane="0" toLane="0" via=":2506085493_1_0" dir="r" state="m" />\n  <connection from="-243188629" to="706105951#4" fromLane="0" toLane="0" via=":1107926639_3_0" dir="l" state="M" />\n  <connection from="-243188629" to="-243188351#1" fromLane="0" toLane="0" via=":1107926639_4_0" dir="s" state="M" />\n  <connection from="-243188629" to="-706105951#3" fromLane="0" toLane="0" via=":1107926639_5_0" dir="r" state="m" />\n  <connection from="-243188644" to="497290263#2" fromLane="0" toLane="0" via=":2506089545_2_0" dir="l" state="m" />\n  <connection from="-243188644" to="-497290263#1" fromLane="0" toLane="0" via=":2506089545_3_0" dir="r" state="m" />\n  <connection from="-243188705" to="243188225" fromLane="0" toLane="0" via=":1123399683_0_0" dir="s" state="M" />\n  <connection from="-243188705" to="-452036923#7" fromLane="0" toLane="0" via=":1123399683_1_0" dir="r" state="m" />\n  <connection from="-243188710#0" to="-243314395" fromLane="0" toLane="0" via=":1107926968_1_0" dir="r" state="M" />\n  <connection from="-243188710#1" to="-243188710#0" fromLane="0" toLane="0" via=":1107926467_4_0" dir="s" state="M" />\n  <connection from="-243188710#1" to="498671427" fromLane="0" toLane="0" via=":1107926467_5_0" dir="r" state="m" />\n  <connection from="-243188710#2" to="-243188710#1" fromLane="0" toLane="0" via=":1369020939_4_0" dir="s" state="M" />\n  <connection from="-243188710#2" to="-122540020#7" fromLane="0" toLane="0" via=":1369020939_5_0" dir="r" state="m" />\n  <connection from="-243188741#0" to="-706105951#5" fromLane="0" toLane="0" via=":1107926537_4_0" dir="l" state="M" />\n  <connection from="-243188741#0" to="706105950" fromLane="0" toLane="0" via=":1107926537_5_0" dir="s" state="M" />\n  <connection from="-243188765#0" to="96941841#0" fromLane="0" toLane="0" via=":2505176697_2_0" dir="l" state="M" />\n  <connection from="-243188765#0" to="-95809258" fromLane="0" toLane="0" via=":2505176697_3_0" dir="s" state="M" />\n  <connection from="-243188765#1" to="-243188765#0" fromLane="0" toLane="0" via=":1110416742_0_0" dir="s" state="M" />\n  <connection from="-243188765#1" to="-497290263#4" fromLane="0" toLane="0" via=":1110416742_1_0" dir="r" state="m" />\n  <connection from="-243188765#2" to="-243188765#1" fromLane="0" toLane="0" via=":4900465069_0_0" dir="s" state="M" />\n  <connection from="-243188765#2" to="-498671432" fromLane="0" toLane="0" via=":4900465069_1_0" dir="r" state="m" />\n  <connection from="-243188786#0" to="95809267#0" fromLane="0" toLane="0" via=":1110416631_0_0" dir="s" state="M" />\n  <connection from="-243188786#1" to="-243188786#0" fromLane="0" toLane="0" via=":1123399502_5_0" dir="s" state="M" />\n  <connection from="-243188786#1" to="-684741141#3" fromLane="0" toLane="0" via=":1123399502_6_0" dir="r" state="m" />\n  <connection from="-243188786#2" to="-243188786#1" fromLane="0" toLane="0" via=":2506088938_0_0" dir="s" state="M" />\n  <connection from="-243188786#2" to="-243188085" fromLane="0" toLane="0" via=":2506088938_1_0" dir="r" state="m" />\n  <connection from="-243188786#3" to="-243188786#2" fromLane="0" toLane="0" via=":2506089058_0_0" dir="s" state="M" />\n  <connection from="-243188786#3" to="-243188361" fromLane="0" toLane="0" via=":2506089058_1_0" dir="r" state="m" />\n  <connection from="-243188786#4" to="-243188786#3" fromLane="0" toLane="0" via=":2506089487_0_0" dir="s" state="M" />\n  <connection from="-243188786#4" to="-243188161" fromLane="0" toLane="0" via=":2506089487_1_0" dir="r" state="m" />\n  <connection from="-243314073" to="-243314112" fromLane="0" toLane="0" via=":2507400689_5_0" dir="s" state="m" />\n  <connection from="-243314073" to="66984613#6" fromLane="0" toLane="0" via=":2507400689_6_0" dir="r" state="m" />\n  <connection from="-243314137" to="-95542482#10" fromLane="0" toLane="0" via=":2507400769_2_0" dir="l" state="m" />\n  <connection from="-243314137" to="95542482#11" fromLane="0" toLane="0" via=":2507400769_3_0" dir="r" state="m" />\n  <connection from="-243314160" to="1257722739#2" fromLane="0" toLane="0" via=":1123399863_0_0" dir="l" state="M" />\n  <connection from="-243314165#1" to="-243314165#0" fromLane="0" toLane="0" via=":8523326825_0_0" dir="s" state="M" />\n  <connection from="-243314165#1" to="-917695719" fromLane="0" toLane="0" via=":8523326825_1_0" dir="r" state="m" />\n  <connection from="-243314186" to="-96713525" fromLane="0" toLane="0" via=":2505176388_5_0" dir="s" state="m" />\n  <connection from="-243314186" to="66984613#4" fromLane="0" toLane="0" via=":2505176388_6_0" dir="r" state="m" />\n  <connection from="-243314192" to="1270957861" fromLane="0" toLane="0" via=":2507400639_0_0" dir="l" state="m" />\n  <connection from="-243314213" to="-243314378#3" fromLane="0" toLane="0" via=":2507400696_5_0" dir="s" state="m" />\n  <connection from="-243314213" to="66984613#5" fromLane="0" toLane="0" via=":2507400696_6_0" dir="r" state="m" />\n  <connection from="-243314248#0" to="243314436#2" fromLane="0" toLane="0" via=":cluster_2507400612_2507400618_3_0" dir="l" state="m" />\n  <connection from="-243314248#0" to="-243314165#1" fromLane="0" toLane="0" via=":cluster_2507400612_2507400618_4_0" dir="s" state="m" />\n  <connection from="-243314248#0" to="-243314436#0" fromLane="0" toLane="0" via=":cluster_2507400612_2507400618_5_0" dir="r" state="m" />\n  <connection from="-243314248#1" to="-498671542" fromLane="0" toLane="0" via=":4900465377_2_0" dir="l" state="M" />\n  <connection from="-243314248#1" to="-243314248#0" fromLane="0" toLane="0" via=":4900465377_3_0" dir="s" state="M" />\n  <connection from="-243314310#0" to="932339002#0" fromLane="0" toLane="0" via=":8642866006_0_0" dir="l" state="m" />\n  <connection from="-243314310#1" to="1547102487#1" fromLane="0" toLane="0" via=":2507400591_0_0" dir="l" state="m" />\n  <connection from="-243314310#1" to="-243314310#0" fromLane="0" toLane="0" via=":2507400591_1_0" dir="s" state="m" />\n  <connection from="-243314312" to="969757855#1" fromLane="0" toLane="0" via=":2507400638_0_0" dir="l" state="m" />\n  <connection from="-243314312" to="-243314192" fromLane="0" toLane="0" via=":2507400638_1_0" dir="s" state="m" />\n  <connection from="-243314378#2" to="-243314378#1" fromLane="0" toLane="0" via=":8523449430_4_0" dir="s" state="M" />\n  <connection from="-243314378#2" to="917706996" fromLane="0" toLane="0" via=":8523449430_5_0" dir="r" state="m" />\n  <connection from="-243314378#3" to="-243314378#2" fromLane="0" toLane="0" via=":4900465441_4_0" dir="s" state="M" />\n  <connection from="-243314378#3" to="-498671564" fromLane="0" toLane="0" via=":4900465441_5_0" dir="r" state="m" />\n  <connection from="-243314395" to="65080358#4" fromLane="0" toLane="0" via=":1680668226_0_0" dir="l" state="m" />\n  <connection from="-243314436#2" to="-243314165#1" fromLane="0" toLane="0" via=":cluster_2507400612_2507400618_0_0" dir="l" state="M" />\n  <connection from="-243314436#2" to="-243314436#0" fromLane="0" toLane="0" via=":cluster_2507400612_2507400618_1_0" dir="s" state="M" />\n  <connection from="-243314436#2" to="243314248#0" fromLane="0" toLane="0" via=":cluster_2507400612_2507400618_2_0" dir="r" state="m" />\n  <connection from="-243314436#3" to="-243314436#2" fromLane="0" toLane="0" via=":2507400604_0_0" dir="s" state="M" />\n  <connection from="-243314436#3" to="243314467#0" fromLane="0" toLane="0" via=":2507400604_1_0" dir="r" state="m" />\n  <connection from="-243314440" to="66984613#2" fromLane="0" toLane="0" via=":947896989_0_0" dir="l" state="m" />\n  <connection from="-243314440" to="-243079775" fromLane="0" toLane="0" via=":947896989_1_0" dir="s" state="m" />\n  <connection from="-243314467#0" to="243314436#3" fromLane="0" toLane="0" via=":2507400604_2_0" dir="l" state="m" />\n  <connection from="-243314467#0" to="-243314436#2" fromLane="0" toLane="0" via=":2507400604_3_0" dir="r" state="m" />\n  <connection from="-243314467#1" to="-243314467#0" fromLane="0" toLane="0" via=":2507400609_0_0" dir="s" state="M" />\n  <connection from="-243314467#1" to="498671542" fromLane="0" toLane="0" via=":2507400609_1_0" dir="r" state="m" />\n  <connection from="-330615760" to="1257722738#0" fromLane="0" toLane="0" via=":2506088243_0_0" dir="l" state="M" />\n  <connection from="-330615760" to="1257722736#0" fromLane="0" toLane="0" via=":2506088243_1_0" dir="r" state="M" />\n  <connection from="-330615829#0" to="-81350005#2" fromLane="0" toLane="0" via=":947896964_2_0" tl="947896964" linkIndex="2" dir="s" state="O" />\n  <connection from="-330615829#0" to="706104217#0" fromLane="0" toLane="0" via=":947896964_3_0" tl="947896964" linkIndex="3" dir="r" state="o" />\n  <connection from="-330615829#2" to="-330615829#0" fromLane="0" toLane="0" via=":12346099054_2_0" dir="s" state="M" />\n  <connection from="-330615829#2" to="498671676#0" fromLane="0" toLane="0" via=":12346099054_3_0" dir="r" state="m" />\n  <connection from="-330615844" to="-330615829#2" fromLane="0" toLane="0" via=":3376091528_1_0" dir="s" state="M" />\n  <connection from="-452036922" to="-452036923#1" fromLane="0" toLane="0" via=":1123399561_2_0" dir="l" state="m" />\n  <connection from="-452036922" to="452036923#2" fromLane="0" toLane="0" via=":1123399561_3_0" dir="r" state="m" />\n  <connection from="-452036923#0" to="-72016745#5" fromLane="0" toLane="0" via=":5149825984_6_0" dir="l" state="M" />\n  <connection from="-452036923#0" to="-96941849" fromLane="0" toLane="0" via=":5149825984_7_0" dir="s" state="m" />\n  <connection from="-452036923#0" to="706104218" fromLane="0" toLane="0" via=":5149825984_8_0" dir="r" state="m" />\n  <connection from="-452036923#1" to="-243188225" fromLane="0" toLane="0" via=":4488366488_4_0" dir="l" state="M" />\n  <connection from="-452036923#1" to="-452036923#0" fromLane="0" toLane="0" via=":4488366488_5_0" dir="s" state="M" />\n  <connection from="-452036923#2" to="452036922" fromLane="0" toLane="0" via=":1123399561_4_0" dir="l" state="M" />\n  <connection from="-452036923#2" to="-452036923#1" fromLane="0" toLane="0" via=":1123399561_5_0" dir="s" state="M" />\n  <connection from="-452036923#3" to="452037066" fromLane="0" toLane="0" via=":1123399535_4_0" dir="l" state="M" />\n  <connection from="-452036923#3" to="-452036923#2" fromLane="0" toLane="0" via=":1123399535_5_0" dir="s" state="M" />\n  <connection from="-452036923#4" to="452037067" fromLane="0" toLane="0" via=":4488358857_4_0" dir="l" state="M" />\n  <connection from="-452036923#4" to="-452036923#3" fromLane="0" toLane="0" via=":4488358857_5_0" dir="s" state="M" />\n  <connection from="-452036923#5" to="-452037067" fromLane="0" toLane="0" via=":4488358858_2_0" dir="l" state="M" />\n  <connection from="-452036923#5" to="-452036923#4" fromLane="0" toLane="0" via=":4488358858_3_0" dir="s" state="M" />\n  <connection from="-452036923#6" to="-452037066" fromLane="0" toLane="0" via=":4488358856_2_0" dir="l" state="M" />\n  <connection from="-452036923#6" to="-452036923#5" fromLane="0" toLane="0" via=":4488358856_3_0" dir="s" state="M" />\n  <connection from="-452036923#7" to="-452036922" fromLane="0" toLane="0" via=":4488368090_2_0" dir="l" state="M" />\n  <connection from="-452036923#7" to="-452036923#6" fromLane="0" toLane="0" via=":4488368090_3_0" dir="s" state="M" />\n  <connection from="-452037066" to="-452036923#2" fromLane="0" toLane="0" via=":1123399535_2_0" dir="l" state="m" />\n  <connection from="-452037066" to="452036923#3" fromLane="0" toLane="0" via=":1123399535_3_0" dir="r" state="m" />\n  <connection from="-452037067" to="-452036923#3" fromLane="0" toLane="0" via=":4488358857_2_0" dir="l" state="m" />\n  <connection from="-452037067" to="452036923#4" fromLane="0" toLane="0" via=":4488358857_3_0" dir="r" state="m" />\n  <connection from="-497290263#0" to="497290260" fromLane="0" toLane="0" via=":4888861605_0_0" dir="s" state="M" />\n  <connection from="-497290263#1" to="-497290263#0" fromLane="0" toLane="0" via=":2506089600_0_0" dir="s" state="M" />\n  <connection from="-497290263#1" to="-243188627" fromLane="0" toLane="0" via=":2506089600_1_0" dir="r" state="m" />\n  <connection from="-497290263#2" to="-497290263#1" fromLane="0" toLane="0" via=":2506089545_0_0" dir="s" state="M" />\n  <connection from="-497290263#2" to="243188644" fromLane="0" toLane="0" via=":2506089545_1_0" dir="r" state="m" />\n  <connection from="-497290263#3" to="-497290263#2" fromLane="0" toLane="0" via=":1123399713_0_0" dir="s" state="M" />\n  <connection from="-497290263#3" to="243188115" fromLane="0" toLane="0" via=":1123399713_1_0" dir="r" state="m" />\n  <connection from="-497290263#4" to="-497290263#3" fromLane="0" toLane="0" via=":1123399740_0_0" dir="s" state="M" />\n  <connection from="-497290263#4" to="243188445" fromLane="0" toLane="0" via=":1123399740_1_0" dir="r" state="m" />\n  <connection from="-498671328#0" to="-706104217#19" fromLane="0" toLane="0" via=":2506087987_6_0" dir="l" state="m" />\n  <connection from="-498671328#0" to="706095306" fromLane="0" toLane="0" via=":2506087987_7_0" dir="s" state="m" />\n  <connection from="-498671328#0" to="706104217#20" fromLane="0" toLane="0" via=":2506087987_8_0" dir="r" state="m" />\n  <connection from="-498671328#1" to="1558734968" fromLane="0" toLane="0" via=":14179759489_2_0" dir="l" state="M" />\n  <connection from="-498671328#1" to="-498671328#0" fromLane="0" toLane="0" via=":14179759489_3_0" dir="s" state="M" />\n  <connection from="-498671331" to="243188389#3" fromLane="0" toLane="0" via=":2506088494_4_0" dir="l" state="m" />\n  <connection from="-498671331" to="-243188389#2" fromLane="0" toLane="0" via=":2506088494_5_0" dir="r" state="m" />\n  <connection from="-498671388#0" to="706104219#7" fromLane="0" toLane="0" via=":2506086831_0_0" dir="l" state="m" />\n  <connection from="-498671388#0" to="-706104219#6" fromLane="0" toLane="0" via=":2506086831_1_0" dir="r" state="m" />\n  <connection from="-498671388#1" to="-96941857#0" fromLane="0" toLane="0" via=":4900466448_0_0" dir="l" state="m" />\n  <connection from="-498671388#1" to="-498671388#0" fromLane="0" toLane="0" via=":4900466448_1_0" dir="s" state="m" />\n  <connection from="-498671388#1" to="96941857#1" fromLane="0" toLane="0" via=":4900466448_2_0" dir="r" state="m" />\n  <connection from="-498671389#1" to="706104219#5" fromLane="0" toLane="0" via=":4900464597_0_0" dir="l" state="m" />\n  <connection from="-498671389#1" to="-706104219#4" fromLane="0" toLane="0" via=":4900464597_1_0" dir="r" state="m" />\n  <connection from="-498671389#2" to="-96941857#1" fromLane="0" toLane="0" via=":4900464580_9_0" dir="l" state="m" />\n  <connection from="-498671389#2" to="-498671389#1" fromLane="0" toLane="0" via=":4900464580_10_0" dir="s" state="m" />\n  <connection from="-498671389#2" to="96941857#2" fromLane="0" toLane="0" via=":4900464580_11_0" dir="r" state="m" />\n  <connection from="-498671391" to="530314558#1" fromLane="0" toLane="0" via=":4900464587_4_0" dir="l" state="m" />\n  <connection from="-498671391" to="-530314558#0" fromLane="0" toLane="0" via=":4900464587_5_0" dir="r" state="m" />\n  <connection from="-498671393" to="1365935945#1" fromLane="0" toLane="0" via=":4900465166_3_0" dir="l" state="m" />\n  <connection from="-498671393" to="498671465" fromLane="0" toLane="0" via=":4900465166_4_0" dir="s" state="m" />\n  <connection from="-498671393" to="-1365935945#0" fromLane="0" toLane="0" via=":4900465166_5_0" dir="r" state="m" />\n  <connection from="-498671427" to="243188710#1" fromLane="0" toLane="0" via=":1107926467_0_0" dir="l" state="m" />\n  <connection from="-498671427" to="-243188710#0" fromLane="0" toLane="0" via=":1107926467_1_0" dir="r" state="m" />\n  <connection from="-498671465" to="-1365935945#0" fromLane="0" toLane="0" via=":4900465166_9_0" dir="l" state="m" />\n  <connection from="-498671465" to="498671393" fromLane="0" toLane="0" via=":4900465166_10_0" dir="s" state="m" />\n  <connection from="-498671465" to="1365935945#1" fromLane="0" toLane="0" via=":4900465166_11_0" dir="r" state="m" />\n  <connection from="-498671482" to="-706256379#4" fromLane="0" toLane="0" via=":4900465208_2_0" dir="l" state="m" />\n  <connection from="-498671482" to="706256379#5" fromLane="0" toLane="0" via=":4900465208_3_0" dir="r" state="m" />\n  <connection from="-498671483" to="142455600#2" fromLane="0" toLane="0" via=":cluster_1559016910_4900465210_3_0" dir="l" state="m" />\n  <connection from="-498671483" to="-142455590" fromLane="0" toLane="0" via=":cluster_1559016910_4900465210_4_0" dir="s" state="m" />\n  <connection from="-498671483" to="-142455600#0" fromLane="0" toLane="0" via=":cluster_1559016910_4900465210_5_0" dir="r" state="m" />\n  <connection from="-498671497" to="706094740#2" fromLane="0" toLane="0" via=":cluster_1558314142_1558314144_0_0" dir="l" state="M" />\n  <connection from="-498671497" to="-706094740#0" fromLane="0" toLane="0" via=":cluster_1558314142_1558314144_1_0" dir="s" state="M" />\n  <connection from="-498671497" to="155690513" fromLane="0" toLane="0" via=":cluster_1558314142_1558314144_2_0" dir="r" state="m" />\n  <connection from="-498671536" to="95809261#0" fromLane="0" toLane="0" via=":2506087432_2_0" dir="l" state="M" />\n  <connection from="-498671536" to="-1365892748#1" fromLane="0" toLane="0" via=":2506087432_3_0" dir="r" state="M" />\n  <connection from="-498671539" to="498671330#1" fromLane="0" toLane="0" via=":4900465090_0_0" dir="r" state="m" />\n  <connection from="-498671542" to="243314467#1" fromLane="0" toLane="0" via=":2507400609_2_0" dir="l" state="m" />\n  <connection from="-498671542" to="-243314467#0" fromLane="0" toLane="0" via=":2507400609_3_0" dir="r" state="m" />\n  <connection from="-498671544#1" to="706104219#12" fromLane="0" toLane="0" via=":1140280757_0_0" dir="l" state="m" />\n  <connection from="-498671544#1" to="-706104219#11" fromLane="0" toLane="0" via=":1140280757_1_0" dir="r" state="m" />\n  <connection from="-498671576#0" to="706093509#1" fromLane="0" toLane="0" via=":4900465471_2_0" dir="r" state="m" />\n  <connection from="-498671576#1" to="-498671576#0" fromLane="0" toLane="0" via=":4900465472_3_0" dir="s" state="M" />\n  <connection from="-498671576#1" to="243188064#1" fromLane="0" toLane="0" via=":4900465472_4_0" dir="r" state="m" />\n  <connection from="-498671589" to="-706104217#17" fromLane="0" toLane="0" via=":4900465517_2_0" dir="l" state="m" />\n  <connection from="-498671589" to="706104217#18" fromLane="0" toLane="0" via=":4900465517_3_0" dir="r" state="m" />\n  <connection from="-498671600" to="142395239#1" fromLane="0" toLane="0" via=":1558314095_2_0" dir="s" state="M" />\n  <connection from="-498671600" to="-142395239#0" fromLane="0" toLane="0" via=":1558314095_3_0" dir="r" state="m" />\n  <connection from="-498671615" to="498671616" fromLane="0" toLane="0" via=":4900465999_2_0" dir="l" state="M" />\n  <connection from="-498671615" to="-498671678" fromLane="0" toLane="0" via=":4900465999_3_0" dir="s" state="m" />\n  <connection from="-498671616" to="-498671678" fromLane="0" toLane="0" via=":4900465999_0_0" dir="l" state="M" />\n  <connection from="-498671616" to="498671615" fromLane="0" toLane="0" via=":4900465999_1_0" dir="r" state="M" />\n  <connection from="-498671673" to="-243188174#1" fromLane="0" toLane="0" via=":4900465989_4_0" dir="l" state="m" />\n  <connection from="-498671673" to="243188174#2" fromLane="0" toLane="0" via=":4900465989_5_0" dir="r" state="m" />\n  <connection from="-498671678" to="243188499" fromLane="0" toLane="0" via=":2506085555_2_0" dir="l" state="M" />\n  <connection from="-498671678" to="-243188741#0" fromLane="0" toLane="0" via=":2506085555_3_0" dir="s" state="M" />\n  <connection from="-498671679" to="-95533432#3" fromLane="0" toLane="0" via=":4900465219_2_0" dir="l" state="m" />\n  <connection from="-498671679" to="95533432#4" fromLane="0" toLane="0" via=":4900465219_3_0" dir="r" state="m" />\n  <connection from="-530314558#0" to="96242763#3" fromLane="0" toLane="0" via=":13545019681_0_0" dir="s" state="m" />\n  <connection from="-530314558#1" to="-530314558#0" fromLane="0" toLane="0" via=":4900464587_2_0" dir="s" state="M" />\n  <connection from="-530314558#1" to="498671391" fromLane="0" toLane="0" via=":4900464587_3_0" dir="r" state="m" />\n  <connection from="-66984610#0" to="72016745#0" fromLane="0" toLane="0" via=":856214571_0_0" tl="joinedS_809537820_856214571" linkIndex="4" dir="s" state="o" />\n  <connection from="-66984610#0" to="72016745#0" fromLane="0" toLane="1" via=":856214571_0_1" tl="joinedS_809537820_856214571" linkIndex="5" dir="s" state="o" />\n  <connection from="-66984610#0" to="65080358#0" fromLane="0" toLane="0" via=":856214571_2_0" tl="joinedS_809537820_856214571" linkIndex="6" dir="r" state="o" />\n  <connection from="-684741141#0" to="-243188243" fromLane="0" toLane="0" via=":1123399591_7_0" dir="l" state="m" />\n  <connection from="-684741141#0" to="-95542482#14" fromLane="0" toLane="0" via=":1123399591_8_0" dir="r" state="m" />\n  <connection from="-684741141#1" to="-684741141#0" fromLane="0" toLane="0" via=":1123399675_4_0" dir="s" state="M" />\n  <connection from="-684741141#1" to="243188161" fromLane="0" toLane="0" via=":1123399675_5_0" dir="r" state="m" />\n  <connection from="-684741141#2" to="-684741141#1" fromLane="0" toLane="0" via=":1123399943_4_0" dir="s" state="M" />\n  <connection from="-684741141#2" to="243188361" fromLane="0" toLane="0" via=":1123399943_5_0" dir="r" state="m" />\n  <connection from="-684741141#3" to="-684741141#2" fromLane="0" toLane="0" via=":1123399486_4_0" dir="s" state="M" />\n  <connection from="-684741141#3" to="243188085" fromLane="0" toLane="0" via=":1123399486_5_0" dir="r" state="m" />\n  <connection from="-684741338" to="-706094741#3" fromLane="0" toLane="0" via=":1123399489_3_0" dir="s" state="m" />\n  <connection from="-706087699#0" to="-142395239#0" fromLane="0" toLane="0" via=":1558314095_0_0" dir="l" state="M" />\n  <connection from="-706087699#0" to="498671600" fromLane="0" toLane="0" via=":1558314095_1_0" dir="s" state="M" />\n  <connection from="-706087699#1" to="-498671694" fromLane="0" toLane="0" via=":4900465031_4_0" dir="l" state="M" />\n  <connection from="-706087699#1" to="-706087699#0" fromLane="0" toLane="0" via=":4900465031_5_0" dir="s" state="M" />\n  <connection from="-706094740#0" to="970010765#1" fromLane="0" toLane="0" via=":1107926433_0_0" dir="l" state="m" />\n  <connection from="-706094740#2" to="-706094740#0" fromLane="0" toLane="0" via=":cluster_1558314142_1558314144_9_0" dir="l" state="m" />\n  <connection from="-706094740#2" to="155690513" fromLane="0" toLane="0" via=":cluster_1558314142_1558314144_10_0" dir="s" state="m" />\n  <connection from="-706094740#2" to="498671497" fromLane="0" toLane="0" via=":cluster_1558314142_1558314144_11_0" dir="r" state="m" />\n  <connection from="-706094741#0" to="243188552#0" fromLane="0" toLane="0" via=":1123399516_4_0" dir="l" state="m" />\n  <connection from="-706094741#0" to="-243188705" fromLane="0" toLane="0" via=":1123399516_5_0" dir="s" state="m" />\n  <connection from="-706094741#1" to="243188212#1" fromLane="0" toLane="0" via=":1123399641_6_0" dir="l" state="m" />\n  <connection from="-706094741#1" to="-706094741#0" fromLane="0" toLane="0" via=":1123399641_7_0" dir="s" state="m" />\n  <connection from="-706094741#1" to="-243188212#0" fromLane="0" toLane="0" via=":1123399641_8_0" dir="r" state="m" />\n  <connection from="-706094741#2" to="-243188565" fromLane="0" toLane="0" via=":1123399483_4_0" dir="l" state="M" />\n  <connection from="-706094741#2" to="-706094741#1" fromLane="0" toLane="0" via=":1123399483_5_0" dir="s" state="M" />\n  <connection from="-706094741#3" to="-706094741#2" fromLane="0" toLane="0" via=":1123399659_3_0" dir="s" state="m" />\n  <connection from="-706094742" to="970010765#3" fromLane="0" toLane="0" via=":1558997825_0_0" dir="l" state="m" />\n  <connection from="-706095305" to="-706104217#18" fromLane="0" toLane="0" via=":947896956_5_0" dir="l" state="M" />\n  <connection from="-706095305" to="706104217#19" fromLane="0" toLane="0" via=":947896956_6_0" dir="r" state="m" />\n  <connection from="-706095306" to="706104217#20" fromLane="0" toLane="0" via=":2506087987_0_0" dir="l" state="m" />\n  <connection from="-706095306" to="-243188576" fromLane="0" toLane="0" via=":2506087987_1_0" dir="s" state="m" />\n  <connection from="-706095306" to="-706104217#19" fromLane="0" toLane="0" via=":2506087987_2_0" dir="r" state="m" />\n  <connection from="-706095308#0" to="706104217#21" fromLane="0" toLane="0" via=":6634630594_4_0" dir="l" state="m" />\n  <connection from="-706095308#0" to="-706104217#20" fromLane="0" toLane="0" via=":6634630594_5_0" dir="r" state="m" />\n  <connection from="-706095308#1" to="-243188174#0" fromLane="0" toLane="0" via=":6634630596_9_0" dir="l" state="m" />\n  <connection from="-706095308#1" to="-706095308#0" fromLane="0" toLane="0" via=":6634630596_10_0" dir="s" state="m" />\n  <connection from="-706095308#1" to="243188174#1" fromLane="0" toLane="0" via=":6634630596_11_0" dir="r" state="m" />\n  <connection from="-706096644#1" to="96837172#0" fromLane="0" toLane="0" via=":1174378788_1_0" dir="s" state="M" />\n  <connection from="-706103887" to="95809268#2" fromLane="0" toLane="0" via=":2506087863_0_0" dir="l" state="m" />\n  <connection from="-706104217#1" to="330615829#0" fromLane="0" toLane="0" via=":947896964_4_0" tl="947896964" linkIndex="4" dir="l" state="o" />\n  <connection from="-706104217#1" to="-81350005#2" fromLane="0" toLane="0" via=":947896964_5_0" tl="947896964" linkIndex="5" dir="r" state="o" />\n  <connection from="-706104217#10" to="-706104217#8" fromLane="0" toLane="0" via=":2506088002_2_0" dir="s" state="m" />\n  <connection from="-706104217#10" to="-243188563" fromLane="0" toLane="0" via=":2506088002_3_0" dir="r" state="m" />\n  <connection from="-706104217#11" to="-330615760" fromLane="0" toLane="0" via=":2506088024_10_0" dir="l" state="m" />\n  <connection from="-706104217#11" to="-706104217#10" fromLane="0" toLane="0" via=":2506088024_11_0" dir="s" state="m" />\n  <connection from="-706104217#11" to="243188491" fromLane="0" toLane="0" via=":2506088024_12_0" dir="r" state="m" />\n  <connection from="-706104217#12" to="498671702" fromLane="0" toLane="0" via=":12339345632_4_0" dir="l" state="M" />\n  <connection from="-706104217#12" to="-706104217#11" fromLane="0" toLane="0" via=":12339345632_5_0" dir="s" state="M" />\n  <connection from="-706104217#13" to="-706104217#12" fromLane="0" toLane="0" via=":2506088084_3_0" dir="s" state="m" />\n  <connection from="-706104217#14" to="-706104217#13" fromLane="0" toLane="0" via=":1140280758_2_0" dir="s" state="M" />\n  <connection from="-706104217#14" to="98567749" fromLane="0" toLane="0" via=":1140280758_3_0" dir="r" state="m" />\n  <connection from="-706104217#16" to="917704692" fromLane="0" toLane="0" via=":cluster_2506088039_8523406713_7_0" dir="l" state="M" />\n  <connection from="-706104217#16" to="-706104217#14" fromLane="0" toLane="0" via=":cluster_2506088039_8523406713_8_0" dir="s" state="M" />\n  <connection from="-706104217#17" to="-706104217#16" fromLane="0" toLane="0" via=":1110416470_5_0" dir="s" state="m" />\n  <connection from="-706104217#17" to="95809268#0" fromLane="0" toLane="0" via=":1110416470_6_0" dir="r" state="m" />\n  <connection from="-706104217#18" to="498671589" fromLane="0" toLane="0" via=":4900465517_4_0" dir="l" state="M" />\n  <connection from="-706104217#18" to="-706104217#17" fromLane="0" toLane="0" via=":4900465517_5_0" dir="s" state="M" />\n  <connection from="-706104217#19" to="706095305" fromLane="0" toLane="0" via=":947896956_7_0" dir="l" state="m" />\n  <connection from="-706104217#19" to="-706104217#18" fromLane="0" toLane="0" via=":947896956_8_0" dir="s" state="m" />\n  <connection from="-706104217#2" to="-706104217#1" fromLane="0" toLane="0" via=":8523431062_4_0" dir="s" state="M" />\n  <connection from="-706104217#2" to="917705248" fromLane="0" toLane="0" via=":8523431062_5_0" dir="r" state="m" />\n  <connection from="-706104217#20" to="-243188576" fromLane="0" toLane="0" via=":2506087987_9_0" dir="l" state="M" />\n  <connection from="-706104217#20" to="-706104217#19" fromLane="0" toLane="0" via=":2506087987_10_0" dir="s" state="M" />\n  <connection from="-706104217#20" to="706095306" fromLane="0" toLane="0" via=":2506087987_11_0" dir="r" state="m" />\n  <connection from="-706104217#21" to="-706104217#20" fromLane="0" toLane="0" via=":6634630594_2_0" dir="s" state="M" />\n  <connection from="-706104217#21" to="706095308#0" fromLane="0" toLane="0" via=":6634630594_3_0" dir="r" state="m" />\n  <connection from="-706104217#3" to="-706104217#2" fromLane="0" toLane="0" via=":8523431064_2_0" dir="s" state="M" />\n  <connection from="-706104217#3" to="917705249" fromLane="0" toLane="0" via=":8523431064_3_0" dir="r" state="m" />\n  <connection from="-706104217#7" to="-706104217#3" fromLane="0" toLane="0" via=":2506087962_2_0" dir="s" state="M" />\n  <connection from="-706104217#7" to="-243188208" fromLane="0" toLane="0" via=":2506087962_3_0" dir="r" state="m" />\n  <connection from="-706104217#8" to="-706104217#7" fromLane="0" toLane="0" via=":947896950_2_0" dir="s" state="M" />\n  <connection from="-706104217#8" to="243188432#0" fromLane="0" toLane="0" via=":947896950_3_0" dir="r" state="m" />\n  <connection from="-706104218" to="452036923#0" fromLane="0" toLane="0" via=":5149825984_9_0" dir="l" state="M" />\n  <connection from="-706104218" to="-72016745#5" fromLane="0" toLane="1" via=":5149825984_10_0" dir="s" state="M" />\n  <connection from="-706104218" to="-96941849" fromLane="0" toLane="0" via=":5149825984_11_0" dir="r" state="m" />\n  <connection from="-706104219#0" to="-706104218" fromLane="0" toLane="0" via=":1123399639_4_0" dir="s" state="M" />\n  <connection from="-706104219#0" to="-530314558#1" fromLane="0" toLane="0" via=":1123399639_5_0" dir="r" state="m" />\n  <connection from="-706104219#11" to="96713729" fromLane="0" toLane="0" via=":cluster_1966399241_8523333972_6_0" dir="l" state="M" />\n  <connection from="-706104219#11" to="-706104219#9" fromLane="0" toLane="0" via=":cluster_1966399241_8523333972_7_0" dir="s" state="M" />\n  <connection from="-706104219#11" to="917696759" fromLane="0" toLane="0" via=":cluster_1966399241_8523333972_8_0" dir="r" state="m" />\n  <connection from="-706104219#12" to="-706104219#11" fromLane="0" toLane="0" via=":1140280757_7_0" dir="s" state="M" />\n  <connection from="-706104219#12" to="498671544#0" fromLane="0" toLane="0" via=":1140280757_8_0" dir="r" state="m" />\n  <connection from="-706104219#13" to="1365892748#0" fromLane="0" toLane="0" via=":1110416444_6_0" dir="l" state="M" />\n  <connection from="-706104219#13" to="-706104219#12" fromLane="0" toLane="0" via=":1110416444_7_0" dir="s" state="M" />\n  <connection from="-706104219#13" to="706092523#0" fromLane="0" toLane="0" via=":1110416444_8_0" dir="r" state="m" />\n  <connection from="-706104219#14" to="-706104219#13" fromLane="0" toLane="0" via=":1110416654_3_0" dir="s" state="M" />\n  <connection from="-706104219#15" to="706093506#0" fromLane="0" toLane="0" via=":1107784887_5_0" dir="l" state="M" />\n  <connection from="-706104219#15" to="-706104219#14" fromLane="0" toLane="0" via=":1107784887_6_0" dir="s" state="M" />\n  <connection from="-706104219#16" to="-706104219#15" fromLane="0" toLane="0" via=":2506086181_3_0" dir="s" state="M" />\n  <connection from="-706104219#4" to="81350005#0" fromLane="0" toLane="0" via=":cluster_1115585393_13472440789_947896971_9_0" tl="cluster_1115585393_13472440789_947896971" linkIndex="9" dir="l" state="O" />\n  <connection from="-706104219#4" to="-706104219#0" fromLane="0" toLane="0" via=":cluster_1115585393_13472440789_947896971_10_0" tl="cluster_1115585393_13472440789_947896971" linkIndex="10" dir="s" state="O" />\n  <connection from="-706104219#4" to="96242763#0" fromLane="0" toLane="0" via=":cluster_1115585393_13472440789_947896971_11_0" tl="cluster_1115585393_13472440789_947896971" linkIndex="11" dir="r" state="o" />\n  <connection from="-706104219#6" to="-706104219#4" fromLane="0" toLane="0" via=":4900464597_4_0" dir="s" state="M" />\n  <connection from="-706104219#6" to="498671389#0" fromLane="0" toLane="0" via=":4900464597_5_0" dir="r" state="m" />\n  <connection from="-706104219#7" to="-706104219#6" fromLane="0" toLane="0" via=":2506086831_7_0" dir="s" state="M" />\n  <connection from="-706104219#7" to="498671388#0" fromLane="0" toLane="0" via=":2506086831_8_0" dir="r" state="m" />\n  <connection from="-706104219#8" to="243188563" fromLane="0" toLane="0" via=":2506086795_4_0" dir="l" state="M" />\n  <connection from="-706104219#8" to="-706104219#7" fromLane="0" toLane="0" via=":2506086795_5_0" dir="s" state="M" />\n  <connection from="-706104219#9" to="-243188491" fromLane="0" toLane="0" via=":1966399248_9_0" dir="l" state="M" />\n  <connection from="-706104219#9" to="-706104219#8" fromLane="0" toLane="0" via=":1966399248_10_0" dir="s" state="M" />\n  <connection from="-706104219#9" to="-96941842#3" fromLane="0" toLane="0" via=":1966399248_11_0" dir="r" state="m" />\n  <connection from="-706105950" to="243188741#0" fromLane="0" toLane="0" via=":1107926537_0_0" dir="s" state="M" />\n  <connection from="-706105950" to="-706105951#5" fromLane="0" toLane="0" via=":1107926537_1_0" dir="r" state="m" />\n  <connection from="-706105951#0" to="72016745#4" fromLane="0" toLane="0" via=":cluster_13472440775_1666900747_0_0" dir="l" state="m" />\n  <connection from="-706105951#0" to="-95542485#3" fromLane="0" toLane="0" via=":cluster_13472440775_1666900747_1_0" dir="s" state="m" />\n  <connection from="-706105951#0" to="-72016745#2" fromLane="0" toLane="1" via=":cluster_13472440775_1666900747_2_0" dir="r" state="m" />\n  <connection from="-706105951#2" to="-706105951#0" fromLane="0" toLane="0" via=":1107926513_0_0" dir="s" state="M" />\n  <connection from="-706105951#2" to="900244003#0" fromLane="0" toLane="0" via=":1107926513_1_0" dir="r" state="m" />\n  <connection from="-706105951#3" to="185951074#0" fromLane="0" toLane="0" via=":856214524_9_0" dir="l" state="m" />\n  <connection from="-706105951#3" to="-706105951#2" fromLane="0" toLane="0" via=":856214524_10_0" dir="s" state="m" />\n  <connection from="-706105951#3" to="-1270957863" fromLane="0" toLane="0" via=":856214524_11_0" dir="r" state="m" />\n  <connection from="-706105951#5" to="-243188351#1" fromLane="0" toLane="0" via=":1107926639_0_0" dir="l" state="m" />\n  <connection from="-706105951#5" to="-706105951#3" fromLane="0" toLane="0" via=":1107926639_1_0" dir="s" state="m" />\n  <connection from="-706105951#5" to="243188629" fromLane="0" toLane="0" via=":1107926639_2_0" dir="r" state="m" />\n  <connection from="-706256377#0" to="-969765782" fromLane="0" toLane="0" via=":1110416410_3_0" dir="s" state="M" />\n  <connection from="-706256377#1" to="-706256377#0" fromLane="0" toLane="0" via=":2052547075_5_0" dir="s" state="M" />\n  <connection from="-706256377#1" to="154707523#1" fromLane="0" toLane="0" via=":2052547075_6_0" dir="r" state="m" />\n  <connection from="-706256377#2" to="95533433#1" fromLane="0" toLane="0" via=":1107784883_5_0" dir="l" state="M" />\n  <connection from="-706256377#2" to="-706256377#1" fromLane="0" toLane="0" via=":1107784883_6_0" dir="s" state="M" />\n  <connection from="-706256377#3" to="-706256377#2" fromLane="0" toLane="0" via=":4900465209_4_0" dir="s" state="M" />\n  <connection from="-706256377#3" to="-498671482" fromLane="0" toLane="0" via=":4900465209_5_0" dir="r" state="m" />\n  <connection from="-706256377#4" to="706096640#1" fromLane="0" toLane="0" via=":2052547074_5_0" tl="2052547074" linkIndex="5" dir="l" state="O" />\n  <connection from="-706256377#4" to="-706256377#3" fromLane="0" toLane="0" via=":2052547074_6_0" tl="2052547074" linkIndex="6" dir="s" state="O" />\n  <connection from="-706256379#0" to="-1454593202#2" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_10_0" dir="s" state="M" />\n  <connection from="-706256379#0" to="-148941242#5" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_11_0" dir="R" state="m" />\n  <connection from="-706256379#0" to="-129827554#4" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_12_0" dir="r" state="m" />\n  <connection from="-706256379#1" to="96941844" fromLane="0" toLane="0" via=":1431715409_2_0" dir="l" state="M" />\n  <connection from="-706256379#1" to="-706256379#0" fromLane="0" toLane="0" via=":1431715409_3_0" dir="s" state="M" />\n  <connection from="-706256379#3" to="-706256379#1" fromLane="0" toLane="0" via=":cluster_1122072658_14138648129_5_0" dir="s" state="M" />\n  <connection from="-706256379#3" to="96837170#0" fromLane="0" toLane="0" via=":cluster_1122072658_14138648129_6_0" dir="r" state="m" />\n  <connection from="-706256379#4" to="95533433#0" fromLane="0" toLane="0" via=":1107784878_2_0" dir="l" state="M" />\n  <connection from="-706256379#4" to="-706256379#3" fromLane="0" toLane="0" via=":1107784878_3_0" dir="s" state="M" />\n  <connection from="-706256379#6" to="498671482" fromLane="0" toLane="0" via=":4900465208_4_0" dir="l" state="M" />\n  <connection from="-706256379#6" to="-706256379#4" fromLane="0" toLane="0" via=":4900465208_5_0" dir="s" state="M" />\n  <connection from="-706256380#1" to="155690968" fromLane="0" toLane="0" via=":1680666322_4_0" dir="l" state="M" />\n  <connection from="-706256380#1" to="-706256380#0" fromLane="0" toLane="0" via=":1680666322_5_0" dir="s" state="M" />\n  <connection from="-706256380#3" to="155690981" fromLane="0" toLane="0" via=":cluster_1110416379_1680666461_6_0" dir="l" state="M" />\n  <connection from="-706256380#3" to="-706256380#1" fromLane="0" toLane="0" via=":cluster_1110416379_1680666461_7_0" dir="s" state="M" />\n  <connection from="-706256380#3" to="95809262#0" fromLane="0" toLane="0" via=":cluster_1110416379_1680666461_8_0" dir="r" state="m" />\n  <connection from="-706256380#4" to="155690340" fromLane="0" toLane="0" via=":1680666686_6_0" dir="l" state="M" />\n  <connection from="-706256380#4" to="-706256380#3" fromLane="0" toLane="0" via=":1680666686_7_0" dir="s" state="M" />\n  <connection from="-706256380#4" to="-498671483" fromLane="0" toLane="0" via=":1680666686_8_0" dir="r" state="m" />\n  <connection from="-706256380#6" to="155690878#0" fromLane="0" toLane="0" via=":cluster_1559016919_1680666950_6_0" dir="l" state="M" />\n  <connection from="-706256380#6" to="-706256380#4" fromLane="0" toLane="0" via=":cluster_1559016919_1680666950_7_0" dir="s" state="M" />\n  <connection from="-706256380#6" to="-706256386" fromLane="0" toLane="0" via=":cluster_1559016919_1680666950_8_0" dir="r" state="m" />\n  <connection from="-706256381" to="101939921" fromLane="0" toLane="0" via=":1160612969_0_0" dir="l" state="M" />\n  <connection from="-706256386" to="142455600#0" fromLane="0" toLane="0" via=":1559016918_1_0" dir="l" state="M" />\n  <connection from="-706520617#0" to="-142453602#2" fromLane="0" toLane="0" via=":1115585416_4_0" dir="l" state="M" />\n  <connection from="-706520617#0" to="-969765780#2" fromLane="0" toLane="0" via=":1115585416_5_0" dir="s" state="m" />\n  <connection from="-706520617#1" to="-1365935945#1" fromLane="0" toLane="0" via=":12649445299_4_0" dir="l" state="M" />\n  <connection from="-706520617#1" to="-706520617#0" fromLane="0" toLane="0" via=":12649445299_5_0" dir="s" state="M" />\n  <connection from="-706520617#2" to="96941842#0" fromLane="0" toLane="0" via=":1123399562_4_0" dir="l" state="M" />\n  <connection from="-706520617#2" to="-706520617#1" fromLane="0" toLane="0" via=":1123399562_5_0" dir="s" state="m" />\n  <connection from="-706520617#3" to="931354570" fromLane="0" toLane="0" via=":1115585792_2_0" dir="l" state="M" />\n  <connection from="-706520617#3" to="-706520617#2" fromLane="0" toLane="0" via=":1115585792_3_0" dir="s" state="M" />\n  <connection from="-706520617#4" to="-706520617#3" fromLane="0" toLane="0" via=":2052489649_3_0" dir="s" state="m" />\n  <connection from="-72016745#1" to="65080358#0" fromLane="0" toLane="0" via=":856214571_3_0" tl="joinedS_809537820_856214571" linkIndex="7" dir="l" state="o" />\n  <connection from="-72016745#1" to="66984610#0" fromLane="1" toLane="0" via=":856214571_4_0" tl="joinedS_809537820_856214571" linkIndex="8" dir="s" state="o" />\n  <connection from="-72016745#2" to="122540020#0" fromLane="0" toLane="0" via=":1369020950_5_0" dir="l" state="M" />\n  <connection from="-72016745#2" to="-72016745#1" fromLane="0" toLane="0" via=":1369020950_6_0" dir="s" state="M" />\n  <connection from="-72016745#2" to="-72016745#1" fromLane="1" toLane="1" via=":1369020950_6_1" dir="s" state="M" />\n  <connection from="-72016745#4" to="-95542485#3" fromLane="0" toLane="0" via=":cluster_13472440775_1666900747_10_0" dir="l" state="M" />\n  <connection from="-72016745#4" to="-72016745#2" fromLane="0" toLane="0" via=":cluster_13472440775_1666900747_11_0" dir="s" state="M" />\n  <connection from="-72016745#4" to="-72016745#2" fromLane="1" toLane="1" via=":cluster_13472440775_1666900747_11_1" dir="s" state="M" />\n  <connection from="-72016745#4" to="706105951#0" fromLane="1" toLane="0" via=":cluster_13472440775_1666900747_13_0" dir="r" state="m" />\n  <connection from="-72016745#5" to="243188585" fromLane="0" toLane="0" via=":2506087136_10_0" dir="l" state="M" />\n  <connection from="-72016745#5" to="-72016745#4" fromLane="0" toLane="0" via=":2506087136_11_0" dir="s" state="M" />\n  <connection from="-72016745#5" to="-72016745#4" fromLane="1" toLane="1" via=":2506087136_11_1" dir="s" state="M" />\n  <connection from="-72016745#5" to="-498671569" fromLane="1" toLane="0" via=":2506087136_13_0" dir="r" state="m" />\n  <connection from="-81350005#2" to="-706104219#0" fromLane="0" toLane="0" via=":cluster_1115585393_13472440789_947896971_6_0" tl="cluster_1115585393_13472440789_947896971" linkIndex="6" dir="l" state="o" />\n  <connection from="-81350005#2" to="96242763#0" fromLane="0" toLane="0" via=":cluster_1115585393_13472440789_947896971_7_0" tl="cluster_1115585393_13472440789_947896971" linkIndex="7" dir="s" state="o" />\n  <connection from="-81350005#2" to="706104219#3" fromLane="0" toLane="0" via=":cluster_1115585393_13472440789_947896971_8_0" tl="cluster_1115585393_13472440789_947896971" linkIndex="8" dir="r" state="o" />\n  <connection from="-81350006#2" to="-81350006#1" fromLane="0" toLane="0" via=":2507400773_4_0" dir="s" state="M" />\n  <connection from="-81350006#2" to="-243314177" fromLane="0" toLane="0" via=":2507400773_5_0" dir="r" state="m" />\n  <connection from="-898440288" to="84842974#2" fromLane="0" toLane="0" via=":8348961008_0_0" dir="l" state="m" />\n  <connection from="-900244003#0" to="706105951#1" fromLane="0" toLane="0" via=":1107926513_2_0" dir="l" state="m" />\n  <connection from="-900244003#0" to="-706105951#0" fromLane="0" toLane="0" via=":1107926513_3_0" dir="r" state="m" />\n  <connection from="-900244003#1" to="900244004" fromLane="0" toLane="0" via=":8363308288_2_0" dir="l" state="M" />\n  <connection from="-900244003#1" to="-900244003#0" fromLane="0" toLane="0" via=":8363308288_3_0" dir="s" state="M" />\n  <connection from="-900244004" to="-900244003#0" fromLane="0" toLane="0" via=":8363308288_0_0" dir="l" state="m" />\n  <connection from="-900244004" to="900244003#1" fromLane="0" toLane="0" via=":8363308288_1_0" dir="r" state="m" />\n  <connection from="-917695557" to="151072453#1" fromLane="0" toLane="0" via=":8523328986_0_0" dir="l" state="m" />\n  <connection from="-917695719" to="-243314312" fromLane="0" toLane="0" via=":8523326826_0_0" dir="l" state="M" />\n  <connection from="-917696759" to="706104219#11" fromLane="0" toLane="0" via=":cluster_1966399241_8523333972_0_0" dir="l" state="m" />\n  <connection from="-917696759" to="96713729" fromLane="0" toLane="0" via=":cluster_1966399241_8523333972_1_0" dir="s" state="m" />\n  <connection from="-917696759" to="-706104219#9" fromLane="0" toLane="0" via=":cluster_1966399241_8523333972_2_0" dir="r" state="m" />\n  <connection from="-917697169" to="84842974#4" fromLane="0" toLane="0" via=":cluster_620881097_8523329816_0_0" dir="l" state="m" />\n  <connection from="-917704692" to="-706104217#14" fromLane="0" toLane="0" via=":cluster_2506088039_8523406713_5_0" dir="l" state="m" />\n  <connection from="-917704692" to="706104217#16" fromLane="0" toLane="0" via=":cluster_2506088039_8523406713_6_0" dir="r" state="m" />\n  <connection from="-917705248" to="706104217#2" fromLane="0" toLane="0" via=":8523431062_0_0" dir="l" state="m" />\n  <connection from="-917705248" to="-706104217#1" fromLane="0" toLane="0" via=":8523431062_1_0" dir="r" state="m" />\n  <connection from="-917705249" to="706104217#3" fromLane="0" toLane="0" via=":8523431064_4_0" dir="l" state="m" />\n  <connection from="-917705249" to="-706104217#2" fromLane="0" toLane="0" via=":8523431064_5_0" dir="r" state="m" />\n  <connection from="-917706996" to="243314378#2" fromLane="0" toLane="0" via=":8523449430_0_0" dir="l" state="m" />\n  <connection from="-917706996" to="-243314378#1" fromLane="0" toLane="0" via=":8523449430_1_0" dir="r" state="m" />\n  <connection from="-931354569#0" to="243188561#0" fromLane="0" toLane="0" via=":2506085690_7_0" dir="l" state="M" />\n  <connection from="-931354569#0" to="-969757849#3" fromLane="0" toLane="0" via=":2506085690_8_0" dir="s" state="M" />\n  <connection from="-931354569#1" to="-931354569#0" fromLane="0" toLane="0" via=":856214503_5_0" dir="s" state="M" />\n  <connection from="-931354569#1" to="706092523#4" fromLane="0" toLane="0" via=":856214503_6_0" dir="r" state="m" />\n  <connection from="-931354569#3" to="-931354569#1" fromLane="0" toLane="0" via=":cluster_13424220808_2506085647_7_0" dir="s" state="M" />\n  <connection from="-931354569#3" to="243188465#0" fromLane="0" toLane="0" via=":cluster_13424220808_2506085647_8_0" dir="r" state="m" />\n  <connection from="-931354569#4" to="-931354569#3" fromLane="0" toLane="0" via=":2506085628_3_0" dir="s" state="M" />\n  <connection from="-932324369" to="-129842397#1" fromLane="0" toLane="0" via=":8642733850_2_0" dir="l" state="m" />\n  <connection from="-932324369" to="129842397#2" fromLane="0" toLane="0" via=":8642733850_3_0" dir="r" state="m" />\n  <connection from="-932339005" to="932339002#4" fromLane="0" toLane="0" via=":1680666775_0_0" dir="l" state="m" />\n  <connection from="-95533432#1" to="-243188318" fromLane="0" toLane="0" via=":2506085476_4_0" dir="l" state="M" />\n  <connection from="-95533432#1" to="-95533432#0" fromLane="0" toLane="0" via=":2506085476_5_0" dir="s" state="M" />\n  <connection from="-95533432#3" to="243188458" fromLane="0" toLane="0" via=":cluster_1680667233_2506085475_9_0" dir="l" state="M" />\n  <connection from="-95533432#3" to="-95533432#1" fromLane="0" toLane="0" via=":cluster_1680667233_2506085475_10_0" dir="s" state="M" />\n  <connection from="-95533432#3" to="-155690875" fromLane="0" toLane="0" via=":cluster_1680667233_2506085475_11_0" dir="r" state="m" />\n  <connection from="-95533432#4" to="498671679" fromLane="0" toLane="0" via=":4900465219_4_0" dir="l" state="M" />\n  <connection from="-95533432#4" to="-95533432#3" fromLane="0" toLane="0" via=":4900465219_5_0" dir="s" state="M" />\n  <connection from="-95533432#5" to="-95533432#4" fromLane="0" toLane="0" via=":1680667181_4_0" dir="s" state="M" />\n  <connection from="-95533432#5" to="-155690528" fromLane="0" toLane="0" via=":1680667181_5_0" dir="r" state="m" />\n  <connection from="-95542480" to="-81350006#2" fromLane="0" toLane="0" via=":947896981_5_0" dir="s" state="M" />\n  <connection from="-95542480" to="66984613#0" fromLane="0" toLane="0" via=":947896981_6_0" dir="r" state="m" />\n  <connection from="-95542482#0" to="95809258" fromLane="0" toLane="0" via=":1107926481_6_0" dir="l" state="m" />\n  <connection from="-95542482#0" to="-243188710#2" fromLane="0" toLane="0" via=":1107926481_7_0" dir="s" state="m" />\n  <connection from="-95542482#0" to="95542485#0" fromLane="0" toLane="0" via=":1107926481_8_0" dir="r" state="m" />\n  <connection from="-95542482#1" to="-95542482#0" fromLane="0" toLane="0" via=":1107926561_2_0" dir="s" state="M" />\n  <connection from="-95542482#1" to="-243188585" fromLane="0" toLane="0" via=":1107926561_3_0" dir="r" state="m" />\n  <connection from="-95542482#10" to="243188273#0" fromLane="0" toLane="0" via=":1110416526_4_0" dir="l" state="M" />\n  <connection from="-95542482#10" to="-95542482#9" fromLane="0" toLane="0" via=":1110416526_5_0" dir="s" state="M" />\n  <connection from="-95542482#11" to="243314137" fromLane="0" toLane="0" via=":2507400769_4_0" dir="l" state="M" />\n  <connection from="-95542482#11" to="-95542482#10" fromLane="0" toLane="0" via=":2507400769_5_0" dir="s" state="M" />\n  <connection from="-95542482#12" to="100401320" fromLane="0" toLane="0" via=":1110416469_4_0" dir="l" state="M" />\n  <connection from="-95542482#12" to="-95542482#11" fromLane="0" toLane="0" via=":1110416469_5_0" dir="s" state="M" />\n  <connection from="-95542482#13" to="-243314160" fromLane="0" toLane="0" via=":2507400772_4_0" dir="l" state="M" />\n  <connection from="-95542482#13" to="-95542482#12" fromLane="0" toLane="0" via=":2507400772_5_0" dir="s" state="M" />\n  <connection from="-95542482#14" to="-95542482#13" fromLane="0" toLane="0" via=":1110416754_2_0" dir="s" state="M" />\n  <connection from="-95542482#14" to="-243188786#4" fromLane="0" toLane="0" via=":1110416754_3_0" dir="r" state="m" />\n  <connection from="-95542482#3" to="-95542482#1" fromLane="0" toLane="0" via=":1107926940_2_0" dir="s" state="M" />\n  <connection from="-95542482#3" to="-243188552#2" fromLane="0" toLane="0" via=":1107926940_3_0" dir="r" state="m" />\n  <connection from="-95542482#4" to="-95542482#3" fromLane="0" toLane="0" via=":1107926599_2_0" dir="s" state="M" />\n  <connection from="-95542482#4" to="243188641#0" fromLane="0" toLane="0" via=":1107926599_3_0" dir="r" state="m" />\n  <connection from="-95542482#5" to="-980451721#1" fromLane="0" toLane="0" via=":1107926977_6_0" dir="l" state="M" />\n  <connection from="-95542482#5" to="-95542482#4" fromLane="0" toLane="0" via=":1107926977_7_0" dir="s" state="M" />\n  <connection from="-95542482#5" to="243188594#0" fromLane="0" toLane="0" via=":1107926977_8_0" dir="r" state="m" />\n  <connection from="-95542482#9" to="243188670#2" fromLane="0" toLane="0" via=":1107926734_2_0" dir="l" state="m" />\n  <connection from="-95542482#9" to="-95542482#5" fromLane="0" toLane="0" via=":1107926734_3_0" dir="s" state="m" />\n  <connection from="-95542485#3" to="95542482#0" fromLane="0" toLane="0" via=":1107926481_9_0" dir="l" state="M" />\n  <connection from="-95542485#3" to="95809258" fromLane="0" toLane="0" via=":1107926481_10_0" dir="s" state="M" />\n  <connection from="-95542485#3" to="-243188710#2" fromLane="0" toLane="0" via=":1107926481_11_0" dir="r" state="m" />\n  <connection from="-95809258" to="-243188710#2" fromLane="0" toLane="0" via=":1107926481_3_0" dir="l" state="M" />\n  <connection from="-95809258" to="95542485#0" fromLane="0" toLane="0" via=":1107926481_4_0" dir="s" state="M" />\n  <connection from="-95809258" to="95542482#0" fromLane="0" toLane="0" via=":1107926481_5_0" dir="r" state="m" />\n  <connection from="-95809262#0" to="706256380#3" fromLane="0" toLane="0" via=":cluster_1110416379_1680666461_9_0" dir="l" state="m" />\n  <connection from="-95809262#0" to="155690981" fromLane="0" toLane="0" via=":cluster_1110416379_1680666461_10_0" dir="s" state="m" />\n  <connection from="-95809262#0" to="-706256380#1" fromLane="0" toLane="0" via=":cluster_1110416379_1680666461_11_0" dir="r" state="m" />\n  <connection from="-95809262#1" to="-142455600#3" fromLane="0" toLane="0" via=":1559016903_9_0" dir="l" state="m" />\n  <connection from="-95809262#1" to="-95809262#0" fromLane="0" toLane="0" via=":1559016903_10_0" dir="s" state="m" />\n  <connection from="-95809262#1" to="142455600#4" fromLane="0" toLane="0" via=":1559016903_11_0" dir="r" state="m" />\n  <connection from="-96837170#2" to="706256379#3" fromLane="0" toLane="0" via=":cluster_1122072658_14138648129_7_0" dir="l" state="m" />\n  <connection from="-96837170#2" to="-706256379#1" fromLane="0" toLane="0" via=":cluster_1122072658_14138648129_8_0" dir="r" state="m" />\n  <connection from="-96837170#4" to="-129827552#1" fromLane="0" toLane="0" via=":1122072674_0_0" dir="l" state="M" />\n  <connection from="-96837170#4" to="-96837170#2" fromLane="0" toLane="0" via=":1122072674_1_0" dir="s" state="M" />\n  <connection from="-96837172#0" to="706096644#0" fromLane="0" toLane="0" via=":1174378788_0_0" dir="s" state="M" />\n  <connection from="-96837172#1" to="-96837172#0" fromLane="0" toLane="0" via=":1431715395_0_0" dir="s" state="M" />\n  <connection from="-96837172#1" to="129827552#0" fromLane="0" toLane="0" via=":1431715395_1_0" dir="r" state="m" />\n  <connection from="-96941841#0" to="-95809258" fromLane="0" toLane="0" via=":2505176697_0_0" dir="l" state="m" />\n  <connection from="-96941841#0" to="243188765#0" fromLane="0" toLane="0" via=":2505176697_1_0" dir="r" state="m" />\n  <connection from="-96941841#1" to="-96941841#0" fromLane="0" toLane="0" via=":1123399700_0_0" dir="s" state="M" />\n  <connection from="-96941841#1" to="243188503#0" fromLane="0" toLane="0" via=":1123399700_1_0" dir="r" state="m" />\n  <connection from="-96941841#2" to="-96941841#1" fromLane="0" toLane="0" via=":1123399513_0_0" dir="s" state="M" />\n  <connection from="-96941841#2" to="243188196" fromLane="0" toLane="0" via=":1123399513_1_0" dir="r" state="m" />\n  <connection from="-96941842#0" to="-706520617#1" fromLane="0" toLane="0" via=":1123399562_2_0" dir="l" state="M" />\n  <connection from="-96941842#0" to="706520617#2" fromLane="0" toLane="0" via=":1123399562_3_0" dir="r" state="M" />\n  <connection from="-96941842#1" to="-498671465" fromLane="0" toLane="0" via=":12649445300_4_0" dir="l" state="M" />\n  <connection from="-96941842#1" to="-96941842#0" fromLane="0" toLane="0" via=":12649445300_5_0" dir="s" state="M" />\n  <connection from="-96941842#2" to="-969757849#2" fromLane="0" toLane="0" via=":1123399706_6_0" dir="l" state="m" />\n  <connection from="-96941842#2" to="-96941842#1" fromLane="0" toLane="0" via=":1123399706_7_0" dir="s" state="m" />\n  <connection from="-96941842#2" to="969757849#3" fromLane="0" toLane="0" via=":1123399706_8_0" dir="r" state="m" />\n  <connection from="-96941842#3" to="96941857#0" fromLane="0" toLane="0" via=":1123399595_4_0" dir="l" state="M" />\n  <connection from="-96941842#3" to="-96941842#2" fromLane="0" toLane="0" via=":1123399595_5_0" dir="s" state="M" />\n  <connection from="-96941849" to="-185951074#3" fromLane="0" toLane="0" via=":1123399600_3_0" dir="l" state="m" />\n  <connection from="-96941849" to="-498671615" fromLane="0" toLane="0" via=":1123399600_4_0" dir="s" state="m" />\n  <connection from="-96941849" to="185951074#4" fromLane="0" toLane="0" via=":1123399600_5_0" dir="r" state="m" />\n  <connection from="-96941857#0" to="-96941842#2" fromLane="0" toLane="0" via=":1123399595_2_0" dir="l" state="m" />\n  <connection from="-96941857#0" to="96941842#3" fromLane="0" toLane="0" via=":1123399595_3_0" dir="r" state="m" />\n  <connection from="-96941857#1" to="498671388#1" fromLane="0" toLane="0" via=":4900466448_3_0" dir="l" state="M" />\n  <connection from="-96941857#1" to="-96941857#0" fromLane="0" toLane="0" via=":4900466448_4_0" dir="s" state="M" />\n  <connection from="-96941857#1" to="-498671388#0" fromLane="0" toLane="0" via=":4900466448_5_0" dir="r" state="m" />\n  <connection from="-96941857#2" to="498671389#2" fromLane="0" toLane="0" via=":4900464580_0_0" dir="l" state="M" />\n  <connection from="-96941857#2" to="-96941857#1" fromLane="0" toLane="0" via=":4900464580_1_0" dir="s" state="M" />\n  <connection from="-96941857#2" to="-498671389#1" fromLane="0" toLane="0" via=":4900464580_2_0" dir="r" state="m" />\n  <connection from="-969757849#0" to="498671395#0" fromLane="0" toLane="0" via=":1115585385_3_0" dir="l" state="M" />\n  <connection from="-969757849#0" to="-1204844051" fromLane="0" toLane="0" via=":1115585385_4_0" dir="s" state="M" />\n  <connection from="-969757849#0" to="142453602#0" fromLane="0" toLane="0" via=":1115585385_5_0" dir="r" state="m" />\n  <connection from="-969757849#1" to="-498671389#2" fromLane="0" toLane="0" via=":4900464598_4_0" dir="l" state="M" />\n  <connection from="-969757849#1" to="-969757849#0" fromLane="0" toLane="0" via=":4900464598_5_0" dir="s" state="M" />\n  <connection from="-969757849#2" to="-498671388#1" fromLane="0" toLane="0" via=":856214538_9_0" dir="l" state="M" />\n  <connection from="-969757849#2" to="-969757849#1" fromLane="0" toLane="0" via=":856214538_10_0" dir="s" state="M" />\n  <connection from="-969757849#2" to="1365935945#0" fromLane="0" toLane="0" via=":856214538_11_0" dir="r" state="m" />\n  <connection from="-969757849#3" to="96941842#2" fromLane="0" toLane="0" via=":1123399706_9_0" dir="l" state="M" />\n  <connection from="-969757849#3" to="-969757849#2" fromLane="0" toLane="0" via=":1123399706_10_0" dir="s" state="M" />\n  <connection from="-969757849#3" to="-96941842#1" fromLane="0" toLane="0" via=":1123399706_11_0" dir="r" state="m" />\n  <connection from="-969765780#1" to="969757851#1" fromLane="0" toLane="0" via=":cluster_1123399587_8976429007_2_0" dir="s" state="m" />\n  <connection from="-969765780#1" to="970010765#2" fromLane="0" toLane="0" via=":cluster_1123399587_8976429007_3_0" dir="r" state="m" />\n  <connection from="-969765780#2" to="1556742613" fromLane="0" toLane="0" via=":14159239427_4_0" dir="l" state="M" />\n  <connection from="-969765780#2" to="-969765780#1" fromLane="0" toLane="0" via=":14159239427_5_0" dir="s" state="M" />\n  <connection from="-969765781" to="706256385" fromLane="0" toLane="0" via=":1431715406_5_0" dir="l" state="M" />\n  <connection from="-969765781" to="-129827561#1" fromLane="0" toLane="0" via=":1431715406_6_0" dir="r" state="m" />\n  <connection from="-969765782" to="1365892989" fromLane="0" toLane="0" via=":1431715392_3_0" dir="l" state="M" />\n  <connection from="-969765782" to="-969765781" fromLane="0" toLane="0" via=":1431715392_4_0" dir="s" state="M" />\n  <connection from="-969765782" to="706256383" fromLane="0" toLane="0" via=":1431715392_5_0" dir="r" state="m" />\n  <connection from="-980451721#0" to="980451722#0" fromLane="0" toLane="0" via=":1123399676_0_0" dir="s" state="M" />\n  <connection from="-980451721#0" to="497290269" fromLane="0" toLane="0" via=":1123399676_1_0" dir="r" state="M" />\n  <connection from="-980451721#1" to="-980451721#0" fromLane="0" toLane="0" via=":2506089309_0_0" dir="s" state="m" />\n  <connection from="100401250#0" to="100401250#1" fromLane="0" toLane="0" via=":8976428985_1_0" dir="s" state="m" />\n  <connection from="100401250#1" to="706256381" fromLane="0" toLane="0" via=":1160612969_1_0" dir="s" state="M" />\n  <connection from="100401250#1" to="101939921" fromLane="0" toLane="1" via=":1160612969_2_0" dir="r" state="M" />\n  <connection from="101706795#0" to="101706795#1" fromLane="0" toLane="0" via=":1966412828_4_0" dir="s" state="m" />\n  <connection from="101706795#0" to="84842974#17" fromLane="0" toLane="1" via=":1966412828_5_0" dir="r" state="m" />\n  <connection from="101706795#0" to="84842974#17" fromLane="0" toLane="2" via=":1966412828_5_1" dir="r" state="m" />\n  <connection from="101706795#1" to="706096644#0" fromLane="0" toLane="0" via=":1174378788_2_0" dir="l" state="m" />\n  <connection from="101706795#1" to="96837172#0" fromLane="0" toLane="0" via=":1174378788_3_0" dir="r" state="m" />\n  <connection from="101939921" to="-129827561#1" fromLane="0" toLane="0" via=":1431715406_2_0" dir="l" state="M" />\n  <connection from="101939921" to="969765781" fromLane="0" toLane="0" via=":1431715406_3_0" dir="s" state="M" />\n  <connection from="101939921" to="706256385" fromLane="1" toLane="0" via=":1431715406_4_0" dir="r" state="m" />\n  <connection from="1168168079" to="706093506#2" fromLane="0" toLane="0" via=":4900465364_1_0" dir="r" state="m" />\n  <connection from="1204844051" to="142453602#0" fromLane="0" toLane="0" via=":1115585385_0_0" dir="l" state="M" />\n  <connection from="1204844051" to="969757849#0" fromLane="0" toLane="0" via=":1115585385_1_0" dir="s" state="M" />\n  <connection from="1204844051" to="498671395#0" fromLane="0" toLane="0" via=":1115585385_2_0" dir="r" state="m" />\n  <connection from="1218225518#0" to="-1218747872" fromLane="0" toLane="0" via=":2505176175_1_0" dir="l" state="M" />\n  <connection from="1218225518#0" to="1218225518#1" fromLane="0" toLane="0" via=":2505176175_2_0" dir="s" state="M" />\n  <connection from="1218225518#0" to="1218225518#1" fromLane="1" toLane="1" via=":2505176175_2_1" dir="s" state="M" />\n  <connection from="1218225518#1" to="1270957862" fromLane="0" toLane="0" via=":cluster_2505176169_856214432_0_0" dir="s" state="M" />\n  <connection from="1218225518#1" to="1270957862" fromLane="1" toLane="1" via=":cluster_2505176169_856214432_0_1" dir="s" state="M" />\n  <connection from="1218225518#1" to="1270957863" fromLane="1" toLane="0" via=":cluster_2505176169_856214432_2_0" dir="r" state="m" />\n  <connection from="1218225522#0" to="-243188629" fromLane="0" toLane="0" via=":796747634_1_0" dir="l" state="M" />\n  <connection from="1218225522#0" to="1270957860#0" fromLane="0" toLane="0" via=":796747634_2_0" dir="s" state="M" />\n  <connection from="1218225524#1" to="72016745#0" fromLane="0" toLane="0" via=":856214571_5_0" tl="joinedS_809537820_856214571" linkIndex="9" dir="l" state="O" />\n  <connection from="1218225524#1" to="65080358#0" fromLane="0" toLane="0" via=":856214571_6_0" tl="joinedS_809537820_856214571" linkIndex="10" dir="s" state="O" />\n  <connection from="1218225524#1" to="66984610#0" fromLane="0" toLane="0" via=":856214571_7_0" tl="joinedS_809537820_856214571" linkIndex="11" dir="r" state="O" />\n  <connection from="1218747873" to="1218225518#1" fromLane="0" toLane="0" via=":2505176175_0_0" dir="l" state="m" />\n  <connection from="1218747874" to="243079844#0" fromLane="0" toLane="0" via=":11292971036_0_0" dir="s" state="M" />\n  <connection from="122540020#0" to="1557537524" fromLane="0" toLane="0" via=":14165547215_4_0" dir="l" state="M" />\n  <connection from="122540020#0" to="122540020#1" fromLane="0" toLane="0" via=":14165547215_5_0" dir="s" state="M" />\n  <connection from="122540020#1" to="122540020#3" fromLane="0" toLane="0" via=":1369020943_4_0" dir="s" state="M" />\n  <connection from="122540020#1" to="243188283" fromLane="0" toLane="0" via=":1369020943_5_0" dir="r" state="m" />\n  <connection from="122540020#3" to="122540020#4" fromLane="0" toLane="0" via=":2506088023_4_0" dir="s" state="M" />\n  <connection from="122540020#3" to="243188139" fromLane="0" toLane="0" via=":2506088023_5_0" dir="r" state="m" />\n  <connection from="122540020#4" to="122540020#5" fromLane="0" toLane="0" via=":2506088274_4_0" dir="s" state="M" />\n  <connection from="122540020#4" to="243188167" fromLane="0" toLane="0" via=":2506088274_5_0" dir="r" state="m" />\n  <connection from="122540020#5" to="122540020#6" fromLane="0" toLane="0" via=":2506088528_4_0" dir="s" state="M" />\n  <connection from="122540020#5" to="243188092" fromLane="0" toLane="0" via=":2506088528_5_0" dir="r" state="m" />\n  <connection from="122540020#6" to="122540020#7" fromLane="0" toLane="0" via=":1369020942_4_0" dir="s" state="M" />\n  <connection from="122540020#6" to="243188056" fromLane="0" toLane="0" via=":1369020942_5_0" dir="r" state="m" />\n  <connection from="122540020#7" to="243188710#2" fromLane="0" toLane="0" via=":1369020939_0_0" dir="l" state="m" />\n  <connection from="122540020#7" to="-243188710#1" fromLane="0" toLane="0" via=":1369020939_1_0" dir="r" state="m" />\n  <connection from="1227721991#0" to="1218225524#1" fromLane="0" toLane="0" via=":cluster_11292971032_2406954578_2406954579_2406954580_5_0" dir="s" state="M" />\n  <connection from="1227721991#0" to="1218747874" fromLane="0" toLane="0" via=":cluster_11292971032_2406954578_2406954579_2406954580_6_0" dir="r" state="m" />\n  <connection from="1257722736#0" to="243188670#0" fromLane="0" toLane="0" via=":cluster_1123399715_3376084911_3376091818_3_0" dir="l" state="M" />\n  <connection from="1257722737" to="-706104217#10" fromLane="0" toLane="0" via=":2506088024_7_0" dir="l" state="M" />\n  <connection from="1257722737" to="243188491" fromLane="0" toLane="0" via=":2506088024_8_0" dir="s" state="M" />\n  <connection from="1257722737" to="706104217#11" fromLane="0" toLane="0" via=":2506088024_9_0" dir="r" state="m" />\n  <connection from="1257722738#0" to="-243188786#0" fromLane="0" toLane="0" via=":1123399502_2_0" dir="l" state="m" />\n  <connection from="1257722738#0" to="-684741141#3" fromLane="0" toLane="0" via=":1123399502_3_0" dir="s" state="m" />\n  <connection from="1257722738#0" to="243188786#1" fromLane="0" toLane="0" via=":1123399502_4_0" dir="r" state="m" />\n  <connection from="1257722739#2" to="-95542482#14" fromLane="0" toLane="0" via=":1123399591_2_0" dir="l" state="m" />\n  <connection from="1257722739#2" to="684741141#0" fromLane="0" toLane="0" via=":1123399591_3_0" dir="s" state="m" />\n  <connection from="1257722739#2" to="-243188243" fromLane="0" toLane="0" via=":1123399591_4_0" dir="r" state="m" />\n  <connection from="1270957860#0" to="1270957863" fromLane="0" toLane="0" via=":cluster_2505176169_856214432_5_0" dir="l" state="M" />\n  <connection from="1270957860#0" to="1227721991#0" fromLane="0" toLane="0" via=":cluster_2505176169_856214432_6_0" dir="s" state="M" />\n  <connection from="1270957861" to="969757852" fromLane="0" toLane="0" via=":cluster_849403384_849403500_8976428987_8976428988_3_0" tl="cluster_849403384_849403500_8976428987_8976428988" linkIndex="3" dir="l" state="O" />\n  <connection from="1270957861" to="990212968" fromLane="0" toLane="0" via=":cluster_849403384_849403500_8976428987_8976428988_4_0" tl="cluster_849403384_849403500_8976428987_8976428988" linkIndex="4" dir="s" state="O" />\n  <connection from="1270957861" to="990212968" fromLane="1" toLane="1" via=":cluster_849403384_849403500_8976428987_8976428988_4_1" tl="cluster_849403384_849403500_8976428987_8976428988" linkIndex="5" dir="s" state="O" />\n  <connection from="1270957861" to="970010765#0" fromLane="1" toLane="0" via=":cluster_849403384_849403500_8976428987_8976428988_6_0" tl="cluster_849403384_849403500_8976428987_8976428988" linkIndex="6" dir="r" state="o" />\n  <connection from="1270957862" to="243314192" fromLane="0" toLane="0" via=":2507400639_1_0" dir="l" state="M" />\n  <connection from="1270957862" to="1270957861" fromLane="0" toLane="0" via=":2507400639_2_0" dir="s" state="M" />\n  <connection from="1270957862" to="1270957861" fromLane="1" toLane="1" via=":2507400639_2_1" dir="s" state="M" />\n  <connection from="1270957863" to="706105951#3" fromLane="0" toLane="0" via=":856214524_0_0" dir="l" state="M" />\n  <connection from="1270957863" to="185951074#0" fromLane="0" toLane="0" via=":856214524_1_0" dir="s" state="M" />\n  <connection from="1270957863" to="-706105951#2" fromLane="0" toLane="0" via=":856214524_2_0" dir="r" state="m" />\n  <connection from="129827552#0" to="129827552#1" fromLane="0" toLane="0" via=":1680666546_4_0" dir="s" state="m" />\n  <connection from="129827552#0" to="155690791#0" fromLane="0" toLane="0" via=":1680666546_5_0" dir="r" state="m" />\n  <connection from="129827552#1" to="-96837170#2" fromLane="0" toLane="0" via=":1122072674_4_0" dir="l" state="m" />\n  <connection from="129827552#1" to="96837170#3" fromLane="0" toLane="0" via=":1122072674_5_0" dir="r" state="m" />\n  <connection from="129827554#0" to="129827554#1" fromLane="0" toLane="0" via=":1680666468_0_0" dir="s" state="M" />\n  <connection from="129827554#0" to="155690741#0" fromLane="0" toLane="0" via=":1680666468_1_0" dir="r" state="m" />\n  <connection from="129827554#1" to="1374050353" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_13_0" dir="l" state="m" />\n  <connection from="129827554#1" to="-1454593202#2" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_14_0" dir="s" state="m" />\n  <connection from="129827554#1" to="-148941242#5" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_15_0" dir="r" state="m" />\n  <connection from="129827561#0" to="129827561#1" fromLane="0" toLane="0" via=":14188764645_0_0" dir="s" state="M" />\n  <connection from="129827561#0" to="1559841160#0" fromLane="0" toLane="0" via=":14188764645_1_0" dir="r" state="m" />\n  <connection from="129827561#1" to="969765781" fromLane="0" toLane="0" via=":1431715406_0_0" dir="l" state="m" />\n  <connection from="129827561#1" to="706256385" fromLane="0" toLane="0" via=":1431715406_1_0" dir="s" state="m" />\n  <connection from="129827562" to="706096643#1" fromLane="0" toLane="0" via=":1431715421_0_0" dir="l" state="M" />\n  <connection from="129842388#1" to="155691088#1" fromLane="0" toLane="0" via=":1680666463_9_0" dir="l" state="M" />\n  <connection from="129842388#1" to="129842388#2" fromLane="0" toLane="0" via=":1680666463_10_0" dir="s" state="M" />\n  <connection from="129842388#1" to="-155691088#0" fromLane="0" toLane="0" via=":1680666463_11_0" dir="r" state="m" />\n  <connection from="129842388#2" to="142455592#0" fromLane="0" toLane="0" via=":1431867166_4_0" dir="l" state="M" />\n  <connection from="129842388#2" to="1457057266#0" fromLane="0" toLane="0" via=":1431867166_5_0" dir="r" state="M" />\n  <connection from="129842397#0" to="-706087699#1" fromLane="0" toLane="0" via=":1431867103_2_0" dir="l" state="M" />\n  <connection from="129842397#0" to="129842397#1" fromLane="0" toLane="0" via=":1431867103_3_0" dir="s" state="M" />\n  <connection from="129842397#1" to="129842397#2" fromLane="0" toLane="0" via=":8642733850_0_0" dir="s" state="M" />\n  <connection from="129842397#1" to="932324369" fromLane="0" toLane="0" via=":8642733850_1_0" dir="r" state="m" />\n  <connection from="129842397#2" to="142395239#0" fromLane="0" toLane="0" via=":1431867150_2_0" dir="l" state="m" />\n  <connection from="129842397#2" to="129842397#4" fromLane="0" toLane="0" via=":1431867150_3_0" dir="r" state="m" />\n  <connection from="129842397#4" to="-1457057266#1" fromLane="0" toLane="0" via=":1431867100_0_0" dir="r" state="M" />\n  <connection from="1335076429" to="243188212#0" fromLane="0" toLane="0" via=":12351390200_1_0" dir="s" state="M" />\n  <connection from="1365892748#0" to="498671536" fromLane="0" toLane="0" via=":2506087432_0_0" dir="l" state="M" />\n  <connection from="1365892748#0" to="95809261#0" fromLane="0" toLane="0" via=":2506087432_1_0" dir="s" state="m" />\n  <connection from="1365892989" to="71380773#0" fromLane="0" toLane="0" via=":1431715366_0_0" dir="r" state="m" />\n  <connection from="1365892991" to="-969765782" fromLane="0" toLane="0" via=":1110416410_1_0" dir="l" state="m" />\n  <connection from="1365892991" to="706256377#0" fromLane="0" toLane="0" via=":1110416410_2_0" dir="r" state="m" />\n  <connection from="1365935945#0" to="498671393" fromLane="0" toLane="0" via=":4900465166_6_0" dir="l" state="M" />\n  <connection from="1365935945#0" to="1365935945#1" fromLane="0" toLane="0" via=":4900465166_7_0" dir="s" state="M" />\n  <connection from="1365935945#0" to="498671465" fromLane="0" toLane="0" via=":4900465166_8_0" dir="r" state="m" />\n  <connection from="1365935945#1" to="-706520617#0" fromLane="0" toLane="0" via=":12649445299_2_0" dir="l" state="m" />\n  <connection from="1365935945#1" to="706520617#1" fromLane="0" toLane="0" via=":12649445299_3_0" dir="r" state="m" />\n  <connection from="1374050353" to="706256379#1" fromLane="0" toLane="0" via=":1431715409_0_0" dir="s" state="M" />\n  <connection from="1374050353" to="96941844" fromLane="0" toLane="0" via=":1431715409_1_0" dir="r" state="m" />\n  <connection from="142395239#0" to="498671600" fromLane="0" toLane="0" via=":1558314095_4_0" dir="l" state="m" />\n  <connection from="142395239#0" to="142395239#1" fromLane="0" toLane="0" via=":1558314095_5_0" dir="r" state="m" />\n  <connection from="142395239#1" to="706087699#1" fromLane="0" toLane="0" via=":4900465031_0_0" dir="s" state="M" />\n  <connection from="142395239#1" to="-498671694" fromLane="0" toLane="0" via=":4900465031_1_0" dir="r" state="m" />\n  <connection from="142453602#0" to="142453602#1" fromLane="0" toLane="0" via=":1115585909_0_0" dir="s" state="M" />\n  <connection from="142453602#0" to="-498671393" fromLane="0" toLane="0" via=":1115585909_1_0" dir="r" state="m" />\n  <connection from="142453602#1" to="243188024" fromLane="0" toLane="0" via=":1115585463_2_0" dir="l" state="M" />\n  <connection from="142453602#1" to="142453602#2" fromLane="0" toLane="0" via=":1115585463_3_0" dir="s" state="M" />\n  <connection from="142453602#2" to="-969765780#2" fromLane="0" toLane="0" via=":1115585416_2_0" dir="l" state="M" />\n  <connection from="142453602#2" to="706520617#0" fromLane="0" toLane="0" via=":1115585416_3_0" dir="r" state="M" />\n  <connection from="142455590" to="-142455600#0" fromLane="0" toLane="0" via=":cluster_1559016910_4900465210_9_0" dir="l" state="m" />\n  <connection from="142455590" to="498671483" fromLane="0" toLane="0" via=":cluster_1559016910_4900465210_10_0" dir="s" state="m" />\n  <connection from="142455590" to="142455600#2" fromLane="0" toLane="0" via=":cluster_1559016910_4900465210_11_0" dir="r" state="m" />\n  <connection from="142455595" to="-142455600#2" fromLane="0" toLane="0" via=":1559016905_4_0" dir="l" state="m" />\n  <connection from="142455595" to="142455600#3" fromLane="0" toLane="0" via=":1559016905_5_0" dir="r" state="m" />\n  <connection from="142455599" to="706256386" fromLane="0" toLane="0" via=":1559016918_2_0" dir="s" state="M" />\n  <connection from="142455599" to="142455600#0" fromLane="0" toLane="0" via=":1559016918_3_0" dir="r" state="m" />\n  <connection from="142455600#0" to="498671483" fromLane="0" toLane="0" via=":cluster_1559016910_4900465210_6_0" dir="l" state="M" />\n  <connection from="142455600#0" to="142455600#2" fromLane="0" toLane="0" via=":cluster_1559016910_4900465210_7_0" dir="s" state="M" />\n  <connection from="142455600#0" to="-142455590" fromLane="0" toLane="0" via=":cluster_1559016910_4900465210_8_0" dir="r" state="m" />\n  <connection from="142455600#2" to="142455600#3" fromLane="0" toLane="0" via=":1559016905_2_0" dir="s" state="M" />\n  <connection from="142455600#2" to="-142455595" fromLane="0" toLane="0" via=":1559016905_3_0" dir="r" state="m" />\n  <connection from="142455600#3" to="-95809262#0" fromLane="0" toLane="0" via=":1559016903_6_0" dir="l" state="M" />\n  <connection from="142455600#3" to="142455600#4" fromLane="0" toLane="0" via=":1559016903_7_0" dir="s" state="M" />\n  <connection from="142455600#3" to="95809262#1" fromLane="0" toLane="0" via=":1559016903_8_0" dir="r" state="m" />\n  <connection from="142455600#4" to="-148941242#0" fromLane="0" toLane="0" via=":1110416726_3_0" dir="l" state="M" />\n  <connection from="142455600#4" to="129827554#0" fromLane="0" toLane="0" via=":1110416726_4_0" dir="s" state="M" />\n  <connection from="142455600#4" to="148941242#1" fromLane="0" toLane="0" via=":1110416726_5_0" dir="r" state="m" />\n  <connection from="142456544" to="84842974#12" fromLane="0" toLane="0" via=":1559029797_0_0" dir="l" state="m" />\n  <connection from="1454593202#0" to="1454593202#1" fromLane="0" toLane="0" via=":14188764647_0_0" dir="s" state="M" />\n  <connection from="1454593202#0" to="-1559841160#1" fromLane="0" toLane="0" via=":14188764647_1_0" dir="r" state="m" />\n  <connection from="1454593202#1" to="1454593202#2" fromLane="0" toLane="0" via=":1431715415_0_0" dir="s" state="M" />\n  <connection from="1454593202#1" to="129827561#0" fromLane="0" toLane="0" via=":1431715415_1_0" dir="r" state="m" />\n  <connection from="1454593202#2" to="-148941242#5" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_3_0" dir="l" state="M" />\n  <connection from="1454593202#2" to="-129827554#4" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_4_0" dir="L" state="M" />\n  <connection from="1454593202#2" to="1374050353" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_5_0" dir="s" state="M" />\n  <connection from="1454593203#1" to="-706256380#6" fromLane="0" toLane="0" via=":13339819888_1_0" dir="l" state="M" />\n  <connection from="1454593203#1" to="1454593203#2" fromLane="0" toLane="0" via=":13339819888_2_0" dir="s" state="M" />\n  <connection from="1454593203#2" to="1454593202#0" fromLane="0" toLane="0" via=":13339819885_0_0" dir="l" state="M" />\n  <connection from="1454593203#2" to="1454593203#3" fromLane="0" toLane="0" via=":13339819885_1_0" dir="s" state="M" />\n  <connection from="1454593203#3" to="-706256381" fromLane="0" toLane="0" via=":13339819893_1_0" dir="l" state="M" />\n  <connection from="1454593203#3" to="1454593203#4" fromLane="0" toLane="0" via=":13339819893_2_0" dir="s" state="M" />\n  <connection from="1454593203#4" to="-95533432#5" fromLane="0" toLane="0" via=":13339844003_1_0" dir="l" state="M" />\n  <connection from="1454593203#4" to="1454593203#1" fromLane="0" toLane="0" via=":13339844003_2_0" dir="s" state="M" />\n  <connection from="1457057266#0" to="-129842397#4" fromLane="0" toLane="0" via=":1431867100_1_0" dir="l" state="M" />\n  <connection from="1457057267" to="129842391" fromLane="0" toLane="0" via=":13362936997_0_0" dir="s" state="M" />\n  <connection from="148941242#1" to="148941242#2" fromLane="0" toLane="0" via=":1431715372_0_0" dir="s" state="M" />\n  <connection from="148941242#1" to="-95809262#1" fromLane="0" toLane="0" via=":1431715372_1_0" dir="r" state="m" />\n  <connection from="148941242#2" to="148941242#3" fromLane="0" toLane="0" via=":1559016904_0_0" dir="s" state="M" />\n  <connection from="148941242#2" to="142455595" fromLane="0" toLane="0" via=":1559016904_1_0" dir="r" state="m" />\n  <connection from="148941242#3" to="-155690741#2" fromLane="0" toLane="0" via=":1559016909_0_0" dir="l" state="M" />\n  <connection from="148941242#3" to="148941242#4" fromLane="0" toLane="0" via=":1559016909_1_0" dir="s" state="M" />\n  <connection from="148941242#3" to="142455590" fromLane="0" toLane="0" via=":1559016909_2_0" dir="r" state="m" />\n  <connection from="148941242#4" to="148941242#5" fromLane="0" toLane="0" via=":1559016916_0_0" dir="s" state="M" />\n  <connection from="148941242#4" to="142455599" fromLane="0" toLane="0" via=":1559016916_1_0" dir="r" state="m" />\n  <connection from="148941242#5" to="-129827554#4" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_0_0" dir="l" state="m" />\n  <connection from="148941242#5" to="1374050353" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_1_0" dir="L" state="m" />\n  <connection from="148941242#5" to="-1454593202#2" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_2_0" dir="r" state="m" />\n  <connection from="148941245" to="129827554#0" fromLane="0" toLane="0" via=":1110416726_0_0" dir="l" state="m" />\n  <connection from="148941245" to="148941242#1" fromLane="0" toLane="0" via=":1110416726_1_0" dir="s" state="m" />\n  <connection from="148941245" to="-142455604#1" fromLane="0" toLane="0" via=":1110416726_2_0" dir="r" state="m" />\n  <connection from="151072453#0" to="917695557" fromLane="0" toLane="0" via=":8523328986_1_0" dir="l" state="M" />\n  <connection from="151072453#0" to="151072453#1" fromLane="0" toLane="0" via=":8523328986_2_0" dir="s" state="M" />\n  <connection from="151072453#1" to="-243079757#3" fromLane="0" toLane="0" via=":1639088658_2_0" dir="l" state="M" />\n  <connection from="151072453#1" to="151072453#2" fromLane="0" toLane="0" via=":1639088658_3_0" dir="s" state="M" />\n  <connection from="151072453#1" to="1218747873" fromLane="0" toLane="0" via=":1639088658_4_0" dir="r" state="M" />\n  <connection from="151072453#2" to="151072451#0" fromLane="0" toLane="0" via=":1639088623_1_0" dir="l" state="M" />\n  <connection from="151072453#2" to="151072453#3" fromLane="0" toLane="0" via=":1639088623_2_0" dir="s" state="m" />\n  <connection from="151072453#3" to="243314312" fromLane="0" toLane="0" via=":2507400638_2_0" dir="l" state="M" />\n  <connection from="151072453#3" to="969757855#1" fromLane="0" toLane="0" via=":2507400638_3_0" dir="s" state="M" />\n  <connection from="151072453#3" to="-243314192" fromLane="0" toLane="0" via=":2507400638_4_0" dir="r" state="M" />\n  <connection from="154097280" to="-96941857#2" fromLane="0" toLane="0" via=":1966404349_3_0" dir="s" state="M" />\n  <connection from="154097280" to="498671395#1" fromLane="0" toLane="0" via=":1966404349_4_0" dir="r" state="m" />\n  <connection from="154707523#0" to="-706256377#0" fromLane="0" toLane="0" via=":2052547075_2_0" dir="l" state="m" />\n  <connection from="154707523#0" to="154707523#1" fromLane="0" toLane="0" via=":2052547075_3_0" dir="s" state="m" />\n  <connection from="154707523#0" to="706256377#1" fromLane="0" toLane="0" via=":2052547075_4_0" dir="r" state="m" />\n  <connection from="154707523#1" to="-706256379#1" fromLane="0" toLane="0" via=":cluster_1122072658_14138648129_2_0" dir="l" state="m" />\n  <connection from="154707523#1" to="96837170#0" fromLane="0" toLane="0" via=":cluster_1122072658_14138648129_3_0" dir="s" state="m" />\n  <connection from="154707523#1" to="706256379#3" fromLane="0" toLane="0" via=":cluster_1122072658_14138648129_4_0" dir="r" state="m" />\n  <connection from="1547102487#0" to="243314310#1" fromLane="0" toLane="0" via=":2507400591_2_0" dir="l" state="M" />\n  <connection from="1547102487#0" to="1547102487#1" fromLane="0" toLane="0" via=":2507400591_3_0" dir="s" state="M" />\n  <connection from="1547102487#0" to="-243314310#0" fromLane="0" toLane="0" via=":2507400591_4_0" dir="r" state="M" />\n  <connection from="1547102487#1" to="155691084#1" fromLane="0" toLane="0" via=":4900465949_2_0" dir="l" state="m" />\n  <connection from="1547102487#1" to="705855017#1" fromLane="0" toLane="0" via=":4900465949_3_0" dir="s" state="m" />\n  <connection from="1547102487#1" to="-155691084#0" fromLane="0" toLane="0" via=":4900465949_4_0" dir="r" state="m" />\n  <connection from="1552748564#0" to="-498671328#1" fromLane="0" toLane="0" via=":4900465706_0_0" dir="l" state="M" />\n  <connection from="1552748564#0" to="1552748564#1" fromLane="0" toLane="0" via=":4900465706_1_0" dir="s" state="M" />\n  <connection from="1552748564#1" to="77967090#1" fromLane="0" toLane="0" via=":916327912_0_0" dir="l" state="m" />\n  <connection from="155690480#0" to="-498671497" fromLane="0" toLane="0" via=":4900465244_4_0" dir="l" state="M" />\n  <connection from="155690480#0" to="155690480#1" fromLane="0" toLane="0" via=":4900465244_5_0" dir="s" state="M" />\n  <connection from="155690480#1" to="990212966#1" fromLane="0" toLane="0" via=":796747714_0_0" dir="l" state="m" />\n  <connection from="155690513" to="990212966#2" fromLane="0" toLane="0" via=":1680667730_0_0" dir="l" state="m" />\n  <connection from="155690528" to="95533432#5" fromLane="0" toLane="0" via=":1680667181_0_0" dir="l" state="m" />\n  <connection from="155690528" to="-95533432#4" fromLane="0" toLane="0" via=":1680667181_1_0" dir="r" state="m" />\n  <connection from="155690734#0" to="155691015#1" fromLane="0" toLane="0" via=":1680667079_6_0" dir="l" state="M" />\n  <connection from="155690734#0" to="155690734#1" fromLane="0" toLane="0" via=":1680667079_7_0" dir="s" state="M" />\n  <connection from="155690734#0" to="-155691015#0" fromLane="0" toLane="0" via=":1680667079_8_0" dir="r" state="m" />\n  <connection from="155690741#0" to="155690741#2" fromLane="0" toLane="0" via=":14182422401_4_0" dir="s" state="M" />\n  <connection from="155690741#0" to="1559020882" fromLane="0" toLane="0" via=":14182422401_5_0" dir="r" state="m" />\n  <connection from="155690741#2" to="148941242#4" fromLane="0" toLane="0" via=":1559016909_9_0" dir="l" state="m" />\n  <connection from="155690741#2" to="142455590" fromLane="0" toLane="0" via=":1559016909_10_0" dir="s" state="m" />\n  <connection from="155690741#2" to="-148941242#3" fromLane="0" toLane="0" via=":1559016909_11_0" dir="r" state="m" />\n  <connection from="155690791#0" to="1546907364" fromLane="0" toLane="0" via=":14070071629_1_0" dir="l" state="M" />\n  <connection from="155690812#0" to="155690812#1" fromLane="0" toLane="0" via=":1680667164_2_0" dir="s" state="M" />\n  <connection from="155690812#0" to="-155691015#1" fromLane="0" toLane="0" via=":1680667164_3_0" dir="r" state="m" />\n  <connection from="155690875" to="95533432#3" fromLane="0" toLane="0" via=":cluster_1680667233_2506085475_0_0" dir="l" state="m" />\n  <connection from="155690875" to="243188458" fromLane="0" toLane="0" via=":cluster_1680667233_2506085475_1_0" dir="s" state="m" />\n  <connection from="155690875" to="-95533432#1" fromLane="0" toLane="0" via=":cluster_1680667233_2506085475_2_0" dir="r" state="m" />\n  <connection from="155690878#0" to="155690878#1" fromLane="0" toLane="0" via=":1680667029_4_0" dir="s" state="M" />\n  <connection from="155690878#0" to="155691032" fromLane="0" toLane="0" via=":1680667029_5_0" dir="r" state="m" />\n  <connection from="155691015#0" to="-155690734#0" fromLane="0" toLane="0" via=":1680667079_9_0" dir="l" state="m" />\n  <connection from="155691015#0" to="155691015#1" fromLane="0" toLane="0" via=":1680667079_10_0" dir="s" state="m" />\n  <connection from="155691015#0" to="155690734#1" fromLane="0" toLane="0" via=":1680667079_11_0" dir="r" state="m" />\n  <connection from="155691015#1" to="-155690812#0" fromLane="0" toLane="0" via=":1680667164_4_0" dir="l" state="m" />\n  <connection from="155691015#1" to="155690812#1" fromLane="0" toLane="0" via=":1680667164_5_0" dir="r" state="m" />\n  <connection from="155691022#0" to="155691022#1" fromLane="0" toLane="0" via=":4900465071_0_0" dir="s" state="M" />\n  <connection from="155691022#0" to="-498671433" fromLane="0" toLane="0" via=":4900465071_1_0" dir="r" state="m" />\n  <connection from="155691084#0" to="155691084#1" fromLane="0" toLane="0" via=":4900465949_5_0" dir="s" state="M" />\n  <connection from="155691084#0" to="705855017#1" fromLane="0" toLane="0" via=":4900465949_6_0" dir="r" state="m" />\n  <connection from="155691084#1" to="155691015#0" fromLane="0" toLane="0" via=":1680666827_4_0" dir="l" state="M" />\n  <connection from="155691084#1" to="155691084#2" fromLane="0" toLane="0" via=":1680666827_5_0" dir="s" state="M" />\n  <connection from="155691084#2" to="155691022#0" fromLane="0" toLane="0" via=":1680666891_4_0" dir="l" state="M" />\n  <connection from="155691084#2" to="155691084#3" fromLane="0" toLane="0" via=":1680666891_5_0" dir="s" state="M" />\n  <connection from="155691088#0" to="-129842388#1" fromLane="0" toLane="0" via=":1680666463_0_0" dir="l" state="m" />\n  <connection from="155691088#0" to="155691088#1" fromLane="0" toLane="0" via=":1680666463_1_0" dir="s" state="m" />\n  <connection from="155691088#0" to="129842388#2" fromLane="0" toLane="0" via=":1680666463_2_0" dir="r" state="m" />\n  <connection from="1558734968" to="-498671331" fromLane="0" toLane="0" via=":2506088476_4_0" dir="s" state="M" />\n  <connection from="1558734968" to="-706095305" fromLane="0" toLane="0" via=":2506088476_5_0" dir="r" state="m" />\n  <connection from="1559802495#0" to="1559802496" fromLane="0" toLane="0" via=":14188374629_0_0" dir="l" state="M" />\n  <connection from="1559802495#0" to="1559802495#1" fromLane="0" toLane="0" via=":14188374629_1_0" dir="s" state="M" />\n  <connection from="1559802496" to="243188508#3" fromLane="0" toLane="0" via=":cluster_14188374630_2506088509_6_0" dir="l" state="m" />\n  <connection from="1559802496" to="-243188212#1" fromLane="0" toLane="0" via=":cluster_14188374630_2506088509_7_0" dir="s" state="m" />\n  <connection from="1559802496" to="-243188508#1" fromLane="0" toLane="0" via=":cluster_14188374630_2506088509_8_0" dir="r" state="m" />\n  <connection from="1559841160#0" to="1559841161" fromLane="0" toLane="0" via=":14188764646_4_0" dir="s" state="M" />\n  <connection from="1559841160#0" to="1559841160#1" fromLane="0" toLane="0" via=":14188764646_5_0" dir="r" state="m" />\n  <connection from="1559841160#1" to="-1454593202#0" fromLane="0" toLane="0" via=":14188764647_2_0" dir="l" state="m" />\n  <connection from="1559841160#1" to="1454593202#1" fromLane="0" toLane="0" via=":14188764647_3_0" dir="r" state="m" />\n  <connection from="185951074#0" to="1447303746" fromLane="0" toLane="0" via=":13279030938_2_0" dir="l" state="M" />\n  <connection from="185951074#0" to="185951074#1" fromLane="0" toLane="0" via=":13279030938_3_0" dir="s" state="M" />\n  <connection from="185951074#1" to="185951074#2" fromLane="0" toLane="0" via=":4900465707_0_0" dir="s" state="M" />\n  <connection from="185951074#2" to="-498671615" fromLane="0" toLane="0" via=":1123399600_0_0" dir="l" state="M" />\n  <connection from="185951074#2" to="185951074#4" fromLane="0" toLane="0" via=":1123399600_1_0" dir="s" state="M" />\n  <connection from="185951074#2" to="96941849" fromLane="0" toLane="0" via=":1123399600_2_0" dir="r" state="m" />\n  <connection from="185951074#4" to="1204844051" fromLane="0" toLane="0" via=":4900464584_0_0" dir="s" state="M" />\n  <connection from="194743656#0" to="194743656#1" fromLane="0" toLane="0" via=":2506086743_0_0" dir="s" state="M" />\n  <connection from="194743656#0" to="-243188536#2" fromLane="0" toLane="0" via=":2506086743_1_0" dir="r" state="M" />\n  <connection from="194743656#1" to="84842974#7" fromLane="0" toLane="0" via=":2052522376_0_0" dir="l" state="m" />\n  <connection from="243079656" to="497290261#3" fromLane="0" toLane="0" via=":1115585857_1_0" dir="s" state="m" visibility="9.00" />\n  <connection from="243079673" to="970323546#3" fromLane="0" toLane="0" via=":2505176494_0_0" dir="l" state="m" />\n  <connection from="243079723" to="66984613#1" fromLane="0" toLane="0" via=":947896972_0_0" dir="l" state="m" />\n  <connection from="243079757#1" to="151072453#2" fromLane="0" toLane="0" via=":1639088658_0_0" dir="l" state="m" />\n  <connection from="243079757#1" to="1218747873" fromLane="0" toLane="0" via=":1639088658_1_0" dir="s" state="m" />\n  <connection from="243079775" to="243314440" fromLane="0" toLane="0" via=":947896989_5_0" dir="s" state="m" />\n  <connection from="243079775" to="66984613#2" fromLane="0" toLane="0" via=":947896989_6_0" dir="r" state="m" />\n  <connection from="243079805" to="71380773#0" fromLane="0" toLane="0" via=":1431715366_1_0" dir="s" state="M" />\n  <connection from="243079844#0" to="243079844#1" fromLane="0" toLane="0" via=":809537690_2_0" dir="s" state="m" />\n  <connection from="243079844#0" to="706105952#1" fromLane="0" toLane="0" via=":809537690_3_0" dir="r" state="m" />\n  <connection from="243188024" to="969757851#3" fromLane="0" toLane="0" via=":2506085525_0_0" dir="l" state="m" />\n  <connection from="243188064#0" to="-498671576#0" fromLane="0" toLane="0" via=":4900465472_0_0" dir="l" state="m" />\n  <connection from="243188064#0" to="243188064#1" fromLane="0" toLane="0" via=":4900465472_1_0" dir="s" state="m" />\n  <connection from="243188064#0" to="498671576#1" fromLane="0" toLane="0" via=":4900465472_2_0" dir="r" state="m" />\n  <connection from="243188064#1" to="931354569#4" fromLane="0" toLane="0" via=":2506085628_0_0" dir="l" state="m" />\n  <connection from="243188064#1" to="-931354569#3" fromLane="0" toLane="0" via=":2506085628_1_0" dir="r" state="m" />\n  <connection from="243188085" to="243188786#2" fromLane="0" toLane="0" via=":2506088938_2_0" dir="l" state="m" />\n  <connection from="243188085" to="-243188786#1" fromLane="0" toLane="0" via=":2506088938_3_0" dir="r" state="m" />\n  <connection from="243188161" to="243188786#4" fromLane="0" toLane="0" via=":2506089487_2_0" dir="l" state="m" />\n  <connection from="243188161" to="-243188786#3" fromLane="0" toLane="0" via=":2506089487_3_0" dir="r" state="m" />\n  <connection from="243188174#0" to="-706095308#0" fromLane="0" toLane="0" via=":6634630596_6_0" dir="l" state="M" />\n  <connection from="243188174#0" to="243188174#1" fromLane="0" toLane="0" via=":6634630596_7_0" dir="s" state="M" />\n  <connection from="243188174#0" to="706095308#1" fromLane="0" toLane="0" via=":6634630596_8_0" dir="r" state="m" />\n  <connection from="243188174#1" to="243188174#2" fromLane="0" toLane="0" via=":4900465989_2_0" dir="s" state="M" />\n  <connection from="243188174#1" to="498671673" fromLane="0" toLane="0" via=":4900465989_3_0" dir="r" state="m" />\n  <connection from="243188174#2" to="-706095306" fromLane="0" toLane="0" via=":2506087732_1_0" dir="l" state="M" />\n  <connection from="243188196" to="-243188503#0" fromLane="0" toLane="0" via=":2506089901_2_0" dir="l" state="m" />\n  <connection from="243188196" to="243188503#1" fromLane="0" toLane="0" via=":2506089901_3_0" dir="r" state="m" />\n  <connection from="243188208" to="706104217#4" fromLane="0" toLane="0" via=":2506087962_4_0" dir="l" state="m" />\n  <connection from="243188208" to="-706104217#3" fromLane="0" toLane="0" via=":2506087962_5_0" dir="r" state="m" />\n  <connection from="243188212#0" to="706094741#1" fromLane="0" toLane="0" via=":1123399641_9_0" dir="l" state="M" />\n  <connection from="243188212#0" to="243188212#1" fromLane="0" toLane="0" via=":1123399641_10_0" dir="s" state="M" />\n  <connection from="243188212#0" to="-706094741#0" fromLane="0" toLane="0" via=":1123399641_11_0" dir="r" state="m" />\n  <connection from="243188212#1" to="-243188508#1" fromLane="0" toLane="0" via=":cluster_14188374630_2506088509_0_0" dir="l" state="m" />\n  <connection from="243188212#1" to="-1559802496" fromLane="0" toLane="0" via=":cluster_14188374630_2506088509_1_0" dir="s" state="m" />\n  <connection from="243188212#1" to="243188508#3" fromLane="0" toLane="0" via=":cluster_14188374630_2506088509_2_0" dir="r" state="m" />\n  <connection from="243188225" to="-452036923#0" fromLane="0" toLane="0" via=":4488366488_2_0" dir="l" state="m" />\n  <connection from="243188225" to="452036923#1" fromLane="0" toLane="0" via=":4488366488_3_0" dir="r" state="m" />\n  <connection from="243188243" to="-95542482#14" fromLane="0" toLane="0" via=":1123399591_5_0" dir="s" state="M" />\n  <connection from="243188243" to="684741141#0" fromLane="0" toLane="0" via=":1123399591_6_0" dir="r" state="m" />\n  <connection from="243188298" to="980451721#0" fromLane="0" toLane="0" via=":1123399676_2_0" dir="l" state="M" />\n  <connection from="243188298" to="980451722#0" fromLane="0" toLane="0" via=":1123399676_3_0" dir="r" state="m" />\n  <connection from="243188318" to="-95533432#0" fromLane="0" toLane="0" via=":2506085476_2_0" dir="l" state="m" />\n  <connection from="243188318" to="95533432#1" fromLane="0" toLane="0" via=":2506085476_3_0" dir="r" state="m" />\n  <connection from="243188351#0" to="-706105951#3" fromLane="0" toLane="0" via=":1107926639_9_0" dir="l" state="M" />\n  <connection from="243188351#0" to="243188629" fromLane="0" toLane="0" via=":1107926639_10_0" dir="s" state="M" />\n  <connection from="243188351#0" to="706105951#4" fromLane="0" toLane="0" via=":1107926639_11_0" dir="r" state="m" />\n  <connection from="243188361" to="243188786#3" fromLane="0" toLane="0" via=":2506089058_2_0" dir="l" state="m" />\n  <connection from="243188361" to="-243188786#2" fromLane="0" toLane="0" via=":2506089058_3_0" dir="r" state="m" />\n  <connection from="243188389#1" to="-498671589" fromLane="0" toLane="0" via=":2506088491_2_0" dir="l" state="M" />\n  <connection from="243188389#1" to="243188389#2" fromLane="0" toLane="0" via=":2506088491_3_0" dir="s" state="M" />\n  <connection from="243188389#2" to="498671331" fromLane="0" toLane="0" via=":2506088494_0_0" dir="l" state="M" />\n  <connection from="243188389#2" to="243188389#3" fromLane="0" toLane="0" via=":2506088494_1_0" dir="s" state="M" />\n  <connection from="243188389#3" to="243188243" fromLane="0" toLane="0" via=":2506090118_1_0" dir="r" state="M" />\n  <connection from="243188432#0" to="-706104219#6" fromLane="0" toLane="0" via=":2506086831_4_0" dir="l" state="m" />\n  <connection from="243188432#0" to="498671388#0" fromLane="0" toLane="0" via=":2506086831_5_0" dir="s" state="m" />\n  <connection from="243188432#0" to="706104219#7" fromLane="0" toLane="0" via=":2506086831_6_0" dir="r" state="m" />\n  <connection from="243188447" to="911123067" fromLane="0" toLane="0" via=":2506085849_0_0" dir="s" state="m" />\n  <connection from="243188447" to="684714761#3" fromLane="0" toLane="0" via=":2506085849_1_0" dir="r" state="m" />\n  <connection from="243188452#0" to="243188452#1" fromLane="0" toLane="0" via=":2506088612_2_0" dir="s" state="m" />\n  <connection from="243188452#0" to="243188594#1" fromLane="0" toLane="0" via=":2506088612_3_0" dir="r" state="m" />\n  <connection from="243188452#1" to="243188641#1" fromLane="0" toLane="0" via=":2506088529_1_0" dir="r" state="m" />\n  <connection from="243188458" to="243188607#1" fromLane="0" toLane="0" via=":2506085493_2_0" dir="l" state="m" />\n  <connection from="243188458" to="-243188607#0" fromLane="0" toLane="0" via=":2506085493_3_0" dir="r" state="m" />\n  <connection from="243188465#0" to="243188465#1" fromLane="0" toLane="0" via=":4900465473_2_0" dir="s" state="M" />\n  <connection from="243188465#0" to="-498671576#1" fromLane="0" toLane="0" via=":4900465473_3_0" dir="r" state="m" />\n  <connection from="243188465#1" to="40264697#9" fromLane="0" toLane="0" via=":cluster_2052547073_2506085499_0_0" dir="l" state="m" />\n  <connection from="243188491" to="-706104219#8" fromLane="0" toLane="0" via=":1966399248_6_0" dir="l" state="m" />\n  <connection from="243188491" to="-96941842#3" fromLane="0" toLane="0" via=":1966399248_7_0" dir="s" state="m" />\n  <connection from="243188491" to="706104219#9" fromLane="0" toLane="0" via=":1966399248_8_0" dir="r" state="m" />\n  <connection from="243188499" to="-498671616" fromLane="0" toLane="0" via=":2506085631_4_0" dir="l" state="m" />\n  <connection from="243188499" to="498671639" fromLane="0" toLane="0" via=":2506085631_5_0" dir="s" state="m" />\n  <connection from="243188499" to="243188351#0" fromLane="0" toLane="0" via=":2506085631_6_0" dir="r" state="m" />\n  <connection from="243188503#0" to="243188503#1" fromLane="0" toLane="0" via=":2506089901_0_0" dir="s" state="M" />\n  <connection from="243188503#0" to="-243188196" fromLane="0" toLane="0" via=":2506089901_1_0" dir="r" state="m" />\n  <connection from="243188505" to="95809261#2" fromLane="0" toLane="0" via=":2506087968_2_0" dir="l" state="M" />\n  <connection from="243188508#0" to="-1559802496" fromLane="0" toLane="0" via=":cluster_14188374630_2506088509_9_0" dir="l" state="M" />\n  <connection from="243188508#0" to="243188508#3" fromLane="0" toLane="0" via=":cluster_14188374630_2506088509_10_0" dir="s" state="M" />\n  <connection from="243188508#0" to="-243188212#1" fromLane="0" toLane="0" via=":cluster_14188374630_2506088509_11_0" dir="r" state="m" />\n  <connection from="243188508#3" to="243188552#1" fromLane="0" toLane="0" via=":2506088492_4_0" dir="l" state="m" />\n  <connection from="243188508#3" to="-243188552#0" fromLane="0" toLane="0" via=":2506088492_5_0" dir="r" state="m" />\n  <connection from="243188527" to="706103887" fromLane="0" toLane="0" via=":2506087863_3_0" dir="s" state="m" />\n  <connection from="243188527" to="95809268#2" fromLane="0" toLane="0" via=":2506087863_4_0" dir="r" state="m" />\n  <connection from="243188536#0" to="243188536#1" fromLane="0" toLane="0" via=":8348961009_2_0" dir="s" state="M" />\n  <connection from="243188536#0" to="-898440288" fromLane="0" toLane="0" via=":8348961009_3_0" dir="r" state="m" />\n  <connection from="243188536#1" to="243188536#2" fromLane="0" toLane="0" via=":2506087304_2_0" dir="s" state="M" />\n  <connection from="243188536#1" to="243188103" fromLane="0" toLane="0" via=":2506087304_3_0" dir="r" state="m" />\n  <connection from="243188536#2" to="194743656#1" fromLane="0" toLane="0" via=":2506086743_2_0" dir="r" state="m" />\n  <connection from="243188552#0" to="-243188508#3" fromLane="0" toLane="0" via=":2506088492_0_0" dir="l" state="M" />\n  <connection from="243188552#0" to="243188552#1" fromLane="0" toLane="0" via=":2506088492_1_0" dir="s" state="M" />\n  <connection from="243188552#1" to="1559802495#0" fromLane="0" toLane="0" via=":4900465288_0_0" dir="l" state="M" />\n  <connection from="243188552#1" to="243188552#2" fromLane="0" toLane="0" via=":4900465288_1_0" dir="s" state="M" />\n  <connection from="243188552#2" to="95542482#2" fromLane="0" toLane="0" via=":1107926940_4_0" dir="l" state="m" />\n  <connection from="243188552#2" to="-95542482#1" fromLane="0" toLane="0" via=":1107926940_5_0" dir="r" state="m" />\n  <connection from="243188563" to="706104217#9" fromLane="0" toLane="0" via=":2506088002_4_0" dir="l" state="M" />\n  <connection from="243188563" to="-706104217#8" fromLane="0" toLane="0" via=":2506088002_5_0" dir="r" state="M" />\n  <connection from="243188565" to="-706094741#1" fromLane="0" toLane="0" via=":1123399483_2_0" dir="l" state="m" />\n  <connection from="243188565" to="706094741#2" fromLane="0" toLane="0" via=":1123399483_3_0" dir="r" state="m" />\n  <connection from="243188585" to="95542482#1" fromLane="0" toLane="0" via=":1107926561_4_0" dir="l" state="m" />\n  <connection from="243188585" to="-95542482#0" fromLane="0" toLane="0" via=":1107926561_5_0" dir="r" state="m" />\n  <connection from="243188594#0" to="243188452#1" fromLane="0" toLane="0" via=":2506088612_0_0" dir="l" state="M" />\n  <connection from="243188594#0" to="243188594#1" fromLane="0" toLane="0" via=":2506088612_1_0" dir="s" state="M" />\n  <connection from="243188594#1" to="-706094741#3" fromLane="0" toLane="0" via=":1123399489_1_0" dir="l" state="M" />\n  <connection from="243188594#1" to="684740716" fromLane="0" toLane="0" via=":1123399489_2_0" dir="r" state="M" />\n  <connection from="243188607#0" to="-243188458" fromLane="0" toLane="0" via=":2506085493_4_0" dir="l" state="M" />\n  <connection from="243188607#0" to="243188607#1" fromLane="0" toLane="0" via=":2506085493_5_0" dir="s" state="M" />\n  <connection from="243188627" to="497290263#1" fromLane="0" toLane="0" via=":2506089600_2_0" dir="l" state="m" />\n  <connection from="243188627" to="-497290263#0" fromLane="0" toLane="0" via=":2506089600_3_0" dir="r" state="m" />\n  <connection from="243188629" to="1270957860#0" fromLane="0" toLane="0" via=":796747634_0_0" dir="l" state="m" />\n  <connection from="243188641#0" to="243188641#1" fromLane="0" toLane="0" via=":2506088529_0_0" dir="s" state="M" />\n  <connection from="243188641#1" to="-706094741#2" fromLane="0" toLane="0" via=":1123399659_1_0" dir="l" state="M" />\n  <connection from="243188641#1" to="706094741#3" fromLane="0" toLane="0" via=":1123399659_2_0" dir="r" state="M" />\n  <connection from="243188646#0" to="243188646#1" fromLane="0" toLane="0" via=":2506087522_0_0" dir="s" state="M" />\n  <connection from="243188646#0" to="-498671673" fromLane="0" toLane="0" via=":2506087522_1_0" dir="r" state="M" />\n  <connection from="243188646#1" to="84842974#1" fromLane="0" toLane="0" via=":cluster_2506087462_8523307708_0_0" dir="l" state="M" />\n  <connection from="243188646#1" to="-706095308#1" fromLane="0" toLane="0" via=":cluster_2506087462_8523307708_1_0" dir="r" state="M" />\n  <connection from="243188670#0" to="243188670#1" fromLane="0" toLane="0" via=":2506088715_0_0" dir="s" state="M" />\n  <connection from="243188670#0" to="243188670#1" fromLane="1" toLane="1" via=":2506088715_0_1" dir="s" state="M" />\n  <connection from="243188670#0" to="243188452#0" fromLane="1" toLane="0" via=":2506088715_2_0" dir="r" state="M" />\n  <connection from="243188670#1" to="95542482#6" fromLane="0" toLane="0" via=":1107926734_4_0" dir="l" state="M" />\n  <connection from="243188670#1" to="243188670#2" fromLane="0" toLane="0" via=":1107926734_5_0" dir="s" state="M" />\n  <connection from="243188670#1" to="243188670#2" fromLane="1" toLane="1" via=":1107926734_5_1" dir="s" state="M" />\n  <connection from="243188670#1" to="-95542482#5" fromLane="1" toLane="0" via=":1107926734_7_0" dir="r" state="M" />\n  <connection from="243188670#2" to="-980451721#0" fromLane="0" toLane="0" via=":2506089309_2_0" dir="l" state="M" />\n  <connection from="243188670#2" to="980451721#1" fromLane="1" toLane="0" via=":2506089309_3_0" dir="r" state="M" />\n  <connection from="243188705" to="706094741#0" fromLane="0" toLane="0" via=":1123399516_0_0" dir="L" state="M" />\n  <connection from="243188705" to="243188552#0" fromLane="0" toLane="0" via=":1123399516_1_0" dir="s" state="M" />\n  <connection from="243188710#0" to="498671427" fromLane="0" toLane="0" via=":1107926467_2_0" dir="l" state="M" />\n  <connection from="243188710#0" to="243188710#1" fromLane="0" toLane="0" via=":1107926467_3_0" dir="s" state="M" />\n  <connection from="243188710#1" to="-122540020#7" fromLane="0" toLane="0" via=":1369020939_2_0" dir="l" state="M" />\n  <connection from="243188710#1" to="243188710#2" fromLane="0" toLane="0" via=":1369020939_3_0" dir="s" state="M" />\n  <connection from="243188710#2" to="95542485#0" fromLane="0" toLane="0" via=":1107926481_0_0" dir="l" state="m" />\n  <connection from="243188710#2" to="95542482#0" fromLane="0" toLane="0" via=":1107926481_1_0" dir="s" state="m" />\n  <connection from="243188710#2" to="95809258" fromLane="0" toLane="0" via=":1107926481_2_0" dir="r" state="m" />\n  <connection from="243188741#0" to="243188741#1" fromLane="0" toLane="0" via=":2506085555_0_0" dir="s" state="M" />\n  <connection from="243188741#0" to="243188499" fromLane="0" toLane="0" via=":2506085555_1_0" dir="r" state="m" />\n  <connection from="243188741#1" to="498671615" fromLane="0" toLane="0" via=":4900465999_4_0" dir="s" state="m" />\n  <connection from="243188741#1" to="498671616" fromLane="0" toLane="0" via=":4900465999_5_0" dir="r" state="m" />\n  <connection from="243188765#0" to="-497290263#4" fromLane="0" toLane="0" via=":1110416742_4_0" dir="l" state="M" />\n  <connection from="243188765#0" to="243188765#1" fromLane="0" toLane="0" via=":1110416742_5_0" dir="s" state="M" />\n  <connection from="243188765#1" to="-498671432" fromLane="0" toLane="0" via=":4900465069_4_0" dir="l" state="M" />\n  <connection from="243188765#1" to="243188765#2" fromLane="0" toLane="0" via=":4900465069_5_0" dir="s" state="M" />\n  <connection from="243188785" to="-684741141#3" fromLane="0" toLane="0" via=":1123399502_0_0" dir="l" state="M" />\n  <connection from="243188785" to="243188786#1" fromLane="0" toLane="0" via=":1123399502_1_0" dir="s" state="M" />\n  <connection from="243188786#1" to="-243188085" fromLane="0" toLane="0" via=":2506088938_4_0" dir="l" state="M" />\n  <connection from="243188786#1" to="243188786#2" fromLane="0" toLane="0" via=":2506088938_5_0" dir="s" state="M" />\n  <connection from="243188786#2" to="-243188361" fromLane="0" toLane="0" via=":2506089058_4_0" dir="l" state="M" />\n  <connection from="243188786#2" to="243188786#3" fromLane="0" toLane="0" via=":2506089058_5_0" dir="s" state="M" />\n  <connection from="243188786#3" to="-243188161" fromLane="0" toLane="0" via=":2506089487_4_0" dir="l" state="M" />\n  <connection from="243188786#3" to="243188786#4" fromLane="0" toLane="0" via=":2506089487_5_0" dir="s" state="M" />\n  <connection from="243188786#4" to="95542482#14" fromLane="0" toLane="0" via=":1110416754_4_0" dir="l" state="m" />\n  <connection from="243188786#4" to="-95542482#13" fromLane="0" toLane="0" via=":1110416754_5_0" dir="r" state="m" />\n  <connection from="243314073" to="970323546#6" fromLane="0" toLane="0" via=":2507400690_0_0" dir="l" state="m" />\n  <connection from="243314103" to="65080358#2" fromLane="0" toLane="0" via=":2507400748_0_0" dir="l" state="m" />\n  <connection from="243314112" to="66984613#6" fromLane="0" toLane="0" via=":2507400689_0_0" dir="l" state="m" />\n  <connection from="243314112" to="243314073" fromLane="0" toLane="0" via=":2507400689_1_0" dir="s" state="m" />\n  <connection from="243314160" to="-95542482#12" fromLane="0" toLane="0" via=":2507400772_2_0" dir="l" state="m" />\n  <connection from="243314160" to="95542482#13" fromLane="0" toLane="0" via=":2507400772_3_0" dir="r" state="m" />\n  <connection from="243314165#0" to="-917695719" fromLane="0" toLane="0" via=":8523326825_4_0" dir="l" state="M" />\n  <connection from="243314165#0" to="243314165#1" fromLane="0" toLane="0" via=":8523326825_5_0" dir="s" state="M" />\n  <connection from="243314165#1" to="-243314436#0" fromLane="0" toLane="0" via=":cluster_2507400612_2507400618_9_0" dir="l" state="m" />\n  <connection from="243314165#1" to="243314248#0" fromLane="0" toLane="0" via=":cluster_2507400612_2507400618_10_0" dir="s" state="m" />\n  <connection from="243314165#1" to="243314436#2" fromLane="0" toLane="0" via=":cluster_2507400612_2507400618_11_0" dir="r" state="m" />\n  <connection from="243314177" to="81350006#2" fromLane="0" toLane="0" via=":2507400773_0_0" dir="l" state="m" />\n  <connection from="243314177" to="-81350006#1" fromLane="0" toLane="0" via=":2507400773_1_0" dir="r" state="m" />\n  <connection from="243314186" to="970323546#4" fromLane="0" toLane="0" via=":cluster_2505176422_2505176434_0_0" dir="l" state="m" />\n  <connection from="243314186" to="65080358#1" fromLane="0" toLane="0" via=":cluster_2505176422_2505176434_1_0" dir="r" state="m" />\n  <connection from="243314192" to="243314312" fromLane="0" toLane="0" via=":2507400638_5_0" dir="s" state="m" />\n  <connection from="243314192" to="969757855#1" fromLane="0" toLane="0" via=":2507400638_6_0" dir="r" state="m" />\n  <connection from="243314213" to="970323546#5" fromLane="0" toLane="0" via=":2507400698_0_0" dir="l" state="m" />\n  <connection from="243314248#0" to="243314248#1" fromLane="0" toLane="0" via=":4900465377_4_0" dir="s" state="M" />\n  <connection from="243314248#0" to="-498671542" fromLane="0" toLane="0" via=":4900465377_5_0" dir="r" state="m" />\n  <connection from="243314310#1" to="970010767#1" fromLane="0" toLane="0" via=":2505176020_0_0" dir="l" state="m" />\n  <connection from="243314312" to="917695719" fromLane="0" toLane="0" via=":8523326826_1_0" dir="r" state="M" />\n  <connection from="243314378#0" to="917706996" fromLane="0" toLane="0" via=":8523449430_2_0" dir="l" state="M" />\n  <connection from="243314378#0" to="243314378#2" fromLane="0" toLane="0" via=":8523449430_3_0" dir="s" state="M" />\n  <connection from="243314378#2" to="-498671564" fromLane="0" toLane="0" via=":4900465441_2_0" dir="l" state="M" />\n  <connection from="243314378#2" to="243314378#3" fromLane="0" toLane="0" via=":4900465441_3_0" dir="s" state="M" />\n  <connection from="243314378#3" to="66984613#5" fromLane="0" toLane="0" via=":2507400696_0_0" dir="l" state="m" />\n  <connection from="243314378#3" to="243314213" fromLane="0" toLane="0" via=":2507400696_1_0" dir="s" state="m" />\n  <connection from="243314395" to="243188710#0" fromLane="0" toLane="0" via=":1107926968_0_0" dir="l" state="M" />\n  <connection from="243314436#0" to="243314248#0" fromLane="0" toLane="0" via=":cluster_2507400612_2507400618_6_0" dir="l" state="M" />\n  <connection from="243314436#0" to="243314436#2" fromLane="0" toLane="0" via=":cluster_2507400612_2507400618_7_0" dir="s" state="M" />\n  <connection from="243314436#0" to="-243314165#1" fromLane="0" toLane="0" via=":cluster_2507400612_2507400618_8_0" dir="r" state="m" />\n  <connection from="243314436#2" to="243314467#0" fromLane="0" toLane="0" via=":2507400604_4_0" dir="l" state="M" />\n  <connection from="243314436#2" to="243314436#3" fromLane="0" toLane="0" via=":2507400604_5_0" dir="s" state="M" />\n  <connection from="243314436#3" to="969757853#2" fromLane="0" toLane="0" via=":13449811511_0_0" dir="l" state="m" />\n  <connection from="243314467#0" to="498671542" fromLane="0" toLane="0" via=":2507400609_4_0" dir="l" state="M" />\n  <connection from="243314467#0" to="243314467#1" fromLane="0" toLane="0" via=":2507400609_5_0" dir="s" state="M" />\n  <connection from="307427662" to="40264696#3" fromLane="0" toLane="0" via=":3125684630_3_0" dir="l" state="m" />\n  <connection from="324267161#0" to="77967093#3" fromLane="0" toLane="0" via=":916327839_3_0" dir="l" state="m" visibility="9.00" />\n  <connection from="330615829#0" to="498671676#0" fromLane="0" toLane="0" via=":12346099054_0_0" dir="l" state="M" />\n  <connection from="330615829#0" to="330615829#1" fromLane="0" toLane="0" via=":12346099054_1_0" dir="s" state="M" />\n  <connection from="330615829#1" to="330615844" fromLane="0" toLane="0" via=":3376091528_0_0" dir="s" state="M" />\n  <connection from="330615844" to="243188670#0" fromLane="0" toLane="1" via=":cluster_1123399715_3376084911_3376091818_0_0" dir="s" state="M" />\n  <connection from="40264696#1" to="40264696#2" fromLane="0" toLane="0" via=":1196113662_0_0" dir="s" state="M" />\n  <connection from="40264696#1" to="40264696#2" fromLane="1" toLane="1" via=":1196113662_0_1" dir="s" state="M" />\n  <connection from="40264696#1" to="40264696#2" fromLane="2" toLane="2" via=":1196113662_0_2" dir="s" state="M" />\n  <connection from="40264696#1" to="101706795#0" fromLane="2" toLane="0" via=":1196113662_3_0" dir="r" state="M" />\n  <connection from="40264696#2" to="40264696#3" fromLane="0" toLane="0" via=":3125684630_0_0" dir="s" state="M" />\n  <connection from="40264696#2" to="40264696#3" fromLane="1" toLane="1" via=":3125684630_0_1" dir="s" state="M" />\n  <connection from="40264696#2" to="40264696#3" fromLane="2" toLane="2" via=":3125684630_0_2" dir="s" state="M" />\n  <connection from="40264696#3" to="684704215#1" fromLane="0" toLane="0" via=":617234535_0_0" dir="l" state="M" />\n  <connection from="40264696#3" to="40264696#4" fromLane="0" toLane="0" via=":617234535_1_0" dir="s" state="M" />\n  <connection from="40264696#3" to="40264696#4" fromLane="1" toLane="1" via=":617234535_1_1" dir="s" state="M" />\n  <connection from="40264696#3" to="40264696#4" fromLane="2" toLane="2" via=":617234535_1_2" dir="s" state="M" />\n  <connection from="40264696#4" to="40264696#5" fromLane="0" toLane="0" via=":617234280_0_0" dir="s" state="M" />\n  <connection from="40264696#4" to="40264696#5" fromLane="1" toLane="1" via=":617234280_0_1" dir="s" state="M" />\n  <connection from="40264696#4" to="40264696#5" fromLane="2" toLane="2" via=":617234280_0_2" dir="s" state="M" />\n  <connection from="40264696#4" to="40264697#2" fromLane="2" toLane="1" via=":617234280_3_0" dir="r" state="M" />\n  <connection from="40264697#1" to="40264696#5" fromLane="0" toLane="0" via=":617234280_4_0" dir="l" state="m" />\n  <connection from="40264697#1" to="40264696#5" fromLane="1" toLane="1" via=":617234280_4_1" dir="l" state="m" />\n  <connection from="40264697#1" to="40264697#2" fromLane="1" toLane="0" via=":617234280_6_0" dir="s" state="m" />\n  <connection from="40264697#1" to="40264697#2" fromLane="1" toLane="1" via=":617234280_6_1" dir="s" state="m" />\n  <connection from="40264697#2" to="40264697#3" fromLane="0" toLane="0" via=":617234576_4_0" dir="s" state="m" />\n  <connection from="40264697#2" to="40264697#3" fromLane="1" toLane="1" via=":617234576_4_1" dir="s" state="m" />\n  <connection from="40264697#2" to="84842974#13" fromLane="1" toLane="1" via=":617234576_6_0" dir="r" state="m" />\n  <connection from="40264697#2" to="84842974#13" fromLane="1" toLane="2" via=":617234576_6_1" dir="r" state="m" />\n  <connection from="40264697#3" to="706097788#0" fromLane="0" toLane="0" via=":617234281_2_0" dir="l" state="M" />\n  <connection from="40264697#3" to="40264697#4" fromLane="0" toLane="0" via=":617234281_3_0" dir="s" state="M" />\n  <connection from="40264697#3" to="40264697#4" fromLane="1" toLane="1" via=":617234281_3_1" dir="s" state="M" />\n  <connection from="40264697#4" to="1546894905" fromLane="0" toLane="0" via=":14069947700_1_0" dir="l" state="M" />\n  <connection from="40264697#4" to="40264697#5" fromLane="0" toLane="0" via=":14069947700_2_0" dir="s" state="M" />\n  <connection from="40264697#4" to="40264697#5" fromLane="1" toLane="1" via=":14069947700_2_1" dir="s" state="M" />\n  <connection from="40264697#5" to="706093509#0" fromLane="0" toLane="0" via=":1107784884_2_0" dir="l" state="M" />\n  <connection from="40264697#5" to="40264697#6" fromLane="0" toLane="0" via=":1107784884_3_0" dir="s" state="M" />\n  <connection from="40264697#5" to="40264697#6" fromLane="1" toLane="1" via=":1107784884_3_1" dir="s" state="M" />\n  <connection from="40264697#6" to="243188064#0" fromLane="0" toLane="0" via=":2506085496_0_0" dir="l" state="M" />\n  <connection from="40264697#6" to="40264697#7" fromLane="0" toLane="0" via=":2506085496_1_0" dir="s" state="M" />\n  <connection from="40264697#6" to="40264697#7" fromLane="1" toLane="1" via=":2506085496_1_1" dir="s" state="M" />\n  <connection from="40264697#7" to="-243188465#1" fromLane="0" toLane="0" via=":cluster_2052547073_2506085499_1_0" dir="l" state="M" />\n  <connection from="40264697#7" to="40264697#9" fromLane="0" toLane="0" via=":cluster_2052547073_2506085499_2_0" dir="s" state="M" />\n  <connection from="40264697#7" to="40264697#9" fromLane="1" toLane="1" via=":cluster_2052547073_2506085499_2_1" dir="s" state="M" />\n  <connection from="40264697#7" to="154707523#0" fromLane="1" toLane="0" via=":cluster_2052547073_2506085499_4_0" dir="r" state="M" />\n  <connection from="40264697#9" to="-706520617#4" fromLane="0" toLane="0" via=":1110416568_2_0" dir="l" state="M" />\n  <connection from="40264697#9" to="243079805" fromLane="0" toLane="0" via=":1110416568_3_0" dir="s" state="M" />\n  <connection from="40264697#9" to="1365892991" fromLane="1" toLane="0" via=":1110416568_4_0" dir="r" state="M" />\n  <connection from="452036922" to="-452036923#6" fromLane="0" toLane="0" via=":4488368090_0_0" dir="l" state="m" />\n  <connection from="452036922" to="452036923#7" fromLane="0" toLane="0" via=":4488368090_1_0" dir="r" state="m" />\n  <connection from="452036923#0" to="452036923#1" fromLane="0" toLane="0" via=":4488366488_0_0" dir="s" state="M" />\n  <connection from="452036923#0" to="-243188225" fromLane="0" toLane="0" via=":4488366488_1_0" dir="r" state="m" />\n  <connection from="452036923#1" to="452036923#2" fromLane="0" toLane="0" via=":1123399561_0_0" dir="s" state="M" />\n  <connection from="452036923#1" to="452036922" fromLane="0" toLane="0" via=":1123399561_1_0" dir="r" state="m" />\n  <connection from="452036923#2" to="452036923#3" fromLane="0" toLane="0" via=":1123399535_0_0" dir="s" state="M" />\n  <connection from="452036923#2" to="452037066" fromLane="0" toLane="0" via=":1123399535_1_0" dir="r" state="m" />\n  <connection from="452036923#3" to="452036923#4" fromLane="0" toLane="0" via=":4488358857_0_0" dir="s" state="M" />\n  <connection from="452036923#3" to="452037067" fromLane="0" toLane="0" via=":4488358857_1_0" dir="r" state="m" />\n  <connection from="452036923#4" to="452036923#5" fromLane="0" toLane="0" via=":4488358858_4_0" dir="s" state="M" />\n  <connection from="452036923#4" to="-452037067" fromLane="0" toLane="0" via=":4488358858_5_0" dir="r" state="m" />\n  <connection from="452036923#5" to="452036923#6" fromLane="0" toLane="0" via=":4488358856_4_0" dir="s" state="M" />\n  <connection from="452036923#5" to="-452037066" fromLane="0" toLane="0" via=":4488358856_5_0" dir="r" state="m" />\n  <connection from="452036923#6" to="452036923#7" fromLane="0" toLane="0" via=":4488368090_4_0" dir="s" state="M" />\n  <connection from="452036923#6" to="-452036922" fromLane="0" toLane="0" via=":4488368090_5_0" dir="r" state="m" />\n  <connection from="452036923#7" to="243188705" fromLane="0" toLane="0" via=":1123399683_2_0" dir="l" state="m" />\n  <connection from="452036923#7" to="243188225" fromLane="0" toLane="0" via=":1123399683_3_0" dir="r" state="m" />\n  <connection from="452037066" to="-452036923#5" fromLane="0" toLane="0" via=":4488358856_0_0" dir="l" state="m" />\n  <connection from="452037066" to="452036923#6" fromLane="0" toLane="0" via=":4488358856_1_0" dir="r" state="m" />\n  <connection from="452037067" to="-452036923#4" fromLane="0" toLane="0" via=":4488358858_0_0" dir="l" state="m" />\n  <connection from="452037067" to="452036923#5" fromLane="0" toLane="0" via=":4488358858_1_0" dir="r" state="m" />\n  <connection from="497290260" to="497290261#1" fromLane="0" toLane="0" via=":4888862222_0_0" dir="l" state="m" visibility="9.00" />\n  <connection from="497290261#1" to="497290262" fromLane="0" toLane="0" via=":4888861608_0_0" dir="l" state="M" />\n  <connection from="497290261#1" to="497290261#2" fromLane="0" toLane="0" via=":4888861608_1_0" dir="s" state="M" />\n  <connection from="497290261#2" to="497290261#3" fromLane="0" toLane="0" via=":1115585857_0_0" dir="s" state="M" />\n  <connection from="497290261#3" to="497290265" fromLane="0" toLane="0" via=":4888861618_0_0" dir="l" state="M" />\n  <connection from="497290261#3" to="497290261#4" fromLane="0" toLane="0" via=":4888861618_1_0" dir="s" state="M" />\n  <connection from="497290261#4" to="497290261#5" fromLane="0" toLane="0" via=":4888861612_1_0" dir="s" state="M" />\n  <connection from="497290261#5" to="497290264" fromLane="0" toLane="0" via=":4888861620_0_0" dir="s" state="M" />\n  <connection from="497290261#5" to="497290261#7" fromLane="0" toLane="0" via=":4888861620_1_0" dir="s" state="M" />\n  <connection from="497290261#7" to="497290261#1" fromLane="0" toLane="0" via=":4888862222_1_0" dir="s" state="M" />\n  <connection from="497290262" to="243188298" fromLane="0" toLane="0" via=":2506089779_0_0" dir="s" state="M" />\n  <connection from="497290262" to="243079656" fromLane="0" toLane="0" via=":2506089779_1_0" dir="r" state="m" />\n  <connection from="497290263#0" to="-243188627" fromLane="0" toLane="0" via=":2506089600_4_0" dir="l" state="M" />\n  <connection from="497290263#0" to="497290263#1" fromLane="0" toLane="0" via=":2506089600_5_0" dir="s" state="M" />\n  <connection from="497290263#1" to="243188644" fromLane="0" toLane="0" via=":2506089545_4_0" dir="l" state="M" />\n  <connection from="497290263#1" to="497290263#2" fromLane="0" toLane="0" via=":2506089545_5_0" dir="s" state="M" />\n  <connection from="497290263#2" to="243188115" fromLane="0" toLane="0" via=":1123399713_4_0" dir="l" state="M" />\n  <connection from="497290263#2" to="497290263#3" fromLane="0" toLane="0" via=":1123399713_5_0" dir="s" state="M" />\n  <connection from="497290263#3" to="243188445" fromLane="0" toLane="0" via=":1123399740_4_0" dir="l" state="M" />\n  <connection from="497290263#3" to="497290263#4" fromLane="0" toLane="0" via=":1123399740_5_0" dir="s" state="M" />\n  <connection from="497290263#4" to="243188765#1" fromLane="0" toLane="0" via=":1110416742_2_0" dir="l" state="m" />\n  <connection from="497290263#4" to="-243188765#0" fromLane="0" toLane="0" via=":1110416742_3_0" dir="r" state="m" />\n  <connection from="497290264" to="497290263#0" fromLane="0" toLane="0" via=":4888861605_1_0" dir="s" state="M" />\n  <connection from="497290265" to="497290261#5" fromLane="0" toLane="0" via=":4888861612_0_0" dir="s" state="m" visibility="9.00" />\n  <connection from="497290269" to="243079656" fromLane="0" toLane="0" via=":2506089779_4_0" dir="s" state="M" />\n  <connection from="497371052#0" to="129842391" fromLane="0" toLane="0" via=":13362936997_1_0" dir="r" state="m" />\n  <connection from="498671328#1" to="1552748564#1" fromLane="0" toLane="0" via=":4900465706_4_0" dir="l" state="m" />\n  <connection from="498671328#1" to="-1552748564#0" fromLane="0" toLane="0" via=":4900465706_5_0" dir="r" state="m" />\n  <connection from="498671330#0" to="498671330#1" fromLane="0" toLane="0" via=":4900465090_1_0" dir="s" state="M" />\n  <connection from="498671330#0" to="498671539" fromLane="0" toLane="0" via=":4900465090_2_0" dir="r" state="M" />\n  <connection from="498671330#1" to="931348308" fromLane="0" toLane="0" via=":4900466297_0_0" dir="l" state="m" />\n  <connection from="498671331" to="-706095305" fromLane="0" toLane="0" via=":2506088476_2_0" dir="l" state="M" />\n  <connection from="498671331" to="-1558734968" fromLane="0" toLane="0" via=":2506088476_3_0" dir="s" state="M" />\n  <connection from="498671388#0" to="96941857#1" fromLane="0" toLane="0" via=":4900466448_6_0" dir="l" state="m" />\n  <connection from="498671388#0" to="498671388#1" fromLane="0" toLane="0" via=":4900466448_7_0" dir="s" state="m" />\n  <connection from="498671388#0" to="-96941857#0" fromLane="0" toLane="0" via=":4900466448_8_0" dir="r" state="m" />\n  <connection from="498671388#1" to="-969757849#1" fromLane="0" toLane="0" via=":856214538_6_0" dir="l" state="m" />\n  <connection from="498671388#1" to="1365935945#0" fromLane="0" toLane="0" via=":856214538_7_0" dir="s" state="m" />\n  <connection from="498671388#1" to="969757849#2" fromLane="0" toLane="0" via=":856214538_8_0" dir="r" state="m" />\n  <connection from="498671389#0" to="96941857#2" fromLane="0" toLane="0" via=":4900464580_3_0" dir="l" state="m" />\n  <connection from="498671389#0" to="498671389#2" fromLane="0" toLane="0" via=":4900464580_4_0" dir="s" state="m" />\n  <connection from="498671389#0" to="-96941857#1" fromLane="0" toLane="0" via=":4900464580_5_0" dir="r" state="m" />\n  <connection from="498671389#2" to="-969757849#0" fromLane="0" toLane="0" via=":4900464598_2_0" dir="l" state="m" />\n  <connection from="498671389#2" to="969757849#1" fromLane="0" toLane="0" via=":4900464598_3_0" dir="r" state="m" />\n  <connection from="498671391" to="96242763#1" fromLane="0" toLane="0" via=":1123399771_0_0" dir="l" state="m" />\n  <connection from="498671393" to="-142453602#0" fromLane="0" toLane="0" via=":1115585909_2_0" dir="l" state="m" />\n  <connection from="498671393" to="142453602#1" fromLane="0" toLane="0" via=":1115585909_3_0" dir="r" state="m" />\n  <connection from="498671395#0" to="-96941857#2" fromLane="0" toLane="0" via=":1966404349_0_0" dir="l" state="m" />\n  <connection from="498671395#0" to="498671395#1" fromLane="0" toLane="0" via=":1966404349_1_0" dir="s" state="m" />\n  <connection from="498671395#0" to="-154097280" fromLane="0" toLane="0" via=":1966404349_2_0" dir="r" state="m" />\n  <connection from="498671395#1" to="706104219#3" fromLane="0" toLane="0" via=":cluster_1115585393_13472440789_947896971_0_0" tl="cluster_1115585393_13472440789_947896971" linkIndex="0" dir="l" state="o" />\n  <connection from="498671395#1" to="81350005#0" fromLane="0" toLane="0" via=":cluster_1115585393_13472440789_947896971_1_0" tl="cluster_1115585393_13472440789_947896971" linkIndex="1" dir="s" state="o" />\n  <connection from="498671395#1" to="-706104219#0" fromLane="0" toLane="0" via=":cluster_1115585393_13472440789_947896971_2_0" tl="cluster_1115585393_13472440789_947896971" linkIndex="2" dir="r" state="o" />\n  <connection from="498671427" to="65080358#3" fromLane="0" toLane="0" via=":4900465052_0_0" dir="l" state="m" />\n  <connection from="498671432" to="243188765#2" fromLane="0" toLane="0" via=":4900465069_2_0" dir="l" state="m" />\n  <connection from="498671432" to="-243188765#1" fromLane="0" toLane="0" via=":4900465069_3_0" dir="r" state="m" />\n  <connection from="498671433" to="-155691022#0" fromLane="0" toLane="0" via=":4900465071_2_0" dir="l" state="m" />\n  <connection from="498671433" to="155691022#1" fromLane="0" toLane="0" via=":4900465071_3_0" dir="r" state="m" />\n  <connection from="498671442" to="706093505#1" fromLane="0" toLane="0" via=":4900465099_0_0" dir="r" state="m" />\n  <connection from="498671465" to="-96941842#0" fromLane="0" toLane="0" via=":12649445300_2_0" dir="l" state="m" />\n  <connection from="498671465" to="96941842#1" fromLane="0" toLane="0" via=":12649445300_3_0" dir="r" state="m" />\n  <connection from="498671482" to="706256377#3" fromLane="0" toLane="0" via=":4900465209_0_0" dir="l" state="m" />\n  <connection from="498671482" to="-706256377#2" fromLane="0" toLane="0" via=":4900465209_1_0" dir="r" state="m" />\n  <connection from="498671483" to="706256380#4" fromLane="0" toLane="0" via=":1680666686_9_0" dir="l" state="m" />\n  <connection from="498671483" to="155690340" fromLane="0" toLane="0" via=":1680666686_10_0" dir="s" state="m" />\n  <connection from="498671483" to="-706256380#3" fromLane="0" toLane="0" via=":1680666686_11_0" dir="r" state="m" />\n  <connection from="498671497" to="155690480#1" fromLane="0" toLane="0" via=":4900465244_2_0" dir="l" state="m" />\n  <connection from="498671497" to="-155690480#0" fromLane="0" toLane="0" via=":4900465244_3_0" dir="r" state="m" />\n  <connection from="498671536" to="95809268#3" fromLane="0" toLane="0" via=":4900465363_0_0" dir="l" state="m" />\n  <connection from="498671536" to="1168168079" fromLane="0" toLane="0" via=":4900465363_1_0" dir="s" state="m" />\n  <connection from="498671542" to="-243314248#0" fromLane="0" toLane="0" via=":4900465377_0_0" dir="l" state="m" />\n  <connection from="498671542" to="243314248#1" fromLane="0" toLane="0" via=":4900465377_1_0" dir="r" state="m" />\n  <connection from="498671564" to="243314378#3" fromLane="0" toLane="0" via=":4900465441_0_0" dir="l" state="m" />\n  <connection from="498671564" to="-243314378#2" fromLane="0" toLane="0" via=":4900465441_1_0" dir="r" state="m" />\n  <connection from="498671569" to="72016745#5" fromLane="0" toLane="0" via=":2506087136_0_0" dir="l" state="m" />\n  <connection from="498671569" to="243188585" fromLane="0" toLane="0" via=":2506087136_1_0" dir="s" state="m" />\n  <connection from="498671569" to="-72016745#4" fromLane="0" toLane="1" via=":2506087136_2_0" dir="r" state="m" />\n  <connection from="498671576#0" to="243188064#1" fromLane="0" toLane="0" via=":4900465472_5_0" dir="l" state="M" />\n  <connection from="498671576#0" to="498671576#1" fromLane="0" toLane="0" via=":4900465472_6_0" dir="s" state="M" />\n  <connection from="498671576#1" to="-243188465#0" fromLane="0" toLane="0" via=":4900465473_4_0" dir="l" state="m" />\n  <connection from="498671576#1" to="243188465#1" fromLane="0" toLane="0" via=":4900465473_5_0" dir="r" state="m" />\n  <connection from="498671586" to="684714761#2" fromLane="0" toLane="0" via=":4900465513_0_0" dir="r" state="m" />\n  <connection from="498671589" to="243188389#2" fromLane="0" toLane="0" via=":2506088491_0_0" dir="l" state="m" />\n  <connection from="498671589" to="-243188389#1" fromLane="0" toLane="0" via=":2506088491_1_0" dir="r" state="m" />\n  <connection from="498671615" to="185951074#4" fromLane="0" toLane="0" via=":1123399600_9_0" dir="l" state="m" />\n  <connection from="498671615" to="96941849" fromLane="0" toLane="0" via=":1123399600_10_0" dir="s" state="m" />\n  <connection from="498671615" to="-185951074#3" fromLane="0" toLane="0" via=":1123399600_11_0" dir="r" state="m" />\n  <connection from="498671616" to="498671639" fromLane="0" toLane="0" via=":2506085631_2_0" dir="l" state="M" />\n  <connection from="498671616" to="243188351#0" fromLane="0" toLane="0" via=":2506085631_3_0" dir="s" state="M" />\n  <connection from="498671639" to="185951074#2" fromLane="0" toLane="0" via=":4900465707_2_0" dir="l" state="m" />\n  <connection from="498671639" to="-185951074#1" fromLane="0" toLane="0" via=":4900465707_3_0" dir="r" state="m" />\n  <connection from="498671673" to="243188646#1" fromLane="0" toLane="0" via=":2506087522_2_0" dir="r" state="m" />\n  <connection from="498671676#0" to="-706104217#10" fromLane="0" toLane="0" via=":2506088024_3_0" dir="l" state="m" />\n  <connection from="498671676#0" to="243188491" fromLane="0" toLane="0" via=":2506088024_4_0" dir="L" state="m" />\n  <connection from="498671676#0" to="706104217#11" fromLane="0" toLane="0" via=":2506088024_5_0" dir="s" state="m" />\n  <connection from="498671676#0" to="-330615760" fromLane="0" toLane="0" via=":2506088024_6_0" dir="r" state="m" />\n  <connection from="498671694" to="-706087699#0" fromLane="0" toLane="0" via=":4900465031_2_0" dir="l" state="m" />\n  <connection from="498671694" to="706087699#1" fromLane="0" toLane="0" via=":4900465031_3_0" dir="r" state="m" />\n  <connection from="498671702" to="95809267#1" fromLane="0" toLane="0" via=":12346099051_0_0" dir="l" state="m" />\n  <connection from="498671702" to="243188389#1" fromLane="0" toLane="0" via=":12346099051_1_0" dir="s" state="m" />\n  <connection from="530314558#1" to="706104219#0" fromLane="0" toLane="0" via=":1123399639_0_0" dir="l" state="m" />\n  <connection from="530314558#1" to="-706104218" fromLane="0" toLane="0" via=":1123399639_1_0" dir="r" state="m" />\n  <connection from="65080358#0" to="65080358#1" fromLane="0" toLane="0" via=":cluster_2505176422_2505176434_5_0" dir="s" state="M" />\n  <connection from="65080358#0" to="-243314186" fromLane="0" toLane="0" via=":cluster_2505176422_2505176434_6_0" dir="r" state="m" />\n  <connection from="65080358#1" to="-243314103" fromLane="0" toLane="0" via=":2507400748_1_0" dir="l" state="M" />\n  <connection from="65080358#1" to="65080358#2" fromLane="0" toLane="0" via=":2507400748_2_0" dir="s" state="M" />\n  <connection from="65080358#2" to="-498671427" fromLane="0" toLane="0" via=":4900465052_1_0" dir="l" state="M" />\n  <connection from="65080358#2" to="65080358#3" fromLane="0" toLane="0" via=":4900465052_2_0" dir="s" state="M" />\n  <connection from="65080358#3" to="243314395" fromLane="0" toLane="0" via=":1680668226_1_0" dir="l" state="M" />\n  <connection from="65080358#3" to="65080358#4" fromLane="0" toLane="0" via=":1680668226_2_0" dir="s" state="M" />\n  <connection from="66984604#1" to="151072453#0" fromLane="0" toLane="0" via=":809537820_0_0" tl="joinedS_809537820_856214571" linkIndex="0" dir="l" state="O" />\n  <connection from="66984604#1" to="66984604#2" fromLane="0" toLane="0" via=":809537820_1_0" tl="joinedS_809537820_856214571" linkIndex="1" dir="s" state="O" />\n  <connection from="66984604#2" to="1218225518#0" fromLane="0" toLane="0" via=":cluster_11292971032_2406954578_2406954579_2406954580_0_0" dir="l" state="m" />\n  <connection from="66984604#2" to="1218225524#1" fromLane="0" toLane="0" via=":cluster_11292971032_2406954578_2406954579_2406954580_1_0" dir="r" state="m" />\n  <connection from="66984610#0" to="66984610#1" fromLane="0" toLane="0" via=":2505176258_6_0" dir="s" state="m" />\n  <connection from="66984610#0" to="970323546#7" fromLane="0" toLane="1" via=":2505176258_7_0" dir="r" state="m" />\n  <connection from="66984610#1" to="706105952#0" fromLane="0" toLane="0" via=":2505176265_2_0" dir="r" state="m" />\n  <connection from="66984613#0" to="-243079723" fromLane="0" toLane="0" via=":947896972_1_0" dir="l" state="M" />\n  <connection from="66984613#0" to="66984613#1" fromLane="0" toLane="0" via=":947896972_2_0" dir="s" state="M" />\n  <connection from="66984613#1" to="243314440" fromLane="0" toLane="0" via=":947896989_2_0" dir="l" state="M" />\n  <connection from="66984613#1" to="66984613#2" fromLane="0" toLane="0" via=":947896989_3_0" dir="s" state="M" />\n  <connection from="66984613#1" to="-243079775" fromLane="0" toLane="0" via=":947896989_4_0" dir="r" state="M" />\n  <connection from="66984613#2" to="66984613#3" fromLane="0" toLane="0" via=":947896979_0_0" dir="s" state="M" />\n  <connection from="66984613#2" to="243079673" fromLane="0" toLane="0" via=":947896979_1_0" dir="r" state="M" />\n  <connection from="66984613#3" to="-96713525" fromLane="0" toLane="0" via=":2505176388_2_0" dir="l" state="M" />\n  <connection from="66984613#3" to="66984613#4" fromLane="0" toLane="0" via=":2505176388_3_0" dir="s" state="M" />\n  <connection from="66984613#3" to="243314186" fromLane="0" toLane="0" via=":2505176388_4_0" dir="r" state="M" />\n  <connection from="66984613#4" to="-243314378#3" fromLane="0" toLane="0" via=":2507400696_2_0" dir="l" state="M" />\n  <connection from="66984613#4" to="66984613#5" fromLane="0" toLane="0" via=":2507400696_3_0" dir="s" state="M" />\n  <connection from="66984613#4" to="243314213" fromLane="0" toLane="0" via=":2507400696_4_0" dir="r" state="M" />\n  <connection from="66984613#5" to="-243314112" fromLane="0" toLane="0" via=":2507400689_2_0" dir="l" state="M" />\n  <connection from="66984613#5" to="66984613#6" fromLane="0" toLane="0" via=":2507400689_3_0" dir="s" state="M" />\n  <connection from="66984613#5" to="243314073" fromLane="0" toLane="0" via=":2507400689_4_0" dir="r" state="M" />\n  <connection from="66984613#6" to="243079738" fromLane="0" toLane="0" via=":947896958_0_0" dir="l" state="M" />\n  <connection from="66984613#6" to="66984613#7" fromLane="0" toLane="0" via=":947896958_1_0" dir="s" state="M" />\n  <connection from="66984613#7" to="706105952#0" fromLane="0" toLane="0" via=":2505176265_0_0" dir="s" state="M" />\n  <connection from="66984613#7" to="-1218747879" fromLane="0" toLane="0" via=":2505176265_1_0" dir="r" state="M" />\n  <connection from="684704215#0" to="684704215#1" fromLane="0" toLane="0" via=":617234535_4_0" dir="s" state="m" />\n  <connection from="684704215#0" to="40264696#4" fromLane="0" toLane="1" via=":617234535_5_0" dir="r" state="m" />\n  <connection from="684704215#0" to="40264696#4" fromLane="0" toLane="2" via=":617234535_5_1" dir="r" state="m" />\n  <connection from="684714761#0" to="684714761#1" fromLane="0" toLane="0" via=":856214578_4_0" dir="s" state="m" />\n  <connection from="684714761#0" to="84842974#10" fromLane="0" toLane="1" via=":856214578_5_0" dir="r" state="m" />\n  <connection from="684714761#0" to="84842974#10" fromLane="0" toLane="2" via=":856214578_5_1" dir="r" state="m" />\n  <connection from="684714761#1" to="684714761#2" fromLane="0" toLane="0" via=":4900465513_1_0" dir="s" state="M" />\n  <connection from="684714761#2" to="911123067" fromLane="0" toLane="0" via=":2506085849_2_0" dir="l" state="M" />\n  <connection from="684714761#2" to="684714761#3" fromLane="0" toLane="0" via=":2506085849_3_0" dir="s" state="M" />\n  <connection from="684714761#2" to="-243188447" fromLane="0" toLane="0" via=":2506085849_4_0" dir="r" state="M" />\n  <connection from="684714761#3" to="706093504#1" fromLane="0" toLane="0" via=":1107784879_2_0" dir="l" state="M" />\n  <connection from="684714761#3" to="706093505#0" fromLane="0" toLane="0" via=":1107784879_3_0" dir="s" state="M" />\n  <connection from="684740716" to="-330615844" fromLane="0" toLane="0" via=":cluster_1123399715_3376084911_3376091818_1_0" dir="l" state="M" />\n  <connection from="684740716" to="243188670#0" fromLane="0" toLane="1" via=":cluster_1123399715_3376084911_3376091818_2_0" dir="r" state="m" />\n  <connection from="684741141#0" to="243188161" fromLane="0" toLane="0" via=":1123399675_2_0" dir="l" state="M" />\n  <connection from="684741141#0" to="684741141#1" fromLane="0" toLane="0" via=":1123399675_3_0" dir="s" state="M" />\n  <connection from="684741141#1" to="243188361" fromLane="0" toLane="0" via=":1123399943_2_0" dir="l" state="M" />\n  <connection from="684741141#1" to="684741141#2" fromLane="0" toLane="0" via=":1123399943_3_0" dir="s" state="M" />\n  <connection from="684741141#2" to="243188085" fromLane="0" toLane="0" via=":1123399486_2_0" dir="l" state="M" />\n  <connection from="684741141#2" to="684741141#3" fromLane="0" toLane="0" via=":1123399486_3_0" dir="s" state="M" />\n  <connection from="684741141#3" to="243188786#1" fromLane="0" toLane="0" via=":1123399502_7_0" dir="l" state="m" />\n  <connection from="684741141#3" to="-243188786#0" fromLane="0" toLane="0" via=":1123399502_8_0" dir="r" state="m" />\n  <connection from="705855017#1" to="129842388#1" fromLane="0" toLane="0" via=":1431867108_2_0" dir="l" state="m" />\n  <connection from="705855017#1" to="1457057267" fromLane="0" toLane="0" via=":1431867108_3_0" dir="s" state="m" />\n  <connection from="705855017#1" to="-129842388#0" fromLane="0" toLane="0" via=":1431867108_4_0" dir="r" state="m" />\n  <connection from="706087699#1" to="129842397#1" fromLane="0" toLane="0" via=":1431867103_0_0" dir="l" state="m" />\n  <connection from="706087699#1" to="-129842397#0" fromLane="0" toLane="0" via=":1431867103_1_0" dir="r" state="m" />\n  <connection from="706092523#0" to="706092523#2" fromLane="0" toLane="0" via=":856214470_0_0" dir="s" state="M" />\n  <connection from="706092523#2" to="-931354569#0" fromLane="0" toLane="0" via=":856214503_2_0" dir="l" state="m" />\n  <connection from="706092523#2" to="706092523#4" fromLane="0" toLane="0" via=":856214503_3_0" dir="s" state="m" />\n  <connection from="706092523#2" to="931354569#1" fromLane="0" toLane="0" via=":856214503_4_0" dir="r" state="m" />\n  <connection from="706092523#4" to="-706520617#3" fromLane="0" toLane="0" via=":2052489649_1_0" dir="l" state="m" />\n  <connection from="706092523#4" to="706520617#4" fromLane="0" toLane="0" via=":2052489649_2_0" dir="s" state="M" />\n  <connection from="706093504#0" to="706093504#1" fromLane="0" toLane="0" via=":1107784879_0_0" dir="s" state="m" />\n  <connection from="706093504#0" to="706093505#0" fromLane="0" toLane="0" via=":1107784879_1_0" dir="r" state="m" />\n  <connection from="706093504#1" to="706093504#2" fromLane="0" toLane="0" via=":12344163778_0_0" dir="s" state="M" />\n  <connection from="706093504#1" to="1334274432" fromLane="0" toLane="0" via=":12344163778_1_0" dir="r" state="M" />\n  <connection from="706093504#2" to="706104219#15" fromLane="0" toLane="0" via=":1107784887_0_0" dir="l" state="m" />\n  <connection from="706093504#2" to="706093506#0" fromLane="0" toLane="0" via=":1107784887_1_0" dir="s" state="m" />\n  <connection from="706093504#2" to="-706104219#14" fromLane="0" toLane="0" via=":1107784887_2_0" dir="r" state="m" />\n  <connection from="706093505#0" to="706093505#1" fromLane="0" toLane="0" via=":4900465099_1_0" dir="s" state="M" />\n  <connection from="706093505#0" to="-498671442" fromLane="0" toLane="0" via=":4900465099_2_0" dir="r" state="M" />\n  <connection from="706093505#1" to="706093505#2" fromLane="0" toLane="0" via=":8523334139_0_0" dir="s" state="M" />\n  <connection from="706093505#1" to="917696883" fromLane="0" toLane="0" via=":8523334139_1_0" dir="r" state="M" />\n  <connection from="706093505#2" to="706092523#2" fromLane="0" toLane="0" via=":856214470_1_0" dir="r" state="m" />\n  <connection from="706093506#0" to="194743656#0" fromLane="0" toLane="0" via=":2052497441_0_0" dir="l" state="M" />\n  <connection from="706093506#0" to="706093506#1" fromLane="0" toLane="0" via=":2052497441_1_0" dir="s" state="M" />\n  <connection from="706093506#1" to="706093506#2" fromLane="0" toLane="0" via=":4900465364_0_0" dir="s" state="M" />\n  <connection from="706093506#2" to="706093506#3" fromLane="0" toLane="0" via=":2506087810_0_0" dir="s" state="M" />\n  <connection from="706093506#2" to="243188527" fromLane="0" toLane="0" via=":2506087810_1_0" dir="r" state="M" />\n  <connection from="706093506#3" to="706104217#19" fromLane="0" toLane="0" via=":947896956_0_0" dir="l" state="M" />\n  <connection from="706093506#3" to="706095305" fromLane="0" toLane="0" via=":947896956_1_0" dir="s" state="M" />\n  <connection from="706093506#3" to="-706104217#18" fromLane="0" toLane="0" via=":947896956_2_0" dir="r" state="m" />\n  <connection from="706093508#0" to="706093508#1" fromLane="0" toLane="0" via=":8523344319_0_0" dir="s" state="M" />\n  <connection from="706093508#0" to="498671586" fromLane="0" toLane="0" via=":8523344319_1_0" dir="r" state="M" />\n  <connection from="706093508#1" to="706093508#2" fromLane="0" toLane="0" via=":617234843_1_0" dir="s" state="M" />\n  <connection from="706093508#2" to="84842974#11" fromLane="0" toLane="0" via=":485580842_0_0" dir="l" state="m" />\n  <connection from="706093508#2" to="706093508#3" fromLane="0" toLane="0" via=":485580842_1_0" dir="s" state="m" />\n  <connection from="706093509#0" to="706093509#1" fromLane="0" toLane="0" via=":4900465471_0_0" dir="s" state="M" />\n  <connection from="706093509#0" to="498671576#0" fromLane="0" toLane="0" via=":4900465471_1_0" dir="r" state="M" />\n  <connection from="706093509#1" to="706093508#0" fromLane="0" toLane="0" via=":1966386254_0_0" dir="l" state="m" />\n  <connection from="706093509#1" to="706093504#0" fromLane="0" toLane="0" via=":1966386254_1_0" dir="s" state="m" />\n  <connection from="706093509#1" to="-931354569#4" fromLane="0" toLane="0" via=":1966386254_2_0" dir="r" state="m" />\n  <connection from="706094740#0" to="155690513" fromLane="0" toLane="0" via=":cluster_1558314142_1558314144_6_0" dir="l" state="M" />\n  <connection from="706094740#0" to="498671497" fromLane="0" toLane="0" via=":cluster_1558314142_1558314144_7_0" dir="s" state="M" />\n  <connection from="706094740#0" to="706094740#2" fromLane="0" toLane="0" via=":cluster_1558314142_1558314144_8_0" dir="r" state="m" />\n  <connection from="706094740#2" to="243188607#0" fromLane="0" toLane="0" via=":2506085508_2_0" dir="l" state="M" />\n  <connection from="706094740#2" to="-706094742" fromLane="0" toLane="0" via=":2506085508_3_0" dir="s" state="M" />\n  <connection from="706094741#0" to="-243188212#0" fromLane="0" toLane="0" via=":1123399641_0_0" dir="l" state="m" />\n  <connection from="706094741#0" to="706094741#1" fromLane="0" toLane="0" via=":1123399641_1_0" dir="s" state="m" />\n  <connection from="706094741#0" to="243188212#1" fromLane="0" toLane="0" via=":1123399641_2_0" dir="r" state="m" />\n  <connection from="706094741#1" to="706094741#2" fromLane="0" toLane="0" via=":1123399483_0_0" dir="s" state="M" />\n  <connection from="706094741#1" to="-243188565" fromLane="0" toLane="0" via=":1123399483_1_0" dir="r" state="m" />\n  <connection from="706094741#2" to="706094741#3" fromLane="0" toLane="0" via=":1123399659_0_0" dir="s" state="m" />\n  <connection from="706094741#3" to="684740716" fromLane="0" toLane="0" via=":1123399489_0_0" dir="s" state="m" />\n  <connection from="706094742" to="-706094740#2" fromLane="0" toLane="0" via=":2506085508_4_0" dir="s" state="M" />\n  <connection from="706094742" to="243188607#0" fromLane="0" toLane="0" via=":2506085508_5_0" dir="r" state="m" />\n  <connection from="706095305" to="-1558734968" fromLane="0" toLane="0" via=":2506088476_0_0" dir="l" state="m" />\n  <connection from="706095305" to="-498671331" fromLane="0" toLane="0" via=":2506088476_1_0" dir="r" state="m" />\n  <connection from="706095306" to="-243188174#2" fromLane="0" toLane="0" via=":2506087732_0_0" dir="r" state="M" />\n  <connection from="706095308#0" to="243188174#1" fromLane="0" toLane="0" via=":6634630596_3_0" dir="l" state="m" />\n  <connection from="706095308#0" to="706095308#1" fromLane="0" toLane="0" via=":6634630596_4_0" dir="s" state="m" />\n  <connection from="706095308#0" to="-243188174#0" fromLane="0" toLane="0" via=":6634630596_5_0" dir="r" state="m" />\n  <connection from="706095308#1" to="84842974#1" fromLane="0" toLane="0" via=":cluster_2506087462_8523307708_2_0" dir="s" state="m" />\n  <connection from="706095308#1" to="84842974#1" fromLane="0" toLane="1" via=":cluster_2506087462_8523307708_2_1" dir="s" state="m" />\n  <connection from="706095308#1" to="84842974#1" fromLane="0" toLane="2" via=":cluster_2506087462_8523307708_2_2" dir="s" state="m" />\n  <connection from="706096637" to="40264696#2" fromLane="0" toLane="0" via=":1196113662_4_0" dir="l" state="m" />\n  <connection from="706096637" to="101706795#0" fromLane="0" toLane="0" via=":1196113662_5_0" dir="s" state="m" />\n  <connection from="706096640#0" to="706256377#4" fromLane="0" toLane="0" via=":2052547074_0_0" tl="2052547074" linkIndex="0" dir="l" state="o" />\n  <connection from="706096640#0" to="706096640#1" fromLane="0" toLane="0" via=":2052547074_1_0" tl="2052547074" linkIndex="1" dir="s" state="o" />\n  <connection from="706096640#0" to="-706256377#3" fromLane="0" toLane="0" via=":2052547074_2_0" tl="2052547074" linkIndex="2" dir="r" state="o" />\n  <connection from="706096640#1" to="706097788#0" fromLane="0" toLane="0" via=":617234281_0_0" dir="s" state="m" />\n  <connection from="706096640#1" to="40264697#4" fromLane="0" toLane="1" via=":617234281_1_0" dir="r" state="m" />\n  <connection from="706096642" to="-706104219#16" fromLane="0" toLane="0" via=":4085816302_5_0" tl="4085816302" linkIndex="5" dir="s" state="o" />\n  <connection from="706096642" to="84842974#9" fromLane="0" toLane="2" via=":4085816302_6_0" tl="4085816302" linkIndex="6" dir="r" state="o" />\n  <connection from="706096643#0" to="-129827562" fromLane="0" toLane="0" via=":1431715421_1_0" dir="l" state="M" />\n  <connection from="706096643#0" to="706096643#1" fromLane="0" toLane="0" via=":1431715421_2_0" dir="s" state="m" />\n  <connection from="706096644#0" to="706096640#0" fromLane="0" toLane="0" via=":1122072681_0_0" dir="s" state="M" />\n  <connection from="706096644#0" to="-706256379#6" fromLane="0" toLane="0" via=":1122072681_1_0" dir="r" state="M" />\n  <connection from="706097788#0" to="706097788#1" fromLane="0" toLane="0" via=":14069981565_0_0" dir="s" state="M" />\n  <connection from="706097788#0" to="1546896992" fromLane="0" toLane="0" via=":14069981565_1_0" dir="r" state="M" />\n  <connection from="706097788#1" to="142456544" fromLane="0" toLane="0" via=":1559029798_0_0" dir="l" state="M" />\n  <connection from="706097788#1" to="706097788#2" fromLane="0" toLane="0" via=":1559029798_1_0" dir="s" state="M" />\n  <connection from="706097788#2" to="706093508#2" fromLane="0" toLane="0" via=":617234843_0_0" dir="l" state="m" />\n  <connection from="706102885" to="243079656" fromLane="0" toLane="0" via=":2506089779_2_0" dir="l" state="m" />\n  <connection from="706102885" to="243188298" fromLane="0" toLane="0" via=":2506089779_3_0" dir="r" state="m" />\n  <connection from="706103887" to="95809261#1" fromLane="0" toLane="0" via=":2506087889_0_0" dir="l" state="M" />\n  <connection from="706104217#0" to="917705248" fromLane="0" toLane="0" via=":8523431062_2_0" dir="l" state="M" />\n  <connection from="706104217#0" to="706104217#2" fromLane="0" toLane="0" via=":8523431062_3_0" dir="s" state="M" />\n  <connection from="706104217#11" to="706104217#12" fromLane="0" toLane="0" via=":12339345632_0_0" dir="s" state="M" />\n  <connection from="706104217#11" to="498671702" fromLane="0" toLane="0" via=":12339345632_1_0" dir="r" state="m" />\n  <connection from="706104217#12" to="706104217#13" fromLane="0" toLane="0" via=":2506088084_2_0" dir="s" state="m" />\n  <connection from="706104217#13" to="98567749" fromLane="0" toLane="0" via=":1140280758_0_0" dir="l" state="M" />\n  <connection from="706104217#13" to="706104217#14" fromLane="0" toLane="0" via=":1140280758_1_0" dir="s" state="M" />\n  <connection from="706104217#14" to="706104217#16" fromLane="0" toLane="0" via=":cluster_2506088039_8523406713_3_0" dir="s" state="M" />\n  <connection from="706104217#14" to="917704692" fromLane="0" toLane="0" via=":cluster_2506088039_8523406713_4_0" dir="r" state="m" />\n  <connection from="706104217#16" to="95809268#0" fromLane="0" toLane="0" via=":1110416470_0_0" dir="l" state="m" />\n  <connection from="706104217#16" to="706104217#17" fromLane="0" toLane="0" via=":1110416470_1_0" dir="s" state="m" />\n  <connection from="706104217#17" to="706104217#18" fromLane="0" toLane="0" via=":4900465517_0_0" dir="s" state="M" />\n  <connection from="706104217#17" to="498671589" fromLane="0" toLane="0" via=":4900465517_1_0" dir="r" state="m" />\n  <connection from="706104217#18" to="706104217#19" fromLane="0" toLane="0" via=":947896956_3_0" dir="s" state="m" />\n  <connection from="706104217#18" to="706095305" fromLane="0" toLane="0" via=":947896956_4_0" dir="r" state="m" />\n  <connection from="706104217#19" to="706095306" fromLane="0" toLane="0" via=":2506087987_3_0" dir="l" state="M" />\n  <connection from="706104217#19" to="706104217#20" fromLane="0" toLane="0" via=":2506087987_4_0" dir="s" state="M" />\n  <connection from="706104217#19" to="-243188576" fromLane="0" toLane="0" via=":2506087987_5_0" dir="r" state="m" />\n  <connection from="706104217#2" to="917705249" fromLane="0" toLane="0" via=":8523431064_0_0" dir="l" state="M" />\n  <connection from="706104217#2" to="706104217#3" fromLane="0" toLane="0" via=":8523431064_1_0" dir="s" state="M" />\n  <connection from="706104217#20" to="706095308#0" fromLane="0" toLane="0" via=":6634630594_0_0" dir="l" state="M" />\n  <connection from="706104217#20" to="706104217#21" fromLane="0" toLane="0" via=":6634630594_1_0" dir="s" state="M" />\n  <connection from="706104217#21" to="77967090#3" fromLane="0" toLane="0" via=":cluster_916327730_916327962_985092239_985092700_0_0" tl="cluster_916327730_916327962_985092239_985092700" linkIndex="0" dir="l" state="o" />\n  <connection from="706104217#21" to="77967084#2" fromLane="0" toLane="0" via=":cluster_916327730_916327962_985092239_985092700_1_0" tl="cluster_916327730_916327962_985092239_985092700" linkIndex="1" dir="r" state="o" />\n  <connection from="706104217#3" to="-243188208" fromLane="0" toLane="0" via=":2506087962_0_0" dir="l" state="M" />\n  <connection from="706104217#3" to="706104217#4" fromLane="0" toLane="0" via=":2506087962_1_0" dir="s" state="M" />\n  <connection from="706104217#4" to="243188432#0" fromLane="0" toLane="0" via=":947896950_0_0" dir="l" state="M" />\n  <connection from="706104217#4" to="706104217#8" fromLane="0" toLane="0" via=":947896950_1_0" dir="s" state="M" />\n  <connection from="706104217#8" to="-243188563" fromLane="0" toLane="0" via=":2506088002_0_0" dir="l" state="M" />\n  <connection from="706104217#8" to="706104217#9" fromLane="0" toLane="0" via=":2506088002_1_0" dir="s" state="m" />\n  <connection from="706104217#9" to="243188491" fromLane="0" toLane="0" via=":2506088024_0_0" dir="l" state="m" />\n  <connection from="706104217#9" to="706104217#11" fromLane="0" toLane="0" via=":2506088024_1_0" dir="s" state="m" />\n  <connection from="706104217#9" to="-330615760" fromLane="0" toLane="0" via=":2506088024_2_0" dir="r" state="m" />\n  <connection from="706104218" to="-530314558#1" fromLane="0" toLane="0" via=":1123399639_2_0" dir="l" state="M" />\n  <connection from="706104218" to="706104219#0" fromLane="0" toLane="0" via=":1123399639_3_0" dir="s" state="M" />\n  <connection from="706104219#0" to="96242763#0" fromLane="0" toLane="0" via=":cluster_1115585393_13472440789_947896971_3_0" tl="cluster_1115585393_13472440789_947896971" linkIndex="3" dir="l" state="O" />\n  <connection from="706104219#0" to="706104219#3" fromLane="0" toLane="0" via=":cluster_1115585393_13472440789_947896971_4_0" tl="cluster_1115585393_13472440789_947896971" linkIndex="4" dir="s" state="O" />\n  <connection from="706104219#0" to="81350005#0" fromLane="0" toLane="0" via=":cluster_1115585393_13472440789_947896971_5_0" tl="cluster_1115585393_13472440789_947896971" linkIndex="5" dir="r" state="o" />\n  <connection from="706104219#11" to="498671544#0" fromLane="0" toLane="0" via=":1140280757_2_0" dir="l" state="M" />\n  <connection from="706104219#11" to="706104219#12" fromLane="0" toLane="0" via=":1140280757_3_0" dir="s" state="M" />\n  <connection from="706104219#12" to="706092523#0" fromLane="0" toLane="0" via=":1110416444_0_0" dir="l" state="M" />\n  <connection from="706104219#12" to="706104219#13" fromLane="0" toLane="0" via=":1110416444_1_0" dir="s" state="M" />\n  <connection from="706104219#12" to="1365892748#0" fromLane="0" toLane="0" via=":1110416444_2_0" dir="r" state="m" />\n  <connection from="706104219#13" to="706104219#14" fromLane="0" toLane="0" via=":1110416654_0_0" dir="s" state="M" />\n  <connection from="706104219#14" to="706104219#15" fromLane="0" toLane="0" via=":1107784887_3_0" dir="s" state="M" />\n  <connection from="706104219#14" to="706093506#0" fromLane="0" toLane="0" via=":1107784887_4_0" dir="r" state="m" />\n  <connection from="706104219#15" to="706104219#16" fromLane="0" toLane="0" via=":2506086181_2_0" dir="s" state="M" />\n  <connection from="706104219#16" to="84842974#9" fromLane="0" toLane="0" via=":4085816302_0_0" tl="4085816302" linkIndex="0" dir="l" state="o" />\n  <connection from="706104219#3" to="498671389#0" fromLane="0" toLane="0" via=":4900464597_2_0" dir="l" state="M" />\n  <connection from="706104219#3" to="706104219#5" fromLane="0" toLane="0" via=":4900464597_3_0" dir="s" state="M" />\n  <connection from="706104219#5" to="498671388#0" fromLane="0" toLane="0" via=":2506086831_2_0" dir="l" state="M" />\n  <connection from="706104219#5" to="706104219#7" fromLane="0" toLane="0" via=":2506086831_3_0" dir="s" state="M" />\n  <connection from="706104219#7" to="706104219#8" fromLane="0" toLane="0" via=":2506086795_0_0" dir="s" state="M" />\n  <connection from="706104219#7" to="243188563" fromLane="0" toLane="0" via=":2506086795_1_0" dir="r" state="m" />\n  <connection from="706104219#8" to="-96941842#3" fromLane="0" toLane="0" via=":1966399248_3_0" dir="l" state="M" />\n  <connection from="706104219#8" to="706104219#9" fromLane="0" toLane="0" via=":1966399248_4_0" dir="s" state="M" />\n  <connection from="706104219#8" to="-243188491" fromLane="0" toLane="0" via=":1966399248_5_0" dir="r" state="m" />\n  <connection from="706104219#9" to="917696759" fromLane="0" toLane="0" via=":cluster_1966399241_8523333972_3_0" dir="l" state="M" />\n  <connection from="706104219#9" to="706104219#11" fromLane="0" toLane="0" via=":cluster_1966399241_8523333972_4_0" dir="s" state="M" />\n  <connection from="706104219#9" to="96713729" fromLane="0" toLane="0" via=":cluster_1966399241_8523333972_5_0" dir="r" state="m" />\n  <connection from="706105950" to="969757851#4" fromLane="0" toLane="0" via=":8976429009_0_0" dir="l" state="m" />\n  <connection from="706105951#0" to="900244003#0" fromLane="0" toLane="0" via=":1107926513_4_0" dir="l" state="M" />\n  <connection from="706105951#0" to="706105951#1" fromLane="0" toLane="0" via=":1107926513_5_0" dir="s" state="M" />\n  <connection from="706105951#1" to="-1270957863" fromLane="0" toLane="0" via=":856214524_3_0" dir="l" state="m" />\n  <connection from="706105951#1" to="706105951#3" fromLane="0" toLane="0" via=":856214524_4_0" dir="s" state="m" />\n  <connection from="706105951#1" to="185951074#0" fromLane="0" toLane="0" via=":856214524_5_0" dir="r" state="m" />\n  <connection from="706105951#3" to="243188629" fromLane="0" toLane="0" via=":1107926639_6_0" dir="l" state="m" />\n  <connection from="706105951#3" to="706105951#4" fromLane="0" toLane="0" via=":1107926639_7_0" dir="s" state="m" />\n  <connection from="706105951#3" to="-243188351#1" fromLane="0" toLane="0" via=":1107926639_8_0" dir="r" state="m" />\n  <connection from="706105951#4" to="706105950" fromLane="0" toLane="0" via=":1107926537_2_0" dir="l" state="m" />\n  <connection from="706105951#4" to="243188741#0" fromLane="0" toLane="0" via=":1107926537_3_0" dir="r" state="m" />\n  <connection from="706105952#0" to="243079844#1" fromLane="0" toLane="0" via=":809537690_0_0" dir="l" state="M" />\n  <connection from="706105952#0" to="706105952#1" fromLane="0" toLane="0" via=":809537690_1_0" dir="s" state="M" />\n  <connection from="706105952#1" to="151072453#0" fromLane="0" toLane="0" via=":809537820_2_0" tl="joinedS_809537820_856214571" linkIndex="2" dir="s" state="o" />\n  <connection from="706105952#1" to="66984604#2" fromLane="0" toLane="0" via=":809537820_3_0" tl="joinedS_809537820_856214571" linkIndex="3" dir="r" state="o" />\n  <connection from="706256377#0" to="154707523#1" fromLane="0" toLane="0" via=":2052547075_0_0" dir="l" state="M" />\n  <connection from="706256377#0" to="706256377#1" fromLane="0" toLane="0" via=":2052547075_1_0" dir="s" state="M" />\n  <connection from="706256377#1" to="706256377#2" fromLane="0" toLane="0" via=":1107784883_3_0" dir="s" state="M" />\n  <connection from="706256377#1" to="95533433#1" fromLane="0" toLane="0" via=":1107784883_4_0" dir="r" state="m" />\n  <connection from="706256377#2" to="-498671482" fromLane="0" toLane="0" via=":4900465209_2_0" dir="l" state="M" />\n  <connection from="706256377#2" to="706256377#3" fromLane="0" toLane="0" via=":4900465209_3_0" dir="s" state="M" />\n  <connection from="706256377#3" to="706256377#4" fromLane="0" toLane="0" via=":2052547074_3_0" tl="2052547074" linkIndex="3" dir="s" state="O" />\n  <connection from="706256377#3" to="706096640#1" fromLane="0" toLane="0" via=":2052547074_4_0" tl="2052547074" linkIndex="4" dir="r" state="o" />\n  <connection from="706256377#4" to="84842974#14" fromLane="0" toLane="0" via=":1176926303_0_0" dir="l" state="m" />\n  <connection from="706256379#1" to="96837170#0" fromLane="0" toLane="0" via=":cluster_1122072658_14138648129_0_0" dir="l" state="M" />\n  <connection from="706256379#1" to="706256379#3" fromLane="0" toLane="0" via=":cluster_1122072658_14138648129_1_0" dir="s" state="M" />\n  <connection from="706256379#3" to="706256379#4" fromLane="0" toLane="0" via=":1107784878_0_0" dir="s" state="M" />\n  <connection from="706256379#3" to="95533433#0" fromLane="0" toLane="0" via=":1107784878_1_0" dir="r" state="m" />\n  <connection from="706256379#4" to="706256379#5" fromLane="0" toLane="0" via=":4900465208_0_0" dir="s" state="M" />\n  <connection from="706256379#4" to="498671482" fromLane="0" toLane="0" via=":4900465208_1_0" dir="r" state="m" />\n  <connection from="706256379#5" to="-706096644#1" fromLane="0" toLane="0" via=":1122072681_2_0" dir="l" state="M" />\n  <connection from="706256379#5" to="706096640#0" fromLane="0" toLane="0" via=":1122072681_3_0" dir="r" state="m" />\n  <connection from="706256380#0" to="706256380#1" fromLane="0" toLane="0" via=":1680666322_0_0" dir="s" state="M" />\n  <connection from="706256380#0" to="155690968" fromLane="0" toLane="0" via=":1680666322_1_0" dir="r" state="m" />\n  <connection from="706256380#1" to="95809262#0" fromLane="0" toLane="0" via=":cluster_1110416379_1680666461_0_0" dir="l" state="M" />\n  <connection from="706256380#1" to="706256380#3" fromLane="0" toLane="0" via=":cluster_1110416379_1680666461_1_0" dir="s" state="M" />\n  <connection from="706256380#1" to="155690981" fromLane="0" toLane="0" via=":cluster_1110416379_1680666461_2_0" dir="r" state="m" />\n  <connection from="706256380#3" to="-498671483" fromLane="0" toLane="0" via=":1680666686_0_0" dir="l" state="M" />\n  <connection from="706256380#3" to="706256380#4" fromLane="0" toLane="0" via=":1680666686_1_0" dir="s" state="M" />\n  <connection from="706256380#3" to="155690340" fromLane="0" toLane="0" via=":1680666686_2_0" dir="r" state="m" />\n  <connection from="706256380#4" to="-706256386" fromLane="0" toLane="0" via=":cluster_1559016919_1680666950_0_0" dir="l" state="M" />\n  <connection from="706256380#4" to="706256380#6" fromLane="0" toLane="0" via=":cluster_1559016919_1680666950_1_0" dir="s" state="M" />\n  <connection from="706256380#4" to="155690878#0" fromLane="0" toLane="0" via=":cluster_1559016919_1680666950_2_0" dir="r" state="m" />\n  <connection from="706256380#6" to="1454593203#2" fromLane="0" toLane="0" via=":13339819888_0_0" dir="l" state="m" visibility="9.00" />\n  <connection from="706256381" to="1454593203#4" fromLane="0" toLane="0" via=":13339819893_0_0" dir="l" state="m" visibility="9.00" />\n  <connection from="706256383" to="-1454593202#2" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_6_0" dir="l" state="m" />\n  <connection from="706256383" to="-148941242#5" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_7_0" dir="L" state="m" />\n  <connection from="706256383" to="-129827554#4" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_8_0" dir="s" state="m" />\n  <connection from="706256383" to="1374050353" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_9_0" dir="r" state="m" />\n  <connection from="706256385" to="71380773#1" fromLane="0" toLane="0" via=":1431715363_1_0" dir="r" state="m" />\n  <connection from="706256386" to="706256380#6" fromLane="0" toLane="0" via=":cluster_1559016919_1680666950_9_0" dir="l" state="m" />\n  <connection from="706256386" to="155690878#0" fromLane="0" toLane="0" via=":cluster_1559016919_1680666950_10_0" dir="s" state="m" />\n  <connection from="706256386" to="-706256380#4" fromLane="0" toLane="0" via=":cluster_1559016919_1680666950_11_0" dir="r" state="m" />\n  <connection from="706520617#0" to="706520617#1" fromLane="0" toLane="0" via=":12649445299_0_0" dir="s" state="M" />\n  <connection from="706520617#0" to="-1365935945#1" fromLane="0" toLane="0" via=":12649445299_1_0" dir="r" state="m" />\n  <connection from="706520617#1" to="706520617#2" fromLane="0" toLane="0" via=":1123399562_0_0" dir="s" state="m" />\n  <connection from="706520617#1" to="96941842#0" fromLane="0" toLane="0" via=":1123399562_1_0" dir="r" state="m" />\n  <connection from="706520617#2" to="706520617#3" fromLane="0" toLane="0" via=":1115585792_0_0" dir="s" state="M" />\n  <connection from="706520617#2" to="931354570" fromLane="0" toLane="0" via=":1115585792_1_0" dir="r" state="m" />\n  <connection from="706520617#3" to="706520617#4" fromLane="0" toLane="0" via=":2052489649_0_0" dir="s" state="m" />\n  <connection from="706520617#4" to="243079805" fromLane="0" toLane="0" via=":1110416568_0_0" dir="l" state="m" />\n  <connection from="706520617#4" to="1365892991" fromLane="0" toLane="0" via=":1110416568_1_0" dir="s" state="m" />\n  <connection from="71380773#0" to="71380773#1" fromLane="0" toLane="0" via=":1431715363_0_0" dir="s" state="M" />\n  <connection from="71380773#1" to="969757851#0" fromLane="0" toLane="0" via=":1160612974_0_0" dir="s" state="M" />\n  <connection from="71380773#1" to="100401250#0" fromLane="0" toLane="0" via=":1160612974_1_0" dir="r" state="M" />\n  <connection from="72016745#0" to="72016745#2" fromLane="0" toLane="0" via=":1369020950_0_0" dir="s" state="M" />\n  <connection from="72016745#0" to="72016745#2" fromLane="1" toLane="1" via=":1369020950_0_1" dir="s" state="M" />\n  <connection from="72016745#0" to="122540020#0" fromLane="1" toLane="0" via=":1369020950_2_0" dir="r" state="m" />\n  <connection from="72016745#2" to="706105951#0" fromLane="0" toLane="0" via=":cluster_13472440775_1666900747_3_0" dir="l" state="M" />\n  <connection from="72016745#2" to="72016745#4" fromLane="0" toLane="0" via=":cluster_13472440775_1666900747_4_0" dir="s" state="M" />\n  <connection from="72016745#2" to="72016745#4" fromLane="1" toLane="1" via=":cluster_13472440775_1666900747_4_1" dir="s" state="M" />\n  <connection from="72016745#2" to="-95542485#3" fromLane="1" toLane="0" via=":cluster_13472440775_1666900747_6_0" dir="r" state="m" />\n  <connection from="72016745#4" to="-498671569" fromLane="0" toLane="0" via=":2506087136_3_0" dir="l" state="M" />\n  <connection from="72016745#4" to="72016745#5" fromLane="0" toLane="0" via=":2506087136_4_0" dir="s" state="M" />\n  <connection from="72016745#4" to="72016745#5" fromLane="1" toLane="1" via=":2506087136_4_1" dir="s" state="M" />\n  <connection from="72016745#4" to="243188585" fromLane="1" toLane="0" via=":2506087136_6_0" dir="r" state="m" />\n  <connection from="72016745#5" to="-96941849" fromLane="0" toLane="0" via=":5149825984_3_0" dir="l" state="M" />\n  <connection from="72016745#5" to="706104218" fromLane="0" toLane="0" via=":5149825984_4_0" dir="s" state="M" />\n  <connection from="72016745#5" to="452036923#0" fromLane="1" toLane="0" via=":5149825984_5_0" dir="r" state="m" />\n  <connection from="77967082#0" to="77967084#2" fromLane="0" toLane="0" via=":cluster_916327730_916327962_985092239_985092700_6_0" tl="cluster_916327730_916327962_985092239_985092700" linkIndex="6" dir="l" state="O" />\n  <connection from="77967082#0" to="-706104217#21" fromLane="0" toLane="0" via=":cluster_916327730_916327962_985092239_985092700_7_0" tl="cluster_916327730_916327962_985092239_985092700" linkIndex="7" dir="s" state="o" />\n  <connection from="77967082#0" to="77967090#3" fromLane="1" toLane="2" via=":cluster_916327730_916327962_985092239_985092700_8_0" tl="cluster_916327730_916327962_985092239_985092700" linkIndex="8" dir="r" state="o" />\n  <connection from="77967084#2" to="77967092" fromLane="0" toLane="0" via=":916327810_0_0" dir="l" state="M" />\n  <connection from="77967084#2" to="77967092" fromLane="0" toLane="1" via=":916327810_0_1" dir="l" state="M" />\n  <connection from="77967084#2" to="77967084#3" fromLane="0" toLane="0" via=":916327810_2_0" dir="s" state="M" />\n  <connection from="77967084#3" to="77967084#4" fromLane="0" toLane="0" via=":916327702_1_0" dir="s" state="M" />\n  <connection from="77967084#4" to="498671329" fromLane="0" toLane="0" via=":4900466295_0_0" dir="l" state="M" />\n  <connection from="77967084#4" to="77967084#5" fromLane="0" toLane="0" via=":4900466295_1_0" dir="s" state="M" />\n  <connection from="77967084#5" to="931348308" fromLane="0" toLane="0" via=":4900466297_1_0" dir="s" state="M" />\n  <connection from="77967085" to="77967084#4" fromLane="0" toLane="0" via=":916327702_0_0" dir="l" state="m" />\n  <connection from="77967090#0" to="-1552748564#1" fromLane="0" toLane="0" via=":916327912_1_0" dir="l" state="M" />\n  <connection from="77967090#0" to="77967090#1" fromLane="0" toLane="0" via=":916327912_2_0" dir="s" state="M" />\n  <connection from="77967090#0" to="77967090#1" fromLane="1" toLane="1" via=":916327912_2_1" dir="s" state="M" />\n  <connection from="77967090#0" to="77967090#1" fromLane="2" toLane="2" via=":916327912_2_2" dir="s" state="M" />\n  <connection from="77967090#1" to="-706104217#21" fromLane="0" toLane="0" via=":cluster_916327730_916327962_985092239_985092700_2_0" tl="cluster_916327730_916327962_985092239_985092700" linkIndex="2" dir="l" state="O" />\n  <connection from="77967090#1" to="77967090#3" fromLane="0" toLane="0" via=":cluster_916327730_916327962_985092239_985092700_3_0" tl="cluster_916327730_916327962_985092239_985092700" linkIndex="3" dir="s" state="O" />\n  <connection from="77967090#1" to="77967090#3" fromLane="1" toLane="1" via=":cluster_916327730_916327962_985092239_985092700_3_1" tl="cluster_916327730_916327962_985092239_985092700" linkIndex="4" dir="s" state="O" />\n  <connection from="77967090#1" to="77967090#3" fromLane="2" toLane="2" via=":cluster_916327730_916327962_985092239_985092700_3_2" tl="cluster_916327730_916327962_985092239_985092700" linkIndex="5" dir="s" state="O" />\n  <connection from="77967090#3" to="243188174#0" fromLane="0" toLane="0" via=":2506087709_0_0" dir="l" state="M" />\n  <connection from="77967093#0" to="77967090#0" fromLane="0" toLane="0" via=":916327686_0_0" dir="s" state="M" />\n  <connection from="77967093#0" to="77967090#0" fromLane="0" toLane="1" via=":916327686_0_1" dir="s" state="M" />\n  <connection from="77967093#0" to="77967090#0" fromLane="0" toLane="2" via=":916327686_0_2" dir="s" state="M" />\n  <connection from="77967093#0" to="77967093#1" fromLane="0" toLane="0" via=":916327686_3_0" dir="s" state="M" />\n  <connection from="77967093#0" to="77967093#1" fromLane="1" toLane="1" via=":916327686_3_1" dir="s" state="M" />\n  <connection from="77967093#1" to="77967093#2" fromLane="0" toLane="0" via=":916327921_1_0" dir="s" state="M" />\n  <connection from="77967093#1" to="77967093#2" fromLane="1" toLane="1" via=":916327921_1_1" dir="s" state="M" />\n  <connection from="77967093#2" to="-324267161#1" fromLane="0" toLane="0" via=":916327839_0_0" dir="l" state="M" />\n  <connection from="77967093#2" to="77967093#3" fromLane="0" toLane="0" via=":916327839_1_0" dir="s" state="M" />\n  <connection from="77967093#2" to="77967093#3" fromLane="1" toLane="1" via=":916327839_1_1" dir="s" state="M" />\n  <connection from="77967093#3" to="77967093#0" fromLane="0" toLane="0" via=":916327888_1_0" dir="s" state="M" />\n  <connection from="77967093#3" to="77967093#0" fromLane="1" toLane="1" via=":916327888_1_1" dir="s" state="M" />\n  <connection from="81350005#0" to="706104217#0" fromLane="0" toLane="0" via=":947896964_0_0" tl="947896964" linkIndex="0" dir="l" state="O" />\n  <connection from="81350005#0" to="330615829#0" fromLane="0" toLane="0" via=":947896964_1_0" tl="947896964" linkIndex="1" dir="s" state="O" />\n  <connection from="81350006#1" to="-243314177" fromLane="0" toLane="0" via=":2507400773_2_0" dir="l" state="M" />\n  <connection from="81350006#1" to="81350006#2" fromLane="0" toLane="0" via=":2507400773_3_0" dir="s" state="M" />\n  <connection from="81350006#2" to="66984613#0" fromLane="0" toLane="0" via=":947896981_0_0" dir="l" state="M" />\n  <connection from="81350006#2" to="95542480" fromLane="0" toLane="0" via=":947896981_1_0" dir="s" state="M" />\n  <connection from="84842974#0" to="84842974#1" fromLane="0" toLane="0" via=":cluster_2506087462_8523307708_5_0" dir="r" state="m" />\n  <connection from="84842974#0" to="84842974#1" fromLane="1" toLane="1" via=":cluster_2506087462_8523307708_5_1" dir="r" state="m" />\n  <connection from="84842974#0" to="84842974#1" fromLane="2" toLane="2" via=":cluster_2506087462_8523307708_5_2" dir="r" state="m" />\n  <connection from="84842974#1" to="898440288" fromLane="0" toLane="0" via=":8348961008_1_0" dir="l" state="M" />\n  <connection from="84842974#1" to="84842974#2" fromLane="0" toLane="0" via=":8348961008_2_0" dir="s" state="M" />\n  <connection from="84842974#1" to="84842974#2" fromLane="1" toLane="1" via=":8348961008_2_1" dir="s" state="M" />\n  <connection from="84842974#1" to="84842974#2" fromLane="2" toLane="2" via=":8348961008_2_2" dir="s" state="M" />\n  <connection from="84842974#10" to="84842974#11" fromLane="0" toLane="0" via=":485580842_2_0" dir="s" state="M" />\n  <connection from="84842974#10" to="84842974#11" fromLane="1" toLane="1" via=":485580842_2_1" dir="s" state="M" />\n  <connection from="84842974#10" to="84842974#11" fromLane="2" toLane="2" via=":485580842_2_2" dir="s" state="M" />\n  <connection from="84842974#10" to="706093508#3" fromLane="2" toLane="0" via=":485580842_5_0" dir="r" state="M" />\n  <connection from="84842974#11" to="84842974#12" fromLane="0" toLane="0" via=":1559029797_1_0" dir="s" state="M" />\n  <connection from="84842974#11" to="84842974#12" fromLane="1" toLane="1" via=":1559029797_1_1" dir="s" state="M" />\n  <connection from="84842974#11" to="84842974#12" fromLane="2" toLane="2" via=":1559029797_1_2" dir="s" state="M" />\n  <connection from="84842974#12" to="40264697#3" fromLane="0" toLane="0" via=":617234576_0_0" dir="l" state="M" />\n  <connection from="84842974#12" to="84842974#13" fromLane="0" toLane="0" via=":617234576_1_0" dir="s" state="M" />\n  <connection from="84842974#12" to="84842974#13" fromLane="1" toLane="1" via=":617234576_1_1" dir="s" state="M" />\n  <connection from="84842974#12" to="84842974#13" fromLane="2" toLane="2" via=":617234576_1_2" dir="s" state="M" />\n  <connection from="84842974#13" to="-706256377#4" fromLane="0" toLane="0" via=":1176926303_1_0" dir="l" state="M" />\n  <connection from="84842974#13" to="84842974#14" fromLane="0" toLane="0" via=":1176926303_2_0" dir="s" state="M" />\n  <connection from="84842974#13" to="84842974#14" fromLane="1" toLane="1" via=":1176926303_2_1" dir="s" state="M" />\n  <connection from="84842974#13" to="84842974#14" fromLane="2" toLane="2" via=":1176926303_2_2" dir="s" state="M" />\n  <connection from="84842974#14" to="84842974#15" fromLane="0" toLane="0" via=":1431715369_0_0" dir="s" state="M" />\n  <connection from="84842974#14" to="84842974#15" fromLane="1" toLane="1" via=":1431715369_0_1" dir="s" state="M" />\n  <connection from="84842974#14" to="84842974#15" fromLane="2" toLane="2" via=":1431715369_0_2" dir="s" state="M" />\n  <connection from="84842974#14" to="684704215#0" fromLane="2" toLane="0" via=":1431715369_3_0" dir="r" state="M" />\n  <connection from="84842974#15" to="101706795#1" fromLane="0" toLane="0" via=":1966412828_0_0" dir="l" state="M" />\n  <connection from="84842974#15" to="84842974#17" fromLane="0" toLane="0" via=":1966412828_1_0" dir="s" state="M" />\n  <connection from="84842974#15" to="84842974#17" fromLane="1" toLane="1" via=":1966412828_1_1" dir="s" state="M" />\n  <connection from="84842974#15" to="84842974#17" fromLane="2" toLane="2" via=":1966412828_1_2" dir="s" state="M" />\n  <connection from="84842974#2" to="917697169" fromLane="0" toLane="0" via=":cluster_620881097_8523329816_1_0" dir="l" state="M" />\n  <connection from="84842974#2" to="84842974#4" fromLane="0" toLane="0" via=":cluster_620881097_8523329816_2_0" dir="s" state="M" />\n  <connection from="84842974#2" to="84842974#4" fromLane="1" toLane="1" via=":cluster_620881097_8523329816_2_1" dir="s" state="M" />\n  <connection from="84842974#2" to="84842974#4" fromLane="2" toLane="2" via=":cluster_620881097_8523329816_2_2" dir="s" state="M" />\n  <connection from="84842974#2" to="243188120" fromLane="2" toLane="0" via=":cluster_620881097_8523329816_5_0" dir="r" state="M" />\n  <connection from="84842974#4" to="84842974#7" fromLane="0" toLane="0" via=":2052522376_1_0" dir="s" state="M" />\n  <connection from="84842974#4" to="84842974#7" fromLane="1" toLane="1" via=":2052522376_1_1" dir="s" state="M" />\n  <connection from="84842974#4" to="84842974#7" fromLane="2" toLane="2" via=":2052522376_1_2" dir="s" state="M" />\n  <connection from="84842974#7" to="-706104219#16" fromLane="0" toLane="0" via=":4085816302_1_0" tl="4085816302" linkIndex="1" dir="l" state="O" />\n  <connection from="84842974#7" to="84842974#9" fromLane="0" toLane="0" via=":4085816302_2_0" tl="4085816302" linkIndex="2" dir="s" state="O" />\n  <connection from="84842974#7" to="84842974#9" fromLane="1" toLane="1" via=":4085816302_2_1" tl="4085816302" linkIndex="3" dir="s" state="O" />\n  <connection from="84842974#7" to="84842974#9" fromLane="2" toLane="2" via=":4085816302_2_2" tl="4085816302" linkIndex="4" dir="s" state="O" />\n  <connection from="84842974#9" to="684714761#1" fromLane="0" toLane="0" via=":856214578_0_0" dir="l" state="M" />\n  <connection from="84842974#9" to="84842974#10" fromLane="0" toLane="0" via=":856214578_1_0" dir="s" state="M" />\n  <connection from="84842974#9" to="84842974#10" fromLane="1" toLane="1" via=":856214578_1_1" dir="s" state="M" />\n  <connection from="84842974#9" to="84842974#10" fromLane="2" toLane="2" via=":856214578_1_2" dir="s" state="M" />\n  <connection from="898440288" to="-243188536#0" fromLane="0" toLane="0" via=":8348961009_4_0" dir="l" state="m" />\n  <connection from="898440288" to="243188536#1" fromLane="0" toLane="0" via=":8348961009_5_0" dir="r" state="m" />\n  <connection from="900244003#0" to="900244003#1" fromLane="0" toLane="0" via=":8363308288_4_0" dir="s" state="M" />\n  <connection from="900244003#0" to="900244004" fromLane="0" toLane="0" via=":8363308288_5_0" dir="r" state="m" />\n  <connection from="911123067" to="706104219#16" fromLane="0" toLane="0" via=":2506086181_0_0" dir="l" state="m" />\n  <connection from="911123067" to="-706104219#15" fromLane="0" toLane="0" via=":2506086181_1_0" dir="r" state="m" />\n  <connection from="917695719" to="243314165#1" fromLane="0" toLane="0" via=":8523326825_2_0" dir="l" state="m" />\n  <connection from="917695719" to="-243314165#0" fromLane="0" toLane="0" via=":8523326825_3_0" dir="r" state="m" />\n  <connection from="917696883" to="-931354569#1" fromLane="0" toLane="0" via=":cluster_13424220808_2506085647_4_0" dir="l" state="m" />\n  <connection from="917696883" to="243188465#0" fromLane="0" toLane="0" via=":cluster_13424220808_2506085647_5_0" dir="s" state="m" />\n  <connection from="917696883" to="931354569#3" fromLane="0" toLane="0" via=":cluster_13424220808_2506085647_6_0" dir="r" state="m" />\n  <connection from="931348308" to="77967093#2" fromLane="0" toLane="0" via=":916327921_0_0" dir="s" state="m" visibility="9.00" />\n  <connection from="931354569#0" to="706092523#4" fromLane="0" toLane="0" via=":856214503_0_0" dir="l" state="M" />\n  <connection from="931354569#0" to="931354569#1" fromLane="0" toLane="0" via=":856214503_1_0" dir="s" state="M" />\n  <connection from="931354569#1" to="243188465#0" fromLane="0" toLane="0" via=":cluster_13424220808_2506085647_2_0" dir="l" state="M" />\n  <connection from="931354569#1" to="931354569#3" fromLane="0" toLane="0" via=":cluster_13424220808_2506085647_3_0" dir="s" state="M" />\n  <connection from="931354569#3" to="931354569#4" fromLane="0" toLane="0" via=":2506085628_2_0" dir="s" state="M" />\n  <connection from="931354569#4" to="706093508#0" fromLane="0" toLane="0" via=":1966386254_3_0" dir="s" state="M" />\n  <connection from="931354569#4" to="706093504#0" fromLane="0" toLane="0" via=":1966386254_4_0" dir="r" state="M" />\n  <connection from="931354570" to="931354569#0" fromLane="0" toLane="0" via=":2506085690_0_0" dir="l" state="m" />\n  <connection from="931354570" to="243188561#0" fromLane="0" toLane="0" via=":2506085690_1_0" dir="s" state="m" />\n  <connection from="931354570" to="-969757849#3" fromLane="0" toLane="0" via=":2506085690_2_0" dir="r" state="m" />\n  <connection from="932339001" to="243314310#1" fromLane="0" toLane="0" via=":2507400591_5_0" dir="s" state="m" />\n  <connection from="932339001" to="1547102487#1" fromLane="0" toLane="0" via=":2507400591_6_0" dir="r" state="m" />\n  <connection from="932339002#0" to="932339006" fromLane="0" toLane="0" via=":1680666775_1_0" dir="l" state="M" />\n  <connection from="932339002#0" to="932339002#4" fromLane="0" toLane="0" via=":1680666775_2_0" dir="s" state="M" />\n  <connection from="932339002#0" to="932339002#4" fromLane="1" toLane="1" via=":1680666775_2_1" dir="s" state="M" />\n  <connection from="932339002#4" to="932339004" fromLane="0" toLane="0" via=":1431867160_1_0" dir="l" state="M" />\n  <connection from="932339002#4" to="932339002#5" fromLane="0" toLane="0" via=":1431867160_2_0" dir="s" state="M" />\n  <connection from="932339002#4" to="932339002#5" fromLane="1" toLane="1" via=":1431867160_2_1" dir="s" state="M" />\n  <connection from="932339004" to="129842388#1" fromLane="0" toLane="0" via=":1431867108_5_0" dir="s" state="M" />\n  <connection from="932339004" to="1457057267" fromLane="0" toLane="0" via=":1431867108_6_0" dir="r" state="m" />\n  <connection from="932339006" to="155691084#0" fromLane="0" toLane="0" via=":8642866010_1_0" dir="s" state="M" />\n  <connection from="95533432#0" to="95533432#1" fromLane="0" toLane="0" via=":2506085476_0_0" dir="s" state="M" />\n  <connection from="95533432#0" to="-243188318" fromLane="0" toLane="0" via=":2506085476_1_0" dir="r" state="m" />\n  <connection from="95533432#1" to="-155690875" fromLane="0" toLane="0" via=":cluster_1680667233_2506085475_3_0" dir="l" state="M" />\n  <connection from="95533432#1" to="95533432#3" fromLane="0" toLane="0" via=":cluster_1680667233_2506085475_4_0" dir="s" state="M" />\n  <connection from="95533432#1" to="243188458" fromLane="0" toLane="0" via=":cluster_1680667233_2506085475_5_0" dir="r" state="m" />\n  <connection from="95533432#3" to="95533432#4" fromLane="0" toLane="0" via=":4900465219_0_0" dir="s" state="M" />\n  <connection from="95533432#3" to="498671679" fromLane="0" toLane="0" via=":4900465219_1_0" dir="r" state="m" />\n  <connection from="95533432#4" to="-155690528" fromLane="0" toLane="0" via=":1680667181_2_0" dir="l" state="M" />\n  <connection from="95533432#4" to="95533432#5" fromLane="0" toLane="0" via=":1680667181_3_0" dir="s" state="M" />\n  <connection from="95533432#5" to="1454593203#1" fromLane="0" toLane="0" via=":13339844003_0_0" dir="l" state="m" visibility="9.00" />\n  <connection from="95533433#0" to="706256377#2" fromLane="0" toLane="0" via=":1107784883_0_0" dir="l" state="m" />\n  <connection from="95533433#0" to="95533433#1" fromLane="0" toLane="0" via=":1107784883_1_0" dir="s" state="m" />\n  <connection from="95533433#0" to="-706256377#1" fromLane="0" toLane="0" via=":1107784883_2_0" dir="r" state="m" />\n  <connection from="95533433#1" to="706093509#0" fromLane="0" toLane="0" via=":1107784884_0_0" dir="s" state="m" />\n  <connection from="95533433#1" to="40264697#6" fromLane="0" toLane="1" via=":1107784884_1_0" dir="r" state="m" />\n  <connection from="95542480" to="970323546#1" fromLane="0" toLane="0" via=":1680668223_0_0" dir="l" state="m" />\n  <connection from="95542482#0" to="-243188585" fromLane="0" toLane="0" via=":1107926561_0_0" dir="l" state="M" />\n  <connection from="95542482#0" to="95542482#1" fromLane="0" toLane="0" via=":1107926561_1_0" dir="s" state="M" />\n  <connection from="95542482#1" to="-243188552#2" fromLane="0" toLane="0" via=":1107926940_0_0" dir="l" state="M" />\n  <connection from="95542482#1" to="95542482#2" fromLane="0" toLane="0" via=":1107926940_1_0" dir="s" state="M" />\n  <connection from="95542482#10" to="95542482#11" fromLane="0" toLane="0" via=":2507400769_0_0" dir="s" state="M" />\n  <connection from="95542482#10" to="243314137" fromLane="0" toLane="0" via=":2507400769_1_0" dir="r" state="m" />\n  <connection from="95542482#11" to="95542482#12" fromLane="0" toLane="0" via=":1110416469_0_0" dir="s" state="M" />\n  <connection from="95542482#11" to="100401320" fromLane="0" toLane="0" via=":1110416469_1_0" dir="r" state="m" />\n  <connection from="95542482#12" to="95542482#13" fromLane="0" toLane="0" via=":2507400772_0_0" dir="s" state="M" />\n  <connection from="95542482#12" to="-243314160" fromLane="0" toLane="0" via=":2507400772_1_0" dir="r" state="m" />\n  <connection from="95542482#13" to="-243188786#4" fromLane="0" toLane="0" via=":1110416754_0_0" dir="l" state="M" />\n  <connection from="95542482#13" to="95542482#14" fromLane="0" toLane="0" via=":1110416754_1_0" dir="s" state="M" />\n  <connection from="95542482#14" to="684741141#0" fromLane="0" toLane="0" via=":1123399591_0_0" dir="l" state="M" />\n  <connection from="95542482#14" to="-243188243" fromLane="0" toLane="0" via=":1123399591_1_0" dir="s" state="M" />\n  <connection from="95542482#2" to="243188641#0" fromLane="0" toLane="0" via=":1107926599_0_0" dir="l" state="M" />\n  <connection from="95542482#2" to="95542482#4" fromLane="0" toLane="0" via=":1107926599_1_0" dir="s" state="M" />\n  <connection from="95542482#4" to="243188594#0" fromLane="0" toLane="0" via=":1107926977_0_0" dir="l" state="M" />\n  <connection from="95542482#4" to="95542482#5" fromLane="0" toLane="0" via=":1107926977_1_0" dir="s" state="M" />\n  <connection from="95542482#4" to="-980451721#1" fromLane="0" toLane="0" via=":1107926977_2_0" dir="R" state="m" />\n  <connection from="95542482#5" to="95542482#6" fromLane="0" toLane="0" via=":1107926734_0_0" dir="s" state="m" />\n  <connection from="95542482#5" to="243188670#2" fromLane="0" toLane="1" via=":1107926734_1_0" dir="r" state="m" />\n  <connection from="95542482#6" to="95542482#10" fromLane="0" toLane="0" via=":1110416526_0_0" dir="s" state="M" />\n  <connection from="95542482#6" to="243188273#0" fromLane="0" toLane="0" via=":1110416526_1_0" dir="r" state="m" />\n  <connection from="95542485#0" to="-72016745#2" fromLane="0" toLane="0" via=":cluster_13472440775_1666900747_7_0" dir="l" state="m" />\n  <connection from="95542485#0" to="706105951#0" fromLane="0" toLane="0" via=":cluster_13472440775_1666900747_8_0" dir="s" state="m" />\n  <connection from="95542485#0" to="72016745#4" fromLane="0" toLane="1" via=":cluster_13472440775_1666900747_9_0" dir="r" state="m" />\n  <connection from="95809258" to="243188765#0" fromLane="0" toLane="0" via=":2505176697_4_0" dir="s" state="M" />\n  <connection from="95809258" to="96941841#0" fromLane="0" toLane="0" via=":2505176697_5_0" dir="r" state="m" />\n  <connection from="95809261#0" to="-706103887" fromLane="0" toLane="0" via=":2506087889_1_0" dir="l" state="M" />\n  <connection from="95809261#0" to="95809261#1" fromLane="0" toLane="0" via=":2506087889_2_0" dir="s" state="m" />\n  <connection from="95809261#1" to="-243188505" fromLane="0" toLane="0" via=":2506087968_0_0" dir="l" state="M" />\n  <connection from="95809261#1" to="95809261#2" fromLane="0" toLane="0" via=":2506087968_1_0" dir="s" state="m" />\n  <connection from="95809261#2" to="706104217#16" fromLane="0" toLane="0" via=":cluster_2506088039_8523406713_0_0" dir="l" state="m" />\n  <connection from="95809261#2" to="917704692" fromLane="0" toLane="0" via=":cluster_2506088039_8523406713_1_0" dir="s" state="m" />\n  <connection from="95809261#2" to="-706104217#14" fromLane="0" toLane="0" via=":cluster_2506088039_8523406713_2_0" dir="r" state="m" />\n  <connection from="95809262#0" to="142455600#4" fromLane="0" toLane="0" via=":1559016903_3_0" dir="l" state="m" />\n  <connection from="95809262#0" to="95809262#1" fromLane="0" toLane="0" via=":1559016903_4_0" dir="s" state="m" />\n  <connection from="95809262#0" to="-142455600#3" fromLane="0" toLane="0" via=":1559016903_5_0" dir="r" state="m" />\n  <connection from="95809262#1" to="-148941242#1" fromLane="0" toLane="0" via=":1431715372_2_0" dir="l" state="m" />\n  <connection from="95809262#1" to="148941242#2" fromLane="0" toLane="0" via=":1431715372_3_0" dir="r" state="m" />\n  <connection from="95809267#0" to="-243188389#0" fromLane="0" toLane="0" via=":12346099051_2_0" dir="l" state="M" />\n  <connection from="95809267#0" to="95809267#1" fromLane="0" toLane="0" via=":12346099051_3_0" dir="s" state="M" />\n  <connection from="95809267#0" to="243188389#1" fromLane="0" toLane="0" via=":12346099051_4_0" dir="r" state="M" />\n  <connection from="95809267#1" to="-706104217#16" fromLane="0" toLane="0" via=":1110416470_2_0" dir="l" state="M" />\n  <connection from="95809267#1" to="95809268#0" fromLane="0" toLane="0" via=":1110416470_3_0" dir="s" state="M" />\n  <connection from="95809267#1" to="706104217#17" fromLane="0" toLane="0" via=":1110416470_4_0" dir="r" state="M" />\n  <connection from="95809268#0" to="243188505" fromLane="0" toLane="0" via=":2506087955_1_0" dir="l" state="M" />\n  <connection from="95809268#0" to="95809268#1" fromLane="0" toLane="0" via=":2506087955_2_0" dir="s" state="M" />\n  <connection from="95809268#1" to="706103887" fromLane="0" toLane="0" via=":2506087863_1_0" dir="l" state="M" />\n  <connection from="95809268#1" to="95809268#2" fromLane="0" toLane="0" via=":2506087863_2_0" dir="s" state="M" />\n  <connection from="95809268#2" to="-498671536" fromLane="0" toLane="0" via=":4900465363_2_0" dir="l" state="M" />\n  <connection from="95809268#2" to="95809268#3" fromLane="0" toLane="0" via=":4900465363_3_0" dir="s" state="M" />\n  <connection from="95809268#2" to="1168168079" fromLane="0" toLane="0" via=":4900465363_4_0" dir="r" state="M" />\n  <connection from="95809268#3" to="-706104219#13" fromLane="0" toLane="0" via=":1110416654_1_0" dir="l" state="m" />\n  <connection from="95809268#3" to="706104219#14" fromLane="0" toLane="0" via=":1110416654_2_0" dir="r" state="m" />\n  <connection from="96242763#0" to="-498671391" fromLane="0" toLane="0" via=":1123399771_1_0" dir="l" state="M" />\n  <connection from="96242763#0" to="96242763#1" fromLane="0" toLane="0" via=":1123399771_2_0" dir="s" state="M" />\n  <connection from="96242763#1" to="96242763#2" fromLane="0" toLane="0" via=":1115585403_0_0" dir="s" state="M" />\n  <connection from="96242763#1" to="154097280" fromLane="0" toLane="0" via=":1115585403_1_0" dir="r" state="M" />\n  <connection from="96242763#2" to="-1476708031" fromLane="0" toLane="0" via=":13545019681_1_0" dir="l" state="M" />\n  <connection from="96242763#2" to="96242763#3" fromLane="0" toLane="0" via=":13545019681_2_0" dir="s" state="M" />\n  <connection from="96242763#3" to="-1204844052" fromLane="0" toLane="0" via=":4900464584_1_0" dir="l" state="m" />\n  <connection from="96713525" to="66984613#4" fromLane="0" toLane="0" via=":2505176388_0_0" dir="l" state="m" />\n  <connection from="96713525" to="243314186" fromLane="0" toLane="0" via=":2505176388_1_0" dir="s" state="m" />\n  <connection from="96713729" to="706104217#13" fromLane="0" toLane="0" via=":2506088084_0_0" dir="l" state="M" />\n  <connection from="96713729" to="-706104217#12" fromLane="0" toLane="0" via=":2506088084_1_0" dir="r" state="M" />\n  <connection from="96837170#0" to="96837170#3" fromLane="0" toLane="0" via=":1122072674_2_0" dir="s" state="M" />\n  <connection from="96837170#0" to="-129827552#1" fromLane="0" toLane="0" via=":1122072674_3_0" dir="r" state="m" />\n  <connection from="96837172#0" to="129827552#0" fromLane="0" toLane="0" via=":1431715395_4_0" dir="l" state="M" />\n  <connection from="96837172#0" to="96837172#1" fromLane="0" toLane="0" via=":1431715395_5_0" dir="s" state="M" />\n  <connection from="96837172#1" to="129827558" fromLane="0" toLane="0" via=":1122072673_2_0" dir="s" state="M" />\n  <connection from="96837172#1" to="706096643#0" fromLane="0" toLane="0" via=":1122072673_3_0" dir="r" state="m" />\n  <connection from="96941841#0" to="243188503#0" fromLane="0" toLane="0" via=":1123399700_4_0" dir="l" state="M" />\n  <connection from="96941841#0" to="96941841#1" fromLane="0" toLane="0" via=":1123399700_5_0" dir="s" state="M" />\n  <connection from="96941841#1" to="243188196" fromLane="0" toLane="0" via=":1123399513_4_0" dir="l" state="M" />\n  <connection from="96941841#1" to="96941841#2" fromLane="0" toLane="0" via=":1123399513_5_0" dir="s" state="M" />\n  <connection from="96941842#0" to="96941842#1" fromLane="0" toLane="0" via=":12649445300_0_0" dir="s" state="M" />\n  <connection from="96941842#0" to="-498671465" fromLane="0" toLane="0" via=":12649445300_1_0" dir="r" state="m" />\n  <connection from="96941842#1" to="969757849#3" fromLane="0" toLane="0" via=":1123399706_0_0" dir="l" state="m" />\n  <connection from="96941842#1" to="96941842#2" fromLane="0" toLane="0" via=":1123399706_1_0" dir="s" state="m" />\n  <connection from="96941842#1" to="-969757849#2" fromLane="0" toLane="0" via=":1123399706_2_0" dir="r" state="m" />\n  <connection from="96941842#2" to="96941842#3" fromLane="0" toLane="0" via=":1123399595_0_0" dir="s" state="M" />\n  <connection from="96941842#2" to="96941857#0" fromLane="0" toLane="0" via=":1123399595_1_0" dir="r" state="m" />\n  <connection from="96941842#3" to="706104219#9" fromLane="0" toLane="0" via=":1966399248_0_0" dir="l" state="m" />\n  <connection from="96941842#3" to="-243188491" fromLane="0" toLane="0" via=":1966399248_1_0" dir="s" state="m" />\n  <connection from="96941842#3" to="-706104219#8" fromLane="0" toLane="0" via=":1966399248_2_0" dir="r" state="m" />\n  <connection from="96941844" to="706256377#0" fromLane="0" toLane="0" via=":1110416410_4_0" dir="l" state="m" />\n  <connection from="96941844" to="-969765782" fromLane="0" toLane="0" via=":1110416410_5_0" dir="r" state="m" />\n  <connection from="96941849" to="706104218" fromLane="0" toLane="0" via=":5149825984_0_0" dir="l" state="m" />\n  <connection from="96941849" to="452036923#0" fromLane="0" toLane="0" via=":5149825984_1_0" dir="s" state="m" />\n  <connection from="96941849" to="-72016745#5" fromLane="0" toLane="1" via=":5149825984_2_0" dir="r" state="m" />\n  <connection from="96941857#0" to="-498671388#0" fromLane="0" toLane="0" via=":4900466448_9_0" dir="l" state="M" />\n  <connection from="96941857#0" to="96941857#1" fromLane="0" toLane="0" via=":4900466448_10_0" dir="s" state="M" />\n  <connection from="96941857#0" to="498671388#1" fromLane="0" toLane="0" via=":4900466448_11_0" dir="r" state="m" />\n  <connection from="96941857#1" to="-498671389#1" fromLane="0" toLane="0" via=":4900464580_6_0" dir="l" state="M" />\n  <connection from="96941857#1" to="96941857#2" fromLane="0" toLane="0" via=":4900464580_7_0" dir="s" state="M" />\n  <connection from="96941857#1" to="498671389#2" fromLane="0" toLane="0" via=":4900464580_8_0" dir="r" state="m" />\n  <connection from="96941857#2" to="498671395#1" fromLane="0" toLane="0" via=":1966404349_5_0" dir="l" state="M" />\n  <connection from="96941857#2" to="-154097280" fromLane="0" toLane="0" via=":1966404349_6_0" dir="s" state="M" />\n  <connection from="969757849#0" to="969757849#1" fromLane="0" toLane="0" via=":4900464598_0_0" dir="s" state="M" />\n  <connection from="969757849#0" to="-498671389#2" fromLane="0" toLane="0" via=":4900464598_1_0" dir="r" state="m" />\n  <connection from="969757849#1" to="1365935945#0" fromLane="0" toLane="0" via=":856214538_3_0" dir="l" state="M" />\n  <connection from="969757849#1" to="969757849#2" fromLane="0" toLane="0" via=":856214538_4_0" dir="s" state="M" />\n  <connection from="969757849#1" to="-498671388#1" fromLane="0" toLane="0" via=":856214538_5_0" dir="r" state="m" />\n  <connection from="969757849#2" to="-96941842#1" fromLane="0" toLane="0" via=":1123399706_3_0" dir="l" state="M" />\n  <connection from="969757849#2" to="969757849#3" fromLane="0" toLane="0" via=":1123399706_4_0" dir="s" state="M" />\n  <connection from="969757849#2" to="96941842#2" fromLane="0" toLane="0" via=":1123399706_5_0" dir="r" state="m" />\n  <connection from="969757849#3" to="931354569#0" fromLane="0" toLane="0" via=":2506085690_3_0" dir="s" state="M" />\n  <connection from="969757849#3" to="243188561#0" fromLane="0" toLane="0" via=":2506085690_4_0" dir="r" state="m" />\n  <connection from="969757851#0" to="969765780#1" fromLane="0" toLane="0" via=":cluster_1123399587_8976429007_4_0" dir="l" state="M" />\n  <connection from="969757851#0" to="969757851#1" fromLane="0" toLane="0" via=":cluster_1123399587_8976429007_5_0" dir="s" state="M" />\n  <connection from="969757851#1" to="1556742612" fromLane="0" toLane="0" via=":14159239425_1_0" dir="l" state="M" />\n  <connection from="969757851#1" to="969757851#2" fromLane="0" toLane="0" via=":14159239425_2_0" dir="s" state="M" />\n  <connection from="969757851#2" to="-243188024" fromLane="0" toLane="0" via=":2506085525_1_0" dir="l" state="M" />\n  <connection from="969757851#2" to="969757851#3" fromLane="0" toLane="0" via=":2506085525_2_0" dir="s" state="M" />\n  <connection from="969757851#3" to="-706105950" fromLane="0" toLane="0" via=":8976429009_1_0" dir="l" state="M" />\n  <connection from="969757851#3" to="969757851#4" fromLane="0" toLane="0" via=":8976429009_2_0" dir="s" state="M" />\n  <connection from="969757851#4" to="1218225522#0" fromLane="0" toLane="0" via=":cluster_849403384_849403500_8976428987_8976428988_7_0" tl="cluster_849403384_849403500_8976428987_8976428988" linkIndex="7" dir="l" state="o" />\n  <connection from="969757851#4" to="969757852" fromLane="0" toLane="0" via=":cluster_849403384_849403500_8976428987_8976428988_8_0" tl="cluster_849403384_849403500_8976428987_8976428988" linkIndex="8" dir="s" state="o" />\n  <connection from="969757851#4" to="990212968" fromLane="0" toLane="1" via=":cluster_849403384_849403500_8976428987_8976428988_9_0" tl="cluster_849403384_849403500_8976428987_8976428988" linkIndex="9" dir="r" state="o" />\n  <connection from="969757852" to="969757853#1" fromLane="0" toLane="0" via=":8976429008_1_0" dir="s" state="m" />\n  <connection from="969757853#1" to="-243314436#3" fromLane="0" toLane="0" via=":13449811511_1_0" dir="l" state="M" />\n  <connection from="969757853#1" to="969757853#2" fromLane="0" toLane="0" via=":13449811511_2_0" dir="s" state="M" />\n  <connection from="969757853#2" to="-243314467#1" fromLane="0" toLane="0" via=":2505176027_0_0" dir="l" state="M" />\n  <connection from="969757855#1" to="969757853#1" fromLane="0" toLane="0" via=":8976429008_0_0" dir="l" state="M" />\n  <connection from="969765780#1" to="969765780#2" fromLane="0" toLane="0" via=":14159239427_0_0" dir="s" state="M" />\n  <connection from="969765780#1" to="1556742613" fromLane="0" toLane="0" via=":14159239427_1_0" dir="r" state="m" />\n  <connection from="969765780#2" to="706520617#0" fromLane="0" toLane="0" via=":1115585416_0_0" dir="s" state="m" />\n  <connection from="969765780#2" to="-142453602#2" fromLane="0" toLane="0" via=":1115585416_1_0" dir="r" state="m" />\n  <connection from="969765781" to="706256383" fromLane="0" toLane="0" via=":1431715392_0_0" dir="l" state="M" />\n  <connection from="969765781" to="969765782" fromLane="0" toLane="0" via=":1431715392_1_0" dir="s" state="M" />\n  <connection from="969765781" to="1365892989" fromLane="0" toLane="0" via=":1431715392_2_0" dir="r" state="m" />\n  <connection from="969765782" to="706256377#0" fromLane="0" toLane="0" via=":1110416410_0_0" dir="s" state="M" />\n  <connection from="970010765#0" to="706094740#0" fromLane="0" toLane="0" via=":1107926433_1_0" dir="l" state="M" />\n  <connection from="970010765#0" to="970010765#1" fromLane="0" toLane="0" via=":1107926433_2_0" dir="s" state="M" />\n  <connection from="970010765#1" to="970010765#2" fromLane="0" toLane="0" via=":cluster_1123399587_8976429007_0_0" dir="s" state="M" />\n  <connection from="970010765#1" to="969765780#1" fromLane="0" toLane="0" via=":cluster_1123399587_8976429007_1_0" dir="R" state="m" />\n  <connection from="970010765#2" to="706094742" fromLane="0" toLane="0" via=":1558997825_1_0" dir="l" state="M" />\n  <connection from="970010765#2" to="970010765#3" fromLane="0" toLane="0" via=":1558997825_2_0" dir="s" state="M" />\n  <connection from="970010765#3" to="100401250#1" fromLane="0" toLane="0" via=":8976428985_0_0" dir="s" state="M" />\n  <connection from="970010767#0" to="-243314310#1" fromLane="0" toLane="0" via=":2505176020_1_0" dir="l" state="M" />\n  <connection from="970010767#0" to="970010767#1" fromLane="0" toLane="0" via=":2505176020_2_0" dir="s" state="M" />\n  <connection from="970010767#1" to="1547102487#0" fromLane="0" toLane="0" via=":2507400593_0_0" dir="l" state="M" />\n  <connection from="970010767#1" to="970010767#2" fromLane="0" toLane="0" via=":2507400593_1_0" dir="s" state="M" />\n  <connection from="970010767#2" to="990212968" fromLane="0" toLane="0" via=":cluster_849403384_849403500_8976428987_8976428988_0_0" tl="cluster_849403384_849403500_8976428987_8976428988" linkIndex="0" dir="l" state="o" />\n  <connection from="970010767#2" to="970010765#0" fromLane="0" toLane="0" via=":cluster_849403384_849403500_8976428987_8976428988_1_0" tl="cluster_849403384_849403500_8976428987_8976428988" linkIndex="1" dir="s" state="o" />\n  <connection from="970010767#2" to="1218225522#0" fromLane="0" toLane="0" via=":cluster_849403384_849403500_8976428987_8976428988_2_0" tl="cluster_849403384_849403500_8976428987_8976428988" linkIndex="2" dir="r" state="o" />\n  <connection from="970311985#1" to="77967093#0" fromLane="0" toLane="0" via=":916327888_0_0" dir="s" state="m" visibility="9.00" />\n  <connection from="970323545" to="-81350006#2" fromLane="0" toLane="0" via=":947896981_2_0" dir="l" state="m" />\n  <connection from="970323545" to="66984613#0" fromLane="0" toLane="0" via=":947896981_3_0" dir="s" state="m" />\n  <connection from="970323545" to="95542480" fromLane="0" toLane="0" via=":947896981_4_0" dir="r" state="m" />\n  <connection from="970323546#0" to="-95542480" fromLane="0" toLane="0" via=":1680668223_1_0" dir="l" state="M" />\n  <connection from="970323546#0" to="970323546#1" fromLane="0" toLane="0" via=":1680668223_2_0" dir="s" state="M" />\n  <connection from="970323546#0" to="970323546#1" fromLane="1" toLane="1" via=":1680668223_2_1" dir="s" state="M" />\n  <connection from="970323546#1" to="243079775" fromLane="0" toLane="0" via=":2505176514_1_0" dir="l" state="M" />\n  <connection from="970323546#1" to="970323546#2" fromLane="0" toLane="0" via=":2505176514_2_0" dir="s" state="M" />\n  <connection from="970323546#1" to="970323546#2" fromLane="1" toLane="1" via=":2505176514_2_1" dir="s" state="M" />\n  <connection from="970323546#2" to="-243079673" fromLane="0" toLane="0" via=":2505176494_1_0" dir="l" state="M" />\n  <connection from="970323546#2" to="970323546#3" fromLane="0" toLane="0" via=":2505176494_2_0" dir="s" state="M" />\n  <connection from="970323546#2" to="970323546#3" fromLane="1" toLane="1" via=":2505176494_2_1" dir="s" state="M" />\n  <connection from="970323546#3" to="-243314186" fromLane="0" toLane="0" via=":cluster_2505176422_2505176434_2_0" dir="l" state="M" />\n  <connection from="970323546#3" to="970323546#4" fromLane="0" toLane="0" via=":cluster_2505176422_2505176434_3_0" dir="s" state="M" />\n  <connection from="970323546#3" to="970323546#4" fromLane="1" toLane="1" via=":cluster_2505176422_2505176434_3_1" dir="s" state="M" />\n  <connection from="970323546#4" to="-243314213" fromLane="0" toLane="0" via=":2507400698_1_0" dir="l" state="M" />\n  <connection from="970323546#4" to="970323546#5" fromLane="0" toLane="0" via=":2507400698_2_0" dir="s" state="M" />\n  <connection from="970323546#4" to="970323546#5" fromLane="1" toLane="1" via=":2507400698_2_1" dir="s" state="M" />\n  <connection from="970323546#5" to="-243314073" fromLane="0" toLane="0" via=":2507400690_1_0" dir="l" state="M" />\n  <connection from="970323546#5" to="970323546#6" fromLane="0" toLane="0" via=":2507400690_2_0" dir="s" state="M" />\n  <connection from="970323546#5" to="970323546#6" fromLane="1" toLane="1" via=":2507400690_2_1" dir="s" state="M" />\n  <connection from="970323546#6" to="66984610#1" fromLane="0" toLane="0" via=":2505176258_2_0" dir="l" state="M" />\n  <connection from="970323546#6" to="970323546#7" fromLane="0" toLane="0" via=":2505176258_3_0" dir="s" state="M" />\n  <connection from="970323546#6" to="970323546#7" fromLane="1" toLane="1" via=":2505176258_3_1" dir="s" state="M" />\n  <connection from="970323546#6" to="-66984610#0" fromLane="1" toLane="0" via=":2505176258_5_0" dir="r" state="M" />\n  <connection from="970323546#7" to="1218747874" fromLane="0" toLane="0" via=":cluster_11292971032_2406954578_2406954579_2406954580_2_0" dir="l" state="M" />\n  <connection from="970323546#7" to="1218225518#0" fromLane="0" toLane="0" via=":cluster_11292971032_2406954578_2406954579_2406954580_3_0" dir="s" state="M" />\n  <connection from="970323546#7" to="1218225518#0" fromLane="1" toLane="1" via=":cluster_11292971032_2406954578_2406954579_2406954580_3_1" dir="s" state="M" />\n  <connection from="980451721#0" to="980451721#1" fromLane="0" toLane="0" via=":2506089309_1_0" dir="s" state="m" />\n  <connection from="980451721#1" to="-95542482#4" fromLane="0" toLane="0" via=":1107926977_3_0" dir="s" state="m" />\n  <connection from="980451721#1" to="243188594#0" fromLane="0" toLane="0" via=":1107926977_4_0" dir="R" state="m" />\n  <connection from="980451721#1" to="95542482#5" fromLane="0" toLane="0" via=":1107926977_5_0" dir="r" state="m" />\n  <connection from="980451722#0" to="-243188273#1" fromLane="0" toLane="0" via=":1123399569_0_0" dir="l" state="M" />\n  <connection from="98567749" to="-706104219#11" fromLane="0" toLane="0" via=":1140280757_4_0" dir="l" state="m" />\n  <connection from="98567749" to="498671544#0" fromLane="0" toLane="0" via=":1140280757_5_0" dir="s" state="m" />\n  <connection from="98567749" to="706104219#12" fromLane="0" toLane="0" via=":1140280757_6_0" dir="r" state="m" />\n  <connection from="990212966#0" to="-155690480#1" fromLane="0" toLane="0" via=":796747714_1_0" dir="l" state="M" />\n  <connection from="990212966#0" to="990212966#1" fromLane="0" toLane="0" via=":796747714_2_0" dir="s" state="M" />\n  <connection from="990212966#1" to="-155690513" fromLane="0" toLane="0" via=":1680667730_1_0" dir="l" state="M" />\n  <connection from="990212966#1" to="990212966#2" fromLane="0" toLane="0" via=":1680667730_2_0" dir="s" state="M" />\n  <connection from="990212966#2" to="970010765#0" fromLane="0" toLane="0" via=":cluster_849403384_849403500_8976428987_8976428988_10_0" tl="cluster_849403384_849403500_8976428987_8976428988" linkIndex="10" dir="l" state="O" />\n  <connection from="990212966#2" to="1218225522#0" fromLane="0" toLane="0" via=":cluster_849403384_849403500_8976428987_8976428988_11_0" tl="cluster_849403384_849403500_8976428987_8976428988" linkIndex="11" dir="s" state="O" />\n  <connection from="990212966#2" to="969757852" fromLane="0" toLane="0" via=":cluster_849403384_849403500_8976428987_8976428988_12_0" tl="cluster_849403384_849403500_8976428987_8976428988" linkIndex="12" dir="r" state="o" />\n  <connection from="990212968" to="932339001" fromLane="0" toLane="0" via=":8642866006_1_0" dir="l" state="M" />\n  <connection from="990212968" to="932339002#0" fromLane="0" toLane="0" via=":8642866006_2_0" dir="s" state="M" />\n  <connection from="990212968" to="932339002#0" fromLane="1" toLane="1" via=":8642866006_2_1" dir="s" state="M" />\n  <connection from=":1107784878_0" to="706256379#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107784878_1" to="95533433#0" fromLane="0" toLane="0" via=":1107784878_4_0" dir="r" state="m" />\n  <connection from=":1107784878_4" to="95533433#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107784878_2" to="95533433#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107784878_3" to="-706256379#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107784879_0" to="706093504#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107784879_1" to="706093505#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107784879_2" to="706093504#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107784879_3" to="706093505#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107784883_0" to="706256377#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107784883_1" to="95533433#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107784883_2" to="-706256377#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107784883_3" to="706256377#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107784883_4" to="95533433#1" fromLane="0" toLane="0" via=":1107784883_7_0" dir="r" state="m" />\n  <connection from=":1107784883_7" to="95533433#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107784883_5" to="95533433#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107784883_6" to="-706256377#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107784884_0" to="706093509#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107784884_1" to="40264697#6" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":1107784884_2" to="706093509#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107784884_3" to="40264697#6" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107784884_3" to="40264697#6" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":1107784887_0" to="706104219#15" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107784887_1" to="706093506#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107784887_2" to="-706104219#14" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107784887_3" to="706104219#15" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107784887_4" to="706093506#0" fromLane="0" toLane="0" via=":1107784887_7_0" dir="r" state="m" />\n  <connection from=":1107784887_7" to="706093506#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107784887_5" to="706093506#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107784887_6" to="-706104219#14" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926433_0" to="970010765#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926433_1" to="706094740#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926433_2" to="970010765#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926467_0" to="243188710#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926467_1" to="-243188710#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926467_2" to="498671427" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926467_3" to="243188710#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926467_4" to="-243188710#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926467_5" to="498671427" fromLane="0" toLane="0" via=":1107926467_6_0" dir="r" state="m" />\n  <connection from=":1107926467_6" to="498671427" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926481_0" to="95542485#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926481_1" to="95542482#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926481_2" to="95809258" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926481_3" to="-243188710#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926481_4" to="95542485#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926481_5" to="95542482#0" fromLane="0" toLane="0" via=":1107926481_12_0" dir="r" state="m" />\n  <connection from=":1107926481_12" to="95542482#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926481_6" to="95809258" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926481_7" to="-243188710#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926481_8" to="95542485#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926481_9" to="95542482#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926481_10" to="95809258" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926481_11" to="-243188710#2" fromLane="0" toLane="0" via=":1107926481_13_0" dir="r" state="m" />\n  <connection from=":1107926481_13" to="-243188710#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926513_0" to="-706105951#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926513_1" to="900244003#0" fromLane="0" toLane="0" via=":1107926513_6_0" dir="r" state="m" />\n  <connection from=":1107926513_6" to="900244003#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926513_2" to="706105951#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926513_3" to="-706105951#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926513_4" to="900244003#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926513_5" to="706105951#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926537_0" to="243188741#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926537_1" to="-706105951#5" fromLane="0" toLane="0" via=":1107926537_6_0" dir="r" state="m" />\n  <connection from=":1107926537_6" to="-706105951#5" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926537_2" to="706105950" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926537_3" to="243188741#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926537_4" to="-706105951#5" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926537_5" to="706105950" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926561_0" to="-243188585" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926561_1" to="95542482#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926561_2" to="-95542482#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926561_3" to="-243188585" fromLane="0" toLane="0" via=":1107926561_6_0" dir="r" state="m" />\n  <connection from=":1107926561_6" to="-243188585" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926561_4" to="95542482#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926561_5" to="-95542482#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926599_0" to="243188641#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926599_1" to="95542482#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926599_2" to="-95542482#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926599_3" to="243188641#0" fromLane="0" toLane="0" via=":1107926599_4_0" dir="r" state="m" />\n  <connection from=":1107926599_4" to="243188641#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926639_0" to="-243188351#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926639_1" to="-706105951#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926639_2" to="243188629" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926639_3" to="706105951#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926639_4" to="-243188351#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926639_5" to="-706105951#3" fromLane="0" toLane="0" via=":1107926639_12_0" dir="r" state="m" />\n  <connection from=":1107926639_12" to="-706105951#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926639_6" to="243188629" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926639_7" to="706105951#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926639_8" to="-243188351#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926639_9" to="-706105951#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926639_10" to="243188629" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926639_11" to="706105951#4" fromLane="0" toLane="0" via=":1107926639_13_0" dir="r" state="m" />\n  <connection from=":1107926639_13" to="706105951#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926734_0" to="95542482#6" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926734_1" to="243188670#2" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":1107926734_2" to="243188670#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926734_3" to="-95542482#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926734_4" to="95542482#6" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926734_5" to="243188670#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926734_5" to="243188670#2" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":1107926734_7" to="-95542482#5" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926940_0" to="-243188552#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926940_1" to="95542482#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926940_2" to="-95542482#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926940_3" to="-243188552#2" fromLane="0" toLane="0" via=":1107926940_6_0" dir="r" state="m" />\n  <connection from=":1107926940_6" to="-243188552#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926940_4" to="95542482#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926940_5" to="-95542482#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926968_0" to="243188710#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926968_1" to="-243314395" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926977_0" to="243188594#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926977_1" to="95542482#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926977_2" to="-980451721#1" fromLane="0" toLane="0" via=":1107926977_9_0" dir="R" state="m" />\n  <connection from=":1107926977_9" to="-980451721#1" fromLane="0" toLane="0" dir="R" state="M" />\n  <connection from=":1107926977_3" to="-95542482#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926977_4" to="243188594#0" fromLane="0" toLane="0" dir="R" state="M" />\n  <connection from=":1107926977_5" to="95542482#5" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1107926977_6" to="-980451721#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1107926977_7" to="-95542482#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1107926977_8" to="243188594#0" fromLane="0" toLane="0" via=":1107926977_10_0" dir="r" state="m" />\n  <connection from=":1107926977_10" to="243188594#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416410_0" to="706256377#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416410_1" to="-969765782" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416410_2" to="706256377#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416410_3" to="-969765782" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416410_4" to="706256377#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416410_5" to="-969765782" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416444_0" to="706092523#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416444_1" to="706104219#13" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416444_2" to="1365892748#0" fromLane="0" toLane="0" via=":1110416444_9_0" dir="r" state="m" />\n  <connection from=":1110416444_9" to="1365892748#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416444_3" to="-706104219#12" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416444_4" to="706092523#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416444_5" to="706104219#13" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416444_6" to="1365892748#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416444_7" to="-706104219#12" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416444_8" to="706092523#0" fromLane="0" toLane="0" via=":1110416444_10_0" dir="r" state="m" />\n  <connection from=":1110416444_10" to="706092523#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416469_0" to="95542482#12" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416469_1" to="100401320" fromLane="0" toLane="0" via=":1110416469_6_0" dir="r" state="m" />\n  <connection from=":1110416469_6" to="100401320" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416469_2" to="-95542482#11" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416469_3" to="95542482#12" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416469_4" to="100401320" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416469_5" to="-95542482#11" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416470_0" to="95809268#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416470_1" to="706104217#17" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416470_2" to="-706104217#16" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416470_3" to="95809268#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416470_4" to="706104217#17" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416470_5" to="-706104217#16" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416470_6" to="95809268#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416526_0" to="95542482#10" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416526_1" to="243188273#0" fromLane="0" toLane="0" via=":1110416526_6_0" dir="r" state="m" />\n  <connection from=":1110416526_6" to="243188273#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416526_2" to="-95542482#9" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416526_3" to="95542482#10" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416526_4" to="243188273#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416526_5" to="-95542482#9" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416568_0" to="243079805" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416568_1" to="1365892991" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416568_2" to="-706520617#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416568_3" to="243079805" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416568_4" to="1365892991" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416631_0" to="95809267#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416654_0" to="706104219#14" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416654_1" to="-706104219#13" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416654_2" to="706104219#14" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416654_3" to="-706104219#13" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416726_0" to="129827554#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416726_1" to="148941242#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416726_2" to="-142455604#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416726_3" to="-148941242#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416726_4" to="129827554#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416726_5" to="148941242#1" fromLane="0" toLane="0" via=":1110416726_12_0" dir="r" state="m" />\n  <connection from=":1110416726_12" to="148941242#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416726_6" to="-142455604#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416726_7" to="-148941242#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416726_8" to="129827554#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416726_9" to="148941242#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416726_10" to="-142455604#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416726_11" to="-148941242#0" fromLane="0" toLane="0" via=":1110416726_13_0" dir="r" state="m" />\n  <connection from=":1110416726_13" to="-148941242#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416742_0" to="-243188765#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416742_1" to="-497290263#4" fromLane="0" toLane="0" via=":1110416742_6_0" dir="r" state="m" />\n  <connection from=":1110416742_6" to="-497290263#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416742_2" to="243188765#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416742_3" to="-243188765#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416742_4" to="-497290263#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416742_5" to="243188765#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416754_0" to="-243188786#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416754_1" to="95542482#14" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416754_2" to="-95542482#13" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1110416754_3" to="-243188786#4" fromLane="0" toLane="0" via=":1110416754_6_0" dir="r" state="m" />\n  <connection from=":1110416754_6" to="-243188786#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1110416754_4" to="95542482#14" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1110416754_5" to="-95542482#13" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1115585385_0" to="142453602#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1115585385_1" to="969757849#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1115585385_2" to="498671395#0" fromLane="0" toLane="0" via=":1115585385_9_0" dir="r" state="m" />\n  <connection from=":1115585385_9" to="498671395#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1115585385_3" to="498671395#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1115585385_4" to="-1204844051" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1115585385_5" to="142453602#0" fromLane="0" toLane="0" via=":1115585385_10_0" dir="r" state="m" />\n  <connection from=":1115585385_10" to="142453602#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1115585385_6" to="969757849#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1115585385_7" to="498671395#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1115585385_8" to="-1204844051" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1115585403_0" to="96242763#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1115585403_1" to="154097280" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1115585403_2" to="96242763#2" fromLane="0" toLane="0" via=":1115585403_3_0" dir="r" state="m" />\n  <connection from=":1115585403_3" to="96242763#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1115585416_0" to="706520617#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1115585416_1" to="-142453602#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1115585416_2" to="-969765780#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1115585416_3" to="706520617#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1115585416_4" to="-142453602#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1115585416_5" to="-969765780#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1115585463_0" to="142453602#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1115585463_1" to="-142453602#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1115585463_2" to="243188024" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1115585463_3" to="142453602#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1115585463_4" to="-142453602#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1115585463_5" to="243188024" fromLane="0" toLane="0" via=":1115585463_6_0" dir="r" state="m" />\n  <connection from=":1115585463_6" to="243188024" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1115585792_0" to="706520617#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1115585792_1" to="931354570" fromLane="0" toLane="0" via=":1115585792_4_0" dir="r" state="m" />\n  <connection from=":1115585792_4" to="931354570" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1115585792_2" to="931354570" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1115585792_3" to="-706520617#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1115585857_0" to="497290261#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1115585857_1" to="497290261#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1115585909_0" to="142453602#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1115585909_1" to="-498671393" fromLane="0" toLane="0" via=":1115585909_6_0" dir="r" state="m" />\n  <connection from=":1115585909_6" to="-498671393" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1115585909_2" to="-142453602#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1115585909_3" to="142453602#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1115585909_4" to="-498671393" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1115585909_5" to="-142453602#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1122072673_0" to="706096643#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1122072673_1" to="-96837172#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1122072673_2" to="129827558" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1122072673_3" to="706096643#0" fromLane="0" toLane="0" via=":1122072673_4_0" dir="r" state="m" />\n  <connection from=":1122072673_4" to="706096643#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1122072674_0" to="-129827552#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1122072674_1" to="-96837170#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1122072674_2" to="96837170#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1122072674_3" to="-129827552#1" fromLane="0" toLane="0" via=":1122072674_6_0" dir="r" state="m" />\n  <connection from=":1122072674_6" to="-129827552#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1122072674_4" to="-96837170#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1122072674_5" to="96837170#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1122072681_0" to="706096640#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1122072681_1" to="-706256379#6" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1122072681_2" to="-706096644#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1122072681_3" to="706096640#0" fromLane="0" toLane="0" via=":1122072681_4_0" dir="r" state="m" />\n  <connection from=":1122072681_4" to="706096640#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399483_0" to="706094741#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399483_1" to="-243188565" fromLane="0" toLane="0" via=":1123399483_6_0" dir="r" state="m" />\n  <connection from=":1123399483_6" to="-243188565" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399483_2" to="-706094741#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399483_3" to="706094741#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399483_4" to="-243188565" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399483_5" to="-706094741#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399486_0" to="684741141#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399486_1" to="-684741141#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399486_2" to="243188085" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399486_3" to="684741141#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399486_4" to="-684741141#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399486_5" to="243188085" fromLane="0" toLane="0" via=":1123399486_6_0" dir="r" state="m" />\n  <connection from=":1123399486_6" to="243188085" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399489_0" to="684740716" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399489_1" to="-706094741#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399489_2" to="684740716" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399489_3" to="-706094741#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399502_0" to="-684741141#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399502_1" to="243188786#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399502_2" to="-243188786#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399502_3" to="-684741141#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399502_4" to="243188786#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399502_5" to="-243188786#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399502_6" to="-684741141#3" fromLane="0" toLane="0" via=":1123399502_9_0" dir="r" state="m" />\n  <connection from=":1123399502_9" to="-684741141#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399502_7" to="243188786#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399502_8" to="-243188786#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399513_0" to="-96941841#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399513_1" to="243188196" fromLane="0" toLane="0" via=":1123399513_6_0" dir="r" state="m" />\n  <connection from=":1123399513_6" to="243188196" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399513_2" to="96941841#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399513_3" to="-96941841#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399513_4" to="243188196" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399513_5" to="96941841#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399516_0" to="706094741#0" fromLane="0" toLane="0" dir="L" state="M" />\n  <connection from=":1123399516_1" to="243188552#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399516_2" to="-243188705" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399516_3" to="706094741#0" fromLane="0" toLane="0" via=":1123399516_6_0" dir="r" state="m" />\n  <connection from=":1123399516_6" to="706094741#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399516_4" to="243188552#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399516_5" to="-243188705" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399535_0" to="452036923#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399535_1" to="452037066" fromLane="0" toLane="0" via=":1123399535_6_0" dir="r" state="m" />\n  <connection from=":1123399535_6" to="452037066" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399535_2" to="-452036923#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399535_3" to="452036923#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399535_4" to="452037066" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399535_5" to="-452036923#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399561_0" to="452036923#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399561_1" to="452036922" fromLane="0" toLane="0" via=":1123399561_6_0" dir="r" state="m" />\n  <connection from=":1123399561_6" to="452036922" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399561_2" to="-452036923#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399561_3" to="452036923#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399561_4" to="452036922" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399561_5" to="-452036923#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399562_0" to="706520617#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399562_1" to="96941842#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399562_2" to="-706520617#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399562_3" to="706520617#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399562_4" to="96941842#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399562_5" to="-706520617#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399569_0" to="-243188273#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399591_0" to="684741141#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399591_1" to="-243188243" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399591_2" to="-95542482#14" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399591_3" to="684741141#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399591_4" to="-243188243" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399591_5" to="-95542482#14" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399591_6" to="684741141#0" fromLane="0" toLane="0" via=":1123399591_9_0" dir="r" state="m" />\n  <connection from=":1123399591_9" to="684741141#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399591_7" to="-243188243" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399591_8" to="-95542482#14" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399595_0" to="96941842#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399595_1" to="96941857#0" fromLane="0" toLane="0" via=":1123399595_6_0" dir="r" state="m" />\n  <connection from=":1123399595_6" to="96941857#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399595_2" to="-96941842#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399595_3" to="96941842#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399595_4" to="96941857#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399595_5" to="-96941842#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399600_0" to="-498671615" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399600_1" to="185951074#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399600_2" to="96941849" fromLane="0" toLane="0" via=":1123399600_12_0" dir="r" state="m" />\n  <connection from=":1123399600_12" to="96941849" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399600_3" to="-185951074#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399600_4" to="-498671615" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399600_5" to="185951074#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399600_6" to="96941849" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399600_7" to="-185951074#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399600_8" to="-498671615" fromLane="0" toLane="0" via=":1123399600_13_0" dir="r" state="m" />\n  <connection from=":1123399600_13" to="-498671615" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399600_9" to="185951074#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399600_10" to="96941849" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399600_11" to="-185951074#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399639_0" to="706104219#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399639_1" to="-706104218" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399639_2" to="-530314558#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399639_3" to="706104219#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399639_4" to="-706104218" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399639_5" to="-530314558#1" fromLane="0" toLane="0" via=":1123399639_6_0" dir="r" state="m" />\n  <connection from=":1123399639_6" to="-530314558#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399641_0" to="-243188212#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399641_1" to="706094741#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399641_2" to="243188212#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399641_3" to="-706094741#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399641_4" to="-243188212#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399641_5" to="706094741#1" fromLane="0" toLane="0" via=":1123399641_12_0" dir="r" state="m" />\n  <connection from=":1123399641_12" to="706094741#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399641_6" to="243188212#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399641_7" to="-706094741#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399641_8" to="-243188212#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399641_9" to="706094741#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399641_10" to="243188212#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399641_11" to="-706094741#0" fromLane="0" toLane="0" via=":1123399641_13_0" dir="r" state="m" />\n  <connection from=":1123399641_13" to="-706094741#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399659_0" to="706094741#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399659_1" to="-706094741#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399659_2" to="706094741#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399659_3" to="-706094741#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399675_0" to="684741141#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399675_1" to="-684741141#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399675_2" to="243188161" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399675_3" to="684741141#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399675_4" to="-684741141#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399675_5" to="243188161" fromLane="0" toLane="0" via=":1123399675_6_0" dir="r" state="m" />\n  <connection from=":1123399675_6" to="243188161" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399676_0" to="980451722#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399676_1" to="497290269" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399676_2" to="980451721#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399676_3" to="980451722#0" fromLane="0" toLane="0" via=":1123399676_4_0" dir="r" state="m" />\n  <connection from=":1123399676_4" to="980451722#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399683_0" to="243188225" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399683_1" to="-452036923#7" fromLane="0" toLane="0" via=":1123399683_6_0" dir="r" state="m" />\n  <connection from=":1123399683_6" to="-452036923#7" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399683_2" to="243188705" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399683_3" to="243188225" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399683_4" to="-452036923#7" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399683_5" to="243188705" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399700_0" to="-96941841#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399700_1" to="243188503#0" fromLane="0" toLane="0" via=":1123399700_6_0" dir="r" state="m" />\n  <connection from=":1123399700_6" to="243188503#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399700_2" to="96941841#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399700_3" to="-96941841#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399700_4" to="243188503#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399700_5" to="96941841#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399706_0" to="969757849#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399706_1" to="96941842#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399706_2" to="-969757849#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399706_3" to="-96941842#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399706_4" to="969757849#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399706_5" to="96941842#2" fromLane="0" toLane="0" via=":1123399706_12_0" dir="r" state="m" />\n  <connection from=":1123399706_12" to="96941842#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399706_6" to="-969757849#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399706_7" to="-96941842#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399706_8" to="969757849#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399706_9" to="96941842#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399706_10" to="-969757849#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399706_11" to="-96941842#1" fromLane="0" toLane="0" via=":1123399706_13_0" dir="r" state="m" />\n  <connection from=":1123399706_13" to="-96941842#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399713_0" to="-497290263#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399713_1" to="243188115" fromLane="0" toLane="0" via=":1123399713_6_0" dir="r" state="m" />\n  <connection from=":1123399713_6" to="243188115" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399713_2" to="497290263#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399713_3" to="-497290263#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399713_4" to="243188115" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399713_5" to="497290263#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399740_0" to="-497290263#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399740_1" to="243188445" fromLane="0" toLane="0" via=":1123399740_6_0" dir="r" state="m" />\n  <connection from=":1123399740_6" to="243188445" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399740_2" to="497290263#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399740_3" to="-497290263#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399740_4" to="243188445" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399740_5" to="497290263#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399771_0" to="96242763#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399771_1" to="-498671391" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399771_2" to="96242763#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399863_0" to="1257722739#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399943_0" to="684741141#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399943_1" to="-684741141#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1123399943_2" to="243188361" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1123399943_3" to="684741141#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399943_4" to="-684741141#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1123399943_5" to="243188361" fromLane="0" toLane="0" via=":1123399943_6_0" dir="r" state="m" />\n  <connection from=":1123399943_6" to="243188361" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":11292971036_0" to="243079844#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1140280757_0" to="706104219#12" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1140280757_1" to="-706104219#11" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1140280757_2" to="498671544#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1140280757_3" to="706104219#12" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1140280757_4" to="-706104219#11" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1140280757_5" to="498671544#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1140280757_6" to="706104219#12" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1140280757_7" to="-706104219#11" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1140280757_8" to="498671544#0" fromLane="0" toLane="0" via=":1140280757_9_0" dir="r" state="m" />\n  <connection from=":1140280757_9" to="498671544#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1140280758_0" to="98567749" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1140280758_1" to="706104217#14" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1140280758_2" to="-706104217#13" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1140280758_3" to="98567749" fromLane="0" toLane="0" via=":1140280758_4_0" dir="r" state="m" />\n  <connection from=":1140280758_4" to="98567749" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1160612969_0" to="101939921" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1160612969_1" to="706256381" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1160612969_2" to="101939921" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":1160612974_0" to="969757851#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1160612974_1" to="100401250#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1174378788_0" to="706096644#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1174378788_1" to="96837172#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1174378788_2" to="706096644#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1174378788_3" to="96837172#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1176926303_0" to="84842974#14" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1176926303_1" to="-706256377#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1176926303_2" to="84842974#14" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1176926303_2" to="84842974#14" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":1176926303_2" to="84842974#14" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":1196113662_0" to="40264696#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1196113662_0" to="40264696#2" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":1196113662_0" to="40264696#2" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":1196113662_3" to="101706795#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1196113662_4" to="40264696#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1196113662_5" to="101706795#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":12339345632_0" to="706104217#12" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":12339345632_1" to="498671702" fromLane="0" toLane="0" via=":12339345632_6_0" dir="r" state="m" />\n  <connection from=":12339345632_6" to="498671702" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":12339345632_2" to="-706104217#11" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":12339345632_3" to="706104217#12" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":12339345632_4" to="498671702" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":12339345632_5" to="-706104217#11" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":12344163778_0" to="706093504#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":12344163778_1" to="1334274432" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":12344163778_2" to="706093504#2" fromLane="0" toLane="0" via=":12344163778_3_0" dir="r" state="m" />\n  <connection from=":12344163778_3" to="706093504#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":12346099051_0" to="95809267#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":12346099051_1" to="243188389#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":12346099051_2" to="-243188389#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":12346099051_3" to="95809267#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":12346099051_4" to="243188389#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":12346099051_5" to="-243188389#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":12346099051_6" to="95809267#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":12346099054_0" to="498671676#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":12346099054_1" to="330615829#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":12346099054_2" to="-330615829#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":12346099054_3" to="498671676#0" fromLane="0" toLane="0" via=":12346099054_4_0" dir="r" state="m" />\n  <connection from=":12346099054_4" to="498671676#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":12351390200_0" to="-243188101" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":12351390200_1" to="243188212#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":12649445299_0" to="706520617#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":12649445299_1" to="-1365935945#1" fromLane="0" toLane="0" via=":12649445299_6_0" dir="r" state="m" />\n  <connection from=":12649445299_6" to="-1365935945#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":12649445299_2" to="-706520617#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":12649445299_3" to="706520617#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":12649445299_4" to="-1365935945#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":12649445299_5" to="-706520617#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":12649445300_0" to="96941842#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":12649445300_1" to="-498671465" fromLane="0" toLane="0" via=":12649445300_6_0" dir="r" state="m" />\n  <connection from=":12649445300_6" to="-498671465" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":12649445300_2" to="-96941842#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":12649445300_3" to="96941842#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":12649445300_4" to="-498671465" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":12649445300_5" to="-96941842#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":13279030938_0" to="185951074#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":13279030938_1" to="-185951074#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":13279030938_2" to="1447303746" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":13279030938_3" to="185951074#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":13279030938_4" to="-185951074#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":13279030938_5" to="1447303746" fromLane="0" toLane="0" via=":13279030938_6_0" dir="r" state="m" />\n  <connection from=":13279030938_6" to="1447303746" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":13339819885_0" to="1454593202#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":13339819885_1" to="1454593203#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":13339819885_2" to="1454593203#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":13339819888_0" to="1454593203#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":13339819888_1" to="-706256380#6" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":13339819888_2" to="1454593203#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":13339819893_0" to="1454593203#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":13339819893_1" to="-706256381" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":13339819893_2" to="1454593203#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":13339844003_0" to="1454593203#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":13339844003_1" to="-95533432#5" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":13339844003_2" to="1454593203#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":13362936997_0" to="129842391" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":13362936997_1" to="129842391" fromLane="0" toLane="0" via=":13362936997_2_0" dir="r" state="m" />\n  <connection from=":13362936997_2" to="129842391" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":13449811511_0" to="969757853#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":13449811511_1" to="-243314436#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":13449811511_2" to="969757853#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":13545019681_0" to="96242763#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":13545019681_1" to="-1476708031" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":13545019681_2" to="96242763#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1369020939_0" to="243188710#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1369020939_1" to="-243188710#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1369020939_2" to="-122540020#7" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1369020939_3" to="243188710#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1369020939_4" to="-243188710#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1369020939_5" to="-122540020#7" fromLane="0" toLane="0" via=":1369020939_6_0" dir="r" state="m" />\n  <connection from=":1369020939_6" to="-122540020#7" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1369020942_0" to="-122540020#6" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1369020942_1" to="122540020#7" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1369020942_2" to="243188056" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1369020942_3" to="-122540020#6" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1369020942_4" to="122540020#7" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1369020942_5" to="243188056" fromLane="0" toLane="0" via=":1369020942_6_0" dir="r" state="m" />\n  <connection from=":1369020942_6" to="243188056" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1369020943_0" to="-122540020#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1369020943_1" to="122540020#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1369020943_2" to="243188283" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1369020943_3" to="-122540020#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1369020943_4" to="122540020#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1369020943_5" to="243188283" fromLane="0" toLane="0" via=":1369020943_6_0" dir="r" state="m" />\n  <connection from=":1369020943_6" to="243188283" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1369020950_0" to="72016745#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1369020950_0" to="72016745#2" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":1369020950_2" to="122540020#0" fromLane="0" toLane="0" via=":1369020950_8_0" dir="r" state="m" />\n  <connection from=":1369020950_8" to="122540020#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1369020950_3" to="-72016745#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1369020950_4" to="72016745#2" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":1369020950_5" to="122540020#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1369020950_6" to="-72016745#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1369020950_6" to="-72016745#1" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":14069947700_0" to="40264697#5" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14069947700_1" to="1546894905" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14069947700_2" to="40264697#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14069947700_2" to="40264697#5" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":14069981565_0" to="706097788#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14069981565_1" to="1546896992" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14069981565_2" to="706097788#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14070071629_0" to="-155690791#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14070071629_1" to="1546907364" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14159239425_0" to="969757851#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14159239425_1" to="1556742612" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14159239425_2" to="969757851#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14159239427_0" to="969765780#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14159239427_1" to="1556742613" fromLane="0" toLane="0" via=":14159239427_6_0" dir="r" state="m" />\n  <connection from=":14159239427_6" to="1556742613" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14159239427_2" to="-969765780#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14159239427_3" to="969765780#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14159239427_4" to="1556742613" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14159239427_5" to="-969765780#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14165547215_0" to="-122540020#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14165547215_1" to="1557537524" fromLane="0" toLane="0" via=":14165547215_6_0" dir="r" state="m" />\n  <connection from=":14165547215_6" to="1557537524" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14165547215_2" to="122540020#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14165547215_3" to="-122540020#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14165547215_4" to="1557537524" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14165547215_5" to="122540020#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14179759489_0" to="-498671328#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14179759489_1" to="498671328#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14179759489_2" to="1558734968" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14179759489_3" to="-498671328#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14179759489_4" to="498671328#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14179759489_5" to="1558734968" fromLane="0" toLane="0" via=":14179759489_6_0" dir="r" state="m" />\n  <connection from=":14179759489_6" to="1558734968" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14182422401_0" to="-155690741#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14182422401_1" to="155690741#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14182422401_2" to="1559020882" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14182422401_3" to="-155690741#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14182422401_4" to="155690741#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14182422401_5" to="1559020882" fromLane="0" toLane="0" via=":14182422401_6_0" dir="r" state="m" />\n  <connection from=":14182422401_6" to="1559020882" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14188374629_0" to="1559802496" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14188374629_1" to="1559802495#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14188374629_2" to="-1559802495#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14188374629_3" to="1559802496" fromLane="0" toLane="0" via=":14188374629_6_0" dir="r" state="m" />\n  <connection from=":14188374629_6" to="1559802496" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14188374629_4" to="1559802495#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14188374629_5" to="-1559802495#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14188764645_0" to="129827561#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14188764645_1" to="1559841160#0" fromLane="0" toLane="0" via=":14188764645_6_0" dir="r" state="m" />\n  <connection from=":14188764645_6" to="1559841160#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14188764645_2" to="-129827561#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14188764645_3" to="129827561#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14188764645_4" to="1559841160#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14188764645_5" to="-129827561#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14188764646_0" to="-1559841160#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14188764646_1" to="1559841161" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14188764646_2" to="1559841160#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14188764646_3" to="-1559841160#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14188764646_4" to="1559841161" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14188764646_5" to="1559841160#1" fromLane="0" toLane="0" via=":14188764646_6_0" dir="r" state="m" />\n  <connection from=":14188764646_6" to="1559841160#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14188764647_0" to="1454593202#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":14188764647_1" to="-1559841160#1" fromLane="0" toLane="0" via=":14188764647_6_0" dir="r" state="m" />\n  <connection from=":14188764647_6" to="-1559841160#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14188764647_2" to="-1454593202#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14188764647_3" to="1454593202#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":14188764647_4" to="-1559841160#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":14188764647_5" to="-1454593202#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431715363_0" to="71380773#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431715363_1" to="71380773#1" fromLane="0" toLane="0" via=":1431715363_2_0" dir="r" state="m" />\n  <connection from=":1431715363_2" to="71380773#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431715366_0" to="71380773#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431715366_1" to="71380773#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431715369_0" to="84842974#15" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431715369_0" to="84842974#15" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":1431715369_0" to="84842974#15" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":1431715369_3" to="684704215#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431715372_0" to="148941242#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431715372_1" to="-95809262#1" fromLane="0" toLane="0" via=":1431715372_6_0" dir="r" state="m" />\n  <connection from=":1431715372_6" to="-95809262#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431715372_2" to="-148941242#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431715372_3" to="148941242#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431715372_4" to="-95809262#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431715372_5" to="-148941242#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431715392_0" to="706256383" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431715392_1" to="969765782" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431715392_2" to="1365892989" fromLane="0" toLane="0" via=":1431715392_6_0" dir="r" state="m" />\n  <connection from=":1431715392_6" to="1365892989" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431715392_3" to="1365892989" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431715392_4" to="-969765781" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431715392_5" to="706256383" fromLane="0" toLane="0" via=":1431715392_7_0" dir="r" state="m" />\n  <connection from=":1431715392_7" to="706256383" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431715395_0" to="-96837172#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431715395_1" to="129827552#0" fromLane="0" toLane="0" via=":1431715395_6_0" dir="r" state="m" />\n  <connection from=":1431715395_6" to="129827552#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431715395_2" to="96837172#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431715395_3" to="-96837172#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431715395_4" to="129827552#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431715395_5" to="96837172#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431715406_0" to="969765781" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431715406_1" to="706256385" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431715406_2" to="-129827561#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431715406_3" to="969765781" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431715406_4" to="706256385" fromLane="0" toLane="0" via=":1431715406_7_0" dir="r" state="m" />\n  <connection from=":1431715406_7" to="706256385" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431715406_5" to="706256385" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431715406_6" to="-129827561#1" fromLane="0" toLane="0" via=":1431715406_8_0" dir="r" state="m" />\n  <connection from=":1431715406_8" to="-129827561#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431715409_0" to="706256379#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431715409_1" to="96941844" fromLane="0" toLane="0" via=":1431715409_4_0" dir="r" state="m" />\n  <connection from=":1431715409_4" to="96941844" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431715409_2" to="96941844" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431715409_3" to="-706256379#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431715415_0" to="1454593202#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431715415_1" to="129827561#0" fromLane="0" toLane="0" via=":1431715415_6_0" dir="r" state="m" />\n  <connection from=":1431715415_6" to="129827561#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431715415_2" to="-1454593202#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431715415_3" to="1454593202#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431715415_4" to="129827561#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431715415_5" to="-1454593202#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431715421_0" to="706096643#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431715421_1" to="-129827562" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431715421_2" to="706096643#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431867100_0" to="-1457057266#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431867100_1" to="-129842397#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431867103_0" to="129842397#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431867103_1" to="-129842397#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431867103_2" to="-706087699#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431867103_3" to="129842397#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431867103_4" to="-129842397#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431867103_5" to="-706087699#1" fromLane="0" toLane="0" via=":1431867103_6_0" dir="r" state="m" />\n  <connection from=":1431867103_6" to="-706087699#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431867108_0" to="1457057267" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431867108_1" to="-129842388#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431867108_2" to="129842388#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431867108_3" to="1457057267" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431867108_4" to="-129842388#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431867108_5" to="129842388#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431867108_6" to="1457057267" fromLane="0" toLane="0" via=":1431867108_7_0" dir="r" state="m" />\n  <connection from=":1431867108_7" to="1457057267" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431867150_0" to="129842397#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431867150_1" to="-129842397#3" fromLane="0" toLane="0" via=":1431867150_6_0" dir="r" state="m" />\n  <connection from=":1431867150_6" to="-129842397#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431867150_2" to="142395239#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431867150_3" to="129842397#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431867150_4" to="-129842397#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431867150_5" to="142395239#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431867160_0" to="932339002#5" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431867160_1" to="932339004" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431867160_2" to="932339002#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431867160_2" to="932339002#5" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":1431867166_0" to="-129842388#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431867166_1" to="142455592#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431867166_2" to="1457057266#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1431867166_3" to="-129842388#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1431867166_4" to="142455592#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1431867166_5" to="1457057266#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1558314095_0" to="-142395239#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1558314095_1" to="498671600" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1558314095_2" to="142395239#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1558314095_3" to="-142395239#0" fromLane="0" toLane="0" via=":1558314095_6_0" dir="r" state="m" />\n  <connection from=":1558314095_6" to="-142395239#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1558314095_4" to="498671600" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1558314095_5" to="142395239#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1558997825_0" to="970010765#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1558997825_1" to="706094742" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1558997825_2" to="970010765#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1559016903_0" to="95809262#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1559016903_1" to="-142455600#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1559016903_2" to="-95809262#0" fromLane="0" toLane="0" via=":1559016903_12_0" dir="r" state="m" />\n  <connection from=":1559016903_12" to="-95809262#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1559016903_3" to="142455600#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1559016903_4" to="95809262#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1559016903_5" to="-142455600#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1559016903_6" to="-95809262#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1559016903_7" to="142455600#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1559016903_8" to="95809262#1" fromLane="0" toLane="0" via=":1559016903_13_0" dir="r" state="m" />\n  <connection from=":1559016903_13" to="95809262#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1559016903_9" to="-142455600#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1559016903_10" to="-95809262#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1559016903_11" to="142455600#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1559016904_0" to="148941242#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1559016904_1" to="142455595" fromLane="0" toLane="0" via=":1559016904_6_0" dir="r" state="m" />\n  <connection from=":1559016904_6" to="142455595" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1559016904_2" to="-148941242#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1559016904_3" to="148941242#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1559016904_4" to="142455595" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1559016904_5" to="-148941242#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1559016905_0" to="-142455595" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1559016905_1" to="-142455600#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1559016905_2" to="142455600#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1559016905_3" to="-142455595" fromLane="0" toLane="0" via=":1559016905_6_0" dir="r" state="m" />\n  <connection from=":1559016905_6" to="-142455595" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1559016905_4" to="-142455600#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1559016905_5" to="142455600#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1559016909_0" to="-155690741#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1559016909_1" to="148941242#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1559016909_2" to="142455590" fromLane="0" toLane="0" via=":1559016909_12_0" dir="r" state="m" />\n  <connection from=":1559016909_12" to="142455590" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1559016909_3" to="-148941242#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1559016909_4" to="-155690741#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1559016909_5" to="148941242#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1559016909_6" to="142455590" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1559016909_7" to="-148941242#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1559016909_8" to="-155690741#2" fromLane="0" toLane="0" via=":1559016909_13_0" dir="r" state="m" />\n  <connection from=":1559016909_13" to="-155690741#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1559016909_9" to="148941242#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1559016909_10" to="142455590" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1559016909_11" to="-148941242#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1559016916_0" to="148941242#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1559016916_1" to="142455599" fromLane="0" toLane="0" via=":1559016916_4_0" dir="r" state="m" />\n  <connection from=":1559016916_4" to="142455599" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1559016916_2" to="142455599" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1559016916_3" to="-148941242#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1559016918_0" to="706256386" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1559016918_1" to="142455600#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1559016918_2" to="706256386" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1559016918_3" to="142455600#0" fromLane="0" toLane="0" via=":1559016918_4_0" dir="r" state="m" />\n  <connection from=":1559016918_4" to="142455600#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1559029797_0" to="84842974#12" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1559029797_1" to="84842974#12" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1559029797_1" to="84842974#12" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":1559029797_1" to="84842974#12" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":1559029798_0" to="142456544" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1559029798_1" to="706097788#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1639088623_0" to="151072453#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1639088623_1" to="151072451#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1639088623_2" to="151072453#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1639088658_0" to="151072453#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1639088658_1" to="1218747873" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1639088658_2" to="-243079757#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1639088658_3" to="151072453#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1639088658_4" to="1218747873" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1639088658_5" to="-243079757#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1639088658_6" to="151072453#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666322_0" to="706256380#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666322_1" to="155690968" fromLane="0" toLane="0" via=":1680666322_6_0" dir="r" state="m" />\n  <connection from=":1680666322_6" to="155690968" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666322_2" to="-706256380#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666322_3" to="706256380#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666322_4" to="155690968" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666322_5" to="-706256380#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666463_0" to="-129842388#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666463_1" to="155691088#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666463_2" to="129842388#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666463_3" to="-155691088#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666463_4" to="-129842388#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666463_5" to="155691088#1" fromLane="0" toLane="0" via=":1680666463_12_0" dir="r" state="m" />\n  <connection from=":1680666463_12" to="155691088#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666463_6" to="129842388#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666463_7" to="-155691088#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666463_8" to="-129842388#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666463_9" to="155691088#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666463_10" to="129842388#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666463_11" to="-155691088#0" fromLane="0" toLane="0" via=":1680666463_13_0" dir="r" state="m" />\n  <connection from=":1680666463_13" to="-155691088#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666468_0" to="129827554#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666468_1" to="155690741#0" fromLane="0" toLane="0" via=":1680666468_6_0" dir="r" state="m" />\n  <connection from=":1680666468_6" to="155690741#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666468_2" to="-129827554#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666468_3" to="129827554#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666468_4" to="155690741#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666468_5" to="-129827554#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666546_0" to="-129827552#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666546_1" to="129827552#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666546_2" to="155690791#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666546_3" to="-129827552#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666546_4" to="129827552#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666546_5" to="155690791#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666686_0" to="-498671483" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666686_1" to="706256380#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666686_2" to="155690340" fromLane="0" toLane="0" via=":1680666686_12_0" dir="r" state="m" />\n  <connection from=":1680666686_12" to="155690340" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666686_3" to="-706256380#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666686_4" to="-498671483" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666686_5" to="706256380#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666686_6" to="155690340" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666686_7" to="-706256380#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666686_8" to="-498671483" fromLane="0" toLane="0" via=":1680666686_13_0" dir="r" state="m" />\n  <connection from=":1680666686_13" to="-498671483" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666686_9" to="706256380#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666686_10" to="155690340" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666686_11" to="-706256380#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666775_0" to="932339002#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666775_1" to="932339006" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666775_2" to="932339002#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666775_2" to="932339002#4" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":1680666827_0" to="-155691084#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666827_1" to="155691015#0" fromLane="0" toLane="0" via=":1680666827_6_0" dir="r" state="m" />\n  <connection from=":1680666827_6" to="155691015#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666827_2" to="155691084#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666827_3" to="-155691084#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666827_4" to="155691015#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666827_5" to="155691084#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666891_0" to="-155691084#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680666891_1" to="155691022#0" fromLane="0" toLane="0" via=":1680666891_6_0" dir="r" state="m" />\n  <connection from=":1680666891_6" to="155691022#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666891_2" to="155691084#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666891_3" to="-155691084#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680666891_4" to="155691022#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680666891_5" to="155691084#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680667029_0" to="-155690878#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680667029_1" to="155690878#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680667029_2" to="155691032" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680667029_3" to="-155690878#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680667029_4" to="155690878#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680667029_5" to="155691032" fromLane="0" toLane="0" via=":1680667029_6_0" dir="r" state="m" />\n  <connection from=":1680667029_6" to="155691032" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680667079_0" to="-155691015#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680667079_1" to="-155690734#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680667079_2" to="155691015#1" fromLane="0" toLane="0" via=":1680667079_12_0" dir="r" state="m" />\n  <connection from=":1680667079_12" to="155691015#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680667079_3" to="155690734#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680667079_4" to="-155691015#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680667079_5" to="-155690734#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680667079_6" to="155691015#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680667079_7" to="155690734#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680667079_8" to="-155691015#0" fromLane="0" toLane="0" via=":1680667079_13_0" dir="r" state="m" />\n  <connection from=":1680667079_13" to="-155691015#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680667079_9" to="-155690734#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680667079_10" to="155691015#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680667079_11" to="155690734#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680667164_0" to="-155691015#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680667164_1" to="-155690812#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680667164_2" to="155690812#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680667164_3" to="-155691015#1" fromLane="0" toLane="0" via=":1680667164_6_0" dir="r" state="m" />\n  <connection from=":1680667164_6" to="-155691015#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680667164_4" to="-155690812#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680667164_5" to="155690812#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680667181_0" to="95533432#5" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680667181_1" to="-95533432#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680667181_2" to="-155690528" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680667181_3" to="95533432#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680667181_4" to="-95533432#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680667181_5" to="-155690528" fromLane="0" toLane="0" via=":1680667181_6_0" dir="r" state="m" />\n  <connection from=":1680667181_6" to="-155690528" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1680667730_0" to="990212966#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680667730_1" to="-155690513" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680667730_2" to="990212966#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680668223_0" to="970323546#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680668223_1" to="-95542480" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680668223_2" to="970323546#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1680668223_2" to="970323546#1" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":1680668226_0" to="65080358#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680668226_1" to="243314395" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1680668226_2" to="65080358#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1966386254_0" to="706093508#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1966386254_1" to="706093504#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1966386254_2" to="-931354569#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1966386254_3" to="706093508#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1966386254_4" to="706093504#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1966399248_0" to="706104219#9" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1966399248_1" to="-243188491" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1966399248_2" to="-706104219#8" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1966399248_3" to="-96941842#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1966399248_4" to="706104219#9" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1966399248_5" to="-243188491" fromLane="0" toLane="0" via=":1966399248_12_0" dir="r" state="m" />\n  <connection from=":1966399248_12" to="-243188491" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1966399248_6" to="-706104219#8" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1966399248_7" to="-96941842#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1966399248_8" to="706104219#9" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1966399248_9" to="-243188491" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1966399248_10" to="-706104219#8" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1966399248_11" to="-96941842#3" fromLane="0" toLane="0" via=":1966399248_13_0" dir="r" state="m" />\n  <connection from=":1966399248_13" to="-96941842#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1966404349_0" to="-96941857#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1966404349_1" to="498671395#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1966404349_2" to="-154097280" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1966404349_3" to="-96941857#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1966404349_4" to="498671395#1" fromLane="0" toLane="0" via=":1966404349_7_0" dir="r" state="m" />\n  <connection from=":1966404349_7" to="498671395#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":1966404349_5" to="498671395#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1966404349_6" to="-154097280" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1966412828_0" to="101706795#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":1966412828_1" to="84842974#17" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1966412828_1" to="84842974#17" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":1966412828_1" to="84842974#17" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":1966412828_4" to="101706795#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":1966412828_5" to="84842974#17" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":1966412828_5" to="84842974#17" fromLane="1" toLane="2" dir="r" state="M" />\n  <connection from=":2052489649_0" to="706520617#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2052489649_1" to="-706520617#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2052489649_2" to="706520617#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2052489649_3" to="-706520617#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2052497441_0" to="194743656#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2052497441_1" to="706093506#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2052522376_0" to="84842974#7" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2052522376_1" to="84842974#7" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2052522376_1" to="84842974#7" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":2052522376_1" to="84842974#7" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":2052547074_0" to="706256377#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2052547074_1" to="706096640#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2052547074_2" to="-706256377#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2052547074_3" to="706256377#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2052547074_4" to="706096640#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2052547074_5" to="706096640#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2052547074_6" to="-706256377#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2052547075_0" to="154707523#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2052547075_1" to="706256377#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2052547075_2" to="-706256377#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2052547075_3" to="154707523#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2052547075_4" to="706256377#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2052547075_5" to="-706256377#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2052547075_6" to="154707523#1" fromLane="0" toLane="0" via=":2052547075_7_0" dir="r" state="m" />\n  <connection from=":2052547075_7" to="154707523#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2505176020_0" to="970010767#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2505176020_1" to="-243314310#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2505176020_2" to="970010767#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2505176027_0" to="-243314467#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2505176175_0" to="1218225518#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2505176175_1" to="-1218747872" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2505176175_2" to="1218225518#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2505176175_2" to="1218225518#1" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":2505176258_0" to="970323546#7" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2505176258_1" to="-66984610#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2505176258_2" to="66984610#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2505176258_3" to="970323546#7" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2505176258_3" to="970323546#7" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":2505176258_5" to="-66984610#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2505176258_6" to="66984610#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2505176258_7" to="970323546#7" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":2505176265_0" to="706105952#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2505176265_1" to="-1218747879" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2505176265_2" to="706105952#0" fromLane="0" toLane="0" via=":2505176265_3_0" dir="r" state="m" />\n  <connection from=":2505176265_3" to="706105952#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2505176388_0" to="66984613#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2505176388_1" to="243314186" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2505176388_2" to="-96713525" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2505176388_3" to="66984613#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2505176388_4" to="243314186" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2505176388_5" to="-96713525" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2505176388_6" to="66984613#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2505176494_0" to="970323546#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2505176494_1" to="-243079673" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2505176494_2" to="970323546#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2505176494_2" to="970323546#3" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":2505176514_0" to="970323546#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2505176514_1" to="243079775" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2505176514_2" to="970323546#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2505176514_2" to="970323546#2" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":2505176697_0" to="-95809258" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2505176697_1" to="243188765#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2505176697_2" to="96941841#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2505176697_3" to="-95809258" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2505176697_4" to="243188765#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2505176697_5" to="96941841#0" fromLane="0" toLane="0" via=":2505176697_6_0" dir="r" state="m" />\n  <connection from=":2505176697_6" to="96941841#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506085476_0" to="95533432#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085476_1" to="-243188318" fromLane="0" toLane="0" via=":2506085476_6_0" dir="r" state="m" />\n  <connection from=":2506085476_6" to="-243188318" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506085476_2" to="-95533432#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506085476_3" to="95533432#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506085476_4" to="-243188318" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506085476_5" to="-95533432#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085493_0" to="-243188607#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085493_1" to="-243188458" fromLane="0" toLane="0" via=":2506085493_6_0" dir="r" state="m" />\n  <connection from=":2506085493_6" to="-243188458" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506085493_2" to="243188607#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506085493_3" to="-243188607#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506085493_4" to="-243188458" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506085493_5" to="243188607#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085496_0" to="243188064#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506085496_1" to="40264697#7" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085496_1" to="40264697#7" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":2506085508_0" to="-706094742" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506085508_1" to="-706094740#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506085508_2" to="243188607#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506085508_3" to="-706094742" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085508_4" to="-706094740#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085508_5" to="243188607#0" fromLane="0" toLane="0" via=":2506085508_6_0" dir="r" state="m" />\n  <connection from=":2506085508_6" to="243188607#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506085525_0" to="969757851#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506085525_1" to="-243188024" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506085525_2" to="969757851#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085555_0" to="243188741#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085555_1" to="243188499" fromLane="0" toLane="0" via=":2506085555_4_0" dir="r" state="m" />\n  <connection from=":2506085555_4" to="243188499" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506085555_2" to="243188499" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506085555_3" to="-243188741#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085628_0" to="931354569#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506085628_1" to="-931354569#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506085628_2" to="931354569#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085628_3" to="-931354569#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085631_0" to="-498671616" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085631_1" to="498671639" fromLane="0" toLane="0" via=":2506085631_7_0" dir="r" state="m" />\n  <connection from=":2506085631_7" to="498671639" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506085631_2" to="498671639" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506085631_3" to="243188351#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085631_4" to="-498671616" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506085631_5" to="498671639" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085631_6" to="243188351#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506085690_0" to="931354569#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506085690_1" to="243188561#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085690_2" to="-969757849#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506085690_3" to="931354569#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085690_4" to="243188561#0" fromLane="0" toLane="0" via=":2506085690_9_0" dir="r" state="m" />\n  <connection from=":2506085690_9" to="243188561#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506085690_5" to="-969757849#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506085690_6" to="931354569#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506085690_7" to="243188561#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506085690_8" to="-969757849#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085849_0" to="911123067" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085849_1" to="684714761#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506085849_2" to="911123067" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506085849_3" to="684714761#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506085849_4" to="-243188447" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506086181_0" to="706104219#16" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506086181_1" to="-706104219#15" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506086181_2" to="706104219#16" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506086181_3" to="-706104219#15" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506086743_0" to="194743656#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506086743_1" to="-243188536#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506086743_2" to="194743656#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506086795_0" to="706104219#8" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506086795_1" to="243188563" fromLane="0" toLane="0" via=":2506086795_6_0" dir="r" state="m" />\n  <connection from=":2506086795_6" to="243188563" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506086795_2" to="-706104219#7" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506086795_3" to="706104219#8" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506086795_4" to="243188563" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506086795_5" to="-706104219#7" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506086831_0" to="706104219#7" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506086831_1" to="-706104219#6" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506086831_2" to="498671388#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506086831_3" to="706104219#7" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506086831_4" to="-706104219#6" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506086831_5" to="498671388#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506086831_6" to="706104219#7" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506086831_7" to="-706104219#6" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506086831_8" to="498671388#0" fromLane="0" toLane="0" via=":2506086831_9_0" dir="r" state="m" />\n  <connection from=":2506086831_9" to="498671388#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506087136_0" to="72016745#5" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087136_1" to="243188585" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087136_2" to="-72016745#4" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":2506087136_3" to="-498671569" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087136_4" to="72016745#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087136_4" to="72016745#5" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":2506087136_6" to="243188585" fromLane="0" toLane="0" via=":2506087136_14_0" dir="r" state="m" />\n  <connection from=":2506087136_14" to="243188585" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506087136_7" to="-72016745#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087136_8" to="-498671569" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087136_9" to="72016745#5" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":2506087136_10" to="243188585" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087136_11" to="-72016745#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087136_11" to="-72016745#4" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":2506087136_13" to="-498671569" fromLane="0" toLane="0" via=":2506087136_15_0" dir="r" state="m" />\n  <connection from=":2506087136_15" to="-498671569" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506087304_0" to="243188103" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087304_1" to="-243188536#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087304_2" to="243188536#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087304_3" to="243188103" fromLane="0" toLane="0" via=":2506087304_6_0" dir="r" state="m" />\n  <connection from=":2506087304_6" to="243188103" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506087304_4" to="-243188536#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087304_5" to="243188536#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506087432_0" to="498671536" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087432_1" to="95809261#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087432_2" to="95809261#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087432_3" to="-1365892748#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506087522_0" to="243188646#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087522_1" to="-498671673" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506087522_2" to="243188646#1" fromLane="0" toLane="0" via=":2506087522_3_0" dir="r" state="m" />\n  <connection from=":2506087522_3" to="243188646#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506087563_0" to="243188646#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087563_1" to="1316153439" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506087563_2" to="243188536#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087563_3" to="243188646#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087709_0" to="243188174#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087732_0" to="-243188174#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506087732_1" to="-706095306" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087810_0" to="706093506#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087810_1" to="243188527" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506087863_0" to="95809268#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087863_1" to="706103887" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087863_2" to="95809268#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087863_3" to="706103887" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087863_4" to="95809268#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506087889_0" to="95809261#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087889_1" to="-706103887" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087889_2" to="95809261#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087955_0" to="95809268#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087955_1" to="243188505" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087955_2" to="95809268#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087962_0" to="-243188208" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087962_1" to="706104217#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087962_2" to="-706104217#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087962_3" to="-243188208" fromLane="0" toLane="0" via=":2506087962_6_0" dir="r" state="m" />\n  <connection from=":2506087962_6" to="-243188208" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506087962_4" to="706104217#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087962_5" to="-706104217#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506087968_0" to="-243188505" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087968_1" to="95809261#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087968_2" to="95809261#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087987_0" to="706104217#20" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087987_1" to="-243188576" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087987_2" to="-706104217#19" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506087987_3" to="706095306" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087987_4" to="706104217#20" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087987_5" to="-243188576" fromLane="0" toLane="0" via=":2506087987_12_0" dir="r" state="m" />\n  <connection from=":2506087987_12" to="-243188576" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506087987_6" to="-706104217#19" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087987_7" to="706095306" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087987_8" to="706104217#20" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506087987_9" to="-243188576" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506087987_10" to="-706104217#19" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506087987_11" to="706095306" fromLane="0" toLane="0" via=":2506087987_13_0" dir="r" state="m" />\n  <connection from=":2506087987_13" to="706095306" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088002_0" to="-243188563" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088002_1" to="706104217#9" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088002_2" to="-706104217#8" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088002_3" to="-243188563" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088002_4" to="706104217#9" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088002_5" to="-706104217#8" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088023_0" to="-122540020#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088023_1" to="122540020#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088023_2" to="243188139" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088023_3" to="-122540020#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088023_4" to="122540020#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088023_5" to="243188139" fromLane="0" toLane="0" via=":2506088023_6_0" dir="r" state="m" />\n  <connection from=":2506088023_6" to="243188139" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088024_0" to="243188491" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088024_1" to="706104217#11" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088024_2" to="-330615760" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088024_3" to="-706104217#10" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088024_4" to="243188491" fromLane="0" toLane="0" dir="L" state="M" />\n  <connection from=":2506088024_5" to="706104217#11" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088024_6" to="-330615760" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088024_7" to="-706104217#10" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088024_8" to="243188491" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088024_9" to="706104217#11" fromLane="0" toLane="0" via=":2506088024_16_0" dir="r" state="m" />\n  <connection from=":2506088024_16" to="706104217#11" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088024_10" to="-330615760" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088024_11" to="-706104217#10" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088024_12" to="243188491" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088024_13" to="706104217#11" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088024_14" to="-330615760" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088024_15" to="-706104217#10" fromLane="0" toLane="0" via=":2506088024_17_0" dir="r" state="m" />\n  <connection from=":2506088024_17" to="-706104217#10" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088084_0" to="706104217#13" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088084_1" to="-706104217#12" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088084_2" to="706104217#13" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088084_3" to="-706104217#12" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088243_0" to="1257722738#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088243_1" to="1257722736#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088274_0" to="-122540020#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088274_1" to="122540020#5" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088274_2" to="243188167" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088274_3" to="-122540020#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088274_4" to="122540020#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088274_5" to="243188167" fromLane="0" toLane="0" via=":2506088274_6_0" dir="r" state="m" />\n  <connection from=":2506088274_6" to="243188167" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088476_0" to="-1558734968" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088476_1" to="-498671331" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088476_2" to="-706095305" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088476_3" to="-1558734968" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088476_4" to="-498671331" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088476_5" to="-706095305" fromLane="0" toLane="0" via=":2506088476_6_0" dir="r" state="m" />\n  <connection from=":2506088476_6" to="-706095305" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088491_0" to="243188389#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088491_1" to="-243188389#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088491_2" to="-498671589" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088491_3" to="243188389#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088491_4" to="-243188389#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088491_5" to="-498671589" fromLane="0" toLane="0" via=":2506088491_6_0" dir="r" state="m" />\n  <connection from=":2506088491_6" to="-498671589" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088492_0" to="-243188508#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088492_1" to="243188552#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088492_2" to="-243188552#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088492_3" to="-243188508#3" fromLane="0" toLane="0" via=":2506088492_6_0" dir="r" state="m" />\n  <connection from=":2506088492_6" to="-243188508#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088492_4" to="243188552#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088492_5" to="-243188552#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088494_0" to="498671331" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088494_1" to="243188389#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088494_2" to="-243188389#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088494_3" to="498671331" fromLane="0" toLane="0" via=":2506088494_6_0" dir="r" state="m" />\n  <connection from=":2506088494_6" to="498671331" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088494_4" to="243188389#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088494_5" to="-243188389#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088518_0" to="1552748146" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088518_1" to="243188508#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088518_2" to="243188565" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088518_3" to="1552748146" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088518_4" to="243188508#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088518_5" to="243188565" fromLane="0" toLane="0" via=":2506088518_6_0" dir="r" state="m" />\n  <connection from=":2506088518_6" to="243188565" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088528_0" to="-122540020#5" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088528_1" to="122540020#6" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088528_2" to="243188092" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088528_3" to="-122540020#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088528_4" to="122540020#6" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088528_5" to="243188092" fromLane="0" toLane="0" via=":2506088528_6_0" dir="r" state="m" />\n  <connection from=":2506088528_6" to="243188092" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088529_0" to="243188641#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088529_1" to="243188641#1" fromLane="0" toLane="0" via=":2506088529_2_0" dir="r" state="m" />\n  <connection from=":2506088529_2" to="243188641#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088612_0" to="243188452#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088612_1" to="243188594#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088612_2" to="243188452#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088612_3" to="243188594#1" fromLane="0" toLane="0" via=":2506088612_4_0" dir="r" state="m" />\n  <connection from=":2506088612_4" to="243188594#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088715_0" to="243188670#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088715_0" to="243188670#1" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":2506088715_2" to="243188452#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088938_0" to="-243188786#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506088938_1" to="-243188085" fromLane="0" toLane="0" via=":2506088938_6_0" dir="r" state="m" />\n  <connection from=":2506088938_6" to="-243188085" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088938_2" to="243188786#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088938_3" to="-243188786#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506088938_4" to="-243188085" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506088938_5" to="243188786#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506089058_0" to="-243188786#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506089058_1" to="-243188361" fromLane="0" toLane="0" via=":2506089058_6_0" dir="r" state="m" />\n  <connection from=":2506089058_6" to="-243188361" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506089058_2" to="243188786#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506089058_3" to="-243188786#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506089058_4" to="-243188361" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506089058_5" to="243188786#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506089309_0" to="-980451721#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506089309_1" to="980451721#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506089309_2" to="-980451721#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506089309_3" to="980451721#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506089487_0" to="-243188786#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506089487_1" to="-243188161" fromLane="0" toLane="0" via=":2506089487_6_0" dir="r" state="m" />\n  <connection from=":2506089487_6" to="-243188161" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506089487_2" to="243188786#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506089487_3" to="-243188786#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506089487_4" to="-243188161" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506089487_5" to="243188786#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506089545_0" to="-497290263#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506089545_1" to="243188644" fromLane="0" toLane="0" via=":2506089545_6_0" dir="r" state="m" />\n  <connection from=":2506089545_6" to="243188644" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506089545_2" to="497290263#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506089545_3" to="-497290263#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506089545_4" to="243188644" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506089545_5" to="497290263#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506089600_0" to="-497290263#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506089600_1" to="-243188627" fromLane="0" toLane="0" via=":2506089600_6_0" dir="r" state="m" />\n  <connection from=":2506089600_6" to="-243188627" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506089600_2" to="497290263#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506089600_3" to="-497290263#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506089600_4" to="-243188627" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506089600_5" to="497290263#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506089779_0" to="243188298" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506089779_1" to="243079656" fromLane="0" toLane="0" via=":2506089779_5_0" dir="r" state="m" />\n  <connection from=":2506089779_5" to="243079656" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506089779_2" to="243079656" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506089779_3" to="243188298" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506089779_4" to="243079656" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506089901_0" to="243188503#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506089901_1" to="-243188196" fromLane="0" toLane="0" via=":2506089901_6_0" dir="r" state="m" />\n  <connection from=":2506089901_6" to="-243188196" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506089901_2" to="-243188503#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506089901_3" to="243188503#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2506089901_4" to="-243188196" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506089901_5" to="-243188503#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2506090118_0" to="-243188389#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2506090118_1" to="243188243" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400591_0" to="1547102487#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400591_1" to="-243314310#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400591_2" to="243314310#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400591_3" to="1547102487#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400591_4" to="-243314310#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400591_5" to="243314310#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400591_6" to="1547102487#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400593_0" to="1547102487#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400593_1" to="970010767#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400604_0" to="-243314436#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400604_1" to="243314467#0" fromLane="0" toLane="0" via=":2507400604_6_0" dir="r" state="m" />\n  <connection from=":2507400604_6" to="243314467#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400604_2" to="243314436#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400604_3" to="-243314436#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400604_4" to="243314467#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400604_5" to="243314436#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400609_0" to="-243314467#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400609_1" to="498671542" fromLane="0" toLane="0" via=":2507400609_6_0" dir="r" state="m" />\n  <connection from=":2507400609_6" to="498671542" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400609_2" to="243314467#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400609_3" to="-243314467#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400609_4" to="498671542" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400609_5" to="243314467#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400638_0" to="969757855#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400638_1" to="-243314192" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400638_2" to="243314312" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400638_3" to="969757855#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400638_4" to="-243314192" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400638_5" to="243314312" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400638_6" to="969757855#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400639_0" to="1270957861" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400639_1" to="243314192" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400639_2" to="1270957861" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400639_2" to="1270957861" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":2507400689_0" to="66984613#6" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400689_1" to="243314073" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400689_2" to="-243314112" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400689_3" to="66984613#6" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400689_4" to="243314073" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400689_5" to="-243314112" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400689_6" to="66984613#6" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400690_0" to="970323546#6" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400690_1" to="-243314073" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400690_2" to="970323546#6" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400690_2" to="970323546#6" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":2507400696_0" to="66984613#5" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400696_1" to="243314213" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400696_2" to="-243314378#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400696_3" to="66984613#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400696_4" to="243314213" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400696_5" to="-243314378#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400696_6" to="66984613#5" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400698_0" to="970323546#5" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400698_1" to="-243314213" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400698_2" to="970323546#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400698_2" to="970323546#5" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":2507400748_0" to="65080358#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400748_1" to="-243314103" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400748_2" to="65080358#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400769_0" to="95542482#11" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400769_1" to="243314137" fromLane="0" toLane="0" via=":2507400769_6_0" dir="r" state="m" />\n  <connection from=":2507400769_6" to="243314137" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400769_2" to="-95542482#10" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400769_3" to="95542482#11" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400769_4" to="243314137" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400769_5" to="-95542482#10" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400772_0" to="95542482#13" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400772_1" to="-243314160" fromLane="0" toLane="0" via=":2507400772_6_0" dir="r" state="m" />\n  <connection from=":2507400772_6" to="-243314160" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400772_2" to="-95542482#12" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400772_3" to="95542482#13" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400772_4" to="-243314160" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400772_5" to="-95542482#12" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400773_0" to="81350006#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400773_1" to="-81350006#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":2507400773_2" to="-243314177" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":2507400773_3" to="81350006#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400773_4" to="-81350006#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":2507400773_5" to="-243314177" fromLane="0" toLane="0" via=":2507400773_6_0" dir="r" state="m" />\n  <connection from=":2507400773_6" to="-243314177" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":3125684630_0" to="40264696#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":3125684630_0" to="40264696#3" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":3125684630_0" to="40264696#3" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":3125684630_3" to="40264696#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":3376091528_0" to="330615844" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":3376091528_1" to="-330615829#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4085816302_0" to="84842974#9" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4085816302_1" to="-706104219#16" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4085816302_2" to="84842974#9" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4085816302_2" to="84842974#9" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":4085816302_2" to="84842974#9" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":4085816302_5" to="-706104219#16" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4085816302_6" to="84842974#9" fromLane="0" toLane="2" dir="r" state="M" />\n  <connection from=":4488358856_0" to="-452036923#5" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4488358856_1" to="452036923#6" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4488358856_2" to="-452037066" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4488358856_3" to="-452036923#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4488358856_4" to="452036923#6" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4488358856_5" to="-452037066" fromLane="0" toLane="0" via=":4488358856_6_0" dir="r" state="m" />\n  <connection from=":4488358856_6" to="-452037066" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4488358857_0" to="452036923#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4488358857_1" to="452037067" fromLane="0" toLane="0" via=":4488358857_6_0" dir="r" state="m" />\n  <connection from=":4488358857_6" to="452037067" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4488358857_2" to="-452036923#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4488358857_3" to="452036923#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4488358857_4" to="452037067" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4488358857_5" to="-452036923#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4488358858_0" to="-452036923#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4488358858_1" to="452036923#5" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4488358858_2" to="-452037067" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4488358858_3" to="-452036923#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4488358858_4" to="452036923#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4488358858_5" to="-452037067" fromLane="0" toLane="0" via=":4488358858_6_0" dir="r" state="m" />\n  <connection from=":4488358858_6" to="-452037067" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4488366488_0" to="452036923#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4488366488_1" to="-243188225" fromLane="0" toLane="0" via=":4488366488_6_0" dir="r" state="m" />\n  <connection from=":4488366488_6" to="-243188225" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4488366488_2" to="-452036923#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4488366488_3" to="452036923#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4488366488_4" to="-243188225" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4488366488_5" to="-452036923#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4488368090_0" to="-452036923#6" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4488368090_1" to="452036923#7" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4488368090_2" to="-452036922" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4488368090_3" to="-452036923#6" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4488368090_4" to="452036923#7" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4488368090_5" to="-452036922" fromLane="0" toLane="0" via=":4488368090_6_0" dir="r" state="m" />\n  <connection from=":4488368090_6" to="-452036922" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":485580842_0" to="84842974#11" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":485580842_1" to="706093508#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":485580842_2" to="84842974#11" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":485580842_2" to="84842974#11" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":485580842_2" to="84842974#11" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":485580842_5" to="706093508#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4888861605_0" to="497290260" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4888861605_1" to="497290263#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4888861608_0" to="497290262" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4888861608_1" to="497290261#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4888861612_0" to="497290261#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4888861612_1" to="497290261#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4888861618_0" to="497290265" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4888861618_1" to="497290261#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4888861620_0" to="497290264" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4888861620_1" to="497290261#7" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4888862222_0" to="497290261#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4888862222_1" to="497290261#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900464580_0" to="498671389#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900464580_1" to="-96941857#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900464580_2" to="-498671389#1" fromLane="0" toLane="0" via=":4900464580_12_0" dir="r" state="m" />\n  <connection from=":4900464580_12" to="-498671389#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900464580_3" to="96941857#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900464580_4" to="498671389#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900464580_5" to="-96941857#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900464580_6" to="-498671389#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900464580_7" to="96941857#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900464580_8" to="498671389#2" fromLane="0" toLane="0" via=":4900464580_13_0" dir="r" state="m" />\n  <connection from=":4900464580_13" to="498671389#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900464580_9" to="-96941857#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900464580_10" to="-498671389#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900464580_11" to="96941857#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900464584_0" to="1204844051" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900464584_1" to="-1204844052" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900464584_2" to="-1204844052" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900464587_0" to="498671391" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900464587_1" to="530314558#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900464587_2" to="-530314558#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900464587_3" to="498671391" fromLane="0" toLane="0" via=":4900464587_6_0" dir="r" state="m" />\n  <connection from=":4900464587_6" to="498671391" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900464587_4" to="530314558#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900464587_5" to="-530314558#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900464597_0" to="706104219#5" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900464597_1" to="-706104219#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900464597_2" to="498671389#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900464597_3" to="706104219#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900464597_4" to="-706104219#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900464597_5" to="498671389#0" fromLane="0" toLane="0" via=":4900464597_6_0" dir="r" state="m" />\n  <connection from=":4900464597_6" to="498671389#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900464598_0" to="969757849#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900464598_1" to="-498671389#2" fromLane="0" toLane="0" via=":4900464598_6_0" dir="r" state="m" />\n  <connection from=":4900464598_6" to="-498671389#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900464598_2" to="-969757849#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900464598_3" to="969757849#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900464598_4" to="-498671389#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900464598_5" to="-969757849#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465031_0" to="706087699#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465031_1" to="-498671694" fromLane="0" toLane="0" via=":4900465031_6_0" dir="r" state="m" />\n  <connection from=":4900465031_6" to="-498671694" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465031_2" to="-706087699#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465031_3" to="706087699#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465031_4" to="-498671694" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465031_5" to="-706087699#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465052_0" to="65080358#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465052_1" to="-498671427" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465052_2" to="65080358#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465069_0" to="-243188765#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465069_1" to="-498671432" fromLane="0" toLane="0" via=":4900465069_6_0" dir="r" state="m" />\n  <connection from=":4900465069_6" to="-498671432" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465069_2" to="243188765#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465069_3" to="-243188765#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465069_4" to="-498671432" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465069_5" to="243188765#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465071_0" to="155691022#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465071_1" to="-498671433" fromLane="0" toLane="0" via=":4900465071_6_0" dir="r" state="m" />\n  <connection from=":4900465071_6" to="-498671433" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465071_2" to="-155691022#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465071_3" to="155691022#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465071_4" to="-498671433" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465071_5" to="-155691022#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465090_0" to="498671330#1" fromLane="0" toLane="0" via=":4900465090_3_0" dir="r" state="m" />\n  <connection from=":4900465090_3" to="498671330#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465090_1" to="498671330#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465090_2" to="498671539" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465099_0" to="706093505#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465099_1" to="706093505#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465099_2" to="-498671442" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465166_0" to="498671465" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465166_1" to="-1365935945#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465166_2" to="498671393" fromLane="0" toLane="0" via=":4900465166_12_0" dir="r" state="m" />\n  <connection from=":4900465166_12" to="498671393" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465166_3" to="1365935945#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465166_4" to="498671465" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465166_5" to="-1365935945#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465166_6" to="498671393" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465166_7" to="1365935945#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465166_8" to="498671465" fromLane="0" toLane="0" via=":4900465166_13_0" dir="r" state="m" />\n  <connection from=":4900465166_13" to="498671465" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465166_9" to="-1365935945#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465166_10" to="498671393" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465166_11" to="1365935945#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465208_0" to="706256379#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465208_1" to="498671482" fromLane="0" toLane="0" via=":4900465208_6_0" dir="r" state="m" />\n  <connection from=":4900465208_6" to="498671482" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465208_2" to="-706256379#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465208_3" to="706256379#5" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465208_4" to="498671482" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465208_5" to="-706256379#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465209_0" to="706256377#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465209_1" to="-706256377#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465209_2" to="-498671482" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465209_3" to="706256377#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465209_4" to="-706256377#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465209_5" to="-498671482" fromLane="0" toLane="0" via=":4900465209_6_0" dir="r" state="m" />\n  <connection from=":4900465209_6" to="-498671482" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465219_0" to="95533432#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465219_1" to="498671679" fromLane="0" toLane="0" via=":4900465219_6_0" dir="r" state="m" />\n  <connection from=":4900465219_6" to="498671679" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465219_2" to="-95533432#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465219_3" to="95533432#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465219_4" to="498671679" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465219_5" to="-95533432#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465244_0" to="-155690480#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465244_1" to="-498671497" fromLane="0" toLane="0" via=":4900465244_6_0" dir="r" state="m" />\n  <connection from=":4900465244_6" to="-498671497" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465244_2" to="155690480#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465244_3" to="-155690480#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465244_4" to="-498671497" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465244_5" to="155690480#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465288_0" to="1559802495#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465288_1" to="243188552#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465288_2" to="-243188552#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465288_3" to="1559802495#0" fromLane="0" toLane="0" via=":4900465288_6_0" dir="r" state="m" />\n  <connection from=":4900465288_6" to="1559802495#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465288_4" to="243188552#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465288_5" to="-243188552#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465363_0" to="95809268#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465363_1" to="1168168079" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465363_2" to="-498671536" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465363_3" to="95809268#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465363_4" to="1168168079" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465364_0" to="706093506#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465364_1" to="706093506#2" fromLane="0" toLane="0" via=":4900465364_2_0" dir="r" state="m" />\n  <connection from=":4900465364_2" to="706093506#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465377_0" to="-243314248#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465377_1" to="243314248#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465377_2" to="-498671542" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465377_3" to="-243314248#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465377_4" to="243314248#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465377_5" to="-498671542" fromLane="0" toLane="0" via=":4900465377_6_0" dir="r" state="m" />\n  <connection from=":4900465377_6" to="-498671542" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465441_0" to="243314378#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465441_1" to="-243314378#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465441_2" to="-498671564" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465441_3" to="243314378#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465441_4" to="-243314378#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465441_5" to="-498671564" fromLane="0" toLane="0" via=":4900465441_6_0" dir="r" state="m" />\n  <connection from=":4900465441_6" to="-498671564" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465471_0" to="706093509#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465471_1" to="498671576#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465471_2" to="706093509#1" fromLane="0" toLane="0" via=":4900465471_3_0" dir="r" state="m" />\n  <connection from=":4900465471_3" to="706093509#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465472_0" to="-498671576#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465472_1" to="243188064#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465472_2" to="498671576#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465472_3" to="-498671576#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465472_4" to="243188064#1" fromLane="0" toLane="0" via=":4900465472_7_0" dir="r" state="m" />\n  <connection from=":4900465472_7" to="243188064#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465472_5" to="243188064#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465472_6" to="498671576#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465473_0" to="-498671576#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465473_1" to="-243188465#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465473_2" to="243188465#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465473_3" to="-498671576#1" fromLane="0" toLane="0" via=":4900465473_6_0" dir="r" state="m" />\n  <connection from=":4900465473_6" to="-498671576#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465473_4" to="-243188465#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465473_5" to="243188465#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465513_0" to="684714761#2" fromLane="0" toLane="0" via=":4900465513_2_0" dir="r" state="m" />\n  <connection from=":4900465513_2" to="684714761#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465513_1" to="684714761#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465517_0" to="706104217#18" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465517_1" to="498671589" fromLane="0" toLane="0" via=":4900465517_6_0" dir="r" state="m" />\n  <connection from=":4900465517_6" to="498671589" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465517_2" to="-706104217#17" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465517_3" to="706104217#18" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465517_4" to="498671589" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465517_5" to="-706104217#17" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465706_0" to="-498671328#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465706_1" to="1552748564#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465706_2" to="-1552748564#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465706_3" to="-498671328#1" fromLane="0" toLane="0" via=":4900465706_6_0" dir="r" state="m" />\n  <connection from=":4900465706_6" to="-498671328#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465706_4" to="1552748564#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465706_5" to="-1552748564#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465707_0" to="185951074#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465707_1" to="-185951074#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465707_2" to="185951074#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465707_3" to="-185951074#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465949_0" to="705855017#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465949_1" to="-155691084#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465949_2" to="155691084#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465949_3" to="705855017#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465949_4" to="-155691084#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465949_5" to="155691084#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465949_6" to="705855017#1" fromLane="0" toLane="0" via=":4900465949_7_0" dir="r" state="m" />\n  <connection from=":4900465949_7" to="705855017#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465989_0" to="498671673" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465989_1" to="-243188174#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465989_2" to="243188174#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465989_3" to="498671673" fromLane="0" toLane="0" via=":4900465989_6_0" dir="r" state="m" />\n  <connection from=":4900465989_6" to="498671673" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465989_4" to="-243188174#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465989_5" to="243188174#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465999_0" to="-498671678" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465999_1" to="498671615" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900465999_2" to="498671616" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900465999_3" to="-498671678" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465999_4" to="498671615" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900465999_5" to="498671616" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900466295_0" to="498671329" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900466295_1" to="77967084#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900466297_0" to="931348308" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900466297_1" to="931348308" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900466448_0" to="-96941857#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900466448_1" to="-498671388#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900466448_2" to="96941857#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900466448_3" to="498671388#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900466448_4" to="-96941857#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900466448_5" to="-498671388#0" fromLane="0" toLane="0" via=":4900466448_12_0" dir="r" state="m" />\n  <connection from=":4900466448_12" to="-498671388#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900466448_6" to="96941857#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900466448_7" to="498671388#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900466448_8" to="-96941857#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":4900466448_9" to="-498671388#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":4900466448_10" to="96941857#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":4900466448_11" to="498671388#1" fromLane="0" toLane="0" via=":4900466448_13_0" dir="r" state="m" />\n  <connection from=":4900466448_13" to="498671388#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":5149825984_0" to="706104218" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":5149825984_1" to="452036923#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":5149825984_2" to="-72016745#5" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":5149825984_3" to="-96941849" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":5149825984_4" to="706104218" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":5149825984_5" to="452036923#0" fromLane="0" toLane="0" via=":5149825984_12_0" dir="r" state="m" />\n  <connection from=":5149825984_12" to="452036923#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":5149825984_6" to="-72016745#5" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":5149825984_7" to="-96941849" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":5149825984_8" to="706104218" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":5149825984_9" to="452036923#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":5149825984_10" to="-72016745#5" fromLane="0" toLane="1" dir="s" state="M" />\n  <connection from=":5149825984_11" to="-96941849" fromLane="0" toLane="0" via=":5149825984_13_0" dir="r" state="m" />\n  <connection from=":5149825984_13" to="-96941849" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":617234280_0" to="40264696#5" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":617234280_0" to="40264696#5" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":617234280_0" to="40264696#5" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":617234280_3" to="40264697#2" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":617234280_4" to="40264696#5" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":617234280_4" to="40264696#5" fromLane="1" toLane="1" dir="l" state="M" />\n  <connection from=":617234280_6" to="40264697#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":617234280_6" to="40264697#2" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":617234281_0" to="706097788#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":617234281_1" to="40264697#4" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":617234281_2" to="706097788#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":617234281_3" to="40264697#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":617234281_3" to="40264697#4" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":617234535_0" to="684704215#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":617234535_1" to="40264696#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":617234535_1" to="40264696#4" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":617234535_1" to="40264696#4" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":617234535_4" to="684704215#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":617234535_5" to="40264696#4" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":617234535_5" to="40264696#4" fromLane="1" toLane="2" dir="r" state="M" />\n  <connection from=":617234576_0" to="40264697#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":617234576_1" to="84842974#13" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":617234576_1" to="84842974#13" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":617234576_1" to="84842974#13" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":617234576_4" to="40264697#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":617234576_4" to="40264697#3" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":617234576_6" to="84842974#13" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":617234576_6" to="84842974#13" fromLane="1" toLane="2" dir="r" state="M" />\n  <connection from=":617234843_0" to="706093508#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":617234843_1" to="706093508#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":6634630594_0" to="706095308#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":6634630594_1" to="706104217#21" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":6634630594_2" to="-706104217#20" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":6634630594_3" to="706095308#0" fromLane="0" toLane="0" via=":6634630594_6_0" dir="r" state="m" />\n  <connection from=":6634630594_6" to="706095308#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":6634630594_4" to="706104217#21" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":6634630594_5" to="-706104217#20" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":6634630596_0" to="706095308#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":6634630596_1" to="-243188174#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":6634630596_2" to="-706095308#0" fromLane="0" toLane="0" via=":6634630596_12_0" dir="r" state="m" />\n  <connection from=":6634630596_12" to="-706095308#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":6634630596_3" to="243188174#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":6634630596_4" to="706095308#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":6634630596_5" to="-243188174#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":6634630596_6" to="-706095308#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":6634630596_7" to="243188174#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":6634630596_8" to="706095308#1" fromLane="0" toLane="0" via=":6634630596_13_0" dir="r" state="m" />\n  <connection from=":6634630596_13" to="706095308#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":6634630596_9" to="-243188174#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":6634630596_10" to="-706095308#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":6634630596_11" to="243188174#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":796747634_0" to="1270957860#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":796747634_1" to="-243188629" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":796747634_2" to="1270957860#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":796747714_0" to="990212966#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":796747714_1" to="-155690480#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":796747714_2" to="990212966#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":809537690_0" to="243079844#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":809537690_1" to="706105952#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":809537690_2" to="243079844#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":809537690_3" to="706105952#1" fromLane="0" toLane="0" via=":809537690_4_0" dir="r" state="m" />\n  <connection from=":809537690_4" to="706105952#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":809537820_0" to="151072453#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":809537820_1" to="66984604#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":809537820_2" to="151072453#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":809537820_3" to="66984604#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":8348961008_0" to="84842974#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8348961008_1" to="898440288" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8348961008_2" to="84842974#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8348961008_2" to="84842974#2" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":8348961008_2" to="84842974#2" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":8348961009_0" to="-898440288" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8348961009_1" to="-243188536#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8348961009_2" to="243188536#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8348961009_3" to="-898440288" fromLane="0" toLane="0" via=":8348961009_6_0" dir="r" state="m" />\n  <connection from=":8348961009_6" to="-898440288" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":8348961009_4" to="-243188536#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8348961009_5" to="243188536#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":8363308288_0" to="-900244003#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8363308288_1" to="900244003#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":8363308288_2" to="900244004" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8363308288_3" to="-900244003#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8363308288_4" to="900244003#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8363308288_5" to="900244004" fromLane="0" toLane="0" via=":8363308288_6_0" dir="r" state="m" />\n  <connection from=":8363308288_6" to="900244004" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":8523326825_0" to="-243314165#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8523326825_1" to="-917695719" fromLane="0" toLane="0" via=":8523326825_6_0" dir="r" state="m" />\n  <connection from=":8523326825_6" to="-917695719" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":8523326825_2" to="243314165#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8523326825_3" to="-243314165#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":8523326825_4" to="-917695719" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8523326825_5" to="243314165#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8523326826_0" to="-243314312" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8523326826_1" to="917695719" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":8523328986_0" to="151072453#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8523328986_1" to="917695557" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8523328986_2" to="151072453#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8523334139_0" to="706093505#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8523334139_1" to="917696883" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":8523344319_0" to="706093508#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8523344319_1" to="498671586" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":8523431062_0" to="706104217#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8523431062_1" to="-706104217#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":8523431062_2" to="917705248" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8523431062_3" to="706104217#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8523431062_4" to="-706104217#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8523431062_5" to="917705248" fromLane="0" toLane="0" via=":8523431062_6_0" dir="r" state="m" />\n  <connection from=":8523431062_6" to="917705248" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":8523431064_0" to="917705249" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8523431064_1" to="706104217#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8523431064_2" to="-706104217#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8523431064_3" to="917705249" fromLane="0" toLane="0" via=":8523431064_6_0" dir="r" state="m" />\n  <connection from=":8523431064_6" to="917705249" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":8523431064_4" to="706104217#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8523431064_5" to="-706104217#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":8523449430_0" to="243314378#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8523449430_1" to="-243314378#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":8523449430_2" to="917706996" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8523449430_3" to="243314378#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8523449430_4" to="-243314378#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8523449430_5" to="917706996" fromLane="0" toLane="0" via=":8523449430_6_0" dir="r" state="m" />\n  <connection from=":8523449430_6" to="917706996" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":856214470_0" to="706092523#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":856214470_1" to="706092523#2" fromLane="0" toLane="0" via=":856214470_2_0" dir="r" state="m" />\n  <connection from=":856214470_2" to="706092523#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":856214503_0" to="706092523#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":856214503_1" to="931354569#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":856214503_2" to="-931354569#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":856214503_3" to="706092523#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":856214503_4" to="931354569#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":856214503_5" to="-931354569#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":856214503_6" to="706092523#4" fromLane="0" toLane="0" via=":856214503_7_0" dir="r" state="m" />\n  <connection from=":856214503_7" to="706092523#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":856214524_0" to="706105951#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":856214524_1" to="185951074#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":856214524_2" to="-706105951#2" fromLane="0" toLane="0" via=":856214524_12_0" dir="r" state="m" />\n  <connection from=":856214524_12" to="-706105951#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":856214524_3" to="-1270957863" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":856214524_4" to="706105951#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":856214524_5" to="185951074#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":856214524_6" to="-706105951#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":856214524_7" to="-1270957863" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":856214524_8" to="706105951#3" fromLane="0" toLane="0" via=":856214524_13_0" dir="r" state="m" />\n  <connection from=":856214524_13" to="706105951#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":856214524_9" to="185951074#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":856214524_10" to="-706105951#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":856214524_11" to="-1270957863" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":856214538_0" to="969757849#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":856214538_1" to="-498671388#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":856214538_2" to="-969757849#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":856214538_3" to="1365935945#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":856214538_4" to="969757849#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":856214538_5" to="-498671388#1" fromLane="0" toLane="0" via=":856214538_12_0" dir="r" state="m" />\n  <connection from=":856214538_12" to="-498671388#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":856214538_6" to="-969757849#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":856214538_7" to="1365935945#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":856214538_8" to="969757849#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":856214538_9" to="-498671388#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":856214538_10" to="-969757849#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":856214538_11" to="1365935945#0" fromLane="0" toLane="0" via=":856214538_13_0" dir="r" state="m" />\n  <connection from=":856214538_13" to="1365935945#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":856214571_0" to="72016745#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":856214571_0" to="72016745#0" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":856214571_2" to="65080358#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":856214571_3" to="65080358#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":856214571_4" to="66984610#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":856214571_5" to="72016745#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":856214571_6" to="65080358#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":856214571_7" to="66984610#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":856214578_0" to="684714761#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":856214578_1" to="84842974#10" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":856214578_1" to="84842974#10" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":856214578_1" to="84842974#10" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":856214578_4" to="684714761#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":856214578_5" to="84842974#10" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":856214578_5" to="84842974#10" fromLane="1" toLane="2" dir="r" state="M" />\n  <connection from=":8642733850_0" to="129842397#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8642733850_1" to="932324369" fromLane="0" toLane="0" via=":8642733850_6_0" dir="r" state="m" />\n  <connection from=":8642733850_6" to="932324369" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":8642733850_2" to="-129842397#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8642733850_3" to="129842397#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":8642733850_4" to="932324369" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8642733850_5" to="-129842397#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8642866006_0" to="932339002#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8642866006_1" to="932339001" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8642866006_2" to="932339002#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8642866006_2" to="932339002#0" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":8642866010_0" to="-932339005" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8642866010_1" to="155691084#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8976428985_0" to="100401250#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8976428985_1" to="100401250#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8976429008_0" to="969757853#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8976429008_1" to="969757853#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":8976429009_0" to="969757851#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8976429009_1" to="-706105950" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":8976429009_2" to="969757851#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":916327686_0" to="77967090#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":916327686_0" to="77967090#0" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":916327686_0" to="77967090#0" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":916327686_3" to="77967093#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":916327686_3" to="77967093#1" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":916327702_0" to="77967084#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":916327702_1" to="77967084#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":916327810_0" to="77967092" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":916327810_0" to="77967092" fromLane="1" toLane="1" dir="l" state="M" />\n  <connection from=":916327810_2" to="77967084#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":916327839_0" to="-324267161#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":916327839_1" to="77967093#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":916327839_1" to="77967093#3" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":916327839_3" to="77967093#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":916327888_0" to="77967093#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":916327888_1" to="77967093#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":916327888_1" to="77967093#0" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":916327912_0" to="77967090#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":916327912_1" to="-1552748564#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":916327912_2" to="77967090#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":916327912_2" to="77967090#1" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":916327912_2" to="77967090#1" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":916327921_0" to="77967093#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":916327921_1" to="77967093#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":916327921_1" to="77967093#2" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":947896950_0" to="243188432#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":947896950_1" to="706104217#8" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":947896950_2" to="-706104217#7" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":947896950_3" to="243188432#0" fromLane="0" toLane="0" via=":947896950_4_0" dir="r" state="m" />\n  <connection from=":947896950_4" to="243188432#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":947896956_0" to="706104217#19" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":947896956_1" to="706095305" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":947896956_2" to="-706104217#18" fromLane="0" toLane="0" via=":947896956_9_0" dir="r" state="m" />\n  <connection from=":947896956_9" to="-706104217#18" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":947896956_3" to="706104217#19" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":947896956_4" to="706095305" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":947896956_5" to="-706104217#18" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":947896956_6" to="706104217#19" fromLane="0" toLane="0" via=":947896956_10_0" dir="r" state="m" />\n  <connection from=":947896956_10" to="706104217#19" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":947896956_7" to="706095305" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":947896956_8" to="-706104217#18" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":947896958_0" to="243079738" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":947896958_1" to="66984613#7" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":947896964_0" to="706104217#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":947896964_1" to="330615829#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":947896964_2" to="-81350005#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":947896964_3" to="706104217#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":947896964_4" to="330615829#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":947896964_5" to="-81350005#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":947896972_0" to="66984613#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":947896972_1" to="-243079723" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":947896972_2" to="66984613#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":947896979_0" to="66984613#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":947896979_1" to="243079673" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":947896979_2" to="66984613#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":947896981_0" to="66984613#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":947896981_1" to="95542480" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":947896981_2" to="-81350006#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":947896981_3" to="66984613#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":947896981_4" to="95542480" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":947896981_5" to="-81350006#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":947896981_6" to="66984613#0" fromLane="0" toLane="0" via=":947896981_7_0" dir="r" state="m" />\n  <connection from=":947896981_7" to="66984613#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":947896989_0" to="66984613#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":947896989_1" to="-243079775" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":947896989_2" to="243314440" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":947896989_3" to="66984613#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":947896989_4" to="-243079775" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":947896989_5" to="243314440" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":947896989_6" to="66984613#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1107773211_1110416556_0" to="-129827554#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1107773211_1110416556_1" to="1374050353" fromLane="0" toLane="0" dir="L" state="M" />\n  <connection from=":cluster_1107773211_1110416556_2" to="-1454593202#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1107773211_1110416556_3" to="-148941242#5" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1107773211_1110416556_4" to="-129827554#4" fromLane="0" toLane="0" dir="L" state="M" />\n  <connection from=":cluster_1107773211_1110416556_5" to="1374050353" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1107773211_1110416556_6" to="-1454593202#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1107773211_1110416556_7" to="-148941242#5" fromLane="0" toLane="0" dir="L" state="M" />\n  <connection from=":cluster_1107773211_1110416556_8" to="-129827554#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1107773211_1110416556_9" to="1374050353" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1107773211_1110416556_10" to="-1454593202#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1107773211_1110416556_11" to="-148941242#5" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_16_0" dir="R" state="m" />\n  <connection from=":cluster_1107773211_1110416556_16" to="-148941242#5" fromLane="0" toLane="0" dir="R" state="M" />\n  <connection from=":cluster_1107773211_1110416556_12" to="-129827554#4" fromLane="0" toLane="0" via=":cluster_1107773211_1110416556_17_0" dir="r" state="m" />\n  <connection from=":cluster_1107773211_1110416556_17" to="-129827554#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1107773211_1110416556_13" to="1374050353" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1107773211_1110416556_14" to="-1454593202#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1107773211_1110416556_15" to="-148941242#5" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1110416379_1680666461_0" to="95809262#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1110416379_1680666461_1" to="706256380#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1110416379_1680666461_2" to="155690981" fromLane="0" toLane="0" via=":cluster_1110416379_1680666461_12_0" dir="r" state="m" />\n  <connection from=":cluster_1110416379_1680666461_12" to="155690981" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1110416379_1680666461_3" to="-706256380#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1110416379_1680666461_4" to="95809262#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1110416379_1680666461_5" to="706256380#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1110416379_1680666461_6" to="155690981" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1110416379_1680666461_7" to="-706256380#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1110416379_1680666461_8" to="95809262#0" fromLane="0" toLane="0" via=":cluster_1110416379_1680666461_13_0" dir="r" state="m" />\n  <connection from=":cluster_1110416379_1680666461_13" to="95809262#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1110416379_1680666461_9" to="706256380#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1110416379_1680666461_10" to="155690981" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1110416379_1680666461_11" to="-706256380#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1115585393_13472440789_947896971_0" to="706104219#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1115585393_13472440789_947896971_1" to="81350005#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1115585393_13472440789_947896971_2" to="-706104219#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1115585393_13472440789_947896971_3" to="96242763#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1115585393_13472440789_947896971_4" to="706104219#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1115585393_13472440789_947896971_5" to="81350005#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1115585393_13472440789_947896971_6" to="-706104219#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1115585393_13472440789_947896971_7" to="96242763#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1115585393_13472440789_947896971_8" to="706104219#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1115585393_13472440789_947896971_9" to="81350005#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1115585393_13472440789_947896971_10" to="-706104219#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1115585393_13472440789_947896971_11" to="96242763#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1122072658_14138648129_0" to="96837170#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1122072658_14138648129_1" to="706256379#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1122072658_14138648129_2" to="-706256379#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1122072658_14138648129_3" to="96837170#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1122072658_14138648129_4" to="706256379#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1122072658_14138648129_5" to="-706256379#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1122072658_14138648129_6" to="96837170#0" fromLane="0" toLane="0" via=":cluster_1122072658_14138648129_9_0" dir="r" state="m" />\n  <connection from=":cluster_1122072658_14138648129_9" to="96837170#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1122072658_14138648129_7" to="706256379#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1122072658_14138648129_8" to="-706256379#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1123399587_8976429007_0" to="970010765#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1123399587_8976429007_1" to="969765780#1" fromLane="0" toLane="0" via=":cluster_1123399587_8976429007_6_0" dir="R" state="m" />\n  <connection from=":cluster_1123399587_8976429007_6" to="969765780#1" fromLane="0" toLane="0" dir="R" state="M" />\n  <connection from=":cluster_1123399587_8976429007_2" to="969757851#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1123399587_8976429007_3" to="970010765#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1123399587_8976429007_4" to="969765780#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1123399587_8976429007_5" to="969757851#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1123399715_3376084911_3376091818_0" to="243188670#0" fromLane="0" toLane="1" dir="s" state="M" />\n  <connection from=":cluster_1123399715_3376084911_3376091818_1" to="-330615844" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1123399715_3376084911_3376091818_2" to="243188670#0" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":cluster_1123399715_3376084911_3376091818_3" to="243188670#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_11292971032_2406954578_2406954579_2406954580_0" to="1218225518#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_11292971032_2406954578_2406954579_2406954580_1" to="1218225524#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_11292971032_2406954578_2406954579_2406954580_2" to="1218747874" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_11292971032_2406954578_2406954579_2406954580_3" to="1218225518#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_11292971032_2406954578_2406954579_2406954580_3" to="1218225518#0" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":cluster_11292971032_2406954578_2406954579_2406954580_5" to="1218225524#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_11292971032_2406954578_2406954579_2406954580_6" to="1218747874" fromLane="0" toLane="0" via=":cluster_11292971032_2406954578_2406954579_2406954580_7_0" dir="r" state="m" />\n  <connection from=":cluster_11292971032_2406954578_2406954579_2406954580_7" to="1218747874" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_13424220808_2506085647_0" to="931354569#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_13424220808_2506085647_1" to="-931354569#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_13424220808_2506085647_2" to="243188465#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_13424220808_2506085647_3" to="931354569#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_13424220808_2506085647_4" to="-931354569#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_13424220808_2506085647_5" to="243188465#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_13424220808_2506085647_6" to="931354569#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_13424220808_2506085647_7" to="-931354569#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_13424220808_2506085647_8" to="243188465#0" fromLane="0" toLane="0" via=":cluster_13424220808_2506085647_9_0" dir="r" state="m" />\n  <connection from=":cluster_13424220808_2506085647_9" to="243188465#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_13472440775_1666900747_0" to="72016745#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_13472440775_1666900747_1" to="-95542485#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_13472440775_1666900747_2" to="-72016745#2" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":cluster_13472440775_1666900747_3" to="706105951#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_13472440775_1666900747_4" to="72016745#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_13472440775_1666900747_4" to="72016745#4" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":cluster_13472440775_1666900747_6" to="-95542485#3" fromLane="0" toLane="0" via=":cluster_13472440775_1666900747_14_0" dir="r" state="m" />\n  <connection from=":cluster_13472440775_1666900747_14" to="-95542485#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_13472440775_1666900747_7" to="-72016745#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_13472440775_1666900747_8" to="706105951#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_13472440775_1666900747_9" to="72016745#4" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":cluster_13472440775_1666900747_10" to="-95542485#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_13472440775_1666900747_11" to="-72016745#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_13472440775_1666900747_11" to="-72016745#2" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":cluster_13472440775_1666900747_13" to="706105951#0" fromLane="0" toLane="0" via=":cluster_13472440775_1666900747_15_0" dir="r" state="m" />\n  <connection from=":cluster_13472440775_1666900747_15" to="706105951#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_14188374630_2506088509_0" to="-243188508#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_14188374630_2506088509_1" to="-1559802496" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_14188374630_2506088509_2" to="243188508#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_14188374630_2506088509_3" to="-243188212#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_14188374630_2506088509_4" to="-243188508#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_14188374630_2506088509_5" to="-1559802496" fromLane="0" toLane="0" via=":cluster_14188374630_2506088509_12_0" dir="r" state="m" />\n  <connection from=":cluster_14188374630_2506088509_12" to="-1559802496" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_14188374630_2506088509_6" to="243188508#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_14188374630_2506088509_7" to="-243188212#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_14188374630_2506088509_8" to="-243188508#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_14188374630_2506088509_9" to="-1559802496" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_14188374630_2506088509_10" to="243188508#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_14188374630_2506088509_11" to="-243188212#1" fromLane="0" toLane="0" via=":cluster_14188374630_2506088509_13_0" dir="r" state="m" />\n  <connection from=":cluster_14188374630_2506088509_13" to="-243188212#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1558314142_1558314144_0" to="706094740#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1558314142_1558314144_1" to="-706094740#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1558314142_1558314144_2" to="155690513" fromLane="0" toLane="0" via=":cluster_1558314142_1558314144_12_0" dir="r" state="m" />\n  <connection from=":cluster_1558314142_1558314144_12" to="155690513" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1558314142_1558314144_3" to="498671497" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1558314142_1558314144_4" to="706094740#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1558314142_1558314144_5" to="-706094740#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1558314142_1558314144_6" to="155690513" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1558314142_1558314144_7" to="498671497" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1558314142_1558314144_8" to="706094740#2" fromLane="0" toLane="0" via=":cluster_1558314142_1558314144_13_0" dir="r" state="m" />\n  <connection from=":cluster_1558314142_1558314144_13" to="706094740#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1558314142_1558314144_9" to="-706094740#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1558314142_1558314144_10" to="155690513" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1558314142_1558314144_11" to="498671497" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1559016910_4900465210_0" to="-142455590" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1559016910_4900465210_1" to="-142455600#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1559016910_4900465210_2" to="498671483" fromLane="0" toLane="0" via=":cluster_1559016910_4900465210_12_0" dir="r" state="m" />\n  <connection from=":cluster_1559016910_4900465210_12" to="498671483" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1559016910_4900465210_3" to="142455600#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1559016910_4900465210_4" to="-142455590" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1559016910_4900465210_5" to="-142455600#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1559016910_4900465210_6" to="498671483" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1559016910_4900465210_7" to="142455600#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1559016910_4900465210_8" to="-142455590" fromLane="0" toLane="0" via=":cluster_1559016910_4900465210_13_0" dir="r" state="m" />\n  <connection from=":cluster_1559016910_4900465210_13" to="-142455590" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1559016910_4900465210_9" to="-142455600#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1559016910_4900465210_10" to="498671483" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1559016910_4900465210_11" to="142455600#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1559016919_1680666950_0" to="-706256386" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1559016919_1680666950_1" to="706256380#6" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1559016919_1680666950_2" to="155690878#0" fromLane="0" toLane="0" via=":cluster_1559016919_1680666950_12_0" dir="r" state="m" />\n  <connection from=":cluster_1559016919_1680666950_12" to="155690878#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1559016919_1680666950_3" to="-706256380#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1559016919_1680666950_4" to="-706256386" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1559016919_1680666950_5" to="706256380#6" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1559016919_1680666950_6" to="155690878#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1559016919_1680666950_7" to="-706256380#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1559016919_1680666950_8" to="-706256386" fromLane="0" toLane="0" via=":cluster_1559016919_1680666950_13_0" dir="r" state="m" />\n  <connection from=":cluster_1559016919_1680666950_13" to="-706256386" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1559016919_1680666950_9" to="706256380#6" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1559016919_1680666950_10" to="155690878#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1559016919_1680666950_11" to="-706256380#4" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1680667233_2506085475_0" to="95533432#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1680667233_2506085475_1" to="243188458" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1680667233_2506085475_2" to="-95533432#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1680667233_2506085475_3" to="-155690875" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1680667233_2506085475_4" to="95533432#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1680667233_2506085475_5" to="243188458" fromLane="0" toLane="0" via=":cluster_1680667233_2506085475_12_0" dir="r" state="m" />\n  <connection from=":cluster_1680667233_2506085475_12" to="243188458" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1680667233_2506085475_6" to="-95533432#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1680667233_2506085475_7" to="-155690875" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1680667233_2506085475_8" to="95533432#3" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1680667233_2506085475_9" to="243188458" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1680667233_2506085475_10" to="-95533432#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1680667233_2506085475_11" to="-155690875" fromLane="0" toLane="0" via=":cluster_1680667233_2506085475_13_0" dir="r" state="m" />\n  <connection from=":cluster_1680667233_2506085475_13" to="-155690875" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1966399241_8523333972_0" to="706104219#11" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1966399241_8523333972_1" to="96713729" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1966399241_8523333972_2" to="-706104219#9" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1966399241_8523333972_3" to="917696759" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1966399241_8523333972_4" to="706104219#11" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1966399241_8523333972_5" to="96713729" fromLane="0" toLane="0" via=":cluster_1966399241_8523333972_9_0" dir="r" state="m" />\n  <connection from=":cluster_1966399241_8523333972_9" to="96713729" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_1966399241_8523333972_6" to="96713729" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_1966399241_8523333972_7" to="-706104219#9" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_1966399241_8523333972_8" to="917696759" fromLane="0" toLane="0" via=":cluster_1966399241_8523333972_10_0" dir="r" state="m" />\n  <connection from=":cluster_1966399241_8523333972_10" to="917696759" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_2052547073_2506085499_0" to="40264697#9" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_2052547073_2506085499_1" to="-243188465#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_2052547073_2506085499_2" to="40264697#9" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_2052547073_2506085499_2" to="40264697#9" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":cluster_2052547073_2506085499_4" to="154707523#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_2505176169_856214432_0" to="1270957862" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_2505176169_856214432_0" to="1270957862" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":cluster_2505176169_856214432_2" to="1270957863" fromLane="0" toLane="0" via=":cluster_2505176169_856214432_7_0" dir="r" state="m" />\n  <connection from=":cluster_2505176169_856214432_7" to="1270957863" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_2505176169_856214432_3" to="1227721991#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_2505176169_856214432_4" to="1270957862" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":cluster_2505176169_856214432_5" to="1270957863" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_2505176169_856214432_6" to="1227721991#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_2505176422_2505176434_0" to="970323546#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_2505176422_2505176434_1" to="65080358#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_2505176422_2505176434_2" to="-243314186" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_2505176422_2505176434_3" to="970323546#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_2505176422_2505176434_3" to="970323546#4" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":cluster_2505176422_2505176434_5" to="65080358#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_2505176422_2505176434_6" to="-243314186" fromLane="0" toLane="0" via=":cluster_2505176422_2505176434_7_0" dir="r" state="m" />\n  <connection from=":cluster_2505176422_2505176434_7" to="-243314186" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_2506087462_8523307708_0" to="84842974#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_2506087462_8523307708_1" to="-706095308#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_2506087462_8523307708_2" to="84842974#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_2506087462_8523307708_2" to="84842974#1" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":cluster_2506087462_8523307708_2" to="84842974#1" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":cluster_2506087462_8523307708_5" to="84842974#1" fromLane="0" toLane="0" via=":cluster_2506087462_8523307708_8_0" dir="r" state="m" />\n  <connection from=":cluster_2506087462_8523307708_8" to="84842974#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_2506087462_8523307708_5" to="84842974#1" fromLane="1" toLane="1" dir="r" state="M" />\n  <connection from=":cluster_2506087462_8523307708_5" to="84842974#1" fromLane="2" toLane="2" dir="r" state="M" />\n  <connection from=":cluster_2506088039_8523406713_0" to="706104217#16" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_2506088039_8523406713_1" to="917704692" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_2506088039_8523406713_2" to="-706104217#14" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_2506088039_8523406713_3" to="706104217#16" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_2506088039_8523406713_4" to="917704692" fromLane="0" toLane="0" via=":cluster_2506088039_8523406713_9_0" dir="r" state="m" />\n  <connection from=":cluster_2506088039_8523406713_9" to="917704692" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_2506088039_8523406713_5" to="-706104217#14" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_2506088039_8523406713_6" to="706104217#16" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_2506088039_8523406713_7" to="917704692" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_2506088039_8523406713_8" to="-706104217#14" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_2507400612_2507400618_0" to="-243314165#1" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_2507400612_2507400618_1" to="-243314436#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_2507400612_2507400618_2" to="243314248#0" fromLane="0" toLane="0" via=":cluster_2507400612_2507400618_12_0" dir="r" state="m" />\n  <connection from=":cluster_2507400612_2507400618_12" to="243314248#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_2507400612_2507400618_3" to="243314436#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_2507400612_2507400618_4" to="-243314165#1" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_2507400612_2507400618_5" to="-243314436#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_2507400612_2507400618_6" to="243314248#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_2507400612_2507400618_7" to="243314436#2" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_2507400612_2507400618_8" to="-243314165#1" fromLane="0" toLane="0" via=":cluster_2507400612_2507400618_13_0" dir="r" state="m" />\n  <connection from=":cluster_2507400612_2507400618_13" to="-243314165#1" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_2507400612_2507400618_9" to="-243314436#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_2507400612_2507400618_10" to="243314248#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_2507400612_2507400618_11" to="243314436#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_620881097_8523329816_0" to="84842974#4" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_620881097_8523329816_1" to="917697169" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_620881097_8523329816_2" to="84842974#4" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_620881097_8523329816_2" to="84842974#4" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":cluster_620881097_8523329816_2" to="84842974#4" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":cluster_620881097_8523329816_5" to="243188120" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_849403384_849403500_8976428987_8976428988_0" to="990212968" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_849403384_849403500_8976428987_8976428988_1" to="970010765#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_849403384_849403500_8976428987_8976428988_2" to="1218225522#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_849403384_849403500_8976428987_8976428988_3" to="969757852" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_849403384_849403500_8976428987_8976428988_4" to="990212968" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_849403384_849403500_8976428987_8976428988_4" to="990212968" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":cluster_849403384_849403500_8976428987_8976428988_6" to="970010765#0" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_849403384_849403500_8976428987_8976428988_7" to="1218225522#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_849403384_849403500_8976428987_8976428988_8" to="969757852" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_849403384_849403500_8976428987_8976428988_9" to="990212968" fromLane="0" toLane="1" dir="r" state="M" />\n  <connection from=":cluster_849403384_849403500_8976428987_8976428988_10" to="970010765#0" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_849403384_849403500_8976428987_8976428988_11" to="1218225522#0" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_849403384_849403500_8976428987_8976428988_12" to="969757852" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_916327730_916327962_985092239_985092700_0" to="77967090#3" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_916327730_916327962_985092239_985092700_1" to="77967084#2" fromLane="0" toLane="0" dir="r" state="M" />\n  <connection from=":cluster_916327730_916327962_985092239_985092700_2" to="-706104217#21" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_916327730_916327962_985092239_985092700_3" to="77967090#3" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_916327730_916327962_985092239_985092700_3" to="77967090#3" fromLane="1" toLane="1" dir="s" state="M" />\n  <connection from=":cluster_916327730_916327962_985092239_985092700_3" to="77967090#3" fromLane="2" toLane="2" dir="s" state="M" />\n  <connection from=":cluster_916327730_916327962_985092239_985092700_6" to="77967084#2" fromLane="0" toLane="0" dir="l" state="M" />\n  <connection from=":cluster_916327730_916327962_985092239_985092700_7" to="-706104217#21" fromLane="0" toLane="0" dir="s" state="M" />\n  <connection from=":cluster_916327730_916327962_985092239_985092700_8" to="77967090#3" fromLane="0" toLane="2" dir="r" state="M" />\n  <roundabout nodes="13339819885 13339819888 13339819893 13339844003" edges="1454593203#1 1454593203#2 1454593203#3 1454593203#4" />\n  <roundabout nodes="1115585857 4888861608 4888861612 4888861618 4888861620 4888862222" edges="497290261#1 497290261#2 497290261#3 497290261#4 497290261#5 497290261#7" />\n  <roundabout nodes="916327686 916327839 916327888 916327921" edges="77967093#0 77967093#1 77967093#2 77967093#3" />\n</net>', 'data/hat_yai_roads.osm.xml': '<?xml version=\'1.0\' encoding=\'utf-8\'?>\n<osm version="0.6" generator="AdaptiveSignals bounded-road extraction" copyright="OpenStreetMap and contributors" attribution="https://www.openstreetmap.org/copyright" license="https://opendatacommons.org/licenses/odbl/1-0/">\n  <node id="1107773191" visible="true" version="2" changeset="7021523" timestamp="2011-01-19T15:25:36Z" user="iPudding" uid="395923" lat="7.0011123" lon="100.4744635" />\n  <node id="1107773194" visible="true" version="4" changeset="28369342" timestamp="2015-01-24T10:49:07Z" user="Pathumthani" uid="2302962" lat="7.0003361" lon="100.4713553" />\n  <node id="1107773211" visible="true" version="2" changeset="7408976" timestamp="2011-02-27T10:51:36Z" user="sdayu" uid="300693" lat="7.0030741" lon="100.4741969" />\n  <node id="1107773215" visible="true" version="2" changeset="9287848" timestamp="2011-09-13T09:25:40Z" user="iPudding" uid="395923" lat="7.0015941" lon="100.4743295" />\n  <node id="1107773217" visible="true" version="2" changeset="9287848" timestamp="2011-09-13T09:25:43Z" user="iPudding" uid="395923" lat="7.0022767" lon="100.4742481" />\n  <node id="1107773241" visible="true" version="3" changeset="189039218" timestamp="2026-09-15T03:15:39Z" user="GrabTH_Monnapat" uid="24164407" lat="6.9992978" lon="100.4715028" />\n  <node id="1107773286" visible="true" version="2" changeset="9287848" timestamp="2011-09-13T09:25:42Z" user="iPudding" uid="395923" lat="7.0017597" lon="100.4743062" />\n  <node id="1107784878" visible="true" version="2" changeset="7408976" timestamp="2011-02-27T10:51:32Z" user="sdayu" uid="300693" lat="7.0029742" lon="100.4729750" />\n  <node id="1107784879" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:03Z" user="Bernhard Hiller" uid="326704" lat="7.0075010" lon="100.4720857" />\n  <node id="1107784883" visible="true" version="2" changeset="7519651" timestamp="2011-03-11T00:01:41Z" user="Beddhist" uid="140550" lat="7.0037451" lon="100.4728685" />\n  <node id="1107784884" visible="true" version="2" changeset="7524843" timestamp="2011-03-11T16:16:05Z" user="sdayu" uid="300693" lat="7.0043929" lon="100.4727667" />\n  <node id="1107784887" visible="true" version="4" changeset="159211955" timestamp="2024-11-16T15:16:41Z" user="nabilersyad" uid="11419275" lat="7.0083069" lon="100.4719493" />\n  <node id="1107926433" visible="true" version="5" changeset="175444533" timestamp="2025-12-03T12:26:55Z" user="WorayutTK" uid="23562084" lat="7.0050294" lon="100.4797720" />\n  <node id="1107926450" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:34:46Z" user="Bernhard Hiller" uid="326704" lat="7.0139005" lon="100.4816632" />\n  <node id="1107926467" visible="true" version="6" changeset="176054561" timestamp="2025-12-17T14:53:03Z" user="Wim Keijsers" uid="22692008" lat="7.0136240" lon="100.4809111" />\n  <node id="1107926481" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0125080" lon="100.4793910" />\n  <node id="1107926513" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0086352" lon="100.4793306" />\n  <node id="1107926537" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:04Z" user="Bernhard Hiller" uid="326704" lat="7.0056747" lon="100.4793325" />\n  <node id="1107926561" visible="true" version="4" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0126294" lon="100.4789070" />\n  <node id="1107926563" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0094300" lon="100.4793377" />\n  <node id="1107926599" visible="true" version="5" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0129339" lon="100.4775048" />\n  <node id="1107926605" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0132586" lon="100.4805697" />\n  <node id="1107926607" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0113992" lon="100.4793143" />\n  <node id="1107926639" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:05Z" user="Bernhard Hiller" uid="326704" lat="7.0063521" lon="100.4793298" />\n  <node id="1107926734" visible="true" version="6" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0130502" lon="100.4765834" />\n  <node id="1107926761" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0130204" lon="100.4802886" />\n  <node id="1107926777" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:05Z" user="Bernhard Hiller" uid="326704" lat="7.0059100" lon="100.4793444" />\n  <node id="1107926818" visible="true" version="3" changeset="18505195" timestamp="2013-10-23T15:58:05Z" user="Bernhard Hiller" uid="326704" lat="7.0052020" lon="100.4796960" />\n  <node id="1107926840" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:34:47Z" user="Bernhard Hiller" uid="326704" lat="7.0053794" lon="100.4834443" />\n  <node id="1107926845" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0129146" lon="100.4777506" />\n  <node id="1107926897" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0080039" lon="100.4793293" />\n  <node id="1107926905" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0108060" lon="100.4793180" />\n  <node id="1107926934" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:06Z" user="Bernhard Hiller" uid="326704" lat="7.0055682" lon="100.4794897" />\n  <node id="1107926940" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:06Z" user="Bernhard Hiller" uid="326704" lat="7.0127437" lon="100.4785183" />\n  <node id="1107926968" visible="true" version="5" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0140297" lon="100.4811349" />\n  <node id="1107926977" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0129809" lon="100.4770600" />\n  <node id="1108909723" visible="true" version="3" changeset="10185152" timestamp="2011-12-23T08:17:14Z" user="iPudding" uid="395923" lat="6.9996535" lon="100.4707878" />\n  <node id="1110416312" visible="true" version="1" changeset="7021523" timestamp="2011-01-19T15:25:06Z" user="iPudding" uid="395923" lat="7.0026435" lon="100.4736406" />\n  <node id="1110416318" visible="true" version="1" changeset="7021523" timestamp="2011-01-19T15:25:06Z" user="iPudding" uid="395923" lat="7.0014877" lon="100.4734423" />\n  <node id="1110416325" visible="true" version="3" changeset="176632477" timestamp="2025-12-30T15:05:11Z" user="Tharhtetaung" uid="16875287" lat="7.0034984" lon="100.4739155" />\n  <node id="1110416342" visible="true" version="4" changeset="176397735" timestamp="2025-12-25T14:44:47Z" user="11aye" uid="22221182" lat="7.0142744" lon="100.4799142" />\n  <node id="1110416356" visible="true" version="3" changeset="189575613" timestamp="2026-09-26T08:15:30Z" user="Tonushka" uid="3419975" lat="7.0021499" lon="100.4728952" />\n  <node id="1110416379" visible="true" version="3" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0015440" lon="100.4756624" />\n  <node id="1110416388" visible="true" version="1" changeset="7021523" timestamp="2011-01-19T15:25:09Z" user="iPudding" uid="395923" lat="7.0112424" lon="100.4724116" />\n  <node id="1110416410" visible="true" version="3" changeset="176632477" timestamp="2025-12-30T15:05:11Z" user="Tharhtetaung" uid="16875287" lat="7.0038374" lon="100.4739745" />\n  <node id="1110416439" visible="true" version="1" changeset="7021523" timestamp="2011-01-19T15:25:12Z" user="iPudding" uid="395923" lat="7.0011516" lon="100.4742187" />\n  <node id="1110416444" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:09Z" user="Bernhard Hiller" uid="326704" lat="7.0085628" lon="100.4730775" />\n  <node id="1110416454" visible="true" version="3" changeset="188756079" timestamp="2026-09-09T06:45:22Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0035246" lon="100.4790258" />\n  <node id="1110416469" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:10Z" user="Bernhard Hiller" uid="326704" lat="7.0134345" lon="100.4740667" />\n  <node id="1110416470" visible="true" version="3" changeset="144432300" timestamp="2023-11-25T08:10:05Z" user="pLoskutov" uid="17472737" lat="7.0109310" lon="100.4724426" />\n  <node id="1110416484" visible="true" version="1" changeset="7021523" timestamp="2011-01-19T15:25:14Z" user="iPudding" uid="395923" lat="7.0017172" lon="100.4731946" />\n  <node id="1110416498" visible="true" version="3" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0020616" lon="100.4756061" />\n  <node id="1110416526" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0132825" lon="100.4750909" />\n  <node id="1110416528" visible="true" version="1" changeset="7021523" timestamp="2011-01-19T15:25:16Z" user="iPudding" uid="395923" lat="7.0088024" lon="100.4723617" />\n  <node id="1110416537" visible="true" version="1" changeset="7021523" timestamp="2011-01-19T15:25:16Z" user="iPudding" uid="395923" lat="7.0011352" lon="100.4739544" />\n  <node id="1110416556" visible="true" version="4" changeset="182517970" timestamp="2026-05-11T15:45:42Z" user="Emma Js" uid="22434897" lat="7.0030551" lon="100.4740731" />\n  <node id="1110416567" visible="true" version="6" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0141017" lon="100.4808389" />\n  <node id="1110416568" visible="true" version="3" changeset="176632477" timestamp="2025-12-30T15:05:11Z" user="Tharhtetaung" uid="16875287" lat="7.0045161" lon="100.4739438" />\n  <node id="1110416593" visible="true" version="2" changeset="10185152" timestamp="2011-12-23T08:17:15Z" user="iPudding" uid="395923" lat="7.0018921" lon="100.4729864" />\n  <node id="1110416631" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:10Z" user="Bernhard Hiller" uid="326704" lat="7.0119313" lon="100.4723830" />\n  <node id="1110416635" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0014610" lon="100.4756742" />\n  <node id="1110416654" visible="true" version="3" changeset="159211955" timestamp="2024-11-16T15:16:41Z" user="nabilersyad" uid="11419275" lat="7.0084292" lon="100.4723864" />\n  <node id="1110416655" visible="true" version="2" changeset="103186836" timestamp="2021-04-19T09:59:02Z" user="mummadi1" uid="11589192" lat="7.0024918" lon="100.4734594" />\n  <node id="1110416663" visible="true" version="1" changeset="7021523" timestamp="2011-01-19T15:25:23Z" user="iPudding" uid="395923" lat="7.0012254" lon="100.4737644" />\n  <node id="1110416677" visible="true" version="3" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0002589" lon="100.4757901" />\n  <node id="1110416683" visible="true" version="1" changeset="7021523" timestamp="2011-01-19T15:25:25Z" user="iPudding" uid="395923" lat="7.0009238" lon="100.4745986" />\n  <node id="1110416716" visible="true" version="2" changeset="10185152" timestamp="2011-12-23T08:17:15Z" user="iPudding" uid="395923" lat="7.0020251" lon="100.4728627" />\n  <node id="1110416726" visible="true" version="1" changeset="7021523" timestamp="2011-01-19T15:25:27Z" user="iPudding" uid="395923" lat="7.0011631" lon="100.4744325" />\n  <node id="1110416742" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0131278" lon="100.4794188" />\n  <node id="1110416745" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:10Z" user="Bernhard Hiller" uid="326704" lat="7.0115970" lon="100.4723966" />\n  <node id="1110416754" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:11Z" user="Bernhard Hiller" uid="326704" lat="7.0138154" lon="100.4727813" />\n  <node id="1115585385" visible="true" version="2" changeset="49318745" timestamp="2017-06-06T20:56:23Z" user="RVR004" uid="5314349" lat="7.0072325" lon="100.4765604" />\n  <node id="1115585393" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0089076" lon="100.4767571" />\n  <node id="1115585403" visible="true" version="2" changeset="49318745" timestamp="2017-06-06T20:56:23Z" user="RVR004" uid="5314349" lat="7.0080654" lon="100.4768967" />\n  <node id="1115585416" visible="true" version="3" changeset="18505195" timestamp="2013-10-23T15:58:13Z" user="Bernhard Hiller" uid="326704" lat="7.0060349" lon="100.4759068" />\n  <node id="1115585435" visible="true" version="3" changeset="18505195" timestamp="2013-10-23T15:58:13Z" user="Bernhard Hiller" uid="326704" lat="7.0057924" lon="100.4739999" />\n  <node id="1115585439" visible="true" version="2" changeset="49318745" timestamp="2017-06-06T20:56:23Z" user="RVR004" uid="5314349" lat="7.0075906" lon="100.4768647" />\n  <node id="1115585463" visible="true" version="3" changeset="49318745" timestamp="2017-06-06T20:56:23Z" user="RVR004" uid="5314349" lat="7.0063144" lon="100.4762745" />\n  <node id="1115585518" visible="true" version="3" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0161422" lon="100.4758201" />\n  <node id="1115585600" visible="true" version="4" changeset="18505195" timestamp="2013-10-23T15:58:14Z" user="Bernhard Hiller" uid="326704" lat="7.0063133" lon="100.4752321" />\n  <node id="1115585690" visible="true" version="2" changeset="49318745" timestamp="2017-06-06T20:56:23Z" user="RVR004" uid="5314349" lat="7.0072998" lon="100.4767653" />\n  <node id="1115585732" visible="true" version="2" changeset="49318745" timestamp="2017-06-06T20:56:23Z" user="RVR004" uid="5314349" lat="7.0074552" lon="100.4768032" />\n  <node id="1115585744" visible="true" version="5" changeset="176632477" timestamp="2025-12-30T15:05:11Z" user="Tharhtetaung" uid="16875287" lat="7.0052191" lon="100.4737834" />\n  <node id="1115585792" visible="true" version="6" changeset="182517970" timestamp="2026-05-11T15:45:42Z" user="Emma Js" uid="22434897" lat="7.0059882" lon="100.4742830" />\n  <node id="1115585846" visible="true" version="2" changeset="49318745" timestamp="2017-06-06T20:56:23Z" user="RVR004" uid="5314349" lat="7.0073835" lon="100.4767791" />\n  <node id="1115585857" visible="true" version="4" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0138821" lon="100.4765306" />\n  <node id="1115585909" visible="true" version="2" changeset="49318745" timestamp="2017-06-06T20:56:23Z" user="RVR004" uid="5314349" lat="7.0067270" lon="100.4763647" />\n  <node id="1120012968" visible="true" version="1" changeset="7084993" timestamp="2011-01-25T16:10:03Z" user="makissy" uid="397712" lat="7.0111931" lon="100.4831963" />\n  <node id="1120012970" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0114003" lon="100.4850376" />\n  <node id="1120012971" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0111615" lon="100.4832184" />\n  <node id="1120012973" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0112162" lon="100.4841881" />\n  <node id="1120012974" visible="true" version="3" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0113097" lon="100.4831710" />\n  <node id="1120012975" visible="true" version="1" changeset="7084993" timestamp="2011-01-25T16:10:04Z" user="makissy" uid="397712" lat="7.0113081" lon="100.4830116" />\n  <node id="1120012976" visible="true" version="4" changeset="49318745" timestamp="2017-06-06T20:56:23Z" user="RVR004" uid="5314349" lat="7.0143721" lon="100.4794703" />\n  <node id="1120012977" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0111417" lon="100.4832614" />\n  <node id="1120012979" visible="true" version="3" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0112751" lon="100.4831789" />\n  <node id="1120012981" visible="true" version="1" changeset="7084993" timestamp="2011-01-25T16:10:05Z" user="makissy" uid="397712" lat="7.0113081" lon="100.4830683" />\n  <node id="1120012985" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0111362" lon="100.4833282" />\n  <node id="1120012986" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0112279" lon="100.4831792" />\n  <node id="1120012989" visible="true" version="3" changeset="23214413" timestamp="2014-06-27T12:42:53Z" user="stephenwilks" uid="695426" lat="7.0148135" lon="100.4795108" />\n  <node id="1120015438" visible="true" version="2" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0109129" lon="100.4738369" />\n  <node id="1122072640" visible="true" version="2" changeset="7524843" timestamp="2011-03-11T16:16:00Z" user="sdayu" uid="300693" lat="7.0009155" lon="100.4706000" />\n  <node id="1122072645" visible="true" version="2" changeset="7524843" timestamp="2011-03-11T16:16:15Z" user="sdayu" uid="300693" lat="7.0009887" lon="100.4712661" />\n  <node id="1122072646" visible="true" version="2" changeset="10185152" timestamp="2011-12-23T08:17:15Z" user="iPudding" uid="395923" lat="7.0018120" lon="100.4726594" />\n  <node id="1122072657" visible="true" version="1" changeset="7095434" timestamp="2011-01-26T17:13:35Z" user="iPudding" uid="395923" lat="7.0012900" lon="100.4732995" />\n  <node id="1122072658" visible="true" version="4" changeset="188339904" timestamp="2026-09-01T02:58:55Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0030225" lon="100.4735518" />\n  <node id="1122072661" visible="true" version="2" changeset="189575613" timestamp="2026-09-26T08:15:30Z" user="Tonushka" uid="3419975" lat="7.0023698" lon="100.4726799" />\n  <node id="1122072673" visible="true" version="2" changeset="7524843" timestamp="2011-03-11T16:16:16Z" user="sdayu" uid="300693" lat="7.0010895" lon="100.4721166" />\n  <node id="1122072674" visible="true" version="2" changeset="10185152" timestamp="2011-12-23T08:17:15Z" user="iPudding" uid="395923" lat="7.0019712" lon="100.4725470" />\n  <node id="1122072681" visible="true" version="1" changeset="7095434" timestamp="2011-01-26T17:13:36Z" user="iPudding" uid="395923" lat="7.0028447" lon="100.4717066" />\n  <node id="1122072686" visible="true" version="1" changeset="7095434" timestamp="2011-01-26T17:13:36Z" user="iPudding" uid="395923" lat="7.0015153" lon="100.4730044" />\n  <node id="1122072692" visible="true" version="2" changeset="10185152" timestamp="2011-12-23T08:17:15Z" user="iPudding" uid="395923" lat="7.0025405" lon="100.4729738" />\n  <node id="1122072702" visible="true" version="2" changeset="10185152" timestamp="2011-12-23T08:17:15Z" user="iPudding" uid="395923" lat="7.0021907" lon="100.4725401" />\n  <node id="1122072706" visible="true" version="3" changeset="188339904" timestamp="2026-09-01T02:58:55Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0028182" lon="100.4734554" />\n  <node id="1123399465" visible="true" version="2" changeset="175444256" timestamp="2025-12-03T12:18:40Z" user="WorayutTK" uid="23562084" lat="7.0080224" lon="100.4775255" />\n  <node id="1123399466" visible="true" version="2" changeset="10184991" timestamp="2011-12-23T07:35:43Z" user="iPudding" uid="395923" lat="7.0050140" lon="100.4769404" />\n  <node id="1123399468" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0098848" lon="100.4768618" />\n  <node id="1123399478" visible="true" version="5" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0102924" lon="100.4774677" />\n  <node id="1123399480" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0114081" lon="100.4757640" />\n  <node id="1123399483" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0112687" lon="100.4777493" />\n  <node id="1123399484" visible="true" version="4" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0120157" lon="100.4750807" />\n  <node id="1123399486" visible="true" version="3" changeset="189038359" timestamp="2026-09-15T02:25:53Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0129324" lon="100.4718305" />\n  <node id="1123399488" visible="true" version="4" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0142276" lon="100.4757722" />\n  <node id="1123399489" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0110591" lon="100.4767109" />\n  <node id="1123399492" visible="true" version="3" changeset="74914002" timestamp="2019-09-25T14:34:34Z" user="icosiam" uid="10316678" lat="7.0142485" lon="100.4742818" />\n  <node id="1123399493" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0094266" lon="100.4768014" />\n  <node id="1123399500" visible="true" version="4" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0106012" lon="100.4784727" />\n  <node id="1123399502" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:15Z" user="Bernhard Hiller" uid="326704" lat="7.0121475" lon="100.4723816" />\n  <node id="1123399505" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0092420" lon="100.4770134" />\n  <node id="1123399512" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0110577" lon="100.4760190" />\n  <node id="1123399513" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0136980" lon="100.4806302" />\n  <node id="1123399514" visible="true" version="3" changeset="189038359" timestamp="2026-09-15T02:25:53Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0121380" lon="100.4719097" />\n  <node id="1123399516" visible="true" version="4" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0109282" lon="100.4784760" />\n  <node id="1123399524" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0092323" lon="100.4768874" />\n  <node id="1123399535" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0095530" lon="100.4775237" />\n  <node id="1123399537" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0129010" lon="100.4797437" />\n  <node id="1123399540" visible="true" version="3" changeset="175444256" timestamp="2025-12-03T12:18:40Z" user="WorayutTK" uid="23562084" lat="7.0079271" lon="100.4785602" />\n  <node id="1123399547" visible="true" version="3" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0113927" lon="100.4745469" />\n  <node id="1123399549" visible="true" version="4" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0143156" lon="100.4756106" />\n  <node id="1123399550" visible="true" version="3" changeset="29150194" timestamp="2015-02-28T07:41:03Z" user="Pathumthani" uid="2302962" lat="7.0113928" lon="100.4741218" />\n  <node id="1123399553" visible="true" version="1" changeset="7103873" timestamp="2011-01-27T14:00:15Z" user="iPudding" uid="395923" lat="7.0078260" lon="100.4772728" />\n  <node id="1123399561" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0095359" lon="100.4779695" />\n  <node id="1123399562" visible="true" version="4" changeset="182517970" timestamp="2026-05-11T15:45:42Z" user="Emma Js" uid="22434897" lat="7.0062176" lon="100.4746858" />\n  <node id="1123399566" visible="true" version="4" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0102230" lon="100.4771311" />\n  <node id="1123399569" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:16Z" user="Bernhard Hiller" uid="326704" lat="7.0143049" lon="100.4752262" />\n  <node id="1123399575" visible="true" version="1" changeset="7103873" timestamp="2011-01-27T14:00:16Z" user="iPudding" uid="395923" lat="7.0075360" lon="100.4771150" />\n  <node id="1123399576" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0101656" lon="100.4776799" />\n  <node id="1123399578" visible="true" version="3" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0120629" lon="100.4751688" />\n  <node id="1123399580" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:17Z" user="Bernhard Hiller" uid="326704" lat="7.0142997" lon="100.4731187" />\n  <node id="1123399584" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0094790" lon="100.4781116" />\n  <node id="1123399585" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:17Z" user="Bernhard Hiller" uid="326704" lat="7.0121175" lon="100.4728006" />\n  <node id="1123399586" visible="true" version="2" changeset="175444256" timestamp="2025-12-03T12:18:40Z" user="WorayutTK" uid="23562084" lat="7.0080373" lon="100.4777154" />\n  <node id="1123399587" visible="true" version="4" changeset="18489522" timestamp="2013-10-22T17:34:51Z" user="Bernhard Hiller" uid="326704" lat="7.0048322" lon="100.4772008" />\n  <node id="1123399589" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0100458" lon="100.4769594" />\n  <node id="1123399591" visible="true" version="3" changeset="28539150" timestamp="2015-02-01T08:05:21Z" user="Pathumthani" uid="2302962" lat="7.0139162" lon="100.4724430" />\n  <node id="1123399595" visible="true" version="3" changeset="49318745" timestamp="2017-06-06T20:56:23Z" user="RVR004" uid="5314349" lat="7.0078391" lon="100.4744019" />\n  <node id="1123399598" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0134441" lon="100.4804567" />\n  <node id="1123399600" visible="true" version="2" changeset="108306644" timestamp="2021-07-20T09:48:02Z" user="appik" uid="11515808" lat="7.0072955" lon="100.4770573" />\n  <node id="1123399603" visible="true" version="5" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0119286" lon="100.4756330" />\n  <node id="1123399605" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:17Z" user="Bernhard Hiller" uid="326704" lat="7.0140823" lon="100.4726335" />\n  <node id="1123399607" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0112649" lon="100.4774913" />\n  <node id="1123399611" visible="true" version="3" changeset="29150156" timestamp="2015-02-28T07:37:20Z" user="Pathumthani" uid="2302962" lat="7.0110081" lon="100.4764540" />\n  <node id="1123399612" visible="true" version="3" changeset="148237597" timestamp="2024-03-05T07:55:56Z" user="GrabTH_Panuwit" uid="16483995" lat="7.0119996" lon="100.4734042" />\n  <node id="1123399616" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0096853" lon="100.4768092" />\n  <node id="1123399630" visible="true" version="4" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0102838" lon="100.4772279" />\n  <node id="1123399632" visible="true" version="5" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0112314" lon="100.4758290" />\n  <node id="1123399637" visible="true" version="3" changeset="189038359" timestamp="2026-09-15T02:25:53Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0122428" lon="100.4717765" />\n  <node id="1123399639" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0090525" lon="100.4783910" />\n  <node id="1123399641" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0112234" lon="100.4781114" />\n  <node id="1123399648" visible="true" version="3" changeset="189038359" timestamp="2026-09-15T02:25:53Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0126678" lon="100.4717763" />\n  <node id="1123399659" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0112512" lon="100.4772180" />\n  <node id="1123399662" visible="true" version="3" changeset="18505195" timestamp="2013-10-23T15:58:19Z" user="Bernhard Hiller" uid="326704" lat="7.0056021" lon="100.4762604" />\n  <node id="1123399664" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0092994" lon="100.4768097" />\n  <node id="1123399675" visible="true" version="3" changeset="189038359" timestamp="2026-09-15T02:25:53Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0135359" lon="100.4721098" />\n  <node id="1123399676" visible="true" version="3" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0135968" lon="100.4762773" />\n  <node id="1123399683" visible="true" version="4" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0102105" lon="100.4784443" />\n  <node id="1123399687" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:19Z" user="Bernhard Hiller" uid="326704" lat="7.0121449" lon="100.4726367" />\n  <node id="1123399697" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0094804" lon="100.4773680" />\n  <node id="1123399700" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0132451" lon="100.4802259" />\n  <node id="1123399706" visible="true" version="3" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0069454" lon="100.4745272" />\n  <node id="1123399711" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0079861" lon="100.4787276" />\n  <node id="1123399713" visible="true" version="5" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0133490" lon="100.4783117" />\n  <node id="1123399715" visible="true" version="5" changeset="188456911" timestamp="2026-09-03T07:55:18Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0110080" lon="100.4763430" />\n  <node id="1123399730" visible="true" version="3" changeset="189038359" timestamp="2026-09-15T02:25:53Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0121313" lon="100.4720617" />\n  <node id="1123399740" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0132460" lon="100.4788636" />\n  <node id="1123399749" visible="true" version="3" changeset="189038359" timestamp="2026-09-15T02:25:53Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0124312" lon="100.4717528" />\n  <node id="1123399768" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:20Z" user="Bernhard Hiller" uid="326704" lat="7.0118722" lon="100.4736361" />\n  <node id="1123399771" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0084357" lon="100.4768435" />\n  <node id="1123399778" visible="true" version="2" changeset="175444256" timestamp="2025-12-03T12:18:40Z" user="WorayutTK" uid="23562084" lat="7.0079535" lon="100.4773795" />\n  <node id="1123399809" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:20Z" user="Bernhard Hiller" uid="326704" lat="7.0065520" lon="100.4745938" />\n  <node id="1123399815" visible="true" version="2" changeset="175444256" timestamp="2025-12-03T12:18:40Z" user="WorayutTK" uid="23562084" lat="7.0079513" lon="100.4783496" />\n  <node id="1123399836" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:21Z" user="Bernhard Hiller" uid="326704" lat="7.0114821" lon="100.4746189" />\n  <node id="1123399840" visible="true" version="2" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0143175" lon="100.4753732" />\n  <node id="1123399846" visible="true" version="3" changeset="18505195" timestamp="2013-10-23T15:58:21Z" user="Bernhard Hiller" uid="326704" lat="7.0062942" lon="100.4748249" />\n  <node id="1123399848" visible="true" version="2" changeset="49318745" timestamp="2017-06-06T20:56:23Z" user="RVR004" uid="5314349" lat="7.0079659" lon="100.4758069" />\n  <node id="1123399852" visible="true" version="1" changeset="7103873" timestamp="2011-01-27T14:00:27Z" user="iPudding" uid="395923" lat="7.0076960" lon="100.4771721" />\n  <node id="1123399863" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:21Z" user="Bernhard Hiller" uid="326704" lat="7.0142618" lon="100.4737446" />\n  <node id="1123399866" visible="true" version="3" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0102088" lon="100.4782092" />\n  <node id="1123399906" visible="true" version="2" changeset="7524843" timestamp="2011-03-11T16:16:06Z" user="sdayu" uid="300693" lat="7.0014849" lon="100.4705320" />\n  <node id="1123399909" visible="true" version="2" changeset="49318745" timestamp="2017-06-06T20:56:23Z" user="RVR004" uid="5314349" lat="7.0078974" lon="100.4748897" />\n  <node id="1123399917" visible="true" version="1" changeset="7103873" timestamp="2011-01-27T14:00:28Z" user="iPudding" uid="395923" lat="7.0073760" lon="100.4770814" />\n  <node id="1123399932" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0120325" lon="100.4755238" />\n  <node id="1123399938" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:21Z" user="Bernhard Hiller" uid="326704" lat="7.0141714" lon="100.4728415" />\n  <node id="1123399943" visible="true" version="3" changeset="189038359" timestamp="2026-09-15T02:25:53Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0131697" lon="100.4719226" />\n  <node id="1123399962" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:21Z" user="Bernhard Hiller" uid="326704" lat="7.0120618" lon="100.4731494" />\n  <node id="11292971032" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0090558" lon="100.4811930" />\n  <node id="11292971033" visible="true" version="1" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145" lat="7.0079828" lon="100.4811410" />\n  <node id="11292971034" visible="true" version="1" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145" lat="7.0079773" lon="100.4812364" />\n  <node id="11292971035" visible="true" version="1" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145" lat="7.0090322" lon="100.4813437" />\n  <node id="11292971036" visible="true" version="1" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145" lat="7.0090218" lon="100.4813993" />\n  <node id="11292971037" visible="true" version="1" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145" lat="7.0089226" lon="100.4813279" />\n  <node id="11292971038" visible="true" version="1" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145" lat="7.0089108" lon="100.4813892" />\n  <node id="11292971039" visible="true" version="1" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145" lat="7.0092290" lon="100.4813808" />\n  <node id="11292971040" visible="true" version="1" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145" lat="7.0092243" lon="100.4814287" />\n  <node id="1140277055" visible="true" version="4" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0152148" lon="100.4843753" />\n  <node id="1140277056" visible="true" version="2" changeset="176054561" timestamp="2025-12-17T14:53:03Z" user="Wim Keijsers" uid="22692008" lat="7.0152329" lon="100.4835016" />\n  <node id="1140280757" visible="true" version="1" changeset="7212332" timestamp="2011-02-07T03:27:54Z" user="makissy" uid="397712" lat="7.0086169" lon="100.4735109" />\n  <node id="1140280758" visible="true" version="3" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0110800" lon="100.4733872" />\n  <node id="1160612969" visible="true" version="3" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0039760" lon="100.4754019" />\n  <node id="1160612974" visible="true" version="5" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0046686" lon="100.4753243" />\n  <node id="1160612987" visible="true" version="2" changeset="176632477" timestamp="2025-12-30T15:05:11Z" user="Tharhtetaung" uid="16875287" lat="7.0041770" lon="100.4753823" />\n  <node id="1160617588" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:22Z" user="Bernhard Hiller" uid="326704" lat="7.0153059" lon="100.4740319" />\n  <node id="1160617590" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:22Z" user="Bernhard Hiller" uid="326704" lat="7.0145950" lon="100.4742373" />\n  <node id="11693248213" visible="true" version="1" changeset="148237597" timestamp="2024-03-05T07:55:56Z" user="GrabTH_Panuwit" uid="16483995" lat="7.0114241" lon="100.4745775" />\n  <node id="1174378788" visible="true" version="2" changeset="13502820" timestamp="2012-10-15T09:29:30Z" user="Tungmar" uid="427807" lat="7.0020490" lon="100.4717988" />\n  <node id="1174378892" visible="true" version="3" changeset="177498262" timestamp="2026-01-20T22:53:08Z" user="talbertdoi" uid="23344915" lat="7.0018011" lon="100.4697782" />\n  <node id="1176926303" visible="true" version="2" changeset="7524843" timestamp="2011-03-11T16:16:16Z" user="sdayu" uid="300693" lat="7.0035690" lon="100.4708694" />\n  <node id="1196113662" visible="true" version="2" changeset="7524843" timestamp="2011-03-11T16:16:00Z" user="sdayu" uid="300693" lat="7.0018818" lon="100.4704629" />\n  <node id="12187377521" visible="true" version="1" changeset="156846444" timestamp="2024-09-20T02:29:38Z" user="djmorkman" uid="5547549" lat="7.0074645" lon="100.4687375" />\n  <node id="12187377522" visible="true" version="1" changeset="156846444" timestamp="2024-09-20T02:29:38Z" user="djmorkman" uid="5547549" lat="7.0074968" lon="100.4688858" />\n  <node id="12284916697" visible="true" version="2" changeset="158436906" timestamp="2024-10-28T01:56:33Z" user="Madi-Red" uid="14550990" lat="7.0128527" lon="100.4748626" />\n  <node id="12284916698" visible="true" version="2" changeset="158436906" timestamp="2024-10-28T01:56:33Z" user="Madi-Red" uid="14550990" lat="7.0128490" lon="100.4748998" />\n  <node id="12334148767" visible="true" version="1" changeset="159040528" timestamp="2024-11-12T08:48:17Z" user="nabilersyad" uid="11419275" lat="7.0111376" lon="100.4746257" />\n  <node id="12334148768" visible="true" version="2" changeset="159209920" timestamp="2024-11-16T14:15:41Z" user="nabilersyad" uid="11419275" lat="7.0110764" lon="100.4745304" />\n  <node id="12334148769" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0106744" lon="100.4763755" />\n  <node id="12334148770" visible="true" version="2" changeset="159211010" timestamp="2024-11-16T14:48:54Z" user="nabilersyad" uid="11419275" lat="7.0106917" lon="100.4762162" />\n  <node id="12334148771" visible="true" version="2" changeset="159211010" timestamp="2024-11-16T14:48:54Z" user="nabilersyad" uid="11419275" lat="7.0107309" lon="100.4760687" />\n  <node id="12334148772" visible="true" version="1" changeset="159040528" timestamp="2024-11-12T08:48:17Z" user="nabilersyad" uid="11419275" lat="7.0109186" lon="100.4757100" />\n  <node id="12334158707" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0117035" lon="100.4754894" />\n  <node id="12334158708" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0118199" lon="100.4753995" />\n  <node id="12334158709" visible="true" version="1" changeset="159040528" timestamp="2024-11-12T08:48:17Z" user="nabilersyad" uid="11419275" lat="7.0118337" lon="100.4751568" />\n  <node id="12334158710" visible="true" version="1" changeset="159040528" timestamp="2024-11-12T08:48:17Z" user="nabilersyad" uid="11419275" lat="7.0116593" lon="100.4750106" />\n  <node id="12334158711" visible="true" version="1" changeset="159040528" timestamp="2024-11-12T08:48:17Z" user="nabilersyad" uid="11419275" lat="7.0115595" lon="100.4749449" />\n  <node id="12334158712" visible="true" version="2" changeset="159209920" timestamp="2024-11-16T14:15:41Z" user="nabilersyad" uid="11419275" lat="7.0111855" lon="100.4746759" />\n  <node id="12334158713" visible="true" version="2" changeset="159209920" timestamp="2024-11-16T14:15:41Z" user="nabilersyad" uid="11419275" lat="7.0111142" lon="100.4745922" />\n  <node id="12334158714" visible="true" version="2" changeset="159209920" timestamp="2024-11-16T14:15:41Z" user="nabilersyad" uid="11419275" lat="7.0110271" lon="100.4744339" />\n  <node id="12334158715" visible="true" version="1" changeset="159040528" timestamp="2024-11-12T08:48:17Z" user="nabilersyad" uid="11419275" lat="7.0110215" lon="100.4742733" />\n  <node id="12339345626" visible="true" version="2" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0108065" lon="100.4753282" />\n  <node id="12339345627" visible="true" version="2" changeset="159211010" timestamp="2024-11-16T14:48:54Z" user="nabilersyad" uid="11419275" lat="7.0110249" lon="100.4750960" />\n  <node id="12339345628" visible="true" version="3" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0110965" lon="100.4741044" />\n  <node id="12339345629" visible="true" version="2" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0109895" lon="100.4753031" />\n  <node id="12339345630" visible="true" version="5" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0104256" lon="100.4763376" />\n  <node id="12339345631" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0106322" lon="100.4753593" />\n  <node id="12339345632" visible="true" version="2" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0109890" lon="100.4740867" />\n  <node id="12342535359" visible="true" version="1" changeset="159152002" timestamp="2024-11-15T01:37:20Z" user="nabilersyad" uid="11419275" lat="7.0108466" lon="100.4747874">\n    <tag k="barrier" v="gate" />\n  </node>\n  <node id="12342535364" visible="true" version="2" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0106790" lon="100.4753611">\n    <tag k="barrier" v="gate" />\n  </node>\n  <node id="12342535365" visible="true" version="2" changeset="159211010" timestamp="2024-11-16T14:48:54Z" user="nabilersyad" uid="11419275" lat="7.0110446" lon="100.4748204" />\n  <node id="12342535366" visible="true" version="3" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0107785" lon="100.4747772" />\n  <node id="12344163775" visible="true" version="1" changeset="159178805" timestamp="2024-11-15T16:39:46Z" user="nabilersyad" uid="11419275" lat="7.0081801" lon="100.4724084" />\n  <node id="12344163776" visible="true" version="1" changeset="159178805" timestamp="2024-11-15T16:39:46Z" user="nabilersyad" uid="11419275" lat="7.0081093" lon="100.4719864" />\n  <node id="12344163777" visible="true" version="1" changeset="159178805" timestamp="2024-11-15T16:39:46Z" user="nabilersyad" uid="11419275" lat="7.0079193" lon="100.4725318" />\n  <node id="12344163778" visible="true" version="1" changeset="159178805" timestamp="2024-11-15T16:39:46Z" user="nabilersyad" uid="11419275" lat="7.0078192" lon="100.4720337" />\n  <node id="12345104265" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0105432" lon="100.4757332" />\n  <node id="12345104268" visible="true" version="2" changeset="159209920" timestamp="2024-11-16T14:15:41Z" user="nabilersyad" uid="11419275" lat="7.0105668" lon="100.4757389" />\n  <node id="12346099049" visible="true" version="1" changeset="159211010" timestamp="2024-11-16T14:48:54Z" user="nabilersyad" uid="11419275" lat="7.0108727" lon="100.4753190" />\n  <node id="12346099050" visible="true" version="1" changeset="159211010" timestamp="2024-11-16T14:48:54Z" user="nabilersyad" uid="11419275" lat="7.0121322" lon="100.4723829" />\n  <node id="12346099051" visible="true" version="1" changeset="159211010" timestamp="2024-11-16T14:48:54Z" user="nabilersyad" uid="11419275" lat="7.0119113" lon="100.4723837" />\n  <node id="12346099052" visible="true" version="1" changeset="159211010" timestamp="2024-11-16T14:48:54Z" user="nabilersyad" uid="11419275" lat="7.0117152" lon="100.4732980" />\n  <node id="12346099053" visible="true" version="2" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0111575" lon="100.4740631" />\n  <node id="12346099054" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0106818" lon="100.4764688" />\n  <node id="12346099060" visible="true" version="1" changeset="159211010" timestamp="2024-11-16T14:48:54Z" user="nabilersyad" uid="11419275" lat="7.0105100" lon="100.4759809" />\n  <node id="12346117423" visible="true" version="1" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0101978" lon="100.4752949" />\n  <node id="12346117424" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0105875" lon="100.4755463" />\n  <node id="12346117425" visible="true" version="1" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0104414" lon="100.4755115" />\n  <node id="12346117426" visible="true" version="1" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0104828" lon="100.4753288" />\n  <node id="12346117457" visible="true" version="1" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0110723" lon="100.4736024" />\n  <node id="12346123215" visible="true" version="1" changeset="159211712" timestamp="2024-11-16T15:09:46Z" user="nabilersyad" uid="11419275" lat="7.0040462" lon="100.4685541" />\n  <node id="12346141721" visible="true" version="1" changeset="159211955" timestamp="2024-11-16T15:16:41Z" user="nabilersyad" uid="11419275" lat="7.0078757" lon="100.4708645" />\n  <node id="12351390166" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0103116" lon="100.4769785" />\n  <node id="12351390167" visible="true" version="1" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0104860" lon="100.4771946" />\n  <node id="12351390168" visible="true" version="1" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0105486" lon="100.4775533" />\n  <node id="12351390169" visible="true" version="1" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0105486" lon="100.4773521" />\n  <node id="12351390175" visible="true" version="1" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0109572" lon="100.4778517" />\n  <node id="12351390176" visible="true" version="1" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0109792" lon="100.4777303" />\n  <node id="12351390177" visible="true" version="1" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0109852" lon="100.4773542" />\n  <node id="12351390178" visible="true" version="1" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0109026" lon="100.4770276" />\n  <node id="12351390179" visible="true" version="1" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0107855" lon="100.4766541" />\n  <node id="12351390199" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0109828" lon="100.4780208" />\n  <node id="12351390200" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0111224" lon="100.4780786" />\n  <node id="12351400001" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0110324" lon="100.4760973" />\n  <node id="12351400002" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0110909" lon="100.4759535" />\n  <node id="12351400003" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0111475" lon="100.4758893" />\n  <node id="12351400004" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0113176" lon="100.4757871" />\n  <node id="12351400010" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0117139" lon="100.4757566" />\n  <node id="12351400019" visible="true" version="1" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0119375" lon="100.4749623" />\n  <node id="12351400020" visible="true" version="1" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0118324" lon="100.4748617" />\n  <node id="12351400021" visible="true" version="1" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0116074" lon="100.4746887" />\n  <node id="12351400022" visible="true" version="1" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0113492" lon="100.4744849" />\n  <node id="12518064113" visible="true" version="1" changeset="161557772" timestamp="2025-01-20T10:53:05Z" user="GrabTH_Praiya" uid="18100118" lat="7.0099926" lon="100.4834593" />\n  <node id="12649445299" visible="true" version="1" changeset="163432352" timestamp="2025-03-10T09:06:18Z" user="GrabTH_Praiya" uid="18100118" lat="7.0063169" lon="100.4754109" />\n  <node id="12649445300" visible="true" version="1" changeset="163432352" timestamp="2025-03-10T09:06:18Z" user="GrabTH_Praiya" uid="18100118" lat="7.0066366" lon="100.4745790" />\n  <node id="12722329099" visible="true" version="1" changeset="164407032" timestamp="2025-04-02T07:53:15Z" user="GrabTH_Panuwit" uid="16483995" lat="7.0023836" lon="100.4717600" />\n  <node id="12722329100" visible="true" version="1" changeset="164407032" timestamp="2025-04-02T07:53:15Z" user="GrabTH_Panuwit" uid="16483995" lat="7.0024270" lon="100.4720495" />\n  <node id="12722387501" visible="true" version="1" changeset="164407032" timestamp="2025-04-02T07:53:15Z" user="GrabTH_Panuwit" uid="16483995" lat="7.0028753" lon="100.4719869" />\n  <node id="12722387502" visible="true" version="2" changeset="186900771" timestamp="2026-08-04T03:39:24Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0024106" lon="100.4719296" />\n  <node id="12722387503" visible="true" version="2" changeset="186900771" timestamp="2026-08-04T03:39:24Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0027444" lon="100.4718877" />\n  <node id="12722387504" visible="true" version="1" changeset="164407032" timestamp="2025-04-02T07:53:15Z" user="GrabTH_Panuwit" uid="16483995" lat="7.0027612" lon="100.4720028" />\n  <node id="12722387505" visible="true" version="2" changeset="182517970" timestamp="2026-05-11T15:45:42Z" user="Emma Js" uid="22434897" lat="7.0030503" lon="100.4738818" />\n  <node id="12722387506" visible="true" version="2" changeset="182517970" timestamp="2026-05-11T15:45:42Z" user="Emma Js" uid="22434897" lat="7.0030554" lon="100.4740195" />\n  <node id="13279030938" visible="true" version="3" changeset="177551471" timestamp="2026-01-22T07:08:36Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0072979" lon="100.4790742" />\n  <node id="13279030939" visible="true" version="2" changeset="175448516" timestamp="2025-12-03T14:01:18Z" user="WorayutTK" uid="23562084" lat="7.0069617" lon="100.4790914" />\n  <node id="13279030940" visible="true" version="2" changeset="175448516" timestamp="2025-12-03T14:01:18Z" user="WorayutTK" uid="23562084" lat="7.0069610" lon="100.4788156" />\n  <node id="13335035783" visible="true" version="1" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="6.9984733" lon="100.4794359" />\n  <node id="13336792952" visible="true" version="2" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0071936" lon="100.4756547" />\n  <node id="13339819885" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0032122" lon="100.4754091" />\n  <node id="13339819886" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0033230" lon="100.4754643" />\n  <node id="13339819887" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0032361" lon="100.4756274" />\n  <node id="13339819888" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0031810" lon="100.4754888" />\n  <node id="13339819889" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0031756" lon="100.4754709" />\n  <node id="13339819890" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0031763" lon="100.4754516" />\n  <node id="13339819891" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0031832" lon="100.4754336" />\n  <node id="13339819892" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0032880" lon="100.4754490" />\n  <node id="13339819893" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0032896" lon="100.4754676" />\n  <node id="13339819894" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0032852" lon="100.4754858" />\n  <node id="13339819895" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0032752" lon="100.4755016" />\n  <node id="13339819896" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0031957" lon="100.4754189" />\n  <node id="13339819897" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0032311" lon="100.4754054" />\n  <node id="13339819898" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0032501" lon="100.4754082" />\n  <node id="13339819899" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0032671" lon="100.4754172" />\n  <node id="13339819900" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0032802" lon="100.4754313" />\n  <node id="13339844001" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0032608" lon="100.4755134" />\n  <node id="13339844002" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0032433" lon="100.4755199" />\n  <node id="13339844003" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0032247" lon="100.4755204" />\n  <node id="13339844004" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0032070" lon="100.4755149" />\n  <node id="13339844005" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0031919" lon="100.4755040" />\n  <node id="13340060055" visible="true" version="3" changeset="181643473" timestamp="2026-04-21T19:49:51Z" user="E1ectros" uid="24018730" lat="7.0041968" lon="100.4808443" />\n  <node id="13340060822" visible="true" version="3" changeset="183219774" timestamp="2026-05-26T17:09:26Z" user="anthaas" uid="5438501" lat="7.0041733" lon="100.4815018" />\n  <node id="13340060823" visible="true" version="2" changeset="177509855" timestamp="2026-01-21T08:26:46Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0042316" lon="100.4817118" />\n  <node id="13343573410" visible="true" version="2" changeset="188339904" timestamp="2026-09-01T02:58:55Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0027204" lon="100.4733653" />\n  <node id="13347563565" visible="true" version="1" changeset="175394828" timestamp="2025-12-02T10:05:36Z" user="JethroChen" uid="23301387" lat="7.0018563" lon="100.4721752" />\n  <node id="13347665462" visible="true" version="2" changeset="175398381" timestamp="2025-12-02T11:34:55Z" user="Riie" uid="23542111" lat="7.0091756" lon="100.4694738" />\n  <node id="13347665463" visible="true" version="1" changeset="175398360" timestamp="2025-12-02T11:34:36Z" user="Riie" uid="23542111" lat="7.0090653" lon="100.4694357" />\n  <node id="13348166868" visible="true" version="2" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0065861" lon="100.4741100" />\n  <node id="13348305823" visible="true" version="1" changeset="175413044" timestamp="2025-12-02T17:35:03Z" user="Jostein Heggebø" uid="23570667" lat="7.0127357" lon="100.4849431" />\n  <node id="13348305832" visible="true" version="1" changeset="175413044" timestamp="2025-12-02T17:35:03Z" user="Jostein Heggebø" uid="23570667" lat="7.0126384" lon="100.4835824" />\n  <node id="13351099349" visible="true" version="2" changeset="181556965" timestamp="2026-04-19T22:13:33Z" user="wireguy" uid="9564652" lat="7.0033634" lon="100.4809414" />\n  <node id="13351100149" visible="true" version="2" changeset="177513068" timestamp="2026-01-21T09:47:33Z" user="Lauriegisborne" uid="23651680" lat="7.0082037" lon="100.4760403" />\n  <node id="13351102290" visible="true" version="1" changeset="175442252" timestamp="2025-12-03T11:20:57Z" user="WorayutTK" uid="23562084" lat="7.0043558" lon="100.4780512" />\n  <node id="13351102291" visible="true" version="1" changeset="175442252" timestamp="2025-12-03T11:20:57Z" user="WorayutTK" uid="23562084" lat="7.0042728" lon="100.4780578" />\n  <node id="13351118871" visible="true" version="1" changeset="175441976" timestamp="2025-12-03T11:14:45Z" user="WorayutTK" uid="23562084" lat="7.0072268" lon="100.4763833" />\n  <node id="13351118872" visible="true" version="1" changeset="175441976" timestamp="2025-12-03T11:14:45Z" user="WorayutTK" uid="23562084" lat="7.0072230" lon="100.4762686" />\n  <node id="13351118905" visible="true" version="1" changeset="175441976" timestamp="2025-12-03T11:14:45Z" user="WorayutTK" uid="23562084" lat="7.0078270" lon="100.4768806" />\n  <node id="13351186096" visible="true" version="1" changeset="175444256" timestamp="2025-12-03T12:18:40Z" user="WorayutTK" uid="23562084" lat="7.0077526" lon="100.4778652" />\n  <node id="13351202525" visible="true" version="1" changeset="175444533" timestamp="2025-12-03T12:26:55Z" user="WorayutTK" uid="23562084" lat="7.0047606" lon="100.4790238" />\n  <node id="13351219789" visible="true" version="1" changeset="175446219" timestamp="2025-12-03T13:12:39Z" user="WorayutTK" uid="23562084" lat="7.0061177" lon="100.4760384" />\n  <node id="13351323871" visible="true" version="1" changeset="175446735" timestamp="2025-12-03T13:24:02Z" user="WorayutTK" uid="23562084" lat="7.0052501" lon="100.4794298" />\n  <node id="13351323872" visible="true" version="1" changeset="175446735" timestamp="2025-12-03T13:24:02Z" user="WorayutTK" uid="23562084" lat="7.0051149" lon="100.4794797" />\n  <node id="13351357060" visible="true" version="1" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084" lat="7.0048051" lon="100.4800764" />\n  <node id="13351357076" visible="true" version="1" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084" lat="7.0048088" lon="100.4802706" />\n  <node id="13351357081" visible="true" version="1" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084" lat="7.0049381" lon="100.4805102" />\n  <node id="13351357082" visible="true" version="1" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084" lat="7.0048129" lon="100.4804837" />\n  <node id="13351357083" visible="true" version="1" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084" lat="7.0048111" lon="100.4803909" />\n  <node id="13351357158" visible="true" version="1" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084" lat="7.0041818" lon="100.4782660" />\n  <node id="13351357162" visible="true" version="1" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084" lat="7.0039351" lon="100.4783164" />\n  <node id="13351357163" visible="true" version="1" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084" lat="7.0038849" lon="100.4783267" />\n  <node id="13351357166" visible="true" version="1" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084" lat="7.0037236" lon="100.4783596" />\n  <node id="13351357179" visible="true" version="1" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084" lat="7.0037341" lon="100.4787159" />\n  <node id="13351357180" visible="true" version="1" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084" lat="7.0035665" lon="100.4787190" />\n  <node id="13351357183" visible="true" version="1" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084" lat="7.0039152" lon="100.4787127" />\n  <node id="13351357184" visible="true" version="1" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084" lat="7.0037562" lon="100.4787155" />\n  <node id="13351357188" visible="true" version="1" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084" lat="7.0039420" lon="100.4787122" />\n  <node id="13351357189" visible="true" version="1" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084" lat="7.0041035" lon="100.4787092" />\n  <node id="13362936996" visible="true" version="1" changeset="175694769" timestamp="2025-12-09T03:57:22Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0002028" lon="100.4822090" />\n  <node id="13362936997" visible="true" version="1" changeset="175694769" timestamp="2025-12-09T03:57:22Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0015737" lon="100.4802120" />\n  <node id="13362936998" visible="true" version="2" changeset="177509855" timestamp="2026-01-21T08:26:46Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0045105" lon="100.4821473" />\n  <node id="13362950402" visible="true" version="1" changeset="175694769" timestamp="2025-12-09T03:57:22Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0012013" lon="100.4818389" />\n  <node id="13366620095" visible="true" version="1" changeset="175751416" timestamp="2025-12-10T10:18:27Z" user="neeli_srikanth" uid="12625709" lat="7.0093256" lon="100.4695255" />\n  <node id="13369238458" visible="true" version="1" changeset="175796005" timestamp="2025-12-11T10:36:57Z" user="neeli_srikanth" uid="12625709" lat="7.0127202" lon="100.4847271" />\n  <node id="13383541754" visible="true" version="2" changeset="182517970" timestamp="2026-05-11T15:45:42Z" user="Emma Js" uid="22434897" lat="7.0063590" lon="100.4726339" />\n  <node id="13383541755" visible="true" version="1" changeset="176054515" timestamp="2025-12-17T14:51:59Z" user="Sarah Goy" uid="23622628" lat="7.0063213" lon="100.4724713" />\n  <node id="13383541761" visible="true" version="2" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0066679" lon="100.4736570" />\n  <node id="13383541762" visible="true" version="2" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0067344" lon="100.4738280" />\n  <node id="13383541790" visible="true" version="2" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0072670" lon="100.4734300" />\n  <node id="13383672794" visible="true" version="1" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0152211" lon="100.4839468" />\n  <node id="13386913506" visible="true" version="1" changeset="176104364" timestamp="2025-12-18T16:45:43Z" user="angela_wl" uid="6666296" lat="7.0069688" lon="100.4712767" />\n  <node id="13386931999" visible="true" version="2" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0073266" lon="100.4730228" />\n  <node id="13388461897" visible="true" version="1" changeset="176134821" timestamp="2025-12-19T10:51:27Z" user="Klank123" uid="23623862" lat="7.0074468" lon="100.4830147" />\n  <node id="13388461898" visible="true" version="1" changeset="176134821" timestamp="2025-12-19T10:51:27Z" user="Klank123" uid="23623862" lat="7.0074941" lon="100.4832340" />\n  <node id="13388478046" visible="true" version="1" changeset="176134821" timestamp="2025-12-19T10:51:27Z" user="Klank123" uid="23623862" lat="7.0061579" lon="100.4839720" />\n  <node id="13388478047" visible="true" version="1" changeset="176134821" timestamp="2025-12-19T10:51:27Z" user="Klank123" uid="23623862" lat="7.0059391" lon="100.4840030" />\n  <node id="13388478155" visible="true" version="1" changeset="176134821" timestamp="2025-12-19T10:51:27Z" user="Klank123" uid="23623862" lat="7.0067780" lon="100.4811079" />\n  <node id="13388503443" visible="true" version="1" changeset="176135435" timestamp="2025-12-19T11:03:00Z" user="Klank123" uid="23623862" lat="7.0062832" lon="100.4835987" />\n  <node id="13388503445" visible="true" version="1" changeset="176135435" timestamp="2025-12-19T11:03:00Z" user="Klank123" uid="23623862" lat="7.0062237" lon="100.4831490" />\n  <node id="13418313524" visible="true" version="2" changeset="182517970" timestamp="2026-05-11T15:45:42Z" user="Emma Js" uid="22434897" lat="7.0049791" lon="100.4733813" />\n  <node id="13424210990" visible="true" version="1" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0078631" lon="100.4736661" />\n  <node id="13424220803" visible="true" version="2" changeset="186902224" timestamp="2026-08-04T04:46:49Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0082565" lon="100.4735882" />\n  <node id="13424220804" visible="true" version="2" changeset="186902224" timestamp="2026-08-04T04:46:49Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0081828" lon="100.4731894" />\n  <node id="13424220807" visible="true" version="1" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0067964" lon="100.4732089" />\n  <node id="13424220808" visible="true" version="1" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0065491" lon="100.4732780" />\n  <node id="13424220819" visible="true" version="1" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0060454" lon="100.4747542" />\n  <node id="13424220820" visible="true" version="1" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0059548" lon="100.4745585" />\n  <node id="13424220821" visible="true" version="1" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0061085" lon="100.4748872" />\n  <node id="13424220822" visible="true" version="1" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0061335" lon="100.4749646" />\n  <node id="13424220823" visible="true" version="1" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0061266" lon="100.4750246" />\n  <node id="13424220824" visible="true" version="1" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0060567" lon="100.4750551" />\n  <node id="13424220825" visible="true" version="1" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0060066" lon="100.4751977" />\n  <node id="13424220826" visible="true" version="1" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0058884" lon="100.4754438" />\n  <node id="13424220827" visible="true" version="1" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0058167" lon="100.4755734" />\n  <node id="13433676469" visible="true" version="1" changeset="176885166" timestamp="2026-01-06T02:15:15Z" user="ahartman113" uid="23417960" lat="7.0055594" lon="100.4822893" />\n  <node id="13433688987" visible="true" version="1" changeset="176885644" timestamp="2026-01-06T02:46:55Z" user="Rida K" uid="23358460" lat="7.0006214" lon="100.4718172" />\n  <node id="13449811511" visible="true" version="1" changeset="177131310" timestamp="2026-01-12T10:34:41Z" user="djmorkman" uid="5547549" lat="7.0052931" lon="100.4823578" />\n  <node id="13457981819" visible="true" version="2" changeset="183269728" timestamp="2026-05-27T18:24:44Z" user="anthaas" uid="5438501" lat="7.0079170" lon="100.4702473" />\n  <node id="13458847473" visible="true" version="1" changeset="177274810" timestamp="2026-01-15T20:57:54Z" user="Callipo" uid="23344489" lat="7.0038754" lon="100.4833424" />\n  <node id="13458847530" visible="true" version="1" changeset="177274810" timestamp="2026-01-15T20:57:54Z" user="Callipo" uid="23344489" lat="7.0061540" lon="100.4827611" />\n  <node id="13472156175" visible="true" version="1" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0142156" lon="100.4761845" />\n  <node id="13472156176" visible="true" version="1" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0143370" lon="100.4761073" />\n  <node id="13472156177" visible="true" version="1" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0143955" lon="100.4760631" />\n  <node id="13472222395" visible="true" version="1" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0143211" lon="100.4755124" />\n  <node id="13472222396" visible="true" version="1" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0142925" lon="100.4756931" />\n  <node id="13472222397" visible="true" version="1" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0142657" lon="100.4757327" />\n  <node id="13472222398" visible="true" version="1" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0143074" lon="100.4756536" />\n  <node id="13472222399" visible="true" version="1" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0143195" lon="100.4755649" />\n  <node id="13472222400" visible="true" version="1" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0145853" lon="100.4758287" />\n  <node id="13472222401" visible="true" version="1" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0145964" lon="100.4757947" />\n  <node id="13472222402" visible="true" version="1" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0145120" lon="100.4759475" />\n  <node id="13472222423" visible="true" version="1" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0146130" lon="100.4756684" />\n  <node id="13472222424" visible="true" version="1" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0146089" lon="100.4757238" />\n  <node id="13472440726" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0114230" lon="100.4785208" />\n  <node id="13472440727" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0112558" lon="100.4779279" />\n  <node id="13472440746" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0099653" lon="100.4769051" />\n  <node id="13472440747" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0097915" lon="100.4768318" />\n  <node id="13472440748" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0092610" lon="100.4768357" />\n  <node id="13472440749" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0092296" lon="100.4769484" />\n  <node id="13472440750" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0092794" lon="100.4768187" />\n  <node id="13472440751" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0092415" lon="100.4768591" />\n  <node id="13472440752" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0092608" lon="100.4770590" />\n  <node id="13472440753" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0092346" lon="100.4769833" />\n  <node id="13472440754" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0092301" lon="100.4769187" />\n  <node id="13472440755" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0093614" lon="100.4768016" />\n  <node id="13472440756" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0093304" lon="100.4768024" />\n  <node id="13472440757" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0095225" lon="100.4774453" />\n  <node id="13472440758" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0095036" lon="100.4774061" />\n  <node id="13472440759" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0095383" lon="100.4774856" />\n  <node id="13472440760" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0095678" lon="100.4777342" />\n  <node id="13472440761" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0095688" lon="100.4776328" />\n  <node id="13472440762" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0095661" lon="100.4775815" />\n  <node id="13472440763" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0095696" lon="100.4776848" />\n  <node id="13472440764" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0095584" lon="100.4778332" />\n  <node id="13472440765" visible="true" version="2" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0102614" lon="100.4771820" />\n  <node id="13472440766" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0103050" lon="100.4773288" />\n  <node id="13472440767" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0103073" lon="100.4773904" />\n  <node id="13472440768" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0102977" lon="100.4772775" />\n  <node id="13472440769" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0102042" lon="100.4775909" />\n  <node id="13472440770" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0101805" lon="100.4776240" />\n  <node id="13472440771" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0102747" lon="100.4775008" />\n  <node id="13472440772" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0101691" lon="100.4776480" />\n  <node id="13472440773" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0093635" lon="100.4783624" />\n  <node id="13472440774" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0092557" lon="100.4785057" />\n  <node id="13472440775" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0091205" lon="100.4793166" />\n  <node id="13472440778" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0082093" lon="100.4788029" />\n  <node id="13472440779" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0081694" lon="100.4788046" />\n  <node id="13472440780" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0081459" lon="100.4788026" />\n  <node id="13472440781" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0081870" lon="100.4788046" />\n  <node id="13472440782" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0081512" lon="100.4785448" />\n  <node id="13472440783" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0081620" lon="100.4785633" />\n  <node id="13472440784" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0082034" lon="100.4785694" />\n  <node id="13472440785" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0081469" lon="100.4785231" />\n  <node id="13472440786" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0082698" lon="100.4775870" />\n  <node id="13472440787" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0082704" lon="100.4776281" />\n  <node id="13472440788" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0082644" lon="100.4775434" />\n  <node id="13472440789" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0088927" lon="100.4765344" />\n  <node id="13472440790" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0097584" lon="100.4766068" />\n  <node id="13472440791" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0098316" lon="100.4766599" />\n  <node id="13472440792" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0102402" lon="100.4769141" />\n  <node id="13472440793" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0110031" lon="100.4762515" />\n  <node id="13472440794" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0118242" lon="100.4757117" />\n  <node id="13472440795" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0120641" lon="100.4754809" />\n  <node id="13472440796" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0119864" lon="100.4755740" />\n  <node id="13472440797" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0117783" lon="100.4757376" />\n  <node id="13472440798" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0117461" lon="100.4757511" />\n  <node id="13472440799" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0111192" lon="100.4759188" />\n  <node id="13472440800" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0110727" lon="100.4759852" />\n  <node id="13472452501" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0113629" lon="100.4757716" />\n  <node id="13472452502" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0112745" lon="100.4758048" />\n  <node id="13472452524" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0116309" lon="100.4755139" />\n  <node id="13472452525" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0116632" lon="100.4755045" />\n  <node id="13472452526" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0118373" lon="100.4753642" />\n  <node id="13472452527" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0117970" lon="100.4754260" />\n  <node id="13472452528" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0117348" lon="100.4754697" />\n  <node id="13472452529" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0109547" lon="100.4756698" />\n  <node id="13472452530" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0109353" lon="100.4756886" />\n  <node id="13472452531" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0109788" lon="100.4756527" />\n  <node id="13472452532" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0106749" lon="100.4764213" />\n  <node id="13472452542" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0128574" lon="100.4780954" />\n  <node id="13472452543" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0128905" lon="100.4779335" />\n  <node id="13472452544" visible="true" version="2" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0128181" lon="100.4795800" />\n  <node id="13472452545" visible="true" version="2" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0128579" lon="100.4796690" />\n  <node id="13472452546" visible="true" version="2" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0127956" lon="100.4794991" />\n  <node id="13472452547" visible="true" version="2" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0128390" lon="100.4796294" />\n  <node id="13472452548" visible="true" version="2" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0128783" lon="100.4797037" />\n  <node id="13472452549" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0135711" lon="100.4805579" />\n  <node id="13472452550" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0135051" lon="100.4805129" />\n  <node id="13472452551" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0134728" lon="100.4804862" />\n  <node id="13472452552" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0130662" lon="100.4799838" />\n  <node id="13472452553" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0129786" lon="100.4798608" />\n  <node id="13472452554" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0080516" lon="100.4787757" />\n  <node id="13472452555" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0080886" lon="100.4787900" />\n  <node id="13472452556" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0080179" lon="100.4787556" />\n  <node id="13472452557" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0079404" lon="100.4786524" />\n  <node id="13472452558" visible="true" version="1" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0079633" lon="100.4786959" />\n  <node id="13472483840" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0126562" lon="100.4797573" />\n  <node id="13472483841" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0126825" lon="100.4798081" />\n  <node id="13472483842" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0131395" lon="100.4804389" />\n  <node id="13472483843" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0134328" lon="100.4807465" />\n  <node id="13472483844" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0133457" lon="100.4806618" />\n  <node id="13472483853" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0110727" lon="100.4783524" />\n  <node id="13472483854" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0110125" lon="100.4784215" />\n  <node id="13472483855" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0110438" lon="100.4783894" />\n  <node id="13472483856" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0109752" lon="100.4784537" />\n  <node id="13472483857" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0109523" lon="100.4784675" />\n  <node id="13472483858" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0107706" lon="100.4784804" />\n  <node id="13472483859" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0108494" lon="100.4784805" />\n  <node id="13472483860" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0106829" lon="100.4784780" />\n  <node id="13472483861" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0104200" lon="100.4784607" />\n  <node id="13472483862" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0105031" lon="100.4784682" />\n  <node id="13472483863" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0103130" lon="100.4784532" />\n  <node id="13472483864" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0102161" lon="100.4783276" />\n  <node id="13472483865" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0102165" lon="100.4783860" />\n  <node id="13472483866" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0103038" lon="100.4774307" />\n  <node id="13472484404" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0108341" lon="100.4781367" />\n  <node id="13472484405" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0105739" lon="100.4780973" />\n  <node id="13472484406" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0107286" lon="100.4781644" />\n  <node id="13472484407" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0106982" lon="100.4781641" />\n  <node id="13472484408" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0106428" lon="100.4781542" />\n  <node id="13472484409" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0105943" lon="100.4781226" />\n  <node id="13472484410" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0105607" lon="100.4780684" />\n  <node id="13472484411" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0105363" lon="100.4779592" />\n  <node id="13472484412" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0109039" lon="100.4780508" />\n  <node id="13472484413" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0108568" lon="100.4781210" />\n  <node id="13472484414" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0108095" lon="100.4781493" />\n  <node id="13472484415" visible="true" version="1" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0107567" lon="100.4781623" />\n  <node id="13472970566" visible="true" version="1" changeset="177509855" timestamp="2026-01-21T08:26:46Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0041706" lon="100.4813673" />\n  <node id="13472970567" visible="true" version="2" changeset="183219774" timestamp="2026-05-26T17:09:27Z" user="anthaas" uid="5438501" lat="7.0041829" lon="100.4816060" />\n  <node id="13472970568" visible="true" version="1" changeset="177509855" timestamp="2026-01-21T08:26:46Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0040976" lon="100.4817489" />\n  <node id="13472970569" visible="true" version="1" changeset="177509855" timestamp="2026-01-21T08:26:46Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0042280" lon="100.4822275" />\n  <node id="13472974657" visible="true" version="1" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0046307" lon="100.4754080" />\n  <node id="13472974658" visible="true" version="1" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0045864" lon="100.4753693" />\n  <node id="13473121453" visible="true" version="1" changeset="177513068" timestamp="2026-01-21T09:47:33Z" user="Lauriegisborne" uid="23651680" lat="7.0085314" lon="100.4760360" />\n  <node id="13473142237" visible="true" version="1" changeset="177513068" timestamp="2026-01-21T09:47:33Z" user="Lauriegisborne" uid="23651680" lat="7.0094459" lon="100.4773114" />\n  <node id="13473142490" visible="true" version="1" changeset="177513068" timestamp="2026-01-21T09:47:33Z" user="Lauriegisborne" uid="23651680" lat="7.0116320" lon="100.4781143" />\n  <node id="13473142491" visible="true" version="1" changeset="177513068" timestamp="2026-01-21T09:47:33Z" user="Lauriegisborne" uid="23651680" lat="7.0118333" lon="100.4781079" />\n  <node id="13475596787" visible="true" version="1" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0114136" lon="100.4742767">\n    <tag k="access" v="private" />\n    <tag k="barrier" v="gate" />\n  </node>\n  <node id="13475596788" visible="true" version="1" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0121866" lon="100.4753385">\n    <tag k="access" v="private" />\n    <tag k="barrier" v="gate" />\n  </node>\n  <node id="13475596789" visible="true" version="1" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0123455" lon="100.4745944" />\n  <node id="13475596790" visible="true" version="1" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0123085" lon="100.4745171" />\n  <node id="13545019681" visible="true" version="1" changeset="178433158" timestamp="2026-02-12T03:02:18Z" user="GrabTH_Tanyaporn" uid="22904804" lat="7.0077171" lon="100.4768992" />\n  <node id="13575308874" visible="true" version="1" changeset="178716277" timestamp="2026-02-18T09:37:15Z" user="neeli_srikanth" uid="12625709" lat="7.0071822" lon="100.4721545" />\n  <node id="13579736619" visible="true" version="1" changeset="178756408" timestamp="2026-02-19T06:59:09Z" user="neeli_srikanth" uid="12625709" lat="7.0082561" lon="100.4777794" />\n  <node id="1369020925" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0098191" lon="100.4797749" />\n  <node id="1369020939" visible="true" version="4" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0127196" lon="100.4798711" />\n  <node id="1369020941" visible="true" version="1" changeset="8788516" timestamp="2011-07-21T13:17:15Z" user="sdayu" uid="300693" lat="7.0074191" lon="100.4685290" />\n  <node id="1369020942" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0123335" lon="100.4800029" />\n  <node id="1369020943" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0105515" lon="100.4798697" />\n  <node id="1369020950" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0091619" lon="100.4797938" />\n  <node id="13884411592" visible="true" version="1" changeset="183220852" timestamp="2026-05-26T17:32:35Z" user="anthaas" uid="5438501" lat="7.0065247" lon="100.4763597" />\n  <node id="13884935873" visible="true" version="1" changeset="183223422" timestamp="2026-05-26T18:31:19Z" user="anthaas" uid="5438501" lat="7.0016490" lon="100.4824606" />\n  <node id="13985351056" visible="true" version="1" changeset="185031751" timestamp="2026-07-03T10:22:57Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0000090" lon="100.4799402" />\n  <node id="14063214194" visible="true" version="1" changeset="186687676" timestamp="2026-07-31T06:26:34Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0007573" lon="100.4733255" />\n  <node id="14063214195" visible="true" version="1" changeset="186687676" timestamp="2026-07-31T06:26:34Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0010447" lon="100.4734283" />\n  <node id="14063214196" visible="true" version="1" changeset="186687676" timestamp="2026-07-31T06:26:34Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0010954" lon="100.4735361" />\n  <node id="14063214198" visible="true" version="1" changeset="186687676" timestamp="2026-07-31T06:26:34Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0009592" lon="100.4737192" />\n  <node id="14063214199" visible="true" version="2" changeset="188530562" timestamp="2026-09-04T15:11:35Z" user="Pan_Nug" uid="17280926" lat="7.0008537" lon="100.4734916" />\n  <node id="14063214200" visible="true" version="1" changeset="186687676" timestamp="2026-07-31T06:26:34Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0009612" lon="100.4737153" />\n  <node id="14063217001" visible="true" version="1" changeset="186687774" timestamp="2026-07-31T06:29:47Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0079166" lon="100.4809991" />\n  <node id="14063217002" visible="true" version="2" changeset="188752824" timestamp="2026-09-09T04:48:29Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0076755" lon="100.4800257" />\n  <node id="14063217003" visible="true" version="1" changeset="186687774" timestamp="2026-07-31T06:29:47Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0078957" lon="100.4809085">\n    <tag k="access" v="private" />\n    <tag k="barrier" v="lift_gate" />\n    <tag k="lift_gate:type" v="double" />\n  </node>\n  <node id="14063278749" visible="true" version="1" changeset="186690029" timestamp="2026-07-31T07:37:35Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0080499" lon="100.4797848" />\n  <node id="14063449046" visible="true" version="1" changeset="186694625" timestamp="2026-07-31T09:25:56Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0087314" lon="100.4871931" />\n  <node id="14063449049" visible="true" version="1" changeset="186694625" timestamp="2026-07-31T09:25:56Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0087372" lon="100.4870014" />\n  <node id="14063537293" visible="true" version="1" changeset="186695948" timestamp="2026-07-31T09:54:46Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0094245" lon="100.4833350" />\n  <node id="14063655439" visible="true" version="1" changeset="186698599" timestamp="2026-07-31T10:51:39Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0074343" lon="100.4829554" />\n  <node id="14069947700" visible="true" version="1" changeset="186853397" timestamp="2026-08-03T07:28:04Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0043831" lon="100.4724126" />\n  <node id="14069967501" visible="true" version="1" changeset="186853397" timestamp="2026-08-03T07:28:04Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0047209" lon="100.4723416" />\n  <node id="14069967502" visible="true" version="1" changeset="186853397" timestamp="2026-08-03T07:28:04Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0044457" lon="100.4723996">\n    <tag k="access" v="private" />\n    <tag k="barrier" v="gate" />\n  </node>\n  <node id="14069981565" visible="true" version="1" changeset="186854324" timestamp="2026-08-03T07:49:55Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0050867" lon="100.4713947" />\n  <node id="14069981566" visible="true" version="1" changeset="186854324" timestamp="2026-08-03T07:49:55Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0051037" lon="100.4717904" />\n  <node id="14069981567" visible="true" version="1" changeset="186854324" timestamp="2026-08-03T07:49:55Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0050563" lon="100.4718534" />\n  <node id="14069981568" visible="true" version="1" changeset="186854324" timestamp="2026-08-03T07:49:55Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0048051" lon="100.4718920" />\n  <node id="14069981569" visible="true" version="1" changeset="186854324" timestamp="2026-08-03T07:49:55Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0050921" lon="100.4715200">\n    <tag k="access" v="private" />\n    <tag k="barrier" v="gate" />\n  </node>\n  <node id="14069981570" visible="true" version="1" changeset="186854324" timestamp="2026-08-03T07:49:55Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0051027" lon="100.4717672">\n    <tag k="access" v="private" />\n    <tag k="barrier" v="gate" />\n  </node>\n  <node id="14069996590" visible="true" version="1" changeset="186855589" timestamp="2026-08-03T08:16:33Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0011509" lon="100.4802731" />\n  <node id="14070071629" visible="true" version="1" changeset="186857988" timestamp="2026-08-03T09:07:18Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0009871" lon="100.4725976" />\n  <node id="14070071630" visible="true" version="1" changeset="186857988" timestamp="2026-08-03T09:07:18Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0011730" lon="100.4731726" />\n  <node id="14071847283" visible="true" version="1" changeset="186900335" timestamp="2026-08-04T03:24:25Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0034718" lon="100.4803712" />\n  <node id="14071847284" visible="true" version="1" changeset="186900335" timestamp="2026-08-04T03:24:25Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0034458" lon="100.4805149" />\n  <node id="14071847285" visible="true" version="1" changeset="186900335" timestamp="2026-08-04T03:24:25Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0034607" lon="100.4804354" />\n  <node id="14071847286" visible="true" version="1" changeset="186900335" timestamp="2026-08-04T03:24:25Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0034015" lon="100.4807533" />\n  <node id="14071847287" visible="true" version="1" changeset="186900335" timestamp="2026-08-04T03:24:25Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0033049" lon="100.4807374" />\n  <node id="14071847288" visible="true" version="1" changeset="186900335" timestamp="2026-08-04T03:24:25Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0038267" lon="100.4804304" />\n  <node id="14071847289" visible="true" version="1" changeset="186900335" timestamp="2026-08-04T03:24:25Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0038034" lon="100.4805694" />\n  <node id="14071847290" visible="true" version="1" changeset="186900335" timestamp="2026-08-04T03:24:25Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0038148" lon="100.4804937" />\n  <node id="14071882647" visible="true" version="1" changeset="186901490" timestamp="2026-08-04T04:06:24Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0054573" lon="100.4713437" />\n  <node id="14071882648" visible="true" version="1" changeset="186901490" timestamp="2026-08-04T04:06:24Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0055773" lon="100.4720774" />\n  <node id="14071882649" visible="true" version="1" changeset="186901490" timestamp="2026-08-04T04:06:24Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0057004" lon="100.4720581" />\n  <node id="14071882650" visible="true" version="1" changeset="186901490" timestamp="2026-08-04T04:06:24Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0057945" lon="100.4724998" />\n  <node id="14071882651" visible="true" version="1" changeset="186901490" timestamp="2026-08-04T04:06:24Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0057806" lon="100.4724344">\n    <tag k="access" v="private" />\n    <tag k="barrier" v="lift_gate" />\n    <tag k="lift_gate:type" v="single" />\n  </node>\n  <node id="14071937367" visible="true" version="1" changeset="186901930" timestamp="2026-08-04T04:32:47Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0104615" lon="100.4793229" />\n  <node id="14071937368" visible="true" version="1" changeset="186901930" timestamp="2026-08-04T04:32:47Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0103874" lon="100.4795318" />\n  <node id="14071937369" visible="true" version="1" changeset="186901930" timestamp="2026-08-04T04:32:47Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0103827" lon="100.4796508" />\n  <node id="14071937370" visible="true" version="1" changeset="186901930" timestamp="2026-08-04T04:32:47Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0102182" lon="100.4796415" />\n  <node id="14071937371" visible="true" version="1" changeset="186901930" timestamp="2026-08-04T04:32:47Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0102160" lon="100.4793264" />\n  <node id="14071937372" visible="true" version="1" changeset="186901930" timestamp="2026-08-04T04:32:47Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0103981" lon="100.4793238" />\n  <node id="14071937373" visible="true" version="1" changeset="186901930" timestamp="2026-08-04T04:32:47Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0104645" lon="100.4795634" />\n  <node id="14071937374" visible="true" version="1" changeset="186901930" timestamp="2026-08-04T04:32:47Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0104631" lon="100.4794502" />\n  <node id="14071937375" visible="true" version="1" changeset="186901930" timestamp="2026-08-04T04:32:47Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0103916" lon="100.4794503" />\n  <node id="14071937376" visible="true" version="1" changeset="186901930" timestamp="2026-08-04T04:32:47Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0103869" lon="100.4795434" />\n  <node id="14071937377" visible="true" version="1" changeset="186901930" timestamp="2026-08-04T04:32:47Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0102174" lon="100.4795363" />\n  <node id="14072021538" visible="true" version="1" changeset="186902224" timestamp="2026-08-04T04:46:49Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0081998" lon="100.4732768">\n    <tag k="access" v="private" />\n    <tag k="barrier" v="gate" />\n  </node>\n  <node id="14072021539" visible="true" version="1" changeset="186902224" timestamp="2026-08-04T04:46:49Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0082494" lon="100.4735479">\n    <tag k="access" v="private" />\n    <tag k="barrier" v="gate" />\n  </node>\n  <node id="14124372093" visible="true" version="1" changeset="188028426" timestamp="2026-08-26T07:58:56Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0041811" lon="100.4825251" />\n  <node id="14124519874" visible="true" version="1" changeset="188031708" timestamp="2026-08-26T09:16:03Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0136130" lon="100.4762963" />\n  <node id="14124541207" visible="true" version="1" changeset="188032201" timestamp="2026-08-26T09:26:47Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0138607" lon="100.4767074" />\n  <node id="14127561644" visible="true" version="1" changeset="188075198" timestamp="2026-08-27T03:41:25Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0119622" lon="100.4734809" />\n  <node id="14127561645" visible="true" version="2" changeset="188075209" timestamp="2026-08-27T03:42:00Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0120491" lon="100.4734912">\n    <tag k="access" v="private" />\n    <tag k="barrier" v="gate" />\n  </node>\n  <node id="14127602686" visible="true" version="1" changeset="188076387" timestamp="2026-08-27T04:49:30Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0119280" lon="100.4712331" />\n  <node id="14127615692" visible="true" version="2" changeset="189137799" timestamp="2026-09-17T02:46:06Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0120370" lon="100.4778431" />\n  <node id="14127615693" visible="true" version="2" changeset="189137799" timestamp="2026-09-17T02:46:06Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0126109" lon="100.4778775" />\n  <node id="14127615694" visible="true" version="1" changeset="188076207" timestamp="2026-08-27T04:39:10Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0120472" lon="100.4776052" />\n  <node id="14138648129" visible="true" version="1" changeset="188339904" timestamp="2026-09-01T02:58:55Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0030315" lon="100.4736599" />\n  <node id="14138865123" visible="true" version="1" changeset="188345456" timestamp="2026-09-01T06:28:57Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0136267" lon="100.4752133">\n    <tag k="access" v="no" />\n    <tag k="barrier" v="gate" />\n  </node>\n  <node id="14138865124" visible="true" version="1" changeset="188345456" timestamp="2026-09-01T06:28:57Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0133362" lon="100.4752556">\n    <tag k="access" v="no" />\n    <tag k="barrier" v="gate" />\n  </node>\n  <node id="14144046929" visible="true" version="1" changeset="188459881" timestamp="2026-09-03T08:57:10Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0097957" lon="100.4784392" />\n  <node id="14148238620" visible="true" version="1" changeset="188513846" timestamp="2026-09-04T08:48:15Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0137525" lon="100.4693935" />\n  <node id="14148238623" visible="true" version="1" changeset="188513846" timestamp="2026-09-04T08:48:15Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0137958" lon="100.4697387" />\n  <node id="14148246304" visible="true" version="1" changeset="188513991" timestamp="2026-09-04T08:51:03Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0144281" lon="100.4702893" />\n  <node id="14148246306" visible="true" version="1" changeset="188513991" timestamp="2026-09-04T08:51:03Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0140191" lon="100.4703476" />\n  <node id="14148305416" visible="true" version="1" changeset="188516048" timestamp="2026-09-04T09:37:04Z" user="GrabTH_Pimwipa" uid="24297485" lat="6.9985315" lon="100.4715885" />\n  <node id="14148398712" visible="true" version="1" changeset="188517464" timestamp="2026-09-04T10:11:00Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0071056" lon="100.4734744" />\n  <node id="14148398713" visible="true" version="1" changeset="188517464" timestamp="2026-09-04T10:11:00Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0072546" lon="100.4739451" />\n  <node id="14159239425" visible="true" version="1" changeset="188662419" timestamp="2026-09-07T10:20:56Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0048615" lon="100.4775054" />\n  <node id="14159239426" visible="true" version="1" changeset="188662419" timestamp="2026-09-07T10:20:56Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0053586" lon="100.4776189" />\n  <node id="14159239427" visible="true" version="1" changeset="188662419" timestamp="2026-09-07T10:20:56Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0049763" lon="100.4769905" />\n  <node id="14159239428" visible="true" version="1" changeset="188662419" timestamp="2026-09-07T10:20:56Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0052546" lon="100.4771187" />\n  <node id="14159239429" visible="true" version="1" changeset="188662419" timestamp="2026-09-07T10:20:56Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0053121" lon="100.4771636" />\n  <node id="14165395419" visible="true" version="1" changeset="188752824" timestamp="2026-09-09T04:48:29Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0079510" lon="100.4799977" />\n  <node id="14165395420" visible="true" version="1" changeset="188752824" timestamp="2026-09-09T04:48:29Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0080079" lon="100.4800445" />\n  <node id="14165395421" visible="true" version="1" changeset="188752824" timestamp="2026-09-09T04:48:29Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0080956" lon="100.4806752" />\n  <node id="14165395422" visible="true" version="1" changeset="188752824" timestamp="2026-09-09T04:48:29Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0078499" lon="100.4807110" />\n  <node id="14165536477" visible="true" version="1" changeset="188756079" timestamp="2026-09-09T06:45:22Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0035284" lon="100.4791144" />\n  <node id="14165536478" visible="true" version="1" changeset="188756079" timestamp="2026-09-09T06:45:22Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0037233" lon="100.4794503" />\n  <node id="14165536479" visible="true" version="1" changeset="188756079" timestamp="2026-09-09T06:45:22Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0041541" lon="100.4794361" />\n  <node id="14165536480" visible="true" version="1" changeset="188756079" timestamp="2026-09-09T06:45:22Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0035267" lon="100.4790620">\n    <tag k="access" v="private" />\n    <tag k="barrier" v="gate" />\n  </node>\n  <node id="14165547215" visible="true" version="1" changeset="188756440" timestamp="2026-09-09T06:55:23Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0099084" lon="100.4797907" />\n  <node id="14165547216" visible="true" version="1" changeset="188756440" timestamp="2026-09-09T06:55:23Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0099047" lon="100.4794682" />\n  <node id="14165547217" visible="true" version="1" changeset="188756440" timestamp="2026-09-09T06:55:23Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0099491" lon="100.4803425" />\n  <node id="14165547218" visible="true" version="1" changeset="188756440" timestamp="2026-09-09T06:55:23Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0100085" lon="100.4798893" />\n  <node id="14165547219" visible="true" version="1" changeset="188756440" timestamp="2026-09-09T06:55:23Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0102627" lon="100.4799435" />\n  <node id="14165895527" visible="true" version="2" changeset="188759478" timestamp="2026-09-09T08:13:12Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0081276" lon="100.4823933" />\n  <node id="14165895528" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0079871" lon="100.4823733" />\n  <node id="14165895529" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0079067" lon="100.4820308" />\n  <node id="14165895530" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0079033" lon="100.4819993" />\n  <node id="14165895531" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0078856" lon="100.4819730" />\n  <node id="14165895532" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0077808" lon="100.4819992" />\n  <node id="14165895533" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0077781" lon="100.4820354" />\n  <node id="14165895534" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0077950" lon="100.4820676" />\n  <node id="14165895535" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0078262" lon="100.4820858" />\n  <node id="14165895536" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0078578" lon="100.4819582" />\n  <node id="14165895537" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0078263" lon="100.4819581" />\n  <node id="14165895538" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0077985" lon="100.4819729" />\n  <node id="14165895539" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0078623" lon="100.4820845" />\n  <node id="14165895540" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0078922" lon="100.4820641" />\n  <node id="14165895541" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0073516" lon="100.4825642" />\n  <node id="14165895542" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0076368" lon="100.4825013" />\n  <node id="14165895543" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0078068" lon="100.4828703" />\n  <node id="14165895544" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0078771" lon="100.4832163" />\n  <node id="14165895545" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0080113" lon="100.4831825" />\n  <node id="14165895546" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0078547" lon="100.4824888" />\n  <node id="14165895547" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0077285" lon="100.4825323" />\n  <node id="14165895548" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0081609" lon="100.4832806" />\n  <node id="14165895549" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0080938" lon="100.4830433" />\n  <node id="14165895550" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0082533" lon="100.4829850" />\n  <node id="14165895551" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0083027" lon="100.4831134" />\n  <node id="14165895552" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0081323" lon="100.4831795" />\n  <node id="14166007986" visible="true" version="1" changeset="188760331" timestamp="2026-09-09T08:34:32Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0106019" lon="100.4828077" />\n  <node id="14166007987" visible="true" version="1" changeset="188760331" timestamp="2026-09-09T08:34:32Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0102596" lon="100.4828254" />\n  <node id="14166086160" visible="true" version="1" changeset="188760598" timestamp="2026-09-09T08:41:34Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0074666" lon="100.4831063" />\n  <node id="14172689681" visible="true" version="1" changeset="188857437" timestamp="2026-09-11T07:30:20Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0136759" lon="100.4687836" />\n  <node id="14172689682" visible="true" version="1" changeset="188857437" timestamp="2026-09-11T07:30:20Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0132663" lon="100.4688386" />\n  <node id="14179759489" visible="true" version="1" changeset="188989640" timestamp="2026-09-14T04:03:52Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0113344" lon="100.4708517" />\n  <node id="14182032117" visible="true" version="1" changeset="189038359" timestamp="2026-09-15T02:25:53Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0138153" lon="100.4723066" />\n  <node id="14182032118" visible="true" version="1" changeset="189038359" timestamp="2026-09-15T02:25:53Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0136790" lon="100.4721874" />\n  <node id="14182032119" visible="true" version="1" changeset="189038359" timestamp="2026-09-15T02:25:53Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0121784" lon="100.4718212" />\n  <node id="14182062679" visible="true" version="1" changeset="189039564" timestamp="2026-09-15T03:36:04Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0090901" lon="100.4730675" />\n  <node id="14182062680" visible="true" version="1" changeset="189039564" timestamp="2026-09-15T03:36:04Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0091529" lon="100.4728649" />\n  <node id="14182062681" visible="true" version="1" changeset="189039564" timestamp="2026-09-15T03:36:04Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0091667" lon="100.4725624" />\n  <node id="14182062682" visible="true" version="1" changeset="189039564" timestamp="2026-09-15T03:36:04Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0090289" lon="100.4725597" />\n  <node id="14182062683" visible="true" version="1" changeset="189039564" timestamp="2026-09-15T03:36:04Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0090139" lon="100.4728606" />\n  <node id="14182062684" visible="true" version="1" changeset="189039564" timestamp="2026-09-15T03:36:04Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0090940" lon="100.4728634" />\n  <node id="14182422401" visible="true" version="1" changeset="189045792" timestamp="2026-09-15T07:53:00Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0021488" lon="100.4738360" />\n  <node id="14182422402" visible="true" version="1" changeset="189045792" timestamp="2026-09-15T07:53:00Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0018447" lon="100.4738733" />\n  <node id="14188069236" visible="true" version="1" changeset="189137799" timestamp="2026-09-17T02:46:06Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0129022" lon="100.4778444" />\n  <node id="14188069237" visible="true" version="1" changeset="189137799" timestamp="2026-09-17T02:46:06Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0127308" lon="100.4778272" />\n  <node id="14188374628" visible="true" version="1" changeset="189138033" timestamp="2026-09-17T03:00:23Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0124382" lon="100.4780666" />\n  <node id="14188374629" visible="true" version="1" changeset="189138033" timestamp="2026-09-17T03:00:23Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0124199" lon="100.4782493" />\n  <node id="14188374630" visible="true" version="1" changeset="189138033" timestamp="2026-09-17T03:00:23Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0120009" lon="100.4782259" />\n  <node id="14188374631" visible="true" version="1" changeset="189138033" timestamp="2026-09-17T03:00:23Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0123783" lon="100.4782470">\n    <tag k="access" v="no" />\n    <tag k="barrier" v="gate" />\n  </node>\n  <node id="14188374632" visible="true" version="1" changeset="189138033" timestamp="2026-09-17T03:00:23Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0120807" lon="100.4782303">\n    <tag k="access" v="no" />\n    <tag k="barrier" v="gate" />\n  </node>\n  <node id="14188450535" visible="true" version="1" changeset="189138667" timestamp="2026-09-17T03:37:59Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0069613" lon="100.4789486" />\n  <node id="14188450536" visible="true" version="1" changeset="189138667" timestamp="2026-09-17T03:37:59Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0065377" lon="100.4787451" />\n  <node id="14188637663" visible="true" version="1" changeset="189143922" timestamp="2026-09-17T07:11:29Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0068190" lon="100.4808227" />\n  <node id="14188637664" visible="true" version="1" changeset="189143922" timestamp="2026-09-17T07:11:29Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0067606" lon="100.4803066" />\n  <node id="14188637665" visible="true" version="1" changeset="189143922" timestamp="2026-09-17T07:11:29Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0068015" lon="100.4806682">\n    <tag k="amenity" v="parking_entrance" />\n  </node>\n  <node id="14188679740" visible="true" version="1" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0019696" lon="100.4812547" />\n  <node id="14188679741" visible="true" version="1" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0022094" lon="100.4818106" />\n  <node id="14188679742" visible="true" version="1" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0019749" lon="100.4819185" />\n  <node id="14188679743" visible="true" version="1" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0019503" lon="100.4819194" />\n  <node id="14188679744" visible="true" version="1" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0019284" lon="100.4819123" />\n  <node id="14188679745" visible="true" version="1" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0019117" lon="100.4819026" />\n  <node id="14188679746" visible="true" version="1" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0019012" lon="100.4818875" />\n  <node id="14188679747" visible="true" version="1" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0018845" lon="100.4818557" />\n  <node id="14188679748" visible="true" version="1" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0018739" lon="100.4817699" />\n  <node id="14188679749" visible="true" version="1" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0017075" lon="100.4814335" />\n  <node id="14188679750" visible="true" version="1" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0020266" lon="100.4813868" />\n  <node id="14188679751" visible="true" version="1" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0017544" lon="100.4815283" />\n  <node id="14188679752" visible="true" version="1" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0020249" lon="100.4818962" />\n  <node id="14188679753" visible="true" version="1" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0020321" lon="100.4819110">\n    <tag k="access" v="no" />\n    <tag k="barrier" v="gate" />\n  </node>\n  <node id="14188701065" visible="true" version="1" changeset="189147130" timestamp="2026-09-17T08:28:52Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0009934" lon="100.4732840" />\n  <node id="14188730088" visible="true" version="1" changeset="189148524" timestamp="2026-09-17T09:00:53Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0082337" lon="100.4760399" />\n  <node id="14188730089" visible="true" version="1" changeset="189148524" timestamp="2026-09-17T09:00:53Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0082360" lon="100.4762053" />\n  <node id="14188730090" visible="true" version="1" changeset="189148524" timestamp="2026-09-17T09:00:53Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0081057" lon="100.4762136" />\n  <node id="14188730091" visible="true" version="1" changeset="189148524" timestamp="2026-09-17T09:00:53Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0087790" lon="100.4754812" />\n  <node id="14188730092" visible="true" version="1" changeset="189148524" timestamp="2026-09-17T09:00:53Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0083081" lon="100.4755288" />\n  <node id="14188730093" visible="true" version="1" changeset="189148524" timestamp="2026-09-17T09:00:53Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0083229" lon="100.4756927" />\n  <node id="14188730094" visible="true" version="1" changeset="189148524" timestamp="2026-09-17T09:00:53Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0087488" lon="100.4754842">\n    <tag k="access" v="private" />\n    <tag k="barrier" v="gate" />\n  </node>\n  <node id="14188764645" visible="true" version="1" changeset="189148039" timestamp="2026-09-17T08:49:06Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0036236" lon="100.4745525" />\n  <node id="14188764646" visible="true" version="1" changeset="189148039" timestamp="2026-09-17T08:49:06Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0036582" lon="100.4749618" />\n  <node id="14188764647" visible="true" version="1" changeset="189148039" timestamp="2026-09-17T08:49:06Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0031734" lon="100.4750206" />\n  <node id="14188764648" visible="true" version="1" changeset="189148039" timestamp="2026-09-17T08:49:06Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0036844" lon="100.4751821" />\n  <node id="14190776892" visible="true" version="1" changeset="189189346" timestamp="2026-09-18T03:56:45Z" user="GrabTH_Pimwipa" uid="24297485" lat="6.9999131" lon="100.4796748" />\n  <node id="14190776895" visible="true" version="1" changeset="189189346" timestamp="2026-09-18T03:56:45Z" user="GrabTH_Pimwipa" uid="24297485" lat="6.9999702" lon="100.4796845" />\n  <node id="14190776896" visible="true" version="1" changeset="189189346" timestamp="2026-09-18T03:56:45Z" user="GrabTH_Pimwipa" uid="24297485" lat="6.9995215" lon="100.4796088" />\n  <node id="1421027659" visible="true" version="2" changeset="18522412" timestamp="2013-10-24T16:26:26Z" user="Bernhard Hiller" uid="326704" lat="7.0166590" lon="100.4843092" />\n  <node id="1421027788" visible="true" version="4" changeset="179592394" timestamp="2026-03-09T15:58:28Z" user="thapa prativa" uid="18508229" lat="7.0186788" lon="100.4845869" />\n  <node id="1421027903" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0177321" lon="100.4845946" />\n  <node id="1421027965" visible="true" version="2" changeset="18522412" timestamp="2013-10-24T16:26:26Z" user="Bernhard Hiller" uid="326704" lat="7.0167578" lon="100.4844267" />\n  <node id="1421027993" visible="true" version="1" changeset="9193932" timestamp="2011-09-02T15:35:58Z" user="iPudding" uid="395923" lat="7.0169650" lon="100.4845690" />\n  <node id="14219025415" visible="true" version="1" changeset="189575613" timestamp="2026-09-26T08:15:30Z" user="Tonushka" uid="3419975" lat="7.0022131" lon="100.4730004" />\n  <node id="1431715359" visible="true" version="1" changeset="9287848" timestamp="2011-09-13T09:25:29Z" user="iPudding" uid="395923" lat="7.0003803" lon="100.4723581" />\n  <node id="1431715363" visible="true" version="4" changeset="72473162" timestamp="2019-07-21T03:32:09Z" user="JAAS" uid="3321532" lat="7.0046024" lon="100.4745803" />\n  <node id="1431715366" visible="true" version="4" changeset="186899564" timestamp="2026-08-04T02:55:22Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0045541" lon="100.4741966" />\n  <node id="1431715369" visible="true" version="2" changeset="56270838" timestamp="2018-02-11T18:28:11Z" user="Megre" uid="6684068" lat="7.0028845" lon="100.4709996" />\n  <node id="1431715372" visible="true" version="1" changeset="9287848" timestamp="2011-09-13T09:25:30Z" user="iPudding" uid="395923" lat="7.0014148" lon="100.4743636" />\n  <node id="1431715380" visible="true" version="2" changeset="189039218" timestamp="2026-09-15T03:15:39Z" user="GrabTH_Monnapat" uid="24164407" lat="6.9994427" lon="100.4720237" />\n  <node id="1431715392" visible="true" version="2" changeset="186899564" timestamp="2026-08-04T02:55:22Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0038581" lon="100.4742053" />\n  <node id="1431715395" visible="true" version="1" changeset="9287848" timestamp="2011-09-13T09:25:30Z" user="iPudding" uid="395923" lat="7.0017674" lon="100.4718874" />\n  <node id="1431715406" visible="true" version="2" changeset="176632477" timestamp="2025-12-30T15:05:11Z" user="Tharhtetaung" uid="16875287" lat="7.0038840" lon="100.4745281" />\n  <node id="1431715409" visible="true" version="3" changeset="182517970" timestamp="2026-05-11T15:45:42Z" user="Emma Js" uid="22434897" lat="7.0030489" lon="100.4738636" />\n  <node id="1431715415" visible="true" version="2" changeset="176632477" timestamp="2025-12-30T15:05:11Z" user="Tharhtetaung" uid="16875287" lat="7.0031313" lon="100.4745985" />\n  <node id="1431715417" visible="true" version="2" changeset="186899564" timestamp="2026-08-04T02:55:22Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0032220" lon="100.4741442" />\n  <node id="1431715421" visible="true" version="1" changeset="9287848" timestamp="2011-09-13T09:25:31Z" user="iPudding" uid="395923" lat="7.0010478" lon="100.4717648" />\n  <node id="1431715423" visible="true" version="2" changeset="28369342" timestamp="2015-01-24T10:49:07Z" user="Pathumthani" uid="2302962" lat="7.0003760" lon="100.4718515" />\n  <node id="1431715437" visible="true" version="2" changeset="186899564" timestamp="2026-08-04T02:55:22Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0043770" lon="100.4741997" />\n  <node id="1431715439" visible="true" version="2" changeset="176632477" timestamp="2025-12-30T15:05:11Z" user="Tharhtetaung" uid="16875287" lat="7.0032802" lon="100.4739025" />\n  <node id="1431715458" visible="true" version="2" changeset="186899564" timestamp="2026-08-04T02:55:22Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0034935" lon="100.4741710" />\n  <node id="1431867100" visible="true" version="4" changeset="177256748" timestamp="2026-01-15T12:17:04Z" user="Maptes" uid="23330268" lat="7.0009210" lon="100.4819782" />\n  <node id="1431867103" visible="true" version="1" changeset="9289410" timestamp="2011-09-13T12:52:53Z" user="iPudding" uid="395923" lat="7.0024468" lon="100.4827725" />\n  <node id="1431867108" visible="true" version="2" changeset="103180368" timestamp="2021-04-19T08:40:06Z" user="bomma_1" uid="11659195" lat="7.0016294" lon="100.4802204" />\n  <node id="1431867112" visible="true" version="2" changeset="175694769" timestamp="2025-12-09T03:57:22Z" user="GrabTH_Bulakorn" uid="23192083" lat="6.9998015" lon="100.4812314" />\n  <node id="1431867121" visible="true" version="2" changeset="103176827" timestamp="2021-04-19T07:54:51Z" user="bomma_1" uid="11659195" lat="7.0020378" lon="100.4819463" />\n  <node id="1431867140" visible="true" version="2" changeset="175694243" timestamp="2025-12-09T03:29:04Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0039029" lon="100.4837406" />\n  <node id="1431867141" visible="true" version="2" changeset="175694243" timestamp="2025-12-09T03:29:04Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0030167" lon="100.4839077" />\n  <node id="1431867150" visible="true" version="2" changeset="103176827" timestamp="2021-04-19T07:54:51Z" user="bomma_1" uid="11659195" lat="7.0011288" lon="100.4823233" />\n  <node id="1431867155" visible="true" version="1" changeset="9289410" timestamp="2011-09-13T12:52:56Z" user="iPudding" uid="395923" lat="7.0014450" lon="100.4830727" />\n  <node id="1431867160" visible="true" version="3" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0016511" lon="100.4800676" />\n  <node id="1431867166" visible="true" version="3" changeset="175269776" timestamp="2025-11-29T10:38:56Z" user="Cyrus muqbil" uid="23555462" lat="7.0018751" lon="100.4812957" />\n  <node id="1431867172" visible="true" version="3" changeset="175694769" timestamp="2025-12-09T03:57:22Z" user="GrabTH_Bulakorn" uid="23192083" lat="6.9999394" lon="100.4817107" />\n  <node id="1558314092" visible="true" version="4" changeset="177256748" timestamp="2026-01-15T12:17:04Z" user="Maptes" uid="23330268" lat="7.0007313" lon="100.4826557" />\n  <node id="1558314093" visible="true" version="4" changeset="177256748" timestamp="2026-01-15T12:17:04Z" user="Maptes" uid="23330268" lat="7.0007391" lon="100.4834364" />\n  <node id="1558314094" visible="true" version="2" changeset="175694769" timestamp="2025-12-09T03:57:22Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0011098" lon="100.4826076" />\n  <node id="1558314095" visible="true" version="2" changeset="49318745" timestamp="2017-06-06T20:56:23Z" user="RVR004" uid="5314349" lat="7.0011681" lon="100.4825926" />\n  <node id="1558314107" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:35:19Z" user="Bernhard Hiller" uid="326704" lat="7.0045280" lon="100.4776629" />\n  <node id="1558314109" visible="true" version="3" changeset="18489522" timestamp="2013-10-22T17:35:19Z" user="Bernhard Hiller" uid="326704" lat="7.0045352" lon="100.4773367" />\n  <node id="1558314112" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:35:19Z" user="Bernhard Hiller" uid="326704" lat="7.0045646" lon="100.4770046" />\n  <node id="1558314114" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:35:19Z" user="Bernhard Hiller" uid="326704" lat="7.0045672" lon="100.4778394" />\n  <node id="1558314134" visible="true" version="4" changeset="175444533" timestamp="2025-12-03T12:26:55Z" user="WorayutTK" uid="23562084" lat="7.0046685" lon="100.4780648" />\n  <node id="1558314136" visible="true" version="3" changeset="18489522" timestamp="2013-10-22T17:35:19Z" user="Bernhard Hiller" uid="326704" lat="7.0046465" lon="100.4768503" />\n  <node id="1558314138" visible="true" version="3" changeset="175444533" timestamp="2025-12-03T12:26:55Z" user="WorayutTK" uid="23562084" lat="7.0047150" lon="100.4783865" />\n  <node id="1558314142" visible="true" version="2" changeset="175444533" timestamp="2025-12-03T12:26:55Z" user="WorayutTK" uid="23562084" lat="7.0047449" lon="100.4796058" />\n  <node id="1558314144" visible="true" version="3" changeset="175444533" timestamp="2025-12-03T12:26:55Z" user="WorayutTK" uid="23562084" lat="7.0047995" lon="100.4797866" />\n  <node id="1558997817" visible="true" version="2" changeset="175694769" timestamp="2025-12-09T03:57:22Z" user="GrabTH_Bulakorn" uid="23192083" lat="6.9998501" lon="100.4815138" />\n  <node id="1558997821" visible="true" version="2" changeset="175694769" timestamp="2025-12-09T03:57:22Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0007799" lon="100.4820562" />\n  <node id="1558997823" visible="true" version="2" changeset="175694769" timestamp="2025-12-09T03:57:22Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0014266" lon="100.4816645" />\n  <node id="1558997825" visible="true" version="3" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0047489" lon="100.4767760" />\n  <node id="1558997826" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:28Z" user="Bernhard Hiller" uid="326704" lat="7.0053091" lon="100.4796704" />\n  <node id="1558997827" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:28Z" user="Bernhard Hiller" uid="326704" lat="7.0054831" lon="100.4795796" />\n  <node id="1559016892" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:56Z" user="iPudding" uid="395923" lat="7.0005657" lon="100.4755071" />\n  <node id="1559016893" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:56Z" user="iPudding" uid="395923" lat="7.0007637" lon="100.4753242" />\n  <node id="1559016894" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:56Z" user="iPudding" uid="395923" lat="7.0008231" lon="100.4740238" />\n  <node id="1559016895" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:56Z" user="iPudding" uid="395923" lat="7.0008523" lon="100.4752186" />\n  <node id="1559016896" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:56Z" user="iPudding" uid="395923" lat="7.0008535" lon="100.4739184" />\n  <node id="1559016897" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:56Z" user="iPudding" uid="395923" lat="7.0008535" lon="100.4745850" />\n  <node id="1559016898" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:56Z" user="iPudding" uid="395923" lat="7.0008564" lon="100.4746307" />\n  <node id="1559016899" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:56Z" user="iPudding" uid="395923" lat="7.0009554" lon="100.4750378" />\n  <node id="1559016900" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:56Z" user="iPudding" uid="395923" lat="7.0009852" lon="100.4736701" />\n  <node id="1559016901" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:56Z" user="iPudding" uid="395923" lat="7.0010206" lon="100.4749005" />\n  <node id="1559016902" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:56Z" user="iPudding" uid="395923" lat="7.0011253" lon="100.4746964" />\n  <node id="1559016903" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:56Z" user="iPudding" uid="395923" lat="7.0014894" lon="100.4751872" />\n  <node id="1559016904" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:56Z" user="iPudding" uid="395923" lat="7.0018016" lon="100.4743015" />\n  <node id="1559016905" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:56Z" user="iPudding" uid="395923" lat="7.0018846" lon="100.4751469" />\n  <node id="1559016906" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:56Z" user="iPudding" uid="395923" lat="7.0019624" lon="100.4729106" />\n  <node id="1559016907" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:57Z" user="iPudding" uid="395923" lat="7.0020924" lon="100.4725274" />\n  <node id="1559016908" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:57Z" user="iPudding" uid="395923" lat="7.0020975" lon="100.4728664" />\n  <node id="1559016909" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:57Z" user="iPudding" uid="395923" lat="7.0022036" lon="100.4742563" />\n  <node id="1559016910" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:57Z" user="iPudding" uid="395923" lat="7.0022886" lon="100.4751056" />\n  <node id="1559016911" visible="true" version="2" changeset="175269776" timestamp="2025-11-29T10:38:56Z" user="Cyrus muqbil" uid="23555462" lat="7.0023189" lon="100.4811030" />\n  <node id="1559016912" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:57Z" user="iPudding" uid="395923" lat="7.0023085" lon="100.4726226" />\n  <node id="1559016913" visible="true" version="2" changeset="189575613" timestamp="2026-09-26T08:15:30Z" user="Tonushka" uid="3419975" lat="7.0024894" lon="100.4728522" />\n  <node id="1559016914" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:57Z" user="iPudding" uid="395923" lat="7.0025886" lon="100.4731225" />\n  <node id="1559016915" visible="true" version="3" changeset="177498262" timestamp="2026-01-20T22:53:08Z" user="talbertdoi" uid="23344915" lat="7.0025990" lon="100.4687698" />\n  <node id="1559016916" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:57Z" user="iPudding" uid="395923" lat="7.0025994" lon="100.4742274" />\n  <node id="1559016917" visible="true" version="2" changeset="188339904" timestamp="2026-09-01T02:58:55Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0026340" lon="100.4732506" />\n  <node id="1559016918" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:57Z" user="iPudding" uid="395923" lat="7.0027040" lon="100.4750632" />\n  <node id="1559016919" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0027555" lon="100.4755275" />\n  <node id="1559016920" visible="true" version="1" changeset="10185152" timestamp="2011-12-23T08:16:57Z" user="iPudding" uid="395923" lat="7.0028452" lon="100.4738902" />\n  <node id="1559016921" visible="true" version="2" changeset="182517970" timestamp="2026-05-11T15:45:42Z" user="Emma Js" uid="22434897" lat="7.0029509" lon="100.4739909" />\n  <node id="1559029797" visible="true" version="1" changeset="10185324" timestamp="2011-12-23T08:44:29Z" user="iPudding" uid="395923" lat="7.0051100" lon="100.4706208" />\n  <node id="1559029798" visible="true" version="1" changeset="10185324" timestamp="2011-12-23T08:44:29Z" user="iPudding" uid="395923" lat="7.0051873" lon="100.4713797" />\n  <node id="1597535079" visible="true" version="5" changeset="177239886" timestamp="2026-01-14T23:32:32Z" user="talbertdoi" uid="23344915" lat="7.0138254" lon="100.4852033" />\n  <node id="1612097191" visible="true" version="3" changeset="18489522" timestamp="2013-10-22T17:35:22Z" user="Bernhard Hiller" uid="326704" lat="7.0074193" lon="100.4689898" />\n  <node id="1612097195" visible="true" version="3" changeset="18489522" timestamp="2013-10-22T17:35:22Z" user="Bernhard Hiller" uid="326704" lat="7.0074917" lon="100.4690592" />\n  <node id="1639088623" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:35:24Z" user="Bernhard Hiller" uid="326704" lat="7.0070840" lon="100.4811021" />\n  <node id="1639088625" visible="true" version="2" changeset="176134821" timestamp="2025-12-19T10:51:27Z" user="Klank123" uid="23623862" lat="7.0070655" lon="100.4812108" />\n  <node id="1639088636" visible="true" version="4" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0074097" lon="100.4850281" />\n  <node id="1639088652" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0075743" lon="100.4834965" />\n  <node id="1639088656" visible="true" version="3" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0075498" lon="100.4842060" />\n  <node id="1639088658" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:35:24Z" user="Bernhard Hiller" uid="326704" lat="7.0079759" lon="100.4812598" />\n  <node id="1666900740" visible="true" version="3" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0088045" lon="100.4851124">\n    <tag k="highway" v="crossing" />\n  </node>\n  <node id="1666900742" visible="true" version="2" changeset="34177905" timestamp="2015-09-22T08:33:17Z" user="GerdP" uid="533465" lat="7.0089029" lon="100.4851126">\n    <tag k="highway" v="crossing" />\n  </node>\n  <node id="1666900747" visible="true" version="2" changeset="52655906" timestamp="2017-10-05T14:16:28Z" user="Sal73x" uid="1293386" lat="7.0091232" lon="100.4793527" />\n  <node id="1666915318" visible="true" version="2" changeset="49318745" timestamp="2017-06-06T20:56:23Z" user="RVR004" uid="5314349" lat="7.0080605" lon="100.4767042" />\n  <node id="1680666013" visible="true" version="3" changeset="177551215" timestamp="2026-01-22T06:58:23Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0000323" lon="100.4798053" />\n  <node id="1680666115" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:08:38Z" user="Bernhard Hiller" uid="326704" lat="7.0004163" lon="100.4727931" />\n  <node id="1680666119" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0004484" lon="100.4757767" />\n  <node id="1680666122" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:08:38Z" user="Bernhard Hiller" uid="326704" lat="7.0004554" lon="100.4814040" />\n  <node id="1680666232" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:08:40Z" user="Bernhard Hiller" uid="326704" lat="7.0009396" lon="100.4752400" />\n  <node id="1680666281" visible="true" version="2" changeset="108283189" timestamp="2021-07-20T04:11:19Z" user="p_raj" uid="11589169" lat="7.0011335" lon="100.4766674" />\n  <node id="1680666290" visible="true" version="2" changeset="175694769" timestamp="2025-12-09T03:57:22Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0011675" lon="100.4810153" />\n  <node id="1680666294" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:08:42Z" user="Bernhard Hiller" uid="326704" lat="7.0011606" lon="100.4778200" />\n  <node id="1680666322" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0012381" lon="100.4756954" />\n  <node id="1680666330" visible="true" version="2" changeset="108283189" timestamp="2021-07-20T04:11:19Z" user="p_raj" uid="11589169" lat="7.0012663" lon="100.4765735" />\n  <node id="1680666347" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:08:43Z" user="Bernhard Hiller" uid="326704" lat="7.0013679" lon="100.4724671" />\n  <node id="1680666461" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0017241" lon="100.4756420" />\n  <node id="1680666463" visible="true" version="3" changeset="175694769" timestamp="2025-12-09T03:57:22Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0017578" lon="100.4807566" />\n  <node id="1680666468" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:08:45Z" user="Bernhard Hiller" uid="326704" lat="7.0017505" lon="100.4731549" />\n  <node id="1680666473" visible="true" version="2" changeset="176833293" timestamp="2026-01-04T19:37:23Z" user="gingerChickpea" uid="20574989" lat="7.0017918" lon="100.4768732" />\n  <node id="1680666534" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:08:46Z" user="Bernhard Hiller" uid="326704" lat="7.0018779" lon="100.4772722" />\n  <node id="1680666541" visible="true" version="2" changeset="176833293" timestamp="2026-01-04T19:37:23Z" user="gingerChickpea" uid="20574989" lat="7.0018885" lon="100.4768738" />\n  <node id="1680666546" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:08:46Z" user="Bernhard Hiller" uid="326704" lat="7.0018998" lon="100.4723161" />\n  <node id="1680666638" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:08:48Z" user="Bernhard Hiller" uid="326704" lat="7.0021107" lon="100.4735447" />\n  <node id="1680666683" visible="true" version="2" changeset="137080718" timestamp="2023-06-08T06:20:08Z" user="Dmitriev_DA" uid="9356997" lat="7.0020769" lon="100.4782878" />\n  <node id="1680666686" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0021455" lon="100.4755960" />\n  <node id="1680666699" visible="true" version="2" changeset="103184937" timestamp="2021-04-19T09:35:01Z" user="bomma_1" uid="11659195" lat="7.0022112" lon="100.4770978" />\n  <node id="1680666716" visible="true" version="2" changeset="175694769" timestamp="2025-12-09T03:57:22Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0022266" lon="100.4806151" />\n  <node id="1680666775" visible="true" version="3" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0025363" lon="100.4802152" />\n  <node id="1680666789" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:08:50Z" user="Bernhard Hiller" uid="326704" lat="7.0025589" lon="100.4770867" />\n  <node id="1680666793" visible="true" version="2" changeset="177510665" timestamp="2026-01-21T08:47:43Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0025822" lon="100.4761573" />\n  <node id="1680666823" visible="true" version="2" changeset="177510665" timestamp="2026-01-21T08:47:43Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0026152" lon="100.4765776" />\n  <node id="1680666827" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:08:51Z" user="Bernhard Hiller" uid="326704" lat="7.0026311" lon="100.4812195" />\n  <node id="1680666876" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:08:52Z" user="Bernhard Hiller" uid="326704" lat="7.0026954" lon="100.4817123" />\n  <node id="1680666891" visible="true" version="2" changeset="176134821" timestamp="2025-12-19T10:51:27Z" user="Klank123" uid="23623862" lat="7.0027569" lon="100.4820085" />\n  <node id="1680666950" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0029456" lon="100.4755105" />\n  <node id="1680666958" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:08:54Z" user="Bernhard Hiller" uid="326704" lat="7.0029234" lon="100.4827418" />\n  <node id="1680666961" visible="true" version="2" changeset="181556965" timestamp="2026-04-19T22:13:33Z" user="wireguy" uid="9564652" lat="7.0029259" lon="100.4812038" />\n  <node id="1680666977" visible="true" version="2" changeset="137080718" timestamp="2023-06-08T06:20:08Z" user="Dmitriev_DA" uid="9356997" lat="7.0029342" lon="100.4783372" />\n  <node id="1680667029" visible="true" version="2" changeset="177510665" timestamp="2026-01-21T08:47:43Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0029974" lon="100.4761107" />\n  <node id="1680667034" visible="true" version="2" changeset="177510665" timestamp="2026-01-21T08:47:43Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0030430" lon="100.4765766" />\n  <node id="1680667079" visible="true" version="2" changeset="181556965" timestamp="2026-04-19T22:13:33Z" user="wireguy" uid="9564652" lat="7.0031019" lon="100.4812279" />\n  <node id="1680667083" visible="true" version="2" changeset="181556965" timestamp="2026-04-19T22:13:33Z" user="wireguy" uid="9564652" lat="7.0031140" lon="100.4808292" />\n  <node id="1680667090" visible="true" version="2" changeset="181556965" timestamp="2026-04-19T22:13:33Z" user="wireguy" uid="9564652" lat="7.0031126" lon="100.4816290" />\n  <node id="1680667148" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0032699" lon="100.4760430" />\n  <node id="1680667158" visible="true" version="3" changeset="181556965" timestamp="2026-04-19T22:13:33Z" user="wireguy" uid="9564652" lat="7.0033087" lon="100.4816428" />\n  <node id="1680667164" visible="true" version="2" changeset="181556965" timestamp="2026-04-19T22:13:33Z" user="wireguy" uid="9564652" lat="7.0033362" lon="100.4812407" />\n  <node id="1680667181" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:08:57Z" user="Bernhard Hiller" uid="326704" lat="7.0033453" lon="100.4770236" />\n  <node id="1680667233" visible="true" version="2" changeset="137080718" timestamp="2023-06-08T06:20:08Z" user="Dmitriev_DA" uid="9356997" lat="7.0034672" lon="100.4782680" />\n  <node id="1680667323" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:08:59Z" user="Bernhard Hiller" uid="326704" lat="7.0036826" lon="100.4818460" />\n  <node id="1680667342" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:09:00Z" user="Bernhard Hiller" uid="326704" lat="7.0038442" lon="100.4824829" />\n  <node id="1680667437" visible="true" version="3" changeset="177509855" timestamp="2026-01-21T08:26:46Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0041682" lon="100.4812426" />\n  <node id="1680667445" visible="true" version="3" changeset="188028426" timestamp="2026-08-26T07:58:56Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0044240" lon="100.4824532" />\n  <node id="1680667498" visible="true" version="3" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0042344" lon="100.4804984" />\n  <node id="1680667509" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:09:03Z" user="Bernhard Hiller" uid="326704" lat="7.0042541" lon="100.4793799" />\n  <node id="1680667519" visible="true" version="4" changeset="177509855" timestamp="2026-01-21T08:26:46Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0043634" lon="100.4821900" />\n  <node id="1680667686" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:36:19Z" user="Bernhard Hiller" uid="326704" lat="7.0047480" lon="100.4835610" />\n  <node id="1680667730" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0048130" lon="100.4804894" />\n  <node id="1680667749" visible="true" version="3" changeset="103180368" timestamp="2021-04-19T08:40:06Z" user="bomma_1" uid="11659195" lat="7.0048917" lon="100.4806303" />\n  <node id="1680667774" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:36:19Z" user="Bernhard Hiller" uid="326704" lat="7.0049368" lon="100.4820039" />\n  <node id="1680668223" visible="true" version="3" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0137995" lon="100.4820937" />\n  <node id="1680668226" visible="true" version="3" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0138198" lon="100.4819861" />\n  <node id="1680668239" visible="true" version="4" changeset="176052935" timestamp="2025-12-17T14:14:54Z" user="Wim Keijsers" uid="22692008" lat="7.0161927" lon="100.4823996" />\n  <node id="1727934245" visible="true" version="3" changeset="182517970" timestamp="2026-05-11T15:45:42Z" user="Emma Js" uid="22434897" lat="7.0064663" lon="100.4730218" />\n  <node id="1966386254" visible="true" version="2" changeset="38156304" timestamp="2016-03-30T02:05:18Z" user="Pathumthani" uid="2302962" lat="7.0062913" lon="100.4723468" />\n  <node id="1966399241" visible="true" version="1" changeset="13502820" timestamp="2012-10-15T09:17:40Z" user="Tungmar" uid="427807" lat="7.0086385" lon="100.4737868" />\n  <node id="1966399248" visible="true" version="1" changeset="13502820" timestamp="2012-10-15T09:17:40Z" user="Tungmar" uid="427807" lat="7.0086758" lon="100.4742638" />\n  <node id="1966404349" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0080445" lon="100.4765438" />\n  <node id="1966412828" visible="true" version="2" changeset="28369316" timestamp="2015-01-24T10:47:53Z" user="Pathumthani" uid="2302962" lat="7.0019787" lon="100.4711756" />\n  <node id="2052489649" visible="true" version="4" changeset="182517970" timestamp="2026-05-11T15:45:42Z" user="Emma Js" uid="22434897" lat="7.0056364" lon="100.4738499">\n    <tag k="note" v="add conditional oneway restriction" />\n  </node>\n  <node id="2052497441" visible="true" version="3" changeset="159211955" timestamp="2024-11-16T15:16:41Z" user="nabilersyad" uid="11419275" lat="7.0087990" lon="100.4718620" />\n  <node id="2052522376" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:36Z" user="Bernhard Hiller" uid="326704" lat="7.0084964" lon="100.4701957" />\n  <node id="2052547073" visible="true" version="2" changeset="176632477" timestamp="2025-12-30T15:05:11Z" user="Tharhtetaung" uid="16875287" lat="7.0044641" lon="100.4735222" />\n  <node id="2052547074" visible="true" version="2" changeset="43434341" timestamp="2016-11-06T03:57:36Z" user="Johnny Carlsen" uid="38985" lat="7.0036271" lon="100.4716016">\n    <tag k="highway" v="traffic_signals" />\n  </node>\n  <node id="2052547075" visible="true" version="2" changeset="176632477" timestamp="2025-12-30T15:05:11Z" user="Tharhtetaung" uid="16875287" lat="7.0038031" lon="100.4735586" />\n  <node id="2406954571" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:36:20Z" user="Bernhard Hiller" uid="326704" lat="7.0087127" lon="100.4878101" />\n  <node id="2406954573" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:36:20Z" user="Bernhard Hiller" uid="326704" lat="7.0088071" lon="100.4878141" />\n  <node id="2406954575" visible="true" version="4" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0087947" lon="100.4852537">\n    <tag k="highway" v="traffic_signals" />\n    <tag k="traffic_signals" v="signal" />\n    <tag k="traffic_signals:direction" v="forward" />\n  </node>\n  <node id="2406954577" visible="true" version="4" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0088921" lon="100.4852547">\n    <tag k="highway" v="traffic_signals" />\n    <tag k="traffic_signals:direction" v="both" />\n  </node>\n  <node id="2406954578" visible="true" version="3" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145" lat="7.0089311" lon="100.4812836" />\n  <node id="2406954579" visible="true" version="4" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0089478" lon="100.4811765" />\n  <node id="2406954580" visible="true" version="3" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145" lat="7.0090399" lon="100.4813022" />\n  <node id="2505176020" visible="true" version="2" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0052277" lon="100.4819810" />\n  <node id="2505176027" visible="true" version="3" changeset="176134821" timestamp="2025-12-19T10:51:27Z" user="Klank123" uid="23623862" lat="7.0054160" lon="100.4840329" />\n  <node id="2505176169" visible="true" version="3" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0073582" lon="100.4810194" />\n  <node id="2505176175" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0079842" lon="100.4811238" />\n  <node id="2505176182" visible="true" version="2" changeset="101099622" timestamp="2021-03-16T07:36:22Z" user="oliviar_grab" uid="10664132" lat="7.0081224" lon="100.4823568" />\n  <node id="2505176189" visible="true" version="2" changeset="101099622" timestamp="2021-03-16T07:36:22Z" user="oliviar_grab" uid="10664132" lat="7.0081396" lon="100.4824783" />\n  <node id="2505176202" visible="true" version="3" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0081798" lon="100.4827031" />\n  <node id="2505176258" visible="true" version="3" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0092345" lon="100.4813323" />\n  <node id="2505176265" visible="true" version="2" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145" lat="7.0092202" lon="100.4814688" />\n  <node id="2505176271" visible="true" version="1" changeset="18489522" timestamp="2013-10-22T17:31:18Z" user="Bernhard Hiller" uid="326704" lat="7.0093760" lon="100.4840111" />\n  <node id="2505176284" visible="true" version="2" changeset="189189783" timestamp="2026-09-18T04:25:48Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0094170" lon="100.4836969" />\n  <node id="2505176388" visible="true" version="1" changeset="18489522" timestamp="2013-10-22T17:31:20Z" user="Bernhard Hiller" uid="326704" lat="7.0112538" lon="100.4817930" />\n  <node id="2505176422" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0112676" lon="100.4816714" />\n  <node id="2505176434" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0112821" lon="100.4815644" />\n  <node id="2505176470" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0114505" lon="100.4852841" />\n  <node id="2505176494" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0117036" lon="100.4817441" />\n  <node id="2505176514" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0118889" lon="100.4817750" />\n  <node id="2505176520" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0119288" lon="100.4825334" />\n  <node id="2505176529" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0120666" lon="100.4833182" />\n  <node id="2505176570" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0122427" lon="100.4842572" />\n  <node id="2505176598" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0123906" lon="100.4852601" />\n  <node id="2505176654" visible="true" version="1" changeset="18489522" timestamp="2013-10-22T17:31:22Z" user="Bernhard Hiller" uid="326704" lat="7.0126334" lon="100.4835129" />\n  <node id="2505176686" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0127513" lon="100.4852434" />\n  <node id="2505176697" visible="true" version="4" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0127754" lon="100.4794044" />\n  <node id="2505176734" visible="true" version="2" changeset="176054561" timestamp="2025-12-17T14:53:03Z" user="Wim Keijsers" uid="22692008" lat="7.0131488" lon="100.4833918" />\n  <node id="2505176747" visible="true" version="1" changeset="18489522" timestamp="2013-10-22T17:31:23Z" user="Bernhard Hiller" uid="326704" lat="7.0131699" lon="100.4829596" />\n  <node id="2505176800" visible="true" version="2" changeset="176054561" timestamp="2025-12-17T14:53:03Z" user="Wim Keijsers" uid="22692008" lat="7.0132433" lon="100.4840114" />\n  <node id="2505176867" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0136529" lon="100.4769058" />\n  <node id="2505176923" visible="true" version="1" changeset="18489522" timestamp="2013-10-22T17:31:26Z" user="Bernhard Hiller" uid="326704" lat="7.0147347" lon="100.4728718" />\n  <node id="2505176985" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0153783" lon="100.4823570" />\n  <node id="2505177021" visible="true" version="1" changeset="18489522" timestamp="2013-10-22T17:31:28Z" user="Bernhard Hiller" uid="326704" lat="7.0157936" lon="100.4750410" />\n  <node id="2505177476" visible="true" version="1" changeset="18489522" timestamp="2013-10-22T17:31:37Z" user="Bernhard Hiller" uid="326704" lat="7.0183312" lon="100.4810284" />\n  <node id="2505177499" visible="true" version="1" changeset="18489522" timestamp="2013-10-22T17:31:37Z" user="Bernhard Hiller" uid="326704" lat="7.0183988" lon="100.4817328" />\n  <node id="2506085434" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0009790" lon="100.4757223" />\n  <node id="2506085475" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:21Z" user="Bernhard Hiller" uid="326704" lat="7.0034807" lon="100.4784093" />\n  <node id="2506085476" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:21Z" user="Bernhard Hiller" uid="326704" lat="7.0035052" lon="100.4787201" />\n  <node id="2506085486" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:22Z" user="Bernhard Hiller" uid="326704" lat="7.0041766" lon="100.4787079" />\n  <node id="2506085487" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:22Z" user="Bernhard Hiller" uid="326704" lat="7.0041916" lon="100.4780609" />\n  <node id="2506085489" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:22Z" user="Bernhard Hiller" uid="326704" lat="7.0042324" lon="100.4782556" />\n  <node id="2506085493" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:23Z" user="Bernhard Hiller" uid="326704" lat="7.0043055" lon="100.4780565" />\n  <node id="2506085496" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:23Z" user="Bernhard Hiller" uid="326704" lat="7.0044432" lon="100.4732391" />\n  <node id="2506085498" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:23Z" user="Bernhard Hiller" uid="326704" lat="7.0044902" lon="100.4780371" />\n  <node id="2506085499" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:24Z" user="Bernhard Hiller" uid="326704" lat="7.0044924" lon="100.4736910" />\n  <node id="2506085508" visible="true" version="2" changeset="175444533" timestamp="2025-12-03T12:26:55Z" user="WorayutTK" uid="23562084" lat="7.0046140" lon="100.4779563" />\n  <node id="2506085525" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:27Z" user="Bernhard Hiller" uid="326704" lat="7.0050442" lon="100.4794879" />\n  <node id="2506085526" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:27Z" user="Bernhard Hiller" uid="326704" lat="7.0051497" lon="100.4794757" />\n  <node id="2506085532" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:27Z" user="Bernhard Hiller" uid="326704" lat="7.0053108" lon="100.4794021" />\n  <node id="2506085537" visible="true" version="2" changeset="182517970" timestamp="2026-05-11T15:45:42Z" user="Emma Js" uid="22434897" lat="7.0054113" lon="100.4734646" />\n  <node id="2506085538" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:28Z" user="Bernhard Hiller" uid="326704" lat="7.0054224" lon="100.4792333" />\n  <node id="2506085541" visible="true" version="3" changeset="182517970" timestamp="2026-05-11T15:45:42Z" user="Emma Js" uid="22434897" lat="7.0054273" lon="100.4737829" />\n  <node id="2506085544" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:29Z" user="Bernhard Hiller" uid="326704" lat="7.0055041" lon="100.4779609" />\n  <node id="2506085545" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:29Z" user="Bernhard Hiller" uid="326704" lat="7.0055083" lon="100.4777965" />\n  <node id="2506085547" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:29Z" user="Bernhard Hiller" uid="326704" lat="7.0055083" lon="100.4790494" />\n  <node id="2506085550" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:30Z" user="Bernhard Hiller" uid="326704" lat="7.0055298" lon="100.4788460" />\n  <node id="2506085551" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:30Z" user="Bernhard Hiller" uid="326704" lat="7.0056093" lon="100.4773767" />\n  <node id="2506085555" visible="true" version="2" changeset="189138277" timestamp="2026-09-17T03:16:49Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0057224" lon="100.4782417" />\n  <node id="2506085568" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:30Z" user="Bernhard Hiller" uid="326704" lat="7.0057446" lon="100.4770867" />\n  <node id="2506085570" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:30Z" user="Bernhard Hiller" uid="326704" lat="7.0057571" lon="100.4778915" />\n  <node id="2506085581" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:32Z" user="Bernhard Hiller" uid="326704" lat="7.0058483" lon="100.4774631" />\n  <node id="2506085591" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:33Z" user="Bernhard Hiller" uid="326704" lat="7.0059637" lon="100.4771754" />\n  <node id="2506085598" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:34Z" user="Bernhard Hiller" uid="326704" lat="7.0060474" lon="100.4765825" />\n  <node id="2506085605" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:35Z" user="Bernhard Hiller" uid="326704" lat="7.0061333" lon="100.4768768" />\n  <node id="2506085611" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:35Z" user="Bernhard Hiller" uid="326704" lat="7.0061950" lon="100.4757116" />\n  <node id="2506085619" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:36Z" user="Bernhard Hiller" uid="326704" lat="7.0063028" lon="100.4754791" />\n  <node id="2506085622" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:37Z" user="Bernhard Hiller" uid="326704" lat="7.0063271" lon="100.4753612" />\n  <node id="2506085623" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:37Z" user="Bernhard Hiller" uid="326704" lat="7.0063309" lon="100.4767123" />\n  <node id="2506085628" visible="true" version="4" changeset="182517970" timestamp="2026-05-11T15:45:42Z" user="Emma Js" uid="22434897" lat="7.0064079" lon="100.4728117" />\n  <node id="2506085631" visible="true" version="2" changeset="189138277" timestamp="2026-09-17T03:16:49Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0064184" lon="100.4782986" />\n  <node id="2506085634" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0063982" lon="100.4796578" />\n  <node id="2506085647" visible="true" version="3" changeset="182517970" timestamp="2026-05-11T15:45:42Z" user="Emma Js" uid="22434897" lat="7.0065253" lon="100.4732095" />\n  <node id="2506085650" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0065115" lon="100.4803640" />\n  <node id="2506085670" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:46:42Z" user="Bernhard Hiller" uid="326704" lat="7.0066415" lon="100.4716838" />\n  <node id="2506085690" visible="true" version="2" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0067946" lon="100.4740473" />\n  <node id="2506085784" visible="true" version="2" changeset="176196082" timestamp="2025-12-20T20:47:05Z" user="RAytoun" uid="1414812" lat="7.0070669" lon="100.4697616" />\n  <node id="2506085849" visible="true" version="2" changeset="159211955" timestamp="2024-11-16T15:16:41Z" user="nabilersyad" uid="11419275" lat="7.0073887" lon="100.4715142" />\n  <node id="2506085878" visible="true" version="3" changeset="176196082" timestamp="2025-12-20T20:47:05Z" user="RAytoun" uid="1414812" lat="7.0073984" lon="100.4680895">\n    <tag k="name" v="One way" />\n  </node>\n  <node id="2506086151" visible="true" version="2" changeset="108306644" timestamp="2021-07-20T09:48:02Z" user="appik" uid="11515808" lat="7.0081425" lon="100.4737477" />\n  <node id="2506086181" visible="true" version="3" changeset="159211955" timestamp="2024-11-16T15:16:41Z" user="nabilersyad" uid="11419275" lat="7.0080839" lon="100.4713155" />\n  <node id="2506086249" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0088043" lon="100.4804085" />\n  <node id="2506086743" visible="true" version="2" changeset="74914002" timestamp="2019-09-25T14:34:34Z" user="icosiam" uid="10316678" lat="7.0086851" lon="100.4712546" />\n  <node id="2506086795" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:09Z" user="Bernhard Hiller" uid="326704" lat="7.0087117" lon="100.4747225" />\n  <node id="2506086831" visible="true" version="2" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0087546" lon="100.4751959" />\n  <node id="2506086896" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0088289" lon="100.4806013" />\n  <node id="2506087028" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:12Z" user="Bernhard Hiller" uid="326704" lat="7.0089162" lon="100.4695471" />\n  <node id="2506087114" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:12Z" user="Bernhard Hiller" uid="326704" lat="7.0089896" lon="100.4694096" />\n  <node id="2506087136" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0090881" lon="100.4788840" />\n  <node id="2506087252" visible="true" version="2" changeset="74914002" timestamp="2019-09-25T14:34:34Z" user="icosiam" uid="10316678" lat="7.0091751" lon="100.4708068" />\n  <node id="2506087296" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0092254" lon="100.4806138" />\n  <node id="2506087304" visible="true" version="2" changeset="74914002" timestamp="2019-09-25T14:34:34Z" user="icosiam" uid="10316678" lat="7.0092343" lon="100.4711429" />\n  <node id="2506087316" visible="true" version="2" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0092812" lon="100.4752300" />\n  <node id="2506087432" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:18Z" user="Bernhard Hiller" uid="326704" lat="7.0095067" lon="100.4730603" />\n  <node id="2506087450" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:19Z" user="Bernhard Hiller" uid="326704" lat="7.0095452" lon="100.4742446" />\n  <node id="2506087462" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:19Z" user="Bernhard Hiller" uid="326704" lat="7.0095558" lon="100.4700054" />\n  <node id="2506087487" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:19Z" user="Bernhard Hiller" uid="326704" lat="7.0095935" lon="100.4696179" />\n  <node id="2506087509" visible="true" version="2" changeset="38156259" timestamp="2016-03-30T02:00:25Z" user="Pathumthani" uid="2302962" lat="7.0095374" lon="100.4697662" />\n  <node id="2506087522" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:20Z" user="Bernhard Hiller" uid="326704" lat="7.0096430" lon="100.4705138" />\n  <node id="2506087550" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0096675" lon="100.4765217" />\n  <node id="2506087563" visible="true" version="2" changeset="74914002" timestamp="2019-09-25T14:34:34Z" user="icosiam" uid="10316678" lat="7.0097094" lon="100.4710568" />\n  <node id="2506087592" visible="true" version="2" changeset="101102450" timestamp="2021-03-16T08:13:40Z" user="oliviar_grab" uid="10664132" lat="7.0097025" lon="100.4756106" />\n  <node id="2506087593" visible="true" version="2" changeset="72473287" timestamp="2019-07-21T03:47:56Z" user="JAAS" uid="3321532" lat="7.0097948" lon="100.4717153" />\n  <node id="2506087659" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0099019" lon="100.4767009" />\n  <node id="2506087709" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:24Z" user="Bernhard Hiller" uid="326704" lat="7.0100312" lon="100.4699250" />\n  <node id="2506087732" visible="true" version="2" changeset="38156259" timestamp="2016-03-30T02:00:25Z" user="Pathumthani" uid="2302962" lat="7.0101519" lon="100.4711345" />\n  <node id="2506087772" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:25Z" user="Bernhard Hiller" uid="326704" lat="7.0101143" lon="100.4698017" />\n  <node id="2506087810" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:26Z" user="Bernhard Hiller" uid="326704" lat="7.0101382" lon="100.4716751" />\n  <node id="2506087846" visible="true" version="3" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0101688" lon="100.4768617" />\n  <node id="2506087863" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:27Z" user="Bernhard Hiller" uid="326704" lat="7.0102240" lon="100.4723732" />\n  <node id="2506087889" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:27Z" user="Bernhard Hiller" uid="326704" lat="7.0102799" lon="100.4730689" />\n  <node id="2506087923" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:28Z" user="Bernhard Hiller" uid="326704" lat="7.0103535" lon="100.4788464" />\n  <node id="2506087934" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:30Z" user="Bernhard Hiller" uid="326704" lat="7.0104129" lon="100.4747726" />\n  <node id="2506087952" visible="true" version="2" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0104250" lon="100.4771045" />\n  <node id="2506087955" visible="true" version="2" changeset="144432300" timestamp="2023-11-25T08:10:05Z" user="pLoskutov" uid="17472737" lat="7.0104991" lon="100.4724048" />\n  <node id="2506087957" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0104847" lon="100.4807211" />\n  <node id="2506087958" visible="true" version="3" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0105331" lon="100.4778685" />\n  <node id="2506087959" visible="true" version="3" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0105491" lon="100.4780330" />\n  <node id="2506087962" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0105371" lon="100.4757566" />\n  <node id="2506087964" visible="true" version="2" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0105374" lon="100.4777011" />\n  <node id="2506087965" visible="true" version="2" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0105310" lon="100.4772815" />\n  <node id="2506087966" visible="true" version="2" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0105539" lon="100.4774119" />\n  <node id="2506087967" visible="true" version="3" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0106179" lon="100.4781415" />\n  <node id="2506087968" visible="true" version="2" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0105356" lon="100.4730642" />\n  <node id="2506087975" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:33Z" user="Bernhard Hiller" uid="326704" lat="7.0106603" lon="100.4733879" />\n  <node id="2506087987" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:34Z" user="Bernhard Hiller" uid="326704" lat="7.0106820" lon="100.4710191" />\n  <node id="2506087989" visible="true" version="3" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0106677" lon="100.4781606" />\n  <node id="2506088002" visible="true" version="2" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0107692" lon="100.4748007" />\n  <node id="2506088006" visible="true" version="3" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0107832" lon="100.4781570" />\n  <node id="2506088009" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0107486" lon="100.4764645" />\n  <node id="2506088011" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0108640" lon="100.4807645" />\n  <node id="2506088015" visible="true" version="2" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0108794" lon="100.4780971" />\n  <node id="2506088019" visible="true" version="3" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0109203" lon="100.4779965" />\n  <node id="2506088023" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0109252" lon="100.4799006" />\n  <node id="2506088024" visible="true" version="2" changeset="159194346" timestamp="2024-11-16T04:47:27Z" user="nabilersyad" uid="11419275" lat="7.0109563" lon="100.4742706" />\n  <node id="2506088039" visible="true" version="2" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0110002" lon="100.4729942" />\n  <node id="2506088046" visible="true" version="2" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0109637" lon="100.4772197" />\n  <node id="2506088052" visible="true" version="2" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0109879" lon="100.4774773" />\n  <node id="2506088080" visible="true" version="2" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0109918" lon="100.4776412" />\n  <node id="2506088084" visible="true" version="2" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0110406" lon="100.4738246" />\n  <node id="2506088223" visible="true" version="2" changeset="49318745" timestamp="2017-06-06T20:56:24Z" user="RVR004" uid="5314349" lat="7.0112922" lon="100.4708484" />\n  <node id="2506088243" visible="true" version="5" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0113330" lon="100.4742789" />\n  <node id="2506088254" visible="true" version="2" changeset="159211010" timestamp="2024-11-16T14:48:54Z" user="nabilersyad" uid="11419275" lat="7.0115211" lon="100.4735878" />\n  <node id="2506088265" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0115541" lon="100.4810311" />\n  <node id="2506088273" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0116181" lon="100.4772728" />\n  <node id="2506088274" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0116340" lon="100.4799488" />\n  <node id="2506088279" visible="true" version="2" changeset="159211010" timestamp="2024-11-16T14:48:54Z" user="nabilersyad" uid="11419275" lat="7.0116051" lon="100.4734768" />\n  <node id="2506088280" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0116869" lon="100.4768154" />\n  <node id="2506088404" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0117912" lon="100.4788701" />\n  <node id="2506088411" visible="true" version="2" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0116822" lon="100.4747392" />\n  <node id="2506088423" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0112262" lon="100.4762685" />\n  <node id="2506088450" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:53Z" user="Bernhard Hiller" uid="326704" lat="7.0118069" lon="100.4730729" />\n  <node id="2506088476" visible="true" version="3" changeset="188989640" timestamp="2026-09-14T04:03:52Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0116720" lon="100.4714820" />\n  <node id="2506088485" visible="true" version="2" changeset="159211010" timestamp="2024-11-16T14:48:54Z" user="nabilersyad" uid="11419275" lat="7.0118520" lon="100.4728996" />\n  <node id="2506088491" visible="true" version="2" changeset="188989640" timestamp="2026-09-14T04:03:52Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0118924" lon="100.4720935" />\n  <node id="2506088492" visible="true" version="4" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0119695" lon="100.4785078" />\n  <node id="2506088494" visible="true" version="2" changeset="188989640" timestamp="2026-09-14T04:03:52Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0119096" lon="100.4718175" />\n  <node id="2506088504" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:57Z" user="Bernhard Hiller" uid="326704" lat="7.0120066" lon="100.4716598" />\n  <node id="2506088507" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0120768" lon="100.4807712" />\n  <node id="2506088509" visible="true" version="4" changeset="188076207" timestamp="2026-08-27T04:39:10Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0120146" lon="100.4781026" />\n  <node id="2506088518" visible="true" version="5" changeset="188076207" timestamp="2026-08-27T04:39:10Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0120469" lon="100.4777187" />\n  <node id="2506088522" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:47:59Z" user="Bernhard Hiller" uid="326704" lat="7.0120968" lon="100.4715971" />\n  <node id="2506088528" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0121179" lon="100.4799702" />\n  <node id="2506088529" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0121213" lon="100.4773570" />\n  <node id="2506088612" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0122013" lon="100.4769031" />\n  <node id="2506088675" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:48:04Z" user="Bernhard Hiller" uid="326704" lat="7.0122751" lon="100.4715300" />\n  <node id="2506088695" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0122660" lon="100.4811774" />\n  <node id="2506088707" visible="true" version="2" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0123274" lon="100.4745565" />\n  <node id="2506088715" visible="true" version="2" changeset="29150156" timestamp="2015-02-28T07:37:20Z" user="Pathumthani" uid="2302962" lat="7.0122966" lon="100.4764419" />\n  <node id="2506088733" visible="true" version="2" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0123229" lon="100.4742735" />\n  <node id="2506088747" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:48:07Z" user="Bernhard Hiller" uid="326704" lat="7.0124490" lon="100.4714673" />\n  <node id="2506088775" visible="true" version="2" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0126195" lon="100.4742570" />\n  <node id="2506088791" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:48:10Z" user="Bernhard Hiller" uid="326704" lat="7.0126659" lon="100.4714781" />\n  <node id="2506088792" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:48:11Z" user="Bernhard Hiller" uid="326704" lat="7.0126793" lon="100.4673857" />\n  <node id="2506088938" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:48:14Z" user="Bernhard Hiller" uid="326704" lat="7.0128083" lon="100.4725400" />\n  <node id="2506088971" visible="true" version="2" changeset="74914002" timestamp="2019-09-25T14:34:34Z" user="icosiam" uid="10316678" lat="7.0128286" lon="100.4745719" />\n  <node id="2506089058" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:48:16Z" user="Bernhard Hiller" uid="326704" lat="7.0130105" lon="100.4725884" />\n  <node id="2506089170" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:48:18Z" user="Bernhard Hiller" uid="326704" lat="7.0131664" lon="100.4716642" />\n  <node id="2506089309" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0132406" lon="100.4766265" />\n  <node id="2506089487" visible="true" version="2" changeset="189038359" timestamp="2026-09-15T02:25:53Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0133476" lon="100.4726685" />\n  <node id="2506089522" visible="true" version="2" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0134416" lon="100.4764150" />\n  <node id="2506089545" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0134396" lon="100.4778672" />\n  <node id="2506089593" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:48:24Z" user="Bernhard Hiller" uid="326704" lat="7.0135615" lon="100.4718416" />\n  <node id="2506089600" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0135431" lon="100.4773766" />\n  <node id="2506089735" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:48:25Z" user="Bernhard Hiller" uid="326704" lat="7.0137613" lon="100.4719693" />\n  <node id="2506089779" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0137578" lon="100.4764742" />\n  <node id="2506089826" visible="true" version="2" changeset="176396228" timestamp="2025-12-25T14:07:27Z" user="11aye" uid="22221182" lat="7.0138299" lon="100.4787505" />\n  <node id="2506089901" visible="true" version="3" changeset="176397735" timestamp="2025-12-25T14:44:47Z" user="11aye" uid="22221182" lat="7.0138710" lon="100.4800334" />\n  <node id="2506090118" visible="true" version="2" changeset="32198572" timestamp="2015-06-25T06:54:58Z" user="Pathumthani" uid="2302962" lat="7.0140452" lon="100.4720768" />\n  <node id="2506090181" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0141831" lon="100.4773364" />\n  <node id="2506090372" visible="true" version="3" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0143856" lon="100.4777769" />\n  <node id="2506090379" visible="true" version="2" changeset="32198572" timestamp="2015-06-25T06:54:58Z" user="Pathumthani" uid="2302962" lat="7.0143820" lon="100.4727326" />\n  <node id="2506090385" visible="true" version="2" changeset="176396228" timestamp="2025-12-25T14:07:27Z" user="11aye" uid="22221182" lat="7.0143488" lon="100.4786080" />\n  <node id="2506090412" visible="true" version="3" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0144506" lon="100.4760143" />\n  <node id="2506090451" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0143540" lon="100.4781308" />\n  <node id="2506090499" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:48:38Z" user="Bernhard Hiller" uid="326704" lat="7.0145237" lon="100.4731378" />\n  <node id="2506090505" visible="true" version="3" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0145152" lon="100.4776931" />\n  <node id="2506090510" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:48:39Z" user="Bernhard Hiller" uid="326704" lat="7.0145387" lon="100.4735771" />\n  <node id="2506090521" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0145101" lon="100.4773186" />\n  <node id="2506090533" visible="true" version="3" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0145586" lon="100.4758831" />\n  <node id="2506090538" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:48:39Z" user="Bernhard Hiller" uid="326704" lat="7.0146182" lon="100.4748624" />\n  <node id="2506090550" visible="true" version="3" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0146029" lon="100.4757607" />\n  <node id="2506090556" visible="true" version="2" changeset="177496670" timestamp="2026-01-20T21:52:31Z" user="talbertdoi" uid="23344915" lat="7.0146157" lon="100.4754264" />\n  <node id="2506090779" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0149931" lon="100.4771709" />\n  <node id="2506090932" visible="true" version="3" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0151505" lon="100.4775227" />\n  <node id="2506090979" visible="true" version="2" changeset="23214413" timestamp="2014-06-27T12:42:53Z" user="stephenwilks" uid="695426" lat="7.0152419" lon="100.4795427" />\n  <node id="2507400589" visible="true" version="4" changeset="177509855" timestamp="2026-01-21T08:26:46Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0042120" lon="100.4806316" />\n  <node id="2507400591" visible="true" version="2" changeset="103180368" timestamp="2021-04-19T08:40:06Z" user="bomma_1" uid="11659195" lat="7.0048863" lon="100.4807529" />\n  <node id="2507400593" visible="true" version="3" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0051391" lon="100.4808031" />\n  <node id="2507400604" visible="true" version="2" changeset="176134821" timestamp="2025-12-19T10:51:27Z" user="Klank123" uid="23623862" lat="7.0060459" lon="100.4822272" />\n  <node id="2507400605" visible="true" version="2" changeset="176134821" timestamp="2025-12-19T10:51:27Z" user="Klank123" uid="23623862" lat="7.0060913" lon="100.4839922" />\n  <node id="2507400609" visible="true" version="2" changeset="176134821" timestamp="2025-12-19T10:51:27Z" user="Klank123" uid="23623862" lat="7.0062024" lon="100.4829999" />\n  <node id="2507400610" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:29Z" user="Bernhard Hiller" uid="326704" lat="7.0062199" lon="100.4839347" />\n  <node id="2507400612" visible="true" version="2" changeset="176134821" timestamp="2025-12-19T10:51:27Z" user="Klank123" uid="23623862" lat="7.0062726" lon="100.4821874" />\n  <node id="2507400613" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:29Z" user="Bernhard Hiller" uid="326704" lat="7.0062752" lon="100.4835102" />\n  <node id="2507400614" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:29Z" user="Bernhard Hiller" uid="326704" lat="7.0062955" lon="100.4813157" />\n  <node id="2507400615" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:29Z" user="Bernhard Hiller" uid="326704" lat="7.0063029" lon="100.4838169" />\n  <node id="2507400618" visible="true" version="2" changeset="176134821" timestamp="2025-12-19T10:51:27Z" user="Klank123" uid="23623862" lat="7.0064097" lon="100.4821668" />\n  <node id="2507400619" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:30Z" user="Bernhard Hiller" uid="326704" lat="7.0064259" lon="100.4826738" />\n  <node id="2507400624" visible="true" version="2" changeset="176134821" timestamp="2025-12-19T10:51:27Z" user="Klank123" uid="23623862" lat="7.0065665" lon="100.4821520" />\n  <node id="2507400625" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:31Z" user="Bernhard Hiller" uid="326704" lat="7.0065735" lon="100.4830517" />\n  <node id="2507400628" visible="true" version="2" changeset="176134821" timestamp="2025-12-19T10:51:27Z" user="Klank123" uid="23623862" lat="7.0066591" lon="100.4812702" />\n  <node id="2507400630" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:31Z" user="Bernhard Hiller" uid="326704" lat="7.0067137" lon="100.4817773" />\n  <node id="2507400638" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0067511" lon="100.4810494" />\n  <node id="2507400639" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0067613" lon="100.4809199" />\n  <node id="2507400649" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0075007" lon="100.4845654" />\n  <node id="2507400677" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0088112" lon="100.4847675" />\n  <node id="2507400689" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:37Z" user="Bernhard Hiller" uid="326704" lat="7.0099696" lon="100.4815863" />\n  <node id="2507400690" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0099721" lon="100.4814553" />\n  <node id="2507400696" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:38Z" user="Bernhard Hiller" uid="326704" lat="7.0105513" lon="100.4816799" />\n  <node id="2507400698" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0105590" lon="100.4815532" />\n  <node id="2507400700" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:38Z" user="Bernhard Hiller" uid="326704" lat="7.0106013" lon="100.4826848" />\n  <node id="2507400701" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:38Z" user="Bernhard Hiller" uid="326704" lat="7.0106043" lon="100.4833385" />\n  <node id="2507400702" visible="true" version="2" changeset="108223926" timestamp="2021-07-19T06:37:27Z" user="gunnala_a" uid="13354536" lat="7.0106707" lon="100.4843638" />\n  <node id="2507400742" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:43Z" user="Bernhard Hiller" uid="326704" lat="7.0128976" lon="100.4827128" />\n  <node id="2507400748" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0130290" lon="100.4818547" />\n  <node id="2507400752" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:44Z" user="Bernhard Hiller" uid="326704" lat="7.0131585" lon="100.4810528" />\n  <node id="2507400769" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:45Z" user="Bernhard Hiller" uid="326704" lat="7.0133358" lon="100.4746463" />\n  <node id="2507400772" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:46Z" user="Bernhard Hiller" uid="326704" lat="7.0135753" lon="100.4735914" />\n  <node id="2507400773" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:46Z" user="Bernhard Hiller" uid="326704" lat="7.0136879" lon="100.4826451" />\n  <node id="2507400792" visible="true" version="2" changeset="74914002" timestamp="2019-09-25T14:34:34Z" user="icosiam" uid="10316678" lat="7.0142835" lon="100.4748006" />\n  <node id="2507400837" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0152060" lon="100.4844766" />\n  <node id="2507400842" visible="true" version="2" changeset="176054561" timestamp="2025-12-17T14:53:03Z" user="Wim Keijsers" uid="22692008" lat="7.0152400" lon="100.4833896" />\n  <node id="2507400845" visible="true" version="2" changeset="176054561" timestamp="2025-12-17T14:53:03Z" user="Wim Keijsers" uid="22692008" lat="7.0152917" lon="100.4829427" />\n  <node id="2507400847" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:53Z" user="Bernhard Hiller" uid="326704" lat="7.0153635" lon="100.4824872" />\n  <node id="2507400851" visible="true" version="2" changeset="23214413" timestamp="2014-06-27T12:42:53Z" user="stephenwilks" uid="695426" lat="7.0154356" lon="100.4795571" />\n  <node id="2507400852" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0154939" lon="100.4822643" />\n  <node id="2507400879" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0159698" lon="100.4754228" />\n  <node id="2507400897" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:22:59Z" user="Bernhard Hiller" uid="326704" lat="7.0163150" lon="100.4826303" />\n  <node id="2507400917" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:23:01Z" user="Bernhard Hiller" uid="326704" lat="7.0165118" lon="100.4838757" />\n  <node id="2507400919" visible="true" version="3" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0165308" lon="100.4766086" />\n  <node id="2507400921" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:23:01Z" user="Bernhard Hiller" uid="326704" lat="7.0165487" lon="100.4832406" />\n  <node id="2507400923" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:23:02Z" user="Bernhard Hiller" uid="326704" lat="7.0165549" lon="100.4840895" />\n  <node id="2507400926" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:23:02Z" user="Bernhard Hiller" uid="326704" lat="7.0166010" lon="100.4833460" />\n  <node id="2507400995" visible="true" version="2" changeset="32199614" timestamp="2015-06-25T08:06:08Z" user="Pathumthani" uid="2302962" lat="7.0170326" lon="100.4775490" />\n  <node id="2507401012" visible="true" version="2" changeset="32199603" timestamp="2015-06-25T08:05:12Z" user="Pathumthani" uid="2302962" lat="7.0174280" lon="100.4783510" />\n  <node id="2507401021" visible="true" version="2" changeset="176395264" timestamp="2025-12-25T13:41:17Z" user="11aye" uid="22221182" lat="7.0176144" lon="100.4788622" />\n  <node id="2507401050" visible="true" version="2" changeset="176394779" timestamp="2025-12-25T13:31:07Z" user="11aye" uid="22221182" lat="7.0178153" lon="100.4793366" />\n  <node id="2507401129" visible="true" version="2" changeset="188398529" timestamp="2026-09-02T03:56:56Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0182955" lon="100.4807817" />\n  <node id="2507401162" visible="true" version="2" changeset="176052935" timestamp="2025-12-17T14:14:54Z" user="Wim Keijsers" uid="22692008" lat="7.0183574" lon="100.4813016" />\n  <node id="2935421242" visible="true" version="2" changeset="176396856" timestamp="2025-12-25T14:25:05Z" user="11aye" uid="22221182" lat="7.0156968" lon="100.4795779" />\n  <node id="3125684629" visible="true" version="2" changeset="177498262" timestamp="2026-01-20T22:53:08Z" user="talbertdoi" uid="23344915" lat="7.0023527" lon="100.4697049" />\n  <node id="3125684630" visible="true" version="2" changeset="177498262" timestamp="2026-01-20T22:53:08Z" user="talbertdoi" uid="23344915" lat="7.0024449" lon="100.4703896" />\n  <node id="3307271116" visible="true" version="1" changeset="28369127" timestamp="2015-01-24T10:36:36Z" user="Pathumthani" uid="2302962" lat="6.9979262" lon="100.4710975" />\n  <node id="3307271117" visible="true" version="2" changeset="69365375" timestamp="2019-04-19T05:21:53Z" user="hathaichanok" uid="9622871" lat="6.9979100" lon="100.4716972" />\n  <node id="3307296462" visible="true" version="1" changeset="28369303" timestamp="2015-01-24T10:47:02Z" user="Pathumthani" uid="2302962" lat="6.9986150" lon="100.4715739" />\n  <node id="3307296644" visible="true" version="3" changeset="183274597" timestamp="2026-05-27T20:34:13Z" user="anthaas" uid="5438501" lat="7.0020628" lon="100.4711110" />\n  <node id="3307296645" visible="true" version="1" changeset="28369316" timestamp="2015-01-24T10:47:53Z" user="Pathumthani" uid="2302962" lat="7.0020705" lon="100.4712158" />\n  <node id="3307300892" visible="true" version="1" changeset="28369342" timestamp="2015-01-24T10:49:07Z" user="Pathumthani" uid="2302962" lat="6.9986471" lon="100.4715702" />\n  <node id="3309949761" visible="true" version="1" changeset="28399115" timestamp="2015-01-25T16:03:53Z" user="Bernhard Hiller" uid="326704" lat="7.0134850" lon="100.4672637" />\n  <node id="3309949861" visible="true" version="1" changeset="28399115" timestamp="2015-01-25T16:03:53Z" user="Bernhard Hiller" uid="326704" lat="7.0139336" lon="100.4708351" />\n  <node id="3309949961" visible="true" version="1" changeset="28399115" timestamp="2015-01-25T16:03:54Z" user="Bernhard Hiller" uid="326704" lat="7.0141933" lon="100.4717628" />\n  <node id="3324717301" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0131249" lon="100.4761378" />\n  <node id="3324717336" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0131116" lon="100.4762156" />\n  <node id="3324717337" visible="true" version="6" changeset="188455657" timestamp="2026-09-03T07:33:19Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0132017" lon="100.4762257">\n    <tag k="entrance" v="yes" />\n  </node>\n  <node id="3324717486" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0132524" lon="100.4752891" />\n  <node id="3324717487" visible="true" version="1" changeset="28539190" timestamp="2015-02-01T08:08:44Z" user="Pathumthani" uid="2302962" lat="7.0132946" lon="100.4753103" />\n  <node id="3324717488" visible="true" version="1" changeset="28539190" timestamp="2015-02-01T08:08:44Z" user="Pathumthani" uid="2302962" lat="7.0133625" lon="100.4754162" />\n  <node id="3324717489" visible="true" version="1" changeset="28539190" timestamp="2015-02-01T08:08:44Z" user="Pathumthani" uid="2302962" lat="7.0133772" lon="100.4755087" />\n  <node id="3324717490" visible="true" version="1" changeset="28539190" timestamp="2015-02-01T08:08:44Z" user="Pathumthani" uid="2302962" lat="7.0133772" lon="100.4756201" />\n  <node id="3324717491" visible="true" version="1" changeset="28539190" timestamp="2015-02-01T08:08:44Z" user="Pathumthani" uid="2302962" lat="7.0133677" lon="100.4757022" />\n  <node id="3324718280" visible="true" version="3" changeset="188345456" timestamp="2026-09-01T06:28:57Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0132576" lon="100.4752544" />\n  <node id="3324718281" visible="true" version="2" changeset="188345456" timestamp="2026-09-01T06:28:57Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0134363" lon="100.4752571" />\n  <node id="3324718282" visible="true" version="1" changeset="28539174" timestamp="2015-02-01T08:07:09Z" user="Pathumthani" uid="2302962" lat="7.0136221" lon="100.4752513" />\n  <node id="3324718283" visible="true" version="2" changeset="188345456" timestamp="2026-09-01T06:28:57Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0136261" lon="100.4752255" />\n  <node id="3324718284" visible="true" version="2" changeset="188345456" timestamp="2026-09-01T06:28:57Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0136284" lon="100.4751775" />\n  <node id="3324718285" visible="true" version="2" changeset="188345456" timestamp="2026-09-01T06:28:57Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0136320" lon="100.4751347" />\n  <node id="3324718905" visible="true" version="2" changeset="177500370" timestamp="2026-01-21T01:02:59Z" user="talbertdoi" uid="23344915" lat="7.0133458" lon="100.4758004" />\n  <node id="3324718906" visible="true" version="2" changeset="177500370" timestamp="2026-01-21T01:02:59Z" user="talbertdoi" uid="23344915" lat="7.0133072" lon="100.4759326" />\n  <node id="3324719149" visible="true" version="3" changeset="188455690" timestamp="2026-09-03T07:33:55Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0132753" lon="100.4760707">\n    <tag k="entrance" v="yes" />\n  </node>\n  <node id="3324721380" visible="true" version="2" changeset="188345456" timestamp="2026-09-01T06:28:57Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0134823" lon="100.4752591" />\n  <node id="3324721381" visible="true" version="1" changeset="28539264" timestamp="2015-02-01T08:13:26Z" user="Pathumthani" uid="2302962" lat="7.0134729" lon="100.4753036" />\n  <node id="3324723462" visible="true" version="1" changeset="28539266" timestamp="2015-02-01T08:13:41Z" user="Pathumthani" uid="2302962" lat="7.0134464" lon="100.4754417" />\n  <node id="3376078960" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0110284" lon="100.4766161" />\n  <node id="3376078961" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0120858" lon="100.4754430" />\n  <node id="3376078962" visible="true" version="2" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0120955" lon="100.4752444" />\n  <node id="3376084776" visible="true" version="2" changeset="148237597" timestamp="2024-03-05T07:55:56Z" user="GrabTH_Panuwit" uid="16483995" lat="7.0110979" lon="100.4742750" />\n  <node id="3376084894" visible="true" version="1" changeset="29150100" timestamp="2015-02-28T07:32:32Z" user="Pathumthani" uid="2302962" lat="7.0137408" lon="100.4710427" />\n  <node id="3376084911" visible="true" version="1" changeset="29150323" timestamp="2015-02-28T07:51:59Z" user="Pathumthani" uid="2302962" lat="7.0110215" lon="100.4765611" />\n  <node id="3376085828" visible="true" version="2" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0138824" lon="100.4709713" />\n  <node id="3376088758" visible="true" version="3" changeset="159286562" timestamp="2024-11-18T14:45:58Z" user="nabilersyad" uid="11419275" lat="7.0113319" lon="100.4743960" />\n  <node id="3376090057" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0115869" lon="100.4757532" />\n  <node id="3376090058" visible="true" version="1" changeset="29150293" timestamp="2015-02-28T07:49:47Z" user="Pathumthani" uid="2302962" lat="7.0115804" lon="100.4756802" />\n  <node id="3376091310" visible="true" version="3" changeset="148237597" timestamp="2024-03-05T07:55:56Z" user="GrabTH_Panuwit" uid="16483995" lat="7.0112637" lon="100.4742794" />\n  <node id="3376091512" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0115678" lon="100.4755264" />\n  <node id="3376091513" visible="true" version="1" changeset="29150298" timestamp="2015-02-28T07:49:59Z" user="Pathumthani" uid="2302962" lat="7.0115711" lon="100.4755689" />\n  <node id="3376091528" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0107833" lon="100.4764638" />\n  <node id="3376091818" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0108948" lon="100.4764583" />\n  <node id="3376092311" visible="true" version="1" changeset="29150328" timestamp="2015-02-28T07:52:35Z" user="Pathumthani" uid="2302962" lat="7.0108377" lon="100.4768389" />\n  <node id="3615054036" visible="true" version="1" changeset="32182188" timestamp="2015-06-24T12:18:30Z" user="Pathumthani" uid="2302962" lat="7.0131646" lon="100.4708556" />\n  <node id="3615054037" visible="true" version="1" changeset="32182188" timestamp="2015-06-24T12:18:30Z" user="Pathumthani" uid="2302962" lat="7.0131955" lon="100.4707704" />\n  <node id="3615054038" visible="true" version="1" changeset="32182188" timestamp="2015-06-24T12:18:30Z" user="Pathumthani" uid="2302962" lat="7.0132354" lon="100.4707423" />\n  <node id="3615054039" visible="true" version="1" changeset="32182188" timestamp="2015-06-24T12:18:30Z" user="Pathumthani" uid="2302962" lat="7.0132886" lon="100.4707404" />\n  <node id="3615056548" visible="true" version="1" changeset="32182197" timestamp="2015-06-24T12:18:52Z" user="Pathumthani" uid="2302962" lat="7.0133708" lon="100.4707604" />\n  <node id="3615056549" visible="true" version="1" changeset="32182197" timestamp="2015-06-24T12:18:52Z" user="Pathumthani" uid="2302962" lat="7.0134018" lon="100.4707744" />\n  <node id="3615056550" visible="true" version="1" changeset="32182197" timestamp="2015-06-24T12:18:52Z" user="Pathumthani" uid="2302962" lat="7.0134258" lon="100.4708241" />\n  <node id="3615056551" visible="true" version="1" changeset="32182197" timestamp="2015-06-24T12:18:52Z" user="Pathumthani" uid="2302962" lat="7.0134524" lon="100.4709005" />\n  <node id="3615056552" visible="true" version="1" changeset="32182197" timestamp="2015-06-24T12:18:52Z" user="Pathumthani" uid="2302962" lat="7.0134641" lon="100.4709642" />\n  <node id="3616902435" visible="true" version="1" changeset="32198572" timestamp="2015-06-25T06:54:58Z" user="Pathumthani" uid="2302962" lat="7.0141650" lon="100.4723182" />\n  <node id="4085816301" visible="true" version="1" changeset="38156259" timestamp="2016-03-30T02:00:25Z" user="Pathumthani" uid="2302962" lat="7.0077738" lon="100.4703107" />\n  <node id="4085816302" visible="true" version="2" changeset="159211955" timestamp="2024-11-16T15:16:41Z" user="nabilersyad" uid="11419275" lat="7.0076598" lon="100.4702926">\n    <tag k="highway" v="traffic_signals" />\n    <tag k="source" v="survey" />\n    <tag k="source:date" v="2016-03-29" />\n    <tag k="traffic_signals" v="crossing" />\n  </node>\n  <node id="4085817731" visible="true" version="2" changeset="108306644" timestamp="2021-07-20T09:48:02Z" user="appik" uid="11515808" lat="7.0061458" lon="100.4718374" />\n  <node id="4085817732" visible="true" version="1" changeset="38156304" timestamp="2016-03-30T02:05:18Z" user="Pathumthani" uid="2302962" lat="7.0062286" lon="100.4724184" />\n  <node id="4488358856" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0102586" lon="100.4775228" />\n  <node id="4488358857" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0092835" lon="100.4771021" />\n  <node id="4488358858" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0101632" lon="100.4770649" />\n  <node id="4488366488" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0093167" lon="100.4784333" />\n  <node id="4488368089" visible="true" version="1" changeset="43482822" timestamp="2016-11-08T09:42:19Z" user="Johnny Carlsen" uid="38985" lat="7.0091849" lon="100.4785658" />\n  <node id="4488368090" visible="true" version="3" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0101754" lon="100.4779658" />\n  <node id="4488368092" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0094006" lon="100.4782855" />\n  <node id="485580840" visible="true" version="7" changeset="177057772" timestamp="2026-01-10T11:39:54Z" user="11aye" uid="22221182" lat="7.0039504" lon="100.4678729" />\n  <node id="485580842" visible="true" version="6" changeset="38156304" timestamp="2016-03-30T02:05:18Z" user="Pathumthani" uid="2302962" lat="7.0059709" lon="100.4705028" />\n  <node id="485580845" visible="true" version="4" changeset="176196082" timestamp="2025-12-20T20:47:05Z" user="RAytoun" uid="1414812" lat="7.0075612" lon="100.4696840" />\n  <node id="4888861604" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0140707" lon="100.4765066" />\n  <node id="4888861605" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0137780" lon="100.4767697" />\n  <node id="4888861606" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0138288" lon="100.4766224" />\n  <node id="4888861607" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0138260" lon="100.4765989" />\n  <node id="4888861608" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0138308" lon="100.4765757" />\n  <node id="4888861609" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0138427" lon="100.4765552" />\n  <node id="4888861610" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0138604" lon="100.4765397" />\n  <node id="4888861611" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0139705" lon="100.4765887" />\n  <node id="4888861612" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0139715" lon="100.4766119" />\n  <node id="4888861613" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0139630" lon="100.4766369" />\n  <node id="4888861614" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0139479" lon="100.4766563" />\n  <node id="4888861615" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0139272" lon="100.4766697" />\n  <node id="4888861616" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0139056" lon="100.4765289" />\n  <node id="4888861617" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0139283" lon="100.4765349" />\n  <node id="4888861618" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0139480" lon="100.4765478" />\n  <node id="4888861619" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0139626" lon="100.4765664" />\n  <node id="4888861620" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0139034" lon="100.4766753" />\n  <node id="4888862221" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0138790" lon="100.4766727" />\n  <node id="4888862222" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0138568" lon="100.4766621" />\n  <node id="4888862223" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0138395" lon="100.4766446" />\n  <node id="4888862224" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0140565" lon="100.4767313" />\n  <node id="4888862230" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0136276" lon="100.4763135" />\n  <node id="4889402216" visible="true" version="1" changeset="49159040" timestamp="2017-06-01T09:25:30Z" user="jeka p" uid="2385757" lat="6.9995832" lon="100.4798688" />\n  <node id="4889402217" visible="true" version="2" changeset="103180368" timestamp="2021-04-19T08:40:06Z" user="bomma_1" uid="11659195" lat="6.9983790" lon="100.4797098" />\n  <node id="4900464580" visible="true" version="2" changeset="177513068" timestamp="2026-01-21T09:47:33Z" user="Lauriegisborne" uid="23651680" lat="7.0079891" lon="100.4760178" />\n  <node id="4900464584" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:38Z" user="RVR004" uid="5314349" lat="7.0072545" lon="100.4767552" />\n  <node id="4900464587" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0086417" lon="100.4784834" />\n  <node id="4900464588" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:38Z" user="RVR004" uid="5314349" lat="7.0085548" lon="100.4775328" />\n  <node id="4900464597" visible="true" version="2" changeset="177513068" timestamp="2026-01-21T09:47:33Z" user="Lauriegisborne" uid="23651680" lat="7.0088257" lon="100.4760274" />\n  <node id="4900464598" visible="true" version="2" changeset="175441976" timestamp="2025-12-03T11:14:45Z" user="WorayutTK" uid="23562084" lat="7.0072148" lon="100.4760142" />\n  <node id="4900465025" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0113314" lon="100.4755512" />\n  <node id="4900465026" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0110045" lon="100.4756413" />\n  <node id="4900465027" visible="true" version="2" changeset="159040528" timestamp="2024-11-12T08:48:17Z" user="nabilersyad" uid="11419275" lat="7.0107630" lon="100.4759911" />\n  <node id="4900465030" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:39Z" user="RVR004" uid="5314349" lat="7.0022403" lon="100.4832773" />\n  <node id="4900465031" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:39Z" user="RVR004" uid="5314349" lat="7.0020653" lon="100.4828823" />\n  <node id="4900465052" visible="true" version="3" changeset="176054561" timestamp="2025-12-17T14:53:03Z" user="Wim Keijsers" uid="22692008" lat="7.0134523" lon="100.4819247" />\n  <node id="4900465067" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:40Z" user="RVR004" uid="5314349" lat="7.0140659" lon="100.4791978" />\n  <node id="4900465068" visible="true" version="2" changeset="57398285" timestamp="2018-03-21T20:05:06Z" user="VLD009" uid="5620206" lat="7.0136653" lon="100.4791775" />\n  <node id="4900465069" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0136552" lon="100.4794472" />\n  <node id="4900465070" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:40Z" user="RVR004" uid="5314349" lat="7.0032277" lon="100.4823528" />\n  <node id="4900465071" visible="true" version="2" changeset="176134821" timestamp="2025-12-19T10:51:27Z" user="Klank123" uid="23623862" lat="7.0031313" lon="100.4819359" />\n  <node id="4900465090" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:40Z" user="RVR004" uid="5314349" lat="7.0134331" lon="100.4701188" />\n  <node id="4900465091" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:40Z" user="RVR004" uid="5314349" lat="7.0132446" lon="100.4686692" />\n  <node id="4900465092" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:40Z" user="RVR004" uid="5314349" lat="7.0130920" lon="100.4673184" />\n  <node id="4900465098" visible="true" version="2" changeset="176104364" timestamp="2025-12-18T16:45:43Z" user="angela_wl" uid="6666296" lat="7.0067521" lon="100.4728311" />\n  <node id="4900465099" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:40Z" user="RVR004" uid="5314349" lat="7.0075899" lon="100.4725064" />\n  <node id="4900465166" visible="true" version="3" changeset="163432352" timestamp="2025-03-10T09:06:18Z" user="GrabTH_Praiya" uid="18100118" lat="7.0067018" lon="100.4753863" />\n  <node id="4900465181" visible="true" version="2" changeset="176365051" timestamp="2025-12-24T14:36:05Z" user="11aye" uid="22221182" lat="7.0120937" lon="100.4693501" />\n  <node id="4900465182" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:41Z" user="RVR004" uid="5314349" lat="7.0126792" lon="100.4693424" />\n  <node id="4900465198" visible="true" version="2" changeset="188989640" timestamp="2026-09-14T04:03:52Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0118005" lon="100.4717455" />\n  <node id="4900465208" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:42Z" user="RVR004" uid="5314349" lat="7.0029360" lon="100.4725423" />\n  <node id="4900465209" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:42Z" user="RVR004" uid="5314349" lat="7.0037124" lon="100.4724480" />\n  <node id="4900465210" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:42Z" user="RVR004" uid="5314349" lat="7.0021012" lon="100.4751228" />\n  <node id="4900465216" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:42Z" user="RVR004" uid="5314349" lat="7.0099281" lon="100.4706349" />\n  <node id="4900465219" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:42Z" user="RVR004" uid="5314349" lat="7.0034072" lon="100.4776405" />\n  <node id="4900465244" visible="true" version="2" changeset="175444533" timestamp="2025-12-03T12:26:55Z" user="WorayutTK" uid="23562084" lat="7.0042517" lon="100.4796545" />\n  <node id="4900465277" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:43Z" user="RVR004" uid="5314349" lat="7.0087433" lon="100.4868002" />\n  <node id="4900465288" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0123942" lon="100.4785045" />\n  <node id="4900465324" visible="true" version="2" changeset="148237597" timestamp="2024-03-05T07:55:56Z" user="GrabTH_Panuwit" uid="16483995" lat="7.0119591" lon="100.4734872" />\n  <node id="4900465339" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:44Z" user="RVR004" uid="5314349" lat="7.0088286" lon="100.4835600" />\n  <node id="4900465363" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:44Z" user="RVR004" uid="5314349" lat="7.0094928" lon="100.4723627" />\n  <node id="4900465364" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:44Z" user="RVR004" uid="5314349" lat="7.0094389" lon="100.4717569" />\n  <node id="4900465369" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:44Z" user="RVR004" uid="5314349" lat="7.0131100" lon="100.4701771" />\n  <node id="4900465370" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:44Z" user="RVR004" uid="5314349" lat="7.0130965" lon="100.4704554" />\n  <node id="4900465377" visible="true" version="2" changeset="176134821" timestamp="2025-12-19T10:51:27Z" user="Klank123" uid="23623862" lat="7.0065167" lon="100.4829633" />\n  <node id="4900465379" visible="true" version="2" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0080946" lon="100.4736247" />\n  <node id="4900465413" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:45Z" user="RVR004" uid="5314349" lat="7.0046952" lon="100.4756030" />\n  <node id="4900465425" visible="true" version="2" changeset="177256748" timestamp="2026-01-15T12:17:04Z" user="Maptes" uid="23330268" lat="7.0007323" lon="100.4830486" />\n  <node id="4900465440" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:45Z" user="RVR004" uid="5314349" lat="7.0102467" lon="100.4820745" />\n  <node id="4900465441" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:45Z" user="RVR004" uid="5314349" lat="7.0105743" lon="100.4820566" />\n  <node id="4900465446" visible="true" version="3" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0122705" lon="100.4753460" />\n  <node id="4900465451" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0085994" lon="100.4788935" />\n  <node id="4900465471" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:46Z" user="RVR004" uid="5314349" lat="7.0048972" lon="100.4726679" />\n  <node id="4900465472" visible="true" version="2" changeset="176632477" timestamp="2025-12-30T15:05:11Z" user="Tharhtetaung" uid="16875287" lat="7.0049424" lon="100.4731341" />\n  <node id="4900465473" visible="true" version="3" changeset="182517970" timestamp="2026-05-11T15:45:42Z" user="Emma Js" uid="22434897" lat="7.0050009" lon="100.4735593" />\n  <node id="4900465512" visible="true" version="4" changeset="152871448" timestamp="2024-06-19T02:26:33Z" user="GrabTH_Napittha" uid="15509145" lat="7.0064757" lon="100.4713810" />\n  <node id="4900465513" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:46Z" user="RVR004" uid="5314349" lat="7.0073296" lon="100.4712004" />\n  <node id="4900465517" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:46Z" user="RVR004" uid="5314349" lat="7.0108975" lon="100.4721743" />\n  <node id="4900465641" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:47Z" user="RVR004" uid="5314349" lat="7.0018095" lon="100.4823932" />\n  <node id="4900465665" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:47Z" user="RVR004" uid="5314349" lat="7.0067776" lon="100.4767609" />\n  <node id="4900465668" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:47Z" user="RVR004" uid="5314349" lat="7.0127285" lon="100.4701009" />\n  <node id="4900465688" visible="true" version="3" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0113653" lon="100.4738109" />\n  <node id="4900465692" visible="true" version="3" changeset="176057240" timestamp="2025-12-17T15:59:32Z" user="Wim Keijsers" uid="22692008" lat="7.0136709" lon="100.4837973" />\n  <node id="4900465706" visible="true" version="2" changeset="188076387" timestamp="2026-08-27T04:49:30Z" user="GrabTH_Tanyamai" uid="23350424" lat="7.0119198" lon="100.4708966" />\n  <node id="4900465707" visible="true" version="2" changeset="189138277" timestamp="2026-09-17T03:16:49Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0072973" lon="100.4783660" />\n  <node id="4900465708" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:48Z" user="RVR004" uid="5314349" lat="7.0066071" lon="100.4766846" />\n  <node id="4900465935" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0110130" lon="100.4761822" />\n  <node id="4900465939" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0115665" lon="100.4817213" />\n  <node id="4900465941" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0053117" lon="100.4806781" />\n  <node id="4900465942" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="6.9990356" lon="100.4796314" />\n  <node id="4900465943" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:48Z" user="RVR004" uid="5314349" lat="7.0146493" lon="100.4726814" />\n  <node id="4900465949" visible="true" version="2" changeset="103180368" timestamp="2021-04-19T08:40:06Z" user="bomma_1" uid="11659195" lat="7.0025218" lon="100.4803742" />\n  <node id="4900465955" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:49Z" user="RVR004" uid="5314349" lat="7.0108167" lon="100.4684987" />\n  <node id="4900465957" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:49Z" user="RVR004" uid="5314349" lat="7.0049061" lon="100.4779681" />\n  <node id="4900465971" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:49Z" user="RVR004" uid="5314349" lat="7.0029494" lon="100.4726814" />\n  <node id="4900465972" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:49Z" user="RVR004" uid="5314349" lat="7.0097710" lon="100.4723672" />\n  <node id="4900465989" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:49Z" user="RVR004" uid="5314349" lat="7.0101087" lon="100.4707020" />\n  <node id="4900465991" visible="true" version="2" changeset="159040528" timestamp="2024-11-12T08:48:17Z" user="nabilersyad" uid="11419275" lat="7.0108436" lon="100.4758237" />\n  <node id="4900465992" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0111239" lon="100.4756064" />\n  <node id="4900465993" visible="true" version="3" changeset="159211010" timestamp="2024-11-16T14:48:54Z" user="nabilersyad" uid="11419275" lat="7.0107056" lon="100.4761527" />\n  <node id="4900465994" visible="true" version="3" changeset="159211010" timestamp="2024-11-16T14:48:54Z" user="nabilersyad" uid="11419275" lat="7.0106798" lon="100.4763092" />\n  <node id="4900465995" visible="true" version="2" changeset="159040528" timestamp="2024-11-12T08:48:17Z" user="nabilersyad" uid="11419275" lat="7.0108003" lon="100.4759103" />\n  <node id="4900465996" visible="true" version="2" changeset="159040528" timestamp="2024-11-12T08:48:17Z" user="nabilersyad" uid="11419275" lat="7.0108857" lon="100.4757595" />\n  <node id="4900465999" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:49Z" user="RVR004" uid="5314349" lat="7.0066732" lon="100.4767217" />\n  <node id="4900466000" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:49Z" user="RVR004" uid="5314349" lat="7.0065068" lon="100.4766604" />\n  <node id="4900466001" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:49Z" user="RVR004" uid="5314349" lat="7.0064137" lon="100.4766766" />\n  <node id="4900466002" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:49Z" user="RVR004" uid="5314349" lat="7.0039098" lon="100.4780711" />\n  <node id="4900466003" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:49Z" user="RVR004" uid="5314349" lat="7.0038740" lon="100.4776061" />\n  <node id="4900466225" visible="true" version="2" changeset="103175191" timestamp="2021-04-19T07:33:31Z" user="bomma_1" uid="11659195" lat="7.0013518" lon="100.4829954" />\n  <node id="4900466295" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:51Z" user="RVR004" uid="5314349" lat="7.0127171" lon="100.4706904" />\n  <node id="4900466296" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:51Z" user="RVR004" uid="5314349" lat="7.0134962" lon="100.4705932" />\n  <node id="4900466297" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:51Z" user="RVR004" uid="5314349" lat="7.0135386" lon="100.4709912" />\n  <node id="4900466448" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:52Z" user="RVR004" uid="5314349" lat="7.0079170" lon="100.4752638" />\n  <node id="4900466449" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:52Z" user="RVR004" uid="5314349" lat="7.0078701" lon="100.4746753" />\n  <node id="5149825981" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0081783" lon="100.4785697" />\n  <node id="5149825982" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0081249" lon="100.4788003" />\n  <node id="5149825983" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0083019" lon="100.4787851" />\n  <node id="5149825984" visible="true" version="1" changeset="52655906" timestamp="2017-10-05T14:16:28Z" user="Sal73x" uid="1293386" lat="7.0090671" lon="100.4786303" />\n  <node id="5405819483" visible="true" version="1" changeset="56270838" timestamp="2018-02-11T18:28:11Z" user="Megre" uid="6684068" lat="7.0020622" lon="100.4711588">\n    <tag k="highway" v="crossing" />\n  </node>\n  <node id="5405820221" visible="true" version="1" changeset="56270853" timestamp="2018-02-11T18:28:52Z" user="Megre" uid="6684068" lat="7.0167544" lon="100.4769749" />\n  <node id="5405820222" visible="true" version="1" changeset="56270853" timestamp="2018-02-11T18:28:52Z" user="Megre" uid="6684068" lat="7.0168662" lon="100.4772190" />\n  <node id="617233014" visible="true" version="6" changeset="176196082" timestamp="2025-12-20T20:47:05Z" user="RAytoun" uid="1414812" lat="7.0059052" lon="100.4699124" />\n  <node id="617234280" visible="true" version="2" changeset="7524843" timestamp="2011-03-11T16:16:12Z" user="sdayu" uid="300693" lat="7.0042411" lon="100.4700794" />\n  <node id="617234281" visible="true" version="3" changeset="7524843" timestamp="2011-03-11T16:16:02Z" user="sdayu" uid="300693" lat="7.0043579" lon="100.4715036" />\n  <node id="617234534" visible="true" version="5" changeset="177498262" timestamp="2026-01-20T22:53:08Z" user="talbertdoi" uid="23344915" lat="7.0024922" lon="100.4679467" />\n  <node id="617234535" visible="true" version="5" changeset="177498262" timestamp="2026-01-20T22:53:08Z" user="talbertdoi" uid="23344915" lat="7.0028085" lon="100.4703374" />\n  <node id="617234576" visible="true" version="2" changeset="7524843" timestamp="2011-03-11T16:16:02Z" user="sdayu" uid="300693" lat="7.0043138" lon="100.4707399" />\n  <node id="617234843" visible="true" version="7" changeset="38156304" timestamp="2016-03-30T02:05:18Z" user="Pathumthani" uid="2302962" lat="7.0060498" lon="100.4712646" />\n  <node id="620881097" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:37Z" user="Bernhard Hiller" uid="326704" lat="7.0089826" lon="100.4701012" />\n  <node id="620881181" visible="true" version="5" changeset="159211955" timestamp="2024-11-16T15:16:41Z" user="nabilersyad" uid="11419275" lat="7.0077182" lon="100.4705536" />\n  <node id="620881522" visible="true" version="5" changeset="177498262" timestamp="2026-01-20T22:53:08Z" user="talbertdoi" uid="23344915" lat="7.0025909" lon="100.4687079" />\n  <node id="620881529" visible="true" version="4" changeset="159211712" timestamp="2024-11-16T15:09:46Z" user="nabilersyad" uid="11419275" lat="7.0040355" lon="100.4684890" />\n  <node id="620881544" visible="true" version="6" changeset="177498262" timestamp="2026-01-20T22:53:08Z" user="talbertdoi" uid="23344915" lat="7.0027276" lon="100.4696464">\n    <tag k="highway" v="traffic_signals" />\n  </node>\n  <node id="620881566" visible="true" version="5" changeset="29154973" timestamp="2015-02-28T12:54:36Z" user="Pathumthani" uid="2302962" lat="7.0041770" lon="100.4694581">\n    <tag k="highway" v="traffic_signals" />\n  </node>\n  <node id="620881567" visible="true" version="3" changeset="7524843" timestamp="2011-03-11T16:16:15Z" user="sdayu" uid="300693" lat="7.0058148" lon="100.4691974" />\n  <node id="620881568" visible="true" version="3" changeset="18489522" timestamp="2013-10-22T17:36:44Z" user="Bernhard Hiller" uid="326704" lat="7.0074992" lon="100.4688970" />\n  <node id="6634608141" visible="true" version="1" changeset="72473162" timestamp="2019-07-21T03:32:09Z" user="JAAS" uid="3321532" lat="7.0045910" lon="100.4744510" />\n  <node id="6634608142" visible="true" version="2" changeset="184635890" timestamp="2026-06-26T09:22:45Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0044560" lon="100.4744932" />\n  <node id="6634630594" visible="true" version="1" changeset="72473287" timestamp="2019-07-21T03:47:56Z" user="JAAS" uid="3321532" lat="7.0105369" lon="100.4703752" />\n  <node id="6634630596" visible="true" version="1" changeset="72473287" timestamp="2019-07-21T03:47:56Z" user="JAAS" uid="3321532" lat="7.0100600" lon="100.4702142" />\n  <node id="6634648309" visible="true" version="3" changeset="176397735" timestamp="2025-12-25T14:44:47Z" user="11aye" uid="22221182" lat="7.0147109" lon="100.4794878" />\n  <node id="6634655046" visible="true" version="2" changeset="159040528" timestamp="2024-11-12T08:48:17Z" user="nabilersyad" uid="11419275" lat="7.0118651" lon="100.4752471" />\n  <node id="6634655047" visible="true" version="2" changeset="159040528" timestamp="2024-11-12T08:48:17Z" user="nabilersyad" uid="11419275" lat="7.0117638" lon="100.4750885" />\n  <node id="6634655048" visible="true" version="2" changeset="159040528" timestamp="2024-11-12T08:48:17Z" user="nabilersyad" uid="11419275" lat="7.0113167" lon="100.4747743" />\n  <node id="6634655049" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0118467" lon="100.4753290" />\n  <node id="6634655051" visible="true" version="1" changeset="72473869" timestamp="2019-07-21T05:10:52Z" user="JAAS" uid="3321532" lat="7.0097174" lon="100.4711362" />\n  <node id="6634655052" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0117740" lon="100.4754445" />\n  <node id="6634656806" visible="true" version="3" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915" lat="7.0117873" lon="100.4785174" />\n  <node id="6822269571" visible="true" version="1" changeset="74898854" timestamp="2019-09-25T09:53:19Z" user="icosiam" uid="10316678" lat="7.0020110" lon="100.4734327" />\n  <node id="6822929410" visible="true" version="1" changeset="74914002" timestamp="2019-09-25T14:34:34Z" user="icosiam" uid="10316678" lat="7.0141138" lon="100.4747869" />\n  <node id="7401374741" visible="true" version="2" changeset="176196082" timestamp="2025-12-20T20:47:05Z" user="RAytoun" uid="1414812" lat="7.0075370" lon="100.4694806" />\n  <node id="7500979776" visible="true" version="3" changeset="176196082" timestamp="2025-12-20T20:47:05Z" user="RAytoun" uid="1414812" lat="7.0075900" lon="100.4697921">\n    <tag k="tactile_paving" v="no" />\n  </node>\n  <node id="7501768906" visible="true" version="2" changeset="176196082" timestamp="2025-12-20T20:47:05Z" user="RAytoun" uid="1414812" lat="7.0075525" lon="100.4695955" />\n  <node id="7504254568" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0076201" lon="100.4694095" />\n  <node id="7504254569" visible="true" version="2" changeset="183269728" timestamp="2026-05-27T18:24:44Z" user="anthaas" uid="5438501" lat="7.0077037" lon="100.4702844" />\n  <node id="7504254570" visible="true" version="2" changeset="183269728" timestamp="2026-05-27T18:24:44Z" user="anthaas" uid="5438501" lat="7.0078348" lon="100.4702634" />\n  <node id="7504254571" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0083509" lon="100.4701272" />\n  <node id="7504254572" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0076340" lon="100.4694895" />\n  <node id="7504254574" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0076468" lon="100.4695631" />\n  <node id="7504254576" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0076978" lon="100.4698571" />\n  <node id="7504254577" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0082923" lon="100.4697568" />\n  <node id="7504254578" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0083248" lon="100.4699619" />\n  <node id="7504254579" visible="true" version="2" changeset="183269728" timestamp="2026-05-27T18:24:44Z" user="anthaas" uid="5438501" lat="7.0076774" lon="100.4700834" />\n  <node id="7504254580" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0082031" lon="100.4699858" />\n  <node id="7504254581" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0082344" lon="100.4701483" />\n  <node id="7504254582" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0082196" lon="100.4700716" />\n  <node id="7504254583" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0083382" lon="100.4700470" />\n  <node id="7504254584" visible="true" version="2" changeset="183269728" timestamp="2026-05-27T18:24:44Z" user="anthaas" uid="5438501" lat="7.0080322" lon="100.4700168" />\n  <node id="7504272685" visible="true" version="2" changeset="183269728" timestamp="2026-05-27T18:24:44Z" user="anthaas" uid="5438501" lat="7.0080624" lon="100.4701774" />\n  <node id="7504272686" visible="true" version="2" changeset="183269728" timestamp="2026-05-27T18:24:44Z" user="anthaas" uid="5438501" lat="7.0080285" lon="100.4701863" />\n  <node id="7504272687" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0079318" lon="100.4700390" />\n  <node id="7504272688" visible="true" version="2" changeset="183269728" timestamp="2026-05-27T18:24:44Z" user="anthaas" uid="5438501" lat="7.0080339" lon="100.4702367" />\n  <node id="7504272689" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0078177" lon="100.4700613" />\n  <node id="7504272691" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0081441" lon="100.4699974" />\n  <node id="7504272692" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0079547" lon="100.4698137" />\n  <node id="7504272693" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0079885" lon="100.4700279" />\n  <node id="7504272694" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0082656" lon="100.4695878" />\n  <node id="7504272695" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0076679" lon="100.4696834" />\n  <node id="7504272696" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0076876" lon="100.4697830" />\n  <node id="7504272697" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0082818" lon="100.4696903" />\n  <node id="7504272700" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0082392" lon="100.4694034" />\n  <node id="7504272703" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0076249" lon="100.4694371" />\n  <node id="7504272706" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0076423" lon="100.4695373" />\n  <node id="7504272713" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0082578" lon="100.4695330" />\n  <node id="7504272714" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0076576" lon="100.4696250" />\n  <node id="7504272723" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0088446" lon="100.4696036" />\n  <node id="7504272724" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0089125" lon="100.4700339" />\n  <node id="7504272725" visible="true" version="1" changeset="84983944" timestamp="2020-05-10T19:22:08Z" user="zorree" uid="10282398" lat="7.0088779" lon="100.4698145" />\n  <node id="7508595062" visible="true" version="1" changeset="85043654" timestamp="2020-05-11T20:00:41Z" user="zorree" uid="10282398" lat="7.0083888" lon="100.4694530" />\n  <node id="7508595063" visible="true" version="1" changeset="85043654" timestamp="2020-05-11T20:00:41Z" user="zorree" uid="10282398" lat="7.0084924" lon="100.4701032" />\n  <node id="7508595064" visible="true" version="1" changeset="85043654" timestamp="2020-05-11T20:00:41Z" user="zorree" uid="10282398" lat="7.0083136" lon="100.4694280" />\n  <node id="7508595065" visible="true" version="1" changeset="85043654" timestamp="2020-05-11T20:00:41Z" user="zorree" uid="10282398" lat="7.0084286" lon="100.4701143" />\n  <node id="7508595067" visible="true" version="1" changeset="85043654" timestamp="2020-05-11T20:00:41Z" user="zorree" uid="10282398" lat="7.0085444" lon="100.4700946" />\n  <node id="7508595068" visible="true" version="1" changeset="85043654" timestamp="2020-05-11T20:00:41Z" user="zorree" uid="10282398" lat="7.0085693" lon="100.4701815" />\n  <node id="7508595069" visible="true" version="1" changeset="85043654" timestamp="2020-05-11T20:00:41Z" user="zorree" uid="10282398" lat="7.0080431" lon="100.4701828" />\n  <node id="7508595070" visible="true" version="1" changeset="85043654" timestamp="2020-05-11T20:00:41Z" user="zorree" uid="10282398" lat="7.0080520" lon="100.4702664" />\n  <node id="7508595072" visible="true" version="1" changeset="85043654" timestamp="2020-05-11T20:00:41Z" user="zorree" uid="10282398" lat="7.0087763" lon="100.4695810" />\n  <node id="7508595073" visible="true" version="1" changeset="85043654" timestamp="2020-05-11T20:00:41Z" user="zorree" uid="10282398" lat="7.0088165" lon="100.4700497" />\n  <node id="7508595074" visible="true" version="1" changeset="85043654" timestamp="2020-05-11T20:00:41Z" user="zorree" uid="10282398" lat="7.0088328" lon="100.4701303" />\n  <node id="7508595075" visible="true" version="1" changeset="85043654" timestamp="2020-05-11T20:00:41Z" user="zorree" uid="10282398" lat="7.0088697" lon="100.4697626" />\n  <node id="7508595076" visible="true" version="1" changeset="85043654" timestamp="2020-05-11T20:00:41Z" user="zorree" uid="10282398" lat="7.0089410" lon="100.4697541" />\n  <node id="7508595077" visible="true" version="1" changeset="85043654" timestamp="2020-05-11T20:00:41Z" user="zorree" uid="10282398" lat="7.0076510" lon="100.4695869" />\n  <node id="7691192942" visible="true" version="2" changeset="141710429" timestamp="2023-09-25T03:32:32Z" user="dmatted" uid="3047966" lat="7.0088101" lon="100.4877319">\n    <tag k="highway" v="traffic_signals" />\n    <tag k="traffic_signals" v="traffic_lights" />\n    <tag k="traffic_signals:countdown" v="yes" />\n    <tag k="traffic_signals:direction" v="forward" />\n  </node>\n  <node id="7691192946" visible="true" version="2" changeset="176057240" timestamp="2025-12-17T15:59:32Z" user="Wim Keijsers" uid="22692008" lat="7.0138341" lon="100.4851134">\n    <tag k="highway" v="traffic_signals" />\n    <tag k="traffic_signals" v="signal" />\n  </node>\n  <node id="796720075" visible="true" version="5" changeset="177599388" timestamp="2026-01-23T09:23:50Z" user="GrabTH_Bulakorn" uid="23192083" lat="6.9983906" lon="100.4794398" />\n  <node id="796747251" visible="true" version="3" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145" lat="7.0079967" lon="100.4810115" />\n  <node id="796747479" visible="true" version="3" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145" lat="7.0086723" lon="100.4811304" />\n  <node id="796747582" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0009730" lon="100.4798513" />\n  <node id="796747634" visible="true" version="3" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0065342" lon="100.4807754" />\n  <node id="796747714" visible="true" version="4" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0042416" lon="100.4803944" />\n  <node id="796747766" visible="true" version="3" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0129005" lon="100.4818333" />\n  <node id="809537378" visible="true" version="4" changeset="18489522" timestamp="2013-10-22T17:36:51Z" user="Bernhard Hiller" uid="326704" lat="7.0179486" lon="100.4796474" />\n  <node id="809537390" visible="true" version="3" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="6.9985915" lon="100.4795573" />\n  <node id="809537392" visible="true" version="1" changeset="5179747" timestamp="2010-07-10T06:56:01Z" user="sdayu" uid="300693" lat="7.0181923" lon="100.4803316" />\n  <node id="809537546" visible="true" version="3" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0064668" lon="100.4808707" />\n  <node id="809537690" visible="true" version="4" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145" lat="7.0090140" lon="100.4814405" />\n  <node id="809537730" visible="true" version="3" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="6.9983836" lon="100.4795413" />\n  <node id="809537754" visible="true" version="3" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0089172" lon="100.4840805" />\n  <node id="809537790" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="6.9984690" lon="100.4795369" />\n  <node id="809537792" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0044496" lon="100.4805343" />\n  <node id="809537797" visible="true" version="3" changeset="18489522" timestamp="2013-10-22T17:36:53Z" user="Bernhard Hiller" uid="326704" lat="7.0180708" lon="100.4795991" />\n  <node id="809537800" visible="true" version="1" changeset="5179747" timestamp="2010-07-10T06:56:10Z" user="sdayu" uid="300693" lat="7.0180355" lon="100.4799202" />\n  <node id="809537807" visible="true" version="1" changeset="5179747" timestamp="2010-07-10T06:56:11Z" user="sdayu" uid="300693" lat="7.0183085" lon="100.4808003" />\n  <node id="809537820" visible="true" version="5" changeset="189908183" timestamp="2026-10-03T03:44:58Z" user="dmatted" uid="3047966" lat="7.0089067" lon="100.4814243">\n    <tag k="highway" v="traffic_signals" />\n    <tag k="traffic_signals:direction" v="both" />\n  </node>\n  <node id="823954361" visible="true" version="4" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0133764" lon="100.4852179" />\n  <node id="823954364" visible="true" version="3" changeset="176054561" timestamp="2025-12-17T14:53:03Z" user="Wim Keijsers" uid="22692008" lat="7.0133499" lon="100.4849123" />\n  <node id="8348961008" visible="true" version="1" changeset="97972178" timestamp="2021-01-22T11:47:27Z" user="Dmitriev_DA" uid="9356997" lat="7.0092661" lon="100.4700538" />\n  <node id="8348961009" visible="true" version="1" changeset="97972178" timestamp="2021-01-22T11:47:27Z" user="Dmitriev_DA" uid="9356997" lat="7.0094509" lon="100.4711036" />\n  <node id="8363308288" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0086418" lon="100.4795820" />\n  <node id="8363308289" visible="true" version="1" changeset="98153886" timestamp="2021-01-26T06:16:04Z" user="Dmitriev_DA" uid="9356997" lat="7.0087297" lon="100.4801772" />\n  <node id="8363308290" visible="true" version="1" changeset="98153886" timestamp="2021-01-26T06:16:04Z" user="Dmitriev_DA" uid="9356997" lat="7.0084218" lon="100.4795958" />\n  <node id="8363308291" visible="true" version="1" changeset="98153886" timestamp="2021-01-26T06:16:04Z" user="Dmitriev_DA" uid="9356997" lat="7.0085453" lon="100.4802608" />\n  <node id="849403384" visible="true" version="3" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0051541" lon="100.4805461">\n    <tag k="highway" v="traffic_signals" />\n  </node>\n  <node id="849403490" visible="true" version="4" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0093218" lon="100.4852835" />\n  <node id="849403500" visible="true" version="3" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0051611" lon="100.4806530">\n    <tag k="highway" v="traffic_signals" />\n  </node>\n  <node id="8523307708" visible="true" version="1" changeset="101100223" timestamp="2021-03-16T07:43:59Z" user="oliviar_grab" uid="10664132" lat="7.0095651" lon="100.4700595" />\n  <node id="8523314484" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0059365" lon="100.4806761" />\n  <node id="8523314485" visible="true" version="1" changeset="101099745" timestamp="2021-03-16T07:37:49Z" user="oliviar_grab" uid="10664132" lat="7.0058571" lon="100.4801611" />\n  <node id="8523314486" visible="true" version="1" changeset="101099745" timestamp="2021-03-16T07:37:49Z" user="oliviar_grab" uid="10664132" lat="7.0058598" lon="100.4801229" />\n  <node id="8523314487" visible="true" version="1" changeset="101099745" timestamp="2021-03-16T07:37:49Z" user="oliviar_grab" uid="10664132" lat="7.0058784" lon="100.4801141" />\n  <node id="8523314488" visible="true" version="1" changeset="101099745" timestamp="2021-03-16T07:37:49Z" user="oliviar_grab" uid="10664132" lat="7.0059230" lon="100.4801068" />\n  <node id="8523314489" visible="true" version="1" changeset="101099745" timestamp="2021-03-16T07:37:49Z" user="oliviar_grab" uid="10664132" lat="7.0059457" lon="100.4800900" />\n  <node id="8523314490" visible="true" version="1" changeset="101099745" timestamp="2021-03-16T07:37:49Z" user="oliviar_grab" uid="10664132" lat="7.0059503" lon="100.4800464" />\n  <node id="8523314491" visible="true" version="1" changeset="101099745" timestamp="2021-03-16T07:37:49Z" user="oliviar_grab" uid="10664132" lat="7.0059517" lon="100.4799230" />\n  <node id="8523314492" visible="true" version="1" changeset="101099745" timestamp="2021-03-16T07:37:49Z" user="oliviar_grab" uid="10664132" lat="7.0059304" lon="100.4796756" />\n  <node id="8523314493" visible="true" version="2" changeset="186689791" timestamp="2026-07-31T07:31:36Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0059224" lon="100.4795140">\n    <tag k="access" v="private" />\n    <tag k="barrier" v="lift_gate" />\n    <tag k="lift_gate:type" v="single" />\n  </node>\n  <node id="8523314742" visible="true" version="2" changeset="181556965" timestamp="2026-04-19T22:13:33Z" user="wireguy" uid="9564652" lat="7.0053992" lon="100.4837896" />\n  <node id="8523321638" visible="true" version="2" changeset="175694243" timestamp="2025-12-09T03:29:04Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0032763" lon="100.4838523" />\n  <node id="8523326792" visible="true" version="3" changeset="176365051" timestamp="2025-12-24T14:36:05Z" user="11aye" uid="22221182" lat="7.0121159" lon="100.4698378" />\n  <node id="8523326797" visible="true" version="1" changeset="101100291" timestamp="2021-03-16T07:45:01Z" user="oliviar_grab" uid="10664132" lat="7.0112841" lon="100.4698379" />\n  <node id="8523326798" visible="true" version="1" changeset="101100291" timestamp="2021-03-16T07:45:01Z" user="oliviar_grab" uid="10664132" lat="7.0113039" lon="100.4688629" />\n  <node id="8523326805" visible="true" version="2" changeset="176365051" timestamp="2025-12-24T14:36:05Z" user="11aye" uid="22221182" lat="7.0120907" lon="100.4688871" />\n  <node id="8523326825" visible="true" version="2" changeset="181599991" timestamp="2026-04-20T20:10:17Z" user="wireguy" uid="9564652" lat="7.0063411" lon="100.4816237" />\n  <node id="8523326826" visible="true" version="2" changeset="181599991" timestamp="2026-04-20T20:10:17Z" user="wireguy" uid="9564652" lat="7.0066940" lon="100.4815818" />\n  <node id="8523328981" visible="true" version="1" changeset="101099622" timestamp="2021-03-16T07:36:22Z" user="oliviar_grab" uid="10664132" lat="7.0084701" lon="100.4820755" />\n  <node id="8523328982" visible="true" version="1" changeset="101099622" timestamp="2021-03-16T07:36:22Z" user="oliviar_grab" uid="10664132" lat="7.0083769" lon="100.4819521" />\n  <node id="8523328983" visible="true" version="1" changeset="101099622" timestamp="2021-03-16T07:36:22Z" user="oliviar_grab" uid="10664132" lat="7.0083343" lon="100.4818261" />\n  <node id="8523328984" visible="true" version="1" changeset="101099622" timestamp="2021-03-16T07:36:22Z" user="oliviar_grab" uid="10664132" lat="7.0083223" lon="100.4816839" />\n  <node id="8523328985" visible="true" version="1" changeset="101099622" timestamp="2021-03-16T07:36:22Z" user="oliviar_grab" uid="10664132" lat="7.0083210" lon="100.4815149" />\n  <node id="8523328986" visible="true" version="1" changeset="101099622" timestamp="2021-03-16T07:36:22Z" user="oliviar_grab" uid="10664132" lat="7.0083313" lon="100.4813212" />\n  <node id="8523328987" visible="true" version="1" changeset="101099622" timestamp="2021-03-16T07:36:22Z" user="oliviar_grab" uid="10664132" lat="7.0084994" lon="100.4832208" />\n  <node id="8523328988" visible="true" version="2" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0083346" lon="100.4832132" />\n  <node id="8523328989" visible="true" version="3" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485" lat="7.0082853" lon="100.4830591" />\n  <node id="8523328991" visible="true" version="2" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008" lat="7.0088240" lon="100.4839976" />\n  <node id="8523329815" visible="true" version="1" changeset="101100194" timestamp="2021-03-16T07:43:36Z" user="oliviar_grab" uid="10664132" lat="7.0089959" lon="100.4706272" />\n  <node id="8523329816" visible="true" version="1" changeset="101100194" timestamp="2021-03-16T07:43:36Z" user="oliviar_grab" uid="10664132" lat="7.0088778" lon="100.4701216" />\n  <node id="8523330992" visible="true" version="3" changeset="186690029" timestamp="2026-07-31T07:37:35Z" user="GrabTH_Monnapat" uid="24164407" lat="7.0080157" lon="100.4795970" />\n  <node id="8523332286" visible="true" version="2" changeset="115208364" timestamp="2021-12-21T12:32:31Z" user="than_mayee" uid="10626093" lat="7.0078851" lon="100.4776143">\n    <tag k="noexit" v="yes" />\n  </node>\n  <node id="8523332287" visible="true" version="1" changeset="101099828" timestamp="2021-03-16T07:38:52Z" user="oliviar_grab" uid="10664132" lat="7.0078897" lon="100.4777095" />\n  <node id="8523332288" visible="true" version="1" changeset="101099828" timestamp="2021-03-16T07:38:52Z" user="oliviar_grab" uid="10664132" lat="7.0078857" lon="100.4777947" />\n  <node id="8523332289" visible="true" version="2" changeset="175444256" timestamp="2025-12-03T12:18:40Z" user="WorayutTK" uid="23562084" lat="7.0078575" lon="100.4778316" />\n  <node id="8523332290" visible="true" version="2" changeset="175444256" timestamp="2025-12-03T12:18:40Z" user="WorayutTK" uid="23562084" lat="7.0077264" lon="100.4778736" />\n  <node id="8523332291" visible="true" version="2" changeset="175444256" timestamp="2025-12-03T12:18:40Z" user="WorayutTK" uid="23562084" lat="7.0072921" lon="100.4778641" />\n  <node id="8523332292" visible="true" version="2" changeset="175448516" timestamp="2025-12-03T14:01:18Z" user="WorayutTK" uid="23562084" lat="7.0065707" lon="100.4787450" />\n  <node id="8523332293" visible="true" version="2" changeset="175444256" timestamp="2025-12-03T12:18:40Z" user="WorayutTK" uid="23562084" lat="7.0063805" lon="100.4787458" />\n  <node id="8523333971" visible="true" version="2" changeset="175411611" timestamp="2025-12-02T16:59:47Z" user="Julie Kongsparten" uid="23570745" lat="7.0081934" lon="100.4739460" />\n  <node id="8523333972" visible="true" version="1" changeset="101100007" timestamp="2021-03-16T07:41:08Z" user="oliviar_grab" uid="10664132" lat="7.0086454" lon="100.4738754" />\n  <node id="8523334136" visible="true" version="1" changeset="101100046" timestamp="2021-03-16T07:41:44Z" user="oliviar_grab" uid="10664132" lat="7.0070278" lon="100.4731257" />\n  <node id="8523334137" visible="true" version="2" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0072625" lon="100.4730590" />\n  <node id="8523334139" visible="true" version="2" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0076557" lon="100.4729263" />\n  <node id="8523334169" visible="true" version="2" changeset="176364176" timestamp="2025-12-24T14:14:00Z" user="11aye" uid="22221182" lat="7.0126785" lon="100.4691339" />\n  <node id="8523334172" visible="true" version="2" changeset="176364176" timestamp="2025-12-24T14:14:00Z" user="11aye" uid="22221182" lat="7.0132931" lon="100.4690713" />\n  <node id="8523334173" visible="true" version="2" changeset="176364176" timestamp="2025-12-24T14:14:00Z" user="11aye" uid="22221182" lat="7.0126832" lon="100.4687590" />\n  <node id="8523334176" visible="true" version="2" changeset="176364176" timestamp="2025-12-24T14:14:00Z" user="11aye" uid="22221182" lat="7.0132515" lon="100.4687100" />\n  <node id="8523339090" visible="true" version="1" changeset="101100560" timestamp="2021-03-16T07:48:16Z" user="oliviar_grab" uid="10664132" lat="7.0136736" lon="100.4687655" />\n  <node id="8523343958" visible="true" version="2" changeset="176364176" timestamp="2025-12-24T14:14:00Z" user="11aye" uid="22221182" lat="7.0135982" lon="100.4682451" />\n  <node id="8523344319" visible="true" version="2" changeset="152871448" timestamp="2024-06-19T02:26:33Z" user="GrabTH_Napittha" uid="15509145" lat="7.0060549" lon="100.4712970" />\n  <node id="8523344320" visible="true" version="2" changeset="152871448" timestamp="2024-06-19T02:26:33Z" user="GrabTH_Napittha" uid="15509145" lat="7.0064464" lon="100.4712272" />\n  <node id="8523348885" visible="true" version="1" changeset="101100503" timestamp="2021-03-16T07:47:37Z" user="oliviar_grab" uid="10664132" lat="7.0138749" lon="100.4703681" />\n  <node id="8523348886" visible="true" version="1" changeset="101100503" timestamp="2021-03-16T07:47:37Z" user="oliviar_grab" uid="10664132" lat="7.0146911" lon="100.4702518" />\n  <node id="8523406712" visible="true" version="2" changeset="115207737" timestamp="2021-12-21T12:17:36Z" user="than_mayee" uid="10626093" lat="7.0117565" lon="100.4730056">\n    <tag k="noexit" v="yes" />\n  </node>\n  <node id="8523406713" visible="true" version="2" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275" lat="7.0110379" lon="100.4731558" />\n  <node id="8523423311" visible="true" version="1" changeset="101102777" timestamp="2021-03-16T08:18:04Z" user="oliviar_grab" uid="10664132" lat="7.0102058" lon="100.4803924" />\n  <node id="8523423312" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0102738" lon="100.4798556" />\n  <node id="8523424937" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0095301" lon="100.4765299" />\n  <node id="8523424938" visible="true" version="1" changeset="101102362" timestamp="2021-03-16T08:12:37Z" user="oliviar_grab" uid="10664132" lat="7.0095003" lon="100.4761921" />\n  <node id="8523424940" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0088685" lon="100.4762443" />\n  <node id="8523431061" visible="true" version="1" changeset="101102450" timestamp="2021-03-16T08:13:40Z" user="oliviar_grab" uid="10664132" lat="7.0098637" lon="100.4762202" />\n  <node id="8523431062" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0104273" lon="100.4761999" />\n  <node id="8523431063" visible="true" version="1" changeset="101102450" timestamp="2021-03-16T08:13:40Z" user="oliviar_grab" uid="10664132" lat="7.0098171" lon="100.4758260" />\n  <node id="8523431064" visible="true" version="3" changeset="183219774" timestamp="2026-05-26T17:09:26Z" user="anthaas" uid="5438501" lat="7.0104824" lon="100.4759552" />\n  <node id="8523449429" visible="true" version="1" changeset="101103145" timestamp="2021-03-16T08:22:42Z" user="oliviar_grab" uid="10664132" lat="7.0102783" lon="100.4825060" />\n  <node id="8523449430" visible="true" version="1" changeset="101103145" timestamp="2021-03-16T08:22:42Z" user="oliviar_grab" uid="10664132" lat="7.0105927" lon="100.4824839" />\n  <node id="8523455240" visible="true" version="2" changeset="158286490" timestamp="2024-10-24T08:56:11Z" user="Madi-Red" uid="14550990" lat="7.0128720" lon="100.4746402" />\n  <node id="8523455241" visible="true" version="1" changeset="101103288" timestamp="2021-03-16T08:24:40Z" user="oliviar_grab" uid="10664132" lat="7.0127927" lon="100.4753398" />\n  <node id="8523455242" visible="true" version="1" changeset="101103288" timestamp="2021-03-16T08:24:40Z" user="oliviar_grab" uid="10664132" lat="7.0127754" lon="100.4755651" />\n  <node id="8523455243" visible="true" version="1" changeset="101103288" timestamp="2021-03-16T08:24:40Z" user="oliviar_grab" uid="10664132" lat="7.0127781" lon="100.4757442" />\n  <node id="8523455244" visible="true" version="1" changeset="101103288" timestamp="2021-03-16T08:24:40Z" user="oliviar_grab" uid="10664132" lat="7.0125811" lon="100.4757542" />\n  <node id="8523455245" visible="true" version="1" changeset="101103288" timestamp="2021-03-16T08:24:40Z" user="oliviar_grab" uid="10664132" lat="7.0122796" lon="100.4756590" />\n  <node id="8523455246" visible="true" version="1" changeset="101103288" timestamp="2021-03-16T08:24:40Z" user="oliviar_grab" uid="10664132" lat="7.0122570" lon="100.4756409" />\n  <node id="8523455247" visible="true" version="1" changeset="101103288" timestamp="2021-03-16T08:24:40Z" user="oliviar_grab" uid="10664132" lat="7.0122443" lon="100.4755893" />\n  <node id="8523455248" visible="true" version="2" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0122545" lon="100.4754734" />\n  <node id="8523455250" visible="true" version="2" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0121106" lon="100.4753358" />\n  <node id="8523455251" visible="true" version="2" changeset="148237597" timestamp="2024-03-05T07:55:56Z" user="GrabTH_Panuwit" uid="16483995" lat="7.0114347" lon="100.4742761" />\n  <node id="8523455252" visible="true" version="2" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0115082" lon="100.4741861" />\n  <node id="8523455253" visible="true" version="2" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0120784" lon="100.4735638" />\n  <node id="8523455254" visible="true" version="2" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0121434" lon="100.4735024" />\n  <node id="8523455255" visible="true" version="2" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0124887" lon="100.4735391" />\n  <node id="8523455256" visible="true" version="2" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0126286" lon="100.4735564" />\n  <node id="8523455257" visible="true" version="2" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0126699" lon="100.4735603" />\n  <node id="8523461356" visible="true" version="1" changeset="101103384" timestamp="2021-03-16T08:25:53Z" user="oliviar_grab" uid="10664132" lat="7.0145688" lon="100.4739297" />\n  <node id="8523461357" visible="true" version="1" changeset="101103384" timestamp="2021-03-16T08:25:53Z" user="oliviar_grab" uid="10664132" lat="7.0151543" lon="100.4737240" />\n  <node id="856214432" visible="true" version="4" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0073565" lon="100.4809120" />\n  <node id="856214434" visible="true" version="3" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145" lat="7.0073482" lon="100.4806426" />\n  <node id="856214470" visible="true" version="4" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0077329" lon="100.4733037" />\n  <node id="856214496" visible="true" version="4" changeset="159211955" timestamp="2024-11-16T15:16:41Z" user="nabilersyad" uid="11419275" lat="7.0079310" lon="100.4709336" />\n  <node id="856214503" visible="true" version="4" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0066584" lon="100.4735976" />\n  <node id="856214510" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0092684" lon="100.4811203" />\n  <node id="856214524" visible="true" version="2" changeset="52655906" timestamp="2017-10-05T14:16:28Z" user="Sal73x" uid="1293386" lat="7.0072938" lon="100.4793429" />\n  <node id="856214538" visible="true" version="2" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559" lat="7.0071878" lon="100.4753600" />\n  <node id="856214562" visible="true" version="1" changeset="5468857" timestamp="2010-08-12T06:13:03Z" user="sdayu" uid="300693" lat="7.0072906" lon="100.4775277" />\n  <node id="856214571" visible="true" version="5" changeset="189908183" timestamp="2026-10-03T03:44:58Z" user="dmatted" uid="3047966" lat="7.0092474" lon="100.4812263">\n    <tag k="highway" v="traffic_signals" />\n    <tag k="traffic_signals" v="signal" />\n    <tag k="traffic_signals:direction" v="both" />\n  </node>\n  <node id="856214578" visible="true" version="3" changeset="18505195" timestamp="2013-10-23T15:58:37Z" user="Bernhard Hiller" uid="326704" lat="7.0071506" lon="100.4703267" />\n  <node id="856214580" visible="true" version="3" changeset="159211955" timestamp="2024-11-16T15:16:41Z" user="nabilersyad" uid="11419275" lat="7.0078027" lon="100.4708057" />\n  <node id="8642306621" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="6.9988719" lon="100.4795021" />\n  <node id="8642643253" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0004209" lon="100.4797595" />\n  <node id="8642643269" visible="true" version="4" changeset="186852595" timestamp="2026-08-03T07:05:47Z" user="GrabTH_Monnapat" uid="24164407" lat="6.9993045" lon="100.4795722" />\n  <node id="8642662383" visible="true" version="1" changeset="103175191" timestamp="2021-04-19T07:33:31Z" user="bomma_1" uid="11659195" lat="7.0013699" lon="100.4829960" />\n  <node id="8642662384" visible="true" version="1" changeset="103175191" timestamp="2021-04-19T07:33:31Z" user="bomma_1" uid="11659195" lat="7.0013860" lon="100.4830015" />\n  <node id="8642662385" visible="true" version="1" changeset="103175191" timestamp="2021-04-19T07:33:31Z" user="bomma_1" uid="11659195" lat="7.0014100" lon="100.4830237" />\n  <node id="8642666190" visible="true" version="1" changeset="103176827" timestamp="2021-04-19T07:54:51Z" user="bomma_1" uid="11659195" lat="7.0020963" lon="100.4819776" />\n  <node id="8642666191" visible="true" version="1" changeset="103176827" timestamp="2021-04-19T07:54:51Z" user="bomma_1" uid="11659195" lat="7.0020678" lon="100.4819520" />\n  <node id="8642666192" visible="true" version="1" changeset="103176827" timestamp="2021-04-19T07:54:51Z" user="bomma_1" uid="11659195" lat="7.0020497" lon="100.4819468" />\n  <node id="8642666193" visible="true" version="1" changeset="103176827" timestamp="2021-04-19T07:54:51Z" user="bomma_1" uid="11659195" lat="7.0020832" lon="100.4819615" />\n  <node id="8642666194" visible="true" version="1" changeset="103176827" timestamp="2021-04-19T07:54:51Z" user="bomma_1" uid="11659195" lat="7.0020602" lon="100.4819489" />\n  <node id="8642666195" visible="true" version="1" changeset="103176827" timestamp="2021-04-19T07:54:51Z" user="bomma_1" uid="11659195" lat="7.0020900" lon="100.4819686" />\n  <node id="8642666196" visible="true" version="1" changeset="103176827" timestamp="2021-04-19T07:54:51Z" user="bomma_1" uid="11659195" lat="7.0020748" lon="100.4819562" />\n  <node id="8642666197" visible="true" version="1" changeset="103176827" timestamp="2021-04-19T07:54:51Z" user="bomma_1" uid="11659195" lat="7.0021008" lon="100.4819897" />\n  <node id="8642733849" visible="true" version="1" changeset="103176847" timestamp="2021-04-19T07:55:06Z" user="bomma_1" uid="11659195" lat="7.0023710" lon="100.4820353" />\n  <node id="8642733850" visible="true" version="1" changeset="103176847" timestamp="2021-04-19T07:55:06Z" user="bomma_1" uid="11659195" lat="7.0021584" lon="100.4821199" />\n  <node id="8642858903" visible="true" version="1" changeset="103180514" timestamp="2021-04-19T08:42:09Z" user="bomma_1" uid="11659195" lat="7.0012207" lon="100.4801482" />\n  <node id="8642858904" visible="true" version="1" changeset="103180514" timestamp="2021-04-19T08:42:09Z" user="bomma_1" uid="11659195" lat="7.0012266" lon="100.4801108" />\n  <node id="8642858905" visible="true" version="1" changeset="103180514" timestamp="2021-04-19T08:42:09Z" user="bomma_1" uid="11659195" lat="7.0012362" lon="100.4800468" />\n  <node id="8642858906" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0012439" lon="100.4799997" />\n  <node id="8642866004" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0042286" lon="100.4805255" />\n  <node id="8642866005" visible="true" version="1" changeset="103180368" timestamp="2021-04-19T08:40:06Z" user="bomma_1" uid="11659195" lat="7.0042178" lon="100.4805954" />\n  <node id="8642866006" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0048921" lon="100.4806081" />\n  <node id="8642866007" visible="true" version="1" changeset="103180368" timestamp="2021-04-19T08:40:06Z" user="bomma_1" uid="11659195" lat="7.0048889" lon="100.4807153" />\n  <node id="8642866008" visible="true" version="1" changeset="103180368" timestamp="2021-04-19T08:40:06Z" user="bomma_1" uid="11659195" lat="7.0016342" lon="100.4801778" />\n  <node id="8642866009" visible="true" version="1" changeset="103180368" timestamp="2021-04-19T08:40:06Z" user="bomma_1" uid="11659195" lat="7.0016436" lon="100.4801135" />\n  <node id="8642866010" visible="true" version="1" changeset="103180368" timestamp="2021-04-19T08:40:06Z" user="bomma_1" uid="11659195" lat="7.0025254" lon="100.4803281" />\n  <node id="8642866011" visible="true" version="1" changeset="103180368" timestamp="2021-04-19T08:40:06Z" user="bomma_1" uid="11659195" lat="7.0025334" lon="100.4802516" />\n  <node id="8643114367" visible="true" version="1" changeset="103186836" timestamp="2021-04-19T09:59:02Z" user="mummadi1" uid="11589192" lat="7.0021216" lon="100.4736280" />\n  <node id="8643114368" visible="true" version="1" changeset="103186836" timestamp="2021-04-19T09:59:02Z" user="mummadi1" uid="11589192" lat="7.0021940" lon="100.4736359" />\n  <node id="8643114369" visible="true" version="1" changeset="103186836" timestamp="2021-04-19T09:59:02Z" user="mummadi1" uid="11589192" lat="7.0022259" lon="100.4736319" />\n  <node id="8643114370" visible="true" version="1" changeset="103186836" timestamp="2021-04-19T09:59:02Z" user="mummadi1" uid="11589192" lat="7.0022525" lon="100.4736145" />\n  <node id="8643114371" visible="true" version="1" changeset="103186836" timestamp="2021-04-19T09:59:02Z" user="mummadi1" uid="11589192" lat="7.0022778" lon="100.4736011" />\n  <node id="8643560669" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0002827" lon="100.4797366" />\n  <node id="8643560679" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0000333" lon="100.4796951" />\n  <node id="8932057454" visible="true" version="2" changeset="178433158" timestamp="2026-02-12T03:02:18Z" user="GrabTH_Tanyaporn" uid="22904804" lat="7.0078988" lon="100.4770423" />\n  <node id="8932057455" visible="true" version="1" changeset="108222315" timestamp="2021-07-19T06:15:26Z" user="gunnala_a" uid="13354536" lat="7.0079257" lon="100.4770665" />\n  <node id="8932057456" visible="true" version="1" changeset="108222315" timestamp="2021-07-19T06:15:26Z" user="gunnala_a" uid="13354536" lat="7.0080215" lon="100.4771295" />\n  <node id="8932057457" visible="true" version="1" changeset="108222315" timestamp="2021-07-19T06:15:26Z" user="gunnala_a" uid="13354536" lat="7.0081440" lon="100.4772596" />\n  <node id="8932057458" visible="true" version="1" changeset="108222315" timestamp="2021-07-19T06:15:26Z" user="gunnala_a" uid="13354536" lat="7.0082145" lon="100.4773924" />\n  <node id="8932057459" visible="true" version="1" changeset="108222315" timestamp="2021-07-19T06:15:26Z" user="gunnala_a" uid="13354536" lat="7.0082558" lon="100.4774983" />\n  <node id="8932057460" visible="true" version="1" changeset="108222315" timestamp="2021-07-19T06:15:26Z" user="gunnala_a" uid="13354536" lat="7.0082678" lon="100.4776660" />\n  <node id="8932057461" visible="true" version="1" changeset="108222315" timestamp="2021-07-19T06:15:26Z" user="gunnala_a" uid="13354536" lat="7.0082465" lon="100.4778256" />\n  <node id="8932057462" visible="true" version="1" changeset="108222315" timestamp="2021-07-19T06:15:26Z" user="gunnala_a" uid="13354536" lat="7.0082105" lon="100.4781085" />\n  <node id="8932057463" visible="true" version="1" changeset="108222315" timestamp="2021-07-19T06:15:26Z" user="gunnala_a" uid="13354536" lat="7.0081573" lon="100.4783821" />\n  <node id="8932057464" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0081450" lon="100.4784990" />\n  <node id="8934566430" visible="true" version="1" changeset="108283189" timestamp="2021-07-20T04:11:19Z" user="p_raj" uid="11589169" lat="7.0011367" lon="100.4766492" />\n  <node id="8934566431" visible="true" version="1" changeset="108283189" timestamp="2021-07-20T04:11:19Z" user="p_raj" uid="11589169" lat="7.0011466" lon="100.4766346" />\n  <node id="8934566432" visible="true" version="1" changeset="108283189" timestamp="2021-07-20T04:11:19Z" user="p_raj" uid="11589169" lat="7.0011611" lon="100.4766217" />\n  <node id="8934566433" visible="true" version="1" changeset="108283189" timestamp="2021-07-20T04:11:19Z" user="p_raj" uid="11589169" lat="7.0012515" lon="100.4766054" />\n  <node id="8934566434" visible="true" version="1" changeset="108283189" timestamp="2021-07-20T04:11:19Z" user="p_raj" uid="11589169" lat="7.0012613" lon="100.4765926" />\n  <node id="8934566435" visible="true" version="1" changeset="108283189" timestamp="2021-07-20T04:11:19Z" user="p_raj" uid="11589169" lat="7.0012358" lon="100.4766136" />\n  <node id="8974100552" visible="true" version="1" changeset="109022523" timestamp="2021-08-02T10:26:09Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0072605" lon="100.4767991" />\n  <node id="8974100553" visible="true" version="1" changeset="109022523" timestamp="2021-08-02T10:26:09Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0072744" lon="100.4769018" />\n  <node id="8974100554" visible="true" version="1" changeset="109022523" timestamp="2021-08-02T10:26:09Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0051717" lon="100.4808055" />\n  <node id="8974100558" visible="true" version="1" changeset="109022523" timestamp="2021-08-02T10:26:09Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0068174" lon="100.4810595" />\n  <node id="8974100559" visible="true" version="1" changeset="109022523" timestamp="2021-08-02T10:26:09Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0070398" lon="100.4810950" />\n  <node id="8976428973" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0053171" lon="100.4835667" />\n  <node id="8976428974" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0052749" lon="100.4827326" />\n  <node id="8976428975" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0052176" lon="100.4818195" />\n  <node id="8976428976" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0051578" lon="100.4810196" />\n  <node id="8976428977" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0050864" lon="100.4801876" />\n  <node id="8976428978" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0049945" lon="100.4794470" />\n  <node id="8976428979" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0048862" lon="100.4782476" />\n  <node id="8976428980" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0047656" lon="100.4770053" />\n  <node id="8976428981" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0046809" lon="100.4758439" />\n  <node id="8976428982" visible="true" version="2" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0046465" lon="100.4754399" />\n  <node id="8976428984" visible="true" version="3" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0046139" lon="100.4753884" />\n  <node id="8976428985" visible="true" version="3" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083" lat="7.0045239" lon="100.4753458" />\n  <node id="8976428986" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0047965" lon="100.4767568" />\n  <node id="8976428987" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0051282" lon="100.4806475">\n    <tag k="highway" v="traffic_signals" />\n    <tag k="traffic_signals" v="signal" />\n  </node>\n  <node id="8976428988" visible="true" version="2" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949" lat="7.0051201" lon="100.4805404">\n    <tag k="highway" v="traffic_signals" />\n    <tag k="traffic_signals" v="signal" />\n  </node>\n  <node id="8976428989" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0051521" lon="100.4805184" />\n  <node id="8976428990" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0051369" lon="100.4807777" />\n  <node id="8976428991" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0051286" lon="100.4806593" />\n  <node id="8976428992" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0052642" lon="100.4819804" />\n  <node id="8976428993" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0053122" lon="100.4834698" />\n  <node id="8976428994" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0053447" lon="100.4839871" />\n  <node id="8976428996" visible="true" version="2" changeset="181691651" timestamp="2026-04-22T21:16:53Z" user="RAytoun" uid="1414812" lat="7.0053467" lon="100.4840324" />\n  <node id="8976429006" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0053806" lon="100.4834639" />\n  <node id="8976429007" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0047853" lon="100.4772087" />\n  <node id="8976429008" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0051720" lon="100.4808105" />\n  <node id="8976429009" visible="true" version="1" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0050662" lon="100.4797255" />\n  <node id="9085920065" visible="true" version="1" changeset="111130124" timestamp="2021-09-13T08:17:27Z" user="GrabTH_Kulyada" uid="13325735" lat="7.0051504" lon="100.4805027" />\n  <node id="916327670" visible="true" version="5" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0141365" lon="100.4713793" />\n  <node id="916327674" visible="true" version="2" changeset="29150100" timestamp="2015-02-28T07:32:33Z" user="Pathumthani" uid="2302962" lat="7.0140669" lon="100.4715469" />\n  <node id="916327684" visible="true" version="5" changeset="29150113" timestamp="2015-02-28T07:33:25Z" user="Pathumthani" uid="2302962" lat="7.0137814" lon="100.4713611" />\n  <node id="916327686" visible="true" version="5" changeset="29150113" timestamp="2015-02-28T07:33:25Z" user="Pathumthani" uid="2302962" lat="7.0137451" lon="100.4713034" />\n  <node id="916327693" visible="true" version="5" changeset="29150116" timestamp="2015-02-28T07:33:36Z" user="Pathumthani" uid="2302962" lat="7.0142080" lon="100.4712143" />\n  <node id="916327702" visible="true" version="4" changeset="176365051" timestamp="2025-12-24T14:36:05Z" user="11aye" uid="22221182" lat="7.0121213" lon="100.4704568" />\n  <node id="916327710" visible="true" version="4" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0141896" lon="100.4710998" />\n  <node id="916327728" visible="true" version="4" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0141213" lon="100.4710108" />\n  <node id="916327730" visible="true" version="3" changeset="18489522" timestamp="2013-10-22T17:37:10Z" user="Bernhard Hiller" uid="326704" lat="7.0107398" lon="100.4700070" />\n  <node id="916327738" visible="true" version="5" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0142025" lon="100.4711353" />\n  <node id="916327746" visible="true" version="1" changeset="5808908" timestamp="2010-09-18T06:05:23Z" user="sdayu" uid="300693" lat="7.0132644" lon="100.4710549" />\n  <node id="916327748" visible="true" version="5" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0140984" lon="100.4714095" />\n  <node id="916327753" visible="true" version="4" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0141485" lon="100.4710368" />\n  <node id="916327773" visible="true" version="4" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0140567" lon="100.4709729" />\n  <node id="916327777" visible="true" version="4" changeset="18505195" timestamp="2013-10-23T15:58:47Z" user="Bernhard Hiller" uid="326704" lat="7.0108552" lon="100.4676131" />\n  <node id="916327792" visible="true" version="3" changeset="14172634" timestamp="2012-12-06T09:47:10Z" user="Tungmar" uid="427807" lat="7.0113312" lon="100.4675226" />\n  <node id="916327796" visible="true" version="5" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0141680" lon="100.4713421" />\n  <node id="916327799" visible="true" version="4" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0139837" lon="100.4709564" />\n  <node id="916327801" visible="true" version="4" changeset="29150113" timestamp="2015-02-28T07:33:25Z" user="Pathumthani" uid="2302962" lat="7.0137316" lon="100.4711386" />\n  <node id="916327804" visible="true" version="4" changeset="29150113" timestamp="2015-02-28T07:33:25Z" user="Pathumthani" uid="2302962" lat="7.0137457" lon="100.4710990" />\n  <node id="916327808" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915" lat="7.0163196" lon="100.4762197" />\n  <node id="916327810" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:37:10Z" user="Bernhard Hiller" uid="326704" lat="7.0112770" lon="100.4701876" />\n  <node id="916327824" visible="true" version="4" changeset="176365051" timestamp="2025-12-24T14:36:05Z" user="11aye" uid="22221182" lat="7.0120816" lon="100.4674841" />\n  <node id="916327826" visible="true" version="4" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0137929" lon="100.4710299" />\n  <node id="916327839" visible="true" version="5" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0139462" lon="100.4709568" />\n  <node id="916327846" visible="true" version="5" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0140904" lon="100.4709894" />\n  <node id="916327857" visible="true" version="5" changeset="29150113" timestamp="2015-02-28T07:33:25Z" user="Pathumthani" uid="2302962" lat="7.0137242" lon="100.4712222" />\n  <node id="916327868" visible="true" version="4" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0140208" lon="100.4709618" />\n  <node id="916327875" visible="true" version="4" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0141715" lon="100.4710667" />\n  <node id="916327885" visible="true" version="5" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0141916" lon="100.4712994" />\n  <node id="916327888" visible="true" version="5" changeset="29150113" timestamp="2015-02-28T07:33:25Z" user="Pathumthani" uid="2302962" lat="7.0139598" lon="100.4714477" />\n  <node id="916327891" visible="true" version="4" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0137664" lon="100.4710624" />\n  <node id="916327893" visible="true" version="2" changeset="29150083" timestamp="2015-02-28T07:31:42Z" user="Pathumthani" uid="2302962" lat="7.0136463" lon="100.4711593" />\n  <node id="916327906" visible="true" version="5" changeset="29150113" timestamp="2015-02-28T07:33:25Z" user="Pathumthani" uid="2302962" lat="7.0137243" lon="100.4711801" />\n  <node id="916327908" visible="true" version="5" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0140082" lon="100.4714444" />\n  <node id="916327912" visible="true" version="3" changeset="49318745" timestamp="2017-06-06T20:56:23Z" user="RVR004" uid="5314349" lat="7.0119120" lon="100.4705560" />\n  <node id="916327921" visible="true" version="6" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0138246" lon="100.4710024" />\n  <node id="916327930" visible="true" version="1" changeset="5808908" timestamp="2010-09-18T06:05:31Z" user="sdayu" uid="300693" lat="7.0175349" lon="100.4785890" />\n  <node id="916327943" visible="true" version="5" changeset="29150113" timestamp="2015-02-28T07:33:25Z" user="Pathumthani" uid="2302962" lat="7.0137312" lon="100.4712637" />\n  <node id="916327945" visible="true" version="5" changeset="176393370" timestamp="2025-12-25T12:50:25Z" user="11aye" uid="22221182" lat="7.0168049" lon="100.4770885">\n    <tag k="highway" v="traffic_signals" />\n    <tag k="traffic_signals" v="crossing" />\n  </node>\n  <node id="916327949" visible="true" version="5" changeset="29150113" timestamp="2015-02-28T07:33:26Z" user="Pathumthani" uid="2302962" lat="7.0140550" lon="100.4714315" />\n  <node id="916327950" visible="true" version="5" changeset="29150113" timestamp="2015-02-28T07:33:25Z" user="Pathumthani" uid="2302962" lat="7.0138930" lon="100.4714361" />\n  <node id="916327952" visible="true" version="5" changeset="29150113" timestamp="2015-02-28T07:33:25Z" user="Pathumthani" uid="2302962" lat="7.0138320" lon="100.4714065" />\n  <node id="916327962" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:48Z" user="Bernhard Hiller" uid="326704" lat="7.0107100" lon="100.4701457">\n    <tag k="highway" v="traffic_signals" />\n  </node>\n  <node id="9179528432" visible="true" version="1" changeset="112639467" timestamp="2021-10-18T07:37:12Z" user="Kwanhathai03" uid="12762907" lat="7.0165507" lon="100.4836449" />\n  <node id="947896945" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0103979" lon="100.4763344" />\n  <node id="947896950" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0106338" lon="100.4753495" />\n  <node id="947896953" visible="true" version="3" changeset="176054561" timestamp="2025-12-17T14:53:03Z" user="Wim Keijsers" uid="22692008" lat="7.0136503" lon="100.4828787" />\n  <node id="947896956" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:48Z" user="Bernhard Hiller" uid="326704" lat="7.0108468" lon="100.4715861" />\n  <node id="947896958" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:37:12Z" user="Bernhard Hiller" uid="326704" lat="7.0094407" lon="100.4815012" />\n  <node id="947896961" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0096773" lon="100.4765170" />\n  <node id="947896964" visible="true" version="4" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915" lat="7.0103707" lon="100.4764769">\n    <tag k="highway" v="traffic_signals" />\n  </node>\n  <node id="947896966" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:48Z" user="Bernhard Hiller" uid="326704" lat="7.0106485" lon="100.4709038" />\n  <node id="947896971" visible="true" version="5" changeset="189908183" timestamp="2026-10-03T03:44:58Z" user="dmatted" uid="3047966" lat="7.0088968" lon="100.4765956">\n    <tag k="highway" v="traffic_signals" />\n    <tag k="traffic_signals" v="signal" />\n    <tag k="traffic_signals:direction" v="both" />\n  </node>\n  <node id="947896972" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:37:12Z" user="Bernhard Hiller" uid="326704" lat="7.0125629" lon="100.4820136" />\n  <node id="947896979" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:37:13Z" user="Bernhard Hiller" uid="326704" lat="7.0116953" lon="100.4818636" />\n  <node id="947896981" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:37:13Z" user="Bernhard Hiller" uid="326704" lat="7.0137749" lon="100.4822200" />\n  <node id="947896985" visible="true" version="3" changeset="176054561" timestamp="2025-12-17T14:53:03Z" user="Wim Keijsers" uid="22692008" lat="7.0136152" lon="100.4833871" />\n  <node id="947896989" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:37:13Z" user="Bernhard Hiller" uid="326704" lat="7.0118832" lon="100.4818992" />\n  <node id="947896995" visible="true" version="4" changeset="176057240" timestamp="2025-12-17T15:59:32Z" user="Wim Keijsers" uid="22692008" lat="7.0137047" lon="100.4840514" />\n  <node id="947896997" visible="true" version="3" changeset="18505195" timestamp="2013-10-23T15:58:49Z" user="Bernhard Hiller" uid="326704" lat="7.0105253" lon="100.4703203" />\n  <node id="985092239" visible="true" version="4" changeset="28538968" timestamp="2015-02-01T07:50:35Z" user="Pathumthani" uid="2302962" lat="7.0105593" lon="100.4699593" />\n  <node id="985092392" visible="true" version="2" changeset="18489522" timestamp="2013-10-22T17:37:13Z" user="Bernhard Hiller" uid="326704" lat="7.0076624" lon="100.4689519" />\n  <node id="985092568" visible="true" version="1" changeset="6342391" timestamp="2010-11-11T10:17:18Z" user="Beddhist" uid="140550" lat="7.0136631" lon="100.4710363" />\n  <node id="985092634" visible="true" version="4" changeset="176052935" timestamp="2025-12-17T14:14:54Z" user="Wim Keijsers" uid="22692008" lat="7.0184683" lon="100.4823974" />\n  <node id="985092700" visible="true" version="2" changeset="18505195" timestamp="2013-10-23T15:58:49Z" user="Bernhard Hiller" uid="326704" lat="7.0105275" lon="100.4700939" />\n  <way id="100401250" visible="true" version="11" changeset="184504425" timestamp="2026-06-24T03:58:02Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1160612974" />\n    <nd ref="8976428985" />\n    <nd ref="1160612987" />\n    <nd ref="1160612969" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนเทพสงเคราะห์" />\n    <tag k="name:en" v="Thep Sonkhro Road" />\n    <tag k="name:th" v="ถนนเทพสงเคราะห์" />\n    <tag k="name:th-Latn" v="Thanon Tephsongkrew" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="100401320" visible="true" version="3" changeset="177547487" timestamp="2026-01-22T03:38:52Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1110416469" />\n    <nd ref="1123399492" />\n    <tag k="highway" v="tertiary" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="101706795" visible="true" version="5" changeset="187967768" timestamp="2026-08-25T03:17:18Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1196113662" />\n    <nd ref="1966412828" />\n    <nd ref="1174378788" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนปรีดารมย์" />\n    <tag k="name:en" v="Preedarom Road" />\n    <tag k="name:th" v="ถนนปรีดารมย์" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="101939921" visible="true" version="10" changeset="187927516" timestamp="2026-08-24T09:21:58Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1160612969" />\n    <nd ref="1431715406" />\n    <tag k="highway" v="tertiary" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนชีอุทิศ" />\n    <tag k="name:en" v="She U Tit Road" />\n    <tag k="name:th" v="ถนนชีอุทิศ" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1168168079" visible="true" version="1" changeset="135638961" timestamp="2023-05-03T02:45:09Z" user="GrabTH_Napittha" uid="15509145">\n    <nd ref="4900465363" />\n    <nd ref="4900465364" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="name" v="Soi 1/1 Sophasarnrungsen" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="1204844051" visible="true" version="2" changeset="177551471" timestamp="2026-01-22T07:08:36Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="4900464584" />\n    <nd ref="1115585385" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนประชาธิปัตย์" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1204844052" visible="true" version="2" changeset="177551471" timestamp="2026-01-22T07:08:36Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="8974100552" />\n    <nd ref="4900464584" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนประชาธิปัตย์" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1218225518" visible="true" version="3" changeset="149747000" timestamp="2024-04-08T18:10:15Z" user="GrabTH_Napittha" uid="15509145">\n    <nd ref="2406954578" />\n    <nd ref="2505176175" />\n    <nd ref="2505176169" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="2" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนราษฎร์ยินดี" />\n    <tag k="name:en" v="Rajyindee road" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1218225520" visible="true" version="1" changeset="143104733" timestamp="2023-10-25T09:27:31Z" user="GrabTH_Praiya" uid="18100118">\n    <nd ref="2406954580" />\n    <nd ref="2406954578" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="2" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนราษฎร์ยินดี" />\n    <tag k="name:en" v="Rajyindee road" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1218225521" visible="true" version="2" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145">\n    <nd ref="11292971032" />\n    <nd ref="2406954580" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="maxspeed" v="30" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="1218225522" visible="true" version="3" changeset="149747000" timestamp="2024-04-08T18:10:15Z" user="GrabTH_Napittha" uid="15509145">\n    <nd ref="849403384" />\n    <nd ref="8523314484" />\n    <nd ref="796747634" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="2" />\n    <tag k="name" v="ถนนราษฎร์ยินดี" />\n    <tag k="name:en" v="Thanon Rajyindee" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n  </way>\n  <way id="1218225523" visible="true" version="1" changeset="143104733" timestamp="2023-10-25T09:27:31Z" user="GrabTH_Praiya" uid="18100118">\n    <nd ref="2406954578" />\n    <nd ref="2406954579" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="1218225524" visible="true" version="2" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145">\n    <nd ref="2406954579" />\n    <nd ref="11292971032" />\n    <nd ref="856214571" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="2" />\n    <tag k="name" v="ถนนราษฎร์ยินดี" />\n    <tag k="name:en" v="Thanon Rajyindee" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n  </way>\n  <way id="1218747872" visible="true" version="3" changeset="188397820" timestamp="2026-09-02T03:18:10Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="11292971033" />\n    <nd ref="2505176175" />\n    <tag k="highway" v="secondary_link" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1218747873" visible="true" version="3" changeset="188397820" timestamp="2026-09-02T03:18:10Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1639088658" />\n    <nd ref="11292971034" />\n    <tag k="highway" v="secondary_link" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1218747874" visible="true" version="1" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145">\n    <nd ref="2406954580" />\n    <nd ref="11292971035" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="maxspeed" v="30" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="1218747875" visible="true" version="1" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145">\n    <nd ref="11292971035" />\n    <nd ref="11292971036" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="layer" v="1" />\n    <tag k="maxspeed" v="30" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="1218747876" visible="true" version="1" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145">\n    <nd ref="11292971037" />\n    <nd ref="2406954578" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="1218747877" visible="true" version="2" changeset="183219774" timestamp="2026-05-26T17:09:28Z" user="anthaas" uid="5438501">\n    <nd ref="11292971038" />\n    <nd ref="11292971037" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="layer" v="1" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="1218747878" visible="true" version="2" changeset="183219774" timestamp="2026-05-26T17:09:28Z" user="anthaas" uid="5438501">\n    <nd ref="11292971039" />\n    <nd ref="11292971040" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="layer" v="1" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n  </way>\n  <way id="1218747879" visible="true" version="1" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145">\n    <nd ref="11292971040" />\n    <nd ref="2505176265" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n  </way>\n  <way id="122540020" visible="true" version="8" changeset="188756440" timestamp="2026-09-09T06:55:23Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="1369020950" />\n    <nd ref="1369020925" />\n    <nd ref="14165547215" />\n    <nd ref="8523423312" />\n    <nd ref="1369020943" />\n    <nd ref="2506088023" />\n    <nd ref="2506088274" />\n    <nd ref="2506088528" />\n    <nd ref="1369020942" />\n    <nd ref="1369020939" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ถนนเฉลียวอุทิศ" />\n    <tag k="name:en" v="Chaleaw U-Thit Road" />\n    <tag k="name:th" v="ถนนเฉลียวอุทิศ" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1227721990" visible="true" version="3" changeset="177551471" timestamp="2026-01-22T07:08:36Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="2505176169" />\n    <nd ref="856214432" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนประชาธิปัตย์" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1227721991" visible="true" version="2" changeset="186687774" timestamp="2026-07-31T06:29:47Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="856214432" />\n    <nd ref="14063217001" />\n    <nd ref="796747251" />\n    <nd ref="796747479" />\n    <nd ref="2406954579" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="2" />\n    <tag k="name" v="ถนนราษฎร์ยินดี" />\n    <tag k="name:en" v="Thanon Rajyindee" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n  </way>\n  <way id="1257722736" visible="true" version="7" changeset="188457192" timestamp="2026-09-03T08:00:30Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506088243" />\n    <nd ref="3376088758" />\n    <nd ref="12351400022" />\n    <nd ref="1123399547" />\n    <nd ref="11693248213" />\n    <nd ref="1123399836" />\n    <nd ref="12351400021" />\n    <nd ref="2506088411" />\n    <nd ref="12351400020" />\n    <nd ref="12351400019" />\n    <nd ref="1123399484" />\n    <nd ref="1123399578" />\n    <nd ref="3376078962" />\n    <nd ref="8523455250" />\n    <nd ref="3376078961" />\n    <nd ref="13472440795" />\n    <nd ref="1123399932" />\n    <nd ref="13472440796" />\n    <nd ref="1123399603" />\n    <nd ref="13472440794" />\n    <nd ref="13472440797" />\n    <nd ref="13472440798" />\n    <nd ref="12351400010" />\n    <nd ref="3376090057" />\n    <nd ref="1123399480" />\n    <nd ref="13472452501" />\n    <nd ref="12351400004" />\n    <nd ref="13472452502" />\n    <nd ref="1123399632" />\n    <nd ref="12351400003" />\n    <nd ref="13472440799" />\n    <nd ref="12351400002" />\n    <nd ref="13472440800" />\n    <nd ref="1123399512" />\n    <nd ref="12351400001" />\n    <nd ref="4900465935" />\n    <nd ref="13472440793" />\n    <nd ref="1123399715" />\n    <tag k="highway" v="residential" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1257722737" visible="true" version="3" changeset="188455914" timestamp="2026-09-03T07:37:43Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506088243" />\n    <nd ref="3376091310" />\n    <tag k="highway" v="residential" />\n    <tag k="maxspeed" v="30" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1257722738" visible="true" version="4" changeset="188455914" timestamp="2026-09-03T07:37:43Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506088243" />\n    <nd ref="1123399550" />\n    <nd ref="1123399768" />\n    <nd ref="4900465324" />\n    <nd ref="14127561644" />\n    <nd ref="1123399612" />\n    <nd ref="1123399962" />\n    <nd ref="1123399585" />\n    <nd ref="1123399687" />\n    <nd ref="1123399502" />\n    <tag k="highway" v="residential" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1257722739" visible="true" version="3" changeset="188031708" timestamp="2026-08-26T09:16:03Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2507400792" />\n    <nd ref="1123399492" />\n    <nd ref="1123399863" />\n    <nd ref="1123399580" />\n    <nd ref="1123399938" />\n    <nd ref="1123399605" />\n    <nd ref="1123399591" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1270957860" visible="true" version="2" changeset="189143922" timestamp="2026-09-17T07:11:29Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="796747634" />\n    <nd ref="14188637663" />\n    <nd ref="856214432" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="2" />\n    <tag k="name" v="ถนนราษฎร์ยินดี" />\n    <tag k="name:en" v="Thanon Rajyindee" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n  </way>\n  <way id="1270957861" visible="true" version="1" changeset="149747000" timestamp="2024-04-08T18:10:15Z" user="GrabTH_Napittha" uid="15509145">\n    <nd ref="2507400639" />\n    <nd ref="809537546" />\n    <nd ref="4900465941" />\n    <nd ref="849403500" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="2" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนราษฎร์ยินดี" />\n    <tag k="name:en" v="Rajyindee road" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1270957862" visible="true" version="1" changeset="149747000" timestamp="2024-04-08T18:10:15Z" user="GrabTH_Napittha" uid="15509145">\n    <nd ref="2505176169" />\n    <nd ref="2507400639" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="2" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนราษฎร์ยินดี" />\n    <tag k="name:en" v="Rajyindee road" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1270957863" visible="true" version="2" changeset="177551471" timestamp="2026-01-22T07:08:36Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="856214432" />\n    <nd ref="856214434" />\n    <nd ref="856214524" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนประชาธิปัตย์" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="129827552" visible="true" version="4" changeset="175394828" timestamp="2025-12-02T10:05:36Z" user="JethroChen" uid="23301387">\n    <nd ref="1431715395" />\n    <nd ref="13347563565" />\n    <nd ref="1680666546" />\n    <nd ref="1122072674" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="129827554" visible="true" version="11" changeset="189575613" timestamp="2026-09-26T08:15:30Z" user="Tonushka" uid="3419975">\n    <nd ref="1110416726" />\n    <nd ref="1110416439" />\n    <nd ref="1110416537" />\n    <nd ref="1110416663" />\n    <nd ref="1110416318" />\n    <nd ref="1110416484" />\n    <nd ref="1680666468" />\n    <nd ref="1110416593" />\n    <nd ref="1559016906" />\n    <nd ref="1110416716" />\n    <nd ref="1559016908" />\n    <nd ref="1110416356" />\n    <nd ref="14219025415" />\n    <nd ref="1110416655" />\n    <nd ref="1110416312" />\n    <nd ref="1559016920" />\n    <nd ref="1559016921" />\n    <nd ref="1110416556" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนหอยมุกต์" />\n    <tag k="name:en" v="Hoimook Road" />\n    <tag k="name:th" v="ถนนหอยมุกต์" />\n    <tag k="oneway" v="no" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="129827558" visible="true" version="4" changeset="185384563" timestamp="2026-07-09T04:44:44Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1122072673" />\n    <nd ref="1431715359" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนเสน่หานุสรณ์" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="129827561" visible="true" version="9" changeset="189148039" timestamp="2026-09-17T08:49:06Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="1431715415" />\n    <nd ref="14188764645" />\n    <nd ref="1431715406" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนหอมนวล" />\n    <tag k="name:en" v="Homhaun Road" />\n    <tag k="name:th" v="ถนนหอมนวล" />\n    <tag k="name:th-Latn" v="Thamon Homhaun" />\n    <tag k="oneway" v="no" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="129827562" visible="true" version="4" changeset="187969315" timestamp="2026-08-25T04:56:37Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1431715380" />\n    <nd ref="1431715423" />\n    <nd ref="13433688987" />\n    <nd ref="1431715421" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนอำนวยวิทย์" />\n    <tag k="name:en" v="Amnuaywit Road" />\n    <tag k="name:th" v="ถนนอำนวยวิทย์" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="129842388" visible="true" version="10" changeset="187971023" timestamp="2026-08-25T06:09:20Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="8642866008" />\n    <nd ref="1431867108" />\n    <nd ref="1680666463" />\n    <nd ref="1431867166" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนราษฎร์ยินดี ซอย 9" />\n    <tag k="name:en" v="Rajyindee Road Soi 9" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี ซอย 9" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="129842391" visible="true" version="5" changeset="187971305" timestamp="2026-08-25T06:19:02Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="13362936997" />\n    <nd ref="1431867112" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ถนนพัฒโนอุทิศ" />\n    <tag k="name:en" v="Pattano Uthit Road" />\n    <tag k="name:th" v="ถนนพัฒโนอุทิศ" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="129842397" visible="true" version="5" changeset="175694769" timestamp="2025-12-09T03:57:22Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="1431867141" />\n    <nd ref="1431867103" />\n    <nd ref="8642733850" />\n    <nd ref="8642666197" />\n    <nd ref="8642666190" />\n    <nd ref="8642666195" />\n    <nd ref="8642666193" />\n    <nd ref="8642666196" />\n    <nd ref="8642666191" />\n    <nd ref="8642666194" />\n    <nd ref="8642666192" />\n    <nd ref="1431867121" />\n    <nd ref="1431867150" />\n    <nd ref="1431867100" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Thanon Klongren 2 Soi 7" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1315837497" visible="true" version="1" changeset="156669871" timestamp="2024-09-16T03:25:34Z" user="djmorkman" uid="5547549">\n    <nd ref="1612097195" />\n    <nd ref="620881568" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n  </way>\n  <way id="1316153439" visible="true" version="1" changeset="156729462" timestamp="2024-09-17T13:07:41Z" user="floringoia_grab" uid="13905585">\n    <nd ref="2506087563" />\n    <nd ref="6634655051" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="1327815071" visible="true" version="1" changeset="158286490" timestamp="2024-10-24T08:56:11Z" user="Madi-Red" uid="14550990">\n    <nd ref="12284916697" />\n    <nd ref="8523455240" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1327815072" visible="true" version="1" changeset="158286490" timestamp="2024-10-24T08:56:11Z" user="Madi-Red" uid="14550990">\n    <nd ref="12284916698" />\n    <nd ref="12284916697" />\n    <tag k="covered" v="yes" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1333722464" visible="true" version="1" changeset="159129691" timestamp="2024-11-14T13:08:41Z" user="nabilersyad" uid="11419275">\n    <nd ref="947896945" />\n    <nd ref="12339345630" />\n    <tag k="crossing" v="marked" />\n    <tag k="crossing:markings" v="yes" />\n    <tag k="footway" v="crossing" />\n    <tag k="highway" v="footway" />\n  </way>\n  <way id="1333722465" visible="true" version="4" changeset="177548426" timestamp="2026-01-22T04:37:46Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="12339345631" />\n    <nd ref="12342535364" />\n    <nd ref="12339345626" />\n    <nd ref="12346099049" />\n    <nd ref="12339345629" />\n    <nd ref="12339345627" />\n    <nd ref="12342535365" />\n    <nd ref="12342535359" />\n    <nd ref="12342535366" />\n    <tag k="highway" v="service" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="paving_stones" />\n  </way>\n  <way id="1334274432" visible="true" version="2" changeset="186899418" timestamp="2026-08-04T02:48:00Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="12344163778" />\n    <nd ref="12344163777" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="1334274433" visible="true" version="1" changeset="159178805" timestamp="2024-11-15T16:39:46Z" user="nabilersyad" uid="11419275">\n    <nd ref="12344163776" />\n    <nd ref="12344163775" />\n    <tag k="highway" v="path" />\n  </way>\n  <way id="1334385282" visible="true" version="3" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915">\n    <nd ref="12345104265" />\n    <nd ref="12345104268" />\n    <nd ref="12346099060" />\n    <nd ref="12339345630" />\n    <tag k="footway" v="sidewalk" />\n    <tag k="highway" v="footway" />\n  </way>\n  <way id="1334522325" visible="true" version="1" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275">\n    <nd ref="12346117426" />\n    <nd ref="12346117425" />\n    <nd ref="12346117424" />\n    <tag k="highway" v="service" />\n    <tag k="service" v="driveway" />\n  </way>\n  <way id="1335076429" visible="true" version="2" changeset="188035073" timestamp="2026-08-26T10:34:18Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506088019" />\n    <nd ref="12351390199" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1365892748" visible="true" version="2" changeset="189039564" timestamp="2026-09-15T03:36:04Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1110416444" />\n    <nd ref="14182062679" />\n    <nd ref="2506087432" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Thanon Pracharak" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1365892989" visible="true" version="3" changeset="186899564" timestamp="2026-08-04T02:55:22Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1431715392" />\n    <nd ref="1431715437" />\n    <nd ref="1431715366" />\n    <tag k="highway" v="residential" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1365892991" visible="true" version="1" changeset="163428132" timestamp="2025-03-10T06:52:20Z" user="GrabTH_Panuwit" uid="16483995">\n    <nd ref="1110416568" />\n    <nd ref="1110416410" />\n    <tag k="highway" v="residential" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="1365935945" visible="true" version="1" changeset="163432352" timestamp="2025-03-10T09:06:18Z" user="GrabTH_Praiya" uid="18100118">\n    <nd ref="856214538" />\n    <nd ref="4900465166" />\n    <nd ref="12649445299" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="1374050350" visible="true" version="2" changeset="186900657" timestamp="2026-08-04T03:36:12Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="12722329099" />\n    <nd ref="12722387502" />\n    <nd ref="12722329100" />\n    <nd ref="12722387504" />\n    <nd ref="12722387501" />\n    <tag k="highway" v="service" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="1374050351" visible="true" version="2" changeset="186900771" timestamp="2026-08-04T03:39:24Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="12722387502" />\n    <nd ref="12722387503" />\n    <nd ref="12722387504" />\n    <tag k="highway" v="service" />\n    <tag k="service" v="parking_aisle" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="1374050352" visible="true" version="3" changeset="187927395" timestamp="2026-08-24T09:18:39Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="12722387506" />\n    <nd ref="12722387505" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="layer" v="1" />\n    <tag k="name" v="ถนนกิมประดิษฐ์" />\n    <tag k="name:en" v="Kimpradit Road" />\n    <tag k="name:th" v="ถนนกิมประดิษฐ์" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1374050353" visible="true" version="3" changeset="187927395" timestamp="2026-08-24T09:18:39Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1110416556" />\n    <nd ref="12722387506" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนกิมประดิษฐ์" />\n    <tag k="name:en" v="Kimpradit Road" />\n    <tag k="name:th" v="ถนนกิมประดิษฐ์" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="142395239" visible="true" version="3" changeset="177026634" timestamp="2026-01-09T13:39:37Z" user="cycy145" uid="23690693">\n    <nd ref="1431867150" />\n    <nd ref="1558314095" />\n    <nd ref="1558314094" />\n    <nd ref="1558314092" />\n    <nd ref="4900465425" />\n    <nd ref="1558314093" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="142453602" visible="true" version="7" changeset="187972355" timestamp="2026-08-25T06:40:53Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1115585385" />\n    <nd ref="1115585909" />\n    <nd ref="13884411592" />\n    <nd ref="1115585463" />\n    <nd ref="13351219789" />\n    <nd ref="1115585416" />\n    <tag k="highway" v="tertiary" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="142455590" visible="true" version="4" changeset="187972573" timestamp="2026-08-25T06:47:20Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1559016909" />\n    <nd ref="1559016910" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ถนนฉัยยากุลอุทิศ ซอย 2" />\n    <tag k="name:en" v="Chaiyakuluthit Road Soi 2" />\n    <tag k="name:th" v="ถนนฉัยยากุลอุทิศ ซอย 2" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="142455592" visible="true" version="5" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1431867166" />\n    <nd ref="14188679740" />\n    <nd ref="1559016911" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Rajyindee 9 Road" />\n    <tag k="name:th" v="ราษฎร์ยินดี 9" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="142455595" visible="true" version="4" changeset="187972846" timestamp="2026-08-25T06:56:04Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1559016904" />\n    <nd ref="1559016905" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนฉัยยากุลอุทิศ ซอย 3" />\n    <tag k="name:en" v="Chaiyakuluthit Road Soi 3" />\n    <tag k="name:th" v="ถนนฉัยยากุลอุทิศ ซอย 3" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="142455599" visible="true" version="4" changeset="187973629" timestamp="2026-08-25T07:21:01Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1559016916" />\n    <nd ref="1559016918" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนฉัยยากุลอุทิศ ซอย 1" />\n    <tag k="name:en" v="Chaiyakuluthit Road Soi 1" />\n    <tag k="name:th" v="ถนนฉัยยากุลอุทิศ ซอย 1" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="142455600" visible="true" version="5" changeset="187974419" timestamp="2026-08-25T07:42:15Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1559016918" />\n    <nd ref="1559016910" />\n    <nd ref="4900465210" />\n    <nd ref="1559016905" />\n    <nd ref="1559016903" />\n    <nd ref="1680666232" />\n    <nd ref="1559016895" />\n    <tag k="highway" v="tertiary" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="142455604" visible="true" version="3" changeset="187974471" timestamp="2026-08-25T07:43:39Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1110416677" />\n    <nd ref="1559016892" />\n    <nd ref="1559016893" />\n    <nd ref="1559016895" />\n    <nd ref="1559016899" />\n    <nd ref="1559016901" />\n    <nd ref="1559016902" />\n    <nd ref="1110416726" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนละม้ายสงเคราะห์" />\n    <tag k="name:en" v="Lamai Songkhro Road" />\n    <tag k="name:th" v="ถนนละม้ายสงเคราะห์" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="142456544" visible="true" version="4" changeset="187974708" timestamp="2026-08-25T07:50:18Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1559029798" />\n    <nd ref="1559029797" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ตรอกสุคนธหงส์" />\n    <tag k="name:en" v="Trok Sukonthahong" />\n    <tag k="name:th" v="ตรอกสุคนธหงส์" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1447303746" visible="true" version="3" changeset="189138667" timestamp="2026-09-17T03:37:59Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="13279030938" />\n    <nd ref="13279030939" />\n    <nd ref="14188450535" />\n    <nd ref="13279030940" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="1454593202" visible="true" version="3" changeset="189148039" timestamp="2026-09-17T08:49:06Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="13339819885" />\n    <nd ref="14188764647" />\n    <nd ref="1431715415" />\n    <nd ref="1107773211" />\n    <nd ref="1110416556" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนหอยมุกต์" />\n    <tag k="name:en" v="Hoimook Road" />\n    <tag k="name:th" v="ถนนหอยมุกต์" />\n    <tag k="name:th-Latn" v="Thanon Hoimook" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1454593203" visible="true" version="3" changeset="177753655" timestamp="2026-01-27T09:03:04Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="13339844001" />\n    <nd ref="13339844002" />\n    <nd ref="13339844003" />\n    <nd ref="13339844004" />\n    <nd ref="13339844005" />\n    <nd ref="13339819888" />\n    <nd ref="13339819889" />\n    <nd ref="13339819890" />\n    <nd ref="13339819891" />\n    <nd ref="13339819896" />\n    <nd ref="13339819885" />\n    <nd ref="13339819897" />\n    <nd ref="13339819898" />\n    <nd ref="13339819899" />\n    <nd ref="13339819900" />\n    <nd ref="13339819892" />\n    <nd ref="13339819893" />\n    <nd ref="13339819894" />\n    <nd ref="13339819895" />\n    <nd ref="13339844001" />\n    <tag k="highway" v="tertiary" />\n    <tag k="junction" v="roundabout" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1457057266" visible="true" version="3" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1431867166" />\n    <nd ref="14188679749" />\n    <nd ref="1558997823" />\n    <nd ref="13362950402" />\n    <nd ref="1431867100" />\n    <nd ref="1558997821" />\n    <nd ref="13362936996" />\n    <nd ref="1431867172" />\n    <nd ref="1558997817" />\n    <nd ref="1431867112" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Rajyindee 9 Road" />\n    <tag k="name:th" v="ราษฎร์ยินดี 9" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1457057267" visible="true" version="2" changeset="187917632" timestamp="2026-08-24T04:51:45Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1431867108" />\n    <nd ref="13362936997" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="Thanon Pattano Uthit" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1457057268" visible="true" version="3" changeset="177509915" timestamp="2026-01-21T08:28:25Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="8642866005" />\n    <nd ref="2507400589" />\n    <tag k="highway" v="service" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="1463496591" visible="true" version="2" changeset="186902224" timestamp="2026-08-04T04:46:49Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="13424220804" />\n    <nd ref="14072021538" />\n    <nd ref="14072021539" />\n    <nd ref="13424220803" />\n    <tag k="highway" v="service" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="1463496593" visible="true" version="2" changeset="186902285" timestamp="2026-08-04T04:50:21Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1123399562" />\n    <nd ref="13424220819" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="service" />\n    <tag k="layer" v="1" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="1463496594" visible="true" version="2" changeset="186902285" timestamp="2026-08-04T04:50:21Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="13424220819" />\n    <nd ref="13424220820" />\n    <tag k="highway" v="service" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="1463496595" visible="true" version="2" changeset="186902285" timestamp="2026-08-04T04:50:21Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="13424220819" />\n    <nd ref="13424220821" />\n    <nd ref="13424220822" />\n    <nd ref="13424220823" />\n    <nd ref="13424220824" />\n    <nd ref="13424220825" />\n    <nd ref="13424220826" />\n    <nd ref="13424220827" />\n    <tag k="highway" v="service" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="1469071362" visible="true" version="2" changeset="177509915" timestamp="2026-01-21T08:28:25Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="13340060823" />\n    <nd ref="13472970568" />\n    <nd ref="13472970569" />\n    <nd ref="1680667519" />\n    <tag k="highway" v="service" />\n    <tag k="service" v="parking_aisle" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="1476708031" visible="true" version="2" changeset="186904897" timestamp="2026-08-04T06:26:55Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="8932057454" />\n    <nd ref="13545019681" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="148108188" visible="true" version="8" changeset="156846444" timestamp="2024-09-20T02:29:38Z" user="djmorkman" uid="5547549">\n    <nd ref="1369020941" />\n    <nd ref="12187377521" />\n    <nd ref="12187377522" />\n    <nd ref="620881568" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="1" />\n    <tag k="name" v="ถนนเพชรเกษม" />\n    <tag k="name:en" v="Phet Kasem Road" />\n    <tag k="oneway" v="yes" />\n    <tag k="unsigned_ref" v="สข.ถ3-0013" />\n  </way>\n  <way id="148941242" visible="true" version="4" changeset="187926462" timestamp="2026-08-24T08:56:47Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1107773191" />\n    <nd ref="1110416726" />\n    <nd ref="1431715372" />\n    <nd ref="1107773215" />\n    <nd ref="1107773286" />\n    <nd ref="1559016904" />\n    <nd ref="1559016909" />\n    <nd ref="1107773217" />\n    <nd ref="1559016916" />\n    <nd ref="1107773211" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนฉัยยากุลอุทิศ" />\n    <tag k="name:en" v="Chaiyakuluthit Roafd" />\n    <tag k="name:th" v="ถนนฉัยยากุลอุทิศ" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="148941245" visible="true" version="3" changeset="187926462" timestamp="2026-08-24T08:56:47Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1110416683" />\n    <nd ref="1107773191" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="layer" v="1" />\n    <tag k="name" v="ถนนฉัยยากุลอุทิศ" />\n    <tag k="name:en" v="Chaiyakuluthit Roafd" />\n    <tag k="name:th" v="ถนนฉัยยากุลอุทิศ" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="151072451" visible="true" version="9" changeset="188760598" timestamp="2026-09-09T08:41:34Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="1639088623" />\n    <nd ref="1639088625" />\n    <nd ref="14165895541" />\n    <nd ref="14063655439" />\n    <nd ref="13388461897" />\n    <nd ref="14166086160" />\n    <nd ref="13388461898" />\n    <nd ref="1639088652" />\n    <nd ref="1639088656" />\n    <nd ref="2507400649" />\n    <nd ref="1639088636" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนราษฎร์ยินดี ซอย 7" />\n    <tag k="name:en" v="Rajyindee Road Soi 7" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี ซอย 7" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="151072453" visible="true" version="10" changeset="187917632" timestamp="2026-08-24T04:51:45Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="809537820" />\n    <nd ref="8523328986" />\n    <nd ref="1639088658" />\n    <nd ref="1639088623" />\n    <nd ref="8974100559" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1536475432" visible="true" version="1" changeset="185327371" timestamp="2026-07-08T09:13:38Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="8523326826" />\n    <nd ref="2507400630" />\n    <tag k="highway" v="service" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="154097280" visible="true" version="7" changeset="187967517" timestamp="2026-08-25T03:01:35Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1115585403" />\n    <nd ref="1666915318" />\n    <nd ref="1966404349" />\n    <tag k="addr:city" v="Hat Yai" />\n    <tag k="addr:country" v="TH" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Supasarnrungsan Road Soi 2" />\n    <tag k="name:th" v="ถนนศุภสารรังสรรค์" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1546208563" visible="true" version="1" changeset="186687676" timestamp="2026-07-31T06:26:34Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="14063214200" />\n    <nd ref="14063214199" />\n    <nd ref="14063214195" />\n    <nd ref="14063214196" />\n    <tag k="highway" v="service" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="1546208910" visible="true" version="2" changeset="188752824" timestamp="2026-09-09T04:48:29Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="14063217001" />\n    <nd ref="14063217003" />\n    <nd ref="14165395422" />\n    <nd ref="14063217002" />\n    <nd ref="14165395419" />\n    <nd ref="14165395420" />\n    <nd ref="14165395421" />\n    <nd ref="14165395422" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1546217294" visible="true" version="1" changeset="186689791" timestamp="2026-07-31T07:31:36Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="8523314489" />\n    <nd ref="8523314488" />\n    <nd ref="8523314487" />\n    <nd ref="8523314486" />\n    <nd ref="8523314485" />\n    <nd ref="8523314484" />\n    <tag k="highway" v="service" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="1546257454" visible="true" version="2" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14063655439" />\n    <nd ref="14165895543" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1546894905" visible="true" version="1" changeset="186853397" timestamp="2026-08-03T07:28:04Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14069947700" />\n    <nd ref="14069967502" />\n    <nd ref="14069967501" />\n    <tag k="highway" v="residential" />\n    <tag k="motorcar" v="no" />\n    <tag k="motorcycle" v="yes" />\n  </way>\n  <way id="1546896992" visible="true" version="2" changeset="187667962" timestamp="2026-08-19T03:09:11Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14069981565" />\n    <nd ref="14069981569" />\n    <nd ref="14069981570" />\n    <nd ref="14069981566" />\n    <nd ref="14069981567" />\n    <nd ref="14069981568" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="1546907364" visible="true" version="2" changeset="187667899" timestamp="2026-08-19T03:05:11Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14070071629" />\n    <nd ref="14070071630" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="154707523" visible="true" version="8" changeset="188339904" timestamp="2026-09-01T02:58:55Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2052547073" />\n    <nd ref="2052547075" />\n    <nd ref="14138648129" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1547102484" visible="true" version="1" changeset="186900335" timestamp="2026-08-04T03:24:25Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14071847284" />\n    <nd ref="14071847286" />\n    <nd ref="14071847287" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1547102485" visible="true" version="1" changeset="186900335" timestamp="2026-08-04T03:24:25Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14071847283" />\n    <nd ref="14071847285" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1547102487" visible="true" version="2" changeset="187917632" timestamp="2026-08-24T04:51:45Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2507400593" />\n    <nd ref="2507400591" />\n    <nd ref="2507400589" />\n    <nd ref="14071847289" />\n    <nd ref="14071847284" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1547102488" visible="true" version="1" changeset="186900335" timestamp="2026-08-04T03:24:25Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14071847285" />\n    <nd ref="14071847284" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="service" />\n    <tag k="layer" v="1" />\n  </way>\n  <way id="1547102489" visible="true" version="1" changeset="186900335" timestamp="2026-08-04T03:24:25Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14071847288" />\n    <nd ref="14071847290" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1547102491" visible="true" version="1" changeset="186900335" timestamp="2026-08-04T03:24:25Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14071847290" />\n    <nd ref="14071847289" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="service" />\n    <tag k="layer" v="1" />\n  </way>\n  <way id="1547119925" visible="true" version="2" changeset="187667962" timestamp="2026-08-19T03:09:11Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14071882647" />\n    <nd ref="14071882648" />\n    <nd ref="14071882649" />\n    <nd ref="14071882651" />\n    <nd ref="14071882650" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1547121994" visible="true" version="2" changeset="187667980" timestamp="2026-08-19T03:10:34Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14071937372" />\n    <nd ref="14071937375" />\n    <nd ref="14071937368" />\n    <nd ref="14071937376" />\n    <nd ref="14071937369" />\n    <nd ref="14071937370" />\n    <nd ref="14071937377" />\n    <nd ref="14071937371" />\n    <tag k="highway" v="service" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="1547121995" visible="true" version="2" changeset="187667980" timestamp="2026-08-19T03:10:34Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14071937367" />\n    <nd ref="14071937374" />\n    <nd ref="14071937373" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1547121996" visible="true" version="2" changeset="187667980" timestamp="2026-08-19T03:10:34Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14071937374" />\n    <nd ref="14071937375" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1547121997" visible="true" version="2" changeset="187667980" timestamp="2026-08-19T03:10:34Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14071937376" />\n    <nd ref="14071937377" />\n    <tag k="highway" v="service" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="1552741344" visible="true" version="1" changeset="188075198" timestamp="2026-08-27T03:41:25Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="8523455254" />\n    <nd ref="14127561645" />\n    <nd ref="14127561644" />\n    <tag k="highway" v="service" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="1552748145" visible="true" version="1" changeset="188076207" timestamp="2026-08-27T04:39:10Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="14127615692" />\n    <nd ref="14127615693" />\n    <tag k="highway" v="service" />\n    <tag k="surface" v="unpaved" />\n  </way>\n  <way id="1552748146" visible="true" version="1" changeset="188076207" timestamp="2026-08-27T04:39:10Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506088518" />\n    <nd ref="14127615694" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="1552748564" visible="true" version="2" changeset="188076387" timestamp="2026-08-27T04:49:30Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="14127602686" />\n    <nd ref="4900465706" />\n    <nd ref="916327912" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="1554703880" visible="true" version="1" changeset="188457192" timestamp="2026-09-03T08:00:30Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="3376084911" />\n    <nd ref="1123399611" />\n    <nd ref="1123399715" />\n    <tag k="highway" v="residential" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1555071541" visible="true" version="1" changeset="188517464" timestamp="2026-09-04T10:11:00Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14148398712" />\n    <nd ref="14148398713" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1556742612" visible="true" version="1" changeset="188662419" timestamp="2026-09-07T10:20:56Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14159239425" />\n    <nd ref="14159239426" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="1556742613" visible="true" version="1" changeset="188662419" timestamp="2026-09-07T10:20:56Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14159239427" />\n    <nd ref="14159239428" />\n    <nd ref="14159239429" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="unpaved" />\n  </way>\n  <way id="155690340" visible="true" version="3" changeset="188028182" timestamp="2026-08-26T07:51:44Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1680666686" />\n    <nd ref="1680666699" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ซอยละม้ายสงเคราะห์ 1" />\n    <tag k="name:en" v="Soi Lamai Songkraw 1" />\n    <tag k="name:th" v="ซอยละม้ายสงเคราะห์ 1" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="155690480" visible="true" version="2" changeset="49318745" timestamp="2017-06-06T20:56:35Z" user="RVR004" uid="5314349">\n    <nd ref="1680667509" />\n    <nd ref="4900465244" />\n    <nd ref="796747714" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="155690513" visible="true" version="3" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084">\n    <nd ref="1558314144" />\n    <nd ref="13351357060" />\n    <nd ref="13351357076" />\n    <nd ref="13351357083" />\n    <nd ref="13351357082" />\n    <nd ref="1680667730" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi 14 Rajyindee Road" />\n  </way>\n  <way id="155690528" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:10:06Z" user="Bernhard Hiller" uid="326704">\n    <nd ref="1680666789" />\n    <nd ref="1680667181" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="155690734" visible="true" version="3" changeset="181556965" timestamp="2026-04-19T22:13:33Z" user="wireguy" uid="9564652">\n    <nd ref="1680667083" />\n    <nd ref="1680667079" />\n    <nd ref="1680667090" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="155690741" visible="true" version="7" changeset="189045792" timestamp="2026-09-15T07:53:00Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1680666468" />\n    <nd ref="6822269571" />\n    <nd ref="1680666638" />\n    <nd ref="8643114367" />\n    <nd ref="14182422401" />\n    <nd ref="1559016909" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Thanon Chaiyakuluthit Soi 2" />\n    <tag k="surface" v="concrete:plates" />\n  </way>\n  <way id="155690766" visible="true" version="3" changeset="188028426" timestamp="2026-08-26T07:58:56Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="14124372093" />\n    <nd ref="1680667445" />\n    <nd ref="1680667686" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนคลองเรียน 2 ซอย 9" />\n    <tag k="name:en" v="Klongrien 2 Soi 9" />\n    <tag k="name:th" v="ถนนคลองเรียน 2 ซอย 9" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="155690791" visible="true" version="3" changeset="188028988" timestamp="2026-08-26T08:12:24Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1680666546" />\n    <nd ref="1680666347" />\n    <nd ref="14070071629" />\n    <nd ref="1680666115" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ซอยสเน่หานุสรณ์" />\n    <tag k="name:en" v="Soi Sanehanusorn" />\n    <tag k="name:th" v="ซอยสเน่หานุสรณ์" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="155690812" visible="true" version="3" changeset="181556965" timestamp="2026-04-19T22:13:33Z" user="wireguy" uid="9564652">\n    <nd ref="13351099349" />\n    <nd ref="1680667164" />\n    <nd ref="1680667158" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="155690866" visible="true" version="5" changeset="185032029" timestamp="2026-07-03T10:30:02Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1680667749" />\n    <nd ref="8642866007" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="residential" />\n    <tag k="layer" v="1" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="155690872" visible="true" version="5" changeset="177509915" timestamp="2026-01-21T08:28:25Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="8642866004" />\n    <nd ref="8642866005" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="service" />\n    <tag k="layer" v="1" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="155690875" visible="true" version="1" changeset="11023270" timestamp="2012-03-18T20:11:11Z" user="Bernhard Hiller" uid="326704">\n    <nd ref="1680666683" />\n    <nd ref="1680666977" />\n    <nd ref="1680667233" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="155690878" visible="true" version="2" changeset="177510665" timestamp="2026-01-21T08:47:43Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="1680666950" />\n    <nd ref="1680667029" />\n    <nd ref="1680667034" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="155690968" visible="true" version="3" changeset="188030351" timestamp="2026-08-26T08:44:57Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1680666322" />\n    <nd ref="1680666330" />\n    <nd ref="8934566434" />\n    <nd ref="8934566433" />\n    <nd ref="8934566435" />\n    <nd ref="8934566432" />\n    <nd ref="8934566431" />\n    <nd ref="8934566430" />\n    <nd ref="1680666281" />\n    <nd ref="1680666294" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="155690981" visible="true" version="2" changeset="72496129" timestamp="2019-07-22T02:29:12Z" user="JAAS" uid="3321532">\n    <nd ref="1680666461" />\n    <nd ref="1680666473" />\n    <nd ref="1680666541" />\n    <nd ref="1680666534" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi 2 Lamaisongkraw" />\n  </way>\n  <way id="155691015" visible="true" version="3" changeset="181556965" timestamp="2026-04-19T22:13:33Z" user="wireguy" uid="9564652">\n    <nd ref="1680666827" />\n    <nd ref="1680666961" />\n    <nd ref="1680667079" />\n    <nd ref="1680667164" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="155691022" visible="true" version="4" changeset="177274810" timestamp="2026-01-15T20:57:54Z" user="Callipo" uid="23344489">\n    <nd ref="1680666891" />\n    <nd ref="4900465071" />\n    <nd ref="1680667323" />\n    <nd ref="1680667342" />\n    <nd ref="13458847473" />\n    <nd ref="1431867140" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi 8 Rajyindee" />\n  </way>\n  <way id="155691032" visible="true" version="3" changeset="177510665" timestamp="2026-01-21T08:47:43Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="1680667029" />\n    <nd ref="1680666793" />\n    <nd ref="1680666823" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="155691084" visible="true" version="6" changeset="188031059" timestamp="2026-08-26T08:59:57Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="8642866010" />\n    <nd ref="4900465949" />\n    <nd ref="1680666827" />\n    <nd ref="1680666876" />\n    <nd ref="1680666891" />\n    <nd ref="1680666958" />\n    <nd ref="8523321638" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ซอย 8 ราษฎร์ยินดี" />\n    <tag k="name:en" v="Soi 8 Rayindee" />\n    <tag k="name:th" v="ซอย 8 ราษฎร์ยินดี" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="155691088" visible="true" version="3" changeset="175694769" timestamp="2025-12-09T03:57:22Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="1431867172" />\n    <nd ref="1680666122" />\n    <nd ref="1680666290" />\n    <nd ref="1680666463" />\n    <nd ref="1680666716" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi Vachira-Anan 1" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="1557535966" visible="true" version="1" changeset="188756079" timestamp="2026-09-09T06:45:22Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="1110416454" />\n    <nd ref="14165536480" />\n    <nd ref="14165536477" />\n    <nd ref="14165536478" />\n    <nd ref="14165536479" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1557537524" visible="true" version="1" changeset="188756440" timestamp="2026-09-09T06:55:23Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14165547215" />\n    <nd ref="14165547216" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="1557537525" visible="true" version="1" changeset="188756440" timestamp="2026-09-09T06:55:23Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14165547217" />\n    <nd ref="14165547218" />\n    <nd ref="14165547219" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1557556611" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14165895527" />\n    <nd ref="14165895528" />\n    <nd ref="14165895529" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1557556612" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14165895539" />\n    <nd ref="14165895540" />\n    <nd ref="14165895529" />\n    <nd ref="14165895530" />\n    <nd ref="14165895531" />\n    <nd ref="14165895536" />\n    <nd ref="14165895537" />\n    <nd ref="14165895538" />\n    <nd ref="14165895532" />\n    <nd ref="14165895533" />\n    <nd ref="14165895534" />\n    <nd ref="14165895535" />\n    <nd ref="14165895539" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1557556613" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14165895541" />\n    <nd ref="14165895542" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1557556614" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14165895543" />\n    <nd ref="14165895544" />\n    <nd ref="14165895545" />\n    <nd ref="14165895546" />\n    <nd ref="14165895547" />\n    <nd ref="14165895543" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1557556615" visible="true" version="1" changeset="188759397" timestamp="2026-09-09T08:10:48Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="8523328988" />\n    <nd ref="14165895548" />\n    <nd ref="14165895552" />\n    <nd ref="14165895549" />\n    <nd ref="14165895550" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1557565368" visible="true" version="1" changeset="188760331" timestamp="2026-09-09T08:34:32Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14166007986" />\n    <nd ref="14166007987" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1558734968" visible="true" version="1" changeset="188989640" timestamp="2026-09-14T04:03:52Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="14179759489" />\n    <nd ref="2506088476" />\n    <tag k="highway" v="residential" />\n    <tag k="motorcar" v="no" />\n    <tag k="motorcycle" v="yes" />\n  </way>\n  <way id="1558986534" visible="true" version="1" changeset="189039564" timestamp="2026-09-15T03:36:04Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="14182062679" />\n    <nd ref="14182062684" />\n    <nd ref="14182062680" />\n    <nd ref="14182062681" />\n    <nd ref="14182062682" />\n    <nd ref="14182062683" />\n    <nd ref="14182062684" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1559020882" visible="true" version="1" changeset="189045792" timestamp="2026-09-15T07:53:00Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="14182422401" />\n    <nd ref="14182422402" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="1559800895" visible="true" version="1" changeset="189137799" timestamp="2026-09-17T02:46:06Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="14188069236" />\n    <nd ref="14188069237" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1559802495" visible="true" version="1" changeset="189138033" timestamp="2026-09-17T03:00:23Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="4900465288" />\n    <nd ref="14188374629" />\n    <nd ref="14188374628" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="1559802496" visible="true" version="1" changeset="189138033" timestamp="2026-09-17T03:00:23Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="14188374629" />\n    <nd ref="14188374631" />\n    <nd ref="14188374632" />\n    <nd ref="14188374630" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="1559827446" visible="true" version="1" changeset="189143922" timestamp="2026-09-17T07:11:29Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="14188637663" />\n    <nd ref="14188637665" />\n    <nd ref="14188637664" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1559831193" visible="true" version="1" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="14188679740" />\n    <nd ref="14188679750" />\n    <nd ref="14188679741" />\n    <nd ref="14188679752" />\n    <nd ref="14188679742" />\n    <nd ref="14188679743" />\n    <nd ref="14188679744" />\n    <nd ref="14188679745" />\n    <nd ref="14188679746" />\n    <nd ref="14188679747" />\n    <nd ref="14188679748" />\n    <nd ref="14188679751" />\n    <nd ref="14188679749" />\n    <tag k="highway" v="service" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="1559831194" visible="true" version="1" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="14188679750" />\n    <nd ref="14188679751" />\n    <tag k="highway" v="service" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="1559831195" visible="true" version="1" changeset="189145001" timestamp="2026-09-17T07:38:16Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="14188679752" />\n    <nd ref="14188679753" />\n    <nd ref="8642666192" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1559838040" visible="true" version="1" changeset="189147130" timestamp="2026-09-17T08:28:52Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14063214194" />\n    <nd ref="14188701065" />\n    <nd ref="14063214195" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1559841160" visible="true" version="1" changeset="189148039" timestamp="2026-09-17T08:49:06Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14188764645" />\n    <nd ref="14188764646" />\n    <nd ref="14188764647" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="1559841161" visible="true" version="1" changeset="189148039" timestamp="2026-09-17T08:49:06Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14188764646" />\n    <nd ref="14188764648" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="1559842629" visible="true" version="1" changeset="189148524" timestamp="2026-09-17T09:00:53Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14188730088" />\n    <nd ref="14188730089" />\n    <nd ref="14188730090" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1559842630" visible="true" version="1" changeset="189148524" timestamp="2026-09-17T09:00:53Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="14188730091" />\n    <nd ref="14188730094" />\n    <nd ref="14188730092" />\n    <nd ref="14188730093" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="1562470074" visible="true" version="1" changeset="189575613" timestamp="2026-09-26T08:15:30Z" user="Tonushka" uid="3419975">\n    <nd ref="1122072661" />\n    <nd ref="1110416356" />\n    <tag k="bridge" v="yes" />\n    <tag k="bridge:structure" v="arch" />\n    <tag k="highway" v="pedestrian" />\n  </way>\n  <way id="1562470075" visible="true" version="1" changeset="189575613" timestamp="2026-09-26T08:15:30Z" user="Tonushka" uid="3419975">\n    <nd ref="14219025415" />\n    <nd ref="1559016913" />\n    <tag k="bridge" v="yes" />\n    <tag k="bridge:structure" v="arch" />\n    <tag k="highway" v="pedestrian" />\n  </way>\n  <way id="185951074" visible="true" version="16" changeset="177551471" timestamp="2026-01-22T07:08:36Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="856214524" />\n    <nd ref="13279030938" />\n    <nd ref="4900465707" />\n    <nd ref="8523332291" />\n    <nd ref="856214562" />\n    <nd ref="1123399600" />\n    <nd ref="8974100553" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนประชาธิปัตย์" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="194743656" visible="true" version="4" changeset="188031215" timestamp="2026-08-26T09:03:37Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2052497441" />\n    <nd ref="2506086743" />\n    <nd ref="2052522376" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนแสงอาทิตย์" />\n    <tag k="name:en" v="Sang Aretit Road" />\n    <tag k="name:th" v="ถนนแสงอาทิตย์" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243079656" visible="true" version="6" changeset="188031708" timestamp="2026-08-26T09:16:03Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506089779" />\n    <nd ref="1115585857" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243079673" visible="true" version="2" changeset="18522412" timestamp="2013-10-24T16:26:11Z" user="Bernhard Hiller" uid="326704">\n    <nd ref="947896979" />\n    <nd ref="2505176494" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="unclassified" />\n    <tag k="layer" v="1" />\n  </way>\n  <way id="243079723" visible="true" version="5" changeset="175796005" timestamp="2025-12-11T10:36:57Z" user="neeli_srikanth" uid="12625709">\n    <nd ref="2505176686" />\n    <nd ref="13348305823" />\n    <nd ref="13369238458" />\n    <nd ref="13348305832" />\n    <nd ref="2505176654" />\n    <nd ref="947896972" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ถนนราษฏร์ยินดี ซอย 2" />\n    <tag k="name:en" v="Rajyindee Road Soi 2" />\n    <tag k="name:th" v="ถนนราษฏร์ยินดี ซอย 2" />\n  </way>\n  <way id="243079738" visible="true" version="4" changeset="186695948" timestamp="2026-07-31T09:54:46Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="947896958" />\n    <nd ref="14063537293" />\n    <nd ref="2505176284" />\n    <nd ref="2505176271" />\n    <nd ref="849403490" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Thanon Rajyindee Soi 6" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="243079757" visible="true" version="8" changeset="188759478" timestamp="2026-09-09T08:13:12Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="8523328987" />\n    <nd ref="8523328988" />\n    <nd ref="14165895551" />\n    <nd ref="8523328989" />\n    <nd ref="14165895550" />\n    <nd ref="2505176202" />\n    <nd ref="2505176189" />\n    <nd ref="14165895527" />\n    <nd ref="2505176182" />\n    <nd ref="1639088658" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi Rajyindee Road 6/ 1" />\n    <tag k="name:th" v="ซอย" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243079775" visible="true" version="2" changeset="18522412" timestamp="2013-10-24T16:26:11Z" user="Bernhard Hiller" uid="326704">\n    <nd ref="2505176514" />\n    <nd ref="947896989" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="residential" />\n    <tag k="layer" v="1" />\n  </way>\n  <way id="243079805" visible="true" version="4" changeset="137382428" timestamp="2023-06-15T17:15:11Z" user="Nithinat S" uid="10217273">\n    <nd ref="1110416568" />\n    <nd ref="1431715366" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="2" />\n    <tag k="layer" v="1" />\n    <tag k="name" v="ถนนธรรมนูญวิถี" />\n    <tag k="name:en" v="Thammanun Withi Road" />\n    <tag k="name:th" v="ถนนธรรมนูญวิถี" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="243079844" visible="true" version="10" changeset="145861546" timestamp="2024-01-04T02:30:19Z" user="GrabTH_Praiya" uid="18100118">\n    <nd ref="11292971036" />\n    <nd ref="809537690" />\n    <nd ref="809537754" />\n    <nd ref="1666900742" />\n    <nd ref="2406954577" />\n    <nd ref="7691192942" />\n    <nd ref="2406954573" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="243188024" visible="true" version="2" changeset="175446735" timestamp="2025-12-03T13:24:02Z" user="WorayutTK" uid="23562084">\n    <nd ref="1115585463" />\n    <nd ref="2506085598" />\n    <nd ref="2506085568" />\n    <nd ref="2506085551" />\n    <nd ref="2506085545" />\n    <nd ref="2506085544" />\n    <nd ref="2506085550" />\n    <nd ref="2506085547" />\n    <nd ref="2506085538" />\n    <nd ref="2506085532" />\n    <nd ref="13351323871" />\n    <nd ref="2506085526" />\n    <nd ref="13351323872" />\n    <nd ref="2506085525" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="243188053" visible="true" version="8" changeset="188034338" timestamp="2026-08-26T10:16:37Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506087550" />\n    <nd ref="13472440790" />\n    <nd ref="13472440791" />\n    <nd ref="2506087659" />\n    <nd ref="2506087846" />\n    <nd ref="13472440792" />\n    <nd ref="12351390166" />\n    <nd ref="2506087952" />\n    <nd ref="12351390167" />\n    <nd ref="2506087965" />\n    <nd ref="12351390169" />\n    <nd ref="2506087966" />\n    <nd ref="12351390168" />\n    <nd ref="2506087964" />\n    <nd ref="2506087958" />\n    <nd ref="13472484411" />\n    <nd ref="2506087959" />\n    <nd ref="13472484410" />\n    <nd ref="13472484405" />\n    <nd ref="13472484409" />\n    <nd ref="2506087967" />\n    <nd ref="13472484408" />\n    <nd ref="2506087989" />\n    <nd ref="13472484407" />\n    <nd ref="13472484406" />\n    <nd ref="13472484415" />\n    <nd ref="2506088006" />\n    <nd ref="13472484414" />\n    <nd ref="13472484404" />\n    <nd ref="13472484413" />\n    <nd ref="2506088015" />\n    <nd ref="13472484412" />\n    <nd ref="2506088019" />\n    <nd ref="12351390175" />\n    <nd ref="12351390176" />\n    <nd ref="2506088080" />\n    <nd ref="2506088052" />\n    <nd ref="12351390177" />\n    <nd ref="2506088046" />\n    <nd ref="12351390178" />\n    <nd ref="3376092311" />\n    <nd ref="12351390179" />\n    <nd ref="2506088009" />\n    <tag k="highway" v="service" />\n    <tag k="name" v="ซอยเพชรสยาม" />\n    <tag k="name:en" v="Soi Phetsaim" />\n    <tag k="name:th" v="ซอยเพชรสยาม" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188056" visible="true" version="2" changeset="72474083" timestamp="2019-07-21T05:34:00Z" user="JAAS" uid="3321532">\n    <nd ref="1369020942" />\n    <nd ref="2506088695" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi Boonraj" />\n  </way>\n  <way id="243188064" visible="true" version="5" changeset="188034595" timestamp="2026-08-26T10:22:18Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506085496" />\n    <nd ref="4900465472" />\n    <nd ref="2506085628" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ถนนภราดร" />\n    <tag k="name:en" v="Paradorn Road" />\n    <tag k="name:th" v="ถนนภราดร" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188085" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:55:20Z" user="Bernhard Hiller" uid="326704">\n    <nd ref="1123399486" />\n    <nd ref="2506088938" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="243188092" visible="true" version="2" changeset="72474083" timestamp="2019-07-21T05:34:00Z" user="JAAS" uid="3321532">\n    <nd ref="2506088528" />\n    <nd ref="2506088507" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi Boonraj 1" />\n  </way>\n  <way id="243188101" visible="true" version="3" changeset="188035073" timestamp="2026-08-26T10:34:18Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="12351390199" />\n    <nd ref="12351390200" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="residential" />\n    <tag k="layer" v="1" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188103" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:55:23Z" user="Bernhard Hiller" uid="326704">\n    <nd ref="2506087304" />\n    <nd ref="2506087252" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="243188115" visible="true" version="2" changeset="72474083" timestamp="2019-07-21T05:34:00Z" user="JAAS" uid="3321532">\n    <nd ref="1123399713" />\n    <nd ref="2506090451" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi Thongdee 3" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188120" visible="true" version="4" changeset="188035473" timestamp="2026-08-26T10:43:39Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="620881097" />\n    <nd ref="7508595076" />\n    <nd ref="2506087028" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188139" visible="true" version="2" changeset="72474083" timestamp="2019-07-21T05:34:00Z" user="JAAS" uid="3321532">\n    <nd ref="2506088023" />\n    <nd ref="2506088011" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi Boonraj 3" />\n  </way>\n  <way id="243188161" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:55:32Z" user="Bernhard Hiller" uid="326704">\n    <nd ref="1123399675" />\n    <nd ref="2506089487" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="243188167" visible="true" version="2" changeset="72474083" timestamp="2019-07-21T05:34:00Z" user="JAAS" uid="3321532">\n    <nd ref="2506088274" />\n    <nd ref="2506088265" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi Boonraj 2" />\n  </way>\n  <way id="243188174" visible="true" version="4" changeset="188073890" timestamp="2026-08-27T02:28:27Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506087709" />\n    <nd ref="6634630596" />\n    <nd ref="4900465989" />\n    <nd ref="2506087732" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนทิพรัตน์" />\n    <tag k="name:en" v="Tiparat Road" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188196" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:55:37Z" user="Bernhard Hiller" uid="326704">\n    <nd ref="1123399513" />\n    <nd ref="2506089901" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="243188208" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:55:38Z" user="Bernhard Hiller" uid="326704">\n    <nd ref="2506087592" />\n    <nd ref="2506087962" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="243188212" visible="true" version="5" changeset="188074120" timestamp="2026-08-27T02:44:28Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="12351390200" />\n    <nd ref="1123399641" />\n    <nd ref="13473142490" />\n    <nd ref="13473142491" />\n    <nd ref="2506088509" />\n    <tag k="highway" v="residential" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์ ซอย 4/2" />\n    <tag k="name:en" v="Supasarnrangsan Road Soi 4/2" />\n    <tag k="name:th" v="ถนนศุภสารรังสรรค์ ซอย 4/2" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188225" visible="true" version="5" changeset="188459881" timestamp="2026-09-03T08:57:10Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1123399683" />\n    <nd ref="14144046929" />\n    <nd ref="4488366488" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์ ซอย 4" />\n    <tag k="name:en" v="Suphasanrangsan Road Soi 4" />\n    <tag k="name:th" v="ถนนศุภสารรังสรรค์ ซอย 4" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188243" visible="true" version="4" changeset="83467076" timestamp="2020-04-13T09:57:26Z" user="Bernhard Hiller" uid="326704">\n    <nd ref="2506090118" />\n    <nd ref="1123399591" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="layer" v="1" />\n    <tag k="name" v="ถนนจุติอนุสรณ์" />\n    <tag k="name:th" v="ถนนจุติอนุสรณ์" />\n  </way>\n  <way id="243188273" visible="true" version="3" changeset="188074330" timestamp="2026-08-27T02:56:26Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1110416526" />\n    <nd ref="3324718285" />\n    <nd ref="1123399569" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนจุติอุทิศ 2" />\n    <tag k="name:en" v="Juti Uthit 2 Road" />\n    <tag k="name:th" v="ถนนจุติอุทิศ 2" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188283" visible="true" version="2" changeset="72474083" timestamp="2019-07-21T05:34:00Z" user="JAAS" uid="3321532">\n    <nd ref="1369020943" />\n    <nd ref="2506087957" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi Boonraj 4" />\n  </way>\n  <way id="243188298" visible="true" version="3" changeset="188031708" timestamp="2026-08-26T09:16:03Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506089779" />\n    <nd ref="4888862230" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="layer" v="1" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188318" visible="true" version="2" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084">\n    <nd ref="2506085486" />\n    <nd ref="13351357189" />\n    <nd ref="13351357188" />\n    <nd ref="13351357183" />\n    <nd ref="13351357184" />\n    <nd ref="13351357179" />\n    <nd ref="13351357180" />\n    <nd ref="2506085476" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="243188351" visible="true" version="5" changeset="188081802" timestamp="2026-08-27T07:44:26Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506085631" />\n    <nd ref="8523332293" />\n    <nd ref="1107926639" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนประชาธิปัตย์ ซอย 3" />\n    <tag k="name:en" v="Prachathipat Road Soi 3" />\n    <tag k="name:th" v="ถนนประชาธิปัตย์ ซอย 3" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188361" visible="true" version="1" changeset="18505195" timestamp="2013-10-23T15:55:54Z" user="Bernhard Hiller" uid="326704">\n    <nd ref="1123399943" />\n    <nd ref="2506089058" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="243188389" visible="true" version="6" changeset="177592101" timestamp="2026-01-23T04:39:18Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506088254" />\n    <nd ref="2506088279" />\n    <nd ref="12346099052" />\n    <nd ref="2506088450" />\n    <nd ref="2506088485" />\n    <nd ref="12346099051" />\n    <nd ref="2506088491" />\n    <nd ref="2506088494" />\n    <nd ref="2506088504" />\n    <nd ref="2506088522" />\n    <nd ref="2506088675" />\n    <nd ref="2506088747" />\n    <nd ref="2506088791" />\n    <nd ref="2506089170" />\n    <nd ref="2506089593" />\n    <nd ref="2506089735" />\n    <nd ref="2506090118" />\n    <nd ref="3616902435" />\n    <nd ref="2506090379" />\n    <nd ref="2506090499" />\n    <nd ref="2506090510" />\n    <nd ref="8523461356" />\n    <nd ref="1160617590" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188432" visible="true" version="7" changeset="188074619" timestamp="2026-08-27T03:11:57Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="947896950" />\n    <nd ref="12346117426" />\n    <nd ref="12346117423" />\n    <nd ref="2506087316" />\n    <nd ref="2506086831" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนแสงศรี ซอย 8" />\n    <tag k="name:en" v="Sangsri Road Soi 8" />\n    <tag k="name:th" v="ถนนแสงศรี ซอย 8" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188445" visible="true" version="2" changeset="72474083" timestamp="2019-07-21T05:34:00Z" user="JAAS" uid="3321532">\n    <nd ref="1123399740" />\n    <nd ref="2506089826" />\n    <nd ref="2506090385" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi Thongdee 1" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188447" visible="true" version="3" changeset="100026862" timestamp="2021-02-26T07:10:30Z" user="Dmitriev_DA" uid="9356997">\n    <nd ref="2506085670" />\n    <nd ref="2506085849" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi Duangchan" />\n  </way>\n  <way id="243188452" visible="true" version="3" changeset="188074806" timestamp="2026-08-27T03:21:09Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506088715" />\n    <nd ref="2506088612" />\n    <nd ref="2506088529" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188458" visible="true" version="2" changeset="175447880" timestamp="2025-12-03T13:47:28Z" user="WorayutTK" uid="23562084">\n    <nd ref="2506085475" />\n    <nd ref="13351357166" />\n    <nd ref="13351357163" />\n    <nd ref="13351357162" />\n    <nd ref="13351357158" />\n    <nd ref="2506085489" />\n    <nd ref="2506085493" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="243188465" visible="true" version="4" changeset="188074990" timestamp="2026-08-27T03:31:02Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506085647" />\n    <nd ref="2506085537" />\n    <nd ref="4900465473" />\n    <nd ref="2506085499" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ถนนประชารมย์" />\n    <tag k="name:en" v="Pracharom Road" />\n    <tag k="name:th" v="ถนนประชารมย์" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188488" visible="true" version="3" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="2506088243" />\n    <nd ref="13475596787" />\n    <nd ref="8523455251" />\n    <nd ref="2506088733" />\n    <nd ref="2506088775" />\n    <nd ref="8523455257" />\n    <nd ref="8523455256" />\n    <nd ref="8523455255" />\n    <nd ref="8523455254" />\n    <nd ref="8523455253" />\n    <nd ref="8523455252" />\n    <nd ref="8523455251" />\n    <tag k="highway" v="service" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="243188491" visible="true" version="6" changeset="190585296" timestamp="2026-10-16T04:19:17Z" user="cfgy" uid="11410742">\n    <nd ref="2506088024" />\n    <nd ref="2506087450" />\n    <nd ref="1966399248" />\n    <tag k="highway" v="tertiary" />\n    <tag k="lane_markings" v="yes" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนประชาธิปัตย์ 1" />\n    <tag k="name:en" v="Prachathipat 1 Road" />\n    <tag k="name:th" v="ถนนประชาธิปัตย์ 1" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188499" visible="true" version="4" changeset="189138277" timestamp="2026-09-17T03:16:49Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="2506085555" />\n    <nd ref="2506085631" />\n    <tag k="highway" v="residential" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188503" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915">\n    <nd ref="1123399700" />\n    <nd ref="2506089901" />\n    <nd ref="1110416342" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="243188505" visible="true" version="2" changeset="72473869" timestamp="2019-07-21T05:10:52Z" user="JAAS" uid="3321532">\n    <nd ref="2506087955" />\n    <nd ref="2506087968" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="243188508" visible="true" version="3" changeset="189138033" timestamp="2026-09-17T03:00:23Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="2506088518" />\n    <nd ref="14127615692" />\n    <nd ref="2506088509" />\n    <nd ref="14188374630" />\n    <nd ref="2506088492" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="243188517" visible="true" version="2" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="2506088733" />\n    <nd ref="13475596790" />\n    <nd ref="2506088707" />\n    <nd ref="2506088971" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="243188527" visible="true" version="3" changeset="72474411" timestamp="2019-07-21T06:06:19Z" user="JAAS" uid="3321532">\n    <nd ref="2506087810" />\n    <nd ref="2506087863" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi Watthayorn" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="243188534" visible="true" version="3" changeset="177592425" timestamp="2026-01-23T04:59:07Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506086249" />\n    <nd ref="2506086896" />\n    <nd ref="2506087296" />\n    <tag k="highway" v="service" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="243188536" visible="true" version="3" changeset="188075603" timestamp="2026-08-27T04:09:21Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506087563" />\n    <nd ref="8348961009" />\n    <nd ref="2506087304" />\n    <nd ref="2506086743" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188552" visible="true" version="6" changeset="188075697" timestamp="2026-08-27T04:16:02Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1123399516" />\n    <nd ref="13472440726" />\n    <nd ref="6634656806" />\n    <nd ref="2506088492" />\n    <nd ref="4900465288" />\n    <nd ref="1107926940" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนกุลวัฒน์อุทิศ" />\n    <tag k="name:en" v="Kullawat Uthit Road" />\n    <tag k="name:th" v="ถนนกุลวัฒน์อุทิศ" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="243188561" visible="true" version="4" changeset="188517464" timestamp="2026-09-04T10:11:00Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="2506085690" />\n    <nd ref="14148398713" />\n    <nd ref="2506086151" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188563" visible="true" version="6" changeset="188076110" timestamp="2026-08-27T04:34:36Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506086795" />\n    <nd ref="2506087934" />\n    <nd ref="2506088002" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนแสงศรี ซอย 6" />\n    <tag k="name:en" v="Sangsi Road Soi 6" />\n    <tag k="name:th" v="ถนนแสงศรี ซอย 6" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188565" visible="true" version="4" changeset="188076207" timestamp="2026-08-27T04:39:10Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506088518" />\n    <nd ref="1123399483" />\n    <tag k="highway" v="residential" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์ ซอย 4/3" />\n    <tag k="name:en" v="Supasarnrangsan Road Soi 4/3" />\n    <tag k="name:th" v="ถนนศุภสารรังสรรค์ ซอย 4/3" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188576" visible="true" version="4" changeset="188076359" timestamp="2026-08-27T04:47:39Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506088223" />\n    <nd ref="2506087987" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ถนนแสงศรี ซอย 1" />\n    <tag k="name:en" v="Sangsri Road Soi 1" />\n    <tag k="name:th" v="ถนนแสงศรี ซอย 1" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188585" visible="true" version="6" changeset="188080435" timestamp="2026-08-27T07:12:05Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506087136" />\n    <nd ref="2506087923" />\n    <nd ref="2506088404" />\n    <nd ref="1107926561" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์ ซอย 5" />\n    <tag k="name:en" v="Suphasarnrangsan Road Soi 5" />\n    <tag k="name:th" v="ถนนศุภสารรังสรรค์ ซอย 5" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188594" visible="true" version="3" changeset="188074806" timestamp="2026-08-27T03:21:09Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1107926977" />\n    <nd ref="2506088612" />\n    <nd ref="2506088280" />\n    <nd ref="1123399489" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188607" visible="true" version="3" changeset="188081186" timestamp="2026-08-27T07:30:31Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506085508" />\n    <nd ref="2506085498" />\n    <nd ref="13351102290" />\n    <nd ref="2506085493" />\n    <nd ref="13351102291" />\n    <nd ref="2506085487" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188627" visible="true" version="2" changeset="72474083" timestamp="2019-07-21T05:34:00Z" user="JAAS" uid="3321532">\n    <nd ref="2506090779" />\n    <nd ref="2506090521" />\n    <nd ref="2506090181" />\n    <nd ref="2506089600" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi Thongdee 4" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188629" visible="true" version="3" changeset="188081802" timestamp="2026-08-27T07:44:26Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1107926639" />\n    <nd ref="2506085634" />\n    <nd ref="2506085650" />\n    <nd ref="796747634" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนประชาธิปัตย์ ซอย 3" />\n    <tag k="name:en" v="Prachathipat Road Soi 3" />\n    <tag k="name:th" v="ถนนประชาธิปัตย์ ซอย 3" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188641" visible="true" version="4" changeset="188074806" timestamp="2026-08-27T03:21:09Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1107926599" />\n    <nd ref="2506088529" />\n    <nd ref="2506088273" />\n    <nd ref="1123399659" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="Thanon Suvanvong" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188644" visible="true" version="2" changeset="72474083" timestamp="2019-07-21T05:34:00Z" user="JAAS" uid="3321532">\n    <nd ref="2506089545" />\n    <nd ref="2506090372" />\n    <nd ref="2506090505" />\n    <nd ref="2506090932" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi Thongdee 3" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188646" visible="true" version="6" changeset="188711452" timestamp="2026-09-08T09:17:40Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506087563" />\n    <nd ref="2506087522" />\n    <nd ref="8523307708" />\n    <nd ref="2506087462" />\n    <tag k="highway" v="residential" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="243188670" visible="true" version="5" changeset="190585296" timestamp="2026-10-16T04:21:15Z" user="cfgy" uid="11410742">\n    <nd ref="1123399715" />\n    <nd ref="2506088423" />\n    <nd ref="2506088715" />\n    <nd ref="1107926734" />\n    <nd ref="2506089309" />\n    <tag k="highway" v="tertiary" />\n    <tag k="lane_markings" v="yes" />\n    <tag k="lanes" v="2" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188705" visible="true" version="13" changeset="188350873" timestamp="2026-09-01T08:42:17Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1123399683" />\n    <nd ref="13472483863" />\n    <nd ref="13472483861" />\n    <nd ref="13472483862" />\n    <nd ref="1123399500" />\n    <nd ref="13472483860" />\n    <nd ref="13472483858" />\n    <nd ref="13472483859" />\n    <nd ref="1123399516" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์ ซอย 4" />\n    <tag k="name:en" v="Suphasanrangsan Road Soi 4" />\n    <tag k="name:th" v="ถนนศุภสารรังสรรค์ ซอย 4" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188710" visible="true" version="4" changeset="177596216" timestamp="2026-01-23T07:50:17Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1107926968" />\n    <nd ref="1107926467" />\n    <nd ref="13472483843" />\n    <nd ref="13472483844" />\n    <nd ref="1107926605" />\n    <nd ref="13472483842" />\n    <nd ref="1107926761" />\n    <nd ref="1369020939" />\n    <nd ref="13472483841" />\n    <nd ref="13472483840" />\n    <nd ref="1107926481" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ซอยบุญราช" />\n    <tag k="name:en" v="Soi Boonraj" />\n    <tag k="name:th" v="ซอยบุญราช" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188741" visible="true" version="2" changeset="185331124" timestamp="2026-07-08T10:24:19Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1107926537" />\n    <nd ref="2506085555" />\n    <nd ref="2506085570" />\n    <nd ref="2506085581" />\n    <nd ref="2506085591" />\n    <nd ref="2506085605" />\n    <nd ref="2506085623" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188765" visible="true" version="7" changeset="188351672" timestamp="2026-09-01T08:58:56Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2505176697" />\n    <nd ref="1110416742" />\n    <nd ref="4900465069" />\n    <nd ref="1120012976" />\n    <nd ref="6634648309" />\n    <nd ref="1120012989" />\n    <nd ref="2506090979" />\n    <nd ref="2507400851" />\n    <nd ref="2935421242" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนสามชัย" />\n    <tag k="name:en" v="Samchai Road" />\n    <tag k="name:th" v="ถนนสามชัย" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188785" visible="true" version="4" changeset="188352318" timestamp="2026-09-01T09:14:55Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1110416631" />\n    <nd ref="12346099050" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="layer" v="1" />\n    <tag k="name" v="ถนนสวนศิริ" />\n    <tag k="name:en" v="Suansiri Road" />\n    <tag k="name:th" v="ถนนสวนศิริ" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188786" visible="true" version="3" changeset="188352318" timestamp="2026-09-01T09:14:55Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="12346099050" />\n    <nd ref="1123399502" />\n    <nd ref="2506088938" />\n    <nd ref="2506089058" />\n    <nd ref="2506089487" />\n    <nd ref="1110416754" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนสวนศิริ" />\n    <tag k="name:en" v="Suansiri Road" />\n    <tag k="name:th" v="ถนนสวนศิริ" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243188787" visible="true" version="17" changeset="143103736" timestamp="2023-10-25T09:07:25Z" user="GrabTH_Praiya" uid="18100118">\n    <nd ref="985092634" />\n    <nd ref="2505177499" />\n    <nd ref="2507401162" />\n    <nd ref="2505177476" />\n    <nd ref="809537807" />\n    <nd ref="2507401129" />\n    <nd ref="809537392" />\n    <nd ref="809537800" />\n    <nd ref="809537378" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="1" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนเพชรเกษม" />\n    <tag k="name:en" v="Phet Kasem Road" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n    <tag k="unsigned_ref" v="สข.ถ3-0013" />\n  </way>\n  <way id="243314064" visible="true" version="4" changeset="187917632" timestamp="2026-08-24T04:51:45Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2507400847" />\n    <nd ref="2507400897" />\n    <nd ref="2507400921" />\n    <nd ref="2507400926" />\n    <nd ref="9179528432" />\n    <nd ref="2507400917" />\n    <nd ref="2507400923" />\n    <nd ref="1421027659" />\n    <nd ref="1421027965" />\n    <nd ref="1421027993" />\n    <nd ref="1421027903" />\n    <nd ref="1421027788" />\n    <tag k="highway" v="tertiary" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243314073" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:24:58Z" user="Bernhard Hiller" uid="326704">\n    <nd ref="2507400689" />\n    <nd ref="2507400690" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="residential" />\n    <tag k="layer" v="1" />\n  </way>\n  <way id="243314101" visible="true" version="5" changeset="188397820" timestamp="2026-09-02T03:18:10Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="11292971034" />\n    <nd ref="11292971033" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="secondary_link" />\n    <tag k="layer" v="1" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243314103" visible="true" version="2" changeset="72474083" timestamp="2019-07-21T05:34:00Z" user="JAAS" uid="3321532">\n    <nd ref="2507400752" />\n    <nd ref="2507400748" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ซอยยินดี" />\n  </way>\n  <way id="243314112" visible="true" version="3" changeset="161557772" timestamp="2025-01-20T10:53:05Z" user="GrabTH_Praiya" uid="18100118">\n    <nd ref="12518064113" />\n    <nd ref="2507400689" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi Charoensook" />\n  </way>\n  <way id="243314123" visible="true" version="2" changeset="177598145" timestamp="2026-01-23T08:49:53Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2505176422" />\n    <nd ref="2505176434" />\n    <tag k="highway" v="secondary_link" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243314137" visible="true" version="3" changeset="188398109" timestamp="2026-09-02T03:33:50Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2507400769" />\n    <nd ref="6822929410" />\n    <nd ref="2507400792" />\n    <tag k="highway" v="tertiary" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243314160" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:25:13Z" user="Bernhard Hiller" uid="326704">\n    <nd ref="1123399863" />\n    <nd ref="2507400772" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="243314165" visible="true" version="5" changeset="188398480" timestamp="2026-09-02T03:54:33Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2507400614" />\n    <nd ref="8523326825" />\n    <nd ref="2507400618" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ซอยเตชะโต" />\n    <tag k="name:en" v="Soi Tachato" />\n    <tag k="name:th" v="ซอยเตชะโต" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243314177" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:25:16Z" user="Bernhard Hiller" uid="326704">\n    <nd ref="2507400742" />\n    <nd ref="2507400773" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="243314186" visible="true" version="2" changeset="177598435" timestamp="2026-01-23T08:57:43Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2505176388" />\n    <nd ref="2505176422" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="residential" />\n    <tag k="layer" v="1" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="243314192" visible="true" version="3" changeset="188399074" timestamp="2026-09-02T04:24:14Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2507400639" />\n    <nd ref="2507400638" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary_link" />\n    <tag k="layer" v="1" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="243314213" visible="true" version="1" changeset="18522412" timestamp="2013-10-24T16:25:22Z" user="Bernhard Hiller" uid="326704">\n    <nd ref="2507400696" />\n    <nd ref="2507400698" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="residential" />\n    <tag k="layer" v="1" />\n  </way>\n  <way id="243314248" visible="true" version="2" changeset="49318745" timestamp="2017-06-06T20:56:39Z" user="RVR004" uid="5314349">\n    <nd ref="2507400612" />\n    <nd ref="2507400619" />\n    <nd ref="4900465377" />\n    <nd ref="2507400625" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="243314296" visible="true" version="7" changeset="181643473" timestamp="2026-04-21T19:49:51Z" user="E1ectros" uid="24018730">\n    <nd ref="2507400589" />\n    <nd ref="13340060055" />\n    <nd ref="1680667437" />\n    <nd ref="13472970566" />\n    <nd ref="13340060822" />\n    <nd ref="13472970567" />\n    <nd ref="13340060823" />\n    <nd ref="1680667519" />\n    <nd ref="13362936998" />\n    <tag k="highway" v="service" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="243314310" visible="true" version="5" changeset="185327145" timestamp="2026-07-08T09:10:09Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="8642866007" />\n    <nd ref="2507400591" />\n    <nd ref="1680667774" />\n    <nd ref="2505176020" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="243314312" visible="true" version="4" changeset="185327371" timestamp="2026-07-08T09:13:38Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="2507400638" />\n    <nd ref="13388478155" />\n    <nd ref="2507400628" />\n    <nd ref="8523326826" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243314378" visible="true" version="5" changeset="188760331" timestamp="2026-09-09T08:34:32Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="2507400702" />\n    <nd ref="2507400701" />\n    <nd ref="14166007986" />\n    <nd ref="2507400700" />\n    <nd ref="8523449430" />\n    <nd ref="4900465441" />\n    <nd ref="2507400696" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Thanon Rajyindee Soi 5" />\n  </way>\n  <way id="243314395" visible="true" version="3" changeset="177592009" timestamp="2026-01-23T04:33:30Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1680668226" />\n    <nd ref="1107926450" />\n    <nd ref="1107926968" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนสามชัย ซอย 3" />\n    <tag k="name:en" v="Samchai Road Soi 3" />\n    <tag k="name:th" v="ถนนสามชัย ซอย 3" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243314436" visible="true" version="5" changeset="185327371" timestamp="2026-07-08T09:13:38Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="2507400624" />\n    <nd ref="2507400618" />\n    <nd ref="2507400612" />\n    <nd ref="2507400604" />\n    <nd ref="13433676469" />\n    <nd ref="13449811511" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi Te Ta Cho" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="243314440" visible="true" version="2" changeset="72472571" timestamp="2019-07-21T01:59:52Z" user="JAAS" uid="3321532">\n    <nd ref="947896989" />\n    <nd ref="2505176520" />\n    <nd ref="2505176529" />\n    <nd ref="2505176570" />\n    <nd ref="2505176598" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Thanon Rajyindee Soi 3" />\n  </way>\n  <way id="243314467" visible="true" version="4" changeset="177274810" timestamp="2026-01-15T20:57:54Z" user="Callipo" uid="23344489">\n    <nd ref="2507400604" />\n    <nd ref="13458847530" />\n    <nd ref="2507400609" />\n    <nd ref="13388503445" />\n    <nd ref="2507400613" />\n    <nd ref="13388503443" />\n    <nd ref="2507400615" />\n    <nd ref="2507400610" />\n    <nd ref="13388478046" />\n    <nd ref="2507400605" />\n    <nd ref="13388478047" />\n    <nd ref="2505176027" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="243314469" visible="true" version="4" changeset="176056856" timestamp="2025-12-17T15:50:41Z" user="Wim Keijsers" uid="22692008">\n    <nd ref="2507400847" />\n    <nd ref="2507400845" />\n    <nd ref="2507400842" />\n    <nd ref="1140277056" />\n    <nd ref="13383672794" />\n    <nd ref="1140277055" />\n    <nd ref="2507400837" />\n    <tag k="highway" v="unclassified" />\n    <tag k="name" v="Thanon Banja" />\n  </way>\n  <way id="307427662" visible="true" version="3" changeset="177709773" timestamp="2026-01-26T08:19:02Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="3125684629" />\n    <nd ref="3125684630" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ตรอกจิระพัทธ์" />\n    <tag k="name:en" v="Trok Chirapat" />\n    <tag k="name:th" v="ตรอกจิระพัทธ์" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="323962802" visible="true" version="2" changeset="56270838" timestamp="2018-02-11T18:28:11Z" user="Megre" uid="6684068">\n    <nd ref="3307296644" />\n    <nd ref="5405819483" />\n    <nd ref="3307296645" />\n    <tag k="crossing" v="zebra" />\n    <tag k="footway" v="crossing" />\n    <tag k="highway" v="footway" />\n  </way>\n  <way id="324267161" visible="true" version="9" changeset="188857437" timestamp="2026-09-11T07:30:20Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="3309949761" />\n    <nd ref="8523343958" />\n    <nd ref="8523339090" />\n    <nd ref="14172689681" />\n    <nd ref="14148238620" />\n    <nd ref="14148238623" />\n    <nd ref="8523348885" />\n    <nd ref="3309949861" />\n    <nd ref="916327839" />\n    <tag k="country" v="ไทย" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="name" v="Thanon Prathanuthit" />\n    <tag k="name:th" v="ถนนประธานอุทิศ" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="325853661" visible="true" version="4" changeset="188451441" timestamp="2026-09-03T04:55:48Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="3324718280" />\n    <nd ref="14138865124" />\n    <nd ref="3324718281" />\n    <nd ref="3324721380" />\n    <nd ref="3324718282" />\n    <nd ref="3324718283" />\n    <nd ref="14138865123" />\n    <nd ref="3324718284" />\n    <nd ref="3324718285" />\n    <tag k="highway" v="service" />\n    <tag k="oneway" v="yes" />\n    <tag k="service" v="driveway" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="325853778" visible="true" version="2" changeset="188451504" timestamp="2026-09-03T04:58:39Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="3324717486" />\n    <nd ref="3324717487" />\n    <nd ref="3324717488" />\n    <nd ref="3324717489" />\n    <nd ref="3324717490" />\n    <nd ref="3324717491" />\n    <tag k="highway" v="service" />\n    <tag k="oneway" v="yes" />\n    <tag k="service" v="driveway" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="325853926" visible="true" version="1" changeset="28539201" timestamp="2015-02-01T08:09:22Z" user="Pathumthani" uid="2302962">\n    <nd ref="3324718905" />\n    <nd ref="3324718906" />\n    <tag k="highway" v="service" />\n    <tag k="oneway" v="yes" />\n    <tag k="service" v="driveway" />\n  </way>\n  <way id="325853927" visible="true" version="2" changeset="183219774" timestamp="2026-05-26T17:09:28Z" user="anthaas" uid="5438501">\n    <nd ref="3324717491" />\n    <nd ref="3324718905" />\n    <tag k="highway" v="service" />\n    <tag k="layer" v="-1" />\n    <tag k="oneway" v="yes" />\n    <tag k="tunnel" v="yes" />\n  </way>\n  <way id="325853929" visible="true" version="2" changeset="183219774" timestamp="2026-05-26T17:09:28Z" user="anthaas" uid="5438501">\n    <nd ref="3324718906" />\n    <nd ref="3324719149" />\n    <tag k="highway" v="service" />\n    <tag k="layer" v="-1" />\n    <tag k="oneway" v="yes" />\n    <tag k="tunnel" v="yes" />\n  </way>\n  <way id="325853964" visible="true" version="2" changeset="188455657" timestamp="2026-09-03T07:33:19Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="3324717336" />\n    <nd ref="3324717337" />\n    <tag k="highway" v="service" />\n    <tag k="oneway" v="yes" />\n    <tag k="service" v="driveway" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="325854199" visible="true" version="1" changeset="28539264" timestamp="2015-02-01T08:13:26Z" user="Pathumthani" uid="2302962">\n    <nd ref="3324721380" />\n    <nd ref="3324721381" />\n    <tag k="highway" v="footway" />\n    <tag k="lit" v="yes" />\n  </way>\n  <way id="325854200" visible="true" version="1" changeset="28539266" timestamp="2015-02-01T08:13:41Z" user="Pathumthani" uid="2302962">\n    <nd ref="3324721381" />\n    <nd ref="3324723462" />\n    <tag k="highway" v="footway" />\n    <tag k="tunnel" v="yes" />\n  </way>\n  <way id="330615760" visible="true" version="5" changeset="188455914" timestamp="2026-09-03T07:37:43Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="3376084776" />\n    <nd ref="12334158715" />\n    <nd ref="2506088024" />\n    <tag k="highway" v="residential" />\n    <tag k="maxspeed" v="30" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="330615766" visible="true" version="4" changeset="188455914" timestamp="2026-09-03T07:37:43Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="3376091310" />\n    <nd ref="3376084776" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="residential" />\n    <tag k="layer" v="1" />\n    <tag k="maxspeed" v="30" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="330615796" visible="true" version="1" changeset="29150293" timestamp="2015-02-28T07:49:47Z" user="Pathumthani" uid="2302962">\n    <nd ref="3376090057" />\n    <nd ref="3376090058" />\n    <tag k="highway" v="service" />\n    <tag k="service" v="driveway" />\n  </way>\n  <way id="330615806" visible="true" version="1" changeset="29150298" timestamp="2015-02-28T07:49:59Z" user="Pathumthani" uid="2302962">\n    <nd ref="3376091512" />\n    <nd ref="3376091513" />\n    <tag k="highway" v="service" />\n    <tag k="service" v="driveway" />\n  </way>\n  <way id="330615812" visible="true" version="2" changeset="183219774" timestamp="2026-05-26T17:09:28Z" user="anthaas" uid="5438501">\n    <nd ref="3376091513" />\n    <nd ref="3376090058" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="service" />\n    <tag k="layer" v="1" />\n  </way>\n  <way id="330615829" visible="true" version="4" changeset="177710672" timestamp="2026-01-26T08:49:43Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="947896964" />\n    <nd ref="12346099054" />\n    <nd ref="2506088009" />\n    <nd ref="3376091528" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์ ซอย 3" />\n    <tag k="name:en" v="Supasarnrangsan Road Soi 3" />\n    <tag k="name:th" v="ถนนศุภสารรังสรรค์ ซอย 3" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="330615834" visible="true" version="3" changeset="188456911" timestamp="2026-09-03T07:55:18Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="3376091818" />\n    <nd ref="1123399715" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="330615837" visible="true" version="3" changeset="188456911" timestamp="2026-09-03T07:55:18Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="3376084911" />\n    <nd ref="3376091818" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="330615844" visible="true" version="3" changeset="183219774" timestamp="2026-05-26T17:09:28Z" user="anthaas" uid="5438501">\n    <nd ref="3376091528" />\n    <nd ref="3376091818" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="layer" v="1" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="356023461" visible="true" version="1" changeset="32182188" timestamp="2015-06-24T12:18:30Z" user="Pathumthani" uid="2302962">\n    <nd ref="3615054036" />\n    <nd ref="3615054037" />\n    <nd ref="3615054038" />\n    <nd ref="3615054039" />\n    <tag k="highway" v="service" />\n    <tag k="service" v="driveway" />\n  </way>\n  <way id="356023480" visible="true" version="1" changeset="32182197" timestamp="2015-06-24T12:18:52Z" user="Pathumthani" uid="2302962">\n    <nd ref="3615056548" />\n    <nd ref="3615056549" />\n    <nd ref="3615056550" />\n    <nd ref="3615056551" />\n    <nd ref="3615056552" />\n    <tag k="highway" v="service" />\n    <tag k="service" v="driveway" />\n  </way>\n  <way id="356023646" visible="true" version="1" changeset="32182223" timestamp="2015-06-24T12:20:06Z" user="Pathumthani" uid="2302962">\n    <nd ref="3615054039" />\n    <nd ref="3615056548" />\n    <tag k="covered" v="yes" />\n    <tag k="highway" v="service" />\n    <tag k="service" v="driveway" />\n  </way>\n  <way id="40264696" visible="true" version="21" changeset="136194977" timestamp="2023-05-17T06:09:46Z" user="GrabTH_Napittha" uid="15509145">\n    <nd ref="3307271116" />\n    <nd ref="1108909723" />\n    <nd ref="1122072640" />\n    <nd ref="1123399906" />\n    <nd ref="1196113662" />\n    <nd ref="3125684630" />\n    <nd ref="617234535" />\n    <nd ref="617234280" />\n    <nd ref="617233014" />\n    <nd ref="2506085784" />\n    <nd ref="485580845" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="lanes" v="3" />\n    <tag k="name" v="ถนนนิพัทธ์อุทิศ 2" />\n    <tag k="name:de" v="Thanon Niphat Uthit 2" />\n    <tag k="name:en" v="Thanon Niphat Uthit 2" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="40264697" visible="true" version="31" changeset="186853397" timestamp="2026-08-03T07:28:04Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="485580840" />\n    <nd ref="620881529" />\n    <nd ref="12346123215" />\n    <nd ref="620881566" />\n    <nd ref="617234280" />\n    <nd ref="617234576" />\n    <nd ref="617234281" />\n    <nd ref="14069947700" />\n    <nd ref="1107784884" />\n    <nd ref="2506085496" />\n    <nd ref="2052547073" />\n    <nd ref="2506085499" />\n    <nd ref="1110416568" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="2" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนธรรมนูญวิถี" />\n    <tag k="name:de" v="Thamanoonvithee Rd" />\n    <tag k="name:en" v="Thammanun Withi Road" />\n    <tag k="name:th" v="ถนนธรรมนูญวิถี" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="452036922" visible="true" version="3" changeset="188459881" timestamp="2026-09-03T08:57:10Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1123399561" />\n    <nd ref="4488368090" />\n    <tag k="highway" v="tertiary" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์ ซอย 6" />\n    <tag k="name:en" v="Suphasanrangsan Road Soi 6" />\n    <tag k="name:th" v="ถนนศุภสารรังสรรค์ ซอย 6" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="452036923" visible="true" version="9" changeset="188350873" timestamp="2026-09-01T08:42:17Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="5149825984" />\n    <nd ref="4488368089" />\n    <nd ref="13472440774" />\n    <nd ref="4488366488" />\n    <nd ref="13472440773" />\n    <nd ref="4488368092" />\n    <nd ref="1123399584" />\n    <nd ref="1123399561" />\n    <nd ref="13472440764" />\n    <nd ref="13472440760" />\n    <nd ref="13472440763" />\n    <nd ref="13472440761" />\n    <nd ref="13472440762" />\n    <nd ref="1123399535" />\n    <nd ref="13472440759" />\n    <nd ref="13472440757" />\n    <nd ref="13472440758" />\n    <nd ref="1123399697" />\n    <nd ref="13473142237" />\n    <nd ref="4488358857" />\n    <nd ref="13472440752" />\n    <nd ref="1123399505" />\n    <nd ref="13472440753" />\n    <nd ref="13472440749" />\n    <nd ref="13472440754" />\n    <nd ref="1123399524" />\n    <nd ref="13472440751" />\n    <nd ref="13472440748" />\n    <nd ref="13472440750" />\n    <nd ref="1123399664" />\n    <nd ref="13472440756" />\n    <nd ref="13472440755" />\n    <nd ref="1123399493" />\n    <nd ref="1123399616" />\n    <nd ref="13472440747" />\n    <nd ref="1123399468" />\n    <nd ref="13472440746" />\n    <nd ref="1123399589" />\n    <nd ref="4488358858" />\n    <nd ref="1123399566" />\n    <nd ref="13472440765" />\n    <nd ref="1123399630" />\n    <nd ref="13472440768" />\n    <nd ref="13472440766" />\n    <nd ref="13472440767" />\n    <nd ref="13472483866" />\n    <nd ref="1123399478" />\n    <nd ref="13472440771" />\n    <nd ref="4488358856" />\n    <nd ref="13472440769" />\n    <nd ref="13472440770" />\n    <nd ref="13472440772" />\n    <nd ref="1123399576" />\n    <nd ref="4488368090" />\n    <nd ref="1123399866" />\n    <nd ref="13472483864" />\n    <nd ref="13472483865" />\n    <nd ref="1123399683" />\n    <tag k="highway" v="tertiary" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="452037066" visible="true" version="3" changeset="188461989" timestamp="2026-09-03T09:45:48Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1123399535" />\n    <nd ref="4488358856" />\n    <tag k="highway" v="tertiary" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์ ซอย 7" />\n    <tag k="name:en" v="Suphasanrangsan Road Soi 7" />\n    <tag k="name:th" v="ถนนศุภสารรังสรรค์ ซอย 7" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="452037067" visible="true" version="2" changeset="72474083" timestamp="2019-07-21T05:34:00Z" user="JAAS" uid="3321532">\n    <nd ref="4488358857" />\n    <nd ref="4488358858" />\n    <tag k="highway" v="residential" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="Thanon Supasarnrangsan Soi 8" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="48923856" visible="true" version="24" changeset="156669871" timestamp="2024-09-16T03:25:34Z" user="djmorkman" uid="5547549">\n    <nd ref="620881568" />\n    <nd ref="1612097191" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="lanes" v="3" />\n    <tag k="name" v="ถนนนิพัทธ์อุทิศ 1" />\n    <tag k="name:de" v="Thanon Nipat Uthid 1" />\n    <tag k="name:en" v="Thanon Nipat Uthid 1" />\n    <tag k="name:th" v="ถนนนิพัทธ์อุทิศ 1" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="497290260" visible="true" version="2" changeset="188031708" timestamp="2026-08-26T09:16:03Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="4888861605" />\n    <nd ref="4888862222" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="497290261" visible="true" version="3" changeset="188031708" timestamp="2026-08-26T09:16:03Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="4888862221" />\n    <nd ref="4888862222" />\n    <nd ref="4888862223" />\n    <nd ref="4888861606" />\n    <nd ref="4888861607" />\n    <nd ref="4888861608" />\n    <nd ref="4888861609" />\n    <nd ref="4888861610" />\n    <nd ref="1115585857" />\n    <nd ref="4888861616" />\n    <nd ref="4888861617" />\n    <nd ref="4888861618" />\n    <nd ref="4888861619" />\n    <nd ref="4888861611" />\n    <nd ref="4888861612" />\n    <nd ref="4888861613" />\n    <nd ref="4888861614" />\n    <nd ref="4888861615" />\n    <nd ref="4888861620" />\n    <nd ref="4888862221" />\n    <tag k="highway" v="tertiary" />\n    <tag k="junction" v="roundabout" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="497290262" visible="true" version="2" changeset="188031708" timestamp="2026-08-26T09:16:03Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="4888861608" />\n    <nd ref="2506089779" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="497290263" visible="true" version="3" changeset="188031708" timestamp="2026-08-26T09:16:03Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="4888861605" />\n    <nd ref="2505176867" />\n    <nd ref="2506089600" />\n    <nd ref="2506089545" />\n    <nd ref="1123399713" />\n    <nd ref="1123399740" />\n    <nd ref="1110416742" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="Thanon Jirnakorn" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="497290264" visible="true" version="3" changeset="188032201" timestamp="2026-08-26T09:26:47Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="4888861620" />\n    <nd ref="14124541207" />\n    <nd ref="4888861605" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="497290265" visible="true" version="2" changeset="188031708" timestamp="2026-08-26T09:16:03Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="4888861618" />\n    <nd ref="4888861604" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="497290266" visible="true" version="2" changeset="188031708" timestamp="2026-08-26T09:16:03Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="4888861604" />\n    <nd ref="4888861612" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="497290267" visible="true" version="2" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915">\n    <nd ref="4888861613" />\n    <nd ref="4888862224" />\n    <tag k="highway" v="service" />\n    <tag k="service" v="driveway" />\n  </way>\n  <way id="497290269" visible="true" version="2" changeset="188031708" timestamp="2026-08-26T09:16:03Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1123399676" />\n    <nd ref="14124519874" />\n    <nd ref="4888862230" />\n    <tag k="highway" v="tertiary" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="497371052" visible="true" version="7" changeset="187917775" timestamp="2026-08-24T04:58:47Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="4889402217" />\n    <nd ref="4889402216" />\n    <nd ref="13985351056" />\n    <nd ref="8642858903" />\n    <nd ref="13362936997" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="498671328" visible="true" version="3" changeset="188989640" timestamp="2026-09-14T04:03:52Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="2506088223" />\n    <nd ref="14179759489" />\n    <nd ref="4900465706" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="name" v="ถนนแสงศรี ซอย 1" />\n    <tag k="name:en" v="Sangsri Road Soi 1" />\n    <tag k="name:th" v="ถนนแสงศรี ซอย 1" />\n    <tag k="source" v="digitalglobe" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="498671329" visible="true" version="4" changeset="175471223" timestamp="2025-12-04T03:52:33Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="4900466295" />\n    <nd ref="4900465668" />\n    <nd ref="4900465182" />\n    <nd ref="8523334169" />\n    <nd ref="8523334173" />\n    <nd ref="2506088792" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="name" v="ถนนพูลสุวรรณ" />\n    <tag k="name:en" v="Poonsuwan Road" />\n    <tag k="name:th" v="ถนนพูลสุวรรณ" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="digitalglobe" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="498671330" visible="true" version="6" changeset="188857437" timestamp="2026-09-11T07:30:20Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="4900465092" />\n    <nd ref="4900465091" />\n    <nd ref="8523334176" />\n    <nd ref="14172689682" />\n    <nd ref="8523334172" />\n    <nd ref="4900465090" />\n    <nd ref="4900466296" />\n    <nd ref="4900466297" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="name" v="ถนนมุสลิม" />\n    <tag k="name:en" v="Moslem Road" />\n    <tag k="name:th" v="ถนนมุสลิม" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="digitalglobe" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="498671331" visible="true" version="3" changeset="188989640" timestamp="2026-09-14T04:03:52Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="2506088494" />\n    <nd ref="4900465198" />\n    <nd ref="2506088476" />\n    <tag k="highway" v="residential" />\n    <tag k="motorcar" v="no" />\n    <tag k="motorcycle" v="yes" />\n  </way>\n  <way id="498671388" visible="true" version="2" changeset="72472952" timestamp="2019-07-21T02:59:01Z" user="JAAS" uid="3321532">\n    <nd ref="2506086831" />\n    <nd ref="4900466448" />\n    <nd ref="856214538" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="name" v="Thanon Prachathipat 2" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671389" visible="true" version="5" changeset="189148524" timestamp="2026-09-17T09:00:53Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="4900464597" />\n    <nd ref="13473121453" />\n    <nd ref="14188730088" />\n    <nd ref="13351100149" />\n    <nd ref="4900464580" />\n    <nd ref="4900464598" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="name" v="Thanon Boonrong Uthit" />\n    <tag k="name:en" v="Thanon Boonrong Uthit" />\n    <tag k="name:th" v="ถนน บุญรองอุทิศ 2" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671391" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:55:59Z" user="RVR004" uid="5314349">\n    <nd ref="4900464587" />\n    <nd ref="4900464588" />\n    <nd ref="1123399771" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671393" visible="true" version="2" changeset="163432352" timestamp="2025-03-10T09:06:18Z" user="GrabTH_Praiya" uid="18100118">\n    <nd ref="4900465166" />\n    <nd ref="1115585909" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671395" visible="true" version="4" changeset="184639335" timestamp="2026-06-26T10:53:18Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1115585385" />\n    <nd ref="1966404349" />\n    <nd ref="13472440789" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="digitalglobe" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="498671427" visible="true" version="2" changeset="72474083" timestamp="2019-07-21T05:34:00Z" user="JAAS" uid="3321532">\n    <nd ref="1107926467" />\n    <nd ref="4900465052" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="name" v="ซอยทินกร" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671432" visible="true" version="3" changeset="72474083" timestamp="2019-07-21T05:34:00Z" user="JAAS" uid="3321532">\n    <nd ref="4900465067" />\n    <nd ref="4900465068" />\n    <nd ref="4900465069" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="lanes" v="2" />\n    <tag k="source" v="digitalglobe" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="498671433" visible="true" version="2" changeset="185328343" timestamp="2026-07-08T09:31:28Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="4900465070" />\n    <nd ref="4900465071" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n    <tag k="surface" v="unpaved" />\n  </way>\n  <way id="498671442" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:56:03Z" user="RVR004" uid="5314349">\n    <nd ref="4900465098" />\n    <nd ref="4900465099" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671465" visible="true" version="3" changeset="188290636" timestamp="2026-08-31T08:04:02Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="4900465166" />\n    <nd ref="12649445300" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="motorcar" v="no" />\n    <tag k="motorcycle" v="yes" />\n    <tag k="source" v="digitalglobe" />\n    <tag k="surface" v="unpaved" />\n  </way>\n  <way id="498671482" visible="true" version="2" changeset="150687772" timestamp="2024-04-30T04:01:15Z" user="RotBlau" uid="357344">\n    <nd ref="4900465208" />\n    <nd ref="4900465209" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="498671483" visible="true" version="2" changeset="184637996" timestamp="2026-06-26T10:14:01Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="4900465210" />\n    <nd ref="1680666686" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="498671497" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:56:07Z" user="RVR004" uid="5314349">\n    <nd ref="1558314142" />\n    <nd ref="4900465244" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671536" visible="true" version="4" changeset="135638961" timestamp="2023-05-03T02:45:09Z" user="GrabTH_Napittha" uid="15509145">\n    <nd ref="2506087432" />\n    <nd ref="4900465363" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="name" v="Soi 1/1 Sophasarnrungsen" />\n    <tag k="oneway" v="no" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671539" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:56:10Z" user="RVR004" uid="5314349">\n    <nd ref="4900465090" />\n    <nd ref="4900465369" />\n    <nd ref="4900465370" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671542" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:56:10Z" user="RVR004" uid="5314349">\n    <nd ref="2507400609" />\n    <nd ref="4900465377" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671544" visible="true" version="2" changeset="176715174" timestamp="2026-01-01T18:53:57Z" user="JohanHoogland" uid="6534559">\n    <nd ref="1140280757" />\n    <nd ref="13424220803" />\n    <nd ref="4900465379" />\n    <nd ref="13424210990" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671564" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:56:12Z" user="RVR004" uid="5314349">\n    <nd ref="4900465440" />\n    <nd ref="4900465441" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671566" visible="true" version="3" changeset="177550755" timestamp="2026-01-22T06:37:55Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="2506088707" />\n    <nd ref="13475596789" />\n    <nd ref="4900465446" />\n    <nd ref="13475596788" />\n    <nd ref="8523455250" />\n    <tag k="highway" v="service" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671569" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:56:12Z" user="RVR004" uid="5314349">\n    <nd ref="4900465451" />\n    <nd ref="2506087136" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671576" visible="true" version="2" changeset="176632477" timestamp="2025-12-30T15:05:11Z" user="Tharhtetaung" uid="16875287">\n    <nd ref="4900465471" />\n    <nd ref="4900465472" />\n    <nd ref="13418313524" />\n    <nd ref="4900465473" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671586" visible="true" version="7" changeset="185384607" timestamp="2026-07-09T04:46:21Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="8523344319" />\n    <nd ref="8523344320" />\n    <nd ref="4900465512" />\n    <nd ref="13386913506" />\n    <nd ref="4900465513" />\n    <tag k="highway" v="tertiary" />\n    <tag k="import" v="yes" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="digitalglobe" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="498671589" visible="true" version="2" changeset="72473869" timestamp="2019-07-21T05:10:52Z" user="JAAS" uid="3321532">\n    <nd ref="4900465517" />\n    <nd ref="2506088491" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="name" v="Thanon Sangsri Soi 3" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671600" visible="true" version="4" changeset="183223422" timestamp="2026-05-26T18:31:57Z" user="anthaas" uid="5438501">\n    <nd ref="1558314095" />\n    <nd ref="13884935873" />\n    <nd ref="4900465641" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671615" visible="true" version="2" changeset="185331124" timestamp="2026-07-08T10:24:19Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="4900465999" />\n    <nd ref="4900465665" />\n    <nd ref="1123399600" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="498671616" visible="true" version="4" changeset="188081802" timestamp="2026-08-27T07:44:26Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="4900465999" />\n    <nd ref="2506085631" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนประชาธิปัตย์ ซอย 3" />\n    <tag k="name:en" v="Prachathipat Road Soi 3" />\n    <tag k="name:th" v="ถนนประชาธิปัตย์ ซอย 3" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="498671639" visible="true" version="6" changeset="189138277" timestamp="2026-09-17T03:16:49Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="2506085631" />\n    <nd ref="4900465707" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="digitalglobe" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="498671673" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:56:19Z" user="RVR004" uid="5314349">\n    <nd ref="4900465989" />\n    <nd ref="4900465216" />\n    <nd ref="2506087522" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671676" visible="true" version="7" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915">\n    <nd ref="12346099054" />\n    <nd ref="13472452532" />\n    <nd ref="12334148769" />\n    <nd ref="4900465994" />\n    <nd ref="12334148770" />\n    <nd ref="4900465993" />\n    <nd ref="12334148771" />\n    <nd ref="4900465027" />\n    <nd ref="4900465995" />\n    <nd ref="4900465991" />\n    <nd ref="4900465996" />\n    <nd ref="12334148772" />\n    <nd ref="13472452530" />\n    <nd ref="13472452529" />\n    <nd ref="13472452531" />\n    <nd ref="4900465026" />\n    <nd ref="4900465992" />\n    <nd ref="4900465025" />\n    <nd ref="3376091512" />\n    <nd ref="13472452524" />\n    <nd ref="13472452525" />\n    <nd ref="12334158707" />\n    <nd ref="13472452528" />\n    <nd ref="6634655052" />\n    <nd ref="13472452527" />\n    <nd ref="12334158708" />\n    <nd ref="13472452526" />\n    <nd ref="6634655049" />\n    <nd ref="6634655046" />\n    <nd ref="12334158709" />\n    <nd ref="6634655047" />\n    <nd ref="12334158710" />\n    <nd ref="12334158711" />\n    <nd ref="6634655048" />\n    <nd ref="12334158712" />\n    <nd ref="12334148767" />\n    <nd ref="12334158713" />\n    <nd ref="12334148768" />\n    <nd ref="12334158714" />\n    <nd ref="2506088024" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671678" visible="true" version="2" changeset="185331124" timestamp="2026-07-08T10:24:19Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="2506085623" />\n    <nd ref="4900466001" />\n    <nd ref="4900466000" />\n    <nd ref="4900465708" />\n    <nd ref="4900465999" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="498671679" visible="true" version="1" changeset="49318745" timestamp="2017-06-06T20:56:19Z" user="RVR004" uid="5314349">\n    <nd ref="4900465219" />\n    <nd ref="4900466003" />\n    <nd ref="4900466002" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671694" visible="true" version="2" changeset="185383612" timestamp="2026-07-09T04:14:09Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="4900465030" />\n    <nd ref="4900465031" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="498671695" visible="true" version="3" changeset="103175191" timestamp="2021-04-19T07:33:31Z" user="bomma_1" uid="11659195">\n    <nd ref="4900465425" />\n    <nd ref="4900466225" />\n    <nd ref="8642662383" />\n    <nd ref="8642662384" />\n    <nd ref="8642662385" />\n    <nd ref="1431867155" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n  </way>\n  <way id="498671702" visible="true" version="5" changeset="188074512" timestamp="2026-08-27T03:04:54Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="12339345632" />\n    <nd ref="12339345628" />\n    <nd ref="12346099053" />\n    <nd ref="4900465688" />\n    <nd ref="2506088254" />\n    <tag k="highway" v="residential" />\n    <tag k="import" v="yes" />\n    <tag k="source" v="digitalglobe" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="530314558" visible="true" version="6" changeset="186904897" timestamp="2026-08-04T06:26:55Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="8932057454" />\n    <nd ref="8932057455" />\n    <nd ref="8932057456" />\n    <nd ref="8932057457" />\n    <nd ref="8932057458" />\n    <nd ref="8932057459" />\n    <nd ref="13472440788" />\n    <nd ref="13472440786" />\n    <nd ref="13472440787" />\n    <nd ref="8932057460" />\n    <nd ref="13579736619" />\n    <nd ref="8932057461" />\n    <nd ref="8932057462" />\n    <nd ref="8932057463" />\n    <nd ref="8932057464" />\n    <nd ref="13472440785" />\n    <nd ref="13472440782" />\n    <nd ref="13472440783" />\n    <nd ref="5149825981" />\n    <nd ref="13472440784" />\n    <nd ref="4900464587" />\n    <nd ref="1123399639" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="65080358" visible="true" version="28" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145">\n    <nd ref="856214571" />\n    <nd ref="2505176434" />\n    <nd ref="796747766" />\n    <nd ref="2507400748" />\n    <nd ref="4900465052" />\n    <nd ref="1680668226" />\n    <nd ref="2507400852" />\n    <nd ref="1680668239" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="2" />\n    <tag k="name" v="ถนนราษฎร์ยินดี" />\n    <tag k="name:en" v="Thanon Rajyindee" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n  </way>\n  <way id="66984604" visible="true" version="25" changeset="186694625" timestamp="2026-07-31T09:25:56Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="2406954571" />\n    <nd ref="14063449046" />\n    <nd ref="14063449049" />\n    <nd ref="4900465277" />\n    <nd ref="2406954575" />\n    <nd ref="1666900740" />\n    <nd ref="2507400677" />\n    <nd ref="8523328991" />\n    <nd ref="4900465339" />\n    <nd ref="809537820" />\n    <nd ref="11292971038" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="66984610" visible="true" version="14" changeset="143205428" timestamp="2023-10-27T10:48:34Z" user="GrabTH_Napittha" uid="15509145">\n    <nd ref="856214571" />\n    <nd ref="2505176258" />\n    <nd ref="11292971039" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n  </way>\n  <way id="66984613" visible="true" version="8" changeset="187917632" timestamp="2026-08-24T04:51:45Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="947896981" />\n    <nd ref="947896972" />\n    <nd ref="947896989" />\n    <nd ref="947896979" />\n    <nd ref="2505176388" />\n    <nd ref="2507400696" />\n    <nd ref="2507400689" />\n    <nd ref="947896958" />\n    <nd ref="2505176265" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนราษฎร์ยินดี 2" />\n    <tag k="name:en" v="Ratyindi 2 Road" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี 2" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="684704215" visible="true" version="2" changeset="72473287" timestamp="2019-07-21T03:47:56Z" user="JAAS" uid="3321532">\n    <nd ref="1431715369" />\n    <nd ref="617234535" />\n    <nd ref="620881544" />\n    <nd ref="1559016915" />\n    <nd ref="620881522" />\n    <nd ref="617234534" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Thanon Manasruedee" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n  </way>\n  <way id="684714761" visible="true" version="6" changeset="187921238" timestamp="2026-08-24T06:56:15Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2506085784" />\n    <nd ref="856214578" />\n    <nd ref="4900465513" />\n    <nd ref="2506085849" />\n    <nd ref="1107784879" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="Duangchan Road" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="684740716" visible="true" version="4" changeset="188455914" timestamp="2026-09-03T07:37:43Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1123399489" />\n    <nd ref="3376078960" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="684741141" visible="true" version="6" changeset="189038359" timestamp="2026-09-15T02:25:53Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1123399591" />\n    <nd ref="14182032117" />\n    <nd ref="14182032118" />\n    <nd ref="1123399675" />\n    <nd ref="1123399943" />\n    <nd ref="1123399486" />\n    <nd ref="1123399648" />\n    <nd ref="1123399749" />\n    <nd ref="1123399637" />\n    <nd ref="14182032119" />\n    <nd ref="1123399514" />\n    <nd ref="1123399730" />\n    <nd ref="1123399502" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="684741338" visible="true" version="2" changeset="188455914" timestamp="2026-09-03T07:37:43Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="3376078960" />\n    <nd ref="3376084911" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="685375581" visible="true" version="4" changeset="145861254" timestamp="2024-01-04T02:11:37Z" user="GrabTH_Praiya" uid="18100118">\n    <nd ref="2506085878" />\n    <nd ref="620881568" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="705855017" visible="true" version="5" changeset="187917632" timestamp="2026-08-24T04:51:45Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="14071847284" />\n    <nd ref="4900465949" />\n    <nd ref="1431867108" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706087699" visible="true" version="2" changeset="185383612" timestamp="2026-07-09T04:14:09Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1431867155" />\n    <nd ref="4900465031" />\n    <nd ref="1431867103" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706092523" visible="true" version="6" changeset="188517464" timestamp="2026-09-04T10:11:00Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="1110416444" />\n    <nd ref="13424220804" />\n    <nd ref="856214470" />\n    <nd ref="13383541790" />\n    <nd ref="14148398712" />\n    <nd ref="856214503" />\n    <nd ref="2052489649" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="Thanon Pracharak" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706093504" visible="true" version="5" changeset="178716277" timestamp="2026-02-18T09:37:15Z" user="neeli_srikanth" uid="12625709">\n    <nd ref="1966386254" />\n    <nd ref="13575308874" />\n    <nd ref="1107784879" />\n    <nd ref="12344163778" />\n    <nd ref="12344163776" />\n    <nd ref="1107784887" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Thanon Sangchan" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="706093505" visible="true" version="5" changeset="187921238" timestamp="2026-08-24T06:56:15Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1107784879" />\n    <nd ref="4900465099" />\n    <nd ref="8523334139" />\n    <nd ref="856214470" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="Duangchan Road" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706093506" visible="true" version="2" changeset="72473287" timestamp="2019-07-21T03:47:56Z" user="JAAS" uid="3321532">\n    <nd ref="1107784887" />\n    <nd ref="2052497441" />\n    <nd ref="4900465364" />\n    <nd ref="2506087593" />\n    <nd ref="2506087810" />\n    <nd ref="947896956" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Thanon Sangchan" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="706093508" visible="true" version="4" changeset="177551471" timestamp="2026-01-22T07:08:36Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="1966386254" />\n    <nd ref="4085817731" />\n    <nd ref="8523344319" />\n    <nd ref="617234843" />\n    <nd ref="485580842" />\n    <nd ref="617233014" />\n    <nd ref="620881567" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนประชาธิปัตย์" />\n    <tag k="name:en" v="Thanon Prachathipat" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706093509" visible="true" version="4" changeset="186901490" timestamp="2026-08-04T04:06:24Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="1107784884" />\n    <nd ref="4900465471" />\n    <nd ref="14071882650" />\n    <nd ref="4085817732" />\n    <nd ref="1966386254" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Thanon Sangchan" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706094740" visible="true" version="2" changeset="175444533" timestamp="2025-12-03T12:26:55Z" user="WorayutTK" uid="23562084">\n    <nd ref="1107926433" />\n    <nd ref="1558314144" />\n    <nd ref="1558314142" />\n    <nd ref="13351202525" />\n    <nd ref="1558314138" />\n    <nd ref="1558314134" />\n    <nd ref="2506085508" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ถนนสามชัย" />\n    <tag k="oneway" v="no" />\n  </way>\n  <way id="706094741" visible="true" version="4" changeset="177500038" timestamp="2026-01-21T00:41:37Z" user="talbertdoi" uid="23344915">\n    <nd ref="1123399516" />\n    <nd ref="13472483857" />\n    <nd ref="13472483856" />\n    <nd ref="13472483854" />\n    <nd ref="13472483855" />\n    <nd ref="13472483853" />\n    <nd ref="1123399641" />\n    <nd ref="13472440727" />\n    <nd ref="1123399483" />\n    <nd ref="1123399607" />\n    <nd ref="1123399659" />\n    <nd ref="1123399489" />\n    <tag k="highway" v="residential" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="Thanon Supasarnrangsan Soi 4/1" />\n    <tag k="oneway" v="no" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706094742" visible="true" version="2" changeset="155440101" timestamp="2024-08-19T02:40:08Z" user="GrabTH_Praiya" uid="18100118">\n    <nd ref="1558997825" />\n    <nd ref="1558314136" />\n    <nd ref="1558314112" />\n    <nd ref="1558314109" />\n    <nd ref="1558314107" />\n    <nd ref="1558314114" />\n    <nd ref="2506085508" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ถนนสามชัย" />\n    <tag k="oneway" v="no" />\n  </way>\n  <way id="706095305" visible="true" version="2" changeset="188989640" timestamp="2026-09-14T04:03:52Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="947896956" />\n    <nd ref="2506088476" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Thanon Sangchan" />\n  </way>\n  <way id="706095306" visible="true" version="1" changeset="72473287" timestamp="2019-07-21T03:47:56Z" user="JAAS" uid="3321532">\n    <nd ref="2506087987" />\n    <nd ref="2506087732" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="706095308" visible="true" version="2" changeset="101100223" timestamp="2021-03-16T07:43:59Z" user="oliviar_grab" uid="10664132">\n    <nd ref="6634630594" />\n    <nd ref="6634630596" />\n    <nd ref="8523307708" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="706096636" visible="true" version="9" changeset="176196082" timestamp="2025-12-20T20:47:05Z" user="RAytoun" uid="1414812">\n    <nd ref="485580845" />\n    <nd ref="7501768906" />\n    <nd ref="7401374741" />\n    <nd ref="1612097195" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n  </way>\n  <way id="706096637" visible="true" version="2" changeset="187967768" timestamp="2026-08-25T03:17:18Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1174378892" />\n    <nd ref="1196113662" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนปรีดารมย์" />\n    <tag k="name:en" v="Preedarom Road" />\n    <tag k="name:th" v="ถนนปรีดารมย์" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706096640" visible="true" version="5" changeset="185384563" timestamp="2026-07-09T04:44:44Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1122072681" />\n    <nd ref="2052547074" />\n    <nd ref="617234281" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนเสน่หานุสรณ์" />\n    <tag k="name:en" v="Saneha Nusorn Road" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706096642" visible="true" version="7" changeset="176196082" timestamp="2025-12-20T20:47:05Z" user="RAytoun" uid="1414812">\n    <nd ref="485580845" />\n    <nd ref="7500979776" />\n    <nd ref="4085816302" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706096643" visible="true" version="2" changeset="187967133" timestamp="2026-08-25T02:39:13Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1122072673" />\n    <nd ref="1431715421" />\n    <nd ref="1122072645" />\n    <nd ref="1122072640" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนนิยมรัฐ" />\n    <tag k="name:en" v="Niyomrat Road" />\n    <tag k="name:th" v="ถนนนิยมรัฐ" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706096644" visible="true" version="3" changeset="185384563" timestamp="2026-07-09T04:44:44Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1174378788" />\n    <nd ref="12722329099" />\n    <nd ref="1122072681" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนเสน่หานุสรณ์" />\n    <tag k="oneway" v="no" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706097788" visible="true" version="5" changeset="186901490" timestamp="2026-08-04T04:06:24Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="617234281" />\n    <nd ref="14069981565" />\n    <nd ref="1559029798" />\n    <nd ref="14071882647" />\n    <nd ref="617234843" />\n    <tag k="highway" v="tertiary" />\n    <tag k="maxspeed" v="20" />\n    <tag k="name" v="ถนนเสน่หานุสรณ์" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706102885" visible="true" version="5" changeset="177752629" timestamp="2026-01-27T08:34:12Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1160617590" />\n    <nd ref="2506090538" />\n    <nd ref="2506090556" />\n    <nd ref="13472222423" />\n    <nd ref="13472222424" />\n    <nd ref="2506090550" />\n    <nd ref="13472222401" />\n    <nd ref="13472222400" />\n    <nd ref="2506090533" />\n    <nd ref="13472222402" />\n    <nd ref="2506090412" />\n    <nd ref="13472156177" />\n    <nd ref="13472156176" />\n    <nd ref="13472156175" />\n    <nd ref="2506089779" />\n    <tag k="highway" v="residential" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706103887" visible="true" version="1" changeset="72474411" timestamp="2019-07-21T06:06:19Z" user="JAAS" uid="3321532">\n    <nd ref="2506087863" />\n    <nd ref="2506087889" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi Watthayorn" />\n    <tag k="oneway" v="no" />\n  </way>\n  <way id="706104217" visible="true" version="11" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275">\n    <nd ref="947896964" />\n    <nd ref="947896945" />\n    <nd ref="8523431062" />\n    <nd ref="8523431064" />\n    <nd ref="2506087962" />\n    <nd ref="12345104265" />\n    <nd ref="12346117424" />\n    <nd ref="12339345631" />\n    <nd ref="947896950" />\n    <nd ref="2506088002" />\n    <nd ref="12342535366" />\n    <nd ref="2506088024" />\n    <nd ref="12339345632" />\n    <nd ref="2506088084" />\n    <nd ref="12346117457" />\n    <nd ref="1140280758" />\n    <nd ref="8523406713" />\n    <nd ref="2506088039" />\n    <nd ref="1110416470" />\n    <nd ref="4900465517" />\n    <nd ref="947896956" />\n    <nd ref="2506087987" />\n    <nd ref="947896966" />\n    <nd ref="6634630594" />\n    <nd ref="947896997" />\n    <nd ref="985092700" />\n    <nd ref="985092239" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ถนนเเสงศรี" />\n    <tag k="name:en" v="Sangsri Road" />\n    <tag k="source" v="GPS" />\n  </way>\n  <way id="706104218" visible="true" version="3" changeset="183219774" timestamp="2026-05-26T17:09:28Z" user="anthaas" uid="5438501">\n    <nd ref="5149825984" />\n    <nd ref="1123399639" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="layer" v="1" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n  </way>\n  <way id="706104219" visible="true" version="10" changeset="189148524" timestamp="2026-09-17T09:00:53Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="1123399639" />\n    <nd ref="1115585393" />\n    <nd ref="947896971" />\n    <nd ref="13472440789" />\n    <nd ref="8523424940" />\n    <nd ref="4900464597" />\n    <nd ref="14188730091" />\n    <nd ref="2506086831" />\n    <nd ref="2506086795" />\n    <nd ref="1966399248" />\n    <nd ref="8523333972" />\n    <nd ref="1966399241" />\n    <nd ref="1140280757" />\n    <nd ref="1110416444" />\n    <nd ref="1110416654" />\n    <nd ref="1107784887" />\n    <nd ref="2506086181" />\n    <nd ref="856214496" />\n    <nd ref="12346141721" />\n    <nd ref="856214580" />\n    <nd ref="620881181" />\n    <nd ref="4085816302" />\n    <tag k="country" v="Thailand" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706105950" visible="true" version="2" changeset="109072743" timestamp="2021-08-03T07:25:29Z" user="GrabTH_Kulyada" uid="13325735">\n    <nd ref="1107926537" />\n    <nd ref="1107926934" />\n    <nd ref="1558997827" />\n    <nd ref="1558997826" />\n    <nd ref="1107926818" />\n    <nd ref="8976429009" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ถนนสามชัย" />\n  </way>\n  <way id="706105951" visible="true" version="5" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915">\n    <nd ref="13472440775" />\n    <nd ref="1107926513" />\n    <nd ref="1107926897" />\n    <nd ref="856214524" />\n    <nd ref="1107926639" />\n    <nd ref="1107926777" />\n    <nd ref="1107926537" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ถนนสามชัย" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706105952" visible="true" version="5" changeset="187917632" timestamp="2026-08-24T04:51:45Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2505176265" />\n    <nd ref="809537690" />\n    <nd ref="809537820" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706256377" visible="true" version="5" changeset="187927516" timestamp="2026-08-24T09:21:58Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1110416410" />\n    <nd ref="2052547075" />\n    <nd ref="1107784883" />\n    <nd ref="4900465209" />\n    <nd ref="2052547074" />\n    <nd ref="1176926303" />\n    <tag k="highway" v="tertiary" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนชีอุทิศ" />\n    <tag k="name:en" v="She U Tit Road" />\n    <tag k="name:th" v="ถนนชีอุทิศ" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706256379" visible="true" version="5" changeset="188339904" timestamp="2026-09-01T02:58:55Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="12722387505" />\n    <nd ref="1431715409" />\n    <nd ref="14138648129" />\n    <nd ref="1122072658" />\n    <nd ref="1107784878" />\n    <nd ref="4900465971" />\n    <nd ref="4900465208" />\n    <nd ref="12722387501" />\n    <nd ref="1122072681" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนกิมประดิษฐ์" />\n    <tag k="name:en" v="Kimpradit Road" />\n    <tag k="name:th" v="ถนนกิมประดิษฐ์" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706256380" visible="true" version="4" changeset="177753655" timestamp="2026-01-27T09:03:04Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1110416677" />\n    <nd ref="1680666119" />\n    <nd ref="2506085434" />\n    <nd ref="1680666322" />\n    <nd ref="1110416635" />\n    <nd ref="1110416379" />\n    <nd ref="1680666461" />\n    <nd ref="1110416498" />\n    <nd ref="1680666686" />\n    <nd ref="1559016919" />\n    <nd ref="1680666950" />\n    <nd ref="13339819888" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนละม้ายสงเคราะห์" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706256381" visible="true" version="4" changeset="177753602" timestamp="2026-01-27T09:02:11Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1160612969" />\n    <nd ref="13339819886" />\n    <nd ref="13339819893" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนเทพสงเคราะห์" />\n    <tag k="name:en" v="Thep Sonkhro Road" />\n    <tag k="name:th" v="ถนนเทพสงเคราะห์" />\n    <tag k="name:th-Latn" v="Thanon Tephsongkrew" />\n    <tag k="oneway" v="no" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706256383" visible="true" version="7" changeset="186899564" timestamp="2026-08-04T02:55:22Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1431715392" />\n    <nd ref="1431715458" />\n    <nd ref="1431715417" />\n    <nd ref="1110416556" />\n    <tag k="highway" v="residential" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706256385" visible="true" version="4" changeset="187969237" timestamp="2026-08-25T04:53:10Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1431715406" />\n    <nd ref="6634608142" />\n    <nd ref="1431715363" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนหอมนวล" />\n    <tag k="name:en" v="Homhaun Road" />\n    <tag k="name:th" v="ถนนหอมนวล" />\n    <tag k="name:th-Latn" v="Thamon Homhaun" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706256386" visible="true" version="3" changeset="187973629" timestamp="2026-08-25T07:21:01Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1559016918" />\n    <nd ref="1559016919" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนฉัยยากุลอุทิศ ซอย 1" />\n    <tag k="name:en" v="Chaiyakuluthit Road Soi 1" />\n    <tag k="name:th" v="ถนนฉัยยากุลอุทิศ ซอย 1" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="706520617" visible="true" version="6" changeset="163432352" timestamp="2025-03-10T09:06:18Z" user="GrabTH_Praiya" uid="18100118">\n    <nd ref="1115585416" />\n    <nd ref="2506085611" />\n    <nd ref="2506085619" />\n    <nd ref="12649445299" />\n    <nd ref="2506085622" />\n    <nd ref="1115585600" />\n    <nd ref="1123399846" />\n    <nd ref="1123399562" />\n    <nd ref="1115585792" />\n    <nd ref="1115585435" />\n    <nd ref="2052489649" />\n    <nd ref="2506085541" />\n    <nd ref="1115585744" />\n    <nd ref="1110416568" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Thumnoonvithi Road" />\n    <tag k="note" v="comditional oneway" />\n    <tag k="oneway" v="no" />\n  </way>\n  <way id="71380773" visible="true" version="16" changeset="137382428" timestamp="2023-06-15T17:15:11Z" user="Nithinat S" uid="10217273">\n    <nd ref="1431715366" />\n    <nd ref="6634608141" />\n    <nd ref="1431715363" />\n    <nd ref="1160612974" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="2" />\n    <tag k="name" v="ถนนธรรมนูญวิถี" />\n    <tag k="name:en" v="Thammanun Withi Road" />\n    <tag k="name:th" v="ถนนธรรมนูญวิถี" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="71380788" visible="true" version="3" changeset="72472571" timestamp="2019-07-21T01:59:52Z" user="JAAS" uid="3321532">\n    <nd ref="823954361" />\n    <nd ref="823954364" />\n    <nd ref="2505176800" />\n    <nd ref="2505176734" />\n    <nd ref="2505176747" />\n    <tag k="highway" v="residential" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="Thanon Rajyindee Soi 1/1" />\n  </way>\n  <way id="72016745" visible="true" version="27" changeset="189908183" timestamp="2026-10-03T03:44:58Z" user="dmatted" uid="3047966">\n    <nd ref="856214571" />\n    <nd ref="856214510" />\n    <nd ref="2506087296" />\n    <nd ref="1369020950" />\n    <nd ref="1666900747" />\n    <nd ref="13472440775" />\n    <nd ref="2506087136" />\n    <nd ref="5149825984" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="lanes" v="4" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Thanon Suphasanrangsan" />\n    <tag k="oneway" v="no" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="77967082" visible="true" version="8" changeset="187921306" timestamp="2026-08-24T06:58:20Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="916327777" />\n    <nd ref="4900465955" />\n    <nd ref="916327730" />\n    <nd ref="916327962" />\n    <tag k="highway" v="tertiary" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนเชื่อมรัฐอุทิศ" />\n    <tag k="name:en" v="Cheumrat Uthit Road" />\n    <tag k="name:th" v="ถนนเชื่อมรัฐอุทิศ" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="77967084" visible="true" version="20" changeset="183269728" timestamp="2026-05-27T18:24:44Z" user="anthaas" uid="5438501">\n    <nd ref="620881568" />\n    <nd ref="985092392" />\n    <nd ref="2506087114" />\n    <nd ref="13347665463" />\n    <nd ref="13347665462" />\n    <nd ref="13366620095" />\n    <nd ref="2506087487" />\n    <nd ref="2506087772" />\n    <nd ref="985092239" />\n    <nd ref="916327730" />\n    <nd ref="916327810" />\n    <nd ref="916327702" />\n    <nd ref="4900466295" />\n    <nd ref="3615054036" />\n    <nd ref="3615056552" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="1" />\n    <tag k="name" v="ถนนเพชรเกษม" />\n    <tag k="name:en" v="Phet Kasem Road" />\n    <tag k="oneway" v="yes" />\n    <tag k="unsigned_ref" v="สข.ถ3-0013" />\n  </way>\n  <way id="77967085" visible="true" version="8" changeset="187925615" timestamp="2026-08-24T08:41:34Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="916327824" />\n    <nd ref="8523326805" />\n    <nd ref="4900465181" />\n    <nd ref="8523326792" />\n    <nd ref="916327702" />\n    <tag k="highway" v="tertiary" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนมนตรี 2" />\n    <tag k="name:en" v="Montri 2 Road" />\n    <tag k="name:th" v="ถนนมนตรี 2" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="77967090" visible="true" version="19" changeset="141097449" timestamp="2023-09-11T08:04:08Z" user="GrabTH_Pattanapong" uid="14833323">\n    <nd ref="916327686" />\n    <nd ref="916327893" />\n    <nd ref="916327746" />\n    <nd ref="916327912" />\n    <nd ref="916327962" />\n    <nd ref="985092700" />\n    <nd ref="2506087709" />\n    <nd ref="2506087509" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="1" />\n    <tag k="lanes" v="3" />\n    <tag k="name" v="ถนนเพชรเกษม" />\n    <tag k="name:en" v="Phet Kasem Road" />\n    <tag k="oneway" v="yes" />\n    <tag k="unsigned_ref" v="สข.ถ3-0013" />\n  </way>\n  <way id="77967092" visible="true" version="7" changeset="187925704" timestamp="2026-08-24T08:43:32Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="916327810" />\n    <nd ref="8523326797" />\n    <nd ref="8523326798" />\n    <nd ref="916327792" />\n    <tag k="highway" v="tertiary" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนมนตรี 1" />\n    <tag k="name:en" v="Montri 1 Road" />\n    <tag k="name:th" v="ถนนมนตรี 1" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="77967093" visible="true" version="8" changeset="123152878" timestamp="2022-07-03T14:11:10Z" user="Nithinat S" uid="10217273">\n    <nd ref="916327888" />\n    <nd ref="916327950" />\n    <nd ref="916327952" />\n    <nd ref="916327684" />\n    <nd ref="916327686" />\n    <nd ref="916327943" />\n    <nd ref="916327857" />\n    <nd ref="916327906" />\n    <nd ref="916327801" />\n    <nd ref="916327804" />\n    <nd ref="916327891" />\n    <nd ref="916327826" />\n    <nd ref="916327921" />\n    <nd ref="3376085828" />\n    <nd ref="916327839" />\n    <nd ref="916327799" />\n    <nd ref="916327868" />\n    <nd ref="916327773" />\n    <nd ref="916327846" />\n    <nd ref="916327728" />\n    <nd ref="916327753" />\n    <nd ref="916327875" />\n    <nd ref="916327710" />\n    <nd ref="916327738" />\n    <nd ref="916327693" />\n    <nd ref="916327885" />\n    <nd ref="916327796" />\n    <nd ref="916327670" />\n    <nd ref="916327748" />\n    <nd ref="916327949" />\n    <nd ref="916327908" />\n    <nd ref="916327888" />\n    <tag k="direction" v="clockwise" />\n    <tag k="dual_carriage" v="yes" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="0" />\n    <tag k="junction" v="roundabout" />\n    <tag k="lanes" v="2" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="802258183" visible="true" version="4" changeset="177258805" timestamp="2026-01-15T13:12:36Z" user="arpprasad07" uid="23689086">\n    <nd ref="7504254568" />\n    <nd ref="7504272703" />\n    <nd ref="7504254572" />\n    <nd ref="7504272706" />\n    <nd ref="7504254574" />\n    <nd ref="7508595077" />\n    <nd ref="7504272714" />\n    <nd ref="7504272695" />\n    <nd ref="7504272696" />\n    <nd ref="7504254576" />\n    <nd ref="7504254579" />\n    <nd ref="7504254569" />\n    <nd ref="7504254570" />\n    <nd ref="13457981819" />\n    <nd ref="7504272688" />\n    <nd ref="7504272686" />\n    <nd ref="7508595069" />\n    <nd ref="7504272685" />\n    <nd ref="7504254581" />\n    <nd ref="7504254571" />\n    <nd ref="7504254583" />\n    <nd ref="7504254578" />\n    <nd ref="7504254577" />\n    <nd ref="7504272697" />\n    <nd ref="7504272694" />\n    <nd ref="7504272713" />\n    <nd ref="7504272700" />\n    <tag k="bicycle" v="no" />\n    <tag k="highway" v="footway" />\n    <tag k="horse" v="no" />\n  </way>\n  <way id="802258185" visible="true" version="2" changeset="85044207" timestamp="2020-05-11T20:17:29Z" user="zorree" uid="10282398">\n    <nd ref="7504254578" />\n    <nd ref="7504254580" />\n    <nd ref="7504272691" />\n    <nd ref="7504254584" />\n    <nd ref="7504272693" />\n    <nd ref="7504272687" />\n    <nd ref="7504272689" />\n    <nd ref="7504254579" />\n    <tag k="bicycle" v="no" />\n    <tag k="highway" v="footway" />\n    <tag k="horse" v="no" />\n  </way>\n  <way id="802258186" visible="true" version="2" changeset="85044207" timestamp="2020-05-11T20:17:29Z" user="zorree" uid="10282398">\n    <nd ref="7504254580" />\n    <nd ref="7504254582" />\n    <nd ref="7504254581" />\n    <tag k="bicycle" v="no" />\n    <tag k="highway" v="footway" />\n    <tag k="horse" v="no" />\n  </way>\n  <way id="802258187" visible="true" version="2" changeset="85044207" timestamp="2020-05-11T20:17:29Z" user="zorree" uid="10282398">\n    <nd ref="7504254582" />\n    <nd ref="7504254583" />\n    <tag k="bicycle" v="no" />\n    <tag k="highway" v="footway" />\n    <tag k="horse" v="no" />\n  </way>\n  <way id="802258188" visible="true" version="2" changeset="85044207" timestamp="2020-05-11T20:17:29Z" user="zorree" uid="10282398">\n    <nd ref="7504254584" />\n    <nd ref="7504272685" />\n    <tag k="bicycle" v="no" />\n    <tag k="highway" v="footway" />\n    <tag k="horse" v="no" />\n  </way>\n  <way id="802258192" visible="true" version="2" changeset="85044207" timestamp="2020-05-11T20:17:29Z" user="zorree" uid="10282398">\n    <nd ref="7504272692" />\n    <nd ref="7504272693" />\n    <tag k="bicycle" v="no" />\n    <tag k="highway" v="footway" />\n    <tag k="horse" v="no" />\n  </way>\n  <way id="802258206" visible="true" version="3" changeset="85044207" timestamp="2020-05-11T20:17:29Z" user="zorree" uid="10282398">\n    <nd ref="7504272700" />\n    <nd ref="7508595064" />\n    <nd ref="7508595062" />\n    <nd ref="7508595072" />\n    <nd ref="7504272723" />\n    <nd ref="7508595075" />\n    <nd ref="7504272725" />\n    <nd ref="7504272724" />\n    <nd ref="7508595073" />\n    <nd ref="7508595067" />\n    <nd ref="7508595063" />\n    <nd ref="7508595065" />\n    <nd ref="7504254571" />\n    <tag k="bicycle" v="no" />\n    <tag k="highway" v="footway" />\n    <tag k="horse" v="no" />\n  </way>\n  <way id="802697329" visible="true" version="2" changeset="85044207" timestamp="2020-05-11T20:17:29Z" user="zorree" uid="10282398">\n    <nd ref="7508595062" />\n    <nd ref="7508595063" />\n    <tag k="bicycle" v="no" />\n    <tag k="highway" v="footway" />\n    <tag k="horse" v="no" />\n  </way>\n  <way id="802697333" visible="true" version="2" changeset="85044207" timestamp="2020-05-11T20:17:29Z" user="zorree" uid="10282398">\n    <nd ref="7508595067" />\n    <nd ref="7508595068" />\n    <tag k="bicycle" v="no" />\n    <tag k="highway" v="footway" />\n    <tag k="horse" v="no" />\n  </way>\n  <way id="802697334" visible="true" version="2" changeset="85044207" timestamp="2020-05-11T20:17:29Z" user="zorree" uid="10282398">\n    <nd ref="7508595069" />\n    <nd ref="7508595070" />\n    <tag k="bicycle" v="no" />\n    <tag k="highway" v="footway" />\n    <tag k="horse" v="no" />\n  </way>\n  <way id="802697336" visible="true" version="2" changeset="85044207" timestamp="2020-05-11T20:17:29Z" user="zorree" uid="10282398">\n    <nd ref="7508595073" />\n    <nd ref="7508595074" />\n    <tag k="bicycle" v="no" />\n    <tag k="highway" v="footway" />\n    <tag k="horse" v="no" />\n  </way>\n  <way id="81350005" visible="true" version="12" changeset="177710672" timestamp="2026-01-26T08:49:43Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="947896971" />\n    <nd ref="8523424937" />\n    <nd ref="2506087550" />\n    <nd ref="947896961" />\n    <nd ref="947896964" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="Tiparat Road" />\n    <tag k="name:en" v="Suphasanrangsan Soi 3" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="81350006" visible="true" version="9" changeset="177514261" timestamp="2026-01-21T10:19:00Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1597535079" />\n    <nd ref="7691192946" />\n    <nd ref="947896995" />\n    <nd ref="4900465692" />\n    <nd ref="947896985" />\n    <nd ref="947896953" />\n    <nd ref="2507400773" />\n    <nd ref="947896981" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="Thanon Rajyindee Soi 1" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="84842971" visible="true" version="5" changeset="130155505" timestamp="2022-12-16T12:58:49Z" user="tuupola" uid="16456957">\n    <nd ref="809537378" />\n    <nd ref="809537797" />\n    <tag k="highway" v="secondary_link" />\n    <tag k="hwpi" v="0" />\n    <tag k="lanes" v="1" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="84842974" visible="true" version="31" changeset="188516048" timestamp="2026-09-04T09:37:04Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="2506087509" />\n    <nd ref="2506087462" />\n    <nd ref="8348961008" />\n    <nd ref="620881097" />\n    <nd ref="8523329816" />\n    <nd ref="7508595074" />\n    <nd ref="7508595068" />\n    <nd ref="2052522376" />\n    <nd ref="7508595070" />\n    <nd ref="4085816301" />\n    <nd ref="4085816302" />\n    <nd ref="856214578" />\n    <nd ref="485580842" />\n    <nd ref="1559029797" />\n    <nd ref="617234576" />\n    <nd ref="1176926303" />\n    <nd ref="1431715369" />\n    <nd ref="5405819483" />\n    <nd ref="1966412828" />\n    <nd ref="1122072645" />\n    <nd ref="1107773194" />\n    <nd ref="1107773241" />\n    <nd ref="3307300892" />\n    <nd ref="3307296462" />\n    <nd ref="14148305416" />\n    <nd ref="3307271117" />\n    <tag k="highway" v="tertiary" />\n    <tag k="lanes" v="3" />\n    <tag k="name" v="ถนนนิพัทธ์อุทิศ 3" />\n    <tag k="name:en" v="Thanon Nipatutid 3" />\n    <tag k="name:th" v="ถนนนิพัทธ์อุทิศ 3" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="898440288" visible="true" version="1" changeset="97972178" timestamp="2021-01-22T11:47:27Z" user="Dmitriev_DA" uid="9356997">\n    <nd ref="8348961008" />\n    <nd ref="8348961009" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="900244003" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915">\n    <nd ref="1107926513" />\n    <nd ref="8363308288" />\n    <nd ref="8363308289" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="900244004" visible="true" version="1" changeset="98153886" timestamp="2021-01-26T06:16:04Z" user="Dmitriev_DA" uid="9356997">\n    <nd ref="8363308288" />\n    <nd ref="8363308290" />\n    <nd ref="8363308291" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="911123067" visible="true" version="1" changeset="100026862" timestamp="2021-02-26T07:10:30Z" user="Dmitriev_DA" uid="9356997">\n    <nd ref="2506085849" />\n    <nd ref="2506086181" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Soi Duangchan" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="917695557" visible="true" version="1" changeset="101099622" timestamp="2021-03-16T07:36:22Z" user="oliviar_grab" uid="10664132">\n    <nd ref="8523328986" />\n    <nd ref="8523328985" />\n    <nd ref="8523328984" />\n    <nd ref="8523328983" />\n    <nd ref="8523328982" />\n    <nd ref="8523328981" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="917695719" visible="true" version="6" changeset="181599991" timestamp="2026-04-20T20:10:17Z" user="wireguy" uid="9564652">\n    <nd ref="8523326826" />\n    <nd ref="8523326825" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ซอยเตชะโต" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="917695829" visible="true" version="2" changeset="186689791" timestamp="2026-07-31T07:31:36Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1107926777" />\n    <nd ref="8523314493" />\n    <nd ref="8523314492" />\n    <nd ref="8523314491" />\n    <nd ref="8523314490" />\n    <nd ref="8523314489" />\n    <tag k="highway" v="service" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="917695954" visible="true" version="3" changeset="186689904" timestamp="2026-07-31T07:34:30Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="8523332291" />\n    <nd ref="8523332290" />\n    <nd ref="13351186096" />\n    <nd ref="8523332289" />\n    <nd ref="8523332288" />\n    <nd ref="8523332287" />\n    <nd ref="8523332286" />\n    <tag k="highway" v="service" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="917695955" visible="true" version="2" changeset="189138667" timestamp="2026-09-17T03:37:59Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="8523332293" />\n    <nd ref="14188450536" />\n    <nd ref="8523332292" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="917696153" visible="true" version="3" changeset="186690029" timestamp="2026-07-31T07:37:35Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1107926897" />\n    <nd ref="8523330992" />\n    <nd ref="14063278749" />\n    <tag k="highway" v="service" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="917696759" visible="true" version="1" changeset="101100007" timestamp="2021-03-16T07:41:08Z" user="oliviar_grab" uid="10664132">\n    <nd ref="8523333972" />\n    <nd ref="8523333971" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="917696883" visible="true" version="4" changeset="188345799" timestamp="2026-09-01T06:39:12Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="8523334139" />\n    <nd ref="13386931999" />\n    <nd ref="8523334137" />\n    <nd ref="8523334136" />\n    <nd ref="13424220807" />\n    <nd ref="13424220808" />\n    <tag k="highway" v="residential" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="917697169" visible="true" version="2" changeset="159193271" timestamp="2024-11-16T03:03:32Z" user="nabilersyad" uid="11419275">\n    <nd ref="8523329816" />\n    <nd ref="8523329815" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Thanon Lamaipradit" />\n  </way>\n  <way id="917698181" visible="true" version="2" changeset="188513991" timestamp="2026-09-04T08:51:03Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="8523348886" />\n    <nd ref="14148246304" />\n    <nd ref="14148246306" />\n    <nd ref="8523348885" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="917704692" visible="true" version="1" changeset="101102313" timestamp="2021-03-16T08:11:52Z" user="oliviar_grab" uid="10664132">\n    <nd ref="8523406713" />\n    <nd ref="8523406712" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="917705097" visible="true" version="2" changeset="177499671" timestamp="2026-01-21T00:14:55Z" user="talbertdoi" uid="23344915">\n    <nd ref="8523424940" />\n    <nd ref="8523424938" />\n    <nd ref="8523424937" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="917705248" visible="true" version="3" changeset="159211401" timestamp="2024-11-16T15:00:22Z" user="nabilersyad" uid="11419275">\n    <nd ref="8523431062" />\n    <nd ref="8523431061" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="917705249" visible="true" version="3" changeset="183219774" timestamp="2026-05-26T17:09:28Z" user="anthaas" uid="5438501">\n    <nd ref="8523431064" />\n    <nd ref="8523431063" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="917706194" visible="true" version="3" changeset="188756440" timestamp="2026-09-09T06:55:23Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="8523423312" />\n    <nd ref="14165547219" />\n    <nd ref="8523423311" />\n    <nd ref="14165547217" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="917706996" visible="true" version="1" changeset="101103145" timestamp="2021-03-16T08:22:42Z" user="oliviar_grab" uid="10664132">\n    <nd ref="8523449430" />\n    <nd ref="8523449429" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="917707580" visible="true" version="3" changeset="158286490" timestamp="2024-10-24T08:56:11Z" user="Madi-Red" uid="14550990">\n    <nd ref="4900465446" />\n    <nd ref="8523455248" />\n    <nd ref="8523455247" />\n    <nd ref="8523455246" />\n    <nd ref="8523455245" />\n    <nd ref="8523455244" />\n    <nd ref="8523455243" />\n    <nd ref="8523455242" />\n    <nd ref="8523455241" />\n    <nd ref="12284916698" />\n    <tag k="highway" v="service" />\n  </way>\n  <way id="931348308" visible="true" version="3" changeset="128209489" timestamp="2022-10-29T12:37:55Z" user="Nithinat S" uid="10217273">\n    <nd ref="4900466297" />\n    <nd ref="985092568" />\n    <nd ref="3376084894" />\n    <nd ref="916327921" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="1" />\n    <tag k="name" v="ถนนเพชรเกษม" />\n    <tag k="name:en" v="Phet Kasem Road" />\n    <tag k="oneway" v="yes" />\n    <tag k="unsigned_ref" v="สข.ถ3-0013" />\n  </way>\n  <way id="931348309" visible="true" version="3" changeset="128209489" timestamp="2022-10-29T12:37:55Z" user="Nithinat S" uid="10217273">\n    <nd ref="3615056552" />\n    <nd ref="4900466297" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="1" />\n    <tag k="name" v="ถนนเพชรเกษม" />\n    <tag k="name:en" v="Phet Kasem Road" />\n    <tag k="oneway" v="yes" />\n    <tag k="unsigned_ref" v="สข.ถ3-0013" />\n  </way>\n  <way id="931354569" visible="true" version="5" changeset="177551471" timestamp="2026-01-22T07:08:36Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="2506085690" />\n    <nd ref="13383541762" />\n    <nd ref="13383541761" />\n    <nd ref="856214503" />\n    <nd ref="13424220808" />\n    <nd ref="2506085647" />\n    <nd ref="1727934245" />\n    <nd ref="2506085628" />\n    <nd ref="13383541754" />\n    <nd ref="13383541755" />\n    <nd ref="1966386254" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนประชาธิปัตย์" />\n    <tag k="name:en" v="Prachathipat Road" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="931354570" visible="true" version="2" changeset="175411110" timestamp="2025-12-02T16:48:25Z" user="Julie Kongsparten" uid="23570745">\n    <nd ref="1115585792" />\n    <nd ref="13348166868" />\n    <nd ref="2506085690" />\n    <tag k="highway" v="residential" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="932324369" visible="true" version="1" changeset="103176847" timestamp="2021-04-19T07:55:06Z" user="bomma_1" uid="11659195">\n    <nd ref="8642733850" />\n    <nd ref="8642733849" />\n    <tag k="highway" v="residential" />\n  </way>\n  <way id="932339000" visible="true" version="3" changeset="177509915" timestamp="2026-01-21T08:28:25Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="1680667498" />\n    <nd ref="8642866004" />\n    <tag k="highway" v="service" />\n    <tag k="surface" v="concrete" />\n  </way>\n  <way id="932339001" visible="true" version="2" changeset="185032029" timestamp="2026-07-03T10:30:02Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="8642866006" />\n    <nd ref="1680667749" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="932339002" visible="true" version="4" changeset="186900335" timestamp="2026-08-04T03:24:25Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="8642866006" />\n    <nd ref="809537792" />\n    <nd ref="1680667498" />\n    <nd ref="14071847288" />\n    <nd ref="14071847283" />\n    <nd ref="1680666775" />\n    <nd ref="1431867160" />\n    <nd ref="8642858906" />\n    <nd ref="1680666013" />\n    <nd ref="4900465942" />\n    <nd ref="809537390" />\n    <nd ref="809537790" />\n    <nd ref="809537730" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="2" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนราษฎร์ยินดี" />\n    <tag k="name:en" v="Rajyindee road" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="932339003" visible="true" version="4" changeset="187971023" timestamp="2026-08-25T06:09:20Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="8642866009" />\n    <nd ref="8642866008" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="layer" v="1" />\n    <tag k="name" v="ถนนราษฎร์ยินดี ซอย 9" />\n    <tag k="name:en" v="Rajyindee Road Soi 9" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี ซอย 9" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="932339004" visible="true" version="5" changeset="187971023" timestamp="2026-08-25T06:09:20Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1431867160" />\n    <nd ref="8642866009" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนราษฎร์ยินดี ซอย 9" />\n    <tag k="name:en" v="Rajyindee Road Soi 9" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี ซอย 9" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="932339005" visible="true" version="2" changeset="186855356" timestamp="2026-08-03T08:11:55Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="8642866011" />\n    <nd ref="8642866010" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="residential" />\n    <tag k="layer" v="1" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="932339006" visible="true" version="2" changeset="186855356" timestamp="2026-08-03T08:11:55Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1680666775" />\n    <nd ref="8642866011" />\n    <tag k="highway" v="residential" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="932339889" visible="true" version="2" changeset="186855589" timestamp="2026-08-03T08:16:33Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="8642858905" />\n    <nd ref="8642858904" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="service" />\n    <tag k="layer" v="1" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="932339890" visible="true" version="2" changeset="186855589" timestamp="2026-08-03T08:16:33Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="8642858906" />\n    <nd ref="8642858905" />\n    <tag k="highway" v="service" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="932339891" visible="true" version="2" changeset="186855589" timestamp="2026-08-03T08:16:33Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="8642858904" />\n    <nd ref="8642858903" />\n    <nd ref="14069996590" />\n    <tag k="highway" v="service" />\n    <tag k="surface" v="paved" />\n  </way>\n  <way id="932368267" visible="true" version="4" changeset="186856349" timestamp="2026-08-03T08:32:17Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1110416655" />\n    <nd ref="8643114371" />\n    <nd ref="8643114370" />\n    <nd ref="8643114369" />\n    <nd ref="8643114368" />\n    <nd ref="8643114367" />\n    <tag k="highway" v="service" />\n    <tag k="surface" v="paving_stones" />\n  </way>\n  <way id="95533432" visible="true" version="12" changeset="177753655" timestamp="2026-01-27T09:03:04Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1110416454" />\n    <nd ref="2506085476" />\n    <nd ref="2506085475" />\n    <nd ref="1680667233" />\n    <nd ref="4900465219" />\n    <nd ref="1680667181" />\n    <nd ref="1680667148" />\n    <nd ref="13339819887" />\n    <nd ref="13339844003" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนหอยมุกต์" />\n    <tag k="name:en" v="Hoimook Road" />\n    <tag k="name:th" v="ถนนหอยมุกต์" />\n    <tag k="name:th-Latn" v="Thanon Hoimook" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="95533433" visible="true" version="10" changeset="187927395" timestamp="2026-08-24T09:18:39Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1107784878" />\n    <nd ref="1107784883" />\n    <nd ref="1107784884" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="Thanon Sangchan" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="95542480" visible="true" version="7" changeset="177514261" timestamp="2026-01-21T10:19:00Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="947896981" />\n    <nd ref="1680668223" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="layer" v="1" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="95542482" visible="true" version="17" changeset="190585279" timestamp="2026-10-16T04:21:56Z" user="cfgy" uid="11410742">\n    <nd ref="1107926481" />\n    <nd ref="1107926561" />\n    <nd ref="1107926940" />\n    <nd ref="13472452542" />\n    <nd ref="13472452543" />\n    <nd ref="14188069236" />\n    <nd ref="1107926845" />\n    <nd ref="1107926599" />\n    <nd ref="1107926977" />\n    <nd ref="1107926734" />\n    <nd ref="3324717336" />\n    <nd ref="3324717301" />\n    <nd ref="3324717486" />\n    <nd ref="3324718280" />\n    <nd ref="1110416526" />\n    <nd ref="2507400769" />\n    <nd ref="1110416469" />\n    <nd ref="2507400772" />\n    <nd ref="1110416754" />\n    <nd ref="1123399591" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="3" />\n    <tag k="lane_markings" v="yes" />\n    <tag k="name" v="ถนนจุติอนุสรณ์" />\n    <tag k="name:th" v="ถนนจุติอนุสรณ์" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="95542485" visible="true" version="12" changeset="187927888" timestamp="2026-08-24T09:30:02Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1107926481" />\n    <nd ref="1107926607" />\n    <nd ref="1107926905" />\n    <nd ref="14071937367" />\n    <nd ref="14071937372" />\n    <nd ref="14071937371" />\n    <nd ref="1107926563" />\n    <nd ref="1666900747" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนสามชัย" />\n    <tag k="name:en" v="Samchai Road" />\n    <tag k="name:th" v="ถนนสามชัย" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="95809258" visible="true" version="5" changeset="177514539" timestamp="2026-01-21T10:25:33Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1107926481" />\n    <nd ref="2505176697" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="layer" v="1" />\n    <tag k="name" v="ถนนสามชัย" />\n    <tag k="name:en" v="Samchai Rd" />\n    <tag k="name:th" v="ถนนสามชัย" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="95809261" visible="true" version="10" changeset="163428017" timestamp="2025-03-10T06:47:42Z" user="GrabTH_Panuwit" uid="16483995">\n    <nd ref="2506087432" />\n    <nd ref="2506087889" />\n    <nd ref="2506087968" />\n    <nd ref="2506088039" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Thanon Pracharak" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="95809262" visible="true" version="6" changeset="187928306" timestamp="2026-08-24T09:37:43Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1110416379" />\n    <nd ref="1559016903" />\n    <nd ref="1431715372" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนฉัยยากุลอุทิศ ซอย 4" />\n    <tag k="name:en" v="Chaiyakuluthit Road Soi 4" />\n    <tag k="name:th" v="ถนนฉัยยากุลอุทิศ ซอย 4" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="95809267" visible="true" version="7" changeset="188352318" timestamp="2026-09-01T09:14:55Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1110416631" />\n    <nd ref="12346099051" />\n    <nd ref="1110416745" />\n    <nd ref="1110416388" />\n    <nd ref="1110416470" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนสวนศิริ" />\n    <tag k="name:en" v="Suansiri Road" />\n    <tag k="name:th" v="ถนนสวนศิริ" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="95809268" visible="true" version="8" changeset="187929110" timestamp="2026-08-24T09:53:21Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1110416470" />\n    <nd ref="2506087955" />\n    <nd ref="2506087863" />\n    <nd ref="4900465972" />\n    <nd ref="4900465363" />\n    <nd ref="1110416528" />\n    <nd ref="1110416654" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์ ซอย 1" />\n    <tag k="name:en" v="Suphassrnrangsan Road Soi 1" />\n    <tag k="name:th" v="ถนนศุภสารรังสรรค์ ซอย 1" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="96242763" visible="true" version="12" changeset="187929230" timestamp="2026-08-24T09:56:37Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1115585393" />\n    <nd ref="1123399771" />\n    <nd ref="1115585403" />\n    <nd ref="13351118905" />\n    <nd ref="13545019681" />\n    <nd ref="1115585439" />\n    <nd ref="1115585732" />\n    <nd ref="1115585846" />\n    <nd ref="1115585690" />\n    <nd ref="4900464584" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนบุญรองอุทิศ" />\n    <tag k="name:en" v="Boonrong Uthit Road" />\n    <tag k="name:th" v="ถนนบุญรองอุทิศ" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="96713525" visible="true" version="6" changeset="177554268" timestamp="2026-01-22T08:36:48Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2505176470" />\n    <nd ref="1120012970" />\n    <nd ref="1120012973" />\n    <nd ref="1120012985" />\n    <nd ref="1120012977" />\n    <nd ref="1120012971" />\n    <nd ref="1120012968" />\n    <nd ref="1120012986" />\n    <nd ref="1120012979" />\n    <nd ref="1120012974" />\n    <nd ref="1120012981" />\n    <nd ref="1120012975" />\n    <nd ref="2505176388" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ถนนราฎร์ยินดี ซอย 4" />\n    <tag k="name:en" v="Rajyindee Road Soi 4" />\n    <tag k="name:th" v="ถนนราฎร์ยินดี ซอย 4" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="96713729" visible="true" version="8" changeset="187929348" timestamp="2026-08-24T09:59:00Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1966399241" />\n    <nd ref="1120015438" />\n    <nd ref="2506088084" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนแสงศรี ซอย 4" />\n    <tag k="name:en" v="Sangsri Road Soi 4" />\n    <tag k="name:th" v="ถนนแสงศรี ซอย 4" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="96837170" visible="true" version="9" changeset="189575613" timestamp="2026-09-26T08:15:30Z" user="Tonushka" uid="3419975">\n    <nd ref="1122072658" />\n    <nd ref="1122072706" />\n    <nd ref="13343573410" />\n    <nd ref="1559016917" />\n    <nd ref="1559016914" />\n    <nd ref="1122072692" />\n    <nd ref="1559016913" />\n    <nd ref="1122072661" />\n    <nd ref="1559016912" />\n    <nd ref="1122072702" />\n    <nd ref="1559016907" />\n    <nd ref="1122072674" />\n    <nd ref="1122072646" />\n    <nd ref="1122072686" />\n    <nd ref="1122072657" />\n    <nd ref="14063214196" />\n    <nd ref="1559016900" />\n    <nd ref="14063214200" />\n    <nd ref="14063214198" />\n    <nd ref="1559016896" />\n    <nd ref="1559016894" />\n    <nd ref="1559016897" />\n    <nd ref="1559016898" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนชนตารา" />\n    <tag k="name:en" v="Chontara Road" />\n    <tag k="name:th" v="ถนนชนตารา" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="96837172" visible="true" version="11" changeset="185760150" timestamp="2026-07-15T04:54:12Z" user="GrabTH_Monnapat" uid="24164407">\n    <nd ref="1174378788" />\n    <nd ref="1431715395" />\n    <nd ref="1122072673" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนเสน่หานุสรณ์" />\n    <tag k="oneway" v="no" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="96941841" visible="true" version="5" changeset="187967177" timestamp="2026-08-25T02:42:15Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2505176697" />\n    <nd ref="13472452546" />\n    <nd ref="13472452544" />\n    <nd ref="13472452547" />\n    <nd ref="13472452545" />\n    <nd ref="13472452548" />\n    <nd ref="1123399537" />\n    <nd ref="13472452553" />\n    <nd ref="13472452552" />\n    <nd ref="1123399700" />\n    <nd ref="1123399598" />\n    <nd ref="13472452551" />\n    <nd ref="13472452550" />\n    <nd ref="13472452549" />\n    <nd ref="1123399513" />\n    <nd ref="1110416567" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ซอยชุมอุทิศ 1" />\n    <tag k="name:en" v="Soi Chum Uthit 1" />\n    <tag k="name:th" v="ซอยชุมอุทิศ 1" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="96941842" visible="true" version="5" changeset="187967241" timestamp="2026-08-25T02:45:07Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1123399562" />\n    <nd ref="1123399809" />\n    <nd ref="12649445300" />\n    <nd ref="1123399706" />\n    <nd ref="1123399595" />\n    <nd ref="1966399248" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนประชาธิปัตย์ 1" />\n    <tag k="name:en" v="Prachathipat 1 Road" />\n    <tag k="name:th" v="ถนนประชาธิปัตย์ 1" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="96941844" visible="true" version="12" changeset="163428132" timestamp="2025-03-10T06:52:20Z" user="GrabTH_Panuwit" uid="16483995">\n    <nd ref="1431715409" />\n    <nd ref="1431715439" />\n    <nd ref="1110416325" />\n    <nd ref="1110416410" />\n    <tag k="highway" v="residential" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="96941849" visible="true" version="9" changeset="187967453" timestamp="2026-08-25T02:57:11Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1123399600" />\n    <nd ref="1123399917" />\n    <nd ref="1123399575" />\n    <nd ref="1123399852" />\n    <nd ref="1123399553" />\n    <nd ref="1123399778" />\n    <nd ref="1123399465" />\n    <nd ref="1123399586" />\n    <nd ref="1123399815" />\n    <nd ref="1123399540" />\n    <nd ref="13472452557" />\n    <nd ref="13472452558" />\n    <nd ref="1123399711" />\n    <nd ref="13472452556" />\n    <nd ref="13472452554" />\n    <nd ref="13472452555" />\n    <nd ref="5149825982" />\n    <nd ref="13472440780" />\n    <nd ref="13472440779" />\n    <nd ref="13472440781" />\n    <nd ref="13472440778" />\n    <nd ref="5149825983" />\n    <nd ref="5149825984" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์ ซอย 9" />\n    <tag k="name:en" v="Supasarnrangsan Soi 9 Rd" />\n    <tag k="name:th" v="ถนนศุภสารรังสรรค์ ซอย 9" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="96941857" visible="true" version="5" changeset="187967517" timestamp="2026-08-25T03:01:35Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1123399595" />\n    <nd ref="4900466449" />\n    <nd ref="1123399909" />\n    <nd ref="4900466448" />\n    <nd ref="1123399848" />\n    <nd ref="4900464580" />\n    <nd ref="1966404349" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนศุภสารรังสรรค์" />\n    <tag k="name:en" v="Supasarnrungsan Road Soi 2" />\n    <tag k="name:th" v="ถนนศุภสารรังสรรค์" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="969757849" visible="true" version="5" changeset="177551471" timestamp="2026-01-22T07:08:36Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="1115585385" />\n    <nd ref="13351118871" />\n    <nd ref="13351118872" />\n    <nd ref="4900464598" />\n    <nd ref="13336792952" />\n    <nd ref="856214538" />\n    <nd ref="1123399706" />\n    <nd ref="2506085690" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนประชาธิปัตย์" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="969757850" visible="true" version="2" changeset="177551471" timestamp="2026-01-22T07:08:36Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="8974100553" />\n    <nd ref="8974100552" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="layer" v="1" />\n    <tag k="name" v="ถนนประชาธิปัตย์" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="969757851" visible="true" version="6" changeset="188662419" timestamp="2026-09-07T10:20:56Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="1160612974" />\n    <nd ref="4900465413" />\n    <nd ref="8976428986" />\n    <nd ref="1123399587" />\n    <nd ref="14159239425" />\n    <nd ref="4900465957" />\n    <nd ref="2506085525" />\n    <nd ref="8976429009" />\n    <nd ref="9085920065" />\n    <nd ref="8976428989" />\n    <nd ref="849403384" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="2" />\n    <tag k="name" v="ถนนธรรมนูญวิถี" />\n    <tag k="name:en" v="Thammanun Withi Road" />\n    <tag k="name:th" v="ถนนธรรมนูญวิถี" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="969757852" visible="true" version="4" changeset="175179777" timestamp="2025-11-27T03:54:52Z" user="Earthkk" uid="9334949">\n    <nd ref="849403500" />\n    <nd ref="8974100554" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="2" />\n    <tag k="layer" v="1" />\n    <tag k="name" v="ถนนธรรมนูญวิถี" />\n    <tag k="name:en" v="Thammanun Withi Road" />\n    <tag k="name:th" v="ถนนธรรมนูญวิถี" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="969757853" visible="true" version="5" changeset="177593861" timestamp="2026-01-23T06:17:39Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="8974100554" />\n    <nd ref="8976429008" />\n    <nd ref="8976428992" />\n    <nd ref="13449811511" />\n    <nd ref="1107926840" />\n    <nd ref="8976429006" />\n    <nd ref="8523314742" />\n    <nd ref="2505176027" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="2" />\n    <tag k="name" v="ถนนธรรมนูญวิถี" />\n    <tag k="name:en" v="Thammanun Withi Road" />\n    <tag k="name:th" v="ถนนธรรมนูญวิถี" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="969757855" visible="true" version="3" changeset="187917632" timestamp="2026-08-24T04:51:45Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="8974100558" />\n    <nd ref="2507400638" />\n    <nd ref="8976429008" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="969757856" visible="true" version="2" changeset="187917632" timestamp="2026-08-24T04:51:45Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="8974100559" />\n    <nd ref="8974100558" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="layer" v="1" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="969765780" visible="true" version="3" changeset="188662419" timestamp="2026-09-07T10:20:56Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="8976429007" />\n    <nd ref="1123399587" />\n    <nd ref="14159239427" />\n    <nd ref="1123399466" />\n    <nd ref="1123399662" />\n    <nd ref="1115585416" />\n    <tag k="highway" v="residential" />\n    <tag k="name" v="Thumnoonvithi Road" />\n    <tag k="note" v="comditional oneway" />\n    <tag k="oneway" v="no" />\n  </way>\n  <way id="969765781" visible="true" version="5" changeset="187927516" timestamp="2026-08-24T09:21:58Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1431715406" />\n    <nd ref="1431715392" />\n    <tag k="highway" v="tertiary" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนชีอุทิศ" />\n    <tag k="name:en" v="She U Tit Road" />\n    <tag k="name:th" v="ถนนชีอุทิศ" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="969765782" visible="true" version="5" changeset="187927516" timestamp="2026-08-24T09:21:58Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1431715392" />\n    <nd ref="1110416410" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="lanes" v="2" />\n    <tag k="layer" v="1" />\n    <tag k="name" v="ถนนชีอุทิศ" />\n    <tag k="name:en" v="She U Tit Road" />\n    <tag k="name:th" v="ถนนชีอุทิศ" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="970010765" visible="true" version="4" changeset="177510550" timestamp="2026-01-21T08:44:58Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="8976428988" />\n    <nd ref="8976428977" />\n    <nd ref="1107926433" />\n    <nd ref="8976428978" />\n    <nd ref="8976428979" />\n    <nd ref="8976429007" />\n    <nd ref="8976428980" />\n    <nd ref="1558997825" />\n    <nd ref="8976428981" />\n    <nd ref="8976428982" />\n    <nd ref="13472974657" />\n    <nd ref="8976428984" />\n    <nd ref="13472974658" />\n    <nd ref="8976428985" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนธรรมนูญวิถี" />\n    <tag k="name:en" v="Thammanun Withi Road" />\n    <tag k="name:th" v="ถนนธรรมนูญวิถี" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="970010766" visible="true" version="2" changeset="137382428" timestamp="2023-06-15T17:15:11Z" user="Nithinat S" uid="10217273">\n    <nd ref="8976428990" />\n    <nd ref="8976428991" />\n    <tag k="bridge" v="yes" />\n    <tag k="highway" v="tertiary" />\n    <tag k="layer" v="1" />\n    <tag k="name" v="ถนนธรรมนูญวิถี" />\n    <tag k="name:en" v="Thammanun Withi Road" />\n    <tag k="name:th" v="ถนนธรรมนูญวิถี" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="970010767" visible="true" version="3" changeset="177593861" timestamp="2026-01-23T06:17:39Z" user="GrabTH_Bulakorn" uid="23192083">\n    <nd ref="8976428996" />\n    <nd ref="8976428994" />\n    <nd ref="8976428973" />\n    <nd ref="8976428993" />\n    <nd ref="8976428974" />\n    <nd ref="2505176020" />\n    <nd ref="8976428975" />\n    <nd ref="8976428976" />\n    <nd ref="2507400593" />\n    <nd ref="8976428990" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนธรรมนูญวิถี" />\n    <tag k="name:en" v="Thammanun Withi Road" />\n    <tag k="name:th" v="ถนนธรรมนูญวิถี" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="970311985" visible="true" version="4" changeset="177496146" timestamp="2026-01-20T21:34:53Z" user="talbertdoi" uid="23344915">\n    <nd ref="809537378" />\n    <nd ref="2507401050" />\n    <nd ref="2507401021" />\n    <nd ref="916327930" />\n    <nd ref="2507401012" />\n    <nd ref="2507400995" />\n    <nd ref="5405820222" />\n    <nd ref="916327945" />\n    <nd ref="5405820221" />\n    <nd ref="2507400919" />\n    <nd ref="916327808" />\n    <nd ref="1115585518" />\n    <nd ref="2507400879" />\n    <nd ref="2505177021" />\n    <nd ref="1160617588" />\n    <nd ref="8523461357" />\n    <nd ref="2505176923" />\n    <nd ref="4900465943" />\n    <nd ref="3309949961" />\n    <nd ref="916327674" />\n    <nd ref="916327888" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="1" />\n    <tag k="name" v="ถนนเพชรเกษม" />\n    <tag k="name:en" v="Phet Kasem Road" />\n    <tag k="oneway" v="yes" />\n    <tag k="unsigned_ref" v="สข.ถ3-0013" />\n  </way>\n  <way id="970323545" visible="true" version="2" changeset="187917632" timestamp="2026-08-24T04:51:45Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="2507400847" />\n    <nd ref="947896981" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="970323546" visible="true" version="3" changeset="143104733" timestamp="2023-10-25T09:27:31Z" user="GrabTH_Praiya" uid="18100118">\n    <nd ref="2505176985" />\n    <nd ref="1680668223" />\n    <nd ref="2505176514" />\n    <nd ref="2505176494" />\n    <nd ref="4900465939" />\n    <nd ref="2505176422" />\n    <nd ref="2507400698" />\n    <nd ref="2507400690" />\n    <nd ref="2505176258" />\n    <nd ref="2406954580" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="2" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนราษฎร์ยินดี" />\n    <tag k="name:en" v="Rajyindee road" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="980451721" visible="true" version="3" changeset="188031708" timestamp="2026-08-26T09:16:03Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1123399676" />\n    <nd ref="2506089522" />\n    <nd ref="2506089309" />\n    <nd ref="1107926977" />\n    <tag k="highway" v="tertiary" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="980451722" visible="true" version="5" changeset="188031708" timestamp="2026-08-26T09:16:03Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1123399676" />\n    <nd ref="1123399488" />\n    <nd ref="13472222397" />\n    <nd ref="13472222396" />\n    <nd ref="13472222398" />\n    <nd ref="1123399549" />\n    <nd ref="13472222399" />\n    <nd ref="13472222395" />\n    <nd ref="1123399840" />\n    <nd ref="1123399569" />\n    <nd ref="2507400792" />\n    <tag k="highway" v="tertiary" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="98567749" visible="true" version="7" changeset="187967665" timestamp="2026-08-25T03:10:52Z" user="GrabTH_Tanyamai" uid="23350424">\n    <nd ref="1140280758" />\n    <nd ref="2506087975" />\n    <nd ref="1140280757" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนแสงศรี ซอย 2" />\n    <tag k="name:en" v="Sangsri Road Soi 2" />\n    <tag k="name:th" v="ถนนแสงศรี ซอย 2" />\n    <tag k="oneway" v="yes" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="990212966" visible="true" version="5" changeset="189189346" timestamp="2026-09-18T03:56:45Z" user="GrabTH_Pimwipa" uid="24297485">\n    <nd ref="796720075" />\n    <nd ref="13335035783" />\n    <nd ref="8642306621" />\n    <nd ref="8642643269" />\n    <nd ref="14190776896" />\n    <nd ref="14190776892" />\n    <nd ref="14190776895" />\n    <nd ref="8643560679" />\n    <nd ref="8643560669" />\n    <nd ref="8642643253" />\n    <nd ref="796747582" />\n    <nd ref="796747714" />\n    <nd ref="1680667730" />\n    <nd ref="13351357081" />\n    <nd ref="8976428988" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="2" />\n    <tag k="name" v="ถนนราษฎร์ยินดี" />\n    <tag k="name:en" v="Thanon Rajyindee" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n  </way>\n  <way id="990212967" visible="true" version="2" changeset="137382428" timestamp="2023-06-15T17:15:11Z" user="Nithinat S" uid="10217273">\n    <nd ref="849403384" />\n    <nd ref="849403500" />\n    <tag k="highway" v="tertiary" />\n    <tag k="hwpi" v="2" />\n    <tag k="name" v="ถนนธรรมนูญวิถี" />\n    <tag k="name:en" v="Thammanun Withi Road" />\n    <tag k="name:th" v="ถนนธรรมนูญวิถี" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="990212968" visible="true" version="1" changeset="112147059" timestamp="2021-10-06T02:50:39Z" user="GrabTH_Kulyada" uid="13325735">\n    <nd ref="8976428987" />\n    <nd ref="8642866006" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="2" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนราษฎร์ยินดี" />\n    <tag k="name:en" v="Rajyindee road" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="990212970" visible="true" version="2" changeset="137382428" timestamp="2023-06-15T17:15:11Z" user="Nithinat S" uid="10217273">\n    <nd ref="8976428991" />\n    <nd ref="8976428987" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนธรรมนูญวิถี" />\n    <tag k="name:en" v="Thammanun Withi Road" />\n    <tag k="name:th" v="ถนนธรรมนูญวิถี" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="990212971" visible="true" version="1" changeset="112147059" timestamp="2021-10-06T02:50:39Z" user="GrabTH_Kulyada" uid="13325735">\n    <nd ref="849403500" />\n    <nd ref="8976428987" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="2" />\n    <tag k="lanes" v="2" />\n    <tag k="name" v="ถนนราษฎร์ยินดี" />\n    <tag k="name:en" v="Rajyindee road" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n    <tag k="surface" v="asphalt" />\n  </way>\n  <way id="990212972" visible="true" version="2" changeset="137382428" timestamp="2023-06-15T17:15:11Z" user="Nithinat S" uid="10217273">\n    <nd ref="8976428987" />\n    <nd ref="8976428988" />\n    <tag k="highway" v="tertiary" />\n    <tag k="name" v="ถนนธรรมนูญวิถี" />\n    <tag k="name:en" v="Thammanun Withi Road" />\n    <tag k="name:th" v="ถนนธรรมนูญวิถี" />\n    <tag k="oneway" v="yes" />\n  </way>\n  <way id="990212973" visible="true" version="1" changeset="112147059" timestamp="2021-10-06T02:50:39Z" user="GrabTH_Kulyada" uid="13325735">\n    <nd ref="8976428988" />\n    <nd ref="849403384" />\n    <tag k="highway" v="secondary" />\n    <tag k="hwpi" v="2" />\n    <tag k="name" v="ถนนราษฎร์ยินดี" />\n    <tag k="name:en" v="Thanon Rajyindee" />\n    <tag k="name:th" v="ถนนราษฎร์ยินดี" />\n    <tag k="oneway" v="yes" />\n    <tag k="source" v="GPS" />\n  </way>\n  <relation id="12588212" visible="true" version="1" changeset="103048954" timestamp="2021-04-16T09:50:13Z" user="Kesinee" uid="12597826">\n    <member type="way" ref="931348309" role="from" />\n    <member type="node" ref="4900466297" role="via" />\n    <member type="way" ref="498671330" role="to" />\n    <tag k="restriction" v="no_left_turn" />\n    <tag k="type" v="restriction" />\n  </relation>\n  <relation id="13049971" visible="true" version="2" changeset="175264223" timestamp="2025-11-29T06:48:49Z" user="Kri_ttt" uid="13669494">\n    <member type="way" ref="40264697" role="from" />\n    <member type="way" ref="243188465" role="to" />\n    <tag k="restriction" v="no_left_turn" />\n    <tag k="type" v="restriction" />\n  </relation>\n  <relation id="13054234" visible="true" version="1" changeset="109132857" timestamp="2021-08-04T08:37:06Z" user="GrabTH_Kulyada" uid="13325735">\n    <member type="way" ref="243188787" role="from" />\n    <member type="node" ref="809537378" role="via" />\n    <member type="way" ref="84842971" role="to" />\n    <tag k="restriction" v="no_right_turn" />\n    <tag k="type" v="restriction" />\n  </relation>\n  <relation id="13054411" visible="true" version="1" changeset="109136401" timestamp="2021-08-04T09:47:42Z" user="GrabTH_Kulyada" uid="13325735">\n    <member type="way" ref="243314469" role="from" />\n    <member type="node" ref="2507400847" role="via" />\n    <member type="way" ref="243314064" role="to" />\n    <tag k="restriction" v="no_right_turn" />\n    <tag k="type" v="restriction" />\n  </relation>\n  <relation id="13273283" visible="true" version="2" changeset="112147059" timestamp="2021-10-06T02:50:39Z" user="GrabTH_Kulyada" uid="13325735">\n    <member type="way" ref="990212971" role="from" />\n    <member type="way" ref="990212972" role="via" />\n    <member type="way" ref="990212973" role="to" />\n    <tag k="restriction" v="no_u_turn" />\n    <tag k="type" v="restriction" />\n  </relation>\n  <relation id="13273285" visible="true" version="2" changeset="112147059" timestamp="2021-10-06T02:50:39Z" user="GrabTH_Kulyada" uid="13325735">\n    <member type="way" ref="990212973" role="from" />\n    <member type="way" ref="990212967" role="via" />\n    <member type="way" ref="990212971" role="to" />\n    <tag k="restriction" v="no_u_turn" />\n    <tag k="type" v="restriction" />\n  </relation>\n  <relation id="13295605" visible="true" version="1" changeset="112147059" timestamp="2021-10-06T02:50:39Z" user="GrabTH_Kulyada" uid="13325735">\n    <member type="way" ref="990212967" role="from" />\n    <member type="way" ref="990212971" role="via" />\n    <member type="way" ref="990212972" role="to" />\n    <tag k="restriction" v="no_u_turn" />\n    <tag k="type" v="restriction" />\n  </relation>\n  <relation id="16278487" visible="true" version="1" changeset="140795016" timestamp="2023-09-04T09:29:09Z" user="GrabTH_Napittha" uid="15509145">\n    <member type="way" ref="96242763" role="from" />\n    <member type="node" ref="4900464584" role="via" />\n    <member type="way" ref="1204844051" role="to" />\n    <tag k="restriction" v="no_right_turn" />\n    <tag k="type" v="restriction" />\n  </relation>\n  <relation id="16558672" visible="true" version="1" changeset="143104733" timestamp="2023-10-25T09:27:31Z" user="GrabTH_Praiya" uid="18100118">\n    <member type="way" ref="1218225520" role="from" />\n    <member type="way" ref="1218225523" role="via" />\n    <member type="way" ref="1218225524" role="to" />\n    <tag k="restriction" v="no_u_turn" />\n    <tag k="type" v="restriction" />\n  </relation>\n  <relation id="17445577" visible="true" version="1" changeset="149747000" timestamp="2024-04-08T18:10:15Z" user="GrabTH_Napittha" uid="15509145">\n    <member type="way" ref="1270957863" role="from" />\n    <member type="node" ref="856214432" role="via" />\n    <member type="way" ref="1227721990" role="to" />\n    <tag k="restriction" v="no_right_turn" />\n    <tag k="type" v="restriction" />\n  </relation>\n  <relation id="18053324" visible="true" version="1" changeset="156711096" timestamp="2024-09-17T02:58:22Z" user="djmorkman" uid="5547549">\n    <member type="way" ref="685375581" role="from" />\n    <member type="node" ref="620881568" role="via" />\n    <member type="way" ref="1315837497" role="to" />\n    <tag k="restriction" v="no_straight_on" />\n    <tag k="type" v="restriction" />\n  </relation>\n  <relation id="18053325" visible="true" version="1" changeset="156711096" timestamp="2024-09-17T02:58:22Z" user="djmorkman" uid="5547549">\n    <member type="way" ref="685375581" role="from" />\n    <member type="node" ref="620881568" role="via" />\n    <member type="way" ref="48923856" role="to" />\n    <tag k="restriction" v="no_right_turn" />\n    <tag k="type" v="restriction" />\n  </relation>\n  <relation id="18069064" visible="true" version="1" changeset="156846444" timestamp="2024-09-20T02:29:38Z" user="djmorkman" uid="5547549">\n    <member type="way" ref="148108188" role="from" />\n    <member type="node" ref="620881568" role="via" />\n    <member type="way" ref="1315837497" role="to" />\n    <tag k="restriction" v="no_straight_on" />\n    <tag k="type" v="restriction" />\n  </relation>\n  <relation id="18069065" visible="true" version="1" changeset="156846444" timestamp="2024-09-20T02:29:38Z" user="djmorkman" uid="5547549">\n    <member type="way" ref="148108188" role="from" />\n    <member type="node" ref="620881568" role="via" />\n    <member type="way" ref="48923856" role="to" />\n    <tag k="restriction" v="no_right_turn" />\n    <tag k="type" v="restriction" />\n  </relation>\n  <relation id="18080764" visible="true" version="1" changeset="156975838" timestamp="2024-09-23T04:12:50Z" user="djmorkman" uid="5547549">\n    <member type="way" ref="706096636" role="from" />\n    <member type="node" ref="485580845" role="via" />\n    <member type="way" ref="706096636" role="to" />\n    <tag k="restriction" v="no_u_turn" />\n    <tag k="type" v="restriction" />\n  </relation>\n</osm>', 'data/provenance.json': '{\n  "source_url": "https://api.openstreetmap.org/api/0.6/map?bbox=100.470,7.001,100.483,7.014",\n  "requested_bbox_wgs84": [\n    100.47,\n    7.001,\n    100.483,\n    7.014\n  ],\n  "retrieved_utc": "2026-10-16T07:37:58Z",\n  "raw_sha256": "0be555b95dd9b81d3ee22995aea15fe8c41cae5760e64c35f750a391f22105de",\n  "additional_restriction_member_requests": [\n    {\n      "url": "https://api.openstreetmap.org/api/0.6/ways?ways=1315837497,148108188,243188787,243314064,243314469,48923856,685375581,706096636,84842971",\n      "sha256": "bb26b24f768c2d5d405b127ee44d7bf67c71a717398167ea1848d968aa9eb967",\n      "bytes": 4417\n    },\n    {\n      "url": "https://api.openstreetmap.org/api/0.6/nodes?nodes=1140277055,1140277056,12187377521,12187377522,13383672794,1369020941,1421027659,1421027788,1421027903,1421027965,1421027993,1612097191,1612097195,2505177476,2505177499,2506085878,2507400837,2507400842,2507400845,2507400897,2507400917,2507400921,2507400923,2507400926,2507401129,2507401162,7401374741,7501768906,809537392,809537797,809537800,809537807,9179528432,985092634",\n      "sha256": "2adcdcdb53988d693f4f5907fb72655fcdc401dc2cc65222752856897d5bda74",\n      "bytes": 6237\n    }\n  ],\n  "road_ways": 489,\n  "nodes": 1731,\n  "completed_restriction_relations": 15,\n  "data_attribution": "\\u00a9 OpenStreetMap contributors; ODbL 1.0",\n  "limits": [\n    "not a surveyed lane/signal inventory",\n    "full member ways may extend outside requested bbox",\n    "unmeasured demand"\n  ]\n}\n', 'data/network_audit.json': '{\n  "source_sha256": "17d8703340b57cb9d11dda3dedece5282a6b11142d1d2cf2319464011795d59f",\n  "network_sha256": "1bbfb6ee1a4a8a8ff718ade9c46232eba660007f92e055604828425db39a0a2a",\n  "sumo_version": "SUMO 1.25.0",\n  "road_edges": 1054,\n  "network_nodes": 479,\n  "signal_controllers": 7,\n  "green_stage_counts": [\n    3,\n    3,\n    3,\n    3,\n    4,\n    4,\n    5\n  ],\n  "source_highway_ways": 489,\n  "ways_with_explicit_maxspeed": 7,\n  "ways_with_explicit_lanes": 35,\n  "lanes_with_demo_speed_cap": 1156,\n  "demo_speed_cap_mps": 13.89,\n  "warnings": [\n    "Warning: to-edge \'498671330\' of restriction relation could not be determined",\n    "Warning: Ignoring restriction relation \'12588212\'.",\n    "Warning: Ignoring restriction relation \'13049971\' with unknown via.",\n    "Warning: from-edge \'243188787\' of restriction relation could not be determined",\n    "Warning: Ignoring restriction relation \'13054234\'.",\n    "Warning: from-edge \'243314469\' of restriction relation could not be determined",\n    "Warning: Ignoring restriction relation \'13054411\'.",\n    "Warning: direction of restriction relation could not be determined",\n    "Warning: Ignoring restriction relation \'13273283\'.",\n    "Warning: direction of restriction relation could not be determined",\n    "Warning: Ignoring restriction relation \'13273285\'.",\n    "Warning: direction of restriction relation could not be determined",\n    "Warning: direction of restriction relation could not be determined",\n    "Warning: from-edge \'685375581\' of restriction relation could not be determined",\n    "Warning: from-edge \'685375581\' of restriction relation could not be determined",\n    "Warning: from-edge \'148108188\' of restriction relation could not be determined",\n    "Warning: Not joining junctions 1107926734,2506089309 (parallel incoming -980451721#1,95542482#5).",\n    "Warning: Not joining junctions 1123399639,5149825984 (parallel incoming 530314558#1,96941849).",\n    "Warning: Reducing junction cluster 11292971032,11292971035,11292971036,11292971037,11292971038,11292971039,11292971040,2406954578,2406954579,2406954580,2505176258,2505176265,809537690,809537820,856214571 (parallel incoming 66984613#7,970323546#6).",\n    "Warning: Not joining junctions 1160612974,8976428985 (only 1 edge within and no cross-traffic).",\n    "Warning: Not joining junctions 12339345632,2506088024 (6 incoming edges).",\n    "Warning: Not joining junctions 2506087962,8523431064 (parallel incoming -917705249,243188208).",\n    "Warning: Reducing junction cluster 13362936997,1431867108,1431867160,8642866008,8642866009 (parallel incoming 705855017#1,932339002#4).",\n    "Warning: Reducing junction cluster 1559016905,1559016910,4900465210 (parallel incoming 142455590,142455595).",\n    "Warning: Reducing junction cluster 1680666775,4900465949,8642866010,8642866011 (parallel incoming 705855017#0,932339002#3).",\n    "Warning: Reducing junction cluster 1680667749,2507400591,8642866006,8642866007 (parallel incoming 1547102487#0,990212968).",\n    "Warning: Ambiguity in turnarounds computation at junction \'4488358858\'.",\n    "Warning: Found angle of 105.94 degrees at edge \'-142455604#1\', segment 3.",\n    "Warning: Found angle of 105.94 degrees at edge \'142455600#4\', segment 1.",\n    "Warning: Found angle of 151.88 degrees at edge \'497290265\', segment 0.",\n    "Warning: Found sharp turn with radius 3.58 at the start of edge \'497290265\'.",\n    "Warning: Replacing loaded roundabout \'497290261#0 497290261#1 497290261#2 497290261#3 497290261#4 497290261#5 497290261#6 497290261#7\' with \'497290261#1 497290261#2 497290261#3 497290261#4 497290261#5 497290261#7\'.",\n    "Warning: Replacing loaded roundabout \'1454593203#0 1454593203#1 1454593203#2 1454593203#3 1454593203#4\' with \'1454593203#1 1454593203#2 1454593203#3 1454593203#4\'.",\n    "Warning: Lane \'-684741338_0\' is not connected from any incoming edge at junction \'cluster_1123399715_3376084911_3376091818\'.",\n    "Warning: The traffic light \'2406954575\' does not control any links; it will not be build.",\n    "Warning: Could not build program \'0\' for traffic light \'2406954575\'",\n    "Warning: The traffic light \'2406954577\' does not control any links; it will not be build.",\n    "Warning: Could not build program \'0\' for traffic light \'2406954577\'",\n    "Warning: The traffic light \'620881544\' does not control any links; it will not be build.",\n    "Warning: Could not build program \'0\' for traffic light \'620881544\'",\n    "Warning: The traffic light \'620881566\' does not control any links; it will not be build.",\n    "Warning: Could not build program \'0\' for traffic light \'620881566\'",\n    "Warning: The traffic light \'916327945\' does not control any links; it will not be build.",\n    "Warning: Could not build program \'0\' for traffic light \'916327945\'",\n    "Warning: The traffic light \'cluster_1115585393_13472440789_947896971\' has a high cycle time of 100.00.",\n    "Warning: The traffic light \'cluster_849403384_849403500_8976428987_8976428988\' has a high cycle time of 100.00.",\n    "Warning: The traffic light \'joinedS_809537820_856214571\' has a high cycle time of 125.00.",\n    "Warning: Speed of straight connection \'243188710#2_0->95542482#0_0\' reduced by 6.04 due to turning radius of 47.62 (length=17.28, angle=36.15).",\n    "Warning: Speed of straight connection \'-95542482#0_0->-243188710#2_0\' reduced by 6.97 due to turning radius of 42.30 (length=15.31, angle=36.15).",\n    "Warning: Speed of straight connection \'-142453602#2_0->-142453602#1_0\' reduced by 5.96 due to turning radius of 48.09 (length=10.85, angle=28.15).",\n    "Warning: Speed of straight connection \'1457057267_0->129842391_0\' reduced by 6.50 due to turning radius of 24.26 (length=9.60, angle=38.46).",\n    "Warning: Speed of straight connection \'970010765#3_0->100401250#1_0\' reduced by 11.11 due to turning radius of 22.46 (length=9.73, angle=40.74).",\n    "Warning: Intersecting left turns at junction \'cluster_1558314142_1558314144\' from lane \'706094740#0_0\' and lane \'-498671497_0\' (increase junction radius to avoid this).",\n    "Warning: Intersecting left turns at junction \'cluster_2506088039_8523406713\' from lane \'95809261#2_0\' and lane \'-917704692_0\' (increase junction radius to avoid this).",\n    "Warning: Intersecting left turns at junction \'cluster_916327730_916327962_985092239_985092700\' from lane \'77967082#0_1\' and lane \'706104217#21_0\' (increase junction radius to avoid this).",\n    "Warning: 12 total messages of type: Ignoring restriction relation \'%\'.",\n    "Warning: 17 total messages of type: Not joining junctions % (%).",\n    "Warning: 7 total messages of type: Reducing junction cluster % (%).",\n    "Warning: 7 total messages of type: from-edge \'%\' of restriction relation could not be determined"\n  ],\n  "unresolved_restriction_warning_lines": 18,\n  "field_ready": false,\n  "signal_inventory": [\n    {\n      "label": "J01",\n      "controller_id": "2052547074",\n      "latitude": 7.003627070874894,\n      "longitude": 100.47160158188208,\n      "green_stages": 3,\n      "signal_plan_source": "SUMO-generated; not field verified",\n      "site_verified": false\n    },\n    {\n      "label": "J02",\n      "controller_id": "4085816302",\n      "latitude": 7.007659845965738,\n      "longitude": 100.47029259916944,\n      "green_stages": 3,\n      "signal_plan_source": "SUMO-generated; not field verified",\n      "site_verified": false\n    },\n    {\n      "label": "J03",\n      "controller_id": "947896964",\n      "latitude": 7.010370727315322,\n      "longitude": 100.47647690879944,\n      "green_stages": 3,\n      "signal_plan_source": "SUMO-generated; not field verified",\n      "site_verified": false\n    },\n    {\n      "label": "J04",\n      "controller_id": "cluster_1115585393_13472440789_947896971",\n      "latitude": 7.008899039143243,\n      "longitude": 100.47662903919118,\n      "green_stages": 4,\n      "signal_plan_source": "SUMO-generated; not field verified",\n      "site_verified": false\n    },\n    {\n      "label": "J05",\n      "controller_id": "cluster_849403384_849403500_8976428987_8976428988",\n      "latitude": 7.005140847144839,\n      "longitude": 100.48059672820655,\n      "green_stages": 4,\n      "signal_plan_source": "SUMO-generated; not field verified",\n      "site_verified": false\n    },\n    {\n      "label": "J06",\n      "controller_id": "cluster_916327730_916327962_985092239_985092700",\n      "latitude": 7.010634136925007,\n      "longitude": 100.47005147985605,\n      "green_stages": 3,\n      "signal_plan_source": "SUMO-generated; not field verified",\n      "site_verified": false\n    },\n    {\n      "label": "J07",\n      "controller_id": "joinedS_809537820_856214571",\n      "latitude": 7.009077053512301,\n      "longitude": 100.48132531162341,\n      "green_stages": 5,\n      "signal_plan_source": "SUMO-generated; not field verified",\n      "site_verified": false\n    }\n  ],\n  "status": "real road geometry; inferred lane/turn/signal model; synthetic traffic by default",\n  "attribution": "\\u00a9 OpenStreetMap contributors, ODbL 1.0",\n  "conversion_command": [\n    "/root/.local/lib/python3.12/site-packages/sumo/bin/netconvert",\n    "--osm-files",\n    "/workspace/scratch/020145e11beb/AdaptiveSignals_Stage3_RealRoad/data/hat_yai_roads.osm.xml",\n    "--output-file",\n    "/workspace/scratch/020145e11beb/AdaptiveSignals_Stage3_RealRoad/data/hat_yai.raw.net.xml",\n    "--lefthand",\n    "--geometry.remove",\n    "--junctions.join",\n    "--tls.guess-signals",\n    "--tls.discard-simple",\n    "--tls.join",\n    "--tls.layout",\n    "incoming",\n    "--tls.yellow.time",\n    "3",\n    "--tls.allred.time",\n    "2",\n    "--tls.green.time",\n    "20",\n    "--no-turnarounds",\n    "--keep-edges.by-vclass",\n    "passenger",\n    "--keep-edges.in-geo-boundary",\n    "100.470,7.001,100.483,7.014",\n    "--osm.turn-lanes",\n    "true",\n    "--output.street-names",\n    "true",\n    "--output.original-names",\n    "true"\n  ]\n}\n', 'data/signal_inventory.csv': 'label,controller_id,latitude,longitude,green_stages,signal_plan_source,site_verified\nJ01,2052547074,7.003627070874894,100.47160158188208,3,SUMO-generated; not field verified,False\nJ02,4085816302,7.007659845965738,100.47029259916944,3,SUMO-generated; not field verified,False\nJ03,947896964,7.010370727315322,100.47647690879944,3,SUMO-generated; not field verified,False\nJ04,cluster_1115585393_13472440789_947896971,7.008899039143243,100.47662903919118,4,SUMO-generated; not field verified,False\nJ05,cluster_849403384_849403500_8976428987_8976428988,7.005140847144839,100.48059672820655,4,SUMO-generated; not field verified,False\nJ06,cluster_916327730_916327962_985092239_985092700,7.010634136925007,100.47005147985605,3,SUMO-generated; not field verified,False\nJ07,joinedS_809537820_856214571,7.009077053512301,100.48132531162341,5,SUMO-generated; not field verified,False\n', 'data/entry_edges.csv': 'entry_edge_id,road_name,longitude,latitude,allowed_destination_count\n-1546907364,,100.47317263594033,7.001173023995683,15\n-155690968,,100.47782004096604,7.001160615785931,14\n-243188644,Soi Thongdee 3,100.47752270758924,7.015150474301877,16\n-243314137,,100.4748005629839,7.014283487666805,13\n-243314248#1,,100.48305170439683,7.006573478156663,19\n-498671539,,100.47045536861415,7.013096470972506,18\n-917706996,,100.4825059657412,7.010278309829284,18\n1257722739#2,,100.47374458956115,7.014261844718087,13\n1552748564#0,,100.47123308519062,7.011928024499651,14\n307427662,ตรอกจิระพัทธ์,100.46970494926393,7.002352687039977,14\n40264696#1,ถนนนิพัทธ์อุทิศ 2,100.47060001870716,7.000915479864999,14\n498671432,,100.47919781733567,7.0140659153215505,16\n66984604#1,ถนนศุภสารรังสรรค์,100.48525368209464,7.008794718384378,20\n706096642,ถนนศุภสารรังสรรค์,100.46968397079738,7.007561191012753,19\n84842974#0,ถนนนิพัทธ์อุทิศ 3,100.46976619175318,7.0095373837925905,17\n970323546#0,ถนนราษฎร์ยินดี,100.48235700265806,7.015378309952922,14\n', 'data/sensor_lane_mapping.csv': 'lane_id,controller_id,road_name,longitude,latitude,length_m\n-706256377#4_0,2052547074,ถนนชีอุทิศ,100.47154830412407,7.003637365830733,66.19\n706096640#0_0,2052547074,ถนนเสน่หานุสรณ์,100.47161033837811,7.003562023949667,72.87\n706256377#3_0,2052547074,ถนนชีอุทิศ,100.47165441049634,7.003617862485043,81.15\n706096642_0,4085816302,ถนนศุภสารรังสรรค์,100.47021255320483,7.00764870215167,59.27\n706104219#16_0,4085816302,ถนนศุภสารรังสรรค์,100.47038792675703,7.007666329994414,108.44\n84842974#7_0,4085816302,ถนนนิพัทธ์อุทิศ 3,100.47033448118611,7.007738751204457,78.9\n84842974#7_1,4085816302,ถนนนิพัทธ์อุทิศ 3,100.47030589411945,7.007743271742899,78.9\n84842974#7_2,4085816302,ถนนนิพัทธ์อุทิศ 3,100.47027721682595,7.007747882992531,78.9\n-330615829#0_0,947896964,ถนนศุภสารรังสรรค์ ซอย 3,100.4764895369709,7.010443212699586,20.45\n-706104217#1_0,947896964,ถนนเเสงศรี,100.47640875139331,7.010398522765352,15.1\n81350005#0_0,947896964,Tiparat Road,100.47646586029744,7.010311258923682,149.84\n-706104219#4_0,cluster_1115585393_13472440789_947896971,ถนนศุภสารรังสรรค์,100.47648061415522,7.008902761068942,44.23\n-81350005#2_0,cluster_1115585393_13472440789_947896971,Tiparat Road,100.47660317891089,7.0089646823595855,149.69\n498671395#1_0,cluster_1115585393_13472440789_947896971,,100.47653541236213,7.008800312229216,76.23\n706104219#0_0,cluster_1115585393_13472440789_947896971,ถนนศุภสารรังสรรค์,100.4768116852757,7.008897922519533,168.53\n1270957861_0,cluster_849403384_849403500_8976428987_8976428988,ถนนราษฎร์ยินดี,100.48067679431465,7.005215923258812,166.05\n1270957861_1,cluster_849403384_849403500_8976428987_8976428988,ถนนราษฎร์ยินดี,100.4806482081988,7.005220625232483,166.05\n969757851#4_0,cluster_849403384_849403500_8976428987_8976428988,ถนนธรรมนูญวิถี,100.48048243802506,7.005148170209952,77.07\n970010767#2_0,cluster_849403384_849403500_8976428987_8976428988,ถนนธรรมนูญวิถี,100.48072459378602,7.005133210002857,3.57\n990212966#2_0,cluster_849403384_849403500_8976428987_8976428988,ถนนราษฎร์ยินดี,100.48053141556635,7.005065905408386,21.38\n706104217#21_0,cluster_916327730_916327962_985092239_985092700,ถนนเเสงศรี,100.47017482586027,7.010512212283329,13.01\n77967082#0_0,cluster_916327730_916327962_985092239_985092700,ถนนเชื่อมรัฐอุทิศ,100.46996470150745,7.010756489698482,260.04\n77967082#0_1,cluster_916327730_916327962_985092239_985092700,ถนนเชื่อมรัฐอุทิศ,100.4699631626322,7.010727556745801,260.04\n77967090#1_0,cluster_916327730_916327962_985092239_985092700,ถนนเพชรเกษม,100.47018813539823,7.010744577058722,129.67\n77967090#1_1,cluster_916327730_916327962_985092239_985092700,ถนนเพชรเกษม,100.47016064901366,7.010753796567675,129.67\n77967090#1_2,cluster_916327730_916327962_985092239_985092700,ถนนเพชรเกษม,100.47013316262772,7.010763016075242,129.67\n-66984610#0_0,joinedS_809537820_856214571,ถนนศุภสารรังสรรค์,100.48126962536128,7.009227523874541,0.2\n-72016745#1_0,joinedS_809537820_856214571,ถนนศุภสารรังสรรค์,100.48118323653763,7.009300140297841,146.54\n-72016745#1_1,joinedS_809537820_856214571,ถนนศุภสารรังสรรค์,100.4811776258007,7.009271762939212,146.54\n1218225524#1_0,joinedS_809537820_856214571,ถนนราษฎร์ยินดี,100.4812102932434,7.0091550051418805,5.24\n66984604#1_0,joinedS_809537820_856214571,ถนนศุภสารรังสรรค์,100.48147189211988,7.008904955066819,418.04\n706105952#1_0,joinedS_809537820_856214571,,100.48143140747673,7.008953824526395,1.19\n', 'data/entry_counts_TEMPLATE.csv': 'start_s,end_s,entry_edge_id,vehicles\n', 'data/queue_observations_TEMPLATE.csv': 'time_s,lane_id,queue_vehicles,vehicles\n', 'data/EXAMPLE_SYNTHETIC_entry_counts.csv': 'start_s,end_s,entry_edge_id,vehicles,source_note\n0,60,706096642,12,SYNTHETIC EXAMPLE — not measured traffic\n', 'data/netconvert.log': "Success.\nWarning: to-edge '498671330' of restriction relation could not be determined\nWarning: Ignoring restriction relation '12588212'.\nWarning: Ignoring restriction relation '13049971' with unknown via.\nWarning: from-edge '243188787' of restriction relation could not be determined\nWarning: Ignoring restriction relation '13054234'.\nWarning: from-edge '243314469' of restriction relation could not be determined\nWarning: Ignoring restriction relation '13054411'.\nWarning: direction of restriction relation could not be determined\nWarning: Ignoring restriction relation '13273283'.\nWarning: direction of restriction relation could not be determined\nWarning: Ignoring restriction relation '13273285'.\nWarning: direction of restriction relation could not be determined\nWarning: direction of restriction relation could not be determined\nWarning: from-edge '685375581' of restriction relation could not be determined\nWarning: from-edge '685375581' of restriction relation could not be determined\nWarning: from-edge '148108188' of restriction relation could not be determined\nWarning: Not joining junctions 1107926734,2506089309 (parallel incoming -980451721#1,95542482#5).\nWarning: Not joining junctions 1123399639,5149825984 (parallel incoming 530314558#1,96941849).\nWarning: Reducing junction cluster 11292971032,11292971035,11292971036,11292971037,11292971038,11292971039,11292971040,2406954578,2406954579,2406954580,2505176258,2505176265,809537690,809537820,856214571 (parallel incoming 66984613#7,970323546#6).\nWarning: Not joining junctions 1160612974,8976428985 (only 1 edge within and no cross-traffic).\nWarning: Not joining junctions 12339345632,2506088024 (6 incoming edges).\nWarning: Not joining junctions 2506087962,8523431064 (parallel incoming -917705249,243188208).\nWarning: Reducing junction cluster 13362936997,1431867108,1431867160,8642866008,8642866009 (parallel incoming 705855017#1,932339002#4).\nWarning: Reducing junction cluster 1559016905,1559016910,4900465210 (parallel incoming 142455590,142455595).\nWarning: Reducing junction cluster 1680666775,4900465949,8642866010,8642866011 (parallel incoming 705855017#0,932339002#3).\nWarning: Reducing junction cluster 1680667749,2507400591,8642866006,8642866007 (parallel incoming 1547102487#0,990212968).\nWarning: Ambiguity in turnarounds computation at junction '4488358858'.\nWarning: Found angle of 105.94 degrees at edge '-142455604#1', segment 3.\nWarning: Found angle of 105.94 degrees at edge '142455600#4', segment 1.\nWarning: Found angle of 151.88 degrees at edge '497290265', segment 0.\nWarning: Found sharp turn with radius 3.58 at the start of edge '497290265'.\nWarning: Replacing loaded roundabout '497290261#0 497290261#1 497290261#2 497290261#3 497290261#4 497290261#5 497290261#6 497290261#7' with '497290261#1 497290261#2 497290261#3 497290261#4 497290261#5 497290261#7'.\nWarning: Replacing loaded roundabout '1454593203#0 1454593203#1 1454593203#2 1454593203#3 1454593203#4' with '1454593203#1 1454593203#2 1454593203#3 1454593203#4'.\nWarning: Lane '-684741338_0' is not connected from any incoming edge at junction 'cluster_1123399715_3376084911_3376091818'.\nWarning: The traffic light '2406954575' does not control any links; it will not be build.\nWarning: Could not build program '0' for traffic light '2406954575'\nWarning: The traffic light '2406954577' does not control any links; it will not be build.\nWarning: Could not build program '0' for traffic light '2406954577'\nWarning: The traffic light '620881544' does not control any links; it will not be build.\nWarning: Could not build program '0' for traffic light '620881544'\nWarning: The traffic light '620881566' does not control any links; it will not be build.\nWarning: Could not build program '0' for traffic light '620881566'\nWarning: The traffic light '916327945' does not control any links; it will not be build.\nWarning: Could not build program '0' for traffic light '916327945'\nWarning: The traffic light 'cluster_1115585393_13472440789_947896971' has a high cycle time of 100.00.\nWarning: The traffic light 'cluster_849403384_849403500_8976428987_8976428988' has a high cycle time of 100.00.\nWarning: The traffic light 'joinedS_809537820_856214571' has a high cycle time of 125.00.\nWarning: Speed of straight connection '243188710#2_0->95542482#0_0' reduced by 6.04 due to turning radius of 47.62 (length=17.28, angle=36.15).\nWarning: Speed of straight connection '-95542482#0_0->-243188710#2_0' reduced by 6.97 due to turning radius of 42.30 (length=15.31, angle=36.15).\nWarning: Speed of straight connection '-142453602#2_0->-142453602#1_0' reduced by 5.96 due to turning radius of 48.09 (length=10.85, angle=28.15).\nWarning: Speed of straight connection '1457057267_0->129842391_0' reduced by 6.50 due to turning radius of 24.26 (length=9.60, angle=38.46).\nWarning: Speed of straight connection '970010765#3_0->100401250#1_0' reduced by 11.11 due to turning radius of 22.46 (length=9.73, angle=40.74).\nWarning: Intersecting left turns at junction 'cluster_1558314142_1558314144' from lane '706094740#0_0' and lane '-498671497_0' (increase junction radius to avoid this).\nWarning: Intersecting left turns at junction 'cluster_2506088039_8523406713' from lane '95809261#2_0' and lane '-917704692_0' (increase junction radius to avoid this).\nWarning: Intersecting left turns at junction 'cluster_916327730_916327962_985092239_985092700' from lane '77967082#0_1' and lane '706104217#21_0' (increase junction radius to avoid this).\nWarning: 12 total messages of type: Ignoring restriction relation '%'.\nWarning: 17 total messages of type: Not joining junctions % (%).\nWarning: 7 total messages of type: Reducing junction cluster % (%).\nWarning: 7 total messages of type: from-edge '%' of restriction relation could not be determined\n", 'reference/real_pilot_audit.json': '{\n  "runs": 36,\n  "all_valid": true,\n  "all_metric_ledgers_agree": true,\n  "collision_events": 0,\n  "teleport_events": 0,\n  "cleared_runs_by_controller": {\n    "fixed": 7,\n    "sumo_actuated": 7,\n    "predictive": 1\n  },\n  "field_ready": false,\n  "demand_calibrated": false,\n  "site_signal_plans_verified": false,\n  "interpretation": "software integration on real road geometry, not evidence of field effectiveness"\n}\n'}
for name,text in PAYLOAD.items():
    p=PROJECT/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(text,encoding="utf-8")
if shutil.which("g++") is None:raise RuntimeError("Use a Linux CPU runtime with g++.")
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(PROJECT/"requirements.txt")],check=True)
subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-v"],cwd=PROJECT,check=True)
audit=json.loads((PROJECT/"data/network_audit.json").read_text())
print(f"Hat Yai model: {audit['road_edges']} directed road edges; {audit['signal_controllers']} signal controllers.")
print("Field ready: FALSE. Lane/turn rules and signal plans remain unverified.")
print("The Stage-2 discharge candidate cleared only 1/12 imported-model test streams; references cleared 7/12.")


In [ ]:
# Cell 2 — live Hat Yai road model. Keep the runtime connected.
# Optional: supply CSV paths AFTER mapping your permitted observations to data/*.csv.
COUNTS_CSV=os.environ.get("ADAPTIVE_SIGNALS_COUNTS_CSV", "")
QUEUES_CSV=os.environ.get("ADAPTIVE_SIGNALS_QUEUES_CSV", "")
PORT=int(os.environ.get("ADAPTIVE_SIGNALS_PORT","8765"))
if "LIVE_PROCESS" in globals() and LIVE_PROCESS.poll() is None:
    LIVE_PROCESS.terminate();LIVE_PROCESS.wait(timeout=10)
args=[sys.executable,"-u",str(PROJECT/"real_live.py"),"--port",str(PORT)]
if COUNTS_CSV:args += ["--counts-csv",str(Path(COUNTS_CSV).resolve())]
if QUEUES_CSV:args += ["--queues-csv",str(Path(QUEUES_CSV).resolve())]
LIVE_LOG=(PROJECT/"live_server.log").open("w")
LIVE_PROCESS=subprocess.Popen(args,cwd=PROJECT,stdout=LIVE_LOG,stderr=subprocess.STDOUT)
ready=False
for _ in range(100):
    if LIVE_PROCESS.poll() is not None:break
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/",timeout=1) as r:ready=b"Adaptive Signals" in r.read()
        if ready:break
    except OSError:time.sleep(.1)
if not ready:raise RuntimeError("Live startup failed. "+(PROJECT/"live_server.log").read_text()[-3000:])
# Display the original frontend directly in notebook output.
try:
    from google.colab import output
except ImportError:
    print(f'Open http://127.0.0.1:{PORT}/ in a local browser.')
else:
    import re
    from IPython.display import HTML, JSON, display

    local_origin = f'http://127.0.0.1:{PORT}/'
    with urllib.request.urlopen(local_origin, timeout=5) as connection:
        original_html = connection.read().decode('utf-8')
    key_pattern = r'const ACCESS_KEY=(.*?);'
    access_key = json.loads(re.search(key_pattern, original_html).group(1))

    def request_simulation(path, body=None):
        allowed_requests = {('scene', True), ('state', True), ('control', False)}
        if (path, body is None) not in allowed_requests:
            return JSON({'error': 'Unknown simulation request'})
        request = urllib.request.Request(
            local_origin + 'api/' + path,
            data=None if body is None else json.dumps(body).encode(),
            headers={'X-Simulation-Key': access_key, 'Content-Type': 'application/json'},
        )
        try:
            with urllib.request.urlopen(request, timeout=5) as reply:
                return JSON(json.load(reply))
        except urllib.error.HTTPError as error:
            return JSON(json.load(error))

    output.register_callback('hatyai_realroad_cell.request', request_simulation)
    replacement = """async function api(path, body) {
      const result = await google.colab.kernel.invokeFunction(
        'hatyai_realroad_cell.request', [path, body ?? null], {}
      );
      const value = result.data['application/json'];
      if (!value || value.error) throw new Error(value?.error || 'No simulator response');
      return value;
    }"""
    original_html = re.sub(key_pattern, '', original_html, count=1)
    original_html = chr(10).join(
        replacement if line.startswith('async function api(path,body)') else line
        for line in original_html.splitlines()
    )
    display(HTML(original_html))


In [ ]:
# Cell 3 — optional 36-run integration pilot / source export.
RUN_PILOT=os.environ.get("ADAPTIVE_SIGNALS_RUN_PILOT","0")=="1"  # set True to run
EXPORT_RESULTS=RUN_PILOT
if RUN_PILOT:
    OUTPUT=PROJECT/"runs/reproduced_real_pilot"
    subprocess.run([sys.executable,"-u",str(PROJECT/"real_pilot.py"),"--output",str(OUTPUT)],cwd=PROJECT,check=True)
    report=json.loads((OUTPUT/"audit.json").read_text())
else:
    report=json.loads((PROJECT/"reference/real_pilot_audit.json").read_text())
    print("Saved integration result; no new pilot run:")
print(json.dumps(report,indent=2))
print("Zero simulation errors are not field validation. No live CCTV or physical signals are connected.")
if EXPORT_RESULTS:
    archive=BASE/"AdaptiveSignals_HatYai_RealRoad_Results.zip"
    with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
        for p in PROJECT.rglob("*"):
            if not p.is_file():continue
            rel=p.relative_to(PROJECT)
            if "__pycache__" in rel.parts or "build" in rel.parts or str(rel).startswith("runs/live/"):continue
            if p.suffix==".tmp":continue
            z.write(p,Path(PROJECT.name)/rel)
    print("Exported:",archive)
    try:
        from google.colab import files
    except ImportError:pass
    else:files.download(str(archive))
